# Bibliotecas

In [1]:
# -----------------------------------
# Bibliotecas gerais
# -----------------------------------
import os
import time
import math
import json
import html
import warnings
import itertools
import getpass
import unicodedata
import re
import shutil
import requests
import gc

from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict, Counter
from itertools import combinations

# -----------------------------------
# Bibliotecas de dados
# -----------------------------------
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# -----------------------------------
# Visualização
# -----------------------------------
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px

# -----------------------------------
# Estatística / otimização
# -----------------------------------
from scipy import stats
from scipy.optimize import minimize

# -----------------------------------
# Banco de dados
# -----------------------------------
from sqlalchemy import create_engine, text

# -----------------------------------
# Dados externos
# -----------------------------------
import yfinance as yf

# -----------------------------------
# Configurações globais
# -----------------------------------
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.10f}")

plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["axes.grid"] = True

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


# Configurações banco de dados

In [2]:
# Tenta recuperar a conexão com o banco, se definido antes
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "MF"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

    print("Configuração do Banco de Dados criada com Sucesso!!!")

else:
    print("Configuração já existe. Usando configuração já existente!!!")

# Teste de conexão
try:
    with db_engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Conexão com o banco validada com sucesso!")
except Exception as e:
    print("Falha ao validar conexão com o banco.")
    print(f"Erro: {e}")

Digite seu usuário MySQL:  tomida
Digite sua senha MySQL (não será exibida):  ········


Configuração do Banco de Dados criada com Sucesso!!!
Conexão com o banco validada com sucesso!


# Etapa 1) Estrutura Inicial e Configurações do Projeto

## Etapa 1.1) Estrutura de Pastas e Parâmetros Globais

In [3]:
%%time
# ============================================================
# Etapa 1.1) Estrutura de Pastas e Parâmetros Globais
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 1.1 - ESTRUTURA DE PASTAS E PARÂMETROS GLOBAIS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/9] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")

if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")

print("OK")

# ============================================================
# 2) Parâmetros estruturais do projeto
# ============================================================

print("\n[2/9] Parâmetros estruturais do projeto...")

N_ETAPAS_PROJETO = 15

DIRETORIO_RAIZ_PROJETO = Path.cwd()
DIRETORIO_RESULTADOS = DIRETORIO_RAIZ_PROJETO / "resultados"
DIRETORIO_HTML_FINAL = DIRETORIO_RAIZ_PROJETO / "html_final"

print(f"Diretório raiz identificado : {DIRETORIO_RAIZ_PROJETO}")
print(f"Diretório de resultados     : {DIRETORIO_RESULTADOS}")
print(f"Diretório do HTML final     : {DIRETORIO_HTML_FINAL}")
print("OK")

# ============================================================
# 3) Dicionário central de diretórios
# ============================================================

print("\n[3/9] Dicionário central de diretórios...")

DIRETORIOS_PROJETO = {
    "raiz_projeto": DIRETORIO_RAIZ_PROJETO,
    "resultados": DIRETORIO_RESULTADOS,
    "html_final": DIRETORIO_HTML_FINAL,
    "html_assets": DIRETORIO_HTML_FINAL / "assets",
    "html_css": DIRETORIO_HTML_FINAL / "css",
    "html_js": DIRETORIO_HTML_FINAL / "js",
    "html_dados": DIRETORIO_HTML_FINAL / "dados",
    "html_img": DIRETORIO_HTML_FINAL / "img",
}

for etapa in range(1, N_ETAPAS_PROJETO + 1):
    DIRETORIOS_PROJETO[f"etapa_{etapa}"] = DIRETORIO_RESULTADOS / f"etapa_{etapa}"

print(f"Quantidade total de diretórios mapeados: {len(DIRETORIOS_PROJETO)}")
print("OK")

# ============================================================
# 4) Criação física da estrutura de pastas
# ============================================================

print("\n[4/9] Criação física da estrutura de pastas...")

for chave_diretorio, caminho_diretorio in DIRETORIOS_PROJETO.items():
    Path(caminho_diretorio).mkdir(parents=True, exist_ok=True)

print("Estrutura física de diretórios criada ou validada com sucesso.")
print("OK")

# ============================================================
# 5) Convenções de nomenclatura dos arquivos
# ============================================================

print("\n[5/9] Convenções de nomenclatura dos arquivos...")

PADROES_NOMENCLATURA = {
    "tbl": "{etapa}_{subetapa}_tbl_{nome}.parquet",
    "base": "{etapa}_{subetapa}_base_{nome}.parquet",
    "auditoria": "{etapa}_{subetapa}_auditoria_{nome}.parquet",
    "grafico": "{etapa}_{subetapa}_grafico_{nome}.png",
}

print("Padrões de nomenclatura definidos.")
print("OK")

# ============================================================
# 6) Parâmetros globais do projeto
# ============================================================

print("\n[6/9] Parâmetros globais do projeto...")

PARAMETROS_GLOBAIS = {
    "capital_inicial_estrategia": 100000,
    "cooldown_sinais_pregoes": 63,
    "codigo_sgs_selic_diaria": 11,
    "codigo_sgs_cdi_diaria": 12,
    "benchmark_renda_variavel": "^BVSP",
    "preco_principal_acoes": "close_adj",
    "chave_empresa": "issuer_code",
    "chave_ativo": "ticker",
    "formato_padrao_tabelas": "parquet",
    "formato_padrao_graficos": "png",
}

print(f"Quantidade de parâmetros globais definidos: {len(PARAMETROS_GLOBAIS)}")
print("OK")

# ============================================================
# 7) Funções auxiliares de padronização e salvamento
# ============================================================

print("\n[7/9] Funções auxiliares de padronização e salvamento...")

def padronizar_nome_arquivo(texto):
    """
    Padroniza o nome lógico de um arquivo para o padrão do projeto.
    """
    texto = str(texto).strip().lower()
    texto = texto.replace(" ", "_").replace("-", "_")
    while "__" in texto:
        texto = texto.replace("__", "_")
    return texto

def gerar_nome_arquivo(etapa, subetapa, tipo_arquivo, nome):
    """
    Gera o nome final do arquivo com base na etapa, subetapa, tipo e descrição.
    """
    if tipo_arquivo not in PADROES_NOMENCLATURA:
        raise ValueError(
            f"tipo_arquivo inválido: {tipo_arquivo}. Utilize um dos seguintes: {list(PADROES_NOMENCLATURA.keys())}"
        )

    nome_padronizado = padronizar_nome_arquivo(nome)

    return PADROES_NOMENCLATURA[tipo_arquivo].format(
        etapa=etapa,
        subetapa=subetapa,
        nome=nome_padronizado
    )

def gerar_caminho_arquivo(etapa, subetapa, tipo_arquivo, nome):
    """
    Gera o caminho completo de salvamento de um arquivo dentro da pasta da etapa correspondente.
    """
    chave_etapa = f"etapa_{etapa}"

    if chave_etapa not in DIRETORIOS_PROJETO:
        raise ValueError(f"Diretório da etapa não encontrado: {chave_etapa}")

    nome_arquivo = gerar_nome_arquivo(etapa, subetapa, tipo_arquivo, nome)
    return DIRETORIOS_PROJETO[chave_etapa] / nome_arquivo

def salvar_dataframe(df, caminho_arquivo, index=False):
    """
    Salva um DataFrame em parquet no caminho especificado, garantindo a existência da pasta de destino.
    """
    caminho_arquivo = Path(caminho_arquivo)
    caminho_arquivo.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(caminho_arquivo, index=index)
    return caminho_arquivo

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 8) Construção das tabelas de auditoria da subetapa
# ============================================================

print("\n[8/9] Construção das tabelas de auditoria da subetapa...")

df_diretorios_projeto = pd.DataFrame(
    [
        {"chave_diretorio": chave, "caminho": str(caminho), "existe": Path(caminho).exists()}
        for chave, caminho in DIRETORIOS_PROJETO.items()
    ]
).sort_values(by="chave_diretorio").reset_index(drop=True)

df_padroes_nomenclatura = pd.DataFrame(
    [
        {"tipo_arquivo": chave, "padrao_nome": valor}
        for chave, valor in PADROES_NOMENCLATURA.items()
    ]
).sort_values(by="tipo_arquivo").reset_index(drop=True)

df_parametros_globais = pd.DataFrame(
    [
        {"parametro": chave, "valor": str(valor)}
        for chave, valor in PARAMETROS_GLOBAIS.items()
    ]
).sort_values(by="parametro").reset_index(drop=True)

print("Tabelas de auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 9) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[9/9] Salvamento dos outputs e validação final da subetapa...")

caminho_diretorios = gerar_caminho_arquivo(etapa=1, subetapa=1, tipo_arquivo="tbl", nome="diretorios_projeto")
caminho_padroes = gerar_caminho_arquivo(etapa=1, subetapa=1, tipo_arquivo="tbl", nome="padroes_nomenclatura")
caminho_parametros = gerar_caminho_arquivo(etapa=1, subetapa=1, tipo_arquivo="tbl", nome="parametros_globais")

salvar_dataframe(df_diretorios_projeto, caminho_diretorios, index=False)
salvar_dataframe(df_padroes_nomenclatura, caminho_padroes, index=False)
salvar_dataframe(df_parametros_globais, caminho_parametros, index=False)

print("\nResumo dos diretórios do projeto:")
print(df_diretorios_projeto.to_string(index=False))

print("\nPadrões de nomenclatura definidos:")
print(df_padroes_nomenclatura.to_string(index=False))

print("\nParâmetros globais definidos:")
print(df_parametros_globais.to_string(index=False))

print("\nArquivos salvos na subetapa 1.1:")
print(f"- {caminho_diretorios}")
print(f"- {caminho_padroes}")
print(f"- {caminho_parametros}")

print("\nExemplos de nomes gerados pelo padrão:")
print(gerar_nome_arquivo(etapa=1, subetapa=1, tipo_arquivo="tbl", nome="diagnostico inicial"))
print(gerar_nome_arquivo(etapa=1, subetapa=1, tipo_arquivo="base", nome="mercado operacional"))
print(gerar_nome_arquivo(etapa=1, subetapa=1, tipo_arquivo="auditoria", nome="validacao estrutura"))
print(gerar_nome_arquivo(etapa=1, subetapa=1, tipo_arquivo="grafico", nome="curva patrimonio"))

print("\nETAPA 1.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 1.1 - ESTRUTURA DE PASTAS E PARÂMETROS GLOBAIS

[1/9] Validação inicial do ambiente...
OK

[2/9] Parâmetros estruturais do projeto...
Diretório raiz identificado : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro
Diretório de resultados     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados
Diretório do HTML final     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\html_final
OK

[3/9] Dicionário central de diretórios...
Quantidade total de diretórios mapeados: 23
OK

[4/9] Criação física da estrutura de pastas...
Estrutura física de diretórios criada ou validada com sucesso.
OK

[5/9] Convenções de nomenclatura dos arquivos...
Padrões de nomenclatura definidos.
OK

[6/9] Parâmetros globais do projeto...
Quantidade de parâmetros globais definidos: 10
OK

[7/9] Funções auxil

## Etapa 1.2) Carga e Padronização das Bases-Fonte

In [4]:
%%time
# ============================================================
# Etapa 1.2) Carga e Padronização das Bases-Fonte
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 1.2 - CARGA E PADRONIZAÇÃO DAS BASES-FONTE")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "yf" not in globals():
    raise NameError("A biblioteca yfinance deve estar importada no ambiente antes da execução desta subetapa.")
if "requests" not in globals():
    raise NameError("A biblioteca requests deve estar importada no ambiente antes da execução desta subetapa.")
if "db_engine" not in globals():
    raise NameError("A variável 'db_engine' não está disponível no ambiente atual.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário 'DIRETORIOS_PROJETO' não está disponível no ambiente atual.")
if "PARAMETROS_GLOBAIS" not in globals():
    raise NameError("O dicionário 'PARAMETROS_GLOBAIS' não está disponível no ambiente atual.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")
if "unicodedata" not in globals():
    raise NameError("A biblioteca unicodedata deve estar importada no ambiente antes da execução desta subetapa.")
if "re" not in globals():
    raise NameError("A biblioteca re deve estar importada no ambiente antes da execução desta subetapa.")

print("OK")

# ============================================================
# 2) Parâmetros da subetapa e definição das colunas de carga
# ============================================================

print("\n[2/10] Parâmetros da subetapa e definição das colunas de carga...")

PARAMETROS_SUBETAPA_1_2 = {
    "data_inicial_series_externas": "2010-01-01",
    "data_final_series_externas": pd.Timestamp.today().normalize(),
    "max_dias_janela_bc": 3650,
    "timeout_requests_segundos": 60,
}

COLUNAS_COT_HIST = [
    "ticker",
    "data",
    "close",
    "close_adj",
    "volume_qtd",
    "volume_fin",
    "negocios",
]
COLUNAS_INFO_EMPRESA = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
]
COLUNAS_INDICADORES_TRIMESTRAIS = [
    "issuer_code",
    "ticker",
    "periodo",
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]
TABELA_COT_HIST = "cot_hist_consolidado"
TABELA_INFO_EMPRESA = "dados_info_empresa"
TABELA_INDICADORES_TRIMESTRAIS = "indicadores_trimestrais"

print(f"Tabela de mercado diário         : {TABELA_COT_HIST}")
print(f"Tabela de informações cadastrais : {TABELA_INFO_EMPRESA}")
print(f"Tabela de indicadores trimestrais: {TABELA_INDICADORES_TRIMESTRAIS}")
print(f"Data inicial das séries externas : {PARAMETROS_SUBETAPA_1_2['data_inicial_series_externas']}")
print(f"Data final das séries externas   : {PARAMETROS_SUBETAPA_1_2['data_final_series_externas'].date()}")
print(f"Quantidade de colunas em {TABELA_COT_HIST}: {len(COLUNAS_COT_HIST)}")
print(f"Quantidade de colunas em {TABELA_INFO_EMPRESA}: {len(COLUNAS_INFO_EMPRESA)}")
print(f"Quantidade de colunas em {TABELA_INDICADORES_TRIMESTRAIS}: {len(COLUNAS_INDICADORES_TRIMESTRAIS)}")
print("OK")

# ============================================================
# 3) Funções auxiliares da subetapa
# ============================================================

print("\n[3/10] Funções auxiliares da subetapa...")

def normalizar_nome_coluna(texto):
    """
    Padroniza nomes de colunas para snake_case ASCII.
    """
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("utf-8")
    texto = texto.strip().lower()
    texto = re.sub(r"[^\w\s]", "_", texto)
    texto = re.sub(r"\s+", "_", texto)
    texto = re.sub(r"_+", "_", texto)
    return texto.strip("_")

def normalizar_nomes_colunas_dataframe(df):
    """
    Aplica padronização de nomes de colunas em um DataFrame.
    """
    df = df.copy()
    df.columns = [normalizar_nome_coluna(col) for col in df.columns]
    return df

def converter_colunas_numericas(df, colunas):
    """
    Converte um conjunto de colunas para formato numérico.
    """
    df = df.copy()
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    return df

def converter_colunas_texto(df, colunas, caixa_alta=False):
    """
    Converte um conjunto de colunas para texto padronizado.
    """
    df = df.copy()
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()
            if caixa_alta:
                df[coluna] = df[coluna].str.upper()
    return df

def montar_query_select(nome_tabela, colunas):
    """
    Monta a instrução SQL de leitura com seleção explícita de colunas.
    """
    colunas_sql = ", ".join(colunas)
    return f"SELECT {colunas_sql} FROM {nome_tabela}"

def formatar_data_bc(data):
    """
    Converte uma data para o formato exigido pela API do Banco Central: dd/mm/aaaa.
    """
    return pd.Timestamp(data).strftime("%d/%m/%Y")

def gerar_janelas_bc(data_inicial, data_final, max_dias_por_janela=3650):
    """
    Quebra um intervalo amplo em janelas menores para consulta de séries diárias do Banco Central.
    """
    data_inicial = pd.Timestamp(data_inicial).normalize()
    data_final = pd.Timestamp(data_final).normalize()

    if data_inicial > data_final:
        raise ValueError("data_inicial não pode ser maior que data_final.")

    janelas = []
    data_inicio_janela = data_inicial

    while data_inicio_janela <= data_final:
        data_fim_janela = min(data_inicio_janela + pd.Timedelta(days=max_dias_por_janela - 1), data_final)
        janelas.append((data_inicio_janela, data_fim_janela))
        data_inicio_janela = data_fim_janela + pd.Timedelta(days=1)

    return janelas

def consulta_bc_em_janelas(codigo_bcb, data_inicial, data_final=None, max_dias_por_janela=3650, timeout=60):
    """
    Consulta uma série diária do Banco Central em janelas sucessivas e concatena o resultado final.
    """
    if data_final is None:
        data_final = pd.Timestamp.today().normalize()

    data_inicial = pd.Timestamp(data_inicial).normalize()
    data_final = pd.Timestamp(data_final).normalize()

    janelas = gerar_janelas_bc(
        data_inicial=data_inicial,
        data_final=data_final,
        max_dias_por_janela=max_dias_por_janela,
    )

    resultados = []

    print(f"Série {codigo_bcb} | Intervalo solicitado: {data_inicial.date()} até {data_final.date()}")
    print(f"Quantidade de janelas a consultar: {len(janelas)}")

    for indice_janela, (inicio_janela, fim_janela) in enumerate(janelas, start=1):
        print(
            f"  - Janela {indice_janela}/{len(janelas)}: "
            f"{inicio_janela.date()} até {fim_janela.date()}"
        )

        url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo_bcb}/dados"
        parametros = {
            "formato": "json",
            "dataInicial": formatar_data_bc(inicio_janela),
            "dataFinal": formatar_data_bc(fim_janela),
        }

        resposta = requests.get(url, params=parametros, timeout=timeout)
        resposta.raise_for_status()
        payload = resposta.json()

        if isinstance(payload, dict):
            raise ValueError(
                f"Falha ao consultar a série {codigo_bcb}. "
                f"Resposta da API: {payload}"
            )

        if not isinstance(payload, list):
            raise ValueError(
                f"Falha ao consultar a série {codigo_bcb}. "
                f"Tipo de payload inesperado: {type(payload)}"
            )

        if len(payload) == 0:
            print("    Registros retornados na janela: 0")
            continue

        df_lote = pd.DataFrame(payload)

        if "data" not in df_lote.columns or "valor" not in df_lote.columns:
            raise ValueError(
                f"Falha ao consultar a série {codigo_bcb}. "
                f"Colunas retornadas na janela: {list(df_lote.columns)}"
            )

        df_lote["data"] = pd.to_datetime(df_lote["data"], dayfirst=True, errors="coerce")
        df_lote["valor"] = pd.to_numeric(df_lote["valor"], errors="coerce")
        df_lote = df_lote.dropna(subset=["data", "valor"]).sort_values("data").reset_index(drop=True)

        print(f"    Registros válidos na janela: {len(df_lote):,}")
        resultados.append(df_lote)

    if len(resultados) == 0:
        raise ValueError(f"Nenhum dado válido foi retornado para a série {codigo_bcb}.")

    df_final = pd.concat(resultados, ignore_index=True)
    df_final = df_final.drop_duplicates(subset=["data"]).sort_values("data").reset_index(drop=True)

    return df_final

def baixar_ibovespa_yahoo(ticker_benchmark, data_inicial, data_final):
    """
    Baixa a série histórica do benchmark via Yahoo Finance e padroniza a estrutura.
    """
    data_inicial = pd.Timestamp(data_inicial).normalize()
    data_final = pd.Timestamp(data_final).normalize()

    df = yf.download(
        tickers=ticker_benchmark,
        start=data_inicial.strftime("%Y-%m-%d"),
        end=(data_final + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        progress=False,
        auto_adjust=False,
        actions=False,
    )

    if df.empty:
        raise ValueError(f"Não foi possível carregar dados do benchmark {ticker_benchmark} via Yahoo Finance.")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

    df = df.reset_index()
    df = normalizar_nomes_colunas_dataframe(df)
    df = df.rename(columns={"adj_close": "close_adj"})

    colunas_esperadas = ["date", "open", "high", "low", "close", "close_adj", "volume"]
    colunas_presentes = [col for col in colunas_esperadas if col in df.columns]
    df = df[colunas_presentes].copy()

    if "date" not in df.columns:
        raise ValueError("A série do Yahoo Finance não retornou a coluna de data esperada.")

    df = df.rename(columns={"date": "data"})
    df["ticker_benchmark"] = ticker_benchmark
    df["data"] = pd.to_datetime(df["data"], errors="coerce")
    df = converter_colunas_numericas(df, ["open", "high", "low", "close", "close_adj", "volume"])
    df = df.dropna(subset=["data"]).sort_values("data").reset_index(drop=True)

    colunas_finais = ["ticker_benchmark", "data", "open", "high", "low", "close", "close_adj", "volume"]
    colunas_finais = [col for col in colunas_finais if col in df.columns]

    return df[colunas_finais].copy()

def resumir_base(df, nome_base, coluna_data=None, coluna_ticker=None, coluna_issuer=None):
    """
    Gera um resumo estrutural padronizado de uma base carregada.
    """
    resumo = {
        "base": nome_base,
        "n_linhas": int(len(df)),
        "n_colunas": int(df.shape[1]),
        "colunas": " | ".join(df.columns.astype(str).tolist()),
        "data_min": None,
        "data_max": None,
        "n_tickers": None,
        "n_issuer_code": None,
    }

    if coluna_data and coluna_data in df.columns:
        serie_data = pd.to_datetime(df[coluna_data], errors="coerce")
        resumo["data_min"] = serie_data.min()
        resumo["data_max"] = serie_data.max()

    if coluna_ticker and coluna_ticker in df.columns:
        resumo["n_tickers"] = int(df[coluna_ticker].nunique(dropna=True))

    if coluna_issuer and coluna_issuer in df.columns:
        resumo["n_issuer_code"] = int(df[coluna_issuer].nunique(dropna=True))

    return resumo

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 4) Carga das bases do banco de dados
# ============================================================

print("\n[4/10] Carga das bases do banco de dados...")

query_cot_hist = montar_query_select(TABELA_COT_HIST, COLUNAS_COT_HIST)
query_info_empresa = montar_query_select(TABELA_INFO_EMPRESA, COLUNAS_INFO_EMPRESA)
query_indicadores_trimestrais = montar_query_select(TABELA_INDICADORES_TRIMESTRAIS, COLUNAS_INDICADORES_TRIMESTRAIS)

df_cot_hist_fonte = pd.read_sql(query_cot_hist, con=db_engine)
print(f"{TABELA_COT_HIST}: {df_cot_hist_fonte.shape[0]:,} linhas x {df_cot_hist_fonte.shape[1]} colunas")

df_info_empresa_fonte = pd.read_sql(query_info_empresa, con=db_engine)
print(f"{TABELA_INFO_EMPRESA}: {df_info_empresa_fonte.shape[0]:,} linhas x {df_info_empresa_fonte.shape[1]} colunas")

df_indicadores_trimestrais_fonte = pd.read_sql(query_indicadores_trimestrais, con=db_engine)
print(f"{TABELA_INDICADORES_TRIMESTRAIS}: {df_indicadores_trimestrais_fonte.shape[0]:,} linhas x {df_indicadores_trimestrais_fonte.shape[1]} colunas")
print("OK")

# ============================================================
# 5) Padronização inicial das bases do banco
# ============================================================

print("\n[5/10] Padronização inicial das bases do banco...")

df_cot_hist_fonte = normalizar_nomes_colunas_dataframe(df_cot_hist_fonte)
df_cot_hist_fonte = converter_colunas_texto(df_cot_hist_fonte, ["ticker"], caixa_alta=True)
df_cot_hist_fonte["data"] = pd.to_datetime(df_cot_hist_fonte["data"], errors="coerce")
df_cot_hist_fonte = converter_colunas_numericas(df_cot_hist_fonte, ["close", "close_adj", "volume_qtd", "volume_fin", "negocios"])
df_cot_hist_fonte = df_cot_hist_fonte.sort_values(["ticker", "data"]).reset_index(drop=True)

df_info_empresa_fonte = normalizar_nomes_colunas_dataframe(df_info_empresa_fonte)
df_info_empresa_fonte = converter_colunas_texto(df_info_empresa_fonte, ["ticker", "issuer_code", "nome", "setor", "subsetor", "segmento"], caixa_alta=False)
df_info_empresa_fonte["ticker"] = df_info_empresa_fonte["ticker"].str.upper()
df_info_empresa_fonte = df_info_empresa_fonte.reset_index(drop=True)

df_indicadores_trimestrais_fonte = normalizar_nomes_colunas_dataframe(df_indicadores_trimestrais_fonte)
df_indicadores_trimestrais_fonte = converter_colunas_texto(df_indicadores_trimestrais_fonte, ["ticker", "issuer_code", "periodo"], caixa_alta=False)
df_indicadores_trimestrais_fonte["ticker"] = df_indicadores_trimestrais_fonte["ticker"].str.upper()
df_indicadores_trimestrais_fonte["periodo"] = df_indicadores_trimestrais_fonte["periodo"].astype("string").str.strip()
df_indicadores_trimestrais_fonte = converter_colunas_numericas(
    df_indicadores_trimestrais_fonte,
    [
        "lucro_liquido",
        "patrimonio_liquido",
        "receita_liquida",
        "ebit",
        "liquidez_corrente",
        "liquidez_geral",
        "liquidez_imediata",
    ],
)
df_indicadores_trimestrais_fonte = df_indicadores_trimestrais_fonte.reset_index(drop=True)

print(f"Base padronizada de mercado     : {df_cot_hist_fonte.shape[0]:,} linhas")
print(f"Base padronizada de cadastro    : {df_info_empresa_fonte.shape[0]:,} linhas")
print(f"Base padronizada de indicadores : {df_indicadores_trimestrais_fonte.shape[0]:,} linhas")
print("OK")

# ============================================================
# 6) Carga das séries externas
# ============================================================

print("\n[6/10] Carga das séries externas...")

df_ibov_fonte = baixar_ibovespa_yahoo(
    ticker_benchmark=PARAMETROS_GLOBAIS["benchmark_renda_variavel"],
    data_inicial=PARAMETROS_SUBETAPA_1_2["data_inicial_series_externas"],
    data_final=PARAMETROS_SUBETAPA_1_2["data_final_series_externas"],
)
print(f"Ibovespa via Yahoo Finance: {df_ibov_fonte.shape[0]:,} linhas x {df_ibov_fonte.shape[1]} colunas")

df_selic_fonte = consulta_bc_em_janelas(
    codigo_bcb=PARAMETROS_GLOBAIS["codigo_sgs_selic_diaria"],
    data_inicial=PARAMETROS_SUBETAPA_1_2["data_inicial_series_externas"],
    data_final=PARAMETROS_SUBETAPA_1_2["data_final_series_externas"],
    max_dias_por_janela=PARAMETROS_SUBETAPA_1_2["max_dias_janela_bc"],
    timeout=PARAMETROS_SUBETAPA_1_2["timeout_requests_segundos"],
)
print(f"Selic diária via SGS {PARAMETROS_GLOBAIS['codigo_sgs_selic_diaria']}: {df_selic_fonte.shape[0]:,} linhas")

df_cdi_fonte = consulta_bc_em_janelas(
    codigo_bcb=PARAMETROS_GLOBAIS["codigo_sgs_cdi_diaria"],
    data_inicial=PARAMETROS_SUBETAPA_1_2["data_inicial_series_externas"],
    data_final=PARAMETROS_SUBETAPA_1_2["data_final_series_externas"],
    max_dias_por_janela=PARAMETROS_SUBETAPA_1_2["max_dias_janela_bc"],
    timeout=PARAMETROS_SUBETAPA_1_2["timeout_requests_segundos"],
)
print(f"CDI diário via SGS {PARAMETROS_GLOBAIS['codigo_sgs_cdi_diaria']}: {df_cdi_fonte.shape[0]:,} linhas")
print("OK")

# ============================================================
# 7) Padronização inicial das séries externas
# ============================================================

print("\n[7/10] Padronização inicial das séries externas...")

df_ibov_fonte = normalizar_nomes_colunas_dataframe(df_ibov_fonte)
df_ibov_fonte["data"] = pd.to_datetime(df_ibov_fonte["data"], errors="coerce")
df_ibov_fonte = converter_colunas_numericas(df_ibov_fonte, ["open", "high", "low", "close", "close_adj", "volume"])
df_ibov_fonte = df_ibov_fonte.dropna(subset=["data"]).sort_values("data").reset_index(drop=True)

df_selic_fonte = normalizar_nomes_colunas_dataframe(df_selic_fonte)
df_selic_fonte["data"] = pd.to_datetime(df_selic_fonte["data"], errors="coerce")
df_selic_fonte = converter_colunas_numericas(df_selic_fonte, ["valor"])
df_selic_fonte = df_selic_fonte.dropna(subset=["data"]).sort_values("data").reset_index(drop=True)

df_cdi_fonte = normalizar_nomes_colunas_dataframe(df_cdi_fonte)
df_cdi_fonte["data"] = pd.to_datetime(df_cdi_fonte["data"], errors="coerce")
df_cdi_fonte = converter_colunas_numericas(df_cdi_fonte, ["valor"])
df_cdi_fonte = df_cdi_fonte.dropna(subset=["data"]).sort_values("data").reset_index(drop=True)

print(f"Intervalo do Ibovespa: {df_ibov_fonte['data'].min()} até {df_ibov_fonte['data'].max()}")
print(f"Intervalo da Selic   : {df_selic_fonte['data'].min()} até {df_selic_fonte['data'].max()}")
print(f"Intervalo do CDI     : {df_cdi_fonte['data'].min()} até {df_cdi_fonte['data'].max()}")
print("OK")

# ============================================================
# 8) Construção das tabelas de apoio da subetapa
# ============================================================

print("\n[8/10] Construção das tabelas de apoio da subetapa...")

df_mapa_bases_uso = pd.DataFrame(
    [
        {
            "base": "cot_hist_consolidado",
            "granularidade": "diária por ticker",
            "chave_principal": "ticker",
            "uso_principal": "preços, liquidez, MM200, mínimas e máximas de 52 semanas, execução das compras e marcação a mercado",
        },
        {
            "base": "dados_info_empresa",
            "granularidade": "cadastro por ticker",
            "chave_principal": "ticker / issuer_code",
            "uso_principal": "vínculo entre ticker e empresa, escolha de um ticker por empresa e classificação setorial",
        },
        {
            "base": "indicadores_trimestrais",
            "granularidade": "trimestral por empresa/ticker",
            "chave_principal": "issuer_code / ticker / periodo",
            "uso_principal": "filtro contábil trimestral e definição de tickers aptos por janela de vigência",
        },
        {
            "base": "ibovespa_yahoo",
            "granularidade": "diária",
            "chave_principal": "data",
            "uso_principal": "benchmark principal de renda variável",
        },
        {
            "base": "selic_sgs_11",
            "granularidade": "diária",
            "chave_principal": "data",
            "uso_principal": "taxa livre de risco para métricas como Sharpe",
        },
        {
            "base": "cdi_sgs_12",
            "granularidade": "diária",
            "chave_principal": "data",
            "uso_principal": "remuneração do caixa parado e construção da carteira CDI-only",
        },
    ]
)

df_resumo_carga_fontes = pd.DataFrame(
    [
        resumir_base(df_cot_hist_fonte, "cot_hist_consolidado", coluna_data="data", coluna_ticker="ticker"),
        resumir_base(df_info_empresa_fonte, "dados_info_empresa", coluna_ticker="ticker", coluna_issuer="issuer_code"),
        resumir_base(df_indicadores_trimestrais_fonte, "indicadores_trimestrais", coluna_ticker="ticker", coluna_issuer="issuer_code"),
        resumir_base(df_ibov_fonte, "ibovespa_yahoo", coluna_data="data"),
        resumir_base(df_selic_fonte, "selic_sgs_11", coluna_data="data"),
        resumir_base(df_cdi_fonte, "cdi_sgs_12", coluna_data="data"),
    ]
)

print("Tabelas de apoio construídas com sucesso.")
print("OK")

# ============================================================
# 9) Salvamento dos outputs da subetapa
# ============================================================

print("\n[9/10] Salvamento dos outputs da subetapa...")

caminho_cot_hist = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="cot_hist_fonte")
caminho_info_empresa = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="info_empresa_fonte")
caminho_indicadores = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="indicadores_trimestrais_fonte")
caminho_ibov = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="ibov_fonte")
caminho_selic = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="selic_fonte")
caminho_cdi = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="cdi_fonte")
caminho_resumo = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="tbl", nome="resumo_carga_fontes")
caminho_mapa = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="tbl", nome="mapa_bases_uso")

salvar_dataframe(df_cot_hist_fonte, caminho_cot_hist, index=False)
salvar_dataframe(df_info_empresa_fonte, caminho_info_empresa, index=False)
salvar_dataframe(df_indicadores_trimestrais_fonte, caminho_indicadores, index=False)
salvar_dataframe(df_ibov_fonte, caminho_ibov, index=False)
salvar_dataframe(df_selic_fonte, caminho_selic, index=False)
salvar_dataframe(df_cdi_fonte, caminho_cdi, index=False)
salvar_dataframe(df_resumo_carga_fontes, caminho_resumo, index=False)
salvar_dataframe(df_mapa_bases_uso, caminho_mapa, index=False)

print("Outputs salvos com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo consolidado da carga:")
print(df_resumo_carga_fontes.to_string(index=False))

print("\nMapa de uso das bases no projeto:")
print(df_mapa_bases_uso.to_string(index=False))

print("\nAmostra da base cot_hist_consolidado:")
print(df_cot_hist_fonte.head(5).to_string(index=False))

print("\nAmostra da base dados_info_empresa:")
print(df_info_empresa_fonte.head(5).to_string(index=False))

print("\nAmostra da base indicadores_trimestrais:")
print(df_indicadores_trimestrais_fonte.head(5).to_string(index=False))

print("\nAmostra da base do Ibovespa:")
print(df_ibov_fonte.head(5).to_string(index=False))

print("\nAmostra da série Selic:")
print(df_selic_fonte.head(5).to_string(index=False))

print("\nAmostra da série CDI:")
print(df_cdi_fonte.head(5).to_string(index=False))

print("\nArquivos salvos na subetapa 1.2:")
print(f"- {caminho_cot_hist}")
print(f"- {caminho_info_empresa}")
print(f"- {caminho_indicadores}")
print(f"- {caminho_ibov}")
print(f"- {caminho_selic}")
print(f"- {caminho_cdi}")
print(f"- {caminho_resumo}")
print(f"- {caminho_mapa}")

print("\nETAPA 1.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 1.2 - CARGA E PADRONIZAÇÃO DAS BASES-FONTE

[1/10] Validação inicial do ambiente...
OK

[2/10] Parâmetros da subetapa e definição das colunas de carga...
Tabela de mercado diário         : cot_hist_consolidado
Tabela de informações cadastrais : dados_info_empresa
Tabela de indicadores trimestrais: indicadores_trimestrais
Data inicial das séries externas : 2010-01-01
Data final das séries externas   : 2026-04-27
Quantidade de colunas em cot_hist_consolidado: 7
Quantidade de colunas em dados_info_empresa: 6
Quantidade de colunas em indicadores_trimestrais: 10
OK

[3/10] Funções auxiliares da subetapa...
Funções auxiliares declaradas com sucesso.
OK

[4/10] Carga das bases do banco de dados...
cot_hist_consolidado: 1,262,516 linhas x 7 colunas
dados_info_empresa: 698 linhas x 6 colunas
indicadores_trimestrais: 39,429 linhas x 10 colunas
OK

[5/10] Padronização inicial das bases do banco...
Base padronizada de mercado     : 1,262,516 linhas
Base padronizada de cadastro    :

## Etapa 1.3) Padronização Inicial de Chaves, Datas e Estruturas

In [5]:
%%time
# ============================================================
# Etapa 1.3) Padronização Inicial de Chaves, Datas e Estruturas
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 1.3 - PADRONIZAÇÃO INICIAL DE CHAVES, DATAS E ESTRUTURAS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/11] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "re" not in globals():
    raise NameError("A biblioteca re deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/11] Definição dos caminhos de entrada e saída da subetapa...")

caminho_cot_hist_fonte = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="cot_hist_fonte")
caminho_info_empresa_fonte = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="info_empresa_fonte")
caminho_indicadores_fonte = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="indicadores_trimestrais_fonte")
caminho_ibov_fonte = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="ibov_fonte")
caminho_selic_fonte = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="selic_fonte")
caminho_cdi_fonte = gerar_caminho_arquivo(etapa=1, subetapa=2, tipo_arquivo="base", nome="cdi_fonte")

caminho_cot_hist_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="cot_hist_padronizada")
caminho_info_empresa_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="info_empresa_padronizada")
caminho_indicadores_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="indicadores_trimestrais_padronizada")
caminho_ibov_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="ibov_padronizada")
caminho_selic_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="selic_padronizada")
caminho_cdi_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="cdi_padronizada")

print(f"Entrada - cot_hist        : {caminho_cot_hist_fonte}")
print(f"Entrada - info_empresa    : {caminho_info_empresa_fonte}")
print(f"Entrada - indicadores     : {caminho_indicadores_fonte}")
print(f"Entrada - ibov            : {caminho_ibov_fonte}")
print(f"Entrada - selic           : {caminho_selic_fonte}")
print(f"Entrada - cdi             : {caminho_cdi_fonte}")
print(f"Saída   - cot_hist        : {caminho_cot_hist_padronizada}")
print(f"Saída   - info_empresa    : {caminho_info_empresa_padronizada}")
print(f"Saída   - indicadores     : {caminho_indicadores_padronizada}")
print(f"Saída   - ibov            : {caminho_ibov_padronizada}")
print(f"Saída   - selic           : {caminho_selic_padronizada}")
print(f"Saída   - cdi             : {caminho_cdi_padronizada}")
print("OK")

# ============================================================
# 3) Carga das bases geradas na subetapa 1.2
# ============================================================

print("\n[3/11] Carga das bases geradas na subetapa 1.2...")

df_cot_hist_fonte = pd.read_parquet(caminho_cot_hist_fonte)
df_info_empresa_fonte = pd.read_parquet(caminho_info_empresa_fonte)
df_indicadores_trimestrais_fonte = pd.read_parquet(caminho_indicadores_fonte)
df_ibov_fonte = pd.read_parquet(caminho_ibov_fonte)
df_selic_fonte = pd.read_parquet(caminho_selic_fonte)
df_cdi_fonte = pd.read_parquet(caminho_cdi_fonte)

print(f"cot_hist_consolidado        : {df_cot_hist_fonte.shape[0]:,} linhas x {df_cot_hist_fonte.shape[1]} colunas")
print(f"dados_info_empresa          : {df_info_empresa_fonte.shape[0]:,} linhas x {df_info_empresa_fonte.shape[1]} colunas")
print(f"indicadores_trimestrais     : {df_indicadores_trimestrais_fonte.shape[0]:,} linhas x {df_indicadores_trimestrais_fonte.shape[1]} colunas")
print(f"ibovespa_yahoo              : {df_ibov_fonte.shape[0]:,} linhas x {df_ibov_fonte.shape[1]} colunas")
print(f"selic_sgs_11                : {df_selic_fonte.shape[0]:,} linhas x {df_selic_fonte.shape[1]} colunas")
print(f"cdi_sgs_12                  : {df_cdi_fonte.shape[0]:,} linhas x {df_cdi_fonte.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/11] Funções auxiliares da subetapa...")

def padronizar_chave_textual(serie, caixa_alta=False):
    """
    Padroniza chaves textuais preservando valores ausentes.
    """
    serie = serie.astype("string").str.strip()
    if caixa_alta:
        serie = serie.str.upper()
    return serie

def primeiro_valor_nao_nulo(serie):
    """
    Retorna o primeiro valor não nulo de uma série.
    """
    serie = serie.dropna()
    if len(serie) == 0:
        return pd.NA
    return serie.iloc[0]

def contar_duplicidades_exatas(df):
    """
    Conta o número de duplicidades exatas em um DataFrame.
    """
    return int(df.duplicated().sum())

def padronizar_periodo_trimestral(periodo):
    """
    Padroniza representações trimestrais para o formato YYYYQn.
    """
    if pd.isna(periodo):
        return pd.NA

    texto = str(periodo).strip().upper()
    texto = texto.replace(" ", "")
    texto = texto.replace("-", "")
    texto = texto.replace("/", "")

    padroes = [
        r"^(\d{4})Q([1-4])$",
        r"^(\d{4})T([1-4])$",
        r"^([1-4])T(\d{4})$",
        r"^Q([1-4])(\d{4})$",
        r"^(\d{4})([1-4])T$",
        r"^(\d{4})TRI([1-4])$",
    ]

    for padrao in padroes:
        match = re.match(padrao, texto)
        if match:
            grupos = match.groups()

            if padrao in [r"^(\d{4})Q([1-4])$", r"^(\d{4})T([1-4])$", r"^(\d{4})([1-4])T$", r"^(\d{4})TRI([1-4])$"]:
                ano = grupos[0]
                trimestre = grupos[1]
            else:
                trimestre = grupos[0]
                ano = grupos[1]

            return f"{ano}Q{trimestre}"

    return pd.NA

def obter_datas_periodo(periodo_padronizado):
    """
    Retorna a data inicial e a data final de um trimestre no formato YYYYQn.
    """
    if pd.isna(periodo_padronizado):
        return pd.Series([pd.NaT, pd.NaT], index=["data_inicio_periodo", "data_fim_periodo"])

    periodo_trimestral = pd.Period(periodo_padronizado, freq="Q-DEC")
    data_inicio = periodo_trimestral.start_time.normalize()
    data_fim = periodo_trimestral.end_time.normalize()

    return pd.Series([data_inicio, data_fim], index=["data_inicio_periodo", "data_fim_periodo"])

def resumir_base(df, nome_base, coluna_data=None, coluna_ticker=None, coluna_issuer=None):
    """
    Gera um resumo estrutural padronizado de uma base.
    """
    resumo = {
        "base": nome_base,
        "n_linhas": int(len(df)),
        "n_colunas": int(df.shape[1]),
        "data_min": None,
        "data_max": None,
        "n_tickers": None,
        "n_issuer_code": None,
    }

    if coluna_data and coluna_data in df.columns:
        serie_data = pd.to_datetime(df[coluna_data], errors="coerce")
        resumo["data_min"] = serie_data.min()
        resumo["data_max"] = serie_data.max()

    if coluna_ticker and coluna_ticker in df.columns:
        resumo["n_tickers"] = int(df[coluna_ticker].nunique(dropna=True))

    if coluna_issuer and coluna_issuer in df.columns:
        resumo["n_issuer_code"] = int(df[coluna_issuer].nunique(dropna=True))

    return resumo

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização da base diária de mercado e da série do benchmark
# ============================================================

print("\n[5/11] Padronização da base diária de mercado e da série do benchmark...")

duplicidades_exatas_cot_hist_antes = contar_duplicidades_exatas(df_cot_hist_fonte)
duplicidades_exatas_ibov_antes = contar_duplicidades_exatas(df_ibov_fonte)

df_cot_hist_padronizada = df_cot_hist_fonte.copy()
df_cot_hist_padronizada["ticker"] = padronizar_chave_textual(df_cot_hist_padronizada["ticker"], caixa_alta=True)
df_cot_hist_padronizada["data"] = pd.to_datetime(df_cot_hist_padronizada["data"], errors="coerce")
df_cot_hist_padronizada = df_cot_hist_padronizada.drop_duplicates().sort_values(["ticker", "data"]).reset_index(drop=True)

df_ibov_padronizada = df_ibov_fonte.copy()
df_ibov_padronizada["ticker_benchmark"] = padronizar_chave_textual(df_ibov_padronizada["ticker_benchmark"], caixa_alta=True)
df_ibov_padronizada["data"] = pd.to_datetime(df_ibov_padronizada["data"], errors="coerce")
df_ibov_padronizada = df_ibov_padronizada.drop_duplicates().sort_values("data").reset_index(drop=True)

print(f"Duplicidades exatas removidas em cot_hist_consolidado: {duplicidades_exatas_cot_hist_antes:,}")
print(f"Duplicidades exatas removidas em ibovespa_yahoo      : {duplicidades_exatas_ibov_antes:,}")
print("OK")

# ============================================================
# 6) Padronização das séries do Banco Central
# ============================================================

print("\n[6/11] Padronização das séries do Banco Central...")

duplicidades_exatas_selic_antes = contar_duplicidades_exatas(df_selic_fonte)
duplicidades_exatas_cdi_antes = contar_duplicidades_exatas(df_cdi_fonte)

df_selic_padronizada = df_selic_fonte.copy()
df_selic_padronizada["data"] = pd.to_datetime(df_selic_padronizada["data"], errors="coerce")
df_selic_padronizada["valor"] = pd.to_numeric(df_selic_padronizada["valor"], errors="coerce")
df_selic_padronizada = df_selic_padronizada.drop_duplicates().sort_values("data").reset_index(drop=True)

df_cdi_padronizada = df_cdi_fonte.copy()
df_cdi_padronizada["data"] = pd.to_datetime(df_cdi_padronizada["data"], errors="coerce")
df_cdi_padronizada["valor"] = pd.to_numeric(df_cdi_padronizada["valor"], errors="coerce")
df_cdi_padronizada = df_cdi_padronizada.drop_duplicates().sort_values("data").reset_index(drop=True)

print(f"Duplicidades exatas removidas em selic_sgs_11: {duplicidades_exatas_selic_antes:,}")
print(f"Duplicidades exatas removidas em cdi_sgs_12  : {duplicidades_exatas_cdi_antes:,}")
print("OK")

# ============================================================
# 7) Padronização da base de informações de empresa
# ============================================================

print("\n[7/11] Padronização da base de informações de empresa...")

duplicidades_exatas_info_empresa_antes = contar_duplicidades_exatas(df_info_empresa_fonte)

df_info_empresa_padronizada = df_info_empresa_fonte.copy()
df_info_empresa_padronizada["ticker"] = padronizar_chave_textual(df_info_empresa_padronizada["ticker"], caixa_alta=True)
df_info_empresa_padronizada["issuer_code"] = padronizar_chave_textual(df_info_empresa_padronizada["issuer_code"], caixa_alta=False)
df_info_empresa_padronizada["nome"] = padronizar_chave_textual(df_info_empresa_padronizada["nome"], caixa_alta=False)
df_info_empresa_padronizada["setor"] = padronizar_chave_textual(df_info_empresa_padronizada["setor"], caixa_alta=False)
df_info_empresa_padronizada["subsetor"] = padronizar_chave_textual(df_info_empresa_padronizada["subsetor"], caixa_alta=False)
df_info_empresa_padronizada["segmento"] = padronizar_chave_textual(df_info_empresa_padronizada["segmento"], caixa_alta=False)
df_info_empresa_padronizada = df_info_empresa_padronizada.drop_duplicates()

df_info_empresa_padronizada = (
    df_info_empresa_padronizada
    .groupby(["ticker", "issuer_code"], dropna=False, as_index=False)
    .agg(
        nome=("nome", primeiro_valor_nao_nulo),
        setor=("setor", primeiro_valor_nao_nulo),
        subsetor=("subsetor", primeiro_valor_nao_nulo),
        segmento=("segmento", primeiro_valor_nao_nulo),
    )
    .sort_values(["ticker", "issuer_code"])
    .reset_index(drop=True)
)

print(f"Duplicidades exatas removidas em dados_info_empresa: {duplicidades_exatas_info_empresa_antes:,}")
print(f"Registros finais em dados_info_empresa             : {len(df_info_empresa_padronizada):,}")
print("OK")

# ============================================================
# 8) Padronização da base trimestral e criação das datas do período
# ============================================================

print("\n[8/11] Padronização da base trimestral e criação das datas do período...")

duplicidades_exatas_indicadores_antes = contar_duplicidades_exatas(df_indicadores_trimestrais_fonte)

df_indicadores_trimestrais_padronizada = df_indicadores_trimestrais_fonte.copy()
df_indicadores_trimestrais_padronizada["ticker"] = padronizar_chave_textual(df_indicadores_trimestrais_padronizada["ticker"], caixa_alta=True)
df_indicadores_trimestrais_padronizada["issuer_code"] = padronizar_chave_textual(df_indicadores_trimestrais_padronizada["issuer_code"], caixa_alta=False)
df_indicadores_trimestrais_padronizada["periodo"] = df_indicadores_trimestrais_padronizada["periodo"].apply(padronizar_periodo_trimestral)
df_indicadores_trimestrais_padronizada[["data_inicio_periodo", "data_fim_periodo"]] = df_indicadores_trimestrais_padronizada["periodo"].apply(obter_datas_periodo)
df_indicadores_trimestrais_padronizada = df_indicadores_trimestrais_padronizada.drop_duplicates()
df_indicadores_trimestrais_padronizada = (
    df_indicadores_trimestrais_padronizada
    .sort_values(["issuer_code", "ticker", "data_inicio_periodo", "periodo"])
    .reset_index(drop=True)
)

print(f"Duplicidades exatas removidas em indicadores_trimestrais: {duplicidades_exatas_indicadores_antes:,}")
print(f"Registros com período trimestral inválido               : {int(df_indicadores_trimestrais_padronizada['periodo'].isna().sum()):,}")
print("OK")

# ============================================================
# 9) Construção das tabelas-resumo da subetapa
# ============================================================

print("\n[9/11] Construção das tabelas-resumo da subetapa...")

df_resumo_padronizacao_1_3 = pd.DataFrame(
    [
        resumir_base(df_cot_hist_padronizada, "cot_hist_consolidado", coluna_data="data", coluna_ticker="ticker"),
        resumir_base(df_info_empresa_padronizada, "dados_info_empresa", coluna_ticker="ticker", coluna_issuer="issuer_code"),
        resumir_base(df_indicadores_trimestrais_padronizada, "indicadores_trimestrais", coluna_ticker="ticker", coluna_issuer="issuer_code"),
        resumir_base(df_ibov_padronizada, "ibovespa_yahoo", coluna_data="data"),
        resumir_base(df_selic_padronizada, "selic_sgs_11", coluna_data="data"),
        resumir_base(df_cdi_padronizada, "cdi_sgs_12", coluna_data="data"),
    ]
)

df_auditoria_duplicidades_1_3 = pd.DataFrame(
    [
        {"base": "cot_hist_consolidado", "duplicidades_exatas_removidas": duplicidades_exatas_cot_hist_antes},
        {"base": "dados_info_empresa", "duplicidades_exatas_removidas": duplicidades_exatas_info_empresa_antes},
        {"base": "indicadores_trimestrais", "duplicidades_exatas_removidas": duplicidades_exatas_indicadores_antes},
        {"base": "ibovespa_yahoo", "duplicidades_exatas_removidas": duplicidades_exatas_ibov_antes},
        {"base": "selic_sgs_11", "duplicidades_exatas_removidas": duplicidades_exatas_selic_antes},
        {"base": "cdi_sgs_12", "duplicidades_exatas_removidas": duplicidades_exatas_cdi_antes},
    ]
)

tickers_cot_hist = set(df_cot_hist_padronizada["ticker"].dropna().unique().tolist())
tickers_info_empresa = set(df_info_empresa_padronizada["ticker"].dropna().unique().tolist())
tickers_indicadores = set(df_indicadores_trimestrais_padronizada["ticker"].dropna().unique().tolist())

issuer_info_empresa = set(df_info_empresa_padronizada["issuer_code"].dropna().unique().tolist())
issuer_indicadores = set(df_indicadores_trimestrais_padronizada["issuer_code"].dropna().unique().tolist())

df_consistencia_chaves_1_3 = pd.DataFrame(
    [
        {
            "metrica": "tickers_cot_hist_nao_mapeados_em_info_empresa",
            "valor": len(tickers_cot_hist - tickers_info_empresa),
        },
        {
            "metrica": "tickers_info_empresa_nao_presentes_em_cot_hist",
            "valor": len(tickers_info_empresa - tickers_cot_hist),
        },
        {
            "metrica": "tickers_indicadores_nao_presentes_em_info_empresa",
            "valor": len(tickers_indicadores - tickers_info_empresa),
        },
        {
            "metrica": "issuer_info_empresa_nao_presentes_em_indicadores",
            "valor": len(issuer_info_empresa - issuer_indicadores),
        },
        {
            "metrica": "issuer_indicadores_nao_presentes_em_info_empresa",
            "valor": len(issuer_indicadores - issuer_info_empresa),
        },
    ]
)

print("Tabelas-resumo construídas com sucesso.")
print("OK")

# ============================================================
# 10) Salvamento dos outputs padronizados da subetapa
# ============================================================

print("\n[10/11] Salvamento dos outputs padronizados da subetapa...")

salvar_dataframe(df_cot_hist_padronizada, caminho_cot_hist_padronizada, index=False)
salvar_dataframe(df_info_empresa_padronizada, caminho_info_empresa_padronizada, index=False)
salvar_dataframe(df_indicadores_trimestrais_padronizada, caminho_indicadores_padronizada, index=False)
salvar_dataframe(df_ibov_padronizada, caminho_ibov_padronizada, index=False)
salvar_dataframe(df_selic_padronizada, caminho_selic_padronizada, index=False)
salvar_dataframe(df_cdi_padronizada, caminho_cdi_padronizada, index=False)

print("Outputs padronizados salvos com sucesso.")
print("OK")

# ============================================================
# 11) Validação final da subetapa
# ============================================================

print("\n[11/11] Validação final da subetapa...")

print("\nResumo estrutural das bases padronizadas:")
print(df_resumo_padronizacao_1_3.to_string(index=False))

print("\nAuditoria de duplicidades exatas removidas:")
print(df_auditoria_duplicidades_1_3.to_string(index=False))

print("\nConsistência das chaves entre as bases:")
print(df_consistencia_chaves_1_3.to_string(index=False))

print("\nAmostra da base cot_hist_consolidado padronizada:")
print(df_cot_hist_padronizada.head(5).to_string(index=False))

print("\nAmostra da base dados_info_empresa padronizada:")
print(df_info_empresa_padronizada.head(5).to_string(index=False))

print("\nAmostra da base indicadores_trimestrais padronizada:")
print(df_indicadores_trimestrais_padronizada.head(5).to_string(index=False))

print("\nAmostra da base do Ibovespa padronizada:")
print(df_ibov_padronizada.head(5).to_string(index=False))

print("\nAmostra da série Selic padronizada:")
print(df_selic_padronizada.head(5).to_string(index=False))

print("\nAmostra da série CDI padronizada:")
print(df_cdi_padronizada.head(5).to_string(index=False))

print("\nArquivos salvos na subetapa 1.3:")
print(f"- {caminho_cot_hist_padronizada}")
print(f"- {caminho_info_empresa_padronizada}")
print(f"- {caminho_indicadores_padronizada}")
print(f"- {caminho_ibov_padronizada}")
print(f"- {caminho_selic_padronizada}")
print(f"- {caminho_cdi_padronizada}")

print("\nETAPA 1.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 1.3 - PADRONIZAÇÃO INICIAL DE CHAVES, DATAS E ESTRUTURAS

[1/11] Validação inicial do ambiente...
OK

[2/11] Definição dos caminhos de entrada e saída da subetapa...
Entrada - cot_hist        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_2_base_cot_hist_fonte.parquet
Entrada - info_empresa    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_2_base_info_empresa_fonte.parquet
Entrada - indicadores     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_2_base_indicadores_trimestrais_fonte.parquet
Entrada - ibov            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_2_base_ibov_fonte.parquet
Entrada - selic           : C:\Users\mht-1\

## Etapa 1.4) Diagnóstico Inicial das Bases

In [6]:
%%time
# ============================================================
# Etapa 1.4) Diagnóstico Inicial das Bases
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 1.4 - DIAGNÓSTICO INICIAL DAS BASES")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/11] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/11] Definição dos caminhos de entrada e saída da subetapa...")

caminho_cot_hist_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="cot_hist_padronizada")
caminho_info_empresa_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="info_empresa_padronizada")
caminho_indicadores_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="indicadores_trimestrais_padronizada")
caminho_ibov_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="ibov_padronizada")
caminho_selic_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="selic_padronizada")
caminho_cdi_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="cdi_padronizada")

caminho_diagnostico_bases = gerar_caminho_arquivo(etapa=1, subetapa=4, tipo_arquivo="tbl", nome="diagnostico_bases")
caminho_cobertura_temporal = gerar_caminho_arquivo(etapa=1, subetapa=4, tipo_arquivo="tbl", nome="cobertura_temporal")
caminho_completude_colunas = gerar_caminho_arquivo(etapa=1, subetapa=4, tipo_arquivo="tbl", nome="completude_colunas")
caminho_auditoria_limitacoes = gerar_caminho_arquivo(etapa=1, subetapa=4, tipo_arquivo="auditoria", nome="limitacoes_bases")

print(f"Entrada - cot_hist        : {caminho_cot_hist_padronizada}")
print(f"Entrada - info_empresa    : {caminho_info_empresa_padronizada}")
print(f"Entrada - indicadores     : {caminho_indicadores_padronizada}")
print(f"Entrada - ibov            : {caminho_ibov_padronizada}")
print(f"Entrada - selic           : {caminho_selic_padronizada}")
print(f"Entrada - cdi             : {caminho_cdi_padronizada}")
print(f"Saída   - diagnóstico     : {caminho_diagnostico_bases}")
print(f"Saída   - cobertura       : {caminho_cobertura_temporal}")
print(f"Saída   - completude      : {caminho_completude_colunas}")
print(f"Saída   - auditoria       : {caminho_auditoria_limitacoes}")
print("OK")

# ============================================================
# 3) Carga das bases padronizadas da subetapa 1.3
# ============================================================

print("\n[3/11] Carga das bases padronizadas da subetapa 1.3...")

df_cot_hist_padronizada = pd.read_parquet(caminho_cot_hist_padronizada)
df_info_empresa_padronizada = pd.read_parquet(caminho_info_empresa_padronizada)
df_indicadores_trimestrais_padronizada = pd.read_parquet(caminho_indicadores_padronizada)
df_ibov_padronizada = pd.read_parquet(caminho_ibov_padronizada)
df_selic_padronizada = pd.read_parquet(caminho_selic_padronizada)
df_cdi_padronizada = pd.read_parquet(caminho_cdi_padronizada)

print(f"cot_hist_consolidado        : {df_cot_hist_padronizada.shape[0]:,} linhas x {df_cot_hist_padronizada.shape[1]} colunas")
print(f"dados_info_empresa          : {df_info_empresa_padronizada.shape[0]:,} linhas x {df_info_empresa_padronizada.shape[1]} colunas")
print(f"indicadores_trimestrais     : {df_indicadores_trimestrais_padronizada.shape[0]:,} linhas x {df_indicadores_trimestrais_padronizada.shape[1]} colunas")
print(f"ibovespa_yahoo              : {df_ibov_padronizada.shape[0]:,} linhas x {df_ibov_padronizada.shape[1]} colunas")
print(f"selic_sgs_11                : {df_selic_padronizada.shape[0]:,} linhas x {df_selic_padronizada.shape[1]} colunas")
print(f"cdi_sgs_12                  : {df_cdi_padronizada.shape[0]:,} linhas x {df_cdi_padronizada.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros e colunas centrais do diagnóstico
# ============================================================

print("\n[4/11] Parâmetros e colunas centrais do diagnóstico...")

COLUNAS_CENTRAIS_POR_BASE = {
    "cot_hist_consolidado": ["ticker", "data", "close", "close_adj", "volume_qtd", "volume_fin", "negocios"],
    "dados_info_empresa": ["ticker", "issuer_code", "nome", "setor", "subsetor", "segmento"],
    "indicadores_trimestrais": [
        "issuer_code",
        "ticker",
        "periodo",
        "data_inicio_periodo",
        "data_fim_periodo",
        "lucro_liquido",
        "patrimonio_liquido",
        "receita_liquida",
        "ebit",
        "liquidez_corrente",
        "liquidez_geral",
        "liquidez_imediata",
    ],
    "ibovespa_yahoo": ["ticker_benchmark", "data", "open", "high", "low", "close", "close_adj", "volume"],
    "selic_sgs_11": ["data", "valor"],
    "cdi_sgs_12": ["data", "valor"],
}

print("Colunas centrais por base definidas com sucesso.")
print("OK")

# ============================================================
# 5) Funções auxiliares da subetapa
# ============================================================

print("\n[5/11] Funções auxiliares da subetapa...")

def contar_duplicidades_exatas(df):
    """
    Conta duplicidades exatas em um DataFrame.
    """
    return int(df.duplicated().sum())

def contar_duplicidades_por_chave(df, colunas_chave):
    """
    Conta duplicidades com base em uma chave natural definida por colunas.
    """
    colunas_validas = [col for col in colunas_chave if col in df.columns]
    if len(colunas_validas) == 0:
        return pd.NA
    return int(df.duplicated(subset=colunas_validas).sum())

def contar_vazios_textuais(serie):
    """
    Conta células textuais vazias após remoção de espaços.
    """
    serie_string = serie.astype("string")
    return int(serie_string.str.strip().eq("").fillna(False).sum())

def resumir_base_diagnostico(df, nome_base, coluna_data=None, coluna_ticker=None, coluna_issuer=None, coluna_periodo=None, colunas_chave=None):
    """
    Gera um resumo estrutural completo de uma base.
    """
    resumo = {
        "base": nome_base,
        "n_linhas": int(len(df)),
        "n_colunas": int(df.shape[1]),
        "n_datas_unicas": pd.NA,
        "n_tickers": pd.NA,
        "n_issuer_code": pd.NA,
        "n_periodos": pd.NA,
        "data_min": pd.NaT,
        "data_max": pd.NaT,
        "duplicidades_exatas": contar_duplicidades_exatas(df),
        "duplicidades_chave": contar_duplicidades_por_chave(df, colunas_chave if colunas_chave is not None else []),
    }

    if coluna_data and coluna_data in df.columns:
        serie_data = pd.to_datetime(df[coluna_data], errors="coerce")
        resumo["n_datas_unicas"] = int(serie_data.nunique(dropna=True))
        resumo["data_min"] = serie_data.min()
        resumo["data_max"] = serie_data.max()

    if coluna_ticker and coluna_ticker in df.columns:
        resumo["n_tickers"] = int(df[coluna_ticker].nunique(dropna=True))

    if coluna_issuer and coluna_issuer in df.columns:
        resumo["n_issuer_code"] = int(df[coluna_issuer].nunique(dropna=True))

    if coluna_periodo and coluna_periodo in df.columns:
        resumo["n_periodos"] = int(df[coluna_periodo].nunique(dropna=True))

    return resumo

def construir_completude_colunas(df, nome_base, colunas_centrais):
    """
    Constrói uma tabela de completude para as colunas centrais de uma base.
    """
    registros = []

    for coluna in colunas_centrais:
        if coluna not in df.columns:
            registros.append(
                {
                    "base": nome_base,
                    "coluna": coluna,
                    "n_linhas": int(len(df)),
                    "n_missing": pd.NA,
                    "n_vazios_textuais": pd.NA,
                    "n_preenchidos": pd.NA,
                    "pct_missing": pd.NA,
                    "pct_preenchidos": pd.NA,
                    "coluna_presente": False,
                }
            )
            continue

        n_linhas = int(len(df))
        n_missing = int(df[coluna].isna().sum())

        if pd.api.types.is_string_dtype(df[coluna]) or df[coluna].dtype == "object":
            n_vazios_textuais = contar_vazios_textuais(df[coluna])
        else:
            n_vazios_textuais = 0

        n_preenchidos = int(n_linhas - n_missing - n_vazios_textuais)
        pct_missing = (n_missing + n_vazios_textuais) / n_linhas if n_linhas > 0 else pd.NA
        pct_preenchidos = n_preenchidos / n_linhas if n_linhas > 0 else pd.NA

        registros.append(
            {
                "base": nome_base,
                "coluna": coluna,
                "n_linhas": n_linhas,
                "n_missing": n_missing,
                "n_vazios_textuais": n_vazios_textuais,
                "n_preenchidos": n_preenchidos,
                "pct_missing": pct_missing,
                "pct_preenchidos": pct_preenchidos,
                "coluna_presente": True,
            }
        )

    return pd.DataFrame(registros)

def extrair_datas_unicas(df, coluna_data):
    """
    Extrai o conjunto de datas válidas de uma base temporal.
    """
    return set(pd.to_datetime(df[coluna_data], errors="coerce").dropna().dt.normalize().tolist())

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 6) Construção da tabela de diagnóstico das bases
# ============================================================

print("\n[6/11] Construção da tabela de diagnóstico das bases...")

df_diagnostico_bases = pd.DataFrame(
    [
        resumir_base_diagnostico(
            df_cot_hist_padronizada,
            "cot_hist_consolidado",
            coluna_data="data",
            coluna_ticker="ticker",
            colunas_chave=["ticker", "data"],
        ),
        resumir_base_diagnostico(
            df_info_empresa_padronizada,
            "dados_info_empresa",
            coluna_ticker="ticker",
            coluna_issuer="issuer_code",
            colunas_chave=["ticker", "issuer_code"],
        ),
        resumir_base_diagnostico(
            df_indicadores_trimestrais_padronizada,
            "indicadores_trimestrais",
            coluna_ticker="ticker",
            coluna_issuer="issuer_code",
            coluna_periodo="periodo",
            colunas_chave=["issuer_code", "ticker", "periodo"],
        ),
        resumir_base_diagnostico(
            df_ibov_padronizada,
            "ibovespa_yahoo",
            coluna_data="data",
            colunas_chave=["data"],
        ),
        resumir_base_diagnostico(
            df_selic_padronizada,
            "selic_sgs_11",
            coluna_data="data",
            colunas_chave=["data"],
        ),
        resumir_base_diagnostico(
            df_cdi_padronizada,
            "cdi_sgs_12",
            coluna_data="data",
            colunas_chave=["data"],
        ),
    ]
)

print("Tabela de diagnóstico das bases construída com sucesso.")
print("OK")

# ============================================================
# 7) Construção da tabela de cobertura temporal
# ============================================================

print("\n[7/11] Construção da tabela de cobertura temporal...")

datas_mercado = extrair_datas_unicas(df_cot_hist_padronizada, "data")
datas_ibov = extrair_datas_unicas(df_ibov_padronizada, "data")
datas_selic = extrair_datas_unicas(df_selic_padronizada, "data")
datas_cdi = extrair_datas_unicas(df_cdi_padronizada, "data")

registros_cobertura_temporal = []

for _, linha in df_diagnostico_bases.iterrows():
    registros_cobertura_temporal.append(
        {
            "tipo_registro": "intervalo_base",
            "base_referencia": linha["base"],
            "base_comparada": pd.NA,
            "periodo": pd.NA,
            "data_min": linha["data_min"],
            "data_max": linha["data_max"],
            "n_datas_observadas": linha["n_datas_unicas"],
            "n_datas_em_comum": pd.NA,
            "pct_base_referencia_em_comum": pd.NA,
            "pct_base_comparada_em_comum": pd.NA,
            "n_empresas": pd.NA,
            "n_tickers": pd.NA,
        }
    )

comparacoes_temporais = [
    ("cot_hist_consolidado", "ibovespa_yahoo", datas_mercado, datas_ibov),
    ("cot_hist_consolidado", "selic_sgs_11", datas_mercado, datas_selic),
    ("cot_hist_consolidado", "cdi_sgs_12", datas_mercado, datas_cdi),
]

for base_referencia, base_comparada, datas_base_referencia, datas_base_comparada in comparacoes_temporais:
    n_comum = len(datas_base_referencia & datas_base_comparada)
    pct_referencia = n_comum / len(datas_base_referencia) if len(datas_base_referencia) > 0 else pd.NA
    pct_comparada = n_comum / len(datas_base_comparada) if len(datas_base_comparada) > 0 else pd.NA

    registros_cobertura_temporal.append(
        {
            "tipo_registro": "comparacao_mercado",
            "base_referencia": base_referencia,
            "base_comparada": base_comparada,
            "periodo": pd.NA,
            "data_min": pd.NaT,
            "data_max": pd.NaT,
            "n_datas_observadas": pd.NA,
            "n_datas_em_comum": n_comum,
            "pct_base_referencia_em_comum": pct_referencia,
            "pct_base_comparada_em_comum": pct_comparada,
            "n_empresas": pd.NA,
            "n_tickers": pd.NA,
        }
    )

df_cobertura_contabil_periodo = (
    df_indicadores_trimestrais_padronizada
    .groupby("periodo", dropna=False)
    .agg(
        data_inicio_periodo=("data_inicio_periodo", "min"),
        data_fim_periodo=("data_fim_periodo", "max"),
        n_empresas=("issuer_code", "nunique"),
        n_tickers=("ticker", "nunique"),
        n_registros=("ticker", "size"),
    )
    .reset_index()
    .sort_values("data_inicio_periodo")
    .reset_index(drop=True)
)

for _, linha in df_cobertura_contabil_periodo.iterrows():
    registros_cobertura_temporal.append(
        {
            "tipo_registro": "cobertura_contabil_periodo",
            "base_referencia": "indicadores_trimestrais",
            "base_comparada": pd.NA,
            "periodo": linha["periodo"],
            "data_min": linha["data_inicio_periodo"],
            "data_max": linha["data_fim_periodo"],
            "n_datas_observadas": linha["n_registros"],
            "n_datas_em_comum": pd.NA,
            "pct_base_referencia_em_comum": pd.NA,
            "pct_base_comparada_em_comum": pd.NA,
            "n_empresas": linha["n_empresas"],
            "n_tickers": linha["n_tickers"],
        }
    )

df_cobertura_temporal = pd.DataFrame(registros_cobertura_temporal)

print("Tabela de cobertura temporal construída com sucesso.")
print("OK")

# ============================================================
# 8) Construção da tabela de completude das colunas centrais
# ============================================================

print("\n[8/11] Construção da tabela de completude das colunas centrais...")

df_completude_colunas = pd.concat(
    [
        construir_completude_colunas(df_cot_hist_padronizada, "cot_hist_consolidado", COLUNAS_CENTRAIS_POR_BASE["cot_hist_consolidado"]),
        construir_completude_colunas(df_info_empresa_padronizada, "dados_info_empresa", COLUNAS_CENTRAIS_POR_BASE["dados_info_empresa"]),
        construir_completude_colunas(df_indicadores_trimestrais_padronizada, "indicadores_trimestrais", COLUNAS_CENTRAIS_POR_BASE["indicadores_trimestrais"]),
        construir_completude_colunas(df_ibov_padronizada, "ibovespa_yahoo", COLUNAS_CENTRAIS_POR_BASE["ibovespa_yahoo"]),
        construir_completude_colunas(df_selic_padronizada, "selic_sgs_11", COLUNAS_CENTRAIS_POR_BASE["selic_sgs_11"]),
        construir_completude_colunas(df_cdi_padronizada, "cdi_sgs_12", COLUNAS_CENTRAIS_POR_BASE["cdi_sgs_12"]),
    ],
    ignore_index=True,
)

print("Tabela de completude das colunas centrais construída com sucesso.")
print("OK")

# ============================================================
# 9) Construção da auditoria de limitações das bases
# ============================================================

print("\n[9/11] Construção da auditoria de limitações das bases...")

data_final_mercado = pd.to_datetime(df_cot_hist_padronizada["data"], errors="coerce").max()
data_final_ibov = pd.to_datetime(df_ibov_padronizada["data"], errors="coerce").max()
data_final_selic = pd.to_datetime(df_selic_padronizada["data"], errors="coerce").max()
data_final_cdi = pd.to_datetime(df_cdi_padronizada["data"], errors="coerce").max()

n_info_empresa_sem_classificacao = int(
    df_info_empresa_padronizada[["setor", "subsetor", "segmento"]].isna().any(axis=1).sum()
)

colunas_contabeis_principais = [
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

n_indicadores_com_lacuna = int(
    df_indicadores_trimestrais_padronizada[colunas_contabeis_principais].isna().any(axis=1).sum()
)

n_empresas_unico_trimestre = int(
    (
        df_indicadores_trimestrais_padronizada
        .groupby("issuer_code", dropna=True)["periodo"]
        .nunique()
        .fillna(0)
        .le(1)
        .sum()
    )
)

registros_limitacoes = [
    {
        "categoria": "cobertura_temporal",
        "item": "mercado_termina_antes_do_ibovespa",
        "valor_observado": int((data_final_ibov - data_final_mercado).days),
        "unidade": "dias",
        "descricao": "A base de mercado termina antes da série do Ibovespa e exigirá harmonização temporal posterior.",
    },
    {
        "categoria": "cobertura_temporal",
        "item": "mercado_termina_antes_da_selic",
        "valor_observado": int((data_final_selic - data_final_mercado).days),
        "unidade": "dias",
        "descricao": "A base de mercado termina antes da série da Selic e exigirá harmonização temporal posterior.",
    },
    {
        "categoria": "cobertura_temporal",
        "item": "mercado_termina_antes_do_cdi",
        "valor_observado": int((data_final_cdi - data_final_mercado).days),
        "unidade": "dias",
        "descricao": "A base de mercado termina antes da série do CDI e exigirá harmonização temporal posterior.",
    },
    {
        "categoria": "cadastro_empresa",
        "item": "registros_sem_classificacao_setorial_completa",
        "valor_observado": n_info_empresa_sem_classificacao,
        "unidade": "registros",
        "descricao": "Existem registros cadastrais sem preenchimento completo de setor, subsetor e segmento.",
    },
    {
        "categoria": "base_contabil",
        "item": "registros_com_lacuna_em_variaveis_contabeis_principais",
        "valor_observado": n_indicadores_com_lacuna,
        "unidade": "registros",
        "descricao": "Existem observações trimestrais com ausência em pelo menos uma variável contábil principal.",
    },
    {
        "categoria": "base_contabil",
        "item": "empresas_com_cobertura_trimestral_muito_curta",
        "valor_observado": n_empresas_unico_trimestre,
        "unidade": "empresas",
        "descricao": "Existem empresas com cobertura contábil observada em apenas um trimestre na base padronizada.",
    },
]

df_auditoria_limitacoes_bases = pd.DataFrame(registros_limitacoes)

print("Auditoria de limitações das bases construída com sucesso.")
print("OK")

# ============================================================
# 10) Salvamento dos outputs da subetapa
# ============================================================

print("\n[10/11] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_diagnostico_bases, caminho_diagnostico_bases, index=False)
salvar_dataframe(df_cobertura_temporal, caminho_cobertura_temporal, index=False)
salvar_dataframe(df_completude_colunas, caminho_completude_colunas, index=False)
salvar_dataframe(df_auditoria_limitacoes_bases, caminho_auditoria_limitacoes, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 11) Validação final da subetapa
# ============================================================

print("\n[11/11] Validação final da subetapa...")

print("\nDiagnóstico consolidado das bases:")
print(df_diagnostico_bases.to_string(index=False))

print("\nComparação de cobertura temporal contra a base de mercado:")
print(
    df_cobertura_temporal[
        df_cobertura_temporal["tipo_registro"].eq("comparacao_mercado")
    ].to_string(index=False)
)

print("\nCompletude das colunas centrais com maior percentual de ausência:")
print(
    df_completude_colunas
    .sort_values(["pct_missing", "base", "coluna"], ascending=[False, True, True])
    .head(20)
    .to_string(index=False)
)

print("\nAuditoria de limitações identificadas:")
print(df_auditoria_limitacoes_bases.to_string(index=False))

print("\nCobertura contábil por trimestre - amostra inicial:")
print(df_cobertura_contabil_periodo.head(10).to_string(index=False))

print("\nArquivos salvos na subetapa 1.4:")
print(f"- {caminho_diagnostico_bases}")
print(f"- {caminho_cobertura_temporal}")
print(f"- {caminho_completude_colunas}")
print(f"- {caminho_auditoria_limitacoes}")

print("\nETAPA 1.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 1.4 - DIAGNÓSTICO INICIAL DAS BASES

[1/11] Validação inicial do ambiente...
OK

[2/11] Definição dos caminhos de entrada e saída da subetapa...
Entrada - cot_hist        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_cot_hist_padronizada.parquet
Entrada - info_empresa    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_info_empresa_padronizada.parquet
Entrada - indicadores     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_indicadores_trimestrais_padronizada.parquet
Entrada - ibov            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_ibov_padronizada.parquet
Entrada - selic           : C:\Users\mht

# Etapa 2) Construção da Base de Mercado Operacional

## Etapa 2.1) Organização da Série Diária de Preços e Liquidez

In [7]:
%%time
# ============================================================
# Etapa 2.1) Organização da Série Diária de Preços e Liquidez
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 2.1 - ORGANIZAÇÃO DA SÉRIE DIÁRIA DE PREÇOS E LIQUIDEZ")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_cot_hist_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="cot_hist_padronizada")

caminho_base_mercado_diario_organizada = gerar_caminho_arquivo(etapa=2, subetapa=1, tipo_arquivo="base", nome="mercado_diario_organizada")
caminho_tbl_resumo_mercado_diario = gerar_caminho_arquivo(etapa=2, subetapa=1, tipo_arquivo="tbl", nome="resumo_mercado_diario")
caminho_tbl_validacao_continuidade = gerar_caminho_arquivo(etapa=2, subetapa=1, tipo_arquivo="tbl", nome="validacao_continuidade_series")
caminho_tbl_tickers_por_data = gerar_caminho_arquivo(etapa=2, subetapa=1, tipo_arquivo="tbl", nome="tickers_por_data")

print(f"Entrada - cot_hist padronizada   : {caminho_cot_hist_padronizada}")
print(f"Saída   - base organizada        : {caminho_base_mercado_diario_organizada}")
print(f"Saída   - resumo                 : {caminho_tbl_resumo_mercado_diario}")
print(f"Saída   - continuidade           : {caminho_tbl_validacao_continuidade}")
print(f"Saída   - tickers por data       : {caminho_tbl_tickers_por_data}")
print("OK")

# ============================================================
# 3) Carga da base padronizada da subetapa 1.3
# ============================================================

print("\n[3/10] Carga da base padronizada da subetapa 1.3...")

df_cot_hist_padronizada = pd.read_parquet(caminho_cot_hist_padronizada)

print(f"cot_hist_consolidado padronizada: {df_cot_hist_padronizada.shape[0]:,} linhas x {df_cot_hist_padronizada.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros operacionais e funções auxiliares da subetapa
# ============================================================

print("\n[4/10] Parâmetros operacionais e funções auxiliares da subetapa...")

COLUNAS_OPERACIONAIS_MERCADO = [
    "ticker",
    "data",
    "close",
    "close_adj",
    "volume_qtd",
    "volume_fin",
    "negocios",
]

COLUNAS_PRINCIPAIS_OPERACIONAIS = [
    "close_adj",
    "volume_fin",
    "volume_qtd",
    "negocios",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def construir_tabela_continuidade_por_ticker(df):
    """
    Constrói uma tabela de validação de continuidade temporal por ticker.
    """
    registros = []

    for ticker, df_ticker in df.groupby("ticker", dropna=False):
        df_ticker = df_ticker.sort_values("data").reset_index(drop=True)
        diferencas_dias = df_ticker["data"].diff().dt.days

        n_diferencas_nao_positivas = int(diferencas_dias.dropna().le(0).sum())
        n_datas_duplicadas = int(df_ticker.duplicated(subset=["data"]).sum())

        registros.append(
            {
                "ticker": ticker,
                "n_registros": int(len(df_ticker)),
                "n_datas_unicas": int(df_ticker["data"].nunique(dropna=True)),
                "data_min": df_ticker["data"].min(),
                "data_max": df_ticker["data"].max(),
                "n_datas_duplicadas": n_datas_duplicadas,
                "n_diferencas_nao_positivas": n_diferencas_nao_positivas,
                "menor_intervalo_dias": diferencas_dias.dropna().min(),
                "maior_intervalo_dias": diferencas_dias.dropna().max(),
                "serie_temporal_valida": (n_datas_duplicadas == 0) and (n_diferencas_nao_positivas == 0),
            }
        )

    return pd.DataFrame(registros)

print(f"Quantidade de colunas operacionais selecionadas: {len(COLUNAS_OPERACIONAIS_MERCADO)}")
print(f"Colunas principais operacionais: {', '.join(COLUNAS_PRINCIPAIS_OPERACIONAIS)}")
print("OK")

# ============================================================
# 5) Seleção e organização da base diária de mercado
# ============================================================

print("\n[5/10] Seleção e organização da base diária de mercado...")

validar_colunas_obrigatorias(df_cot_hist_padronizada, COLUNAS_OPERACIONAIS_MERCADO)

df_mercado_diario_organizada = df_cot_hist_padronizada[COLUNAS_OPERACIONAIS_MERCADO].copy()
df_mercado_diario_organizada["ticker"] = df_mercado_diario_organizada["ticker"].astype("string").str.strip().str.upper()
df_mercado_diario_organizada["data"] = pd.to_datetime(df_mercado_diario_organizada["data"], errors="coerce")
df_mercado_diario_organizada["close"] = pd.to_numeric(df_mercado_diario_organizada["close"], errors="coerce")
df_mercado_diario_organizada["close_adj"] = pd.to_numeric(df_mercado_diario_organizada["close_adj"], errors="coerce")
df_mercado_diario_organizada["volume_qtd"] = pd.to_numeric(df_mercado_diario_organizada["volume_qtd"], errors="coerce")
df_mercado_diario_organizada["volume_fin"] = pd.to_numeric(df_mercado_diario_organizada["volume_fin"], errors="coerce")
df_mercado_diario_organizada["negocios"] = pd.to_numeric(df_mercado_diario_organizada["negocios"], errors="coerce")
df_mercado_diario_organizada = (
    df_mercado_diario_organizada
    .dropna(subset=["ticker", "data"])
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

print(f"Base diária organizada: {df_mercado_diario_organizada.shape[0]:,} linhas x {df_mercado_diario_organizada.shape[1]} colunas")
print("OK")

# ============================================================
# 6) Validação estrutural e de continuidade temporal
# ============================================================

print("\n[6/10] Validação estrutural e de continuidade temporal...")

df_validacao_continuidade_series = construir_tabela_continuidade_por_ticker(df_mercado_diario_organizada)

n_linhas_com_ticker_nulo = int(df_mercado_diario_organizada["ticker"].isna().sum())
n_linhas_com_data_nula = int(df_mercado_diario_organizada["data"].isna().sum())
n_duplicidades_ticker_data = int(df_mercado_diario_organizada.duplicated(subset=["ticker", "data"]).sum())
n_tickers_com_problema_continuidade = int((~df_validacao_continuidade_series["serie_temporal_valida"]).sum())

if n_linhas_com_ticker_nulo > 0:
    raise ValueError(f"Foram encontradas {n_linhas_com_ticker_nulo} linhas com ticker nulo após a organização da base.")

if n_linhas_com_data_nula > 0:
    raise ValueError(f"Foram encontradas {n_linhas_com_data_nula} linhas com data nula após a organização da base.")

if n_duplicidades_ticker_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_ticker_data} duplicidades na chave natural ticker-data.")

if n_tickers_com_problema_continuidade > 0:
    raise ValueError(
        f"Foram identificados {n_tickers_com_problema_continuidade} tickers com quebra de continuidade temporal "
        "ou datas repetidas na série organizada."
    )

print(f"Duplicidades na chave ticker-data        : {n_duplicidades_ticker_data}")
print(f"Tickers com problema de continuidade     : {n_tickers_com_problema_continuidade}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas-resumo da subetapa...")

df_resumo_mercado_diario = pd.DataFrame(
    [
        {
            "base": "mercado_diario_organizada",
            "n_linhas": int(len(df_mercado_diario_organizada)),
            "n_colunas": int(df_mercado_diario_organizada.shape[1]),
            "n_tickers": int(df_mercado_diario_organizada["ticker"].nunique(dropna=True)),
            "n_datas_unicas": int(df_mercado_diario_organizada["data"].nunique(dropna=True)),
            "data_min": df_mercado_diario_organizada["data"].min(),
            "data_max": df_mercado_diario_organizada["data"].max(),
            "missing_close": int(df_mercado_diario_organizada["close"].isna().sum()),
            "missing_close_adj": int(df_mercado_diario_organizada["close_adj"].isna().sum()),
            "missing_volume_qtd": int(df_mercado_diario_organizada["volume_qtd"].isna().sum()),
            "missing_volume_fin": int(df_mercado_diario_organizada["volume_fin"].isna().sum()),
            "missing_negocios": int(df_mercado_diario_organizada["negocios"].isna().sum()),
        }
    ]
)

df_tickers_por_data = (
    df_mercado_diario_organizada
    .groupby("data", as_index=False)
    .agg(
        n_tickers=("ticker", "nunique"),
        volume_fin_total=("volume_fin", "sum"),
        volume_qtd_total=("volume_qtd", "sum"),
        negocios_total=("negocios", "sum"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

print("Tabelas-resumo construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_mercado_diario_organizada, caminho_base_mercado_diario_organizada, index=False)
salvar_dataframe(df_resumo_mercado_diario, caminho_tbl_resumo_mercado_diario, index=False)
salvar_dataframe(df_validacao_continuidade_series, caminho_tbl_validacao_continuidade, index=False)
salvar_dataframe(df_tickers_por_data, caminho_tbl_tickers_por_data, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_tickers_continuidade = (
    df_validacao_continuidade_series
    .sort_values(["serie_temporal_valida", "ticker"], ascending=[True, True])
    .head(10)
    .reset_index(drop=True)
)

df_amostra_tickers_por_data_inicial = df_tickers_por_data.head(10).copy()
df_amostra_tickers_por_data_final = df_tickers_por_data.tail(10).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base diária de mercado organizada:")
print(df_resumo_mercado_diario.to_string(index=False))

print("\nAmostra da base diária de mercado organizada:")
print(df_mercado_diario_organizada.head(5).to_string(index=False))

print("\nValidação de continuidade temporal por ticker - amostra:")
print(df_amostra_tickers_continuidade.to_string(index=False))

print("\nQuantidade de tickers observados por data - amostra inicial:")
print(df_amostra_tickers_por_data_inicial.to_string(index=False))

print("\nQuantidade de tickers observados por data - amostra final:")
print(df_amostra_tickers_por_data_final.to_string(index=False))

print("\nArquivos salvos na subetapa 2.1:")
print(f"- {caminho_base_mercado_diario_organizada}")
print(f"- {caminho_tbl_resumo_mercado_diario}")
print(f"- {caminho_tbl_validacao_continuidade}")
print(f"- {caminho_tbl_tickers_por_data}")

print("\nETAPA 2.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 2.1 - ORGANIZAÇÃO DA SÉRIE DIÁRIA DE PREÇOS E LIQUIDEZ

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - cot_hist padronizada   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_cot_hist_padronizada.parquet
Saída   - base organizada        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_1_base_mercado_diario_organizada.parquet
Saída   - resumo                 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_1_tbl_resumo_mercado_diario.parquet
Saída   - continuidade           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_1_tbl_validacao_continuidade_series.par

## Etapa 2.2) Tratamento de Cobertura Histórica por Ticker

In [8]:
%%time
# ============================================================
# Etapa 2.2) Tratamento de Cobertura Histórica por Ticker
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 2.2 - TRATAMENTO DE COBERTURA HISTÓRICA POR TICKER")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_organizada = gerar_caminho_arquivo(etapa=2, subetapa=1, tipo_arquivo="base", nome="mercado_diario_organizada")
caminho_tbl_tickers_por_data = gerar_caminho_arquivo(etapa=2, subetapa=1, tipo_arquivo="tbl", nome="tickers_por_data")

caminho_base_cobertura_ticker = gerar_caminho_arquivo(etapa=2, subetapa=2, tipo_arquivo="base", nome="cobertura_historica_ticker")
caminho_tbl_resumo_cobertura_ticker = gerar_caminho_arquivo(etapa=2, subetapa=2, tipo_arquivo="tbl", nome="resumo_cobertura_historica_ticker")
caminho_tbl_distribuicao_cobertura_ticker = gerar_caminho_arquivo(etapa=2, subetapa=2, tipo_arquivo="tbl", nome="distribuicao_cobertura_historica_ticker")

print(f"Entrada - base mercado diária    : {caminho_base_mercado_diario_organizada}")
print(f"Entrada - tickers por data       : {caminho_tbl_tickers_por_data}")
print(f"Saída   - cobertura por ticker   : {caminho_base_cobertura_ticker}")
print(f"Saída   - resumo                 : {caminho_tbl_resumo_cobertura_ticker}")
print(f"Saída   - distribuição           : {caminho_tbl_distribuicao_cobertura_ticker}")
print("OK")

# ============================================================
# 3) Carga dos outputs da subetapa 2.1
# ============================================================

print("\n[3/10] Carga dos outputs da subetapa 2.1...")

df_mercado_diario_organizada = pd.read_parquet(caminho_base_mercado_diario_organizada)
df_tickers_por_data = pd.read_parquet(caminho_tbl_tickers_por_data)

print(f"Base mercado diária organizada : {df_mercado_diario_organizada.shape[0]:,} linhas x {df_mercado_diario_organizada.shape[1]} colunas")
print(f"Tabela de tickers por data     : {df_tickers_por_data.shape[0]:,} linhas x {df_tickers_por_data.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros diagnósticos e funções auxiliares
# ============================================================

print("\n[4/10] Parâmetros diagnósticos e funções auxiliares...")

LIMIAR_DIAGNOSTICO_PREGOES_CURTO = 252
LIMIAR_DIAGNOSTICO_COBERTURA_INSUFICIENTE = 0.80
LIMIAR_DIAGNOSTICO_GAP_LONGO_DIAS = 30

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def classificar_faixa_n_pregoes(n_pregoes):
    """
    Classifica a cobertura histórica por quantidade de pregões observados.
    """
    if pd.isna(n_pregoes):
        return pd.NA
    if n_pregoes < 63:
        return "menos_de_63"
    if n_pregoes < 126:
        return "63_a_125"
    if n_pregoes < 252:
        return "126_a_251"
    if n_pregoes < 504:
        return "252_a_503"
    if n_pregoes < 1000:
        return "504_a_999"
    return "1000_ou_mais"

def classificar_faixa_pct_cobertura(pct_cobertura):
    """
    Classifica a cobertura relativa de pregões observados dentro do intervalo de vida do ticker.
    """
    if pd.isna(pct_cobertura):
        return pd.NA
    if pct_cobertura < 0.50:
        return "abaixo_de_50pct"
    if pct_cobertura < 0.80:
        return "50pct_a_79pct"
    if pct_cobertura < 0.95:
        return "80pct_a_94pct"
    return "95pct_ou_mais"

def contar_pregoes_intervalo(indice_datas_mercado, data_inicial, data_final):
    """
    Conta quantos pregões do calendário de mercado existem dentro de um intervalo fechado.
    """
    if pd.isna(data_inicial) or pd.isna(data_final):
        return 0
    posicao_inicial = int(indice_datas_mercado.searchsorted(pd.Timestamp(data_inicial), side="left"))
    posicao_final = int(indice_datas_mercado.searchsorted(pd.Timestamp(data_final), side="right"))
    return max(0, posicao_final - posicao_inicial)

def construir_base_cobertura_ticker(df_mercado, indice_datas_mercado):
    """
    Constrói a base-resumo de cobertura histórica por ticker.
    """
    registros = []

    for ticker, df_ticker in df_mercado.groupby("ticker", dropna=False):
        df_ticker = df_ticker.sort_values("data").reset_index(drop=True)
        data_min = df_ticker["data"].min()
        data_max = df_ticker["data"].max()
        n_registros = int(len(df_ticker))
        n_datas_unicas = int(df_ticker["data"].nunique(dropna=True))
        n_pregoes_intervalo_mercado = contar_pregoes_intervalo(indice_datas_mercado, data_min, data_max)
        pct_cobertura_pregoes = n_datas_unicas / n_pregoes_intervalo_mercado if n_pregoes_intervalo_mercado > 0 else pd.NA
        diferencas_dias = df_ticker["data"].diff().dt.days.dropna()

        if len(diferencas_dias) > 0:
            menor_intervalo_dias = diferencas_dias.min()
            mediana_intervalo_dias = diferencas_dias.median()
            maior_intervalo_dias = diferencas_dias.max()
            n_gaps_acima_5_dias = int(diferencas_dias.gt(5).sum())
            n_gaps_acima_20_dias = int(diferencas_dias.gt(20).sum())
            n_gaps_acima_30_dias = int(diferencas_dias.gt(30).sum())
        else:
            menor_intervalo_dias = pd.NA
            mediana_intervalo_dias = pd.NA
            maior_intervalo_dias = pd.NA
            n_gaps_acima_5_dias = 0
            n_gaps_acima_20_dias = 0
            n_gaps_acima_30_dias = 0

        registros.append(
            {
                "ticker": ticker,
                "data_min": data_min,
                "data_max": data_max,
                "dias_calendario_vida": int((data_max - data_min).days + 1) if pd.notna(data_min) and pd.notna(data_max) else pd.NA,
                "n_registros": n_registros,
                "n_datas_unicas": n_datas_unicas,
                "n_pregoes_intervalo_mercado": n_pregoes_intervalo_mercado,
                "pct_cobertura_pregoes_intervalo": pct_cobertura_pregoes,
                "menor_intervalo_dias": menor_intervalo_dias,
                "mediana_intervalo_dias": mediana_intervalo_dias,
                "maior_intervalo_dias": maior_intervalo_dias,
                "n_gaps_acima_5_dias": n_gaps_acima_5_dias,
                "n_gaps_acima_20_dias": n_gaps_acima_20_dias,
                "n_gaps_acima_30_dias": n_gaps_acima_30_dias,
                "faixa_n_pregoes": classificar_faixa_n_pregoes(n_datas_unicas),
                "faixa_pct_cobertura": classificar_faixa_pct_cobertura(pct_cobertura_pregoes),
                "flag_diagnostico_historico_curto": bool(n_datas_unicas < LIMIAR_DIAGNOSTICO_PREGOES_CURTO),
                "flag_diagnostico_cobertura_insuficiente": bool(pd.notna(pct_cobertura_pregoes) and pct_cobertura_pregoes < LIMIAR_DIAGNOSTICO_COBERTURA_INSUFICIENTE),
                "flag_diagnostico_gap_longo": bool(pd.notna(maior_intervalo_dias) and maior_intervalo_dias > LIMIAR_DIAGNOSTICO_GAP_LONGO_DIAS),
            }
        )

    return pd.DataFrame(registros)

print(f"Limiar diagnóstico - histórico curto         : {LIMIAR_DIAGNOSTICO_PREGOES_CURTO} pregões")
print(f"Limiar diagnóstico - cobertura insuficiente  : {LIMIAR_DIAGNOSTICO_COBERTURA_INSUFICIENTE:.0%}")
print(f"Limiar diagnóstico - gap longo               : {LIMIAR_DIAGNOSTICO_GAP_LONGO_DIAS} dias")
print("OK")

# ============================================================
# 5) Preparação do calendário de referência do mercado
# ============================================================

print("\n[5/10] Preparação do calendário de referência do mercado...")

validar_colunas_obrigatorias(df_mercado_diario_organizada, ["ticker", "data"])
validar_colunas_obrigatorias(df_tickers_por_data, ["data"])

df_tickers_por_data["data"] = pd.to_datetime(df_tickers_por_data["data"], errors="coerce")
df_tickers_por_data = df_tickers_por_data.dropna(subset=["data"]).sort_values("data").reset_index(drop=True)

indice_datas_mercado = pd.Index(df_tickers_por_data["data"].drop_duplicates().sort_values())

print(f"Quantidade de pregões únicos no calendário de referência: {len(indice_datas_mercado):,}")
print(f"Data inicial do calendário de referência                : {indice_datas_mercado.min()}")
print(f"Data final do calendário de referência                  : {indice_datas_mercado.max()}")
print("OK")

# ============================================================
# 6) Construção da base-resumo de cobertura histórica por ticker
# ============================================================

print("\n[6/10] Construção da base-resumo de cobertura histórica por ticker...")

df_cobertura_historica_ticker = construir_base_cobertura_ticker(df_mercado_diario_organizada, indice_datas_mercado)
df_cobertura_historica_ticker = df_cobertura_historica_ticker.sort_values(["ticker"]).reset_index(drop=True)

print(f"Base de cobertura histórica por ticker: {df_cobertura_historica_ticker.shape[0]:,} linhas x {df_cobertura_historica_ticker.shape[1]} colunas")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas-resumo da subetapa...")

df_resumo_cobertura_historica_ticker = pd.DataFrame(
    [
        {
            "n_tickers_total": int(df_cobertura_historica_ticker["ticker"].nunique(dropna=True)),
            "n_tickers_historico_curto": int(df_cobertura_historica_ticker["flag_diagnostico_historico_curto"].sum()),
            "n_tickers_cobertura_insuficiente": int(df_cobertura_historica_ticker["flag_diagnostico_cobertura_insuficiente"].sum()),
            "n_tickers_gap_longo": int(df_cobertura_historica_ticker["flag_diagnostico_gap_longo"].sum()),
            "n_tickers_com_pelo_menos_252_pregoes": int(df_cobertura_historica_ticker["n_datas_unicas"].ge(252).sum()),
            "pct_tickers_historico_curto": float(df_cobertura_historica_ticker["flag_diagnostico_historico_curto"].mean()),
            "pct_tickers_cobertura_insuficiente": float(df_cobertura_historica_ticker["flag_diagnostico_cobertura_insuficiente"].mean()),
            "pct_tickers_gap_longo": float(df_cobertura_historica_ticker["flag_diagnostico_gap_longo"].mean()),
            "n_pregoes_minimo_observado": int(df_cobertura_historica_ticker["n_datas_unicas"].min()),
            "n_pregoes_mediano_observado": float(df_cobertura_historica_ticker["n_datas_unicas"].median()),
            "n_pregoes_maximo_observado": int(df_cobertura_historica_ticker["n_datas_unicas"].max()),
            "pct_cobertura_minima_observada": float(df_cobertura_historica_ticker["pct_cobertura_pregoes_intervalo"].min()),
            "pct_cobertura_mediana_observada": float(df_cobertura_historica_ticker["pct_cobertura_pregoes_intervalo"].median()),
            "pct_cobertura_maxima_observada": float(df_cobertura_historica_ticker["pct_cobertura_pregoes_intervalo"].max()),
        }
    ]
)

df_distribuicao_cobertura_historica_ticker = (
    df_cobertura_historica_ticker
    .groupby(["faixa_n_pregoes", "faixa_pct_cobertura"], dropna=False)
    .agg(
        n_tickers=("ticker", "nunique"),
        pct_cobertura_mediana=("pct_cobertura_pregoes_intervalo", "median"),
        n_pregoes_mediana=("n_datas_unicas", "median"),
    )
    .reset_index()
    .sort_values(["faixa_n_pregoes", "faixa_pct_cobertura"])
    .reset_index(drop=True)
)

print("Tabelas-resumo construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_cobertura_historica_ticker, caminho_base_cobertura_ticker, index=False)
salvar_dataframe(df_resumo_cobertura_historica_ticker, caminho_tbl_resumo_cobertura_ticker, index=False)
salvar_dataframe(df_distribuicao_cobertura_historica_ticker, caminho_tbl_distribuicao_cobertura_ticker, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_menor_historico = (
    df_cobertura_historica_ticker
    .sort_values(["n_datas_unicas", "pct_cobertura_pregoes_intervalo", "ticker"], ascending=[True, True, True])
    .head(15)
    .reset_index(drop=True)
)

df_amostra_menor_cobertura = (
    df_cobertura_historica_ticker
    .sort_values(["pct_cobertura_pregoes_intervalo", "n_datas_unicas", "ticker"], ascending=[True, True, True])
    .head(15)
    .reset_index(drop=True)
)

df_amostra_maior_gap = (
    df_cobertura_historica_ticker
    .sort_values(["maior_intervalo_dias", "n_datas_unicas", "ticker"], ascending=[False, True, True])
    .head(15)
    .reset_index(drop=True)
)

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da cobertura histórica por ticker:")
print(df_resumo_cobertura_historica_ticker.to_string(index=False))

print("\nDistribuição da cobertura histórica por faixa:")
print(df_distribuicao_cobertura_historica_ticker.to_string(index=False))

print("\nTickers com menor histórico observado - amostra:")
print(df_amostra_menor_historico.to_string(index=False))

print("\nTickers com menor cobertura relativa dentro do próprio intervalo - amostra:")
print(df_amostra_menor_cobertura.to_string(index=False))

print("\nTickers com maior intervalo entre observações - amostra:")
print(df_amostra_maior_gap.to_string(index=False))

print("\nArquivos salvos na subetapa 2.2:")
print(f"- {caminho_base_cobertura_ticker}")
print(f"- {caminho_tbl_resumo_cobertura_ticker}")
print(f"- {caminho_tbl_distribuicao_cobertura_ticker}")

print("\nETAPA 2.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 2.2 - TRATAMENTO DE COBERTURA HISTÓRICA POR TICKER

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - base mercado diária    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_1_base_mercado_diario_organizada.parquet
Entrada - tickers por data       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_1_tbl_tickers_por_data.parquet
Saída   - cobertura por ticker   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_2_base_cobertura_historica_ticker.parquet
Saída   - resumo                 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_2_tbl_resumo_cobertura_historica_ticker.pa

## Etapa 2.3) Consolidação de Informações de Empresa por Ticker

In [9]:
%%time
# ============================================================
# Etapa 2.3) Consolidação de Informações de Empresa por Ticker
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 2.3 - CONSOLIDAÇÃO DE INFORMAÇÕES DE EMPRESA POR TICKER")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_organizada = gerar_caminho_arquivo(etapa=2, subetapa=1, tipo_arquivo="base", nome="mercado_diario_organizada")
caminho_info_empresa_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="info_empresa_padronizada")

caminho_base_mercado_diario_com_empresa = gerar_caminho_arquivo(etapa=2, subetapa=3, tipo_arquivo="base", nome="mercado_diario_com_empresa")
caminho_tbl_resumo_cruzamento = gerar_caminho_arquivo(etapa=2, subetapa=3, tipo_arquivo="tbl", nome="resumo_cruzamento_mercado_empresa")
caminho_tbl_auditoria_mapeamento = gerar_caminho_arquivo(etapa=2, subetapa=3, tipo_arquivo="tbl", nome="auditoria_mapeamento_ticker_empresa")
caminho_tbl_inconsistencias_mapeamento = gerar_caminho_arquivo(etapa=2, subetapa=3, tipo_arquivo="tbl", nome="inconsistencias_mapeamento_ticker_empresa")
caminho_tbl_tickers_sem_mapeamento = gerar_caminho_arquivo(etapa=2, subetapa=3, tipo_arquivo="tbl", nome="tickers_sem_mapeamento_empresa")

print(f"Entrada - mercado diário organizada : {caminho_base_mercado_diario_organizada}")
print(f"Entrada - info empresa padronizada  : {caminho_info_empresa_padronizada}")
print(f"Saída   - base consolidada          : {caminho_base_mercado_diario_com_empresa}")
print(f"Saída   - resumo cruzamento         : {caminho_tbl_resumo_cruzamento}")
print(f"Saída   - auditoria mapeamento      : {caminho_tbl_auditoria_mapeamento}")
print(f"Saída   - inconsistências           : {caminho_tbl_inconsistencias_mapeamento}")
print(f"Saída   - tickers sem mapeamento    : {caminho_tbl_tickers_sem_mapeamento}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_mercado_diario_organizada = pd.read_parquet(caminho_base_mercado_diario_organizada)
df_info_empresa_padronizada = pd.read_parquet(caminho_info_empresa_padronizada)

print(f"Base mercado diária organizada : {df_mercado_diario_organizada.shape[0]:,} linhas x {df_mercado_diario_organizada.shape[1]} colunas")
print(f"Base info empresa padronizada  : {df_info_empresa_padronizada.shape[0]:,} linhas x {df_info_empresa_padronizada.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/10] Funções auxiliares da subetapa...")

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def primeiro_valor_nao_nulo(serie):
    """
    Retorna o primeiro valor não nulo de uma série.
    """
    serie = serie.dropna()
    if len(serie) == 0:
        return pd.NA
    return serie.iloc[0]

def contar_unicos_nao_nulos(serie):
    """
    Conta valores únicos não nulos em uma série.
    """
    return int(serie.dropna().nunique())

def construir_cadastro_por_ticker(df_info_empresa):
    """
    Consolida o cadastro para uma linha por ticker, preservando auditoria das possíveis inconsistências.
    """
    df_cadastro_auditoria = (
        df_info_empresa
        .groupby("ticker", dropna=False)
        .agg(
            n_registros=("ticker", "size"),
            n_issuer_code_unicos=("issuer_code", contar_unicos_nao_nulos),
            n_nome_unicos=("nome", contar_unicos_nao_nulos),
            n_setor_unicos=("setor", contar_unicos_nao_nulos),
            n_subsetor_unicos=("subsetor", contar_unicos_nao_nulos),
            n_segmento_unicos=("segmento", contar_unicos_nao_nulos),
            issuer_code_exemplo=("issuer_code", primeiro_valor_nao_nulo),
            nome_exemplo=("nome", primeiro_valor_nao_nulo),
            setor_exemplo=("setor", primeiro_valor_nao_nulo),
            subsetor_exemplo=("subsetor", primeiro_valor_nao_nulo),
            segmento_exemplo=("segmento", primeiro_valor_nao_nulo),
        )
        .reset_index()
        .sort_values("ticker")
        .reset_index(drop=True)
    )

    df_cadastro_auditoria["flag_inconsistencia_issuer_code"] = df_cadastro_auditoria["n_issuer_code_unicos"].gt(1)
    df_cadastro_auditoria["flag_inconsistencia_nome"] = df_cadastro_auditoria["n_nome_unicos"].gt(1)
    df_cadastro_auditoria["flag_inconsistencia_setor"] = df_cadastro_auditoria["n_setor_unicos"].gt(1)
    df_cadastro_auditoria["flag_inconsistencia_subsetor"] = df_cadastro_auditoria["n_subsetor_unicos"].gt(1)
    df_cadastro_auditoria["flag_inconsistencia_segmento"] = df_cadastro_auditoria["n_segmento_unicos"].gt(1)
    df_cadastro_auditoria["flag_inconsistencia_qualquer"] = (
        df_cadastro_auditoria["flag_inconsistencia_issuer_code"]
        | df_cadastro_auditoria["flag_inconsistencia_nome"]
        | df_cadastro_auditoria["flag_inconsistencia_setor"]
        | df_cadastro_auditoria["flag_inconsistencia_subsetor"]
        | df_cadastro_auditoria["flag_inconsistencia_segmento"]
    )

    df_cadastro_por_ticker = (
        df_info_empresa
        .groupby("ticker", dropna=False, as_index=False)
        .agg(
            issuer_code=("issuer_code", primeiro_valor_nao_nulo),
            nome=("nome", primeiro_valor_nao_nulo),
            setor=("setor", primeiro_valor_nao_nulo),
            subsetor=("subsetor", primeiro_valor_nao_nulo),
            segmento=("segmento", primeiro_valor_nao_nulo),
        )
        .sort_values("ticker")
        .reset_index(drop=True)
    )

    return df_cadastro_por_ticker, df_cadastro_auditoria

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Preparação e validação estrutural das bases de entrada
# ============================================================

print("\n[5/10] Preparação e validação estrutural das bases de entrada...")

validar_colunas_obrigatorias(
    df_mercado_diario_organizada,
    ["ticker", "data", "close", "close_adj", "volume_qtd", "volume_fin", "negocios"],
)
validar_colunas_obrigatorias(
    df_info_empresa_padronizada,
    ["ticker", "issuer_code", "nome", "setor", "subsetor", "segmento"],
)

df_mercado_diario_organizada["ticker"] = df_mercado_diario_organizada["ticker"].astype("string").str.strip().str.upper()
df_info_empresa_padronizada["ticker"] = df_info_empresa_padronizada["ticker"].astype("string").str.strip().str.upper()
df_info_empresa_padronizada["issuer_code"] = df_info_empresa_padronizada["issuer_code"].astype("string").str.strip()

n_duplicidades_mercado_ticker_data = int(df_mercado_diario_organizada.duplicated(subset=["ticker", "data"]).sum())
if n_duplicidades_mercado_ticker_data > 0:
    raise ValueError(
        f"Foram encontradas {n_duplicidades_mercado_ticker_data} duplicidades na chave ticker-data da base diária de mercado."
    )

print(f"Duplicidades na chave ticker-data da base de mercado: {n_duplicidades_mercado_ticker_data}")
print("OK")

# ============================================================
# 6) Consolidação do cadastro por ticker e auditoria de inconsistências
# ============================================================

print("\n[6/10] Consolidação do cadastro por ticker e auditoria de inconsistências...")

df_cadastro_por_ticker, df_auditoria_mapeamento_ticker_empresa = construir_cadastro_por_ticker(df_info_empresa_padronizada)
df_inconsistencias_mapeamento_ticker_empresa = (
    df_auditoria_mapeamento_ticker_empresa
    .loc[df_auditoria_mapeamento_ticker_empresa["flag_inconsistencia_qualquer"]]
    .copy()
    .sort_values(["flag_inconsistencia_qualquer", "ticker"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Cadastro consolidado por ticker                 : {df_cadastro_por_ticker.shape[0]:,} linhas")
print(f"Tickers com alguma inconsistência cadastral     : {df_inconsistencias_mapeamento_ticker_empresa.shape[0]:,}")
print("OK")

# ============================================================
# 7) Cruzamento da base diária de mercado com o cadastro empresarial
# ============================================================

print("\n[7/10] Cruzamento da base diária de mercado com o cadastro empresarial...")

df_mercado_diario_com_empresa = df_mercado_diario_organizada.merge(
    df_cadastro_por_ticker,
    on="ticker",
    how="left",
    validate="many_to_one",
)

df_tickers_sem_mapeamento_empresa = (
    df_mercado_diario_com_empresa
    .loc[df_mercado_diario_com_empresa["issuer_code"].isna(), ["ticker"]]
    .drop_duplicates()
    .sort_values("ticker")
    .reset_index(drop=True)
)

df_mercado_diario_com_empresa = (
    df_mercado_diario_com_empresa
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

n_linhas_sem_issuer_code = int(df_mercado_diario_com_empresa["issuer_code"].isna().sum())
n_tickers_sem_mapeamento = int(df_tickers_sem_mapeamento_empresa["ticker"].nunique(dropna=True))

print(f"Linhas sem issuer_code após o cruzamento        : {n_linhas_sem_issuer_code:,}")
print(f"Tickers sem mapeamento cadastral                : {n_tickers_sem_mapeamento:,}")
print("OK")

# ============================================================
# 8) Construção das tabelas-resumo da subetapa
# ============================================================

print("\n[8/10] Construção das tabelas-resumo da subetapa...")

df_resumo_cruzamento_mercado_empresa = pd.DataFrame(
    [
        {
            "n_linhas_base_mercado": int(len(df_mercado_diario_organizada)),
            "n_linhas_base_consolidada": int(len(df_mercado_diario_com_empresa)),
            "n_tickers_base_mercado": int(df_mercado_diario_organizada["ticker"].nunique(dropna=True)),
            "n_tickers_cadastro": int(df_cadastro_por_ticker["ticker"].nunique(dropna=True)),
            "n_tickers_base_consolidada": int(df_mercado_diario_com_empresa["ticker"].nunique(dropna=True)),
            "n_issuer_code_base_consolidada": int(df_mercado_diario_com_empresa["issuer_code"].nunique(dropna=True)),
            "n_linhas_sem_issuer_code": n_linhas_sem_issuer_code,
            "n_tickers_sem_mapeamento": n_tickers_sem_mapeamento,
            "n_tickers_com_inconsistencia_cadastral": int(df_inconsistencias_mapeamento_ticker_empresa["ticker"].nunique(dropna=True)),
            "pct_linhas_sem_issuer_code": n_linhas_sem_issuer_code / len(df_mercado_diario_com_empresa) if len(df_mercado_diario_com_empresa) > 0 else pd.NA,
            "pct_tickers_sem_mapeamento": n_tickers_sem_mapeamento / df_mercado_diario_organizada["ticker"].nunique(dropna=True) if df_mercado_diario_organizada["ticker"].nunique(dropna=True) > 0 else pd.NA,
        }
    ]
)

print("Tabelas-resumo construídas com sucesso.")
print("OK")

# ============================================================
# 9) Salvamento dos outputs da subetapa
# ============================================================

print("\n[9/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_mercado_diario_com_empresa, caminho_base_mercado_diario_com_empresa, index=False)
salvar_dataframe(df_resumo_cruzamento_mercado_empresa, caminho_tbl_resumo_cruzamento, index=False)
salvar_dataframe(df_auditoria_mapeamento_ticker_empresa, caminho_tbl_auditoria_mapeamento, index=False)
salvar_dataframe(df_inconsistencias_mapeamento_ticker_empresa, caminho_tbl_inconsistencias_mapeamento, index=False)
salvar_dataframe(df_tickers_sem_mapeamento_empresa, caminho_tbl_tickers_sem_mapeamento, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo do cruzamento entre mercado e cadastro empresarial:")
print(df_resumo_cruzamento_mercado_empresa.to_string(index=False))

print("\nAmostra da base diária consolidada com informações de empresa:")
print(df_mercado_diario_com_empresa.head(5).to_string(index=False))

print("\nAuditoria do mapeamento ticker-empresa - amostra:")
print(df_auditoria_mapeamento_ticker_empresa.head(15).to_string(index=False))

print("\nTickers com inconsistência cadastral - amostra:")
print(df_inconsistencias_mapeamento_ticker_empresa.head(15).to_string(index=False))

print("\nTickers sem mapeamento empresarial - amostra:")
print(df_tickers_sem_mapeamento_empresa.head(15).to_string(index=False))

print("\nArquivos salvos na subetapa 2.3:")
print(f"- {caminho_base_mercado_diario_com_empresa}")
print(f"- {caminho_tbl_resumo_cruzamento}")
print(f"- {caminho_tbl_auditoria_mapeamento}")
print(f"- {caminho_tbl_inconsistencias_mapeamento}")
print(f"- {caminho_tbl_tickers_sem_mapeamento}")

print("\nETAPA 2.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 2.3 - CONSOLIDAÇÃO DE INFORMAÇÕES DE EMPRESA POR TICKER

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário organizada : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_1_base_mercado_diario_organizada.parquet
Entrada - info empresa padronizada  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_info_empresa_padronizada.parquet
Saída   - base consolidada          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_3_base_mercado_diario_com_empresa.parquet
Saída   - resumo cruzamento         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_3_tbl_resumo_cru

## Etapa 2.4) Base Diária Consolidada de Mercado

In [10]:
%%time
# ============================================================
# Etapa 2.4) Base Diária Consolidada de Mercado
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 2.4 - BASE DIÁRIA CONSOLIDADA DE MERCADO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_com_empresa = gerar_caminho_arquivo(etapa=2, subetapa=3, tipo_arquivo="base", nome="mercado_diario_com_empresa")
caminho_base_cobertura_historica_ticker = gerar_caminho_arquivo(etapa=2, subetapa=2, tipo_arquivo="base", nome="cobertura_historica_ticker")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="base", nome="mercado_diario_consolidada")
caminho_tbl_resumo_mercado_diario_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="tbl", nome="resumo_mercado_diario_consolidada")
caminho_tbl_cobertura_diaria_mercado_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="tbl", nome="cobertura_diaria_mercado_consolidada")
caminho_tbl_empresas_por_data_mercado_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="tbl", nome="empresas_por_data_mercado_consolidada")
caminho_tbl_auditoria_chaves_mercado_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="tbl", nome="auditoria_chaves_mercado_consolidada")

print(f"Entrada - mercado com empresa   : {caminho_base_mercado_diario_com_empresa}")
print(f"Entrada - cobertura por ticker  : {caminho_base_cobertura_historica_ticker}")
print(f"Saída   - base consolidada      : {caminho_base_mercado_diario_consolidada}")
print(f"Saída   - resumo                : {caminho_tbl_resumo_mercado_diario_consolidada}")
print(f"Saída   - cobertura diária      : {caminho_tbl_cobertura_diaria_mercado_consolidada}")
print(f"Saída   - empresas por data     : {caminho_tbl_empresas_por_data_mercado_consolidada}")
print(f"Saída   - auditoria de chaves   : {caminho_tbl_auditoria_chaves_mercado_consolidada}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_mercado_diario_com_empresa = pd.read_parquet(caminho_base_mercado_diario_com_empresa)
df_cobertura_historica_ticker = pd.read_parquet(caminho_base_cobertura_historica_ticker)

print(f"Base mercado diário com empresa : {df_mercado_diario_com_empresa.shape[0]:,} linhas x {df_mercado_diario_com_empresa.shape[1]} colunas")
print(f"Base cobertura por ticker       : {df_cobertura_historica_ticker.shape[0]:,} linhas x {df_cobertura_historica_ticker.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros operacionais e funções auxiliares
# ============================================================

print("\n[4/10] Parâmetros operacionais e funções auxiliares...")

COLUNAS_BASE_FINAL_MERCADO = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close",
    "close_adj",
    "volume_qtd",
    "volume_fin",
    "negocios",
]

COLUNAS_COBERTURA_TICKER_ANEXAS = [
    "ticker",
    "n_datas_unicas",
    "pct_cobertura_pregoes_intervalo",
    "maior_intervalo_dias",
    "flag_diagnostico_historico_curto",
    "flag_diagnostico_cobertura_insuficiente",
    "flag_diagnostico_gap_longo",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def contar_unicos_nao_nulos(serie):
    """
    Conta valores únicos não nulos em uma série.
    """
    return int(serie.dropna().nunique())

print(f"Quantidade de colunas da base final de mercado: {len(COLUNAS_BASE_FINAL_MERCADO)}")
print("OK")

# ============================================================
# 5) Preparação da base diária consolidada de mercado
# ============================================================

print("\n[5/10] Preparação da base diária consolidada de mercado...")

validar_colunas_obrigatorias(df_mercado_diario_com_empresa, COLUNAS_BASE_FINAL_MERCADO)
validar_colunas_obrigatorias(df_cobertura_historica_ticker, COLUNAS_COBERTURA_TICKER_ANEXAS)

df_mercado_diario_consolidada = df_mercado_diario_com_empresa[COLUNAS_BASE_FINAL_MERCADO].copy()
df_mercado_diario_consolidada["ticker"] = df_mercado_diario_consolidada["ticker"].astype("string").str.strip().str.upper()
df_mercado_diario_consolidada["issuer_code"] = df_mercado_diario_consolidada["issuer_code"].astype("string").str.strip()
df_mercado_diario_consolidada["data"] = pd.to_datetime(df_mercado_diario_consolidada["data"], errors="coerce")
df_mercado_diario_consolidada["close"] = pd.to_numeric(df_mercado_diario_consolidada["close"], errors="coerce")
df_mercado_diario_consolidada["close_adj"] = pd.to_numeric(df_mercado_diario_consolidada["close_adj"], errors="coerce")
df_mercado_diario_consolidada["volume_qtd"] = pd.to_numeric(df_mercado_diario_consolidada["volume_qtd"], errors="coerce")
df_mercado_diario_consolidada["volume_fin"] = pd.to_numeric(df_mercado_diario_consolidada["volume_fin"], errors="coerce")
df_mercado_diario_consolidada["negocios"] = pd.to_numeric(df_mercado_diario_consolidada["negocios"], errors="coerce")

df_mercado_diario_consolidada = df_mercado_diario_consolidada.merge(
    df_cobertura_historica_ticker[COLUNAS_COBERTURA_TICKER_ANEXAS],
    on="ticker",
    how="left",
    validate="many_to_one",
)

df_mercado_diario_consolidada = (
    df_mercado_diario_consolidada
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

print(f"Base diária consolidada preparada: {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print("OK")

# ============================================================
# 6) Auditoria estrutural da base consolidada
# ============================================================

print("\n[6/10] Auditoria estrutural da base consolidada...")

n_duplicidades_ticker_data = int(df_mercado_diario_consolidada.duplicated(subset=["ticker", "data"]).sum())
n_tickers_sem_issuer_code = int(df_mercado_diario_consolidada.loc[df_mercado_diario_consolidada["issuer_code"].isna(), "ticker"].nunique(dropna=True))
n_linhas_sem_issuer_code = int(df_mercado_diario_consolidada["issuer_code"].isna().sum())
n_tickers_com_multiplos_issuer_code = int(
    (
        df_mercado_diario_consolidada
        .groupby("ticker", dropna=False)["issuer_code"]
        .agg(contar_unicos_nao_nulos)
        .gt(1)
        .sum()
    )
)
n_issuer_code_com_multiplos_tickers = int(
    (
        df_mercado_diario_consolidada
        .groupby("issuer_code", dropna=True)["ticker"]
        .agg(contar_unicos_nao_nulos)
        .gt(1)
        .sum()
    )
)

if n_duplicidades_ticker_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_ticker_data} duplicidades na chave natural ticker-data da base consolidada.")

df_auditoria_chaves_mercado_consolidada = pd.DataFrame(
    [
        {"metrica": "n_duplicidades_ticker_data", "valor": n_duplicidades_ticker_data},
        {"metrica": "n_linhas_sem_issuer_code", "valor": n_linhas_sem_issuer_code},
        {"metrica": "n_tickers_sem_issuer_code", "valor": n_tickers_sem_issuer_code},
        {"metrica": "n_tickers_com_multiplos_issuer_code", "valor": n_tickers_com_multiplos_issuer_code},
        {"metrica": "n_issuer_code_com_multiplos_tickers", "valor": n_issuer_code_com_multiplos_tickers},
    ]
)

print(f"Duplicidades na chave ticker-data         : {n_duplicidades_ticker_data}")
print(f"Linhas sem issuer_code                    : {n_linhas_sem_issuer_code}")
print(f"Tickers sem issuer_code                   : {n_tickers_sem_issuer_code}")
print(f"Tickers associados a múltiplos issuer_code: {n_tickers_com_multiplos_issuer_code}")
print(f"Issuer_code associados a múltiplos tickers: {n_issuer_code_com_multiplos_tickers}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas-resumo da subetapa...")

df_resumo_mercado_diario_consolidada = pd.DataFrame(
    [
        {
            "base": "mercado_diario_consolidada",
            "n_linhas": int(len(df_mercado_diario_consolidada)),
            "n_colunas": int(df_mercado_diario_consolidada.shape[1]),
            "n_tickers": int(df_mercado_diario_consolidada["ticker"].nunique(dropna=True)),
            "n_issuer_code": int(df_mercado_diario_consolidada["issuer_code"].nunique(dropna=True)),
            "n_datas_unicas": int(df_mercado_diario_consolidada["data"].nunique(dropna=True)),
            "data_min": df_mercado_diario_consolidada["data"].min(),
            "data_max": df_mercado_diario_consolidada["data"].max(),
            "missing_close_adj": int(df_mercado_diario_consolidada["close_adj"].isna().sum()),
            "missing_volume_fin": int(df_mercado_diario_consolidada["volume_fin"].isna().sum()),
            "missing_negocios": int(df_mercado_diario_consolidada["negocios"].isna().sum()),
        }
    ]
)

df_cobertura_diaria_mercado_consolidada = (
    df_mercado_diario_consolidada
    .groupby("data", as_index=False)
    .agg(
        n_tickers=("ticker", "nunique"),
        n_empresas=("issuer_code", "nunique"),
        volume_fin_total=("volume_fin", "sum"),
        volume_qtd_total=("volume_qtd", "sum"),
        negocios_total=("negocios", "sum"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_empresas_por_data_mercado_consolidada = (
    df_mercado_diario_consolidada
    .groupby("data", as_index=False)
    .agg(
        n_empresas=("issuer_code", "nunique"),
        n_tickers=("ticker", "nunique"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

print("Tabelas-resumo construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_mercado_diario_consolidada, caminho_base_mercado_diario_consolidada, index=False)
salvar_dataframe(df_resumo_mercado_diario_consolidada, caminho_tbl_resumo_mercado_diario_consolidada, index=False)
salvar_dataframe(df_cobertura_diaria_mercado_consolidada, caminho_tbl_cobertura_diaria_mercado_consolidada, index=False)
salvar_dataframe(df_empresas_por_data_mercado_consolidada, caminho_tbl_empresas_por_data_mercado_consolidada, index=False)
salvar_dataframe(df_auditoria_chaves_mercado_consolidada, caminho_tbl_auditoria_chaves_mercado_consolidada, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_base_consolidada = df_mercado_diario_consolidada.head(5).copy()
df_amostra_cobertura_diaria_inicial = df_cobertura_diaria_mercado_consolidada.head(10).copy()
df_amostra_cobertura_diaria_final = df_cobertura_diaria_mercado_consolidada.tail(10).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base diária consolidada de mercado:")
print(df_resumo_mercado_diario_consolidada.to_string(index=False))

print("\nAuditoria de chaves da base consolidada:")
print(df_auditoria_chaves_mercado_consolidada.to_string(index=False))

print("\nAmostra da base diária consolidada de mercado:")
print(df_amostra_base_consolidada.to_string(index=False))

print("\nCobertura diária da base consolidada - amostra inicial:")
print(df_amostra_cobertura_diaria_inicial.to_string(index=False))

print("\nCobertura diária da base consolidada - amostra final:")
print(df_amostra_cobertura_diaria_final.to_string(index=False))

print("\nArquivos salvos na subetapa 2.4:")
print(f"- {caminho_base_mercado_diario_consolidada}")
print(f"- {caminho_tbl_resumo_mercado_diario_consolidada}")
print(f"- {caminho_tbl_cobertura_diaria_mercado_consolidada}")
print(f"- {caminho_tbl_empresas_por_data_mercado_consolidada}")
print(f"- {caminho_tbl_auditoria_chaves_mercado_consolidada}")

print("\nETAPA 2.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 2.4 - BASE DIÁRIA CONSOLIDADA DE MERCADO

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado com empresa   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_3_base_mercado_diario_com_empresa.parquet
Entrada - cobertura por ticker  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_2_base_cobertura_historica_ticker.parquet
Saída   - base consolidada      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Saída   - resumo                : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_tbl_resumo_mercado_diario_consolidada.parq

# Etapa 3) Construção do Universo Elegível de Negociação

## Etapa 3.1) Definição dos Critérios de Liquidez

In [11]:
%%time
# ============================================================
# Etapa 3.1) Definição dos Critérios de Liquidez
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 3.1 - DEFINIÇÃO DOS CRITÉRIOS DE LIQUIDEZ")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "math" not in globals():
    raise NameError("A biblioteca math deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="base", nome="mercado_diario_consolidada")
caminho_base_cobertura_historica_ticker = gerar_caminho_arquivo(etapa=2, subetapa=2, tipo_arquivo="base", nome="cobertura_historica_ticker")

caminho_tbl_parametros_liquidez = gerar_caminho_arquivo(etapa=3, subetapa=1, tipo_arquivo="tbl", nome="parametros_liquidez")
caminho_tbl_regras_liquidez = gerar_caminho_arquivo(etapa=3, subetapa=1, tipo_arquivo="tbl", nome="regras_liquidez")
caminho_auditoria_contexto_liquidez = gerar_caminho_arquivo(etapa=3, subetapa=1, tipo_arquivo="auditoria", nome="contexto_liquidez")

print(f"Entrada - mercado diário consolidada : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - cobertura histórica        : {caminho_base_cobertura_historica_ticker}")
print(f"Saída   - parâmetros de liquidez     : {caminho_tbl_parametros_liquidez}")
print(f"Saída   - regras de liquidez         : {caminho_tbl_regras_liquidez}")
print(f"Saída   - auditoria de contexto      : {caminho_auditoria_contexto_liquidez}")
print("OK")

# ============================================================
# 3) Carga dos insumos diagnósticos da etapa 2
# ============================================================

print("\n[3/10] Carga dos insumos diagnósticos da etapa 2...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_cobertura_historica_ticker = pd.read_parquet(caminho_base_cobertura_historica_ticker)

print(f"Base mercado diário consolidada : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Base cobertura histórica ticker : {df_cobertura_historica_ticker.shape[0]:,} linhas x {df_cobertura_historica_ticker.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros metodológicos da regra de liquidez
# ============================================================

print("\n[4/10] Parâmetros metodológicos da regra de liquidez...")

JANELA_LIQUIDEZ_PREGOES = 252
PCT_MINIMO_PREGOES_NEGOCIADOS = 0.80
N_MINIMO_PREGOES_NEGOCIADOS = int(math.ceil(JANELA_LIQUIDEZ_PREGOES * PCT_MINIMO_PREGOES_NEGOCIADOS))

VOLUME_FINANCEIRO_MEDIANO_MINIMO = 1000000.00
NEGOCIOS_MEDIANO_MINIMO = 50
PRECO_MINIMO_FECHAMENTO_AJUSTADO = 1.00

COLUNA_PRECO_REFERENCIA = "close_adj"
COLUNA_VOLUME_FINANCEIRO = "volume_fin"
COLUNA_NEGOCIOS = "negocios"
COLUNA_QUANTIDADE = "volume_qtd"

DEFINICAO_DIA_NEGOCIADO = "volume_fin > 0 e negocios > 0 e close_adj não nulo"

print(f"Janela de avaliação                : {JANELA_LIQUIDEZ_PREGOES} pregões")
print(f"Percentual mínimo de pregões       : {PCT_MINIMO_PREGOES_NEGOCIADOS:.0%}")
print(f"Número mínimo de pregões           : {N_MINIMO_PREGOES_NEGOCIADOS}")
print(f"Volume financeiro mediano mínimo   : R$ {VOLUME_FINANCEIRO_MEDIANO_MINIMO:,.2f}")
print(f"Número mediano mínimo de negócios  : {NEGOCIOS_MEDIANO_MINIMO}")
print(f"Preço mínimo de fechamento ajustado: R$ {PRECO_MINIMO_FECHAMENTO_AJUSTADO:,.2f}")
print("OK")

# ============================================================
# 5) Funções auxiliares da subetapa
# ============================================================

print("\n[5/10] Funções auxiliares da subetapa...")

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def registrar_valor_misto(metrica, valor, unidade):
    """
    Estrutura valores de auditoria em colunas homogêneas para escrita segura em parquet.
    """
    if isinstance(valor, (int, float)) and not pd.isna(valor):
        return {
            "metrica": metrica,
            "valor_numerico": float(valor),
            "valor_texto": pd.NA,
            "unidade": unidade,
        }
    return {
        "metrica": metrica,
        "valor_numerico": pd.NA,
        "valor_texto": str(valor) if pd.notna(valor) else pd.NA,
        "unidade": unidade,
    }

def resumir_contexto_liquidez(df_mercado, df_cobertura):
    """
    Consolida indicadores diagnósticos da etapa 2 para contextualização da regra de liquidez.
    """
    registros = [
        registrar_valor_misto("n_linhas_mercado_diario_consolidada", int(len(df_mercado)), "linhas"),
        registrar_valor_misto("n_tickers_mercado_diario_consolidada", int(df_mercado["ticker"].nunique(dropna=True)), "tickers"),
        registrar_valor_misto("n_issuer_code_mercado_diario_consolidada", int(df_mercado["issuer_code"].nunique(dropna=True)), "empresas"),
        registrar_valor_misto("n_datas_unicas_mercado_diario_consolidada", int(df_mercado["data"].nunique(dropna=True)), "pregoes"),
        registrar_valor_misto("data_min_mercado_diario_consolidada", str(pd.to_datetime(df_mercado["data"], errors="coerce").min().date()), "data"),
        registrar_valor_misto("data_max_mercado_diario_consolidada", str(pd.to_datetime(df_mercado["data"], errors="coerce").max().date()), "data"),
        registrar_valor_misto("n_tickers_com_pelo_menos_252_pregoes", int(df_cobertura["n_datas_unicas"].ge(252).sum()), "tickers"),
        registrar_valor_misto("n_tickers_com_cobertura_relativa_abaixo_80pct", int(df_cobertura["pct_cobertura_pregoes_intervalo"].lt(0.80).sum()), "tickers"),
        registrar_valor_misto("n_tickers_com_gap_acima_30_dias", int(df_cobertura["maior_intervalo_dias"].gt(30).sum()), "tickers"),
        registrar_valor_misto("n_issuer_code_com_multiplos_tickers", int(df_mercado.groupby("issuer_code", dropna=True)["ticker"].nunique().gt(1).sum()), "empresas"),
    ]
    return pd.DataFrame(registros)

def registrar_parametro(parametro, valor, unidade, descricao, justificativa_tecnica):
    """
    Estrutura parâmetros em colunas homogêneas para escrita segura em parquet.
    """
    if isinstance(valor, (int, float)) and not pd.isna(valor):
        return {
            "parametro": parametro,
            "valor_numerico": float(valor),
            "valor_texto": pd.NA,
            "unidade": unidade,
            "descricao": descricao,
            "justificativa_tecnica": justificativa_tecnica,
        }
    return {
        "parametro": parametro,
        "valor_numerico": pd.NA,
        "valor_texto": str(valor) if pd.notna(valor) else pd.NA,
        "unidade": unidade,
        "descricao": descricao,
        "justificativa_tecnica": justificativa_tecnica,
    }

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 6) Validação dos insumos e consolidação do contexto diagnóstico
# ============================================================

print("\n[6/10] Validação dos insumos e consolidação do contexto diagnóstico...")

validar_colunas_obrigatorias(
    df_mercado_diario_consolidada,
    ["ticker", "issuer_code", "data", "close_adj", "volume_qtd", "volume_fin", "negocios"],
)
validar_colunas_obrigatorias(
    df_cobertura_historica_ticker,
    ["ticker", "n_datas_unicas", "pct_cobertura_pregoes_intervalo", "maior_intervalo_dias"],
)

df_auditoria_contexto_liquidez = resumir_contexto_liquidez(df_mercado_diario_consolidada, df_cobertura_historica_ticker)

print("Contexto diagnóstico consolidado com sucesso.")
print("OK")

# ============================================================
# 7) Construção das tabelas formais de parâmetros e regras
# ============================================================

print("\n[7/10] Construção das tabelas formais de parâmetros e regras...")

df_parametros_liquidez = pd.DataFrame(
    [
        registrar_parametro(
            "janela_liquidez_pregoes",
            JANELA_LIQUIDEZ_PREGOES,
            "pregoes",
            "Janela móvel de observação utilizada na avaliação operacional de liquidez por ticker.",
            "Uma janela anual de pregões reduz sensibilidade a ruídos muito curtos e captura liquidez operacional recorrente.",
        ),
        registrar_parametro(
            "pct_minimo_pregoes_negociados",
            PCT_MINIMO_PREGOES_NEGOCIADOS,
            "proporcao",
            "Percentual mínimo de pregões com negociação válida dentro da janela de liquidez.",
            "Exigir presença em pelo menos 80% da janela reduz a entrada de tickers com negociação episódica ou descontínua.",
        ),
        registrar_parametro(
            "n_minimo_pregoes_negociados",
            N_MINIMO_PREGOES_NEGOCIADOS,
            "pregoes",
            "Número mínimo de pregões negociados correspondente ao percentual mínimo dentro da janela.",
            "A versão absoluta da regra facilita auditoria, implementação e verificação operacional por data.",
        ),
        registrar_parametro(
            "volume_financeiro_mediano_minimo",
            VOLUME_FINANCEIRO_MEDIANO_MINIMO,
            "brl",
            "Volume financeiro mediano diário mínimo dentro da janela de liquidez.",
            "O uso da mediana reduz o efeito de outliers e eventos pontuais, preservando uma medida robusta de negociabilidade.",
        ),
        registrar_parametro(
            "negocios_mediano_minimo",
            NEGOCIOS_MEDIANO_MINIMO,
            "negocios",
            "Número mediano mínimo de negócios diários dentro da janela de liquidez.",
            "Complementa o volume financeiro com uma medida de dispersão de negociação, reduzindo dependência de poucos negócios grandes.",
        ),
        registrar_parametro(
            "preco_minimo_fechamento_ajustado",
            PRECO_MINIMO_FECHAMENTO_AJUSTADO,
            "brl",
            "Preço mínimo no fechamento ajustado da data de avaliação.",
            "Um piso mínimo de preço ajuda a evitar ativos com baixa negociabilidade operacional e maior suscetibilidade a distorções microestruturais.",
        ),
        registrar_parametro(
            "coluna_preco_referencia",
            COLUNA_PRECO_REFERENCIA,
            "coluna",
            "Coluna de preço utilizada na verificação do piso de preço.",
            "O fechamento ajustado preserva comparabilidade histórica ao incorporar ajustes corporativos relevantes.",
        ),
        registrar_parametro(
            "definicao_dia_negociado",
            DEFINICAO_DIA_NEGOCIADO,
            "regra_logica",
            "Critério lógico para contar um pregão como efetivamente negociado.",
            "A regra exige simultaneamente preço válido, negócios positivos e volume financeiro positivo para caracterizar presença operacional.",
        ),
    ]
)

df_regras_liquidez = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "R1",
            "criterio": "contagem de pregões negociados na janela",
            "operador": ">=",
            "limiar": float(N_MINIMO_PREGOES_NEGOCIADOS),
            "unidade": "pregoes",
            "campo_referencia": "pregoes_negociados_janela",
            "descricao_operacional": f"O ticker deve apresentar pelo menos {N_MINIMO_PREGOES_NEGOCIADOS} pregões negociados na janela móvel de {JANELA_LIQUIDEZ_PREGOES} pregões.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "R2",
            "criterio": "percentual de pregões negociados na janela",
            "operador": ">=",
            "limiar": float(PCT_MINIMO_PREGOES_NEGOCIADOS),
            "unidade": "proporcao",
            "campo_referencia": "pct_pregoes_negociados_janela",
            "descricao_operacional": "O ticker deve negociar em pelo menos 80% dos pregões da janela móvel de referência.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "R3",
            "criterio": "mediana do volume financeiro diário na janela",
            "operador": ">=",
            "limiar": float(VOLUME_FINANCEIRO_MEDIANO_MINIMO),
            "unidade": "brl",
            "campo_referencia": "mediana_volume_fin_janela",
            "descricao_operacional": "A mediana do volume financeiro diário deve ser igual ou superior a R$ 1.000.000 na janela.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "R4",
            "criterio": "mediana do número diário de negócios na janela",
            "operador": ">=",
            "limiar": float(NEGOCIOS_MEDIANO_MINIMO),
            "unidade": "negocios",
            "campo_referencia": "mediana_negocios_janela",
            "descricao_operacional": "A mediana do número de negócios diários deve ser igual ou superior a 50 na janela.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "R5",
            "criterio": "piso de preço na data de avaliação",
            "operador": ">=",
            "limiar": float(PRECO_MINIMO_FECHAMENTO_AJUSTADO),
            "unidade": "brl",
            "campo_referencia": "close_adj_data_avaliacao",
            "descricao_operacional": "O fechamento ajustado do ticker na data de avaliação deve ser igual ou superior a R$ 1,00.",
        },
    ]
)

print("Tabelas formais de parâmetros e regras construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_parametros_liquidez, caminho_tbl_parametros_liquidez, index=False)
salvar_dataframe(df_regras_liquidez, caminho_tbl_regras_liquidez, index=False)
salvar_dataframe(df_auditoria_contexto_liquidez, caminho_auditoria_contexto_liquidez, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_parametros_liquidez = df_parametros_liquidez.copy()
df_amostra_regras_liquidez = df_regras_liquidez.copy()
df_amostra_contexto_liquidez = df_auditoria_contexto_liquidez.copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nParâmetros formais da regra de liquidez:")
print(df_amostra_parametros_liquidez.to_string(index=False))

print("\nRegras operacionais da etapa de liquidez:")
print(df_amostra_regras_liquidez.to_string(index=False))

print("\nContexto diagnóstico utilizado para definição dos critérios:")
print(df_amostra_contexto_liquidez.to_string(index=False))

print("\nArquivos salvos na subetapa 3.1:")
print(f"- {caminho_tbl_parametros_liquidez}")
print(f"- {caminho_tbl_regras_liquidez}")
print(f"- {caminho_auditoria_contexto_liquidez}")

print("\nETAPA 3.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 3.1 - DEFINIÇÃO DOS CRITÉRIOS DE LIQUIDEZ

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidada : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - cobertura histórica        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_2_base_cobertura_historica_ticker.parquet
Saída   - parâmetros de liquidez     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_1_tbl_parametros_liquidez.parquet
Saída   - regras de liquidez         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_1_tbl_regras_liquidez.parquet
S

## Etapa 3.2) Aplicação dos Critérios de Liquidez por Data

In [12]:
%%time
# ============================================================
# Etapa 3.2) Aplicação dos Critérios de Liquidez por Data
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 3.2 - APLICAÇÃO DOS CRITÉRIOS DE LIQUIDEZ POR DATA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/11] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/11] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="base", nome="mercado_diario_consolidada")
caminho_tbl_parametros_liquidez = gerar_caminho_arquivo(etapa=3, subetapa=1, tipo_arquivo="tbl", nome="parametros_liquidez")

caminho_base_liquidez_ticker_data = gerar_caminho_arquivo(etapa=3, subetapa=2, tipo_arquivo="base", nome="liquidez_ticker_data")
caminho_tbl_resumo_liquidez_ticker_data = gerar_caminho_arquivo(etapa=3, subetapa=2, tipo_arquivo="tbl", nome="resumo_liquidez_ticker_data")
caminho_tbl_cobertura_diaria_liquidez = gerar_caminho_arquivo(etapa=3, subetapa=2, tipo_arquivo="tbl", nome="cobertura_diaria_liquidez")
caminho_tbl_auditoria_regras_liquidez = gerar_caminho_arquivo(etapa=3, subetapa=2, tipo_arquivo="tbl", nome="auditoria_regras_liquidez")

print(f"Entrada - mercado diário consolidada : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - parâmetros de liquidez     : {caminho_tbl_parametros_liquidez}")
print(f"Saída   - base de liquidez           : {caminho_base_liquidez_ticker_data}")
print(f"Saída   - resumo                     : {caminho_tbl_resumo_liquidez_ticker_data}")
print(f"Saída   - cobertura diária           : {caminho_tbl_cobertura_diaria_liquidez}")
print(f"Saída   - auditoria de regras        : {caminho_tbl_auditoria_regras_liquidez}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/11] Carga dos inputs da subetapa...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_parametros_liquidez = pd.read_parquet(caminho_tbl_parametros_liquidez)

print(f"Base mercado diário consolidada : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Tabela de parâmetros de liquidez: {df_parametros_liquidez.shape[0]:,} linhas x {df_parametros_liquidez.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/11] Funções auxiliares e parâmetros operacionais...")

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def obter_valor_parametro(df_parametros, parametro):
    """
    Recupera o valor efetivo de um parâmetro salvo na subetapa 3.1.
    """
    df_parametro = df_parametros.loc[df_parametros["parametro"].eq(parametro)].copy()

    if df_parametro.empty:
        raise ValueError(f"Parâmetro não encontrado: {parametro}")

    valor_numerico = df_parametro["valor_numerico"].iloc[0]
    valor_texto = df_parametro["valor_texto"].iloc[0]

    if pd.notna(valor_numerico):
        return float(valor_numerico)

    if pd.notna(valor_texto):
        return str(valor_texto)

    return pd.NA

def construir_metricas_liquidez_ticker(df_ticker, calendario_mercado, janela_liquidez_pregoes, n_minimo_pregoes_negociados, pct_minimo_pregoes_negociados, volume_financeiro_mediano_minimo, negocios_mediano_minimo, preco_minimo_fechamento_ajustado):
    """
    Calcula as métricas de liquidez por ticker e data utilizando o calendário completo do mercado entre a primeira e a última observação do ticker.
    """
    df_ticker = df_ticker.sort_values("data").reset_index(drop=True)

    data_min_ticker = df_ticker["data"].min()
    data_max_ticker = df_ticker["data"].max()

    df_calendario_ticker = pd.DataFrame(
        {
            "data": calendario_mercado[(calendario_mercado >= data_min_ticker) & (calendario_mercado <= data_max_ticker)]
        }
    )

    df_base_ticker = df_calendario_ticker.merge(
        df_ticker,
        on="data",
        how="left",
        sort=True,
    )

    colunas_identificacao = [
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "n_datas_unicas",
        "pct_cobertura_pregoes_intervalo",
        "maior_intervalo_dias",
        "flag_diagnostico_historico_curto",
        "flag_diagnostico_cobertura_insuficiente",
        "flag_diagnostico_gap_longo",
    ]

    for coluna in colunas_identificacao:
        if coluna in df_base_ticker.columns:
            df_base_ticker[coluna] = df_base_ticker[coluna].ffill().bfill()

    df_base_ticker["flag_possui_observacao_mercado_original"] = df_base_ticker["close_adj"].notna()
    df_base_ticker["volume_fin_ajustado_liquidez"] = pd.to_numeric(df_base_ticker["volume_fin"], errors="coerce").fillna(0.0)
    df_base_ticker["negocios_ajustado_liquidez"] = pd.to_numeric(df_base_ticker["negocios"], errors="coerce").fillna(0.0)
    df_base_ticker["flag_dia_negociado"] = (
        pd.to_numeric(df_base_ticker["volume_fin"], errors="coerce").gt(0)
        & pd.to_numeric(df_base_ticker["negocios"], errors="coerce").gt(0)
        & pd.to_numeric(df_base_ticker["close_adj"], errors="coerce").notna()
    ).astype(int)

    rolling_flag_negociado = df_base_ticker["flag_dia_negociado"].rolling(window=janela_liquidez_pregoes, min_periods=janela_liquidez_pregoes)
    rolling_volume_fin = df_base_ticker["volume_fin_ajustado_liquidez"].rolling(window=janela_liquidez_pregoes, min_periods=janela_liquidez_pregoes)
    rolling_negocios = df_base_ticker["negocios_ajustado_liquidez"].rolling(window=janela_liquidez_pregoes, min_periods=janela_liquidez_pregoes)

    df_base_ticker["pregoes_referencia_janela"] = janela_liquidez_pregoes
    df_base_ticker["pregoes_negociados_janela"] = rolling_flag_negociado.sum()
    df_base_ticker["pct_pregoes_negociados_janela"] = df_base_ticker["pregoes_negociados_janela"] / janela_liquidez_pregoes
    df_base_ticker["mediana_volume_fin_janela"] = rolling_volume_fin.median()
    df_base_ticker["mediana_negocios_janela"] = rolling_negocios.median()
    df_base_ticker["close_adj_data_avaliacao"] = pd.to_numeric(df_base_ticker["close_adj"], errors="coerce")
    df_base_ticker["flag_janela_liquidez_completa"] = df_base_ticker["pregoes_negociados_janela"].notna()

    df_base_ticker["flag_regra_r1_pregoes_minimos"] = (
        df_base_ticker["flag_janela_liquidez_completa"]
        & df_base_ticker["pregoes_negociados_janela"].ge(n_minimo_pregoes_negociados)
    )
    df_base_ticker["flag_regra_r2_pct_minimo"] = (
        df_base_ticker["flag_janela_liquidez_completa"]
        & df_base_ticker["pct_pregoes_negociados_janela"].ge(pct_minimo_pregoes_negociados)
    )
    df_base_ticker["flag_regra_r3_volume_mediano_minimo"] = (
        df_base_ticker["flag_janela_liquidez_completa"]
        & df_base_ticker["mediana_volume_fin_janela"].ge(volume_financeiro_mediano_minimo)
    )
    df_base_ticker["flag_regra_r4_negocios_mediano_minimo"] = (
        df_base_ticker["flag_janela_liquidez_completa"]
        & df_base_ticker["mediana_negocios_janela"].ge(negocios_mediano_minimo)
    )
    df_base_ticker["flag_regra_r5_preco_minimo"] = df_base_ticker["close_adj_data_avaliacao"].ge(preco_minimo_fechamento_ajustado)

    df_base_ticker["flag_elegivel_liquidez_ticker_data"] = (
        df_base_ticker["flag_regra_r1_pregoes_minimos"]
        & df_base_ticker["flag_regra_r2_pct_minimo"]
        & df_base_ticker["flag_regra_r3_volume_mediano_minimo"]
        & df_base_ticker["flag_regra_r4_negocios_mediano_minimo"]
        & df_base_ticker["flag_regra_r5_preco_minimo"]
        & df_base_ticker["flag_possui_observacao_mercado_original"]
    )

    colunas_saida = [
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "data",
        "close_adj_data_avaliacao",
        "flag_possui_observacao_mercado_original",
        "flag_dia_negociado",
        "pregoes_referencia_janela",
        "pregoes_negociados_janela",
        "pct_pregoes_negociados_janela",
        "mediana_volume_fin_janela",
        "mediana_negocios_janela",
        "flag_janela_liquidez_completa",
        "flag_regra_r1_pregoes_minimos",
        "flag_regra_r2_pct_minimo",
        "flag_regra_r3_volume_mediano_minimo",
        "flag_regra_r4_negocios_mediano_minimo",
        "flag_regra_r5_preco_minimo",
        "flag_elegivel_liquidez_ticker_data",
        "n_datas_unicas",
        "pct_cobertura_pregoes_intervalo",
        "maior_intervalo_dias",
        "flag_diagnostico_historico_curto",
        "flag_diagnostico_cobertura_insuficiente",
        "flag_diagnostico_gap_longo",
    ]

    return (
        df_base_ticker
        .loc[df_base_ticker["flag_possui_observacao_mercado_original"], colunas_saida]
        .sort_values("data")
        .reset_index(drop=True)
    )

validar_colunas_obrigatorias(
    df_mercado_diario_consolidada,
    [
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "data",
        "close_adj",
        "volume_fin",
        "negocios",
        "n_datas_unicas",
        "pct_cobertura_pregoes_intervalo",
        "maior_intervalo_dias",
        "flag_diagnostico_historico_curto",
        "flag_diagnostico_cobertura_insuficiente",
        "flag_diagnostico_gap_longo",
    ],
)
validar_colunas_obrigatorias(
    df_parametros_liquidez,
    ["parametro", "valor_numerico", "valor_texto"],
)

JANELA_LIQUIDEZ_PREGOES = int(obter_valor_parametro(df_parametros_liquidez, "janela_liquidez_pregoes"))
PCT_MINIMO_PREGOES_NEGOCIADOS = float(obter_valor_parametro(df_parametros_liquidez, "pct_minimo_pregoes_negociados"))
N_MINIMO_PREGOES_NEGOCIADOS = int(obter_valor_parametro(df_parametros_liquidez, "n_minimo_pregoes_negociados"))
VOLUME_FINANCEIRO_MEDIANO_MINIMO = float(obter_valor_parametro(df_parametros_liquidez, "volume_financeiro_mediano_minimo"))
NEGOCIOS_MEDIANO_MINIMO = float(obter_valor_parametro(df_parametros_liquidez, "negocios_mediano_minimo"))
PRECO_MINIMO_FECHAMENTO_AJUSTADO = float(obter_valor_parametro(df_parametros_liquidez, "preco_minimo_fechamento_ajustado"))

print(f"Janela operacional utilizada                : {JANELA_LIQUIDEZ_PREGOES}")
print(f"Pregões mínimos utilizados                  : {N_MINIMO_PREGOES_NEGOCIADOS}")
print(f"Percentual mínimo utilizado                 : {PCT_MINIMO_PREGOES_NEGOCIADOS:.0%}")
print(f"Volume financeiro mediano mínimo utilizado  : R$ {VOLUME_FINANCEIRO_MEDIANO_MINIMO:,.2f}")
print(f"Número mediano de negócios mínimo utilizado : {NEGOCIOS_MEDIANO_MINIMO:,.0f}")
print("OK")

# ============================================================
# 5) Preparação do calendário do mercado e ordenação da base
# ============================================================

print("\n[5/11] Preparação do calendário do mercado e ordenação da base...")

df_mercado_diario_consolidada["data"] = pd.to_datetime(df_mercado_diario_consolidada["data"], errors="coerce")
df_mercado_diario_consolidada = (
    df_mercado_diario_consolidada
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

calendario_mercado = pd.Index(
    df_mercado_diario_consolidada["data"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print(f"Quantidade de pregões no calendário de referência: {len(calendario_mercado):,}")
print(f"Data inicial do calendário                    : {calendario_mercado.min()}")
print(f"Data final do calendário                      : {calendario_mercado.max()}")
print("OK")

# ============================================================
# 6) Cálculo das métricas de liquidez por ticker e data
# ============================================================

print("\n[6/11] Cálculo das métricas de liquidez por ticker e data...")

lista_resultados_liquidez = []
n_tickers_total = int(df_mercado_diario_consolidada["ticker"].nunique(dropna=True))

for indice_ticker, (ticker, df_ticker) in enumerate(df_mercado_diario_consolidada.groupby("ticker", sort=True), start=1):
    df_resultado_ticker = construir_metricas_liquidez_ticker(
        df_ticker=df_ticker,
        calendario_mercado=calendario_mercado,
        janela_liquidez_pregoes=JANELA_LIQUIDEZ_PREGOES,
        n_minimo_pregoes_negociados=N_MINIMO_PREGOES_NEGOCIADOS,
        pct_minimo_pregoes_negociados=PCT_MINIMO_PREGOES_NEGOCIADOS,
        volume_financeiro_mediano_minimo=VOLUME_FINANCEIRO_MEDIANO_MINIMO,
        negocios_mediano_minimo=NEGOCIOS_MEDIANO_MINIMO,
        preco_minimo_fechamento_ajustado=PRECO_MINIMO_FECHAMENTO_AJUSTADO,
    )
    lista_resultados_liquidez.append(df_resultado_ticker)

    if (indice_ticker % 100 == 0) or (indice_ticker == n_tickers_total):
        print(f"Tickers processados: {indice_ticker:,}/{n_tickers_total:,}")

df_liquidez_ticker_data = pd.concat(lista_resultados_liquidez, ignore_index=True)
df_liquidez_ticker_data = (
    df_liquidez_ticker_data
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

print(f"Base de liquidez ticker-data gerada: {df_liquidez_ticker_data.shape[0]:,} linhas x {df_liquidez_ticker_data.shape[1]} colunas")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e de auditoria
# ============================================================

print("\n[7/11] Construção das tabelas-resumo e de auditoria...")

df_resumo_liquidez_ticker_data = pd.DataFrame(
    [
        {
            "base": "liquidez_ticker_data",
            "n_linhas": int(len(df_liquidez_ticker_data)),
            "n_tickers": int(df_liquidez_ticker_data["ticker"].nunique(dropna=True)),
            "n_issuer_code": int(df_liquidez_ticker_data["issuer_code"].nunique(dropna=True)),
            "n_datas_unicas": int(df_liquidez_ticker_data["data"].nunique(dropna=True)),
            "data_min": df_liquidez_ticker_data["data"].min(),
            "data_max": df_liquidez_ticker_data["data"].max(),
            "n_linhas_janela_completa": int(df_liquidez_ticker_data["flag_janela_liquidez_completa"].sum()),
            "n_linhas_elegiveis_liquidez": int(df_liquidez_ticker_data["flag_elegivel_liquidez_ticker_data"].sum()),
            "pct_linhas_elegiveis_liquidez": float(df_liquidez_ticker_data["flag_elegivel_liquidez_ticker_data"].mean()),
        }
    ]
)

df_cobertura_diaria_liquidez = (
    df_liquidez_ticker_data
    .groupby("data", as_index=False)
    .agg(
        n_tickers_observados=("ticker", "nunique"),
        n_tickers_janela_completa=("flag_janela_liquidez_completa", "sum"),
        n_tickers_elegiveis_liquidez=("flag_elegivel_liquidez_ticker_data", "sum"),
        n_empresas_elegiveis_liquidez=("issuer_code", lambda x: x[df_liquidez_ticker_data.loc[x.index, "flag_elegivel_liquidez_ticker_data"]].nunique()),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_auditoria_regras_liquidez = pd.DataFrame(
    [
        {
            "regra": "R1",
            "descricao": "pregões negociados mínimos na janela",
            "n_linhas_aprovadas": int(df_liquidez_ticker_data["flag_regra_r1_pregoes_minimos"].sum()),
            "pct_linhas_aprovadas": float(df_liquidez_ticker_data["flag_regra_r1_pregoes_minimos"].mean()),
        },
        {
            "regra": "R2",
            "descricao": "percentual mínimo de pregões negociados",
            "n_linhas_aprovadas": int(df_liquidez_ticker_data["flag_regra_r2_pct_minimo"].sum()),
            "pct_linhas_aprovadas": float(df_liquidez_ticker_data["flag_regra_r2_pct_minimo"].mean()),
        },
        {
            "regra": "R3",
            "descricao": "mediana mínima de volume financeiro",
            "n_linhas_aprovadas": int(df_liquidez_ticker_data["flag_regra_r3_volume_mediano_minimo"].sum()),
            "pct_linhas_aprovadas": float(df_liquidez_ticker_data["flag_regra_r3_volume_mediano_minimo"].mean()),
        },
        {
            "regra": "R4",
            "descricao": "mediana mínima de negócios",
            "n_linhas_aprovadas": int(df_liquidez_ticker_data["flag_regra_r4_negocios_mediano_minimo"].sum()),
            "pct_linhas_aprovadas": float(df_liquidez_ticker_data["flag_regra_r4_negocios_mediano_minimo"].mean()),
        },
        {
            "regra": "R5",
            "descricao": "piso mínimo de preço",
            "n_linhas_aprovadas": int(df_liquidez_ticker_data["flag_regra_r5_preco_minimo"].sum()),
            "pct_linhas_aprovadas": float(df_liquidez_ticker_data["flag_regra_r5_preco_minimo"].mean()),
        },
        {
            "regra": "R_FINAL",
            "descricao": "elegibilidade final de liquidez",
            "n_linhas_aprovadas": int(df_liquidez_ticker_data["flag_elegivel_liquidez_ticker_data"].sum()),
            "pct_linhas_aprovadas": float(df_liquidez_ticker_data["flag_elegivel_liquidez_ticker_data"].mean()),
        },
    ]
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/11] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_liquidez_ticker_data, caminho_base_liquidez_ticker_data, index=False)
salvar_dataframe(df_resumo_liquidez_ticker_data, caminho_tbl_resumo_liquidez_ticker_data, index=False)
salvar_dataframe(df_cobertura_diaria_liquidez, caminho_tbl_cobertura_diaria_liquidez, index=False)
salvar_dataframe(df_auditoria_regras_liquidez, caminho_tbl_auditoria_regras_liquidez, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/11] Preparação das amostras de validação...")

df_amostra_liquidez_inicial = df_liquidez_ticker_data.head(10).copy()
df_amostra_liquidez_elegivel = (
    df_liquidez_ticker_data
    .loc[df_liquidez_ticker_data["flag_elegivel_liquidez_ticker_data"]]
    .head(10)
    .copy()
)

df_amostra_cobertura_diaria_inicial = df_cobertura_diaria_liquidez.head(10).copy()
df_amostra_cobertura_diaria_final = df_cobertura_diaria_liquidez.tail(10).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/11] Validação final da subetapa...")

print("\nResumo da base de liquidez ticker-data:")
print(df_resumo_liquidez_ticker_data.to_string(index=False))

print("\nAuditoria das regras de liquidez:")
print(df_auditoria_regras_liquidez.to_string(index=False))

print("\nAmostra inicial da base de liquidez ticker-data:")
print(df_amostra_liquidez_inicial.to_string(index=False))

print("\nAmostra de linhas elegíveis por liquidez:")
print(df_amostra_liquidez_elegivel.to_string(index=False))

print("\nCobertura diária de liquidez - amostra inicial:")
print(df_amostra_cobertura_diaria_inicial.to_string(index=False))

print("\nCobertura diária de liquidez - amostra final:")
print(df_amostra_cobertura_diaria_final.to_string(index=False))

print("\nArquivos salvos na subetapa 3.2:")
print(f"- {caminho_base_liquidez_ticker_data}")
print(f"- {caminho_tbl_resumo_liquidez_ticker_data}")
print(f"- {caminho_tbl_cobertura_diaria_liquidez}")
print(f"- {caminho_tbl_auditoria_regras_liquidez}")

print("\nETAPA 3.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 3.2 - APLICAÇÃO DOS CRITÉRIOS DE LIQUIDEZ POR DATA

[1/11] Validação inicial do ambiente...
OK

[2/11] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidada : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - parâmetros de liquidez     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_1_tbl_parametros_liquidez.parquet
Saída   - base de liquidez           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_2_base_liquidez_ticker_data.parquet
Saída   - resumo                     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_2_tbl_resumo_liquidez_ticker

## Etapa 3.3) Escolha de Um Único Ticker por Empresa

In [13]:
%%time
# ============================================================
# Etapa 3.3) Escolha de Um Único Ticker por Empresa
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 3.3 - ESCOLHA DE UM ÚNICO TICKER POR EMPRESA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_liquidez_ticker_data = gerar_caminho_arquivo(etapa=3, subetapa=2, tipo_arquivo="base", nome="liquidez_ticker_data")

caminho_base_ticker_vencedor_empresa_data = gerar_caminho_arquivo(etapa=3, subetapa=3, tipo_arquivo="base", nome="ticker_vencedor_empresa_data")
caminho_tbl_resumo_escolha_ticker_empresa = gerar_caminho_arquivo(etapa=3, subetapa=3, tipo_arquivo="tbl", nome="resumo_escolha_ticker_empresa")
caminho_tbl_auditoria_disputas_ticker_empresa = gerar_caminho_arquivo(etapa=3, subetapa=3, tipo_arquivo="tbl", nome="auditoria_disputas_ticker_empresa")
caminho_tbl_resumo_disputas_por_data = gerar_caminho_arquivo(etapa=3, subetapa=3, tipo_arquivo="tbl", nome="resumo_disputas_por_data")
caminho_tbl_criterios_desempate_ticker_empresa = gerar_caminho_arquivo(etapa=3, subetapa=3, tipo_arquivo="tbl", nome="criterios_desempate_ticker_empresa")

print(f"Entrada - liquidez ticker-data    : {caminho_base_liquidez_ticker_data}")
print(f"Saída   - ticker vencedor         : {caminho_base_ticker_vencedor_empresa_data}")
print(f"Saída   - resumo                  : {caminho_tbl_resumo_escolha_ticker_empresa}")
print(f"Saída   - auditoria de disputas   : {caminho_tbl_auditoria_disputas_ticker_empresa}")
print(f"Saída   - resumo por data         : {caminho_tbl_resumo_disputas_por_data}")
print(f"Saída   - critérios de desempate  : {caminho_tbl_criterios_desempate_ticker_empresa}")
print("OK")

# ============================================================
# 3) Carga do input da subetapa 3.2
# ============================================================

print("\n[3/10] Carga do input da subetapa 3.2...")

df_liquidez_ticker_data = pd.read_parquet(caminho_base_liquidez_ticker_data)

print(f"Base liquidez ticker-data: {df_liquidez_ticker_data.shape[0]:,} linhas x {df_liquidez_ticker_data.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e critérios de desempate
# ============================================================

print("\n[4/10] Funções auxiliares e critérios de desempate...")

COLUNAS_OBRIGATORIAS = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj_data_avaliacao",
    "pregoes_negociados_janela",
    "pct_pregoes_negociados_janela",
    "mediana_volume_fin_janela",
    "mediana_negocios_janela",
    "flag_janela_liquidez_completa",
    "flag_elegivel_liquidez_ticker_data",
]

COLUNAS_BASE_VENCEDORES = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj_data_avaliacao",
    "pregoes_negociados_janela",
    "pct_pregoes_negociados_janela",
    "mediana_volume_fin_janela",
    "mediana_negocios_janela",
    "flag_janela_liquidez_completa",
    "flag_elegivel_liquidez_ticker_data",
]

COLUNAS_NUMERICAS_DESEMPATE = [
    "mediana_volume_fin_janela",
    "mediana_negocios_janela",
    "pct_pregoes_negociados_janela",
    "pregoes_negociados_janela",
    "close_adj_data_avaliacao",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def preparar_base_elegivel(df):
    """
    Filtra apenas linhas elegíveis por liquidez e padroniza colunas usadas no desempate.
    """
    df = df.copy()
    df["ticker"] = df["ticker"].astype("string").str.strip().str.upper()
    df["issuer_code"] = df["issuer_code"].astype("string").str.strip()
    df["data"] = pd.to_datetime(df["data"], errors="coerce")

    for coluna in COLUNAS_NUMERICAS_DESEMPATE:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    df = df.loc[df["flag_elegivel_liquidez_ticker_data"]].copy()
    df = df.dropna(subset=["ticker", "issuer_code", "data"])
    df = df.sort_values(["issuer_code", "data", "ticker"]).reset_index(drop=True)

    return df

def construir_criterios_desempate():
    """
    Formaliza a ordem dos critérios de desempate entre tickers elegíveis da mesma empresa na mesma data.
    """
    return pd.DataFrame(
        [
            {
                "ordem_criterio": 1,
                "criterio": "mediana_volume_fin_janela",
                "sentido": "decrescente",
                "papel_metodologico": "critério principal de liquidez",
                "descricao_operacional": "Escolher o ticker com maior mediana de volume financeiro diário na janela de liquidez.",
            },
            {
                "ordem_criterio": 2,
                "criterio": "mediana_negocios_janela",
                "sentido": "decrescente",
                "papel_metodologico": "critério secundário de liquidez",
                "descricao_operacional": "Em caso de empate, escolher o ticker com maior mediana do número diário de negócios.",
            },
            {
                "ordem_criterio": 3,
                "criterio": "pct_pregoes_negociados_janela",
                "sentido": "decrescente",
                "papel_metodologico": "continuidade operacional",
                "descricao_operacional": "Persistindo o empate, priorizar o ticker com maior presença relativa de negociação na janela.",
            },
            {
                "ordem_criterio": 4,
                "criterio": "pregoes_negociados_janela",
                "sentido": "decrescente",
                "papel_metodologico": "continuidade operacional absoluta",
                "descricao_operacional": "Persistindo o empate, priorizar o ticker com maior quantidade absoluta de pregões negociados na janela.",
            },
            {
                "ordem_criterio": 5,
                "criterio": "close_adj_data_avaliacao",
                "sentido": "decrescente",
                "papel_metodologico": "critério residual determinístico",
                "descricao_operacional": "Persistindo o empate, priorizar o ticker com maior preço ajustado na data de avaliação.",
            },
            {
                "ordem_criterio": 6,
                "criterio": "ticker",
                "sentido": "crescente",
                "papel_metodologico": "desempate final determinístico",
                "descricao_operacional": "Persistindo o empate total, aplicar ordenação alfabética do ticker para garantir determinismo.",
            },
        ]
    )

print("Critérios de desempate formalizados com sucesso.")
print("OK")

# ============================================================
# 5) Preparação da base elegível e diagnóstico inicial de disputas
# ============================================================

print("\n[5/10] Preparação da base elegível e diagnóstico inicial de disputas...")

validar_colunas_obrigatorias(df_liquidez_ticker_data, COLUNAS_OBRIGATORIAS)

df_elegivel_liquidez = preparar_base_elegivel(df_liquidez_ticker_data)

df_contagem_empresa_data = (
    df_elegivel_liquidez
    .groupby(["issuer_code", "data"], as_index=False)
    .agg(
        n_tickers_elegiveis_empresa_data=("ticker", "nunique"),
        n_linhas_empresa_data=("ticker", "size"),
    )
)

n_empresa_data_total = int(len(df_contagem_empresa_data))
n_empresa_data_com_disputa = int(df_contagem_empresa_data["n_tickers_elegiveis_empresa_data"].gt(1).sum())

print(f"Linhas elegíveis por liquidez                         : {len(df_elegivel_liquidez):,}")
print(f"Combinações empresa-data elegíveis                    : {n_empresa_data_total:,}")
print(f"Combinações empresa-data com disputa entre tickers    : {n_empresa_data_com_disputa:,}")
print("OK")

# ============================================================
# 6) Escolha do ticker vencedor por empresa e data
# ============================================================

print("\n[6/10] Escolha do ticker vencedor por empresa e data...")

df_elegivel_liquidez["_sort_mediana_volume_fin_janela"] = df_elegivel_liquidez["mediana_volume_fin_janela"].fillna(float("-inf"))
df_elegivel_liquidez["_sort_mediana_negocios_janela"] = df_elegivel_liquidez["mediana_negocios_janela"].fillna(float("-inf"))
df_elegivel_liquidez["_sort_pct_pregoes_negociados_janela"] = df_elegivel_liquidez["pct_pregoes_negociados_janela"].fillna(float("-inf"))
df_elegivel_liquidez["_sort_pregoes_negociados_janela"] = df_elegivel_liquidez["pregoes_negociados_janela"].fillna(float("-inf"))
df_elegivel_liquidez["_sort_close_adj_data_avaliacao"] = df_elegivel_liquidez["close_adj_data_avaliacao"].fillna(float("-inf"))

df_elegivel_liquidez = df_elegivel_liquidez.merge(
    df_contagem_empresa_data,
    on=["issuer_code", "data"],
    how="left",
    validate="many_to_one",
)

df_elegivel_liquidez["flag_disputa_ticker_empresa_data"] = df_elegivel_liquidez["n_tickers_elegiveis_empresa_data"].gt(1)

df_elegivel_liquidez = (
    df_elegivel_liquidez
    .sort_values(
        [
            "issuer_code",
            "data",
            "_sort_mediana_volume_fin_janela",
            "_sort_mediana_negocios_janela",
            "_sort_pct_pregoes_negociados_janela",
            "_sort_pregoes_negociados_janela",
            "_sort_close_adj_data_avaliacao",
            "ticker",
        ],
        ascending=[True, True, False, False, False, False, False, True],
    )
    .reset_index(drop=True)
)

df_elegivel_liquidez["ranking_ticker_empresa_data"] = (
    df_elegivel_liquidez
    .groupby(["issuer_code", "data"], dropna=False)
    .cumcount()
    .add(1)
)

df_elegivel_liquidez["flag_ticker_vencedor_empresa_data"] = df_elegivel_liquidez["ranking_ticker_empresa_data"].eq(1)

df_ticker_vencedor_empresa_data = (
    df_elegivel_liquidez
    .loc[df_elegivel_liquidez["flag_ticker_vencedor_empresa_data"]]
    .copy()
)

df_ticker_vencedor_empresa_data = df_ticker_vencedor_empresa_data[
    COLUNAS_BASE_VENCEDORES
    + [
        "n_tickers_elegiveis_empresa_data",
        "n_linhas_empresa_data",
        "flag_disputa_ticker_empresa_data",
        "ranking_ticker_empresa_data",
        "flag_ticker_vencedor_empresa_data",
    ]
].sort_values(["data", "issuer_code", "ticker"]).reset_index(drop=True)

df_auditoria_disputas_ticker_empresa = (
    df_elegivel_liquidez
    .loc[df_elegivel_liquidez["flag_disputa_ticker_empresa_data"]]
    .copy()
)

df_auditoria_disputas_ticker_empresa = df_auditoria_disputas_ticker_empresa[
    COLUNAS_BASE_VENCEDORES
    + [
        "n_tickers_elegiveis_empresa_data",
        "n_linhas_empresa_data",
        "flag_disputa_ticker_empresa_data",
        "ranking_ticker_empresa_data",
        "flag_ticker_vencedor_empresa_data",
    ]
].sort_values(["data", "issuer_code", "ranking_ticker_empresa_data", "ticker"]).reset_index(drop=True)

n_linhas_vencedoras = int(len(df_ticker_vencedor_empresa_data))
n_empresas_dia_vencedoras = int(
    df_ticker_vencedor_empresa_data[["issuer_code", "data"]].drop_duplicates().shape[0]
)

print(f"Linhas vencedoras após consolidação por empresa-data   : {n_linhas_vencedoras:,}")
print(f"Combinações empresa-data vencedoras                    : {n_empresas_dia_vencedoras:,}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria da subetapa...")

df_resumo_escolha_ticker_empresa = pd.DataFrame(
    [
        {
            "n_linhas_elegiveis_antes_consolidacao": int(len(df_elegivel_liquidez)),
            "n_linhas_vencedoras_apos_consolidacao": int(len(df_ticker_vencedor_empresa_data)),
            "n_empresa_data_antes_consolidacao": int(df_elegivel_liquidez[["issuer_code", "data"]].drop_duplicates().shape[0]),
            "n_empresa_data_apos_consolidacao": int(df_ticker_vencedor_empresa_data[["issuer_code", "data"]].drop_duplicates().shape[0]),
            "n_disputas_empresa_data": int(df_contagem_empresa_data["n_tickers_elegiveis_empresa_data"].gt(1).sum()),
            "n_empresas_com_disputa_em_alguma_data": int(
                df_contagem_empresa_data.loc[df_contagem_empresa_data["n_tickers_elegiveis_empresa_data"].gt(1), "issuer_code"].nunique()
            ),
            "n_datas_com_disputa": int(
                df_contagem_empresa_data.loc[df_contagem_empresa_data["n_tickers_elegiveis_empresa_data"].gt(1), "data"].nunique()
            ),
            "n_tickers_vencedores_distintos": int(df_ticker_vencedor_empresa_data["ticker"].nunique(dropna=True)),
            "n_issuer_code_vencedores_distintos": int(df_ticker_vencedor_empresa_data["issuer_code"].nunique(dropna=True)),
            "pct_linhas_mantidas_apos_consolidacao": float(
                len(df_ticker_vencedor_empresa_data) / len(df_elegivel_liquidez)
            ) if len(df_elegivel_liquidez) > 0 else pd.NA,
        }
    ]
)

df_resumo_disputas_por_data = (
    df_auditoria_disputas_ticker_empresa
    .groupby("data", as_index=False)
    .agg(
        n_empresas_com_disputa=("issuer_code", "nunique"),
        n_tickers_envolvidos_em_disputa=("ticker", "nunique"),
        n_linhas_candidatas_em_disputa=("ticker", "size"),
        n_tickers_vencedores_em_disputa=("flag_ticker_vencedor_empresa_data", "sum"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_criterios_desempate_ticker_empresa = construir_criterios_desempate()

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_ticker_vencedor_empresa_data, caminho_base_ticker_vencedor_empresa_data, index=False)
salvar_dataframe(df_resumo_escolha_ticker_empresa, caminho_tbl_resumo_escolha_ticker_empresa, index=False)
salvar_dataframe(df_auditoria_disputas_ticker_empresa, caminho_tbl_auditoria_disputas_ticker_empresa, index=False)
salvar_dataframe(df_resumo_disputas_por_data, caminho_tbl_resumo_disputas_por_data, index=False)
salvar_dataframe(df_criterios_desempate_ticker_empresa, caminho_tbl_criterios_desempate_ticker_empresa, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_vencedores = df_ticker_vencedor_empresa_data.head(15).copy()
df_amostra_disputas = df_auditoria_disputas_ticker_empresa.head(20).copy()
df_amostra_resumo_disputas_por_data = df_resumo_disputas_por_data.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da escolha de um único ticker por empresa:")
print(df_resumo_escolha_ticker_empresa.to_string(index=False))

print("\nCritérios formais de desempate:")
print(df_criterios_desempate_ticker_empresa.to_string(index=False))

print("\nAmostra da base vencedora por empresa-data:")
print(df_amostra_vencedores.to_string(index=False))

print("\nAmostra da auditoria de disputas entre tickers da mesma empresa:")
print(df_amostra_disputas.to_string(index=False))

print("\nResumo de disputas por data - amostra:")
print(df_amostra_resumo_disputas_por_data.to_string(index=False))

print("\nArquivos salvos na subetapa 3.3:")
print(f"- {caminho_base_ticker_vencedor_empresa_data}")
print(f"- {caminho_tbl_resumo_escolha_ticker_empresa}")
print(f"- {caminho_tbl_auditoria_disputas_ticker_empresa}")
print(f"- {caminho_tbl_resumo_disputas_por_data}")
print(f"- {caminho_tbl_criterios_desempate_ticker_empresa}")

print("\nETAPA 3.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 3.3 - ESCOLHA DE UM ÚNICO TICKER POR EMPRESA

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - liquidez ticker-data    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_2_base_liquidez_ticker_data.parquet
Saída   - ticker vencedor         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_3_base_ticker_vencedor_empresa_data.parquet
Saída   - resumo                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_3_tbl_resumo_escolha_ticker_empresa.parquet
Saída   - auditoria de disputas   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_3_tbl_auditoria_disputas_ticker_em

## Etapa 3.4) Base de Elegibilidade Operacional por Data

In [14]:
%%time
# ============================================================
# Etapa 3.4) Base de Elegibilidade Operacional por Data
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 3.4 - BASE DE ELEGIBILIDADE OPERACIONAL POR DATA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_ticker_vencedor_empresa_data = gerar_caminho_arquivo(etapa=3, subetapa=3, tipo_arquivo="base", nome="ticker_vencedor_empresa_data")

caminho_base_elegibilidade_operacional_data = gerar_caminho_arquivo(etapa=3, subetapa=4, tipo_arquivo="base", nome="elegibilidade_operacional_data")
caminho_tbl_resumo_elegibilidade_operacional = gerar_caminho_arquivo(etapa=3, subetapa=4, tipo_arquivo="tbl", nome="resumo_elegibilidade_operacional")
caminho_tbl_cobertura_diaria_elegibilidade_operacional = gerar_caminho_arquivo(etapa=3, subetapa=4, tipo_arquivo="tbl", nome="cobertura_diaria_elegibilidade_operacional")
caminho_tbl_auditoria_chaves_elegibilidade_operacional = gerar_caminho_arquivo(etapa=3, subetapa=4, tipo_arquivo="tbl", nome="auditoria_chaves_elegibilidade_operacional")
caminho_tbl_auditoria_disputas_elegibilidade_operacional = gerar_caminho_arquivo(etapa=3, subetapa=4, tipo_arquivo="tbl", nome="auditoria_disputas_elegibilidade_operacional")

print(f"Entrada - ticker vencedor por empresa-data   : {caminho_base_ticker_vencedor_empresa_data}")
print(f"Saída   - base final de elegibilidade        : {caminho_base_elegibilidade_operacional_data}")
print(f"Saída   - resumo                             : {caminho_tbl_resumo_elegibilidade_operacional}")
print(f"Saída   - cobertura diária                   : {caminho_tbl_cobertura_diaria_elegibilidade_operacional}")
print(f"Saída   - auditoria de chaves                : {caminho_tbl_auditoria_chaves_elegibilidade_operacional}")
print(f"Saída   - auditoria de disputas              : {caminho_tbl_auditoria_disputas_elegibilidade_operacional}")
print("OK")

# ============================================================
# 3) Carga do input da subetapa 3.3
# ============================================================

print("\n[3/10] Carga do input da subetapa 3.3...")

df_ticker_vencedor_empresa_data = pd.read_parquet(caminho_base_ticker_vencedor_empresa_data)

print(f"Base ticker vencedor empresa-data: {df_ticker_vencedor_empresa_data.shape[0]:,} linhas x {df_ticker_vencedor_empresa_data.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e definição das colunas da base final
# ============================================================

print("\n[4/10] Funções auxiliares e definição das colunas da base final...")

COLUNAS_OBRIGATORIAS = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj_data_avaliacao",
    "pregoes_negociados_janela",
    "pct_pregoes_negociados_janela",
    "mediana_volume_fin_janela",
    "mediana_negocios_janela",
    "flag_janela_liquidez_completa",
    "flag_elegivel_liquidez_ticker_data",
    "n_tickers_elegiveis_empresa_data",
    "flag_disputa_ticker_empresa_data",
    "ranking_ticker_empresa_data",
    "flag_ticker_vencedor_empresa_data",
]

COLUNAS_BASE_FINAL = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj_data_avaliacao",
    "pregoes_negociados_janela",
    "pct_pregoes_negociados_janela",
    "mediana_volume_fin_janela",
    "mediana_negocios_janela",
    "flag_janela_liquidez_completa",
    "flag_elegivel_liquidez_ticker_data",
    "n_tickers_elegiveis_empresa_data",
    "flag_disputa_ticker_empresa_data",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

print(f"Quantidade de colunas esperadas no input : {len(COLUNAS_OBRIGATORIAS)}")
print(f"Quantidade de colunas da base final      : {len(COLUNAS_BASE_FINAL) + 2}")
print("OK")

# ============================================================
# 5) Construção da base final de elegibilidade operacional
# ============================================================

print("\n[5/10] Construção da base final de elegibilidade operacional...")

validar_colunas_obrigatorias(df_ticker_vencedor_empresa_data, COLUNAS_OBRIGATORIAS)

df_elegibilidade_operacional_data = df_ticker_vencedor_empresa_data.copy()
df_elegibilidade_operacional_data["ticker"] = df_elegibilidade_operacional_data["ticker"].astype("string").str.strip().str.upper()
df_elegibilidade_operacional_data["issuer_code"] = df_elegibilidade_operacional_data["issuer_code"].astype("string").str.strip()
df_elegibilidade_operacional_data["data"] = pd.to_datetime(df_elegibilidade_operacional_data["data"], errors="coerce")
df_elegibilidade_operacional_data["close_adj_data_avaliacao"] = pd.to_numeric(df_elegibilidade_operacional_data["close_adj_data_avaliacao"], errors="coerce")
df_elegibilidade_operacional_data["pregoes_negociados_janela"] = pd.to_numeric(df_elegibilidade_operacional_data["pregoes_negociados_janela"], errors="coerce")
df_elegibilidade_operacional_data["pct_pregoes_negociados_janela"] = pd.to_numeric(df_elegibilidade_operacional_data["pct_pregoes_negociados_janela"], errors="coerce")
df_elegibilidade_operacional_data["mediana_volume_fin_janela"] = pd.to_numeric(df_elegibilidade_operacional_data["mediana_volume_fin_janela"], errors="coerce")
df_elegibilidade_operacional_data["mediana_negocios_janela"] = pd.to_numeric(df_elegibilidade_operacional_data["mediana_negocios_janela"], errors="coerce")

df_elegibilidade_operacional_data = df_elegibilidade_operacional_data.loc[
    df_elegibilidade_operacional_data["flag_ticker_vencedor_empresa_data"]
    & df_elegibilidade_operacional_data["flag_elegivel_liquidez_ticker_data"]
].copy()

df_elegibilidade_operacional_data["flag_elegivel_operacional_final"] = True
df_elegibilidade_operacional_data["origem_elegibilidade_operacional"] = "liquidez_mais_consolidacao_empresa"

df_elegibilidade_operacional_data = (
    df_elegibilidade_operacional_data[
        COLUNAS_BASE_FINAL + ["flag_elegivel_operacional_final", "origem_elegibilidade_operacional"]
    ]
    .sort_values(["data", "issuer_code", "ticker"])
    .reset_index(drop=True)
)

print(f"Base final de elegibilidade operacional: {df_elegibilidade_operacional_data.shape[0]:,} linhas x {df_elegibilidade_operacional_data.shape[1]} colunas")
print("OK")

# ============================================================
# 6) Auditoria estrutural da base final
# ============================================================

print("\n[6/10] Auditoria estrutural da base final...")

n_duplicidades_ticker_data = int(df_elegibilidade_operacional_data.duplicated(subset=["ticker", "data"]).sum())
n_duplicidades_issuer_data = int(df_elegibilidade_operacional_data.duplicated(subset=["issuer_code", "data"]).sum())
n_linhas_sem_ticker = int(df_elegibilidade_operacional_data["ticker"].isna().sum())
n_linhas_sem_issuer_code = int(df_elegibilidade_operacional_data["issuer_code"].isna().sum())
n_linhas_sem_data = int(df_elegibilidade_operacional_data["data"].isna().sum())
n_linhas_flag_final_false = int((~df_elegibilidade_operacional_data["flag_elegivel_operacional_final"]).sum())
n_linhas_flag_janela_incompleta = int((~df_elegibilidade_operacional_data["flag_janela_liquidez_completa"]).sum())

if n_duplicidades_ticker_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_ticker_data} duplicidades na chave ticker-data da base final.")
if n_duplicidades_issuer_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_issuer_data} duplicidades na chave issuer_code-data da base final.")
if n_linhas_sem_ticker > 0 or n_linhas_sem_issuer_code > 0 or n_linhas_sem_data > 0:
    raise ValueError("Foram encontradas linhas com chaves obrigatórias ausentes na base final de elegibilidade operacional.")
if n_linhas_flag_final_false > 0:
    raise ValueError("Foram encontradas linhas com flag_elegivel_operacional_final = False na base final.")
if n_linhas_flag_janela_incompleta > 0:
    raise ValueError("Foram encontradas linhas com janela de liquidez incompleta na base final de elegibilidade operacional.")

df_auditoria_chaves_elegibilidade_operacional = pd.DataFrame(
    [
        {"metrica": "n_duplicidades_ticker_data", "valor": n_duplicidades_ticker_data},
        {"metrica": "n_duplicidades_issuer_code_data", "valor": n_duplicidades_issuer_data},
        {"metrica": "n_linhas_sem_ticker", "valor": n_linhas_sem_ticker},
        {"metrica": "n_linhas_sem_issuer_code", "valor": n_linhas_sem_issuer_code},
        {"metrica": "n_linhas_sem_data", "valor": n_linhas_sem_data},
        {"metrica": "n_linhas_flag_final_false", "valor": n_linhas_flag_final_false},
        {"metrica": "n_linhas_janela_incompleta", "valor": n_linhas_flag_janela_incompleta},
    ]
)

df_auditoria_disputas_elegibilidade_operacional = (
    df_elegibilidade_operacional_data
    .groupby("data", as_index=False)
    .agg(
        n_tickers_elegiveis_operacionais=("ticker", "nunique"),
        n_empresas_elegiveis_operacionais=("issuer_code", "nunique"),
        n_empresas_com_disputa_resolvida=("flag_disputa_ticker_empresa_data", "sum"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

print(f"Duplicidades ticker-data         : {n_duplicidades_ticker_data}")
print(f"Duplicidades issuer_code-data    : {n_duplicidades_issuer_data}")
print(f"Linhas com disputa resolvida     : {int(df_elegibilidade_operacional_data['flag_disputa_ticker_empresa_data'].sum()):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas-resumo da subetapa...")

df_resumo_elegibilidade_operacional = pd.DataFrame(
    [
        {
            "base": "elegibilidade_operacional_data",
            "n_linhas": int(len(df_elegibilidade_operacional_data)),
            "n_tickers": int(df_elegibilidade_operacional_data["ticker"].nunique(dropna=True)),
            "n_issuer_code": int(df_elegibilidade_operacional_data["issuer_code"].nunique(dropna=True)),
            "n_datas_unicas": int(df_elegibilidade_operacional_data["data"].nunique(dropna=True)),
            "data_min": df_elegibilidade_operacional_data["data"].min(),
            "data_max": df_elegibilidade_operacional_data["data"].max(),
            "n_linhas_com_disputa_resolvida": int(df_elegibilidade_operacional_data["flag_disputa_ticker_empresa_data"].sum()),
            "pct_linhas_com_disputa_resolvida": float(df_elegibilidade_operacional_data["flag_disputa_ticker_empresa_data"].mean()),
            "n_tickers_com_disputa_em_alguma_data": int(
                df_elegibilidade_operacional_data.loc[df_elegibilidade_operacional_data["flag_disputa_ticker_empresa_data"], "ticker"].nunique()
            ),
            "n_empresas_com_disputa_em_alguma_data": int(
                df_elegibilidade_operacional_data.loc[df_elegibilidade_operacional_data["flag_disputa_ticker_empresa_data"], "issuer_code"].nunique()
            ),
        }
    ]
)

df_cobertura_diaria_elegibilidade_operacional = (
    df_elegibilidade_operacional_data
    .groupby("data", as_index=False)
    .agg(
        n_tickers_elegiveis_operacionais=("ticker", "nunique"),
        n_empresas_elegiveis_operacionais=("issuer_code", "nunique"),
        mediana_volume_fin_janela=("mediana_volume_fin_janela", "median"),
        mediana_negocios_janela=("mediana_negocios_janela", "median"),
        mediana_pct_pregoes_negociados_janela=("pct_pregoes_negociados_janela", "median"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

print("Tabelas-resumo construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_elegibilidade_operacional_data, caminho_base_elegibilidade_operacional_data, index=False)
salvar_dataframe(df_resumo_elegibilidade_operacional, caminho_tbl_resumo_elegibilidade_operacional, index=False)
salvar_dataframe(df_cobertura_diaria_elegibilidade_operacional, caminho_tbl_cobertura_diaria_elegibilidade_operacional, index=False)
salvar_dataframe(df_auditoria_chaves_elegibilidade_operacional, caminho_tbl_auditoria_chaves_elegibilidade_operacional, index=False)
salvar_dataframe(df_auditoria_disputas_elegibilidade_operacional, caminho_tbl_auditoria_disputas_elegibilidade_operacional, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_base_final = df_elegibilidade_operacional_data.head(15).copy()
df_amostra_cobertura_inicial = df_cobertura_diaria_elegibilidade_operacional.head(10).copy()
df_amostra_cobertura_final = df_cobertura_diaria_elegibilidade_operacional.tail(10).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base final de elegibilidade operacional:")
print(df_resumo_elegibilidade_operacional.to_string(index=False))

print("\nAuditoria de chaves da base final:")
print(df_auditoria_chaves_elegibilidade_operacional.to_string(index=False))

print("\nAmostra da base final de elegibilidade operacional:")
print(df_amostra_base_final.to_string(index=False))

print("\nCobertura diária da elegibilidade operacional - amostra inicial:")
print(df_amostra_cobertura_inicial.to_string(index=False))

print("\nCobertura diária da elegibilidade operacional - amostra final:")
print(df_amostra_cobertura_final.to_string(index=False))

print("\nArquivos salvos na subetapa 3.4:")
print(f"- {caminho_base_elegibilidade_operacional_data}")
print(f"- {caminho_tbl_resumo_elegibilidade_operacional}")
print(f"- {caminho_tbl_cobertura_diaria_elegibilidade_operacional}")
print(f"- {caminho_tbl_auditoria_chaves_elegibilidade_operacional}")
print(f"- {caminho_tbl_auditoria_disputas_elegibilidade_operacional}")

print("\nETAPA 3.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 3.4 - BASE DE ELEGIBILIDADE OPERACIONAL POR DATA

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - ticker vencedor por empresa-data   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_3_base_ticker_vencedor_empresa_data.parquet
Saída   - base final de elegibilidade        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_4_base_elegibilidade_operacional_data.parquet
Saída   - resumo                             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_4_tbl_resumo_elegibilidade_operacional.parquet
Saída   - cobertura diária                   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasi

# Etapa 4) Construção da Base Contábil Operacional

## Etapa 4.1) Padronização da Base Trimestral

In [15]:
%%time
# ============================================================
# Etapa 4.1) Padronização da Base Trimestral
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 4.1 - PADRONIZAÇÃO DA BASE TRIMESTRAL")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_indicadores_trimestrais_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="indicadores_trimestrais_padronizada")
caminho_info_empresa_padronizada = gerar_caminho_arquivo(etapa=1, subetapa=3, tipo_arquivo="base", nome="info_empresa_padronizada")

caminho_base_contabil_trimestral_operacional = gerar_caminho_arquivo(etapa=4, subetapa=1, tipo_arquivo="base", nome="contabil_trimestral_operacional")
caminho_tbl_resumo_base_contabil_operacional = gerar_caminho_arquivo(etapa=4, subetapa=1, tipo_arquivo="tbl", nome="resumo_base_contabil_operacional")
caminho_tbl_auditoria_coerencia_contabil = gerar_caminho_arquivo(etapa=4, subetapa=1, tipo_arquivo="tbl", nome="auditoria_coerencia_contabil")
caminho_tbl_duplicidades_empresa_periodo = gerar_caminho_arquivo(etapa=4, subetapa=1, tipo_arquivo="tbl", nome="duplicidades_empresa_periodo_contabil")
caminho_tbl_consistencia_cadastro_contabil = gerar_caminho_arquivo(etapa=4, subetapa=1, tipo_arquivo="tbl", nome="consistencia_cadastro_contabil")

print(f"Entrada - indicadores trimestrais padronizada : {caminho_indicadores_trimestrais_padronizada}")
print(f"Entrada - info empresa padronizada            : {caminho_info_empresa_padronizada}")
print(f"Saída   - base contábil operacional           : {caminho_base_contabil_trimestral_operacional}")
print(f"Saída   - resumo                              : {caminho_tbl_resumo_base_contabil_operacional}")
print(f"Saída   - auditoria de coerência              : {caminho_tbl_auditoria_coerencia_contabil}")
print(f"Saída   - duplicidades empresa-período        : {caminho_tbl_duplicidades_empresa_periodo}")
print(f"Saída   - consistência com cadastro           : {caminho_tbl_consistencia_cadastro_contabil}")
print("OK")

# ============================================================
# 3) Carga das bases de entrada
# ============================================================

print("\n[3/10] Carga das bases de entrada...")

df_indicadores_trimestrais_padronizada = pd.read_parquet(caminho_indicadores_trimestrais_padronizada)
df_info_empresa_padronizada = pd.read_parquet(caminho_info_empresa_padronizada)

print(f"Base indicadores trimestrais padronizada : {df_indicadores_trimestrais_padronizada.shape[0]:,} linhas x {df_indicadores_trimestrais_padronizada.shape[1]} colunas")
print(f"Base info empresa padronizada            : {df_info_empresa_padronizada.shape[0]:,} linhas x {df_info_empresa_padronizada.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/10] Funções auxiliares da subetapa...")

COLUNAS_OBRIGATORIAS_INDICADORES = [
    "issuer_code",
    "ticker",
    "periodo",
    "data_inicio_periodo",
    "data_fim_periodo",
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

COLUNAS_OBRIGATORIAS_CADASTRO = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
]

COLUNAS_NUMERICAS_CONTABEIS = [
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def contar_unicos_nao_nulos(serie):
    """
    Conta valores únicos não nulos em uma série.
    """
    return int(serie.dropna().nunique())

def primeiro_valor_nao_nulo(serie):
    """
    Retorna o primeiro valor não nulo de uma série.
    """
    serie = serie.dropna()
    if len(serie) == 0:
        return pd.NA
    return serie.iloc[0]

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização operacional da base trimestral
# ============================================================

print("\n[5/10] Padronização operacional da base trimestral...")

validar_colunas_obrigatorias(df_indicadores_trimestrais_padronizada, COLUNAS_OBRIGATORIAS_INDICADORES)
validar_colunas_obrigatorias(df_info_empresa_padronizada, COLUNAS_OBRIGATORIAS_CADASTRO)

df_contabil_trimestral_operacional = df_indicadores_trimestrais_padronizada.copy()

df_contabil_trimestral_operacional["ticker"] = df_contabil_trimestral_operacional["ticker"].astype("string").str.strip().str.upper()
df_contabil_trimestral_operacional["issuer_code"] = df_contabil_trimestral_operacional["issuer_code"].astype("string").str.strip()
df_contabil_trimestral_operacional["periodo"] = df_contabil_trimestral_operacional["periodo"].astype("string").str.strip().str.upper()
df_contabil_trimestral_operacional["data_inicio_periodo"] = pd.to_datetime(df_contabil_trimestral_operacional["data_inicio_periodo"], errors="coerce")
df_contabil_trimestral_operacional["data_fim_periodo"] = pd.to_datetime(df_contabil_trimestral_operacional["data_fim_periodo"], errors="coerce")

for coluna in COLUNAS_NUMERICAS_CONTABEIS:
    df_contabil_trimestral_operacional[coluna] = pd.to_numeric(df_contabil_trimestral_operacional[coluna], errors="coerce")

n_duplicidades_exatas_antes = int(df_contabil_trimestral_operacional.duplicated().sum())

df_contabil_trimestral_operacional = (
    df_contabil_trimestral_operacional
    .drop_duplicates()
    .sort_values(["issuer_code", "ticker", "data_inicio_periodo", "periodo"])
    .reset_index(drop=True)
)

print(f"Duplicidades exatas removidas na base contábil: {n_duplicidades_exatas_antes:,}")
print(f"Base contábil operacional após padronização   : {df_contabil_trimestral_operacional.shape[0]:,} linhas")
print("OK")

# ============================================================
# 6) Auditoria de coerência entre ticker, issuer_code e período
# ============================================================

print("\n[6/10] Auditoria de coerência entre ticker, issuer_code e período...")

n_periodo_nulo = int(df_contabil_trimestral_operacional["periodo"].isna().sum())
n_data_inicio_nula = int(df_contabil_trimestral_operacional["data_inicio_periodo"].isna().sum())
n_data_fim_nula = int(df_contabil_trimestral_operacional["data_fim_periodo"].isna().sum())
n_periodo_invertido = int(
    (
        pd.notna(df_contabil_trimestral_operacional["data_inicio_periodo"])
        & pd.notna(df_contabil_trimestral_operacional["data_fim_periodo"])
        & df_contabil_trimestral_operacional["data_inicio_periodo"].gt(df_contabil_trimestral_operacional["data_fim_periodo"])
    ).sum()
)
n_duplicidades_chave_exata = int(
    df_contabil_trimestral_operacional.duplicated(subset=["issuer_code", "ticker", "periodo"]).sum()
)
n_duplicidades_empresa_periodo = int(
    df_contabil_trimestral_operacional.duplicated(subset=["issuer_code", "periodo"]).sum()
)
n_tickers_multiplos_por_empresa_periodo = int(
    (
        df_contabil_trimestral_operacional
        .groupby(["issuer_code", "periodo"], dropna=False)["ticker"]
        .agg(contar_unicos_nao_nulos)
        .gt(1)
        .sum()
    )
)
n_issuer_multiplos_por_ticker_periodo = int(
    (
        df_contabil_trimestral_operacional
        .groupby(["ticker", "periodo"], dropna=False)["issuer_code"]
        .agg(contar_unicos_nao_nulos)
        .gt(1)
        .sum()
    )
)

df_auditoria_coerencia_contabil = pd.DataFrame(
    [
        {"metrica": "n_duplicidades_exatas_removidas", "valor": n_duplicidades_exatas_antes},
        {"metrica": "n_periodo_nulo", "valor": n_periodo_nulo},
        {"metrica": "n_data_inicio_periodo_nula", "valor": n_data_inicio_nula},
        {"metrica": "n_data_fim_periodo_nula", "valor": n_data_fim_nula},
        {"metrica": "n_periodo_invertido", "valor": n_periodo_invertido},
        {"metrica": "n_duplicidades_chave_issuer_ticker_periodo", "valor": n_duplicidades_chave_exata},
        {"metrica": "n_duplicidades_chave_issuer_periodo", "valor": n_duplicidades_empresa_periodo},
        {"metrica": "n_empresa_periodo_com_multiplos_tickers", "valor": n_tickers_multiplos_por_empresa_periodo},
        {"metrica": "n_ticker_periodo_com_multiplos_issuer_code", "valor": n_issuer_multiplos_por_ticker_periodo},
    ]
)

df_duplicidades_empresa_periodo_contabil = (
    df_contabil_trimestral_operacional
    .groupby(["issuer_code", "periodo"], dropna=False)
    .agg(
        n_registros=("ticker", "size"),
        n_tickers=("ticker", "nunique"),
        data_inicio_periodo=("data_inicio_periodo", "min"),
        data_fim_periodo=("data_fim_periodo", "max"),
        ticker_exemplo=("ticker", primeiro_valor_nao_nulo),
    )
    .reset_index()
    .sort_values(["n_tickers", "n_registros", "issuer_code", "periodo"], ascending=[False, False, True, True])
    .reset_index(drop=True)
)

df_duplicidades_empresa_periodo_contabil = df_duplicidades_empresa_periodo_contabil.loc[
    df_duplicidades_empresa_periodo_contabil["n_registros"].gt(1)
].copy()

print(f"Duplicidades por issuer_code-periodo identificadas : {len(df_duplicidades_empresa_periodo_contabil):,}")
print(f"Empresa-período com múltiplos tickers              : {n_tickers_multiplos_por_empresa_periodo:,}")
print("OK")

# ============================================================
# 7) Consistência da base contábil com o cadastro empresarial
# ============================================================

print("\n[7/10] Consistência da base contábil com o cadastro empresarial...")

df_info_empresa_aux = df_info_empresa_padronizada.copy()
df_info_empresa_aux["ticker"] = df_info_empresa_aux["ticker"].astype("string").str.strip().str.upper()
df_info_empresa_aux["issuer_code"] = df_info_empresa_aux["issuer_code"].astype("string").str.strip()

df_contabil_com_cadastro = df_contabil_trimestral_operacional.merge(
    df_info_empresa_aux[["ticker", "issuer_code", "nome", "setor", "subsetor", "segmento"]],
    on=["ticker", "issuer_code"],
    how="left",
    validate="many_to_one",
)

n_linhas_sem_match_cadastro = int(df_contabil_com_cadastro["nome"].isna().sum())
n_tickers_contabil_sem_match_cadastro = int(
    df_contabil_com_cadastro.loc[df_contabil_com_cadastro["nome"].isna(), "ticker"].nunique(dropna=True)
)
n_issuer_contabil_sem_match_cadastro = int(
    df_contabil_com_cadastro.loc[df_contabil_com_cadastro["nome"].isna(), "issuer_code"].nunique(dropna=True)
)

df_consistencia_cadastro_contabil = pd.DataFrame(
    [
        {"metrica": "n_linhas_contabeis_sem_match_no_cadastro", "valor": n_linhas_sem_match_cadastro},
        {"metrica": "n_tickers_contabeis_sem_match_no_cadastro", "valor": n_tickers_contabil_sem_match_cadastro},
        {"metrica": "n_issuer_code_contabeis_sem_match_no_cadastro", "valor": n_issuer_contabil_sem_match_cadastro},
        {"metrica": "n_tickers_contabeis_total", "valor": int(df_contabil_trimestral_operacional["ticker"].nunique(dropna=True))},
        {"metrica": "n_issuer_code_contabeis_total", "valor": int(df_contabil_trimestral_operacional["issuer_code"].nunique(dropna=True))},
    ]
)

print(f"Linhas contábeis sem correspondência no cadastro : {n_linhas_sem_match_cadastro:,}")
print(f"Tickers contábeis sem match no cadastro          : {n_tickers_contabil_sem_match_cadastro:,}")
print("OK")

# ============================================================
# 8) Construção das tabelas-resumo e salvamento
# ============================================================

print("\n[8/10] Construção das tabelas-resumo e salvamento...")

df_resumo_base_contabil_operacional = pd.DataFrame(
    [
        {
            "base": "contabil_trimestral_operacional",
            "n_linhas": int(len(df_contabil_trimestral_operacional)),
            "n_colunas": int(df_contabil_trimestral_operacional.shape[1]),
            "n_tickers": int(df_contabil_trimestral_operacional["ticker"].nunique(dropna=True)),
            "n_issuer_code": int(df_contabil_trimestral_operacional["issuer_code"].nunique(dropna=True)),
            "n_periodos": int(df_contabil_trimestral_operacional["periodo"].nunique(dropna=True)),
            "data_inicio_periodo_min": df_contabil_trimestral_operacional["data_inicio_periodo"].min(),
            "data_fim_periodo_max": df_contabil_trimestral_operacional["data_fim_periodo"].max(),
            "n_empresa_periodo_com_multiplos_tickers": n_tickers_multiplos_por_empresa_periodo,
            "n_linhas_sem_match_cadastro": n_linhas_sem_match_cadastro,
        }
    ]
)

salvar_dataframe(df_contabil_trimestral_operacional, caminho_base_contabil_trimestral_operacional, index=False)
salvar_dataframe(df_resumo_base_contabil_operacional, caminho_tbl_resumo_base_contabil_operacional, index=False)
salvar_dataframe(df_auditoria_coerencia_contabil, caminho_tbl_auditoria_coerencia_contabil, index=False)
salvar_dataframe(df_duplicidades_empresa_periodo_contabil, caminho_tbl_duplicidades_empresa_periodo, index=False)
salvar_dataframe(df_consistencia_cadastro_contabil, caminho_tbl_consistencia_cadastro_contabil, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_contabil_operacional = df_contabil_trimestral_operacional.head(15).copy()
df_amostra_duplicidades_empresa_periodo = df_duplicidades_empresa_periodo_contabil.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base contábil operacional:")
print(df_resumo_base_contabil_operacional.to_string(index=False))

print("\nAuditoria de coerência da base contábil:")
print(df_auditoria_coerencia_contabil.to_string(index=False))

print("\nConsistência da base contábil com o cadastro empresarial:")
print(df_consistencia_cadastro_contabil.to_string(index=False))

print("\nAmostra da base contábil operacional:")
print(df_amostra_contabil_operacional.to_string(index=False))

print("\nAmostra de duplicidades por empresa-período:")
print(df_amostra_duplicidades_empresa_periodo.to_string(index=False))

print("\nArquivos salvos na subetapa 4.1:")
print(f"- {caminho_base_contabil_trimestral_operacional}")
print(f"- {caminho_tbl_resumo_base_contabil_operacional}")
print(f"- {caminho_tbl_auditoria_coerencia_contabil}")
print(f"- {caminho_tbl_duplicidades_empresa_periodo}")
print(f"- {caminho_tbl_consistencia_cadastro_contabil}")

print("\nETAPA 4.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 4.1 - PADRONIZAÇÃO DA BASE TRIMESTRAL

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - indicadores trimestrais padronizada : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_indicadores_trimestrais_padronizada.parquet
Entrada - info empresa padronizada            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_info_empresa_padronizada.parquet
Saída   - base contábil operacional           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_1_base_contabil_trimestral_operacional.parquet
Saída   - resumo                              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\

## Etapa 4.2) Definição do Calendário de Vigência Contábil

In [16]:
%%time
# ============================================================
# Etapa 4.2) Definição do Calendário de Vigência Contábil
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 4.2 - DEFINIÇÃO DO CALENDÁRIO DE VIGÊNCIA CONTÁBIL")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_contabil_trimestral_operacional = gerar_caminho_arquivo(etapa=4, subetapa=1, tipo_arquivo="base", nome="contabil_trimestral_operacional")
caminho_base_elegibilidade_operacional_data = gerar_caminho_arquivo(etapa=3, subetapa=4, tipo_arquivo="base", nome="elegibilidade_operacional_data")

caminho_base_calendario_contabil_operacional = gerar_caminho_arquivo(etapa=4, subetapa=2, tipo_arquivo="base", nome="calendario_contabil_operacional")
caminho_tbl_resumo_calendario_contabil_operacional = gerar_caminho_arquivo(etapa=4, subetapa=2, tipo_arquivo="tbl", nome="resumo_calendario_contabil_operacional")
caminho_tbl_parametros_vigencia_contabil = gerar_caminho_arquivo(etapa=4, subetapa=2, tipo_arquivo="tbl", nome="parametros_vigencia_contabil")
caminho_tbl_auditoria_vigencia_contabil = gerar_caminho_arquivo(etapa=4, subetapa=2, tipo_arquivo="tbl", nome="auditoria_vigencia_contabil")
caminho_tbl_inconsistencias_empresa_periodo_contabil = gerar_caminho_arquivo(etapa=4, subetapa=2, tipo_arquivo="tbl", nome="inconsistencias_empresa_periodo_contabil")

print(f"Entrada - base contábil operacional          : {caminho_base_contabil_trimestral_operacional}")
print(f"Entrada - elegibilidade operacional por data : {caminho_base_elegibilidade_operacional_data}")
print(f"Saída   - calendário contábil operacional    : {caminho_base_calendario_contabil_operacional}")
print(f"Saída   - resumo                             : {caminho_tbl_resumo_calendario_contabil_operacional}")
print(f"Saída   - parâmetros                         : {caminho_tbl_parametros_vigencia_contabil}")
print(f"Saída   - auditoria                          : {caminho_tbl_auditoria_vigencia_contabil}")
print(f"Saída   - inconsistências                    : {caminho_tbl_inconsistencias_empresa_periodo_contabil}")
print("OK")

# ============================================================
# 3) Carga das bases de entrada
# ============================================================

print("\n[3/10] Carga das bases de entrada...")

df_contabil_trimestral_operacional = pd.read_parquet(caminho_base_contabil_trimestral_operacional)
df_elegibilidade_operacional_data = pd.read_parquet(caminho_base_elegibilidade_operacional_data)

print(f"Base contábil operacional          : {df_contabil_trimestral_operacional.shape[0]:,} linhas x {df_contabil_trimestral_operacional.shape[1]} colunas")
print(f"Base elegibilidade operacional data: {df_elegibilidade_operacional_data.shape[0]:,} linhas x {df_elegibilidade_operacional_data.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros metodológicos e funções auxiliares
# ============================================================

print("\n[4/10] Parâmetros metodológicos e funções auxiliares...")

DEFASAGEM_CONTABIL_DIAS = 90

COLUNAS_OBRIGATORIAS_CONTABIL = [
    "issuer_code",
    "ticker",
    "periodo",
    "data_inicio_periodo",
    "data_fim_periodo",
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

COLUNAS_NUMERICAS_CONTABEIS = [
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def primeiro_valor_nao_nulo(serie):
    """
    Retorna o primeiro valor não nulo de uma série.
    """
    serie = serie.dropna()
    if len(serie) == 0:
        return pd.NA
    return serie.iloc[0]

def contar_unicos_nao_nulos(serie):
    """
    Conta valores únicos não nulos em uma série.
    """
    return int(serie.dropna().nunique())

def registrar_valor_misto(chave, valor, unidade, nome_chave):
    """
    Estrutura valores mistos em colunas homogêneas para escrita segura em parquet.
    """
    if isinstance(valor, (int, float)) and not pd.isna(valor):
        return {
            nome_chave: chave,
            "valor_numerico": float(valor),
            "valor_texto": pd.NA,
            "unidade": unidade,
        }
    return {
        nome_chave: chave,
        "valor_numerico": pd.NA,
        "valor_texto": str(valor) if pd.notna(valor) else pd.NA,
        "unidade": unidade,
    }

print(f"Defasagem contábil operacional definida : {DEFASAGEM_CONTABIL_DIAS} dias corridos")
print("OK")

# ============================================================
# 5) Preparação da base e definição do horizonte operacional do projeto
# ============================================================

print("\n[5/10] Preparação da base e definição do horizonte operacional do projeto...")

validar_colunas_obrigatorias(df_contabil_trimestral_operacional, COLUNAS_OBRIGATORIAS_CONTABIL)
validar_colunas_obrigatorias(df_elegibilidade_operacional_data, ["data"])

df_contabil_trimestral_operacional["issuer_code"] = df_contabil_trimestral_operacional["issuer_code"].astype("string").str.strip()
df_contabil_trimestral_operacional["ticker"] = df_contabil_trimestral_operacional["ticker"].astype("string").str.strip().str.upper()
df_contabil_trimestral_operacional["periodo"] = df_contabil_trimestral_operacional["periodo"].astype("string").str.strip().str.upper()
df_contabil_trimestral_operacional["data_inicio_periodo"] = pd.to_datetime(df_contabil_trimestral_operacional["data_inicio_periodo"], errors="coerce")
df_contabil_trimestral_operacional["data_fim_periodo"] = pd.to_datetime(df_contabil_trimestral_operacional["data_fim_periodo"], errors="coerce")

for coluna in COLUNAS_NUMERICAS_CONTABEIS:
    df_contabil_trimestral_operacional[coluna] = pd.to_numeric(df_contabil_trimestral_operacional[coluna], errors="coerce")

df_elegibilidade_operacional_data["data"] = pd.to_datetime(df_elegibilidade_operacional_data["data"], errors="coerce")

DATA_INICIAL_PROJETO_OPERACIONAL = pd.to_datetime(df_elegibilidade_operacional_data["data"], errors="coerce").min()
DATA_FINAL_PROJETO_OPERACIONAL = pd.to_datetime(df_elegibilidade_operacional_data["data"], errors="coerce").max()

print(f"Data inicial do horizonte operacional do projeto: {DATA_INICIAL_PROJETO_OPERACIONAL}")
print(f"Data final do horizonte operacional do projeto  : {DATA_FINAL_PROJETO_OPERACIONAL}")
print("OK")

# ============================================================
# 6) Consolidação empresa-período e construção das janelas de vigência
# ============================================================

print("\n[6/10] Consolidação empresa-período e construção das janelas de vigência...")

colunas_agregacao = {
    "ticker_exemplo": ("ticker", primeiro_valor_nao_nulo),
    "n_tickers_periodo": ("ticker", contar_unicos_nao_nulos),
    "data_inicio_periodo": ("data_inicio_periodo", "min"),
    "data_fim_periodo": ("data_fim_periodo", "max"),
    "lucro_liquido": ("lucro_liquido", primeiro_valor_nao_nulo),
    "patrimonio_liquido": ("patrimonio_liquido", primeiro_valor_nao_nulo),
    "receita_liquida": ("receita_liquida", primeiro_valor_nao_nulo),
    "ebit": ("ebit", primeiro_valor_nao_nulo),
    "liquidez_corrente": ("liquidez_corrente", primeiro_valor_nao_nulo),
    "liquidez_geral": ("liquidez_geral", primeiro_valor_nao_nulo),
    "liquidez_imediata": ("liquidez_imediata", primeiro_valor_nao_nulo),
}

df_calendario_contabil_operacional = (
    df_contabil_trimestral_operacional
    .groupby(["issuer_code", "periodo"], as_index=False)
    .agg(**colunas_agregacao)
    .sort_values(["issuer_code", "data_fim_periodo", "periodo"])
    .reset_index(drop=True)
)

df_calendario_contabil_operacional["data_inicio_vigencia_operacional"] = (
    df_calendario_contabil_operacional["data_fim_periodo"] + pd.to_timedelta(DEFASAGEM_CONTABIL_DIAS, unit="D")
)

df_calendario_contabil_operacional["data_inicio_vigencia_proximo_periodo"] = (
    df_calendario_contabil_operacional
    .groupby("issuer_code", dropna=False)["data_inicio_vigencia_operacional"]
    .shift(-1)
)

df_calendario_contabil_operacional["data_fim_vigencia_operacional"] = (
    df_calendario_contabil_operacional["data_inicio_vigencia_proximo_periodo"] - pd.Timedelta(days=1)
)

df_calendario_contabil_operacional["data_fim_vigencia_operacional"] = df_calendario_contabil_operacional["data_fim_vigencia_operacional"].fillna(DATA_FINAL_PROJETO_OPERACIONAL)

df_calendario_contabil_operacional["flag_vigencia_dentro_horizonte_projeto"] = (
    df_calendario_contabil_operacional["data_inicio_vigencia_operacional"].le(DATA_FINAL_PROJETO_OPERACIONAL)
)

df_calendario_contabil_operacional.loc[
    ~df_calendario_contabil_operacional["flag_vigencia_dentro_horizonte_projeto"],
    "data_fim_vigencia_operacional"
] = pd.NaT

df_calendario_contabil_operacional["duracao_vigencia_dias"] = (
    df_calendario_contabil_operacional["data_fim_vigencia_operacional"] - df_calendario_contabil_operacional["data_inicio_vigencia_operacional"]
).dt.days.add(1)

df_calendario_contabil_operacional["flag_vigencia_valida"] = (
    pd.notna(df_calendario_contabil_operacional["data_inicio_vigencia_operacional"])
    & pd.notna(df_calendario_contabil_operacional["data_fim_vigencia_operacional"])
    & df_calendario_contabil_operacional["data_inicio_vigencia_operacional"].le(df_calendario_contabil_operacional["data_fim_vigencia_operacional"])
)

df_calendario_contabil_operacional["data_inicio_vigencia_operacional"] = pd.to_datetime(df_calendario_contabil_operacional["data_inicio_vigencia_operacional"], errors="coerce")
df_calendario_contabil_operacional["data_fim_vigencia_operacional"] = pd.to_datetime(df_calendario_contabil_operacional["data_fim_vigencia_operacional"], errors="coerce")

print(f"Calendário contábil operacional consolidado: {df_calendario_contabil_operacional.shape[0]:,} linhas")
print("OK")

# ============================================================
# 7) Auditoria de inconsistências empresa-período e da vigência contábil
# ============================================================

print("\n[7/10] Auditoria de inconsistências empresa-período e da vigência contábil...")

df_inconsistencias_empresa_periodo_contabil = (
    df_contabil_trimestral_operacional
    .groupby(["issuer_code", "periodo"], as_index=False)
    .agg(
        n_tickers_periodo=("ticker", contar_unicos_nao_nulos),
        n_data_inicio_periodo_unicas=("data_inicio_periodo", contar_unicos_nao_nulos),
        n_data_fim_periodo_unicas=("data_fim_periodo", contar_unicos_nao_nulos),
        n_lucro_liquido_unicos=("lucro_liquido", contar_unicos_nao_nulos),
        n_patrimonio_liquido_unicos=("patrimonio_liquido", contar_unicos_nao_nulos),
        n_receita_liquida_unicos=("receita_liquida", contar_unicos_nao_nulos),
        n_ebit_unicos=("ebit", contar_unicos_nao_nulos),
        n_liquidez_corrente_unicos=("liquidez_corrente", contar_unicos_nao_nulos),
        n_liquidez_geral_unicos=("liquidez_geral", contar_unicos_nao_nulos),
        n_liquidez_imediata_unicos=("liquidez_imediata", contar_unicos_nao_nulos),
        ticker_exemplo=("ticker", primeiro_valor_nao_nulo),
    )
    .sort_values(["n_tickers_periodo", "issuer_code", "periodo"], ascending=[False, True, True])
    .reset_index(drop=True)
)

df_inconsistencias_empresa_periodo_contabil["flag_inconsistencia_qualquer"] = (
    df_inconsistencias_empresa_periodo_contabil["n_data_inicio_periodo_unicas"].gt(1)
    | df_inconsistencias_empresa_periodo_contabil["n_data_fim_periodo_unicas"].gt(1)
    | df_inconsistencias_empresa_periodo_contabil["n_lucro_liquido_unicos"].gt(1)
    | df_inconsistencias_empresa_periodo_contabil["n_patrimonio_liquido_unicos"].gt(1)
    | df_inconsistencias_empresa_periodo_contabil["n_receita_liquida_unicos"].gt(1)
    | df_inconsistencias_empresa_periodo_contabil["n_ebit_unicos"].gt(1)
    | df_inconsistencias_empresa_periodo_contabil["n_liquidez_corrente_unicos"].gt(1)
    | df_inconsistencias_empresa_periodo_contabil["n_liquidez_geral_unicos"].gt(1)
    | df_inconsistencias_empresa_periodo_contabil["n_liquidez_imediata_unicos"].gt(1)
)

df_inconsistencias_empresa_periodo_contabil = df_inconsistencias_empresa_periodo_contabil.loc[
    df_inconsistencias_empresa_periodo_contabil["flag_inconsistencia_qualquer"]
].copy()

n_vigencias_invalidas = int((~df_calendario_contabil_operacional["flag_vigencia_valida"]).sum())
n_vigencias_fora_horizonte = int((~df_calendario_contabil_operacional["flag_vigencia_dentro_horizonte_projeto"]).sum())
n_empresa_periodo_multiplos_tickers = int(df_calendario_contabil_operacional["n_tickers_periodo"].gt(1).sum())
n_inconsistencias_contabeis_entre_tickers = int(len(df_inconsistencias_empresa_periodo_contabil))

df_auditoria_vigencia_contabil = pd.DataFrame(
    [
        registrar_valor_misto("defasagem_contabil_dias", DEFASAGEM_CONTABIL_DIAS, "dias_corridos", "metrica"),
        registrar_valor_misto("n_linhas_calendario_contabil_operacional", int(len(df_calendario_contabil_operacional)), "linhas", "metrica"),
        registrar_valor_misto("n_issuer_code_calendario_contabil_operacional", int(df_calendario_contabil_operacional["issuer_code"].nunique(dropna=True)), "empresas", "metrica"),
        registrar_valor_misto("n_periodos_calendario_contabil_operacional", int(df_calendario_contabil_operacional["periodo"].nunique(dropna=True)), "periodos", "metrica"),
        registrar_valor_misto("n_empresa_periodo_com_multiplos_tickers", n_empresa_periodo_multiplos_tickers, "empresa_periodo", "metrica"),
        registrar_valor_misto("n_empresa_periodo_com_inconsistencia_contabil_entre_tickers", n_inconsistencias_contabeis_entre_tickers, "empresa_periodo", "metrica"),
        registrar_valor_misto("n_vigencias_invalidas", n_vigencias_invalidas, "linhas", "metrica"),
        registrar_valor_misto("n_vigencias_fora_horizonte_projeto", n_vigencias_fora_horizonte, "linhas", "metrica"),
        registrar_valor_misto("data_inicial_projeto_operacional", str(DATA_INICIAL_PROJETO_OPERACIONAL.date()), "data", "metrica"),
        registrar_valor_misto("data_final_projeto_operacional", str(DATA_FINAL_PROJETO_OPERACIONAL.date()), "data", "metrica"),
    ]
)

print(f"Empresa-período com múltiplos tickers                : {n_empresa_periodo_multiplos_tickers:,}")
print(f"Empresa-período com inconsistência contábil real     : {n_inconsistencias_contabeis_entre_tickers:,}")
print(f"Vigências fora do horizonte operacional do projeto   : {n_vigencias_fora_horizonte:,}")
print("OK")

# ============================================================
# 8) Construção das tabelas-resumo e salvamento
# ============================================================

print("\n[8/10] Construção das tabelas-resumo e salvamento...")

df_parametros_vigencia_contabil = pd.DataFrame(
    [
        registrar_valor_misto("defasagem_contabil_dias", DEFASAGEM_CONTABIL_DIAS, "dias_corridos", "parametro"),
        registrar_valor_misto("data_inicial_projeto_operacional", str(DATA_INICIAL_PROJETO_OPERACIONAL.date()), "data", "parametro"),
        registrar_valor_misto("data_final_projeto_operacional", str(DATA_FINAL_PROJETO_OPERACIONAL.date()), "data", "parametro"),
    ]
)

df_resumo_calendario_contabil_operacional = pd.DataFrame(
    [
        {
            "base": "calendario_contabil_operacional",
            "n_linhas": int(len(df_calendario_contabil_operacional)),
            "n_issuer_code": int(df_calendario_contabil_operacional["issuer_code"].nunique(dropna=True)),
            "n_periodos": int(df_calendario_contabil_operacional["periodo"].nunique(dropna=True)),
            "data_inicio_periodo_min": df_calendario_contabil_operacional["data_inicio_periodo"].min(),
            "data_fim_periodo_max": df_calendario_contabil_operacional["data_fim_periodo"].max(),
            "data_inicio_vigencia_min": df_calendario_contabil_operacional["data_inicio_vigencia_operacional"].min(),
            "data_fim_vigencia_max": df_calendario_contabil_operacional["data_fim_vigencia_operacional"].max(),
            "n_vigencias_validas": int(df_calendario_contabil_operacional["flag_vigencia_valida"].sum()),
            "n_vigencias_fora_horizonte_projeto": n_vigencias_fora_horizonte,
        }
    ]
)

salvar_dataframe(df_calendario_contabil_operacional, caminho_base_calendario_contabil_operacional, index=False)
salvar_dataframe(df_resumo_calendario_contabil_operacional, caminho_tbl_resumo_calendario_contabil_operacional, index=False)
salvar_dataframe(df_parametros_vigencia_contabil, caminho_tbl_parametros_vigencia_contabil, index=False)
salvar_dataframe(df_auditoria_vigencia_contabil, caminho_tbl_auditoria_vigencia_contabil, index=False)
salvar_dataframe(df_inconsistencias_empresa_periodo_contabil, caminho_tbl_inconsistencias_empresa_periodo_contabil, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_calendario_contabil = df_calendario_contabil_operacional.head(15).copy()
df_amostra_inconsistencias_contabeis = df_inconsistencias_empresa_periodo_contabil.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo do calendário contábil operacional:")
print(df_resumo_calendario_contabil_operacional.to_string(index=False))

print("\nParâmetros formais da vigência contábil:")
print(df_parametros_vigencia_contabil.to_string(index=False))

print("\nAuditoria da vigência contábil:")
print(df_auditoria_vigencia_contabil.to_string(index=False))

print("\nAmostra do calendário contábil operacional:")
print(df_amostra_calendario_contabil.to_string(index=False))

print("\nAmostra de inconsistências contábeis entre tickers da mesma empresa-período:")
print(df_amostra_inconsistencias_contabeis.to_string(index=False))

print("\nArquivos salvos na subetapa 4.2:")
print(f"- {caminho_base_calendario_contabil_operacional}")
print(f"- {caminho_tbl_resumo_calendario_contabil_operacional}")
print(f"- {caminho_tbl_parametros_vigencia_contabil}")
print(f"- {caminho_tbl_auditoria_vigencia_contabil}")
print(f"- {caminho_tbl_inconsistencias_empresa_periodo_contabil}")

print("\nETAPA 4.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 4.2 - DEFINIÇÃO DO CALENDÁRIO DE VIGÊNCIA CONTÁBIL

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - base contábil operacional          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_1_base_contabil_trimestral_operacional.parquet
Entrada - elegibilidade operacional por data : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_4_base_elegibilidade_operacional_data.parquet
Saída   - calendário contábil operacional    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_2_base_calendario_contabil_operacional.parquet
Saída   - resumo                             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_

## Etapa 4.3) Definição dos Critérios do Filtro Contábil

In [17]:
%%time
# ============================================================
# Etapa 4.3) Definição dos Critérios do Filtro Contábil
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 4.3 - DEFINIÇÃO DOS CRITÉRIOS DO FILTRO CONTÁBIL")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_calendario_contabil_operacional = gerar_caminho_arquivo(etapa=4, subetapa=2, tipo_arquivo="base", nome="calendario_contabil_operacional")

caminho_tbl_parametros_filtro_contabil = gerar_caminho_arquivo(etapa=4, subetapa=3, tipo_arquivo="tbl", nome="parametros_filtro_contabil")
caminho_tbl_regras_filtro_contabil = gerar_caminho_arquivo(etapa=4, subetapa=3, tipo_arquivo="tbl", nome="regras_filtro_contabil")
caminho_auditoria_contexto_filtro_contabil = gerar_caminho_arquivo(etapa=4, subetapa=3, tipo_arquivo="auditoria", nome="contexto_filtro_contabil")

print(f"Entrada - calendário contábil operacional : {caminho_base_calendario_contabil_operacional}")
print(f"Saída   - parâmetros do filtro contábil   : {caminho_tbl_parametros_filtro_contabil}")
print(f"Saída   - regras do filtro contábil       : {caminho_tbl_regras_filtro_contabil}")
print(f"Saída   - auditoria de contexto           : {caminho_auditoria_contexto_filtro_contabil}")
print("OK")

# ============================================================
# 3) Carga da base de entrada
# ============================================================

print("\n[3/10] Carga da base de entrada...")

df_calendario_contabil_operacional = pd.read_parquet(caminho_base_calendario_contabil_operacional)

print(f"Calendário contábil operacional: {df_calendario_contabil_operacional.shape[0]:,} linhas x {df_calendario_contabil_operacional.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros metodológicos do filtro contábil
# ============================================================

print("\n[4/10] Parâmetros metodológicos do filtro contábil...")

RECEITA_LIQUIDA_MINIMA = 0.00
PATRIMONIO_LIQUIDO_MINIMO = 0.00
MARGEM_LIQUIDA_MINIMA = 0.02
MARGEM_EBIT_MINIMA = 0.05
ROE_MINIMO = 0.015
LIQUIDEZ_CORRENTE_MINIMA = 1.00
LIQUIDEZ_GERAL_MINIMA = 1.00
LIQUIDEZ_IMEDIATA_MINIMA = 0.05

TRATAMENTO_VALOR_AUSENTE = "reprova"
UNIDADE_CONTABIL_MONETARIA = "brl"
UNIDADE_INDICADORES_LIQUIDEZ = "indice"
UNIDADE_RENTABILIDADE = "proporcao"

VARIAVEIS_UTILIZADAS = [
    "receita_liquida",
    "patrimonio_liquido",
    "margem_liquida",
    "margem_ebit",
    "roe",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

print(f"Receita líquida mínima          : {RECEITA_LIQUIDA_MINIMA:.2f}")
print(f"Patrimônio líquido mínimo       : {PATRIMONIO_LIQUIDO_MINIMO:.2f}")
print(f"Margem líquida mínima           : {MARGEM_LIQUIDA_MINIMA:.2%}")
print(f"Margem EBIT mínima              : {MARGEM_EBIT_MINIMA:.2%}")
print(f"ROE mínimo                      : {ROE_MINIMO:.2%}")
print(f"Liquidez corrente mínima        : {LIQUIDEZ_CORRENTE_MINIMA:.2f}")
print(f"Liquidez geral mínima           : {LIQUIDEZ_GERAL_MINIMA:.2f}")
print(f"Liquidez imediata mínima        : {LIQUIDEZ_IMEDIATA_MINIMA:.2f}")
print(f"Tratamento de valor ausente     : {TRATAMENTO_VALOR_AUSENTE}")
print("OK")

# ============================================================
# 5) Funções auxiliares da subetapa
# ============================================================

print("\n[5/10] Funções auxiliares da subetapa...")

COLUNAS_OBRIGATORIAS_CONTABIL = [
    "issuer_code",
    "periodo",
    "data_inicio_periodo",
    "data_fim_periodo",
    "data_inicio_vigencia_operacional",
    "data_fim_vigencia_operacional",
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
    "flag_vigencia_valida",
]

VARIAVEIS_FILTRO_CONTABIL = [
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def registrar_valor_misto(chave, valor, unidade, nome_chave):
    """
    Estrutura valores mistos em colunas homogêneas para escrita segura em parquet.
    """
    if isinstance(valor, (int, float)) and not pd.isna(valor):
        return {
            nome_chave: chave,
            "valor_numerico": float(valor),
            "valor_texto": pd.NA,
            "unidade": unidade,
        }
    return {
        nome_chave: chave,
        "valor_numerico": pd.NA,
        "valor_texto": str(valor) if pd.notna(valor) else pd.NA,
        "unidade": unidade,
    }

def calcular_indicadores_relativos(df_base):
    """
    Calcula indicadores relativos utilizados no filtro contábil.
    """
    df_base = df_base.copy()

    receita_positiva = df_base["receita_liquida"].gt(0)
    patrimonio_positivo = df_base["patrimonio_liquido"].gt(0)

    df_base["margem_liquida"] = pd.NA
    df_base["margem_ebit"] = pd.NA
    df_base["roe"] = pd.NA

    df_base.loc[receita_positiva, "margem_liquida"] = (
        df_base.loc[receita_positiva, "lucro_liquido"] / df_base.loc[receita_positiva, "receita_liquida"]
    )
    df_base.loc[receita_positiva, "margem_ebit"] = (
        df_base.loc[receita_positiva, "ebit"] / df_base.loc[receita_positiva, "receita_liquida"]
    )
    df_base.loc[patrimonio_positivo, "roe"] = (
        df_base.loc[patrimonio_positivo, "lucro_liquido"] / df_base.loc[patrimonio_positivo, "patrimonio_liquido"]
    )

    df_base["margem_liquida"] = pd.to_numeric(df_base["margem_liquida"], errors="coerce")
    df_base["margem_ebit"] = pd.to_numeric(df_base["margem_ebit"], errors="coerce")
    df_base["roe"] = pd.to_numeric(df_base["roe"], errors="coerce")

    return df_base

def construir_auditoria_contexto(df_base):
    """
    Consolida métricas diagnósticas da base contábil operacional para contextualizar o filtro.
    """
    registros = [
        registrar_valor_misto("n_linhas_calendario_contabil_operacional", int(len(df_base)), "linhas", "metrica"),
        registrar_valor_misto("n_issuer_code_calendario_contabil_operacional", int(df_base["issuer_code"].nunique(dropna=True)), "empresas", "metrica"),
        registrar_valor_misto("n_periodos_calendario_contabil_operacional", int(df_base["periodo"].nunique(dropna=True)), "periodos", "metrica"),
        registrar_valor_misto("n_linhas_vigencia_valida", int(df_base["flag_vigencia_valida"].sum()), "linhas", "metrica"),
        registrar_valor_misto("data_inicio_vigencia_min", str(pd.to_datetime(df_base["data_inicio_vigencia_operacional"], errors="coerce").min().date()), "data", "metrica"),
        registrar_valor_misto("data_fim_vigencia_max", str(pd.to_datetime(df_base["data_fim_vigencia_operacional"], errors="coerce").max().date()), "data", "metrica"),
    ]

    for variavel in VARIAVEIS_FILTRO_CONTABIL:
        serie = pd.to_numeric(df_base[variavel], errors="coerce")
        registros.append(registrar_valor_misto(f"n_missing_{variavel}", int(serie.isna().sum()), "linhas", "metrica"))
        registros.append(registrar_valor_misto(f"mediana_{variavel}", float(serie.median()) if serie.notna().any() else pd.NA, "valor", "metrica"))

    for variavel in ["margem_liquida", "margem_ebit", "roe"]:
        serie = pd.to_numeric(df_base[variavel], errors="coerce")
        registros.append(registrar_valor_misto(f"n_missing_{variavel}", int(serie.isna().sum()), "linhas", "metrica"))
        registros.append(registrar_valor_misto(f"mediana_{variavel}", float(serie.median()) if serie.notna().any() else pd.NA, "proporcao", "metrica"))

    return pd.DataFrame(registros)

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 6) Validação da base, cálculo dos indicadores e contexto diagnóstico
# ============================================================

print("\n[6/10] Validação da base, cálculo dos indicadores e contexto diagnóstico...")

validar_colunas_obrigatorias(df_calendario_contabil_operacional, COLUNAS_OBRIGATORIAS_CONTABIL)

for coluna in [
    "data_inicio_periodo",
    "data_fim_periodo",
    "data_inicio_vigencia_operacional",
    "data_fim_vigencia_operacional",
]:
    df_calendario_contabil_operacional[coluna] = pd.to_datetime(df_calendario_contabil_operacional[coluna], errors="coerce")

for coluna in VARIAVEIS_FILTRO_CONTABIL:
    df_calendario_contabil_operacional[coluna] = pd.to_numeric(df_calendario_contabil_operacional[coluna], errors="coerce")

df_calendario_contabil_operacional = calcular_indicadores_relativos(df_calendario_contabil_operacional)
df_auditoria_contexto_filtro_contabil = construir_auditoria_contexto(df_calendario_contabil_operacional)

print("Contexto diagnóstico consolidado com sucesso.")
print("OK")

# ============================================================
# 7) Construção das tabelas formais de parâmetros e regras
# ============================================================

print("\n[7/10] Construção das tabelas formais de parâmetros e regras...")

df_parametros_filtro_contabil = pd.DataFrame(
    [
        registrar_valor_misto(
            "receita_liquida_minima",
            RECEITA_LIQUIDA_MINIMA,
            UNIDADE_CONTABIL_MONETARIA,
            "parametro",
        ),
        registrar_valor_misto(
            "patrimonio_liquido_minimo",
            PATRIMONIO_LIQUIDO_MINIMO,
            UNIDADE_CONTABIL_MONETARIA,
            "parametro",
        ),
        registrar_valor_misto(
            "margem_liquida_minima",
            MARGEM_LIQUIDA_MINIMA,
            UNIDADE_RENTABILIDADE,
            "parametro",
        ),
        registrar_valor_misto(
            "margem_ebit_minima",
            MARGEM_EBIT_MINIMA,
            UNIDADE_RENTABILIDADE,
            "parametro",
        ),
        registrar_valor_misto(
            "roe_minimo",
            ROE_MINIMO,
            UNIDADE_RENTABILIDADE,
            "parametro",
        ),
        registrar_valor_misto(
            "liquidez_corrente_minima",
            LIQUIDEZ_CORRENTE_MINIMA,
            UNIDADE_INDICADORES_LIQUIDEZ,
            "parametro",
        ),
        registrar_valor_misto(
            "liquidez_geral_minima",
            LIQUIDEZ_GERAL_MINIMA,
            UNIDADE_INDICADORES_LIQUIDEZ,
            "parametro",
        ),
        registrar_valor_misto(
            "liquidez_imediata_minima",
            LIQUIDEZ_IMEDIATA_MINIMA,
            UNIDADE_INDICADORES_LIQUIDEZ,
            "parametro",
        ),
        registrar_valor_misto(
            "tratamento_valor_ausente",
            TRATAMENTO_VALOR_AUSENTE,
            "regra_logica",
            "parametro",
        ),
        registrar_valor_misto(
            "variaveis_utilizadas",
            " | ".join(VARIAVEIS_UTILIZADAS),
            "lista_variaveis",
            "parametro",
        ),
        registrar_valor_misto(
            "uso_ebitda",
            "nao_utilizar",
            "regra_metodologica",
            "parametro",
        ),
        registrar_valor_misto(
            "uso_divida_como_filtro_principal",
            "nao_utilizar",
            "regra_metodologica",
            "parametro",
        ),
    ]
)

df_regras_filtro_contabil = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "C1",
            "variavel": "receita_liquida",
            "operador": ">",
            "limiar": float(RECEITA_LIQUIDA_MINIMA),
            "unidade": UNIDADE_CONTABIL_MONETARIA,
            "descricao_operacional": "A empresa deve apresentar receita líquida estritamente positiva no período contábil vigente.",
            "justificativa_tecnica": "Garante atividade operacional com geração efetiva de receita no período analisado."
        },
        {
            "ordem_regra": 2,
            "regra_id": "C2",
            "variavel": "patrimonio_liquido",
            "operador": ">",
            "limiar": float(PATRIMONIO_LIQUIDO_MINIMO),
            "unidade": UNIDADE_CONTABIL_MONETARIA,
            "descricao_operacional": "A empresa deve apresentar patrimônio líquido estritamente positivo no período contábil vigente.",
            "justificativa_tecnica": "Evita empresas em situação patrimonial negativa, reforçando um critério mínimo de solvência contábil."
        },
        {
            "ordem_regra": 3,
            "regra_id": "C3",
            "variavel": "margem_liquida",
            "operador": ">=",
            "limiar": float(MARGEM_LIQUIDA_MINIMA),
            "unidade": UNIDADE_RENTABILIDADE,
            "descricao_operacional": "A empresa deve apresentar margem líquida igual ou superior a 2,00% no período contábil vigente.",
            "justificativa_tecnica": "Evita aprovar empresas com lucro líquido apenas marginalmente positivo e melhora a qualidade econômica do filtro."
        },
        {
            "ordem_regra": 4,
            "regra_id": "C4",
            "variavel": "margem_ebit",
            "operador": ">=",
            "limiar": float(MARGEM_EBIT_MINIMA),
            "unidade": UNIDADE_RENTABILIDADE,
            "descricao_operacional": "A empresa deve apresentar margem EBIT igual ou superior a 5,00% no período contábil vigente.",
            "justificativa_tecnica": "Exige geração operacional minimamente robusta, substituindo um simples EBIT positivo por um critério relativo ao porte da operação."
        },
        {
            "ordem_regra": 5,
            "regra_id": "C5",
            "variavel": "roe",
            "operador": ">=",
            "limiar": float(ROE_MINIMO),
            "unidade": UNIDADE_RENTABILIDADE,
            "descricao_operacional": "A empresa deve apresentar ROE igual ou superior a 1,50% no período contábil vigente.",
            "justificativa_tecnica": "Impõe retorno mínimo sobre o patrimônio líquido, evitando aprovar empresas com rentabilidade excessivamente baixa."
        },
        {
            "ordem_regra": 6,
            "regra_id": "C6",
            "variavel": "liquidez_corrente",
            "operador": ">=",
            "limiar": float(LIQUIDEZ_CORRENTE_MINIMA),
            "unidade": UNIDADE_INDICADORES_LIQUIDEZ,
            "descricao_operacional": "A empresa deve apresentar liquidez corrente igual ou superior a 1,00 no período contábil vigente.",
            "justificativa_tecnica": "Impõe um critério mínimo clássico de cobertura do passivo circulante por ativos circulantes."
        },
        {
            "ordem_regra": 7,
            "regra_id": "C7",
            "variavel": "liquidez_geral",
            "operador": ">=",
            "limiar": float(LIQUIDEZ_GERAL_MINIMA),
            "unidade": UNIDADE_INDICADORES_LIQUIDEZ,
            "descricao_operacional": "A empresa deve apresentar liquidez geral igual ou superior a 1,00 no período contábil vigente.",
            "justificativa_tecnica": "Complementa a liquidez corrente com uma visão mais ampla da estrutura de solvência."
        },
        {
            "ordem_regra": 8,
            "regra_id": "C8",
            "variavel": "liquidez_imediata",
            "operador": ">=",
            "limiar": float(LIQUIDEZ_IMEDIATA_MINIMA),
            "unidade": UNIDADE_INDICADORES_LIQUIDEZ,
            "descricao_operacional": "A empresa deve apresentar liquidez imediata igual ou superior a 0,05 no período contábil vigente.",
            "justificativa_tecnica": "Estabelece um piso mínimo de caixa e equivalentes sem transformar a dívida em filtro principal."
        },
        {
            "ordem_regra": 9,
            "regra_id": "C9",
            "variavel": "tratamento_valor_ausente",
            "operador": "=",
            "limiar": float("nan"),
            "unidade": "regra_logica",
            "descricao_operacional": "Qualquer valor ausente em variável do filtro contábil reprova a empresa na janela contábil correspondente.",
            "justificativa_tecnica": "Evita assumir aptidão contábil quando a informação essencial do período não está disponível."
        },
    ]
)

print("Tabelas formais de parâmetros e regras construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_parametros_filtro_contabil, caminho_tbl_parametros_filtro_contabil, index=False)
salvar_dataframe(df_regras_filtro_contabil, caminho_tbl_regras_filtro_contabil, index=False)
salvar_dataframe(df_auditoria_contexto_filtro_contabil, caminho_auditoria_contexto_filtro_contabil, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_parametros_filtro_contabil = df_parametros_filtro_contabil.copy()
df_amostra_regras_filtro_contabil = df_regras_filtro_contabil.copy()
df_amostra_contexto_filtro_contabil = df_auditoria_contexto_filtro_contabil.copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nParâmetros formais do filtro contábil:")
print(df_amostra_parametros_filtro_contabil.to_string(index=False))

print("\nRegras operacionais do filtro contábil:")
print(df_amostra_regras_filtro_contabil.to_string(index=False))

print("\nContexto diagnóstico utilizado para definição do filtro contábil:")
print(df_amostra_contexto_filtro_contabil.to_string(index=False))

print("\nArquivos salvos na subetapa 4.3:")
print(f"- {caminho_tbl_parametros_filtro_contabil}")
print(f"- {caminho_tbl_regras_filtro_contabil}")
print(f"- {caminho_auditoria_contexto_filtro_contabil}")

print("\nETAPA 4.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 4.3 - DEFINIÇÃO DOS CRITÉRIOS DO FILTRO CONTÁBIL

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - calendário contábil operacional : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_2_base_calendario_contabil_operacional.parquet
Saída   - parâmetros do filtro contábil   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_3_tbl_parametros_filtro_contabil.parquet
Saída   - regras do filtro contábil       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_3_tbl_regras_filtro_contabil.parquet
Saída   - auditoria de contexto           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4

## Etapa 4.4) Base Contábil de Elegibilidade por Janela

In [18]:
%%time
# ============================================================
# Etapa 4.4) Base Contábil de Elegibilidade por Janela
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 4.4 - BASE CONTÁBIL DE ELEGIBILIDADE POR JANELA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_calendario_contabil_operacional = gerar_caminho_arquivo(etapa=4, subetapa=2, tipo_arquivo="base", nome="calendario_contabil_operacional")
caminho_tbl_parametros_vigencia_contabil = gerar_caminho_arquivo(etapa=4, subetapa=2, tipo_arquivo="tbl", nome="parametros_vigencia_contabil")
caminho_tbl_parametros_filtro_contabil = gerar_caminho_arquivo(etapa=4, subetapa=3, tipo_arquivo="tbl", nome="parametros_filtro_contabil")

caminho_base_elegibilidade_contabil_janela = gerar_caminho_arquivo(etapa=4, subetapa=4, tipo_arquivo="base", nome="elegibilidade_contabil_janela")
caminho_tbl_resumo_elegibilidade_contabil_janela = gerar_caminho_arquivo(etapa=4, subetapa=4, tipo_arquivo="tbl", nome="resumo_elegibilidade_contabil_janela")
caminho_tbl_cobertura_janelas_contabeis = gerar_caminho_arquivo(etapa=4, subetapa=4, tipo_arquivo="tbl", nome="cobertura_janelas_contabeis")
caminho_tbl_auditoria_regras_filtro_contabil = gerar_caminho_arquivo(etapa=4, subetapa=4, tipo_arquivo="tbl", nome="auditoria_regras_filtro_contabil")
caminho_tbl_motivos_reprovacao_contabil = gerar_caminho_arquivo(etapa=4, subetapa=4, tipo_arquivo="tbl", nome="motivos_reprovacao_contabil")

print(f"Entrada - calendário contábil operacional : {caminho_base_calendario_contabil_operacional}")
print(f"Entrada - parâmetros de vigência          : {caminho_tbl_parametros_vigencia_contabil}")
print(f"Entrada - parâmetros do filtro contábil   : {caminho_tbl_parametros_filtro_contabil}")
print(f"Saída   - base de elegibilidade contábil  : {caminho_base_elegibilidade_contabil_janela}")
print(f"Saída   - resumo                          : {caminho_tbl_resumo_elegibilidade_contabil_janela}")
print(f"Saída   - cobertura por janela            : {caminho_tbl_cobertura_janelas_contabeis}")
print(f"Saída   - auditoria das regras            : {caminho_tbl_auditoria_regras_filtro_contabil}")
print(f"Saída   - motivos de reprovação           : {caminho_tbl_motivos_reprovacao_contabil}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_calendario_contabil_operacional = pd.read_parquet(caminho_base_calendario_contabil_operacional)
df_parametros_vigencia_contabil = pd.read_parquet(caminho_tbl_parametros_vigencia_contabil)
df_parametros_filtro_contabil = pd.read_parquet(caminho_tbl_parametros_filtro_contabil)

print(f"Calendário contábil operacional : {df_calendario_contabil_operacional.shape[0]:,} linhas x {df_calendario_contabil_operacional.shape[1]} colunas")
print(f"Parâmetros de vigência          : {df_parametros_vigencia_contabil.shape[0]:,} linhas x {df_parametros_vigencia_contabil.shape[1]} colunas")
print(f"Parâmetros do filtro contábil   : {df_parametros_filtro_contabil.shape[0]:,} linhas x {df_parametros_filtro_contabil.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

COLUNAS_OBRIGATORIAS = [
    "issuer_code",
    "periodo",
    "ticker_exemplo",
    "n_tickers_periodo",
    "data_inicio_periodo",
    "data_fim_periodo",
    "data_inicio_vigencia_operacional",
    "data_fim_vigencia_operacional",
    "flag_vigencia_dentro_horizonte_projeto",
    "flag_vigencia_valida",
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

COLUNAS_NUMERICAS_BASE = [
    "lucro_liquido",
    "patrimonio_liquido",
    "receita_liquida",
    "ebit",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

VIGENCIA_CONTABIL_MAXIMA_DIAS = 365

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def obter_valor_parametro(df_parametros, parametro):
    """
    Recupera o valor efetivo de um parâmetro salvo em tabela de parâmetros.
    """
    df_parametro = df_parametros.loc[df_parametros["parametro"].eq(parametro)].copy()

    if df_parametro.empty:
        raise ValueError(f"Parâmetro não encontrado: {parametro}")

    valor_numerico = df_parametro["valor_numerico"].iloc[0]
    valor_texto = df_parametro["valor_texto"].iloc[0]

    if pd.notna(valor_numerico):
        return float(valor_numerico)

    if pd.notna(valor_texto):
        return str(valor_texto)

    return pd.NA

def calcular_indicadores_relativos(df_base):
    """
    Calcula os indicadores relativos utilizados no filtro contábil.
    """
    df_base = df_base.copy()

    receita_positiva = df_base["receita_liquida"].gt(0)
    patrimonio_positivo = df_base["patrimonio_liquido"].gt(0)

    df_base["margem_liquida"] = pd.NA
    df_base["margem_ebit"] = pd.NA
    df_base["roe"] = pd.NA

    df_base.loc[receita_positiva, "margem_liquida"] = (
        df_base.loc[receita_positiva, "lucro_liquido"] / df_base.loc[receita_positiva, "receita_liquida"]
    )
    df_base.loc[receita_positiva, "margem_ebit"] = (
        df_base.loc[receita_positiva, "ebit"] / df_base.loc[receita_positiva, "receita_liquida"]
    )
    df_base.loc[patrimonio_positivo, "roe"] = (
        df_base.loc[patrimonio_positivo, "lucro_liquido"] / df_base.loc[patrimonio_positivo, "patrimonio_liquido"]
    )

    df_base["margem_liquida"] = pd.to_numeric(df_base["margem_liquida"], errors="coerce")
    df_base["margem_ebit"] = pd.to_numeric(df_base["margem_ebit"], errors="coerce")
    df_base["roe"] = pd.to_numeric(df_base["roe"], errors="coerce")

    return df_base

print("Funções auxiliares declaradas com sucesso.")
print(f"Vigência contábil máxima definida          : {VIGENCIA_CONTABIL_MAXIMA_DIAS} dias após data_fim_periodo")
print("OK")

# ============================================================
# 5) Preparação da base e leitura dos parâmetros vigentes
# ============================================================

print("\n[5/10] Preparação da base e leitura dos parâmetros vigentes...")

validar_colunas_obrigatorias(df_calendario_contabil_operacional, COLUNAS_OBRIGATORIAS)
validar_colunas_obrigatorias(df_parametros_vigencia_contabil, ["parametro", "valor_numerico", "valor_texto", "unidade"])
validar_colunas_obrigatorias(df_parametros_filtro_contabil, ["parametro", "valor_numerico", "valor_texto", "unidade"])

df_elegibilidade_contabil_janela = df_calendario_contabil_operacional.copy()

df_elegibilidade_contabil_janela["issuer_code"] = df_elegibilidade_contabil_janela["issuer_code"].astype("string").str.strip()
df_elegibilidade_contabil_janela["ticker_exemplo"] = df_elegibilidade_contabil_janela["ticker_exemplo"].astype("string").str.strip().str.upper()
df_elegibilidade_contabil_janela["periodo"] = df_elegibilidade_contabil_janela["periodo"].astype("string").str.strip().str.upper()

for coluna in [
    "data_inicio_periodo",
    "data_fim_periodo",
    "data_inicio_vigencia_operacional",
    "data_fim_vigencia_operacional",
]:
    df_elegibilidade_contabil_janela[coluna] = pd.to_datetime(df_elegibilidade_contabil_janela[coluna], errors="coerce")

for coluna in COLUNAS_NUMERICAS_BASE:
    df_elegibilidade_contabil_janela[coluna] = pd.to_numeric(df_elegibilidade_contabil_janela[coluna], errors="coerce")

df_elegibilidade_contabil_janela["n_tickers_periodo"] = pd.to_numeric(df_elegibilidade_contabil_janela["n_tickers_periodo"], errors="coerce")

RECEITA_LIQUIDA_MINIMA = float(obter_valor_parametro(df_parametros_filtro_contabil, "receita_liquida_minima"))
PATRIMONIO_LIQUIDO_MINIMO = float(obter_valor_parametro(df_parametros_filtro_contabil, "patrimonio_liquido_minimo"))
MARGEM_LIQUIDA_MINIMA = float(obter_valor_parametro(df_parametros_filtro_contabil, "margem_liquida_minima"))
MARGEM_EBIT_MINIMA = float(obter_valor_parametro(df_parametros_filtro_contabil, "margem_ebit_minima"))
ROE_MINIMO = float(obter_valor_parametro(df_parametros_filtro_contabil, "roe_minimo"))
LIQUIDEZ_CORRENTE_MINIMA = float(obter_valor_parametro(df_parametros_filtro_contabil, "liquidez_corrente_minima"))
LIQUIDEZ_GERAL_MINIMA = float(obter_valor_parametro(df_parametros_filtro_contabil, "liquidez_geral_minima"))
LIQUIDEZ_IMEDIATA_MINIMA = float(obter_valor_parametro(df_parametros_filtro_contabil, "liquidez_imediata_minima"))
TRATAMENTO_VALOR_AUSENTE = str(obter_valor_parametro(df_parametros_filtro_contabil, "tratamento_valor_ausente"))

DATA_INICIAL_PROJETO_OPERACIONAL = pd.to_datetime(
    obter_valor_parametro(df_parametros_vigencia_contabil, "data_inicial_projeto_operacional"),
    errors="coerce",
)
DATA_FINAL_PROJETO_OPERACIONAL = pd.to_datetime(
    obter_valor_parametro(df_parametros_vigencia_contabil, "data_final_projeto_operacional"),
    errors="coerce",
)

print(f"Receita líquida mínima vigente   : {RECEITA_LIQUIDA_MINIMA:.2f}")
print(f"Patrimônio líquido mínimo vigente: {PATRIMONIO_LIQUIDO_MINIMO:.2f}")
print(f"Margem líquida mínima vigente    : {MARGEM_LIQUIDA_MINIMA:.2%}")
print(f"Margem EBIT mínima vigente       : {MARGEM_EBIT_MINIMA:.2%}")
print(f"ROE mínimo vigente               : {ROE_MINIMO:.2%}")
print(f"Data inicial do projeto          : {DATA_INICIAL_PROJETO_OPERACIONAL}")
print(f"Data final do projeto            : {DATA_FINAL_PROJETO_OPERACIONAL}")
print(f"Tratamento de ausentes           : {TRATAMENTO_VALOR_AUSENTE}")
print("OK")

# ============================================================
# 6) Aplicação dos critérios do filtro contábil por janela
# ============================================================

print("\n[6/10] Aplicação dos critérios do filtro contábil por janela...")

df_elegibilidade_contabil_janela = calcular_indicadores_relativos(df_elegibilidade_contabil_janela)

df_elegibilidade_contabil_janela["data_fim_vigencia_limite_tecnico"] = (
    df_elegibilidade_contabil_janela["data_fim_periodo"] + pd.Timedelta(days=VIGENCIA_CONTABIL_MAXIMA_DIAS)
)

df_elegibilidade_contabil_janela["data_fim_vigencia_operacional_ajustada"] = df_elegibilidade_contabil_janela["data_fim_vigencia_operacional"]
df_elegibilidade_contabil_janela.loc[
    pd.notna(df_elegibilidade_contabil_janela["data_fim_vigencia_limite_tecnico"]),
    "data_fim_vigencia_operacional_ajustada"
] = df_elegibilidade_contabil_janela.loc[
    pd.notna(df_elegibilidade_contabil_janela["data_fim_vigencia_limite_tecnico"]),
    ["data_fim_vigencia_operacional", "data_fim_vigencia_limite_tecnico"]
].min(axis=1)

df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_valida"]
    & df_elegibilidade_contabil_janela["data_inicio_vigencia_operacional"].le(DATA_FINAL_PROJETO_OPERACIONAL)
    & df_elegibilidade_contabil_janela["data_fim_vigencia_operacional_ajustada"].ge(DATA_INICIAL_PROJETO_OPERACIONAL)
    & df_elegibilidade_contabil_janela["data_inicio_vigencia_operacional"].le(df_elegibilidade_contabil_janela["data_fim_vigencia_operacional_ajustada"])
)

df_elegibilidade_contabil_janela["data_inicio_vigencia_efetiva_projeto"] = df_elegibilidade_contabil_janela["data_inicio_vigencia_operacional"]
df_elegibilidade_contabil_janela["data_fim_vigencia_efetiva_projeto"] = df_elegibilidade_contabil_janela["data_fim_vigencia_operacional_ajustada"]

df_elegibilidade_contabil_janela.loc[
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"],
    "data_inicio_vigencia_efetiva_projeto"
] = df_elegibilidade_contabil_janela.loc[
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"],
    "data_inicio_vigencia_operacional"
].clip(lower=DATA_INICIAL_PROJETO_OPERACIONAL)

df_elegibilidade_contabil_janela.loc[
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"],
    "data_fim_vigencia_efetiva_projeto"
] = df_elegibilidade_contabil_janela.loc[
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"],
    "data_fim_vigencia_operacional_ajustada"
].clip(upper=DATA_FINAL_PROJETO_OPERACIONAL)

df_elegibilidade_contabil_janela.loc[
    ~df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"],
    ["data_inicio_vigencia_efetiva_projeto", "data_fim_vigencia_efetiva_projeto"]
] = pd.NaT

df_elegibilidade_contabil_janela["duracao_vigencia_efetiva_projeto_dias"] = (
    df_elegibilidade_contabil_janela["data_fim_vigencia_efetiva_projeto"]
    - df_elegibilidade_contabil_janela["data_inicio_vigencia_efetiva_projeto"]
).dt.days.add(1)

colunas_essenciais_filtro = [
    "receita_liquida",
    "patrimonio_liquido",
    "margem_liquida",
    "margem_ebit",
    "roe",
    "liquidez_corrente",
    "liquidez_geral",
    "liquidez_imediata",
]

df_elegibilidade_contabil_janela["n_variaveis_ausentes_filtro_contabil"] = (
    df_elegibilidade_contabil_janela[colunas_essenciais_filtro].isna().sum(axis=1)
)

df_elegibilidade_contabil_janela["flag_regra_c1_receita_liquida"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["receita_liquida"].gt(RECEITA_LIQUIDA_MINIMA)
)

df_elegibilidade_contabil_janela["flag_regra_c2_patrimonio_liquido"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["patrimonio_liquido"].gt(PATRIMONIO_LIQUIDO_MINIMO)
)

df_elegibilidade_contabil_janela["flag_regra_c3_margem_liquida"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["margem_liquida"].ge(MARGEM_LIQUIDA_MINIMA)
)

df_elegibilidade_contabil_janela["flag_regra_c4_margem_ebit"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["margem_ebit"].ge(MARGEM_EBIT_MINIMA)
)

df_elegibilidade_contabil_janela["flag_regra_c5_roe"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["roe"].ge(ROE_MINIMO)
)

df_elegibilidade_contabil_janela["flag_regra_c6_liquidez_corrente"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["liquidez_corrente"].ge(LIQUIDEZ_CORRENTE_MINIMA)
)

df_elegibilidade_contabil_janela["flag_regra_c7_liquidez_geral"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["liquidez_geral"].ge(LIQUIDEZ_GERAL_MINIMA)
)

df_elegibilidade_contabil_janela["flag_regra_c8_liquidez_imediata"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["liquidez_imediata"].ge(LIQUIDEZ_IMEDIATA_MINIMA)
)

if TRATAMENTO_VALOR_AUSENTE == "reprova":
    df_elegibilidade_contabil_janela["flag_regra_c9_sem_ausencia_variaveis"] = (
        df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
        & df_elegibilidade_contabil_janela["n_variaveis_ausentes_filtro_contabil"].eq(0)
    )
else:
    raise ValueError(f"Tratamento de valor ausente não suportado: {TRATAMENTO_VALOR_AUSENTE}")

df_elegibilidade_contabil_janela["flag_reprovacao_por_ausencia"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["n_variaveis_ausentes_filtro_contabil"].gt(0)
)

df_elegibilidade_contabil_janela["flag_reprovacao_por_rentabilidade"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & (
        ~df_elegibilidade_contabil_janela["flag_regra_c3_margem_liquida"]
        | ~df_elegibilidade_contabil_janela["flag_regra_c4_margem_ebit"]
        | ~df_elegibilidade_contabil_janela["flag_regra_c5_roe"]
    )
)

df_elegibilidade_contabil_janela["flag_reprovacao_por_liquidez"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & (
        ~df_elegibilidade_contabil_janela["flag_regra_c6_liquidez_corrente"]
        | ~df_elegibilidade_contabil_janela["flag_regra_c7_liquidez_geral"]
        | ~df_elegibilidade_contabil_janela["flag_regra_c8_liquidez_imediata"]
    )
)

df_elegibilidade_contabil_janela["flag_reprovacao_por_estrutura"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & (
        ~df_elegibilidade_contabil_janela["flag_regra_c1_receita_liquida"]
        | ~df_elegibilidade_contabil_janela["flag_regra_c2_patrimonio_liquido"]
    )
)

colunas_flags_regras = [
    "flag_regra_c1_receita_liquida",
    "flag_regra_c2_patrimonio_liquido",
    "flag_regra_c3_margem_liquida",
    "flag_regra_c4_margem_ebit",
    "flag_regra_c5_roe",
    "flag_regra_c6_liquidez_corrente",
    "flag_regra_c7_liquidez_geral",
    "flag_regra_c8_liquidez_imediata",
    "flag_regra_c9_sem_ausencia_variaveis",
]

df_elegibilidade_contabil_janela["n_regras_aprovadas_filtro_contabil"] = (
    df_elegibilidade_contabil_janela[colunas_flags_regras].sum(axis=1)
)

df_elegibilidade_contabil_janela["n_regras_reprovadas_filtro_contabil"] = (
    len(colunas_flags_regras) - df_elegibilidade_contabil_janela["n_regras_aprovadas_filtro_contabil"]
)

df_elegibilidade_contabil_janela["flag_apta_contabil_janela"] = (
    df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]
    & df_elegibilidade_contabil_janela["flag_regra_c1_receita_liquida"]
    & df_elegibilidade_contabil_janela["flag_regra_c2_patrimonio_liquido"]
    & df_elegibilidade_contabil_janela["flag_regra_c3_margem_liquida"]
    & df_elegibilidade_contabil_janela["flag_regra_c4_margem_ebit"]
    & df_elegibilidade_contabil_janela["flag_regra_c5_roe"]
    & df_elegibilidade_contabil_janela["flag_regra_c6_liquidez_corrente"]
    & df_elegibilidade_contabil_janela["flag_regra_c7_liquidez_geral"]
    & df_elegibilidade_contabil_janela["flag_regra_c8_liquidez_imediata"]
    & df_elegibilidade_contabil_janela["flag_regra_c9_sem_ausencia_variaveis"]
)

df_elegibilidade_contabil_janela = (
    df_elegibilidade_contabil_janela
    .sort_values(["data_inicio_vigencia_operacional", "issuer_code", "periodo"])
    .reset_index(drop=True)
)

print(f"Base contábil de elegibilidade por janela: {df_elegibilidade_contabil_janela.shape[0]:,} linhas x {df_elegibilidade_contabil_janela.shape[1]} colunas")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

n_linhas_total = int(len(df_elegibilidade_contabil_janela))
n_linhas_aplicaveis = int(df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"].sum())
n_linhas_aptas = int(df_elegibilidade_contabil_janela["flag_apta_contabil_janela"].sum())

df_resumo_elegibilidade_contabil_janela = pd.DataFrame(
    [
        {
            "base": "elegibilidade_contabil_janela",
            "n_linhas": n_linhas_total,
            "n_issuer_code": int(df_elegibilidade_contabil_janela["issuer_code"].nunique(dropna=True)),
            "n_periodos": int(df_elegibilidade_contabil_janela["periodo"].nunique(dropna=True)),
            "n_janelas_aplicaveis": n_linhas_aplicaveis,
            "n_janelas_aptas": n_linhas_aptas,
            "pct_janelas_aptas_sobre_aplicaveis": (n_linhas_aptas / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
            "data_inicio_vigencia_min": df_elegibilidade_contabil_janela["data_inicio_vigencia_operacional"].min(),
            "data_fim_vigencia_max": df_elegibilidade_contabil_janela["data_fim_vigencia_operacional"].max(),
            "data_fim_vigencia_ajustada_max": df_elegibilidade_contabil_janela["data_fim_vigencia_operacional_ajustada"].max(),
            "data_inicio_vigencia_efetiva_min": df_elegibilidade_contabil_janela["data_inicio_vigencia_efetiva_projeto"].min(),
            "data_fim_vigencia_efetiva_max": df_elegibilidade_contabil_janela["data_fim_vigencia_efetiva_projeto"].max(),
            "vigencia_contabil_maxima_dias": VIGENCIA_CONTABIL_MAXIMA_DIAS,
            "n_janelas_com_reprovacao_por_ausencia": int(df_elegibilidade_contabil_janela["flag_reprovacao_por_ausencia"].sum()),
            "n_janelas_com_reprovacao_por_rentabilidade": int(df_elegibilidade_contabil_janela["flag_reprovacao_por_rentabilidade"].sum()),
            "n_janelas_com_reprovacao_por_liquidez": int(df_elegibilidade_contabil_janela["flag_reprovacao_por_liquidez"].sum()),
        }
    ]
)

df_cobertura_janelas_contabeis = (
    df_elegibilidade_contabil_janela
    .loc[df_elegibilidade_contabil_janela["flag_vigencia_aplicavel"]]
    .groupby(
        [
            "periodo",
            "data_inicio_vigencia_operacional",
            "data_fim_vigencia_operacional",
            "data_fim_vigencia_operacional_ajustada",
            "data_inicio_vigencia_efetiva_projeto",
            "data_fim_vigencia_efetiva_projeto",
        ],
        as_index=False
    )
    .agg(
        n_empresas_janela=("issuer_code", "nunique"),
        n_empresas_aptas_janela=("flag_apta_contabil_janela", "sum"),
        n_empresas_reprovadas_ausencia=("flag_reprovacao_por_ausencia", "sum"),
        n_empresas_reprovadas_rentabilidade=("flag_reprovacao_por_rentabilidade", "sum"),
        n_empresas_reprovadas_liquidez=("flag_reprovacao_por_liquidez", "sum"),
        n_empresas_reprovadas_estrutura=("flag_reprovacao_por_estrutura", "sum"),
    )
    .sort_values(["data_inicio_vigencia_efetiva_projeto", "periodo"])
    .reset_index(drop=True)
)

df_cobertura_janelas_contabeis["pct_empresas_aptas_janela"] = (
    df_cobertura_janelas_contabeis["n_empresas_aptas_janela"] / df_cobertura_janelas_contabeis["n_empresas_janela"]
)

df_auditoria_regras_filtro_contabil = pd.DataFrame(
    [
        {
            "regra": "C1",
            "descricao": "receita líquida positiva",
            "n_janelas_aprovadas": int(df_elegibilidade_contabil_janela["flag_regra_c1_receita_liquida"].sum()),
            "pct_janelas_aprovadas_sobre_aplicaveis": float(df_elegibilidade_contabil_janela["flag_regra_c1_receita_liquida"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "regra": "C2",
            "descricao": "patrimônio líquido positivo",
            "n_janelas_aprovadas": int(df_elegibilidade_contabil_janela["flag_regra_c2_patrimonio_liquido"].sum()),
            "pct_janelas_aprovadas_sobre_aplicaveis": float(df_elegibilidade_contabil_janela["flag_regra_c2_patrimonio_liquido"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "regra": "C3",
            "descricao": "margem líquida mínima",
            "n_janelas_aprovadas": int(df_elegibilidade_contabil_janela["flag_regra_c3_margem_liquida"].sum()),
            "pct_janelas_aprovadas_sobre_aplicaveis": float(df_elegibilidade_contabil_janela["flag_regra_c3_margem_liquida"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "regra": "C4",
            "descricao": "margem EBIT mínima",
            "n_janelas_aprovadas": int(df_elegibilidade_contabil_janela["flag_regra_c4_margem_ebit"].sum()),
            "pct_janelas_aprovadas_sobre_aplicaveis": float(df_elegibilidade_contabil_janela["flag_regra_c4_margem_ebit"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "regra": "C5",
            "descricao": "ROE mínimo",
            "n_janelas_aprovadas": int(df_elegibilidade_contabil_janela["flag_regra_c5_roe"].sum()),
            "pct_janelas_aprovadas_sobre_aplicaveis": float(df_elegibilidade_contabil_janela["flag_regra_c5_roe"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "regra": "C6",
            "descricao": "liquidez corrente mínima",
            "n_janelas_aprovadas": int(df_elegibilidade_contabil_janela["flag_regra_c6_liquidez_corrente"].sum()),
            "pct_janelas_aprovadas_sobre_aplicaveis": float(df_elegibilidade_contabil_janela["flag_regra_c6_liquidez_corrente"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "regra": "C7",
            "descricao": "liquidez geral mínima",
            "n_janelas_aprovadas": int(df_elegibilidade_contabil_janela["flag_regra_c7_liquidez_geral"].sum()),
            "pct_janelas_aprovadas_sobre_aplicaveis": float(df_elegibilidade_contabil_janela["flag_regra_c7_liquidez_geral"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "regra": "C8",
            "descricao": "liquidez imediata mínima",
            "n_janelas_aprovadas": int(df_elegibilidade_contabil_janela["flag_regra_c8_liquidez_imediata"].sum()),
            "pct_janelas_aprovadas_sobre_aplicaveis": float(df_elegibilidade_contabil_janela["flag_regra_c8_liquidez_imediata"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "regra": "C9",
            "descricao": "ausência de valores faltantes",
            "n_janelas_aprovadas": int(df_elegibilidade_contabil_janela["flag_regra_c9_sem_ausencia_variaveis"].sum()),
            "pct_janelas_aprovadas_sobre_aplicaveis": float(df_elegibilidade_contabil_janela["flag_regra_c9_sem_ausencia_variaveis"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "regra": "C_FINAL",
            "descricao": "aptidão contábil final",
            "n_janelas_aprovadas": n_linhas_aptas,
            "pct_janelas_aprovadas_sobre_aplicaveis": float(n_linhas_aptas / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
    ]
)

df_motivos_reprovacao_contabil = pd.DataFrame(
    [
        {
            "motivo_reprovacao": "ausencia_variaveis_essenciais",
            "n_janelas": int(df_elegibilidade_contabil_janela["flag_reprovacao_por_ausencia"].sum()),
            "pct_sobre_janelas_aplicaveis": float(df_elegibilidade_contabil_janela["flag_reprovacao_por_ausencia"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "motivo_reprovacao": "estrutura_basica_receita_ou_patrimonio",
            "n_janelas": int(df_elegibilidade_contabil_janela["flag_reprovacao_por_estrutura"].sum()),
            "pct_sobre_janelas_aplicaveis": float(df_elegibilidade_contabil_janela["flag_reprovacao_por_estrutura"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "motivo_reprovacao": "rentabilidade_relativa_insuficiente",
            "n_janelas": int(df_elegibilidade_contabil_janela["flag_reprovacao_por_rentabilidade"].sum()),
            "pct_sobre_janelas_aplicaveis": float(df_elegibilidade_contabil_janela["flag_reprovacao_por_rentabilidade"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
        {
            "motivo_reprovacao": "liquidez_contabil_insuficiente",
            "n_janelas": int(df_elegibilidade_contabil_janela["flag_reprovacao_por_liquidez"].sum()),
            "pct_sobre_janelas_aplicaveis": float(df_elegibilidade_contabil_janela["flag_reprovacao_por_liquidez"].sum() / n_linhas_aplicaveis) if n_linhas_aplicaveis > 0 else pd.NA,
        },
    ]
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_elegibilidade_contabil_janela, caminho_base_elegibilidade_contabil_janela, index=False)
salvar_dataframe(df_resumo_elegibilidade_contabil_janela, caminho_tbl_resumo_elegibilidade_contabil_janela, index=False)
salvar_dataframe(df_cobertura_janelas_contabeis, caminho_tbl_cobertura_janelas_contabeis, index=False)
salvar_dataframe(df_auditoria_regras_filtro_contabil, caminho_tbl_auditoria_regras_filtro_contabil, index=False)
salvar_dataframe(df_motivos_reprovacao_contabil, caminho_tbl_motivos_reprovacao_contabil, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_base_elegibilidade_contabil = df_elegibilidade_contabil_janela.head(15).copy()
df_amostra_janelas_aptas = (
    df_elegibilidade_contabil_janela
    .loc[df_elegibilidade_contabil_janela["flag_apta_contabil_janela"]]
    .head(15)
    .copy()
)
df_amostra_cobertura_janelas = df_cobertura_janelas_contabeis.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base contábil de elegibilidade por janela:")
print(df_resumo_elegibilidade_contabil_janela.to_string(index=False))

print("\nAuditoria das regras do filtro contábil:")
print(df_auditoria_regras_filtro_contabil.to_string(index=False))

print("\nMotivos agregados de reprovação contábil:")
print(df_motivos_reprovacao_contabil.to_string(index=False))

print("\nAmostra da base contábil de elegibilidade por janela:")
print(df_amostra_base_elegibilidade_contabil.to_string(index=False))

print("\nAmostra de janelas aptas contabilmente:")
print(df_amostra_janelas_aptas.to_string(index=False))

print("\nCobertura das janelas contábeis - amostra:")
print(df_amostra_cobertura_janelas.to_string(index=False))

print("\nArquivos salvos na subetapa 4.4:")
print(f"- {caminho_base_elegibilidade_contabil_janela}")
print(f"- {caminho_tbl_resumo_elegibilidade_contabil_janela}")
print(f"- {caminho_tbl_cobertura_janelas_contabeis}")
print(f"- {caminho_tbl_auditoria_regras_filtro_contabil}")
print(f"- {caminho_tbl_motivos_reprovacao_contabil}")

print("\nETAPA 4.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 4.4 - BASE CONTÁBIL DE ELEGIBILIDADE POR JANELA

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - calendário contábil operacional : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_2_base_calendario_contabil_operacional.parquet
Entrada - parâmetros de vigência          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_2_tbl_parametros_vigencia_contabil.parquet
Entrada - parâmetros do filtro contábil   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_3_tbl_parametros_filtro_contabil.parquet
Saída   - base de elegibilidade contábil  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\et

## Etapa 4.5) Base Final de Tickers Aptos por Janela Contábil

In [19]:
%%time
# ============================================================
# Etapa 4.5) Base Final de Tickers Aptos por Janela Contábil
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 4.5 - BASE FINAL DE TICKERS APTOS POR JANELA CONTÁBIL")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_elegibilidade_operacional_data = gerar_caminho_arquivo(etapa=3, subetapa=4, tipo_arquivo="base", nome="elegibilidade_operacional_data")
caminho_base_elegibilidade_contabil_janela = gerar_caminho_arquivo(etapa=4, subetapa=4, tipo_arquivo="base", nome="elegibilidade_contabil_janela")

caminho_base_tickers_aptos_janela_contabil = gerar_caminho_arquivo(etapa=4, subetapa=5, tipo_arquivo="base", nome="tickers_aptos_janela_contabil")
caminho_tbl_resumo_tickers_aptos_janela_contabil = gerar_caminho_arquivo(etapa=4, subetapa=5, tipo_arquivo="tbl", nome="resumo_tickers_aptos_janela_contabil")
caminho_tbl_cobertura_diaria_tickers_aptos_janela_contabil = gerar_caminho_arquivo(etapa=4, subetapa=5, tipo_arquivo="tbl", nome="cobertura_diaria_tickers_aptos_janela_contabil")
caminho_tbl_auditoria_cruzamento_operacional_contabil = gerar_caminho_arquivo(etapa=4, subetapa=5, tipo_arquivo="tbl", nome="auditoria_cruzamento_operacional_contabil")
caminho_tbl_janelas_contabeis_sem_match_operacional = gerar_caminho_arquivo(etapa=4, subetapa=5, tipo_arquivo="tbl", nome="janelas_contabeis_sem_match_operacional")

print(f"Entrada - elegibilidade operacional por data : {caminho_base_elegibilidade_operacional_data}")
print(f"Entrada - elegibilidade contábil por janela  : {caminho_base_elegibilidade_contabil_janela}")
print(f"Saída   - base final de tickers aptos        : {caminho_base_tickers_aptos_janela_contabil}")
print(f"Saída   - resumo                             : {caminho_tbl_resumo_tickers_aptos_janela_contabil}")
print(f"Saída   - cobertura diária                   : {caminho_tbl_cobertura_diaria_tickers_aptos_janela_contabil}")
print(f"Saída   - auditoria de cruzamento            : {caminho_tbl_auditoria_cruzamento_operacional_contabil}")
print(f"Saída   - janelas sem match operacional      : {caminho_tbl_janelas_contabeis_sem_match_operacional}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_elegibilidade_operacional_data = pd.read_parquet(caminho_base_elegibilidade_operacional_data)
df_elegibilidade_contabil_janela = pd.read_parquet(caminho_base_elegibilidade_contabil_janela)

print(f"Elegibilidade operacional por data : {df_elegibilidade_operacional_data.shape[0]:,} linhas x {df_elegibilidade_operacional_data.shape[1]} colunas")
print(f"Elegibilidade contábil por janela  : {df_elegibilidade_contabil_janela.shape[0]:,} linhas x {df_elegibilidade_contabil_janela.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e colunas obrigatórias
# ============================================================

print("\n[4/10] Funções auxiliares e colunas obrigatórias...")

COLUNAS_OBRIGATORIAS_OPERACIONAL = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj_data_avaliacao",
    "pregoes_negociados_janela",
    "pct_pregoes_negociados_janela",
    "mediana_volume_fin_janela",
    "mediana_negocios_janela",
    "flag_janela_liquidez_completa",
    "flag_elegivel_liquidez_ticker_data",
    "flag_elegivel_operacional_final",
]

COLUNAS_OBRIGATORIAS_CONTABIL = [
    "issuer_code",
    "periodo",
    "ticker_exemplo",
    "data_inicio_vigencia_operacional",
    "data_fim_vigencia_operacional",
    "data_fim_vigencia_operacional_ajustada",
    "data_inicio_vigencia_efetiva_projeto",
    "data_fim_vigencia_efetiva_projeto",
    "flag_vigencia_aplicavel",
    "flag_apta_contabil_janela",
    "n_tickers_periodo",
    "n_regras_aprovadas_filtro_contabil",
    "n_regras_reprovadas_filtro_contabil",
    "n_variaveis_ausentes_filtro_contabil",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Preparação das bases para o cruzamento final
# ============================================================

print("\n[5/10] Preparação das bases para o cruzamento final...")

validar_colunas_obrigatorias(df_elegibilidade_operacional_data, COLUNAS_OBRIGATORIAS_OPERACIONAL)
validar_colunas_obrigatorias(df_elegibilidade_contabil_janela, COLUNAS_OBRIGATORIAS_CONTABIL)

df_operacional = df_elegibilidade_operacional_data.copy()
df_contabil = df_elegibilidade_contabil_janela.copy()

df_operacional["ticker"] = df_operacional["ticker"].astype("string").str.strip().str.upper()
df_operacional["issuer_code"] = df_operacional["issuer_code"].astype("string").str.strip()
df_operacional["data"] = pd.to_datetime(df_operacional["data"], errors="coerce")

df_contabil["issuer_code"] = df_contabil["issuer_code"].astype("string").str.strip()
df_contabil["ticker_exemplo"] = df_contabil["ticker_exemplo"].astype("string").str.strip().str.upper()
df_contabil["periodo"] = df_contabil["periodo"].astype("string").str.strip().str.upper()

for coluna in [
    "data_inicio_vigencia_operacional",
    "data_fim_vigencia_operacional",
    "data_fim_vigencia_operacional_ajustada",
    "data_inicio_vigencia_efetiva_projeto",
    "data_fim_vigencia_efetiva_projeto",
]:
    df_contabil[coluna] = pd.to_datetime(df_contabil[coluna], errors="coerce")

df_operacional = df_operacional.loc[
    df_operacional["flag_elegivel_operacional_final"]
    & df_operacional["flag_elegivel_liquidez_ticker_data"]
    & df_operacional["flag_janela_liquidez_completa"]
].copy()

df_contabil = df_contabil.loc[
    df_contabil["flag_vigencia_aplicavel"]
    & df_contabil["flag_apta_contabil_janela"]
].copy()

print(f"Linhas operacionais elegíveis utilizadas no cruzamento : {len(df_operacional):,}")
print(f"Janelas contábeis aptas utilizadas no cruzamento       : {len(df_contabil):,}")
print("OK")

# ============================================================
# 6) Cruzamento entre a base operacional diária e a base contábil apta
# ============================================================

print("\n[6/10] Cruzamento entre a base operacional diária e a base contábil apta...")

df_cruzamento = df_operacional.merge(
    df_contabil[
        [
            "issuer_code",
            "periodo",
            "ticker_exemplo",
            "data_inicio_vigencia_operacional",
            "data_fim_vigencia_operacional",
            "data_fim_vigencia_operacional_ajustada",
            "data_inicio_vigencia_efetiva_projeto",
            "data_fim_vigencia_efetiva_projeto",
            "n_tickers_periodo",
            "n_regras_aprovadas_filtro_contabil",
            "n_regras_reprovadas_filtro_contabil",
            "n_variaveis_ausentes_filtro_contabil",
        ]
    ],
    on="issuer_code",
    how="left",
    validate="many_to_many",
)

df_cruzamento["flag_data_dentro_janela_contabil_apta"] = (
    pd.notna(df_cruzamento["data_inicio_vigencia_efetiva_projeto"])
    & pd.notna(df_cruzamento["data_fim_vigencia_efetiva_projeto"])
    & df_cruzamento["data"].ge(df_cruzamento["data_inicio_vigencia_efetiva_projeto"])
    & df_cruzamento["data"].le(df_cruzamento["data_fim_vigencia_efetiva_projeto"])
)

df_tickers_aptos_janela_contabil = (
    df_cruzamento
    .loc[df_cruzamento["flag_data_dentro_janela_contabil_apta"]]
    .copy()
)

n_duplicidades_ticker_data = int(df_tickers_aptos_janela_contabil.duplicated(subset=["ticker", "data"]).sum())
n_duplicidades_issuer_data = int(df_tickers_aptos_janela_contabil.duplicated(subset=["issuer_code", "data"]).sum())

if n_duplicidades_ticker_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_ticker_data} duplicidades na chave ticker-data após o cruzamento operacional-contábil.")
if n_duplicidades_issuer_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_issuer_data} duplicidades na chave issuer_code-data após o cruzamento operacional-contábil.")

df_tickers_aptos_janela_contabil["flag_ticker_apto_compra_final"] = True
df_tickers_aptos_janela_contabil["origem_aptidao_final"] = "elegibilidade_operacional_mais_filtro_contabil"

COLUNAS_BASE_FINAL = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj_data_avaliacao",
    "pregoes_negociados_janela",
    "pct_pregoes_negociados_janela",
    "mediana_volume_fin_janela",
    "mediana_negocios_janela",
    "flag_janela_liquidez_completa",
    "flag_elegivel_liquidez_ticker_data",
    "flag_elegivel_operacional_final",
    "periodo",
    "ticker_exemplo",
    "data_inicio_vigencia_operacional",
    "data_fim_vigencia_operacional",
    "data_fim_vigencia_operacional_ajustada",
    "data_inicio_vigencia_efetiva_projeto",
    "data_fim_vigencia_efetiva_projeto",
    "n_tickers_periodo",
    "n_regras_aprovadas_filtro_contabil",
    "n_regras_reprovadas_filtro_contabil",
    "n_variaveis_ausentes_filtro_contabil",
    "flag_ticker_apto_compra_final",
    "origem_aptidao_final",
]

df_tickers_aptos_janela_contabil = (
    df_tickers_aptos_janela_contabil[COLUNAS_BASE_FINAL]
    .sort_values(["data", "issuer_code", "ticker"])
    .reset_index(drop=True)
)

print(f"Base final de tickers aptos após cruzamento operacional-contábil: {df_tickers_aptos_janela_contabil.shape[0]:,} linhas x {df_tickers_aptos_janela_contabil.shape[1]} colunas")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria da subetapa...")

n_linhas_operacionais = int(len(df_operacional))
n_linhas_finais_aptas = int(len(df_tickers_aptos_janela_contabil))
n_tickers_finais = int(df_tickers_aptos_janela_contabil["ticker"].nunique(dropna=True))
n_empresas_finais = int(df_tickers_aptos_janela_contabil["issuer_code"].nunique(dropna=True))
n_datas_finais = int(df_tickers_aptos_janela_contabil["data"].nunique(dropna=True))

df_resumo_tickers_aptos_janela_contabil = pd.DataFrame(
    [
        {
            "base": "tickers_aptos_janela_contabil",
            "n_linhas_operacionais_entrada": n_linhas_operacionais,
            "n_linhas_finais_aptas": n_linhas_finais_aptas,
            "n_tickers_finais": n_tickers_finais,
            "n_issuer_code_finais": n_empresas_finais,
            "n_datas_finais": n_datas_finais,
            "data_min": df_tickers_aptos_janela_contabil["data"].min(),
            "data_max": df_tickers_aptos_janela_contabil["data"].max(),
            "pct_linhas_operacionais_preservadas": (n_linhas_finais_aptas / n_linhas_operacionais) if n_linhas_operacionais > 0 else pd.NA,
            "n_duplicidades_ticker_data_pos_cruzamento": n_duplicidades_ticker_data,
            "n_duplicidades_issuer_data_pos_cruzamento": n_duplicidades_issuer_data,
        }
    ]
)

df_cobertura_diaria_tickers_aptos_janela_contabil = (
    df_tickers_aptos_janela_contabil
    .groupby("data", as_index=False)
    .agg(
        n_tickers_aptos_compra=("ticker", "nunique"),
        n_empresas_aptas_compra=("issuer_code", "nunique"),
        mediana_volume_fin_janela=("mediana_volume_fin_janela", "median"),
        mediana_pct_pregoes_negociados_janela=("pct_pregoes_negociados_janela", "median"),
        mediana_regras_contabeis_aprovadas=("n_regras_aprovadas_filtro_contabil", "median"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_auditoria_cruzamento_operacional_contabil = pd.DataFrame(
    [
        {"metrica": "n_linhas_operacionais_entrada", "valor": n_linhas_operacionais},
        {"metrica": "n_linhas_contabeis_aptas_janela", "valor": int(len(df_contabil))},
        {"metrica": "n_linhas_pos_merge_bruto", "valor": int(len(df_cruzamento))},
        {"metrica": "n_linhas_finais_aptas", "valor": n_linhas_finais_aptas},
        {"metrica": "n_tickers_finais", "valor": n_tickers_finais},
        {"metrica": "n_issuer_code_finais", "valor": n_empresas_finais},
        {"metrica": "n_datas_finais", "valor": n_datas_finais},
        {"metrica": "n_duplicidades_ticker_data_pos_cruzamento", "valor": n_duplicidades_ticker_data},
        {"metrica": "n_duplicidades_issuer_data_pos_cruzamento", "valor": n_duplicidades_issuer_data},
    ]
)

df_janelas_contabeis_sem_match_operacional = (
    df_contabil
    .merge(
        df_tickers_aptos_janela_contabil[
            [
                "issuer_code",
                "periodo",
                "data_inicio_vigencia_efetiva_projeto",
                "data_fim_vigencia_efetiva_projeto",
            ]
        ].drop_duplicates(),
        on=[
            "issuer_code",
            "periodo",
            "data_inicio_vigencia_efetiva_projeto",
            "data_fim_vigencia_efetiva_projeto",
        ],
        how="left",
        indicator=True,
    )
    .loc[lambda df: df["_merge"].eq("left_only")]
    .drop(columns="_merge")
    .sort_values(["data_inicio_vigencia_efetiva_projeto", "issuer_code", "periodo"])
    .reset_index(drop=True)
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_tickers_aptos_janela_contabil, caminho_base_tickers_aptos_janela_contabil, index=False)
salvar_dataframe(df_resumo_tickers_aptos_janela_contabil, caminho_tbl_resumo_tickers_aptos_janela_contabil, index=False)
salvar_dataframe(df_cobertura_diaria_tickers_aptos_janela_contabil, caminho_tbl_cobertura_diaria_tickers_aptos_janela_contabil, index=False)
salvar_dataframe(df_auditoria_cruzamento_operacional_contabil, caminho_tbl_auditoria_cruzamento_operacional_contabil, index=False)
salvar_dataframe(df_janelas_contabeis_sem_match_operacional, caminho_tbl_janelas_contabeis_sem_match_operacional, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_tickers_aptos = df_tickers_aptos_janela_contabil.head(15).copy()
df_amostra_cobertura_diaria = df_cobertura_diaria_tickers_aptos_janela_contabil.head(15).copy()
df_amostra_janelas_sem_match = df_janelas_contabeis_sem_match_operacional.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base final de tickers aptos por janela contábil:")
print(df_resumo_tickers_aptos_janela_contabil.to_string(index=False))

print("\nAuditoria do cruzamento operacional-contábil:")
print(df_auditoria_cruzamento_operacional_contabil.to_string(index=False))

print("\nAmostra da base final de tickers aptos por janela contábil:")
print(df_amostra_tickers_aptos.to_string(index=False))

print("\nCobertura diária dos tickers aptos - amostra:")
print(df_amostra_cobertura_diaria.to_string(index=False))

print("\nJanelas contábeis aptas sem match operacional - amostra:")
print(df_amostra_janelas_sem_match.to_string(index=False))

print("\nArquivos salvos na subetapa 4.5:")
print(f"- {caminho_base_tickers_aptos_janela_contabil}")
print(f"- {caminho_tbl_resumo_tickers_aptos_janela_contabil}")
print(f"- {caminho_tbl_cobertura_diaria_tickers_aptos_janela_contabil}")
print(f"- {caminho_tbl_auditoria_cruzamento_operacional_contabil}")
print(f"- {caminho_tbl_janelas_contabeis_sem_match_operacional}")

print("\nETAPA 4.5 FINALIZADA COM SUCESSO.")
print("=" * 100)


INICIANDO ETAPA 4.5 - BASE FINAL DE TICKERS APTOS POR JANELA CONTÁBIL

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - elegibilidade operacional por data : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_4_base_elegibilidade_operacional_data.parquet
Entrada - elegibilidade contábil por janela  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_4_base_elegibilidade_contabil_janela.parquet
Saída   - base final de tickers aptos        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_5_base_tickers_aptos_janela_contabil.parquet
Saída   - resumo                             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_b

# Etapa 5) Construção dos Indicadores de Amplitude

## Etapa 5.1) Cálculo da Média Móvel de 200 Dias

In [20]:
%%time
# ============================================================
# Etapa 5.1) Cálculo da Média Móvel de 200 Dias
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 5.1 - CÁLCULO DA MÉDIA MÓVEL DE 200 DIAS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="base", nome="mercado_diario_consolidada")

caminho_base_mm200_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=1, tipo_arquivo="base", nome="mm200_ticker_data")
caminho_tbl_resumo_mm200_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=1, tipo_arquivo="tbl", nome="resumo_mm200_ticker_data")
caminho_tbl_cobertura_diaria_mm200 = gerar_caminho_arquivo(etapa=5, subetapa=1, tipo_arquivo="tbl", nome="cobertura_diaria_mm200")
caminho_tbl_auditoria_mm200 = gerar_caminho_arquivo(etapa=5, subetapa=1, tipo_arquivo="tbl", nome="auditoria_mm200")

print(f"Entrada - mercado diário consolidada : {caminho_base_mercado_diario_consolidada}")
print(f"Saída   - base MM200 por ticker-data : {caminho_base_mm200_ticker_data}")
print(f"Saída   - resumo                     : {caminho_tbl_resumo_mm200_ticker_data}")
print(f"Saída   - cobertura diária           : {caminho_tbl_cobertura_diaria_mm200}")
print(f"Saída   - auditoria                  : {caminho_tbl_auditoria_mm200}")
print("OK")

# ============================================================
# 3) Carga da base de entrada
# ============================================================

print("\n[3/10] Carga da base de entrada...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)

print(f"Base mercado diário consolidada: {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros operacionais e funções auxiliares
# ============================================================

print("\n[4/10] Parâmetros operacionais e funções auxiliares...")

JANELA_MM200_PREGOES = 200

COLUNAS_OBRIGATORIAS = [
    "ticker",
    "data",
    "close_adj",
]

COLUNAS_BASE_SAIDA = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj",
    "n_obs_validas_mm200",
    "mm200",
    "flag_mm200_valida",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

print(f"Janela operacional da MM200: {JANELA_MM200_PREGOES} pregões")
print("OK")

# ============================================================
# 5) Preparação da base para cálculo da MM200
# ============================================================

print("\n[5/10] Preparação da base para cálculo da MM200...")

validar_colunas_obrigatorias(df_mercado_diario_consolidada, COLUNAS_OBRIGATORIAS)

df_mm200_ticker_data = df_mercado_diario_consolidada.copy()

df_mm200_ticker_data["ticker"] = df_mm200_ticker_data["ticker"].astype("string").str.strip().str.upper()
if "issuer_code" in df_mm200_ticker_data.columns:
    df_mm200_ticker_data["issuer_code"] = df_mm200_ticker_data["issuer_code"].astype("string").str.strip()

for coluna_texto in ["nome", "setor", "subsetor", "segmento"]:
    if coluna_texto not in df_mm200_ticker_data.columns:
        df_mm200_ticker_data[coluna_texto] = pd.NA

df_mm200_ticker_data["data"] = pd.to_datetime(df_mm200_ticker_data["data"], errors="coerce")
df_mm200_ticker_data["close_adj"] = pd.to_numeric(df_mm200_ticker_data["close_adj"], errors="coerce")

n_duplicidades_ticker_data = int(df_mm200_ticker_data.duplicated(subset=["ticker", "data"]).sum())
n_close_adj_nulo = int(df_mm200_ticker_data["close_adj"].isna().sum())
n_data_nula = int(df_mm200_ticker_data["data"].isna().sum())
n_ticker_nulo = int(df_mm200_ticker_data["ticker"].isna().sum())

if n_duplicidades_ticker_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_ticker_data} duplicidades na chave ticker-data da base de entrada.")
if n_data_nula > 0:
    raise ValueError(f"Foram encontradas {n_data_nula} linhas com data nula na base de entrada.")
if n_ticker_nulo > 0:
    raise ValueError(f"Foram encontradas {n_ticker_nulo} linhas com ticker nulo na base de entrada.")

df_mm200_ticker_data = (
    df_mm200_ticker_data
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

print(f"Duplicidades ticker-data na entrada : {n_duplicidades_ticker_data}")
print(f"Linhas com close_adj nulo           : {n_close_adj_nulo}")
print("OK")

# ============================================================
# 6) Cálculo da MM200 por ticker
# ============================================================

print("\n[6/10] Cálculo da MM200 por ticker...")

agrupador_ticker = df_mm200_ticker_data.groupby("ticker", dropna=False)["close_adj"]

df_mm200_ticker_data["n_obs_validas_mm200"] = (
    agrupador_ticker
    .rolling(window=JANELA_MM200_PREGOES, min_periods=1)
    .count()
    .reset_index(level=0, drop=True)
)

df_mm200_ticker_data["mm200"] = (
    agrupador_ticker
    .rolling(window=JANELA_MM200_PREGOES, min_periods=JANELA_MM200_PREGOES)
    .mean()
    .reset_index(level=0, drop=True)
)

df_mm200_ticker_data["flag_mm200_valida"] = (
    df_mm200_ticker_data["n_obs_validas_mm200"].ge(JANELA_MM200_PREGOES)
    & df_mm200_ticker_data["mm200"].notna()
)

df_mm200_ticker_data = (
    df_mm200_ticker_data[COLUNAS_BASE_SAIDA]
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

print(f"Base MM200 por ticker-data: {df_mm200_ticker_data.shape[0]:,} linhas x {df_mm200_ticker_data.shape[1]} colunas")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

n_linhas_total = int(len(df_mm200_ticker_data))
n_tickers_total = int(df_mm200_ticker_data["ticker"].nunique(dropna=True))
n_datas_total = int(df_mm200_ticker_data["data"].nunique(dropna=True))
n_linhas_mm200_valida = int(df_mm200_ticker_data["flag_mm200_valida"].sum())
n_tickers_com_mm200_em_alguma_data = int(
    df_mm200_ticker_data.loc[df_mm200_ticker_data["flag_mm200_valida"], "ticker"].nunique(dropna=True)
)

df_resumo_mm200_ticker_data = pd.DataFrame(
    [
        {
            "base": "mm200_ticker_data",
            "n_linhas": n_linhas_total,
            "n_tickers": n_tickers_total,
            "n_datas": n_datas_total,
            "data_min": df_mm200_ticker_data["data"].min(),
            "data_max": df_mm200_ticker_data["data"].max(),
            "n_linhas_mm200_valida": n_linhas_mm200_valida,
            "pct_linhas_mm200_valida": (n_linhas_mm200_valida / n_linhas_total) if n_linhas_total > 0 else pd.NA,
            "n_tickers_com_mm200_em_alguma_data": n_tickers_com_mm200_em_alguma_data,
        }
    ]
)

df_cobertura_diaria_mm200 = (
    df_mm200_ticker_data
    .groupby("data", as_index=False)
    .agg(
        n_tickers_total=("ticker", "nunique"),
        n_tickers_mm200_valida=("flag_mm200_valida", "sum"),
        mediana_close_adj=("close_adj", "median"),
        mediana_mm200=("mm200", "median"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_cobertura_diaria_mm200["pct_tickers_mm200_valida"] = (
    df_cobertura_diaria_mm200["n_tickers_mm200_valida"] / df_cobertura_diaria_mm200["n_tickers_total"]
)

df_auditoria_mm200 = pd.DataFrame(
    [
        {"metrica": "janela_mm200_pregoes", "valor": JANELA_MM200_PREGOES},
        {"metrica": "n_linhas_total", "valor": n_linhas_total},
        {"metrica": "n_tickers_total", "valor": n_tickers_total},
        {"metrica": "n_datas_total", "valor": n_datas_total},
        {"metrica": "n_linhas_mm200_valida", "valor": n_linhas_mm200_valida},
        {"metrica": "n_tickers_com_mm200_em_alguma_data", "valor": n_tickers_com_mm200_em_alguma_data},
        {"metrica": "n_close_adj_nulo_entrada", "valor": n_close_adj_nulo},
        {"metrica": "n_duplicidades_ticker_data_entrada", "valor": n_duplicidades_ticker_data},
    ]
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_mm200_ticker_data, caminho_base_mm200_ticker_data, index=False)
salvar_dataframe(df_resumo_mm200_ticker_data, caminho_tbl_resumo_mm200_ticker_data, index=False)
salvar_dataframe(df_cobertura_diaria_mm200, caminho_tbl_cobertura_diaria_mm200, index=False)
salvar_dataframe(df_auditoria_mm200, caminho_tbl_auditoria_mm200, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_mm200_inicial = df_mm200_ticker_data.head(15).copy()
df_amostra_mm200_valida = (
    df_mm200_ticker_data
    .loc[df_mm200_ticker_data["flag_mm200_valida"]]
    .head(15)
    .copy()
)
df_amostra_cobertura_diaria_mm200 = df_cobertura_diaria_mm200.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base MM200 por ticker-data:")
print(df_resumo_mm200_ticker_data.to_string(index=False))

print("\nAuditoria da subetapa MM200:")
print(df_auditoria_mm200.to_string(index=False))

print("\nAmostra inicial da base MM200 por ticker-data:")
print(df_amostra_mm200_inicial.to_string(index=False))

print("\nAmostra de linhas com MM200 válida:")
print(df_amostra_mm200_valida.to_string(index=False))

print("\nCobertura diária da MM200 - amostra:")
print(df_amostra_cobertura_diaria_mm200.to_string(index=False))

print("\nArquivos salvos na subetapa 5.1:")
print(f"- {caminho_base_mm200_ticker_data}")
print(f"- {caminho_tbl_resumo_mm200_ticker_data}")
print(f"- {caminho_tbl_cobertura_diaria_mm200}")
print(f"- {caminho_tbl_auditoria_mm200}")

print("\nETAPA 5.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 5.1 - CÁLCULO DA MÉDIA MÓVEL DE 200 DIAS

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidada : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Saída   - base MM200 por ticker-data : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_1_base_mm200_ticker_data.parquet
Saída   - resumo                     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_1_tbl_resumo_mm200_ticker_data.parquet
Saída   - cobertura diária           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_1_tbl_cobertura_diaria_mm200.parquet

## Etapa 5.2) Cálculo das Mínimas de 52 Semanas

In [21]:
%%time
# ============================================================
# Etapa 5.2) Cálculo das Mínimas de 52 Semanas
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 5.2 - CÁLCULO DAS MÍNIMAS DE 52 SEMANAS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="base", nome="mercado_diario_consolidada")

caminho_base_minima_52s_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=2, tipo_arquivo="base", nome="minima_52s_ticker_data")
caminho_tbl_resumo_minima_52s_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=2, tipo_arquivo="tbl", nome="resumo_minima_52s_ticker_data")
caminho_tbl_cobertura_diaria_minima_52s = gerar_caminho_arquivo(etapa=5, subetapa=2, tipo_arquivo="tbl", nome="cobertura_diaria_minima_52s")
caminho_tbl_auditoria_minima_52s = gerar_caminho_arquivo(etapa=5, subetapa=2, tipo_arquivo="tbl", nome="auditoria_minima_52s")

print(f"Entrada - mercado diário consolidada        : {caminho_base_mercado_diario_consolidada}")
print(f"Saída   - base mínima 52 semanas por ticker : {caminho_base_minima_52s_ticker_data}")
print(f"Saída   - resumo                            : {caminho_tbl_resumo_minima_52s_ticker_data}")
print(f"Saída   - cobertura diária                  : {caminho_tbl_cobertura_diaria_minima_52s}")
print(f"Saída   - auditoria                         : {caminho_tbl_auditoria_minima_52s}")
print("OK")

# ============================================================
# 3) Carga da base de entrada
# ============================================================

print("\n[3/10] Carga da base de entrada...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)

print(f"Base mercado diário consolidada: {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros operacionais e funções auxiliares
# ============================================================

print("\n[4/10] Parâmetros operacionais e funções auxiliares...")

JANELA_52S_PREGOES = 252

COLUNAS_OBRIGATORIAS = [
    "ticker",
    "data",
    "close_adj",
]

COLUNAS_BASE_SAIDA = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj",
    "n_obs_validas_minima_52s",
    "minima_52s",
    "flag_minima_52s_valida",
    "flag_em_minima_52s",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

print(f"Janela operacional da mínima de 52 semanas: {JANELA_52S_PREGOES} pregões")
print("OK")

# ============================================================
# 5) Preparação da base para cálculo da mínima de 52 semanas
# ============================================================

print("\n[5/10] Preparação da base para cálculo da mínima de 52 semanas...")

validar_colunas_obrigatorias(df_mercado_diario_consolidada, COLUNAS_OBRIGATORIAS)

df_minima_52s_ticker_data = df_mercado_diario_consolidada.copy()

df_minima_52s_ticker_data["ticker"] = df_minima_52s_ticker_data["ticker"].astype("string").str.strip().str.upper()
if "issuer_code" in df_minima_52s_ticker_data.columns:
    df_minima_52s_ticker_data["issuer_code"] = df_minima_52s_ticker_data["issuer_code"].astype("string").str.strip()

for coluna_texto in ["nome", "setor", "subsetor", "segmento"]:
    if coluna_texto not in df_minima_52s_ticker_data.columns:
        df_minima_52s_ticker_data[coluna_texto] = pd.NA

df_minima_52s_ticker_data["data"] = pd.to_datetime(df_minima_52s_ticker_data["data"], errors="coerce")
df_minima_52s_ticker_data["close_adj"] = pd.to_numeric(df_minima_52s_ticker_data["close_adj"], errors="coerce")

n_duplicidades_ticker_data = int(df_minima_52s_ticker_data.duplicated(subset=["ticker", "data"]).sum())
n_close_adj_nulo = int(df_minima_52s_ticker_data["close_adj"].isna().sum())
n_data_nula = int(df_minima_52s_ticker_data["data"].isna().sum())
n_ticker_nulo = int(df_minima_52s_ticker_data["ticker"].isna().sum())

if n_duplicidades_ticker_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_ticker_data} duplicidades na chave ticker-data da base de entrada.")
if n_data_nula > 0:
    raise ValueError(f"Foram encontradas {n_data_nula} linhas com data nula na base de entrada.")
if n_ticker_nulo > 0:
    raise ValueError(f"Foram encontradas {n_ticker_nulo} linhas com ticker nulo na base de entrada.")

df_minima_52s_ticker_data = (
    df_minima_52s_ticker_data
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

print(f"Duplicidades ticker-data na entrada : {n_duplicidades_ticker_data}")
print(f"Linhas com close_adj nulo           : {n_close_adj_nulo}")
print("OK")

# ============================================================
# 6) Cálculo da mínima móvel de 52 semanas por ticker
# ============================================================

print("\n[6/10] Cálculo da mínima móvel de 52 semanas por ticker...")

agrupador_ticker = df_minima_52s_ticker_data.groupby("ticker", dropna=False)["close_adj"]

df_minima_52s_ticker_data["n_obs_validas_minima_52s"] = (
    agrupador_ticker
    .rolling(window=JANELA_52S_PREGOES, min_periods=1)
    .count()
    .reset_index(level=0, drop=True)
)

df_minima_52s_ticker_data["minima_52s"] = (
    agrupador_ticker
    .rolling(window=JANELA_52S_PREGOES, min_periods=JANELA_52S_PREGOES)
    .min()
    .reset_index(level=0, drop=True)
)

df_minima_52s_ticker_data["flag_minima_52s_valida"] = (
    df_minima_52s_ticker_data["n_obs_validas_minima_52s"].ge(JANELA_52S_PREGOES)
    & df_minima_52s_ticker_data["minima_52s"].notna()
)

df_minima_52s_ticker_data["flag_em_minima_52s"] = (
    df_minima_52s_ticker_data["flag_minima_52s_valida"]
    & df_minima_52s_ticker_data["close_adj"].eq(df_minima_52s_ticker_data["minima_52s"])
)

df_minima_52s_ticker_data = (
    df_minima_52s_ticker_data[COLUNAS_BASE_SAIDA]
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

print(f"Base mínima 52 semanas por ticker-data: {df_minima_52s_ticker_data.shape[0]:,} linhas x {df_minima_52s_ticker_data.shape[1]} colunas")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

n_linhas_total = int(len(df_minima_52s_ticker_data))
n_tickers_total = int(df_minima_52s_ticker_data["ticker"].nunique(dropna=True))
n_datas_total = int(df_minima_52s_ticker_data["data"].nunique(dropna=True))
n_linhas_minima_52s_valida = int(df_minima_52s_ticker_data["flag_minima_52s_valida"].sum())
n_tickers_com_minima_52s_em_alguma_data = int(
    df_minima_52s_ticker_data.loc[df_minima_52s_ticker_data["flag_em_minima_52s"], "ticker"].nunique(dropna=True)
)
n_linhas_em_minima_52s = int(df_minima_52s_ticker_data["flag_em_minima_52s"].sum())

df_resumo_minima_52s_ticker_data = pd.DataFrame(
    [
        {
            "base": "minima_52s_ticker_data",
            "n_linhas": n_linhas_total,
            "n_tickers": n_tickers_total,
            "n_datas": n_datas_total,
            "data_min": df_minima_52s_ticker_data["data"].min(),
            "data_max": df_minima_52s_ticker_data["data"].max(),
            "n_linhas_minima_52s_valida": n_linhas_minima_52s_valida,
            "pct_linhas_minima_52s_valida": (n_linhas_minima_52s_valida / n_linhas_total) if n_linhas_total > 0 else pd.NA,
            "n_linhas_em_minima_52s": n_linhas_em_minima_52s,
            "pct_linhas_em_minima_52s": (n_linhas_em_minima_52s / n_linhas_total) if n_linhas_total > 0 else pd.NA,
            "n_tickers_com_minima_52s_em_alguma_data": n_tickers_com_minima_52s_em_alguma_data,
        }
    ]
)

df_cobertura_diaria_minima_52s = (
    df_minima_52s_ticker_data
    .groupby("data", as_index=False)
    .agg(
        n_tickers_total=("ticker", "nunique"),
        n_tickers_minima_52s_valida=("flag_minima_52s_valida", "sum"),
        n_tickers_em_minima_52s=("flag_em_minima_52s", "sum"),
        mediana_close_adj=("close_adj", "median"),
        mediana_minima_52s=("minima_52s", "median"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_cobertura_diaria_minima_52s["pct_tickers_minima_52s_valida"] = (
    df_cobertura_diaria_minima_52s["n_tickers_minima_52s_valida"] / df_cobertura_diaria_minima_52s["n_tickers_total"]
)

df_cobertura_diaria_minima_52s["pct_tickers_em_minima_52s"] = (
    df_cobertura_diaria_minima_52s["n_tickers_em_minima_52s"] / df_cobertura_diaria_minima_52s["n_tickers_total"]
)

df_auditoria_minima_52s = pd.DataFrame(
    [
        {"metrica": "janela_52s_pregoes", "valor": JANELA_52S_PREGOES},
        {"metrica": "n_linhas_total", "valor": n_linhas_total},
        {"metrica": "n_tickers_total", "valor": n_tickers_total},
        {"metrica": "n_datas_total", "valor": n_datas_total},
        {"metrica": "n_linhas_minima_52s_valida", "valor": n_linhas_minima_52s_valida},
        {"metrica": "n_linhas_em_minima_52s", "valor": n_linhas_em_minima_52s},
        {"metrica": "n_tickers_com_minima_52s_em_alguma_data", "valor": n_tickers_com_minima_52s_em_alguma_data},
        {"metrica": "n_close_adj_nulo_entrada", "valor": n_close_adj_nulo},
        {"metrica": "n_duplicidades_ticker_data_entrada", "valor": n_duplicidades_ticker_data},
    ]
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_minima_52s_ticker_data, caminho_base_minima_52s_ticker_data, index=False)
salvar_dataframe(df_resumo_minima_52s_ticker_data, caminho_tbl_resumo_minima_52s_ticker_data, index=False)
salvar_dataframe(df_cobertura_diaria_minima_52s, caminho_tbl_cobertura_diaria_minima_52s, index=False)
salvar_dataframe(df_auditoria_minima_52s, caminho_tbl_auditoria_minima_52s, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_minima_52s_inicial = df_minima_52s_ticker_data.head(15).copy()
df_amostra_minima_52s_valida = (
    df_minima_52s_ticker_data
    .loc[df_minima_52s_ticker_data["flag_minima_52s_valida"]]
    .head(15)
    .copy()
)
df_amostra_em_minima_52s = (
    df_minima_52s_ticker_data
    .loc[df_minima_52s_ticker_data["flag_em_minima_52s"]]
    .head(15)
    .copy()
)
df_amostra_cobertura_diaria_minima_52s = df_cobertura_diaria_minima_52s.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base de mínima de 52 semanas por ticker-data:")
print(df_resumo_minima_52s_ticker_data.to_string(index=False))

print("\nAuditoria da subetapa de mínima de 52 semanas:")
print(df_auditoria_minima_52s.to_string(index=False))

print("\nAmostra inicial da base de mínima de 52 semanas:")
print(df_amostra_minima_52s_inicial.to_string(index=False))

print("\nAmostra de linhas com mínima de 52 semanas válida:")
print(df_amostra_minima_52s_valida.to_string(index=False))

print("\nAmostra de linhas em mínima de 52 semanas:")
print(df_amostra_em_minima_52s.to_string(index=False))

print("\nCobertura diária da mínima de 52 semanas - amostra:")
print(df_amostra_cobertura_diaria_minima_52s.to_string(index=False))

print("\nArquivos salvos na subetapa 5.2:")
print(f"- {caminho_base_minima_52s_ticker_data}")
print(f"- {caminho_tbl_resumo_minima_52s_ticker_data}")
print(f"- {caminho_tbl_cobertura_diaria_minima_52s}")
print(f"- {caminho_tbl_auditoria_minima_52s}")

print("\nETAPA 5.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 5.2 - CÁLCULO DAS MÍNIMAS DE 52 SEMANAS

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidada        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Saída   - base mínima 52 semanas por ticker : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_2_base_minima_52s_ticker_data.parquet
Saída   - resumo                            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_2_tbl_resumo_minima_52s_ticker_data.parquet
Saída   - cobertura diária                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5

## Etapa 5.3) Cálculo das Máximas de 52 Semanas

In [22]:
%%time
# ============================================================
# Etapa 5.3) Cálculo das Máximas de 52 Semanas
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 5.3 - CÁLCULO DAS MÁXIMAS DE 52 SEMANAS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(etapa=2, subetapa=4, tipo_arquivo="base", nome="mercado_diario_consolidada")

caminho_base_maxima_52s_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=3, tipo_arquivo="base", nome="maxima_52s_ticker_data")
caminho_tbl_resumo_maxima_52s_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=3, tipo_arquivo="tbl", nome="resumo_maxima_52s_ticker_data")
caminho_tbl_cobertura_diaria_maxima_52s = gerar_caminho_arquivo(etapa=5, subetapa=3, tipo_arquivo="tbl", nome="cobertura_diaria_maxima_52s")
caminho_tbl_auditoria_maxima_52s = gerar_caminho_arquivo(etapa=5, subetapa=3, tipo_arquivo="tbl", nome="auditoria_maxima_52s")

print(f"Entrada - mercado diário consolidada        : {caminho_base_mercado_diario_consolidada}")
print(f"Saída   - base máxima 52 semanas por ticker : {caminho_base_maxima_52s_ticker_data}")
print(f"Saída   - resumo                            : {caminho_tbl_resumo_maxima_52s_ticker_data}")
print(f"Saída   - cobertura diária                  : {caminho_tbl_cobertura_diaria_maxima_52s}")
print(f"Saída   - auditoria                         : {caminho_tbl_auditoria_maxima_52s}")
print("OK")

# ============================================================
# 3) Carga da base de entrada
# ============================================================

print("\n[3/10] Carga da base de entrada...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)

print(f"Base mercado diário consolidada: {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Parâmetros operacionais e funções auxiliares
# ============================================================

print("\n[4/10] Parâmetros operacionais e funções auxiliares...")

JANELA_52S_PREGOES = 252

COLUNAS_OBRIGATORIAS = [
    "ticker",
    "data",
    "close_adj",
]

COLUNAS_BASE_SAIDA = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj",
    "n_obs_validas_maxima_52s",
    "maxima_52s",
    "flag_maxima_52s_valida",
    "flag_em_maxima_52s",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

print(f"Janela operacional da máxima de 52 semanas: {JANELA_52S_PREGOES} pregões")
print("OK")

# ============================================================
# 5) Preparação da base para cálculo da máxima de 52 semanas
# ============================================================

print("\n[5/10] Preparação da base para cálculo da máxima de 52 semanas...")

validar_colunas_obrigatorias(df_mercado_diario_consolidada, COLUNAS_OBRIGATORIAS)

df_maxima_52s_ticker_data = df_mercado_diario_consolidada.copy()

df_maxima_52s_ticker_data["ticker"] = df_maxima_52s_ticker_data["ticker"].astype("string").str.strip().str.upper()
if "issuer_code" in df_maxima_52s_ticker_data.columns:
    df_maxima_52s_ticker_data["issuer_code"] = df_maxima_52s_ticker_data["issuer_code"].astype("string").str.strip()

for coluna_texto in ["nome", "setor", "subsetor", "segmento"]:
    if coluna_texto not in df_maxima_52s_ticker_data.columns:
        df_maxima_52s_ticker_data[coluna_texto] = pd.NA

df_maxima_52s_ticker_data["data"] = pd.to_datetime(df_maxima_52s_ticker_data["data"], errors="coerce")
df_maxima_52s_ticker_data["close_adj"] = pd.to_numeric(df_maxima_52s_ticker_data["close_adj"], errors="coerce")

n_duplicidades_ticker_data = int(df_maxima_52s_ticker_data.duplicated(subset=["ticker", "data"]).sum())
n_close_adj_nulo = int(df_maxima_52s_ticker_data["close_adj"].isna().sum())
n_data_nula = int(df_maxima_52s_ticker_data["data"].isna().sum())
n_ticker_nulo = int(df_maxima_52s_ticker_data["ticker"].isna().sum())

if n_duplicidades_ticker_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_ticker_data} duplicidades na chave ticker-data da base de entrada.")
if n_data_nula > 0:
    raise ValueError(f"Foram encontradas {n_data_nula} linhas com data nula na base de entrada.")
if n_ticker_nulo > 0:
    raise ValueError(f"Foram encontradas {n_ticker_nulo} linhas com ticker nulo na base de entrada.")

df_maxima_52s_ticker_data = (
    df_maxima_52s_ticker_data
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

print(f"Duplicidades ticker-data na entrada : {n_duplicidades_ticker_data}")
print(f"Linhas com close_adj nulo           : {n_close_adj_nulo}")
print("OK")

# ============================================================
# 6) Cálculo da máxima móvel de 52 semanas por ticker
# ============================================================

print("\n[6/10] Cálculo da máxima móvel de 52 semanas por ticker...")

agrupador_ticker = df_maxima_52s_ticker_data.groupby("ticker", dropna=False)["close_adj"]

df_maxima_52s_ticker_data["n_obs_validas_maxima_52s"] = (
    agrupador_ticker
    .rolling(window=JANELA_52S_PREGOES, min_periods=1)
    .count()
    .reset_index(level=0, drop=True)
)

df_maxima_52s_ticker_data["maxima_52s"] = (
    agrupador_ticker
    .rolling(window=JANELA_52S_PREGOES, min_periods=JANELA_52S_PREGOES)
    .max()
    .reset_index(level=0, drop=True)
)

df_maxima_52s_ticker_data["flag_maxima_52s_valida"] = (
    df_maxima_52s_ticker_data["n_obs_validas_maxima_52s"].ge(JANELA_52S_PREGOES)
    & df_maxima_52s_ticker_data["maxima_52s"].notna()
)

df_maxima_52s_ticker_data["flag_em_maxima_52s"] = (
    df_maxima_52s_ticker_data["flag_maxima_52s_valida"]
    & df_maxima_52s_ticker_data["close_adj"].eq(df_maxima_52s_ticker_data["maxima_52s"])
)

df_maxima_52s_ticker_data = (
    df_maxima_52s_ticker_data[COLUNAS_BASE_SAIDA]
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

print(f"Base máxima 52 semanas por ticker-data: {df_maxima_52s_ticker_data.shape[0]:,} linhas x {df_maxima_52s_ticker_data.shape[1]} colunas")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

n_linhas_total = int(len(df_maxima_52s_ticker_data))
n_tickers_total = int(df_maxima_52s_ticker_data["ticker"].nunique(dropna=True))
n_datas_total = int(df_maxima_52s_ticker_data["data"].nunique(dropna=True))
n_linhas_maxima_52s_valida = int(df_maxima_52s_ticker_data["flag_maxima_52s_valida"].sum())
n_tickers_com_maxima_52s_em_alguma_data = int(
    df_maxima_52s_ticker_data.loc[df_maxima_52s_ticker_data["flag_em_maxima_52s"], "ticker"].nunique(dropna=True)
)
n_linhas_em_maxima_52s = int(df_maxima_52s_ticker_data["flag_em_maxima_52s"].sum())

df_resumo_maxima_52s_ticker_data = pd.DataFrame(
    [
        {
            "base": "maxima_52s_ticker_data",
            "n_linhas": n_linhas_total,
            "n_tickers": n_tickers_total,
            "n_datas": n_datas_total,
            "data_min": df_maxima_52s_ticker_data["data"].min(),
            "data_max": df_maxima_52s_ticker_data["data"].max(),
            "n_linhas_maxima_52s_valida": n_linhas_maxima_52s_valida,
            "pct_linhas_maxima_52s_valida": (n_linhas_maxima_52s_valida / n_linhas_total) if n_linhas_total > 0 else pd.NA,
            "n_linhas_em_maxima_52s": n_linhas_em_maxima_52s,
            "pct_linhas_em_maxima_52s": (n_linhas_em_maxima_52s / n_linhas_total) if n_linhas_total > 0 else pd.NA,
            "n_tickers_com_maxima_52s_em_alguma_data": n_tickers_com_maxima_52s_em_alguma_data,
        }
    ]
)

df_cobertura_diaria_maxima_52s = (
    df_maxima_52s_ticker_data
    .groupby("data", as_index=False)
    .agg(
        n_tickers_total=("ticker", "nunique"),
        n_tickers_maxima_52s_valida=("flag_maxima_52s_valida", "sum"),
        n_tickers_em_maxima_52s=("flag_em_maxima_52s", "sum"),
        mediana_close_adj=("close_adj", "median"),
        mediana_maxima_52s=("maxima_52s", "median"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_cobertura_diaria_maxima_52s["pct_tickers_maxima_52s_valida"] = (
    df_cobertura_diaria_maxima_52s["n_tickers_maxima_52s_valida"] / df_cobertura_diaria_maxima_52s["n_tickers_total"]
)

df_cobertura_diaria_maxima_52s["pct_tickers_em_maxima_52s"] = (
    df_cobertura_diaria_maxima_52s["n_tickers_em_maxima_52s"] / df_cobertura_diaria_maxima_52s["n_tickers_total"]
)

df_auditoria_maxima_52s = pd.DataFrame(
    [
        {"metrica": "janela_52s_pregoes", "valor": JANELA_52S_PREGOES},
        {"metrica": "n_linhas_total", "valor": n_linhas_total},
        {"metrica": "n_tickers_total", "valor": n_tickers_total},
        {"metrica": "n_datas_total", "valor": n_datas_total},
        {"metrica": "n_linhas_maxima_52s_valida", "valor": n_linhas_maxima_52s_valida},
        {"metrica": "n_linhas_em_maxima_52s", "valor": n_linhas_em_maxima_52s},
        {"metrica": "n_tickers_com_maxima_52s_em_alguma_data", "valor": n_tickers_com_maxima_52s_em_alguma_data},
        {"metrica": "n_close_adj_nulo_entrada", "valor": n_close_adj_nulo},
        {"metrica": "n_duplicidades_ticker_data_entrada", "valor": n_duplicidades_ticker_data},
    ]
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_maxima_52s_ticker_data, caminho_base_maxima_52s_ticker_data, index=False)
salvar_dataframe(df_resumo_maxima_52s_ticker_data, caminho_tbl_resumo_maxima_52s_ticker_data, index=False)
salvar_dataframe(df_cobertura_diaria_maxima_52s, caminho_tbl_cobertura_diaria_maxima_52s, index=False)
salvar_dataframe(df_auditoria_maxima_52s, caminho_tbl_auditoria_maxima_52s, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_maxima_52s_inicial = df_maxima_52s_ticker_data.head(15).copy()
df_amostra_maxima_52s_valida = (
    df_maxima_52s_ticker_data
    .loc[df_maxima_52s_ticker_data["flag_maxima_52s_valida"]]
    .head(15)
    .copy()
)
df_amostra_em_maxima_52s = (
    df_maxima_52s_ticker_data
    .loc[df_maxima_52s_ticker_data["flag_em_maxima_52s"]]
    .head(15)
    .copy()
)
df_amostra_cobertura_diaria_maxima_52s = df_cobertura_diaria_maxima_52s.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base de máxima de 52 semanas por ticker-data:")
print(df_resumo_maxima_52s_ticker_data.to_string(index=False))

print("\nAuditoria da subetapa de máxima de 52 semanas:")
print(df_auditoria_maxima_52s.to_string(index=False))

print("\nAmostra inicial da base de máxima de 52 semanas:")
print(df_amostra_maxima_52s_inicial.to_string(index=False))

print("\nAmostra de linhas com máxima de 52 semanas válida:")
print(df_amostra_maxima_52s_valida.to_string(index=False))

print("\nAmostra de linhas em máxima de 52 semanas:")
print(df_amostra_em_maxima_52s.to_string(index=False))

print("\nCobertura diária da máxima de 52 semanas - amostra:")
print(df_amostra_cobertura_diaria_maxima_52s.to_string(index=False))

print("\nArquivos salvos na subetapa 5.3:")
print(f"- {caminho_base_maxima_52s_ticker_data}")
print(f"- {caminho_tbl_resumo_maxima_52s_ticker_data}")
print(f"- {caminho_tbl_cobertura_diaria_maxima_52s}")
print(f"- {caminho_tbl_auditoria_maxima_52s}")

print("\nETAPA 5.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 5.3 - CÁLCULO DAS MÁXIMAS DE 52 SEMANAS

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidada        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Saída   - base máxima 52 semanas por ticker : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_3_base_maxima_52s_ticker_data.parquet
Saída   - resumo                            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_3_tbl_resumo_maxima_52s_ticker_data.parquet
Saída   - cobertura diária                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5

## Etapa 5.4) Flags Diárias por Ticker

In [23]:
%%time
# ============================================================
# Etapa 5.4) Flags Diárias por Ticker
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 5.4 - FLAGS DIÁRIAS POR TICKER")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mm200_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=1, tipo_arquivo="base", nome="mm200_ticker_data")
caminho_base_minima_52s_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=2, tipo_arquivo="base", nome="minima_52s_ticker_data")
caminho_base_maxima_52s_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=3, tipo_arquivo="base", nome="maxima_52s_ticker_data")

caminho_base_flags_amplitude_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=4, tipo_arquivo="base", nome="flags_amplitude_ticker_data")
caminho_tbl_resumo_flags_amplitude_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=4, tipo_arquivo="tbl", nome="resumo_flags_amplitude_ticker_data")
caminho_tbl_cobertura_diaria_flags_amplitude = gerar_caminho_arquivo(etapa=5, subetapa=4, tipo_arquivo="tbl", nome="cobertura_diaria_flags_amplitude")
caminho_tbl_auditoria_flags_amplitude = gerar_caminho_arquivo(etapa=5, subetapa=4, tipo_arquivo="tbl", nome="auditoria_flags_amplitude")
caminho_tbl_inconsistencias_flags_amplitude = gerar_caminho_arquivo(etapa=5, subetapa=4, tipo_arquivo="tbl", nome="inconsistencias_flags_amplitude")
caminho_tbl_auditoria_janelas_52s_sem_amplitude = gerar_caminho_arquivo(etapa=5, subetapa=4, tipo_arquivo="tbl", nome="auditoria_janelas_52s_sem_amplitude")

print(f"Entrada - base MM200                    : {caminho_base_mm200_ticker_data}")
print(f"Entrada - base mínima 52 semanas        : {caminho_base_minima_52s_ticker_data}")
print(f"Entrada - base máxima 52 semanas        : {caminho_base_maxima_52s_ticker_data}")
print(f"Saída   - base flags amplitude          : {caminho_base_flags_amplitude_ticker_data}")
print(f"Saída   - resumo                        : {caminho_tbl_resumo_flags_amplitude_ticker_data}")
print(f"Saída   - cobertura diária              : {caminho_tbl_cobertura_diaria_flags_amplitude}")
print(f"Saída   - auditoria                     : {caminho_tbl_auditoria_flags_amplitude}")
print(f"Saída   - inconsistências               : {caminho_tbl_inconsistencias_flags_amplitude}")
print(f"Saída   - auditoria janelas 52s         : {caminho_tbl_auditoria_janelas_52s_sem_amplitude}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_mm200_ticker_data = pd.read_parquet(caminho_base_mm200_ticker_data)
df_minima_52s_ticker_data = pd.read_parquet(caminho_base_minima_52s_ticker_data)
df_maxima_52s_ticker_data = pd.read_parquet(caminho_base_maxima_52s_ticker_data)

print(f"Base MM200             : {df_mm200_ticker_data.shape[0]:,} linhas x {df_mm200_ticker_data.shape[1]} colunas")
print(f"Base mínima 52 semanas : {df_minima_52s_ticker_data.shape[0]:,} linhas x {df_minima_52s_ticker_data.shape[1]} colunas")
print(f"Base máxima 52 semanas : {df_maxima_52s_ticker_data.shape[0]:,} linhas x {df_maxima_52s_ticker_data.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e colunas obrigatórias
# ============================================================

print("\n[4/10] Funções auxiliares e colunas obrigatórias...")

COLUNAS_CHAVE = ["ticker", "data"]

COLUNAS_OBRIGATORIAS_MM200 = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj",
    "n_obs_validas_mm200",
    "mm200",
    "flag_mm200_valida",
]

COLUNAS_OBRIGATORIAS_MINIMA = [
    "ticker",
    "data",
    "n_obs_validas_minima_52s",
    "minima_52s",
    "flag_minima_52s_valida",
    "flag_em_minima_52s",
]

COLUNAS_OBRIGATORIAS_MAXIMA = [
    "ticker",
    "data",
    "n_obs_validas_maxima_52s",
    "maxima_52s",
    "flag_maxima_52s_valida",
    "flag_em_maxima_52s",
]

COLUNAS_BASE_SAIDA = [
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "data",
    "close_adj",
    "n_obs_validas_mm200",
    "mm200",
    "flag_mm200_valida",
    "flag_acima_mm200",
    "flag_abaixo_mm200",
    "flag_na_mm200",
    "n_obs_validas_minima_52s",
    "minima_52s",
    "flag_minima_52s_valida",
    "flag_em_minima_52s",
    "n_obs_validas_maxima_52s",
    "maxima_52s",
    "flag_maxima_52s_valida",
    "flag_em_maxima_52s",
    "flag_janela_52s_com_amplitude_valida",
    "flag_extremos_52s_validos_sem_conflito",
    "flag_amplitude_completa_ticker_data",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def validar_duplicidades_chave(df, nome_base):
    """
    Valida duplicidades na chave ticker-data.
    """
    n_duplicidades = int(df.duplicated(subset=COLUNAS_CHAVE).sum())
    if n_duplicidades > 0:
        raise ValueError(f"Foram encontradas {n_duplicidades} duplicidades na chave ticker-data da base {nome_base}.")
    return n_duplicidades

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Preparação e validação estrutural das bases de indicadores
# ============================================================

print("\n[5/10] Preparação e validação estrutural das bases de indicadores...")

validar_colunas_obrigatorias(df_mm200_ticker_data, COLUNAS_OBRIGATORIAS_MM200)
validar_colunas_obrigatorias(df_minima_52s_ticker_data, COLUNAS_OBRIGATORIAS_MINIMA)
validar_colunas_obrigatorias(df_maxima_52s_ticker_data, COLUNAS_OBRIGATORIAS_MAXIMA)

for df_base in [df_mm200_ticker_data, df_minima_52s_ticker_data, df_maxima_52s_ticker_data]:
    df_base["ticker"] = df_base["ticker"].astype("string").str.strip().str.upper()
    df_base["data"] = pd.to_datetime(df_base["data"], errors="coerce")

if "issuer_code" in df_mm200_ticker_data.columns:
    df_mm200_ticker_data["issuer_code"] = df_mm200_ticker_data["issuer_code"].astype("string").str.strip()

for coluna_texto in ["nome", "setor", "subsetor", "segmento"]:
    df_mm200_ticker_data[coluna_texto] = df_mm200_ticker_data[coluna_texto].astype("string")

for coluna_numerica in [
    "close_adj",
    "n_obs_validas_mm200",
    "mm200",
    "n_obs_validas_minima_52s",
    "minima_52s",
    "n_obs_validas_maxima_52s",
    "maxima_52s",
]:
    if coluna_numerica in df_mm200_ticker_data.columns:
        df_mm200_ticker_data[coluna_numerica] = pd.to_numeric(df_mm200_ticker_data[coluna_numerica], errors="coerce")
    if coluna_numerica in df_minima_52s_ticker_data.columns:
        df_minima_52s_ticker_data[coluna_numerica] = pd.to_numeric(df_minima_52s_ticker_data[coluna_numerica], errors="coerce")
    if coluna_numerica in df_maxima_52s_ticker_data.columns:
        df_maxima_52s_ticker_data[coluna_numerica] = pd.to_numeric(df_maxima_52s_ticker_data[coluna_numerica], errors="coerce")

for coluna_flag in [
    "flag_mm200_valida",
    "flag_minima_52s_valida",
    "flag_em_minima_52s",
    "flag_maxima_52s_valida",
    "flag_em_maxima_52s",
]:
    if coluna_flag in df_mm200_ticker_data.columns:
        df_mm200_ticker_data[coluna_flag] = df_mm200_ticker_data[coluna_flag].fillna(False).astype(bool)
    if coluna_flag in df_minima_52s_ticker_data.columns:
        df_minima_52s_ticker_data[coluna_flag] = df_minima_52s_ticker_data[coluna_flag].fillna(False).astype(bool)
    if coluna_flag in df_maxima_52s_ticker_data.columns:
        df_maxima_52s_ticker_data[coluna_flag] = df_maxima_52s_ticker_data[coluna_flag].fillna(False).astype(bool)

n_duplicidades_mm200 = validar_duplicidades_chave(df_mm200_ticker_data, "MM200")
n_duplicidades_minima = validar_duplicidades_chave(df_minima_52s_ticker_data, "mínima_52s")
n_duplicidades_maxima = validar_duplicidades_chave(df_maxima_52s_ticker_data, "máxima_52s")

print(f"Duplicidades na base MM200             : {n_duplicidades_mm200}")
print(f"Duplicidades na base mínima 52 semanas : {n_duplicidades_minima}")
print(f"Duplicidades na base máxima 52 semanas : {n_duplicidades_maxima}")
print("OK")

# ============================================================
# 6) Consolidação da base diária de flags por ticker
# ============================================================

print("\n[6/10] Consolidação da base diária de flags por ticker...")

df_flags_amplitude_ticker_data = (
    df_mm200_ticker_data
    .merge(
        df_minima_52s_ticker_data[
            [
                "ticker",
                "data",
                "n_obs_validas_minima_52s",
                "minima_52s",
                "flag_minima_52s_valida",
                "flag_em_minima_52s",
            ]
        ],
        on=["ticker", "data"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        df_maxima_52s_ticker_data[
            [
                "ticker",
                "data",
                "n_obs_validas_maxima_52s",
                "maxima_52s",
                "flag_maxima_52s_valida",
                "flag_em_maxima_52s",
            ]
        ],
        on=["ticker", "data"],
        how="left",
        validate="one_to_one",
    )
)

df_flags_amplitude_ticker_data["flag_minima_52s_valida"] = df_flags_amplitude_ticker_data["flag_minima_52s_valida"].fillna(False).astype(bool)
df_flags_amplitude_ticker_data["flag_em_minima_52s"] = df_flags_amplitude_ticker_data["flag_em_minima_52s"].fillna(False).astype(bool)
df_flags_amplitude_ticker_data["flag_maxima_52s_valida"] = df_flags_amplitude_ticker_data["flag_maxima_52s_valida"].fillna(False).astype(bool)
df_flags_amplitude_ticker_data["flag_em_maxima_52s"] = df_flags_amplitude_ticker_data["flag_em_maxima_52s"].fillna(False).astype(bool)

df_flags_amplitude_ticker_data["flag_em_minima_52s_original"] = df_flags_amplitude_ticker_data["flag_em_minima_52s"].copy()
df_flags_amplitude_ticker_data["flag_em_maxima_52s_original"] = df_flags_amplitude_ticker_data["flag_em_maxima_52s"].copy()
df_flags_amplitude_ticker_data["flag_minima_52s_valida_original"] = df_flags_amplitude_ticker_data["flag_minima_52s_valida"].copy()
df_flags_amplitude_ticker_data["flag_maxima_52s_valida_original"] = df_flags_amplitude_ticker_data["flag_maxima_52s_valida"].copy()

df_flags_amplitude_ticker_data["flag_janela_52s_com_amplitude_valida"] = (
    df_flags_amplitude_ticker_data["flag_minima_52s_valida"]
    & df_flags_amplitude_ticker_data["flag_maxima_52s_valida"]
    & df_flags_amplitude_ticker_data["minima_52s"].notna()
    & df_flags_amplitude_ticker_data["maxima_52s"].notna()
    & df_flags_amplitude_ticker_data["maxima_52s"].gt(df_flags_amplitude_ticker_data["minima_52s"])
)

mask_janela_52s_sem_amplitude = (
    df_flags_amplitude_ticker_data["flag_minima_52s_valida"]
    & df_flags_amplitude_ticker_data["flag_maxima_52s_valida"]
    & df_flags_amplitude_ticker_data["minima_52s"].notna()
    & df_flags_amplitude_ticker_data["maxima_52s"].notna()
    & df_flags_amplitude_ticker_data["maxima_52s"].le(df_flags_amplitude_ticker_data["minima_52s"])
)

n_janelas_52s_sem_amplitude = int(mask_janela_52s_sem_amplitude.sum())
n_flags_minima_52s_desativadas_por_janela_sem_amplitude = int(
    (mask_janela_52s_sem_amplitude & df_flags_amplitude_ticker_data["flag_em_minima_52s"]).sum()
)
n_flags_maxima_52s_desativadas_por_janela_sem_amplitude = int(
    (mask_janela_52s_sem_amplitude & df_flags_amplitude_ticker_data["flag_em_maxima_52s"]).sum()
)
n_inconsistencia_minima_maxima_52s_original = int(
    (
        df_flags_amplitude_ticker_data["flag_em_minima_52s_original"]
        & df_flags_amplitude_ticker_data["flag_em_maxima_52s_original"]
    ).sum()
)

df_auditoria_janelas_52s_sem_amplitude = (
    df_flags_amplitude_ticker_data
    .loc[
        mask_janela_52s_sem_amplitude,
        [
            "ticker",
            "issuer_code",
            "nome",
            "setor",
            "subsetor",
            "segmento",
            "data",
            "close_adj",
            "n_obs_validas_minima_52s",
            "minima_52s",
            "flag_minima_52s_valida_original",
            "flag_em_minima_52s_original",
            "n_obs_validas_maxima_52s",
            "maxima_52s",
            "flag_maxima_52s_valida_original",
            "flag_em_maxima_52s_original",
        ],
    ]
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

df_flags_amplitude_ticker_data.loc[
    mask_janela_52s_sem_amplitude,
    ["flag_minima_52s_valida", "flag_maxima_52s_valida", "flag_em_minima_52s", "flag_em_maxima_52s"],
] = False

df_flags_amplitude_ticker_data["flag_extremos_52s_validos_sem_conflito"] = (
    df_flags_amplitude_ticker_data["flag_janela_52s_com_amplitude_valida"]
    & ~(
        df_flags_amplitude_ticker_data["flag_em_minima_52s"]
        & df_flags_amplitude_ticker_data["flag_em_maxima_52s"]
    )
)

print(f"Janelas de 52 semanas sem amplitude efetiva tratadas: {n_janelas_52s_sem_amplitude:,}")
print(f"Flags de mínima desativadas por janela sem amplitude: {n_flags_minima_52s_desativadas_por_janela_sem_amplitude:,}")
print(f"Flags de máxima desativadas por janela sem amplitude: {n_flags_maxima_52s_desativadas_por_janela_sem_amplitude:,}")

df_flags_amplitude_ticker_data["flag_acima_mm200"] = (
    df_flags_amplitude_ticker_data["flag_mm200_valida"]
    & df_flags_amplitude_ticker_data["close_adj"].gt(df_flags_amplitude_ticker_data["mm200"])
)

df_flags_amplitude_ticker_data["flag_abaixo_mm200"] = (
    df_flags_amplitude_ticker_data["flag_mm200_valida"]
    & df_flags_amplitude_ticker_data["close_adj"].lt(df_flags_amplitude_ticker_data["mm200"])
)

df_flags_amplitude_ticker_data["flag_na_mm200"] = (
    df_flags_amplitude_ticker_data["flag_mm200_valida"]
    & df_flags_amplitude_ticker_data["close_adj"].eq(df_flags_amplitude_ticker_data["mm200"])
)

df_flags_amplitude_ticker_data["flag_amplitude_completa_ticker_data"] = (
    df_flags_amplitude_ticker_data["flag_mm200_valida"]
    & df_flags_amplitude_ticker_data["flag_minima_52s_valida"]
    & df_flags_amplitude_ticker_data["flag_maxima_52s_valida"]
    & df_flags_amplitude_ticker_data["flag_janela_52s_com_amplitude_valida"]
)

df_flags_amplitude_ticker_data = (
    df_flags_amplitude_ticker_data[COLUNAS_BASE_SAIDA]
    .sort_values(["ticker", "data"])
    .reset_index(drop=True)
)

n_inconsistencia_acima_abaixo = int(
    (df_flags_amplitude_ticker_data["flag_acima_mm200"] & df_flags_amplitude_ticker_data["flag_abaixo_mm200"]).sum()
)
n_inconsistencia_minima_maxima = int(
    (df_flags_amplitude_ticker_data["flag_em_minima_52s"] & df_flags_amplitude_ticker_data["flag_em_maxima_52s"]).sum()
)

print(f"Base consolidada de flags amplitude: {df_flags_amplitude_ticker_data.shape[0]:,} linhas x {df_flags_amplitude_ticker_data.shape[1]} colunas")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

n_linhas_total = int(len(df_flags_amplitude_ticker_data))
n_tickers_total = int(df_flags_amplitude_ticker_data["ticker"].nunique(dropna=True))
n_datas_total = int(df_flags_amplitude_ticker_data["data"].nunique(dropna=True))
n_linhas_mm200_valida = int(df_flags_amplitude_ticker_data["flag_mm200_valida"].sum())
n_linhas_minima_52s_valida = int(df_flags_amplitude_ticker_data["flag_minima_52s_valida"].sum())
n_linhas_maxima_52s_valida = int(df_flags_amplitude_ticker_data["flag_maxima_52s_valida"].sum())
n_linhas_acima_mm200 = int(df_flags_amplitude_ticker_data["flag_acima_mm200"].sum())
n_linhas_abaixo_mm200 = int(df_flags_amplitude_ticker_data["flag_abaixo_mm200"].sum())
n_linhas_na_mm200 = int(df_flags_amplitude_ticker_data["flag_na_mm200"].sum())
n_linhas_em_minima_52s = int(df_flags_amplitude_ticker_data["flag_em_minima_52s"].sum())
n_linhas_em_maxima_52s = int(df_flags_amplitude_ticker_data["flag_em_maxima_52s"].sum())
n_linhas_janela_52s_com_amplitude_valida = int(df_flags_amplitude_ticker_data["flag_janela_52s_com_amplitude_valida"].sum())
n_linhas_extremos_52s_validos_sem_conflito = int(df_flags_amplitude_ticker_data["flag_extremos_52s_validos_sem_conflito"].sum())
n_linhas_amplitude_completa = int(df_flags_amplitude_ticker_data["flag_amplitude_completa_ticker_data"].sum())

df_resumo_flags_amplitude_ticker_data = pd.DataFrame(
    [
        {
            "base": "flags_amplitude_ticker_data",
            "n_linhas": n_linhas_total,
            "n_tickers": n_tickers_total,
            "n_datas": n_datas_total,
            "data_min": df_flags_amplitude_ticker_data["data"].min(),
            "data_max": df_flags_amplitude_ticker_data["data"].max(),
            "n_linhas_mm200_valida": n_linhas_mm200_valida,
            "n_linhas_minima_52s_valida": n_linhas_minima_52s_valida,
            "n_linhas_maxima_52s_valida": n_linhas_maxima_52s_valida,
            "n_linhas_acima_mm200": n_linhas_acima_mm200,
            "n_linhas_abaixo_mm200": n_linhas_abaixo_mm200,
            "n_linhas_na_mm200": n_linhas_na_mm200,
            "n_linhas_em_minima_52s": n_linhas_em_minima_52s,
            "n_linhas_em_maxima_52s": n_linhas_em_maxima_52s,
            "n_linhas_janela_52s_com_amplitude_valida": n_linhas_janela_52s_com_amplitude_valida,
            "n_linhas_extremos_52s_validos_sem_conflito": n_linhas_extremos_52s_validos_sem_conflito,
            "n_linhas_amplitude_completa": n_linhas_amplitude_completa,
        }
    ]
)

df_cobertura_diaria_flags_amplitude = (
    df_flags_amplitude_ticker_data
    .groupby("data", as_index=False)
    .agg(
        n_tickers_total=("ticker", "nunique"),
        n_tickers_mm200_valida=("flag_mm200_valida", "sum"),
        n_tickers_acima_mm200=("flag_acima_mm200", "sum"),
        n_tickers_abaixo_mm200=("flag_abaixo_mm200", "sum"),
        n_tickers_na_mm200=("flag_na_mm200", "sum"),
        n_tickers_em_minima_52s=("flag_em_minima_52s", "sum"),
        n_tickers_em_maxima_52s=("flag_em_maxima_52s", "sum"),
        n_tickers_janela_52s_com_amplitude_valida=("flag_janela_52s_com_amplitude_valida", "sum"),
        n_tickers_extremos_52s_validos_sem_conflito=("flag_extremos_52s_validos_sem_conflito", "sum"),
        n_tickers_amplitude_completa=("flag_amplitude_completa_ticker_data", "sum"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_cobertura_diaria_flags_amplitude["pct_tickers_acima_mm200"] = (
    df_cobertura_diaria_flags_amplitude["n_tickers_acima_mm200"] / df_cobertura_diaria_flags_amplitude["n_tickers_total"]
)
df_cobertura_diaria_flags_amplitude["pct_tickers_abaixo_mm200"] = (
    df_cobertura_diaria_flags_amplitude["n_tickers_abaixo_mm200"] / df_cobertura_diaria_flags_amplitude["n_tickers_total"]
)
df_cobertura_diaria_flags_amplitude["pct_tickers_em_minima_52s"] = (
    df_cobertura_diaria_flags_amplitude["n_tickers_em_minima_52s"] / df_cobertura_diaria_flags_amplitude["n_tickers_total"]
)
df_cobertura_diaria_flags_amplitude["pct_tickers_em_maxima_52s"] = (
    df_cobertura_diaria_flags_amplitude["n_tickers_em_maxima_52s"] / df_cobertura_diaria_flags_amplitude["n_tickers_total"]
)

df_auditoria_flags_amplitude = pd.DataFrame(
    [
        {"metrica": "n_linhas_total", "valor": n_linhas_total},
        {"metrica": "n_tickers_total", "valor": n_tickers_total},
        {"metrica": "n_datas_total", "valor": n_datas_total},
        {"metrica": "n_linhas_mm200_valida", "valor": n_linhas_mm200_valida},
        {"metrica": "n_linhas_minima_52s_valida", "valor": n_linhas_minima_52s_valida},
        {"metrica": "n_linhas_maxima_52s_valida", "valor": n_linhas_maxima_52s_valida},
        {"metrica": "n_linhas_acima_mm200", "valor": n_linhas_acima_mm200},
        {"metrica": "n_linhas_abaixo_mm200", "valor": n_linhas_abaixo_mm200},
        {"metrica": "n_linhas_na_mm200", "valor": n_linhas_na_mm200},
        {"metrica": "n_linhas_em_minima_52s", "valor": n_linhas_em_minima_52s},
        {"metrica": "n_linhas_em_maxima_52s", "valor": n_linhas_em_maxima_52s},
        {"metrica": "n_linhas_janela_52s_com_amplitude_valida", "valor": n_linhas_janela_52s_com_amplitude_valida},
        {"metrica": "n_linhas_extremos_52s_validos_sem_conflito", "valor": n_linhas_extremos_52s_validos_sem_conflito},
        {"metrica": "n_linhas_amplitude_completa", "valor": n_linhas_amplitude_completa},
        {"metrica": "n_janelas_52s_sem_amplitude", "valor": n_janelas_52s_sem_amplitude},
        {"metrica": "n_flags_minima_52s_desativadas_por_janela_sem_amplitude", "valor": n_flags_minima_52s_desativadas_por_janela_sem_amplitude},
        {"metrica": "n_flags_maxima_52s_desativadas_por_janela_sem_amplitude", "valor": n_flags_maxima_52s_desativadas_por_janela_sem_amplitude},
        {"metrica": "n_inconsistencia_acima_abaixo_mm200", "valor": n_inconsistencia_acima_abaixo},
        {"metrica": "n_inconsistencia_minima_maxima_52s_original", "valor": n_inconsistencia_minima_maxima_52s_original},
        {"metrica": "n_inconsistencia_minima_maxima_52s", "valor": n_inconsistencia_minima_maxima},
    ]
)

df_inconsistencias_flags_amplitude = pd.DataFrame(
    [
        {
            "tipo_inconsistencia": "acima_e_abaixo_mm200_simultaneamente",
            "n_linhas": n_inconsistencia_acima_abaixo,
        },
        {
            "tipo_inconsistencia": "minima_e_maxima_52s_simultaneamente",
            "n_linhas": n_inconsistencia_minima_maxima,
        },
    ]
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_flags_amplitude_ticker_data, caminho_base_flags_amplitude_ticker_data, index=False)
salvar_dataframe(df_resumo_flags_amplitude_ticker_data, caminho_tbl_resumo_flags_amplitude_ticker_data, index=False)
salvar_dataframe(df_cobertura_diaria_flags_amplitude, caminho_tbl_cobertura_diaria_flags_amplitude, index=False)
salvar_dataframe(df_auditoria_flags_amplitude, caminho_tbl_auditoria_flags_amplitude, index=False)
salvar_dataframe(df_inconsistencias_flags_amplitude, caminho_tbl_inconsistencias_flags_amplitude, index=False)
salvar_dataframe(df_auditoria_janelas_52s_sem_amplitude, caminho_tbl_auditoria_janelas_52s_sem_amplitude, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_flags_amplitude = df_flags_amplitude_ticker_data.head(15).copy()
df_amostra_flags_ativas = (
    df_flags_amplitude_ticker_data
    .loc[
        df_flags_amplitude_ticker_data["flag_acima_mm200"]
        | df_flags_amplitude_ticker_data["flag_abaixo_mm200"]
        | df_flags_amplitude_ticker_data["flag_em_minima_52s"]
        | df_flags_amplitude_ticker_data["flag_em_maxima_52s"]
    ]
    .head(15)
    .copy()
)
df_amostra_cobertura_diaria_flags_amplitude = df_cobertura_diaria_flags_amplitude.head(15).copy()
df_amostra_janelas_52s_sem_amplitude = df_auditoria_janelas_52s_sem_amplitude.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da base diária de flags por ticker:")
print(df_resumo_flags_amplitude_ticker_data.to_string(index=False))

print("\nAuditoria das flags de amplitude:")
print(df_auditoria_flags_amplitude.to_string(index=False))

print("\nInconsistências estruturais das flags de amplitude:")
print(df_inconsistencias_flags_amplitude.to_string(index=False))

print("\nAuditoria de janelas de 52 semanas sem amplitude efetiva - amostra:")
print(df_amostra_janelas_52s_sem_amplitude.to_string(index=False))

print("\nAmostra da base diária de flags por ticker:")
print(df_amostra_flags_amplitude.to_string(index=False))

print("\nAmostra de linhas com flags ativas:")
print(df_amostra_flags_ativas.to_string(index=False))

print("\nCobertura diária das flags de amplitude - amostra:")
print(df_amostra_cobertura_diaria_flags_amplitude.to_string(index=False))

print("\nArquivos salvos na subetapa 5.4:")
print(f"- {caminho_base_flags_amplitude_ticker_data}")
print(f"- {caminho_tbl_resumo_flags_amplitude_ticker_data}")
print(f"- {caminho_tbl_cobertura_diaria_flags_amplitude}")
print(f"- {caminho_tbl_auditoria_flags_amplitude}")
print(f"- {caminho_tbl_inconsistencias_flags_amplitude}")
print(f"- {caminho_tbl_auditoria_janelas_52s_sem_amplitude}")

print("\nETAPA 5.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 5.4 - FLAGS DIÁRIAS POR TICKER

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - base MM200                    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_1_base_mm200_ticker_data.parquet
Entrada - base mínima 52 semanas        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_2_base_minima_52s_ticker_data.parquet
Entrada - base máxima 52 semanas        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_3_base_maxima_52s_ticker_data.parquet
Saída   - base flags amplitude          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_4_base_flags_amplitude_ticker_data.parq

## Etapa 5.5) Indicadores Agregados de Amplitude por Pregão

In [24]:
%%time
# ============================================================
# Etapa 5.5) Indicadores Agregados de Amplitude por Pregão
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 5.5 - INDICADORES AGREGADOS DE AMPLITUDE POR PREGÃO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_elegibilidade_operacional_data = gerar_caminho_arquivo(etapa=3, subetapa=4, tipo_arquivo="base", nome="elegibilidade_operacional_data")
caminho_base_flags_amplitude_ticker_data = gerar_caminho_arquivo(etapa=5, subetapa=4, tipo_arquivo="base", nome="flags_amplitude_ticker_data")

caminho_base_indicadores_agregados_amplitude_pregao = gerar_caminho_arquivo(etapa=5, subetapa=5, tipo_arquivo="base", nome="indicadores_agregados_amplitude_pregao")
caminho_tbl_resumo_indicadores_agregados_amplitude_pregao = gerar_caminho_arquivo(etapa=5, subetapa=5, tipo_arquivo="tbl", nome="resumo_indicadores_agregados_amplitude_pregao")
caminho_tbl_cobertura_indicadores_agregados_amplitude_pregao = gerar_caminho_arquivo(etapa=5, subetapa=5, tipo_arquivo="tbl", nome="cobertura_indicadores_agregados_amplitude_pregao")
caminho_tbl_auditoria_indicadores_agregados_amplitude_pregao = gerar_caminho_arquivo(etapa=5, subetapa=5, tipo_arquivo="tbl", nome="auditoria_indicadores_agregados_amplitude_pregao")
caminho_tbl_datas_cobertura_insuficiente_amplitude_pregao = gerar_caminho_arquivo(etapa=5, subetapa=5, tipo_arquivo="tbl", nome="datas_cobertura_insuficiente_amplitude_pregao")

print(f"Entrada - elegibilidade operacional por data : {caminho_base_elegibilidade_operacional_data}")
print(f"Entrada - flags amplitude por ticker-data    : {caminho_base_flags_amplitude_ticker_data}")
print(f"Saída   - base breadth por pregão            : {caminho_base_indicadores_agregados_amplitude_pregao}")
print(f"Saída   - resumo                             : {caminho_tbl_resumo_indicadores_agregados_amplitude_pregao}")
print(f"Saída   - cobertura                          : {caminho_tbl_cobertura_indicadores_agregados_amplitude_pregao}")
print(f"Saída   - auditoria                          : {caminho_tbl_auditoria_indicadores_agregados_amplitude_pregao}")
print(f"Saída   - datas com cobertura insuficiente   : {caminho_tbl_datas_cobertura_insuficiente_amplitude_pregao}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_elegibilidade_operacional_data = pd.read_parquet(caminho_base_elegibilidade_operacional_data)
df_flags_amplitude_ticker_data = pd.read_parquet(caminho_base_flags_amplitude_ticker_data)

print(f"Elegibilidade operacional por data : {df_elegibilidade_operacional_data.shape[0]:,} linhas x {df_elegibilidade_operacional_data.shape[1]} colunas")
print(f"Flags amplitude por ticker-data    : {df_flags_amplitude_ticker_data.shape[0]:,} linhas x {df_flags_amplitude_ticker_data.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e premissas operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e premissas operacionais...")

COLUNAS_OBRIGATORIAS_OPERACIONAL = [
    "ticker",
    "issuer_code",
    "data",
    "flag_elegivel_operacional_final",
]

COLUNAS_OBRIGATORIAS_FLAGS = [
    "ticker",
    "issuer_code",
    "data",
    "flag_mm200_valida",
    "flag_acima_mm200",
    "flag_abaixo_mm200",
    "flag_na_mm200",
    "flag_minima_52s_valida",
    "flag_em_minima_52s",
    "flag_maxima_52s_valida",
    "flag_em_maxima_52s",
    "flag_amplitude_completa_ticker_data",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def validar_duplicidades_ticker_data(df, nome_base):
    """
    Valida duplicidades na chave ticker-data.
    """
    n_duplicidades = int(df.duplicated(subset=["ticker", "data"]).sum())
    if n_duplicidades > 0:
        raise ValueError(f"Foram encontradas {n_duplicidades} duplicidades na chave ticker-data da base {nome_base}.")
    return n_duplicidades

print("Premissa operacional adotada para os percentuais:")
print("- O denominador de cada indicador será a base válida do próprio indicador dentro do universo elegível operacional.")
print("- Isso evita distorções artificiais no início da série por insuficiência de janela histórica.")
print("OK")

# ============================================================
# 5) Preparação das bases e definição do universo operacional
# ============================================================

print("\n[5/10] Preparação das bases e definição do universo operacional...")

validar_colunas_obrigatorias(df_elegibilidade_operacional_data, COLUNAS_OBRIGATORIAS_OPERACIONAL)
validar_colunas_obrigatorias(df_flags_amplitude_ticker_data, COLUNAS_OBRIGATORIAS_FLAGS)

df_operacional = df_elegibilidade_operacional_data.copy()
df_flags = df_flags_amplitude_ticker_data.copy()

for df_base in [df_operacional, df_flags]:
    df_base["ticker"] = df_base["ticker"].astype("string").str.strip().str.upper()
    df_base["data"] = pd.to_datetime(df_base["data"], errors="coerce")

if "issuer_code" in df_operacional.columns:
    df_operacional["issuer_code"] = df_operacional["issuer_code"].astype("string").str.strip()
if "issuer_code" in df_flags.columns:
    df_flags["issuer_code"] = df_flags["issuer_code"].astype("string").str.strip()

for coluna_flag in [
    "flag_elegivel_operacional_final",
    "flag_mm200_valida",
    "flag_acima_mm200",
    "flag_abaixo_mm200",
    "flag_na_mm200",
    "flag_minima_52s_valida",
    "flag_em_minima_52s",
    "flag_maxima_52s_valida",
    "flag_em_maxima_52s",
    "flag_amplitude_completa_ticker_data",
]:
    if coluna_flag in df_operacional.columns:
        df_operacional[coluna_flag] = df_operacional[coluna_flag].fillna(False).astype(bool)
    if coluna_flag in df_flags.columns:
        df_flags[coluna_flag] = df_flags[coluna_flag].fillna(False).astype(bool)

n_duplicidades_operacional = validar_duplicidades_ticker_data(df_operacional, "elegibilidade_operacional_data")
n_duplicidades_flags = validar_duplicidades_ticker_data(df_flags, "flags_amplitude_ticker_data")

df_universo_operacional = df_operacional.loc[df_operacional["flag_elegivel_operacional_final"]].copy()

print(f"Duplicidades na base operacional         : {n_duplicidades_operacional}")
print(f"Duplicidades na base de flags amplitude  : {n_duplicidades_flags}")
print(f"Linhas do universo elegível operacional  : {len(df_universo_operacional):,}")
print("OK")

# ============================================================
# 6) Cruzamento com a base de flags e agregação diária
# ============================================================

print("\n[6/10] Cruzamento com a base de flags e agregação diária...")

df_base_breadth = df_universo_operacional.merge(
    df_flags[
        [
            "ticker",
            "issuer_code",
            "data",
            "flag_mm200_valida",
            "flag_acima_mm200",
            "flag_abaixo_mm200",
            "flag_na_mm200",
            "flag_minima_52s_valida",
            "flag_em_minima_52s",
            "flag_maxima_52s_valida",
            "flag_em_maxima_52s",
            "flag_amplitude_completa_ticker_data",
        ]
    ],
    on=["ticker", "data"],
    how="left",
    validate="one_to_one",
    suffixes=("_operacional", "_flags"),
)

df_base_breadth["issuer_code"] = df_base_breadth["issuer_code_operacional"].fillna(df_base_breadth["issuer_code_flags"])

n_linhas_sem_match_flags = int(df_base_breadth["flag_mm200_valida"].isna().sum())
if n_linhas_sem_match_flags > 0:
    raise ValueError(f"Foram encontradas {n_linhas_sem_match_flags} linhas do universo operacional sem correspondência na base de flags amplitude.")

for coluna_flag in [
    "flag_mm200_valida",
    "flag_acima_mm200",
    "flag_abaixo_mm200",
    "flag_na_mm200",
    "flag_minima_52s_valida",
    "flag_em_minima_52s",
    "flag_maxima_52s_valida",
    "flag_em_maxima_52s",
    "flag_amplitude_completa_ticker_data",
]:
    df_base_breadth[coluna_flag] = df_base_breadth[coluna_flag].fillna(False).astype(bool)

df_indicadores_agregados_amplitude_pregao = (
    df_base_breadth
    .groupby("data", as_index=False)
    .agg(
        n_tickers_elegiveis_operacionais=("ticker", "nunique"),
        n_empresas_elegiveis_operacionais=("issuer_code", "nunique"),
        n_tickers_mm200_validos=("flag_mm200_valida", "sum"),
        n_tickers_acima_mm200=("flag_acima_mm200", "sum"),
        n_tickers_abaixo_mm200=("flag_abaixo_mm200", "sum"),
        n_tickers_na_mm200=("flag_na_mm200", "sum"),
        n_tickers_minima_52s_validos=("flag_minima_52s_valida", "sum"),
        n_tickers_em_minima_52s=("flag_em_minima_52s", "sum"),
        n_tickers_maxima_52s_validos=("flag_maxima_52s_valida", "sum"),
        n_tickers_em_maxima_52s=("flag_em_maxima_52s", "sum"),
        n_tickers_amplitude_completa=("flag_amplitude_completa_ticker_data", "sum"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_indicadores_agregados_amplitude_pregao["pct_tickers_acima_mm200"] = (
    df_indicadores_agregados_amplitude_pregao["n_tickers_acima_mm200"] / df_indicadores_agregados_amplitude_pregao["n_tickers_mm200_validos"]
)
df_indicadores_agregados_amplitude_pregao["pct_tickers_abaixo_mm200"] = (
    df_indicadores_agregados_amplitude_pregao["n_tickers_abaixo_mm200"] / df_indicadores_agregados_amplitude_pregao["n_tickers_mm200_validos"]
)
df_indicadores_agregados_amplitude_pregao["pct_tickers_em_minima_52s"] = (
    df_indicadores_agregados_amplitude_pregao["n_tickers_em_minima_52s"] / df_indicadores_agregados_amplitude_pregao["n_tickers_minima_52s_validos"]
)
df_indicadores_agregados_amplitude_pregao["pct_tickers_em_maxima_52s"] = (
    df_indicadores_agregados_amplitude_pregao["n_tickers_em_maxima_52s"] / df_indicadores_agregados_amplitude_pregao["n_tickers_maxima_52s_validos"]
)

df_indicadores_agregados_amplitude_pregao["pct_tickers_acima_mm200_sobre_universo_operacional"] = (
    df_indicadores_agregados_amplitude_pregao["n_tickers_acima_mm200"] / df_indicadores_agregados_amplitude_pregao["n_tickers_elegiveis_operacionais"]
)
df_indicadores_agregados_amplitude_pregao["pct_tickers_abaixo_mm200_sobre_universo_operacional"] = (
    df_indicadores_agregados_amplitude_pregao["n_tickers_abaixo_mm200"] / df_indicadores_agregados_amplitude_pregao["n_tickers_elegiveis_operacionais"]
)
df_indicadores_agregados_amplitude_pregao["pct_tickers_em_minima_52s_sobre_universo_operacional"] = (
    df_indicadores_agregados_amplitude_pregao["n_tickers_em_minima_52s"] / df_indicadores_agregados_amplitude_pregao["n_tickers_elegiveis_operacionais"]
)
df_indicadores_agregados_amplitude_pregao["pct_tickers_em_maxima_52s_sobre_universo_operacional"] = (
    df_indicadores_agregados_amplitude_pregao["n_tickers_em_maxima_52s"] / df_indicadores_agregados_amplitude_pregao["n_tickers_elegiveis_operacionais"]
)

df_indicadores_agregados_amplitude_pregao["saldo_mm200"] = (
    df_indicadores_agregados_amplitude_pregao["pct_tickers_acima_mm200"] - df_indicadores_agregados_amplitude_pregao["pct_tickers_abaixo_mm200"]
)
df_indicadores_agregados_amplitude_pregao["saldo_extremos_52s"] = (
    df_indicadores_agregados_amplitude_pregao["pct_tickers_em_maxima_52s"] - df_indicadores_agregados_amplitude_pregao["pct_tickers_em_minima_52s"]
)

print(f"Série agregada final de breadth: {df_indicadores_agregados_amplitude_pregao.shape[0]:,} linhas x {df_indicadores_agregados_amplitude_pregao.shape[1]} colunas")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo, cobertura e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo, cobertura e auditoria...")

n_datas_total = int(len(df_indicadores_agregados_amplitude_pregao))
n_datas_mm200_com_denominador_valido = int(df_indicadores_agregados_amplitude_pregao["n_tickers_mm200_validos"].gt(0).sum())
n_datas_minima_52s_com_denominador_valido = int(df_indicadores_agregados_amplitude_pregao["n_tickers_minima_52s_validos"].gt(0).sum())
n_datas_maxima_52s_com_denominador_valido = int(df_indicadores_agregados_amplitude_pregao["n_tickers_maxima_52s_validos"].gt(0).sum())
n_datas_amplitude_completa = int(df_indicadores_agregados_amplitude_pregao["n_tickers_amplitude_completa"].gt(0).sum())

df_resumo_indicadores_agregados_amplitude_pregao = pd.DataFrame(
    [
        {
            "base": "indicadores_agregados_amplitude_pregao",
            "n_datas": n_datas_total,
            "data_min": df_indicadores_agregados_amplitude_pregao["data"].min(),
            "data_max": df_indicadores_agregados_amplitude_pregao["data"].max(),
            "n_datas_mm200_com_denominador_valido": n_datas_mm200_com_denominador_valido,
            "n_datas_minima_52s_com_denominador_valido": n_datas_minima_52s_com_denominador_valido,
            "n_datas_maxima_52s_com_denominador_valido": n_datas_maxima_52s_com_denominador_valido,
            "n_datas_amplitude_completa": n_datas_amplitude_completa,
            "mediana_tickers_elegiveis_operacionais": float(df_indicadores_agregados_amplitude_pregao["n_tickers_elegiveis_operacionais"].median()),
            "mediana_pct_tickers_acima_mm200": float(df_indicadores_agregados_amplitude_pregao["pct_tickers_acima_mm200"].dropna().median()) if df_indicadores_agregados_amplitude_pregao["pct_tickers_acima_mm200"].notna().any() else pd.NA,
            "mediana_pct_tickers_abaixo_mm200": float(df_indicadores_agregados_amplitude_pregao["pct_tickers_abaixo_mm200"].dropna().median()) if df_indicadores_agregados_amplitude_pregao["pct_tickers_abaixo_mm200"].notna().any() else pd.NA,
            "mediana_pct_tickers_em_minima_52s": float(df_indicadores_agregados_amplitude_pregao["pct_tickers_em_minima_52s"].dropna().median()) if df_indicadores_agregados_amplitude_pregao["pct_tickers_em_minima_52s"].notna().any() else pd.NA,
            "mediana_pct_tickers_em_maxima_52s": float(df_indicadores_agregados_amplitude_pregao["pct_tickers_em_maxima_52s"].dropna().median()) if df_indicadores_agregados_amplitude_pregao["pct_tickers_em_maxima_52s"].notna().any() else pd.NA,
        }
    ]
)

df_cobertura_indicadores_agregados_amplitude_pregao = (
    df_indicadores_agregados_amplitude_pregao[
        [
            "data",
            "n_tickers_elegiveis_operacionais",
            "n_empresas_elegiveis_operacionais",
            "n_tickers_mm200_validos",
            "n_tickers_minima_52s_validos",
            "n_tickers_maxima_52s_validos",
            "n_tickers_amplitude_completa",
        ]
    ]
    .copy()
)

df_auditoria_indicadores_agregados_amplitude_pregao = pd.DataFrame(
    [
        {"metrica": "n_linhas_universo_operacional", "valor": int(len(df_universo_operacional))},
        {"metrica": "n_linhas_flags_amplitude", "valor": int(len(df_flags))},
        {"metrica": "n_linhas_base_merge", "valor": int(len(df_base_breadth))},
        {"metrica": "n_datas_serie_breadth", "valor": n_datas_total},
        {"metrica": "n_linhas_sem_match_flags", "valor": n_linhas_sem_match_flags},
        {"metrica": "n_datas_mm200_com_denominador_valido", "valor": n_datas_mm200_com_denominador_valido},
        {"metrica": "n_datas_minima_52s_com_denominador_valido", "valor": n_datas_minima_52s_com_denominador_valido},
        {"metrica": "n_datas_maxima_52s_com_denominador_valido", "valor": n_datas_maxima_52s_com_denominador_valido},
        {"metrica": "n_datas_amplitude_completa", "valor": n_datas_amplitude_completa},
    ]
)

df_datas_cobertura_insuficiente_amplitude_pregao = (
    df_indicadores_agregados_amplitude_pregao
    .loc[
        (df_indicadores_agregados_amplitude_pregao["n_tickers_mm200_validos"].eq(0))
        | (df_indicadores_agregados_amplitude_pregao["n_tickers_minima_52s_validos"].eq(0))
        | (df_indicadores_agregados_amplitude_pregao["n_tickers_maxima_52s_validos"].eq(0))
    ]
    .copy()
)

print("Tabelas-resumo, cobertura e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_indicadores_agregados_amplitude_pregao, caminho_base_indicadores_agregados_amplitude_pregao, index=False)
salvar_dataframe(df_resumo_indicadores_agregados_amplitude_pregao, caminho_tbl_resumo_indicadores_agregados_amplitude_pregao, index=False)
salvar_dataframe(df_cobertura_indicadores_agregados_amplitude_pregao, caminho_tbl_cobertura_indicadores_agregados_amplitude_pregao, index=False)
salvar_dataframe(df_auditoria_indicadores_agregados_amplitude_pregao, caminho_tbl_auditoria_indicadores_agregados_amplitude_pregao, index=False)
salvar_dataframe(df_datas_cobertura_insuficiente_amplitude_pregao, caminho_tbl_datas_cobertura_insuficiente_amplitude_pregao, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_indicadores_agregados_amplitude_pregao = df_indicadores_agregados_amplitude_pregao.head(15).copy()
df_amostra_indicadores_agregados_validos = (
    df_indicadores_agregados_amplitude_pregao
    .loc[
        df_indicadores_agregados_amplitude_pregao["n_tickers_mm200_validos"].gt(0)
        & df_indicadores_agregados_amplitude_pregao["n_tickers_minima_52s_validos"].gt(0)
        & df_indicadores_agregados_amplitude_pregao["n_tickers_maxima_52s_validos"].gt(0)
    ]
    .head(15)
    .copy()
)
df_amostra_datas_cobertura_insuficiente = df_datas_cobertura_insuficiente_amplitude_pregao.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da série histórica final de breadth do mercado:")
print(df_resumo_indicadores_agregados_amplitude_pregao.to_string(index=False))

print("\nAuditoria da subetapa de indicadores agregados de amplitude:")
print(df_auditoria_indicadores_agregados_amplitude_pregao.to_string(index=False))

print("\nAmostra inicial da série histórica de breadth:")
print(df_amostra_indicadores_agregados_amplitude_pregao.to_string(index=False))

print("\nAmostra de datas com breadth plenamente válido:")
print(df_amostra_indicadores_agregados_validos.to_string(index=False))

print("\nAmostra de datas com cobertura insuficiente dos indicadores:")
print(df_amostra_datas_cobertura_insuficiente.to_string(index=False))

print("\nArquivos salvos na subetapa 5.5:")
print(f"- {caminho_base_indicadores_agregados_amplitude_pregao}")
print(f"- {caminho_tbl_resumo_indicadores_agregados_amplitude_pregao}")
print(f"- {caminho_tbl_cobertura_indicadores_agregados_amplitude_pregao}")
print(f"- {caminho_tbl_auditoria_indicadores_agregados_amplitude_pregao}")
print(f"- {caminho_tbl_datas_cobertura_insuficiente_amplitude_pregao}")

print("\nETAPA 5.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 5.5 - INDICADORES AGREGADOS DE AMPLITUDE POR PREGÃO

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - elegibilidade operacional por data : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_4_base_elegibilidade_operacional_data.parquet
Entrada - flags amplitude por ticker-data    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_4_base_flags_amplitude_ticker_data.parquet
Saída   - base breadth por pregão            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_5_base_indicadores_agregados_amplitude_pregao.parquet
Saída   - resumo                             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_merc

# Etapa 6) Definição dos Sinais de Capitulação e Euforia

## Etapa 6.1) Regra de Capitulação

In [25]:
%%time
# ============================================================
# Etapa 6.1) Regra de Capitulação
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 6.1 - REGRA DE CAPITULAÇÃO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_indicadores_agregados_amplitude_pregao = gerar_caminho_arquivo(etapa=5, subetapa=5, tipo_arquivo="base", nome="indicadores_agregados_amplitude_pregao")

caminho_tbl_parametros_capitulacao = gerar_caminho_arquivo(etapa=6, subetapa=1, tipo_arquivo="tbl", nome="parametros_capitulacao")
caminho_tbl_regras_capitulacao = gerar_caminho_arquivo(etapa=6, subetapa=1, tipo_arquivo="tbl", nome="regras_capitulacao")
caminho_base_capitulacao_bruta_pregao = gerar_caminho_arquivo(etapa=6, subetapa=1, tipo_arquivo="base", nome="capitulacao_bruta_pregao")
caminho_tbl_resumo_capitulacao_bruta = gerar_caminho_arquivo(etapa=6, subetapa=1, tipo_arquivo="tbl", nome="resumo_capitulacao_bruta")
caminho_tbl_auditoria_capitulacao_bruta = gerar_caminho_arquivo(etapa=6, subetapa=1, tipo_arquivo="tbl", nome="auditoria_capitulacao_bruta")
caminho_tbl_datas_capitulacao_bruta = gerar_caminho_arquivo(etapa=6, subetapa=1, tipo_arquivo="tbl", nome="datas_capitulacao_bruta")
caminho_tbl_limiares_capitulacao_dinamicos = gerar_caminho_arquivo(etapa=6, subetapa=1, tipo_arquivo="tbl", nome="limiares_capitulacao_dinamicos")

print(f"Entrada - breadth por pregão              : {caminho_base_indicadores_agregados_amplitude_pregao}")
print(f"Saída   - parâmetros de capitulação       : {caminho_tbl_parametros_capitulacao}")
print(f"Saída   - regras de capitulação           : {caminho_tbl_regras_capitulacao}")
print(f"Saída   - base de capitulação bruta       : {caminho_base_capitulacao_bruta_pregao}")
print(f"Saída   - resumo                          : {caminho_tbl_resumo_capitulacao_bruta}")
print(f"Saída   - auditoria                       : {caminho_tbl_auditoria_capitulacao_bruta}")
print(f"Saída   - datas de capitulação bruta      : {caminho_tbl_datas_capitulacao_bruta}")
print(f"Saída   - limiares dinâmicos              : {caminho_tbl_limiares_capitulacao_dinamicos}")
print("OK")

# ============================================================
# 3) Carga da base de entrada
# ============================================================

print("\n[3/10] Carga da base de entrada...")

df_indicadores_agregados_amplitude_pregao = pd.read_parquet(caminho_base_indicadores_agregados_amplitude_pregao)

print(f"Base de breadth por pregão: {df_indicadores_agregados_amplitude_pregao.shape[0]:,} linhas x {df_indicadores_agregados_amplitude_pregao.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e premissas metodológicas
# ============================================================

print("\n[4/10] Funções auxiliares e premissas metodológicas...")

COLUNAS_OBRIGATORIAS = [
    "data",
    "n_tickers_elegiveis_operacionais",
    "n_tickers_mm200_validos",
    "n_tickers_minima_52s_validos",
    "n_tickers_maxima_52s_validos",
    "pct_tickers_abaixo_mm200",
    "pct_tickers_em_minima_52s",
    "pct_tickers_em_maxima_52s",
]

QUANTIL_CAPITULACAO_ABAIXO_MM200 = 0.80
QUANTIL_CAPITULACAO_MINIMA_52S = 0.90
QUANTIL_CAPITULACAO_MAXIMA_52S = 0.20
N_CRITERIOS_MINIMOS_CAPITULACAO = 2
N_HISTORICO_MINIMO_CALIBRACAO_CAPITULACAO = 756
DEFASAGEM_CALIBRACAO_PREGOES = 1

COLUNAS_NUMERICAS_BREADTH_CAPITULACAO = [
    "n_tickers_elegiveis_operacionais",
    "n_tickers_mm200_validos",
    "n_tickers_minima_52s_validos",
    "n_tickers_maxima_52s_validos",
    "pct_tickers_abaixo_mm200",
    "pct_tickers_em_minima_52s",
    "pct_tickers_em_maxima_52s",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def registrar_valor_misto(chave, valor, unidade, nome_chave):
    """
    Estrutura valores mistos em colunas homogêneas para escrita segura em parquet.
    """
    if isinstance(valor, (int, float)) and not pd.isna(valor):
        return {
            nome_chave: chave,
            "valor_numerico": float(valor),
            "valor_texto": pd.NA,
            "unidade": unidade,
        }
    return {
        nome_chave: chave,
        "valor_numerico": pd.NA,
        "valor_texto": str(valor) if pd.notna(valor) else pd.NA,
        "unidade": unidade,
    }

def calcular_limiar_expansivo_sem_lookahead(serie, quantil, historico_minimo, defasagem_pregoes):
    """
    Calcula limiares expansivos usando apenas observações disponíveis antes da data avaliada.
    """
    serie_historica = pd.to_numeric(serie, errors="coerce").shift(defasagem_pregoes)
    return serie_historica.expanding(min_periods=historico_minimo).quantile(quantil)

print("Premissa metodológica adotada:")
print("- A capitulação será identificada por score de exaustão de breadth.")
print("- Os limiares serão calculados por quantis expansivos sem uso da própria data ou de datas futuras.")
print("- A data será marcada como capitulação bruta quando pelo menos 2 dos 3 critérios forem satisfeitos.")
print("OK")

# ============================================================
# 5) Preparação da base e calibração dinâmica dos limiares
# ============================================================

print("\n[5/10] Preparação da base e calibração dinâmica dos limiares...")

validar_colunas_obrigatorias(df_indicadores_agregados_amplitude_pregao, COLUNAS_OBRIGATORIAS)

df_capitulacao_bruta_pregao = df_indicadores_agregados_amplitude_pregao.copy()

df_capitulacao_bruta_pregao["data"] = pd.to_datetime(df_capitulacao_bruta_pregao["data"], errors="coerce")
df_capitulacao_bruta_pregao = df_capitulacao_bruta_pregao.sort_values("data").reset_index(drop=True)

for coluna in COLUNAS_NUMERICAS_BREADTH_CAPITULACAO:
    df_capitulacao_bruta_pregao[coluna] = pd.to_numeric(df_capitulacao_bruta_pregao[coluna], errors="coerce")

n_duplicidades_data = int(df_capitulacao_bruta_pregao.duplicated(subset=["data"]).sum())
if n_duplicidades_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_data} duplicidades por data na base de breadth.")

if df_capitulacao_bruta_pregao["data"].isna().any():
    raise ValueError("Foram encontradas datas inválidas na base de breadth.")

df_capitulacao_bruta_pregao["flag_denominadores_validos_capitulacao"] = (
    df_capitulacao_bruta_pregao["n_tickers_mm200_validos"].gt(0)
    & df_capitulacao_bruta_pregao["n_tickers_minima_52s_validos"].gt(0)
    & df_capitulacao_bruta_pregao["n_tickers_maxima_52s_validos"].gt(0)
)

serie_valida_capitulacao = df_capitulacao_bruta_pregao["flag_denominadores_validos_capitulacao"].astype(bool)

df_capitulacao_bruta_pregao["n_observacoes_historico_calibracao_capitulacao"] = (
    serie_valida_capitulacao.shift(DEFASAGEM_CALIBRACAO_PREGOES, fill_value=False).astype(int).cumsum()
)

df_capitulacao_bruta_pregao["limiar_pct_abaixo_mm200_capitulacao"] = calcular_limiar_expansivo_sem_lookahead(
    serie=df_capitulacao_bruta_pregao["pct_tickers_abaixo_mm200"].where(serie_valida_capitulacao),
    quantil=QUANTIL_CAPITULACAO_ABAIXO_MM200,
    historico_minimo=N_HISTORICO_MINIMO_CALIBRACAO_CAPITULACAO,
    defasagem_pregoes=DEFASAGEM_CALIBRACAO_PREGOES,
)

df_capitulacao_bruta_pregao["limiar_pct_minima_52s_capitulacao"] = calcular_limiar_expansivo_sem_lookahead(
    serie=df_capitulacao_bruta_pregao["pct_tickers_em_minima_52s"].where(serie_valida_capitulacao),
    quantil=QUANTIL_CAPITULACAO_MINIMA_52S,
    historico_minimo=N_HISTORICO_MINIMO_CALIBRACAO_CAPITULACAO,
    defasagem_pregoes=DEFASAGEM_CALIBRACAO_PREGOES,
)

df_capitulacao_bruta_pregao["limiar_pct_maxima_52s_capitulacao"] = calcular_limiar_expansivo_sem_lookahead(
    serie=df_capitulacao_bruta_pregao["pct_tickers_em_maxima_52s"].where(serie_valida_capitulacao),
    quantil=QUANTIL_CAPITULACAO_MAXIMA_52S,
    historico_minimo=N_HISTORICO_MINIMO_CALIBRACAO_CAPITULACAO,
    defasagem_pregoes=DEFASAGEM_CALIBRACAO_PREGOES,
)

df_capitulacao_bruta_pregao["flag_historico_minimo_calibracao_capitulacao"] = (
    df_capitulacao_bruta_pregao["n_observacoes_historico_calibracao_capitulacao"].ge(N_HISTORICO_MINIMO_CALIBRACAO_CAPITULACAO)
)

df_capitulacao_bruta_pregao["flag_limiares_validos_capitulacao"] = (
    df_capitulacao_bruta_pregao["limiar_pct_abaixo_mm200_capitulacao"].notna()
    & df_capitulacao_bruta_pregao["limiar_pct_minima_52s_capitulacao"].notna()
    & df_capitulacao_bruta_pregao["limiar_pct_maxima_52s_capitulacao"].notna()
)

df_capitulacao_bruta_pregao["flag_contexto_operacional_valido_capitulacao"] = (
    df_capitulacao_bruta_pregao["flag_denominadores_validos_capitulacao"]
    & df_capitulacao_bruta_pregao["flag_historico_minimo_calibracao_capitulacao"]
    & df_capitulacao_bruta_pregao["flag_limiares_validos_capitulacao"]
)

n_datas_total = int(len(df_capitulacao_bruta_pregao))
n_datas_denominadores_validos = int(df_capitulacao_bruta_pregao["flag_denominadores_validos_capitulacao"].sum())
n_datas_com_historico_minimo = int(df_capitulacao_bruta_pregao["flag_historico_minimo_calibracao_capitulacao"].sum())
n_datas_limiares_validos = int(df_capitulacao_bruta_pregao["flag_limiares_validos_capitulacao"].sum())
n_datas_validas_calibracao = int(df_capitulacao_bruta_pregao["flag_contexto_operacional_valido_capitulacao"].sum())

if n_datas_validas_calibracao == 0:
    raise ValueError("Não há datas com histórico mínimo e limiares válidos para aplicar a regra de capitulação.")

print(f"Histórico mínimo para calibração dinâmica : {N_HISTORICO_MINIMO_CALIBRACAO_CAPITULACAO:,} pregões válidos")
print(f"Defasagem da calibração                  : {DEFASAGEM_CALIBRACAO_PREGOES:,} pregão")
print(f"Datas totais                             : {n_datas_total:,}")
print(f"Datas com denominadores válidos          : {n_datas_denominadores_validos:,}")
print(f"Datas com histórico mínimo               : {n_datas_com_historico_minimo:,}")
print(f"Datas com limiares válidos               : {n_datas_limiares_validos:,}")
print(f"Datas válidas para aplicação da regra    : {n_datas_validas_calibracao:,}")
print("OK")

# ============================================================
# 6) Aplicação da regra bruta de capitulação
# ============================================================

print("\n[6/10] Aplicação da regra bruta de capitulação...")

flag_contexto_valido = df_capitulacao_bruta_pregao["flag_contexto_operacional_valido_capitulacao"]

df_capitulacao_bruta_pregao["flag_criterio_capitulacao_abaixo_mm200_extremo"] = (
    flag_contexto_valido
    & df_capitulacao_bruta_pregao["pct_tickers_abaixo_mm200"].ge(df_capitulacao_bruta_pregao["limiar_pct_abaixo_mm200_capitulacao"])
)

df_capitulacao_bruta_pregao["flag_criterio_capitulacao_minima_52s_extrema"] = (
    flag_contexto_valido
    & df_capitulacao_bruta_pregao["pct_tickers_em_minima_52s"].ge(df_capitulacao_bruta_pregao["limiar_pct_minima_52s_capitulacao"])
)

df_capitulacao_bruta_pregao["flag_criterio_capitulacao_escassez_maxima_52s"] = (
    flag_contexto_valido
    & df_capitulacao_bruta_pregao["pct_tickers_em_maxima_52s"].le(df_capitulacao_bruta_pregao["limiar_pct_maxima_52s_capitulacao"])
)

colunas_flags_criterios_capitulacao = [
    "flag_criterio_capitulacao_abaixo_mm200_extremo",
    "flag_criterio_capitulacao_minima_52s_extrema",
    "flag_criterio_capitulacao_escassez_maxima_52s",
]

for coluna_flag in colunas_flags_criterios_capitulacao:
    df_capitulacao_bruta_pregao[coluna_flag] = df_capitulacao_bruta_pregao[coluna_flag].fillna(False).astype(bool)

df_capitulacao_bruta_pregao["score_capitulacao_bruta"] = (
    df_capitulacao_bruta_pregao[colunas_flags_criterios_capitulacao]
    .sum(axis=1)
)

df_capitulacao_bruta_pregao["flag_capitulacao_bruta"] = (
    flag_contexto_valido
    & df_capitulacao_bruta_pregao["score_capitulacao_bruta"].ge(N_CRITERIOS_MINIMOS_CAPITULACAO)
)

df_capitulacao_bruta_pregao["intensidade_capitulacao_bruta"] = (
    df_capitulacao_bruta_pregao["score_capitulacao_bruta"] / len(colunas_flags_criterios_capitulacao)
)

df_datas_capitulacao_bruta = (
    df_capitulacao_bruta_pregao
    .loc[df_capitulacao_bruta_pregao["flag_capitulacao_bruta"]]
    .copy()
)

n_datas_capitulacao_bruta = int(df_capitulacao_bruta_pregao["flag_capitulacao_bruta"].sum())
n_sinais_sem_historico_minimo = int(
    (
        df_capitulacao_bruta_pregao["flag_capitulacao_bruta"]
        & ~df_capitulacao_bruta_pregao["flag_historico_minimo_calibracao_capitulacao"]
    ).sum()
)

if n_sinais_sem_historico_minimo > 0:
    raise ValueError("Foram encontrados sinais de capitulação sem histórico mínimo de calibração.")

print(f"Quantidade de datas com capitulação bruta: {n_datas_capitulacao_bruta:,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas formais da subetapa...")

df_parametros_capitulacao = pd.DataFrame(
    [
        registrar_valor_misto("quantil_capitulacao_abaixo_mm200", QUANTIL_CAPITULACAO_ABAIXO_MM200, "quantil", "parametro"),
        registrar_valor_misto("quantil_capitulacao_minima_52s", QUANTIL_CAPITULACAO_MINIMA_52S, "quantil", "parametro"),
        registrar_valor_misto("quantil_capitulacao_maxima_52s", QUANTIL_CAPITULACAO_MAXIMA_52S, "quantil", "parametro"),
        registrar_valor_misto("n_historico_minimo_calibracao_capitulacao", N_HISTORICO_MINIMO_CALIBRACAO_CAPITULACAO, "pregoes_validos", "parametro"),
        registrar_valor_misto("defasagem_calibracao_pregoes", DEFASAGEM_CALIBRACAO_PREGOES, "pregoes", "parametro"),
        registrar_valor_misto("n_criterios_minimos_capitulacao", N_CRITERIOS_MINIMOS_CAPITULACAO, "criterios", "parametro"),
        registrar_valor_misto("metodo_calibracao", "quantis_expansivos_defasados_sem_uso_de_dados_futuros", "regra_metodologica", "parametro"),
        registrar_valor_misto("logica_decisao", "score_maior_ou_igual_a_2_em_3", "regra_metodologica", "parametro"),
    ]
)

df_regras_capitulacao = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "CAP1",
            "variavel": "pct_tickers_abaixo_mm200",
            "operador": ">=",
            "limiar_estatico": pd.NA,
            "limiar_dinamico_coluna": "limiar_pct_abaixo_mm200_capitulacao",
            "quantil_calibracao": float(QUANTIL_CAPITULACAO_ABAIXO_MM200),
            "unidade": "proporcao",
            "descricao_operacional": "O percentual de tickers abaixo da MM200 deve estar em patamar extremo frente ao histórico disponível até a véspera.",
            "justificativa_tecnica": "Captura deterioração disseminada do breadth de tendência sem uso de dados futuros."
        },
        {
            "ordem_regra": 2,
            "regra_id": "CAP2",
            "variavel": "pct_tickers_em_minima_52s",
            "operador": ">=",
            "limiar_estatico": pd.NA,
            "limiar_dinamico_coluna": "limiar_pct_minima_52s_capitulacao",
            "quantil_calibracao": float(QUANTIL_CAPITULACAO_MINIMA_52S),
            "unidade": "proporcao",
            "descricao_operacional": "O percentual de tickers em mínima de 52 semanas deve estar em patamar extremo frente ao histórico disponível até a véspera.",
            "justificativa_tecnica": "Captura pressão vendedora intensa e extensão do movimento de queda sem uso de dados futuros."
        },
        {
            "ordem_regra": 3,
            "regra_id": "CAP3",
            "variavel": "pct_tickers_em_maxima_52s",
            "operador": "<=",
            "limiar_estatico": pd.NA,
            "limiar_dinamico_coluna": "limiar_pct_maxima_52s_capitulacao",
            "quantil_calibracao": float(QUANTIL_CAPITULACAO_MAXIMA_52S),
            "unidade": "proporcao",
            "descricao_operacional": "O percentual de tickers em máxima de 52 semanas deve estar em patamar baixo frente ao histórico disponível até a véspera.",
            "justificativa_tecnica": "Impõe escassez de liderança positiva durante o evento de estresse sem uso de dados futuros."
        },
        {
            "ordem_regra": 4,
            "regra_id": "CAP_FINAL",
            "variavel": "score_capitulacao_bruta",
            "operador": ">=",
            "limiar_estatico": float(N_CRITERIOS_MINIMOS_CAPITULACAO),
            "limiar_dinamico_coluna": pd.NA,
            "quantil_calibracao": pd.NA,
            "unidade": "criterios",
            "descricao_operacional": "A data é marcada como capitulação bruta quando pelo menos 2 dos 3 critérios são satisfeitos.",
            "justificativa_tecnica": "Evita gatilhos excessivamente rígidos de interseção total e também evita sinalização por critério isolado."
        },
    ]
)

df_resumo_capitulacao_bruta = pd.DataFrame(
    [
        {
            "base": "capitulacao_bruta_pregao",
            "metodo_calibracao": "quantis_expansivos_defasados_sem_uso_de_dados_futuros",
            "n_datas_total": n_datas_total,
            "n_datas_denominadores_validos": n_datas_denominadores_validos,
            "n_datas_com_historico_minimo": n_datas_com_historico_minimo,
            "n_datas_limiares_validos": n_datas_limiares_validos,
            "n_datas_validas_calibracao": n_datas_validas_calibracao,
            "n_datas_capitulacao_bruta": n_datas_capitulacao_bruta,
            "pct_datas_capitulacao_bruta": (n_datas_capitulacao_bruta / n_datas_validas_calibracao) if n_datas_validas_calibracao > 0 else pd.NA,
            "data_min": df_capitulacao_bruta_pregao["data"].min(),
            "data_max": df_capitulacao_bruta_pregao["data"].max(),
            "primeira_data_com_limiares_validos": df_capitulacao_bruta_pregao.loc[df_capitulacao_bruta_pregao["flag_limiares_validos_capitulacao"], "data"].min(),
            "primeira_data_capitulacao_bruta": df_datas_capitulacao_bruta["data"].min() if not df_datas_capitulacao_bruta.empty else pd.NaT,
            "mediana_score_capitulacao_bruta": float(df_capitulacao_bruta_pregao["score_capitulacao_bruta"].median()),
            "max_score_capitulacao_bruta": float(df_capitulacao_bruta_pregao["score_capitulacao_bruta"].max()),
        }
    ]
)

df_auditoria_capitulacao_bruta = pd.DataFrame(
    [
        {"metrica": "n_datas_total", "valor": n_datas_total},
        {"metrica": "n_datas_denominadores_validos", "valor": n_datas_denominadores_validos},
        {"metrica": "n_datas_com_historico_minimo", "valor": n_datas_com_historico_minimo},
        {"metrica": "n_datas_limiares_validos", "valor": n_datas_limiares_validos},
        {"metrica": "n_datas_validas_calibracao", "valor": n_datas_validas_calibracao},
        {"metrica": "n_datas_capitulacao_bruta", "valor": n_datas_capitulacao_bruta},
        {"metrica": "n_datas_criterio_cap1", "valor": int(df_capitulacao_bruta_pregao["flag_criterio_capitulacao_abaixo_mm200_extremo"].sum())},
        {"metrica": "n_datas_criterio_cap2", "valor": int(df_capitulacao_bruta_pregao["flag_criterio_capitulacao_minima_52s_extrema"].sum())},
        {"metrica": "n_datas_criterio_cap3", "valor": int(df_capitulacao_bruta_pregao["flag_criterio_capitulacao_escassez_maxima_52s"].sum())},
        {"metrica": "n_datas_score_0", "valor": int(df_capitulacao_bruta_pregao["score_capitulacao_bruta"].eq(0).sum())},
        {"metrica": "n_datas_score_1", "valor": int(df_capitulacao_bruta_pregao["score_capitulacao_bruta"].eq(1).sum())},
        {"metrica": "n_datas_score_2", "valor": int(df_capitulacao_bruta_pregao["score_capitulacao_bruta"].eq(2).sum())},
        {"metrica": "n_datas_score_3", "valor": int(df_capitulacao_bruta_pregao["score_capitulacao_bruta"].eq(3).sum())},
        {"metrica": "n_sinais_sem_historico_minimo", "valor": n_sinais_sem_historico_minimo},
    ]
)

df_limiares_capitulacao_dinamicos = df_capitulacao_bruta_pregao[
    [
        "data",
        "n_observacoes_historico_calibracao_capitulacao",
        "flag_denominadores_validos_capitulacao",
        "flag_historico_minimo_calibracao_capitulacao",
        "flag_limiares_validos_capitulacao",
        "flag_contexto_operacional_valido_capitulacao",
        "limiar_pct_abaixo_mm200_capitulacao",
        "limiar_pct_minima_52s_capitulacao",
        "limiar_pct_maxima_52s_capitulacao",
    ]
].copy()

print("Tabelas formais da subetapa construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_parametros_capitulacao, caminho_tbl_parametros_capitulacao, index=False)
salvar_dataframe(df_regras_capitulacao, caminho_tbl_regras_capitulacao, index=False)
salvar_dataframe(df_capitulacao_bruta_pregao, caminho_base_capitulacao_bruta_pregao, index=False)
salvar_dataframe(df_resumo_capitulacao_bruta, caminho_tbl_resumo_capitulacao_bruta, index=False)
salvar_dataframe(df_auditoria_capitulacao_bruta, caminho_tbl_auditoria_capitulacao_bruta, index=False)
salvar_dataframe(df_datas_capitulacao_bruta, caminho_tbl_datas_capitulacao_bruta, index=False)
salvar_dataframe(df_limiares_capitulacao_dinamicos, caminho_tbl_limiares_capitulacao_dinamicos, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_capitulacao_bruta_inicial = df_capitulacao_bruta_pregao.head(15).copy()
df_amostra_limiares_validos = (
    df_limiares_capitulacao_dinamicos
    .loc[df_limiares_capitulacao_dinamicos["flag_limiares_validos_capitulacao"]]
    .head(15)
    .copy()
)
df_amostra_datas_capitulacao_bruta = df_datas_capitulacao_bruta.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nParâmetros formais da regra de capitulação:")
print(df_parametros_capitulacao.to_string(index=False))

print("\nRegras formais da regra de capitulação:")
print(df_regras_capitulacao.to_string(index=False))

print("\nResumo da base de capitulação bruta:")
print(df_resumo_capitulacao_bruta.to_string(index=False))

print("\nAuditoria da regra bruta de capitulação:")
print(df_auditoria_capitulacao_bruta.to_string(index=False))

print("\nAmostra inicial da base diária de capitulação bruta:")
print(df_amostra_capitulacao_bruta_inicial.to_string(index=False))

print("\nAmostra inicial dos limiares dinâmicos válidos:")
print(df_amostra_limiares_validos.to_string(index=False))

print("\nAmostra das datas sinalizadas como capitulação bruta:")
print(df_amostra_datas_capitulacao_bruta.to_string(index=False))

print("\nArquivos salvos na subetapa 6.1:")
print(f"- {caminho_tbl_parametros_capitulacao}")
print(f"- {caminho_tbl_regras_capitulacao}")
print(f"- {caminho_base_capitulacao_bruta_pregao}")
print(f"- {caminho_tbl_resumo_capitulacao_bruta}")
print(f"- {caminho_tbl_auditoria_capitulacao_bruta}")
print(f"- {caminho_tbl_datas_capitulacao_bruta}")
print(f"- {caminho_tbl_limiares_capitulacao_dinamicos}")

print("\nETAPA 6.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 6.1 - REGRA DE CAPITULAÇÃO

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - breadth por pregão              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_5_base_indicadores_agregados_amplitude_pregao.parquet
Saída   - parâmetros de capitulação       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_1_tbl_parametros_capitulacao.parquet
Saída   - regras de capitulação           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_1_tbl_regras_capitulacao.parquet
Saída   - base de capitulação bruta       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_1_base_capitulacao_b

## Etapa 6.2) Regra de Euforia

In [26]:
%%time
# ============================================================
# Etapa 6.2) Regra de Euforia
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 6.2 - REGRA DE EUFORIA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_indicadores_agregados_amplitude_pregao = gerar_caminho_arquivo(etapa=5, subetapa=5, tipo_arquivo="base", nome="indicadores_agregados_amplitude_pregao")

caminho_tbl_parametros_euforia = gerar_caminho_arquivo(etapa=6, subetapa=2, tipo_arquivo="tbl", nome="parametros_euforia")
caminho_tbl_regras_euforia = gerar_caminho_arquivo(etapa=6, subetapa=2, tipo_arquivo="tbl", nome="regras_euforia")
caminho_base_euforia_bruta_pregao = gerar_caminho_arquivo(etapa=6, subetapa=2, tipo_arquivo="base", nome="euforia_bruta_pregao")
caminho_tbl_resumo_euforia_bruta = gerar_caminho_arquivo(etapa=6, subetapa=2, tipo_arquivo="tbl", nome="resumo_euforia_bruta")
caminho_tbl_auditoria_euforia_bruta = gerar_caminho_arquivo(etapa=6, subetapa=2, tipo_arquivo="tbl", nome="auditoria_euforia_bruta")
caminho_tbl_datas_euforia_bruta = gerar_caminho_arquivo(etapa=6, subetapa=2, tipo_arquivo="tbl", nome="datas_euforia_bruta")
caminho_tbl_limiares_euforia_dinamicos = gerar_caminho_arquivo(etapa=6, subetapa=2, tipo_arquivo="tbl", nome="limiares_euforia_dinamicos")

print(f"Entrada - breadth por pregão          : {caminho_base_indicadores_agregados_amplitude_pregao}")
print(f"Saída   - parâmetros de euforia       : {caminho_tbl_parametros_euforia}")
print(f"Saída   - regras de euforia           : {caminho_tbl_regras_euforia}")
print(f"Saída   - base de euforia bruta       : {caminho_base_euforia_bruta_pregao}")
print(f"Saída   - resumo                      : {caminho_tbl_resumo_euforia_bruta}")
print(f"Saída   - auditoria                   : {caminho_tbl_auditoria_euforia_bruta}")
print(f"Saída   - datas de euforia bruta      : {caminho_tbl_datas_euforia_bruta}")
print(f"Saída   - limiares dinâmicos          : {caminho_tbl_limiares_euforia_dinamicos}")
print("OK")

# ============================================================
# 3) Carga da base de entrada
# ============================================================

print("\n[3/10] Carga da base de entrada...")

df_indicadores_agregados_amplitude_pregao = pd.read_parquet(caminho_base_indicadores_agregados_amplitude_pregao)

print(f"Base de breadth por pregão: {df_indicadores_agregados_amplitude_pregao.shape[0]:,} linhas x {df_indicadores_agregados_amplitude_pregao.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e premissas metodológicas
# ============================================================

print("\n[4/10] Funções auxiliares e premissas metodológicas...")

COLUNAS_OBRIGATORIAS = [
    "data",
    "n_tickers_elegiveis_operacionais",
    "n_tickers_mm200_validos",
    "n_tickers_minima_52s_validos",
    "n_tickers_maxima_52s_validos",
    "pct_tickers_acima_mm200",
    "pct_tickers_em_minima_52s",
    "pct_tickers_em_maxima_52s",
]

QUANTIL_EUFORIA_ACIMA_MM200 = 0.80
QUANTIL_EUFORIA_MAXIMA_52S = 0.90
QUANTIL_EUFORIA_MINIMA_52S = 0.20
N_CRITERIOS_MINIMOS_EUFORIA = 2
N_HISTORICO_MINIMO_CALIBRACAO_EUFORIA = 756
DEFASAGEM_CALIBRACAO_PREGOES = 1

COLUNAS_NUMERICAS_BREADTH_EUFORIA = [
    "n_tickers_elegiveis_operacionais",
    "n_tickers_mm200_validos",
    "n_tickers_minima_52s_validos",
    "n_tickers_maxima_52s_validos",
    "pct_tickers_acima_mm200",
    "pct_tickers_em_minima_52s",
    "pct_tickers_em_maxima_52s",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def registrar_valor_misto(chave, valor, unidade, nome_chave):
    """
    Estrutura valores mistos em colunas homogêneas para escrita segura em parquet.
    """
    if isinstance(valor, (int, float)) and not pd.isna(valor):
        return {
            nome_chave: chave,
            "valor_numerico": float(valor),
            "valor_texto": pd.NA,
            "unidade": unidade,
        }
    return {
        nome_chave: chave,
        "valor_numerico": pd.NA,
        "valor_texto": str(valor) if pd.notna(valor) else pd.NA,
        "unidade": unidade,
    }

def calcular_limiar_expansivo_sem_lookahead(serie, quantil, historico_minimo, defasagem_pregoes):
    """
    Calcula limiares expansivos usando apenas observações disponíveis antes da data avaliada.
    """
    serie_historica = pd.to_numeric(serie, errors="coerce").shift(defasagem_pregoes)
    return serie_historica.expanding(min_periods=historico_minimo).quantile(quantil)

print("Premissa metodológica adotada:")
print("- A euforia será identificada por score de breadth favorável.")
print("- Os limiares serão calculados por quantis expansivos sem uso da própria data ou de datas futuras.")
print("- A data será marcada como euforia bruta quando pelo menos 2 dos 3 critérios forem satisfeitos.")
print("OK")

# ============================================================
# 5) Preparação da base e calibração dinâmica dos limiares
# ============================================================

print("\n[5/10] Preparação da base e calibração dinâmica dos limiares...")

validar_colunas_obrigatorias(df_indicadores_agregados_amplitude_pregao, COLUNAS_OBRIGATORIAS)

df_euforia_bruta_pregao = df_indicadores_agregados_amplitude_pregao.copy()

df_euforia_bruta_pregao["data"] = pd.to_datetime(df_euforia_bruta_pregao["data"], errors="coerce")
df_euforia_bruta_pregao = df_euforia_bruta_pregao.sort_values("data").reset_index(drop=True)

for coluna in COLUNAS_NUMERICAS_BREADTH_EUFORIA:
    df_euforia_bruta_pregao[coluna] = pd.to_numeric(df_euforia_bruta_pregao[coluna], errors="coerce")

n_duplicidades_data = int(df_euforia_bruta_pregao.duplicated(subset=["data"]).sum())
if n_duplicidades_data > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_data} duplicidades por data na base de breadth.")

if df_euforia_bruta_pregao["data"].isna().any():
    raise ValueError("Foram encontradas datas inválidas na base de breadth.")

df_euforia_bruta_pregao["flag_denominadores_validos_euforia"] = (
    df_euforia_bruta_pregao["n_tickers_mm200_validos"].gt(0)
    & df_euforia_bruta_pregao["n_tickers_minima_52s_validos"].gt(0)
    & df_euforia_bruta_pregao["n_tickers_maxima_52s_validos"].gt(0)
)

serie_valida_euforia = df_euforia_bruta_pregao["flag_denominadores_validos_euforia"].astype(bool)

df_euforia_bruta_pregao["n_observacoes_historico_calibracao_euforia"] = (
    serie_valida_euforia.shift(DEFASAGEM_CALIBRACAO_PREGOES, fill_value=False).astype(int).cumsum()
)

df_euforia_bruta_pregao["limiar_pct_acima_mm200_euforia"] = calcular_limiar_expansivo_sem_lookahead(
    serie=df_euforia_bruta_pregao["pct_tickers_acima_mm200"].where(serie_valida_euforia),
    quantil=QUANTIL_EUFORIA_ACIMA_MM200,
    historico_minimo=N_HISTORICO_MINIMO_CALIBRACAO_EUFORIA,
    defasagem_pregoes=DEFASAGEM_CALIBRACAO_PREGOES,
)

df_euforia_bruta_pregao["limiar_pct_maxima_52s_euforia"] = calcular_limiar_expansivo_sem_lookahead(
    serie=df_euforia_bruta_pregao["pct_tickers_em_maxima_52s"].where(serie_valida_euforia),
    quantil=QUANTIL_EUFORIA_MAXIMA_52S,
    historico_minimo=N_HISTORICO_MINIMO_CALIBRACAO_EUFORIA,
    defasagem_pregoes=DEFASAGEM_CALIBRACAO_PREGOES,
)

df_euforia_bruta_pregao["limiar_pct_minima_52s_euforia"] = calcular_limiar_expansivo_sem_lookahead(
    serie=df_euforia_bruta_pregao["pct_tickers_em_minima_52s"].where(serie_valida_euforia),
    quantil=QUANTIL_EUFORIA_MINIMA_52S,
    historico_minimo=N_HISTORICO_MINIMO_CALIBRACAO_EUFORIA,
    defasagem_pregoes=DEFASAGEM_CALIBRACAO_PREGOES,
)

df_euforia_bruta_pregao["flag_historico_minimo_calibracao_euforia"] = (
    df_euforia_bruta_pregao["n_observacoes_historico_calibracao_euforia"].ge(N_HISTORICO_MINIMO_CALIBRACAO_EUFORIA)
)

df_euforia_bruta_pregao["flag_limiares_validos_euforia"] = (
    df_euforia_bruta_pregao["limiar_pct_acima_mm200_euforia"].notna()
    & df_euforia_bruta_pregao["limiar_pct_maxima_52s_euforia"].notna()
    & df_euforia_bruta_pregao["limiar_pct_minima_52s_euforia"].notna()
)

df_euforia_bruta_pregao["flag_contexto_operacional_valido_euforia"] = (
    df_euforia_bruta_pregao["flag_denominadores_validos_euforia"]
    & df_euforia_bruta_pregao["flag_historico_minimo_calibracao_euforia"]
    & df_euforia_bruta_pregao["flag_limiares_validos_euforia"]
)

n_datas_total = int(len(df_euforia_bruta_pregao))
n_datas_denominadores_validos = int(df_euforia_bruta_pregao["flag_denominadores_validos_euforia"].sum())
n_datas_com_historico_minimo = int(df_euforia_bruta_pregao["flag_historico_minimo_calibracao_euforia"].sum())
n_datas_limiares_validos = int(df_euforia_bruta_pregao["flag_limiares_validos_euforia"].sum())
n_datas_validas_calibracao = int(df_euforia_bruta_pregao["flag_contexto_operacional_valido_euforia"].sum())

if n_datas_validas_calibracao == 0:
    raise ValueError("Não há datas com histórico mínimo e limiares válidos para aplicar a regra de euforia.")

print(f"Histórico mínimo para calibração dinâmica : {N_HISTORICO_MINIMO_CALIBRACAO_EUFORIA:,} pregões válidos")
print(f"Defasagem da calibração                  : {DEFASAGEM_CALIBRACAO_PREGOES:,} pregão")
print(f"Datas totais                             : {n_datas_total:,}")
print(f"Datas com denominadores válidos          : {n_datas_denominadores_validos:,}")
print(f"Datas com histórico mínimo               : {n_datas_com_historico_minimo:,}")
print(f"Datas com limiares válidos               : {n_datas_limiares_validos:,}")
print(f"Datas válidas para aplicação da regra    : {n_datas_validas_calibracao:,}")
print("OK")

# ============================================================
# 6) Aplicação da regra bruta de euforia
# ============================================================

print("\n[6/10] Aplicação da regra bruta de euforia...")

flag_contexto_valido = df_euforia_bruta_pregao["flag_contexto_operacional_valido_euforia"]

df_euforia_bruta_pregao["flag_criterio_euforia_acima_mm200_extremo"] = (
    flag_contexto_valido
    & df_euforia_bruta_pregao["pct_tickers_acima_mm200"].ge(df_euforia_bruta_pregao["limiar_pct_acima_mm200_euforia"])
)

df_euforia_bruta_pregao["flag_criterio_euforia_maxima_52s_extrema"] = (
    flag_contexto_valido
    & df_euforia_bruta_pregao["pct_tickers_em_maxima_52s"].ge(df_euforia_bruta_pregao["limiar_pct_maxima_52s_euforia"])
)

df_euforia_bruta_pregao["flag_criterio_euforia_escassez_minima_52s"] = (
    flag_contexto_valido
    & df_euforia_bruta_pregao["pct_tickers_em_minima_52s"].le(df_euforia_bruta_pregao["limiar_pct_minima_52s_euforia"])
)

colunas_flags_criterios_euforia = [
    "flag_criterio_euforia_acima_mm200_extremo",
    "flag_criterio_euforia_maxima_52s_extrema",
    "flag_criterio_euforia_escassez_minima_52s",
]

for coluna_flag in colunas_flags_criterios_euforia:
    df_euforia_bruta_pregao[coluna_flag] = df_euforia_bruta_pregao[coluna_flag].fillna(False).astype(bool)

df_euforia_bruta_pregao["score_euforia_bruta"] = (
    df_euforia_bruta_pregao[colunas_flags_criterios_euforia]
    .sum(axis=1)
)

df_euforia_bruta_pregao["flag_euforia_bruta"] = (
    flag_contexto_valido
    & df_euforia_bruta_pregao["score_euforia_bruta"].ge(N_CRITERIOS_MINIMOS_EUFORIA)
)

df_euforia_bruta_pregao["intensidade_euforia_bruta"] = (
    df_euforia_bruta_pregao["score_euforia_bruta"] / len(colunas_flags_criterios_euforia)
)

df_datas_euforia_bruta = (
    df_euforia_bruta_pregao
    .loc[df_euforia_bruta_pregao["flag_euforia_bruta"]]
    .copy()
)

n_datas_euforia_bruta = int(df_euforia_bruta_pregao["flag_euforia_bruta"].sum())
n_sinais_sem_historico_minimo = int(
    (
        df_euforia_bruta_pregao["flag_euforia_bruta"]
        & ~df_euforia_bruta_pregao["flag_historico_minimo_calibracao_euforia"]
    ).sum()
)

if n_sinais_sem_historico_minimo > 0:
    raise ValueError("Foram encontrados sinais de euforia sem histórico mínimo de calibração.")

print(f"Quantidade de datas com euforia bruta: {n_datas_euforia_bruta:,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas formais da subetapa...")

df_parametros_euforia = pd.DataFrame(
    [
        registrar_valor_misto("quantil_euforia_acima_mm200", QUANTIL_EUFORIA_ACIMA_MM200, "quantil", "parametro"),
        registrar_valor_misto("quantil_euforia_maxima_52s", QUANTIL_EUFORIA_MAXIMA_52S, "quantil", "parametro"),
        registrar_valor_misto("quantil_euforia_minima_52s", QUANTIL_EUFORIA_MINIMA_52S, "quantil", "parametro"),
        registrar_valor_misto("n_historico_minimo_calibracao_euforia", N_HISTORICO_MINIMO_CALIBRACAO_EUFORIA, "pregoes_validos", "parametro"),
        registrar_valor_misto("defasagem_calibracao_pregoes", DEFASAGEM_CALIBRACAO_PREGOES, "pregoes", "parametro"),
        registrar_valor_misto("n_criterios_minimos_euforia", N_CRITERIOS_MINIMOS_EUFORIA, "criterios", "parametro"),
        registrar_valor_misto("metodo_calibracao", "quantis_expansivos_defasados_sem_uso_de_dados_futuros", "regra_metodologica", "parametro"),
        registrar_valor_misto("logica_decisao", "score_maior_ou_igual_a_2_em_3", "regra_metodologica", "parametro"),
    ]
)

df_regras_euforia = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "EUF1",
            "variavel": "pct_tickers_acima_mm200",
            "operador": ">=",
            "limiar_estatico": pd.NA,
            "limiar_dinamico_coluna": "limiar_pct_acima_mm200_euforia",
            "quantil_calibracao": float(QUANTIL_EUFORIA_ACIMA_MM200),
            "unidade": "proporcao",
            "descricao_operacional": "O percentual de tickers acima da MM200 deve estar em patamar extremo frente ao histórico disponível até a véspera.",
            "justificativa_tecnica": "Captura difusão ampla de tendência positiva no breadth do mercado sem uso de dados futuros."
        },
        {
            "ordem_regra": 2,
            "regra_id": "EUF2",
            "variavel": "pct_tickers_em_maxima_52s",
            "operador": ">=",
            "limiar_estatico": pd.NA,
            "limiar_dinamico_coluna": "limiar_pct_maxima_52s_euforia",
            "quantil_calibracao": float(QUANTIL_EUFORIA_MAXIMA_52S),
            "unidade": "proporcao",
            "descricao_operacional": "O percentual de tickers em máxima de 52 semanas deve estar em patamar extremo frente ao histórico disponível até a véspera.",
            "justificativa_tecnica": "Captura extensão do movimento comprador e força de liderança positiva sem uso de dados futuros."
        },
        {
            "ordem_regra": 3,
            "regra_id": "EUF3",
            "variavel": "pct_tickers_em_minima_52s",
            "operador": "<=",
            "limiar_estatico": pd.NA,
            "limiar_dinamico_coluna": "limiar_pct_minima_52s_euforia",
            "quantil_calibracao": float(QUANTIL_EUFORIA_MINIMA_52S),
            "unidade": "proporcao",
            "descricao_operacional": "O percentual de tickers em mínima de 52 semanas deve estar em patamar baixo frente ao histórico disponível até a véspera.",
            "justificativa_tecnica": "Impõe escassez de estresse vendedor durante o evento de euforia sem uso de dados futuros."
        },
        {
            "ordem_regra": 4,
            "regra_id": "EUF_FINAL",
            "variavel": "score_euforia_bruta",
            "operador": ">=",
            "limiar_estatico": float(N_CRITERIOS_MINIMOS_EUFORIA),
            "limiar_dinamico_coluna": pd.NA,
            "quantil_calibracao": pd.NA,
            "unidade": "criterios",
            "descricao_operacional": "A data é marcada como euforia bruta quando pelo menos 2 dos 3 critérios são satisfeitos.",
            "justificativa_tecnica": "Evita gatilhos excessivamente rígidos de interseção total e também evita sinalização por critério isolado."
        },
    ]
)

df_resumo_euforia_bruta = pd.DataFrame(
    [
        {
            "base": "euforia_bruta_pregao",
            "metodo_calibracao": "quantis_expansivos_defasados_sem_uso_de_dados_futuros",
            "n_datas_total": n_datas_total,
            "n_datas_denominadores_validos": n_datas_denominadores_validos,
            "n_datas_com_historico_minimo": n_datas_com_historico_minimo,
            "n_datas_limiares_validos": n_datas_limiares_validos,
            "n_datas_validas_calibracao": n_datas_validas_calibracao,
            "n_datas_euforia_bruta": n_datas_euforia_bruta,
            "pct_datas_euforia_bruta": (n_datas_euforia_bruta / n_datas_validas_calibracao) if n_datas_validas_calibracao > 0 else pd.NA,
            "data_min": df_euforia_bruta_pregao["data"].min(),
            "data_max": df_euforia_bruta_pregao["data"].max(),
            "primeira_data_com_limiares_validos": df_euforia_bruta_pregao.loc[df_euforia_bruta_pregao["flag_limiares_validos_euforia"], "data"].min(),
            "primeira_data_euforia_bruta": df_datas_euforia_bruta["data"].min() if not df_datas_euforia_bruta.empty else pd.NaT,
            "mediana_score_euforia_bruta": float(df_euforia_bruta_pregao["score_euforia_bruta"].median()),
            "max_score_euforia_bruta": float(df_euforia_bruta_pregao["score_euforia_bruta"].max()),
        }
    ]
)

df_auditoria_euforia_bruta = pd.DataFrame(
    [
        {"metrica": "n_datas_total", "valor": n_datas_total},
        {"metrica": "n_datas_denominadores_validos", "valor": n_datas_denominadores_validos},
        {"metrica": "n_datas_com_historico_minimo", "valor": n_datas_com_historico_minimo},
        {"metrica": "n_datas_limiares_validos", "valor": n_datas_limiares_validos},
        {"metrica": "n_datas_validas_calibracao", "valor": n_datas_validas_calibracao},
        {"metrica": "n_datas_euforia_bruta", "valor": n_datas_euforia_bruta},
        {"metrica": "n_datas_criterio_euf1", "valor": int(df_euforia_bruta_pregao["flag_criterio_euforia_acima_mm200_extremo"].sum())},
        {"metrica": "n_datas_criterio_euf2", "valor": int(df_euforia_bruta_pregao["flag_criterio_euforia_maxima_52s_extrema"].sum())},
        {"metrica": "n_datas_criterio_euf3", "valor": int(df_euforia_bruta_pregao["flag_criterio_euforia_escassez_minima_52s"].sum())},
        {"metrica": "n_datas_score_0", "valor": int(df_euforia_bruta_pregao["score_euforia_bruta"].eq(0).sum())},
        {"metrica": "n_datas_score_1", "valor": int(df_euforia_bruta_pregao["score_euforia_bruta"].eq(1).sum())},
        {"metrica": "n_datas_score_2", "valor": int(df_euforia_bruta_pregao["score_euforia_bruta"].eq(2).sum())},
        {"metrica": "n_datas_score_3", "valor": int(df_euforia_bruta_pregao["score_euforia_bruta"].eq(3).sum())},
        {"metrica": "n_sinais_sem_historico_minimo", "valor": n_sinais_sem_historico_minimo},
    ]
)

df_limiares_euforia_dinamicos = df_euforia_bruta_pregao[
    [
        "data",
        "n_observacoes_historico_calibracao_euforia",
        "flag_denominadores_validos_euforia",
        "flag_historico_minimo_calibracao_euforia",
        "flag_limiares_validos_euforia",
        "flag_contexto_operacional_valido_euforia",
        "limiar_pct_acima_mm200_euforia",
        "limiar_pct_maxima_52s_euforia",
        "limiar_pct_minima_52s_euforia",
    ]
].copy()

print("Tabelas formais da subetapa construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_parametros_euforia, caminho_tbl_parametros_euforia, index=False)
salvar_dataframe(df_regras_euforia, caminho_tbl_regras_euforia, index=False)
salvar_dataframe(df_euforia_bruta_pregao, caminho_base_euforia_bruta_pregao, index=False)
salvar_dataframe(df_resumo_euforia_bruta, caminho_tbl_resumo_euforia_bruta, index=False)
salvar_dataframe(df_auditoria_euforia_bruta, caminho_tbl_auditoria_euforia_bruta, index=False)
salvar_dataframe(df_datas_euforia_bruta, caminho_tbl_datas_euforia_bruta, index=False)
salvar_dataframe(df_limiares_euforia_dinamicos, caminho_tbl_limiares_euforia_dinamicos, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_euforia_bruta_inicial = df_euforia_bruta_pregao.head(15).copy()
df_amostra_limiares_validos = (
    df_limiares_euforia_dinamicos
    .loc[df_limiares_euforia_dinamicos["flag_limiares_validos_euforia"]]
    .head(15)
    .copy()
)
df_amostra_datas_euforia_bruta = df_datas_euforia_bruta.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nParâmetros formais da regra de euforia:")
print(df_parametros_euforia.to_string(index=False))

print("\nRegras formais da regra de euforia:")
print(df_regras_euforia.to_string(index=False))

print("\nResumo da base de euforia bruta:")
print(df_resumo_euforia_bruta.to_string(index=False))

print("\nAuditoria da regra bruta de euforia:")
print(df_auditoria_euforia_bruta.to_string(index=False))

print("\nAmostra inicial da base diária de euforia bruta:")
print(df_amostra_euforia_bruta_inicial.to_string(index=False))

print("\nAmostra inicial dos limiares dinâmicos válidos:")
print(df_amostra_limiares_validos.to_string(index=False))

print("\nAmostra das datas sinalizadas como euforia bruta:")
print(df_amostra_datas_euforia_bruta.to_string(index=False))

print("\nArquivos salvos na subetapa 6.2:")
print(f"- {caminho_tbl_parametros_euforia}")
print(f"- {caminho_tbl_regras_euforia}")
print(f"- {caminho_base_euforia_bruta_pregao}")
print(f"- {caminho_tbl_resumo_euforia_bruta}")
print(f"- {caminho_tbl_auditoria_euforia_bruta}")
print(f"- {caminho_tbl_datas_euforia_bruta}")
print(f"- {caminho_tbl_limiares_euforia_dinamicos}")

print("\nETAPA 6.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 6.2 - REGRA DE EUFORIA

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - breadth por pregão          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_5_base_indicadores_agregados_amplitude_pregao.parquet
Saída   - parâmetros de euforia       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_2_tbl_parametros_euforia.parquet
Saída   - regras de euforia           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_2_tbl_regras_euforia.parquet
Saída   - base de euforia bruta       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_2_base_euforia_bruta_pregao.parquet
Saída   - re

## Etapa 6.3) Aplicação do Cooldown

In [27]:
%%time
# ============================================================
# Etapa 6.3) Aplicação do Cooldown
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 6.3 - APLICAÇÃO DO COOLDOWN")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_tbl_parametros_globais = gerar_caminho_arquivo(etapa=1, subetapa=1, tipo_arquivo="tbl", nome="parametros_globais")
caminho_base_capitulacao_bruta_pregao = gerar_caminho_arquivo(etapa=6, subetapa=1, tipo_arquivo="base", nome="capitulacao_bruta_pregao")
caminho_base_euforia_bruta_pregao = gerar_caminho_arquivo(etapa=6, subetapa=2, tipo_arquivo="base", nome="euforia_bruta_pregao")

caminho_base_sinais_pos_cooldown_pregao = gerar_caminho_arquivo(etapa=6, subetapa=3, tipo_arquivo="base", nome="sinais_pos_cooldown_pregao")
caminho_tbl_resumo_cooldown_sinais = gerar_caminho_arquivo(etapa=6, subetapa=3, tipo_arquivo="tbl", nome="resumo_cooldown_sinais")
caminho_tbl_auditoria_cooldown_sinais = gerar_caminho_arquivo(etapa=6, subetapa=3, tipo_arquivo="tbl", nome="auditoria_cooldown_sinais")
caminho_tbl_datas_capitulacao_pos_cooldown = gerar_caminho_arquivo(etapa=6, subetapa=3, tipo_arquivo="tbl", nome="datas_capitulacao_pos_cooldown")
caminho_tbl_datas_euforia_pos_cooldown = gerar_caminho_arquivo(etapa=6, subetapa=3, tipo_arquivo="tbl", nome="datas_euforia_pos_cooldown")
caminho_tbl_sinais_rejeitados_cooldown = gerar_caminho_arquivo(etapa=6, subetapa=3, tipo_arquivo="tbl", nome="sinais_rejeitados_cooldown")

print(f"Entrada - parâmetros globais              : {caminho_tbl_parametros_globais}")
print(f"Entrada - base de capitulação bruta       : {caminho_base_capitulacao_bruta_pregao}")
print(f"Entrada - base de euforia bruta           : {caminho_base_euforia_bruta_pregao}")
print(f"Saída   - base de sinais pós-cooldown     : {caminho_base_sinais_pos_cooldown_pregao}")
print(f"Saída   - resumo do cooldown              : {caminho_tbl_resumo_cooldown_sinais}")
print(f"Saída   - auditoria do cooldown           : {caminho_tbl_auditoria_cooldown_sinais}")
print(f"Saída   - datas finais de capitulação     : {caminho_tbl_datas_capitulacao_pos_cooldown}")
print(f"Saída   - datas finais de euforia         : {caminho_tbl_datas_euforia_pos_cooldown}")
print(f"Saída   - sinais rejeitados no cooldown   : {caminho_tbl_sinais_rejeitados_cooldown}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_parametros_globais = pd.read_parquet(caminho_tbl_parametros_globais)
df_capitulacao_bruta_pregao = pd.read_parquet(caminho_base_capitulacao_bruta_pregao)
df_euforia_bruta_pregao = pd.read_parquet(caminho_base_euforia_bruta_pregao)

print(f"Parâmetros globais        : {df_parametros_globais.shape[0]:,} linhas x {df_parametros_globais.shape[1]} colunas")
print(f"Capitulação bruta pregão : {df_capitulacao_bruta_pregao.shape[0]:,} linhas x {df_capitulacao_bruta_pregao.shape[1]} colunas")
print(f"Euforia bruta pregão     : {df_euforia_bruta_pregao.shape[0]:,} linhas x {df_euforia_bruta_pregao.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

COLUNAS_OBRIGATORIAS_PARAMETROS = [
    "parametro",
    "valor",
]

COLUNAS_OBRIGATORIAS_CAPITULACAO = [
    "data",
    "flag_capitulacao_bruta",
    "score_capitulacao_bruta",
    "intensidade_capitulacao_bruta",
]

COLUNAS_OBRIGATORIAS_EUFORIA = [
    "data",
    "flag_euforia_bruta",
    "score_euforia_bruta",
    "intensidade_euforia_bruta",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def obter_parametro_global(df_parametros, parametro):
    """
    Recupera o valor de um parâmetro global salvo na etapa 1.1.
    """
    df_parametro = df_parametros.loc[df_parametros["parametro"].astype("string").str.strip().eq(parametro)].copy()

    if df_parametro.empty:
        raise ValueError(f"Parâmetro global não encontrado: {parametro}")

    return df_parametro["valor"].iloc[0]

def aplicar_cooldown_flags(df_base, coluna_flag_bruta, cooldown_pregoes, prefixo):
    """
    Aplica cooldown sequencial em uma série diária de flags de sinais.
    A convenção adotada é: um novo sinal da mesma natureza só pode ser validado
    quando a distância ao último sinal válido for maior ou igual ao cooldown.
    """
    df_base = df_base.sort_values("data").reset_index(drop=True).copy()

    flags_finais = []
    flags_rejeitadas = []
    pregoes_desde_ultimo = []
    datas_ultimo_sinal = []
    ordens_sinais = []

    ultimo_indice_valido = None
    ultima_data_valida = pd.NaT
    ordem_atual = 0

    for indice_linha, linha in df_base.iterrows():
        flag_bruta = bool(linha[coluna_flag_bruta])

        if ultimo_indice_valido is None:
            distancia = pd.NA
        else:
            distancia = int(indice_linha - ultimo_indice_valido)

        if flag_bruta:
            if (ultimo_indice_valido is None) or (distancia >= cooldown_pregoes):
                flag_final = True
                flag_rejeitada = False
                ultimo_indice_valido = indice_linha
                ultima_data_valida = linha["data"]
                ordem_atual += 1
                ordem_sinal = ordem_atual
            else:
                flag_final = False
                flag_rejeitada = True
                ordem_sinal = pd.NA
        else:
            flag_final = False
            flag_rejeitada = False
            ordem_sinal = pd.NA

        flags_finais.append(flag_final)
        flags_rejeitadas.append(flag_rejeitada)
        pregoes_desde_ultimo.append(distancia)
        datas_ultimo_sinal.append(ultima_data_valida if pd.notna(ultima_data_valida) else pd.NaT)
        ordens_sinais.append(ordem_sinal)

    df_base[f"pregoes_desde_ultimo_{prefixo}_valido"] = pregoes_desde_ultimo
    df_base[f"data_ultimo_{prefixo}_valido"] = datas_ultimo_sinal
    df_base[f"flag_{prefixo}_final"] = flags_finais
    df_base[f"flag_{prefixo}_rejeitado_cooldown"] = flags_rejeitadas
    df_base[f"ordem_{prefixo}_final"] = ordens_sinais

    return df_base

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Preparação das bases e leitura do cooldown oficial
# ============================================================

print("\n[5/10] Preparação das bases e leitura do cooldown oficial...")

validar_colunas_obrigatorias(df_parametros_globais, COLUNAS_OBRIGATORIAS_PARAMETROS)
validar_colunas_obrigatorias(df_capitulacao_bruta_pregao, COLUNAS_OBRIGATORIAS_CAPITULACAO)
validar_colunas_obrigatorias(df_euforia_bruta_pregao, COLUNAS_OBRIGATORIAS_EUFORIA)

COOLDOWN_SINAIS_PREGOES = int(float(obter_parametro_global(df_parametros_globais, "cooldown_sinais_pregoes")))

df_capitulacao = df_capitulacao_bruta_pregao.copy()
df_euforia = df_euforia_bruta_pregao.copy()

for df_base in [df_capitulacao, df_euforia]:
    df_base["data"] = pd.to_datetime(df_base["data"], errors="coerce")

df_capitulacao["flag_capitulacao_bruta"] = df_capitulacao["flag_capitulacao_bruta"].fillna(False).astype(bool)
df_euforia["flag_euforia_bruta"] = df_euforia["flag_euforia_bruta"].fillna(False).astype(bool)

df_capitulacao["score_capitulacao_bruta"] = pd.to_numeric(df_capitulacao["score_capitulacao_bruta"], errors="coerce")
df_capitulacao["intensidade_capitulacao_bruta"] = pd.to_numeric(df_capitulacao["intensidade_capitulacao_bruta"], errors="coerce")
df_euforia["score_euforia_bruta"] = pd.to_numeric(df_euforia["score_euforia_bruta"], errors="coerce")
df_euforia["intensidade_euforia_bruta"] = pd.to_numeric(df_euforia["intensidade_euforia_bruta"], errors="coerce")

n_duplicidades_capitulacao = int(df_capitulacao.duplicated(subset=["data"]).sum())
n_duplicidades_euforia = int(df_euforia.duplicated(subset=["data"]).sum())

if n_duplicidades_capitulacao > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_capitulacao} duplicidades por data na base de capitulação bruta.")
if n_duplicidades_euforia > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_euforia} duplicidades por data na base de euforia bruta.")

print(f"Cooldown oficial carregado             : {COOLDOWN_SINAIS_PREGOES} pregões")
print(f"Duplicidades na base capitulação bruta : {n_duplicidades_capitulacao}")
print(f"Duplicidades na base euforia bruta     : {n_duplicidades_euforia}")
print("OK")

# ============================================================
# 6) Aplicação do cooldown separadamente por natureza do sinal
# ============================================================

print("\n[6/10] Aplicação do cooldown separadamente por natureza do sinal...")

df_capitulacao_cooldown = aplicar_cooldown_flags(
    df_base=df_capitulacao[["data", "flag_capitulacao_bruta", "score_capitulacao_bruta", "intensidade_capitulacao_bruta"]].copy(),
    coluna_flag_bruta="flag_capitulacao_bruta",
    cooldown_pregoes=COOLDOWN_SINAIS_PREGOES,
    prefixo="capitulacao",
)

df_euforia_cooldown = aplicar_cooldown_flags(
    df_base=df_euforia[["data", "flag_euforia_bruta", "score_euforia_bruta", "intensidade_euforia_bruta"]].copy(),
    coluna_flag_bruta="flag_euforia_bruta",
    cooldown_pregoes=COOLDOWN_SINAIS_PREGOES,
    prefixo="euforia",
)

df_sinais_pos_cooldown_pregao = (
    df_capitulacao_cooldown
    .merge(
        df_euforia_cooldown,
        on="data",
        how="outer",
        validate="one_to_one",
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_sinais_pos_cooldown_pregao["ordem_pregao"] = range(1, len(df_sinais_pos_cooldown_pregao) + 1)

for coluna_flag in [
    "flag_capitulacao_bruta",
    "flag_capitulacao_final",
    "flag_capitulacao_rejeitado_cooldown",
    "flag_euforia_bruta",
    "flag_euforia_final",
    "flag_euforia_rejeitado_cooldown",
]:
    df_sinais_pos_cooldown_pregao[coluna_flag] = df_sinais_pos_cooldown_pregao[coluna_flag].fillna(False).astype(bool)

for coluna_numerica in [
    "score_capitulacao_bruta",
    "intensidade_capitulacao_bruta",
    "pregoes_desde_ultimo_capitulacao_valido",
    "ordem_capitulacao_final",
    "score_euforia_bruta",
    "intensidade_euforia_bruta",
    "pregoes_desde_ultimo_euforia_valido",
    "ordem_euforia_final",
]:
    df_sinais_pos_cooldown_pregao[coluna_numerica] = pd.to_numeric(df_sinais_pos_cooldown_pregao[coluna_numerica], errors="coerce")

df_sinais_pos_cooldown_pregao["flag_conflito_mesma_data_bruta"] = (
    df_sinais_pos_cooldown_pregao["flag_capitulacao_bruta"]
    & df_sinais_pos_cooldown_pregao["flag_euforia_bruta"]
)

df_sinais_pos_cooldown_pregao["flag_conflito_mesma_data_final"] = (
    df_sinais_pos_cooldown_pregao["flag_capitulacao_final"]
    & df_sinais_pos_cooldown_pregao["flag_euforia_final"]
)

df_sinais_pos_cooldown_pregao["flag_sinal_final_qualquer"] = (
    df_sinais_pos_cooldown_pregao["flag_capitulacao_final"]
    | df_sinais_pos_cooldown_pregao["flag_euforia_final"]
)

df_datas_capitulacao_pos_cooldown = (
    df_sinais_pos_cooldown_pregao
    .loc[df_sinais_pos_cooldown_pregao["flag_capitulacao_final"]]
    .copy()
)

df_datas_euforia_pos_cooldown = (
    df_sinais_pos_cooldown_pregao
    .loc[df_sinais_pos_cooldown_pregao["flag_euforia_final"]]
    .copy()
)

df_sinais_rejeitados_cooldown = (
    df_sinais_pos_cooldown_pregao
    .loc[
        df_sinais_pos_cooldown_pregao["flag_capitulacao_rejeitado_cooldown"]
        | df_sinais_pos_cooldown_pregao["flag_euforia_rejeitado_cooldown"]
    ]
    .copy()
)

print(f"Datas finais de capitulação após cooldown : {len(df_datas_capitulacao_pos_cooldown):,}")
print(f"Datas finais de euforia após cooldown     : {len(df_datas_euforia_pos_cooldown):,}")
print(f"Sinais rejeitados pelo cooldown           : {len(df_sinais_rejeitados_cooldown):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria do cooldown
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria do cooldown...")

n_capitulacao_bruta = int(df_sinais_pos_cooldown_pregao["flag_capitulacao_bruta"].sum())
n_capitulacao_final = int(df_sinais_pos_cooldown_pregao["flag_capitulacao_final"].sum())
n_capitulacao_rejeitada = int(df_sinais_pos_cooldown_pregao["flag_capitulacao_rejeitado_cooldown"].sum())

n_euforia_bruta = int(df_sinais_pos_cooldown_pregao["flag_euforia_bruta"].sum())
n_euforia_final = int(df_sinais_pos_cooldown_pregao["flag_euforia_final"].sum())
n_euforia_rejeitada = int(df_sinais_pos_cooldown_pregao["flag_euforia_rejeitado_cooldown"].sum())

df_resumo_cooldown_sinais = pd.DataFrame(
    [
        {
            "tipo_sinal": "capitulacao",
            "cooldown_pregoes": COOLDOWN_SINAIS_PREGOES,
            "n_sinais_brutos": n_capitulacao_bruta,
            "n_sinais_finais": n_capitulacao_final,
            "n_sinais_rejeitados_cooldown": n_capitulacao_rejeitada,
            "pct_preservacao_sinais": (n_capitulacao_final / n_capitulacao_bruta) if n_capitulacao_bruta > 0 else pd.NA,
            "data_primeiro_sinal_final": df_datas_capitulacao_pos_cooldown["data"].min() if not df_datas_capitulacao_pos_cooldown.empty else pd.NaT,
            "data_ultimo_sinal_final": df_datas_capitulacao_pos_cooldown["data"].max() if not df_datas_capitulacao_pos_cooldown.empty else pd.NaT,
        },
        {
            "tipo_sinal": "euforia",
            "cooldown_pregoes": COOLDOWN_SINAIS_PREGOES,
            "n_sinais_brutos": n_euforia_bruta,
            "n_sinais_finais": n_euforia_final,
            "n_sinais_rejeitados_cooldown": n_euforia_rejeitada,
            "pct_preservacao_sinais": (n_euforia_final / n_euforia_bruta) if n_euforia_bruta > 0 else pd.NA,
            "data_primeiro_sinal_final": df_datas_euforia_pos_cooldown["data"].min() if not df_datas_euforia_pos_cooldown.empty else pd.NaT,
            "data_ultimo_sinal_final": df_datas_euforia_pos_cooldown["data"].max() if not df_datas_euforia_pos_cooldown.empty else pd.NaT,
        },
    ]
)

df_auditoria_cooldown_sinais = pd.DataFrame(
    [
        {"metrica": "cooldown_sinais_pregoes", "valor": COOLDOWN_SINAIS_PREGOES},
        {"metrica": "n_datas_total_base", "valor": int(len(df_sinais_pos_cooldown_pregao))},
        {"metrica": "n_capitulacao_bruta", "valor": n_capitulacao_bruta},
        {"metrica": "n_capitulacao_final", "valor": n_capitulacao_final},
        {"metrica": "n_capitulacao_rejeitada_cooldown", "valor": n_capitulacao_rejeitada},
        {"metrica": "n_euforia_bruta", "valor": n_euforia_bruta},
        {"metrica": "n_euforia_final", "valor": n_euforia_final},
        {"metrica": "n_euforia_rejeitada_cooldown", "valor": n_euforia_rejeitada},
        {"metrica": "n_conflito_mesma_data_bruta", "valor": int(df_sinais_pos_cooldown_pregao["flag_conflito_mesma_data_bruta"].sum())},
        {"metrica": "n_conflito_mesma_data_final", "valor": int(df_sinais_pos_cooldown_pregao["flag_conflito_mesma_data_final"].sum())},
        {"metrica": "n_datas_com_qualquer_sinal_final", "valor": int(df_sinais_pos_cooldown_pregao["flag_sinal_final_qualquer"].sum())},
    ]
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_sinais_pos_cooldown_pregao, caminho_base_sinais_pos_cooldown_pregao, index=False)
salvar_dataframe(df_resumo_cooldown_sinais, caminho_tbl_resumo_cooldown_sinais, index=False)
salvar_dataframe(df_auditoria_cooldown_sinais, caminho_tbl_auditoria_cooldown_sinais, index=False)
salvar_dataframe(df_datas_capitulacao_pos_cooldown, caminho_tbl_datas_capitulacao_pos_cooldown, index=False)
salvar_dataframe(df_datas_euforia_pos_cooldown, caminho_tbl_datas_euforia_pos_cooldown, index=False)
salvar_dataframe(df_sinais_rejeitados_cooldown, caminho_tbl_sinais_rejeitados_cooldown, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_sinais_pos_cooldown = df_sinais_pos_cooldown_pregao.head(15).copy()
df_amostra_capitulacao_final = df_datas_capitulacao_pos_cooldown.head(15).copy()
df_amostra_euforia_final = df_datas_euforia_pos_cooldown.head(15).copy()
df_amostra_sinais_rejeitados = df_sinais_rejeitados_cooldown.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo do cooldown por natureza de sinal:")
print(df_resumo_cooldown_sinais.to_string(index=False))

print("\nAuditoria consolidada do cooldown:")
print(df_auditoria_cooldown_sinais.to_string(index=False))

print("\nAmostra inicial da base pós-cooldown:")
print(df_amostra_sinais_pos_cooldown.to_string(index=False))

print("\nAmostra das datas finais de capitulação após cooldown:")
print(df_amostra_capitulacao_final.to_string(index=False))

print("\nAmostra das datas finais de euforia após cooldown:")
print(df_amostra_euforia_final.to_string(index=False))

print("\nAmostra dos sinais rejeitados pelo cooldown:")
print(df_amostra_sinais_rejeitados.to_string(index=False))

print("\nArquivos salvos na subetapa 6.3:")
print(f"- {caminho_base_sinais_pos_cooldown_pregao}")
print(f"- {caminho_tbl_resumo_cooldown_sinais}")
print(f"- {caminho_tbl_auditoria_cooldown_sinais}")
print(f"- {caminho_tbl_datas_capitulacao_pos_cooldown}")
print(f"- {caminho_tbl_datas_euforia_pos_cooldown}")
print(f"- {caminho_tbl_sinais_rejeitados_cooldown}")

print("\nETAPA 6.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 6.3 - APLICAÇÃO DO COOLDOWN

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - parâmetros globais              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_1_tbl_parametros_globais.parquet
Entrada - base de capitulação bruta       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_1_base_capitulacao_bruta_pregao.parquet
Entrada - base de euforia bruta           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_2_base_euforia_bruta_pregao.parquet
Saída   - base de sinais pós-cooldown     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_3_base_sinais_pos_cooldown_pregao.

## Etapa 6.4) Calendário Final de Sinais

In [28]:
%%time
# ============================================================
# Etapa 6.4) Calendário Final de Sinais
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 6.4 - CALENDÁRIO FINAL DE SINAIS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_capitulacao_bruta_pregao = gerar_caminho_arquivo(etapa=6, subetapa=1, tipo_arquivo="base", nome="capitulacao_bruta_pregao")
caminho_base_euforia_bruta_pregao = gerar_caminho_arquivo(etapa=6, subetapa=2, tipo_arquivo="base", nome="euforia_bruta_pregao")
caminho_base_sinais_pos_cooldown_pregao = gerar_caminho_arquivo(etapa=6, subetapa=3, tipo_arquivo="base", nome="sinais_pos_cooldown_pregao")
caminho_tbl_resumo_cooldown_sinais = gerar_caminho_arquivo(etapa=6, subetapa=3, tipo_arquivo="tbl", nome="resumo_cooldown_sinais")

caminho_base_calendario_final_sinais = gerar_caminho_arquivo(etapa=6, subetapa=4, tipo_arquivo="base", nome="calendario_final_sinais")
caminho_tbl_datas_finais_capitulacao = gerar_caminho_arquivo(etapa=6, subetapa=4, tipo_arquivo="tbl", nome="datas_finais_capitulacao")
caminho_tbl_datas_finais_euforia = gerar_caminho_arquivo(etapa=6, subetapa=4, tipo_arquivo="tbl", nome="datas_finais_euforia")
caminho_tbl_auditoria_antes_depois_cooldown = gerar_caminho_arquivo(etapa=6, subetapa=4, tipo_arquivo="tbl", nome="auditoria_antes_depois_cooldown")
caminho_tbl_distribuicao_anual_sinais_finais = gerar_caminho_arquivo(etapa=6, subetapa=4, tipo_arquivo="tbl", nome="distribuicao_anual_sinais_finais")
caminho_tbl_resumo_calendario_final_sinais = gerar_caminho_arquivo(etapa=6, subetapa=4, tipo_arquivo="tbl", nome="resumo_calendario_final_sinais")

print(f"Entrada - capitulação bruta             : {caminho_base_capitulacao_bruta_pregao}")
print(f"Entrada - euforia bruta                 : {caminho_base_euforia_bruta_pregao}")
print(f"Entrada - sinais pós-cooldown           : {caminho_base_sinais_pos_cooldown_pregao}")
print(f"Entrada - resumo do cooldown            : {caminho_tbl_resumo_cooldown_sinais}")
print(f"Saída   - calendário final de sinais    : {caminho_base_calendario_final_sinais}")
print(f"Saída   - datas finais de capitulação   : {caminho_tbl_datas_finais_capitulacao}")
print(f"Saída   - datas finais de euforia       : {caminho_tbl_datas_finais_euforia}")
print(f"Saída   - auditoria antes e depois      : {caminho_tbl_auditoria_antes_depois_cooldown}")
print(f"Saída   - distribuição anual            : {caminho_tbl_distribuicao_anual_sinais_finais}")
print(f"Saída   - resumo do calendário final    : {caminho_tbl_resumo_calendario_final_sinais}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_capitulacao_bruta_pregao = pd.read_parquet(caminho_base_capitulacao_bruta_pregao)
df_euforia_bruta_pregao = pd.read_parquet(caminho_base_euforia_bruta_pregao)
df_sinais_pos_cooldown_pregao = pd.read_parquet(caminho_base_sinais_pos_cooldown_pregao)
df_resumo_cooldown_sinais = pd.read_parquet(caminho_tbl_resumo_cooldown_sinais)

print(f"Capitulação bruta pregão : {df_capitulacao_bruta_pregao.shape[0]:,} linhas x {df_capitulacao_bruta_pregao.shape[1]} colunas")
print(f"Euforia bruta pregão     : {df_euforia_bruta_pregao.shape[0]:,} linhas x {df_euforia_bruta_pregao.shape[1]} colunas")
print(f"Sinais pós-cooldown      : {df_sinais_pos_cooldown_pregao.shape[0]:,} linhas x {df_sinais_pos_cooldown_pregao.shape[1]} colunas")
print(f"Resumo do cooldown       : {df_resumo_cooldown_sinais.shape[0]:,} linhas x {df_resumo_cooldown_sinais.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e colunas obrigatórias
# ============================================================

print("\n[4/10] Funções auxiliares e colunas obrigatórias...")

COLUNAS_OBRIGATORIAS_CAPITULACAO = [
    "data",
    "flag_capitulacao_bruta",
    "score_capitulacao_bruta",
    "intensidade_capitulacao_bruta",
]

COLUNAS_OBRIGATORIAS_EUFORIA = [
    "data",
    "flag_euforia_bruta",
    "score_euforia_bruta",
    "intensidade_euforia_bruta",
]

COLUNAS_OBRIGATORIAS_COOLDOWN = [
    "data",
    "flag_capitulacao_bruta",
    "flag_capitulacao_final",
    "flag_capitulacao_rejeitado_cooldown",
    "ordem_capitulacao_final",
    "flag_euforia_bruta",
    "flag_euforia_final",
    "flag_euforia_rejeitado_cooldown",
    "ordem_euforia_final",
    "flag_conflito_mesma_data_bruta",
    "flag_conflito_mesma_data_final",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def preparar_datas(df_base):
    """
    Padroniza a coluna de data para datetime.
    """
    df_base = df_base.copy()
    df_base["data"] = pd.to_datetime(df_base["data"], errors="coerce")
    return df_base

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

validar_colunas_obrigatorias(df_capitulacao_bruta_pregao, COLUNAS_OBRIGATORIAS_CAPITULACAO)
validar_colunas_obrigatorias(df_euforia_bruta_pregao, COLUNAS_OBRIGATORIAS_EUFORIA)
validar_colunas_obrigatorias(df_sinais_pos_cooldown_pregao, COLUNAS_OBRIGATORIAS_COOLDOWN)

df_capitulacao_bruta_pregao = preparar_datas(df_capitulacao_bruta_pregao)
df_euforia_bruta_pregao = preparar_datas(df_euforia_bruta_pregao)
df_sinais_pos_cooldown_pregao = preparar_datas(df_sinais_pos_cooldown_pregao)

for coluna_flag in [
    "flag_capitulacao_bruta",
    "flag_capitulacao_final",
    "flag_capitulacao_rejeitado_cooldown",
    "flag_euforia_bruta",
    "flag_euforia_final",
    "flag_euforia_rejeitado_cooldown",
    "flag_conflito_mesma_data_bruta",
    "flag_conflito_mesma_data_final",
]:
    df_sinais_pos_cooldown_pregao[coluna_flag] = df_sinais_pos_cooldown_pregao[coluna_flag].fillna(False).astype(bool)

for coluna_numerica in [
    "score_capitulacao_bruta",
    "intensidade_capitulacao_bruta",
    "ordem_capitulacao_final",
    "score_euforia_bruta",
    "intensidade_euforia_bruta",
    "ordem_euforia_final",
]:
    if coluna_numerica in df_sinais_pos_cooldown_pregao.columns:
        df_sinais_pos_cooldown_pregao[coluna_numerica] = pd.to_numeric(df_sinais_pos_cooldown_pregao[coluna_numerica], errors="coerce")

n_duplicidades_data_capitulacao = int(df_capitulacao_bruta_pregao.duplicated(subset=["data"]).sum())
n_duplicidades_data_euforia = int(df_euforia_bruta_pregao.duplicated(subset=["data"]).sum())
n_duplicidades_data_cooldown = int(df_sinais_pos_cooldown_pregao.duplicated(subset=["data"]).sum())

if n_duplicidades_data_capitulacao > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_data_capitulacao} duplicidades por data na base de capitulação bruta.")
if n_duplicidades_data_euforia > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_data_euforia} duplicidades por data na base de euforia bruta.")
if n_duplicidades_data_cooldown > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_data_cooldown} duplicidades por data na base pós-cooldown.")

print(f"Duplicidades por data - capitulação bruta : {n_duplicidades_data_capitulacao}")
print(f"Duplicidades por data - euforia bruta     : {n_duplicidades_data_euforia}")
print(f"Duplicidades por data - pós-cooldown      : {n_duplicidades_data_cooldown}")
print("OK")

# ============================================================
# 6) Construção do calendário final de sinais
# ============================================================

print("\n[6/10] Construção do calendário final de sinais...")

df_datas_finais_capitulacao = (
    df_sinais_pos_cooldown_pregao
    .loc[df_sinais_pos_cooldown_pregao["flag_capitulacao_final"]]
    [
        [
            "data",
            "ordem_capitulacao_final",
            "score_capitulacao_bruta",
            "intensidade_capitulacao_bruta",
            "flag_capitulacao_bruta",
            "flag_capitulacao_final",
            "flag_capitulacao_rejeitado_cooldown",
        ]
    ]
    .copy()
    .rename(
        columns={
            "ordem_capitulacao_final": "ordem_sinal_final",
            "score_capitulacao_bruta": "score_sinal_bruto",
            "intensidade_capitulacao_bruta": "intensidade_sinal_bruto",
        }
    )
)

df_datas_finais_capitulacao["tipo_sinal"] = "capitulacao"
df_datas_finais_capitulacao["direcao_sinal"] = "contrarian_bullish"
df_datas_finais_capitulacao["ordem_sinal_final"] = pd.to_numeric(df_datas_finais_capitulacao["ordem_sinal_final"], errors="coerce")
df_datas_finais_capitulacao["ano"] = df_datas_finais_capitulacao["data"].dt.year
df_datas_finais_capitulacao["mes"] = df_datas_finais_capitulacao["data"].dt.month

df_datas_finais_euforia = (
    df_sinais_pos_cooldown_pregao
    .loc[df_sinais_pos_cooldown_pregao["flag_euforia_final"]]
    [
        [
            "data",
            "ordem_euforia_final",
            "score_euforia_bruta",
            "intensidade_euforia_bruta",
            "flag_euforia_bruta",
            "flag_euforia_final",
            "flag_euforia_rejeitado_cooldown",
        ]
    ]
    .copy()
    .rename(
        columns={
            "ordem_euforia_final": "ordem_sinal_final",
            "score_euforia_bruta": "score_sinal_bruto",
            "intensidade_euforia_bruta": "intensidade_sinal_bruto",
        }
    )
)

df_datas_finais_euforia["tipo_sinal"] = "euforia"
df_datas_finais_euforia["direcao_sinal"] = "contrarian_bearish"
df_datas_finais_euforia["ordem_sinal_final"] = pd.to_numeric(df_datas_finais_euforia["ordem_sinal_final"], errors="coerce")
df_datas_finais_euforia["ano"] = df_datas_finais_euforia["data"].dt.year
df_datas_finais_euforia["mes"] = df_datas_finais_euforia["data"].dt.month

df_calendario_final_sinais = (
    pd.concat(
        [
            df_datas_finais_capitulacao[
                [
                    "data",
                    "tipo_sinal",
                    "direcao_sinal",
                    "ordem_sinal_final",
                    "score_sinal_bruto",
                    "intensidade_sinal_bruto",
                    "ano",
                    "mes",
                ]
            ],
            df_datas_finais_euforia[
                [
                    "data",
                    "tipo_sinal",
                    "direcao_sinal",
                    "ordem_sinal_final",
                    "score_sinal_bruto",
                    "intensidade_sinal_bruto",
                    "ano",
                    "mes",
                ]
            ],
        ],
        axis=0,
        ignore_index=True,
    )
    .sort_values(["data", "tipo_sinal"])
    .reset_index(drop=True)
)

df_calendario_final_sinais["ordem_global_sinal"] = range(1, len(df_calendario_final_sinais) + 1)

print(f"Datas finais de capitulação : {len(df_datas_finais_capitulacao):,}")
print(f"Datas finais de euforia     : {len(df_datas_finais_euforia):,}")
print(f"Calendário final consolidado: {len(df_calendario_final_sinais):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas de auditoria e resumo
# ============================================================

print("\n[7/10] Construção das tabelas de auditoria e resumo...")

df_auditoria_antes_depois_cooldown = pd.DataFrame(
    [
        {
            "tipo_sinal": "capitulacao",
            "n_sinais_brutos": int(df_sinais_pos_cooldown_pregao["flag_capitulacao_bruta"].sum()),
            "n_sinais_finais": int(df_sinais_pos_cooldown_pregao["flag_capitulacao_final"].sum()),
            "n_sinais_rejeitados_cooldown": int(df_sinais_pos_cooldown_pregao["flag_capitulacao_rejeitado_cooldown"].sum()),
            "pct_preservacao_sinais": (
                float(df_sinais_pos_cooldown_pregao["flag_capitulacao_final"].sum() / df_sinais_pos_cooldown_pregao["flag_capitulacao_bruta"].sum())
                if int(df_sinais_pos_cooldown_pregao["flag_capitulacao_bruta"].sum()) > 0 else pd.NA
            ),
            "data_primeiro_sinal_final": df_datas_finais_capitulacao["data"].min() if not df_datas_finais_capitulacao.empty else pd.NaT,
            "data_ultimo_sinal_final": df_datas_finais_capitulacao["data"].max() if not df_datas_finais_capitulacao.empty else pd.NaT,
        },
        {
            "tipo_sinal": "euforia",
            "n_sinais_brutos": int(df_sinais_pos_cooldown_pregao["flag_euforia_bruta"].sum()),
            "n_sinais_finais": int(df_sinais_pos_cooldown_pregao["flag_euforia_final"].sum()),
            "n_sinais_rejeitados_cooldown": int(df_sinais_pos_cooldown_pregao["flag_euforia_rejeitado_cooldown"].sum()),
            "pct_preservacao_sinais": (
                float(df_sinais_pos_cooldown_pregao["flag_euforia_final"].sum() / df_sinais_pos_cooldown_pregao["flag_euforia_bruta"].sum())
                if int(df_sinais_pos_cooldown_pregao["flag_euforia_bruta"].sum()) > 0 else pd.NA
            ),
            "data_primeiro_sinal_final": df_datas_finais_euforia["data"].min() if not df_datas_finais_euforia.empty else pd.NaT,
            "data_ultimo_sinal_final": df_datas_finais_euforia["data"].max() if not df_datas_finais_euforia.empty else pd.NaT,
        },
    ]
)

df_distribuicao_anual_sinais_finais = (
    df_calendario_final_sinais
    .groupby(["ano", "tipo_sinal"], as_index=False)
    .agg(
        n_sinais_finais=("data", "count"),
        intensidade_mediana=("intensidade_sinal_bruto", "median"),
        score_medio=("score_sinal_bruto", "mean"),
    )
    .sort_values(["ano", "tipo_sinal"])
    .reset_index(drop=True)
)

df_resumo_calendario_final_sinais = pd.DataFrame(
    [
        {
            "base": "calendario_final_sinais",
            "n_sinais_finais_total": int(len(df_calendario_final_sinais)),
            "n_sinais_finais_capitulacao": int(len(df_datas_finais_capitulacao)),
            "n_sinais_finais_euforia": int(len(df_datas_finais_euforia)),
            "data_min": df_calendario_final_sinais["data"].min() if not df_calendario_final_sinais.empty else pd.NaT,
            "data_max": df_calendario_final_sinais["data"].max() if not df_calendario_final_sinais.empty else pd.NaT,
            "n_anos_com_sinais": int(df_calendario_final_sinais["ano"].nunique()) if not df_calendario_final_sinais.empty else 0,
            "n_conflitos_mesma_data_final": int(df_sinais_pos_cooldown_pregao["flag_conflito_mesma_data_final"].sum()),
        }
    ]
)

print("Tabelas de auditoria e resumo construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_calendario_final_sinais, caminho_base_calendario_final_sinais, index=False)
salvar_dataframe(df_datas_finais_capitulacao, caminho_tbl_datas_finais_capitulacao, index=False)
salvar_dataframe(df_datas_finais_euforia, caminho_tbl_datas_finais_euforia, index=False)
salvar_dataframe(df_auditoria_antes_depois_cooldown, caminho_tbl_auditoria_antes_depois_cooldown, index=False)
salvar_dataframe(df_distribuicao_anual_sinais_finais, caminho_tbl_distribuicao_anual_sinais_finais, index=False)
salvar_dataframe(df_resumo_calendario_final_sinais, caminho_tbl_resumo_calendario_final_sinais, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_calendario_final_sinais = df_calendario_final_sinais.head(15).copy()
df_amostra_datas_finais_capitulacao = df_datas_finais_capitulacao.head(15).copy()
df_amostra_datas_finais_euforia = df_datas_finais_euforia.head(15).copy()
df_amostra_distribuicao_anual = df_distribuicao_anual_sinais_finais.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo do calendário final de sinais:")
print(df_resumo_calendario_final_sinais.to_string(index=False))

print("\nAuditoria antes e depois do cooldown:")
print(df_auditoria_antes_depois_cooldown.to_string(index=False))

print("\nAmostra do calendário final consolidado:")
print(df_amostra_calendario_final_sinais.to_string(index=False))

print("\nAmostra das datas finais de capitulação:")
print(df_amostra_datas_finais_capitulacao.to_string(index=False))

print("\nAmostra das datas finais de euforia:")
print(df_amostra_datas_finais_euforia.to_string(index=False))

print("\nDistribuição anual dos sinais finais - amostra:")
print(df_amostra_distribuicao_anual.to_string(index=False))

print("\nArquivos salvos na subetapa 6.4:")
print(f"- {caminho_base_calendario_final_sinais}")
print(f"- {caminho_tbl_datas_finais_capitulacao}")
print(f"- {caminho_tbl_datas_finais_euforia}")
print(f"- {caminho_tbl_auditoria_antes_depois_cooldown}")
print(f"- {caminho_tbl_distribuicao_anual_sinais_finais}")
print(f"- {caminho_tbl_resumo_calendario_final_sinais}")

print("\nETAPA 6.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 6.4 - CALENDÁRIO FINAL DE SINAIS

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - capitulação bruta             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_1_base_capitulacao_bruta_pregao.parquet
Entrada - euforia bruta                 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_2_base_euforia_bruta_pregao.parquet
Entrada - sinais pós-cooldown           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_3_base_sinais_pos_cooldown_pregao.parquet
Entrada - resumo do cooldown            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_3_tbl_resumo_cooldown_sinais

# Etapa 7) Construção das Bases de Aportes das Estratégias

## Etapa 7.1) Contagem Final de Sinais por Estratégia

In [29]:
%%time
# ============================================================
# Etapa 7.1) Contagem Final de Sinais por Estratégia
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 7.1 - CONTAGEM FINAL DE SINAIS POR ESTRATÉGIA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_tbl_parametros_globais = gerar_caminho_arquivo(etapa=1, subetapa=1, tipo_arquivo="tbl", nome="parametros_globais")
caminho_base_calendario_final_sinais = gerar_caminho_arquivo(etapa=6, subetapa=4, tipo_arquivo="base", nome="calendario_final_sinais")
caminho_tbl_datas_finais_capitulacao = gerar_caminho_arquivo(etapa=6, subetapa=4, tipo_arquivo="tbl", nome="datas_finais_capitulacao")
caminho_tbl_datas_finais_euforia = gerar_caminho_arquivo(etapa=6, subetapa=4, tipo_arquivo="tbl", nome="datas_finais_euforia")

caminho_base_sinais_finais_dimensionamento = gerar_caminho_arquivo(etapa=7, subetapa=1, tipo_arquivo="base", nome="sinais_finais_dimensionamento")
caminho_tbl_contagem_final_sinais_estrategia = gerar_caminho_arquivo(etapa=7, subetapa=1, tipo_arquivo="tbl", nome="contagem_final_sinais_estrategia")
caminho_tbl_parametros_dimensionamento_aportes = gerar_caminho_arquivo(etapa=7, subetapa=1, tipo_arquivo="tbl", nome="parametros_dimensionamento_aportes")
caminho_tbl_auditoria_contagem_sinais = gerar_caminho_arquivo(etapa=7, subetapa=1, tipo_arquivo="tbl", nome="auditoria_contagem_sinais")
caminho_tbl_distribuicao_anual_sinais_estrategia = gerar_caminho_arquivo(etapa=7, subetapa=1, tipo_arquivo="tbl", nome="distribuicao_anual_sinais_estrategia")

print(f"Entrada - parâmetros globais                  : {caminho_tbl_parametros_globais}")
print(f"Entrada - calendário final de sinais          : {caminho_base_calendario_final_sinais}")
print(f"Entrada - datas finais de capitulação         : {caminho_tbl_datas_finais_capitulacao}")
print(f"Entrada - datas finais de euforia             : {caminho_tbl_datas_finais_euforia}")
print(f"Saída   - base de sinais para dimensionamento : {caminho_base_sinais_finais_dimensionamento}")
print(f"Saída   - contagem final por estratégia       : {caminho_tbl_contagem_final_sinais_estrategia}")
print(f"Saída   - parâmetros de dimensionamento       : {caminho_tbl_parametros_dimensionamento_aportes}")
print(f"Saída   - auditoria de contagem               : {caminho_tbl_auditoria_contagem_sinais}")
print(f"Saída   - distribuição anual                  : {caminho_tbl_distribuicao_anual_sinais_estrategia}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_parametros_globais = pd.read_parquet(caminho_tbl_parametros_globais)
df_calendario_final_sinais = pd.read_parquet(caminho_base_calendario_final_sinais)
df_datas_finais_capitulacao = pd.read_parquet(caminho_tbl_datas_finais_capitulacao)
df_datas_finais_euforia = pd.read_parquet(caminho_tbl_datas_finais_euforia)

print(f"Parâmetros globais          : {df_parametros_globais.shape[0]:,} linhas x {df_parametros_globais.shape[1]} colunas")
print(f"Calendário final de sinais  : {df_calendario_final_sinais.shape[0]:,} linhas x {df_calendario_final_sinais.shape[1]} colunas")
print(f"Datas finais de capitulação : {df_datas_finais_capitulacao.shape[0]:,} linhas x {df_datas_finais_capitulacao.shape[1]} colunas")
print(f"Datas finais de euforia     : {df_datas_finais_euforia.shape[0]:,} linhas x {df_datas_finais_euforia.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e colunas obrigatórias
# ============================================================

print("\n[4/10] Funções auxiliares e colunas obrigatórias...")

COLUNAS_OBRIGATORIAS_PARAMETROS = [
    "parametro",
    "valor",
]

COLUNAS_OBRIGATORIAS_CALENDARIO = [
    "data",
    "tipo_sinal",
    "direcao_sinal",
    "ordem_sinal_final",
    "score_sinal_bruto",
    "intensidade_sinal_bruto",
    "ano",
    "mes",
    "ordem_global_sinal",
]

COLUNAS_OBRIGATORIAS_CAPITULACAO = [
    "data",
    "ordem_sinal_final",
    "score_sinal_bruto",
    "intensidade_sinal_bruto",
    "flag_capitulacao_bruta",
    "flag_capitulacao_final",
    "flag_capitulacao_rejeitado_cooldown",
    "tipo_sinal",
    "direcao_sinal",
    "ano",
    "mes",
]

COLUNAS_OBRIGATORIAS_EUFORIA = [
    "data",
    "ordem_sinal_final",
    "score_sinal_bruto",
    "intensidade_sinal_bruto",
    "flag_euforia_bruta",
    "flag_euforia_final",
    "flag_euforia_rejeitado_cooldown",
    "tipo_sinal",
    "direcao_sinal",
    "ano",
    "mes",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def obter_parametro_global(df_parametros, parametro):
    """
    Recupera o valor de um parâmetro global salvo na etapa 1.1.
    """
    df_parametro = df_parametros.loc[df_parametros["parametro"].astype("string").str.strip().eq(parametro)].copy()

    if df_parametro.empty:
        raise ValueError(f"Parâmetro global não encontrado: {parametro}")

    return df_parametro["valor"].iloc[0]

def construir_tabela_contagem(df_base, tipo_sinal, capital_inicial_estrategia):
    """
    Constrói a tabela-resumo de contagem final dos sinais por estratégia.
    """
    n_sinais = int(len(df_base))
    return {
        "tipo_sinal": tipo_sinal,
        "n_sinais_finais": n_sinais,
        "capital_inicial_estrategia": float(capital_inicial_estrategia),
        "data_primeiro_sinal": df_base["data"].min() if n_sinais > 0 else pd.NaT,
        "data_ultimo_sinal": df_base["data"].max() if n_sinais > 0 else pd.NaT,
        "ordem_primeiro_sinal": float(df_base["ordem_sinal_final"].min()) if n_sinais > 0 else pd.NA,
        "ordem_ultimo_sinal": float(df_base["ordem_sinal_final"].max()) if n_sinais > 0 else pd.NA,
        "score_medio_sinal": float(df_base["score_sinal_bruto"].mean()) if n_sinais > 0 else pd.NA,
        "intensidade_mediana_sinal": float(df_base["intensidade_sinal_bruto"].median()) if n_sinais > 0 else pd.NA,
        "n_anos_com_sinais": int(df_base["ano"].nunique(dropna=True)) if n_sinais > 0 else 0,
    }

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

validar_colunas_obrigatorias(df_parametros_globais, COLUNAS_OBRIGATORIAS_PARAMETROS)
validar_colunas_obrigatorias(df_calendario_final_sinais, COLUNAS_OBRIGATORIAS_CALENDARIO)
validar_colunas_obrigatorias(df_datas_finais_capitulacao, COLUNAS_OBRIGATORIAS_CAPITULACAO)
validar_colunas_obrigatorias(df_datas_finais_euforia, COLUNAS_OBRIGATORIAS_EUFORIA)

CAPITAL_INICIAL_ESTRATEGIA = float(obter_parametro_global(df_parametros_globais, "capital_inicial_estrategia"))

for df_base in [df_calendario_final_sinais, df_datas_finais_capitulacao, df_datas_finais_euforia]:
    df_base["data"] = pd.to_datetime(df_base["data"], errors="coerce")
    df_base["tipo_sinal"] = df_base["tipo_sinal"].astype("string").str.strip().str.lower()
    df_base["direcao_sinal"] = df_base["direcao_sinal"].astype("string").str.strip().str.lower()
    df_base["ano"] = pd.to_numeric(df_base["ano"], errors="coerce")
    df_base["mes"] = pd.to_numeric(df_base["mes"], errors="coerce")
    df_base["ordem_sinal_final"] = pd.to_numeric(df_base["ordem_sinal_final"], errors="coerce")
    df_base["score_sinal_bruto"] = pd.to_numeric(df_base["score_sinal_bruto"], errors="coerce")
    df_base["intensidade_sinal_bruto"] = pd.to_numeric(df_base["intensidade_sinal_bruto"], errors="coerce")

df_datas_finais_capitulacao["flag_capitulacao_bruta"] = df_datas_finais_capitulacao["flag_capitulacao_bruta"].fillna(False).astype(bool)
df_datas_finais_capitulacao["flag_capitulacao_final"] = df_datas_finais_capitulacao["flag_capitulacao_final"].fillna(False).astype(bool)
df_datas_finais_capitulacao["flag_capitulacao_rejeitado_cooldown"] = df_datas_finais_capitulacao["flag_capitulacao_rejeitado_cooldown"].fillna(False).astype(bool)

df_datas_finais_euforia["flag_euforia_bruta"] = df_datas_finais_euforia["flag_euforia_bruta"].fillna(False).astype(bool)
df_datas_finais_euforia["flag_euforia_final"] = df_datas_finais_euforia["flag_euforia_final"].fillna(False).astype(bool)
df_datas_finais_euforia["flag_euforia_rejeitado_cooldown"] = df_datas_finais_euforia["flag_euforia_rejeitado_cooldown"].fillna(False).astype(bool)

n_duplicidades_calendario = int(df_calendario_final_sinais.duplicated(subset=["data"]).sum())
n_duplicidades_capitulacao = int(df_datas_finais_capitulacao.duplicated(subset=["data"]).sum())
n_duplicidades_euforia = int(df_datas_finais_euforia.duplicated(subset=["data"]).sum())

if n_duplicidades_calendario > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_calendario} duplicidades por data no calendário final de sinais.")
if n_duplicidades_capitulacao > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_capitulacao} duplicidades por data na base final de capitulação.")
if n_duplicidades_euforia > 0:
    raise ValueError(f"Foram encontradas {n_duplicidades_euforia} duplicidades por data na base final de euforia.")

print(f"Capital inicial oficial por estratégia    : {CAPITAL_INICIAL_ESTRATEGIA:.2f}")
print(f"Duplicidades no calendário final          : {n_duplicidades_calendario}")
print(f"Duplicidades nas datas finais capitulação : {n_duplicidades_capitulacao}")
print(f"Duplicidades nas datas finais euforia     : {n_duplicidades_euforia}")
print("OK")

# ============================================================
# 6) Contagem final dos sinais e construção das bases de saída
# ============================================================

print("\n[6/10] Contagem final dos sinais e construção das bases de saída...")

df_sinais_finais_dimensionamento = df_calendario_final_sinais.copy()
df_sinais_finais_dimensionamento["capital_inicial_estrategia"] = CAPITAL_INICIAL_ESTRATEGIA
df_sinais_finais_dimensionamento["n_sinais_finais_tipo"] = (
    df_sinais_finais_dimensionamento.groupby("tipo_sinal")["tipo_sinal"].transform("size")
)

n_sinais_capitulacao = int(len(df_datas_finais_capitulacao))
n_sinais_euforia = int(len(df_datas_finais_euforia))

n_sinais_capitulacao_calendario = int(df_calendario_final_sinais["tipo_sinal"].eq("capitulacao").sum())
n_sinais_euforia_calendario = int(df_calendario_final_sinais["tipo_sinal"].eq("euforia").sum())

if n_sinais_capitulacao != n_sinais_capitulacao_calendario:
    raise ValueError(
        "A contagem final de capitulação não bate entre a lista final da 6.4 e o calendário consolidado de sinais."
    )

if n_sinais_euforia != n_sinais_euforia_calendario:
    raise ValueError(
        "A contagem final de euforia não bate entre a lista final da 6.4 e o calendário consolidado de sinais."
    )

df_contagem_final_sinais_estrategia = pd.DataFrame(
    [
        construir_tabela_contagem(df_datas_finais_capitulacao, "capitulacao", CAPITAL_INICIAL_ESTRATEGIA),
        construir_tabela_contagem(df_datas_finais_euforia, "euforia", CAPITAL_INICIAL_ESTRATEGIA),
    ]
)

df_parametros_dimensionamento_aportes = pd.DataFrame(
    [
        {
            "parametro": "capital_inicial_estrategia",
            "valor_numerico": float(CAPITAL_INICIAL_ESTRATEGIA),
            "valor_texto": pd.NA,
            "unidade": "brl",
        },
        {
            "parametro": "n_sinais_capitulacao",
            "valor_numerico": float(n_sinais_capitulacao),
            "valor_texto": pd.NA,
            "unidade": "sinais",
        },
        {
            "parametro": "n_sinais_euforia",
            "valor_numerico": float(n_sinais_euforia),
            "valor_texto": pd.NA,
            "unidade": "sinais",
        },
        {
            "parametro": "metodo_dimensionamento_aporte",
            "valor_numerico": pd.NA,
            "valor_texto": "capital_inicial_dividido_pelo_numero_final_de_sinais",
            "unidade": "regra_metodologica",
        },
    ]
)

df_distribuicao_anual_sinais_estrategia = (
    df_sinais_finais_dimensionamento
    .groupby(["ano", "tipo_sinal"], as_index=False)
    .agg(
        n_sinais_finais=("data", "count"),
        intensidade_mediana_sinal=("intensidade_sinal_bruto", "median"),
        score_medio_sinal=("score_sinal_bruto", "mean"),
    )
    .sort_values(["ano", "tipo_sinal"])
    .reset_index(drop=True)
)

print(f"Quantidade final de sinais - capitulação : {n_sinais_capitulacao}")
print(f"Quantidade final de sinais - euforia     : {n_sinais_euforia}")
print("OK")

# ============================================================
# 7) Construção da auditoria da subetapa
# ============================================================

print("\n[7/10] Construção da auditoria da subetapa...")

df_auditoria_contagem_sinais = pd.DataFrame(
    [
        {"metrica": "capital_inicial_estrategia", "valor": float(CAPITAL_INICIAL_ESTRATEGIA)},
        {"metrica": "n_sinais_capitulacao_lista_final", "valor": n_sinais_capitulacao},
        {"metrica": "n_sinais_capitulacao_calendario", "valor": n_sinais_capitulacao_calendario},
        {"metrica": "n_sinais_euforia_lista_final", "valor": n_sinais_euforia},
        {"metrica": "n_sinais_euforia_calendario", "valor": n_sinais_euforia_calendario},
        {"metrica": "n_sinais_total_calendario_final", "valor": int(len(df_calendario_final_sinais))},
        {"metrica": "n_anos_com_capitulacao_final", "valor": int(df_datas_finais_capitulacao["ano"].nunique(dropna=True))},
        {"metrica": "n_anos_com_euforia_final", "valor": int(df_datas_finais_euforia["ano"].nunique(dropna=True))},
        {"metrica": "n_duplicidades_calendario_final", "valor": n_duplicidades_calendario},
        {"metrica": "n_duplicidades_capitulacao_final", "valor": n_duplicidades_capitulacao},
        {"metrica": "n_duplicidades_euforia_final", "valor": n_duplicidades_euforia},
    ]
)

print("Auditoria da subetapa construída com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_sinais_finais_dimensionamento, caminho_base_sinais_finais_dimensionamento, index=False)
salvar_dataframe(df_contagem_final_sinais_estrategia, caminho_tbl_contagem_final_sinais_estrategia, index=False)
salvar_dataframe(df_parametros_dimensionamento_aportes, caminho_tbl_parametros_dimensionamento_aportes, index=False)
salvar_dataframe(df_auditoria_contagem_sinais, caminho_tbl_auditoria_contagem_sinais, index=False)
salvar_dataframe(df_distribuicao_anual_sinais_estrategia, caminho_tbl_distribuicao_anual_sinais_estrategia, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_contagem_final_sinais_estrategia = df_contagem_final_sinais_estrategia.copy()
df_amostra_parametros_dimensionamento_aportes = df_parametros_dimensionamento_aportes.copy()
df_amostra_sinais_finais_dimensionamento = df_sinais_finais_dimensionamento.head(15).copy()
df_amostra_distribuicao_anual_sinais_estrategia = df_distribuicao_anual_sinais_estrategia.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nContagem final de sinais por estratégia:")
print(df_amostra_contagem_final_sinais_estrategia.to_string(index=False))

print("\nParâmetros de dimensionamento dos aportes:")
print(df_amostra_parametros_dimensionamento_aportes.to_string(index=False))

print("\nAuditoria da contagem final dos sinais:")
print(df_auditoria_contagem_sinais.to_string(index=False))

print("\nAmostra da base de sinais finais para dimensionamento:")
print(df_amostra_sinais_finais_dimensionamento.to_string(index=False))

print("\nDistribuição anual dos sinais por estratégia - amostra:")
print(df_amostra_distribuicao_anual_sinais_estrategia.to_string(index=False))

print("\nArquivos salvos na subetapa 7.1:")
print(f"- {caminho_base_sinais_finais_dimensionamento}")
print(f"- {caminho_tbl_contagem_final_sinais_estrategia}")
print(f"- {caminho_tbl_parametros_dimensionamento_aportes}")
print(f"- {caminho_tbl_auditoria_contagem_sinais}")
print(f"- {caminho_tbl_distribuicao_anual_sinais_estrategia}")

print("\nETAPA 7.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 7.1 - CONTAGEM FINAL DE SINAIS POR ESTRATÉGIA

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - parâmetros globais                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_1_tbl_parametros_globais.parquet
Entrada - calendário final de sinais          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_4_base_calendario_final_sinais.parquet
Entrada - datas finais de capitulação         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_4_tbl_datas_finais_capitulacao.parquet
Entrada - datas finais de euforia             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\

## Etapa 7.2) Definição do Valor de Cada Aporte

In [30]:
%%time
# ============================================================
# Etapa 7.2) Definição do Valor de Cada Aporte
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 7.2 - DEFINIÇÃO DO VALOR DE CADA APORTE")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_tbl_contagem_final_sinais_estrategia = gerar_caminho_arquivo(etapa=7, subetapa=1, tipo_arquivo="tbl", nome="contagem_final_sinais_estrategia")
caminho_tbl_parametros_dimensionamento_aportes = gerar_caminho_arquivo(etapa=7, subetapa=1, tipo_arquivo="tbl", nome="parametros_dimensionamento_aportes")
caminho_base_sinais_finais_dimensionamento = gerar_caminho_arquivo(etapa=7, subetapa=1, tipo_arquivo="base", nome="sinais_finais_dimensionamento")

caminho_base_valor_aporte_por_sinal_estrategia = gerar_caminho_arquivo(etapa=7, subetapa=2, tipo_arquivo="base", nome="valor_aporte_por_sinal_estrategia")
caminho_tbl_resumo_valor_aporte_estrategia = gerar_caminho_arquivo(etapa=7, subetapa=2, tipo_arquivo="tbl", nome="resumo_valor_aporte_estrategia")
caminho_tbl_parametros_valor_aporte_estrategia = gerar_caminho_arquivo(etapa=7, subetapa=2, tipo_arquivo="tbl", nome="parametros_valor_aporte_estrategia")
caminho_tbl_auditoria_valor_aporte_estrategia = gerar_caminho_arquivo(etapa=7, subetapa=2, tipo_arquivo="tbl", nome="auditoria_valor_aporte_estrategia")
caminho_tbl_distribuicao_anual_valor_aporte_estrategia = gerar_caminho_arquivo(etapa=7, subetapa=2, tipo_arquivo="tbl", nome="distribuicao_anual_valor_aporte_estrategia")

print(f"Entrada - contagem final de sinais         : {caminho_tbl_contagem_final_sinais_estrategia}")
print(f"Entrada - parâmetros de dimensionamento    : {caminho_tbl_parametros_dimensionamento_aportes}")
print(f"Entrada - base de sinais dimensionamento   : {caminho_base_sinais_finais_dimensionamento}")
print(f"Saída   - base valor do aporte por sinal   : {caminho_base_valor_aporte_por_sinal_estrategia}")
print(f"Saída   - resumo do valor do aporte        : {caminho_tbl_resumo_valor_aporte_estrategia}")
print(f"Saída   - parâmetros do valor do aporte    : {caminho_tbl_parametros_valor_aporte_estrategia}")
print(f"Saída   - auditoria do valor do aporte     : {caminho_tbl_auditoria_valor_aporte_estrategia}")
print(f"Saída   - distribuição anual dos aportes   : {caminho_tbl_distribuicao_anual_valor_aporte_estrategia}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_contagem_final_sinais_estrategia = pd.read_parquet(caminho_tbl_contagem_final_sinais_estrategia)
df_parametros_dimensionamento_aportes = pd.read_parquet(caminho_tbl_parametros_dimensionamento_aportes)
df_sinais_finais_dimensionamento = pd.read_parquet(caminho_base_sinais_finais_dimensionamento)

print(f"Contagem final de sinais       : {df_contagem_final_sinais_estrategia.shape[0]:,} linhas x {df_contagem_final_sinais_estrategia.shape[1]} colunas")
print(f"Parâmetros de dimensionamento  : {df_parametros_dimensionamento_aportes.shape[0]:,} linhas x {df_parametros_dimensionamento_aportes.shape[1]} colunas")
print(f"Base de sinais dimensionamento : {df_sinais_finais_dimensionamento.shape[0]:,} linhas x {df_sinais_finais_dimensionamento.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

CASAS_DECIMAIS_APORTE = 2

COLUNAS_OBRIGATORIAS_CONTAGEM = [
    "tipo_sinal",
    "n_sinais_finais",
    "capital_inicial_estrategia",
    "data_primeiro_sinal",
    "data_ultimo_sinal",
    "ordem_primeiro_sinal",
    "ordem_ultimo_sinal",
]

COLUNAS_OBRIGATORIAS_PARAMETROS = [
    "parametro",
    "valor_numerico",
    "valor_texto",
    "unidade",
]

COLUNAS_OBRIGATORIAS_SINAIS = [
    "data",
    "tipo_sinal",
    "direcao_sinal",
    "ordem_sinal_final",
    "score_sinal_bruto",
    "intensidade_sinal_bruto",
    "ano",
    "mes",
    "ordem_global_sinal",
    "capital_inicial_estrategia",
    "n_sinais_finais_tipo",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    """
    Valida a presença das colunas obrigatórias em um DataFrame.
    """
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def obter_parametro_numerico(df_parametros, parametro):
    """
    Recupera o valor numérico de um parâmetro salvo em tabela de parâmetros.
    """
    df_parametro = df_parametros.loc[df_parametros["parametro"].astype("string").str.strip().eq(parametro)].copy()

    if df_parametro.empty:
        raise ValueError(f"Parâmetro não encontrado: {parametro}")

    valor_numerico = pd.to_numeric(df_parametro["valor_numerico"].iloc[0], errors="coerce")

    if pd.isna(valor_numerico):
        raise ValueError(f"O parâmetro '{parametro}' não possui valor numérico válido.")

    return float(valor_numerico)

def calcular_valor_aporte_por_sinal(df_sinais_tipo, capital_total, n_sinais, casas_decimais):
    """
    Calcula o valor planejado de cada aporte, com ajuste residual no último sinal
    para garantir fechamento exato do capital total da estratégia.
    """
    df_sinais_tipo = df_sinais_tipo.sort_values("ordem_sinal_final").reset_index(drop=True).copy()

    if len(df_sinais_tipo) != int(n_sinais):
        raise ValueError(
            f"A quantidade de linhas da base de sinais ({len(df_sinais_tipo)}) não coincide com n_sinais ({n_sinais})."
        )

    valor_cada_aporte_teorico_exato = float(capital_total / n_sinais)
    valor_cada_aporte_base = round(valor_cada_aporte_teorico_exato, casas_decimais)

    lista_valores_aporte = [valor_cada_aporte_base] * int(n_sinais)

    if int(n_sinais) > 0:
        soma_preliminar = round(sum(lista_valores_aporte[:-1]), casas_decimais) if int(n_sinais) > 1 else 0.00
        valor_ultimo_aporte_ajustado = round(capital_total - soma_preliminar, casas_decimais)
        lista_valores_aporte[-1] = valor_ultimo_aporte_ajustado

    df_sinais_tipo["valor_cada_aporte_teorico_exato"] = valor_cada_aporte_teorico_exato
    df_sinais_tipo["valor_cada_aporte_planejado"] = lista_valores_aporte
    df_sinais_tipo["flag_ajuste_residual_ultimo_aporte"] = False

    if int(n_sinais) > 0:
        df_sinais_tipo.loc[df_sinais_tipo.index.max(), "flag_ajuste_residual_ultimo_aporte"] = True

    df_sinais_tipo["valor_aportado_planejado_acumulado"] = df_sinais_tipo["valor_cada_aporte_planejado"].cumsum()
    df_sinais_tipo["pct_capital_planejado_acumulado"] = (
        df_sinais_tipo["valor_aportado_planejado_acumulado"] / capital_total
    )
    df_sinais_tipo["valor_medio_aporte_planejado_tipo"] = (
        df_sinais_tipo["valor_cada_aporte_planejado"].sum() / n_sinais
    )

    return df_sinais_tipo

print(f"Casas decimais operacionais do aporte : {CASAS_DECIMAIS_APORTE}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

validar_colunas_obrigatorias(df_contagem_final_sinais_estrategia, COLUNAS_OBRIGATORIAS_CONTAGEM)
validar_colunas_obrigatorias(df_parametros_dimensionamento_aportes, COLUNAS_OBRIGATORIAS_PARAMETROS)
validar_colunas_obrigatorias(df_sinais_finais_dimensionamento, COLUNAS_OBRIGATORIAS_SINAIS)

CAPITAL_INICIAL_ESTRATEGIA_PARAMETRO = obter_parametro_numerico(
    df_parametros_dimensionamento_aportes,
    "capital_inicial_estrategia",
)

N_SINAIS_CAPITULACAO_PARAMETRO = int(
    obter_parametro_numerico(df_parametros_dimensionamento_aportes, "n_sinais_capitulacao")
)
N_SINAIS_EUFORIA_PARAMETRO = int(
    obter_parametro_numerico(df_parametros_dimensionamento_aportes, "n_sinais_euforia")
)

for df_base in [df_contagem_final_sinais_estrategia, df_sinais_finais_dimensionamento]:
    df_base["tipo_sinal"] = df_base["tipo_sinal"].astype("string").str.strip().str.lower()

df_sinais_finais_dimensionamento["direcao_sinal"] = df_sinais_finais_dimensionamento["direcao_sinal"].astype("string").str.strip().str.lower()
df_sinais_finais_dimensionamento["data"] = pd.to_datetime(df_sinais_finais_dimensionamento["data"], errors="coerce")
df_sinais_finais_dimensionamento["ano"] = pd.to_numeric(df_sinais_finais_dimensionamento["ano"], errors="coerce")
df_sinais_finais_dimensionamento["mes"] = pd.to_numeric(df_sinais_finais_dimensionamento["mes"], errors="coerce")
df_sinais_finais_dimensionamento["ordem_global_sinal"] = pd.to_numeric(df_sinais_finais_dimensionamento["ordem_global_sinal"], errors="coerce")
df_sinais_finais_dimensionamento["ordem_sinal_final"] = pd.to_numeric(df_sinais_finais_dimensionamento["ordem_sinal_final"], errors="coerce")
df_sinais_finais_dimensionamento["score_sinal_bruto"] = pd.to_numeric(df_sinais_finais_dimensionamento["score_sinal_bruto"], errors="coerce")
df_sinais_finais_dimensionamento["intensidade_sinal_bruto"] = pd.to_numeric(df_sinais_finais_dimensionamento["intensidade_sinal_bruto"], errors="coerce")
df_sinais_finais_dimensionamento["capital_inicial_estrategia"] = pd.to_numeric(df_sinais_finais_dimensionamento["capital_inicial_estrategia"], errors="coerce")
df_sinais_finais_dimensionamento["n_sinais_finais_tipo"] = pd.to_numeric(df_sinais_finais_dimensionamento["n_sinais_finais_tipo"], errors="coerce")

df_contagem_final_sinais_estrategia["n_sinais_finais"] = pd.to_numeric(
    df_contagem_final_sinais_estrategia["n_sinais_finais"], errors="coerce"
)
df_contagem_final_sinais_estrategia["capital_inicial_estrategia"] = pd.to_numeric(
    df_contagem_final_sinais_estrategia["capital_inicial_estrategia"], errors="coerce"
)

n_duplicidades_tipo_ordem = int(
    df_sinais_finais_dimensionamento.duplicated(subset=["tipo_sinal", "ordem_sinal_final"]).sum()
)

if n_duplicidades_tipo_ordem > 0:
    raise ValueError(
        f"Foram encontradas {n_duplicidades_tipo_ordem} duplicidades na chave tipo_sinal + ordem_sinal_final da base de sinais."
    )

print(f"Capital inicial por estratégia carregado : {CAPITAL_INICIAL_ESTRATEGIA_PARAMETRO:.2f}")
print(f"Sinais finais de capitulação             : {N_SINAIS_CAPITULACAO_PARAMETRO}")
print(f"Sinais finais de euforia                 : {N_SINAIS_EUFORIA_PARAMETRO}")
print(f"Duplicidades tipo+ordem na base de sinais: {n_duplicidades_tipo_ordem}")
print("OK")

# ============================================================
# 6) Definição do valor de cada aporte por estratégia
# ============================================================

print("\n[6/10] Definição do valor de cada aporte por estratégia...")

df_sinais_capitulacao = (
    df_sinais_finais_dimensionamento
    .loc[df_sinais_finais_dimensionamento["tipo_sinal"].eq("capitulacao")]
    .copy()
)

df_sinais_euforia = (
    df_sinais_finais_dimensionamento
    .loc[df_sinais_finais_dimensionamento["tipo_sinal"].eq("euforia")]
    .copy()
)

df_valor_aporte_capitulacao = calcular_valor_aporte_por_sinal(
    df_sinais_tipo=df_sinais_capitulacao,
    capital_total=CAPITAL_INICIAL_ESTRATEGIA_PARAMETRO,
    n_sinais=N_SINAIS_CAPITULACAO_PARAMETRO,
    casas_decimais=CASAS_DECIMAIS_APORTE,
)

df_valor_aporte_euforia = calcular_valor_aporte_por_sinal(
    df_sinais_tipo=df_sinais_euforia,
    capital_total=CAPITAL_INICIAL_ESTRATEGIA_PARAMETRO,
    n_sinais=N_SINAIS_EUFORIA_PARAMETRO,
    casas_decimais=CASAS_DECIMAIS_APORTE,
)

df_valor_aporte_por_sinal_estrategia = (
    pd.concat([df_valor_aporte_capitulacao, df_valor_aporte_euforia], axis=0, ignore_index=True)
    .sort_values(["data", "tipo_sinal", "ordem_sinal_final"])
    .reset_index(drop=True)
)

colunas_saida = [
    "data",
    "tipo_sinal",
    "direcao_sinal",
    "ordem_sinal_final",
    "ordem_global_sinal",
    "ano",
    "mes",
    "score_sinal_bruto",
    "intensidade_sinal_bruto",
    "capital_inicial_estrategia",
    "n_sinais_finais_tipo",
    "valor_cada_aporte_teorico_exato",
    "valor_cada_aporte_planejado",
    "flag_ajuste_residual_ultimo_aporte",
    "valor_medio_aporte_planejado_tipo",
    "valor_aportado_planejado_acumulado",
    "pct_capital_planejado_acumulado",
]

df_valor_aporte_por_sinal_estrategia = df_valor_aporte_por_sinal_estrategia[colunas_saida].copy()

valor_medio_aporte_capitulacao = float(
    df_valor_aporte_capitulacao["valor_medio_aporte_planejado_tipo"].iloc[0]
) if not df_valor_aporte_capitulacao.empty else pd.NA
valor_medio_aporte_euforia = float(
    df_valor_aporte_euforia["valor_medio_aporte_planejado_tipo"].iloc[0]
) if not df_valor_aporte_euforia.empty else pd.NA

print(f"Valor médio planejado por aporte - capitulação : {valor_medio_aporte_capitulacao:.2f}")
print(f"Valor médio planejado por aporte - euforia     : {valor_medio_aporte_euforia:.2f}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

df_resumo_valor_aporte_estrategia = (
    df_valor_aporte_por_sinal_estrategia
    .groupby("tipo_sinal", as_index=False)
    .agg(
        n_sinais_finais=("data", "count"),
        capital_inicial_estrategia=("capital_inicial_estrategia", "first"),
        valor_cada_aporte_teorico_exato=("valor_cada_aporte_teorico_exato", "first"),
        valor_medio_aporte_planejado=("valor_cada_aporte_planejado", "mean"),
        valor_menor_aporte_planejado=("valor_cada_aporte_planejado", "min"),
        valor_maior_aporte_planejado=("valor_cada_aporte_planejado", "max"),
        valor_total_aportado_planejado=("valor_cada_aporte_planejado", "sum"),
        data_primeiro_aporte=("data", "min"),
        data_ultimo_aporte=("data", "max"),
        n_aportes_com_ajuste_residual=("flag_ajuste_residual_ultimo_aporte", "sum"),
    )
    .sort_values("tipo_sinal")
    .reset_index(drop=True)
)

df_resumo_valor_aporte_estrategia["diferenca_total_vs_capital"] = (
    df_resumo_valor_aporte_estrategia["valor_total_aportado_planejado"]
    - df_resumo_valor_aporte_estrategia["capital_inicial_estrategia"]
)

df_parametros_valor_aporte_estrategia = pd.DataFrame(
    [
        {
            "parametro": "casas_decimais_aporte",
            "valor_numerico": float(CASAS_DECIMAIS_APORTE),
            "valor_texto": pd.NA,
            "unidade": "casas_decimais",
        },
        {
            "parametro": "metodo_definicao_valor_aporte",
            "valor_numerico": pd.NA,
            "valor_texto": "valor_igual_por_sinal_com_ajuste_residual_no_ultimo_aporte",
            "unidade": "regra_metodologica",
        },
        {
            "parametro": "capital_inicial_por_estrategia",
            "valor_numerico": float(CAPITAL_INICIAL_ESTRATEGIA_PARAMETRO),
            "valor_texto": pd.NA,
            "unidade": "brl",
        },
        {
            "parametro": "n_sinais_capitulacao",
            "valor_numerico": float(N_SINAIS_CAPITULACAO_PARAMETRO),
            "valor_texto": pd.NA,
            "unidade": "sinais",
        },
        {
            "parametro": "n_sinais_euforia",
            "valor_numerico": float(N_SINAIS_EUFORIA_PARAMETRO),
            "valor_texto": pd.NA,
            "unidade": "sinais",
        },
    ]
)

df_distribuicao_anual_valor_aporte_estrategia = (
    df_valor_aporte_por_sinal_estrategia
    .groupby(["ano", "tipo_sinal"], as_index=False)
    .agg(
        n_aportes=("data", "count"),
        valor_total_aportado_planejado=("valor_cada_aporte_planejado", "sum"),
        valor_medio_aporte_planejado=("valor_cada_aporte_planejado", "mean"),
        intensidade_mediana_sinal=("intensidade_sinal_bruto", "median"),
        score_medio_sinal=("score_sinal_bruto", "mean"),
    )
    .sort_values(["ano", "tipo_sinal"])
    .reset_index(drop=True)
)

diferenca_capitulacao = float(
    df_resumo_valor_aporte_estrategia.loc[
        df_resumo_valor_aporte_estrategia["tipo_sinal"].eq("capitulacao"),
        "diferenca_total_vs_capital",
    ].iloc[0]
)

diferenca_euforia = float(
    df_resumo_valor_aporte_estrategia.loc[
        df_resumo_valor_aporte_estrategia["tipo_sinal"].eq("euforia"),
        "diferenca_total_vs_capital",
    ].iloc[0]
)

n_aportes_com_ajuste_residual = int(df_valor_aporte_por_sinal_estrategia["flag_ajuste_residual_ultimo_aporte"].sum())

df_auditoria_valor_aporte_estrategia = pd.DataFrame(
    [
        {"metrica": "capital_inicial_por_estrategia", "valor": float(CAPITAL_INICIAL_ESTRATEGIA_PARAMETRO)},
        {"metrica": "n_sinais_capitulacao_parametro", "valor": float(N_SINAIS_CAPITULACAO_PARAMETRO)},
        {"metrica": "n_sinais_euforia_parametro", "valor": float(N_SINAIS_EUFORIA_PARAMETRO)},
        {"metrica": "n_linhas_base_valor_aporte", "valor": float(len(df_valor_aporte_por_sinal_estrategia))},
        {"metrica": "n_aportes_com_ajuste_residual", "valor": float(n_aportes_com_ajuste_residual)},
        {"metrica": "diferenca_total_vs_capital_capitulacao", "valor": diferenca_capitulacao},
        {"metrica": "diferenca_total_vs_capital_euforia", "valor": diferenca_euforia},
        {"metrica": "valor_medio_aporte_capitulacao", "valor": float(valor_medio_aporte_capitulacao)},
        {"metrica": "valor_medio_aporte_euforia", "valor": float(valor_medio_aporte_euforia)},
    ]
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_valor_aporte_por_sinal_estrategia, caminho_base_valor_aporte_por_sinal_estrategia, index=False)
salvar_dataframe(df_resumo_valor_aporte_estrategia, caminho_tbl_resumo_valor_aporte_estrategia, index=False)
salvar_dataframe(df_parametros_valor_aporte_estrategia, caminho_tbl_parametros_valor_aporte_estrategia, index=False)
salvar_dataframe(df_auditoria_valor_aporte_estrategia, caminho_tbl_auditoria_valor_aporte_estrategia, index=False)
salvar_dataframe(df_distribuicao_anual_valor_aporte_estrategia, caminho_tbl_distribuicao_anual_valor_aporte_estrategia, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_resumo_valor_aporte_estrategia = df_resumo_valor_aporte_estrategia.copy()
df_amostra_parametros_valor_aporte_estrategia = df_parametros_valor_aporte_estrategia.copy()
df_amostra_valor_aporte_por_sinal_estrategia = df_valor_aporte_por_sinal_estrategia.head(15).copy()
df_amostra_distribuicao_anual_valor_aporte_estrategia = df_distribuicao_anual_valor_aporte_estrategia.head(15).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo da definição do valor de cada aporte por estratégia:")
print(df_amostra_resumo_valor_aporte_estrategia.to_string(index=False))

print("\nParâmetros formais da definição do valor de cada aporte:")
print(df_amostra_parametros_valor_aporte_estrategia.to_string(index=False))

print("\nAuditoria da definição do valor de cada aporte:")
print(df_auditoria_valor_aporte_estrategia.to_string(index=False))

print("\nAmostra da base de valor de cada aporte por sinal:")
print(df_amostra_valor_aporte_por_sinal_estrategia.to_string(index=False))

print("\nDistribuição anual do valor dos aportes por estratégia - amostra:")
print(df_amostra_distribuicao_anual_valor_aporte_estrategia.to_string(index=False))

print("\nArquivos salvos na subetapa 7.2:")
print(f"- {caminho_base_valor_aporte_por_sinal_estrategia}")
print(f"- {caminho_tbl_resumo_valor_aporte_estrategia}")
print(f"- {caminho_tbl_parametros_valor_aporte_estrategia}")
print(f"- {caminho_tbl_auditoria_valor_aporte_estrategia}")
print(f"- {caminho_tbl_distribuicao_anual_valor_aporte_estrategia}")

print("\nETAPA 7.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 7.2 - DEFINIÇÃO DO VALOR DE CADA APORTE

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - contagem final de sinais         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_1_tbl_contagem_final_sinais_estrategia.parquet
Entrada - parâmetros de dimensionamento    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_1_tbl_parametros_dimensionamento_aportes.parquet
Entrada - base de sinais dimensionamento   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_1_base_sinais_finais_dimensionamento.parquet
Saída   - base valor do aporte por sinal   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resulta

## Etapa 7.3) Calendário de Aportes da Estratégia de Capitulação

In [31]:
%%time
# ============================================================
# Etapa 7.3) Calendário de Aportes da Estratégia de Capitulação
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 7.3 - CALENDÁRIO DE APORTES DA ESTRATÉGIA DE CAPITULAÇÃO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_valor_aporte_por_sinal_estrategia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=2,
    tipo_arquivo="base",
    nome="valor_aporte_por_sinal_estrategia",
)

caminho_base_calendario_aportes_capitulacao = gerar_caminho_arquivo(
    etapa=7,
    subetapa=3,
    tipo_arquivo="base",
    nome="calendario_aportes_capitulacao",
)

caminho_tbl_resumo_calendario_aportes_capitulacao = gerar_caminho_arquivo(
    etapa=7,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="resumo_calendario_aportes_capitulacao",
)

caminho_tbl_parametros_calendario_aportes_capitulacao = gerar_caminho_arquivo(
    etapa=7,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="parametros_calendario_aportes_capitulacao",
)

caminho_tbl_auditoria_calendario_aportes_capitulacao = gerar_caminho_arquivo(
    etapa=7,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="auditoria_calendario_aportes_capitulacao",
)

caminho_tbl_inconsistencias_calendario_aportes_capitulacao = gerar_caminho_arquivo(
    etapa=7,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="inconsistencias_calendario_aportes_capitulacao",
)

caminho_tbl_distribuicao_anual_aportes_capitulacao = gerar_caminho_arquivo(
    etapa=7,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_aportes_capitulacao",
)

print(f"Entrada - mercado diário consolidada       : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - valor do aporte por sinal        : {caminho_base_valor_aporte_por_sinal_estrategia}")
print(f"Saída   - base calendário aportes          : {caminho_base_calendario_aportes_capitulacao}")
print(f"Saída   - resumo                           : {caminho_tbl_resumo_calendario_aportes_capitulacao}")
print(f"Saída   - parâmetros                       : {caminho_tbl_parametros_calendario_aportes_capitulacao}")
print(f"Saída   - auditoria                        : {caminho_tbl_auditoria_calendario_aportes_capitulacao}")
print(f"Saída   - inconsistências                  : {caminho_tbl_inconsistencias_calendario_aportes_capitulacao}")
print(f"Saída   - distribuição anual               : {caminho_tbl_distribuicao_anual_aportes_capitulacao}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_valor_aporte_por_sinal_estrategia = pd.read_parquet(caminho_base_valor_aporte_por_sinal_estrategia)

print(f"Mercado diário consolidada       : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Valor do aporte por sinal        : {df_valor_aporte_por_sinal_estrategia.shape[0]:,} linhas x {df_valor_aporte_por_sinal_estrategia.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TIPO_SINAL_ALVO = "capitulacao"
TIPO_ESTRATEGIA = "capitulacao"
ESTRATEGIA_ID = "capitulacao"
GRUPO_CONTROLE = "sinal"
FAMILIA_ESTRATEGIA = "capitulacao_sinal"
ESTRATEGIA_REFERENCIA = "capitulacao"
ORIGEM_CALENDARIO = "7_3_capitulacao"
REPLICA_ID_PADRAO = 0
DEFASAGEM_APORTE_PREGOES = 1

COLUNAS_OBRIGATORIAS_MERCADO = [
    "data",
]

COLUNAS_OBRIGATORIAS_VALOR_APORTE = [
    "tipo_sinal",
    "ordem_sinal_final",
    "ordem_global_sinal",
    "data_sinal",
    "ano",
    "mes",
    "score_sinal_bruto",
    "intensidade_sinal_bruto",
    "capital_inicial_estrategia",
    "n_sinais_finais_tipo",
    "valor_cada_aporte_teorico_exato",
    "valor_aporte_estrategia",
    "flag_ajuste_residual_ultimo_aporte",
    "valor_medio_aporte_planejado_tipo",
    "valor_aportado_planejado_acumulado",
    "pct_capital_planejado_acumulado",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def padronizar_texto(df, colunas_texto, upper_cols=None, lower_cols=None):
    upper_cols = [] if upper_cols is None else upper_cols
    lower_cols = [] if lower_cols is None else lower_cols

    for coluna in colunas_texto:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()

    for coluna in upper_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.upper()

    for coluna in lower_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.lower()

    return df

print(f"Defasagem operacional do aporte : {DEFASAGEM_APORTE_PREGOES} pregão após o sinal")
print("OK")

# ============================================================
# 5) Preparação das bases e calendário de referência
# ============================================================

print("\n[5/10] Preparação das bases e calendário de referência...")

validar_colunas_obrigatorias(df_mercado_diario_consolidada, COLUNAS_OBRIGATORIAS_MERCADO)

# A etapa 7.2 salva a data do sinal como "data" e o valor operacional do aporte
# como "valor_cada_aporte_planejado". Para manter compatibilidade com o layout já
# validado anteriormente e com o contrato canônico downstream, a 7.3 normaliza
# esses nomes antes das validações formais.
if "data_sinal" not in df_valor_aporte_por_sinal_estrategia.columns and "data" in df_valor_aporte_por_sinal_estrategia.columns:
    df_valor_aporte_por_sinal_estrategia = df_valor_aporte_por_sinal_estrategia.rename(columns={"data": "data_sinal"})

if "valor_aporte_estrategia" not in df_valor_aporte_por_sinal_estrategia.columns and "valor_cada_aporte_planejado" in df_valor_aporte_por_sinal_estrategia.columns:
    df_valor_aporte_por_sinal_estrategia = df_valor_aporte_por_sinal_estrategia.rename(columns={"valor_cada_aporte_planejado": "valor_aporte_estrategia"})

validar_colunas_obrigatorias(df_valor_aporte_por_sinal_estrategia, COLUNAS_OBRIGATORIAS_VALOR_APORTE)

df_mercado_diario_consolidada["data"] = pd.to_datetime(df_mercado_diario_consolidada["data"], errors="coerce")
df_valor_aporte_por_sinal_estrategia["data_sinal"] = pd.to_datetime(
    df_valor_aporte_por_sinal_estrategia["data_sinal"],
    errors="coerce",
)

df_valor_aporte_por_sinal_estrategia = padronizar_texto(
    df=df_valor_aporte_por_sinal_estrategia.copy(),
    colunas_texto=["tipo_sinal"],
    lower_cols=["tipo_sinal"],
)

for coluna in [
    "ordem_sinal_final",
    "ordem_global_sinal",
    "ano",
    "mes",
    "score_sinal_bruto",
    "intensidade_sinal_bruto",
    "capital_inicial_estrategia",
    "n_sinais_finais_tipo",
    "valor_cada_aporte_teorico_exato",
    "valor_aporte_estrategia",
    "valor_medio_aporte_planejado_tipo",
    "valor_aportado_planejado_acumulado",
    "pct_capital_planejado_acumulado",
]:
    if coluna in df_valor_aporte_por_sinal_estrategia.columns:
        df_valor_aporte_por_sinal_estrategia[coluna] = pd.to_numeric(
            df_valor_aporte_por_sinal_estrategia[coluna],
            errors="coerce",
        )

df_sinais_capitulacao = (
    df_valor_aporte_por_sinal_estrategia
    .loc[df_valor_aporte_por_sinal_estrategia["tipo_sinal"] == TIPO_SINAL_ALVO]
    .copy()
    .sort_values(["ordem_sinal_final", "data_sinal"])
    .reset_index(drop=True)
)

if df_sinais_capitulacao.empty:
    raise ValueError("A base de valor do aporte por sinal não possui registros válidos para a estratégia de capitulação.")

if df_sinais_capitulacao["ordem_sinal_final"].duplicated().any():
    raise ValueError("Foram encontradas duplicidades em 'ordem_sinal_final' na estratégia de capitulação.")

df_pregoes_referencia = (
    df_mercado_diario_consolidada[["data"]]
    .dropna(subset=["data"])
    .drop_duplicates()
    .sort_values("data")
    .reset_index(drop=True)
)

if df_pregoes_referencia.empty:
    raise ValueError("Não foi possível construir o calendário de pregões de referência a partir da base de mercado.")

df_pregoes_referencia["ordem_pregao"] = np.arange(1, len(df_pregoes_referencia) + 1, dtype=int)

df_sinais_capitulacao = df_sinais_capitulacao.merge(
    df_pregoes_referencia.rename(columns={"data": "data_sinal", "ordem_pregao": "ordem_pregao_sinal"}),
    on="data_sinal",
    how="left",
    validate="one_to_one",
)

df_sinais_capitulacao["flag_data_sinal_em_pregao_referencia"] = df_sinais_capitulacao["ordem_pregao_sinal"].notna()
df_sinais_capitulacao["ordem_pregao_aporte_efetiva"] = df_sinais_capitulacao["ordem_pregao_sinal"] + DEFASAGEM_APORTE_PREGOES

df_sinais_capitulacao = df_sinais_capitulacao.merge(
    df_pregoes_referencia.rename(columns={"data": "data_aporte_efetiva", "ordem_pregao": "ordem_pregao_aporte_efetiva"}),
    on="ordem_pregao_aporte_efetiva",
    how="left",
    validate="one_to_one",
)

df_sinais_capitulacao["flag_data_aporte_efetiva_encontrada"] = df_sinais_capitulacao["data_aporte_efetiva"].notna()
df_sinais_capitulacao["gap_pregoes_sinal_para_aporte"] = (
    df_sinais_capitulacao["ordem_pregao_aporte_efetiva"] - df_sinais_capitulacao["ordem_pregao_sinal"]
)

print(f"Quantidade de pregões de referência     : {len(df_pregoes_referencia):,}")
print(f"Quantidade de aportes de capitulação    : {len(df_sinais_capitulacao):,}")
print(f"Duplicidades em ordem de sinal          : {int(df_sinais_capitulacao['ordem_sinal_final'].duplicated().sum())}")
print("Normalização de schema aplicada: data -> data_sinal e valor_cada_aporte_planejado -> valor_aporte_estrategia, quando necessário.")
print("OK")

# ============================================================
# 6) Construção do calendário final de aportes da capitulação
# ============================================================

print("\n[6/10] Construção do calendário final de aportes da capitulação...")

df_calendario_aportes_capitulacao = df_sinais_capitulacao.copy()

df_calendario_aportes_capitulacao["request_id"] = (
    df_calendario_aportes_capitulacao["ordem_sinal_final"]
    .astype(int)
    .astype(str)
    .map(lambda x: f"{ESTRATEGIA_ID}__{x}")
)

df_calendario_aportes_capitulacao["estrategia_id"] = ESTRATEGIA_ID
df_calendario_aportes_capitulacao["grupo_controle"] = GRUPO_CONTROLE
df_calendario_aportes_capitulacao["familia_estrategia"] = FAMILIA_ESTRATEGIA
df_calendario_aportes_capitulacao["estrategia_referencia"] = ESTRATEGIA_REFERENCIA
df_calendario_aportes_capitulacao["tipo_estrategia"] = TIPO_ESTRATEGIA
df_calendario_aportes_capitulacao["replica_id"] = REPLICA_ID_PADRAO
df_calendario_aportes_capitulacao["seed_replicacao"] = pd.NA
df_calendario_aportes_capitulacao["origem_calendario"] = ORIGEM_CALENDARIO
df_calendario_aportes_capitulacao["ordem_aporte_estrategia"] = pd.to_numeric(
    df_calendario_aportes_capitulacao["ordem_sinal_final"],
    errors="coerce",
)
df_calendario_aportes_capitulacao["flag_evento_aporte"] = True

df_calendario_aportes_capitulacao = df_calendario_aportes_capitulacao[
    [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "replica_id",
        "seed_replicacao",
        "origem_calendario",
        "tipo_sinal",
        "ordem_sinal_final",
        "ordem_aporte_estrategia",
        "ordem_global_sinal",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_pregao_sinal",
        "ordem_pregao_aporte_efetiva",
        "gap_pregoes_sinal_para_aporte",
        "flag_data_sinal_em_pregao_referencia",
        "flag_data_aporte_efetiva_encontrada",
        "ano",
        "mes",
        "score_sinal_bruto",
        "intensidade_sinal_bruto",
        "capital_inicial_estrategia",
        "n_sinais_finais_tipo",
        "valor_cada_aporte_teorico_exato",
        "valor_aporte_estrategia",
        "flag_ajuste_residual_ultimo_aporte",
        "valor_medio_aporte_planejado_tipo",
        "valor_aportado_planejado_acumulado",
        "pct_capital_planejado_acumulado",
        "flag_evento_aporte",
    ]
].copy()

df_calendario_aportes_capitulacao = (
    df_calendario_aportes_capitulacao
    .sort_values(["ordem_aporte_estrategia", "data_aporte_efetiva"])
    .reset_index(drop=True)
)

df_inconsistencias_calendario_aportes_capitulacao = df_calendario_aportes_capitulacao.loc[
    (
        ~df_calendario_aportes_capitulacao["flag_data_sinal_em_pregao_referencia"]
    )
    | (
        ~df_calendario_aportes_capitulacao["flag_data_aporte_efetiva_encontrada"]
    )
    | (
        df_calendario_aportes_capitulacao["gap_pregoes_sinal_para_aporte"] != DEFASAGEM_APORTE_PREGOES
    )
].copy()

print(f"Base final de aportes da capitulação : {df_calendario_aportes_capitulacao.shape[0]:,} linhas x {df_calendario_aportes_capitulacao.shape[1]} colunas")
print(f"Inconsistências identificadas        : {len(df_inconsistencias_calendario_aportes_capitulacao):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

df_resumo_calendario_aportes_capitulacao = pd.DataFrame(
    [
        {
            "base": "calendario_aportes_capitulacao",
            "n_aportes": int(len(df_calendario_aportes_capitulacao)),
            "data_primeiro_sinal": df_calendario_aportes_capitulacao["data_sinal"].min(),
            "data_ultimo_sinal": df_calendario_aportes_capitulacao["data_sinal"].max(),
            "data_primeiro_aporte": df_calendario_aportes_capitulacao["data_aporte_efetiva"].min(),
            "data_ultimo_aporte": df_calendario_aportes_capitulacao["data_aporte_efetiva"].max(),
            "valor_total_aportado": float(df_calendario_aportes_capitulacao["valor_aporte_estrategia"].sum()),
            "capital_inicial_estrategia": float(df_calendario_aportes_capitulacao["capital_inicial_estrategia"].iloc[0]),
            "diferenca_total_vs_capital": float(
                df_calendario_aportes_capitulacao["valor_aporte_estrategia"].sum()
                - df_calendario_aportes_capitulacao["capital_inicial_estrategia"].iloc[0]
            ),
            "gap_pregoes_mediano": float(df_calendario_aportes_capitulacao["gap_pregoes_sinal_para_aporte"].median()),
            "n_aportes_com_ajuste_residual": int(
                df_calendario_aportes_capitulacao["flag_ajuste_residual_ultimo_aporte"].fillna(False).sum()
            ),
            "n_inconsistencias": int(len(df_inconsistencias_calendario_aportes_capitulacao)),
        }
    ]
)

df_parametros_calendario_aportes_capitulacao = pd.DataFrame(
    [
        {
            "parametro": "tipo_estrategia",
            "valor_numerico": pd.NA,
            "valor_texto": TIPO_ESTRATEGIA,
            "unidade": "estrategia",
        },
        {
            "parametro": "grupo_controle",
            "valor_numerico": pd.NA,
            "valor_texto": GRUPO_CONTROLE,
            "unidade": "grupo",
        },
        {
            "parametro": "familia_estrategia",
            "valor_numerico": pd.NA,
            "valor_texto": FAMILIA_ESTRATEGIA,
            "unidade": "familia",
        },
        {
            "parametro": "replica_id_padrao",
            "valor_numerico": float(REPLICA_ID_PADRAO),
            "valor_texto": pd.NA,
            "unidade": "replica",
        },
        {
            "parametro": "regra_data_aporte_efetiva",
            "valor_numerico": pd.NA,
            "valor_texto": "proximo_pregao_apos_data_sinal",
            "unidade": "regra_operacional",
        },
        {
            "parametro": "defasagem_aporte_pregoes",
            "valor_numerico": float(DEFASAGEM_APORTE_PREGOES),
            "valor_texto": pd.NA,
            "unidade": "pregoes",
        },
        {
            "parametro": "n_aportes_planejados",
            "valor_numerico": float(len(df_calendario_aportes_capitulacao)),
            "valor_texto": pd.NA,
            "unidade": "aportes",
        },
        {
            "parametro": "capital_inicial_estrategia",
            "valor_numerico": float(df_calendario_aportes_capitulacao["capital_inicial_estrategia"].iloc[0]),
            "valor_texto": pd.NA,
            "unidade": "brl",
        },
    ]
)

df_auditoria_calendario_aportes_capitulacao = pd.DataFrame(
    [
        {"metrica": "n_aportes_planejados", "valor": float(len(df_calendario_aportes_capitulacao))},
        {
            "metrica": "valor_total_aportado",
            "valor": float(df_calendario_aportes_capitulacao["valor_aporte_estrategia"].sum()),
        },
        {
            "metrica": "capital_inicial_estrategia",
            "valor": float(df_calendario_aportes_capitulacao["capital_inicial_estrategia"].iloc[0]),
        },
        {
            "metrica": "diferenca_total_vs_capital",
            "valor": float(
                df_calendario_aportes_capitulacao["valor_aporte_estrategia"].sum()
                - df_calendario_aportes_capitulacao["capital_inicial_estrategia"].iloc[0]
            ),
        },
        {
            "metrica": "n_datas_sinal_fora_calendario_referencia",
            "valor": float((~df_calendario_aportes_capitulacao["flag_data_sinal_em_pregao_referencia"]).sum()),
        },
        {
            "metrica": "n_datas_aporte_nao_encontradas",
            "valor": float((~df_calendario_aportes_capitulacao["flag_data_aporte_efetiva_encontrada"]).sum()),
        },
        {
            "metrica": "gap_pregoes_minimo",
            "valor": float(df_calendario_aportes_capitulacao["gap_pregoes_sinal_para_aporte"].min()),
        },
        {
            "metrica": "gap_pregoes_maximo",
            "valor": float(df_calendario_aportes_capitulacao["gap_pregoes_sinal_para_aporte"].max()),
        },
        {
            "metrica": "n_aportes_com_ajuste_residual",
            "valor": float(df_calendario_aportes_capitulacao["flag_ajuste_residual_ultimo_aporte"].fillna(False).sum()),
        },
        {
            "metrica": "n_inconsistencias_calendario",
            "valor": float(len(df_inconsistencias_calendario_aportes_capitulacao)),
        },
    ]
)

df_distribuicao_anual_aportes_capitulacao = (
    df_calendario_aportes_capitulacao
    .groupby(["ano"], as_index=False)
    .agg(
        n_aportes=("request_id", "count"),
        valor_total_aportado=("valor_aporte_estrategia", "sum"),
        valor_medio_aporte=("valor_aporte_estrategia", "mean"),
        intensidade_mediana_sinal=("intensidade_sinal_bruto", "median"),
        score_medio_sinal=("score_sinal_bruto", "mean"),
    )
    .sort_values("ano")
    .reset_index(drop=True)
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_calendario_aportes_capitulacao, caminho_base_calendario_aportes_capitulacao, index=False)
salvar_dataframe(df_resumo_calendario_aportes_capitulacao, caminho_tbl_resumo_calendario_aportes_capitulacao, index=False)
salvar_dataframe(df_parametros_calendario_aportes_capitulacao, caminho_tbl_parametros_calendario_aportes_capitulacao, index=False)
salvar_dataframe(df_auditoria_calendario_aportes_capitulacao, caminho_tbl_auditoria_calendario_aportes_capitulacao, index=False)
salvar_dataframe(df_inconsistencias_calendario_aportes_capitulacao, caminho_tbl_inconsistencias_calendario_aportes_capitulacao, index=False)
salvar_dataframe(df_distribuicao_anual_aportes_capitulacao, caminho_tbl_distribuicao_anual_aportes_capitulacao, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_resumo_calendario_aportes_capitulacao = df_resumo_calendario_aportes_capitulacao.copy()
df_amostra_parametros_calendario_aportes_capitulacao = df_parametros_calendario_aportes_capitulacao.copy()
df_amostra_auditoria_calendario_aportes_capitulacao = df_auditoria_calendario_aportes_capitulacao.copy()
df_amostra_calendario_aportes_capitulacao = df_calendario_aportes_capitulacao.head(20).copy()
df_amostra_inconsistencias_calendario_aportes_capitulacao = df_inconsistencias_calendario_aportes_capitulacao.head(20).copy()
df_amostra_distribuicao_anual_aportes_capitulacao = df_distribuicao_anual_aportes_capitulacao.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo do calendário de aportes da estratégia de capitulação:")
print(df_amostra_resumo_calendario_aportes_capitulacao.to_string(index=False))

print("\nParâmetros do calendário de aportes da estratégia de capitulação:")
print(df_amostra_parametros_calendario_aportes_capitulacao.to_string(index=False))

print("\nAuditoria do calendário de aportes da estratégia de capitulação:")
print(df_amostra_auditoria_calendario_aportes_capitulacao.to_string(index=False))

print("\nBase final do calendário de aportes da estratégia de capitulação - amostra:")
print(df_amostra_calendario_aportes_capitulacao.to_string(index=False))

print("\nInconsistências do calendário de aportes da estratégia de capitulação - amostra:")
print(df_amostra_inconsistencias_calendario_aportes_capitulacao.to_string(index=False))

print("\nDistribuição anual dos aportes da estratégia de capitulação - amostra:")
print(df_amostra_distribuicao_anual_aportes_capitulacao.to_string(index=False))

print("\nArquivos salvos na subetapa 7.3:")
print(f"- {caminho_base_calendario_aportes_capitulacao}")
print(f"- {caminho_tbl_resumo_calendario_aportes_capitulacao}")
print(f"- {caminho_tbl_parametros_calendario_aportes_capitulacao}")
print(f"- {caminho_tbl_auditoria_calendario_aportes_capitulacao}")
print(f"- {caminho_tbl_inconsistencias_calendario_aportes_capitulacao}")
print(f"- {caminho_tbl_distribuicao_anual_aportes_capitulacao}")

print("\nETAPA 7.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 7.3 - CALENDÁRIO DE APORTES DA ESTRATÉGIA DE CAPITULAÇÃO

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidada       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - valor do aporte por sinal        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_2_base_valor_aporte_por_sinal_estrategia.parquet
Saída   - base calendário aportes          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_3_base_calendario_aportes_capitulacao.parquet
Saída   - resumo                           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasi

## Etapa 7.4) Calendário de Aportes da Estratégia de Euforia

In [32]:
%%time
# ============================================================
# Etapa 7.4) Calendário de Aportes da Estratégia de Euforia
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 7.4 - CALENDÁRIO DE APORTES DA ESTRATÉGIA DE EUFORIA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_valor_aporte_por_sinal_estrategia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=2,
    tipo_arquivo="base",
    nome="valor_aporte_por_sinal_estrategia",
)

caminho_base_calendario_aportes_euforia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=4,
    tipo_arquivo="base",
    nome="calendario_aportes_euforia",
)

caminho_tbl_resumo_calendario_aportes_euforia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="resumo_calendario_aportes_euforia",
)

caminho_tbl_parametros_calendario_aportes_euforia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="parametros_calendario_aportes_euforia",
)

caminho_tbl_auditoria_calendario_aportes_euforia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="auditoria_calendario_aportes_euforia",
)

caminho_tbl_inconsistencias_calendario_aportes_euforia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="inconsistencias_calendario_aportes_euforia",
)

caminho_tbl_distribuicao_anual_aportes_euforia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_aportes_euforia",
)

print(f"Entrada - mercado diário consolidada       : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - valor do aporte por sinal        : {caminho_base_valor_aporte_por_sinal_estrategia}")
print(f"Saída   - base calendário aportes          : {caminho_base_calendario_aportes_euforia}")
print(f"Saída   - resumo                           : {caminho_tbl_resumo_calendario_aportes_euforia}")
print(f"Saída   - parâmetros                       : {caminho_tbl_parametros_calendario_aportes_euforia}")
print(f"Saída   - auditoria                        : {caminho_tbl_auditoria_calendario_aportes_euforia}")
print(f"Saída   - inconsistências                  : {caminho_tbl_inconsistencias_calendario_aportes_euforia}")
print(f"Saída   - distribuição anual               : {caminho_tbl_distribuicao_anual_aportes_euforia}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_valor_aporte_por_sinal_estrategia = pd.read_parquet(caminho_base_valor_aporte_por_sinal_estrategia)

print(f"Mercado diário consolidada       : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Valor do aporte por sinal        : {df_valor_aporte_por_sinal_estrategia.shape[0]:,} linhas x {df_valor_aporte_por_sinal_estrategia.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TIPO_SINAL_ALVO = "euforia"
TIPO_ESTRATEGIA = "euforia"
ESTRATEGIA_ID = "euforia"
GRUPO_CONTROLE = "sinal"
FAMILIA_ESTRATEGIA = "euforia_sinal"
ESTRATEGIA_REFERENCIA = "euforia"
ORIGEM_CALENDARIO = "7_4_euforia"
REPLICA_ID_PADRAO = 0
DEFASAGEM_APORTE_PREGOES = 1

COLUNAS_OBRIGATORIAS_MERCADO = [
    "data",
]

COLUNAS_OBRIGATORIAS_VALOR_APORTE = [
    "tipo_sinal",
    "ordem_sinal_final",
    "ordem_global_sinal",
    "data_sinal",
    "ano",
    "mes",
    "score_sinal_bruto",
    "intensidade_sinal_bruto",
    "capital_inicial_estrategia",
    "n_sinais_finais_tipo",
    "valor_cada_aporte_teorico_exato",
    "valor_aporte_estrategia",
    "flag_ajuste_residual_ultimo_aporte",
    "valor_medio_aporte_planejado_tipo",
    "valor_aportado_planejado_acumulado",
    "pct_capital_planejado_acumulado",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def padronizar_texto(df, colunas_texto, upper_cols=None, lower_cols=None):
    upper_cols = [] if upper_cols is None else upper_cols
    lower_cols = [] if lower_cols is None else lower_cols

    for coluna in colunas_texto:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()

    for coluna in upper_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.upper()

    for coluna in lower_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.lower()

    return df

print(f"Defasagem operacional do aporte : {DEFASAGEM_APORTE_PREGOES} pregão após o sinal")
print("OK")

# ============================================================
# 5) Preparação das bases e calendário de referência
# ============================================================

print("\n[5/10] Preparação das bases e calendário de referência...")

validar_colunas_obrigatorias(df_mercado_diario_consolidada, COLUNAS_OBRIGATORIAS_MERCADO)

# A etapa 7.2 salva a data do sinal como "data" e o valor operacional do aporte
# como "valor_cada_aporte_planejado". Para manter compatibilidade com o layout já
# validado anteriormente e com o contrato canônico downstream, a 7.3 normaliza
# esses nomes antes das validações formais.
if "data_sinal" not in df_valor_aporte_por_sinal_estrategia.columns and "data" in df_valor_aporte_por_sinal_estrategia.columns:
    df_valor_aporte_por_sinal_estrategia = df_valor_aporte_por_sinal_estrategia.rename(columns={"data": "data_sinal"})

if "valor_aporte_estrategia" not in df_valor_aporte_por_sinal_estrategia.columns and "valor_cada_aporte_planejado" in df_valor_aporte_por_sinal_estrategia.columns:
    df_valor_aporte_por_sinal_estrategia = df_valor_aporte_por_sinal_estrategia.rename(columns={"valor_cada_aporte_planejado": "valor_aporte_estrategia"})

validar_colunas_obrigatorias(df_valor_aporte_por_sinal_estrategia, COLUNAS_OBRIGATORIAS_VALOR_APORTE)

df_mercado_diario_consolidada["data"] = pd.to_datetime(df_mercado_diario_consolidada["data"], errors="coerce")
df_valor_aporte_por_sinal_estrategia["data_sinal"] = pd.to_datetime(
    df_valor_aporte_por_sinal_estrategia["data_sinal"],
    errors="coerce",
)

df_valor_aporte_por_sinal_estrategia = padronizar_texto(
    df=df_valor_aporte_por_sinal_estrategia.copy(),
    colunas_texto=["tipo_sinal"],
    lower_cols=["tipo_sinal"],
)

for coluna in [
    "ordem_sinal_final",
    "ordem_global_sinal",
    "ano",
    "mes",
    "score_sinal_bruto",
    "intensidade_sinal_bruto",
    "capital_inicial_estrategia",
    "n_sinais_finais_tipo",
    "valor_cada_aporte_teorico_exato",
    "valor_aporte_estrategia",
    "valor_medio_aporte_planejado_tipo",
    "valor_aportado_planejado_acumulado",
    "pct_capital_planejado_acumulado",
]:
    if coluna in df_valor_aporte_por_sinal_estrategia.columns:
        df_valor_aporte_por_sinal_estrategia[coluna] = pd.to_numeric(
            df_valor_aporte_por_sinal_estrategia[coluna],
            errors="coerce",
        )

df_sinais_euforia = (
    df_valor_aporte_por_sinal_estrategia
    .loc[df_valor_aporte_por_sinal_estrategia["tipo_sinal"] == TIPO_SINAL_ALVO]
    .copy()
    .sort_values(["ordem_sinal_final", "data_sinal"])
    .reset_index(drop=True)
)

if df_sinais_euforia.empty:
    raise ValueError("A base de valor do aporte por sinal não possui registros válidos para a estratégia de capitulação.")

if df_sinais_euforia["ordem_sinal_final"].duplicated().any():
    raise ValueError("Foram encontradas duplicidades em 'ordem_sinal_final' na estratégia de capitulação.")

df_pregoes_referencia = (
    df_mercado_diario_consolidada[["data"]]
    .dropna(subset=["data"])
    .drop_duplicates()
    .sort_values("data")
    .reset_index(drop=True)
)

if df_pregoes_referencia.empty:
    raise ValueError("Não foi possível construir o calendário de pregões de referência a partir da base de mercado.")

df_pregoes_referencia["ordem_pregao"] = np.arange(1, len(df_pregoes_referencia) + 1, dtype=int)

df_sinais_euforia = df_sinais_euforia.merge(
    df_pregoes_referencia.rename(columns={"data": "data_sinal", "ordem_pregao": "ordem_pregao_sinal"}),
    on="data_sinal",
    how="left",
    validate="one_to_one",
)

df_sinais_euforia["flag_data_sinal_em_pregao_referencia"] = df_sinais_euforia["ordem_pregao_sinal"].notna()
df_sinais_euforia["ordem_pregao_aporte_efetiva"] = df_sinais_euforia["ordem_pregao_sinal"] + DEFASAGEM_APORTE_PREGOES

df_sinais_euforia = df_sinais_euforia.merge(
    df_pregoes_referencia.rename(columns={"data": "data_aporte_efetiva", "ordem_pregao": "ordem_pregao_aporte_efetiva"}),
    on="ordem_pregao_aporte_efetiva",
    how="left",
    validate="one_to_one",
)

df_sinais_euforia["flag_data_aporte_efetiva_encontrada"] = df_sinais_euforia["data_aporte_efetiva"].notna()
df_sinais_euforia["gap_pregoes_sinal_para_aporte"] = (
    df_sinais_euforia["ordem_pregao_aporte_efetiva"] - df_sinais_euforia["ordem_pregao_sinal"]
)

print(f"Quantidade de pregões de referência     : {len(df_pregoes_referencia):,}")
print(f"Quantidade de aportes de euforia    : {len(df_sinais_euforia):,}")
print(f"Duplicidades em ordem de sinal          : {int(df_sinais_euforia['ordem_sinal_final'].duplicated().sum())}")
print("Normalização de schema aplicada: data -> data_sinal e valor_cada_aporte_planejado -> valor_aporte_estrategia, quando necessário.")
print("OK")

# ============================================================
# 6) Construção do calendário final de aportes da euforia
# ============================================================

print("\n[6/10] Construção do calendário final de aportes da euforia...")

df_calendario_aportes_euforia = df_sinais_euforia.copy()

df_calendario_aportes_euforia["request_id"] = (
    df_calendario_aportes_euforia["ordem_sinal_final"]
    .astype(int)
    .astype(str)
    .map(lambda x: f"{ESTRATEGIA_ID}__{x}")
)

df_calendario_aportes_euforia["estrategia_id"] = ESTRATEGIA_ID
df_calendario_aportes_euforia["grupo_controle"] = GRUPO_CONTROLE
df_calendario_aportes_euforia["familia_estrategia"] = FAMILIA_ESTRATEGIA
df_calendario_aportes_euforia["estrategia_referencia"] = ESTRATEGIA_REFERENCIA
df_calendario_aportes_euforia["tipo_estrategia"] = TIPO_ESTRATEGIA
df_calendario_aportes_euforia["replica_id"] = REPLICA_ID_PADRAO
df_calendario_aportes_euforia["seed_replicacao"] = pd.NA
df_calendario_aportes_euforia["origem_calendario"] = ORIGEM_CALENDARIO
df_calendario_aportes_euforia["ordem_aporte_estrategia"] = pd.to_numeric(
    df_calendario_aportes_euforia["ordem_sinal_final"],
    errors="coerce",
)
df_calendario_aportes_euforia["flag_evento_aporte"] = True

df_calendario_aportes_euforia = df_calendario_aportes_euforia[
    [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "replica_id",
        "seed_replicacao",
        "origem_calendario",
        "tipo_sinal",
        "ordem_sinal_final",
        "ordem_aporte_estrategia",
        "ordem_global_sinal",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_pregao_sinal",
        "ordem_pregao_aporte_efetiva",
        "gap_pregoes_sinal_para_aporte",
        "flag_data_sinal_em_pregao_referencia",
        "flag_data_aporte_efetiva_encontrada",
        "ano",
        "mes",
        "score_sinal_bruto",
        "intensidade_sinal_bruto",
        "capital_inicial_estrategia",
        "n_sinais_finais_tipo",
        "valor_cada_aporte_teorico_exato",
        "valor_aporte_estrategia",
        "flag_ajuste_residual_ultimo_aporte",
        "valor_medio_aporte_planejado_tipo",
        "valor_aportado_planejado_acumulado",
        "pct_capital_planejado_acumulado",
        "flag_evento_aporte",
    ]
].copy()

df_calendario_aportes_euforia = (
    df_calendario_aportes_euforia
    .sort_values(["ordem_aporte_estrategia", "data_aporte_efetiva"])
    .reset_index(drop=True)
)

df_inconsistencias_calendario_aportes_euforia = df_calendario_aportes_euforia.loc[
    (
        ~df_calendario_aportes_euforia["flag_data_sinal_em_pregao_referencia"]
    )
    | (
        ~df_calendario_aportes_euforia["flag_data_aporte_efetiva_encontrada"]
    )
    | (
        df_calendario_aportes_euforia["gap_pregoes_sinal_para_aporte"] != DEFASAGEM_APORTE_PREGOES
    )
].copy()

print(f"Base final de aportes da euforia : {df_calendario_aportes_euforia.shape[0]:,} linhas x {df_calendario_aportes_euforia.shape[1]} colunas")
print(f"Inconsistências identificadas        : {len(df_inconsistencias_calendario_aportes_euforia):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

df_resumo_calendario_aportes_euforia = pd.DataFrame(
    [
        {
            "base": "calendario_aportes_euforia",
            "n_aportes": int(len(df_calendario_aportes_euforia)),
            "data_primeiro_sinal": df_calendario_aportes_euforia["data_sinal"].min(),
            "data_ultimo_sinal": df_calendario_aportes_euforia["data_sinal"].max(),
            "data_primeiro_aporte": df_calendario_aportes_euforia["data_aporte_efetiva"].min(),
            "data_ultimo_aporte": df_calendario_aportes_euforia["data_aporte_efetiva"].max(),
            "valor_total_aportado": float(df_calendario_aportes_euforia["valor_aporte_estrategia"].sum()),
            "capital_inicial_estrategia": float(df_calendario_aportes_euforia["capital_inicial_estrategia"].iloc[0]),
            "diferenca_total_vs_capital": float(
                df_calendario_aportes_euforia["valor_aporte_estrategia"].sum()
                - df_calendario_aportes_euforia["capital_inicial_estrategia"].iloc[0]
            ),
            "gap_pregoes_mediano": float(df_calendario_aportes_euforia["gap_pregoes_sinal_para_aporte"].median()),
            "n_aportes_com_ajuste_residual": int(
                df_calendario_aportes_euforia["flag_ajuste_residual_ultimo_aporte"].fillna(False).sum()
            ),
            "n_inconsistencias": int(len(df_inconsistencias_calendario_aportes_euforia)),
        }
    ]
)

df_parametros_calendario_aportes_euforia = pd.DataFrame(
    [
        {
            "parametro": "tipo_estrategia",
            "valor_numerico": pd.NA,
            "valor_texto": TIPO_ESTRATEGIA,
            "unidade": "estrategia",
        },
        {
            "parametro": "grupo_controle",
            "valor_numerico": pd.NA,
            "valor_texto": GRUPO_CONTROLE,
            "unidade": "grupo",
        },
        {
            "parametro": "familia_estrategia",
            "valor_numerico": pd.NA,
            "valor_texto": FAMILIA_ESTRATEGIA,
            "unidade": "familia",
        },
        {
            "parametro": "replica_id_padrao",
            "valor_numerico": float(REPLICA_ID_PADRAO),
            "valor_texto": pd.NA,
            "unidade": "replica",
        },
        {
            "parametro": "regra_data_aporte_efetiva",
            "valor_numerico": pd.NA,
            "valor_texto": "proximo_pregao_apos_data_sinal",
            "unidade": "regra_operacional",
        },
        {
            "parametro": "defasagem_aporte_pregoes",
            "valor_numerico": float(DEFASAGEM_APORTE_PREGOES),
            "valor_texto": pd.NA,
            "unidade": "pregoes",
        },
        {
            "parametro": "n_aportes_planejados",
            "valor_numerico": float(len(df_calendario_aportes_euforia)),
            "valor_texto": pd.NA,
            "unidade": "aportes",
        },
        {
            "parametro": "capital_inicial_estrategia",
            "valor_numerico": float(df_calendario_aportes_euforia["capital_inicial_estrategia"].iloc[0]),
            "valor_texto": pd.NA,
            "unidade": "brl",
        },
    ]
)

df_auditoria_calendario_aportes_euforia = pd.DataFrame(
    [
        {"metrica": "n_aportes_planejados", "valor": float(len(df_calendario_aportes_euforia))},
        {
            "metrica": "valor_total_aportado",
            "valor": float(df_calendario_aportes_euforia["valor_aporte_estrategia"].sum()),
        },
        {
            "metrica": "capital_inicial_estrategia",
            "valor": float(df_calendario_aportes_euforia["capital_inicial_estrategia"].iloc[0]),
        },
        {
            "metrica": "diferenca_total_vs_capital",
            "valor": float(
                df_calendario_aportes_euforia["valor_aporte_estrategia"].sum()
                - df_calendario_aportes_euforia["capital_inicial_estrategia"].iloc[0]
            ),
        },
        {
            "metrica": "n_datas_sinal_fora_calendario_referencia",
            "valor": float((~df_calendario_aportes_euforia["flag_data_sinal_em_pregao_referencia"]).sum()),
        },
        {
            "metrica": "n_datas_aporte_nao_encontradas",
            "valor": float((~df_calendario_aportes_euforia["flag_data_aporte_efetiva_encontrada"]).sum()),
        },
        {
            "metrica": "gap_pregoes_minimo",
            "valor": float(df_calendario_aportes_euforia["gap_pregoes_sinal_para_aporte"].min()),
        },
        {
            "metrica": "gap_pregoes_maximo",
            "valor": float(df_calendario_aportes_euforia["gap_pregoes_sinal_para_aporte"].max()),
        },
        {
            "metrica": "n_aportes_com_ajuste_residual",
            "valor": float(df_calendario_aportes_euforia["flag_ajuste_residual_ultimo_aporte"].fillna(False).sum()),
        },
        {
            "metrica": "n_inconsistencias_calendario",
            "valor": float(len(df_inconsistencias_calendario_aportes_euforia)),
        },
    ]
)

df_distribuicao_anual_aportes_euforia = (
    df_calendario_aportes_euforia
    .groupby(["ano"], as_index=False)
    .agg(
        n_aportes=("request_id", "count"),
        valor_total_aportado=("valor_aporte_estrategia", "sum"),
        valor_medio_aporte=("valor_aporte_estrategia", "mean"),
        intensidade_mediana_sinal=("intensidade_sinal_bruto", "median"),
        score_medio_sinal=("score_sinal_bruto", "mean"),
    )
    .sort_values("ano")
    .reset_index(drop=True)
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_calendario_aportes_euforia, caminho_base_calendario_aportes_euforia, index=False)
salvar_dataframe(df_resumo_calendario_aportes_euforia, caminho_tbl_resumo_calendario_aportes_euforia, index=False)
salvar_dataframe(df_parametros_calendario_aportes_euforia, caminho_tbl_parametros_calendario_aportes_euforia, index=False)
salvar_dataframe(df_auditoria_calendario_aportes_euforia, caminho_tbl_auditoria_calendario_aportes_euforia, index=False)
salvar_dataframe(df_inconsistencias_calendario_aportes_euforia, caminho_tbl_inconsistencias_calendario_aportes_euforia, index=False)
salvar_dataframe(df_distribuicao_anual_aportes_euforia, caminho_tbl_distribuicao_anual_aportes_euforia, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_resumo_calendario_aportes_euforia = df_resumo_calendario_aportes_euforia.copy()
df_amostra_parametros_calendario_aportes_euforia = df_parametros_calendario_aportes_euforia.copy()
df_amostra_auditoria_calendario_aportes_euforia = df_auditoria_calendario_aportes_euforia.copy()
df_amostra_calendario_aportes_euforia = df_calendario_aportes_euforia.head(20).copy()
df_amostra_inconsistencias_calendario_aportes_euforia = df_inconsistencias_calendario_aportes_euforia.head(20).copy()
df_amostra_distribuicao_anual_aportes_euforia = df_distribuicao_anual_aportes_euforia.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo do calendário de aportes da estratégia de euforia:")
print(df_amostra_resumo_calendario_aportes_euforia.to_string(index=False))

print("\nParâmetros do calendário de aportes da estratégia de euforia:")
print(df_amostra_parametros_calendario_aportes_euforia.to_string(index=False))

print("\nAuditoria do calendário de aportes da estratégia de euforia:")
print(df_amostra_auditoria_calendario_aportes_euforia.to_string(index=False))

print("\nBase final do calendário de aportes da estratégia de euforia - amostra:")
print(df_amostra_calendario_aportes_euforia.to_string(index=False))

print("\nInconsistências do calendário de aportes da estratégia de euforia - amostra:")
print(df_amostra_inconsistencias_calendario_aportes_euforia.to_string(index=False))

print("\nDistribuição anual dos aportes da estratégia de euforia - amostra:")
print(df_amostra_distribuicao_anual_aportes_euforia.to_string(index=False))

print("\nArquivos salvos na subetapa 7.4:")
print(f"- {caminho_base_calendario_aportes_euforia}")
print(f"- {caminho_tbl_resumo_calendario_aportes_euforia}")
print(f"- {caminho_tbl_parametros_calendario_aportes_euforia}")
print(f"- {caminho_tbl_auditoria_calendario_aportes_euforia}")
print(f"- {caminho_tbl_inconsistencias_calendario_aportes_euforia}")
print(f"- {caminho_tbl_distribuicao_anual_aportes_euforia}")

print("\nETAPA 7.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 7.4 - CALENDÁRIO DE APORTES DA ESTRATÉGIA DE EUFORIA

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidada       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - valor do aporte por sinal        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_2_base_valor_aporte_por_sinal_estrategia.parquet
Saída   - base calendário aportes          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_4_base_calendario_aportes_euforia.parquet
Saída   - resumo                           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\re

## Etapa 7.5) Geração dos Calendários Aleatórios de Controle

In [33]:
%%time
# ============================================================
# Etapa 7.5) Geração dos Calendários Aleatórios de Controle
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 7.5 - GERAÇÃO DOS CALENDÁRIOS ALEATÓRIOS DE CONTROLE")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_tbl_parametros_globais = gerar_caminho_arquivo(
    etapa=1,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="parametros_globais",
)

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_calendario_aportes_capitulacao = gerar_caminho_arquivo(
    etapa=7,
    subetapa=3,
    tipo_arquivo="base",
    nome="calendario_aportes_capitulacao",
)

caminho_base_calendario_aportes_euforia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=4,
    tipo_arquivo="base",
    nome="calendario_aportes_euforia",
)

caminho_base_calendarios_aleatorios_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=5,
    tipo_arquivo="base",
    nome="calendarios_aleatorios_controle",
)

caminho_tbl_resumo_calendarios_aleatorios_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="resumo_calendarios_aleatorios_controle",
)

caminho_tbl_parametros_calendarios_aleatorios_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="parametros_calendarios_aleatorios_controle",
)

caminho_tbl_auditoria_calendarios_aleatorios_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="auditoria_calendarios_aleatorios_controle",
)

caminho_tbl_inconsistencias_calendarios_aleatorios_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="inconsistencias_calendarios_aleatorios_controle",
)

caminho_tbl_distribuicao_anual_calendarios_aleatorios_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_calendarios_aleatorios_controle",
)

print(f"Entrada - parâmetros globais                      : {caminho_tbl_parametros_globais}")
print(f"Entrada - mercado diário consolidada             : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - calendário de aportes capitulação      : {caminho_base_calendario_aportes_capitulacao}")
print(f"Entrada - calendário de aportes euforia          : {caminho_base_calendario_aportes_euforia}")
print(f"Saída   - base calendários aleatórios            : {caminho_base_calendarios_aleatorios_controle}")
print(f"Saída   - resumo                                 : {caminho_tbl_resumo_calendarios_aleatorios_controle}")
print(f"Saída   - parâmetros                             : {caminho_tbl_parametros_calendarios_aleatorios_controle}")
print(f"Saída   - auditoria                              : {caminho_tbl_auditoria_calendarios_aleatorios_controle}")
print(f"Saída   - inconsistências                        : {caminho_tbl_inconsistencias_calendarios_aleatorios_controle}")
print(f"Saída   - distribuição anual                     : {caminho_tbl_distribuicao_anual_calendarios_aleatorios_controle}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_parametros_globais = pd.read_parquet(caminho_tbl_parametros_globais)
df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_calendario_aportes_capitulacao = pd.read_parquet(caminho_base_calendario_aportes_capitulacao)
df_calendario_aportes_euforia = pd.read_parquet(caminho_base_calendario_aportes_euforia)

print(f"Parâmetros globais                 : {df_parametros_globais.shape[0]:,} linhas x {df_parametros_globais.shape[1]} colunas")
print(f"Mercado diário consolidada         : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Calendário aportes capitulação     : {df_calendario_aportes_capitulacao.shape[0]:,} linhas x {df_calendario_aportes_capitulacao.shape[1]} colunas")
print(f"Calendário aportes euforia         : {df_calendario_aportes_euforia.shape[0]:,} linhas x {df_calendario_aportes_euforia.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

N_REPLICAS_ALEATORIAS_POR_ESTRATEGIA = 5
SEED_FIXO_SORTEIO_DATAS = 20260424
SEED_ALEATORIO_SORTEIO_DATAS = int(datetime.now().strftime("%Y%m%d%H%M%S"))
DEFASAGEM_APORTE_PREGOES = 1
ORIGEM_CALENDARIO = "7_5_aleatorias_controle"

COLUNAS_OBRIGATORIAS_PARAMETROS = [
    "parametro",
    "valor",
]

COLUNAS_OBRIGATORIAS_MERCADO = [
    "data",
]

COLUNAS_OBRIGATORIAS_CALENDARIO_REAL = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "tipo_estrategia",
    "replica_id",
    "origem_calendario",
    "tipo_sinal",
    "ordem_sinal_final",
    "ordem_aporte_estrategia",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_pregao_sinal",
    "ordem_pregao_aporte_efetiva",
    "gap_pregoes_sinal_para_aporte",
    "flag_data_sinal_em_pregao_referencia",
    "flag_data_aporte_efetiva_encontrada",
    "capital_inicial_estrategia",
    "n_sinais_finais_tipo",
    "valor_aporte_estrategia",
    "flag_ajuste_residual_ultimo_aporte",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def padronizar_texto(df, colunas_texto, upper_cols=None, lower_cols=None):
    upper_cols = [] if upper_cols is None else upper_cols
    lower_cols = [] if lower_cols is None else lower_cols

    for coluna in colunas_texto:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()

    for coluna in upper_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.upper()

    for coluna in lower_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.lower()

    return df

def extrair_parametro_numerico(df_parametros, nome_parametro, valor_default=None):
    df_aux = df_parametros.loc[df_parametros["parametro"].astype("string").str.strip().str.lower() == nome_parametro.lower()].copy()
    if df_aux.empty:
        if valor_default is None:
            raise ValueError(f"Parâmetro obrigatório não encontrado: {nome_parametro}")
        return float(valor_default)

    valor_bruto = str(df_aux["valor"].iloc[0]).strip().replace(",", ".")
    try:
        return float(valor_bruto)
    except Exception:
        if valor_default is None:
            raise ValueError(f"Não foi possível converter o parâmetro '{nome_parametro}' para numérico.")
        return float(valor_default)

def construir_mapa_pregoes(df_mercado):
    df_pregoes = (
        df_mercado[["data"]]
        .dropna(subset=["data"])
        .drop_duplicates()
        .sort_values("data")
        .reset_index(drop=True)
        .copy()
    )

    if df_pregoes.empty:
        raise ValueError("Não foi possível construir o calendário de pregões de referência a partir da base de mercado.")

    df_pregoes["ordem_pregao"] = np.arange(1, len(df_pregoes) + 1, dtype=int)

    mapa_ordem_para_data = dict(zip(df_pregoes["ordem_pregao"].tolist(), df_pregoes["data"].tolist()))
    mapa_data_para_ordem = dict(zip(df_pregoes["data"].tolist(), df_pregoes["ordem_pregao"].tolist()))

    return df_pregoes, mapa_ordem_para_data, mapa_data_para_ordem

def sortear_ordens_pregao_com_cooldown(ordem_minima, ordem_maxima, n_aportes, cooldown_pregoes, seed_local):
    ordem_minima = int(ordem_minima)
    ordem_maxima = int(ordem_maxima)
    n_aportes = int(n_aportes)
    cooldown_pregoes = int(cooldown_pregoes)

    if n_aportes <= 0:
        raise ValueError("O número de aportes a sortear deve ser positivo.")

    if ordem_maxima < ordem_minima:
        raise ValueError("A janela de sorteio possui ordem máxima menor que a ordem mínima.")

    comprimento_janela = ordem_maxima - ordem_minima + 1
    capacidade_maxima = ((comprimento_janela - 1) // cooldown_pregoes) + 1

    if capacidade_maxima < n_aportes:
        raise ValueError(
            f"A janela de pregões não comporta {n_aportes} aportes com cooldown de {cooldown_pregoes} pregões."
        )

    comprimento_comprimido = comprimento_janela - ((n_aportes - 1) * (cooldown_pregoes - 1))
    universo_comprimido = np.arange(ordem_minima, ordem_minima + comprimento_comprimido, dtype=int)

    if len(universo_comprimido) < n_aportes:
        raise ValueError("O universo comprimido ficou menor que o número de aportes a sortear.")

    rng = np.random.default_rng(seed_local)
    ordens_comprimidas = np.sort(rng.choice(universo_comprimido, size=n_aportes, replace=False))
    deslocamentos = np.arange(n_aportes, dtype=int) * (cooldown_pregoes - 1)
    ordens_sorteadas = ordens_comprimidas + deslocamentos

    return ordens_sorteadas.astype(int)

def mapear_direcao_sinal(tipo_sinal):
    tipo_sinal = str(tipo_sinal).strip().lower()
    if tipo_sinal == "capitulacao":
        return "contrarian_bullish"
    if tipo_sinal == "euforia":
        return "contrarian_bearish"
    return pd.NA

def gerar_seed_replicacao(estrategia_referencia, replica_id, semente_base):
    if str(estrategia_referencia).strip().lower() == "capitulacao":
        return int(semente_base + replica_id)
    if str(estrategia_referencia).strip().lower() == "euforia":
        return int(semente_base + 100 + replica_id)
    return int(semente_base + replica_id)

def seed_esta_vazia(valor):
    if valor is None:
        return True

    texto = str(valor).strip().lower()
    return texto in {"", "none", "null", "nan", "<na>"}

def resolver_seed(seed_fixo, seed_aleatorio):
    if not seed_esta_vazia(seed_fixo):
        return int(seed_fixo), "fixo"

    return int(seed_aleatorio), "aleatorio"

def gerar_calendario_aleatorio_para_estrategia(
    df_calendario_real,
    df_pregoes_referencia,
    mapa_ordem_para_data,
    cooldown_equivalente_pregoes,
    n_replicas,
    semente_base,
):
    lista_replicas = []

    estrategia_referencia = str(df_calendario_real["estrategia_referencia"].iloc[0]).strip().lower()
    tipo_sinal_referencia = str(df_calendario_real["tipo_sinal"].iloc[0]).strip().lower()
    capital_inicial_estrategia = float(df_calendario_real["capital_inicial_estrategia"].iloc[0])
    n_aportes = int(len(df_calendario_real))
    tipo_estrategia_aleatoria = f"aleatoria_{estrategia_referencia}"
    familia_estrategia_aleatoria = f"{estrategia_referencia}_aleatoria_controle"

    ordem_global_sinal_referencia_min = int(pd.to_numeric(df_calendario_real["ordem_global_sinal"], errors="coerce").min())
    ordem_global_sinal_referencia_max = int(pd.to_numeric(df_calendario_real["ordem_global_sinal"], errors="coerce").max())

    ordem_pregao_sinal_min = int(pd.to_numeric(df_calendario_real["ordem_pregao_sinal"], errors="coerce").min())
    ordem_pregao_sinal_max = int(pd.to_numeric(df_calendario_real["ordem_pregao_sinal"], errors="coerce").max())

    ordem_pregao_sinal_max_sorteavel = int(
        min(
            ordem_pregao_sinal_max,
            int(df_pregoes_referencia["ordem_pregao"].max()) - DEFASAGEM_APORTE_PREGOES,
        )
    )

    if ordem_pregao_sinal_max_sorteavel < ordem_pregao_sinal_min:
        raise ValueError(f"A janela de sorteio ficou inválida para a estratégia de referência {estrategia_referencia}.")

    data_janela_inicial_referencia = pd.Timestamp(df_calendario_real["data_sinal"].min())
    data_janela_final_referencia = pd.Timestamp(df_calendario_real["data_sinal"].max())

    for replica_id in range(1, n_replicas + 1):
        seed_replicacao = gerar_seed_replicacao(
            estrategia_referencia=estrategia_referencia,
            replica_id=replica_id,
            semente_base=semente_base,
        )

        ordens_sinal_sorteadas = sortear_ordens_pregao_com_cooldown(
            ordem_minima=ordem_pregao_sinal_min,
            ordem_maxima=ordem_pregao_sinal_max_sorteavel,
            n_aportes=n_aportes,
            cooldown_pregoes=cooldown_equivalente_pregoes,
            seed_local=seed_replicacao,
        )

        ordens_aporte_sorteadas = ordens_sinal_sorteadas + DEFASAGEM_APORTE_PREGOES

        datas_sinal_sorteadas = [pd.Timestamp(mapa_ordem_para_data[int(ordem)]) for ordem in ordens_sinal_sorteadas]
        datas_aporte_sorteadas = [pd.Timestamp(mapa_ordem_para_data[int(ordem)]) for ordem in ordens_aporte_sorteadas]

        df_replica = df_calendario_real.copy().sort_values("ordem_sinal_final").reset_index(drop=True)

        # Preservação explícita das informações do sinal real usado como referência.
        # As datas sorteadas passam a ser as datas operacionais da réplica aleatória;
        # por isso, score e intensidade do sinal real não devem permanecer nos campos canônicos.
        df_replica["data_sinal_referencia"] = pd.to_datetime(df_replica["data_sinal"], errors="coerce")
        df_replica["data_aporte_efetiva_referencia"] = pd.to_datetime(df_replica["data_aporte_efetiva"], errors="coerce")
        df_replica["ordem_sinal_final_referencia"] = pd.to_numeric(df_replica["ordem_sinal_final"], errors="coerce").astype("Int64")
        df_replica["ordem_aporte_estrategia_referencia"] = pd.to_numeric(df_replica["ordem_aporte_estrategia"], errors="coerce").astype("Int64")
        df_replica["ordem_global_sinal_referencia"] = pd.to_numeric(df_replica["ordem_global_sinal"], errors="coerce").astype("Int64")
        df_replica["ordem_pregao_sinal_referencia"] = pd.to_numeric(df_replica["ordem_pregao_sinal"], errors="coerce").astype("Int64")
        df_replica["ordem_pregao_aporte_efetiva_referencia"] = pd.to_numeric(df_replica["ordem_pregao_aporte_efetiva"], errors="coerce").astype("Int64")
        df_replica["tipo_sinal_referencia"] = df_replica["tipo_sinal"].astype("string").str.strip().str.lower()

        if "direcao_sinal" in df_replica.columns:
            df_replica["direcao_sinal_referencia"] = df_replica["direcao_sinal"].astype("string").str.strip().str.lower()
        else:
            df_replica["direcao_sinal_referencia"] = df_replica["tipo_sinal_referencia"].map(mapear_direcao_sinal).astype("string")

        df_replica["score_sinal_referencia"] = pd.to_numeric(df_replica["score_sinal_bruto"], errors="coerce")
        df_replica["intensidade_sinal_referencia"] = pd.to_numeric(df_replica["intensidade_sinal_bruto"], errors="coerce")

        df_replica["request_id"] = [
            f"{tipo_estrategia_aleatoria}__replica_{replica_id}__{int(ordem)}"
            for ordem in df_replica["ordem_sinal_final"].astype(int).tolist()
        ]
        df_replica["estrategia_id"] = f"{tipo_estrategia_aleatoria}_replica_{replica_id}"
        df_replica["grupo_controle"] = "aleatoria"
        df_replica["familia_estrategia"] = familia_estrategia_aleatoria
        df_replica["estrategia_referencia"] = estrategia_referencia
        df_replica["tipo_estrategia"] = tipo_estrategia_aleatoria
        df_replica["replica_id"] = int(replica_id)
        df_replica["seed_replicacao"] = int(seed_replicacao)
        df_replica["origem_calendario"] = ORIGEM_CALENDARIO
        df_replica["tipo_sinal"] = tipo_sinal_referencia
        df_replica["direcao_sinal"] = mapear_direcao_sinal(tipo_sinal_referencia)

        df_replica["data_sinal"] = datas_sinal_sorteadas
        df_replica["data_aporte_efetiva"] = datas_aporte_sorteadas
        df_replica["ordem_pregao_sinal"] = ordens_sinal_sorteadas
        df_replica["ordem_pregao_aporte_efetiva"] = ordens_aporte_sorteadas
        df_replica["data_sinal_aleatoria"] = pd.to_datetime(df_replica["data_sinal"], errors="coerce")
        df_replica["data_aporte_efetiva_aleatoria"] = pd.to_datetime(df_replica["data_aporte_efetiva"], errors="coerce")
        df_replica["ordem_pregao_sinal_aleatorio"] = pd.to_numeric(df_replica["ordem_pregao_sinal"], errors="coerce").astype("Int64")
        df_replica["ordem_pregao_aporte_efetiva_aleatorio"] = pd.to_numeric(df_replica["ordem_pregao_aporte_efetiva"], errors="coerce").astype("Int64")
        df_replica["score_sinal_aleatorio"] = pd.Series(pd.NA, index=df_replica.index, dtype="Float64")
        df_replica["intensidade_sinal_aleatorio"] = pd.Series(pd.NA, index=df_replica.index, dtype="Float64")
        df_replica["score_sinal_bruto"] = pd.Series(pd.NA, index=df_replica.index, dtype="Float64")
        df_replica["intensidade_sinal_bruto"] = pd.Series(pd.NA, index=df_replica.index, dtype="Float64")
        df_replica["gap_pregoes_sinal_para_aporte"] = (
            df_replica["ordem_pregao_aporte_efetiva"] - df_replica["ordem_pregao_sinal"]
        )
        df_replica["flag_data_sinal_em_pregao_referencia"] = True
        df_replica["flag_data_aporte_efetiva_encontrada"] = True
        df_replica["flag_gap_aporte_valido"] = (
            df_replica["gap_pregoes_sinal_para_aporte"] == DEFASAGEM_APORTE_PREGOES
        )

        df_replica["pregoes_desde_sinal_aleatorio_anterior"] = (
            pd.Series(df_replica["ordem_pregao_sinal"]).diff()
        )
        df_replica["flag_cooldown_equivalente_respeitado"] = (
            df_replica["pregoes_desde_sinal_aleatorio_anterior"].isna()
            | (df_replica["pregoes_desde_sinal_aleatorio_anterior"] >= cooldown_equivalente_pregoes)
        )

        df_replica["ano"] = pd.to_datetime(df_replica["data_sinal"], errors="coerce").dt.year
        df_replica["mes"] = pd.to_datetime(df_replica["data_sinal"], errors="coerce").dt.month
        df_replica["janela_data_inicial_referencia"] = data_janela_inicial_referencia
        df_replica["janela_data_final_referencia"] = data_janela_final_referencia
        df_replica["cooldown_equivalente_pregoes"] = int(cooldown_equivalente_pregoes)
        df_replica["ordem_global_sinal_referencia_min"] = int(ordem_global_sinal_referencia_min)
        df_replica["ordem_global_sinal_referencia_max"] = int(ordem_global_sinal_referencia_max)
        df_replica["capital_inicial_estrategia"] = capital_inicial_estrategia
        df_replica["flag_evento_aporte"] = True

        df_replica = df_replica[
            [
                "request_id",
                "estrategia_id",
                "grupo_controle",
                "familia_estrategia",
                "estrategia_referencia",
                "tipo_estrategia",
                "replica_id",
                "seed_replicacao",
                "origem_calendario",
                "tipo_sinal",
                "direcao_sinal",
                "tipo_sinal_referencia",
                "direcao_sinal_referencia",
                "ordem_sinal_final",
                "ordem_sinal_final_referencia",
                "ordem_aporte_estrategia",
                "ordem_aporte_estrategia_referencia",
                "ordem_global_sinal",
                "ordem_global_sinal_referencia",
                "data_sinal",
                "data_sinal_aleatoria",
                "data_sinal_referencia",
                "data_aporte_efetiva",
                "data_aporte_efetiva_aleatoria",
                "data_aporte_efetiva_referencia",
                "ordem_pregao_sinal",
                "ordem_pregao_sinal_aleatorio",
                "ordem_pregao_sinal_referencia",
                "ordem_pregao_aporte_efetiva",
                "ordem_pregao_aporte_efetiva_aleatorio",
                "ordem_pregao_aporte_efetiva_referencia",
                "gap_pregoes_sinal_para_aporte",
                "flag_data_sinal_em_pregao_referencia",
                "flag_data_aporte_efetiva_encontrada",
                "flag_gap_aporte_valido",
                "flag_cooldown_equivalente_respeitado",
                "pregoes_desde_sinal_aleatorio_anterior",
                "ano",
                "mes",
                "janela_data_inicial_referencia",
                "janela_data_final_referencia",
                "cooldown_equivalente_pregoes",
                "ordem_global_sinal_referencia_min",
                "ordem_global_sinal_referencia_max",
                "score_sinal_referencia",
                "intensidade_sinal_referencia",
                "score_sinal_aleatorio",
                "intensidade_sinal_aleatorio",
                "score_sinal_bruto",
                "intensidade_sinal_bruto",
                "capital_inicial_estrategia",
                "n_sinais_finais_tipo",
                "valor_cada_aporte_teorico_exato",
                "valor_aporte_estrategia",
                "flag_ajuste_residual_ultimo_aporte",
                "valor_medio_aporte_planejado_tipo",
                "valor_aportado_planejado_acumulado",
                "pct_capital_planejado_acumulado",
                "flag_evento_aporte",
            ]
        ].copy()

        lista_replicas.append(df_replica)

    return pd.concat(lista_replicas, axis=0, ignore_index=True)

print(f"Número de réplicas aleatórias por estratégia : {N_REPLICAS_ALEATORIAS_POR_ESTRATEGIA}")
print(f"Seed fixo do sorteio de datas                : {SEED_FIXO_SORTEIO_DATAS}")
print(f"Seed aleatório do sorteio de datas           : {SEED_ALEATORIO_SORTEIO_DATAS}")
print(f"Defasagem operacional do aporte              : {DEFASAGEM_APORTE_PREGOES} pregão após o sinal")
print("OK")

# ============================================================
# 5) Preparação das bases e calendário de referência
# ============================================================

print("\n[5/10] Preparação das bases e calendário de referência...")

validar_colunas_obrigatorias(df_parametros_globais, COLUNAS_OBRIGATORIAS_PARAMETROS)
validar_colunas_obrigatorias(df_mercado_diario_consolidada, COLUNAS_OBRIGATORIAS_MERCADO)
validar_colunas_obrigatorias(df_calendario_aportes_capitulacao, COLUNAS_OBRIGATORIAS_CALENDARIO_REAL)
validar_colunas_obrigatorias(df_calendario_aportes_euforia, COLUNAS_OBRIGATORIAS_CALENDARIO_REAL)

df_parametros_globais = padronizar_texto(
    df=df_parametros_globais.copy(),
    colunas_texto=["parametro", "valor"],
    lower_cols=["parametro"],
)

df_mercado_diario_consolidada["data"] = pd.to_datetime(df_mercado_diario_consolidada["data"], errors="coerce")

for df_calendario in [df_calendario_aportes_capitulacao, df_calendario_aportes_euforia]:
    df_calendario["data_sinal"] = pd.to_datetime(df_calendario["data_sinal"], errors="coerce")
    df_calendario["data_aporte_efetiva"] = pd.to_datetime(df_calendario["data_aporte_efetiva"], errors="coerce")

cooldown_equivalente_pregoes = int(extrair_parametro_numerico(
    df_parametros=df_parametros_globais,
    nome_parametro="cooldown_sinais_pregoes",
    valor_default=63,
))

seed_sorteio_datas, modo_seed_sorteio_datas = resolver_seed(
    seed_fixo=SEED_FIXO_SORTEIO_DATAS,
    seed_aleatorio=SEED_ALEATORIO_SORTEIO_DATAS,
)

df_pregoes_referencia, mapa_ordem_para_data, mapa_data_para_ordem = construir_mapa_pregoes(
    df_mercado=df_mercado_diario_consolidada.copy()
)

if int(pd.to_numeric(df_calendario_aportes_capitulacao["gap_pregoes_sinal_para_aporte"], errors="coerce").dropna().median()) != DEFASAGEM_APORTE_PREGOES:
    raise ValueError("A base da etapa 7.3 não está compatível com a defasagem operacional canônica.")
if int(pd.to_numeric(df_calendario_aportes_euforia["gap_pregoes_sinal_para_aporte"], errors="coerce").dropna().median()) != DEFASAGEM_APORTE_PREGOES:
    raise ValueError("A base da etapa 7.4 não está compatível com a defasagem operacional canônica.")

print(f"Cooldown oficial carregado                  : {cooldown_equivalente_pregoes} pregões")
print(f"Seed efetiva do sorteio de datas            : {seed_sorteio_datas}")
print(f"Modo da seed do sorteio de datas            : {modo_seed_sorteio_datas}")
print(f"Quantidade de pregões de referência         : {len(df_pregoes_referencia):,}")
print(f"Aportes de capitulação usados como modelo   : {len(df_calendario_aportes_capitulacao):,}")
print(f"Aportes de euforia usados como modelo       : {len(df_calendario_aportes_euforia):,}")
print("OK")

# ============================================================
# 6) Geração dos calendários aleatórios de controle
# ============================================================

print("\n[6/10] Geração dos calendários aleatórios de controle...")

df_aleatoria_capitulacao = gerar_calendario_aleatorio_para_estrategia(
    df_calendario_real=df_calendario_aportes_capitulacao.copy(),
    df_pregoes_referencia=df_pregoes_referencia.copy(),
    mapa_ordem_para_data=mapa_ordem_para_data,
    cooldown_equivalente_pregoes=cooldown_equivalente_pregoes,
    n_replicas=N_REPLICAS_ALEATORIAS_POR_ESTRATEGIA,
    semente_base=seed_sorteio_datas,
)

df_aleatoria_euforia = gerar_calendario_aleatorio_para_estrategia(
    df_calendario_real=df_calendario_aportes_euforia.copy(),
    df_pregoes_referencia=df_pregoes_referencia.copy(),
    mapa_ordem_para_data=mapa_ordem_para_data,
    cooldown_equivalente_pregoes=cooldown_equivalente_pregoes,
    n_replicas=N_REPLICAS_ALEATORIAS_POR_ESTRATEGIA,
    semente_base=seed_sorteio_datas,
)

df_calendarios_aleatorios_controle = (
    pd.concat([df_aleatoria_capitulacao, df_aleatoria_euforia], axis=0, ignore_index=True)
    .sort_values(["estrategia_referencia", "replica_id", "ordem_aporte_estrategia"])
    .reset_index(drop=True)
)

df_inconsistencias_calendarios_aleatorios_controle = df_calendarios_aleatorios_controle.loc[
    (
        ~df_calendarios_aleatorios_controle["flag_data_sinal_em_pregao_referencia"]
    )
    | (
        ~df_calendarios_aleatorios_controle["flag_data_aporte_efetiva_encontrada"]
    )
    | (
        ~df_calendarios_aleatorios_controle["flag_gap_aporte_valido"]
    )
    | (
        ~df_calendarios_aleatorios_controle["flag_cooldown_equivalente_respeitado"]
    )
].copy()

n_score_referencia_ausente = int(df_calendarios_aleatorios_controle["score_sinal_referencia"].isna().sum())
n_score_bruto_preenchido = int(df_calendarios_aleatorios_controle["score_sinal_bruto"].notna().sum())
n_score_aleatorio_preenchido = int(df_calendarios_aleatorios_controle["score_sinal_aleatorio"].notna().sum())

print(f"Linhas totais dos calendários aleatórios : {len(df_calendarios_aleatorios_controle):,}")
print(f"Réplicas totais geradas                  : {df_calendarios_aleatorios_controle[['estrategia_referencia', 'replica_id']].drop_duplicates().shape[0]:,}")
print(f"Inconsistências identificadas            : {len(df_inconsistencias_calendarios_aleatorios_controle):,}")
print(f"Scores de referência ausentes            : {n_score_referencia_ausente:,}")
print(f"Scores brutos indevidamente preenchidos  : {n_score_bruto_preenchido:,}")
print(f"Scores aleatórios preenchidos            : {n_score_aleatorio_preenchido:,}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

df_resumo_calendarios_aleatorios_controle = (
    df_calendarios_aleatorios_controle
    .groupby(["estrategia_referencia", "replica_id", "tipo_estrategia"], as_index=False)
    .agg(
        n_aportes=("request_id", "count"),
        data_primeiro_sinal=("data_sinal", "min"),
        data_ultimo_sinal=("data_sinal", "max"),
        data_primeiro_aporte=("data_aporte_efetiva", "min"),
        data_ultimo_aporte=("data_aporte_efetiva", "max"),
        valor_total_aportado=("valor_aporte_estrategia", "sum"),
        capital_inicial_estrategia=("capital_inicial_estrategia", "first"),
        gap_pregoes_sinal_para_aporte_mediano=("gap_pregoes_sinal_para_aporte", "median"),
        pregoes_desde_sinal_aleatorio_anterior_minimo=("pregoes_desde_sinal_aleatorio_anterior", "min"),
        n_aportes_com_ajuste_residual=("flag_ajuste_residual_ultimo_aporte", lambda x: int(pd.Series(x).fillna(False).sum())),
        seed_replicacao=("seed_replicacao", "first"),
    )
    .sort_values(["estrategia_referencia", "replica_id"])
    .reset_index(drop=True)
)

df_resumo_calendarios_aleatorios_controle["diferenca_total_vs_capital"] = (
    df_resumo_calendarios_aleatorios_controle["valor_total_aportado"]
    - df_resumo_calendarios_aleatorios_controle["capital_inicial_estrategia"]
)

df_parametros_calendarios_aleatorios_controle = pd.DataFrame(
    [
        {
            "parametro": "n_replicas_aleatorias_por_estrategia",
            "valor_numerico": float(N_REPLICAS_ALEATORIAS_POR_ESTRATEGIA),
            "valor_texto": pd.NA,
            "unidade": "replicas",
        },
        {
            "parametro": "cooldown_equivalente_pregoes",
            "valor_numerico": float(cooldown_equivalente_pregoes),
            "valor_texto": pd.NA,
            "unidade": "pregoes",
        },
        {
            "parametro": "defasagem_aporte_pregoes",
            "valor_numerico": float(DEFASAGEM_APORTE_PREGOES),
            "valor_texto": pd.NA,
            "unidade": "pregoes",
        },
        {
            "parametro": "seed_fixo_sorteio_datas",
            "valor_numerico": float(SEED_FIXO_SORTEIO_DATAS) if not seed_esta_vazia(SEED_FIXO_SORTEIO_DATAS) else pd.NA,
            "valor_texto": pd.NA,
            "unidade": "seed",
        },
        {
            "parametro": "seed_aleatorio_sorteio_datas",
            "valor_numerico": float(SEED_ALEATORIO_SORTEIO_DATAS) if not seed_esta_vazia(SEED_ALEATORIO_SORTEIO_DATAS) else pd.NA,
            "valor_texto": pd.NA,
            "unidade": "seed",
        },
        {
            "parametro": "seed_sorteio_datas",
            "valor_numerico": float(seed_sorteio_datas),
            "valor_texto": pd.NA,
            "unidade": "seed",
        },
        {
            "parametro": "modo_seed_sorteio_datas",
            "valor_numerico": pd.NA,
            "valor_texto": modo_seed_sorteio_datas,
            "unidade": "modo_seed",
        },
        {
            "parametro": "regra_janela_temporal_aleatoria",
            "valor_numerico": pd.NA,
            "valor_texto": "mesma_janela_entre_primeiro_e_ultimo_sinal_da_estrategia_real",
            "unidade": "regra_metodologica",
        },
        {
            "parametro": "regra_valor_aporte_aleatorio",
            "valor_numerico": pd.NA,
            "valor_texto": "mesmo_valor_planejado_por_ordem_da_estrategia_real",
            "unidade": "regra_metodologica",
        },
        {
            "parametro": "regra_sorteio_datas",
            "valor_numerico": pd.NA,
            "valor_texto": "amostragem_aleatoria_em_pregoes_validos_com_cooldown_equivalente",
            "unidade": "regra_metodologica",
        },
        {
            "parametro": "regra_score_intensidade_calendarios_aleatorios",
            "valor_numerico": pd.NA,
            "valor_texto": "score_e_intensidade_do_sinal_real_sao_preservados_apenas_como_referencia",
            "unidade": "regra_metodologica",
        },
    ]
)

df_auditoria_calendarios_aleatorios_controle = pd.DataFrame(
    [
        {
            "metrica": "n_replicas_esperadas",
            "valor": float(N_REPLICAS_ALEATORIAS_POR_ESTRATEGIA * 2),
        },
        {
            "metrica": "n_replicas_geradas",
            "valor": float(df_calendarios_aleatorios_controle[["estrategia_referencia", "replica_id"]].drop_duplicates().shape[0]),
        },
        {
            "metrica": "n_linhas_total_calendarios_aleatorios",
            "valor": float(len(df_calendarios_aleatorios_controle)),
        },
        {
            "metrica": "n_inconsistencias_calendarios_aleatorios",
            "valor": float(len(df_inconsistencias_calendarios_aleatorios_controle)),
        },
        {
            "metrica": "n_linhas_capitulacao_aleatoria",
            "valor": float((df_calendarios_aleatorios_controle["estrategia_referencia"] == "capitulacao").sum()),
        },
        {
            "metrica": "n_linhas_euforia_aleatoria",
            "valor": float((df_calendarios_aleatorios_controle["estrategia_referencia"] == "euforia").sum()),
        },
        {
            "metrica": "n_datas_sinal_fora_calendario_referencia",
            "valor": float((~df_calendarios_aleatorios_controle["flag_data_sinal_em_pregao_referencia"]).sum()),
        },
        {
            "metrica": "n_datas_aporte_nao_encontradas",
            "valor": float((~df_calendarios_aleatorios_controle["flag_data_aporte_efetiva_encontrada"]).sum()),
        },
        {
            "metrica": "n_gaps_aporte_invalidos",
            "valor": float((~df_calendarios_aleatorios_controle["flag_gap_aporte_valido"]).sum()),
        },
        {
            "metrica": "n_violacoes_cooldown_equivalente",
            "valor": float((~df_calendarios_aleatorios_controle["flag_cooldown_equivalente_respeitado"]).sum()),
        },
        {
            "metrica": "n_score_sinal_referencia_ausente",
            "valor": float(df_calendarios_aleatorios_controle["score_sinal_referencia"].isna().sum()),
        },
        {
            "metrica": "n_score_sinal_bruto_preenchido",
            "valor": float(df_calendarios_aleatorios_controle["score_sinal_bruto"].notna().sum()),
        },
        {
            "metrica": "n_score_sinal_aleatorio_preenchido",
            "valor": float(df_calendarios_aleatorios_controle["score_sinal_aleatorio"].notna().sum()),
        },
        {
            "metrica": "seed_sorteio_datas",
            "valor": float(seed_sorteio_datas),
        },
    ]
)

df_distribuicao_anual_calendarios_aleatorios_controle = (
    df_calendarios_aleatorios_controle
    .groupby(["estrategia_referencia", "replica_id", "ano"], as_index=False)
    .agg(
        n_aportes=("request_id", "count"),
        valor_total_aportado=("valor_aporte_estrategia", "sum"),
        valor_medio_aporte=("valor_aporte_estrategia", "mean"),
    )
    .sort_values(["estrategia_referencia", "replica_id", "ano"])
    .reset_index(drop=True)
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_calendarios_aleatorios_controle, caminho_base_calendarios_aleatorios_controle, index=False)
salvar_dataframe(df_resumo_calendarios_aleatorios_controle, caminho_tbl_resumo_calendarios_aleatorios_controle, index=False)
salvar_dataframe(df_parametros_calendarios_aleatorios_controle, caminho_tbl_parametros_calendarios_aleatorios_controle, index=False)
salvar_dataframe(df_auditoria_calendarios_aleatorios_controle, caminho_tbl_auditoria_calendarios_aleatorios_controle, index=False)
salvar_dataframe(df_inconsistencias_calendarios_aleatorios_controle, caminho_tbl_inconsistencias_calendarios_aleatorios_controle, index=False)
salvar_dataframe(df_distribuicao_anual_calendarios_aleatorios_controle, caminho_tbl_distribuicao_anual_calendarios_aleatorios_controle, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_resumo_calendarios_aleatorios_controle = df_resumo_calendarios_aleatorios_controle.copy()
df_amostra_parametros_calendarios_aleatorios_controle = df_parametros_calendarios_aleatorios_controle.copy()
df_amostra_auditoria_calendarios_aleatorios_controle = df_auditoria_calendarios_aleatorios_controle.copy()
df_amostra_calendarios_aleatorios_controle = df_calendarios_aleatorios_controle.head(20).copy()
df_amostra_distribuicao_anual_calendarios_aleatorios_controle = df_distribuicao_anual_calendarios_aleatorios_controle.head(20).copy()
df_amostra_inconsistencias_calendarios_aleatorios_controle = df_inconsistencias_calendarios_aleatorios_controle.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nResumo dos calendários aleatórios de controle:")
print(df_amostra_resumo_calendarios_aleatorios_controle.to_string(index=False))

print("\nParâmetros dos calendários aleatórios de controle:")
print(df_amostra_parametros_calendarios_aleatorios_controle.to_string(index=False))

print("\nAuditoria dos calendários aleatórios de controle:")
print(df_amostra_auditoria_calendarios_aleatorios_controle.to_string(index=False))

print("\nAmostra da base consolidada dos calendários aleatórios de controle:")
print(df_amostra_calendarios_aleatorios_controle.to_string(index=False))

print("\nDistribuição anual dos calendários aleatórios de controle - amostra:")
print(df_amostra_distribuicao_anual_calendarios_aleatorios_controle.to_string(index=False))

print("\nInconsistências dos calendários aleatórios de controle - amostra:")
print(df_amostra_inconsistencias_calendarios_aleatorios_controle.to_string(index=False))

print("\nArquivos salvos na subetapa 7.5:")
print(f"- {caminho_base_calendarios_aleatorios_controle}")
print(f"- {caminho_tbl_resumo_calendarios_aleatorios_controle}")
print(f"- {caminho_tbl_parametros_calendarios_aleatorios_controle}")
print(f"- {caminho_tbl_auditoria_calendarios_aleatorios_controle}")
print(f"- {caminho_tbl_inconsistencias_calendarios_aleatorios_controle}")
print(f"- {caminho_tbl_distribuicao_anual_calendarios_aleatorios_controle}")

print("\nETAPA 7.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 7.5 - GERAÇÃO DOS CALENDÁRIOS ALEATÓRIOS DE CONTROLE

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - parâmetros globais                      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_1_tbl_parametros_globais.parquet
Entrada - mercado diário consolidada             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - calendário de aportes capitulação      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_3_base_calendario_aportes_capitulacao.parquet
Entrada - calendário de aportes euforia          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_

## Etapa 7.6) Calendário de Aportes das Estratégias Mensais de Controle

In [34]:
%%time
# ============================================================
# Etapa 7.6) Calendário de Aportes das Estratégias Mensais de Controle
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 7.6 - CALENDÁRIO DE APORTES DAS ESTRATÉGIAS MENSAIS DE CONTROLE")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_tbl_parametros_globais = gerar_caminho_arquivo(
    etapa=1,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="parametros_globais",
)

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_tickers_aptos_janela_contabil = gerar_caminho_arquivo(
    etapa=4,
    subetapa=5,
    tipo_arquivo="base",
    nome="tickers_aptos_janela_contabil",
)

caminho_base_calendario_aportes_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="base",
    nome="calendario_aportes_estrategias_mensais_controle",
)

caminho_tbl_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="estrategias_mensais_controle",
)

caminho_tbl_resumo_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="resumo_estrategias_mensais_controle",
)

caminho_tbl_parametros_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="parametros_estrategias_mensais_controle",
)

caminho_tbl_auditoria_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="auditoria_estrategias_mensais_controle",
)

caminho_tbl_inconsistencias_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="inconsistencias_estrategias_mensais_controle",
)

caminho_tbl_distribuicao_anual_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_estrategias_mensais_controle",
)

print(f"Entrada - parâmetros globais                           : {caminho_tbl_parametros_globais}")
print(f"Entrada - mercado diário consolidada                  : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - tickers aptos por janela contábil           : {caminho_base_tickers_aptos_janela_contabil}")
print(f"Saída   - base calendários mensais de controle        : {caminho_base_calendario_aportes_estrategias_mensais_controle}")
print(f"Saída   - definição das estratégias mensais           : {caminho_tbl_estrategias_mensais_controle}")
print(f"Saída   - resumo                                      : {caminho_tbl_resumo_estrategias_mensais_controle}")
print(f"Saída   - parâmetros                                  : {caminho_tbl_parametros_estrategias_mensais_controle}")
print(f"Saída   - auditoria                                   : {caminho_tbl_auditoria_estrategias_mensais_controle}")
print(f"Saída   - inconsistências                             : {caminho_tbl_inconsistencias_estrategias_mensais_controle}")
print(f"Saída   - distribuição anual                          : {caminho_tbl_distribuicao_anual_estrategias_mensais_controle}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_parametros_globais = pd.read_parquet(caminho_tbl_parametros_globais)
df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_tickers_aptos_janela_contabil = pd.read_parquet(caminho_base_tickers_aptos_janela_contabil)

print(f"Parâmetros globais                 : {df_parametros_globais.shape[0]:,} linhas x {df_parametros_globais.shape[1]} colunas")
print(f"Mercado diário consolidada         : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Tickers aptos por janela contábil  : {df_tickers_aptos_janela_contabil.shape[0]:,} linhas x {df_tickers_aptos_janela_contabil.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

N_CARTEIRAS_ALEATORIAS_20 = 5
N_CARTEIRAS_ALEATORIAS_30 = 5
TAMANHO_CARTEIRA_ALEATORIA_20 = 20
TAMANHO_CARTEIRA_ALEATORIA_30 = 30
SEED_FIXO_SORTEIO_ACOES = 20260424
SEED_ALEATORIO_SORTEIO_ACOES = int(datetime.now().strftime("%Y%m%d%H%M%S"))
ORIGEM_CALENDARIO = "7_6_mensais"
REPLICA_ID_PADRAO_MENSAL = 0

COLUNAS_OBRIGATORIAS_PARAMETROS = [
    "parametro",
    "valor",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def padronizar_texto(df, colunas_texto, upper_cols=None, lower_cols=None):
    upper_cols = [] if upper_cols is None else upper_cols
    lower_cols = [] if lower_cols is None else lower_cols

    for coluna in colunas_texto:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()

    for coluna in upper_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.upper()

    for coluna in lower_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.lower()

    return df

def detectar_coluna_data(df, candidatos):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    raise ValueError(f"Não foi possível detectar a coluna de data. Candidatos testados: {candidatos}")

def detectar_coluna_identificador(df, candidatos):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    return None

def extrair_parametro_numerico_flexivel(df_parametros, candidatos, valor_default):
    df_aux = df_parametros.copy()
    df_aux["parametro_norm"] = df_aux["parametro"].astype("string").str.strip().str.lower()

    for nome_parametro in candidatos:
        linha = df_aux.loc[df_aux["parametro_norm"] == str(nome_parametro).strip().lower()]
        if not linha.empty:
            valor_bruto = str(linha["valor"].iloc[0]).strip().replace(".", "").replace(",", ".")
            try:
                return float(valor_bruto)
            except Exception:
                break

    return float(valor_default)

def extrair_parametro_texto_flexivel(df_parametros, candidatos, valor_default):
    df_aux = df_parametros.copy()
    df_aux["parametro_norm"] = df_aux["parametro"].astype("string").str.strip().str.lower()

    for nome_parametro in candidatos:
        linha = df_aux.loc[df_aux["parametro_norm"] == str(nome_parametro).strip().lower()]
        if not linha.empty:
            valor_bruto = str(linha["valor"].iloc[0]).strip()
            if valor_bruto != "":
                return valor_bruto

    return str(valor_default)

def calcular_aportes_uniformes(capital_inicial_estrategia, n_aportes):
    valor_teorico = float(capital_inicial_estrategia) / float(n_aportes)
    valor_arredondado = round(valor_teorico, 2)

    lista_aportes = [valor_arredondado] * int(n_aportes)
    soma_parcial = round(sum(lista_aportes[:-1]), 2)
    lista_aportes[-1] = round(float(capital_inicial_estrategia) - soma_parcial, 2)

    return valor_teorico, lista_aportes

def seed_esta_vazia(valor):
    if valor is None:
        return True

    texto = str(valor).strip().lower()
    return texto in {"", "none", "null", "nan", "<na>"}

def resolver_seed(seed_fixo, seed_aleatorio):
    if not seed_esta_vazia(seed_fixo):
        return int(seed_fixo), "fixo"

    return int(seed_aleatorio), "aleatorio"

def seed_base_carteira_aleatoria(seed_sorteio_acoes, n_ativos_alvo, carteira_id):
    if int(n_ativos_alvo) == 20:
        return int(seed_sorteio_acoes + 20 + int(carteira_id))
    if int(n_ativos_alvo) == 30:
        return int(seed_sorteio_acoes + 30 + int(carteira_id))
    raise ValueError("n_ativos_alvo inválido para carteira aleatória mensal.")

def construir_df_estrategias_mensais_controle(benchmark_renda_variavel, seed_sorteio_acoes):
    registros = [
        {
            "estrategia_id": "ibovespa_buy_and_hold",
            "grupo_controle": "benchmark",
            "familia_estrategia": "ibovespa_buy_and_hold",
            "tipo_estrategia": "ibovespa_buy_and_hold",
            "carteira_id": pd.NA,
            "replica_id": pd.NA,
            "n_ativos_alvo": pd.NA,
            "benchmark_renda_variavel": benchmark_renda_variavel,
            "origem_universo_compra": "benchmark_indice",
            "regra_selecao_ativos": "buy_and_hold_sem_aportes",
            "flag_usa_universo_elegivel": False,
            "flag_executar_calendario_mensal": False,
            "seed_base_carteira": pd.NA,
            "observacao_metodologica": "benchmark_sem_aportes_previsto_para_execucao_na_etapa_9_5",
        },
        {
            "estrategia_id": "ibovespa_aportes_mensais",
            "grupo_controle": "mensal",
            "familia_estrategia": "ibovespa_mensal",
            "tipo_estrategia": "ibovespa_aportes_mensais",
            "carteira_id": 1,
            "replica_id": REPLICA_ID_PADRAO_MENSAL,
            "n_ativos_alvo": 1,
            "benchmark_renda_variavel": benchmark_renda_variavel,
            "origem_universo_compra": "benchmark_indice",
            "regra_selecao_ativos": "aplicar_aporte_mensal_no_ibovespa",
            "flag_usa_universo_elegivel": False,
            "flag_executar_calendario_mensal": True,
            "seed_base_carteira": pd.NA,
            "observacao_metodologica": "estrategia_mensal_de_controle_via_indice_benchmark",
        },
        {
            "estrategia_id": "acoes_elegiveis_aportes_mensais",
            "grupo_controle": "mensal",
            "familia_estrategia": "todas_elegiveis_mensal",
            "tipo_estrategia": "acoes_elegiveis_aportes_mensais",
            "carteira_id": 1,
            "replica_id": REPLICA_ID_PADRAO_MENSAL,
            "n_ativos_alvo": pd.NA,
            "benchmark_renda_variavel": pd.NA,
            "origem_universo_compra": "tickers_aptos_janela_contabil",
            "regra_selecao_ativos": "comprar_todas_as_acoes_elegiveis_na_data_do_aporte",
            "flag_usa_universo_elegivel": True,
            "flag_executar_calendario_mensal": True,
            "seed_base_carteira": pd.NA,
            "observacao_metodologica": "controle_mensal_com_toda_a_cesta_elegivel_da_data",
        },
    ]

    for carteira_id in range(1, N_CARTEIRAS_ALEATORIAS_20 + 1):
        registros.append(
            {
                "estrategia_id": f"acoes_aleatorias_20_aportes_mensais_carteira_{carteira_id}",
                "grupo_controle": "mensal",
                "familia_estrategia": "aleatoria_20_mensal",
                "tipo_estrategia": f"acoes_aleatorias_20_aportes_mensais_carteira_{carteira_id}",
                "carteira_id": carteira_id,
                "replica_id": carteira_id,
                "n_ativos_alvo": TAMANHO_CARTEIRA_ALEATORIA_20,
                "benchmark_renda_variavel": pd.NA,
                "origem_universo_compra": "tickers_aptos_janela_contabil",
                "regra_selecao_ativos": "sortear_20_acoes_elegiveis_por_aporte",
                "flag_usa_universo_elegivel": True,
                "flag_executar_calendario_mensal": True,
                "seed_base_carteira": seed_base_carteira_aleatoria(seed_sorteio_acoes, 20, carteira_id),
                "observacao_metodologica": "controle_mensal_com_carteira_aleatoria_de_20_elegiveis",
            }
        )

    for carteira_id in range(1, N_CARTEIRAS_ALEATORIAS_30 + 1):
        registros.append(
            {
                "estrategia_id": f"acoes_aleatorias_30_aportes_mensais_carteira_{carteira_id}",
                "grupo_controle": "mensal",
                "familia_estrategia": "aleatoria_30_mensal",
                "tipo_estrategia": f"acoes_aleatorias_30_aportes_mensais_carteira_{carteira_id}",
                "carteira_id": carteira_id,
                "replica_id": carteira_id,
                "n_ativos_alvo": TAMANHO_CARTEIRA_ALEATORIA_30,
                "benchmark_renda_variavel": pd.NA,
                "origem_universo_compra": "tickers_aptos_janela_contabil",
                "regra_selecao_ativos": "sortear_30_acoes_elegiveis_por_aporte",
                "flag_usa_universo_elegivel": True,
                "flag_executar_calendario_mensal": True,
                "seed_base_carteira": seed_base_carteira_aleatoria(seed_sorteio_acoes, 30, carteira_id),
                "observacao_metodologica": "controle_mensal_com_carteira_aleatoria_de_30_elegiveis",
            }
        )

    df_estrategias = pd.DataFrame(registros)

    df_estrategias["estrategia_referencia"] = df_estrategias["estrategia_id"].astype("string")

    df_estrategias.loc[
        df_estrategias["estrategia_id"].eq("ibovespa_buy_and_hold"),
        "estrategia_referencia",
    ] = "ibovespa_buy_and_hold"

    df_estrategias.loc[
        df_estrategias["estrategia_id"].eq("ibovespa_aportes_mensais"),
        "estrategia_referencia",
    ] = "ibovespa_aportes_mensais"

    df_estrategias.loc[
        df_estrategias["estrategia_id"].eq("acoes_elegiveis_aportes_mensais"),
        "estrategia_referencia",
    ] = "acoes_elegiveis_aportes_mensais"

    df_estrategias.loc[
        df_estrategias["familia_estrategia"].eq("aleatoria_20_mensal"),
        "estrategia_referencia",
    ] = "acoes_aleatorias_20_aportes_mensais"

    df_estrategias.loc[
        df_estrategias["familia_estrategia"].eq("aleatoria_30_mensal"),
        "estrategia_referencia",
    ] = "acoes_aleatorias_30_aportes_mensais"

    return df_estrategias

print(f"Número de carteiras aleatórias de 20 elegíveis : {N_CARTEIRAS_ALEATORIAS_20}")
print(f"Número de carteiras aleatórias de 30 elegíveis : {N_CARTEIRAS_ALEATORIAS_30}")
print(f"Tamanho alvo das carteiras aleatórias          : {TAMANHO_CARTEIRA_ALEATORIA_20} e {TAMANHO_CARTEIRA_ALEATORIA_30} ações")
print(f"Seed fixo do sorteio de ações                  : {SEED_FIXO_SORTEIO_ACOES}")
print(f"Seed aleatório do sorteio de ações             : {SEED_ALEATORIO_SORTEIO_ACOES}")
print("OK")

# ============================================================
# 5) Preparação das bases e construção da grade mensal
# ============================================================

print("\n[5/10] Preparação das bases e construção da grade mensal...")

validar_colunas_obrigatorias(df_parametros_globais, COLUNAS_OBRIGATORIAS_PARAMETROS)

df_parametros_globais = padronizar_texto(
    df=df_parametros_globais.copy(),
    colunas_texto=["parametro", "valor"],
    lower_cols=["parametro"],
)

capital_inicial_estrategia = float(
    extrair_parametro_numerico_flexivel(
        df_parametros=df_parametros_globais,
        candidatos=["capital_inicial_estrategia", "capital_inicial_por_estrategia", "capital_inicial"],
        valor_default=100000.00,
    )
)

seed_sorteio_acoes, modo_seed_sorteio_acoes = resolver_seed(
    seed_fixo=SEED_FIXO_SORTEIO_ACOES,
    seed_aleatorio=SEED_ALEATORIO_SORTEIO_ACOES,
)

benchmark_renda_variavel = str(
    extrair_parametro_texto_flexivel(
        df_parametros=df_parametros_globais,
        candidatos=["benchmark_renda_variavel", "benchmark_equity", "benchmark"],
        valor_default="^BVSP",
    )
).strip()

coluna_data_mercado = detectar_coluna_data(
    df=df_mercado_diario_consolidada,
    candidatos=["data", "date", "dt_ref"],
)

coluna_data_elegivel = detectar_coluna_data(
    df=df_tickers_aptos_janela_contabil,
    candidatos=["data", "data_aporte_efetiva", "dt_ref", "data_pregao"],
)

coluna_ticker_elegivel = detectar_coluna_identificador(
    df=df_tickers_aptos_janela_contabil,
    candidatos=["ticker", "codneg", "ticker_ajustado"],
)

coluna_issuer_elegivel = detectar_coluna_identificador(
    df=df_tickers_aptos_janela_contabil,
    candidatos=["issuer_code", "codigo_empresa", "codigo_emissor"],
)

if coluna_ticker_elegivel is None:
    raise ValueError("Não foi possível detectar a coluna de ticker na base de tickers aptos por janela contábil.")

df_mercado_diario_consolidada[coluna_data_mercado] = pd.to_datetime(
    df_mercado_diario_consolidada[coluna_data_mercado],
    errors="coerce",
)

df_tickers_aptos_janela_contabil[coluna_data_elegivel] = pd.to_datetime(
    df_tickers_aptos_janela_contabil[coluna_data_elegivel],
    errors="coerce",
)

df_tickers_aptos_janela_contabil = padronizar_texto(
    df=df_tickers_aptos_janela_contabil.copy(),
    colunas_texto=[coluna_ticker_elegivel] + ([coluna_issuer_elegivel] if coluna_issuer_elegivel is not None else []),
    upper_cols=[coluna_ticker_elegivel],
)

df_elegiveis_por_data = (
    df_tickers_aptos_janela_contabil
    .dropna(subset=[coluna_data_elegivel, coluna_ticker_elegivel])
    .copy()
)

df_elegiveis_por_data["ano_referencia"] = df_elegiveis_por_data[coluna_data_elegivel].dt.year
df_elegiveis_por_data["mes_referencia"] = df_elegiveis_por_data[coluna_data_elegivel].dt.month

agregacoes = {
    "data_aporte_efetiva": (coluna_data_elegivel, "min"),
    "n_tickers_elegiveis_data_aporte": (coluna_ticker_elegivel, "nunique"),
}

if coluna_issuer_elegivel is not None:
    agregacoes["n_empresas_elegiveis_data_aporte"] = (coluna_issuer_elegivel, "nunique")

df_grade_mensal = (
    df_elegiveis_por_data
    .groupby(["ano_referencia", "mes_referencia"], as_index=False)
    .agg(**agregacoes)
    .sort_values(["ano_referencia", "mes_referencia"])
    .reset_index(drop=True)
)

if "n_empresas_elegiveis_data_aporte" not in df_grade_mensal.columns:
    df_grade_mensal["n_empresas_elegiveis_data_aporte"] = pd.NA

df_grade_mensal["ordem_aporte_mensal"] = np.arange(1, len(df_grade_mensal) + 1, dtype=int)
df_grade_mensal["data_sinal"] = pd.to_datetime(
    {
        "year": df_grade_mensal["ano_referencia"].astype(int),
        "month": df_grade_mensal["mes_referencia"].astype(int),
        "day": 1,
    },
    errors="coerce",
)
df_grade_mensal["ano"] = df_grade_mensal["ano_referencia"].astype(int)
df_grade_mensal["mes"] = df_grade_mensal["mes_referencia"].astype(int)
df_grade_mensal["data_referencia_mensal"] = df_grade_mensal["data_sinal"]

df_pregoes_referencia = (
    df_mercado_diario_consolidada[[coluna_data_mercado]]
    .dropna(subset=[coluna_data_mercado])
    .drop_duplicates()
    .sort_values(coluna_data_mercado)
    .rename(columns={coluna_data_mercado: "data"})
    .reset_index(drop=True)
)

df_pregoes_referencia["ordem_pregao_aporte_efetiva"] = np.arange(1, len(df_pregoes_referencia) + 1, dtype=int)

df_grade_mensal = df_grade_mensal.merge(
    df_pregoes_referencia.rename(columns={"data": "data_aporte_efetiva"}),
    on="data_aporte_efetiva",
    how="left",
    validate="one_to_one",
)

df_grade_mensal["flag_data_aporte_valida"] = df_grade_mensal["ordem_pregao_aporte_efetiva"].notna()

data_janela_inicial = pd.Timestamp(df_grade_mensal["data_aporte_efetiva"].min())
data_janela_final = pd.Timestamp(df_mercado_diario_consolidada[coluna_data_mercado].max())

df_estrategias_mensais_controle = construir_df_estrategias_mensais_controle(
    benchmark_renda_variavel=benchmark_renda_variavel,
    seed_sorteio_acoes=seed_sorteio_acoes,
)

df_estrategias_mensais_executaveis = (
    df_estrategias_mensais_controle
    .loc[df_estrategias_mensais_controle["flag_executar_calendario_mensal"] == True]
    .copy()
    .reset_index(drop=True)
)

n_aportes_planejados_estrategia = int(len(df_grade_mensal))
valor_cada_aporte_teorico_exato, lista_aportes_planejados = calcular_aportes_uniformes(
    capital_inicial_estrategia=capital_inicial_estrategia,
    n_aportes=n_aportes_planejados_estrategia,
)

df_grade_mensal["n_aportes_planejados_estrategia"] = int(n_aportes_planejados_estrategia)
df_grade_mensal["valor_cada_aporte_teorico_exato"] = float(valor_cada_aporte_teorico_exato)
df_grade_mensal["valor_aporte_estrategia"] = lista_aportes_planejados
df_grade_mensal["flag_ajuste_residual_ultimo_aporte"] = False
df_grade_mensal.loc[df_grade_mensal.index.max(), "flag_ajuste_residual_ultimo_aporte"] = True
df_grade_mensal["valor_medio_aporte_planejado_tipo"] = float(np.mean(lista_aportes_planejados))
df_grade_mensal["valor_aportado_planejado_acumulado"] = pd.Series(lista_aportes_planejados).cumsum()
df_grade_mensal["pct_capital_planejado_acumulado"] = (
    df_grade_mensal["valor_aportado_planejado_acumulado"] / float(capital_inicial_estrategia)
)

print(f"Capital inicial por estratégia               : {capital_inicial_estrategia:.2f}")
print(f"Seed efetiva do sorteio de ações             : {seed_sorteio_acoes}")
print(f"Modo da seed do sorteio de ações             : {modo_seed_sorteio_acoes}")
print(f"Benchmark de renda variável                  : {benchmark_renda_variavel}")
print(f"Janela mensal comum de controle              : {data_janela_inicial.date()} até {data_janela_final.date()}")
print(f"Quantidade de datas mensais de aporte        : {len(df_grade_mensal):,}")
print("OK")

# ============================================================
# 6) Construção dos calendários mensais de controle
# ============================================================

print("\n[6/10] Construção dos calendários mensais de controle...")

lista_calendarios = []

for linha_estrategia in df_estrategias_mensais_executaveis.itertuples(index=False):
    df_tmp = df_grade_mensal.copy()

    estrategia_id = str(linha_estrategia.estrategia_id)
    grupo_controle = str(linha_estrategia.grupo_controle)
    familia_estrategia = str(linha_estrategia.familia_estrategia)
    tipo_estrategia = str(linha_estrategia.tipo_estrategia)
    carteira_id = linha_estrategia.carteira_id
    replica_id = linha_estrategia.replica_id
    n_ativos_alvo = linha_estrategia.n_ativos_alvo
    origem_universo_compra = str(linha_estrategia.origem_universo_compra)
    regra_selecao_ativos = str(linha_estrategia.regra_selecao_ativos)
    flag_usa_universo_elegivel = bool(linha_estrategia.flag_usa_universo_elegivel)
    seed_base_carteira = linha_estrategia.seed_base_carteira
    observacao_metodologica = str(linha_estrategia.observacao_metodologica)

    df_tmp["estrategia_id"] = estrategia_id
    df_tmp["grupo_controle"] = grupo_controle
    df_tmp["familia_estrategia"] = familia_estrategia
    df_tmp["tipo_estrategia"] = tipo_estrategia
    df_tmp["estrategia_referencia"] = str(linha_estrategia.estrategia_referencia)
    df_tmp["carteira_id"] = carteira_id
    df_tmp["replica_id"] = replica_id
    df_tmp["n_ativos_alvo"] = n_ativos_alvo
    df_tmp["benchmark_renda_variavel"] = linha_estrategia.benchmark_renda_variavel
    df_tmp["origem_universo_compra"] = origem_universo_compra
    df_tmp["regra_selecao_ativos"] = regra_selecao_ativos
    df_tmp["flag_usa_universo_elegivel"] = flag_usa_universo_elegivel
    df_tmp["flag_executar_calendario_mensal"] = True
    df_tmp["seed_base_carteira"] = seed_base_carteira
    df_tmp["seed_replicacao"] = seed_base_carteira
    df_tmp["observacao_metodologica"] = observacao_metodologica
    df_tmp["origem_calendario"] = ORIGEM_CALENDARIO
    df_tmp["capital_inicial_estrategia"] = float(capital_inicial_estrategia)
    df_tmp["flag_evento_aporte"] = True
    df_tmp["ordem_aporte_estrategia"] = df_tmp["ordem_aporte_mensal"].astype(float)
    df_tmp["ordem_sinal_final"] = df_tmp["ordem_aporte_mensal"].astype(float)

    if pd.notna(seed_base_carteira):
        df_tmp["seed_sorteio_cesta_aporte"] = pd.to_numeric(seed_base_carteira, errors="coerce") + df_tmp["ordem_aporte_mensal"].astype(float)
    else:
        df_tmp["seed_sorteio_cesta_aporte"] = pd.NA

    if pd.notna(n_ativos_alvo):
        df_tmp["flag_universo_elegivel_suficiente"] = (
            pd.to_numeric(df_tmp["n_tickers_elegiveis_data_aporte"], errors="coerce") >= float(n_ativos_alvo)
        )
    else:
        df_tmp["flag_universo_elegivel_suficiente"] = True

    df_tmp["request_id"] = (
        df_tmp["estrategia_id"].astype("string")
        + "__"
        + df_tmp["ordem_aporte_mensal"].astype(int).astype(str)
    )

    df_tmp = df_tmp[
        [
            "request_id",
            "estrategia_id",
            "grupo_controle",
            "familia_estrategia",
            "estrategia_referencia",
            "tipo_estrategia",
            "carteira_id",
            "replica_id",
            "seed_replicacao",
            "origem_calendario",
            "origem_universo_compra",
            "regra_selecao_ativos",
            "flag_usa_universo_elegivel",
            "flag_executar_calendario_mensal",
            "benchmark_renda_variavel",
            "observacao_metodologica",
            "data_referencia_mensal",
            "data_sinal",
            "data_aporte_efetiva",
            "ordem_aporte_mensal",
            "ordem_aporte_estrategia",
            "ordem_sinal_final",
            "ordem_pregao_aporte_efetiva",
            "ano_referencia",
            "mes_referencia",
            "ano",
            "mes",
            "n_tickers_elegiveis_data_aporte",
            "n_empresas_elegiveis_data_aporte",
            "n_ativos_alvo",
            "seed_base_carteira",
            "seed_sorteio_cesta_aporte",
            "flag_data_aporte_valida",
            "flag_universo_elegivel_suficiente",
            "n_aportes_planejados_estrategia",
            "capital_inicial_estrategia",
            "valor_cada_aporte_teorico_exato",
            "valor_aporte_estrategia",
            "flag_ajuste_residual_ultimo_aporte",
            "valor_medio_aporte_planejado_tipo",
            "valor_aportado_planejado_acumulado",
            "pct_capital_planejado_acumulado",
            "flag_evento_aporte",
        ]
    ].copy()

    lista_calendarios.append(df_tmp)

df_calendario_aportes_estrategias_mensais_controle = (
    pd.concat(lista_calendarios, axis=0, ignore_index=True)
    .sort_values(["familia_estrategia", "tipo_estrategia", "ordem_aporte_estrategia"])
    .reset_index(drop=True)
)

df_calendario_aportes_estrategias_mensais_controle["flag_estrategia_referencia_preenchida"] = (
    df_calendario_aportes_estrategias_mensais_controle["estrategia_referencia"]
    .astype("string")
    .str.strip()
    .notna()
)

df_inconsistencias_estrategias_mensais_controle = (
    df_calendario_aportes_estrategias_mensais_controle
    .loc[
        (
            ~df_calendario_aportes_estrategias_mensais_controle["flag_data_aporte_valida"]
        )
        | (
            ~df_calendario_aportes_estrategias_mensais_controle["flag_universo_elegivel_suficiente"]
        )
        | (
            ~df_calendario_aportes_estrategias_mensais_controle["flag_estrategia_referencia_preenchida"]
        )
    ]
    .copy()
)

n_estrategia_referencia_ausente = int(
    (~df_calendario_aportes_estrategias_mensais_controle["flag_estrategia_referencia_preenchida"]).sum()
)

print(f"Estratégias mensais executáveis              : {len(df_estrategias_mensais_executaveis):,}")
print(f"Linhas totais dos calendários mensais        : {len(df_calendario_aportes_estrategias_mensais_controle):,}")
print(f"Estrategia_referencia ausente                : {n_estrategia_referencia_ausente:,}")
print(f"Inconsistências identificadas                : {len(df_inconsistencias_estrategias_mensais_controle):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas-resumo e auditoria
# ============================================================

print("\n[7/10] Construção das tabelas-resumo e auditoria...")

df_resumo_estrategias_mensais_controle = (
    df_calendario_aportes_estrategias_mensais_controle
    .groupby(["familia_estrategia", "estrategia_referencia", "tipo_estrategia", "grupo_controle", "carteira_id", "n_ativos_alvo"], as_index=False, dropna=False)
    .agg(
        n_aportes_planejados_estrategia=("request_id", "count"),
        data_primeiro_aporte=("data_aporte_efetiva", "min"),
        data_ultimo_aporte=("data_aporte_efetiva", "max"),
        valor_total_aportado=("valor_aporte_estrategia", "sum"),
        capital_inicial_estrategia=("capital_inicial_estrategia", "first"),
        valor_cada_aporte_teorico_exato=("valor_cada_aporte_teorico_exato", "first"),
        valor_medio_aporte_planejado=("valor_aporte_estrategia", "mean"),
        n_aportes_com_ajuste_residual=("flag_ajuste_residual_ultimo_aporte", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_aportes_com_universo_suficiente=("flag_universo_elegivel_suficiente", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_anos_com_aporte=("ano", "nunique"),
    )
    .sort_values(["familia_estrategia", "estrategia_referencia", "tipo_estrategia"])
    .reset_index(drop=True)
)

df_resumo_estrategias_mensais_controle["diferenca_total_vs_capital"] = (
    df_resumo_estrategias_mensais_controle["valor_total_aportado"]
    - df_resumo_estrategias_mensais_controle["capital_inicial_estrategia"]
)

df_parametros_estrategias_mensais_controle = pd.DataFrame(
    [
        {
            "parametro": "n_carteiras_aleatorias_20",
            "valor_numerico": float(N_CARTEIRAS_ALEATORIAS_20),
            "valor_texto": pd.NA,
            "unidade": "carteiras",
        },
        {
            "parametro": "n_carteiras_aleatorias_30",
            "valor_numerico": float(N_CARTEIRAS_ALEATORIAS_30),
            "valor_texto": pd.NA,
            "unidade": "carteiras",
        },
        {
            "parametro": "tamanho_carteira_aleatoria_20",
            "valor_numerico": float(TAMANHO_CARTEIRA_ALEATORIA_20),
            "valor_texto": pd.NA,
            "unidade": "ativos",
        },
        {
            "parametro": "tamanho_carteira_aleatoria_30",
            "valor_numerico": float(TAMANHO_CARTEIRA_ALEATORIA_30),
            "valor_texto": pd.NA,
            "unidade": "ativos",
        },
        {
            "parametro": "seed_fixo_sorteio_acoes",
            "valor_numerico": float(SEED_FIXO_SORTEIO_ACOES) if not seed_esta_vazia(SEED_FIXO_SORTEIO_ACOES) else pd.NA,
            "valor_texto": pd.NA,
            "unidade": "seed",
        },
        {
            "parametro": "seed_aleatorio_sorteio_acoes",
            "valor_numerico": float(SEED_ALEATORIO_SORTEIO_ACOES) if not seed_esta_vazia(SEED_ALEATORIO_SORTEIO_ACOES) else pd.NA,
            "valor_texto": pd.NA,
            "unidade": "seed",
        },
        {
            "parametro": "seed_sorteio_acoes",
            "valor_numerico": float(seed_sorteio_acoes),
            "valor_texto": pd.NA,
            "unidade": "seed",
        },
        {
            "parametro": "modo_seed_sorteio_acoes",
            "valor_numerico": pd.NA,
            "valor_texto": modo_seed_sorteio_acoes,
            "unidade": "modo_seed",
        },
        {
            "parametro": "capital_inicial_estrategia",
            "valor_numerico": float(capital_inicial_estrategia),
            "valor_texto": pd.NA,
            "unidade": "brl",
        },
        {
            "parametro": "benchmark_renda_variavel",
            "valor_numerico": pd.NA,
            "valor_texto": benchmark_renda_variavel,
            "unidade": "ticker",
        },
        {
            "parametro": "regra_grade_mensal",
            "valor_numerico": pd.NA,
            "valor_texto": "primeiro_pregao_disponivel_em_cada_mes_com_universo_elegivel_valido",
            "unidade": "regra_operacional",
        },
    ]
)

df_auditoria_estrategias_mensais_controle = pd.DataFrame(
    [
        {
            "metrica": "n_estrategias_formalmente_definidas",
            "valor": float(len(df_estrategias_mensais_controle)),
        },
        {
            "metrica": "n_estrategias_mensais_executaveis",
            "valor": float(len(df_estrategias_mensais_executaveis)),
        },
        {
            "metrica": "n_linhas_total_calendario_mensal",
            "valor": float(len(df_calendario_aportes_estrategias_mensais_controle)),
        },
        {
            "metrica": "n_inconsistencias_calendario_mensal",
            "valor": float(len(df_inconsistencias_estrategias_mensais_controle)),
        },
        {
            "metrica": "n_aportes_grade_mensal",
            "valor": float(len(df_grade_mensal)),
        },
        {
            "metrica": "n_datas_aporte_invalidas",
            "valor": float((~df_calendario_aportes_estrategias_mensais_controle["flag_data_aporte_valida"]).sum()),
        },
        {
            "metrica": "n_aportes_sem_universo_suficiente",
            "valor": float((~df_calendario_aportes_estrategias_mensais_controle["flag_universo_elegivel_suficiente"]).sum()),
        },
        {
            "metrica": "n_estrategia_referencia_ausente",
            "valor": float(n_estrategia_referencia_ausente),
        },
        {
            "metrica": "n_estrategia_referencia_distintas",
            "valor": float(df_calendario_aportes_estrategias_mensais_controle["estrategia_referencia"].nunique(dropna=True)),
        },
        {
            "metrica": "seed_sorteio_acoes",
            "valor": float(seed_sorteio_acoes),
        },
    ]
)

df_distribuicao_anual_estrategias_mensais_controle = (
    df_calendario_aportes_estrategias_mensais_controle
    .groupby(["estrategia_referencia", "tipo_estrategia", "ano"], as_index=False)
    .agg(
        n_aportes=("request_id", "count"),
        valor_total_aportado=("valor_aporte_estrategia", "sum"),
        valor_medio_aporte=("valor_aporte_estrategia", "mean"),
        n_tickers_elegiveis_mediano=("n_tickers_elegiveis_data_aporte", "median"),
    )
    .sort_values(["estrategia_referencia", "tipo_estrategia", "ano"])
    .reset_index(drop=True)
)

print("Tabelas-resumo e auditoria construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_calendario_aportes_estrategias_mensais_controle, caminho_base_calendario_aportes_estrategias_mensais_controle, index=False)
salvar_dataframe(df_estrategias_mensais_controle, caminho_tbl_estrategias_mensais_controle, index=False)
salvar_dataframe(df_resumo_estrategias_mensais_controle, caminho_tbl_resumo_estrategias_mensais_controle, index=False)
salvar_dataframe(df_parametros_estrategias_mensais_controle, caminho_tbl_parametros_estrategias_mensais_controle, index=False)
salvar_dataframe(df_auditoria_estrategias_mensais_controle, caminho_tbl_auditoria_estrategias_mensais_controle, index=False)
salvar_dataframe(df_inconsistencias_estrategias_mensais_controle, caminho_tbl_inconsistencias_estrategias_mensais_controle, index=False)
salvar_dataframe(df_distribuicao_anual_estrategias_mensais_controle, caminho_tbl_distribuicao_anual_estrategias_mensais_controle, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_estrategias_mensais_controle = df_estrategias_mensais_controle.copy()
df_amostra_resumo_estrategias_mensais_controle = df_resumo_estrategias_mensais_controle.copy()
df_amostra_parametros_estrategias_mensais_controle = df_parametros_estrategias_mensais_controle.copy()
df_amostra_auditoria_estrategias_mensais_controle = df_auditoria_estrategias_mensais_controle.copy()
df_amostra_calendario_aportes_estrategias_mensais_controle = df_calendario_aportes_estrategias_mensais_controle.head(20).copy()
df_amostra_inconsistencias_estrategias_mensais_controle = df_inconsistencias_estrategias_mensais_controle.head(20).copy()
df_amostra_distribuicao_anual_estrategias_mensais_controle = df_distribuicao_anual_estrategias_mensais_controle.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nDefinição formal das estratégias mensais de controle:")
print(df_amostra_estrategias_mensais_controle.to_string(index=False))

print("\nResumo dos calendários mensais de controle:")
print(df_amostra_resumo_estrategias_mensais_controle.to_string(index=False))

print("\nParâmetros das estratégias mensais de controle:")
print(df_amostra_parametros_estrategias_mensais_controle.to_string(index=False))

print("\nAuditoria das estratégias mensais de controle:")
print(df_amostra_auditoria_estrategias_mensais_controle.to_string(index=False))

print("\nBase consolidada dos calendários mensais de controle - amostra:")
print(df_amostra_calendario_aportes_estrategias_mensais_controle.to_string(index=False))

print("\nInconsistências dos calendários mensais de controle - amostra:")
print(df_amostra_inconsistencias_estrategias_mensais_controle.to_string(index=False))

print("\nDistribuição anual dos calendários mensais de controle - amostra:")
print(df_amostra_distribuicao_anual_estrategias_mensais_controle.to_string(index=False))

print("\nArquivos salvos na subetapa 7.6:")
print(f"- {caminho_base_calendario_aportes_estrategias_mensais_controle}")
print(f"- {caminho_tbl_estrategias_mensais_controle}")
print(f"- {caminho_tbl_resumo_estrategias_mensais_controle}")
print(f"- {caminho_tbl_parametros_estrategias_mensais_controle}")
print(f"- {caminho_tbl_auditoria_estrategias_mensais_controle}")
print(f"- {caminho_tbl_inconsistencias_estrategias_mensais_controle}")
print(f"- {caminho_tbl_distribuicao_anual_estrategias_mensais_controle}")

print("\nETAPA 7.6 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 7.6 - CALENDÁRIO DE APORTES DAS ESTRATÉGIAS MENSAIS DE CONTROLE

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - parâmetros globais                           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_1_tbl_parametros_globais.parquet
Entrada - mercado diário consolidada                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - tickers aptos por janela contábil           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_5_base_tickers_aptos_janela_contabil.parquet
Saída   - base calendários mensais de controle        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andament

# Etapa 8) Construção das Regras de Carteira

## Etapa 8.1) Regra de Seleção dos Ativos por Aporte

In [35]:
%%time
# ============================================================
# Etapa 8.1) Regra de Seleção dos Ativos por Aporte
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 8.1 - REGRA DE SELEÇÃO DOS ATIVOS POR APORTE")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_tbl_parametros_globais = gerar_caminho_arquivo(
    etapa=1,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="parametros_globais",
)

caminho_base_ibov_padronizada = gerar_caminho_arquivo(
    etapa=1,
    subetapa=3,
    tipo_arquivo="base",
    nome="ibov_padronizada",
)

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_tickers_aptos_janela_contabil = gerar_caminho_arquivo(
    etapa=4,
    subetapa=5,
    tipo_arquivo="base",
    nome="tickers_aptos_janela_contabil",
)

caminho_base_calendario_aportes_capitulacao = gerar_caminho_arquivo(
    etapa=7,
    subetapa=3,
    tipo_arquivo="base",
    nome="calendario_aportes_capitulacao",
)

caminho_base_calendario_aportes_euforia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=4,
    tipo_arquivo="base",
    nome="calendario_aportes_euforia",
)

caminho_base_calendarios_aleatorios_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=5,
    tipo_arquivo="base",
    nome="calendarios_aleatorios_controle",
)

caminho_base_calendario_aportes_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="base",
    nome="calendario_aportes_estrategias_mensais_controle",
)

caminho_tbl_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="estrategias_mensais_controle",
)

caminho_tbl_regras_selecao_ativos_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="regras_selecao_ativos_aporte",
)

caminho_base_cesta_ativos_por_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=1,
    tipo_arquivo="base",
    nome="cesta_ativos_por_aporte",
)

caminho_tbl_auditoria_cesta_ativos_por_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="auditoria_cesta_ativos_por_aporte",
)

caminho_tbl_resumo_cesta_ativos_por_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="resumo_cesta_ativos_por_aporte",
)

caminho_tbl_inconsistencias_cesta_ativos_por_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="inconsistencias_cesta_ativos_por_aporte",
)

caminho_tbl_distribuicao_anual_cesta_ativos_por_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_cesta_ativos_por_aporte",
)

print(f"Entrada - parâmetros globais                      : {caminho_tbl_parametros_globais}")
print(f"Entrada - ibov padronizada                        : {caminho_base_ibov_padronizada}")
print(f"Entrada - mercado diário consolidada             : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - tickers aptos por janela contábil      : {caminho_base_tickers_aptos_janela_contabil}")
print(f"Entrada - calendário aportes capitulação         : {caminho_base_calendario_aportes_capitulacao}")
print(f"Entrada - calendário aportes euforia             : {caminho_base_calendario_aportes_euforia}")
print(f"Entrada - calendários aleatórios de controle     : {caminho_base_calendarios_aleatorios_controle}")
print(f"Entrada - calendários mensais de controle        : {caminho_base_calendario_aportes_estrategias_mensais_controle}")
print(f"Entrada - definição estratégias mensais          : {caminho_tbl_estrategias_mensais_controle}")
print(f"Saída   - regras seleção ativos por aporte       : {caminho_tbl_regras_selecao_ativos_aporte}")
print(f"Saída   - base cesta ativos por aporte           : {caminho_base_cesta_ativos_por_aporte}")
print(f"Saída   - auditoria cesta por aporte             : {caminho_tbl_auditoria_cesta_ativos_por_aporte}")
print(f"Saída   - resumo da cesta por estratégia         : {caminho_tbl_resumo_cesta_ativos_por_aporte}")
print(f"Saída   - inconsistências da cesta por aporte    : {caminho_tbl_inconsistencias_cesta_ativos_por_aporte}")
print(f"Saída   - distribuição anual da cesta            : {caminho_tbl_distribuicao_anual_cesta_ativos_por_aporte}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_parametros_globais = pd.read_parquet(caminho_tbl_parametros_globais)
df_ibov_padronizada = pd.read_parquet(caminho_base_ibov_padronizada)
df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_tickers_aptos_janela_contabil = pd.read_parquet(caminho_base_tickers_aptos_janela_contabil)
df_calendario_aportes_capitulacao = pd.read_parquet(caminho_base_calendario_aportes_capitulacao)
df_calendario_aportes_euforia = pd.read_parquet(caminho_base_calendario_aportes_euforia)
df_calendarios_aleatorios_controle = pd.read_parquet(caminho_base_calendarios_aleatorios_controle)
df_calendario_aportes_estrategias_mensais_controle = pd.read_parquet(caminho_base_calendario_aportes_estrategias_mensais_controle)
df_estrategias_mensais_controle = pd.read_parquet(caminho_tbl_estrategias_mensais_controle)

print(f"Parâmetros globais                        : {df_parametros_globais.shape[0]:,} linhas x {df_parametros_globais.shape[1]} colunas")
print(f"Ibov padronizada                          : {df_ibov_padronizada.shape[0]:,} linhas x {df_ibov_padronizada.shape[1]} colunas")
print(f"Mercado diário consolidada               : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Tickers aptos por janela contábil        : {df_tickers_aptos_janela_contabil.shape[0]:,} linhas x {df_tickers_aptos_janela_contabil.shape[1]} colunas")
print(f"Calendário aportes capitulação           : {df_calendario_aportes_capitulacao.shape[0]:,} linhas x {df_calendario_aportes_capitulacao.shape[1]} colunas")
print(f"Calendário aportes euforia               : {df_calendario_aportes_euforia.shape[0]:,} linhas x {df_calendario_aportes_euforia.shape[1]} colunas")
print(f"Calendários aleatórios de controle       : {df_calendarios_aleatorios_controle.shape[0]:,} linhas x {df_calendarios_aleatorios_controle.shape[1]} colunas")
print(f"Calendários mensais de controle          : {df_calendario_aportes_estrategias_mensais_controle.shape[0]:,} linhas x {df_calendario_aportes_estrategias_mensais_controle.shape[1]} colunas")
print(f"Estratégias mensais de controle          : {df_estrategias_mensais_controle.shape[0]:,} linhas x {df_estrategias_mensais_controle.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e regras operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e regras operacionais...")

COLUNAS_MINIMAS_CALENDARIO = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "origem_calendario",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "valor_aporte_estrategia",
    "capital_inicial_estrategia",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def detectar_coluna(df, candidatos, obrigatoria=True):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna

    if obrigatoria:
        raise ValueError(f"Não foi possível localizar nenhuma das colunas candidatas: {candidatos}")

    return None

def padronizar_colunas_texto(df, colunas_upper=None, colunas_lower=None):
    colunas_upper = [] if colunas_upper is None else colunas_upper
    colunas_lower = [] if colunas_lower is None else colunas_lower

    for coluna in colunas_upper:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.upper()

    for coluna in colunas_lower:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.lower()

    return df

def extrair_parametro_texto(df_parametros, nome_parametro, valor_default=None):
    df_aux = df_parametros.copy()
    df_aux["parametro"] = df_aux["parametro"].astype("string").str.strip().str.lower()

    linha = df_aux.loc[df_aux["parametro"] == str(nome_parametro).strip().lower()]
    if linha.empty:
        if valor_default is None:
            raise ValueError(f"Parâmetro obrigatório não encontrado: {nome_parametro}")
        return str(valor_default)

    valor = str(linha["valor"].iloc[0]).strip()
    if valor == "" and valor_default is not None:
        return str(valor_default)

    return valor

def construir_rng(seed):
    if pd.isna(seed):
        raise ValueError("A seed de sorteio da cesta não pode ser nula para as estratégias aleatórias mensais.")
    return np.random.default_rng(int(seed))

def construir_base_benchmark_ibov(df_ibov, benchmark_renda_variavel):
    coluna_data = detectar_coluna(df_ibov, ["data", "date", "dt_ref"])
    coluna_preco = detectar_coluna(df_ibov, ["close_adj", "close", "adj_close", "preco_fechamento"])

    df_out = df_ibov.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_preco] = pd.to_numeric(df_out[coluna_preco], errors="coerce")

    if "ticker" not in df_out.columns:
        df_out["ticker"] = benchmark_renda_variavel

    df_out = padronizar_colunas_texto(df_out, colunas_upper=["ticker"])
    df_out = (
        df_out
        .rename(columns={coluna_data: "data", coluna_preco: "preco_benchmark"})
        .dropna(subset=["data", "preco_benchmark"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    return df_out[["data", "ticker", "preco_benchmark"]].copy()

def preparar_base_elegivel_com_preco(df_tickers_aptos, df_mercado):
    coluna_data_elegivel = detectar_coluna(df_tickers_aptos, ["data_aporte_efetiva", "data", "dt_ref", "data_pregao"])
    coluna_ticker_elegivel = detectar_coluna(df_tickers_aptos, ["ticker", "codneg", "ticker_ajustado"])
    coluna_issuer_elegivel = detectar_coluna(df_tickers_aptos, ["issuer_code", "codigo_empresa", "codigo_emissor"], obrigatoria=False)

    coluna_data_mercado = detectar_coluna(df_mercado, ["data", "date", "dt_ref"])
    coluna_ticker_mercado = detectar_coluna(df_mercado, ["ticker", "codneg", "ticker_ajustado"])
    coluna_preco_mercado = detectar_coluna(df_mercado, ["close_adj", "close", "adj_close"])

    colunas_meta_mercado = [col for col in ["nome", "setor", "subsetor", "segmento"] if col in df_mercado.columns]
    colunas_meta_elegivel = [col for col in ["nome", "setor", "subsetor", "segmento"] if col in df_tickers_aptos.columns and col not in colunas_meta_mercado]

    df_elegivel = df_tickers_aptos.copy()
    df_elegivel[coluna_data_elegivel] = pd.to_datetime(df_elegivel[coluna_data_elegivel], errors="coerce")
    df_elegivel = padronizar_colunas_texto(df_elegivel, colunas_upper=[coluna_ticker_elegivel])

    df_merc = df_mercado.copy()
    df_merc[coluna_data_mercado] = pd.to_datetime(df_merc[coluna_data_mercado], errors="coerce")
    df_merc[coluna_preco_mercado] = pd.to_numeric(df_merc[coluna_preco_mercado], errors="coerce")
    df_merc = padronizar_colunas_texto(df_merc, colunas_upper=[coluna_ticker_mercado])

    colunas_mercado_merge = [coluna_data_mercado, coluna_ticker_mercado, coluna_preco_mercado] + colunas_meta_mercado
    if coluna_issuer_elegivel is not None and coluna_issuer_elegivel in df_merc.columns:
        colunas_mercado_merge.append(coluna_issuer_elegivel)

    df_merc = (
        df_merc[colunas_mercado_merge]
        .dropna(subset=[coluna_data_mercado, coluna_ticker_mercado, coluna_preco_mercado])
        .rename(
            columns={
                coluna_data_mercado: "data_aporte_efetiva",
                coluna_ticker_mercado: "ticker",
                coluna_preco_mercado: "preco_compra_referencia",
            }
        )
    )

    df_elegivel = df_elegivel.rename(
        columns={
            coluna_data_elegivel: "data_aporte_efetiva",
            coluna_ticker_elegivel: "ticker",
        }
    )

    df_out = df_elegivel.merge(
        df_merc,
        on=["data_aporte_efetiva", "ticker"],
        how="inner",
        suffixes=("", "_mercado"),
    )

    if coluna_issuer_elegivel is None:
        coluna_issuer_elegivel = "issuer_code"
        if coluna_issuer_elegivel not in df_out.columns:
            df_out[coluna_issuer_elegivel] = pd.NA

    for coluna in colunas_meta_elegivel:
        if coluna in df_out.columns and f"{coluna}_mercado" not in df_out.columns:
            continue

    for coluna in ["nome", "setor", "subsetor", "segmento"]:
        coluna_merc = f"{coluna}_mercado"
        if coluna in df_out.columns and coluna_merc in df_out.columns:
            df_out[coluna] = df_out[coluna].fillna(df_out[coluna_merc])
        elif coluna not in df_out.columns and coluna_merc in df_out.columns:
            df_out[coluna] = df_out[coluna_merc]

    df_out = padronizar_colunas_texto(df_out, colunas_upper=["ticker"])

    colunas_finais = ["data_aporte_efetiva", "ticker", coluna_issuer_elegivel, "preco_compra_referencia", "nome", "setor", "subsetor", "segmento"]
    colunas_finais = [col for col in colunas_finais if col in df_out.columns]

    df_out = (
        df_out[colunas_finais]
        .rename(columns={coluna_issuer_elegivel: "issuer_code"})
        .drop_duplicates(subset=["data_aporte_efetiva", "ticker"])
        .sort_values(["data_aporte_efetiva", "ticker"])
        .reset_index(drop=True)
    )

    return df_out

def normalizar_calendario(df_calendario, grupo_selecao_ativos, regra_selecao_cesta, versao_cesta, n_ativos_alvo_padrao=None):
    validar_colunas_obrigatorias(df_calendario, COLUNAS_MINIMAS_CALENDARIO)

    df_out = df_calendario.copy()
    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")

    if "benchmark_renda_variavel" not in df_out.columns:
        df_out["benchmark_renda_variavel"] = pd.NA

    if "estrategia_referencia" not in df_out.columns:
        df_out["estrategia_referencia"] = pd.NA

    if "seed_sorteio_cesta_aporte" not in df_out.columns:
        df_out["seed_sorteio_cesta_aporte"] = pd.NA

    if "replica_id" not in df_out.columns:
        df_out["replica_id"] = pd.NA

    if "carteira_id" not in df_out.columns:
        df_out["carteira_id"] = pd.NA

    if "n_ativos_alvo" not in df_out.columns:
        df_out["n_ativos_alvo"] = n_ativos_alvo_padrao
    else:
        df_out["n_ativos_alvo"] = pd.to_numeric(df_out["n_ativos_alvo"], errors="coerce")
        if n_ativos_alvo_padrao is not None:
            df_out["n_ativos_alvo"] = df_out["n_ativos_alvo"].fillna(n_ativos_alvo_padrao)

    df_out["grupo_selecao_ativos"] = grupo_selecao_ativos
    df_out["regra_selecao_cesta"] = regra_selecao_cesta
    df_out["versao_cesta"] = versao_cesta

    colunas_saida = [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "origem_calendario",
        "grupo_selecao_ativos",
        "benchmark_renda_variavel",
        "regra_selecao_cesta",
        "versao_cesta",
        "carteira_id",
        "replica_id",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_aporte_estrategia",
        "n_ativos_alvo",
        "seed_sorteio_cesta_aporte",
        "capital_inicial_estrategia",
        "valor_aporte_estrategia",
    ]

    return df_out[colunas_saida].copy()

def selecionar_cesta_para_solicitacao(linha, df_elegivel_preco, df_ibov_benchmark):
    request_id = linha.request_id
    estrategia_id = linha.estrategia_id
    grupo_controle = linha.grupo_controle
    familia_estrategia = linha.familia_estrategia
    estrategia_referencia = linha.estrategia_referencia
    tipo_estrategia = linha.tipo_estrategia
    origem_calendario = linha.origem_calendario
    grupo_selecao_ativos = linha.grupo_selecao_ativos
    benchmark_renda_variavel = linha.benchmark_renda_variavel
    regra_selecao_cesta = linha.regra_selecao_cesta
    versao_cesta = linha.versao_cesta
    carteira_id = linha.carteira_id
    replica_id = linha.replica_id
    data_sinal = pd.Timestamp(linha.data_sinal)
    data_aporte_efetiva = pd.Timestamp(linha.data_aporte_efetiva)
    ordem_aporte_estrategia = float(linha.ordem_aporte_estrategia)
    n_ativos_alvo = linha.n_ativos_alvo
    seed_sorteio_cesta_aporte = linha.seed_sorteio_cesta_aporte
    capital_inicial_estrategia = float(linha.capital_inicial_estrategia)
    valor_aporte_estrategia = float(linha.valor_aporte_estrategia)

    deslocamento_dias_corridos_preco = 0
    flag_preco_deslocado = False
    df_cesta = pd.DataFrame()

    if regra_selecao_cesta == "benchmark_ibovespa":
        df_bench = df_ibov_benchmark.loc[df_ibov_benchmark["data"] >= data_aporte_efetiva].copy().sort_values("data").head(1)

        if not df_bench.empty:
            deslocamento_dias_corridos_preco = int((pd.Timestamp(df_bench["data"].iloc[0]) - data_aporte_efetiva).days)
            flag_preco_deslocado = deslocamento_dias_corridos_preco > 0

            df_cesta = pd.DataFrame(
                [
                    {
                        "ticker": str(df_bench["ticker"].iloc[0]),
                        "issuer_code": str(df_bench["ticker"].iloc[0]),
                        "nome": "IBOVESPA",
                        "setor": "ÍNDICE",
                        "subsetor": "ÍNDICE",
                        "segmento": "ÍNDICE",
                        "preco_compra_referencia": float(df_bench["preco_benchmark"].iloc[0]),
                    }
                ]
            )

    else:
        df_disponiveis = (
            df_elegivel_preco
            .loc[df_elegivel_preco["data_aporte_efetiva"] == data_aporte_efetiva]
            .copy()
            .sort_values(["ticker"])
            .reset_index(drop=True)
        )

        if regra_selecao_cesta == "universo_elegivel_integral":
            df_cesta = df_disponiveis.copy()

        elif regra_selecao_cesta == "aleatoria_universo_elegivel":
            if pd.isna(n_ativos_alvo):
                raise ValueError(f"A estratégia {tipo_estrategia} exige n_ativos_alvo definido.")
            if len(df_disponiveis) > 0:
                rng = construir_rng(seed_sorteio_cesta_aporte)
                n_selecionar = int(min(int(n_ativos_alvo), len(df_disponiveis)))
                idx_sorteados = rng.choice(df_disponiveis.index.to_numpy(), size=n_selecionar, replace=False)
                df_cesta = (
                    df_disponiveis.loc[idx_sorteados]
                    .copy()
                    .sort_values(["ticker"])
                    .reset_index(drop=True)
                )
            else:
                df_cesta = df_disponiveis.copy()

        else:
            raise ValueError(f"Regra de seleção de cesta não reconhecida: {regra_selecao_cesta}")

    n_ativos_disponiveis = int(
        len(
            df_elegivel_preco.loc[df_elegivel_preco["data_aporte_efetiva"] == data_aporte_efetiva]
        )
    ) if regra_selecao_cesta != "benchmark_ibovespa" else int(len(df_cesta))

    n_ativos_selecionados = int(len(df_cesta))
    flag_cesta_vazia = n_ativos_selecionados == 0

    if pd.isna(n_ativos_alvo):
        flag_quantidade_alvo_atendida = not flag_cesta_vazia
    else:
        flag_quantidade_alvo_atendida = int(n_ativos_selecionados) == int(n_ativos_alvo)

    if n_ativos_selecionados > 0:
        orcamento_por_ativo = float(valor_aporte_estrategia) / float(n_ativos_selecionados)
        peso_planejado_ativo = 1.0 / float(n_ativos_selecionados)
    else:
        orcamento_por_ativo = 0.0
        peso_planejado_ativo = 0.0

    if not df_cesta.empty:
        df_cesta["request_id"] = request_id
        df_cesta["estrategia_id"] = estrategia_id
        df_cesta["grupo_controle"] = grupo_controle
        df_cesta["familia_estrategia"] = familia_estrategia
        df_cesta["estrategia_referencia"] = estrategia_referencia
        df_cesta["tipo_estrategia"] = tipo_estrategia
        df_cesta["origem_calendario"] = origem_calendario
        df_cesta["grupo_selecao_ativos"] = grupo_selecao_ativos
        df_cesta["benchmark_renda_variavel"] = benchmark_renda_variavel
        df_cesta["regra_selecao_cesta"] = regra_selecao_cesta
        df_cesta["versao_cesta"] = versao_cesta
        df_cesta["carteira_id"] = carteira_id
        df_cesta["replica_id"] = replica_id
        df_cesta["data_sinal"] = data_sinal
        df_cesta["data_aporte_efetiva"] = data_aporte_efetiva
        df_cesta["ordem_aporte_estrategia"] = ordem_aporte_estrategia
        df_cesta["n_ativos_alvo"] = n_ativos_alvo
        df_cesta["n_ativos_disponiveis_data_aporte"] = n_ativos_disponiveis
        df_cesta["n_ativos_selecionados_aporte"] = n_ativos_selecionados
        df_cesta["flag_quantidade_alvo_atendida"] = flag_quantidade_alvo_atendida
        df_cesta["flag_cesta_vazia"] = flag_cesta_vazia
        df_cesta["capital_inicial_estrategia"] = capital_inicial_estrategia
        df_cesta["valor_aporte_estrategia"] = valor_aporte_estrategia
        df_cesta["valor_orcamento_ativo_planejado"] = orcamento_por_ativo
        df_cesta["peso_ativo_planejado"] = peso_planejado_ativo
        df_cesta["seed_sorteio_cesta_aporte"] = seed_sorteio_cesta_aporte
        df_cesta["flag_preco_deslocado"] = flag_preco_deslocado
        df_cesta["deslocamento_dias_corridos_preco"] = deslocamento_dias_corridos_preco

    auditoria = {
        "request_id": request_id,
        "estrategia_id": estrategia_id,
        "grupo_controle": grupo_controle,
        "familia_estrategia": familia_estrategia,
        "estrategia_referencia": estrategia_referencia,
        "tipo_estrategia": tipo_estrategia,
        "origem_calendario": origem_calendario,
        "grupo_selecao_ativos": grupo_selecao_ativos,
        "benchmark_renda_variavel": benchmark_renda_variavel,
        "regra_selecao_cesta": regra_selecao_cesta,
        "versao_cesta": versao_cesta,
        "carteira_id": carteira_id,
        "replica_id": replica_id,
        "data_sinal": data_sinal,
        "data_aporte_efetiva": data_aporte_efetiva,
        "ordem_aporte_estrategia": ordem_aporte_estrategia,
        "n_ativos_alvo": n_ativos_alvo,
        "n_ativos_disponiveis_data_aporte": n_ativos_disponiveis,
        "n_ativos_selecionados_aporte": n_ativos_selecionados,
        "flag_quantidade_alvo_atendida": flag_quantidade_alvo_atendida,
        "flag_cesta_vazia": flag_cesta_vazia,
        "flag_preco_deslocado": flag_preco_deslocado,
        "deslocamento_dias_corridos_preco": deslocamento_dias_corridos_preco,
        "capital_inicial_estrategia": capital_inicial_estrategia,
        "valor_aporte_estrategia": valor_aporte_estrategia,
        "seed_sorteio_cesta_aporte": seed_sorteio_cesta_aporte,
    }

    return df_cesta, auditoria

df_regras_selecao_ativos_aporte = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "SEL1",
            "escopo": "capitulacao_euforia_e_aleatorias",
            "regra_operacional": "utilizar_universo_elegivel_final_da_data_do_aporte",
            "detalhe": "As estratégias baseadas em sinais e os controles aleatórios herdam a cesta integral de tickers aptos da data do aporte.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "SEL2",
            "escopo": "mensal_todas_elegiveis",
            "regra_operacional": "utilizar_universo_elegivel_final_da_data_do_aporte",
            "detalhe": "A estratégia mensal de todas as elegíveis compra toda a cesta apta da data.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "SEL3",
            "escopo": "mensal_aleatoria_20_30",
            "regra_operacional": "sortear_cesta_dentro_do_universo_elegivel_com_seed_reprodutivel",
            "detalhe": "As carteiras aleatórias de 20 e 30 ativos sorteiam os tickers dentro do universo elegível da data do aporte.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "SEL4",
            "escopo": "ibovespa_mensal",
            "regra_operacional": "comprar_ticker_do_benchmark_renda_variavel",
            "detalhe": "A estratégia mensal do Ibovespa utiliza o benchmark como ativo único da cesta.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "SEL5",
            "escopo": "validacao_preco",
            "regra_operacional": "manter_apenas_ativos_com_preco_disponivel_na_data_do_aporte",
            "detalhe": "A cesta comprável exige presença de preço válido na data efetiva do aporte.",
        },
        {
            "ordem_regra": 6,
            "regra_id": "SEL6",
            "escopo": "benchmark_ibovespa_sem_preco_exato",
            "regra_operacional": "usar_primeira_observacao_do_benchmark_em_data_posterior_mais_proxima",
            "detalhe": "Quando o benchmark não possui linha exata na data planejada do aporte, usa-se a primeira observação posterior disponível para evitar cesta vazia por desalinhamento da fonte.",
        },
    ]
)

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Preparação das bases e padronização dos insumos
# ============================================================

print("\n[5/10] Preparação das bases e padronização dos insumos...")

df_parametros_globais["parametro"] = df_parametros_globais["parametro"].astype("string").str.strip().str.lower()
benchmark_renda_variavel = extrair_parametro_texto(
    df_parametros=df_parametros_globais,
    nome_parametro="benchmark_renda_variavel",
    valor_default="^BVSP",
).strip().upper()

df_ibov_benchmark = construir_base_benchmark_ibov(
    df_ibov=df_ibov_padronizada.copy(),
    benchmark_renda_variavel=benchmark_renda_variavel,
)

df_elegivel_preco = preparar_base_elegivel_com_preco(
    df_tickers_aptos=df_tickers_aptos_janela_contabil.copy(),
    df_mercado=df_mercado_diario_consolidada.copy(),
)

df_calendario_aportes_capitulacao_norm = normalizar_calendario(
    df_calendario=df_calendario_aportes_capitulacao.copy(),
    grupo_selecao_ativos="universo_elegivel_integral",
    regra_selecao_cesta="universo_elegivel_integral",
    versao_cesta="cesta_elegivel_integral",
)

df_calendario_aportes_euforia_norm = normalizar_calendario(
    df_calendario=df_calendario_aportes_euforia.copy(),
    grupo_selecao_ativos="universo_elegivel_integral",
    regra_selecao_cesta="universo_elegivel_integral",
    versao_cesta="cesta_elegivel_integral",
)

df_calendarios_aleatorios_controle_norm = normalizar_calendario(
    df_calendario=df_calendarios_aleatorios_controle.copy(),
    grupo_selecao_ativos="universo_elegivel_integral",
    regra_selecao_cesta="universo_elegivel_integral",
    versao_cesta="cesta_elegivel_integral",
)

df_calendario_aportes_estrategias_mensais_controle_norm = normalizar_calendario(
    df_calendario=df_calendario_aportes_estrategias_mensais_controle.copy(),
    grupo_selecao_ativos="mensal",
    regra_selecao_cesta="mensal",
    versao_cesta="mensal",
)

mask_ibov = df_calendario_aportes_estrategias_mensais_controle_norm["tipo_estrategia"].astype("string") == "ibovespa_aportes_mensais"
mask_todas = df_calendario_aportes_estrategias_mensais_controle_norm["tipo_estrategia"].astype("string") == "acoes_elegiveis_aportes_mensais"
mask_aleatoria = df_calendario_aportes_estrategias_mensais_controle_norm["familia_estrategia"].astype("string").str.contains("aleatoria_", regex=False)

df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_ibov, "grupo_selecao_ativos"] = "benchmark_ibovespa"
df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_ibov, "regra_selecao_cesta"] = "benchmark_ibovespa"
df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_ibov, "versao_cesta"] = "benchmark_indice"
df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_ibov, "benchmark_renda_variavel"] = benchmark_renda_variavel
df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_ibov, "n_ativos_alvo"] = 1

df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_todas, "grupo_selecao_ativos"] = "universo_elegivel_integral"
df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_todas, "regra_selecao_cesta"] = "universo_elegivel_integral"
df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_todas, "versao_cesta"] = "cesta_elegivel_integral"

df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_aleatoria, "grupo_selecao_ativos"] = "amostra_aleatoria_universo_elegivel"
df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_aleatoria, "regra_selecao_cesta"] = "aleatoria_universo_elegivel"
df_calendario_aportes_estrategias_mensais_controle_norm.loc[mask_aleatoria, "versao_cesta"] = "cesta_aleatoria_seed"

print(f"Benchmark renda variável                     : {benchmark_renda_variavel}")
print(f"Linhas da base elegível com preço            : {len(df_elegivel_preco):,}")
print(f"Linhas da base padronizada do Ibovespa       : {len(df_ibov_benchmark):,}")
print("OK")

# ============================================================
# 6) Construção da base unificada de solicitações e seleção das cestas
# ============================================================

print("\n[6/10] Construção da base unificada de solicitações e seleção das cestas...")

df_solicitacoes_aporte_unificadas = (
    pd.concat(
        [
            df_calendario_aportes_capitulacao_norm,
            df_calendario_aportes_euforia_norm,
            df_calendarios_aleatorios_controle_norm,
            df_calendario_aportes_estrategias_mensais_controle_norm,
        ],
        axis=0,
        ignore_index=True,
    )
    .sort_values(["tipo_estrategia", "ordem_aporte_estrategia", "request_id"])
    .reset_index(drop=True)
)

lista_cestas = []
lista_auditoria = []

for linha in df_solicitacoes_aporte_unificadas.itertuples(index=False):
    df_cesta_tmp, auditoria_tmp = selecionar_cesta_para_solicitacao(
        linha=linha,
        df_elegivel_preco=df_elegivel_preco,
        df_ibov_benchmark=df_ibov_benchmark,
    )

    lista_auditoria.append(auditoria_tmp)

    if not df_cesta_tmp.empty:
        lista_cestas.append(df_cesta_tmp)

df_cesta_ativos_por_aporte = (
    pd.concat(lista_cestas, axis=0, ignore_index=True)
    if len(lista_cestas) > 0
    else pd.DataFrame()
)

df_auditoria_cesta_ativos_por_aporte = pd.DataFrame(lista_auditoria)

if df_cesta_ativos_por_aporte.empty:
    raise ValueError("A base final da cesta de ativos por aporte ficou vazia.")

df_cesta_ativos_por_aporte = df_cesta_ativos_por_aporte[
    [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "origem_calendario",
        "grupo_selecao_ativos",
        "benchmark_renda_variavel",
        "regra_selecao_cesta",
        "versao_cesta",
        "carteira_id",
        "replica_id",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_aporte_estrategia",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "preco_compra_referencia",
        "n_ativos_alvo",
        "n_ativos_disponiveis_data_aporte",
        "n_ativos_selecionados_aporte",
        "flag_quantidade_alvo_atendida",
        "flag_cesta_vazia",
        "flag_preco_deslocado",
        "deslocamento_dias_corridos_preco",
        "capital_inicial_estrategia",
        "valor_aporte_estrategia",
        "valor_orcamento_ativo_planejado",
        "peso_ativo_planejado",
        "seed_sorteio_cesta_aporte",
    ]
].copy()

df_inconsistencias_cesta_ativos_por_aporte = (
    df_auditoria_cesta_ativos_por_aporte
    .loc[
        (df_auditoria_cesta_ativos_por_aporte["flag_cesta_vazia"])
        | (~df_auditoria_cesta_ativos_por_aporte["flag_quantidade_alvo_atendida"])
    ]
    .copy()
)

print(f"Solicitações unificadas de aporte              : {len(df_solicitacoes_aporte_unificadas):,}")
print(f"Linhas da base final da cesta por aporte       : {len(df_cesta_ativos_por_aporte):,}")
print(f"Linhas da auditoria por aporte                 : {len(df_auditoria_cesta_ativos_por_aporte):,}")
print(f"Inconsistências identificadas                  : {len(df_inconsistencias_cesta_ativos_por_aporte):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas formais da subetapa...")

# Observação metodológica:
# o resumo e a distribuição anual são consolidados por estrategia_id
# para evitar colapsar múltiplas réplicas aleatórias em uma única linha.

df_resumo_cesta_ativos_por_aporte = (
    df_auditoria_cesta_ativos_por_aporte
    .groupby(
        [
            "familia_estrategia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_aportes=("request_id", "count"),
        data_primeiro_aporte=("data_aporte_efetiva", "min"),
        data_ultimo_aporte=("data_aporte_efetiva", "max"),
        n_ativos_alvo=("n_ativos_alvo", "median"),
        n_ativos_medio_disponiveis=("n_ativos_disponiveis_data_aporte", "mean"),
        n_ativos_mediano_selecionados=("n_ativos_selecionados_aporte", "median"),
        n_ativos_minimo_selecionados=("n_ativos_selecionados_aporte", "min"),
        n_ativos_maximo_selecionados=("n_ativos_selecionados_aporte", "max"),
        n_aportes_quantidade_alvo_atendida=("flag_quantidade_alvo_atendida", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_aportes_cesta_vazia=("flag_cesta_vazia", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_aportes_preco_deslocado=("flag_preco_deslocado", lambda x: int(pd.Series(x).fillna(False).sum())),
        deslocamento_dias_corridos_preco_maximo=("deslocamento_dias_corridos_preco", "max"),
        valor_total_orcamento=("valor_aporte_estrategia", "sum"),
        capital_inicial_estrategia=("capital_inicial_estrategia", "first"),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id"])
    .reset_index(drop=True)
)

df_resumo_cesta_ativos_por_aporte["diferenca_total_vs_capital"] = (
    df_resumo_cesta_ativos_por_aporte["valor_total_orcamento"]
    - df_resumo_cesta_ativos_por_aporte["capital_inicial_estrategia"]
)

df_distribuicao_anual_cesta_ativos_por_aporte = (
    df_auditoria_cesta_ativos_por_aporte
    .assign(ano=lambda df: pd.to_datetime(df["data_aporte_efetiva"], errors="coerce").dt.year)
    .groupby(
        [
            "familia_estrategia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_aportes=("request_id", "count"),
        valor_total_orcamento=("valor_aporte_estrategia", "sum"),
        n_ativos_medio_selecionados=("n_ativos_selecionados_aporte", "mean"),
        n_aportes_quantidade_alvo_atendida=("flag_quantidade_alvo_atendida", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_aportes_preco_deslocado=("flag_preco_deslocado", lambda x: int(pd.Series(x).fillna(False).sum())),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id", "ano"])
    .reset_index(drop=True)
)

print("Tabelas formais da subetapa construídas com sucesso.")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_selecao_ativos_aporte, caminho_tbl_regras_selecao_ativos_aporte, index=False)
salvar_dataframe(df_cesta_ativos_por_aporte, caminho_base_cesta_ativos_por_aporte, index=False)
salvar_dataframe(df_auditoria_cesta_ativos_por_aporte, caminho_tbl_auditoria_cesta_ativos_por_aporte, index=False)
salvar_dataframe(df_resumo_cesta_ativos_por_aporte, caminho_tbl_resumo_cesta_ativos_por_aporte, index=False)
salvar_dataframe(df_inconsistencias_cesta_ativos_por_aporte, caminho_tbl_inconsistencias_cesta_ativos_por_aporte, index=False)
salvar_dataframe(df_distribuicao_anual_cesta_ativos_por_aporte, caminho_tbl_distribuicao_anual_cesta_ativos_por_aporte, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_selecao_ativos_aporte = df_regras_selecao_ativos_aporte.copy()
df_amostra_resumo_cesta_ativos_por_aporte = df_resumo_cesta_ativos_por_aporte.copy()
df_amostra_auditoria_cesta_ativos_por_aporte = df_auditoria_cesta_ativos_por_aporte.head(20).copy()
df_amostra_cesta_ativos_por_aporte = df_cesta_ativos_por_aporte.head(20).copy()
df_amostra_inconsistencias_cesta_ativos_por_aporte = df_inconsistencias_cesta_ativos_por_aporte.head(20).copy()
df_amostra_distribuicao_anual_cesta_ativos_por_aporte = df_distribuicao_anual_cesta_ativos_por_aporte.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais de seleção dos ativos por aporte:")
print(df_amostra_regras_selecao_ativos_aporte.to_string(index=False))

print("\nResumo da cesta selecionada por estratégia:")
print(df_amostra_resumo_cesta_ativos_por_aporte.to_string(index=False))

print("\nAuditoria da cesta selecionada por aporte - amostra:")
print(df_amostra_auditoria_cesta_ativos_por_aporte.to_string(index=False))

print("\nBase final da cesta de ativos por aporte - amostra:")
print(df_amostra_cesta_ativos_por_aporte.to_string(index=False))

print("\nDistribuição anual da cesta selecionada - amostra:")
print(df_amostra_distribuicao_anual_cesta_ativos_por_aporte.to_string(index=False))

print("\nInconsistências da seleção dos ativos por aporte - amostra:")
print(df_amostra_inconsistencias_cesta_ativos_por_aporte.to_string(index=False))

print("\nArquivos salvos na subetapa 8.1:")
print(f"- {caminho_tbl_regras_selecao_ativos_aporte}")
print(f"- {caminho_base_cesta_ativos_por_aporte}")
print(f"- {caminho_tbl_auditoria_cesta_ativos_por_aporte}")
print(f"- {caminho_tbl_resumo_cesta_ativos_por_aporte}")
print(f"- {caminho_tbl_inconsistencias_cesta_ativos_por_aporte}")
print(f"- {caminho_tbl_distribuicao_anual_cesta_ativos_por_aporte}")

print("\nETAPA 8.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 8.1 - REGRA DE SELEÇÃO DOS ATIVOS POR APORTE

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - parâmetros globais                      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_1_tbl_parametros_globais.parquet
Entrada - ibov padronizada                        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_ibov_padronizada.parquet
Entrada - mercado diário consolidada             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - tickers aptos por janela contábil      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados

## Etapa 8.2) Regra de Alocação Equal Weight

In [36]:
%%time
# ============================================================
# Etapa 8.2) Regra de Alocação Equal Weight
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 8.2 - REGRA DE ALOCAÇÃO EQUAL WEIGHT")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_cesta_ativos_por_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=1,
    tipo_arquivo="base",
    nome="cesta_ativos_por_aporte",
)

caminho_tbl_auditoria_cesta_ativos_por_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="auditoria_cesta_ativos_por_aporte",
)

caminho_tbl_regras_alocacao_equal_weight = gerar_caminho_arquivo(
    etapa=8,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="regras_alocacao_equal_weight",
)

caminho_base_pesos_alvo_equal_weight_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=2,
    tipo_arquivo="base",
    nome="pesos_alvo_equal_weight_aporte",
)

caminho_tbl_auditoria_equal_weight_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="auditoria_equal_weight_aporte",
)

caminho_tbl_resumo_equal_weight_estrategia = gerar_caminho_arquivo(
    etapa=8,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="resumo_equal_weight_estrategia",
)

caminho_tbl_inconsistencias_equal_weight_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="inconsistencias_equal_weight_aporte",
)

caminho_tbl_distribuicao_anual_equal_weight = gerar_caminho_arquivo(
    etapa=8,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_equal_weight",
)

print(f"Entrada - base cesta ativos por aporte       : {caminho_base_cesta_ativos_por_aporte}")
print(f"Entrada - auditoria cesta por aporte         : {caminho_tbl_auditoria_cesta_ativos_por_aporte}")
print(f"Saída   - regras de alocação equal weight    : {caminho_tbl_regras_alocacao_equal_weight}")
print(f"Saída   - base pesos-alvo por aporte         : {caminho_base_pesos_alvo_equal_weight_aporte}")
print(f"Saída   - auditoria equal weight por aporte  : {caminho_tbl_auditoria_equal_weight_aporte}")
print(f"Saída   - resumo equal weight por estratégia : {caminho_tbl_resumo_equal_weight_estrategia}")
print(f"Saída   - inconsistências equal weight       : {caminho_tbl_inconsistencias_equal_weight_aporte}")
print(f"Saída   - distribuição anual equal weight    : {caminho_tbl_distribuicao_anual_equal_weight}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_cesta_ativos_por_aporte = pd.read_parquet(caminho_base_cesta_ativos_por_aporte)
df_auditoria_cesta_ativos_por_aporte = pd.read_parquet(caminho_tbl_auditoria_cesta_ativos_por_aporte)

print(f"Base cesta ativos por aporte       : {df_cesta_ativos_por_aporte.shape[0]:,} linhas x {df_cesta_ativos_por_aporte.shape[1]} colunas")
print(f"Auditoria cesta por aporte         : {df_auditoria_cesta_ativos_por_aporte.shape[0]:,} linhas x {df_auditoria_cesta_ativos_por_aporte.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_OPERACIONAL_SOMA_PESOS = 1e-08

COLUNAS_MINIMAS_BASE_CESTA = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "origem_calendario",
    "grupo_selecao_ativos",
    "regra_selecao_cesta",
    "versao_cesta",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "ticker",
    "issuer_code",
    "preco_compra_referencia",
    "valor_aporte_estrategia",
]

COLUNAS_MINIMAS_AUDITORIA = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "n_ativos_selecionados_aporte",
    "flag_quantidade_alvo_atendida",
    "flag_cesta_vazia",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

print(f"Tolerância operacional da soma dos pesos : {TOLERANCIA_OPERACIONAL_SOMA_PESOS}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

validar_colunas_obrigatorias(df_cesta_ativos_por_aporte, COLUNAS_MINIMAS_BASE_CESTA)
validar_colunas_obrigatorias(df_auditoria_cesta_ativos_por_aporte, COLUNAS_MINIMAS_AUDITORIA)

for coluna in [
    "estrategia_referencia",
    "benchmark_renda_variavel",
    "carteira_id",
    "replica_id",
    "seed_sorteio_cesta_aporte",
    "n_ativos_alvo",
    "n_ativos_disponiveis_data_aporte",
    "n_ativos_selecionados_aporte",
    "capital_inicial_estrategia",
]:
    df_cesta_ativos_por_aporte = garantir_coluna(df_cesta_ativos_por_aporte, coluna)

for coluna in [
    "carteira_id",
    "replica_id",
]:
    df_auditoria_cesta_ativos_por_aporte = garantir_coluna(df_auditoria_cesta_ativos_por_aporte, coluna)

df_cesta_ativos_por_aporte["data_sinal"] = pd.to_datetime(df_cesta_ativos_por_aporte["data_sinal"], errors="coerce")
df_cesta_ativos_por_aporte["data_aporte_efetiva"] = pd.to_datetime(df_cesta_ativos_por_aporte["data_aporte_efetiva"], errors="coerce")
df_cesta_ativos_por_aporte["ordem_aporte_estrategia"] = pd.to_numeric(df_cesta_ativos_por_aporte["ordem_aporte_estrategia"], errors="coerce")
df_cesta_ativos_por_aporte["preco_compra_referencia"] = pd.to_numeric(df_cesta_ativos_por_aporte["preco_compra_referencia"], errors="coerce")
df_cesta_ativos_por_aporte["valor_aporte_estrategia"] = pd.to_numeric(df_cesta_ativos_por_aporte["valor_aporte_estrategia"], errors="coerce")
df_cesta_ativos_por_aporte["capital_inicial_estrategia"] = pd.to_numeric(df_cesta_ativos_por_aporte["capital_inicial_estrategia"], errors="coerce")
df_cesta_ativos_por_aporte["n_ativos_alvo"] = pd.to_numeric(df_cesta_ativos_por_aporte["n_ativos_alvo"], errors="coerce")
df_cesta_ativos_por_aporte["n_ativos_disponiveis_data_aporte"] = pd.to_numeric(df_cesta_ativos_por_aporte["n_ativos_disponiveis_data_aporte"], errors="coerce")
df_cesta_ativos_por_aporte["n_ativos_selecionados_aporte"] = pd.to_numeric(df_cesta_ativos_por_aporte["n_ativos_selecionados_aporte"], errors="coerce")

df_auditoria_cesta_ativos_por_aporte["data_aporte_efetiva"] = pd.to_datetime(df_auditoria_cesta_ativos_por_aporte["data_aporte_efetiva"], errors="coerce")
df_auditoria_cesta_ativos_por_aporte["ordem_aporte_estrategia"] = pd.to_numeric(df_auditoria_cesta_ativos_por_aporte["ordem_aporte_estrategia"], errors="coerce")
df_auditoria_cesta_ativos_por_aporte["n_ativos_selecionados_aporte"] = pd.to_numeric(df_auditoria_cesta_ativos_por_aporte["n_ativos_selecionados_aporte"], errors="coerce")

df_cesta_ativos_por_aporte["ticker"] = df_cesta_ativos_por_aporte["ticker"].astype("string").str.strip().str.upper()
df_cesta_ativos_por_aporte["issuer_code"] = df_cesta_ativos_por_aporte["issuer_code"].astype("string").str.strip()

duplicidades_request_ticker = int(
    df_cesta_ativos_por_aporte.duplicated(subset=["request_id", "ticker"]).sum()
)

df_inconsistencias_herdadas_8_1 = (
    df_auditoria_cesta_ativos_por_aporte
    .loc[
        (df_auditoria_cesta_ativos_por_aporte["flag_cesta_vazia"])
        | (~df_auditoria_cesta_ativos_por_aporte["flag_quantidade_alvo_atendida"])
    ]
    .copy()
)

if duplicidades_request_ticker > 0:
    raise ValueError(f"Foram encontradas {duplicidades_request_ticker} duplicidades por request_id + ticker na base da etapa 8.1.")

print(f"Requests únicos na base de cesta           : {df_cesta_ativos_por_aporte['request_id'].nunique():,}")
print(f"Requests únicos na auditoria herdada       : {df_auditoria_cesta_ativos_por_aporte['request_id'].nunique():,}")
print(f"Duplicidades request+ticker                : {duplicidades_request_ticker:,}")
print(f"Inconsistências herdadas da 8.1            : {len(df_inconsistencias_herdadas_8_1):,}")
print("OK")

# ============================================================
# 6) Construção da base de pesos-alvo equal weight por aporte
# ============================================================

print("\n[6/10] Construção da base de pesos-alvo equal weight por aporte...")

df_pesos_alvo_equal_weight_aporte = df_cesta_ativos_por_aporte.copy()

df_pesos_alvo_equal_weight_aporte["ordem_sinal_final"] = df_pesos_alvo_equal_weight_aporte["ordem_aporte_estrategia"]
df_pesos_alvo_equal_weight_aporte["ano"] = df_pesos_alvo_equal_weight_aporte["data_aporte_efetiva"].dt.year
df_pesos_alvo_equal_weight_aporte["mes"] = df_pesos_alvo_equal_weight_aporte["data_aporte_efetiva"].dt.month
df_pesos_alvo_equal_weight_aporte["close_adj"] = df_pesos_alvo_equal_weight_aporte["preco_compra_referencia"]

df_pesos_alvo_equal_weight_aporte["n_tickers_cesta_equal_weight"] = (
    df_pesos_alvo_equal_weight_aporte.groupby("request_id")["ticker"].transform("nunique")
)

df_pesos_alvo_equal_weight_aporte["ordem_ticker_no_aporte"] = (
    df_pesos_alvo_equal_weight_aporte
    .sort_values(["request_id", "ticker"])
    .groupby("request_id")
    .cumcount()
    + 1
)

df_pesos_alvo_equal_weight_aporte["peso_alvo_equal_weight"] = (
    1.0 / df_pesos_alvo_equal_weight_aporte["n_tickers_cesta_equal_weight"]
)

df_pesos_alvo_equal_weight_aporte["peso_alvo_percentual"] = (
    df_pesos_alvo_equal_weight_aporte["peso_alvo_equal_weight"] * 100.0
)

df_pesos_alvo_equal_weight_aporte["valor_orcamento_alvo_ticker"] = (
    df_pesos_alvo_equal_weight_aporte["valor_aporte_estrategia"]
    * df_pesos_alvo_equal_weight_aporte["peso_alvo_equal_weight"]
)

df_pesos_alvo_equal_weight_aporte["flag_equal_weight_unitario"] = (
    df_pesos_alvo_equal_weight_aporte["n_tickers_cesta_equal_weight"] == 1
)

df_pesos_alvo_equal_weight_aporte["flag_peso_positivo"] = (
    df_pesos_alvo_equal_weight_aporte["peso_alvo_equal_weight"] > 0
)

df_pesos_alvo_equal_weight_aporte["flag_orcamento_alvo_positivo"] = (
    df_pesos_alvo_equal_weight_aporte["valor_orcamento_alvo_ticker"] > 0
)

df_pesos_alvo_equal_weight_aporte["flag_preco_valido_equal_weight"] = (
    df_pesos_alvo_equal_weight_aporte["close_adj"] > 0
)

df_pesos_alvo_equal_weight_aporte["grupo_selecao_ativos"] = (
    df_pesos_alvo_equal_weight_aporte["grupo_selecao_ativos"]
    .astype("string")
    .fillna("na")
)

print(f"Linhas da base de pesos-alvo equal weight : {len(df_pesos_alvo_equal_weight_aporte):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais e auditorias da subetapa
# ============================================================

print("\n[7/10] Construção das tabelas formais e auditorias da subetapa...")

df_regras_alocacao_equal_weight = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "EW1",
            "escopo": "todos_os_aportes",
            "regra_operacional": "atribuir_peso_igual_para_cada_ticker_da_cesta",
            "detalhe": "Cada aporte distribui igualmente o orçamento entre todos os ativos que compõem a cesta comprável do request.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "EW2",
            "escopo": "orcamento_por_ticker",
            "regra_operacional": "valor_aporte_dividido_pelo_numero_de_tickers_da_cesta",
            "detalhe": "O orçamento-alvo por ativo é calculado como valor do aporte multiplicado pelo peso equal weight do ticker.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "EW3",
            "escopo": "consistencia_da_cesta",
            "regra_operacional": "executar_equal_weight_somente_sobre_cestas_validadas_na_8_1",
            "detalhe": "A subetapa herda apenas aportes cuja cesta não esteja vazia e cuja seleção dos ativos tenha sido validada na etapa 8.1.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "EW4",
            "escopo": "benchmark_unitario",
            "regra_operacional": "peso_alvo_igual_a_100_porcento_quando_cesta_tem_um_unico_ativo",
            "detalhe": "Nos aportes do benchmark Ibovespa, o único ativo da cesta recebe peso-alvo unitário.",
        },
    ]
)

df_auditoria_equal_weight_aporte = (
    df_pesos_alvo_equal_weight_aporte
    .groupby(
        [
            "request_id",
            "estrategia_id",
            "grupo_controle",
            "familia_estrategia",
            "tipo_estrategia",
            "data_aporte_efetiva",
            "ordem_aporte_estrategia",
            "carteira_id",
            "replica_id",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_tickers_cesta_equal_weight=("n_tickers_cesta_equal_weight", "max"),
        n_ativos_alvo=("n_ativos_alvo", "first"),
        n_ativos_disponiveis_data_aporte=("n_ativos_disponiveis_data_aporte", "first"),
        n_ativos_selecionados_aporte=("n_ativos_selecionados_aporte", "first"),
        valor_aporte_estrategia=("valor_aporte_estrategia", "first"),
        capital_inicial_estrategia=("capital_inicial_estrategia", "first"),
        soma_pesos_alvo_equal_weight=("peso_alvo_equal_weight", "sum"),
        soma_orcamento_alvo_ticker=("valor_orcamento_alvo_ticker", "sum"),
        peso_minimo_ticker=("peso_alvo_equal_weight", "min"),
        peso_maximo_ticker=("peso_alvo_equal_weight", "max"),
        flag_equal_weight_unitario=("flag_equal_weight_unitario", "max"),
        n_tickers_com_peso_positivo=("flag_peso_positivo", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_tickers_com_orcamento_alvo_positivo=("flag_orcamento_alvo_positivo", lambda x: int(pd.Series(x).fillna(False).sum())),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "request_id"])
    .reset_index(drop=True)
)

df_auditoria_equal_weight_aporte["desvio_absoluto_soma_pesos"] = (
    df_auditoria_equal_weight_aporte["soma_pesos_alvo_equal_weight"] - 1.0
).abs()

df_auditoria_equal_weight_aporte["desvio_absoluto_soma_orcamento"] = (
    df_auditoria_equal_weight_aporte["soma_orcamento_alvo_ticker"]
    - df_auditoria_equal_weight_aporte["valor_aporte_estrategia"]
).abs()

df_auditoria_equal_weight_aporte["flag_soma_pesos_valida"] = (
    df_auditoria_equal_weight_aporte["desvio_absoluto_soma_pesos"] <= TOLERANCIA_OPERACIONAL_SOMA_PESOS
)

df_auditoria_equal_weight_aporte["flag_soma_orcamento_valida"] = (
    df_auditoria_equal_weight_aporte["desvio_absoluto_soma_orcamento"] <= TOLERANCIA_OPERACIONAL_SOMA_PESOS
)

df_auditoria_equal_weight_aporte["flag_quantidade_tickers_coerente"] = (
    df_auditoria_equal_weight_aporte["n_tickers_cesta_equal_weight"]
    == df_auditoria_equal_weight_aporte["n_ativos_selecionados_aporte"]
)

df_auditoria_equal_weight_aporte["flag_pesos_positivos"] = (
    df_auditoria_equal_weight_aporte["n_tickers_com_peso_positivo"]
    == df_auditoria_equal_weight_aporte["n_tickers_cesta_equal_weight"]
)

df_auditoria_equal_weight_aporte["flag_orcamentos_positivos"] = (
    df_auditoria_equal_weight_aporte["n_tickers_com_orcamento_alvo_positivo"]
    == df_auditoria_equal_weight_aporte["n_tickers_cesta_equal_weight"]
)

df_inconsistencias_equal_weight_aporte = (
    df_auditoria_equal_weight_aporte
    .loc[
        (~df_auditoria_equal_weight_aporte["flag_soma_pesos_valida"])
        | (~df_auditoria_equal_weight_aporte["flag_soma_orcamento_valida"])
        | (~df_auditoria_equal_weight_aporte["flag_quantidade_tickers_coerente"])
        | (~df_auditoria_equal_weight_aporte["flag_pesos_positivos"])
        | (~df_auditoria_equal_weight_aporte["flag_orcamentos_positivos"])
    ]
    .copy()
)

df_resumo_equal_weight_estrategia = (
    df_auditoria_equal_weight_aporte
    .groupby(
        [
            "familia_estrategia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_aportes=("request_id", "count"),
        data_primeiro_aporte=("data_aporte_efetiva", "min"),
        data_ultimo_aporte=("data_aporte_efetiva", "max"),
        n_tickers_medio_cesta=("n_tickers_cesta_equal_weight", "mean"),
        n_tickers_mediano_cesta=("n_tickers_cesta_equal_weight", "median"),
        peso_minimo_global=("peso_minimo_ticker", "min"),
        peso_maximo_global=("peso_maximo_ticker", "max"),
        n_aportes_soma_pesos_valida=("flag_soma_pesos_valida", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_aportes_soma_orcamento_valida=("flag_soma_orcamento_valida", lambda x: int(pd.Series(x).fillna(False).sum())),
        valor_total_orcamento=("valor_aporte_estrategia", "sum"),
        capital_inicial_estrategia=("capital_inicial_estrategia", "first"),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id"])
    .reset_index(drop=True)
)

df_resumo_equal_weight_estrategia["diferenca_total_vs_capital"] = (
    df_resumo_equal_weight_estrategia["valor_total_orcamento"]
    - df_resumo_equal_weight_estrategia["capital_inicial_estrategia"]
)

df_distribuicao_anual_equal_weight = (
    df_auditoria_equal_weight_aporte
    .assign(ano=lambda df: pd.to_datetime(df["data_aporte_efetiva"], errors="coerce").dt.year)
    .groupby(
        [
            "familia_estrategia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_aportes=("request_id", "count"),
        valor_total_orcamento=("valor_aporte_estrategia", "sum"),
        n_tickers_medio_cesta=("n_tickers_cesta_equal_weight", "mean"),
        n_aportes_soma_pesos_valida=("flag_soma_pesos_valida", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_aportes_soma_orcamento_valida=("flag_soma_orcamento_valida", lambda x: int(pd.Series(x).fillna(False).sum())),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id", "ano"])
    .reset_index(drop=True)
)

print(f"Inconsistências equal weight identificadas : {len(df_inconsistencias_equal_weight_aporte):,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_alocacao_equal_weight, caminho_tbl_regras_alocacao_equal_weight, index=False)
salvar_dataframe(df_pesos_alvo_equal_weight_aporte, caminho_base_pesos_alvo_equal_weight_aporte, index=False)
salvar_dataframe(df_auditoria_equal_weight_aporte, caminho_tbl_auditoria_equal_weight_aporte, index=False)
salvar_dataframe(df_resumo_equal_weight_estrategia, caminho_tbl_resumo_equal_weight_estrategia, index=False)
salvar_dataframe(df_inconsistencias_equal_weight_aporte, caminho_tbl_inconsistencias_equal_weight_aporte, index=False)
salvar_dataframe(df_distribuicao_anual_equal_weight, caminho_tbl_distribuicao_anual_equal_weight, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_alocacao_equal_weight = df_regras_alocacao_equal_weight.copy()
df_amostra_pesos_alvo_equal_weight_aporte = df_pesos_alvo_equal_weight_aporte.head(20).copy()
df_amostra_auditoria_equal_weight_aporte = df_auditoria_equal_weight_aporte.head(20).copy()
df_amostra_resumo_equal_weight_estrategia = df_resumo_equal_weight_estrategia.copy()
df_amostra_inconsistencias_equal_weight_aporte = df_inconsistencias_equal_weight_aporte.head(20).copy()
df_amostra_distribuicao_anual_equal_weight = df_distribuicao_anual_equal_weight.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais de alocação equal weight:")
print(df_amostra_regras_alocacao_equal_weight.to_string(index=False))

print("\nBase de pesos-alvo equal weight por aporte - amostra:")
print(df_amostra_pesos_alvo_equal_weight_aporte.to_string(index=False))

print("\nAuditoria equal weight por aporte - amostra:")
print(df_amostra_auditoria_equal_weight_aporte.to_string(index=False))

print("\nResumo equal weight por estratégia:")
print(df_amostra_resumo_equal_weight_estrategia.to_string(index=False))

print("\nDistribuição anual equal weight - amostra:")
print(df_amostra_distribuicao_anual_equal_weight.to_string(index=False))

print("\nInconsistências equal weight - amostra:")
print(df_amostra_inconsistencias_equal_weight_aporte.to_string(index=False))

print("\nArquivos salvos na subetapa 8.2:")
print(f"- {caminho_tbl_regras_alocacao_equal_weight}")
print(f"- {caminho_base_pesos_alvo_equal_weight_aporte}")
print(f"- {caminho_tbl_auditoria_equal_weight_aporte}")
print(f"- {caminho_tbl_resumo_equal_weight_estrategia}")
print(f"- {caminho_tbl_inconsistencias_equal_weight_aporte}")
print(f"- {caminho_tbl_distribuicao_anual_equal_weight}")

print("\nETAPA 8.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 8.2 - REGRA DE ALOCAÇÃO EQUAL WEIGHT

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - base cesta ativos por aporte       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_1_base_cesta_ativos_por_aporte.parquet
Entrada - auditoria cesta por aporte         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_1_tbl_auditoria_cesta_ativos_por_aporte.parquet
Saída   - regras de alocação equal weight    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_2_tbl_regras_alocacao_equal_weight.parquet
Saída   - base pesos-alvo por aporte         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\et

## Etapa 8.3) Regra de Conversão do Orçamento em Quantidades

In [37]:
%%time
# ============================================================
# Etapa 8.3) Regra de Conversão do Orçamento em Quantidades
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 8.3 - REGRA DE CONVERSÃO DO ORÇAMENTO EM QUANTIDADES")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_pesos_alvo_equal_weight_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=2,
    tipo_arquivo="base",
    nome="pesos_alvo_equal_weight_aporte",
)

caminho_tbl_auditoria_equal_weight_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="auditoria_equal_weight_aporte",
)

caminho_tbl_regras_conversao_orcamento_quantidades = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="regras_conversao_orcamento_quantidades",
)

caminho_base_quantidades_alvo_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="base",
    nome="quantidades_alvo_aporte",
)

caminho_tbl_auditoria_quantidades_alvo_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="auditoria_quantidades_alvo_aporte",
)

caminho_tbl_resumo_quantidades_alvo_estrategia = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="resumo_quantidades_alvo_estrategia",
)

caminho_tbl_inviabilidades_quantidade_minima_ticker = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="inviabilidades_quantidade_minima_ticker",
)

caminho_tbl_inconsistencias_quantidades_alvo_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="inconsistencias_quantidades_alvo_aporte",
)

caminho_tbl_distribuicao_anual_quantidades_alvo = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_quantidades_alvo",
)

print(f"Entrada - base pesos-alvo equal weight         : {caminho_base_pesos_alvo_equal_weight_aporte}")
print(f"Entrada - auditoria equal weight               : {caminho_tbl_auditoria_equal_weight_aporte}")
print(f"Saída   - regras de conversão                  : {caminho_tbl_regras_conversao_orcamento_quantidades}")
print(f"Saída   - base quantidades-alvo por aporte     : {caminho_base_quantidades_alvo_aporte}")
print(f"Saída   - auditoria de quantidades por aporte  : {caminho_tbl_auditoria_quantidades_alvo_aporte}")
print(f"Saída   - resumo de quantidades por estratégia : {caminho_tbl_resumo_quantidades_alvo_estrategia}")
print(f"Saída   - inviabilidades quantidade mínima     : {caminho_tbl_inviabilidades_quantidade_minima_ticker}")
print(f"Saída   - inconsistências de quantidades       : {caminho_tbl_inconsistencias_quantidades_alvo_aporte}")
print(f"Saída   - distribuição anual de quantidades    : {caminho_tbl_distribuicao_anual_quantidades_alvo}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_pesos_alvo_equal_weight_aporte = pd.read_parquet(caminho_base_pesos_alvo_equal_weight_aporte)
df_auditoria_equal_weight_aporte = pd.read_parquet(caminho_tbl_auditoria_equal_weight_aporte)

print(f"Base pesos-alvo equal weight         : {df_pesos_alvo_equal_weight_aporte.shape[0]:,} linhas x {df_pesos_alvo_equal_weight_aporte.shape[1]} colunas")
print(f"Auditoria equal weight por aporte    : {df_auditoria_equal_weight_aporte.shape[0]:,} linhas x {df_auditoria_equal_weight_aporte.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_CONSERVACAO_ORCAMENTO = 1e-08
LIMITE_ITERACOES_REDISTRIBUICAO = 100000

COLUNAS_MINIMAS_PESOS = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "origem_calendario",
    "grupo_selecao_ativos",
    "benchmark_renda_variavel",
    "regra_selecao_cesta",
    "versao_cesta",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "ticker",
    "issuer_code",
    "close_adj",
    "ano",
    "mes",
    "capital_inicial_estrategia",
    "valor_aporte_estrategia",
    "n_ativos_alvo",
    "n_ativos_disponiveis_data_aporte",
    "n_ativos_selecionados_aporte",
    "n_tickers_cesta_equal_weight",
    "ordem_ticker_no_aporte",
    "peso_alvo_equal_weight",
    "peso_alvo_percentual",
    "valor_orcamento_alvo_ticker",
]

COLUNAS_MINIMAS_AUDITORIA_8_2 = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "n_tickers_cesta_equal_weight",
    "valor_aporte_estrategia",
    "flag_soma_pesos_valida",
    "flag_soma_orcamento_valida",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

def eh_benchmark_equivalente(df_request):
    if "tipo_estrategia" in df_request.columns:
        if str(df_request["tipo_estrategia"].iloc[0]).strip().lower() == "ibovespa_aportes_mensais":
            return True
    if "benchmark_renda_variavel" in df_request.columns and "n_tickers_cesta_equal_weight" in df_request.columns:
        return (
            pd.notna(df_request["benchmark_renda_variavel"].iloc[0])
            and int(df_request["n_tickers_cesta_equal_weight"].iloc[0]) == 1
        )
    return False

def converter_request_em_quantidades(df_request):
    df_req = df_request.copy().sort_values("ordem_ticker_no_aporte").reset_index(drop=True)

    df_req["flag_quantidade_fracionaria_permitida"] = False
    df_req["tipo_conversao_quantidade"] = "quantidade_inteira_com_redistribuicao"
    df_req["quantidade_inicial_inteira"] = 0.0
    df_req["quantidade_extra_redistribuicao"] = 0.0
    df_req["flag_recebeu_redistribuicao"] = False
    df_req["quantidade_alvo_convertida"] = 0.0

    orcamentos = df_req["valor_orcamento_alvo_ticker"].astype(float).to_numpy()
    precos = df_req["close_adj"].astype(float).to_numpy()

    if eh_benchmark_equivalente(df_req):
        quantidade_fracionaria = np.where(precos > 0, orcamentos / precos, 0.0)

        df_req["flag_quantidade_fracionaria_permitida"] = True
        df_req["tipo_conversao_quantidade"] = "quantidade_fracionaria_equivalente_benchmark"
        df_req["quantidade_inicial_inteira"] = 0.0
        df_req["quantidade_extra_redistribuicao"] = 0.0
        df_req["flag_recebeu_redistribuicao"] = False
        df_req["quantidade_alvo_convertida"] = quantidade_fracionaria

    else:
        quantidade_inicial = np.where(precos > 0, np.floor(orcamentos / precos), 0.0)
        quantidade_extra = np.zeros(len(df_req), dtype=float)

        caixa_residual_global = float(orcamentos.sum() - np.sum(quantidade_inicial * precos))
        iteracao = 0

        while caixa_residual_global > TOLERANCIA_CONSERVACAO_ORCAMENTO and iteracao < LIMITE_ITERACOES_REDISTRIBUICAO:
            quantidade_atual = quantidade_inicial + quantidade_extra
            valor_investido_atual = quantidade_atual * precos

            candidatos_compra = np.where(precos <= caixa_residual_global + TOLERANCIA_CONSERVACAO_ORCAMENTO)[0]
            if len(candidatos_compra) == 0:
                break

            gaps_relativos = np.where(
                orcamentos > 0,
                (orcamentos - valor_investido_atual) / orcamentos,
                -np.inf,
            )

            gaps_candidatos = gaps_relativos[candidatos_compra]
            gap_maximo = np.nanmax(gaps_candidatos)

            candidatos_gap = candidatos_compra[gaps_candidatos == gap_maximo]
            if len(candidatos_gap) > 1:
                idx_escolhido = candidatos_gap[np.argmin(precos[candidatos_gap])]
            else:
                idx_escolhido = int(candidatos_gap[0])

            quantidade_extra[idx_escolhido] += 1.0
            caixa_residual_global -= float(precos[idx_escolhido])
            iteracao += 1

        df_req["quantidade_inicial_inteira"] = quantidade_inicial
        df_req["quantidade_extra_redistribuicao"] = quantidade_extra
        df_req["flag_recebeu_redistribuicao"] = df_req["quantidade_extra_redistribuicao"] > 0
        df_req["quantidade_alvo_convertida"] = df_req["quantidade_inicial_inteira"] + df_req["quantidade_extra_redistribuicao"]

    df_req["flag_quantidade_minima_viavel"] = df_req["quantidade_alvo_convertida"] > 0
    df_req["flag_compra_efetivada"] = df_req["flag_quantidade_minima_viavel"]

    df_req["valor_investido_ticker"] = (
        df_req["quantidade_alvo_convertida"].astype(float)
        * df_req["close_adj"].astype(float)
    )

    df_req["caixa_nao_investido_ticker"] = (
        df_req["valor_orcamento_alvo_ticker"].astype(float)
        - df_req["valor_investido_ticker"].astype(float)
    )

    valor_aporte = float(df_req["valor_aporte_estrategia"].iloc[0])

    if valor_aporte > 0:
        df_req["peso_efetivo_investido_ticker"] = df_req["valor_investido_ticker"] / valor_aporte
    else:
        df_req["peso_efetivo_investido_ticker"] = 0.0

    df_req["flag_caixa_residual_positivo_ticker"] = (
        df_req["caixa_nao_investido_ticker"] > TOLERANCIA_CONSERVACAO_ORCAMENTO
    )

    df_req["desvio_orcamento_pos_execucao_ticker"] = (
        df_req["valor_investido_ticker"] - df_req["valor_orcamento_alvo_ticker"]
    )

    auditoria_request = {
        "request_id": df_req["request_id"].iloc[0],
        "estrategia_id": df_req["estrategia_id"].iloc[0],
        "grupo_controle": df_req["grupo_controle"].iloc[0],
        "familia_estrategia": df_req["familia_estrategia"].iloc[0],
        "tipo_estrategia": df_req["tipo_estrategia"].iloc[0],
        "origem_calendario": df_req["origem_calendario"].iloc[0],
        "carteira_id": df_req["carteira_id"].iloc[0],
        "replica_id": df_req["replica_id"].iloc[0],
        "data_sinal": df_req["data_sinal"].iloc[0],
        "data_aporte_efetiva": df_req["data_aporte_efetiva"].iloc[0],
        "ordem_aporte_estrategia": df_req["ordem_aporte_estrategia"].iloc[0],
        "n_tickers_cesta_equal_weight": int(df_req["n_tickers_cesta_equal_weight"].iloc[0]),
        "n_tickers_com_quantidade_positiva": int(df_req["flag_compra_efetivada"].sum()),
        "n_tickers_inviaveis_quantidade_minima": int((~df_req["flag_quantidade_minima_viavel"]).sum()),
        "quantidade_total_convertida": float(df_req["quantidade_alvo_convertida"].sum()),
        "valor_aporte_estrategia": float(df_req["valor_aporte_estrategia"].iloc[0]),
        "valor_total_investido": float(df_req["valor_investido_ticker"].sum()),
        "caixa_residual_final_aporte": float(df_req["valor_aporte_estrategia"].iloc[0] - df_req["valor_investido_ticker"].sum()),
        "pct_orcamento_investido": float(df_req["valor_investido_ticker"].sum() / float(df_req["valor_aporte_estrategia"].iloc[0])) if float(df_req["valor_aporte_estrategia"].iloc[0]) > 0 else 0.0,
        "flag_conservacao_orcamento": abs(float(df_req["valor_aporte_estrategia"].iloc[0] - df_req["valor_investido_ticker"].sum() - (float(df_req["valor_aporte_estrategia"].iloc[0] - df_req["valor_investido_ticker"].sum())))) <= TOLERANCIA_CONSERVACAO_ORCAMENTO,
        "flag_compra_minima_existe_no_aporte": bool((df_req["flag_quantidade_minima_viavel"]).any()),
        "flag_benchmark_equivalente": bool(eh_benchmark_equivalente(df_req)),
        "n_tickers_redistribuidos": int(df_req["flag_recebeu_redistribuicao"].sum()),
    }

    return df_req, auditoria_request

print(f"Tolerância de conservação do orçamento : {TOLERANCIA_CONSERVACAO_ORCAMENTO}")
print(f"Limite de iterações de redistribuição  : {LIMITE_ITERACOES_REDISTRIBUICAO}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

validar_colunas_obrigatorias(df_pesos_alvo_equal_weight_aporte, COLUNAS_MINIMAS_PESOS)
validar_colunas_obrigatorias(df_auditoria_equal_weight_aporte, COLUNAS_MINIMAS_AUDITORIA_8_2)

for coluna in ["carteira_id", "replica_id", "estrategia_referencia", "seed_sorteio_cesta_aporte", "nome", "setor", "subsetor", "segmento"]:
    df_pesos_alvo_equal_weight_aporte = garantir_coluna(df_pesos_alvo_equal_weight_aporte, coluna)

df_pesos_alvo_equal_weight_aporte["data_sinal"] = pd.to_datetime(df_pesos_alvo_equal_weight_aporte["data_sinal"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["data_aporte_efetiva"] = pd.to_datetime(df_pesos_alvo_equal_weight_aporte["data_aporte_efetiva"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["ordem_aporte_estrategia"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["ordem_aporte_estrategia"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["ordem_sinal_final"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["ordem_sinal_final"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["close_adj"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["close_adj"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["ano"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["ano"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["mes"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["mes"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["capital_inicial_estrategia"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["capital_inicial_estrategia"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["valor_aporte_estrategia"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["valor_aporte_estrategia"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["n_ativos_alvo"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["n_ativos_alvo"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["n_ativos_disponiveis_data_aporte"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["n_ativos_disponiveis_data_aporte"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["n_ativos_selecionados_aporte"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["n_ativos_selecionados_aporte"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["n_tickers_cesta_equal_weight"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["n_tickers_cesta_equal_weight"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["ordem_ticker_no_aporte"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["ordem_ticker_no_aporte"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["peso_alvo_equal_weight"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["peso_alvo_equal_weight"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["peso_alvo_percentual"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["peso_alvo_percentual"], errors="coerce")
df_pesos_alvo_equal_weight_aporte["valor_orcamento_alvo_ticker"] = pd.to_numeric(df_pesos_alvo_equal_weight_aporte["valor_orcamento_alvo_ticker"], errors="coerce")

duplicidades_request_ticker = int(
    df_pesos_alvo_equal_weight_aporte.duplicated(subset=["request_id", "ticker"]).sum()
)

df_inconsistencias_herdadas_8_2 = (
    df_auditoria_equal_weight_aporte
    .loc[
        (~df_auditoria_equal_weight_aporte["flag_soma_pesos_valida"])
        | (~df_auditoria_equal_weight_aporte["flag_soma_orcamento_valida"])
    ]
    .copy()
)

if duplicidades_request_ticker > 0:
    raise ValueError(f"Foram encontradas {duplicidades_request_ticker} duplicidades por request_id+ticker na base da etapa 8.2.")

print(f"Duplicidades request+ticker na base de pesos : {duplicidades_request_ticker:,}")
print(f"Inconsistências herdadas da 8.2              : {len(df_inconsistencias_herdadas_8_2):,}")
print("OK")

# ============================================================
# 6) Conversão do orçamento em quantidades
# ============================================================

print("\n[6/10] Conversão do orçamento em quantidades...")

lista_requests_convertidos = []
lista_auditoria_requests = []

for request_id, df_request in df_pesos_alvo_equal_weight_aporte.groupby("request_id", sort=False, dropna=False):
    df_convertido_request, auditoria_request = converter_request_em_quantidades(df_request=df_request)
    lista_requests_convertidos.append(df_convertido_request)
    lista_auditoria_requests.append(auditoria_request)

df_quantidades_alvo_aporte = pd.concat(lista_requests_convertidos, axis=0, ignore_index=True)
df_auditoria_quantidades_alvo_aporte = pd.DataFrame(lista_auditoria_requests)

print(f"Linhas da base de quantidades-alvo           : {len(df_quantidades_alvo_aporte):,}")
print(f"Requests processados                         : {df_quantidades_alvo_aporte['request_id'].nunique():,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais, inviabilidades e auditorias
# ============================================================

print("\n[7/10] Construção das tabelas formais, inviabilidades e auditorias...")

df_regras_conversao_orcamento_quantidades = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "QTY1",
            "escopo": "acoes_compraveis",
            "regra_operacional": "converter_o_orcamento_alvo_em_quantidade_inteira_por_piso",
            "detalhe": "Para ações individuais, a quantidade inicial de cada ticker é dada pelo piso do orçamento-alvo dividido por close_adj.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "QTY2",
            "escopo": "redistribuicao_intra_aporte",
            "regra_operacional": "redistribuir_o_residual_dentro_do_mesmo_aporte_por_unidades_inteiras",
            "detalhe": "O residual remanescente do aporte é reaplicado na própria cesta por compra unitária adicional dos ativos mais abaixo do alvo relativo, respeitando o caixa disponível.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "QTY3",
            "escopo": "carry_residual_para_etapa_8_4",
            "regra_operacional": "registrar_o_residual_final_pos_redistribuicao_como_caixa_da_estrategia",
            "detalhe": "Após a redistribuição intra-aporte, o eventual residual remanescente é mantido como caixa para a etapa de tratamento do caixa.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "QTY4",
            "escopo": "benchmark_equivalente",
            "regra_operacional": "manter_quantidade_fracionaria_equivalente_para_o_benchmark_indice",
            "detalhe": "O benchmark de índice permanece tratado por unidade equivalente para evitar inviabilidade artificial decorrente de um ativo não negociável diretamente como ação.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "QTY5",
            "escopo": "comparabilidade_entre_estrategias",
            "regra_operacional": "maximizar_o_capital_efetivamente_investido_sem_quebrar_a_execucao_inteira",
            "detalhe": "A redistribuição intra-aporte reduz a sobra de caixa e melhora a comparabilidade de performance entre estratégias sem recorrer a uma trilha fracionária paralela.",
        },
    ]
)

df_inviabilidades_quantidade_minima_ticker = (
    df_quantidades_alvo_aporte
    .loc[~df_quantidades_alvo_aporte["flag_quantidade_minima_viavel"]]
    .copy()
)

df_auditoria_quantidades_alvo_aporte["flag_conservacao_orcamento"] = (
    (
        df_auditoria_quantidades_alvo_aporte["valor_total_investido"]
        + df_auditoria_quantidades_alvo_aporte["caixa_residual_final_aporte"]
        - df_auditoria_quantidades_alvo_aporte["valor_aporte_estrategia"]
    ).abs() <= TOLERANCIA_CONSERVACAO_ORCAMENTO
)

df_inconsistencias_quantidades_alvo_aporte = (
    df_auditoria_quantidades_alvo_aporte
    .loc[
        (~df_auditoria_quantidades_alvo_aporte["flag_conservacao_orcamento"])
        | (~df_auditoria_quantidades_alvo_aporte["flag_compra_minima_existe_no_aporte"])
        | (df_auditoria_quantidades_alvo_aporte["caixa_residual_final_aporte"] < -TOLERANCIA_CONSERVACAO_ORCAMENTO)
    ]
    .copy()
)

df_resumo_quantidades_alvo_estrategia = (
    df_auditoria_quantidades_alvo_aporte
    .groupby(
        [
            "familia_estrategia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_aportes=("request_id", "count"),
        data_primeiro_aporte=("data_aporte_efetiva", "min"),
        data_ultimo_aporte=("data_aporte_efetiva", "max"),
        n_tickers_medio_com_quantidade_positiva=("n_tickers_com_quantidade_positiva", "mean"),
        n_tickers_medio_inviaveis=("n_tickers_inviaveis_quantidade_minima", "mean"),
        quantidade_total_convertida=("quantidade_total_convertida", "sum"),
        valor_total_investido=("valor_total_investido", "sum"),
        caixa_residual_total=("caixa_residual_final_aporte", "sum"),
        pct_medio_orcamento_investido=("pct_orcamento_investido", "mean"),
        n_aportes_conservacao_valida=("flag_conservacao_orcamento", lambda x: int(pd.Series(x).fillna(False).sum())),
        capital_inicial_estrategia=("valor_aporte_estrategia", "sum"),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id"])
    .reset_index(drop=True)
)

df_resumo_quantidades_alvo_estrategia["taxa_media_investimento"] = (
    df_resumo_quantidades_alvo_estrategia["valor_total_investido"]
    / df_resumo_quantidades_alvo_estrategia["capital_inicial_estrategia"]
)

df_distribuicao_anual_quantidades_alvo = (
    df_auditoria_quantidades_alvo_aporte
    .assign(ano=lambda df: pd.to_datetime(df["data_aporte_efetiva"], errors="coerce").dt.year)
    .groupby(
        [
            "familia_estrategia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_aportes=("request_id", "count"),
        n_tickers_medio_com_quantidade_positiva=("n_tickers_com_quantidade_positiva", "mean"),
        n_tickers_medio_inviaveis=("n_tickers_inviaveis_quantidade_minima", "mean"),
        quantidade_total_convertida=("quantidade_total_convertida", "sum"),
        valor_total_investido=("valor_total_investido", "sum"),
        caixa_residual_total=("caixa_residual_final_aporte", "sum"),
        pct_medio_orcamento_investido=("pct_orcamento_investido", "mean"),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id", "ano"])
    .reset_index(drop=True)
)

print(f"Inviabilidades ticker a ticker identificadas : {len(df_inviabilidades_quantidade_minima_ticker):,}")
print(f"Inconsistências por aporte identificadas     : {len(df_inconsistencias_quantidades_alvo_aporte):,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_conversao_orcamento_quantidades, caminho_tbl_regras_conversao_orcamento_quantidades, index=False)
salvar_dataframe(df_quantidades_alvo_aporte, caminho_base_quantidades_alvo_aporte, index=False)
salvar_dataframe(df_auditoria_quantidades_alvo_aporte, caminho_tbl_auditoria_quantidades_alvo_aporte, index=False)
salvar_dataframe(df_resumo_quantidades_alvo_estrategia, caminho_tbl_resumo_quantidades_alvo_estrategia, index=False)
salvar_dataframe(df_inviabilidades_quantidade_minima_ticker, caminho_tbl_inviabilidades_quantidade_minima_ticker, index=False)
salvar_dataframe(df_inconsistencias_quantidades_alvo_aporte, caminho_tbl_inconsistencias_quantidades_alvo_aporte, index=False)
salvar_dataframe(df_distribuicao_anual_quantidades_alvo, caminho_tbl_distribuicao_anual_quantidades_alvo, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_conversao_orcamento_quantidades = df_regras_conversao_orcamento_quantidades.copy()
df_amostra_quantidades_alvo_aporte = df_quantidades_alvo_aporte.head(20).copy()
df_amostra_auditoria_quantidades_alvo_aporte = df_auditoria_quantidades_alvo_aporte.head(20).copy()
df_amostra_resumo_quantidades_alvo_estrategia = df_resumo_quantidades_alvo_estrategia.copy()
df_amostra_inviabilidades_quantidade_minima_ticker = df_inviabilidades_quantidade_minima_ticker.head(20).copy()
df_amostra_inconsistencias_quantidades_alvo_aporte = df_inconsistencias_quantidades_alvo_aporte.head(20).copy()
df_amostra_distribuicao_anual_quantidades_alvo = df_distribuicao_anual_quantidades_alvo.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais de conversão do orçamento em quantidades:")
print(df_amostra_regras_conversao_orcamento_quantidades.to_string(index=False))

print("\nBase de quantidades-alvo por aporte - amostra:")
print(df_amostra_quantidades_alvo_aporte.to_string(index=False))

print("\nAuditoria de quantidades por aporte - amostra:")
print(df_amostra_auditoria_quantidades_alvo_aporte.to_string(index=False))

print("\nResumo de quantidades por estratégia:")
print(df_amostra_resumo_quantidades_alvo_estrategia.to_string(index=False))

print("\nDistribuição anual de quantidades - amostra:")
print(df_amostra_distribuicao_anual_quantidades_alvo.to_string(index=False))

print("\nInviabilidades quantidade mínima ticker a ticker - amostra:")
print(df_amostra_inviabilidades_quantidade_minima_ticker.to_string(index=False))

print("\nInconsistências de quantidades por aporte - amostra:")
print(df_amostra_inconsistencias_quantidades_alvo_aporte.to_string(index=False))

print("\nArquivos salvos na subetapa 8.3:")
print(f"- {caminho_tbl_regras_conversao_orcamento_quantidades}")
print(f"- {caminho_base_quantidades_alvo_aporte}")
print(f"- {caminho_tbl_auditoria_quantidades_alvo_aporte}")
print(f"- {caminho_tbl_resumo_quantidades_alvo_estrategia}")
print(f"- {caminho_tbl_inviabilidades_quantidade_minima_ticker}")
print(f"- {caminho_tbl_inconsistencias_quantidades_alvo_aporte}")
print(f"- {caminho_tbl_distribuicao_anual_quantidades_alvo}")

print("\nETAPA 8.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 8.3 - REGRA DE CONVERSÃO DO ORÇAMENTO EM QUANTIDADES

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - base pesos-alvo equal weight         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_2_base_pesos_alvo_equal_weight_aporte.parquet
Entrada - auditoria equal weight               : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_2_tbl_auditoria_equal_weight_aporte.parquet
Saída   - regras de conversão                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_3_tbl_regras_conversao_orcamento_quantidades.parquet
Saída   - base quantidades-alvo por aporte     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euf

## Etapa 8.4) Regra de Tratamento do Caixa

In [38]:
%%time
# ============================================================
# Etapa 8.4) Regra de Tratamento do Caixa Completo
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 8.4 - REGRA DE TRATAMENTO DO CAIXA COMPLETO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_cdi_padronizada = gerar_caminho_arquivo(
    etapa=1,
    subetapa=3,
    tipo_arquivo="base",
    nome="cdi_padronizada",
)

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_quantidades_alvo_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="base",
    nome="quantidades_alvo_aporte",
)

caminho_tbl_auditoria_quantidades_alvo_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="auditoria_quantidades_alvo_aporte",
)

# Os nomes físicos abaixo são mantidos para compatibilidade com as etapas 8.5 e 9.x.
# O conteúdo passa a representar caixa completo: capital inicial, realocações para ações,
# caixa residual e remuneração diária pelo CDI até o fim do backtest.
caminho_tbl_regras_tratamento_caixa_residual = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="regras_tratamento_caixa_residual",
)

caminho_base_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="caixa_residual_aporte",
)

caminho_base_movimentacao_caixa_diaria = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="movimentacao_caixa_diaria",
)

caminho_tbl_auditoria_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="auditoria_caixa_residual_aporte",
)

caminho_tbl_resumo_caixa_residual_estrategia = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="resumo_caixa_residual_estrategia",
)

caminho_tbl_inconsistencias_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="inconsistencias_caixa_residual_aporte",
)

caminho_tbl_distribuicao_anual_caixa_residual = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_caixa_residual",
)

print(f"Entrada - base CDI padronizada               : {caminho_base_cdi_padronizada}")
print(f"Entrada - mercado diário consolidado         : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - base quantidades-alvo por aporte   : {caminho_base_quantidades_alvo_aporte}")
print(f"Entrada - auditoria quantidades por aporte   : {caminho_tbl_auditoria_quantidades_alvo_aporte}")
print(f"Saída   - regras de tratamento do caixa      : {caminho_tbl_regras_tratamento_caixa_residual}")
print(f"Saída   - base de caixa por aporte           : {caminho_base_caixa_residual_aporte}")
print(f"Saída   - base de movimentação diária caixa  : {caminho_base_movimentacao_caixa_diaria}")
print(f"Saída   - auditoria de caixa por aporte      : {caminho_tbl_auditoria_caixa_residual_aporte}")
print(f"Saída   - resumo de caixa por estratégia     : {caminho_tbl_resumo_caixa_residual_estrategia}")
print(f"Saída   - inconsistências de caixa           : {caminho_tbl_inconsistencias_caixa_residual_aporte}")
print(f"Saída   - distribuição anual do caixa        : {caminho_tbl_distribuicao_anual_caixa_residual}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_cdi_padronizada = pd.read_parquet(caminho_base_cdi_padronizada)
df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_quantidades_alvo_aporte = pd.read_parquet(caminho_base_quantidades_alvo_aporte)
df_auditoria_quantidades_alvo_aporte = pd.read_parquet(caminho_tbl_auditoria_quantidades_alvo_aporte)

print(f"Base CDI padronizada                  : {df_cdi_padronizada.shape[0]:,} linhas x {df_cdi_padronizada.shape[1]} colunas")
print(f"Mercado diário consolidado            : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Base quantidades-alvo por aporte      : {df_quantidades_alvo_aporte.shape[0]:,} linhas x {df_quantidades_alvo_aporte.shape[1]} colunas")
print(f"Auditoria quantidades por aporte      : {df_auditoria_quantidades_alvo_aporte.shape[0]:,} linhas x {df_auditoria_quantidades_alvo_aporte.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_NUMERICA_CAIXA = 0.000001

COLUNAS_MINIMAS_BASE_QUANTIDADES = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "origem_calendario",
    "grupo_selecao_ativos",
    "benchmark_renda_variavel",
    "regra_selecao_cesta",
    "versao_cesta",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "capital_inicial_estrategia",
    "valor_aporte_estrategia",
    "ticker",
    "quantidade_alvo_convertida",
    "flag_compra_efetivada",
    "valor_investido_ticker",
    "caixa_nao_investido_ticker",
]

COLUNAS_MINIMAS_AUDITORIA_8_3 = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "origem_calendario",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "valor_aporte_estrategia",
    "valor_total_investido",
    "caixa_residual_final_aporte",
    "flag_conservacao_orcamento",
    "flag_compra_minima_existe_no_aporte",
    "flag_benchmark_equivalente",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

def detectar_coluna(df, candidatos, obrigatoria=True):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna

    if obrigatoria:
        raise ValueError(f"Não foi possível localizar nenhuma das colunas candidatas: {candidatos}")

    return None

def padronizar_texto(df, colunas_texto, lower_cols=None, upper_cols=None):
    lower_cols = [] if lower_cols is None else lower_cols
    upper_cols = [] if upper_cols is None else upper_cols

    for coluna in colunas_texto:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()

    for coluna in lower_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.lower()

    for coluna in upper_cols:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.upper()

    return df

def preencher_estrategia_referencia(df):
    df = garantir_coluna(df, "estrategia_referencia")
    df["estrategia_referencia"] = df["estrategia_referencia"].astype("string").str.strip().str.lower()

    mask_vazia = df["estrategia_referencia"].isna() | df["estrategia_referencia"].isin(["", "<na>", "nan", "none", "null"])

    if "tipo_sinal" in df.columns:
        tipo_sinal = df["tipo_sinal"].astype("string").str.strip().str.lower()
        df.loc[mask_vazia & tipo_sinal.isin(["capitulacao", "euforia"]), "estrategia_referencia"] = tipo_sinal
        mask_vazia = df["estrategia_referencia"].isna() | df["estrategia_referencia"].isin(["", "<na>", "nan", "none", "null"])

    if "tipo_estrategia" in df.columns:
        tipo_estrategia = df["tipo_estrategia"].astype("string").str.strip().str.lower()
        df.loc[mask_vazia & tipo_estrategia.isin(["capitulacao", "euforia"]), "estrategia_referencia"] = tipo_estrategia
        mask_vazia = df["estrategia_referencia"].isna() | df["estrategia_referencia"].isin(["", "<na>", "nan", "none", "null"])

    if "estrategia_id" in df.columns:
        df.loc[mask_vazia, "estrategia_referencia"] = df.loc[mask_vazia, "estrategia_id"].astype("string").str.strip().str.lower()

    return df

def normalizar_taxa_cdi_diaria(serie_valor):
    serie = pd.to_numeric(serie_valor, errors="coerce").fillna(0.0)
    return serie / 100.0

def obter_primeiro_valor_nao_nulo(serie):
    serie_limpa = pd.Series(serie).dropna()
    if serie_limpa.empty:
        return pd.NA
    return serie_limpa.iloc[0]

def obter_ultimo_valor_nao_nulo(serie):
    serie_limpa = pd.Series(serie).dropna()
    if serie_limpa.empty:
        return pd.NA
    return serie_limpa.iloc[-1]

print(f"Tolerância numérica do caixa : {TOLERANCIA_NUMERICA_CAIXA}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

validar_colunas_obrigatorias(df_quantidades_alvo_aporte, COLUNAS_MINIMAS_BASE_QUANTIDADES)
validar_colunas_obrigatorias(df_auditoria_quantidades_alvo_aporte, COLUNAS_MINIMAS_AUDITORIA_8_3)

for coluna in [
    "carteira_id",
    "replica_id",
    "ano",
    "mes",
    "n_tickers_cesta_equal_weight",
    "flag_quantidade_fracionaria_permitida",
    "flag_preco_valido_equal_weight",
    "flag_recebeu_redistribuicao",
    "estrategia_referencia",
]:
    df_quantidades_alvo_aporte = garantir_coluna(df_quantidades_alvo_aporte, coluna)

for coluna in [
    "carteira_id",
    "replica_id",
    "n_tickers_redistribuidos",
    "pct_orcamento_investido",
    "estrategia_referencia",
]:
    df_auditoria_quantidades_alvo_aporte = garantir_coluna(df_auditoria_quantidades_alvo_aporte, coluna)

coluna_data_cdi = detectar_coluna(df_cdi_padronizada, ["data", "date"])
coluna_valor_cdi = detectar_coluna(df_cdi_padronizada, ["valor", "value"])
coluna_data_mercado = detectar_coluna(df_mercado_diario_consolidada, ["data", "date", "dt_ref"])

if coluna_data_cdi != "data" or coluna_valor_cdi != "valor":
    df_cdi_padronizada = df_cdi_padronizada.rename(columns={coluna_data_cdi: "data", coluna_valor_cdi: "valor"})

if coluna_data_mercado != "data":
    df_mercado_diario_consolidada = df_mercado_diario_consolidada.rename(columns={coluna_data_mercado: "data"})

df_cdi_padronizada["data"] = pd.to_datetime(df_cdi_padronizada["data"], errors="coerce")
df_cdi_padronizada["valor"] = pd.to_numeric(df_cdi_padronizada["valor"], errors="coerce")

df_cdi_padronizada = (
    df_cdi_padronizada[["data", "valor"]]
    .dropna(subset=["data"])
    .drop_duplicates(subset=["data"], keep="last")
    .sort_values("data")
    .reset_index(drop=True)
)

df_cdi_padronizada["taxa_cdi_dia"] = normalizar_taxa_cdi_diaria(df_cdi_padronizada["valor"])

df_mercado_diario_consolidada["data"] = pd.to_datetime(df_mercado_diario_consolidada["data"], errors="coerce")
df_mercado_diario_consolidada = (
    df_mercado_diario_consolidada[["data"]]
    .dropna(subset=["data"])
    .drop_duplicates()
    .sort_values("data")
    .reset_index(drop=True)
)

for df_tmp in [df_quantidades_alvo_aporte, df_auditoria_quantidades_alvo_aporte]:
    df_tmp["data_sinal"] = pd.to_datetime(df_tmp["data_sinal"], errors="coerce")
    df_tmp["data_aporte_efetiva"] = pd.to_datetime(df_tmp["data_aporte_efetiva"], errors="coerce")
    df_tmp["ordem_aporte_estrategia"] = pd.to_numeric(df_tmp["ordem_aporte_estrategia"], errors="coerce")
    df_tmp["capital_inicial_estrategia"] = pd.to_numeric(df_tmp.get("capital_inicial_estrategia", pd.Series(pd.NA, index=df_tmp.index)), errors="coerce")
    df_tmp["valor_aporte_estrategia"] = pd.to_numeric(df_tmp["valor_aporte_estrategia"], errors="coerce")

for coluna in [
    "quantidade_alvo_convertida",
    "valor_investido_ticker",
    "caixa_nao_investido_ticker",
    "n_tickers_cesta_equal_weight",
    "ano",
    "mes",
]:
    if coluna in df_quantidades_alvo_aporte.columns:
        df_quantidades_alvo_aporte[coluna] = pd.to_numeric(df_quantidades_alvo_aporte[coluna], errors="coerce")

for coluna in ["valor_total_investido", "caixa_residual_final_aporte", "pct_orcamento_investido"]:
    if coluna in df_auditoria_quantidades_alvo_aporte.columns:
        df_auditoria_quantidades_alvo_aporte[coluna] = pd.to_numeric(df_auditoria_quantidades_alvo_aporte[coluna], errors="coerce")

df_quantidades_alvo_aporte = padronizar_texto(
    df_quantidades_alvo_aporte,
    colunas_texto=[
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "origem_calendario",
        "grupo_selecao_ativos",
        "benchmark_renda_variavel",
        "regra_selecao_cesta",
        "versao_cesta",
        "estrategia_referencia",
    ],
    lower_cols=[
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "origem_calendario",
        "grupo_selecao_ativos",
        "regra_selecao_cesta",
        "versao_cesta",
        "estrategia_referencia",
    ],
)

df_auditoria_quantidades_alvo_aporte = padronizar_texto(
    df_auditoria_quantidades_alvo_aporte,
    colunas_texto=[
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "origem_calendario",
        "estrategia_referencia",
    ],
    lower_cols=[
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "origem_calendario",
        "estrategia_referencia",
    ],
)

df_quantidades_alvo_aporte = preencher_estrategia_referencia(df_quantidades_alvo_aporte)
df_auditoria_quantidades_alvo_aporte = preencher_estrategia_referencia(df_auditoria_quantidades_alvo_aporte)

duplicidades_request_ticker = int(
    df_quantidades_alvo_aporte.duplicated(subset=["request_id", "ticker"]).sum()
)

if duplicidades_request_ticker > 0:
    raise ValueError(f"Foram encontradas {duplicidades_request_ticker} duplicidades por request_id+ticker na base de quantidades da etapa 8.3.")

df_inconsistencias_herdadas_8_3 = (
    df_auditoria_quantidades_alvo_aporte
    .loc[
        (~df_auditoria_quantidades_alvo_aporte["flag_conservacao_orcamento"].fillna(False))
        | (~df_auditoria_quantidades_alvo_aporte["flag_compra_minima_existe_no_aporte"].fillna(False))
        | (df_auditoria_quantidades_alvo_aporte["caixa_residual_final_aporte"] < -TOLERANCIA_NUMERICA_CAIXA)
    ]
    .copy()
)

DATA_INICIO_BACKTEST_CAIXA = pd.Timestamp(df_quantidades_alvo_aporte["data_aporte_efetiva"].min())
DATA_FIM_BACKTEST_CAIXA = pd.Timestamp(df_mercado_diario_consolidada["data"].max())

if pd.isna(DATA_INICIO_BACKTEST_CAIXA) or pd.isna(DATA_FIM_BACKTEST_CAIXA):
    raise ValueError("Não foi possível determinar as datas de início e fim da trilha de caixa.")

if DATA_FIM_BACKTEST_CAIXA < DATA_INICIO_BACKTEST_CAIXA:
    raise ValueError("A data final da trilha de caixa é anterior à data inicial.")

df_calendario_caixa = (
    df_mercado_diario_consolidada
    .loc[
        (df_mercado_diario_consolidada["data"] >= DATA_INICIO_BACKTEST_CAIXA)
        & (df_mercado_diario_consolidada["data"] <= DATA_FIM_BACKTEST_CAIXA),
        ["data"],
    ]
    .drop_duplicates()
    .sort_values("data")
    .reset_index(drop=True)
    .merge(df_cdi_padronizada[["data", "taxa_cdi_dia"]], on="data", how="left")
)

df_calendario_caixa["taxa_cdi_dia"] = pd.to_numeric(df_calendario_caixa["taxa_cdi_dia"], errors="coerce").fillna(0.0)

print(f"Inconsistências herdadas da 8.3              : {len(df_inconsistencias_herdadas_8_3):,}")
print(f"Duplicidades request+ticker                  : {duplicidades_request_ticker:,}")
print(f"Data inicial da trilha de caixa              : {DATA_INICIO_BACKTEST_CAIXA.date()}")
print(f"Data final da trilha de caixa                : {DATA_FIM_BACKTEST_CAIXA.date()}")
print(f"Datas operacionais na trilha de caixa        : {len(df_calendario_caixa):,}")
print(f"Primeira data do CDI padronizado             : {df_cdi_padronizada['data'].min().date()}")
print(f"Última data do CDI padronizado               : {df_cdi_padronizada['data'].max().date()}")
print("OK")

# ============================================================
# 6) Construção da base de caixa completo por aporte e por dia
# ============================================================

print("\n[6/10] Construção da base de caixa completo por aporte e por dia...")

coluna_flag_preco_valido = detectar_coluna(
    df_quantidades_alvo_aporte,
    ["flag_preco_valido_conversao", "flag_preco_valido_equal_weight"],
    obrigatoria=False,
)

agregacoes_requests = {
    "estrategia_id": ("estrategia_id", "first"),
    "grupo_controle": ("grupo_controle", "first"),
    "familia_estrategia": ("familia_estrategia", "first"),
    "estrategia_referencia": ("estrategia_referencia", "first"),
    "tipo_estrategia": ("tipo_estrategia", "first"),
    "origem_calendario": ("origem_calendario", "first"),
    "grupo_selecao_ativos": ("grupo_selecao_ativos", "first"),
    "benchmark_renda_variavel": ("benchmark_renda_variavel", "first"),
    "regra_selecao_cesta": ("regra_selecao_cesta", "first"),
    "versao_cesta": ("versao_cesta", "first"),
    "carteira_id": ("carteira_id", "first"),
    "replica_id": ("replica_id", "first"),
    "data_sinal": ("data_sinal", "first"),
    "data_aporte_efetiva": ("data_aporte_efetiva", "first"),
    "ordem_aporte_estrategia": ("ordem_aporte_estrategia", "first"),
    "ano": ("ano", "first"),
    "mes": ("mes", "first"),
    "capital_inicial_estrategia": ("capital_inicial_estrategia", "first"),
    "valor_aporte_estrategia": ("valor_aporte_estrategia", "first"),
    "n_tickers_cesta_equal_weight": ("n_tickers_cesta_equal_weight", "first"),
    "n_tickers_quantidade_fracionaria": ("flag_quantidade_fracionaria_permitida", lambda x: int(pd.Series(x).fillna(False).sum())),
    "n_tickers_compra_efetivada": ("flag_compra_efetivada", lambda x: int(pd.Series(x).fillna(False).sum())),
    "n_tickers_quantidade_zero": ("quantidade_alvo_convertida", lambda x: int((pd.to_numeric(pd.Series(x), errors="coerce").fillna(0.0) <= 0).sum())),
    "flag_quantidade_total_nao_negativa": ("quantidade_alvo_convertida", lambda x: bool((pd.to_numeric(pd.Series(x), errors="coerce").fillna(0.0) >= -TOLERANCIA_NUMERICA_CAIXA).all())),
}

if coluna_flag_preco_valido is not None:
    agregacoes_requests["flag_precos_validos_para_toda_cesta"] = (coluna_flag_preco_valido, "min")
else:
    agregacoes_requests["flag_precos_validos_para_toda_cesta"] = ("request_id", lambda x: True)

df_agregado_requests = (
    df_quantidades_alvo_aporte
    .groupby("request_id", as_index=False, dropna=False)
    .agg(**agregacoes_requests)
)

colunas_auditoria_merge = [
    "request_id",
    "valor_total_investido",
    "caixa_residual_final_aporte",
    "pct_orcamento_investido",
    "flag_conservacao_orcamento",
    "flag_compra_minima_existe_no_aporte",
    "flag_benchmark_equivalente",
    "n_tickers_redistribuidos",
]

for coluna in colunas_auditoria_merge:
    df_auditoria_quantidades_alvo_aporte = garantir_coluna(df_auditoria_quantidades_alvo_aporte, coluna)

df_eventos_aporte_base = df_agregado_requests.merge(
    df_auditoria_quantidades_alvo_aporte[colunas_auditoria_merge],
    on="request_id",
    how="left",
)

df_eventos_aporte_base["capital_inicial_estrategia"] = pd.to_numeric(df_eventos_aporte_base["capital_inicial_estrategia"], errors="coerce")
df_eventos_aporte_base["valor_aporte_estrategia"] = pd.to_numeric(df_eventos_aporte_base["valor_aporte_estrategia"], errors="coerce")
df_eventos_aporte_base["valor_total_investido"] = pd.to_numeric(df_eventos_aporte_base["valor_total_investido"], errors="coerce").fillna(0.0)
df_eventos_aporte_base["caixa_residual_final_aporte"] = pd.to_numeric(df_eventos_aporte_base["caixa_residual_final_aporte"], errors="coerce").fillna(0.0)
df_eventos_aporte_base["pct_orcamento_investido"] = pd.to_numeric(df_eventos_aporte_base["pct_orcamento_investido"], errors="coerce")

df_eventos_aporte_base = (
    df_eventos_aporte_base
    .sort_values(["estrategia_id", "data_aporte_efetiva", "ordem_aporte_estrategia", "request_id"])
    .reset_index(drop=True)
)

lista_movimentacao_caixa = []
lista_eventos_caixa = []

for estrategia_id, df_eventos_estrategia in df_eventos_aporte_base.groupby("estrategia_id", sort=False, dropna=False):
    df_eventos_estrategia = (
        df_eventos_estrategia
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia", "request_id"])
        .reset_index(drop=True)
    )

    capital_inicial_estrategia = float(pd.to_numeric(df_eventos_estrategia["capital_inicial_estrategia"], errors="coerce").dropna().iloc[0])
    grupo_controle = obter_primeiro_valor_nao_nulo(df_eventos_estrategia["grupo_controle"])
    familia_estrategia = obter_primeiro_valor_nao_nulo(df_eventos_estrategia["familia_estrategia"])
    estrategia_referencia = obter_primeiro_valor_nao_nulo(df_eventos_estrategia["estrategia_referencia"])
    tipo_estrategia = obter_primeiro_valor_nao_nulo(df_eventos_estrategia["tipo_estrategia"])
    carteira_id = obter_primeiro_valor_nao_nulo(df_eventos_estrategia["carteira_id"])
    replica_id = obter_primeiro_valor_nao_nulo(df_eventos_estrategia["replica_id"])

    mapa_eventos_por_data = {
        pd.Timestamp(data): df_data.sort_values(["ordem_aporte_estrategia", "request_id"]).reset_index(drop=True)
        for data, df_data in df_eventos_estrategia.groupby("data_aporte_efetiva", sort=False, dropna=False)
    }

    saldo_caixa_inicio_dia = float(capital_inicial_estrategia)

    for linha_cal in df_calendario_caixa.itertuples(index=False):
        data_atual = pd.Timestamp(linha_cal.data)
        taxa_cdi_dia = float(linha_cal.taxa_cdi_dia)

        saldo_antes_movimentacao_dia = float(saldo_caixa_inicio_dia)
        saldo_caixa_corrente = float(saldo_antes_movimentacao_dia)

        valor_aporte_planejado_dia = 0.0
        valor_investido_dia = 0.0
        valor_caixa_residual_dia = 0.0
        request_ids_dia = []
        ordens_aporte_dia = []
        flag_evento_aporte = False
        flag_investimento_coberto_por_caixa_dia = True

        if data_atual in mapa_eventos_por_data:
            df_eventos_data = mapa_eventos_por_data[data_atual]
            flag_evento_aporte = True

            for evento in df_eventos_data.itertuples(index=False):
                valor_aporte_evento = float(evento.valor_aporte_estrategia)
                valor_investido_evento = float(evento.valor_total_investido)
                valor_caixa_residual_evento = float(evento.caixa_residual_final_aporte)
                saldo_caixa_antes_aporte = float(saldo_caixa_corrente)
                flag_investimento_coberto_por_caixa = bool(valor_investido_evento <= saldo_caixa_antes_aporte + TOLERANCIA_NUMERICA_CAIXA)
                saldo_caixa_apos_investimento = float(saldo_caixa_antes_aporte - valor_investido_evento)

                if not flag_investimento_coberto_por_caixa:
                    flag_investimento_coberto_por_caixa_dia = False

                lista_eventos_caixa.append(
                    {
                        "request_id": evento.request_id,
                        "estrategia_id": estrategia_id,
                        "grupo_controle": evento.grupo_controle,
                        "familia_estrategia": evento.familia_estrategia,
                        "estrategia_referencia": evento.estrategia_referencia,
                        "tipo_estrategia": evento.tipo_estrategia,
                        "origem_calendario": evento.origem_calendario,
                        "grupo_selecao_ativos": evento.grupo_selecao_ativos,
                        "benchmark_renda_variavel": evento.benchmark_renda_variavel,
                        "regra_selecao_cesta": evento.regra_selecao_cesta,
                        "versao_cesta": evento.versao_cesta,
                        "carteira_id": evento.carteira_id,
                        "replica_id": evento.replica_id,
                        "data_sinal": evento.data_sinal,
                        "data_aporte_efetiva": evento.data_aporte_efetiva,
                        "ordem_aporte_estrategia": evento.ordem_aporte_estrategia,
                        "ano": evento.ano,
                        "mes": evento.mes,
                        "capital_inicial_estrategia": capital_inicial_estrategia,
                        "valor_aporte_estrategia": valor_aporte_evento,
                        "valor_total_investido": valor_investido_evento,
                        "caixa_residual_final_aporte": valor_caixa_residual_evento,
                        "pct_orcamento_investido": evento.pct_orcamento_investido,
                        "flag_conservacao_orcamento": evento.flag_conservacao_orcamento,
                        "flag_compra_minima_existe_no_aporte": evento.flag_compra_minima_existe_no_aporte,
                        "flag_benchmark_equivalente": evento.flag_benchmark_equivalente,
                        "n_tickers_redistribuidos": evento.n_tickers_redistribuidos,
                        "n_tickers_cesta_equal_weight": evento.n_tickers_cesta_equal_weight,
                        "n_tickers_quantidade_fracionaria": evento.n_tickers_quantidade_fracionaria,
                        "n_tickers_compra_efetivada": evento.n_tickers_compra_efetivada,
                        "n_tickers_quantidade_zero": evento.n_tickers_quantidade_zero,
                        "flag_precos_validos_para_toda_cesta": evento.flag_precos_validos_para_toda_cesta,
                        "flag_quantidade_total_nao_negativa": evento.flag_quantidade_total_nao_negativa,
                        "saldo_caixa_antes_aporte": saldo_caixa_antes_aporte,
                        "valor_caixa_trazido_aporte": saldo_caixa_antes_aporte,
                        "saldo_caixa_inicial_periodo": saldo_caixa_antes_aporte,
                        "saldo_caixa_apos_investimento_aporte": saldo_caixa_apos_investimento,
                        "flag_investimento_coberto_por_caixa": flag_investimento_coberto_por_caixa,
                    }
                )

                saldo_caixa_corrente = float(saldo_caixa_apos_investimento)
                valor_aporte_planejado_dia += valor_aporte_evento
                valor_investido_dia += valor_investido_evento
                valor_caixa_residual_dia += valor_caixa_residual_evento
                request_ids_dia.append(str(evento.request_id))
                ordens_aporte_dia.append(float(evento.ordem_aporte_estrategia))

        saldo_caixa_apos_movimentacao_dia = float(saldo_caixa_corrente)
        rendimento_cdi_dia = float(saldo_caixa_apos_movimentacao_dia * taxa_cdi_dia)
        saldo_caixa_fim_dia = float(saldo_caixa_apos_movimentacao_dia + rendimento_cdi_dia)

        lista_movimentacao_caixa.append(
            {
                "data": data_atual,
                "request_id": ";".join(request_ids_dia) if len(request_ids_dia) > 0 else pd.NA,
                "estrategia_id": estrategia_id,
                "grupo_controle": grupo_controle,
                "familia_estrategia": familia_estrategia,
                "estrategia_referencia": estrategia_referencia,
                "tipo_estrategia": tipo_estrategia,
                "carteira_id": carteira_id,
                "replica_id": replica_id,
                "ordem_aporte_estrategia": max(ordens_aporte_dia) if len(ordens_aporte_dia) > 0 else pd.NA,
                "capital_inicial_estrategia": capital_inicial_estrategia,
                "valor_aporte_estrategia": valor_aporte_planejado_dia,
                "valor_investido_efetivo_aporte": valor_investido_dia,
                "valor_caixa_residual_aporte": valor_caixa_residual_dia,
                "valor_caixa_trazido_aporte": saldo_antes_movimentacao_dia if flag_evento_aporte else 0.0,
                "saldo_caixa_inicio_dia": saldo_antes_movimentacao_dia,
                "saldo_caixa_apos_movimentacao_dia": saldo_caixa_apos_movimentacao_dia,
                "saldo_caixa_fim_dia": saldo_caixa_fim_dia,
                "saldo_caixa": saldo_caixa_fim_dia,
                "rendimento_cdi_dia": rendimento_cdi_dia,
                "taxa_cdi_dia": taxa_cdi_dia,
                "flag_evento_aporte": flag_evento_aporte,
                "flag_investimento_coberto_por_caixa_dia": flag_investimento_coberto_por_caixa_dia,
                "flag_caixa_nao_negativo_dia": saldo_caixa_fim_dia >= -TOLERANCIA_NUMERICA_CAIXA,
            }
        )

        saldo_caixa_inicio_dia = float(saldo_caixa_fim_dia)

df_movimentacao_caixa_diaria = pd.DataFrame(lista_movimentacao_caixa)
df_eventos_caixa = pd.DataFrame(lista_eventos_caixa)

if df_eventos_caixa.empty:
    raise ValueError("A base de eventos de caixa ficou vazia após a construção da etapa 8.4.")

if df_movimentacao_caixa_diaria.empty:
    raise ValueError("A base diária de caixa ficou vazia após a construção da etapa 8.4.")

df_eventos_caixa = (
    df_eventos_caixa
    .sort_values(["estrategia_id", "data_aporte_efetiva", "ordem_aporte_estrategia", "request_id"])
    .reset_index(drop=True)
)

df_movimentacao_caixa_diaria = (
    df_movimentacao_caixa_diaria
    .sort_values(["estrategia_id", "data"])
    .reset_index(drop=True)
)

df_eventos_caixa["data_proximo_aporte_estrategia"] = df_eventos_caixa.groupby("estrategia_id")["data_aporte_efetiva"].shift(-1)

df_eventos_caixa["data_inicio_remuneracao_cdi"] = pd.NaT
df_eventos_caixa["data_fim_remuneracao_cdi"] = pd.NaT
df_eventos_caixa["n_dias_cdi_periodo"] = 0
df_eventos_caixa["fator_acumulado_cdi_periodo"] = 1.0
df_eventos_caixa["rendimento_cdi_periodo"] = 0.0
df_eventos_caixa["saldo_caixa_final_periodo"] = 0.0

for idx, evento in df_eventos_caixa.iterrows():
    estrategia_id = evento["estrategia_id"]
    data_aporte = pd.Timestamp(evento["data_aporte_efetiva"])
    data_proximo = evento["data_proximo_aporte_estrategia"]

    if pd.notna(data_proximo):
        mask_periodo = (
            (df_movimentacao_caixa_diaria["estrategia_id"] == estrategia_id)
            & (df_movimentacao_caixa_diaria["data"] >= data_aporte)
            & (df_movimentacao_caixa_diaria["data"] < pd.Timestamp(data_proximo))
        )
    else:
        mask_periodo = (
            (df_movimentacao_caixa_diaria["estrategia_id"] == estrategia_id)
            & (df_movimentacao_caixa_diaria["data"] >= data_aporte)
            & (df_movimentacao_caixa_diaria["data"] <= DATA_FIM_BACKTEST_CAIXA)
        )

    df_periodo = df_movimentacao_caixa_diaria.loc[mask_periodo].copy().sort_values("data")

    if df_periodo.empty:
        continue

    df_eventos_caixa.loc[idx, "data_inicio_remuneracao_cdi"] = pd.Timestamp(df_periodo["data"].min())
    df_eventos_caixa.loc[idx, "data_fim_remuneracao_cdi"] = pd.Timestamp(df_periodo["data"].max())
    df_eventos_caixa.loc[idx, "n_dias_cdi_periodo"] = int(len(df_periodo))
    df_eventos_caixa.loc[idx, "fator_acumulado_cdi_periodo"] = float(np.prod((1.0 + pd.to_numeric(df_periodo["taxa_cdi_dia"], errors="coerce").fillna(0.0)).to_numpy(dtype=float)))
    df_eventos_caixa.loc[idx, "rendimento_cdi_periodo"] = float(pd.to_numeric(df_periodo["rendimento_cdi_dia"], errors="coerce").fillna(0.0).sum())
    df_eventos_caixa.loc[idx, "saldo_caixa_final_periodo"] = float(pd.to_numeric(df_periodo["saldo_caixa_fim_dia"], errors="coerce").iloc[-1])

df_caixa_residual_aporte = df_eventos_caixa.copy()

df_caixa_residual_aporte["pct_investido_efetivo_aporte"] = np.where(
    df_caixa_residual_aporte["valor_aporte_estrategia"] > 0,
    df_caixa_residual_aporte["valor_total_investido"] / df_caixa_residual_aporte["valor_aporte_estrategia"],
    0.0,
)

df_caixa_residual_aporte["pct_caixa_residual_aporte"] = np.where(
    df_caixa_residual_aporte["valor_aporte_estrategia"] > 0,
    df_caixa_residual_aporte["caixa_residual_final_aporte"] / df_caixa_residual_aporte["valor_aporte_estrategia"],
    0.0,
)

df_caixa_residual_aporte["valor_aportes_acumulado_estrategia"] = (
    df_caixa_residual_aporte.groupby("estrategia_id")["valor_aporte_estrategia"].cumsum()
)

df_caixa_residual_aporte["valor_investido_acumulado_estrategia"] = (
    df_caixa_residual_aporte.groupby("estrategia_id")["valor_total_investido"].cumsum()
)

df_caixa_residual_aporte["valor_caixa_residual_acumulado_bruto_estrategia"] = (
    df_caixa_residual_aporte.groupby("estrategia_id")["caixa_residual_final_aporte"].cumsum()
)

df_caixa_residual_aporte["rendimento_cdi_acumulado_estrategia"] = (
    df_caixa_residual_aporte.groupby("estrategia_id")["rendimento_cdi_periodo"].cumsum()
)

df_caixa_residual_aporte["saldo_caixa_final_acumulado_estrategia"] = df_caixa_residual_aporte["saldo_caixa_final_periodo"]

df_caixa_residual_aporte["pct_saldo_caixa_final_sobre_capital_inicial"] = np.where(
    df_caixa_residual_aporte["capital_inicial_estrategia"] > 0,
    df_caixa_residual_aporte["saldo_caixa_final_acumulado_estrategia"] / df_caixa_residual_aporte["capital_inicial_estrategia"],
    0.0,
)

df_caixa_residual_aporte["flag_caixa_residual_positivo_aporte"] = (
    df_caixa_residual_aporte["caixa_residual_final_aporte"] > TOLERANCIA_NUMERICA_CAIXA
)
df_caixa_residual_aporte["flag_caixa_residual_zero_aporte"] = (
    df_caixa_residual_aporte["caixa_residual_final_aporte"].abs() <= TOLERANCIA_NUMERICA_CAIXA
)
df_caixa_residual_aporte["flag_caixa_trazido_positivo_aporte"] = (
    df_caixa_residual_aporte["valor_caixa_trazido_aporte"] > TOLERANCIA_NUMERICA_CAIXA
)
df_caixa_residual_aporte["flag_saldo_caixa_final_positivo"] = (
    df_caixa_residual_aporte["saldo_caixa_final_periodo"] > TOLERANCIA_NUMERICA_CAIXA
)
df_caixa_residual_aporte["flag_conservacao_orcamento_valida"] = (
    (
        df_caixa_residual_aporte["valor_total_investido"]
        + df_caixa_residual_aporte["caixa_residual_final_aporte"]
        - df_caixa_residual_aporte["valor_aporte_estrategia"]
    ).abs() <= TOLERANCIA_NUMERICA_CAIXA
)
df_caixa_residual_aporte["flag_quantidade_compra_positiva_em_algum_ticker"] = (
    df_caixa_residual_aporte["n_tickers_compra_efetivada"] > 0
)
df_caixa_residual_aporte["flag_caixa_completo_sem_saldo_negativo"] = (
    df_caixa_residual_aporte["saldo_caixa_apos_investimento_aporte"] >= -TOLERANCIA_NUMERICA_CAIXA
)

df_movimentacao_caixa_diaria["valor_investido_acumulado_estrategia"] = (
    df_movimentacao_caixa_diaria.groupby("estrategia_id")["valor_investido_efetivo_aporte"].cumsum()
)
df_movimentacao_caixa_diaria["valor_aporte_planejado_acumulado_estrategia"] = (
    df_movimentacao_caixa_diaria.groupby("estrategia_id")["valor_aporte_estrategia"].cumsum()
)
df_movimentacao_caixa_diaria["rendimento_cdi_acumulado_estrategia"] = (
    df_movimentacao_caixa_diaria.groupby("estrategia_id")["rendimento_cdi_dia"].cumsum()
)
df_movimentacao_caixa_diaria["pct_caixa_final_sobre_capital_inicial"] = np.where(
    df_movimentacao_caixa_diaria["capital_inicial_estrategia"] > 0,
    df_movimentacao_caixa_diaria["saldo_caixa_fim_dia"] / df_movimentacao_caixa_diaria["capital_inicial_estrategia"],
    0.0,
)

print(f"Linhas da base de caixa por aporte          : {len(df_caixa_residual_aporte):,}")
print(f"Linhas da base diária de caixa              : {len(df_movimentacao_caixa_diaria):,}")
print(f"Estratégias com trilha de caixa             : {df_movimentacao_caixa_diaria['estrategia_id'].nunique():,}")
print(f"Data inicial efetiva da base diária         : {df_movimentacao_caixa_diaria['data'].min().date()}")
print(f"Data final efetiva da base diária           : {df_movimentacao_caixa_diaria['data'].max().date()}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais, auditorias e inconsistências do caixa
# ============================================================

print("\n[7/10] Construção das tabelas formais, auditorias e inconsistências do caixa...")

df_regras_tratamento_caixa_residual = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "CX1",
            "escopo": "capital_inicial",
            "regra_operacional": "capital_inicial_integral_inicia_como_caixa_disponivel",
            "detalhe": "Cada estratégia inicia a trilha operacional com o capital inicial integral disponível em caixa.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "CX2",
            "escopo": "aportes",
            "regra_operacional": "aportes_sao_realocacoes_internas_de_caixa_para_acoes",
            "detalhe": "O aporte planejado não entra como fluxo externo; a compra reduz o caixa disponível pelo valor efetivamente investido.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "CX3",
            "escopo": "caixa_residual_aporte",
            "regra_operacional": "caixa_residual_do_aporte_permanece_na_trilha_de_caixa",
            "detalhe": "O valor planejado e não executado por arredondamento de quantidade permanece como caixa da própria estratégia.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "CX4",
            "escopo": "remuneracao_cdi",
            "regra_operacional": "caixa_nao_investido_e_remunerado_diariamente_pelo_cdi",
            "detalhe": "O saldo de caixa após movimentações de compra é remunerado pela taxa diária do CDI em cada data operacional.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "CX5",
            "escopo": "fim_do_backtest",
            "regra_operacional": "caixa_remanescente_permanece_remunerado_ate_a_data_final_do_backtest",
            "detalhe": "Depois do último aporte de cada estratégia, o caixa remanescente continua rendendo CDI até o fim comum da amostra.",
        },
        {
            "ordem_regra": 6,
            "regra_id": "CX6",
            "escopo": "compatibilidade_operacional",
            "regra_operacional": "nomes_fisicos_de_arquivos_legados_sao_mantidos_para_consumo_das_etapas_sequenciais",
            "detalhe": "Os arquivos de saída preservam os nomes esperados pelas etapas seguintes, embora o conteúdo represente caixa completo.",
        },
    ]
)

df_auditoria_caixa_residual_aporte = df_caixa_residual_aporte.copy()

mask_inconsistencias_eventos = (
    (~df_auditoria_caixa_residual_aporte["flag_conservacao_orcamento_valida"].fillna(False))
    | (~df_auditoria_caixa_residual_aporte["flag_quantidade_compra_positiva_em_algum_ticker"].fillna(False))
    | (~df_auditoria_caixa_residual_aporte["flag_precos_validos_para_toda_cesta"].fillna(False))
    | (~df_auditoria_caixa_residual_aporte["flag_quantidade_total_nao_negativa"].fillna(False))
    | (~df_auditoria_caixa_residual_aporte["flag_investimento_coberto_por_caixa"].fillna(False))
    | (~df_auditoria_caixa_residual_aporte["flag_caixa_completo_sem_saldo_negativo"].fillna(False))
    | (df_auditoria_caixa_residual_aporte["saldo_caixa_final_periodo"] < -TOLERANCIA_NUMERICA_CAIXA)
)

mask_inconsistencias_diarias = (
    (~df_movimentacao_caixa_diaria["flag_investimento_coberto_por_caixa_dia"].fillna(False))
    | (~df_movimentacao_caixa_diaria["flag_caixa_nao_negativo_dia"].fillna(False))
)

df_inconsistencias_eventos = df_auditoria_caixa_residual_aporte.loc[mask_inconsistencias_eventos].copy()
df_inconsistencias_diarias = df_movimentacao_caixa_diaria.loc[mask_inconsistencias_diarias].copy()

if not df_inconsistencias_diarias.empty:
    df_inconsistencias_diarias = df_inconsistencias_diarias.assign(
        tipo_inconsistencia="inconsistencia_diaria_caixa",
        data_aporte_efetiva=pd.NaT,
        valor_total_investido=pd.NA,
        caixa_residual_final_aporte=pd.NA,
        saldo_caixa_final_periodo=df_inconsistencias_diarias["saldo_caixa_fim_dia"],
        flag_conservacao_orcamento_valida=pd.NA,
        flag_quantidade_compra_positiva_em_algum_ticker=pd.NA,
    )

if not df_inconsistencias_eventos.empty:
    df_inconsistencias_eventos = df_inconsistencias_eventos.assign(tipo_inconsistencia="inconsistencia_evento_aporte")

colunas_inconsistencias = sorted(set(df_inconsistencias_eventos.columns).union(set(df_inconsistencias_diarias.columns)))

df_inconsistencias_caixa_residual_aporte = pd.concat(
    [
        df_inconsistencias_eventos.reindex(columns=colunas_inconsistencias),
        df_inconsistencias_diarias.reindex(columns=colunas_inconsistencias),
    ],
    axis=0,
    ignore_index=True,
)

ultimo_caixa_por_estrategia = (
    df_movimentacao_caixa_diaria
    .sort_values(["estrategia_id", "data"])
    .groupby("estrategia_id", as_index=False, dropna=False)
    .tail(1)[
        [
            "estrategia_id",
            "saldo_caixa_fim_dia",
            "rendimento_cdi_acumulado_estrategia",
            "valor_investido_acumulado_estrategia",
            "valor_aporte_planejado_acumulado_estrategia",
            "data",
        ]
    ]
    .rename(
        columns={
            "saldo_caixa_fim_dia": "saldo_caixa_final_estrategia",
            "rendimento_cdi_acumulado_estrategia": "valor_total_rendimento_cdi",
            "valor_investido_acumulado_estrategia": "valor_total_investido_trilha_diaria",
            "valor_aporte_planejado_acumulado_estrategia": "valor_total_aportes_planejados_trilha_diaria",
            "data": "data_final_caixa",
        }
    )
)

df_resumo_caixa_residual_estrategia = (
    df_auditoria_caixa_residual_aporte
    .groupby(
        [
            "familia_estrategia",
            "estrategia_referencia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_aportes=("request_id", "count"),
        data_primeiro_aporte=("data_aporte_efetiva", "min"),
        data_ultimo_aporte=("data_aporte_efetiva", "max"),
        capital_inicial_estrategia=("capital_inicial_estrategia", "first"),
        valor_total_aportes=("valor_aporte_estrategia", "sum"),
        valor_total_investido=("valor_total_investido", "sum"),
        valor_total_caixa_residual_bruto=("caixa_residual_final_aporte", "sum"),
        saldo_caixa_minimo_apos_aporte=("saldo_caixa_apos_investimento_aporte", "min"),
        pct_medio_investido_efetivo=("pct_investido_efetivo_aporte", "mean"),
        n_aportes_conservacao_valida=("flag_conservacao_orcamento_valida", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_aportes_cobertos_por_caixa=("flag_investimento_coberto_por_caixa", lambda x: int(pd.Series(x).fillna(False).sum())),
    )
    .merge(ultimo_caixa_por_estrategia, on="estrategia_id", how="left")
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id"])
    .reset_index(drop=True)
)

df_resumo_caixa_residual_estrategia["pct_saldo_caixa_final_sobre_capital_inicial"] = np.where(
    df_resumo_caixa_residual_estrategia["capital_inicial_estrategia"] > 0,
    df_resumo_caixa_residual_estrategia["saldo_caixa_final_estrategia"] / df_resumo_caixa_residual_estrategia["capital_inicial_estrategia"],
    0.0,
)

df_resumo_caixa_residual_estrategia["desvio_aportes_planejados_vs_capital"] = (
    df_resumo_caixa_residual_estrategia["valor_total_aportes"]
    - df_resumo_caixa_residual_estrategia["capital_inicial_estrategia"]
)

df_distribuicao_anual_caixa_residual = (
    df_auditoria_caixa_residual_aporte
    .assign(ano=lambda df: pd.to_datetime(df["data_aporte_efetiva"], errors="coerce").dt.year)
    .groupby(
        [
            "familia_estrategia",
            "estrategia_referencia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_aportes=("request_id", "count"),
        valor_total_aportes=("valor_aporte_estrategia", "sum"),
        valor_total_investido=("valor_total_investido", "sum"),
        valor_total_caixa_residual_bruto=("caixa_residual_final_aporte", "sum"),
        saldo_caixa_final_ano=("saldo_caixa_final_periodo", "last"),
        pct_medio_investido_efetivo=("pct_investido_efetivo_aporte", "mean"),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id", "ano"])
    .reset_index(drop=True)
)

n_saldos_negativos_diarios = int((df_movimentacao_caixa_diaria["saldo_caixa_fim_dia"] < -TOLERANCIA_NUMERICA_CAIXA).sum())
n_aportes_descobertos = int((~df_auditoria_caixa_residual_aporte["flag_investimento_coberto_por_caixa"].fillna(False)).sum())
n_estrategia_referencia_ausente = int(
    df_auditoria_caixa_residual_aporte["estrategia_referencia"].isna().sum()
    + (df_auditoria_caixa_residual_aporte["estrategia_referencia"].astype("string").str.strip().isin(["", "<na>", "nan", "none", "null"])).sum()
)

print(f"Inconsistências de caixa identificadas        : {len(df_inconsistencias_caixa_residual_aporte):,}")
print(f"Saldos diários negativos                      : {n_saldos_negativos_diarios:,}")
print(f"Aportes descobertos pelo caixa                : {n_aportes_descobertos:,}")
print(f"Estrategia_referencia ausente em eventos      : {n_estrategia_referencia_ausente:,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_tratamento_caixa_residual, caminho_tbl_regras_tratamento_caixa_residual, index=False)
salvar_dataframe(df_caixa_residual_aporte, caminho_base_caixa_residual_aporte, index=False)
salvar_dataframe(df_movimentacao_caixa_diaria, caminho_base_movimentacao_caixa_diaria, index=False)
salvar_dataframe(df_auditoria_caixa_residual_aporte, caminho_tbl_auditoria_caixa_residual_aporte, index=False)
salvar_dataframe(df_resumo_caixa_residual_estrategia, caminho_tbl_resumo_caixa_residual_estrategia, index=False)
salvar_dataframe(df_inconsistencias_caixa_residual_aporte, caminho_tbl_inconsistencias_caixa_residual_aporte, index=False)
salvar_dataframe(df_distribuicao_anual_caixa_residual, caminho_tbl_distribuicao_anual_caixa_residual, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_tratamento_caixa_residual = df_regras_tratamento_caixa_residual.copy()
df_amostra_caixa_residual_aporte = df_caixa_residual_aporte.head(20).copy()
df_amostra_movimentacao_caixa_diaria = df_movimentacao_caixa_diaria.head(20).copy()
df_amostra_auditoria_caixa_residual_aporte = df_auditoria_caixa_residual_aporte.head(20).copy()
df_amostra_resumo_caixa_residual_estrategia = df_resumo_caixa_residual_estrategia.copy()
df_amostra_inconsistencias_caixa_residual_aporte = df_inconsistencias_caixa_residual_aporte.head(20).copy()
df_amostra_distribuicao_anual_caixa_residual = df_distribuicao_anual_caixa_residual.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais de tratamento do caixa:")
print(df_amostra_regras_tratamento_caixa_residual.to_string(index=False))

print("\nBase de caixa por aporte - amostra:")
print(df_amostra_caixa_residual_aporte.to_string(index=False))

print("\nBase diária de caixa - amostra:")
print(df_amostra_movimentacao_caixa_diaria.to_string(index=False))

print("\nAuditoria de caixa por aporte - amostra:")
print(df_amostra_auditoria_caixa_residual_aporte.to_string(index=False))

print("\nResumo de caixa por estratégia:")
print(df_amostra_resumo_caixa_residual_estrategia.to_string(index=False))

print("\nDistribuição anual do caixa - amostra:")
print(df_amostra_distribuicao_anual_caixa_residual.to_string(index=False))

print("\nInconsistências de caixa - amostra:")
print(df_amostra_inconsistencias_caixa_residual_aporte.to_string(index=False))

print("\nArquivos salvos na subetapa 8.4:")
print(f"- {caminho_tbl_regras_tratamento_caixa_residual}")
print(f"- {caminho_base_caixa_residual_aporte}")
print(f"- {caminho_base_movimentacao_caixa_diaria}")
print(f"- {caminho_tbl_auditoria_caixa_residual_aporte}")
print(f"- {caminho_tbl_resumo_caixa_residual_estrategia}")
print(f"- {caminho_tbl_inconsistencias_caixa_residual_aporte}")
print(f"- {caminho_tbl_distribuicao_anual_caixa_residual}")

print("\nETAPA 8.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 8.4 - REGRA DE TRATAMENTO DO CAIXA COMPLETO

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - base CDI padronizada               : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_cdi_padronizada.parquet
Entrada - mercado diário consolidado         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - base quantidades-alvo por aporte   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_3_base_quantidades_alvo_aporte.parquet
Entrada - auditoria quantidades por aporte   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_3_t

## Etapa 8.5) Estrutura de Registro de Compras e Posições

In [39]:
%%time
# ============================================================
# Etapa 8.5) Estrutura de Registro de Compras e Posições
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 8.5 - ESTRUTURA DE REGISTRO DE COMPRAS E POSIÇÕES")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_quantidades_alvo_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="base",
    nome="quantidades_alvo_aporte",
)

caminho_tbl_auditoria_quantidades_alvo_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="auditoria_quantidades_alvo_aporte",
)

caminho_base_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="caixa_residual_aporte",
)

caminho_tbl_auditoria_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="auditoria_caixa_residual_aporte",
)

caminho_tbl_regras_estrutura_registro_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="regras_estrutura_registro_carteira",
)

caminho_base_compras_planejadas_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="base",
    nome="compras_planejadas_carteira",
)

caminho_tbl_layout_base_compras_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="layout_base_compras_carteira",
)

caminho_tbl_layout_base_posicoes_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="layout_base_posicoes_carteira",
)

caminho_tbl_layout_base_patrimonio_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="layout_base_patrimonio_carteira",
)

caminho_tbl_layout_base_auditoria_aportes_caixa = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="layout_base_auditoria_aportes_caixa",
)

caminho_tbl_resumo_estrutura_registro_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="resumo_estrutura_registro_carteira",
)

caminho_tbl_inconsistencias_estrutura_registro_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="inconsistencias_estrutura_registro_carteira",
)

caminho_tbl_distribuicao_anual_compras_planejadas = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_compras_planejadas",
)

print(f"Entrada - base quantidades por aporte          : {caminho_base_quantidades_alvo_aporte}")
print(f"Entrada - auditoria quantidades por aporte     : {caminho_tbl_auditoria_quantidades_alvo_aporte}")
print(f"Entrada - base de caixa por aporte             : {caminho_base_caixa_residual_aporte}")
print(f"Entrada - auditoria de caixa por aporte        : {caminho_tbl_auditoria_caixa_residual_aporte}")
print(f"Saída   - regras da estrutura da carteira      : {caminho_tbl_regras_estrutura_registro_carteira}")
print(f"Saída   - base de compras planejadas           : {caminho_base_compras_planejadas_carteira}")
print(f"Saída   - layout da base de compras            : {caminho_tbl_layout_base_compras_carteira}")
print(f"Saída   - layout da base de posições           : {caminho_tbl_layout_base_posicoes_carteira}")
print(f"Saída   - layout da base patrimonial           : {caminho_tbl_layout_base_patrimonio_carteira}")
print(f"Saída   - layout da auditoria aportes e caixa  : {caminho_tbl_layout_base_auditoria_aportes_caixa}")
print(f"Saída   - resumo da estrutura da carteira      : {caminho_tbl_resumo_estrutura_registro_carteira}")
print(f"Saída   - inconsistências da estrutura         : {caminho_tbl_inconsistencias_estrutura_registro_carteira}")
print(f"Saída   - distribuição anual das compras       : {caminho_tbl_distribuicao_anual_compras_planejadas}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_quantidades_alvo_aporte = pd.read_parquet(caminho_base_quantidades_alvo_aporte)
df_auditoria_quantidades_alvo_aporte = pd.read_parquet(caminho_tbl_auditoria_quantidades_alvo_aporte)
df_caixa_residual_aporte = pd.read_parquet(caminho_base_caixa_residual_aporte)
df_auditoria_caixa_residual_aporte = pd.read_parquet(caminho_tbl_auditoria_caixa_residual_aporte)

print(f"Base quantidades por aporte              : {df_quantidades_alvo_aporte.shape[0]:,} linhas x {df_quantidades_alvo_aporte.shape[1]} colunas")
print(f"Auditoria quantidades por aporte         : {df_auditoria_quantidades_alvo_aporte.shape[0]:,} linhas x {df_auditoria_quantidades_alvo_aporte.shape[1]} colunas")
print(f"Base de caixa por aporte                 : {df_caixa_residual_aporte.shape[0]:,} linhas x {df_caixa_residual_aporte.shape[1]} colunas")
print(f"Auditoria de caixa por aporte            : {df_auditoria_caixa_residual_aporte.shape[0]:,} linhas x {df_auditoria_caixa_residual_aporte.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_AUDITORIA_ESTRUTURAL = 1e-08

COLUNAS_MINIMAS_BASE_QUANTIDADES = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "origem_calendario",
    "grupo_selecao_ativos",
    "benchmark_renda_variavel",
    "regra_selecao_cesta",
    "versao_cesta",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "close_adj",
    "quantidade_alvo_convertida",
    "flag_compra_efetivada",
    "valor_investido_ticker",
    "peso_efetivo_investido_ticker",
    "valor_aporte_estrategia",
    "valor_orcamento_alvo_ticker",
    "quantidade_inicial_inteira",
    "quantidade_extra_redistribuicao",
    "flag_recebeu_redistribuicao",
    "flag_quantidade_fracionaria_permitida",
    "tipo_conversao_quantidade",
]

COLUNAS_MINIMAS_AUDITORIA_QUANTIDADES = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "valor_aporte_estrategia",
    "valor_total_investido",
    "caixa_residual_final_aporte",
    "flag_conservacao_orcamento",
]

COLUNAS_MINIMAS_BASE_CAIXA = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "valor_aporte_estrategia",
    "valor_total_investido",
    "caixa_residual_final_aporte",
    "valor_caixa_trazido_aporte",
    "saldo_caixa_final_periodo",
]

COLUNAS_MINIMAS_AUDITORIA_CAIXA = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "tipo_estrategia",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "flag_conservacao_orcamento_valida",
    "flag_quantidade_compra_positiva_em_algum_ticker",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

print(f"Tolerância da auditoria estrutural : {TOLERANCIA_AUDITORIA_ESTRUTURAL}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

validar_colunas_obrigatorias(df_quantidades_alvo_aporte, COLUNAS_MINIMAS_BASE_QUANTIDADES)
validar_colunas_obrigatorias(df_auditoria_quantidades_alvo_aporte, COLUNAS_MINIMAS_AUDITORIA_QUANTIDADES)
validar_colunas_obrigatorias(df_caixa_residual_aporte, COLUNAS_MINIMAS_BASE_CAIXA)
validar_colunas_obrigatorias(df_auditoria_caixa_residual_aporte, COLUNAS_MINIMAS_AUDITORIA_CAIXA)

for coluna in [
    "estrategia_referencia",
    "carteira_id",
    "replica_id",
    "quantidade_inicial_inteira",
    "quantidade_extra_redistribuicao",
    "flag_recebeu_redistribuicao",
    "flag_quantidade_fracionaria_permitida",
    "tipo_conversao_quantidade",
]:
    df_quantidades_alvo_aporte = garantir_coluna(df_quantidades_alvo_aporte, coluna)

for coluna in ["carteira_id", "replica_id"]:
    df_auditoria_quantidades_alvo_aporte = garantir_coluna(df_auditoria_quantidades_alvo_aporte, coluna)
    df_caixa_residual_aporte = garantir_coluna(df_caixa_residual_aporte, coluna)
    df_auditoria_caixa_residual_aporte = garantir_coluna(df_auditoria_caixa_residual_aporte, coluna)

df_quantidades_alvo_aporte["data_sinal"] = pd.to_datetime(df_quantidades_alvo_aporte["data_sinal"], errors="coerce")
df_quantidades_alvo_aporte["data_aporte_efetiva"] = pd.to_datetime(df_quantidades_alvo_aporte["data_aporte_efetiva"], errors="coerce")
df_quantidades_alvo_aporte["ordem_aporte_estrategia"] = pd.to_numeric(df_quantidades_alvo_aporte["ordem_aporte_estrategia"], errors="coerce")
df_quantidades_alvo_aporte["close_adj"] = pd.to_numeric(df_quantidades_alvo_aporte["close_adj"], errors="coerce")
df_quantidades_alvo_aporte["quantidade_alvo_convertida"] = pd.to_numeric(df_quantidades_alvo_aporte["quantidade_alvo_convertida"], errors="coerce")
df_quantidades_alvo_aporte["valor_investido_ticker"] = pd.to_numeric(df_quantidades_alvo_aporte["valor_investido_ticker"], errors="coerce")
df_quantidades_alvo_aporte["peso_efetivo_investido_ticker"] = pd.to_numeric(df_quantidades_alvo_aporte["peso_efetivo_investido_ticker"], errors="coerce")
df_quantidades_alvo_aporte["valor_aporte_estrategia"] = pd.to_numeric(df_quantidades_alvo_aporte["valor_aporte_estrategia"], errors="coerce")
df_quantidades_alvo_aporte["valor_orcamento_alvo_ticker"] = pd.to_numeric(df_quantidades_alvo_aporte["valor_orcamento_alvo_ticker"], errors="coerce")
df_quantidades_alvo_aporte["quantidade_inicial_inteira"] = pd.to_numeric(df_quantidades_alvo_aporte["quantidade_inicial_inteira"], errors="coerce")
df_quantidades_alvo_aporte["quantidade_extra_redistribuicao"] = pd.to_numeric(df_quantidades_alvo_aporte["quantidade_extra_redistribuicao"], errors="coerce")
df_quantidades_alvo_aporte["ticker"] = df_quantidades_alvo_aporte["ticker"].astype("string").str.strip().str.upper()

df_inconsistencias_herdadas_8_3 = (
    df_auditoria_quantidades_alvo_aporte
    .loc[
        (~df_auditoria_quantidades_alvo_aporte["flag_conservacao_orcamento"])
        | (~df_auditoria_quantidades_alvo_aporte["flag_compra_minima_existe_no_aporte"])
        | (pd.to_numeric(df_auditoria_quantidades_alvo_aporte["caixa_residual_final_aporte"], errors="coerce") < -TOLERANCIA_AUDITORIA_ESTRUTURAL)
    ]
    .copy()
)

flag_caixa_valida_coluna = "flag_conservacao_orcamento_valida" if "flag_conservacao_orcamento_valida" in df_auditoria_caixa_residual_aporte.columns else "flag_conservacao_orcamento"
df_inconsistencias_herdadas_8_4 = (
    df_auditoria_caixa_residual_aporte
    .loc[
        (~df_auditoria_caixa_residual_aporte[flag_caixa_valida_coluna])
        | (~df_auditoria_caixa_residual_aporte["flag_quantidade_compra_positiva_em_algum_ticker"])
    ]
    .copy()
)

duplicidades_request_ticker = int(
    df_quantidades_alvo_aporte.duplicated(subset=["request_id", "ticker"]).sum()
)

if duplicidades_request_ticker > 0:
    raise ValueError(f"Foram encontradas {duplicidades_request_ticker} duplicidades por request_id+ticker na base de quantidades.")

print(f"Inconsistências herdadas da 8.3            : {len(df_inconsistencias_herdadas_8_3):,}")
print(f"Inconsistências herdadas da 8.4            : {len(df_inconsistencias_herdadas_8_4):,}")
print(f"Duplicidades request+ticker                : {duplicidades_request_ticker:,}")
print("OK")

# ============================================================
# 6) Construção da base de compras planejadas da carteira
# ============================================================

print("\n[6/10] Construção da base de compras planejadas da carteira...")

df_compras_planejadas_carteira = (
    df_quantidades_alvo_aporte
    .loc[df_quantidades_alvo_aporte["flag_compra_efetivada"].fillna(False)]
    .copy()
    .sort_values(["request_id", "ticker"])
    .reset_index(drop=True)
)

df_compras_planejadas_carteira["ordem_compra_no_aporte"] = (
    df_compras_planejadas_carteira
    .groupby("request_id")
    .cumcount()
    + 1
)

df_compras_planejadas_carteira["grupo_registro_carteira"] = "acoes"

df_compras_planejadas_carteira["grupo_selecao_ativos"] = (
    df_compras_planejadas_carteira["grupo_selecao_ativos"]
    .astype("string")
    .fillna("na")
)

df_compras_planejadas_carteira["benchmark_renda_variavel"] = (
    df_compras_planejadas_carteira["benchmark_renda_variavel"]
    .astype("string")
)

mapa_regra_selecao = {
    "universo_elegivel_integral": "universo_elegivel_final",
    "aleatoria_universo_elegivel": "amostra_aleatoria_universo_elegivel_data_aporte",
    "benchmark_ibovespa": "benchmark_equivalente_data_aporte",
}

df_compras_planejadas_carteira["regra_selecao_cesta"] = (
    df_compras_planejadas_carteira["regra_selecao_cesta"]
    .map(mapa_regra_selecao)
    .fillna(df_compras_planejadas_carteira["regra_selecao_cesta"].astype("string"))
)

mapa_versao_cesta = {
    "cesta_elegivel_integral": "universo_elegivel_final",
    "cesta_aleatoria_seed": "universo_elegivel_final",
    "benchmark_indice": "benchmark_equivalente",
}
df_compras_planejadas_carteira["versao_cesta"] = (
    df_compras_planejadas_carteira["versao_cesta"]
    .map(mapa_versao_cesta)
    .fillna(df_compras_planejadas_carteira["versao_cesta"].astype("string"))
)

df_compras_planejadas_carteira["preco_compra_planejado"] = df_compras_planejadas_carteira["close_adj"]
df_compras_planejadas_carteira["quantidade_comprada_planejada"] = df_compras_planejadas_carteira["quantidade_alvo_convertida"]
df_compras_planejadas_carteira["valor_investido_planejado"] = df_compras_planejadas_carteira["valor_investido_ticker"]
df_compras_planejadas_carteira["peso_efetivo_compra_planejada"] = df_compras_planejadas_carteira["peso_efetivo_investido_ticker"]
df_compras_planejadas_carteira["valor_aporte_na_data"] = df_compras_planejadas_carteira["valor_aporte_estrategia"]
df_compras_planejadas_carteira["flag_compra_fracionaria_equivalente"] = df_compras_planejadas_carteira["flag_quantidade_fracionaria_permitida"]

colunas_base_compras = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "tipo_estrategia",
    "origem_calendario",
    "grupo_registro_carteira",
    "benchmark_renda_variavel",
    "regra_selecao_cesta",
    "versao_cesta",
    "carteira_id",
    "replica_id",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "ordem_compra_no_aporte",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "preco_compra_planejado",
    "quantidade_comprada_planejada",
    "valor_investido_planejado",
    "peso_efetivo_compra_planejada",
    "valor_aporte_na_data",
    "valor_orcamento_alvo_ticker",
    "quantidade_inicial_inteira",
    "quantidade_extra_redistribuicao",
    "flag_recebeu_redistribuicao",
    "flag_compra_fracionaria_equivalente",
    "tipo_conversao_quantidade",
    "flag_evento_compra",
]

df_compras_planejadas_carteira["flag_evento_compra"] = True

for coluna in ["estrategia_referencia", "carteira_id", "replica_id"]:
    df_compras_planejadas_carteira = garantir_coluna(df_compras_planejadas_carteira, coluna)

df_compras_planejadas_carteira = df_compras_planejadas_carteira[colunas_base_compras].copy()

print(f"Linhas da base de compras planejadas       : {len(df_compras_planejadas_carteira):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais, layouts e auditorias
# ============================================================

print("\n[7/10] Construção das tabelas formais, layouts e auditorias...")

# Observação metodológica:
# os resumos e distribuições anuais são consolidados por estrategia_id,
# carteira_id e replica_id para não colapsar réplicas aleatórias.

df_regras_estrutura_registro_carteira = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "REG1",
            "escopo": "compras_planejadas",
            "regra_operacional": "registrar_cada_compra_planejada_com_chave_request_id_ticker",
            "detalhe": "Cada linha de compra planejada deve registrar data do aporte, ticker, issuer_code, classificação setorial, preço, quantidade e valor investido.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "REG2",
            "escopo": "base_historica_posicoes",
            "regra_operacional": "preservar_layout_para_quantidade_valor_de_mercado_custo_medio_e_peso",
            "detalhe": "A base histórica de posições deve suportar quantidade em carteira, custo médio, valor de mercado e participação relativa por ticker e por estratégia.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "REG3",
            "escopo": "base_patrimonial",
            "regra_operacional": "preservar_layout_para_patrimonio_carteira_caixa_e_referencias_de_aporte",
            "detalhe": "A base patrimonial deve consolidar valor investido, caixa, patrimônio total, aportes acumulados e métricas derivadas por data e estratégia.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "REG4",
            "escopo": "auditoria_aportes_caixa",
            "regra_operacional": "manter_vinculo_entre_aporte_compra_e_saldo_de_caixa",
            "detalhe": "Cada aporte executado deve poder ser reconciliado com o valor investido, o caixa residual, o caixa trazido e o saldo final do período.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "REG5",
            "escopo": "etapas_9_e_10",
            "regra_operacional": "salvar_layouts_operacionais_para_o_motor_de_backtest_e_auditoria_final",
            "detalhe": "Os layouts salvos nesta subetapa servirão como estrutura base para os registros das etapas 9 e 10.",
        },
    ]
)

df_layout_base_compras_carteira = pd.DataFrame(
    [
        ["base_compras_carteira", 1, "request_id", "string", True, "etapa_8_3", "Identificador do aporte executado."],
        ["base_compras_carteira", 2, "estrategia_id", "string", True, "etapa_7_e_8", "Identificador único da estratégia."],
        ["base_compras_carteira", 3, "data_aporte_efetiva", "datetime64[ns]", True, "etapa_7", "Data efetiva da compra."],
        ["base_compras_carteira", 4, "ticker", "string", True, "etapa_8_1", "Ticker efetivamente comprado."],
        ["base_compras_carteira", 5, "issuer_code", "string", True, "etapa_2_4", "Código consolidado da empresa emissora."],
        ["base_compras_carteira", 6, "nome", "string", True, "etapa_2_4", "Nome da empresa."],
        ["base_compras_carteira", 7, "setor", "string", False, "etapa_2_4", "Setor da empresa."],
        ["base_compras_carteira", 8, "subsetor", "string", False, "etapa_2_4", "Subsetor da empresa."],
        ["base_compras_carteira", 9, "segmento", "string", False, "etapa_2_4", "Segmento da empresa."],
        ["base_compras_carteira", 10, "preco_compra_planejado", "float64", True, "etapa_8_3", "Preço unitário planejado da compra."],
        ["base_compras_carteira", 11, "quantidade_comprada_planejada", "float64", True, "etapa_8_3", "Quantidade planejada no evento de compra."],
        ["base_compras_carteira", 12, "valor_investido_planejado", "float64", True, "etapa_8_3", "Valor planejado no evento de compra."],
        ["base_compras_carteira", 13, "peso_efetivo_compra_planejada", "float64", True, "etapa_8_3", "Peso efetivo planejado do ticker no aporte."],
    ],
    columns=["base_nome", "ordem_coluna", "nome_coluna", "tipo_dado", "obrigatoria", "origem_esperada", "descricao_operacional"],
)

df_layout_base_posicoes_carteira = pd.DataFrame(
    [
        ["base_posicoes_carteira", 1, "data", "datetime64[ns]", True, "etapa_9", "Data diária da posição."],
        ["base_posicoes_carteira", 2, "estrategia_id", "string", True, "etapa_9", "Identificador da estratégia."],
        ["base_posicoes_carteira", 3, "ticker", "string", True, "etapa_9", "Ticker em carteira."],
        ["base_posicoes_carteira", 4, "issuer_code", "string", True, "etapa_9", "Empresa consolidada do ativo."],
        ["base_posicoes_carteira", 5, "quantidade_em_carteira", "float64", True, "etapa_9", "Quantidade carregada na data."],
        ["base_posicoes_carteira", 6, "custo_medio_unitario", "float64", True, "etapa_9", "Custo médio unitário em carteira."],
        ["base_posicoes_carteira", 7, "preco_fechamento", "float64", True, "etapa_2_4", "Preço de fechamento de marcação a mercado."],
        ["base_posicoes_carteira", 8, "valor_mercado_posicao", "float64", True, "etapa_9", "Valor de mercado da posição."],
        ["base_posicoes_carteira", 9, "peso_posicao_no_patrimonio", "float64", True, "etapa_9", "Peso relativo da posição no patrimônio total."],
    ],
    columns=["base_nome", "ordem_coluna", "nome_coluna", "tipo_dado", "obrigatoria", "origem_esperada", "descricao_operacional"],
)

df_layout_base_patrimonio_carteira = pd.DataFrame(
    [
        ["base_patrimonio_carteira", 1, "data", "datetime64[ns]", True, "etapa_9", "Data da curva patrimonial."],
        ["base_patrimonio_carteira", 2, "estrategia_id", "string", True, "etapa_9", "Estratégia consolidada."],
        ["base_patrimonio_carteira", 3, "valor_carteira_investida", "float64", True, "etapa_9", "Somatório do valor investido das posições."],
        ["base_patrimonio_carteira", 4, "saldo_caixa", "float64", True, "etapa_8_4", "Saldo de caixa remunerado na data."],
        ["base_patrimonio_carteira", 5, "patrimonio_total", "float64", True, "etapa_9", "Soma do valor investido com o caixa."],
        ["base_patrimonio_carteira", 6, "valor_aportes_acumulado", "float64", True, "etapa_8_4", "Aportes acumulados até a data."],
        ["base_patrimonio_carteira", 7, "retorno_acumulado", "float64", False, "etapa_9", "Retorno acumulado da estratégia."],
    ],
    columns=["base_nome", "ordem_coluna", "nome_coluna", "tipo_dado", "obrigatoria", "origem_esperada", "descricao_operacional"],
)

df_layout_base_auditoria_aportes_caixa = pd.DataFrame(
    [
        ["base_auditoria_aportes_caixa", 1, "request_id", "string", True, "etapa_8_4", "Identificador do aporte."],
        ["base_auditoria_aportes_caixa", 2, "estrategia_id", "string", True, "etapa_8_4", "Estratégia associada ao aporte."],
        ["base_auditoria_aportes_caixa", 3, "data_aporte_efetiva", "datetime64[ns]", True, "etapa_7", "Data efetiva do aporte."],
        ["base_auditoria_aportes_caixa", 4, "valor_aporte_estrategia", "float64", True, "etapa_7", "Valor nominal do aporte."],
        ["base_auditoria_aportes_caixa", 5, "valor_investido_efetivo_aporte", "float64", True, "etapa_8_4", "Valor efetivamente investido no aporte."],
        ["base_auditoria_aportes_caixa", 6, "valor_caixa_residual_aporte", "float64", True, "etapa_8_4", "Caixa residual originado no aporte."],
        ["base_auditoria_aportes_caixa", 7, "valor_caixa_trazido_aporte", "float64", True, "etapa_8_4", "Caixa transportado do período anterior."],
        ["base_auditoria_aportes_caixa", 8, "rendimento_cdi_periodo", "float64", True, "etapa_8_4", "Rendimento do caixa no intervalo entre aportes."],
        ["base_auditoria_aportes_caixa", 9, "saldo_caixa_final_periodo", "float64", True, "etapa_8_4", "Saldo final carregável para o período seguinte."],
    ],
    columns=["base_nome", "ordem_coluna", "nome_coluna", "tipo_dado", "obrigatoria", "origem_esperada", "descricao_operacional"],
)

df_resumo_estrutura_registro_carteira = (
    df_compras_planejadas_carteira
    .groupby(
        [
            "familia_estrategia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_eventos_compra_planejada=("request_id", "count"),
        n_aportes_com_compra=("request_id", "nunique"),
        n_tickers_unicos_comprados=("ticker", "nunique"),
        data_primeira_compra=("data_aporte_efetiva", "min"),
        data_ultima_compra=("data_aporte_efetiva", "max"),
        valor_total_investido_planejado=("valor_investido_planejado", "sum"),
        quantidade_total_planejada=("quantidade_comprada_planejada", "sum"),
        peso_medio_compra_planejada=("peso_efetivo_compra_planejada", "mean"),
        n_eventos_com_redistribuicao=("flag_recebeu_redistribuicao", lambda x: int(pd.Series(x).fillna(False).sum())),
        n_eventos_compra_fracionaria_equivalente=("flag_compra_fracionaria_equivalente", lambda x: int(pd.Series(x).fillna(False).sum())),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id"])
    .reset_index(drop=True)
)

df_distribuicao_anual_compras_planejadas = (
    df_compras_planejadas_carteira
    .assign(ano=lambda df: pd.to_datetime(df["data_aporte_efetiva"], errors="coerce").dt.year)
    .groupby(
        [
            "familia_estrategia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_eventos_compra_planejada=("request_id", "count"),
        n_aportes_com_compra=("request_id", "nunique"),
        n_tickers_unicos_comprados=("ticker", "nunique"),
        valor_total_investido_planejado=("valor_investido_planejado", "sum"),
        quantidade_total_planejada=("quantidade_comprada_planejada", "sum"),
        n_eventos_com_redistribuicao=("flag_recebeu_redistribuicao", lambda x: int(pd.Series(x).fillna(False).sum())),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id", "ano"])
    .reset_index(drop=True)
)

df_auditoria_estrutura = (
    df_compras_planejadas_carteira
    .groupby("request_id", as_index=False, dropna=False)
    .agg(
        estrategia_id=("estrategia_id", "first"),
        grupo_controle=("grupo_controle", "first"),
        familia_estrategia=("familia_estrategia", "first"),
        tipo_estrategia=("tipo_estrategia", "first"),
        data_aporte_efetiva=("data_aporte_efetiva", "first"),
        ordem_aporte_estrategia=("ordem_aporte_estrategia", "first"),
        n_eventos_compra=("ticker", "count"),
        valor_total_investido_planejado=("valor_investido_planejado", "sum"),
        peso_total_compra_planejada=("peso_efetivo_compra_planejada", "sum"),
        flag_tem_compra_fracionaria_equivalente=("flag_compra_fracionaria_equivalente", "max"),
        flag_tem_redistribuicao=("flag_recebeu_redistribuicao", "max"),
    )
)

df_auditoria_estrutura = df_auditoria_estrutura.merge(
    df_caixa_residual_aporte[
        [
            "request_id",
            "valor_aporte_estrategia",
            "valor_total_investido",
            "caixa_residual_final_aporte",
            "valor_caixa_trazido_aporte",
            "saldo_caixa_final_periodo",
        ]
    ],
    on="request_id",
    how="left",
)

df_auditoria_estrutura["desvio_valor_investido"] = (
    df_auditoria_estrutura["valor_total_investido_planejado"]
    - df_auditoria_estrutura["valor_total_investido"]
).abs()

df_auditoria_estrutura["flag_valor_investido_reconciliado"] = (
    df_auditoria_estrutura["desvio_valor_investido"] <= TOLERANCIA_AUDITORIA_ESTRUTURAL
)

df_inconsistencias_estrutura_registro_carteira = (
    df_auditoria_estrutura
    .loc[
        (~df_auditoria_estrutura["flag_valor_investido_reconciliado"])
        | (df_auditoria_estrutura["n_eventos_compra"] <= 0)
        | (df_auditoria_estrutura["valor_total_investido_planejado"] <= 0)
    ]
    .copy()
)

print(f"Inconsistências estruturais identificadas   : {len(df_inconsistencias_estrutura_registro_carteira):,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_estrutura_registro_carteira, caminho_tbl_regras_estrutura_registro_carteira, index=False)
salvar_dataframe(df_compras_planejadas_carteira, caminho_base_compras_planejadas_carteira, index=False)
salvar_dataframe(df_layout_base_compras_carteira, caminho_tbl_layout_base_compras_carteira, index=False)
salvar_dataframe(df_layout_base_posicoes_carteira, caminho_tbl_layout_base_posicoes_carteira, index=False)
salvar_dataframe(df_layout_base_patrimonio_carteira, caminho_tbl_layout_base_patrimonio_carteira, index=False)
salvar_dataframe(df_layout_base_auditoria_aportes_caixa, caminho_tbl_layout_base_auditoria_aportes_caixa, index=False)
salvar_dataframe(df_resumo_estrutura_registro_carteira, caminho_tbl_resumo_estrutura_registro_carteira, index=False)
salvar_dataframe(df_inconsistencias_estrutura_registro_carteira, caminho_tbl_inconsistencias_estrutura_registro_carteira, index=False)
salvar_dataframe(df_distribuicao_anual_compras_planejadas, caminho_tbl_distribuicao_anual_compras_planejadas, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_estrutura_registro_carteira = df_regras_estrutura_registro_carteira.copy()
df_amostra_compras_planejadas_carteira = df_compras_planejadas_carteira.head(20).copy()
df_amostra_layout_base_compras = df_layout_base_compras_carteira.copy()
df_amostra_layout_base_posicoes = df_layout_base_posicoes_carteira.copy()
df_amostra_layout_base_patrimonio = df_layout_base_patrimonio_carteira.copy()
df_amostra_layout_base_auditoria = df_layout_base_auditoria_aportes_caixa.copy()
df_amostra_resumo_estrutura = df_resumo_estrutura_registro_carteira.copy()
df_amostra_inconsistencias_estrutura = df_inconsistencias_estrutura_registro_carteira.head(20).copy()
df_amostra_distribuicao_anual_compras = df_distribuicao_anual_compras_planejadas.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais da estrutura de registro da carteira:")
print(df_amostra_regras_estrutura_registro_carteira.to_string(index=False))

print("\nBase de compras planejadas da carteira - amostra:")
print(df_amostra_compras_planejadas_carteira.to_string(index=False))

print("\nLayout da base de compras da carteira:")
print(df_amostra_layout_base_compras.to_string(index=False))

print("\nLayout da base de posições da carteira:")
print(df_amostra_layout_base_posicoes.to_string(index=False))

print("\nLayout da base patrimonial da carteira:")
print(df_amostra_layout_base_patrimonio.to_string(index=False))

print("\nLayout da base de auditoria de aportes e caixa:")
print(df_amostra_layout_base_auditoria.to_string(index=False))

print("\nResumo da estrutura de registro da carteira:")
print(df_amostra_resumo_estrutura.to_string(index=False))

print("\nDistribuição anual das compras planejadas - amostra:")
print(df_amostra_distribuicao_anual_compras.to_string(index=False))

print("\nInconsistências estruturais - amostra:")
print(df_amostra_inconsistencias_estrutura.to_string(index=False))

print("\nArquivos salvos na subetapa 8.5:")
print(f"- {caminho_tbl_regras_estrutura_registro_carteira}")
print(f"- {caminho_base_compras_planejadas_carteira}")
print(f"- {caminho_tbl_layout_base_compras_carteira}")
print(f"- {caminho_tbl_layout_base_posicoes_carteira}")
print(f"- {caminho_tbl_layout_base_patrimonio_carteira}")
print(f"- {caminho_tbl_layout_base_auditoria_aportes_caixa}")
print(f"- {caminho_tbl_resumo_estrutura_registro_carteira}")
print(f"- {caminho_tbl_inconsistencias_estrutura_registro_carteira}")
print(f"- {caminho_tbl_distribuicao_anual_compras_planejadas}")

print("\nETAPA 8.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 8.5 - ESTRUTURA DE REGISTRO DE COMPRAS E POSIÇÕES

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - base quantidades por aporte          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_3_base_quantidades_alvo_aporte.parquet
Entrada - auditoria quantidades por aporte     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_3_tbl_auditoria_quantidades_alvo_aporte.parquet
Entrada - base de caixa por aporte             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_4_base_caixa_residual_aporte.parquet
Entrada - auditoria de caixa por aporte        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileir

# Etapa 9) Backtest das Estratégias

## Etapa 9.1) Backtest da Estratégia de Capitulação

In [40]:
%%time
# ============================================================
# Etapa 9.1) Backtest da Estratégia de Capitulação
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 9.1 - BACKTEST DA ESTRATÉGIA DE CAPITULAÇÃO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_calendario_aportes_capitulacao = gerar_caminho_arquivo(
    etapa=7,
    subetapa=3,
    tipo_arquivo="base",
    nome="calendario_aportes_capitulacao",
)

caminho_base_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="caixa_residual_aporte",
)

caminho_base_movimentacao_caixa_diaria = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="movimentacao_caixa_diaria",
)

caminho_base_compras_planejadas_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="base",
    nome="compras_planejadas_carteira",
)

caminho_tbl_regras_backtest_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="regras_backtest_capitulacao",
)

caminho_base_compras_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="compras_capitulacao",
)

caminho_base_posicoes_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="posicoes_capitulacao",
)

caminho_base_caixa_capitulacao_diaria = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="caixa_capitulacao_diaria",
)

caminho_base_patrimonio_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="patrimonio_capitulacao",
)

caminho_tbl_resumo_backtest_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="resumo_backtest_capitulacao",
)

caminho_tbl_auditoria_backtest_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="auditoria_backtest_capitulacao",
)

caminho_tbl_inconsistencias_backtest_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="inconsistencias_backtest_capitulacao",
)

caminho_tbl_distribuicao_anual_backtest_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_backtest_capitulacao",
)

print(f"Entrada - mercado diário consolidado          : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - calendário de aportes capitulação   : {caminho_base_calendario_aportes_capitulacao}")
print(f"Entrada - base de caixa por aporte            : {caminho_base_caixa_residual_aporte}")
print(f"Entrada - base diária de caixa                : {caminho_base_movimentacao_caixa_diaria}")
print(f"Entrada - base de compras planejadas          : {caminho_base_compras_planejadas_carteira}")
print(f"Saída   - regras do backtest                  : {caminho_tbl_regras_backtest_capitulacao}")
print(f"Saída   - base de compras da estratégia       : {caminho_base_compras_capitulacao}")
print(f"Saída   - base diária de posições             : {caminho_base_posicoes_capitulacao}")
print(f"Saída   - base diária de caixa                : {caminho_base_caixa_capitulacao_diaria}")
print(f"Saída   - base diária patrimonial             : {caminho_base_patrimonio_capitulacao}")
print(f"Saída   - resumo do backtest                  : {caminho_tbl_resumo_backtest_capitulacao}")
print(f"Saída   - auditoria do backtest               : {caminho_tbl_auditoria_backtest_capitulacao}")
print(f"Saída   - inconsistências do backtest         : {caminho_tbl_inconsistencias_backtest_capitulacao}")
print(f"Saída   - distribuição anual                  : {caminho_tbl_distribuicao_anual_backtest_capitulacao}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_calendario_aportes_capitulacao = pd.read_parquet(caminho_base_calendario_aportes_capitulacao)
df_base_caixa_residual_aporte = pd.read_parquet(caminho_base_caixa_residual_aporte)
df_movimentacao_caixa_diaria = pd.read_parquet(caminho_base_movimentacao_caixa_diaria)
df_compras_planejadas_carteira = pd.read_parquet(caminho_base_compras_planejadas_carteira)

print(f"Mercado diário consolidado                : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Calendário de aportes capitulação         : {df_calendario_aportes_capitulacao.shape[0]:,} linhas x {df_calendario_aportes_capitulacao.shape[1]} colunas")
print(f"Base de caixa por aporte                  : {df_base_caixa_residual_aporte.shape[0]:,} linhas x {df_base_caixa_residual_aporte.shape[1]} colunas")
print(f"Base diária de caixa                      : {df_movimentacao_caixa_diaria.shape[0]:,} linhas x {df_movimentacao_caixa_diaria.shape[1]} colunas")
print(f"Base de compras planejadas                : {df_compras_planejadas_carteira.shape[0]:,} linhas x {df_compras_planejadas_carteira.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_BACKTEST = 0.000001
TIPO_ESTRATEGIA_ALVO = "capitulacao"
NOME_ESTRATEGIA_REFERENCIA = "capitulacao"
FAMILIA_ESTRATEGIA_ESPERADA = "capitulacao_sinal"

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def detectar_coluna(df, candidatos, obrigatoria=True):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna

    if obrigatoria:
        raise ValueError(f"Não foi possível localizar nenhuma das colunas candidatas: {candidatos}")

    return None

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

def padronizar_texto(df, colunas_upper=None, colunas_lower=None):
    colunas_upper = [] if colunas_upper is None else colunas_upper
    colunas_lower = [] if colunas_lower is None else colunas_lower

    for coluna in colunas_upper:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.upper()

    for coluna in colunas_lower:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.lower()

    return df

def preparar_base_mercado(df_mercado):
    coluna_data = detectar_coluna(df_mercado, ["data", "date", "dt_ref"])
    coluna_ticker = detectar_coluna(df_mercado, ["ticker", "codneg", "ticker_ajustado"])
    coluna_preco = detectar_coluna(df_mercado, ["close_adj", "adj_close", "close", "preco_fechamento"])
    coluna_issuer = detectar_coluna(df_mercado, ["issuer_code", "codigo_empresa", "codigo_emissor"], obrigatoria=False)

    df_out = df_mercado.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_preco] = pd.to_numeric(df_out[coluna_preco], errors="coerce")

    df_out = padronizar_texto(df_out, colunas_upper=[coluna_ticker])

    colunas_saida = [coluna_data, coluna_ticker, coluna_preco]
    if coluna_issuer is not None:
        colunas_saida.append(coluna_issuer)

    for coluna in ["nome", "setor", "subsetor", "segmento"]:
        if coluna in df_out.columns:
            colunas_saida.append(coluna)

    df_out = (
        df_out[colunas_saida]
        .rename(
            columns={
                coluna_data: "data",
                coluna_ticker: "ticker",
                coluna_preco: "close_adj",
            }
        )
        .dropna(subset=["data", "ticker", "close_adj"])
        .sort_values(["ticker", "data"])
        .drop_duplicates(subset=["ticker", "data"], keep="last")
        .reset_index(drop=True)
    )

    if coluna_issuer is not None:
        df_out = df_out.rename(columns={coluna_issuer: "issuer_code"})
    else:
        df_out["issuer_code"] = pd.NA

    for coluna in ["nome", "setor", "subsetor", "segmento"]:
        df_out = garantir_coluna(df_out, coluna)

    return df_out

def preparar_calendario_capitulacao(df_calendario, df_caixa_aporte):
    df_cal = df_calendario.copy()
    df_cal["data_sinal"] = pd.to_datetime(df_cal["data_sinal"], errors="coerce")
    df_cal["data_aporte_efetiva"] = pd.to_datetime(df_cal["data_aporte_efetiva"], errors="coerce")
    df_cal["ordem_aporte_estrategia"] = pd.to_numeric(df_cal["ordem_aporte_estrategia"], errors="coerce")
    df_cal = padronizar_texto(df_cal, colunas_lower=["tipo_estrategia"])

    df_cal = df_cal.loc[df_cal["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO].copy()

    df_meta = (
        df_caixa_aporte.loc[df_caixa_aporte["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO]
        [
            [
                "request_id",
                "estrategia_id",
                "grupo_controle",
                "familia_estrategia",
                "estrategia_referencia",
                "tipo_estrategia",
                "data_sinal",
                "data_aporte_efetiva",
                "ordem_aporte_estrategia",
                "capital_inicial_estrategia",
                "valor_aporte_estrategia",
            ]
        ]
        .drop_duplicates(subset=["request_id"])
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    if df_meta.empty:
        raise ValueError("Não foi possível localizar os metadados operacionais da capitulação na etapa 8.4.")

    colunas_merge = ["tipo_estrategia", "data_aporte_efetiva", "ordem_aporte_estrategia"]

    df_saida = (
        df_cal.merge(
            df_meta,
            on=colunas_merge,
            how="left",
            validate="one_to_one",
            suffixes=("", "_meta"),
        )
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    n_sem_request = int(df_saida["request_id"].isna().sum())
    if n_sem_request > 0:
        raise ValueError(f"{n_sem_request} linhas do calendário de capitulação não encontraram request_id correspondente na etapa 8.4.")

    return df_saida

def preparar_base_caixa_aporte(df_caixa):
    df_out = df_caixa.copy()
    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["valor_total_investido"] = pd.to_numeric(df_out["valor_total_investido"], errors="coerce")
    df_out["caixa_residual_final_aporte"] = pd.to_numeric(df_out["caixa_residual_final_aporte"], errors="coerce")
    df_out["valor_caixa_trazido_aporte"] = pd.to_numeric(df_out["valor_caixa_trazido_aporte"], errors="coerce")
    df_out["saldo_caixa_final_periodo"] = pd.to_numeric(df_out["saldo_caixa_final_periodo"], errors="coerce")

    for coluna in ["carteira_id", "replica_id", "estrategia_referencia", "familia_estrategia", "grupo_controle"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out = padronizar_texto(df_out, colunas_lower=["tipo_estrategia"])
    df_out = df_out.loc[df_out["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO].copy()

    return (
        df_out
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .drop_duplicates(subset=["request_id"], keep="last")
        .reset_index(drop=True)
    )

def preparar_base_movimentacao_caixa(df_mov):
    df_out = df_mov.copy()
    coluna_data = detectar_coluna(df_out, ["data", "date"])
    coluna_saldo = detectar_coluna(df_out, ["saldo_caixa_fim_dia", "saldo_caixa", "saldo_caixa_final_dia", "saldo_caixa_final_periodo"])
    coluna_rendimento = detectar_coluna(df_out, ["rendimento_cdi_dia"], obrigatoria=False)
    coluna_taxa = detectar_coluna(df_out, ["taxa_cdi_dia"], obrigatoria=False)
    coluna_flag_evento = detectar_coluna(df_out, ["flag_evento_aporte"], obrigatoria=False)

    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_saldo] = pd.to_numeric(df_out[coluna_saldo], errors="coerce")

    if coluna_rendimento is not None:
        df_out[coluna_rendimento] = pd.to_numeric(df_out[coluna_rendimento], errors="coerce")
    else:
        df_out["rendimento_cdi_dia"] = 0.0
        coluna_rendimento = "rendimento_cdi_dia"

    if coluna_taxa is not None:
        df_out[coluna_taxa] = pd.to_numeric(df_out[coluna_taxa], errors="coerce")
    else:
        df_out["taxa_cdi_dia"] = 0.0
        coluna_taxa = "taxa_cdi_dia"

    if coluna_flag_evento is None:
        df_out["flag_evento_aporte"] = False
        coluna_flag_evento = "flag_evento_aporte"

    for coluna in ["request_id", "estrategia_id", "grupo_controle", "familia_estrategia", "estrategia_referencia", "tipo_estrategia", "ordem_aporte_estrategia", "valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte", "capital_inicial_estrategia"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["valor_investido_efetivo_aporte"] = pd.to_numeric(df_out["valor_investido_efetivo_aporte"], errors="coerce")
    df_out["valor_caixa_residual_aporte"] = pd.to_numeric(df_out["valor_caixa_residual_aporte"], errors="coerce")
    df_out["valor_caixa_trazido_aporte"] = pd.to_numeric(df_out["valor_caixa_trazido_aporte"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")

    df_out = padronizar_texto(df_out, colunas_lower=["tipo_estrategia"])
    df_out = (
        df_out.loc[df_out["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO]
        .rename(columns={coluna_data: "data", coluna_saldo: "saldo_caixa_fim_dia", coluna_rendimento: "rendimento_cdi_dia", coluna_taxa: "taxa_cdi_dia", coluna_flag_evento: "flag_evento_aporte"})
        .sort_values(["data", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_compras(df_compras):
    df_out = df_compras.copy()
    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["ordem_compra_no_aporte"] = pd.to_numeric(df_out["ordem_compra_no_aporte"], errors="coerce")
    df_out["preco_compra_planejado"] = pd.to_numeric(df_out["preco_compra_planejado"], errors="coerce")
    df_out["quantidade_comprada_planejada"] = pd.to_numeric(df_out["quantidade_comprada_planejada"], errors="coerce")
    df_out["valor_investido_planejado"] = pd.to_numeric(df_out["valor_investido_planejado"], errors="coerce")
    df_out["peso_efetivo_compra_planejada"] = pd.to_numeric(df_out["peso_efetivo_compra_planejada"], errors="coerce")
    df_out["valor_aporte_na_data"] = pd.to_numeric(df_out["valor_aporte_na_data"], errors="coerce")
    df_out["valor_orcamento_alvo_ticker"] = pd.to_numeric(df_out["valor_orcamento_alvo_ticker"], errors="coerce")

    df_out = garantir_coluna(df_out, "quantidade_extra_redistribuicao", 0.0)
    df_out["quantidade_extra_redistribuicao"] = pd.to_numeric(df_out["quantidade_extra_redistribuicao"], errors="coerce").fillna(0.0)
    df_out["quantidade_unidades_extra_redistribuicao"] = df_out["quantidade_extra_redistribuicao"]

    for coluna in ["carteira_id", "replica_id", "estrategia_referencia"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out = padronizar_texto(df_out, colunas_upper=["ticker"], colunas_lower=["tipo_estrategia"])
    df_out = (
        df_out.loc[df_out["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO]
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia", "ordem_compra_no_aporte", "ticker"])
        .reset_index(drop=True)
    )

    return df_out

def construir_calendario_diario(df_mercado_pad, df_mov_caixa_alvo, data_inicio, data_fim):
    datas_mercado = (
        df_mercado_pad.loc[(df_mercado_pad["data"] >= data_inicio) & (df_mercado_pad["data"] <= data_fim), "data"]
        .drop_duplicates()
        .sort_values()
    )

    datas_caixa = (
        df_mov_caixa_alvo.loc[(df_mov_caixa_alvo["data"] >= data_inicio) & (df_mov_caixa_alvo["data"] <= data_fim), "data"]
        .drop_duplicates()
        .sort_values()
    )

    calendario = pd.Series(pd.Index(sorted(set(datas_mercado.tolist()) | set(datas_caixa.tolist()))), name="data")
    return pd.to_datetime(calendario)

def construir_base_caixa_diaria_backtest(df_caixa_aporte_alvo, df_mov_caixa_alvo, calendario_diario):
    if df_mov_caixa_alvo.empty:
        raise ValueError("A base diária de caixa da etapa 8.4 não possui registros para a estratégia de capitulação.")

    estrategia_id = str(df_caixa_aporte_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_caixa_aporte_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0])
    tipo_estrategia = str(df_caixa_aporte_alvo["tipo_estrategia"].iloc[0])
    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])

    df_cash_daily = (
        df_mov_caixa_alvo
        .sort_values(["data", "ordem_aporte_estrategia"])
        .groupby("data", as_index=False, dropna=False)
        .agg(
            request_id=("request_id", "last"),
            ordem_aporte_estrategia=("ordem_aporte_estrategia", "last"),
            valor_aporte_estrategia=("valor_aporte_estrategia", "sum"),
            valor_investido_efetivo_aporte=("valor_investido_efetivo_aporte", "sum"),
            valor_caixa_residual_aporte=("valor_caixa_residual_aporte", "sum"),
            valor_caixa_trazido_aporte=("valor_caixa_trazido_aporte", "sum"),
            estrategia_referencia=("estrategia_referencia", "last"),
            saldo_caixa_fim_dia=("saldo_caixa_fim_dia", "last"),
            rendimento_cdi_dia=("rendimento_cdi_dia", "sum"),
            taxa_cdi_dia=("taxa_cdi_dia", "last"),
            flag_evento_aporte=("flag_evento_aporte", "max"),
        )
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_out = pd.DataFrame({"data": calendario_diario}).merge(df_cash_daily, on="data", how="left")
    df_out["saldo_caixa_fim_dia"] = pd.to_numeric(df_out["saldo_caixa_fim_dia"], errors="coerce").ffill().fillna(0.0)
    for coluna in ["valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte", "rendimento_cdi_dia", "taxa_cdi_dia"]:
        df_out[coluna] = pd.to_numeric(df_out[coluna], errors="coerce").fillna(0.0)

    df_out["flag_evento_aporte"] = df_out["flag_evento_aporte"].fillna(False)
    df_out["request_id"] = df_out["request_id"].astype("string")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")

    df_out["estrategia_id"] = estrategia_id
    df_out["grupo_controle"] = grupo_controle
    df_out["familia_estrategia"] = familia_estrategia
    df_out["estrategia_referencia"] = estrategia_referencia
    df_out["tipo_estrategia"] = tipo_estrategia
    df_out["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_out["saldo_caixa"] = df_out["saldo_caixa_fim_dia"]

    return df_out

def construir_base_posicoes_diarias(df_mercado_pad, df_compras_alvo, calendario_diario, capital_inicial_estrategia):
    # O capital inicial da estratégia deve vir do calendário/base operacional da estratégia,
    # e não da soma linha a linha da base de compras, para manter consistência com o notebook
    # e com as bases de auditoria e patrimônio.
    estrategia_id = str(df_compras_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_compras_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_compras_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_compras_alvo["estrategia_referencia"].iloc[0])
    tipo_estrategia = str(df_compras_alvo["tipo_estrategia"].iloc[0])
    capital_inicial_estrategia = float(capital_inicial_estrategia)
    data_inicio_backtest = pd.Timestamp(df_compras_alvo["data_aporte_efetiva"].min())
    data_fim_backtest = pd.Timestamp(calendario_diario.max())

    df_eventos = (
        df_compras_alvo
        .groupby(["ticker", "data_aporte_efetiva"], as_index=False, dropna=False)
        .agg(
            issuer_code=("issuer_code", "first"),
            nome=("nome", "first"),
            setor=("setor", "first"),
            subsetor=("subsetor", "first"),
            segmento=("segmento", "first"),
            quantidade_evento=("quantidade_comprada_planejada", "sum"),
            valor_investido_evento=("valor_investido_planejado", "sum"),
            preco_compra_evento=("preco_compra_planejado", "last"),
        )
        .rename(columns={"data_aporte_efetiva": "data"})
        .sort_values(["ticker", "data"])
        .reset_index(drop=True)
    )

    tickers_alvo = sorted(df_eventos["ticker"].dropna().unique().tolist())

    df_grade = pd.MultiIndex.from_product(
        [calendario_diario, tickers_alvo],
        names=["data", "ticker"],
    ).to_frame(index=False)

    df_meta_ticker = (
        df_eventos[["ticker", "issuer_code", "nome", "setor", "subsetor", "segmento", "preco_compra_evento"]]
        .drop_duplicates(subset=["ticker"], keep="first")
        .reset_index(drop=True)
    )

    df_precos = (
        df_mercado_pad.loc[df_mercado_pad["ticker"].isin(tickers_alvo), ["data", "ticker", "close_adj"]]
        .drop_duplicates(subset=["data", "ticker"], keep="last")
        .sort_values(["ticker", "data"])
        .reset_index(drop=True)
    )

    df_grade = df_grade.merge(df_precos, on=["data", "ticker"], how="left")
    df_grade = df_grade.merge(df_meta_ticker, on="ticker", how="left")
    df_grade["close_adj"] = (
        df_grade
        .sort_values(["ticker", "data"])
        .groupby("ticker")["close_adj"]
        .ffill()
    )
    df_grade["close_adj"] = df_grade["close_adj"].fillna(df_grade["preco_compra_evento"])

    df_grade = df_grade.merge(
        df_eventos[["ticker", "data", "quantidade_evento", "valor_investido_evento"]],
        on=["ticker", "data"],
        how="left",
    )

    df_grade["quantidade_evento"] = pd.to_numeric(df_grade["quantidade_evento"], errors="coerce").fillna(0.0)
    df_grade["valor_investido_evento"] = pd.to_numeric(df_grade["valor_investido_evento"], errors="coerce").fillna(0.0)

    df_grade = df_grade.sort_values(["ticker", "data"]).reset_index(drop=True)
    df_grade["quantidade_em_carteira"] = df_grade.groupby("ticker")["quantidade_evento"].cumsum()
    df_grade["valor_investido_acumulado_ticker"] = df_grade.groupby("ticker")["valor_investido_evento"].cumsum()

    df_grade = df_grade.loc[df_grade["quantidade_em_carteira"] > 0].copy()

    df_grade["custo_medio_unitario"] = np.where(
        df_grade["quantidade_em_carteira"] > 0,
        df_grade["valor_investido_acumulado_ticker"] / df_grade["quantidade_em_carteira"],
        0.0,
    )

    df_grade["close_adj"] = df_grade["close_adj"].fillna(df_grade["custo_medio_unitario"])
    df_grade["valor_mercado_posicao"] = df_grade["quantidade_em_carteira"] * df_grade["close_adj"]

    df_grade["estrategia_id"] = estrategia_id
    df_grade["grupo_controle"] = grupo_controle
    df_grade["familia_estrategia"] = familia_estrategia
    df_grade["estrategia_referencia"] = estrategia_referencia
    df_grade["tipo_estrategia"] = tipo_estrategia
    df_grade["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_grade["data_inicio_backtest"] = data_inicio_backtest
    df_grade["data_fim_backtest"] = data_fim_backtest

    colunas_saida = [
        "data",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "quantidade_em_carteira",
        "custo_medio_unitario",
        "close_adj",
        "valor_mercado_posicao",
    ]

    return df_grade[colunas_saida].copy()

def construir_base_patrimonial(df_posicoes_diarias, df_caixa_diaria_backtest, df_caixa_aporte_alvo, calendario_diario):
    estrategia_id = str(df_caixa_aporte_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_caixa_aporte_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0])
    tipo_estrategia = str(df_caixa_aporte_alvo["tipo_estrategia"].iloc[0])
    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])
    data_inicio_backtest = pd.Timestamp(calendario_diario.min())
    data_fim_backtest = pd.Timestamp(calendario_diario.max())

    df_pos_agg = (
        df_posicoes_diarias
        .groupby("data", as_index=False, dropna=False)
        .agg(
            valor_carteira_investida=("valor_mercado_posicao", "sum"),
            n_posicoes_ativas=("ticker", "count"),
            n_tickers_unicos_ativos=("ticker", "nunique"),
        )
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_patrimonio = pd.DataFrame({"data": calendario_diario}).merge(df_pos_agg, on="data", how="left")
    df_patrimonio = df_patrimonio.merge(
        df_caixa_diaria_backtest[
            [
                "data",
                "saldo_caixa_fim_dia",
                "saldo_caixa",
                "rendimento_cdi_dia",
                "taxa_cdi_dia",
                "request_id",
                "ordem_aporte_estrategia",
                "valor_aporte_estrategia",
                "valor_investido_efetivo_aporte",
                "valor_caixa_residual_aporte",
                "valor_caixa_trazido_aporte",
                "flag_evento_aporte",
            ]
        ],
        on="data",
        how="left",
    )

    for coluna in ["valor_carteira_investida", "n_posicoes_ativas", "n_tickers_unicos_ativos"]:
        df_patrimonio[coluna] = pd.to_numeric(df_patrimonio[coluna], errors="coerce").fillna(0.0)

    for coluna in ["saldo_caixa_fim_dia", "saldo_caixa", "rendimento_cdi_dia", "taxa_cdi_dia", "valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte"]:
        df_patrimonio[coluna] = pd.to_numeric(df_patrimonio[coluna], errors="coerce").fillna(0.0)

    df_patrimonio["flag_evento_aporte"] = df_patrimonio["flag_evento_aporte"].fillna(False)
    df_patrimonio["patrimonio_total"] = df_patrimonio["valor_carteira_investida"] + df_patrimonio["saldo_caixa_fim_dia"]

    mapa_aportes = (
        df_caixa_aporte_alvo[["data_aporte_efetiva", "valor_aporte_estrategia"]]
        .groupby("data_aporte_efetiva", as_index=False)["valor_aporte_estrategia"]
        .sum()
        .rename(columns={"data_aporte_efetiva": "data", "valor_aporte_estrategia": "valor_aporte_evento"})
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_patrimonio = df_patrimonio.merge(mapa_aportes, on="data", how="left")
    df_patrimonio["valor_aporte_evento"] = pd.to_numeric(df_patrimonio["valor_aporte_evento"], errors="coerce").fillna(0.0)
    df_patrimonio["valor_total_aportado"] = df_patrimonio["valor_aporte_evento"].cumsum()

    df_patrimonio["max_patrimonio_acumulado"] = df_patrimonio["patrimonio_total"].cummax()
    df_patrimonio["retorno_diario"] = df_patrimonio["patrimonio_total"].pct_change().fillna(0.0)

    if capital_inicial_estrategia > 0:
        df_patrimonio["retorno_acumulado"] = (df_patrimonio["patrimonio_total"] / capital_inicial_estrategia) - 1.0
    else:
        df_patrimonio["retorno_acumulado"] = 0.0

    df_patrimonio["drawdown_atual"] = np.where(
        df_patrimonio["max_patrimonio_acumulado"] > 0,
        (df_patrimonio["patrimonio_total"] / df_patrimonio["max_patrimonio_acumulado"]) - 1.0,
        0.0,
    )
    df_patrimonio["flag_em_drawdown"] = df_patrimonio["drawdown_atual"] < 0.0

    df_patrimonio["estrategia_id"] = estrategia_id
    df_patrimonio["grupo_controle"] = grupo_controle
    df_patrimonio["familia_estrategia"] = familia_estrategia
    df_patrimonio["estrategia_referencia"] = estrategia_referencia
    df_patrimonio["tipo_estrategia"] = tipo_estrategia
    df_patrimonio["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_patrimonio["data_inicio_backtest"] = data_inicio_backtest
    df_patrimonio["data_fim_backtest"] = data_fim_backtest

    return df_patrimonio

def enriquecer_pesos_posicoes(df_posicoes_diarias, df_patrimonio):
    df_out = df_posicoes_diarias.merge(
        df_patrimonio[["data", "patrimonio_total"]],
        on="data",
        how="left",
    )

    df_out["peso_posicao_no_patrimonio"] = np.where(
        df_out["patrimonio_total"] > 0,
        df_out["valor_mercado_posicao"] / df_out["patrimonio_total"],
        0.0,
    )

    return df_out

def executar_motor_backtest_unico(df_mercado_pad, df_calendario_alvo, df_caixa_aporte_alvo, df_mov_caixa_alvo, df_compras_alvo):
    data_inicio = pd.Timestamp(df_mov_caixa_alvo["data"].min())
    data_fim = pd.Timestamp(df_mov_caixa_alvo["data"].max())

    calendario_diario = construir_calendario_diario(
        df_mercado_pad=df_mercado_pad,
        df_mov_caixa_alvo=df_mov_caixa_alvo,
        data_inicio=data_inicio,
        data_fim=data_fim,
    )

    if calendario_diario.empty:
        raise ValueError("O calendário diário do backtest ficou vazio.")

    df_caixa_diaria_backtest = construir_base_caixa_diaria_backtest(
        df_caixa_aporte_alvo=df_caixa_aporte_alvo,
        df_mov_caixa_alvo=df_mov_caixa_alvo,
        calendario_diario=calendario_diario,
    )

    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])

    df_posicoes_diarias = construir_base_posicoes_diarias(
        df_mercado_pad=df_mercado_pad,
        df_compras_alvo=df_compras_alvo,
        calendario_diario=calendario_diario,
        capital_inicial_estrategia=capital_inicial_estrategia,
    )

    df_patrimonio = construir_base_patrimonial(
        df_posicoes_diarias=df_posicoes_diarias,
        df_caixa_diaria_backtest=df_caixa_diaria_backtest,
        df_caixa_aporte_alvo=df_caixa_aporte_alvo,
        calendario_diario=calendario_diario,
    )

    df_posicoes_diarias = enriquecer_pesos_posicoes(
        df_posicoes_diarias=df_posicoes_diarias,
        df_patrimonio=df_patrimonio,
    )

    return df_caixa_diaria_backtest, df_posicoes_diarias, df_patrimonio

print(f"Tolerância do backtest capitulação : {TOLERANCIA_BACKTEST}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

df_mercado_pad = preparar_base_mercado(df_mercado_diario_consolidada)
df_caixa_aporte_alvo = preparar_base_caixa_aporte(df_base_caixa_residual_aporte)
df_calendario_alvo = preparar_calendario_capitulacao(df_calendario_aportes_capitulacao, df_caixa_aporte_alvo)
df_mov_caixa_alvo = preparar_base_movimentacao_caixa(df_movimentacao_caixa_diaria)
df_compras_alvo = preparar_base_compras(df_compras_planejadas_carteira)

if df_compras_alvo.empty:
    raise ValueError("A base de compras planejadas não possui registros para a estratégia de capitulação.")
if df_caixa_aporte_alvo.empty:
    raise ValueError("A base de caixa por aporte não possui registros para a estratégia de capitulação.")
if df_mov_caixa_alvo.empty:
    raise ValueError("A base diária de caixa não possui registros para a estratégia de capitulação.")
if df_calendario_alvo.empty:
    raise ValueError("O calendário da estratégia de capitulação ficou vazio após a preparação.")

n_aportes_capitulacao = int(df_calendario_alvo["request_id"].nunique())
n_compras_capitulacao = int(len(df_compras_alvo))
n_tickers_capitulacao = int(df_compras_alvo["ticker"].nunique())

n_estrategia_referencia_ausente_inputs = int(
    df_caixa_aporte_alvo["estrategia_referencia"].isna().sum()
    + df_compras_alvo["estrategia_referencia"].isna().sum()
)

print(f"Compras da estratégia de capitulação        : {n_compras_capitulacao:,}")
print(f"Aportes da estratégia de capitulação        : {n_aportes_capitulacao:,}")
print(f"Tickers comprados na estratégia             : {n_tickers_capitulacao:,}")
print(f"Estrategia_referencia ausente nos inputs    : {n_estrategia_referencia_ausente_inputs:,}")
print("OK")

# ============================================================
# 6) Execução do motor de backtest da estratégia de capitulação
# ============================================================

print("\n[6/10] Execução do motor de backtest da estratégia de capitulação...")

df_caixa_capitulacao_diaria, df_posicoes_capitulacao, df_patrimonio_capitulacao = executar_motor_backtest_unico(
    df_mercado_pad=df_mercado_pad,
    df_calendario_alvo=df_calendario_alvo,
    df_caixa_aporte_alvo=df_caixa_aporte_alvo,
    df_mov_caixa_alvo=df_mov_caixa_alvo,
    df_compras_alvo=df_compras_alvo,
)

df_compras_capitulacao = df_compras_alvo.copy()

n_ref_ausente_outputs = int(
    df_compras_capitulacao["estrategia_referencia"].isna().sum()
    + df_posicoes_capitulacao["estrategia_referencia"].isna().sum()
    + df_caixa_capitulacao_diaria["estrategia_referencia"].isna().sum()
    + df_patrimonio_capitulacao["estrategia_referencia"].isna().sum()
)

print(f"Linhas da base de compras da capitulação   : {len(df_compras_capitulacao):,}")
print(f"Linhas da base de posições da capitulação  : {len(df_posicoes_capitulacao):,}")
print(f"Linhas da base de caixa diária             : {len(df_caixa_capitulacao_diaria):,}")
print(f"Linhas da base patrimonial                 : {len(df_patrimonio_capitulacao):,}")
print(f"Data inicial da curva patrimonial          : {pd.Timestamp(df_patrimonio_capitulacao['data'].min()).date()}")
print(f"Data final da curva patrimonial            : {pd.Timestamp(df_patrimonio_capitulacao['data'].max()).date()}")
print(f"Estrategia_referencia ausente nos outputs  : {n_ref_ausente_outputs:,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais, resumos e auditorias
# ============================================================

print("\n[7/10] Construção das tabelas formais, resumos e auditorias...")

df_regras_backtest_capitulacao = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "BT1",
            "escopo": "motor_da_estrategia",
            "regra_operacional": "executar_os_aportes_da_capitulacao_nas_datas_definidas_no_calendario",
            "detalhe": "O backtest da capitulação utiliza exclusivamente o calendário de aportes da estratégia de capitulação como agenda de execução.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "BT2",
            "escopo": "compras_e_posicoes",
            "regra_operacional": "registrar_cada_compra_e_propagar_quantidades_diariamente_por_ticker",
            "detalhe": "As compras executadas alimentam a base diária de posições com propagação diária por ticker, carregamento do último preço disponível e fallback para o preço de compra na data do aporte quando necessário.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "BT3",
            "escopo": "caixa",
            "regra_operacional": "incorporar_o_saldo_de_caixa_completo_diario_remunerado_pelo_cdi_ate_a_data_final_do_backtest",
            "detalhe": "O caixa diário utilizado no backtest representa o capital não alocado em ações, remunerado pelo CDI desde o início comum da trilha até cada data do calendário patrimonial.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "BT4",
            "escopo": "curva_patrimonial",
            "regra_operacional": "consolidar_diariamente_valor_investido_caixa_e_patrimonio_total",
            "detalhe": "A curva patrimonial diária é a soma do valor de mercado das posições e do saldo de caixa da estratégia.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "BT5",
            "escopo": "auditoria",
            "regra_operacional": "reconciliar_capital_inicial_compras_caixa_completo_e_rendimento_cdi_do_proprio_backtest",
            "detalhe": "A auditoria deve reconciliar capital inicial, compras efetivas, saldo de caixa completo e rendimento CDI efetivamente refletido na trilha diária do backtest.",
        },
    ]
)

capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])
valor_total_aportado = float(df_caixa_aporte_alvo["valor_aporte_estrategia"].sum())
valor_total_investido = float(df_compras_capitulacao["valor_investido_planejado"].sum())
saldo_caixa_final = float(df_caixa_capitulacao_diaria["saldo_caixa_fim_dia"].iloc[-1])
patrimonio_final = float(df_patrimonio_capitulacao["patrimonio_total"].iloc[-1])
retorno_acumulado_final = float(df_patrimonio_capitulacao["retorno_acumulado"].iloc[-1])
drawdown_maximo = float(df_patrimonio_capitulacao["drawdown_atual"].min())
n_posicoes_ativas_finais = int(df_posicoes_capitulacao.loc[df_posicoes_capitulacao["data"] == df_patrimonio_capitulacao["data"].max(), "ticker"].nunique())

valor_caixa_residual_total_aportes = float(df_caixa_aporte_alvo["caixa_residual_final_aporte"].sum())
valor_rendimento_cdi_acumulado_final = float(df_caixa_capitulacao_diaria["rendimento_cdi_dia"].sum())
desvio_fluxo_caixa_completo = abs(
    saldo_caixa_final
    - (capital_inicial_estrategia - valor_total_investido + valor_rendimento_cdi_acumulado_final)
)
desvio_aportes_planejados = abs(valor_total_aportado - (valor_total_investido + valor_caixa_residual_total_aportes))

df_resumo_backtest_capitulacao = pd.DataFrame(
    [
        {
            "estrategia_id": str(df_caixa_aporte_alvo["estrategia_id"].iloc[0]),
            "grupo_controle": str(df_caixa_aporte_alvo["grupo_controle"].iloc[0]),
            "familia_estrategia": str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0]),
            "estrategia_referencia": str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0]),
            "tipo_estrategia": TIPO_ESTRATEGIA_ALVO,
            "data_inicio_backtest": pd.Timestamp(df_patrimonio_capitulacao["data"].min()),
            "data_fim_backtest": pd.Timestamp(df_patrimonio_capitulacao["data"].max()),
            "n_aportes": int(df_caixa_aporte_alvo["request_id"].nunique()),
            "n_compras": int(len(df_compras_capitulacao)),
            "n_tickers_unicos": int(df_compras_capitulacao["ticker"].nunique()),
            "n_datas_curva_patrimonial": int(len(df_patrimonio_capitulacao)),
            "valor_total_aportado": valor_total_aportado,
            "valor_total_investido": valor_total_investido,
            "saldo_caixa_final": saldo_caixa_final,
            "patrimonio_final": patrimonio_final,
            "retorno_acumulado_final": retorno_acumulado_final,
            "drawdown_maximo": drawdown_maximo,
            "n_posicoes_ativas_finais": n_posicoes_ativas_finais,
        }
    ]
)

df_auditoria_backtest_capitulacao = pd.DataFrame(
    [
        {
            "estrategia_id": str(df_caixa_aporte_alvo["estrategia_id"].iloc[0]),
            "grupo_controle": str(df_caixa_aporte_alvo["grupo_controle"].iloc[0]),
            "familia_estrategia": str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0]),
            "estrategia_referencia": str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0]),
            "tipo_estrategia": TIPO_ESTRATEGIA_ALVO,
            "capital_inicial_estrategia": capital_inicial_estrategia,
            "valor_total_aportado": valor_total_aportado,
            "valor_total_investido": valor_total_investido,
            "valor_caixa_residual_total_aportes": valor_caixa_residual_total_aportes,
            "valor_rendimento_cdi_acumulado_final": valor_rendimento_cdi_acumulado_final,
            "saldo_caixa_final": saldo_caixa_final,
            "patrimonio_final": patrimonio_final,
            "desvio_aportes_planejados_vs_investido_mais_residual": desvio_aportes_planejados,
            "desvio_fluxo_caixa_completo": desvio_fluxo_caixa_completo,
            "flag_reconciliacao_aportes_planejados": desvio_aportes_planejados <= TOLERANCIA_BACKTEST,
            "flag_reconciliacao_caixa_completo": desvio_fluxo_caixa_completo <= TOLERANCIA_BACKTEST,
            "flag_patrimonio_final_nao_negativo": patrimonio_final >= -TOLERANCIA_BACKTEST,
            "flag_posicoes_nao_negativas": bool((df_posicoes_capitulacao["quantidade_em_carteira"] >= -TOLERANCIA_BACKTEST).all()),
            "flag_caixa_nao_negativo": bool((df_caixa_capitulacao_diaria["saldo_caixa_fim_dia"] >= -TOLERANCIA_BACKTEST).all()),
            "flag_estrategia_referencia_preenchida": bool(n_ref_ausente_outputs == 0),
        }
    ]
)

df_inconsistencias_backtest_capitulacao = (
    df_auditoria_backtest_capitulacao.loc[
        (~df_auditoria_backtest_capitulacao["flag_reconciliacao_aportes_planejados"])
        | (~df_auditoria_backtest_capitulacao["flag_reconciliacao_caixa_completo"])
        | (~df_auditoria_backtest_capitulacao["flag_patrimonio_final_nao_negativo"])
        | (~df_auditoria_backtest_capitulacao["flag_posicoes_nao_negativas"])
        | (~df_auditoria_backtest_capitulacao["flag_caixa_nao_negativo"])
        | (~df_auditoria_backtest_capitulacao["flag_estrategia_referencia_preenchida"])
    ].copy()
)

df_distribuicao_anual_backtest_capitulacao = (
    df_patrimonio_capitulacao
    .assign(ano=lambda df: pd.to_datetime(df["data"], errors="coerce").dt.year)
    .groupby(["estrategia_id", "grupo_controle", "familia_estrategia", "estrategia_referencia", "tipo_estrategia", "ano"], as_index=False)
    .agg(
        n_datas_curva_patrimonial=("data", "count"),
        patrimonio_medio_ano=("patrimonio_total", "mean"),
        valor_carteira_medio_ano=("valor_carteira_investida", "mean"),
        saldo_caixa_medio_ano=("saldo_caixa_fim_dia", "mean"),
        patrimonio_final_ano=("patrimonio_total", "last"),
        saldo_caixa_final_ano=("saldo_caixa_fim_dia", "last"),
        drawdown_minimo_ano=("drawdown_atual", "min"),
        retorno_ultimo_dia_ano=("retorno_acumulado", "last"),
    )
    .sort_values(["ano"])
    .reset_index(drop=True)
)

print(f"Inconsistências do backtest identificadas   : {len(df_inconsistencias_backtest_capitulacao):,}")
print(f"Desvio do fluxo de caixa completo           : {desvio_fluxo_caixa_completo:.10f}")
print(f"Estrategia_referencia ausente nos outputs   : {n_ref_ausente_outputs:,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_backtest_capitulacao, caminho_tbl_regras_backtest_capitulacao, index=False)
salvar_dataframe(df_compras_capitulacao, caminho_base_compras_capitulacao, index=False)
salvar_dataframe(df_posicoes_capitulacao, caminho_base_posicoes_capitulacao, index=False)
salvar_dataframe(df_caixa_capitulacao_diaria, caminho_base_caixa_capitulacao_diaria, index=False)
salvar_dataframe(df_patrimonio_capitulacao, caminho_base_patrimonio_capitulacao, index=False)
salvar_dataframe(df_resumo_backtest_capitulacao, caminho_tbl_resumo_backtest_capitulacao, index=False)
salvar_dataframe(df_auditoria_backtest_capitulacao, caminho_tbl_auditoria_backtest_capitulacao, index=False)
salvar_dataframe(df_inconsistencias_backtest_capitulacao, caminho_tbl_inconsistencias_backtest_capitulacao, index=False)
salvar_dataframe(df_distribuicao_anual_backtest_capitulacao, caminho_tbl_distribuicao_anual_backtest_capitulacao, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_backtest_capitulacao = df_regras_backtest_capitulacao.copy()
df_amostra_compras_capitulacao = df_compras_capitulacao.head(20).copy()
df_amostra_posicoes_capitulacao = df_posicoes_capitulacao.head(20).copy()
df_amostra_caixa_capitulacao_diaria = df_caixa_capitulacao_diaria.head(20).copy()
df_amostra_patrimonio_capitulacao = df_patrimonio_capitulacao.head(20).copy()
df_amostra_resumo_backtest_capitulacao = df_resumo_backtest_capitulacao.copy()
df_amostra_auditoria_backtest_capitulacao = df_auditoria_backtest_capitulacao.copy()
df_amostra_inconsistencias_backtest_capitulacao = df_inconsistencias_backtest_capitulacao.head(20).copy()
df_amostra_distribuicao_anual_backtest_capitulacao = df_distribuicao_anual_backtest_capitulacao.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais do backtest de capitulação:")
print(df_amostra_regras_backtest_capitulacao.to_string(index=False))

print("\nBase de compras da estratégia de capitulação - amostra:")
print(df_amostra_compras_capitulacao.to_string(index=False))

print("\nBase diária de posições da estratégia de capitulação - amostra:")
print(df_amostra_posicoes_capitulacao.to_string(index=False))

print("\nBase diária de caixa da estratégia de capitulação - amostra:")
print(df_amostra_caixa_capitulacao_diaria.to_string(index=False))

print("\nBase patrimonial da estratégia de capitulação - amostra:")
print(df_amostra_patrimonio_capitulacao.to_string(index=False))

print("\nResumo do backtest de capitulação:")
print(df_amostra_resumo_backtest_capitulacao.to_string(index=False))

print("\nAuditoria do backtest de capitulação:")
print(df_amostra_auditoria_backtest_capitulacao.to_string(index=False))

print("\nInconsistências do backtest de capitulação - amostra:")
print(df_amostra_inconsistencias_backtest_capitulacao.to_string(index=False))

print("\nDistribuição anual do backtest de capitulação - amostra:")
print(df_amostra_distribuicao_anual_backtest_capitulacao.to_string(index=False))

print("\nArquivos salvos na subetapa 9.1:")
print(f"- {caminho_tbl_regras_backtest_capitulacao}")
print(f"- {caminho_base_compras_capitulacao}")
print(f"- {caminho_base_posicoes_capitulacao}")
print(f"- {caminho_base_caixa_capitulacao_diaria}")
print(f"- {caminho_base_patrimonio_capitulacao}")
print(f"- {caminho_tbl_resumo_backtest_capitulacao}")
print(f"- {caminho_tbl_auditoria_backtest_capitulacao}")
print(f"- {caminho_tbl_inconsistencias_backtest_capitulacao}")
print(f"- {caminho_tbl_distribuicao_anual_backtest_capitulacao}")

print("\nETAPA 9.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 9.1 - BACKTEST DA ESTRATÉGIA DE CAPITULAÇÃO

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidado          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - calendário de aportes capitulação   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_3_base_calendario_aportes_capitulacao.parquet
Entrada - base de caixa por aporte            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_4_base_caixa_residual_aporte.parquet
Entrada - base diária de caixa                : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resulta

## Etapa 9.2) Backtest da Estratégia de Euforia

In [41]:
%%time
# ============================================================
# Etapa 9.2) Backtest da Estratégia de Euforia
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 9.2 - BACKTEST DA ESTRATÉGIA DE EUFORIA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_calendario_aportes_euforia = gerar_caminho_arquivo(
    etapa=7,
    subetapa=4,
    tipo_arquivo="base",
    nome="calendario_aportes_euforia",
)

caminho_base_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="caixa_residual_aporte",
)

caminho_base_movimentacao_caixa_diaria = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="movimentacao_caixa_diaria",
)

caminho_base_compras_planejadas_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="base",
    nome="compras_planejadas_carteira",
)

caminho_tbl_regras_backtest_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="regras_backtest_euforia",
)

caminho_base_compras_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="compras_euforia",
)

caminho_base_posicoes_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="posicoes_euforia",
)

caminho_base_caixa_euforia_diaria = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="caixa_euforia_diaria",
)

caminho_base_patrimonio_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="patrimonio_euforia",
)

caminho_tbl_resumo_backtest_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="resumo_backtest_euforia",
)

caminho_tbl_auditoria_backtest_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="auditoria_backtest_euforia",
)

caminho_tbl_inconsistencias_backtest_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="inconsistencias_backtest_euforia",
)

caminho_tbl_distribuicao_anual_backtest_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_backtest_euforia",
)

print(f"Entrada - mercado diário consolidado          : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - calendário de aportes euforia       : {caminho_base_calendario_aportes_euforia}")
print(f"Entrada - base de caixa por aporte            : {caminho_base_caixa_residual_aporte}")
print(f"Entrada - base diária de caixa                : {caminho_base_movimentacao_caixa_diaria}")
print(f"Entrada - base de compras planejadas          : {caminho_base_compras_planejadas_carteira}")
print(f"Saída   - regras do backtest                  : {caminho_tbl_regras_backtest_euforia}")
print(f"Saída   - base de compras da estratégia       : {caminho_base_compras_euforia}")
print(f"Saída   - base diária de posições             : {caminho_base_posicoes_euforia}")
print(f"Saída   - base diária de caixa                : {caminho_base_caixa_euforia_diaria}")
print(f"Saída   - base diária patrimonial             : {caminho_base_patrimonio_euforia}")
print(f"Saída   - resumo do backtest                  : {caminho_tbl_resumo_backtest_euforia}")
print(f"Saída   - auditoria do backtest               : {caminho_tbl_auditoria_backtest_euforia}")
print(f"Saída   - inconsistências do backtest         : {caminho_tbl_inconsistencias_backtest_euforia}")
print(f"Saída   - distribuição anual                  : {caminho_tbl_distribuicao_anual_backtest_euforia}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_calendario_aportes_euforia = pd.read_parquet(caminho_base_calendario_aportes_euforia)
df_base_caixa_residual_aporte = pd.read_parquet(caminho_base_caixa_residual_aporte)
df_movimentacao_caixa_diaria = pd.read_parquet(caminho_base_movimentacao_caixa_diaria)
df_compras_planejadas_carteira = pd.read_parquet(caminho_base_compras_planejadas_carteira)

print(f"Mercado diário consolidado                : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Calendário de aportes euforia             : {df_calendario_aportes_euforia.shape[0]:,} linhas x {df_calendario_aportes_euforia.shape[1]} colunas")
print(f"Base de caixa por aporte                  : {df_base_caixa_residual_aporte.shape[0]:,} linhas x {df_base_caixa_residual_aporte.shape[1]} colunas")
print(f"Base diária de caixa                      : {df_movimentacao_caixa_diaria.shape[0]:,} linhas x {df_movimentacao_caixa_diaria.shape[1]} colunas")
print(f"Base de compras planejadas                : {df_compras_planejadas_carteira.shape[0]:,} linhas x {df_compras_planejadas_carteira.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_BACKTEST = 0.000001
TIPO_ESTRATEGIA_ALVO = "euforia"
NOME_ESTRATEGIA_REFERENCIA = "euforia"
FAMILIA_ESTRATEGIA_ESPERADA = "euforia_sinal"

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def detectar_coluna(df, candidatos, obrigatoria=True):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna

    if obrigatoria:
        raise ValueError(f"Não foi possível localizar nenhuma das colunas candidatas: {candidatos}")

    return None

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

def padronizar_texto(df, colunas_upper=None, colunas_lower=None):
    colunas_upper = [] if colunas_upper is None else colunas_upper
    colunas_lower = [] if colunas_lower is None else colunas_lower

    for coluna in colunas_upper:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.upper()

    for coluna in colunas_lower:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.lower()

    return df

def preparar_base_mercado(df_mercado):
    coluna_data = detectar_coluna(df_mercado, ["data", "date", "dt_ref"])
    coluna_ticker = detectar_coluna(df_mercado, ["ticker", "codneg", "ticker_ajustado"])
    coluna_preco = detectar_coluna(df_mercado, ["close_adj", "adj_close", "close", "preco_fechamento"])
    coluna_issuer = detectar_coluna(df_mercado, ["issuer_code", "codigo_empresa", "codigo_emissor"], obrigatoria=False)

    df_out = df_mercado.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_preco] = pd.to_numeric(df_out[coluna_preco], errors="coerce")

    df_out = padronizar_texto(df_out, colunas_upper=[coluna_ticker])

    colunas_saida = [coluna_data, coluna_ticker, coluna_preco]
    if coluna_issuer is not None:
        colunas_saida.append(coluna_issuer)

    for coluna in ["nome", "setor", "subsetor", "segmento"]:
        if coluna in df_out.columns:
            colunas_saida.append(coluna)

    df_out = (
        df_out[colunas_saida]
        .rename(
            columns={
                coluna_data: "data",
                coluna_ticker: "ticker",
                coluna_preco: "close_adj",
            }
        )
        .dropna(subset=["data", "ticker", "close_adj"])
        .sort_values(["ticker", "data"])
        .drop_duplicates(subset=["ticker", "data"], keep="last")
        .reset_index(drop=True)
    )

    if coluna_issuer is not None:
        df_out = df_out.rename(columns={coluna_issuer: "issuer_code"})
    else:
        df_out["issuer_code"] = pd.NA

    for coluna in ["nome", "setor", "subsetor", "segmento"]:
        df_out = garantir_coluna(df_out, coluna)

    return df_out

def preparar_calendario_euforia(df_calendario, df_caixa_aporte):
    df_cal = df_calendario.copy()
    df_cal["data_sinal"] = pd.to_datetime(df_cal["data_sinal"], errors="coerce")
    df_cal["data_aporte_efetiva"] = pd.to_datetime(df_cal["data_aporte_efetiva"], errors="coerce")
    df_cal["ordem_aporte_estrategia"] = pd.to_numeric(df_cal["ordem_aporte_estrategia"], errors="coerce")
    df_cal = padronizar_texto(df_cal, colunas_lower=["tipo_estrategia"])

    df_cal = df_cal.loc[df_cal["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO].copy()

    df_meta = (
        df_caixa_aporte.loc[df_caixa_aporte["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO]
        [
            [
                "request_id",
                "estrategia_id",
                "grupo_controle",
                "familia_estrategia",
                "estrategia_referencia",
                "tipo_estrategia",
                "data_sinal",
                "data_aporte_efetiva",
                "ordem_aporte_estrategia",
                "capital_inicial_estrategia",
                "valor_aporte_estrategia",
            ]
        ]
        .drop_duplicates(subset=["request_id"])
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    if df_meta.empty:
        raise ValueError("Não foi possível localizar os metadados operacionais da euforia na etapa 8.4.")

    colunas_merge = ["tipo_estrategia", "data_aporte_efetiva", "ordem_aporte_estrategia"]

    df_saida = (
        df_cal.merge(
            df_meta,
            on=colunas_merge,
            how="left",
            validate="one_to_one",
            suffixes=("", "_meta"),
        )
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    n_sem_request = int(df_saida["request_id"].isna().sum())
    if n_sem_request > 0:
        raise ValueError(f"{n_sem_request} linhas do calendário de euforia não encontraram request_id correspondente na etapa 8.4.")

    return df_saida

def preparar_base_caixa_aporte(df_caixa):
    df_out = df_caixa.copy()
    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["valor_total_investido"] = pd.to_numeric(df_out["valor_total_investido"], errors="coerce")
    df_out["caixa_residual_final_aporte"] = pd.to_numeric(df_out["caixa_residual_final_aporte"], errors="coerce")
    df_out["valor_caixa_trazido_aporte"] = pd.to_numeric(df_out["valor_caixa_trazido_aporte"], errors="coerce")
    df_out["saldo_caixa_final_periodo"] = pd.to_numeric(df_out["saldo_caixa_final_periodo"], errors="coerce")

    for coluna in ["carteira_id", "replica_id", "estrategia_referencia", "familia_estrategia", "grupo_controle"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out = padronizar_texto(df_out, colunas_lower=["tipo_estrategia"])
    df_out = df_out.loc[df_out["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO].copy()

    return (
        df_out
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .drop_duplicates(subset=["request_id"], keep="last")
        .reset_index(drop=True)
    )

def preparar_base_movimentacao_caixa(df_mov):
    df_out = df_mov.copy()
    coluna_data = detectar_coluna(df_out, ["data", "date"])
    coluna_saldo = detectar_coluna(df_out, ["saldo_caixa_fim_dia", "saldo_caixa", "saldo_caixa_final_dia", "saldo_caixa_final_periodo"])
    coluna_rendimento = detectar_coluna(df_out, ["rendimento_cdi_dia"], obrigatoria=False)
    coluna_taxa = detectar_coluna(df_out, ["taxa_cdi_dia"], obrigatoria=False)
    coluna_flag_evento = detectar_coluna(df_out, ["flag_evento_aporte"], obrigatoria=False)

    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_saldo] = pd.to_numeric(df_out[coluna_saldo], errors="coerce")

    if coluna_rendimento is not None:
        df_out[coluna_rendimento] = pd.to_numeric(df_out[coluna_rendimento], errors="coerce")
    else:
        df_out["rendimento_cdi_dia"] = 0.0
        coluna_rendimento = "rendimento_cdi_dia"

    if coluna_taxa is not None:
        df_out[coluna_taxa] = pd.to_numeric(df_out[coluna_taxa], errors="coerce")
    else:
        df_out["taxa_cdi_dia"] = 0.0
        coluna_taxa = "taxa_cdi_dia"

    if coluna_flag_evento is None:
        df_out["flag_evento_aporte"] = False
        coluna_flag_evento = "flag_evento_aporte"

    for coluna in ["request_id", "estrategia_id", "grupo_controle", "familia_estrategia", "estrategia_referencia", "tipo_estrategia", "ordem_aporte_estrategia", "valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte", "capital_inicial_estrategia"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["valor_investido_efetivo_aporte"] = pd.to_numeric(df_out["valor_investido_efetivo_aporte"], errors="coerce")
    df_out["valor_caixa_residual_aporte"] = pd.to_numeric(df_out["valor_caixa_residual_aporte"], errors="coerce")
    df_out["valor_caixa_trazido_aporte"] = pd.to_numeric(df_out["valor_caixa_trazido_aporte"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")

    df_out = padronizar_texto(df_out, colunas_lower=["tipo_estrategia"])
    df_out = (
        df_out.loc[df_out["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO]
        .rename(columns={coluna_data: "data", coluna_saldo: "saldo_caixa_fim_dia", coluna_rendimento: "rendimento_cdi_dia", coluna_taxa: "taxa_cdi_dia", coluna_flag_evento: "flag_evento_aporte"})
        .sort_values(["data", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_compras(df_compras):
    df_out = df_compras.copy()
    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["ordem_compra_no_aporte"] = pd.to_numeric(df_out["ordem_compra_no_aporte"], errors="coerce")
    df_out["preco_compra_planejado"] = pd.to_numeric(df_out["preco_compra_planejado"], errors="coerce")
    df_out["quantidade_comprada_planejada"] = pd.to_numeric(df_out["quantidade_comprada_planejada"], errors="coerce")
    df_out["valor_investido_planejado"] = pd.to_numeric(df_out["valor_investido_planejado"], errors="coerce")
    df_out["peso_efetivo_compra_planejada"] = pd.to_numeric(df_out["peso_efetivo_compra_planejada"], errors="coerce")
    df_out["valor_aporte_na_data"] = pd.to_numeric(df_out["valor_aporte_na_data"], errors="coerce")
    df_out["valor_orcamento_alvo_ticker"] = pd.to_numeric(df_out["valor_orcamento_alvo_ticker"], errors="coerce")

    df_out = garantir_coluna(df_out, "quantidade_extra_redistribuicao", 0.0)
    df_out["quantidade_extra_redistribuicao"] = pd.to_numeric(df_out["quantidade_extra_redistribuicao"], errors="coerce").fillna(0.0)
    df_out["quantidade_unidades_extra_redistribuicao"] = df_out["quantidade_extra_redistribuicao"]

    for coluna in ["carteira_id", "replica_id", "estrategia_referencia"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out = padronizar_texto(df_out, colunas_upper=["ticker"], colunas_lower=["tipo_estrategia"])
    df_out = (
        df_out.loc[df_out["tipo_estrategia"] == TIPO_ESTRATEGIA_ALVO]
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia", "ordem_compra_no_aporte", "ticker"])
        .reset_index(drop=True)
    )

    return df_out

def construir_calendario_diario(df_mercado_pad, df_mov_caixa_alvo, data_inicio, data_fim):
    datas_mercado = (
        df_mercado_pad.loc[(df_mercado_pad["data"] >= data_inicio) & (df_mercado_pad["data"] <= data_fim), "data"]
        .drop_duplicates()
        .sort_values()
    )

    datas_caixa = (
        df_mov_caixa_alvo.loc[(df_mov_caixa_alvo["data"] >= data_inicio) & (df_mov_caixa_alvo["data"] <= data_fim), "data"]
        .drop_duplicates()
        .sort_values()
    )

    calendario = pd.Series(pd.Index(sorted(set(datas_mercado.tolist()) | set(datas_caixa.tolist()))), name="data")
    return pd.to_datetime(calendario)

def construir_base_caixa_diaria_backtest(df_caixa_aporte_alvo, df_mov_caixa_alvo, calendario_diario):
    if df_mov_caixa_alvo.empty:
        raise ValueError("A base diária de caixa da etapa 8.4 não possui registros para a estratégia de euforia.")

    estrategia_id = str(df_caixa_aporte_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_caixa_aporte_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0])
    tipo_estrategia = str(df_caixa_aporte_alvo["tipo_estrategia"].iloc[0])
    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])

    df_cash_daily = (
        df_mov_caixa_alvo
        .sort_values(["data", "ordem_aporte_estrategia"])
        .groupby("data", as_index=False, dropna=False)
        .agg(
            request_id=("request_id", "last"),
            ordem_aporte_estrategia=("ordem_aporte_estrategia", "last"),
            valor_aporte_estrategia=("valor_aporte_estrategia", "sum"),
            valor_investido_efetivo_aporte=("valor_investido_efetivo_aporte", "sum"),
            valor_caixa_residual_aporte=("valor_caixa_residual_aporte", "sum"),
            valor_caixa_trazido_aporte=("valor_caixa_trazido_aporte", "sum"),
            estrategia_referencia=("estrategia_referencia", "last"),
            saldo_caixa_fim_dia=("saldo_caixa_fim_dia", "last"),
            rendimento_cdi_dia=("rendimento_cdi_dia", "sum"),
            taxa_cdi_dia=("taxa_cdi_dia", "last"),
            flag_evento_aporte=("flag_evento_aporte", "max"),
        )
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_out = pd.DataFrame({"data": calendario_diario}).merge(df_cash_daily, on="data", how="left")
    df_out["saldo_caixa_fim_dia"] = pd.to_numeric(df_out["saldo_caixa_fim_dia"], errors="coerce").ffill().fillna(0.0)
    for coluna in ["valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte", "rendimento_cdi_dia", "taxa_cdi_dia"]:
        df_out[coluna] = pd.to_numeric(df_out[coluna], errors="coerce").fillna(0.0)

    df_out["flag_evento_aporte"] = df_out["flag_evento_aporte"].fillna(False)
    df_out["request_id"] = df_out["request_id"].astype("string")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")

    df_out["estrategia_id"] = estrategia_id
    df_out["grupo_controle"] = grupo_controle
    df_out["familia_estrategia"] = familia_estrategia
    df_out["estrategia_referencia"] = estrategia_referencia
    df_out["tipo_estrategia"] = tipo_estrategia
    df_out["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_out["saldo_caixa"] = df_out["saldo_caixa_fim_dia"]

    return df_out

def construir_base_posicoes_diarias(df_mercado_pad, df_compras_alvo, calendario_diario, capital_inicial_estrategia):
    # O capital inicial da estratégia deve vir do calendário/base operacional da estratégia,
    # e não da soma linha a linha da base de compras, para manter consistência com o notebook
    # e com as bases de auditoria e patrimônio.
    estrategia_id = str(df_compras_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_compras_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_compras_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_compras_alvo["estrategia_referencia"].iloc[0])
    tipo_estrategia = str(df_compras_alvo["tipo_estrategia"].iloc[0])
    capital_inicial_estrategia = float(capital_inicial_estrategia)
    data_inicio_backtest = pd.Timestamp(df_compras_alvo["data_aporte_efetiva"].min())
    data_fim_backtest = pd.Timestamp(calendario_diario.max())

    df_eventos = (
        df_compras_alvo
        .groupby(["ticker", "data_aporte_efetiva"], as_index=False, dropna=False)
        .agg(
            issuer_code=("issuer_code", "first"),
            nome=("nome", "first"),
            setor=("setor", "first"),
            subsetor=("subsetor", "first"),
            segmento=("segmento", "first"),
            quantidade_evento=("quantidade_comprada_planejada", "sum"),
            valor_investido_evento=("valor_investido_planejado", "sum"),
            preco_compra_evento=("preco_compra_planejado", "last"),
        )
        .rename(columns={"data_aporte_efetiva": "data"})
        .sort_values(["ticker", "data"])
        .reset_index(drop=True)
    )

    tickers_alvo = sorted(df_eventos["ticker"].dropna().unique().tolist())

    df_grade = pd.MultiIndex.from_product(
        [calendario_diario, tickers_alvo],
        names=["data", "ticker"],
    ).to_frame(index=False)

    df_meta_ticker = (
        df_eventos[["ticker", "issuer_code", "nome", "setor", "subsetor", "segmento", "preco_compra_evento"]]
        .drop_duplicates(subset=["ticker"], keep="first")
        .reset_index(drop=True)
    )

    df_precos = (
        df_mercado_pad.loc[df_mercado_pad["ticker"].isin(tickers_alvo), ["data", "ticker", "close_adj"]]
        .drop_duplicates(subset=["data", "ticker"], keep="last")
        .sort_values(["ticker", "data"])
        .reset_index(drop=True)
    )

    df_grade = df_grade.merge(df_precos, on=["data", "ticker"], how="left")
    df_grade = df_grade.merge(df_meta_ticker, on="ticker", how="left")
    df_grade["close_adj"] = (
        df_grade
        .sort_values(["ticker", "data"])
        .groupby("ticker")["close_adj"]
        .ffill()
    )
    df_grade["close_adj"] = df_grade["close_adj"].fillna(df_grade["preco_compra_evento"])

    df_grade = df_grade.merge(
        df_eventos[["ticker", "data", "quantidade_evento", "valor_investido_evento"]],
        on=["ticker", "data"],
        how="left",
    )

    df_grade["quantidade_evento"] = pd.to_numeric(df_grade["quantidade_evento"], errors="coerce").fillna(0.0)
    df_grade["valor_investido_evento"] = pd.to_numeric(df_grade["valor_investido_evento"], errors="coerce").fillna(0.0)

    df_grade = df_grade.sort_values(["ticker", "data"]).reset_index(drop=True)
    df_grade["quantidade_em_carteira"] = df_grade.groupby("ticker")["quantidade_evento"].cumsum()
    df_grade["valor_investido_acumulado_ticker"] = df_grade.groupby("ticker")["valor_investido_evento"].cumsum()

    df_grade = df_grade.loc[df_grade["quantidade_em_carteira"] > 0].copy()

    df_grade["custo_medio_unitario"] = np.where(
        df_grade["quantidade_em_carteira"] > 0,
        df_grade["valor_investido_acumulado_ticker"] / df_grade["quantidade_em_carteira"],
        0.0,
    )

    df_grade["close_adj"] = df_grade["close_adj"].fillna(df_grade["custo_medio_unitario"])
    df_grade["valor_mercado_posicao"] = df_grade["quantidade_em_carteira"] * df_grade["close_adj"]

    df_grade["estrategia_id"] = estrategia_id
    df_grade["grupo_controle"] = grupo_controle
    df_grade["familia_estrategia"] = familia_estrategia
    df_grade["estrategia_referencia"] = estrategia_referencia
    df_grade["tipo_estrategia"] = tipo_estrategia
    df_grade["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_grade["data_inicio_backtest"] = data_inicio_backtest
    df_grade["data_fim_backtest"] = data_fim_backtest

    colunas_saida = [
        "data",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "quantidade_em_carteira",
        "custo_medio_unitario",
        "close_adj",
        "valor_mercado_posicao",
    ]

    return df_grade[colunas_saida].copy()

def construir_base_patrimonial(df_posicoes_diarias, df_caixa_diaria_backtest, df_caixa_aporte_alvo, calendario_diario):
    estrategia_id = str(df_caixa_aporte_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_caixa_aporte_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0])
    tipo_estrategia = str(df_caixa_aporte_alvo["tipo_estrategia"].iloc[0])
    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])
    data_inicio_backtest = pd.Timestamp(calendario_diario.min())
    data_fim_backtest = pd.Timestamp(calendario_diario.max())

    df_pos_agg = (
        df_posicoes_diarias
        .groupby("data", as_index=False, dropna=False)
        .agg(
            valor_carteira_investida=("valor_mercado_posicao", "sum"),
            n_posicoes_ativas=("ticker", "count"),
            n_tickers_unicos_ativos=("ticker", "nunique"),
        )
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_patrimonio = pd.DataFrame({"data": calendario_diario}).merge(df_pos_agg, on="data", how="left")
    df_patrimonio = df_patrimonio.merge(
        df_caixa_diaria_backtest[
            [
                "data",
                "saldo_caixa_fim_dia",
                "saldo_caixa",
                "rendimento_cdi_dia",
                "taxa_cdi_dia",
                "request_id",
                "ordem_aporte_estrategia",
                "valor_aporte_estrategia",
                "valor_investido_efetivo_aporte",
                "valor_caixa_residual_aporte",
                "valor_caixa_trazido_aporte",
                "flag_evento_aporte",
            ]
        ],
        on="data",
        how="left",
    )

    for coluna in ["valor_carteira_investida", "n_posicoes_ativas", "n_tickers_unicos_ativos"]:
        df_patrimonio[coluna] = pd.to_numeric(df_patrimonio[coluna], errors="coerce").fillna(0.0)

    for coluna in ["saldo_caixa_fim_dia", "saldo_caixa", "rendimento_cdi_dia", "taxa_cdi_dia", "valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte"]:
        df_patrimonio[coluna] = pd.to_numeric(df_patrimonio[coluna], errors="coerce").fillna(0.0)

    df_patrimonio["flag_evento_aporte"] = df_patrimonio["flag_evento_aporte"].fillna(False)
    df_patrimonio["patrimonio_total"] = df_patrimonio["valor_carteira_investida"] + df_patrimonio["saldo_caixa_fim_dia"]

    mapa_aportes = (
        df_caixa_aporte_alvo[["data_aporte_efetiva", "valor_aporte_estrategia"]]
        .groupby("data_aporte_efetiva", as_index=False)["valor_aporte_estrategia"]
        .sum()
        .rename(columns={"data_aporte_efetiva": "data", "valor_aporte_estrategia": "valor_aporte_evento"})
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_patrimonio = df_patrimonio.merge(mapa_aportes, on="data", how="left")
    df_patrimonio["valor_aporte_evento"] = pd.to_numeric(df_patrimonio["valor_aporte_evento"], errors="coerce").fillna(0.0)
    df_patrimonio["valor_total_aportado"] = df_patrimonio["valor_aporte_evento"].cumsum()

    df_patrimonio["max_patrimonio_acumulado"] = df_patrimonio["patrimonio_total"].cummax()
    df_patrimonio["retorno_diario"] = df_patrimonio["patrimonio_total"].pct_change().fillna(0.0)

    if capital_inicial_estrategia > 0:
        df_patrimonio["retorno_acumulado"] = (df_patrimonio["patrimonio_total"] / capital_inicial_estrategia) - 1.0
    else:
        df_patrimonio["retorno_acumulado"] = 0.0

    df_patrimonio["drawdown_atual"] = np.where(
        df_patrimonio["max_patrimonio_acumulado"] > 0,
        (df_patrimonio["patrimonio_total"] / df_patrimonio["max_patrimonio_acumulado"]) - 1.0,
        0.0,
    )
    df_patrimonio["flag_em_drawdown"] = df_patrimonio["drawdown_atual"] < 0.0

    df_patrimonio["estrategia_id"] = estrategia_id
    df_patrimonio["grupo_controle"] = grupo_controle
    df_patrimonio["familia_estrategia"] = familia_estrategia
    df_patrimonio["estrategia_referencia"] = estrategia_referencia
    df_patrimonio["tipo_estrategia"] = tipo_estrategia
    df_patrimonio["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_patrimonio["data_inicio_backtest"] = data_inicio_backtest
    df_patrimonio["data_fim_backtest"] = data_fim_backtest

    return df_patrimonio

def enriquecer_pesos_posicoes(df_posicoes_diarias, df_patrimonio):
    df_out = df_posicoes_diarias.merge(
        df_patrimonio[["data", "patrimonio_total"]],
        on="data",
        how="left",
    )

    df_out["peso_posicao_no_patrimonio"] = np.where(
        df_out["patrimonio_total"] > 0,
        df_out["valor_mercado_posicao"] / df_out["patrimonio_total"],
        0.0,
    )

    return df_out

def executar_motor_backtest_unico(df_mercado_pad, df_calendario_alvo, df_caixa_aporte_alvo, df_mov_caixa_alvo, df_compras_alvo):
    data_inicio = pd.Timestamp(df_mov_caixa_alvo["data"].min())
    data_fim = pd.Timestamp(df_mov_caixa_alvo["data"].max())

    calendario_diario = construir_calendario_diario(
        df_mercado_pad=df_mercado_pad,
        df_mov_caixa_alvo=df_mov_caixa_alvo,
        data_inicio=data_inicio,
        data_fim=data_fim,
    )

    if calendario_diario.empty:
        raise ValueError("O calendário diário do backtest ficou vazio.")

    df_caixa_diaria_backtest = construir_base_caixa_diaria_backtest(
        df_caixa_aporte_alvo=df_caixa_aporte_alvo,
        df_mov_caixa_alvo=df_mov_caixa_alvo,
        calendario_diario=calendario_diario,
    )

    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])

    df_posicoes_diarias = construir_base_posicoes_diarias(
        df_mercado_pad=df_mercado_pad,
        df_compras_alvo=df_compras_alvo,
        calendario_diario=calendario_diario,
        capital_inicial_estrategia=capital_inicial_estrategia,
    )

    df_patrimonio = construir_base_patrimonial(
        df_posicoes_diarias=df_posicoes_diarias,
        df_caixa_diaria_backtest=df_caixa_diaria_backtest,
        df_caixa_aporte_alvo=df_caixa_aporte_alvo,
        calendario_diario=calendario_diario,
    )

    df_posicoes_diarias = enriquecer_pesos_posicoes(
        df_posicoes_diarias=df_posicoes_diarias,
        df_patrimonio=df_patrimonio,
    )

    return df_caixa_diaria_backtest, df_posicoes_diarias, df_patrimonio

print(f"Tolerância do backtest euforia : {TOLERANCIA_BACKTEST}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

df_mercado_pad = preparar_base_mercado(df_mercado_diario_consolidada)
df_caixa_aporte_alvo = preparar_base_caixa_aporte(df_base_caixa_residual_aporte)
df_calendario_alvo = preparar_calendario_euforia(df_calendario_aportes_euforia, df_caixa_aporte_alvo)
df_mov_caixa_alvo = preparar_base_movimentacao_caixa(df_movimentacao_caixa_diaria)
df_compras_alvo = preparar_base_compras(df_compras_planejadas_carteira)

if df_compras_alvo.empty:
    raise ValueError("A base de compras planejadas não possui registros para a estratégia de euforia.")
if df_caixa_aporte_alvo.empty:
    raise ValueError("A base de caixa por aporte não possui registros para a estratégia de euforia.")
if df_mov_caixa_alvo.empty:
    raise ValueError("A base diária de caixa não possui registros para a estratégia de euforia.")
if df_calendario_alvo.empty:
    raise ValueError("O calendário da estratégia de euforia ficou vazio após a preparação.")

n_aportes_euforia = int(df_calendario_alvo["request_id"].nunique())
n_compras_euforia = int(len(df_compras_alvo))
n_tickers_euforia = int(df_compras_alvo["ticker"].nunique())

n_estrategia_referencia_ausente_inputs = int(
    df_caixa_aporte_alvo["estrategia_referencia"].isna().sum()
    + df_compras_alvo["estrategia_referencia"].isna().sum()
)

print(f"Compras da estratégia de euforia        : {n_compras_euforia:,}")
print(f"Aportes da estratégia de euforia        : {n_aportes_euforia:,}")
print(f"Tickers comprados na estratégia             : {n_tickers_euforia:,}")
print(f"Estrategia_referencia ausente nos inputs    : {n_estrategia_referencia_ausente_inputs:,}")
print("OK")

# ============================================================
# 6) Execução do motor de backtest da estratégia de euforia
# ============================================================

print("\n[6/10] Execução do motor de backtest da estratégia de euforia...")

df_caixa_euforia_diaria, df_posicoes_euforia, df_patrimonio_euforia = executar_motor_backtest_unico(
    df_mercado_pad=df_mercado_pad,
    df_calendario_alvo=df_calendario_alvo,
    df_caixa_aporte_alvo=df_caixa_aporte_alvo,
    df_mov_caixa_alvo=df_mov_caixa_alvo,
    df_compras_alvo=df_compras_alvo,
)

df_compras_euforia = df_compras_alvo.copy()

n_ref_ausente_outputs = int(
    df_compras_euforia["estrategia_referencia"].isna().sum()
    + df_posicoes_euforia["estrategia_referencia"].isna().sum()
    + df_caixa_euforia_diaria["estrategia_referencia"].isna().sum()
    + df_patrimonio_euforia["estrategia_referencia"].isna().sum()
)

print(f"Linhas da base de compras da euforia   : {len(df_compras_euforia):,}")
print(f"Linhas da base de posições da euforia  : {len(df_posicoes_euforia):,}")
print(f"Linhas da base de caixa diária             : {len(df_caixa_euforia_diaria):,}")
print(f"Linhas da base patrimonial                 : {len(df_patrimonio_euforia):,}")
print(f"Data inicial da curva patrimonial          : {pd.Timestamp(df_patrimonio_euforia['data'].min()).date()}")
print(f"Data final da curva patrimonial            : {pd.Timestamp(df_patrimonio_euforia['data'].max()).date()}")
print(f"Estrategia_referencia ausente nos outputs  : {n_ref_ausente_outputs:,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais, resumos e auditorias
# ============================================================

print("\n[7/10] Construção das tabelas formais, resumos e auditorias...")

df_regras_backtest_euforia = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "BT1",
            "escopo": "motor_da_estrategia",
            "regra_operacional": "executar_os_aportes_da_euforia_nas_datas_definidas_no_calendario",
            "detalhe": "O backtest da euforia utiliza exclusivamente o calendário de aportes da estratégia de euforia como agenda de execução.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "BT2",
            "escopo": "compras_e_posicoes",
            "regra_operacional": "registrar_cada_compra_e_propagar_quantidades_diariamente_por_ticker",
            "detalhe": "As compras executadas alimentam a base diária de posições com propagação diária por ticker, carregamento do último preço disponível e fallback para o preço de compra na data do aporte quando necessário.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "BT3",
            "escopo": "caixa",
            "regra_operacional": "incorporar_o_saldo_de_caixa_completo_diario_remunerado_pelo_cdi_ate_a_data_final_do_backtest",
            "detalhe": "O caixa diário utilizado no backtest representa o capital não alocado em ações, remunerado pelo CDI desde o início comum da trilha até cada data do calendário patrimonial.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "BT4",
            "escopo": "curva_patrimonial",
            "regra_operacional": "consolidar_diariamente_valor_investido_caixa_e_patrimonio_total",
            "detalhe": "A curva patrimonial diária é a soma do valor de mercado das posições e do saldo de caixa da estratégia.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "BT5",
            "escopo": "auditoria",
            "regra_operacional": "reconciliar_capital_inicial_compras_caixa_completo_e_rendimento_cdi_do_proprio_backtest",
            "detalhe": "A auditoria deve reconciliar capital inicial, compras efetivas, saldo de caixa completo e rendimento CDI efetivamente refletido na trilha diária do backtest.",
        },
    ]
)

capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])
valor_total_aportado = float(df_caixa_aporte_alvo["valor_aporte_estrategia"].sum())
valor_total_investido = float(df_compras_euforia["valor_investido_planejado"].sum())
saldo_caixa_final = float(df_caixa_euforia_diaria["saldo_caixa_fim_dia"].iloc[-1])
patrimonio_final = float(df_patrimonio_euforia["patrimonio_total"].iloc[-1])
retorno_acumulado_final = float(df_patrimonio_euforia["retorno_acumulado"].iloc[-1])
drawdown_maximo = float(df_patrimonio_euforia["drawdown_atual"].min())
n_posicoes_ativas_finais = int(df_posicoes_euforia.loc[df_posicoes_euforia["data"] == df_patrimonio_euforia["data"].max(), "ticker"].nunique())

valor_caixa_residual_total_aportes = float(df_caixa_aporte_alvo["caixa_residual_final_aporte"].sum())
valor_rendimento_cdi_acumulado_final = float(df_caixa_euforia_diaria["rendimento_cdi_dia"].sum())
desvio_fluxo_caixa_completo = abs(
    saldo_caixa_final
    - (capital_inicial_estrategia - valor_total_investido + valor_rendimento_cdi_acumulado_final)
)
desvio_aportes_planejados = abs(valor_total_aportado - (valor_total_investido + valor_caixa_residual_total_aportes))

df_resumo_backtest_euforia = pd.DataFrame(
    [
        {
            "estrategia_id": str(df_caixa_aporte_alvo["estrategia_id"].iloc[0]),
            "grupo_controle": str(df_caixa_aporte_alvo["grupo_controle"].iloc[0]),
            "familia_estrategia": str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0]),
            "estrategia_referencia": str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0]),
            "tipo_estrategia": TIPO_ESTRATEGIA_ALVO,
            "data_inicio_backtest": pd.Timestamp(df_patrimonio_euforia["data"].min()),
            "data_fim_backtest": pd.Timestamp(df_patrimonio_euforia["data"].max()),
            "n_aportes": int(df_caixa_aporte_alvo["request_id"].nunique()),
            "n_compras": int(len(df_compras_euforia)),
            "n_tickers_unicos": int(df_compras_euforia["ticker"].nunique()),
            "n_datas_curva_patrimonial": int(len(df_patrimonio_euforia)),
            "valor_total_aportado": valor_total_aportado,
            "valor_total_investido": valor_total_investido,
            "saldo_caixa_final": saldo_caixa_final,
            "patrimonio_final": patrimonio_final,
            "retorno_acumulado_final": retorno_acumulado_final,
            "drawdown_maximo": drawdown_maximo,
            "n_posicoes_ativas_finais": n_posicoes_ativas_finais,
        }
    ]
)

df_auditoria_backtest_euforia = pd.DataFrame(
    [
        {
            "estrategia_id": str(df_caixa_aporte_alvo["estrategia_id"].iloc[0]),
            "grupo_controle": str(df_caixa_aporte_alvo["grupo_controle"].iloc[0]),
            "familia_estrategia": str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0]),
            "estrategia_referencia": str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0]),
            "tipo_estrategia": TIPO_ESTRATEGIA_ALVO,
            "capital_inicial_estrategia": capital_inicial_estrategia,
            "valor_total_aportado": valor_total_aportado,
            "valor_total_investido": valor_total_investido,
            "valor_caixa_residual_total_aportes": valor_caixa_residual_total_aportes,
            "valor_rendimento_cdi_acumulado_final": valor_rendimento_cdi_acumulado_final,
            "saldo_caixa_final": saldo_caixa_final,
            "patrimonio_final": patrimonio_final,
            "desvio_aportes_planejados_vs_investido_mais_residual": desvio_aportes_planejados,
            "desvio_fluxo_caixa_completo": desvio_fluxo_caixa_completo,
            "flag_reconciliacao_aportes_planejados": desvio_aportes_planejados <= TOLERANCIA_BACKTEST,
            "flag_reconciliacao_caixa_completo": desvio_fluxo_caixa_completo <= TOLERANCIA_BACKTEST,
            "flag_patrimonio_final_nao_negativo": patrimonio_final >= -TOLERANCIA_BACKTEST,
            "flag_posicoes_nao_negativas": bool((df_posicoes_euforia["quantidade_em_carteira"] >= -TOLERANCIA_BACKTEST).all()),
            "flag_caixa_nao_negativo": bool((df_caixa_euforia_diaria["saldo_caixa_fim_dia"] >= -TOLERANCIA_BACKTEST).all()),
            "flag_estrategia_referencia_preenchida": bool(n_ref_ausente_outputs == 0),
        }
    ]
)

df_inconsistencias_backtest_euforia = (
    df_auditoria_backtest_euforia.loc[
        (~df_auditoria_backtest_euforia["flag_reconciliacao_aportes_planejados"])
        | (~df_auditoria_backtest_euforia["flag_reconciliacao_caixa_completo"])
        | (~df_auditoria_backtest_euforia["flag_patrimonio_final_nao_negativo"])
        | (~df_auditoria_backtest_euforia["flag_posicoes_nao_negativas"])
        | (~df_auditoria_backtest_euforia["flag_caixa_nao_negativo"])
        | (~df_auditoria_backtest_euforia["flag_estrategia_referencia_preenchida"])
    ].copy()
)

df_distribuicao_anual_backtest_euforia = (
    df_patrimonio_euforia
    .assign(ano=lambda df: pd.to_datetime(df["data"], errors="coerce").dt.year)
    .groupby(["estrategia_id", "grupo_controle", "familia_estrategia", "estrategia_referencia", "tipo_estrategia", "ano"], as_index=False)
    .agg(
        n_datas_curva_patrimonial=("data", "count"),
        patrimonio_medio_ano=("patrimonio_total", "mean"),
        valor_carteira_medio_ano=("valor_carteira_investida", "mean"),
        saldo_caixa_medio_ano=("saldo_caixa_fim_dia", "mean"),
        patrimonio_final_ano=("patrimonio_total", "last"),
        saldo_caixa_final_ano=("saldo_caixa_fim_dia", "last"),
        drawdown_minimo_ano=("drawdown_atual", "min"),
        retorno_ultimo_dia_ano=("retorno_acumulado", "last"),
    )
    .sort_values(["ano"])
    .reset_index(drop=True)
)

print(f"Inconsistências do backtest identificadas   : {len(df_inconsistencias_backtest_euforia):,}")
print(f"Desvio do fluxo de caixa completo           : {desvio_fluxo_caixa_completo:.10f}")
print(f"Estrategia_referencia ausente nos outputs   : {n_ref_ausente_outputs:,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_backtest_euforia, caminho_tbl_regras_backtest_euforia, index=False)
salvar_dataframe(df_compras_euforia, caminho_base_compras_euforia, index=False)
salvar_dataframe(df_posicoes_euforia, caminho_base_posicoes_euforia, index=False)
salvar_dataframe(df_caixa_euforia_diaria, caminho_base_caixa_euforia_diaria, index=False)
salvar_dataframe(df_patrimonio_euforia, caminho_base_patrimonio_euforia, index=False)
salvar_dataframe(df_resumo_backtest_euforia, caminho_tbl_resumo_backtest_euforia, index=False)
salvar_dataframe(df_auditoria_backtest_euforia, caminho_tbl_auditoria_backtest_euforia, index=False)
salvar_dataframe(df_inconsistencias_backtest_euforia, caminho_tbl_inconsistencias_backtest_euforia, index=False)
salvar_dataframe(df_distribuicao_anual_backtest_euforia, caminho_tbl_distribuicao_anual_backtest_euforia, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_backtest_euforia = df_regras_backtest_euforia.copy()
df_amostra_compras_euforia = df_compras_euforia.head(20).copy()
df_amostra_posicoes_euforia = df_posicoes_euforia.head(20).copy()
df_amostra_caixa_euforia_diaria = df_caixa_euforia_diaria.head(20).copy()
df_amostra_patrimonio_euforia = df_patrimonio_euforia.head(20).copy()
df_amostra_resumo_backtest_euforia = df_resumo_backtest_euforia.copy()
df_amostra_auditoria_backtest_euforia = df_auditoria_backtest_euforia.copy()
df_amostra_inconsistencias_backtest_euforia = df_inconsistencias_backtest_euforia.head(20).copy()
df_amostra_distribuicao_anual_backtest_euforia = df_distribuicao_anual_backtest_euforia.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais do backtest de euforia:")
print(df_amostra_regras_backtest_euforia.to_string(index=False))

print("\nBase de compras da estratégia de euforia - amostra:")
print(df_amostra_compras_euforia.to_string(index=False))

print("\nBase diária de posições da estratégia de euforia - amostra:")
print(df_amostra_posicoes_euforia.to_string(index=False))

print("\nBase diária de caixa da estratégia de euforia - amostra:")
print(df_amostra_caixa_euforia_diaria.to_string(index=False))

print("\nBase patrimonial da estratégia de euforia - amostra:")
print(df_amostra_patrimonio_euforia.to_string(index=False))

print("\nResumo do backtest de euforia:")
print(df_amostra_resumo_backtest_euforia.to_string(index=False))

print("\nAuditoria do backtest de euforia:")
print(df_amostra_auditoria_backtest_euforia.to_string(index=False))

print("\nInconsistências do backtest de euforia - amostra:")
print(df_amostra_inconsistencias_backtest_euforia.to_string(index=False))

print("\nDistribuição anual do backtest de euforia - amostra:")
print(df_amostra_distribuicao_anual_backtest_euforia.to_string(index=False))

print("\nArquivos salvos na subetapa 9.2:")
print(f"- {caminho_tbl_regras_backtest_euforia}")
print(f"- {caminho_base_compras_euforia}")
print(f"- {caminho_base_posicoes_euforia}")
print(f"- {caminho_base_caixa_euforia_diaria}")
print(f"- {caminho_base_patrimonio_euforia}")
print(f"- {caminho_tbl_resumo_backtest_euforia}")
print(f"- {caminho_tbl_auditoria_backtest_euforia}")
print(f"- {caminho_tbl_inconsistencias_backtest_euforia}")
print(f"- {caminho_tbl_distribuicao_anual_backtest_euforia}")

print("\nETAPA 9.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 9.2 - BACKTEST DA ESTRATÉGIA DE EUFORIA

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidado          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - calendário de aportes euforia       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_4_base_calendario_aportes_euforia.parquet
Entrada - base de caixa por aporte            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_4_base_caixa_residual_aporte.parquet
Entrada - base diária de caixa                : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etap

## Etapa 9.3) Backtest das Estratégias Aleatórias de Controle

In [42]:
%%time
# ============================================================
# Etapa 9.3) Backtest das Estratégias Aleatórias de Controle
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 9.3 - BACKTEST DAS ESTRATÉGIAS ALEATÓRIAS DE CONTROLE")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_calendarios_aleatorios_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=5,
    tipo_arquivo="base",
    nome="calendarios_aleatorios_controle",
)

caminho_base_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="caixa_residual_aporte",
)

caminho_base_movimentacao_caixa_diaria = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="movimentacao_caixa_diaria",
)

caminho_base_compras_planejadas_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="base",
    nome="compras_planejadas_carteira",
)

caminho_tbl_regras_backtest_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="regras_backtest_aleatorias_controle",
)

caminho_base_compras_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="compras_aleatorias_controle",
)

caminho_base_posicoes_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="posicoes_aleatorias_controle",
)

caminho_base_caixa_aleatorias_controle_diaria = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="caixa_aleatorias_controle_diaria",
)

caminho_base_patrimonio_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="patrimonio_aleatorias_controle",
)

caminho_tbl_resumo_backtest_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="resumo_backtest_aleatorias_controle",
)

caminho_tbl_auditoria_backtest_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="auditoria_backtest_aleatorias_controle",
)

caminho_tbl_inconsistencias_backtest_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="inconsistencias_backtest_aleatorias_controle",
)

caminho_tbl_distribuicao_anual_backtest_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_backtest_aleatorias_controle",
)

print(f"Entrada - mercado diário consolidado          : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - calendários aleatórios de controle  : {caminho_base_calendarios_aleatorios_controle}")
print(f"Entrada - base de caixa por aporte            : {caminho_base_caixa_residual_aporte}")
print(f"Entrada - base diária de caixa                : {caminho_base_movimentacao_caixa_diaria}")
print(f"Entrada - base de compras planejadas          : {caminho_base_compras_planejadas_carteira}")
print(f"Saída   - regras do backtest                  : {caminho_tbl_regras_backtest_aleatorias_controle}")
print(f"Saída   - base de compras das estratégias     : {caminho_base_compras_aleatorias_controle}")
print(f"Saída   - base diária de posições             : {caminho_base_posicoes_aleatorias_controle}")
print(f"Saída   - base diária de caixa                : {caminho_base_caixa_aleatorias_controle_diaria}")
print(f"Saída   - base diária patrimonial             : {caminho_base_patrimonio_aleatorias_controle}")
print(f"Saída   - resumo do backtest                  : {caminho_tbl_resumo_backtest_aleatorias_controle}")
print(f"Saída   - auditoria do backtest               : {caminho_tbl_auditoria_backtest_aleatorias_controle}")
print(f"Saída   - inconsistências do backtest         : {caminho_tbl_inconsistencias_backtest_aleatorias_controle}")
print(f"Saída   - distribuição anual                  : {caminho_tbl_distribuicao_anual_backtest_aleatorias_controle}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_calendarios_aleatorios_controle = pd.read_parquet(caminho_base_calendarios_aleatorios_controle)
df_base_caixa_residual_aporte = pd.read_parquet(caminho_base_caixa_residual_aporte)
df_movimentacao_caixa_diaria = pd.read_parquet(caminho_base_movimentacao_caixa_diaria)
df_compras_planejadas_carteira = pd.read_parquet(caminho_base_compras_planejadas_carteira)

print(f"Mercado diário consolidado                : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Calendários aleatórios de controle        : {df_calendarios_aleatorios_controle.shape[0]:,} linhas x {df_calendarios_aleatorios_controle.shape[1]} colunas")
print(f"Base de caixa por aporte                  : {df_base_caixa_residual_aporte.shape[0]:,} linhas x {df_base_caixa_residual_aporte.shape[1]} colunas")
print(f"Base diária de caixa                      : {df_movimentacao_caixa_diaria.shape[0]:,} linhas x {df_movimentacao_caixa_diaria.shape[1]} colunas")
print(f"Base de compras planejadas                : {df_compras_planejadas_carteira.shape[0]:,} linhas x {df_compras_planejadas_carteira.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_BACKTEST = 0.000001
GRUPO_CONTROLE_ALVO = "aleatoria"
TIPOS_ESTRATEGIA_ALEATORIA_VALIDOS = ["aleatoria_capitulacao", "aleatoria_euforia"]

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def detectar_coluna(df, candidatos, obrigatoria=True):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    if obrigatoria:
        raise ValueError(f"Não foi possível localizar nenhuma das colunas candidatas: {candidatos}")
    return None

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

def padronizar_texto(df, colunas_upper=None, colunas_lower=None):
    colunas_upper = [] if colunas_upper is None else colunas_upper
    colunas_lower = [] if colunas_lower is None else colunas_lower

    for coluna in colunas_upper:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.upper()

    for coluna in colunas_lower:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.lower()

    return df

def extrair_replica_id_de_estrategia_id(serie):
    return (
        serie.astype("string")
        .str.extract(r"replica_(\d+)", expand=False)
        .astype("Float64")
    )

def preparar_base_mercado(df_mercado):
    coluna_data = detectar_coluna(df_mercado, ["data", "date", "dt_ref"])
    coluna_ticker = detectar_coluna(df_mercado, ["ticker", "codneg", "ticker_ajustado"])
    coluna_preco = detectar_coluna(df_mercado, ["close_adj", "adj_close", "close", "preco_fechamento"])
    coluna_issuer = detectar_coluna(df_mercado, ["issuer_code", "codigo_empresa", "codigo_emissor"], obrigatoria=False)

    df_out = df_mercado.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_preco] = pd.to_numeric(df_out[coluna_preco], errors="coerce")
    df_out = padronizar_texto(df_out, colunas_upper=[coluna_ticker])

    colunas_saida = [coluna_data, coluna_ticker, coluna_preco]
    if coluna_issuer is not None:
        colunas_saida.append(coluna_issuer)

    for coluna in ["nome", "setor", "subsetor", "segmento"]:
        if coluna in df_out.columns:
            colunas_saida.append(coluna)

    df_out = (
        df_out[colunas_saida]
        .rename(columns={coluna_data: "data", coluna_ticker: "ticker", coluna_preco: "close_adj"})
        .dropna(subset=["data", "ticker", "close_adj"])
        .sort_values(["ticker", "data"])
        .drop_duplicates(subset=["ticker", "data"], keep="last")
        .reset_index(drop=True)
    )

    if coluna_issuer is not None:
        df_out = df_out.rename(columns={coluna_issuer: "issuer_code"})
    else:
        df_out["issuer_code"] = pd.NA

    for coluna in ["nome", "setor", "subsetor", "segmento"]:
        df_out = garantir_coluna(df_out, coluna)

    return df_out

def preparar_calendarios_aleatorios(df_cal):
    colunas_obrigatorias = [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "replica_id",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_aporte_estrategia",
        "valor_aporte_estrategia",
        "capital_inicial_estrategia",
    ]
    validar_colunas_obrigatorias(df_cal, colunas_obrigatorias)

    df_out = df_cal.copy()
    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")

    df_out = padronizar_texto(
        df_out,
        colunas_upper=[],
        colunas_lower=["grupo_controle", "familia_estrategia", "estrategia_referencia", "tipo_estrategia"],
    )

    df_out = (
        df_out.loc[
            (df_out["grupo_controle"] == GRUPO_CONTROLE_ALVO)
            & (df_out["tipo_estrategia"].isin(TIPOS_ESTRATEGIA_ALEATORIA_VALIDOS))
        ]
        .sort_values(["tipo_estrategia", "replica_id", "data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_caixa_aporte(df_caixa):
    colunas_obrigatorias = [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_aporte_estrategia",
        "capital_inicial_estrategia",
        "valor_aporte_estrategia",
        "valor_total_investido",
        "caixa_residual_final_aporte",
        "valor_caixa_trazido_aporte",
        "saldo_caixa_final_periodo",
    ]
    validar_colunas_obrigatorias(df_caixa, colunas_obrigatorias)

    df_out = df_caixa.copy()
    for coluna in ["replica_id", "estrategia_referencia", "carteira_id"]:
        df_out = garantir_coluna(df_out, coluna)

    if df_out["replica_id"].isna().all():
        df_out["replica_id"] = extrair_replica_id_de_estrategia_id(df_out["estrategia_id"])

    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["valor_total_investido"] = pd.to_numeric(df_out["valor_total_investido"], errors="coerce")
    df_out["caixa_residual_final_aporte"] = pd.to_numeric(df_out["caixa_residual_final_aporte"], errors="coerce")
    df_out["valor_caixa_trazido_aporte"] = pd.to_numeric(df_out["valor_caixa_trazido_aporte"], errors="coerce")
    df_out["saldo_caixa_final_periodo"] = pd.to_numeric(df_out["saldo_caixa_final_periodo"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")

    df_out = padronizar_texto(
        df_out,
        colunas_upper=[],
        colunas_lower=["grupo_controle", "familia_estrategia", "estrategia_referencia", "tipo_estrategia"],
    )

    df_out = (
        df_out.loc[
            (df_out["grupo_controle"] == GRUPO_CONTROLE_ALVO)
            & (df_out["tipo_estrategia"].isin(TIPOS_ESTRATEGIA_ALEATORIA_VALIDOS))
        ]
        .sort_values(["tipo_estrategia", "replica_id", "data_aporte_efetiva", "ordem_aporte_estrategia"])
        .drop_duplicates(subset=["request_id"], keep="last")
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_movimentacao_caixa(df_mov):
    coluna_data = detectar_coluna(df_mov, ["data", "date"])
    coluna_saldo = detectar_coluna(df_mov, ["saldo_caixa_fim_dia", "saldo_caixa", "saldo_caixa_final_dia", "saldo_caixa_final_periodo"])
    coluna_rendimento = detectar_coluna(df_mov, ["rendimento_cdi_dia"], obrigatoria=False)
    coluna_taxa = detectar_coluna(df_mov, ["taxa_cdi_dia"], obrigatoria=False)
    coluna_flag_evento = detectar_coluna(df_mov, ["flag_evento_aporte"], obrigatoria=False)

    colunas_obrigatorias = ["request_id", "estrategia_id", "grupo_controle", "familia_estrategia", "tipo_estrategia", coluna_data, coluna_saldo]
    validar_colunas_obrigatorias(df_mov, colunas_obrigatorias)

    df_out = df_mov.copy()
    for coluna in ["replica_id", "estrategia_referencia", "carteira_id", "ordem_aporte_estrategia", "valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte", "capital_inicial_estrategia"]:
        df_out = garantir_coluna(df_out, coluna)

    if df_out["replica_id"].isna().all():
        df_out["replica_id"] = extrair_replica_id_de_estrategia_id(df_out["estrategia_id"])

    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_saldo] = pd.to_numeric(df_out[coluna_saldo], errors="coerce")
    if coluna_rendimento is None:
        df_out["rendimento_cdi_dia"] = 0.0
        coluna_rendimento = "rendimento_cdi_dia"
    else:
        df_out[coluna_rendimento] = pd.to_numeric(df_out[coluna_rendimento], errors="coerce")

    if coluna_taxa is None:
        df_out["taxa_cdi_dia"] = 0.0
        coluna_taxa = "taxa_cdi_dia"
    else:
        df_out[coluna_taxa] = pd.to_numeric(df_out[coluna_taxa], errors="coerce")

    if coluna_flag_evento is None:
        df_out["flag_evento_aporte"] = False
        coluna_flag_evento = "flag_evento_aporte"

    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["valor_investido_efetivo_aporte"] = pd.to_numeric(df_out["valor_investido_efetivo_aporte"], errors="coerce")
    df_out["valor_caixa_residual_aporte"] = pd.to_numeric(df_out["valor_caixa_residual_aporte"], errors="coerce")
    df_out["valor_caixa_trazido_aporte"] = pd.to_numeric(df_out["valor_caixa_trazido_aporte"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")

    df_out = padronizar_texto(
        df_out,
        colunas_upper=[],
        colunas_lower=["grupo_controle", "familia_estrategia", "estrategia_referencia", "tipo_estrategia"],
    )

    df_out = (
        df_out.loc[
            (df_out["grupo_controle"] == GRUPO_CONTROLE_ALVO)
            & (df_out["tipo_estrategia"].isin(TIPOS_ESTRATEGIA_ALEATORIA_VALIDOS))
        ]
        .rename(columns={coluna_data: "data", coluna_saldo: "saldo_caixa_fim_dia", coluna_rendimento: "rendimento_cdi_dia", coluna_taxa: "taxa_cdi_dia", coluna_flag_evento: "flag_evento_aporte"})
        .sort_values(["estrategia_id", "data", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_compras(df_compras):
    colunas_obrigatorias = [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_aporte_estrategia",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "preco_compra_planejado",
        "quantidade_comprada_planejada",
        "valor_investido_planejado",
    ]
    validar_colunas_obrigatorias(df_compras, colunas_obrigatorias)

    df_out = df_compras.copy()
    for coluna in ["replica_id", "estrategia_referencia", "carteira_id", "ordem_compra_no_aporte", "valor_aporte_na_data", "peso_efetivo_compra_planejada"]:
        df_out = garantir_coluna(df_out, coluna)

    if df_out["replica_id"].isna().all():
        df_out["replica_id"] = extrair_replica_id_de_estrategia_id(df_out["estrategia_id"])

    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["ordem_compra_no_aporte"] = pd.to_numeric(df_out["ordem_compra_no_aporte"], errors="coerce")
    df_out["preco_compra_planejado"] = pd.to_numeric(df_out["preco_compra_planejado"], errors="coerce")
    df_out["quantidade_comprada_planejada"] = pd.to_numeric(df_out["quantidade_comprada_planejada"], errors="coerce")
    df_out["valor_investido_planejado"] = pd.to_numeric(df_out["valor_investido_planejado"], errors="coerce")
    df_out["valor_aporte_na_data"] = pd.to_numeric(df_out["valor_aporte_na_data"], errors="coerce")
    df_out["peso_efetivo_compra_planejada"] = pd.to_numeric(df_out["peso_efetivo_compra_planejada"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")

    df_out = padronizar_texto(
        df_out,
        colunas_upper=["ticker"],
        colunas_lower=["grupo_controle", "familia_estrategia", "estrategia_referencia", "tipo_estrategia"],
    )

    df_out = (
        df_out.loc[
            (df_out["grupo_controle"] == GRUPO_CONTROLE_ALVO)
            & (df_out["tipo_estrategia"].isin(TIPOS_ESTRATEGIA_ALEATORIA_VALIDOS))
        ]
        .sort_values(["tipo_estrategia", "replica_id", "data_aporte_efetiva", "ordem_aporte_estrategia", "ordem_compra_no_aporte", "ticker"])
        .reset_index(drop=True)
    )

    return df_out

def construir_calendario_diario(df_mercado_pad, df_mov_caixa_alvo, data_inicio, data_fim):
    datas_mercado = (
        df_mercado_pad.loc[(df_mercado_pad["data"] >= data_inicio) & (df_mercado_pad["data"] <= data_fim), "data"]
        .drop_duplicates()
        .sort_values()
    )

    datas_caixa = (
        df_mov_caixa_alvo.loc[(df_mov_caixa_alvo["data"] >= data_inicio) & (df_mov_caixa_alvo["data"] <= data_fim), "data"]
        .drop_duplicates()
        .sort_values()
    )

    calendario = pd.Series(pd.Index(sorted(set(datas_mercado.tolist()) | set(datas_caixa.tolist()))), name="data")
    return pd.to_datetime(calendario)

def construir_base_caixa_diaria_backtest(df_caixa_aporte_alvo, df_mov_caixa_alvo, calendario_diario):
    if df_mov_caixa_alvo.empty:
        raise ValueError("A base diária de caixa da etapa 8.4 não possui registros para a réplica aleatória em processamento.")

    estrategia_id = str(df_caixa_aporte_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_caixa_aporte_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0])
    tipo_estrategia = str(df_caixa_aporte_alvo["tipo_estrategia"].iloc[0])
    estrategia_referencia = str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0]) if "estrategia_referencia" in df_caixa_aporte_alvo.columns else pd.NA
    replica_id = pd.to_numeric(df_caixa_aporte_alvo["replica_id"], errors="coerce").iloc[0]
    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])

    df_cash_daily = (
        df_mov_caixa_alvo
        .sort_values(["data", "ordem_aporte_estrategia"])
        .groupby("data", as_index=False, dropna=False)
        .agg(
            request_id=("request_id", "last"),
            ordem_aporte_estrategia=("ordem_aporte_estrategia", "last"),
            valor_aporte_estrategia=("valor_aporte_estrategia", "sum"),
            valor_investido_efetivo_aporte=("valor_investido_efetivo_aporte", "sum"),
            valor_caixa_residual_aporte=("valor_caixa_residual_aporte", "sum"),
            valor_caixa_trazido_aporte=("valor_caixa_trazido_aporte", "sum"),
            saldo_caixa_fim_dia=("saldo_caixa_fim_dia", "last"),
            rendimento_cdi_dia=("rendimento_cdi_dia", "sum"),
            taxa_cdi_dia=("taxa_cdi_dia", "last"),
            flag_evento_aporte=("flag_evento_aporte", "max"),
        )
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_out = pd.DataFrame({"data": calendario_diario}).merge(df_cash_daily, on="data", how="left")
    df_out["saldo_caixa_fim_dia"] = pd.to_numeric(df_out["saldo_caixa_fim_dia"], errors="coerce").ffill().fillna(0.0)

    for coluna in ["valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte", "rendimento_cdi_dia", "taxa_cdi_dia"]:
        df_out[coluna] = pd.to_numeric(df_out[coluna], errors="coerce").fillna(0.0)

    df_out["flag_evento_aporte"] = df_out["flag_evento_aporte"].fillna(False)
    df_out["request_id"] = df_out["request_id"].astype("string")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")

    df_out["estrategia_id"] = estrategia_id
    df_out["grupo_controle"] = grupo_controle
    df_out["familia_estrategia"] = familia_estrategia
    df_out["estrategia_referencia"] = estrategia_referencia
    df_out["tipo_estrategia"] = tipo_estrategia
    df_out["replica_id"] = replica_id
    df_out["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_out["saldo_caixa"] = df_out["saldo_caixa_fim_dia"]

    return df_out

def construir_base_posicoes_diarias(df_mercado_pad, df_compras_alvo, calendario_diario, capital_inicial_estrategia):
    estrategia_id = str(df_compras_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_compras_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_compras_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_compras_alvo["estrategia_referencia"].iloc[0]) if "estrategia_referencia" in df_compras_alvo.columns else pd.NA
    tipo_estrategia = str(df_compras_alvo["tipo_estrategia"].iloc[0])
    replica_id = pd.to_numeric(df_compras_alvo["replica_id"], errors="coerce").iloc[0]
    capital_inicial_estrategia = float(capital_inicial_estrategia)
    data_inicio_backtest = pd.Timestamp(df_compras_alvo["data_aporte_efetiva"].min())
    data_fim_backtest = pd.Timestamp(calendario_diario.max())

    df_eventos = (
        df_compras_alvo
        .groupby(["ticker", "data_aporte_efetiva"], as_index=False, dropna=False)
        .agg(
            issuer_code=("issuer_code", "first"),
            nome=("nome", "first"),
            setor=("setor", "first"),
            subsetor=("subsetor", "first"),
            segmento=("segmento", "first"),
            quantidade_evento=("quantidade_comprada_planejada", "sum"),
            valor_investido_evento=("valor_investido_planejado", "sum"),
            preco_compra_evento=("preco_compra_planejado", "last"),
        )
        .rename(columns={"data_aporte_efetiva": "data"})
        .sort_values(["ticker", "data"])
        .reset_index(drop=True)
    )

    tickers_alvo = sorted(df_eventos["ticker"].dropna().unique().tolist())

    df_grade = pd.MultiIndex.from_product(
        [calendario_diario, tickers_alvo],
        names=["data", "ticker"],
    ).to_frame(index=False)

    df_meta_ticker = (
        df_eventos[["ticker", "issuer_code", "nome", "setor", "subsetor", "segmento", "preco_compra_evento"]]
        .drop_duplicates(subset=["ticker"], keep="first")
        .reset_index(drop=True)
    )

    df_precos = (
        df_mercado_pad.loc[df_mercado_pad["ticker"].isin(tickers_alvo), ["data", "ticker", "close_adj"]]
        .drop_duplicates(subset=["data", "ticker"], keep="last")
        .sort_values(["ticker", "data"])
        .reset_index(drop=True)
    )

    df_grade = df_grade.merge(df_precos, on=["data", "ticker"], how="left")
    df_grade = df_grade.merge(df_meta_ticker, on="ticker", how="left")
    df_grade["close_adj"] = (
        df_grade
        .sort_values(["ticker", "data"])
        .groupby("ticker")["close_adj"]
        .ffill()
    )
    df_grade["close_adj"] = df_grade["close_adj"].fillna(df_grade["preco_compra_evento"])

    df_grade = df_grade.merge(
        df_eventos[["ticker", "data", "quantidade_evento", "valor_investido_evento"]],
        on=["ticker", "data"],
        how="left",
    )

    df_grade["quantidade_evento"] = pd.to_numeric(df_grade["quantidade_evento"], errors="coerce").fillna(0.0)
    df_grade["valor_investido_evento"] = pd.to_numeric(df_grade["valor_investido_evento"], errors="coerce").fillna(0.0)

    df_grade = df_grade.sort_values(["ticker", "data"]).reset_index(drop=True)
    df_grade["quantidade_em_carteira"] = df_grade.groupby("ticker")["quantidade_evento"].cumsum()
    df_grade["valor_investido_acumulado_ticker"] = df_grade.groupby("ticker")["valor_investido_evento"].cumsum()

    df_grade = df_grade.loc[df_grade["quantidade_em_carteira"] > 0].copy()

    df_grade["custo_medio_unitario"] = np.where(
        df_grade["quantidade_em_carteira"] > 0,
        df_grade["valor_investido_acumulado_ticker"] / df_grade["quantidade_em_carteira"],
        0.0,
    )

    df_grade["close_adj"] = df_grade["close_adj"].fillna(df_grade["custo_medio_unitario"])
    df_grade["valor_mercado_posicao"] = df_grade["quantidade_em_carteira"] * df_grade["close_adj"]

    df_grade["estrategia_id"] = estrategia_id
    df_grade["grupo_controle"] = grupo_controle
    df_grade["familia_estrategia"] = familia_estrategia
    df_grade["estrategia_referencia"] = estrategia_referencia
    df_grade["tipo_estrategia"] = tipo_estrategia
    df_grade["replica_id"] = replica_id
    df_grade["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_grade["data_inicio_backtest"] = data_inicio_backtest
    df_grade["data_fim_backtest"] = data_fim_backtest

    colunas_saida = [
        "data",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "replica_id",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "quantidade_em_carteira",
        "custo_medio_unitario",
        "close_adj",
        "valor_mercado_posicao",
    ]

    return df_grade[colunas_saida].copy()

def construir_base_patrimonial(df_posicoes_diarias, df_caixa_diaria_backtest, df_caixa_aporte_alvo, calendario_diario):
    estrategia_id = str(df_caixa_aporte_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_caixa_aporte_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0]) if "estrategia_referencia" in df_caixa_aporte_alvo.columns else pd.NA
    tipo_estrategia = str(df_caixa_aporte_alvo["tipo_estrategia"].iloc[0])
    replica_id = pd.to_numeric(df_caixa_aporte_alvo["replica_id"], errors="coerce").iloc[0]
    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])
    data_inicio_backtest = pd.Timestamp(calendario_diario.min())
    data_fim_backtest = pd.Timestamp(calendario_diario.max())

    df_pos_agg = (
        df_posicoes_diarias
        .groupby("data", as_index=False, dropna=False)
        .agg(
            valor_carteira_investida=("valor_mercado_posicao", "sum"),
            n_posicoes_ativas=("ticker", "count"),
            n_tickers_unicos_ativos=("ticker", "nunique"),
        )
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_patrimonio = pd.DataFrame({"data": calendario_diario}).merge(df_pos_agg, on="data", how="left")
    df_patrimonio = df_patrimonio.merge(
        df_caixa_diaria_backtest[
            [
                "data",
                "saldo_caixa_fim_dia",
                "saldo_caixa",
                "rendimento_cdi_dia",
                "taxa_cdi_dia",
                "request_id",
                "ordem_aporte_estrategia",
                "valor_aporte_estrategia",
                "valor_investido_efetivo_aporte",
                "valor_caixa_residual_aporte",
                "valor_caixa_trazido_aporte",
                "flag_evento_aporte",
            ]
        ],
        on="data",
        how="left",
    )

    for coluna in ["valor_carteira_investida", "n_posicoes_ativas", "n_tickers_unicos_ativos"]:
        df_patrimonio[coluna] = pd.to_numeric(df_patrimonio[coluna], errors="coerce").fillna(0.0)

    for coluna in ["saldo_caixa_fim_dia", "saldo_caixa", "rendimento_cdi_dia", "taxa_cdi_dia", "valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte"]:
        df_patrimonio[coluna] = pd.to_numeric(df_patrimonio[coluna], errors="coerce").fillna(0.0)

    df_patrimonio["flag_evento_aporte"] = df_patrimonio["flag_evento_aporte"].fillna(False)
    df_patrimonio["patrimonio_total"] = df_patrimonio["valor_carteira_investida"] + df_patrimonio["saldo_caixa_fim_dia"]

    mapa_aportes = (
        df_caixa_aporte_alvo[["data_aporte_efetiva", "valor_aporte_estrategia"]]
        .groupby("data_aporte_efetiva", as_index=False)["valor_aporte_estrategia"]
        .sum()
        .rename(columns={"data_aporte_efetiva": "data", "valor_aporte_estrategia": "valor_aporte_evento"})
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_patrimonio = df_patrimonio.merge(mapa_aportes, on="data", how="left")
    df_patrimonio["valor_aporte_evento"] = pd.to_numeric(df_patrimonio["valor_aporte_evento"], errors="coerce").fillna(0.0)
    df_patrimonio["valor_total_aportado"] = df_patrimonio["valor_aporte_evento"].cumsum()

    df_patrimonio["max_patrimonio_acumulado"] = df_patrimonio["patrimonio_total"].cummax()
    df_patrimonio["retorno_diario"] = df_patrimonio["patrimonio_total"].pct_change().fillna(0.0)

    if capital_inicial_estrategia > 0:
        df_patrimonio["retorno_acumulado"] = (df_patrimonio["patrimonio_total"] / capital_inicial_estrategia) - 1.0
    else:
        df_patrimonio["retorno_acumulado"] = 0.0

    df_patrimonio["drawdown_atual"] = np.where(
        df_patrimonio["max_patrimonio_acumulado"] > 0,
        (df_patrimonio["patrimonio_total"] / df_patrimonio["max_patrimonio_acumulado"]) - 1.0,
        0.0,
    )
    df_patrimonio["flag_em_drawdown"] = df_patrimonio["drawdown_atual"] < 0.0

    df_patrimonio["estrategia_id"] = estrategia_id
    df_patrimonio["grupo_controle"] = grupo_controle
    df_patrimonio["familia_estrategia"] = familia_estrategia
    df_patrimonio["estrategia_referencia"] = estrategia_referencia
    df_patrimonio["tipo_estrategia"] = tipo_estrategia
    df_patrimonio["replica_id"] = replica_id
    df_patrimonio["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_patrimonio["data_inicio_backtest"] = data_inicio_backtest
    df_patrimonio["data_fim_backtest"] = data_fim_backtest

    return df_patrimonio

def enriquecer_pesos_posicoes(df_posicoes_diarias, df_patrimonio):
    df_out = df_posicoes_diarias.merge(
        df_patrimonio[["data", "patrimonio_total"]],
        on="data",
        how="left",
    )

    df_out["peso_posicao_no_patrimonio"] = np.where(
        df_out["patrimonio_total"] > 0,
        df_out["valor_mercado_posicao"] / df_out["patrimonio_total"],
        0.0,
    )

    return df_out

def executar_motor_backtest_unico(df_mercado_pad, df_calendario_alvo, df_caixa_aporte_alvo, df_mov_caixa_alvo, df_compras_alvo):
    data_inicio = pd.Timestamp(df_mov_caixa_alvo["data"].min())
    data_fim = pd.Timestamp(df_mov_caixa_alvo["data"].max())

    calendario_diario = construir_calendario_diario(
        df_mercado_pad=df_mercado_pad,
        df_mov_caixa_alvo=df_mov_caixa_alvo,
        data_inicio=data_inicio,
        data_fim=data_fim,
    )

    if calendario_diario.empty:
        raise ValueError("O calendário diário do backtest ficou vazio.")

    df_caixa_diaria_backtest = construir_base_caixa_diaria_backtest(
        df_caixa_aporte_alvo=df_caixa_aporte_alvo,
        df_mov_caixa_alvo=df_mov_caixa_alvo,
        calendario_diario=calendario_diario,
    )

    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])

    df_posicoes_diarias = construir_base_posicoes_diarias(
        df_mercado_pad=df_mercado_pad,
        df_compras_alvo=df_compras_alvo,
        calendario_diario=calendario_diario,
        capital_inicial_estrategia=capital_inicial_estrategia,
    )

    df_patrimonio = construir_base_patrimonial(
        df_posicoes_diarias=df_posicoes_diarias,
        df_caixa_diaria_backtest=df_caixa_diaria_backtest,
        df_caixa_aporte_alvo=df_caixa_aporte_alvo,
        calendario_diario=calendario_diario,
    )

    df_posicoes_diarias = enriquecer_pesos_posicoes(
        df_posicoes_diarias=df_posicoes_diarias,
        df_patrimonio=df_patrimonio,
    )

    return df_caixa_diaria_backtest, df_posicoes_diarias, df_patrimonio

print(f"Tolerância do backtest das aleatórias : {TOLERANCIA_BACKTEST}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

df_mercado_pad = preparar_base_mercado(df_mercado_diario_consolidada)
df_calendarios_aleatorios_validos = preparar_calendarios_aleatorios(df_calendarios_aleatorios_controle)
df_caixa_aleatorias = preparar_base_caixa_aporte(df_base_caixa_residual_aporte)
df_movimentacao_caixa_aleatorias = preparar_base_movimentacao_caixa(df_movimentacao_caixa_diaria)
df_compras_aleatorias = preparar_base_compras(df_compras_planejadas_carteira)

if df_calendarios_aleatorios_validos.empty:
    raise ValueError("A base de calendários aleatórios de controle não possui registros válidos.")
if df_caixa_aleatorias.empty:
    raise ValueError("A base de caixa por aporte não possui registros para as estratégias aleatórias de controle.")
if df_compras_aleatorias.empty:
    raise ValueError("A base de compras planejadas não possui registros para as estratégias aleatórias de controle.")

df_combinacoes_esperadas = (
    df_calendarios_aleatorios_validos[
        ["estrategia_referencia", "tipo_estrategia", "estrategia_id", "replica_id"]
    ]
    .drop_duplicates()
    .sort_values(["estrategia_referencia", "replica_id"])
    .reset_index(drop=True)
)

df_combinacoes_caixa = (
    df_caixa_aleatorias[
        ["tipo_estrategia", "estrategia_id", "replica_id"]
    ]
    .drop_duplicates()
    .sort_values(["tipo_estrategia", "replica_id"])
    .reset_index(drop=True)
)

df_combinacoes_compras = (
    df_compras_aleatorias[
        ["tipo_estrategia", "estrategia_id", "replica_id"]
    ]
    .drop_duplicates()
    .sort_values(["tipo_estrategia", "replica_id"])
    .reset_index(drop=True)
)

df_merge_caixa = df_combinacoes_esperadas.merge(
    df_combinacoes_caixa,
    on=["tipo_estrategia", "estrategia_id", "replica_id"],
    how="left",
    indicator=True,
)

df_merge_compras = df_combinacoes_esperadas.merge(
    df_combinacoes_compras,
    on=["tipo_estrategia", "estrategia_id", "replica_id"],
    how="left",
    indicator=True,
)

if not (df_merge_caixa["_merge"] == "both").all():
    raise ValueError("A quantidade de combinações tipo_estrategia + estrategia_id + replica_id na etapa 8.4 não bate com a etapa 7.5.")
if not (df_merge_compras["_merge"] == "both").all():
    raise ValueError("A quantidade de combinações tipo_estrategia + estrategia_id + replica_id na etapa 8.5 não bate com a etapa 7.5.")

n_replicas_capitulacao = int(
    df_combinacoes_esperadas.loc[df_combinacoes_esperadas["estrategia_referencia"] == "capitulacao", "replica_id"].nunique()
)
n_replicas_euforia = int(
    df_combinacoes_esperadas.loc[df_combinacoes_esperadas["estrategia_referencia"] == "euforia", "replica_id"].nunique()
)

n_estrategia_referencia_ausente_inputs = int(
    df_calendarios_aleatorios_validos["estrategia_referencia"].isna().sum()
    + df_caixa_aleatorias["estrategia_referencia"].isna().sum()
    + df_compras_aleatorias["estrategia_referencia"].isna().sum()
)

print(f"Réplicas aleatórias de capitulação         : {n_replicas_capitulacao}")
print(f"Réplicas aleatórias de euforia             : {n_replicas_euforia}")
print(f"Combinações de estratégia + réplica        : {len(df_combinacoes_esperadas):,}")
print(f"Compras aleatórias de controle             : {len(df_compras_aleatorias):,}")
print(f"Estrategia_referencia ausente nos inputs   : {n_estrategia_referencia_ausente_inputs:,}")
print("OK")

# ============================================================
# 6) Execução do motor de backtest das réplicas aleatórias
# ============================================================

print("\n[6/10] Execução do motor de backtest das réplicas aleatórias...")

lista_compras = []
lista_posicoes = []
lista_caixa = []
lista_patrimonio = []
lista_resumo = []
lista_auditoria = []

for linha in df_combinacoes_esperadas.itertuples(index=False):
    estrategia_referencia = str(linha.estrategia_referencia)
    tipo_estrategia = str(linha.tipo_estrategia)
    estrategia_id = str(linha.estrategia_id)
    replica_id = float(linha.replica_id)

    df_cal_estr = (
        df_calendarios_aleatorios_validos
        .loc[
            (df_calendarios_aleatorios_validos["estrategia_id"] == estrategia_id)
            & (df_calendarios_aleatorios_validos["tipo_estrategia"] == tipo_estrategia)
            & (df_calendarios_aleatorios_validos["replica_id"] == replica_id)
        ]
        .copy()
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    df_caixa_estr = (
        df_caixa_aleatorias
        .loc[
            (df_caixa_aleatorias["estrategia_id"] == estrategia_id)
            & (df_caixa_aleatorias["tipo_estrategia"] == tipo_estrategia)
            & (df_caixa_aleatorias["replica_id"] == replica_id)
        ]
        .copy()
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    df_mov_estr = (
        df_movimentacao_caixa_aleatorias
        .loc[
            (df_movimentacao_caixa_aleatorias["estrategia_id"] == estrategia_id)
            & (df_movimentacao_caixa_aleatorias["tipo_estrategia"] == tipo_estrategia)
            & (df_movimentacao_caixa_aleatorias["replica_id"] == replica_id)
        ]
        .copy()
        .sort_values(["data", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    df_compras_estr = (
        df_compras_aleatorias
        .loc[
            (df_compras_aleatorias["estrategia_id"] == estrategia_id)
            & (df_compras_aleatorias["tipo_estrategia"] == tipo_estrategia)
            & (df_compras_aleatorias["replica_id"] == replica_id)
        ]
        .copy()
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia", "ticker"])
        .reset_index(drop=True)
    )

    if df_cal_estr.empty:
        raise ValueError(f"O calendário da estratégia {estrategia_id} réplica {replica_id} ficou vazio.")
    if df_caixa_estr.empty:
        raise ValueError(f"A base de caixa da estratégia {estrategia_id} réplica {replica_id} ficou vazia.")
    if df_mov_estr.empty:
        raise ValueError(f"A base diária de caixa da estratégia {estrategia_id} réplica {replica_id} ficou vazia.")
    if df_compras_estr.empty:
        raise ValueError(f"A base de compras da estratégia {estrategia_id} réplica {replica_id} ficou vazia.")

    df_caixa_estr["estrategia_referencia"] = estrategia_referencia
    df_compras_estr["estrategia_referencia"] = estrategia_referencia

    df_caixa_diaria_estr, df_posicoes_estr, df_patrimonio_estr = executar_motor_backtest_unico(
        df_mercado_pad=df_mercado_pad,
        df_calendario_alvo=df_cal_estr,
        df_caixa_aporte_alvo=df_caixa_estr,
        df_mov_caixa_alvo=df_mov_estr,
        df_compras_alvo=df_compras_estr,
    )

    df_compras_estr["estrategia_referencia"] = estrategia_referencia
    df_posicoes_estr["estrategia_referencia"] = estrategia_referencia
    df_caixa_diaria_estr["estrategia_referencia"] = estrategia_referencia
    df_patrimonio_estr["estrategia_referencia"] = estrategia_referencia

    lista_compras.append(df_compras_estr.copy())
    lista_posicoes.append(df_posicoes_estr.copy())
    lista_caixa.append(df_caixa_diaria_estr.copy())
    lista_patrimonio.append(df_patrimonio_estr.copy())

    capital_inicial_estrategia = float(df_caixa_estr["capital_inicial_estrategia"].iloc[0])
    valor_total_aportado = float(df_cal_estr["valor_aporte_estrategia"].sum())
    valor_total_investido_compras = float(df_compras_estr["valor_investido_planejado"].sum())
    valor_total_investido_auditoria = float(df_caixa_estr["valor_total_investido"].sum())
    valor_caixa_residual_total_aportes = float(df_caixa_estr["caixa_residual_final_aporte"].sum())
    valor_rendimento_cdi_acumulado_final = float(df_caixa_diaria_estr["rendimento_cdi_dia"].sum())
    saldo_caixa_final = float(df_caixa_diaria_estr["saldo_caixa_fim_dia"].iloc[-1])
    patrimonio_final = float(df_patrimonio_estr["patrimonio_total"].iloc[-1])
    retorno_acumulado_final = float(df_patrimonio_estr["retorno_acumulado"].iloc[-1])
    drawdown_maximo = float(df_patrimonio_estr["drawdown_atual"].min())
    n_posicoes_ativas_finais = int(df_posicoes_estr.loc[df_posicoes_estr["data"] == df_patrimonio_estr["data"].max(), "ticker"].nunique())

    desvio_investido_compras_vs_auditoria = abs(valor_total_investido_compras - valor_total_investido_auditoria)
    desvio_aportes_planejados_vs_investido_mais_residual = abs(valor_total_aportado - (valor_total_investido_auditoria + valor_caixa_residual_total_aportes))
    desvio_fluxo_caixa_completo = abs(
        saldo_caixa_final
        - (capital_inicial_estrategia - valor_total_investido_auditoria + valor_rendimento_cdi_acumulado_final)
    )
    flag_estrategia_referencia_preenchida = pd.notna(estrategia_referencia) and str(estrategia_referencia).strip().lower() not in ["", "<na>", "nan", "none", "null"]

    lista_resumo.append(
        {
            "familia_estrategia": str(df_caixa_estr["familia_estrategia"].iloc[0]),
            "estrategia_referencia": estrategia_referencia,
            "tipo_estrategia": tipo_estrategia,
            "estrategia_id": estrategia_id,
            "grupo_controle": str(df_caixa_estr["grupo_controle"].iloc[0]),
            "carteira_id": pd.NA,
            "replica_id": replica_id,
            "n_aportes": int(df_cal_estr["request_id"].nunique()),
            "data_inicio_backtest": pd.Timestamp(df_patrimonio_estr["data"].min()),
            "data_fim_backtest": pd.Timestamp(df_patrimonio_estr["data"].max()),
            "n_compras": int(len(df_compras_estr)),
            "n_tickers_unicos": int(df_compras_estr["ticker"].nunique()),
            "n_datas_curva_patrimonial": int(len(df_patrimonio_estr)),
            "valor_total_aportado": valor_total_aportado,
            "valor_total_investido": valor_total_investido_compras,
            "saldo_caixa_final": saldo_caixa_final,
            "patrimonio_final": patrimonio_final,
            "retorno_acumulado_final": retorno_acumulado_final,
            "drawdown_maximo": drawdown_maximo,
            "n_posicoes_ativas_finais": n_posicoes_ativas_finais,
        }
    )

    lista_auditoria.append(
        {
            "familia_estrategia": str(df_caixa_estr["familia_estrategia"].iloc[0]),
            "estrategia_referencia": estrategia_referencia,
            "tipo_estrategia": tipo_estrategia,
            "estrategia_id": estrategia_id,
            "grupo_controle": str(df_caixa_estr["grupo_controle"].iloc[0]),
            "carteira_id": pd.NA,
            "replica_id": replica_id,
            "n_aportes_calendario": int(df_cal_estr["request_id"].nunique()),
            "n_aportes_compras": int(df_compras_estr["request_id"].nunique()),
            "n_aportes_caixa": int(df_caixa_estr["request_id"].nunique()),
            "n_datas_patrimonio": int(len(df_patrimonio_estr)),
            "n_datas_caixa": int(len(df_caixa_diaria_estr)),
            "n_tickers_comprados": int(df_compras_estr["ticker"].nunique()),
            "valor_total_investido_compras": valor_total_investido_compras,
            "valor_total_investido_auditoria": valor_total_investido_auditoria,
            "desvio_investido_compras_vs_auditoria": desvio_investido_compras_vs_auditoria,
            "valor_total_aportes": valor_total_aportado,
            "valor_caixa_residual_total_aportes": valor_caixa_residual_total_aportes,
            "valor_rendimento_cdi_acumulado_final": valor_rendimento_cdi_acumulado_final,
            "saldo_caixa_final": saldo_caixa_final,
            "patrimonio_final": patrimonio_final,
            "retorno_acumulado_final": retorno_acumulado_final,
            "drawdown_maximo": drawdown_maximo,
            "flag_aportes_reconciliados": int(df_cal_estr["request_id"].nunique()) == int(df_caixa_estr["request_id"].nunique()),
            "flag_aportes_compras_compativeis": int(df_cal_estr["request_id"].nunique()) == int(df_compras_estr["request_id"].nunique()),
            "desvio_aportes_planejados_vs_investido_mais_residual": desvio_aportes_planejados_vs_investido_mais_residual,
            "desvio_fluxo_caixa_completo": desvio_fluxo_caixa_completo,
            "flag_reconciliacao_aportes_planejados": desvio_aportes_planejados_vs_investido_mais_residual <= TOLERANCIA_BACKTEST,
            "flag_reconciliacao_caixa_completo": desvio_fluxo_caixa_completo <= TOLERANCIA_BACKTEST,
            "flag_patrimonio_final_nao_negativo": patrimonio_final >= 0,
            "flag_posicoes_nao_negativas": bool((df_posicoes_estr["quantidade_em_carteira"] >= 0).all()),
            "flag_caixa_nao_negativo": bool((df_caixa_diaria_estr["saldo_caixa_fim_dia"] >= -TOLERANCIA_BACKTEST).all()),
            "flag_estrategia_referencia_preenchida": flag_estrategia_referencia_preenchida,
        }
    )

df_compras_aleatorias_controle = pd.concat(lista_compras, axis=0, ignore_index=True)
df_posicoes_aleatorias_controle = pd.concat(lista_posicoes, axis=0, ignore_index=True)
df_caixa_aleatorias_controle_diaria = pd.concat(lista_caixa, axis=0, ignore_index=True)
df_patrimonio_aleatorias_controle = pd.concat(lista_patrimonio, axis=0, ignore_index=True)
df_resumo_backtest_aleatorias_controle = pd.DataFrame(lista_resumo)
df_auditoria_backtest_aleatorias_controle = pd.DataFrame(lista_auditoria)

n_estrategia_referencia_ausente_outputs = int(
    df_compras_aleatorias_controle["estrategia_referencia"].isna().sum()
    + df_posicoes_aleatorias_controle["estrategia_referencia"].isna().sum()
    + df_caixa_aleatorias_controle_diaria["estrategia_referencia"].isna().sum()
    + df_patrimonio_aleatorias_controle["estrategia_referencia"].isna().sum()
)

print(f"Linhas da base de compras das aleatórias    : {len(df_compras_aleatorias_controle):,}")
print(f"Linhas da base de posições das aleatórias   : {len(df_posicoes_aleatorias_controle):,}")
print(f"Linhas da base de caixa diária              : {len(df_caixa_aleatorias_controle_diaria):,}")
print(f"Linhas da base patrimonial                  : {len(df_patrimonio_aleatorias_controle):,}")
print(f"Data inicial das curvas patrimoniais        : {pd.Timestamp(df_patrimonio_aleatorias_controle['data'].min()).date()}")
print(f"Data final das curvas patrimoniais          : {pd.Timestamp(df_patrimonio_aleatorias_controle['data'].max()).date()}")
print(f"Estrategia_referencia ausente nos outputs   : {n_estrategia_referencia_ausente_outputs:,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais, resumos e auditorias
# ============================================================

print("\n[7/10] Construção das tabelas formais, resumos e auditorias...")

df_regras_backtest_aleatorias_controle = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "BTA1",
            "escopo": "motor_unico",
            "regra_operacional": "executar_cada_replica_aleatoria_com_o_mesmo_motor_de_backtest_das_estrategias_reais",
            "detalhe": "Cada réplica aleatória herda o mesmo motor de compras, posições, caixa e patrimônio utilizado nas estratégias reais, variando apenas o calendário e a composição das compras.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "BTA2",
            "escopo": "combinacoes_esperadas",
            "regra_operacional": "validar_as_10_combinacoes_tipo_estrategia_mais_replica_entre_as_etapas_7_5_8_4_e_8_5",
            "detalhe": "A subetapa exige consistência entre os calendários aleatórios, a trilha de caixa e a base de compras planejadas para cada réplica.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "BTA3",
            "escopo": "caixa",
            "regra_operacional": "incorporar_o_saldo_de_caixa_completo_diario_remunerado_pelo_cdi_ate_a_data_final_do_backtest",
            "detalhe": "O caixa diário de cada réplica representa o capital não alocado em ações, remunerado pelo CDI desde o início comum da trilha até cada data do calendário patrimonial.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "BTA4",
            "escopo": "curva_patrimonial",
            "regra_operacional": "consolidar_diariamente_valor_investido_caixa_e_patrimonio_total_por_replica",
            "detalhe": "A curva patrimonial de cada réplica corresponde ao somatório diário do valor das posições e do saldo de caixa.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "BTA5",
            "escopo": "auditoria",
            "regra_operacional": "reconciliar_capital_inicial_compras_caixa_completo_e_rendimento_cdi_para_cada_replica",
            "detalhe": "Cada réplica deve fechar a conciliação entre capital inicial, compras efetivas, saldo de caixa completo e rendimento CDI efetivamente refletido na trilha diária do backtest.",
        },
    ]
)

df_inconsistencias_backtest_aleatorias_controle = (
    df_auditoria_backtest_aleatorias_controle
    .loc[
        (~df_auditoria_backtest_aleatorias_controle["flag_aportes_reconciliados"])
        | (~df_auditoria_backtest_aleatorias_controle["flag_aportes_compras_compativeis"])
        | (~df_auditoria_backtest_aleatorias_controle["flag_reconciliacao_aportes_planejados"])
        | (~df_auditoria_backtest_aleatorias_controle["flag_reconciliacao_caixa_completo"])
        | (~df_auditoria_backtest_aleatorias_controle["flag_patrimonio_final_nao_negativo"])
        | (~df_auditoria_backtest_aleatorias_controle["flag_posicoes_nao_negativas"])
        | (~df_auditoria_backtest_aleatorias_controle["flag_caixa_nao_negativo"])
        | (~df_auditoria_backtest_aleatorias_controle["flag_estrategia_referencia_preenchida"])
    ]
    .copy()
)

df_distribuicao_anual_backtest_aleatorias_controle = (
    df_patrimonio_aleatorias_controle
    .assign(ano=lambda df: pd.to_datetime(df["data"], errors="coerce").dt.year)
    .groupby(
        [
            "familia_estrategia",
            "estrategia_referencia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "replica_id",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_datas_curva_patrimonial=("data", "count"),
        patrimonio_medio_ano=("patrimonio_total", "mean"),
        valor_carteira_medio_ano=("valor_carteira_investida", "mean"),
        saldo_caixa_medio_ano=("saldo_caixa_fim_dia", "mean"),
        patrimonio_final_ano=("patrimonio_total", "last"),
        saldo_caixa_final_ano=("saldo_caixa_fim_dia", "last"),
        drawdown_minimo_ano=("drawdown_atual", "min"),
        retorno_ultimo_dia_ano=("retorno_acumulado", "last"),
    )
    .sort_values(["estrategia_referencia", "replica_id", "ano"])
    .reset_index(drop=True)
)

desvio_fluxo_caixa_completo_maximo = float(df_auditoria_backtest_aleatorias_controle["desvio_fluxo_caixa_completo"].abs().max())
n_estrategia_referencia_ausente_auditoria = int((~df_auditoria_backtest_aleatorias_controle["flag_estrategia_referencia_preenchida"]).sum())

print(f"Inconsistências do backtest identificadas   : {len(df_inconsistencias_backtest_aleatorias_controle):,}")
print(f"Desvio máximo do fluxo de caixa completo    : {desvio_fluxo_caixa_completo_maximo:.10f}")
print(f"Estrategia_referencia ausente na auditoria  : {n_estrategia_referencia_ausente_auditoria:,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_backtest_aleatorias_controle, caminho_tbl_regras_backtest_aleatorias_controle, index=False)
salvar_dataframe(df_compras_aleatorias_controle, caminho_base_compras_aleatorias_controle, index=False)
salvar_dataframe(df_posicoes_aleatorias_controle, caminho_base_posicoes_aleatorias_controle, index=False)
salvar_dataframe(df_caixa_aleatorias_controle_diaria, caminho_base_caixa_aleatorias_controle_diaria, index=False)
salvar_dataframe(df_patrimonio_aleatorias_controle, caminho_base_patrimonio_aleatorias_controle, index=False)
salvar_dataframe(df_resumo_backtest_aleatorias_controle, caminho_tbl_resumo_backtest_aleatorias_controle, index=False)
salvar_dataframe(df_auditoria_backtest_aleatorias_controle, caminho_tbl_auditoria_backtest_aleatorias_controle, index=False)
salvar_dataframe(df_inconsistencias_backtest_aleatorias_controle, caminho_tbl_inconsistencias_backtest_aleatorias_controle, index=False)
salvar_dataframe(df_distribuicao_anual_backtest_aleatorias_controle, caminho_tbl_distribuicao_anual_backtest_aleatorias_controle, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_backtest_aleatorias_controle = df_regras_backtest_aleatorias_controle.copy()
df_amostra_compras_aleatorias_controle = df_compras_aleatorias_controle.head(20).copy()
df_amostra_posicoes_aleatorias_controle = df_posicoes_aleatorias_controle.head(20).copy()
df_amostra_caixa_aleatorias_controle_diaria = df_caixa_aleatorias_controle_diaria.head(20).copy()
df_amostra_patrimonio_aleatorias_controle = df_patrimonio_aleatorias_controle.head(20).copy()
df_amostra_resumo_backtest_aleatorias_controle = df_resumo_backtest_aleatorias_controle.copy()
df_amostra_auditoria_backtest_aleatorias_controle = df_auditoria_backtest_aleatorias_controle.copy()
df_amostra_inconsistencias_backtest_aleatorias_controle = df_inconsistencias_backtest_aleatorias_controle.head(20).copy()
df_amostra_distribuicao_anual_backtest_aleatorias_controle = df_distribuicao_anual_backtest_aleatorias_controle.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais do backtest das estratégias aleatórias de controle:")
print(df_amostra_regras_backtest_aleatorias_controle.to_string(index=False))

print("\nBase de compras das estratégias aleatórias de controle - amostra:")
print(df_amostra_compras_aleatorias_controle.to_string(index=False))

print("\nBase diária de posições das estratégias aleatórias de controle - amostra:")
print(df_amostra_posicoes_aleatorias_controle.to_string(index=False))

print("\nBase diária de caixa das estratégias aleatórias de controle - amostra:")
print(df_amostra_caixa_aleatorias_controle_diaria.to_string(index=False))

print("\nBase patrimonial das estratégias aleatórias de controle - amostra:")
print(df_amostra_patrimonio_aleatorias_controle.to_string(index=False))

print("\nResumo do backtest das estratégias aleatórias de controle:")
print(df_amostra_resumo_backtest_aleatorias_controle.to_string(index=False))

print("\nAuditoria do backtest das estratégias aleatórias de controle:")
print(df_amostra_auditoria_backtest_aleatorias_controle.to_string(index=False))

print("\nInconsistências do backtest das estratégias aleatórias de controle - amostra:")
print(df_amostra_inconsistencias_backtest_aleatorias_controle.to_string(index=False))

print("\nDistribuição anual do backtest das estratégias aleatórias de controle - amostra:")
print(df_amostra_distribuicao_anual_backtest_aleatorias_controle.to_string(index=False))

print("\nArquivos salvos na subetapa 9.3:")
print(f"- {caminho_tbl_regras_backtest_aleatorias_controle}")
print(f"- {caminho_base_compras_aleatorias_controle}")
print(f"- {caminho_base_posicoes_aleatorias_controle}")
print(f"- {caminho_base_caixa_aleatorias_controle_diaria}")
print(f"- {caminho_base_patrimonio_aleatorias_controle}")
print(f"- {caminho_tbl_resumo_backtest_aleatorias_controle}")
print(f"- {caminho_tbl_auditoria_backtest_aleatorias_controle}")
print(f"- {caminho_tbl_inconsistencias_backtest_aleatorias_controle}")
print(f"- {caminho_tbl_distribuicao_anual_backtest_aleatorias_controle}")

print("\nETAPA 9.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 9.3 - BACKTEST DAS ESTRATÉGIAS ALEATÓRIAS DE CONTROLE

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidado          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - calendários aleatórios de controle  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_5_base_calendarios_aleatorios_controle.parquet
Entrada - base de caixa por aporte            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_4_base_caixa_residual_aporte.parquet
Entrada - base diária de caixa                : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasile

## Etapa 9.4) Backtest das Estratégias Mensais de Controle

In [43]:
%%time
# ============================================================
# Etapa 9.4) Backtest das Estratégias Mensais de Controle
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 9.4 - BACKTEST DAS ESTRATÉGIAS MENSAIS DE CONTROLE")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_base_ibov_padronizada = gerar_caminho_arquivo(
    etapa=1,
    subetapa=3,
    tipo_arquivo="base",
    nome="ibov_padronizada",
)

caminho_base_calendario_aportes_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="base",
    nome="calendario_aportes_estrategias_mensais_controle",
)

caminho_tbl_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="estrategias_mensais_controle",
)

caminho_base_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="caixa_residual_aporte",
)

caminho_base_movimentacao_caixa_diaria = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="movimentacao_caixa_diaria",
)

caminho_base_compras_planejadas_carteira = gerar_caminho_arquivo(
    etapa=8,
    subetapa=5,
    tipo_arquivo="base",
    nome="compras_planejadas_carteira",
)

caminho_tbl_regras_backtest_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="regras_backtest_mensais_controle",
)

caminho_base_compras_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="compras_mensais_controle",
)

caminho_base_posicoes_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="posicoes_mensais_controle",
)

caminho_base_caixa_mensais_controle_diaria = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="caixa_mensais_controle_diaria",
)

caminho_base_patrimonio_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="patrimonio_mensais_controle",
)

caminho_tbl_resumo_backtest_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="resumo_backtest_mensais_controle",
)

caminho_tbl_auditoria_backtest_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="auditoria_backtest_mensais_controle",
)

caminho_tbl_inconsistencias_backtest_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="inconsistencias_backtest_mensais_controle",
)

caminho_tbl_distribuicao_anual_backtest_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_backtest_mensais_controle",
)

print(f"Entrada - mercado diário consolidado          : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - ibov padronizada                    : {caminho_base_ibov_padronizada}")
print(f"Entrada - calendário mensal de controle       : {caminho_base_calendario_aportes_estrategias_mensais_controle}")
print(f"Entrada - definição estratégias mensais       : {caminho_tbl_estrategias_mensais_controle}")
print(f"Entrada - base de caixa por aporte            : {caminho_base_caixa_residual_aporte}")
print(f"Entrada - base diária de caixa                : {caminho_base_movimentacao_caixa_diaria}")
print(f"Entrada - base de compras planejadas          : {caminho_base_compras_planejadas_carteira}")
print(f"Saída   - regras do backtest                  : {caminho_tbl_regras_backtest_mensais_controle}")
print(f"Saída   - base de compras das estratégias     : {caminho_base_compras_mensais_controle}")
print(f"Saída   - base diária de posições             : {caminho_base_posicoes_mensais_controle}")
print(f"Saída   - base diária de caixa                : {caminho_base_caixa_mensais_controle_diaria}")
print(f"Saída   - base diária patrimonial             : {caminho_base_patrimonio_mensais_controle}")
print(f"Saída   - resumo do backtest                  : {caminho_tbl_resumo_backtest_mensais_controle}")
print(f"Saída   - auditoria do backtest               : {caminho_tbl_auditoria_backtest_mensais_controle}")
print(f"Saída   - inconsistências do backtest         : {caminho_tbl_inconsistencias_backtest_mensais_controle}")
print(f"Saída   - distribuição anual                  : {caminho_tbl_distribuicao_anual_backtest_mensais_controle}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_ibov_padronizada = pd.read_parquet(caminho_base_ibov_padronizada)
df_calendario_mensais_controle = pd.read_parquet(caminho_base_calendario_aportes_estrategias_mensais_controle)
df_estrategias_mensais_controle = pd.read_parquet(caminho_tbl_estrategias_mensais_controle)
df_base_caixa_residual_aporte = pd.read_parquet(caminho_base_caixa_residual_aporte)
df_movimentacao_caixa_diaria = pd.read_parquet(caminho_base_movimentacao_caixa_diaria)
df_compras_planejadas_carteira = pd.read_parquet(caminho_base_compras_planejadas_carteira)

print(f"Mercado diário consolidado                : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Ibov padronizada                          : {df_ibov_padronizada.shape[0]:,} linhas x {df_ibov_padronizada.shape[1]} colunas")
print(f"Calendário mensal de controle             : {df_calendario_mensais_controle.shape[0]:,} linhas x {df_calendario_mensais_controle.shape[1]} colunas")
print(f"Definição estratégias mensais             : {df_estrategias_mensais_controle.shape[0]:,} linhas x {df_estrategias_mensais_controle.shape[1]} colunas")
print(f"Base de caixa por aporte                  : {df_base_caixa_residual_aporte.shape[0]:,} linhas x {df_base_caixa_residual_aporte.shape[1]} colunas")
print(f"Base diária de caixa                      : {df_movimentacao_caixa_diaria.shape[0]:,} linhas x {df_movimentacao_caixa_diaria.shape[1]} colunas")
print(f"Base de compras planejadas                : {df_compras_planejadas_carteira.shape[0]:,} linhas x {df_compras_planejadas_carteira.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_BACKTEST = 0.000001
GRUPO_CONTROLE_ALVO = "mensal"

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def detectar_coluna(df, candidatos, obrigatoria=True):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    if obrigatoria:
        raise ValueError(f"Não foi possível localizar nenhuma das colunas candidatas: {candidatos}")
    return None

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

def padronizar_texto(df, colunas_upper=None, colunas_lower=None):
    colunas_upper = [] if colunas_upper is None else colunas_upper
    colunas_lower = [] if colunas_lower is None else colunas_lower

    for coluna in colunas_upper:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.upper()

    for coluna in colunas_lower:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.lower()

    return df

def preparar_base_mercado(df_mercado):
    coluna_data = detectar_coluna(df_mercado, ["data", "date", "dt_ref"])
    coluna_ticker = detectar_coluna(df_mercado, ["ticker", "codneg", "ticker_ajustado"])
    coluna_preco = detectar_coluna(df_mercado, ["close_adj", "adj_close", "close", "preco_fechamento"])
    coluna_issuer = detectar_coluna(df_mercado, ["issuer_code", "codigo_empresa", "codigo_emissor"], obrigatoria=False)

    df_out = df_mercado.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_preco] = pd.to_numeric(df_out[coluna_preco], errors="coerce")
    df_out = padronizar_texto(df_out, colunas_upper=[coluna_ticker])

    colunas_saida = [coluna_data, coluna_ticker, coluna_preco]
    if coluna_issuer is not None:
        colunas_saida.append(coluna_issuer)

    for coluna in ["nome", "setor", "subsetor", "segmento"]:
        if coluna in df_out.columns:
            colunas_saida.append(coluna)

    df_out = (
        df_out[colunas_saida]
        .rename(columns={coluna_data: "data", coluna_ticker: "ticker", coluna_preco: "close_adj"})
        .dropna(subset=["data", "ticker", "close_adj"])
        .sort_values(["ticker", "data"])
        .drop_duplicates(subset=["ticker", "data"], keep="last")
        .reset_index(drop=True)
    )

    if coluna_issuer is not None:
        df_out = df_out.rename(columns={coluna_issuer: "issuer_code"})
    else:
        df_out["issuer_code"] = pd.NA

    for coluna in ["nome", "setor", "subsetor", "segmento"]:
        df_out = garantir_coluna(df_out, coluna)

    return df_out


def preparar_base_ibov_para_marcacao(df_ibov, ticker_benchmark="^BVSP"):
    coluna_data = detectar_coluna(df_ibov, ["data", "date", "dt_ref"])
    coluna_preco = detectar_coluna(df_ibov, ["close_adj", "adj_close", "close", "preco_fechamento", "ibov_close_adj", "valor"])

    df_out = df_ibov.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_preco] = pd.to_numeric(df_out[coluna_preco], errors="coerce")

    df_out = (
        df_out[[coluna_data, coluna_preco]]
        .rename(columns={coluna_data: "data", coluna_preco: "close_adj"})
        .dropna(subset=["data", "close_adj"])
        .sort_values("data")
        .drop_duplicates(subset=["data"], keep="last")
        .reset_index(drop=True)
    )

    if df_out.empty:
        raise ValueError("A série padronizada do Ibovespa ficou vazia após a preparação para marcação a mercado.")

    df_out["ticker"] = ticker_benchmark
    df_out["issuer_code"] = ticker_benchmark
    df_out["nome"] = "IBOVESPA"
    df_out["setor"] = "ÍNDICE"
    df_out["subsetor"] = "ÍNDICE"
    df_out["segmento"] = "ÍNDICE"
    df_out["fonte_preco"] = "ibov_padronizada"
    df_out["prioridade_fonte_preco"] = 1

    return df_out[["data", "ticker", "close_adj", "issuer_code", "nome", "setor", "subsetor", "segmento", "fonte_preco", "prioridade_fonte_preco"]].copy()

def combinar_mercado_com_benchmark_ibov(df_mercado_pad, df_ibov_marcacao):
    df_mercado_base = df_mercado_pad.copy()
    df_mercado_base["fonte_preco"] = "mercado_diario_consolidada"
    df_mercado_base["prioridade_fonte_preco"] = 2

    for coluna in ["issuer_code", "nome", "setor", "subsetor", "segmento"]:
        if coluna not in df_mercado_base.columns:
            df_mercado_base[coluna] = pd.NA

    colunas_saida = ["data", "ticker", "close_adj", "issuer_code", "nome", "setor", "subsetor", "segmento", "fonte_preco", "prioridade_fonte_preco"]

    df_out = pd.concat(
        [df_mercado_base[colunas_saida], df_ibov_marcacao[colunas_saida]],
        axis=0,
        ignore_index=True,
    )

    df_out = (
        df_out
        .sort_values(["ticker", "data", "prioridade_fonte_preco"])
        .drop_duplicates(subset=["ticker", "data"], keep="first")
        .sort_values(["ticker", "data"])
        .reset_index(drop=True)
    )

    return df_out.copy()

def preparar_definicoes_mensais(df_def):
    colunas_obrigatorias = [
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "flag_executar_calendario_mensal",
    ]
    validar_colunas_obrigatorias(df_def, colunas_obrigatorias)

    df_out = df_def.copy()
    for coluna in ["carteira_id", "replica_id", "n_ativos_alvo", "estrategia_referencia", "benchmark_renda_variavel", "origem_universo_compra", "regra_selecao_ativos", "seed_base_carteira"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out["flag_executar_calendario_mensal"] = df_out["flag_executar_calendario_mensal"].fillna(False)
    df_out["carteira_id"] = pd.to_numeric(df_out["carteira_id"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")
    df_out["n_ativos_alvo"] = pd.to_numeric(df_out["n_ativos_alvo"], errors="coerce")
    df_out = padronizar_texto(df_out, colunas_lower=["grupo_controle", "familia_estrategia", "tipo_estrategia", "estrategia_id", "estrategia_referencia"])

    df_out = (
        df_out.loc[
            (df_out["grupo_controle"] == GRUPO_CONTROLE_ALVO)
            & (df_out["flag_executar_calendario_mensal"])
        ]
        .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id"])
        .reset_index(drop=True)
    )

    return df_out

def preparar_calendario_mensal(df_cal):
    colunas_obrigatorias = [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "carteira_id",
        "replica_id",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_aporte_estrategia",
        "valor_aporte_estrategia",
        "capital_inicial_estrategia",
    ]
    validar_colunas_obrigatorias(df_cal, colunas_obrigatorias)

    df_out = df_cal.copy()
    df_out = garantir_coluna(df_out, "estrategia_referencia")
    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")
    df_out["carteira_id"] = pd.to_numeric(df_out["carteira_id"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")
    df_out = padronizar_texto(df_out, colunas_upper=[], colunas_lower=["estrategia_id", "grupo_controle", "familia_estrategia", "tipo_estrategia", "estrategia_referencia"])

    df_out = (
        df_out.loc[df_out["grupo_controle"] == GRUPO_CONTROLE_ALVO]
        .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id", "data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_caixa_aporte(df_caixa):
    colunas_obrigatorias = [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_aporte_estrategia",
        "capital_inicial_estrategia",
        "valor_aporte_estrategia",
        "valor_total_investido",
        "caixa_residual_final_aporte",
        "valor_caixa_trazido_aporte",
        "saldo_caixa_final_periodo",
    ]
    validar_colunas_obrigatorias(df_caixa, colunas_obrigatorias)

    df_out = df_caixa.copy()
    for coluna in ["replica_id", "estrategia_referencia", "carteira_id"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["valor_total_investido"] = pd.to_numeric(df_out["valor_total_investido"], errors="coerce")
    df_out["caixa_residual_final_aporte"] = pd.to_numeric(df_out["caixa_residual_final_aporte"], errors="coerce")
    df_out["valor_caixa_trazido_aporte"] = pd.to_numeric(df_out["valor_caixa_trazido_aporte"], errors="coerce")
    df_out["saldo_caixa_final_periodo"] = pd.to_numeric(df_out["saldo_caixa_final_periodo"], errors="coerce")
    df_out["carteira_id"] = pd.to_numeric(df_out["carteira_id"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")

    df_out = padronizar_texto(df_out, colunas_upper=[], colunas_lower=["estrategia_id", "grupo_controle", "familia_estrategia", "tipo_estrategia", "estrategia_referencia"])

    df_out = (
        df_out.loc[df_out["grupo_controle"] == GRUPO_CONTROLE_ALVO]
        .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id", "data_aporte_efetiva", "ordem_aporte_estrategia"])
        .drop_duplicates(subset=["request_id"], keep="last")
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_movimentacao_caixa(df_mov):
    coluna_data = detectar_coluna(df_mov, ["data", "date"])
    coluna_saldo = detectar_coluna(df_mov, ["saldo_caixa_fim_dia", "saldo_caixa", "saldo_caixa_final_dia", "saldo_caixa_final_periodo"])
    coluna_rendimento = detectar_coluna(df_mov, ["rendimento_cdi_dia"], obrigatoria=False)
    coluna_taxa = detectar_coluna(df_mov, ["taxa_cdi_dia"], obrigatoria=False)
    coluna_flag_evento = detectar_coluna(df_mov, ["flag_evento_aporte"], obrigatoria=False)

    colunas_obrigatorias = ["request_id", "estrategia_id", "grupo_controle", "familia_estrategia", "tipo_estrategia", coluna_data, coluna_saldo]
    validar_colunas_obrigatorias(df_mov, colunas_obrigatorias)

    df_out = df_mov.copy()
    for coluna in ["replica_id", "estrategia_referencia", "carteira_id", "ordem_aporte_estrategia", "valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte", "capital_inicial_estrategia"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_saldo] = pd.to_numeric(df_out[coluna_saldo], errors="coerce")
    if coluna_rendimento is None:
        df_out["rendimento_cdi_dia"] = 0.0
        coluna_rendimento = "rendimento_cdi_dia"
    else:
        df_out[coluna_rendimento] = pd.to_numeric(df_out[coluna_rendimento], errors="coerce")

    if coluna_taxa is None:
        df_out["taxa_cdi_dia"] = 0.0
        coluna_taxa = "taxa_cdi_dia"
    else:
        df_out[coluna_taxa] = pd.to_numeric(df_out[coluna_taxa], errors="coerce")

    if coluna_flag_evento is None:
        df_out["flag_evento_aporte"] = False
        coluna_flag_evento = "flag_evento_aporte"

    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out["valor_aporte_estrategia"], errors="coerce")
    df_out["valor_investido_efetivo_aporte"] = pd.to_numeric(df_out["valor_investido_efetivo_aporte"], errors="coerce")
    df_out["valor_caixa_residual_aporte"] = pd.to_numeric(df_out["valor_caixa_residual_aporte"], errors="coerce")
    df_out["valor_caixa_trazido_aporte"] = pd.to_numeric(df_out["valor_caixa_trazido_aporte"], errors="coerce")
    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")
    df_out["carteira_id"] = pd.to_numeric(df_out["carteira_id"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")

    df_out = padronizar_texto(df_out, colunas_upper=[], colunas_lower=["estrategia_id", "grupo_controle", "familia_estrategia", "tipo_estrategia", "estrategia_referencia"])

    df_out = (
        df_out.loc[df_out["grupo_controle"] == GRUPO_CONTROLE_ALVO]
        .rename(columns={coluna_data: "data", coluna_saldo: "saldo_caixa_fim_dia", coluna_rendimento: "rendimento_cdi_dia", coluna_taxa: "taxa_cdi_dia", coluna_flag_evento: "flag_evento_aporte"})
        .sort_values(["estrategia_id", "data", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_compras(df_compras):
    colunas_obrigatorias = [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "data_sinal",
        "data_aporte_efetiva",
        "ordem_aporte_estrategia",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "preco_compra_planejado",
        "quantidade_comprada_planejada",
        "valor_investido_planejado",
    ]
    validar_colunas_obrigatorias(df_compras, colunas_obrigatorias)

    df_out = df_compras.copy()
    for coluna in ["replica_id", "estrategia_referencia", "carteira_id", "ordem_compra_no_aporte", "valor_aporte_na_data", "peso_efetivo_compra_planejada"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out["data_sinal"] = pd.to_datetime(df_out["data_sinal"], errors="coerce")
    df_out["data_aporte_efetiva"] = pd.to_datetime(df_out["data_aporte_efetiva"], errors="coerce")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")
    df_out["ordem_compra_no_aporte"] = pd.to_numeric(df_out["ordem_compra_no_aporte"], errors="coerce")
    df_out["preco_compra_planejado"] = pd.to_numeric(df_out["preco_compra_planejado"], errors="coerce")
    df_out["quantidade_comprada_planejada"] = pd.to_numeric(df_out["quantidade_comprada_planejada"], errors="coerce")
    df_out["valor_investido_planejado"] = pd.to_numeric(df_out["valor_investido_planejado"], errors="coerce")
    df_out["valor_aporte_na_data"] = pd.to_numeric(df_out["valor_aporte_na_data"], errors="coerce")
    df_out["peso_efetivo_compra_planejada"] = pd.to_numeric(df_out["peso_efetivo_compra_planejada"], errors="coerce")
    df_out["carteira_id"] = pd.to_numeric(df_out["carteira_id"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")

    df_out = padronizar_texto(df_out, colunas_upper=["ticker"], colunas_lower=["estrategia_id", "grupo_controle", "familia_estrategia", "tipo_estrategia", "estrategia_referencia"])

    df_out = (
        df_out.loc[df_out["grupo_controle"] == GRUPO_CONTROLE_ALVO]
        .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id", "data_aporte_efetiva", "ordem_aporte_estrategia", "ordem_compra_no_aporte", "ticker"])
        .reset_index(drop=True)
    )

    return df_out

def construir_calendario_diario(df_mercado_pad, df_mov_caixa_alvo, data_inicio, data_fim):
    datas_mercado = (
        df_mercado_pad.loc[(df_mercado_pad["data"] >= data_inicio) & (df_mercado_pad["data"] <= data_fim), "data"]
        .drop_duplicates()
        .sort_values()
    )

    datas_caixa = (
        df_mov_caixa_alvo.loc[(df_mov_caixa_alvo["data"] >= data_inicio) & (df_mov_caixa_alvo["data"] <= data_fim), "data"]
        .drop_duplicates()
        .sort_values()
    )

    calendario = pd.Series(pd.Index(sorted(set(datas_mercado.tolist()) | set(datas_caixa.tolist()))), name="data")
    return pd.to_datetime(calendario)

def construir_base_caixa_diaria_backtest(df_caixa_aporte_alvo, df_mov_caixa_alvo, calendario_diario):
    if df_mov_caixa_alvo.empty:
        raise ValueError("A base diária de caixa da etapa 8.4 não possui registros para a estratégia mensal em processamento.")

    estrategia_id = str(df_caixa_aporte_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_caixa_aporte_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0]) if "estrategia_referencia" in df_caixa_aporte_alvo.columns else pd.NA
    tipo_estrategia = str(df_caixa_aporte_alvo["tipo_estrategia"].iloc[0])
    carteira_id = pd.to_numeric(df_caixa_aporte_alvo["carteira_id"], errors="coerce").iloc[0]
    replica_id = pd.to_numeric(df_caixa_aporte_alvo["replica_id"], errors="coerce").iloc[0]
    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])

    df_cash_daily = (
        df_mov_caixa_alvo
        .sort_values(["data", "ordem_aporte_estrategia"])
        .groupby("data", as_index=False, dropna=False)
        .agg(
            request_id=("request_id", "last"),
            ordem_aporte_estrategia=("ordem_aporte_estrategia", "last"),
            valor_aporte_estrategia=("valor_aporte_estrategia", "sum"),
            valor_investido_efetivo_aporte=("valor_investido_efetivo_aporte", "sum"),
            valor_caixa_residual_aporte=("valor_caixa_residual_aporte", "sum"),
            valor_caixa_trazido_aporte=("valor_caixa_trazido_aporte", "sum"),
            saldo_caixa_fim_dia=("saldo_caixa_fim_dia", "last"),
            rendimento_cdi_dia=("rendimento_cdi_dia", "sum"),
            taxa_cdi_dia=("taxa_cdi_dia", "last"),
            flag_evento_aporte=("flag_evento_aporte", "max"),
        )
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_out = pd.DataFrame({"data": calendario_diario}).merge(df_cash_daily, on="data", how="left")
    df_out["saldo_caixa_fim_dia"] = pd.to_numeric(df_out["saldo_caixa_fim_dia"], errors="coerce").ffill().fillna(0.0)

    for coluna in ["valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte", "rendimento_cdi_dia", "taxa_cdi_dia"]:
        df_out[coluna] = pd.to_numeric(df_out[coluna], errors="coerce").fillna(0.0)

    df_out["flag_evento_aporte"] = df_out["flag_evento_aporte"].fillna(False)
    df_out["request_id"] = df_out["request_id"].astype("string")
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out["ordem_aporte_estrategia"], errors="coerce")

    df_out["estrategia_id"] = estrategia_id
    df_out["grupo_controle"] = grupo_controle
    df_out["familia_estrategia"] = familia_estrategia
    df_out["estrategia_referencia"] = estrategia_referencia
    df_out["tipo_estrategia"] = tipo_estrategia
    df_out["carteira_id"] = carteira_id
    df_out["replica_id"] = replica_id
    df_out["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_out["saldo_caixa"] = df_out["saldo_caixa_fim_dia"]

    return df_out

def construir_base_posicoes_diarias(df_mercado_pad, df_compras_alvo, calendario_diario, capital_inicial_estrategia):
    estrategia_id = str(df_compras_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_compras_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_compras_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_compras_alvo["estrategia_referencia"].iloc[0]) if "estrategia_referencia" in df_compras_alvo.columns else pd.NA
    tipo_estrategia = str(df_compras_alvo["tipo_estrategia"].iloc[0])
    carteira_id = pd.to_numeric(df_compras_alvo["carteira_id"], errors="coerce").iloc[0]
    replica_id = pd.to_numeric(df_compras_alvo["replica_id"], errors="coerce").iloc[0]
    capital_inicial_estrategia = float(capital_inicial_estrategia)
    data_inicio_backtest = pd.Timestamp(df_compras_alvo["data_aporte_efetiva"].min())
    data_fim_backtest = pd.Timestamp(calendario_diario.max())

    df_eventos = (
        df_compras_alvo
        .groupby(["ticker", "data_aporte_efetiva"], as_index=False, dropna=False)
        .agg(
            issuer_code=("issuer_code", "first"),
            nome=("nome", "first"),
            setor=("setor", "first"),
            subsetor=("subsetor", "first"),
            segmento=("segmento", "first"),
            quantidade_evento=("quantidade_comprada_planejada", "sum"),
            valor_investido_evento=("valor_investido_planejado", "sum"),
            preco_compra_evento=("preco_compra_planejado", "last"),
        )
        .rename(columns={"data_aporte_efetiva": "data"})
        .sort_values(["ticker", "data"])
        .reset_index(drop=True)
    )

    tickers_alvo = sorted(df_eventos["ticker"].dropna().unique().tolist())

    df_grade = pd.MultiIndex.from_product(
        [calendario_diario, tickers_alvo],
        names=["data", "ticker"],
    ).to_frame(index=False)

    df_meta_ticker = (
        df_eventos[["ticker", "issuer_code", "nome", "setor", "subsetor", "segmento", "preco_compra_evento"]]
        .drop_duplicates(subset=["ticker"], keep="first")
        .reset_index(drop=True)
    )

    df_precos = (
        df_mercado_pad.loc[df_mercado_pad["ticker"].isin(tickers_alvo), ["data", "ticker", "close_adj"]]
        .drop_duplicates(subset=["data", "ticker"], keep="last")
        .sort_values(["ticker", "data"])
        .reset_index(drop=True)
    )

    df_grade = df_grade.merge(df_precos, on=["data", "ticker"], how="left")
    df_grade = df_grade.merge(df_meta_ticker, on="ticker", how="left")
    df_grade["close_adj"] = (
        df_grade
        .sort_values(["ticker", "data"])
        .groupby("ticker")["close_adj"]
        .ffill()
    )

    df_cobertura_preco = (
        df_grade
        .groupby("ticker", as_index=False, dropna=False)
        .agg(n_precos_mercado_ou_carregados=("close_adj", lambda s: int(pd.Series(s).notna().sum())))
    )
    tickers_sem_preco_marcacao = df_cobertura_preco.loc[df_cobertura_preco["n_precos_mercado_ou_carregados"] == 0, "ticker"].astype(str).tolist()
    if len(tickers_sem_preco_marcacao) > 0:
        raise ValueError(
            "Há tickers comprados sem nenhuma série de preço para marcação a mercado no backtest mensal: "
            + str(tickers_sem_preco_marcacao[:20])
        )

    df_grade["close_adj"] = df_grade["close_adj"].fillna(df_grade["preco_compra_evento"])

    df_grade = df_grade.merge(
        df_eventos[["ticker", "data", "quantidade_evento", "valor_investido_evento"]],
        on=["ticker", "data"],
        how="left",
    )

    df_grade["quantidade_evento"] = pd.to_numeric(df_grade["quantidade_evento"], errors="coerce").fillna(0.0)
    df_grade["valor_investido_evento"] = pd.to_numeric(df_grade["valor_investido_evento"], errors="coerce").fillna(0.0)

    df_grade = df_grade.sort_values(["ticker", "data"]).reset_index(drop=True)
    df_grade["quantidade_em_carteira"] = df_grade.groupby("ticker")["quantidade_evento"].cumsum()
    df_grade["valor_investido_acumulado_ticker"] = df_grade.groupby("ticker")["valor_investido_evento"].cumsum()

    df_grade = df_grade.loc[df_grade["quantidade_em_carteira"] > 0].copy()

    df_grade["custo_medio_unitario"] = np.where(
        df_grade["quantidade_em_carteira"] > 0,
        df_grade["valor_investido_acumulado_ticker"] / df_grade["quantidade_em_carteira"],
        0.0,
    )

    df_grade["close_adj"] = df_grade["close_adj"].fillna(df_grade["custo_medio_unitario"])
    df_grade["valor_mercado_posicao"] = df_grade["quantidade_em_carteira"] * df_grade["close_adj"]

    df_grade["estrategia_id"] = estrategia_id
    df_grade["grupo_controle"] = grupo_controle
    df_grade["familia_estrategia"] = familia_estrategia
    df_grade["estrategia_referencia"] = estrategia_referencia
    df_grade["tipo_estrategia"] = tipo_estrategia
    df_grade["carteira_id"] = carteira_id
    df_grade["replica_id"] = replica_id
    df_grade["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_grade["data_inicio_backtest"] = data_inicio_backtest
    df_grade["data_fim_backtest"] = data_fim_backtest

    colunas_saida = [
        "data",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "carteira_id",
        "replica_id",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "quantidade_em_carteira",
        "custo_medio_unitario",
        "close_adj",
        "valor_mercado_posicao",
    ]

    return df_grade[colunas_saida].copy()

def construir_base_patrimonial(df_posicoes_diarias, df_caixa_diaria_backtest, df_caixa_aporte_alvo, calendario_diario):
    estrategia_id = str(df_caixa_aporte_alvo["estrategia_id"].iloc[0])
    grupo_controle = str(df_caixa_aporte_alvo["grupo_controle"].iloc[0])
    familia_estrategia = str(df_caixa_aporte_alvo["familia_estrategia"].iloc[0])
    estrategia_referencia = str(df_caixa_aporte_alvo["estrategia_referencia"].iloc[0]) if "estrategia_referencia" in df_caixa_aporte_alvo.columns else pd.NA
    tipo_estrategia = str(df_caixa_aporte_alvo["tipo_estrategia"].iloc[0])
    carteira_id = pd.to_numeric(df_caixa_aporte_alvo["carteira_id"], errors="coerce").iloc[0]
    replica_id = pd.to_numeric(df_caixa_aporte_alvo["replica_id"], errors="coerce").iloc[0]
    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])
    data_inicio_backtest = pd.Timestamp(calendario_diario.min())
    data_fim_backtest = pd.Timestamp(calendario_diario.max())

    df_pos_agg = (
        df_posicoes_diarias
        .groupby("data", as_index=False, dropna=False)
        .agg(
            valor_carteira_investida=("valor_mercado_posicao", "sum"),
            n_posicoes_ativas=("ticker", "count"),
            n_tickers_unicos_ativos=("ticker", "nunique"),
        )
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_patrimonio = pd.DataFrame({"data": calendario_diario}).merge(df_pos_agg, on="data", how="left")
    df_patrimonio = df_patrimonio.merge(
        df_caixa_diaria_backtest[
            [
                "data",
                "saldo_caixa_fim_dia",
                "saldo_caixa",
                "rendimento_cdi_dia",
                "taxa_cdi_dia",
                "request_id",
                "ordem_aporte_estrategia",
                "valor_aporte_estrategia",
                "valor_investido_efetivo_aporte",
                "valor_caixa_residual_aporte",
                "valor_caixa_trazido_aporte",
                "flag_evento_aporte",
            ]
        ],
        on="data",
        how="left",
    )

    for coluna in ["valor_carteira_investida", "n_posicoes_ativas", "n_tickers_unicos_ativos"]:
        df_patrimonio[coluna] = pd.to_numeric(df_patrimonio[coluna], errors="coerce").fillna(0.0)

    for coluna in ["saldo_caixa_fim_dia", "saldo_caixa", "rendimento_cdi_dia", "taxa_cdi_dia", "valor_aporte_estrategia", "valor_investido_efetivo_aporte", "valor_caixa_residual_aporte", "valor_caixa_trazido_aporte"]:
        df_patrimonio[coluna] = pd.to_numeric(df_patrimonio[coluna], errors="coerce").fillna(0.0)

    df_patrimonio["flag_evento_aporte"] = df_patrimonio["flag_evento_aporte"].fillna(False)
    df_patrimonio["patrimonio_total"] = df_patrimonio["valor_carteira_investida"] + df_patrimonio["saldo_caixa_fim_dia"]

    mapa_aportes = (
        df_caixa_aporte_alvo[["data_aporte_efetiva", "valor_aporte_estrategia"]]
        .groupby("data_aporte_efetiva", as_index=False)["valor_aporte_estrategia"]
        .sum()
        .rename(columns={"data_aporte_efetiva": "data", "valor_aporte_estrategia": "valor_aporte_evento"})
        .sort_values("data")
        .reset_index(drop=True)
    )

    df_patrimonio = df_patrimonio.merge(mapa_aportes, on="data", how="left")
    df_patrimonio["valor_aporte_evento"] = pd.to_numeric(df_patrimonio["valor_aporte_evento"], errors="coerce").fillna(0.0)
    df_patrimonio["valor_total_aportado"] = df_patrimonio["valor_aporte_evento"].cumsum()

    df_patrimonio["max_patrimonio_acumulado"] = df_patrimonio["patrimonio_total"].cummax()
    df_patrimonio["retorno_diario"] = df_patrimonio["patrimonio_total"].pct_change().fillna(0.0)

    if capital_inicial_estrategia > 0:
        df_patrimonio["retorno_acumulado"] = (df_patrimonio["patrimonio_total"] / capital_inicial_estrategia) - 1.0
    else:
        df_patrimonio["retorno_acumulado"] = 0.0

    df_patrimonio["drawdown_atual"] = np.where(
        df_patrimonio["max_patrimonio_acumulado"] > 0,
        (df_patrimonio["patrimonio_total"] / df_patrimonio["max_patrimonio_acumulado"]) - 1.0,
        0.0,
    )
    df_patrimonio["flag_em_drawdown"] = df_patrimonio["drawdown_atual"] < 0.0

    df_patrimonio["estrategia_id"] = estrategia_id
    df_patrimonio["grupo_controle"] = grupo_controle
    df_patrimonio["familia_estrategia"] = familia_estrategia
    df_patrimonio["estrategia_referencia"] = estrategia_referencia
    df_patrimonio["tipo_estrategia"] = tipo_estrategia
    df_patrimonio["carteira_id"] = carteira_id
    df_patrimonio["replica_id"] = replica_id
    df_patrimonio["capital_inicial_estrategia"] = capital_inicial_estrategia
    df_patrimonio["data_inicio_backtest"] = data_inicio_backtest
    df_patrimonio["data_fim_backtest"] = data_fim_backtest

    return df_patrimonio

def enriquecer_pesos_posicoes(df_posicoes_diarias, df_patrimonio):
    df_out = df_posicoes_diarias.merge(
        df_patrimonio[["data", "patrimonio_total"]],
        on="data",
        how="left",
    )

    df_out["peso_posicao_no_patrimonio"] = np.where(
        df_out["patrimonio_total"] > 0,
        df_out["valor_mercado_posicao"] / df_out["patrimonio_total"],
        0.0,
    )

    return df_out

def executar_motor_backtest_unico(df_mercado_pad, df_calendario_alvo, df_caixa_aporte_alvo, df_mov_caixa_alvo, df_compras_alvo):
    data_inicio = pd.Timestamp(df_mov_caixa_alvo["data"].min())

    if "data_aporte_efetiva" not in df_calendario_alvo.columns or df_calendario_alvo["data_aporte_efetiva"].dropna().empty:
        raise ValueError("O calendário mensal não possui data_aporte_efetiva válida para definir o ano final operacional.")

    data_ultimo_aporte = pd.Timestamp(df_calendario_alvo["data_aporte_efetiva"].max())
    data_fim_ano_operacional = pd.Timestamp(year=int(data_ultimo_aporte.year), month=12, day=31)

    candidatos_data_fim_operacional = [
        pd.Timestamp(df_mercado_pad["data"].max()),
        pd.Timestamp(df_mov_caixa_alvo["data"].max()),
        data_fim_ano_operacional,
    ]

    data_fim = min(candidatos_data_fim_operacional)

    calendario_diario = construir_calendario_diario(
        df_mercado_pad=df_mercado_pad,
        df_mov_caixa_alvo=df_mov_caixa_alvo,
        data_inicio=data_inicio,
        data_fim=data_fim,
    )

    if calendario_diario.empty:
        raise ValueError("O calendário diário do backtest ficou vazio.")

    df_caixa_diaria_backtest = construir_base_caixa_diaria_backtest(
        df_caixa_aporte_alvo=df_caixa_aporte_alvo,
        df_mov_caixa_alvo=df_mov_caixa_alvo,
        calendario_diario=calendario_diario,
    )

    capital_inicial_estrategia = float(df_caixa_aporte_alvo["capital_inicial_estrategia"].iloc[0])

    df_posicoes_diarias = construir_base_posicoes_diarias(
        df_mercado_pad=df_mercado_pad,
        df_compras_alvo=df_compras_alvo,
        calendario_diario=calendario_diario,
        capital_inicial_estrategia=capital_inicial_estrategia,
    )

    df_patrimonio = construir_base_patrimonial(
        df_posicoes_diarias=df_posicoes_diarias,
        df_caixa_diaria_backtest=df_caixa_diaria_backtest,
        df_caixa_aporte_alvo=df_caixa_aporte_alvo,
        calendario_diario=calendario_diario,
    )

    df_posicoes_diarias = enriquecer_pesos_posicoes(
        df_posicoes_diarias=df_posicoes_diarias,
        df_patrimonio=df_patrimonio,
    )

    return df_caixa_diaria_backtest, df_posicoes_diarias, df_patrimonio

print(f"Tolerância do backtest mensal : {TOLERANCIA_BACKTEST}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

df_mercado_pad_original = preparar_base_mercado(df_mercado_diario_consolidada)
df_ibov_marcacao = preparar_base_ibov_para_marcacao(df_ibov_padronizada, ticker_benchmark="^BVSP")
df_mercado_pad = combinar_mercado_com_benchmark_ibov(
    df_mercado_pad=df_mercado_pad_original,
    df_ibov_marcacao=df_ibov_marcacao,
)
df_definicoes_mensais_validas = preparar_definicoes_mensais(df_estrategias_mensais_controle)
df_calendario_mensais_validos = preparar_calendario_mensal(df_calendario_mensais_controle)
df_caixa_mensais = preparar_base_caixa_aporte(df_base_caixa_residual_aporte)
df_movimentacao_caixa_mensais = preparar_base_movimentacao_caixa(df_movimentacao_caixa_diaria)
df_compras_mensais = preparar_base_compras(df_compras_planejadas_carteira)

if df_definicoes_mensais_validas.empty:
    raise ValueError("A tabela de estratégias mensais executáveis ficou vazia.")
if df_calendario_mensais_validos.empty:
    raise ValueError("A base de calendários mensais executáveis ficou vazia.")
if df_caixa_mensais.empty:
    raise ValueError("A base de caixa por aporte não possui registros para as estratégias mensais de controle.")
if df_compras_mensais.empty:
    raise ValueError("A base de compras planejadas não possui registros para as estratégias mensais de controle.")

ids_esperados = sorted(df_definicoes_mensais_validas["estrategia_id"].dropna().unique().tolist())
ids_calendario = sorted(df_calendario_mensais_validos["estrategia_id"].dropna().unique().tolist())
ids_caixa = sorted(df_caixa_mensais["estrategia_id"].dropna().unique().tolist())
ids_compras = sorted(df_compras_mensais["estrategia_id"].dropna().unique().tolist())

n_precos_ibov_marcacao = int((df_mercado_pad["ticker"] == "^BVSP").sum())
data_inicial_ibov_marcacao = pd.to_datetime(df_mercado_pad.loc[df_mercado_pad["ticker"] == "^BVSP", "data"], errors="coerce").min()
data_final_ibov_marcacao = pd.to_datetime(df_mercado_pad.loc[df_mercado_pad["ticker"] == "^BVSP", "data"], errors="coerce").max()

if n_precos_ibov_marcacao == 0:
    raise ValueError("A base de marcação a mercado não possui preços para ^BVSP após combinar mercado e ibov_padronizada.")

if ids_esperados != ids_calendario:
    raise ValueError("Os estrategia_id executáveis da etapa 7.6 não batem com o calendário mensal consolidado.")
if ids_esperados != ids_caixa:
    raise ValueError("Os estrategia_id executáveis da etapa 7.6 não batem com a etapa 8.4.")
if ids_esperados != ids_compras:
    raise ValueError("Os estrategia_id executáveis da etapa 7.6 não batem com a etapa 8.5.")


n_estrategia_referencia_ausente_inputs = int(
    df_definicoes_mensais_validas["estrategia_referencia"].isna().sum()
    + df_calendario_mensais_validos["estrategia_referencia"].isna().sum()
    + df_caixa_mensais["estrategia_referencia"].isna().sum()
    + df_compras_mensais["estrategia_referencia"].isna().sum()
)

print(f"Estratégias mensais executáveis            : {len(ids_esperados):,}")
print(f"Linhas do calendário mensal válido         : {len(df_calendario_mensais_validos):,}")
print(f"Compras das estratégias mensais            : {len(df_compras_mensais):,}")
print(f"Preços de ^BVSP para marcação a mercado    : {n_precos_ibov_marcacao:,}")
print(f"Intervalo de marcação ^BVSP                : {pd.Timestamp(data_inicial_ibov_marcacao).date()} até {pd.Timestamp(data_final_ibov_marcacao).date()}")
print(f"Estrategia_referencia ausente nos inputs   : {n_estrategia_referencia_ausente_inputs:,}")
print("OK")

# ============================================================
# 6) Execução do motor de backtest das estratégias mensais
# ============================================================

print("\n[6/10] Execução do motor de backtest das estratégias mensais...")

lista_compras = []
lista_posicoes = []
lista_caixa = []
lista_patrimonio = []
lista_resumo = []
lista_auditoria = []

for estrategia_id in ids_esperados:
    df_def_estr = (
        df_definicoes_mensais_validas
        .loc[df_definicoes_mensais_validas["estrategia_id"] == estrategia_id]
        .copy()
        .reset_index(drop=True)
    )

    df_cal_estr = (
        df_calendario_mensais_validos
        .loc[df_calendario_mensais_validos["estrategia_id"] == estrategia_id]
        .copy()
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    df_caixa_estr = (
        df_caixa_mensais
        .loc[df_caixa_mensais["estrategia_id"] == estrategia_id]
        .copy()
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    df_mov_estr = (
        df_movimentacao_caixa_mensais
        .loc[df_movimentacao_caixa_mensais["estrategia_id"] == estrategia_id]
        .copy()
        .sort_values(["data", "ordem_aporte_estrategia"])
        .reset_index(drop=True)
    )

    df_compras_estr = (
        df_compras_mensais
        .loc[df_compras_mensais["estrategia_id"] == estrategia_id]
        .copy()
        .sort_values(["data_aporte_efetiva", "ordem_aporte_estrategia", "ticker"])
        .reset_index(drop=True)
    )

    if df_cal_estr.empty:
        raise ValueError(f"O calendário da estratégia mensal {estrategia_id} ficou vazio.")
    if df_caixa_estr.empty:
        raise ValueError(f"A base de caixa da estratégia mensal {estrategia_id} ficou vazia.")
    if df_mov_estr.empty:
        raise ValueError(f"A base diária de caixa da estratégia mensal {estrategia_id} ficou vazia.")
    if df_compras_estr.empty:
        raise ValueError(f"A base de compras da estratégia mensal {estrategia_id} ficou vazia.")


    referencia_candidata = pd.NA
    for df_ref in [df_cal_estr, df_caixa_estr, df_compras_estr, df_def_estr]:
        if "estrategia_referencia" in df_ref.columns:
            serie_ref = df_ref["estrategia_referencia"].dropna().astype("string").str.strip().str.lower()
            serie_ref = serie_ref.loc[~serie_ref.isin(["", "<na>", "nan", "none", "null"])]
            if len(serie_ref) > 0:
                referencia_candidata = str(serie_ref.iloc[0])
                break

    if pd.isna(referencia_candidata) or str(referencia_candidata).strip().lower() in ["", "<na>", "nan", "none", "null"]:
        raise ValueError(f"A estrategia_referencia da estratégia mensal {estrategia_id} está ausente.")

    df_cal_estr["estrategia_referencia"] = referencia_candidata
    df_caixa_estr["estrategia_referencia"] = referencia_candidata
    df_mov_estr["estrategia_referencia"] = referencia_candidata
    df_compras_estr["estrategia_referencia"] = referencia_candidata
    df_caixa_diaria_estr, df_posicoes_estr, df_patrimonio_estr = executar_motor_backtest_unico(
        df_mercado_pad=df_mercado_pad,
        df_calendario_alvo=df_cal_estr,
        df_caixa_aporte_alvo=df_caixa_estr,
        df_mov_caixa_alvo=df_mov_estr,
        df_compras_alvo=df_compras_estr,
    )



    df_compras_estr["estrategia_referencia"] = referencia_candidata
    df_posicoes_estr["estrategia_referencia"] = referencia_candidata
    df_caixa_diaria_estr["estrategia_referencia"] = referencia_candidata
    df_patrimonio_estr["estrategia_referencia"] = referencia_candidata

    lista_compras.append(df_compras_estr.copy())
    lista_posicoes.append(df_posicoes_estr.copy())
    lista_caixa.append(df_caixa_diaria_estr.copy())
    lista_patrimonio.append(df_patrimonio_estr.copy())

    capital_inicial_estrategia = float(df_caixa_estr["capital_inicial_estrategia"].iloc[0])
    valor_total_aportado = float(df_cal_estr["valor_aporte_estrategia"].sum())
    valor_total_investido_compras = float(df_compras_estr["valor_investido_planejado"].sum())
    valor_total_investido_auditoria = float(df_caixa_estr["valor_total_investido"].sum())
    valor_caixa_residual_total_aportes = float(df_caixa_estr["caixa_residual_final_aporte"].sum())
    valor_rendimento_cdi_acumulado_final = float(df_caixa_diaria_estr["rendimento_cdi_dia"].sum())
    saldo_caixa_final = float(df_caixa_diaria_estr["saldo_caixa_fim_dia"].iloc[-1])
    patrimonio_final = float(df_patrimonio_estr["patrimonio_total"].iloc[-1])
    retorno_acumulado_final = float(df_patrimonio_estr["retorno_acumulado"].iloc[-1])
    drawdown_maximo = float(df_patrimonio_estr["drawdown_atual"].min())
    n_posicoes_ativas_finais = int(df_posicoes_estr.loc[df_posicoes_estr["data"] == df_patrimonio_estr["data"].max(), "ticker"].nunique())

    desvio_investido_compras_vs_auditoria = abs(valor_total_investido_compras - valor_total_investido_auditoria)
    desvio_aportes_planejados_vs_investido_mais_residual = abs(valor_total_aportado - (valor_total_investido_auditoria + valor_caixa_residual_total_aportes))
    desvio_fluxo_caixa_completo = abs(
        saldo_caixa_final
        - (capital_inicial_estrategia - valor_total_investido_auditoria + valor_rendimento_cdi_acumulado_final)
    )
    flag_estrategia_referencia_preenchida = pd.notna(referencia_candidata) and str(referencia_candidata).strip().lower() not in ["", "<na>", "nan", "none", "null"]

    carteira_id = pd.to_numeric(df_caixa_estr["carteira_id"], errors="coerce").iloc[0]
    replica_id = pd.to_numeric(df_caixa_estr["replica_id"], errors="coerce").iloc[0]

    lista_resumo.append(
        {
            "familia_estrategia": str(df_caixa_estr["familia_estrategia"].iloc[0]),
            "estrategia_referencia": referencia_candidata,
            "tipo_estrategia": str(df_caixa_estr["tipo_estrategia"].iloc[0]),
            "estrategia_id": estrategia_id,
            "grupo_controle": str(df_caixa_estr["grupo_controle"].iloc[0]),
            "carteira_id": carteira_id,
            "replica_id": replica_id,
            "n_aportes": int(df_cal_estr["request_id"].nunique()),
            "data_inicio_backtest": pd.Timestamp(df_patrimonio_estr["data"].min()),
            "data_fim_backtest": pd.Timestamp(df_patrimonio_estr["data"].max()),
            "n_compras": int(len(df_compras_estr)),
            "n_tickers_unicos": int(df_compras_estr["ticker"].nunique()),
            "n_datas_curva_patrimonial": int(len(df_patrimonio_estr)),
            "valor_total_aportado": valor_total_aportado,
            "valor_total_investido": valor_total_investido_compras,
            "saldo_caixa_final": saldo_caixa_final,
            "patrimonio_final": patrimonio_final,
            "retorno_acumulado_final": retorno_acumulado_final,
            "drawdown_maximo": drawdown_maximo,
            "n_posicoes_ativas_finais": n_posicoes_ativas_finais,
            "capital_inicial_estrategia": capital_inicial_estrategia,
        }
    )

    lista_auditoria.append(
        {
            "familia_estrategia": str(df_caixa_estr["familia_estrategia"].iloc[0]),
            "estrategia_referencia": referencia_candidata,
            "tipo_estrategia": str(df_caixa_estr["tipo_estrategia"].iloc[0]),
            "estrategia_id": estrategia_id,
            "grupo_controle": str(df_caixa_estr["grupo_controle"].iloc[0]),
            "carteira_id": carteira_id,
            "replica_id": replica_id,
            "n_aportes_calendario": int(df_cal_estr["request_id"].nunique()),
            "n_aportes_compras": int(df_compras_estr["request_id"].nunique()),
            "n_aportes_caixa": int(df_caixa_estr["request_id"].nunique()),
            "n_datas_patrimonio": int(len(df_patrimonio_estr)),
            "n_datas_caixa": int(len(df_caixa_diaria_estr)),
            "n_tickers_comprados": int(df_compras_estr["ticker"].nunique()),
            "valor_total_investido_compras": valor_total_investido_compras,
            "valor_total_investido_auditoria": valor_total_investido_auditoria,
            "desvio_investido_compras_vs_auditoria": desvio_investido_compras_vs_auditoria,
            "valor_total_aportes": valor_total_aportado,
            "valor_caixa_residual_total_aportes": valor_caixa_residual_total_aportes,
            "valor_rendimento_cdi_acumulado_final": valor_rendimento_cdi_acumulado_final,
            "saldo_caixa_final": saldo_caixa_final,
            "patrimonio_final": patrimonio_final,
            "retorno_acumulado_final": retorno_acumulado_final,
            "drawdown_maximo": drawdown_maximo,
            "flag_aportes_reconciliados": int(df_cal_estr["request_id"].nunique()) == int(df_caixa_estr["request_id"].nunique()),
            "flag_aportes_compras_compativeis": int(df_cal_estr["request_id"].nunique()) == int(df_compras_estr["request_id"].nunique()),
            "desvio_aportes_planejados_vs_investido_mais_residual": desvio_aportes_planejados_vs_investido_mais_residual,
            "desvio_fluxo_caixa_completo": desvio_fluxo_caixa_completo,
            "flag_reconciliacao_aportes_planejados": desvio_aportes_planejados_vs_investido_mais_residual <= TOLERANCIA_BACKTEST,
            "flag_reconciliacao_caixa_completo": desvio_fluxo_caixa_completo <= TOLERANCIA_BACKTEST,
            "flag_patrimonio_final_nao_negativo": patrimonio_final >= 0,
            "flag_posicoes_nao_negativas": bool((df_posicoes_estr["quantidade_em_carteira"] >= 0).all()),
            "flag_caixa_nao_negativo": bool((df_caixa_diaria_estr["saldo_caixa_fim_dia"] >= -TOLERANCIA_BACKTEST).all()),
            "flag_estrategia_referencia_preenchida": flag_estrategia_referencia_preenchida,
        }
    )
df_compras_mensais_controle = pd.concat(lista_compras, axis=0, ignore_index=True)
df_posicoes_mensais_controle = pd.concat(lista_posicoes, axis=0, ignore_index=True)
df_caixa_mensais_controle_diaria = pd.concat(lista_caixa, axis=0, ignore_index=True)
df_patrimonio_mensais_controle = pd.concat(lista_patrimonio, axis=0, ignore_index=True)
df_resumo_backtest_mensais_controle = pd.DataFrame(lista_resumo)
df_auditoria_backtest_mensais_controle = pd.DataFrame(lista_auditoria)

n_estrategia_referencia_ausente_outputs = int(
    df_compras_mensais_controle["estrategia_referencia"].isna().sum()
    + df_posicoes_mensais_controle["estrategia_referencia"].isna().sum()
    + df_caixa_mensais_controle_diaria["estrategia_referencia"].isna().sum()
    + df_patrimonio_mensais_controle["estrategia_referencia"].isna().sum()
)

print(f"Linhas da base de compras mensais          : {len(df_compras_mensais_controle):,}")
print(f"Linhas da base de posições mensais         : {len(df_posicoes_mensais_controle):,}")
print(f"Linhas da base de caixa diária             : {len(df_caixa_mensais_controle_diaria):,}")
print(f"Linhas da base patrimonial                 : {len(df_patrimonio_mensais_controle):,}")
print(f"Data inicial das curvas patrimoniais       : {pd.Timestamp(df_patrimonio_mensais_controle['data'].min()).date()}")
print(f"Data final das curvas patrimoniais         : {pd.Timestamp(df_patrimonio_mensais_controle['data'].max()).date()}")
print(f"Data final do calendário mensal            : {pd.Timestamp(df_calendario_mensais_validos['data_aporte_efetiva'].max()).date()}")
print(f"Ano final operacional das estratégias       : {int(pd.Timestamp(df_calendario_mensais_validos['data_aporte_efetiva'].max()).year)}")
df_validacao_ibov_aportes = (
    df_patrimonio_mensais_controle
    .loc[df_patrimonio_mensais_controle["estrategia_id"].astype("string") == "ibovespa_aportes_mensais"]
    .copy()
)
vol_diaria_ibov_aportes = float(df_validacao_ibov_aportes["retorno_diario"].std(ddof=1)) if len(df_validacao_ibov_aportes) > 1 else np.nan
vol_anual_ibov_aportes = vol_diaria_ibov_aportes * np.sqrt(252) if pd.notna(vol_diaria_ibov_aportes) else np.nan
valor_carteira_final_ibov_aportes = float(df_validacao_ibov_aportes["valor_carteira_investida"].iloc[-1]) if not df_validacao_ibov_aportes.empty else np.nan

print(f"Estrategia_referencia ausente nos outputs  : {n_estrategia_referencia_ausente_outputs:,}")
print(f"Valor final carteira Ibovespa mensal       : {valor_carteira_final_ibov_aportes:,.10f}")
print(f"Vol. anual prévia Ibovespa mensal          : {vol_anual_ibov_aportes:.10f}")
print("OK")
# ============================================================
# 7) Construção das tabelas formais, resumos e auditorias
# ============================================================

print("\n[7/10] Construção das tabelas formais, resumos e auditorias...")

df_regras_backtest_mensais_controle = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "BTM1",
            "escopo": "motor_unico",
            "regra_operacional": "executar_cada_estrategia_mensal_com_o_mesmo_motor_de_backtest_das_demais_estrategias",
            "detalhe": "Cada estratégia mensal reaproveita o mesmo motor de compras, posições, caixa e patrimônio utilizado nas estratégias reais e aleatórias, variando apenas o calendário e a composição das compras.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "BTM2",
            "escopo": "universo_mensal",
            "regra_operacional": "validar_as_12_estrategias_mensais_executaveis_entre_as_etapas_7_6_8_4_e_8_5",
            "detalhe": "A subetapa exige consistência entre a definição formal das estratégias mensais, o calendário consolidado, a trilha de caixa e a base de compras planejadas.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "BTM3",
            "escopo": "caixa",
            "regra_operacional": "incorporar_o_saldo_de_caixa_completo_diario_remunerado_pelo_cdi_ate_a_data_final_do_backtest",
            "detalhe": "O caixa diário de cada estratégia mensal representa o capital não alocado em ações, remunerado pelo CDI desde o início comum da trilha até a data final operacional do ano do último aporte mensal.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "BTM4",
            "escopo": "curva_patrimonial",
            "regra_operacional": "consolidar_diariamente_valor_investido_caixa_e_patrimonio_total_por_estrategia_mensal",
            "detalhe": "A curva patrimonial de cada estratégia mensal corresponde ao somatório diário do valor das posições e do saldo de caixa.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "BTM5",
            "escopo": "benchmark_mensal",
            "regra_operacional": "marcar_posicoes_do_ibovespa_aportes_mensais_pela_serie_diaria_do_ibov_padronizada",
            "detalhe": "A estratégia Ibovespa Aportes Mensais usa o mesmo motor das demais estratégias, mas a posição ^BVSP é marcada diariamente pela série ibov_padronizada da etapa 1.3, evitando manter a posição pelo preço de compra.",
        },
        {
            "ordem_regra": 6,
            "regra_id": "BTM6",
            "escopo": "auditoria",
            "regra_operacional": "reconciliar_capital_inicial_compras_caixa_completo_e_rendimento_cdi_para_cada_estrategia_mensal",
            "detalhe": "Cada estratégia mensal deve fechar a conciliação entre capital inicial, compras efetivas, saldo de caixa completo e rendimento CDI efetivamente refletido na trilha diária do backtest.",
        },
    ]
)


df_inconsistencias_backtest_mensais_controle = (
    df_auditoria_backtest_mensais_controle
    .loc[
        (~df_auditoria_backtest_mensais_controle["flag_aportes_reconciliados"])
        | (~df_auditoria_backtest_mensais_controle["flag_aportes_compras_compativeis"])
        | (~df_auditoria_backtest_mensais_controle["flag_reconciliacao_aportes_planejados"])
        | (~df_auditoria_backtest_mensais_controle["flag_reconciliacao_caixa_completo"])
        | (~df_auditoria_backtest_mensais_controle["flag_patrimonio_final_nao_negativo"])
        | (~df_auditoria_backtest_mensais_controle["flag_posicoes_nao_negativas"])
        | (~df_auditoria_backtest_mensais_controle["flag_caixa_nao_negativo"])
        | (~df_auditoria_backtest_mensais_controle["flag_estrategia_referencia_preenchida"])
    ]
    .copy()
)
df_distribuicao_anual_backtest_mensais_controle = (
    df_patrimonio_mensais_controle
    .assign(ano=lambda df: pd.to_datetime(df["data"], errors="coerce").dt.year)
    .groupby(
        [
            "familia_estrategia",
            "estrategia_referencia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "carteira_id",
            "replica_id",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_datas_curva_patrimonial=("data", "count"),
        patrimonio_medio_ano=("patrimonio_total", "mean"),
        valor_carteira_medio_ano=("valor_carteira_investida", "mean"),
        saldo_caixa_medio_ano=("saldo_caixa_fim_dia", "mean"),
        patrimonio_final_ano=("patrimonio_total", "last"),
        saldo_caixa_final_ano=("saldo_caixa_fim_dia", "last"),
        drawdown_minimo_ano=("drawdown_atual", "min"),
        retorno_ultimo_dia_ano=("retorno_acumulado", "last"),
    )
    .sort_values(["familia_estrategia", "tipo_estrategia", "estrategia_id", "ano"])
    .reset_index(drop=True)
)

desvio_fluxo_caixa_completo_maximo = float(df_auditoria_backtest_mensais_controle["desvio_fluxo_caixa_completo"].abs().max())
n_estrategia_referencia_ausente_auditoria = int((~df_auditoria_backtest_mensais_controle["flag_estrategia_referencia_preenchida"]).sum())

print(f"Inconsistências do backtest identificadas   : {len(df_inconsistencias_backtest_mensais_controle):,}")
print(f"Desvio máximo do fluxo de caixa completo    : {desvio_fluxo_caixa_completo_maximo:.10f}")
print(f"Estrategia_referencia ausente na auditoria  : {n_estrategia_referencia_ausente_auditoria:,}")
print("OK")
# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_backtest_mensais_controle, caminho_tbl_regras_backtest_mensais_controle, index=False)
salvar_dataframe(df_compras_mensais_controle, caminho_base_compras_mensais_controle, index=False)
salvar_dataframe(df_posicoes_mensais_controle, caminho_base_posicoes_mensais_controle, index=False)
salvar_dataframe(df_caixa_mensais_controle_diaria, caminho_base_caixa_mensais_controle_diaria, index=False)
salvar_dataframe(df_patrimonio_mensais_controle, caminho_base_patrimonio_mensais_controle, index=False)
salvar_dataframe(df_resumo_backtest_mensais_controle, caminho_tbl_resumo_backtest_mensais_controle, index=False)
salvar_dataframe(df_auditoria_backtest_mensais_controle, caminho_tbl_auditoria_backtest_mensais_controle, index=False)
salvar_dataframe(df_inconsistencias_backtest_mensais_controle, caminho_tbl_inconsistencias_backtest_mensais_controle, index=False)
salvar_dataframe(df_distribuicao_anual_backtest_mensais_controle, caminho_tbl_distribuicao_anual_backtest_mensais_controle, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_backtest_mensais_controle = df_regras_backtest_mensais_controle.copy()
df_amostra_compras_mensais_controle = df_compras_mensais_controle.head(20).copy()
df_amostra_posicoes_mensais_controle = df_posicoes_mensais_controle.head(20).copy()
df_amostra_caixa_mensais_controle_diaria = df_caixa_mensais_controle_diaria.head(20).copy()
df_amostra_patrimonio_mensais_controle = df_patrimonio_mensais_controle.head(20).copy()
df_amostra_resumo_backtest_mensais_controle = df_resumo_backtest_mensais_controle.copy()
df_amostra_auditoria_backtest_mensais_controle = df_auditoria_backtest_mensais_controle.copy()
df_amostra_inconsistencias_backtest_mensais_controle = df_inconsistencias_backtest_mensais_controle.head(20).copy()
df_amostra_distribuicao_anual_backtest_mensais_controle = df_distribuicao_anual_backtest_mensais_controle.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais do backtest das estratégias mensais de controle:")
print(df_amostra_regras_backtest_mensais_controle.to_string(index=False))

print("\nBase de compras das estratégias mensais de controle - amostra:")
print(df_amostra_compras_mensais_controle.to_string(index=False))

print("\nBase diária de posições das estratégias mensais de controle - amostra:")
print(df_amostra_posicoes_mensais_controle.to_string(index=False))

print("\nBase diária de caixa das estratégias mensais de controle - amostra:")
print(df_amostra_caixa_mensais_controle_diaria.to_string(index=False))

print("\nBase patrimonial das estratégias mensais de controle - amostra:")
print(df_amostra_patrimonio_mensais_controle.to_string(index=False))

print("\nResumo do backtest das estratégias mensais de controle:")
print(df_amostra_resumo_backtest_mensais_controle.to_string(index=False))

print("\nAuditoria do backtest das estratégias mensais de controle:")
print(df_amostra_auditoria_backtest_mensais_controle.to_string(index=False))

print("\nInconsistências do backtest das estratégias mensais de controle - amostra:")
print(df_amostra_inconsistencias_backtest_mensais_controle.to_string(index=False))

print("\nDistribuição anual do backtest das estratégias mensais de controle - amostra:")
print(df_amostra_distribuicao_anual_backtest_mensais_controle.to_string(index=False))

print("\nArquivos salvos na subetapa 9.4:")
print(f"- {caminho_tbl_regras_backtest_mensais_controle}")
print(f"- {caminho_base_compras_mensais_controle}")
print(f"- {caminho_base_posicoes_mensais_controle}")
print(f"- {caminho_base_caixa_mensais_controle_diaria}")
print(f"- {caminho_base_patrimonio_mensais_controle}")
print(f"- {caminho_tbl_resumo_backtest_mensais_controle}")
print(f"- {caminho_tbl_auditoria_backtest_mensais_controle}")
print(f"- {caminho_tbl_inconsistencias_backtest_mensais_controle}")
print(f"- {caminho_tbl_distribuicao_anual_backtest_mensais_controle}")

print("\nETAPA 9.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 9.4 - BACKTEST DAS ESTRATÉGIAS MENSAIS DE CONTROLE

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - mercado diário consolidado          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - ibov padronizada                    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_ibov_padronizada.parquet
Entrada - calendário mensal de controle       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_6_base_calendario_aportes_estrategias_mensais_controle.parquet
Entrada - definição estratégias mensais       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado

## Etapa 9.5) Backtest do Benchmark Ibovespa

In [44]:
%%time
# ============================================================
# Etapa 9.5) Backtest do Benchmark Ibovespa
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 9.5 - BACKTEST DO BENCHMARK IBOVESPA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_ibov_padronizada = gerar_caminho_arquivo(
    etapa=1,
    subetapa=3,
    tipo_arquivo="base",
    nome="ibov_padronizada",
)

caminho_base_mercado_diario_consolidada = gerar_caminho_arquivo(
    etapa=2,
    subetapa=4,
    tipo_arquivo="base",
    nome="mercado_diario_consolidada",
)

caminho_tbl_estrategias_mensais_controle = gerar_caminho_arquivo(
    etapa=7,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="estrategias_mensais_controle",
)

caminho_base_patrimonio_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="patrimonio_capitulacao",
)

caminho_base_patrimonio_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="patrimonio_euforia",
)

caminho_base_patrimonio_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="patrimonio_aleatorias_controle",
)

caminho_base_patrimonio_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="patrimonio_mensais_controle",
)

caminho_tbl_regras_backtest_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="regras_backtest_benchmark_ibovespa",
)

caminho_base_evento_inicial_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="base",
    nome="evento_inicial_benchmark_ibovespa",
)

caminho_base_posicoes_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="base",
    nome="posicoes_benchmark_ibovespa",
)

caminho_base_patrimonio_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="base",
    nome="patrimonio_benchmark_ibovespa",
)

caminho_tbl_resumo_backtest_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="resumo_backtest_benchmark_ibovespa",
)

caminho_tbl_auditoria_backtest_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="auditoria_backtest_benchmark_ibovespa",
)

caminho_tbl_inconsistencias_backtest_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="inconsistencias_backtest_benchmark_ibovespa",
)

caminho_tbl_distribuicao_anual_backtest_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_backtest_benchmark_ibovespa",
)

print(f"Entrada - ibov padronizada                    : {caminho_base_ibov_padronizada}")
print(f"Entrada - mercado diário consolidado          : {caminho_base_mercado_diario_consolidada}")
print(f"Entrada - definição estratégias mensais       : {caminho_tbl_estrategias_mensais_controle}")
print(f"Entrada - patrimônio capitulação              : {caminho_base_patrimonio_capitulacao}")
print(f"Entrada - patrimônio euforia                  : {caminho_base_patrimonio_euforia}")
print(f"Entrada - patrimônio aleatórias controle      : {caminho_base_patrimonio_aleatorias_controle}")
print(f"Entrada - patrimônio mensais controle         : {caminho_base_patrimonio_mensais_controle}")
print(f"Saída   - regras do backtest                  : {caminho_tbl_regras_backtest_benchmark_ibovespa}")
print(f"Saída   - evento inicial do benchmark         : {caminho_base_evento_inicial_benchmark_ibovespa}")
print(f"Saída   - base diária de posições             : {caminho_base_posicoes_benchmark_ibovespa}")
print(f"Saída   - base diária patrimonial             : {caminho_base_patrimonio_benchmark_ibovespa}")
print(f"Saída   - resumo do backtest                  : {caminho_tbl_resumo_backtest_benchmark_ibovespa}")
print(f"Saída   - auditoria do backtest               : {caminho_tbl_auditoria_backtest_benchmark_ibovespa}")
print(f"Saída   - inconsistências do backtest         : {caminho_tbl_inconsistencias_backtest_benchmark_ibovespa}")
print(f"Saída   - distribuição anual                  : {caminho_tbl_distribuicao_anual_backtest_benchmark_ibovespa}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_ibov_padronizada = pd.read_parquet(caminho_base_ibov_padronizada)
df_mercado_diario_consolidada = pd.read_parquet(caminho_base_mercado_diario_consolidada)
df_estrategias_mensais_controle = pd.read_parquet(caminho_tbl_estrategias_mensais_controle)
df_patrimonio_capitulacao = pd.read_parquet(caminho_base_patrimonio_capitulacao)
df_patrimonio_euforia = pd.read_parquet(caminho_base_patrimonio_euforia)
df_patrimonio_aleatorias_controle = pd.read_parquet(caminho_base_patrimonio_aleatorias_controle)
df_patrimonio_mensais_controle = pd.read_parquet(caminho_base_patrimonio_mensais_controle)

print(f"Ibov padronizada                          : {df_ibov_padronizada.shape[0]:,} linhas x {df_ibov_padronizada.shape[1]} colunas")
print(f"Mercado diário consolidado                : {df_mercado_diario_consolidada.shape[0]:,} linhas x {df_mercado_diario_consolidada.shape[1]} colunas")
print(f"Definição estratégias mensais             : {df_estrategias_mensais_controle.shape[0]:,} linhas x {df_estrategias_mensais_controle.shape[1]} colunas")
print(f"Patrimônio capitulação                    : {df_patrimonio_capitulacao.shape[0]:,} linhas x {df_patrimonio_capitulacao.shape[1]} colunas")
print(f"Patrimônio euforia                        : {df_patrimonio_euforia.shape[0]:,} linhas x {df_patrimonio_euforia.shape[1]} colunas")
print(f"Patrimônio aleatórias controle            : {df_patrimonio_aleatorias_controle.shape[0]:,} linhas x {df_patrimonio_aleatorias_controle.shape[1]} colunas")
print(f"Patrimônio mensais controle               : {df_patrimonio_mensais_controle.shape[0]:,} linhas x {df_patrimonio_mensais_controle.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_BACKTEST = 0.000001
CAPITAL_INICIAL_PROJETO = 100000.00

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def detectar_coluna(df, candidatos, obrigatoria=True):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna

    if obrigatoria:
        raise ValueError(f"Não foi possível localizar nenhuma das colunas candidatas: {candidatos}")

    return None

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

def padronizar_texto(df, colunas_upper=None, colunas_lower=None):
    colunas_upper = [] if colunas_upper is None else colunas_upper
    colunas_lower = [] if colunas_lower is None else colunas_lower

    for coluna in colunas_upper:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.upper()

    for coluna in colunas_lower:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.lower()

    return df

def preparar_base_benchmark_ibovespa(df_ibov, df_mercado, ticker_benchmark):
    ticker_benchmark = str(ticker_benchmark).strip().upper()

    # Fonte preferencial do benchmark: ibov_padronizada da etapa 1.3
    coluna_data_ibov = detectar_coluna(df_ibov, ["data", "date", "dt_ref"])
    coluna_preco_ibov = detectar_coluna(df_ibov, ["close_adj", "adj_close", "close", "preco_fechamento", "ibov_close_adj", "valor"])

    df_ibov_out = df_ibov.copy()
    df_ibov_out[coluna_data_ibov] = pd.to_datetime(df_ibov_out[coluna_data_ibov], errors="coerce")
    df_ibov_out[coluna_preco_ibov] = pd.to_numeric(df_ibov_out[coluna_preco_ibov], errors="coerce")

    df_ibov_out = (
        df_ibov_out[[coluna_data_ibov, coluna_preco_ibov]]
        .rename(columns={coluna_data_ibov: "data", coluna_preco_ibov: "close_adj"})
        .dropna(subset=["data", "close_adj"])
        .sort_values("data")
        .drop_duplicates(subset=["data"], keep="last")
        .reset_index(drop=True)
    )

    if not df_ibov_out.empty:
        df_ibov_out["ticker"] = ticker_benchmark
        df_ibov_out["fonte_benchmark"] = "ibov_padronizada"
        return df_ibov_out[["data", "ticker", "close_adj", "fonte_benchmark"]].copy()

    # Fallback defensivo: tentar mercado_diario_consolidada se o índice estiver presente
    coluna_data = detectar_coluna(df_mercado, ["data", "date", "dt_ref"])
    coluna_ticker = detectar_coluna(df_mercado, ["ticker", "codneg", "ticker_ajustado"])
    coluna_preco = detectar_coluna(df_mercado, ["close_adj", "adj_close", "close", "preco_fechamento"])

    df_out = df_mercado.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_preco] = pd.to_numeric(df_out[coluna_preco], errors="coerce")
    df_out = padronizar_texto(df_out, colunas_upper=[coluna_ticker])

    df_out = (
        df_out.loc[df_out[coluna_ticker] == ticker_benchmark, [coluna_data, coluna_ticker, coluna_preco]]
        .rename(columns={coluna_data: "data", coluna_ticker: "ticker", coluna_preco: "close_adj"})
        .dropna(subset=["data", "ticker", "close_adj"])
        .sort_values("data")
        .drop_duplicates(subset=["data"], keep="last")
        .reset_index(drop=True)
    )

    if df_out.empty:
        raise ValueError(
            f"Não foi possível localizar a série diária do benchmark {ticker_benchmark} nem em ibov_padronizada nem em mercado_diario_consolidada."
        )

    df_out["fonte_benchmark"] = "mercado_diario_consolidada"
    return df_out[["data", "ticker", "close_adj", "fonte_benchmark"]].copy()

def preparar_definicao_benchmark(df_def):
    colunas_obrigatorias = [
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "tipo_estrategia",
        "benchmark_renda_variavel",
        "regra_selecao_ativos",
        "flag_executar_calendario_mensal",
    ]
    validar_colunas_obrigatorias(df_def, colunas_obrigatorias)

    df_out = df_def.copy()
    for coluna in ["carteira_id", "replica_id", "observacao_metodologica"]:
        df_out = garantir_coluna(df_out, coluna)

    df_out["carteira_id"] = pd.to_numeric(df_out["carteira_id"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")
    df_out = padronizar_texto(
        df_out,
        colunas_lower=["estrategia_id", "grupo_controle", "familia_estrategia", "tipo_estrategia", "regra_selecao_ativos"],
        colunas_upper=["benchmark_renda_variavel"],
    )

    df_out = (
        df_out.loc[
            (df_out["grupo_controle"] == "benchmark")
            & (df_out["estrategia_id"] == "ibovespa_buy_and_hold")
        ]
        .copy()
        .reset_index(drop=True)
    )

    if df_out.empty:
        raise ValueError("Não foi possível localizar a definição formal de ibovespa_buy_and_hold na etapa 7.6.")

    return df_out

def preparar_base_patrimonial_para_calendario(df_patrimonio):
    coluna_data = detectar_coluna(df_patrimonio, ["data", "date"])
    df_out = df_patrimonio.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    return df_out.rename(columns={coluna_data: "data"})

def construir_calendario_alinhado_estrategias(*dfs_patrimonio):
    lista_datas = []

    for df in dfs_patrimonio:
        df_pad = preparar_base_patrimonial_para_calendario(df)
        lista_datas.extend(df_pad["data"].dropna().drop_duplicates().tolist())

    if len(lista_datas) == 0:
        raise ValueError("Não foi possível construir o calendário de alinhamento das estratégias.")

    calendario = pd.Series(pd.Index(sorted(set(lista_datas))), name="data")
    return pd.to_datetime(calendario)

def construir_evento_inicial_benchmark(df_benchmark, data_inicio_estrategias, data_fim_estrategias, ticker_benchmark):
    df_exec = (
        df_benchmark
        .loc[(df_benchmark["data"] >= data_inicio_estrategias) & (df_benchmark["data"] <= data_fim_estrategias)]
        .copy()
        .sort_values("data")
        .reset_index(drop=True)
    )

    if df_exec.empty:
        raise ValueError("Não foi possível localizar observação do Ibovespa no recorte final de datas das estratégias.")

    data_execucao_inicial = pd.Timestamp(df_exec["data"].iloc[0])
    preco_execucao_inicial = float(df_exec["close_adj"].iloc[0])
    quantidade_equivalente = float(CAPITAL_INICIAL_PROJETO / preco_execucao_inicial)
    valor_investido_inicial = float(quantidade_equivalente * preco_execucao_inicial)

    return pd.DataFrame(
        [
            {
                "estrategia_id": "ibovespa_buy_and_hold",
                "grupo_controle": "benchmark",
                "familia_estrategia": "ibovespa_buy_and_hold",
                "estrategia_referencia": "ibovespa_buy_and_hold",
                "tipo_estrategia": "ibovespa_buy_and_hold",
                "carteira_id": pd.NA,
                "replica_id": pd.NA,
                "ticker_benchmark": ticker_benchmark,
                "data_inicio_estrategias": pd.Timestamp(data_inicio_estrategias),
                "data_fim_estrategias": pd.Timestamp(data_fim_estrategias),
                "data_execucao_inicial": data_execucao_inicial,
                "preco_execucao_inicial": preco_execucao_inicial,
                "quantidade_equivalente_benchmark": quantidade_equivalente,
                "valor_investido_inicial": valor_investido_inicial,
                "capital_inicial_estrategia": float(CAPITAL_INICIAL_PROJETO),
                "fonte_benchmark": str(df_benchmark["fonte_benchmark"].iloc[0]),
                "flag_preco_benchmark_encontrado": True,
            }
        ]
    )

print(f"Tolerância do backtest benchmark : {TOLERANCIA_BACKTEST}")
print(f"Capital inicial do benchmark     : {CAPITAL_INICIAL_PROJETO}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

df_def_benchmark = preparar_definicao_benchmark(df_estrategias_mensais_controle)
ticker_benchmark = str(df_def_benchmark["benchmark_renda_variavel"].iloc[0]).strip().upper()

calendario_alinhado_estrategias = construir_calendario_alinhado_estrategias(
    df_patrimonio_capitulacao,
    df_patrimonio_euforia,
    df_patrimonio_aleatorias_controle,
    df_patrimonio_mensais_controle,
)

data_inicio_estrategias = pd.Timestamp(calendario_alinhado_estrategias.min())
data_fim_estrategias = pd.Timestamp(calendario_alinhado_estrategias.max())

df_benchmark_ibovespa = preparar_base_benchmark_ibovespa(
    df_ibov=df_ibov_padronizada,
    df_mercado=df_mercado_diario_consolidada,
    ticker_benchmark=ticker_benchmark,
)

df_evento_inicial_benchmark_ibovespa = construir_evento_inicial_benchmark(
    df_benchmark=df_benchmark_ibovespa,
    data_inicio_estrategias=data_inicio_estrategias,
    data_fim_estrategias=data_fim_estrategias,
    ticker_benchmark=ticker_benchmark,
)

data_execucao_inicial = pd.Timestamp(df_evento_inicial_benchmark_ibovespa["data_execucao_inicial"].iloc[0])
preco_execucao_inicial = float(df_evento_inicial_benchmark_ibovespa["preco_execucao_inicial"].iloc[0])
quantidade_equivalente_benchmark = float(df_evento_inicial_benchmark_ibovespa["quantidade_equivalente_benchmark"].iloc[0])
fonte_benchmark = str(df_benchmark_ibovespa["fonte_benchmark"].iloc[0])

print(f"Ticker do benchmark                          : {ticker_benchmark}")
print(f"Fonte da série do benchmark                  : {fonte_benchmark}")
print(f"Data inicial do recorte das estratégias      : {data_inicio_estrategias.date()}")
print(f"Data final do recorte das estratégias        : {data_fim_estrategias.date()}")
print(f"Data de execução inicial do benchmark        : {data_execucao_inicial.date()}")
print(f"Preço inicial do benchmark                   : {preco_execucao_inicial:,.10f}")
print(f"Quantidade equivalente comprada              : {quantidade_equivalente_benchmark:,.10f}")
print("OK")

# ============================================================
# 6) Construção das bases diárias do benchmark
# ============================================================

print("\n[6/10] Construção das bases diárias do benchmark...")

df_benchmark_alinhado = (
    pd.DataFrame({"data": calendario_alinhado_estrategias})
    .merge(
        df_benchmark_ibovespa[["data", "close_adj"]],
        on="data",
        how="left",
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_benchmark_alinhado["close_adj"] = pd.to_numeric(df_benchmark_alinhado["close_adj"], errors="coerce").ffill()

if df_benchmark_alinhado.loc[df_benchmark_alinhado["data"] >= data_execucao_inicial, "close_adj"].isna().any():
    raise ValueError("Persistiram valores ausentes de preço do benchmark após o forward fill dentro do recorte alinhado.")

df_benchmark_alinhado["flag_ativo_no_benchmark"] = df_benchmark_alinhado["data"] >= data_execucao_inicial

df_benchmark_alinhado["quantidade_em_carteira"] = np.where(
    df_benchmark_alinhado["flag_ativo_no_benchmark"],
    quantidade_equivalente_benchmark,
    0.0,
)

df_benchmark_alinhado["valor_mercado_posicao"] = (
    df_benchmark_alinhado["quantidade_em_carteira"] * df_benchmark_alinhado["close_adj"]
)

df_benchmark_alinhado["saldo_caixa_fim_dia"] = 0.0
df_benchmark_alinhado["saldo_caixa"] = 0.0
# A base patrimonial precisa nascer já com valor_carteira_investida antes da seleção final,
# para evitar KeyError e manter o mesmo contrato estrutural das demais subetapas.
df_benchmark_alinhado["valor_carteira_investida"] = df_benchmark_alinhado["valor_mercado_posicao"]
df_benchmark_alinhado["patrimonio_total"] = df_benchmark_alinhado["valor_carteira_investida"] + df_benchmark_alinhado["saldo_caixa_fim_dia"]
df_benchmark_alinhado["valor_total_aportado"] = np.where(
    df_benchmark_alinhado["flag_ativo_no_benchmark"],
    float(CAPITAL_INICIAL_PROJETO),
    0.0,
)

df_benchmark_alinhado["valor_aporte_evento"] = np.where(
    df_benchmark_alinhado["data"] == data_execucao_inicial,
    float(CAPITAL_INICIAL_PROJETO),
    0.0,
)

df_benchmark_alinhado["n_posicoes_ativas"] = np.where(
    df_benchmark_alinhado["flag_ativo_no_benchmark"],
    1,
    0,
)

df_benchmark_alinhado["n_tickers_unicos_ativos"] = df_benchmark_alinhado["n_posicoes_ativas"]

df_benchmark_alinhado["max_patrimonio_acumulado"] = df_benchmark_alinhado["patrimonio_total"].cummax()
df_benchmark_alinhado["retorno_diario"] = df_benchmark_alinhado["patrimonio_total"].pct_change().fillna(0.0)
df_benchmark_alinhado["retorno_acumulado"] = np.where(
    df_benchmark_alinhado["flag_ativo_no_benchmark"],
    (df_benchmark_alinhado["patrimonio_total"] / float(CAPITAL_INICIAL_PROJETO)) - 1.0,
    0.0,
)

df_benchmark_alinhado["drawdown_atual"] = np.where(
    df_benchmark_alinhado["max_patrimonio_acumulado"] > 0,
    (df_benchmark_alinhado["patrimonio_total"] / df_benchmark_alinhado["max_patrimonio_acumulado"]) - 1.0,
    0.0,
)

df_benchmark_alinhado["flag_em_drawdown"] = df_benchmark_alinhado["drawdown_atual"] < 0.0
df_benchmark_alinhado["flag_evento_aporte"] = df_benchmark_alinhado["data"] == data_execucao_inicial
df_benchmark_alinhado["rendimento_cdi_dia"] = 0.0
df_benchmark_alinhado["taxa_cdi_dia"] = 0.0
df_benchmark_alinhado["request_id"] = np.where(
    df_benchmark_alinhado["data"] == data_execucao_inicial,
    "ibovespa_buy_and_hold__1",
    pd.NA,
)
df_benchmark_alinhado["ordem_aporte_estrategia"] = np.where(
    df_benchmark_alinhado["data"] == data_execucao_inicial,
    1.0,
    pd.NA,
)
df_benchmark_alinhado["valor_aporte_estrategia"] = df_benchmark_alinhado["valor_aporte_evento"]
df_benchmark_alinhado["valor_investido_efetivo_aporte"] = np.where(
    df_benchmark_alinhado["data"] == data_execucao_inicial,
    float(CAPITAL_INICIAL_PROJETO),
    0.0,
)
df_benchmark_alinhado["valor_caixa_residual_aporte"] = 0.0
df_benchmark_alinhado["valor_caixa_trazido_aporte"] = 0.0

df_benchmark_alinhado["estrategia_id"] = "ibovespa_buy_and_hold"
df_benchmark_alinhado["grupo_controle"] = "benchmark"
df_benchmark_alinhado["familia_estrategia"] = "ibovespa_buy_and_hold"
df_benchmark_alinhado["estrategia_referencia"] = "ibovespa_buy_and_hold"
df_benchmark_alinhado["tipo_estrategia"] = "ibovespa_buy_and_hold"
df_benchmark_alinhado["carteira_id"] = pd.NA
df_benchmark_alinhado["replica_id"] = pd.NA
df_benchmark_alinhado["capital_inicial_estrategia"] = float(CAPITAL_INICIAL_PROJETO)
df_benchmark_alinhado["data_inicio_backtest"] = data_execucao_inicial
df_benchmark_alinhado["data_fim_backtest"] = data_fim_estrategias

df_posicoes_benchmark_ibovespa = df_benchmark_alinhado[
    [
        "data",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "carteira_id",
        "replica_id",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
        "close_adj",
        "quantidade_em_carteira",
        "valor_mercado_posicao",
    ]
].copy()

df_posicoes_benchmark_ibovespa["ticker"] = ticker_benchmark
df_posicoes_benchmark_ibovespa["issuer_code"] = ticker_benchmark
df_posicoes_benchmark_ibovespa["nome"] = "Ibovespa"
df_posicoes_benchmark_ibovespa["setor"] = pd.NA
df_posicoes_benchmark_ibovespa["subsetor"] = pd.NA
df_posicoes_benchmark_ibovespa["segmento"] = pd.NA
df_posicoes_benchmark_ibovespa["custo_medio_unitario"] = np.where(
    df_posicoes_benchmark_ibovespa["quantidade_em_carteira"] > 0,
    float(CAPITAL_INICIAL_PROJETO) / df_posicoes_benchmark_ibovespa["quantidade_em_carteira"],
    0.0,
)

df_posicoes_benchmark_ibovespa["peso_posicao_no_patrimonio"] = np.where(
    df_benchmark_alinhado["patrimonio_total"] > 0,
    df_posicoes_benchmark_ibovespa["valor_mercado_posicao"] / df_benchmark_alinhado["patrimonio_total"],
    0.0,
)

df_posicoes_benchmark_ibovespa = df_posicoes_benchmark_ibovespa.loc[
    df_posicoes_benchmark_ibovespa["quantidade_em_carteira"] > 0
].copy().reset_index(drop=True)

df_patrimonio_benchmark_ibovespa = df_benchmark_alinhado[
    [
        "data",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "carteira_id",
        "replica_id",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
        "n_posicoes_ativas",
        "n_tickers_unicos_ativos",
        "valor_carteira_investida",
        "saldo_caixa_fim_dia",
        "saldo_caixa",
        "patrimonio_total",
        "valor_total_aportado",
        "valor_aporte_evento",
        "max_patrimonio_acumulado",
        "retorno_diario",
        "retorno_acumulado",
        "drawdown_atual",
        "flag_em_drawdown",
        "flag_evento_aporte",
        "request_id",
        "ordem_aporte_estrategia",
        "valor_aporte_estrategia",
        "valor_investido_efetivo_aporte",
        "valor_caixa_residual_aporte",
        "valor_caixa_trazido_aporte",
        "rendimento_cdi_dia",
        "taxa_cdi_dia",
        "close_adj",
    ]
].copy()

# corrigir nome padrao usado nas outras subetapas
df_patrimonio_benchmark_ibovespa = df_patrimonio_benchmark_ibovespa.rename(columns={"close_adj": "preco_benchmark_fechamento"})

print(f"Linhas da base de posições do benchmark    : {len(df_posicoes_benchmark_ibovespa):,}")
print(f"Linhas da base patrimonial do benchmark    : {len(df_patrimonio_benchmark_ibovespa):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais, resumos e auditorias
# ============================================================

print("\n[7/10] Construção das tabelas formais, resumos e auditorias...")

df_regras_backtest_benchmark_ibovespa = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "BTB1",
            "escopo": "execucao_inicial",
            "regra_operacional": "comprar_o_ibovespa_uma_unica_vez_no_primeiro_dia_util_do_recorte_final",
            "detalhe": "O benchmark buy and hold aplica todo o capital inicial de uma só vez na primeira observação disponível do Ibovespa dentro do recorte final das estratégias.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "BTB2",
            "escopo": "calendario_alinhado",
            "regra_operacional": "alinhar_a_curva_do_benchmark_ao_calendario_diario_unificado_das_estrategias",
            "detalhe": "A base diária do benchmark utiliza o mesmo calendário diário consolidado das estratégias reais e de controle para permitir comparações consistentes.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "BTB3",
            "escopo": "mark_to_market",
            "regra_operacional": "marcar_a_posicao_diariamente_pelo_close_adj_do_ibovespa",
            "detalhe": "A posição equivalente do benchmark é recalculada diariamente com base no close_adj do Ibovespa.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "BTB4",
            "escopo": "caixa",
            "regra_operacional": "manter_caixa_zero_apos_a_compra_inicial",
            "detalhe": "O benchmark buy and hold consome integralmente o capital inicial e não mantém trilha de caixa residual no backtest.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "BTB5",
            "escopo": "auditoria",
            "regra_operacional": "reconciliar_capital_inicial_quantidade_equivalente_e_patrimonio_diario",
            "detalhe": "A auditoria verifica se o patrimônio diário corresponde à quantidade equivalente constante multiplicada pelo preço diário do benchmark.",
        },
    ]
)

patrimonio_final = float(df_patrimonio_benchmark_ibovespa["patrimonio_total"].iloc[-1])
retorno_acumulado_final = float(df_patrimonio_benchmark_ibovespa["retorno_acumulado"].iloc[-1])
drawdown_maximo = float(df_patrimonio_benchmark_ibovespa["drawdown_atual"].min())
n_datas_curva = int(len(df_patrimonio_benchmark_ibovespa))
n_datas_posicoes = int(df_posicoes_benchmark_ibovespa["data"].nunique())

df_resumo_backtest_benchmark_ibovespa = pd.DataFrame(
    [
        {
            "familia_estrategia": "ibovespa_buy_and_hold",
            "estrategia_referencia": "ibovespa_buy_and_hold",
            "tipo_estrategia": "ibovespa_buy_and_hold",
            "estrategia_id": "ibovespa_buy_and_hold",
            "grupo_controle": "benchmark",
            "fonte_benchmark": fonte_benchmark,
            "carteira_id": pd.NA,
            "replica_id": pd.NA,
            "n_eventos_compra": 1,
            "data_inicio_backtest": data_execucao_inicial,
            "data_fim_backtest": data_fim_estrategias,
            "n_datas_curva_patrimonial": n_datas_curva,
            "n_datas_posicoes": n_datas_posicoes,
            "capital_inicial_estrategia": float(CAPITAL_INICIAL_PROJETO),
            "patrimonio_inicial": float(CAPITAL_INICIAL_PROJETO),
            "patrimonio_final": patrimonio_final,
            "retorno_acumulado_final": retorno_acumulado_final,
            "drawdown_maximo": drawdown_maximo,
        }
    ]
)

valor_teorico_final = float(quantidade_equivalente_benchmark * df_benchmark_alinhado["close_adj"].iloc[-1])
desvio_patrimonio_final = abs(patrimonio_final - valor_teorico_final)
desvio_compra_inicial = abs(float(CAPITAL_INICIAL_PROJETO) - float(df_evento_inicial_benchmark_ibovespa["valor_investido_inicial"].iloc[0]))
flag_quantidade_constante = bool(
    np.isclose(
        df_posicoes_benchmark_ibovespa["quantidade_em_carteira"].to_numpy(dtype=float),
        quantidade_equivalente_benchmark,
        atol=TOLERANCIA_BACKTEST,
    ).all()
)

df_auditoria_backtest_benchmark_ibovespa = pd.DataFrame(
    [
        {
            "familia_estrategia": "ibovespa_buy_and_hold",
            "estrategia_referencia": "ibovespa_buy_and_hold",
            "tipo_estrategia": "ibovespa_buy_and_hold",
            "estrategia_id": "ibovespa_buy_and_hold",
            "grupo_controle": "benchmark",
            "fonte_benchmark": fonte_benchmark,
            "capital_inicial_estrategia": float(CAPITAL_INICIAL_PROJETO),
            "ticker_benchmark": ticker_benchmark,
            "data_execucao_inicial": data_execucao_inicial,
            "preco_execucao_inicial": preco_execucao_inicial,
            "quantidade_equivalente_benchmark": quantidade_equivalente_benchmark,
            "patrimonio_final": patrimonio_final,
            "valor_teorico_final": valor_teorico_final,
            "desvio_compra_inicial": desvio_compra_inicial,
            "desvio_patrimonio_final": desvio_patrimonio_final,
            "flag_compra_inicial_valida": desvio_compra_inicial <= TOLERANCIA_BACKTEST,
            "flag_patrimonio_final_valido": desvio_patrimonio_final <= TOLERANCIA_BACKTEST,
            "flag_quantidade_constante": flag_quantidade_constante,
            "flag_caixa_zero": bool(np.isclose(df_patrimonio_benchmark_ibovespa["saldo_caixa_fim_dia"].to_numpy(dtype=float), 0.0, atol=TOLERANCIA_BACKTEST).all()),
            "flag_patrimonio_nao_negativo": bool((df_patrimonio_benchmark_ibovespa["patrimonio_total"] >= -TOLERANCIA_BACKTEST).all()),
        }
    ]
)

df_inconsistencias_backtest_benchmark_ibovespa = (
    df_auditoria_backtest_benchmark_ibovespa
    .loc[
        (~df_auditoria_backtest_benchmark_ibovespa["flag_compra_inicial_valida"])
        | (~df_auditoria_backtest_benchmark_ibovespa["flag_patrimonio_final_valido"])
        | (~df_auditoria_backtest_benchmark_ibovespa["flag_quantidade_constante"])
        | (~df_auditoria_backtest_benchmark_ibovespa["flag_caixa_zero"])
        | (~df_auditoria_backtest_benchmark_ibovespa["flag_patrimonio_nao_negativo"])
    ]
    .copy()
)

df_distribuicao_anual_backtest_benchmark_ibovespa = (
    df_patrimonio_benchmark_ibovespa
    .assign(ano=lambda df: pd.to_datetime(df["data"], errors="coerce").dt.year)
    .groupby(
        [
            "familia_estrategia",
            "estrategia_referencia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_datas_curva_patrimonial=("data", "count"),
        patrimonio_medio_ano=("patrimonio_total", "mean"),
        patrimonio_final_ano=("patrimonio_total", "last"),
        drawdown_minimo_ano=("drawdown_atual", "min"),
        retorno_ultimo_dia_ano=("retorno_acumulado", "last"),
    )
    .sort_values(["ano"])
    .reset_index(drop=True)
)

print(f"Inconsistências do backtest identificadas   : {len(df_inconsistencias_backtest_benchmark_ibovespa):,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_backtest_benchmark_ibovespa, caminho_tbl_regras_backtest_benchmark_ibovespa, index=False)
salvar_dataframe(df_evento_inicial_benchmark_ibovespa, caminho_base_evento_inicial_benchmark_ibovespa, index=False)
salvar_dataframe(df_posicoes_benchmark_ibovespa, caminho_base_posicoes_benchmark_ibovespa, index=False)
salvar_dataframe(df_patrimonio_benchmark_ibovespa, caminho_base_patrimonio_benchmark_ibovespa, index=False)
salvar_dataframe(df_resumo_backtest_benchmark_ibovespa, caminho_tbl_resumo_backtest_benchmark_ibovespa, index=False)
salvar_dataframe(df_auditoria_backtest_benchmark_ibovespa, caminho_tbl_auditoria_backtest_benchmark_ibovespa, index=False)
salvar_dataframe(df_inconsistencias_backtest_benchmark_ibovespa, caminho_tbl_inconsistencias_backtest_benchmark_ibovespa, index=False)
salvar_dataframe(df_distribuicao_anual_backtest_benchmark_ibovespa, caminho_tbl_distribuicao_anual_backtest_benchmark_ibovespa, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_backtest_benchmark_ibovespa = df_regras_backtest_benchmark_ibovespa.copy()
df_amostra_evento_inicial_benchmark_ibovespa = df_evento_inicial_benchmark_ibovespa.copy()
df_amostra_posicoes_benchmark_ibovespa = df_posicoes_benchmark_ibovespa.head(20).copy()
df_amostra_patrimonio_benchmark_ibovespa = df_patrimonio_benchmark_ibovespa.head(20).copy()
df_amostra_resumo_backtest_benchmark_ibovespa = df_resumo_backtest_benchmark_ibovespa.copy()
df_amostra_auditoria_backtest_benchmark_ibovespa = df_auditoria_backtest_benchmark_ibovespa.copy()
df_amostra_inconsistencias_backtest_benchmark_ibovespa = df_inconsistencias_backtest_benchmark_ibovespa.head(20).copy()
df_amostra_distribuicao_anual_backtest_benchmark_ibovespa = df_distribuicao_anual_backtest_benchmark_ibovespa.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais do backtest do benchmark Ibovespa:")
print(df_amostra_regras_backtest_benchmark_ibovespa.to_string(index=False))

print("\nEvento inicial do benchmark Ibovespa:")
print(df_amostra_evento_inicial_benchmark_ibovespa.to_string(index=False))

print("\nBase diária de posições do benchmark Ibovespa - amostra:")
print(df_amostra_posicoes_benchmark_ibovespa.to_string(index=False))

print("\nBase patrimonial do benchmark Ibovespa - amostra:")
print(df_amostra_patrimonio_benchmark_ibovespa.to_string(index=False))

print("\nResumo do backtest do benchmark Ibovespa:")
print(df_amostra_resumo_backtest_benchmark_ibovespa.to_string(index=False))

print("\nAuditoria do backtest do benchmark Ibovespa:")
print(df_amostra_auditoria_backtest_benchmark_ibovespa.to_string(index=False))

print("\nInconsistências do backtest do benchmark Ibovespa - amostra:")
print(df_amostra_inconsistencias_backtest_benchmark_ibovespa.to_string(index=False))

print("\nDistribuição anual do backtest do benchmark Ibovespa - amostra:")
print(df_amostra_distribuicao_anual_backtest_benchmark_ibovespa.to_string(index=False))

print("\nArquivos salvos na subetapa 9.5:")
print(f"- {caminho_tbl_regras_backtest_benchmark_ibovespa}")
print(f"- {caminho_base_evento_inicial_benchmark_ibovespa}")
print(f"- {caminho_base_posicoes_benchmark_ibovespa}")
print(f"- {caminho_base_patrimonio_benchmark_ibovespa}")
print(f"- {caminho_tbl_resumo_backtest_benchmark_ibovespa}")
print(f"- {caminho_tbl_auditoria_backtest_benchmark_ibovespa}")
print(f"- {caminho_tbl_inconsistencias_backtest_benchmark_ibovespa}")
print(f"- {caminho_tbl_distribuicao_anual_backtest_benchmark_ibovespa}")

print("\nETAPA 9.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 9.5 - BACKTEST DO BENCHMARK IBOVESPA

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - ibov padronizada                    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_ibov_padronizada.parquet
Entrada - mercado diário consolidado          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_2\2_4_base_mercado_diario_consolidada.parquet
Entrada - definição estratégias mensais       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_6_tbl_estrategias_mensais_controle.parquet
Entrada - patrimônio capitulação              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_1

## Etapa 9.6) Backtest da Carteira CDI-Only

In [45]:
%%time
# ============================================================
# Etapa 9.6) Backtest da Carteira CDI-Only
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 9.6 - BACKTEST DA CARTEIRA CDI-ONLY")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_tbl_parametros_globais = gerar_caminho_arquivo(
    etapa=1,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="parametros_globais",
)

caminho_base_cdi_padronizada = gerar_caminho_arquivo(
    etapa=1,
    subetapa=3,
    tipo_arquivo="base",
    nome="cdi_padronizada",
)

caminho_base_patrimonio_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="patrimonio_capitulacao",
)

caminho_base_patrimonio_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="patrimonio_euforia",
)

caminho_base_patrimonio_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="patrimonio_aleatorias_controle",
)

caminho_base_patrimonio_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="patrimonio_mensais_controle",
)

caminho_base_patrimonio_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="base",
    nome="patrimonio_benchmark_ibovespa",
)

caminho_tbl_regras_backtest_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="regras_backtest_cdi_only",
)

caminho_base_evento_inicial_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="base",
    nome="evento_inicial_cdi_only",
)

caminho_base_posicoes_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="base",
    nome="posicoes_cdi_only",
)

caminho_base_patrimonio_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="base",
    nome="patrimonio_cdi_only",
)

caminho_tbl_resumo_backtest_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="resumo_backtest_cdi_only",
)

caminho_tbl_auditoria_backtest_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="auditoria_backtest_cdi_only",
)

caminho_tbl_inconsistencias_backtest_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="inconsistencias_backtest_cdi_only",
)

caminho_tbl_distribuicao_anual_backtest_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_backtest_cdi_only",
)

print(f"Entrada - parâmetros globais                : {caminho_tbl_parametros_globais}")
print(f"Entrada - CDI padronizada                   : {caminho_base_cdi_padronizada}")
print(f"Entrada - patrimônio capitulação            : {caminho_base_patrimonio_capitulacao}")
print(f"Entrada - patrimônio euforia                : {caminho_base_patrimonio_euforia}")
print(f"Entrada - patrimônio aleatórias controle    : {caminho_base_patrimonio_aleatorias_controle}")
print(f"Entrada - patrimônio mensais controle       : {caminho_base_patrimonio_mensais_controle}")
print(f"Entrada - patrimônio benchmark Ibovespa     : {caminho_base_patrimonio_benchmark_ibovespa}")
print(f"Saída   - regras do backtest                : {caminho_tbl_regras_backtest_cdi_only}")
print(f"Saída   - evento inicial CDI-only           : {caminho_base_evento_inicial_cdi_only}")
print(f"Saída   - base diária de posições           : {caminho_base_posicoes_cdi_only}")
print(f"Saída   - base diária patrimonial           : {caminho_base_patrimonio_cdi_only}")
print(f"Saída   - resumo do backtest                : {caminho_tbl_resumo_backtest_cdi_only}")
print(f"Saída   - auditoria do backtest             : {caminho_tbl_auditoria_backtest_cdi_only}")
print(f"Saída   - inconsistências do backtest       : {caminho_tbl_inconsistencias_backtest_cdi_only}")
print(f"Saída   - distribuição anual                : {caminho_tbl_distribuicao_anual_backtest_cdi_only}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_parametros_globais = pd.read_parquet(caminho_tbl_parametros_globais)
df_cdi_padronizada = pd.read_parquet(caminho_base_cdi_padronizada)
df_patrimonio_capitulacao = pd.read_parquet(caminho_base_patrimonio_capitulacao)
df_patrimonio_euforia = pd.read_parquet(caminho_base_patrimonio_euforia)
df_patrimonio_aleatorias_controle = pd.read_parquet(caminho_base_patrimonio_aleatorias_controle)
df_patrimonio_mensais_controle = pd.read_parquet(caminho_base_patrimonio_mensais_controle)
df_patrimonio_benchmark_ibovespa = pd.read_parquet(caminho_base_patrimonio_benchmark_ibovespa)

print(f"Parâmetros globais                        : {df_parametros_globais.shape[0]:,} linhas x {df_parametros_globais.shape[1]} colunas")
print(f"CDI padronizada                           : {df_cdi_padronizada.shape[0]:,} linhas x {df_cdi_padronizada.shape[1]} colunas")
print(f"Patrimônio capitulação                    : {df_patrimonio_capitulacao.shape[0]:,} linhas x {df_patrimonio_capitulacao.shape[1]} colunas")
print(f"Patrimônio euforia                        : {df_patrimonio_euforia.shape[0]:,} linhas x {df_patrimonio_euforia.shape[1]} colunas")
print(f"Patrimônio aleatórias controle            : {df_patrimonio_aleatorias_controle.shape[0]:,} linhas x {df_patrimonio_aleatorias_controle.shape[1]} colunas")
print(f"Patrimônio mensais controle               : {df_patrimonio_mensais_controle.shape[0]:,} linhas x {df_patrimonio_mensais_controle.shape[1]} colunas")
print(f"Patrimônio benchmark Ibovespa             : {df_patrimonio_benchmark_ibovespa.shape[0]:,} linhas x {df_patrimonio_benchmark_ibovespa.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_BACKTEST = 0.000001
ESTRATEGIA_ID_CDI_ONLY = "cdi_only"
TICKER_CDI_ONLY = "CDI"

def validar_colunas_obrigatorias(df, colunas_obrigatorias):
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"As seguintes colunas obrigatórias não foram encontradas na base: {colunas_ausentes}")

def detectar_coluna(df, candidatos, obrigatoria=True):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    if obrigatoria:
        raise ValueError(f"Não foi possível localizar nenhuma das colunas candidatas: {candidatos}")
    return None

def extrair_parametro(df_parametros, nome_parametro, obrigatorio=True, default=None):
    col_parametro = detectar_coluna(df_parametros, ["parametro", "nome_parametro"])
    col_valor = detectar_coluna(df_parametros, ["valor", "valor_parametro"])
    df_tmp = df_parametros.copy()
    df_tmp[col_parametro] = df_tmp[col_parametro].astype("string").str.strip().str.lower()
    filtro = df_tmp[col_parametro] == str(nome_parametro).strip().lower()

    if not filtro.any():
        if obrigatorio:
            raise ValueError(f"O parâmetro '{nome_parametro}' não foi localizado em parametros_globais.")
        return default

    return df_tmp.loc[filtro, col_valor].iloc[0]

def preparar_base_patrimonial_para_calendario(df_patrimonio):
    coluna_data = detectar_coluna(df_patrimonio, ["data", "date"])
    df_out = df_patrimonio.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    return df_out.rename(columns={coluna_data: "data"})

def construir_calendario_alinhado_estrategias(*dfs_patrimonio):
    lista_datas = []

    for df in dfs_patrimonio:
        df_pad = preparar_base_patrimonial_para_calendario(df)
        lista_datas.extend(df_pad["data"].dropna().drop_duplicates().tolist())

    if len(lista_datas) == 0:
        raise ValueError("Não foi possível construir o calendário de alinhamento das estratégias.")

    calendario = pd.Series(pd.Index(sorted(set(lista_datas))), name="data")
    return pd.to_datetime(calendario)

def preparar_cdi_padronizada(df_cdi):
    coluna_data = detectar_coluna(df_cdi, ["data", "date"])
    coluna_valor = detectar_coluna(df_cdi, ["valor", "value", "cdi", "taxa"])

    df_out = df_cdi.copy()
    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_valor] = pd.to_numeric(df_out[coluna_valor], errors="coerce")

    df_out = (
        df_out[[coluna_data, coluna_valor]]
        .rename(columns={coluna_data: "data", coluna_valor: "valor_cdi_percentual"})
        .dropna(subset=["data", "valor_cdi_percentual"])
        .sort_values("data")
        .drop_duplicates(subset=["data"], keep="last")
        .reset_index(drop=True)
    )

    df_out["taxa_cdi_dia"] = df_out["valor_cdi_percentual"] / 100.0

    return df_out

def construir_evento_inicial_cdi_only(data_inicio_estrategias, data_fim_estrategias, capital_inicial_estrategia):
    return pd.DataFrame(
        [
            {
                "estrategia_id": ESTRATEGIA_ID_CDI_ONLY,
                "grupo_controle": "benchmark",
                "familia_estrategia": ESTRATEGIA_ID_CDI_ONLY,
                "estrategia_referencia": ESTRATEGIA_ID_CDI_ONLY,
                "tipo_estrategia": ESTRATEGIA_ID_CDI_ONLY,
                "carteira_id": pd.NA,
                "replica_id": pd.NA,
                "ticker_referencia": TICKER_CDI_ONLY,
                "data_inicio_estrategias": pd.Timestamp(data_inicio_estrategias),
                "data_fim_estrategias": pd.Timestamp(data_fim_estrategias),
                "data_execucao_inicial": pd.Timestamp(data_inicio_estrategias),
                "capital_inicial_estrategia": float(capital_inicial_estrategia),
                "flag_inicio_valido": True,
            }
        ]
    )

print(f"Tolerância do backtest CDI-only  : {TOLERANCIA_BACKTEST}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

capital_inicial_estrategia = float(pd.to_numeric(extrair_parametro(df_parametros_globais, "capital_inicial_estrategia"), errors="coerce"))
codigo_sgs_cdi_diaria = str(extrair_parametro(df_parametros_globais, "codigo_sgs_cdi_diaria"))

calendario_alinhado_estrategias = construir_calendario_alinhado_estrategias(
    df_patrimonio_capitulacao,
    df_patrimonio_euforia,
    df_patrimonio_aleatorias_controle,
    df_patrimonio_mensais_controle,
    df_patrimonio_benchmark_ibovespa,
)

data_inicio_estrategias = pd.Timestamp(calendario_alinhado_estrategias.min())
data_fim_estrategias = pd.Timestamp(calendario_alinhado_estrategias.max())

df_cdi_preparada = preparar_cdi_padronizada(df_cdi_padronizada)

df_cdi_alinhada = (
    pd.DataFrame({"data": calendario_alinhado_estrategias})
    .merge(
        df_cdi_preparada[["data", "valor_cdi_percentual", "taxa_cdi_dia"]],
        on="data",
        how="left",
    )
    .sort_values("data")
    .reset_index(drop=True)
)

df_cdi_alinhada["valor_cdi_percentual"] = pd.to_numeric(df_cdi_alinhada["valor_cdi_percentual"], errors="coerce").ffill()
df_cdi_alinhada["taxa_cdi_dia"] = pd.to_numeric(df_cdi_alinhada["taxa_cdi_dia"], errors="coerce").ffill()

if df_cdi_alinhada.loc[(df_cdi_alinhada["data"] >= data_inicio_estrategias) & (df_cdi_alinhada["data"] <= data_fim_estrategias), "taxa_cdi_dia"].isna().any():
    raise ValueError("Persistiram valores ausentes da taxa CDI diária dentro do recorte alinhado das estratégias.")

df_evento_inicial_cdi_only = construir_evento_inicial_cdi_only(
    data_inicio_estrategias=data_inicio_estrategias,
    data_fim_estrategias=data_fim_estrategias,
    capital_inicial_estrategia=capital_inicial_estrategia,
)

print(f"Código SGS do CDI diário                    : {codigo_sgs_cdi_diaria}")
print(f"Data inicial do recorte das estratégias      : {data_inicio_estrategias.date()}")
print(f"Data final do recorte das estratégias        : {data_fim_estrategias.date()}")
print(f"Capital inicial da estratégia CDI-only       : {capital_inicial_estrategia:,.10f}")
print("OK")

# ============================================================
# 6) Construção das bases diárias da carteira CDI-only
# ============================================================

print("\n[6/10] Construção das bases diárias da carteira CDI-only...")

df_cdi_alinhada["fator_cdi_dia"] = 1.0 + df_cdi_alinhada["taxa_cdi_dia"]
df_cdi_alinhada["fator_cdi_acumulado"] = df_cdi_alinhada["fator_cdi_dia"].cumprod()
df_cdi_alinhada["patrimonio_total"] = capital_inicial_estrategia * df_cdi_alinhada["fator_cdi_acumulado"]
df_cdi_alinhada["valor_carteira_investida"] = df_cdi_alinhada["patrimonio_total"]
df_cdi_alinhada["saldo_caixa_fim_dia"] = 0.0
df_cdi_alinhada["saldo_caixa"] = 0.0

df_cdi_alinhada["rendimento_cdi_dia"] = df_cdi_alinhada["patrimonio_total"].diff()
df_cdi_alinhada.loc[
    df_cdi_alinhada.index[0],
    "rendimento_cdi_dia",
] = df_cdi_alinhada.loc[df_cdi_alinhada.index[0], "patrimonio_total"] - capital_inicial_estrategia

df_cdi_alinhada["valor_aporte_evento"] = np.where(
    df_cdi_alinhada["data"] == data_inicio_estrategias,
    capital_inicial_estrategia,
    0.0,
)
df_cdi_alinhada["valor_total_aportado"] = df_cdi_alinhada["valor_aporte_evento"].cumsum()
df_cdi_alinhada["max_patrimonio_acumulado"] = df_cdi_alinhada["patrimonio_total"].cummax()
df_cdi_alinhada["retorno_diario"] = df_cdi_alinhada["patrimonio_total"].pct_change().fillna(0.0)
df_cdi_alinhada["retorno_acumulado"] = (df_cdi_alinhada["patrimonio_total"] / capital_inicial_estrategia) - 1.0
df_cdi_alinhada["drawdown_atual"] = np.where(
    df_cdi_alinhada["max_patrimonio_acumulado"] > 0,
    (df_cdi_alinhada["patrimonio_total"] / df_cdi_alinhada["max_patrimonio_acumulado"]) - 1.0,
    0.0,
)
df_cdi_alinhada["flag_em_drawdown"] = df_cdi_alinhada["drawdown_atual"] < 0.0
df_cdi_alinhada["flag_evento_aporte"] = df_cdi_alinhada["data"] == data_inicio_estrategias
df_cdi_alinhada["request_id"] = np.where(
    df_cdi_alinhada["data"] == data_inicio_estrategias,
    "cdi_only__1",
    pd.NA,
)
df_cdi_alinhada["ordem_aporte_estrategia"] = np.where(
    df_cdi_alinhada["data"] == data_inicio_estrategias,
    1.0,
    pd.NA,
)
df_cdi_alinhada["valor_aporte_estrategia"] = df_cdi_alinhada["valor_aporte_evento"]
df_cdi_alinhada["valor_investido_efetivo_aporte"] = np.where(
    df_cdi_alinhada["data"] == data_inicio_estrategias,
    capital_inicial_estrategia,
    0.0,
)
df_cdi_alinhada["valor_caixa_residual_aporte"] = 0.0
df_cdi_alinhada["valor_caixa_trazido_aporte"] = 0.0
df_cdi_alinhada["n_posicoes_ativas"] = 1
df_cdi_alinhada["n_tickers_unicos_ativos"] = 1

df_cdi_alinhada["estrategia_id"] = ESTRATEGIA_ID_CDI_ONLY
df_cdi_alinhada["grupo_controle"] = "benchmark"
df_cdi_alinhada["familia_estrategia"] = ESTRATEGIA_ID_CDI_ONLY
df_cdi_alinhada["estrategia_referencia"] = ESTRATEGIA_ID_CDI_ONLY
df_cdi_alinhada["tipo_estrategia"] = ESTRATEGIA_ID_CDI_ONLY
df_cdi_alinhada["carteira_id"] = pd.NA
df_cdi_alinhada["replica_id"] = pd.NA
df_cdi_alinhada["capital_inicial_estrategia"] = capital_inicial_estrategia
df_cdi_alinhada["data_inicio_backtest"] = data_inicio_estrategias
df_cdi_alinhada["data_fim_backtest"] = data_fim_estrategias

df_posicoes_cdi_only = df_cdi_alinhada[
    [
        "data",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "carteira_id",
        "replica_id",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
        "patrimonio_total",
        "valor_carteira_investida",
    ]
].copy()

df_posicoes_cdi_only["ticker"] = TICKER_CDI_ONLY
df_posicoes_cdi_only["issuer_code"] = TICKER_CDI_ONLY
df_posicoes_cdi_only["nome"] = "Carteira CDI-Only"
df_posicoes_cdi_only["setor"] = pd.NA
df_posicoes_cdi_only["subsetor"] = pd.NA
df_posicoes_cdi_only["segmento"] = pd.NA
df_posicoes_cdi_only["quantidade_em_carteira"] = 1.0
df_posicoes_cdi_only["close_adj"] = df_posicoes_cdi_only["patrimonio_total"]
df_posicoes_cdi_only["custo_medio_unitario"] = capital_inicial_estrategia
df_posicoes_cdi_only["valor_mercado_posicao"] = df_posicoes_cdi_only["patrimonio_total"]
df_posicoes_cdi_only["peso_posicao_no_patrimonio"] = 1.0

df_posicoes_cdi_only = df_posicoes_cdi_only[
    [
        "data",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "carteira_id",
        "replica_id",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "quantidade_em_carteira",
        "custo_medio_unitario",
        "close_adj",
        "valor_mercado_posicao",
        "peso_posicao_no_patrimonio",
    ]
].copy()

df_patrimonio_cdi_only = df_cdi_alinhada[
    [
        "data",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "carteira_id",
        "replica_id",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
        "n_posicoes_ativas",
        "n_tickers_unicos_ativos",
        "valor_carteira_investida",
        "saldo_caixa_fim_dia",
        "saldo_caixa",
        "patrimonio_total",
        "valor_total_aportado",
        "valor_aporte_evento",
        "max_patrimonio_acumulado",
        "retorno_diario",
        "retorno_acumulado",
        "drawdown_atual",
        "flag_em_drawdown",
        "flag_evento_aporte",
        "request_id",
        "ordem_aporte_estrategia",
        "valor_aporte_estrategia",
        "valor_investido_efetivo_aporte",
        "valor_caixa_residual_aporte",
        "valor_caixa_trazido_aporte",
        "rendimento_cdi_dia",
        "taxa_cdi_dia",
        "valor_cdi_percentual",
        "fator_cdi_dia",
        "fator_cdi_acumulado",
    ]
].copy()

print(f"Linhas da base de posições CDI-only        : {len(df_posicoes_cdi_only):,}")
print(f"Linhas da base patrimonial CDI-only        : {len(df_patrimonio_cdi_only):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais, resumos e auditorias
# ============================================================

print("\n[7/10] Construção das tabelas formais, resumos e auditorias...")

df_regras_backtest_cdi_only = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "BTC1",
            "escopo": "execucao_inicial",
            "regra_operacional": "investir_todo_o_capital_inicial_em_cdi_no_primeiro_dia_do_recorte_alinhado",
            "detalhe": "A carteira CDI-only parte do mesmo capital inicial do projeto e inicia no primeiro dia do recorte final das estratégias.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "BTC2",
            "escopo": "calendario_alinhado",
            "regra_operacional": "alinhar_a_curva_cdi_only_ao_calendario_diario_unificado_das_estrategias_e_benchmarks",
            "detalhe": "A carteira CDI-only utiliza o mesmo calendário diário consolidado das estratégias reais, controles e benchmark Ibovespa.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "BTC3",
            "escopo": "remuneracao",
            "regra_operacional": "atualizar_o_patrimonio_diariamente_pela_taxa_cdi_padronizada",
            "detalhe": "O patrimônio CDI-only é atualizado diariamente pela taxa CDI padronizada da etapa 1.3.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "BTC4",
            "escopo": "posicao_sintetica",
            "regra_operacional": "representar_a_estrategia_como_uma_posicao_sintetica_unica_cdi",
            "detalhe": "A estratégia CDI-only é registrada como uma posição sintética única para manter compatibilidade estrutural com as demais bases do projeto.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "BTC5",
            "escopo": "auditoria",
            "regra_operacional": "reconciliar_o_patrimonio_final_com_o_fator_acumulado_do_cdi",
            "detalhe": "A auditoria verifica se o patrimônio final corresponde ao capital inicial multiplicado pelo fator acumulado do CDI no recorte do backtest.",
        },
    ]
)

patrimonio_final = float(df_patrimonio_cdi_only["patrimonio_total"].iloc[-1])
retorno_acumulado_final = float(df_patrimonio_cdi_only["retorno_acumulado"].iloc[-1])
drawdown_maximo = float(df_patrimonio_cdi_only["drawdown_atual"].min())
fator_cdi_acumulado_final = float(df_patrimonio_cdi_only["fator_cdi_acumulado"].iloc[-1])
valor_teorico_final = float(capital_inicial_estrategia * fator_cdi_acumulado_final)
desvio_patrimonio_final = abs(patrimonio_final - valor_teorico_final)
desvio_evento_inicial = abs(capital_inicial_estrategia - float(df_evento_inicial_cdi_only["capital_inicial_estrategia"].iloc[0]))

df_resumo_backtest_cdi_only = pd.DataFrame(
    [
        {
            "familia_estrategia": ESTRATEGIA_ID_CDI_ONLY,
            "estrategia_referencia": ESTRATEGIA_ID_CDI_ONLY,
            "tipo_estrategia": ESTRATEGIA_ID_CDI_ONLY,
            "estrategia_id": ESTRATEGIA_ID_CDI_ONLY,
            "grupo_controle": "benchmark",
            "fonte_benchmark": f"cdi_sgs_{codigo_sgs_cdi_diaria}",
            "carteira_id": pd.NA,
            "replica_id": pd.NA,
            "n_eventos_compra": 1,
            "data_inicio_backtest": data_inicio_estrategias,
            "data_fim_backtest": data_fim_estrategias,
            "n_datas_curva_patrimonial": int(len(df_patrimonio_cdi_only)),
            "n_datas_posicoes": int(df_posicoes_cdi_only["data"].nunique()),
            "capital_inicial_estrategia": capital_inicial_estrategia,
            "patrimonio_inicial": capital_inicial_estrategia,
            "patrimonio_final": patrimonio_final,
            "retorno_acumulado_final": retorno_acumulado_final,
            "drawdown_maximo": drawdown_maximo,
        }
    ]
)

df_auditoria_backtest_cdi_only = pd.DataFrame(
    [
        {
            "familia_estrategia": ESTRATEGIA_ID_CDI_ONLY,
            "estrategia_referencia": ESTRATEGIA_ID_CDI_ONLY,
            "tipo_estrategia": ESTRATEGIA_ID_CDI_ONLY,
            "estrategia_id": ESTRATEGIA_ID_CDI_ONLY,
            "grupo_controle": "benchmark",
            "fonte_benchmark": f"cdi_sgs_{codigo_sgs_cdi_diaria}",
            "capital_inicial_estrategia": capital_inicial_estrategia,
            "data_execucao_inicial": data_inicio_estrategias,
            "fator_cdi_acumulado_final": fator_cdi_acumulado_final,
            "patrimonio_final": patrimonio_final,
            "valor_teorico_final": valor_teorico_final,
            "desvio_evento_inicial": desvio_evento_inicial,
            "desvio_patrimonio_final": desvio_patrimonio_final,
            "flag_evento_inicial_valido": desvio_evento_inicial <= TOLERANCIA_BACKTEST,
            "flag_patrimonio_final_valido": desvio_patrimonio_final <= TOLERANCIA_BACKTEST,
            "flag_caixa_zero": bool(np.isclose(df_patrimonio_cdi_only["saldo_caixa_fim_dia"].to_numpy(dtype=float), 0.0, atol=TOLERANCIA_BACKTEST).all()),
            "flag_patrimonio_nao_negativo": bool((df_patrimonio_cdi_only["patrimonio_total"] >= -TOLERANCIA_BACKTEST).all()),
            "flag_posicao_unica_valida": bool(
                (df_posicoes_cdi_only["ticker"] == TICKER_CDI_ONLY).all()
                and np.isclose(df_posicoes_cdi_only["quantidade_em_carteira"].to_numpy(dtype=float), 1.0, atol=TOLERANCIA_BACKTEST).all()
            ),
        }
    ]
)

df_inconsistencias_backtest_cdi_only = (
    df_auditoria_backtest_cdi_only
    .loc[
        (~df_auditoria_backtest_cdi_only["flag_evento_inicial_valido"])
        | (~df_auditoria_backtest_cdi_only["flag_patrimonio_final_valido"])
        | (~df_auditoria_backtest_cdi_only["flag_caixa_zero"])
        | (~df_auditoria_backtest_cdi_only["flag_patrimonio_nao_negativo"])
        | (~df_auditoria_backtest_cdi_only["flag_posicao_unica_valida"])
    ]
    .copy()
)

df_distribuicao_anual_backtest_cdi_only = (
    df_patrimonio_cdi_only
    .assign(ano=lambda df: pd.to_datetime(df["data"], errors="coerce").dt.year)
    .groupby(
        [
            "familia_estrategia",
            "estrategia_referencia",
            "tipo_estrategia",
            "estrategia_id",
            "grupo_controle",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_datas_curva_patrimonial=("data", "count"),
        patrimonio_medio_ano=("patrimonio_total", "mean"),
        patrimonio_final_ano=("patrimonio_total", "last"),
        drawdown_minimo_ano=("drawdown_atual", "min"),
        retorno_ultimo_dia_ano=("retorno_acumulado", "last"),
    )
    .sort_values(["ano"])
    .reset_index(drop=True)
)

print(f"Inconsistências do backtest identificadas   : {len(df_inconsistencias_backtest_cdi_only):,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_backtest_cdi_only, caminho_tbl_regras_backtest_cdi_only, index=False)
salvar_dataframe(df_evento_inicial_cdi_only, caminho_base_evento_inicial_cdi_only, index=False)
salvar_dataframe(df_posicoes_cdi_only, caminho_base_posicoes_cdi_only, index=False)
salvar_dataframe(df_patrimonio_cdi_only, caminho_base_patrimonio_cdi_only, index=False)
salvar_dataframe(df_resumo_backtest_cdi_only, caminho_tbl_resumo_backtest_cdi_only, index=False)
salvar_dataframe(df_auditoria_backtest_cdi_only, caminho_tbl_auditoria_backtest_cdi_only, index=False)
salvar_dataframe(df_inconsistencias_backtest_cdi_only, caminho_tbl_inconsistencias_backtest_cdi_only, index=False)
salvar_dataframe(df_distribuicao_anual_backtest_cdi_only, caminho_tbl_distribuicao_anual_backtest_cdi_only, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_backtest_cdi_only = df_regras_backtest_cdi_only.copy()
df_amostra_evento_inicial_cdi_only = df_evento_inicial_cdi_only.copy()
df_amostra_posicoes_cdi_only = df_posicoes_cdi_only.head(20).copy()
df_amostra_patrimonio_cdi_only = df_patrimonio_cdi_only.head(20).copy()
df_amostra_resumo_backtest_cdi_only = df_resumo_backtest_cdi_only.copy()
df_amostra_auditoria_backtest_cdi_only = df_auditoria_backtest_cdi_only.copy()
df_amostra_inconsistencias_backtest_cdi_only = df_inconsistencias_backtest_cdi_only.head(20).copy()
df_amostra_distribuicao_anual_backtest_cdi_only = df_distribuicao_anual_backtest_cdi_only.head(20).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais do backtest da carteira CDI-only:")
print(df_amostra_regras_backtest_cdi_only.to_string(index=False))

print("\nEvento inicial da carteira CDI-only:")
print(df_amostra_evento_inicial_cdi_only.to_string(index=False))

print("\nBase diária de posições da carteira CDI-only - amostra:")
print(df_amostra_posicoes_cdi_only.to_string(index=False))

print("\nBase patrimonial da carteira CDI-only - amostra:")
print(df_amostra_patrimonio_cdi_only.to_string(index=False))

print("\nResumo do backtest da carteira CDI-only:")
print(df_amostra_resumo_backtest_cdi_only.to_string(index=False))

print("\nAuditoria do backtest da carteira CDI-only:")
print(df_amostra_auditoria_backtest_cdi_only.to_string(index=False))

print("\nInconsistências do backtest da carteira CDI-only - amostra:")
print(df_amostra_inconsistencias_backtest_cdi_only.to_string(index=False))

print("\nDistribuição anual do backtest da carteira CDI-only - amostra:")
print(df_amostra_distribuicao_anual_backtest_cdi_only.to_string(index=False))

print("\nArquivos salvos na subetapa 9.6:")
print(f"- {caminho_tbl_regras_backtest_cdi_only}")
print(f"- {caminho_base_evento_inicial_cdi_only}")
print(f"- {caminho_base_posicoes_cdi_only}")
print(f"- {caminho_base_patrimonio_cdi_only}")
print(f"- {caminho_tbl_resumo_backtest_cdi_only}")
print(f"- {caminho_tbl_auditoria_backtest_cdi_only}")
print(f"- {caminho_tbl_inconsistencias_backtest_cdi_only}")
print(f"- {caminho_tbl_distribuicao_anual_backtest_cdi_only}")

print("\nETAPA 9.6 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 9.6 - BACKTEST DA CARTEIRA CDI-ONLY

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - parâmetros globais                : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_1_tbl_parametros_globais.parquet
Entrada - CDI padronizada                   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1\1_3_base_cdi_padronizada.parquet
Entrada - patrimônio capitulação            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_1_base_patrimonio_capitulacao.parquet
Entrada - patrimônio euforia                : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_2_base_patrimonio_euforia

## Etapa 9.7) Consolidação das Curvas Patrimoniais

In [46]:
%%time
# ============================================================
# Etapa 9.7) Consolidação das Curvas Patrimoniais
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 9.7 - CONSOLIDAÇÃO DAS CURVAS PATRIMONIAIS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_patrimonio_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="patrimonio_capitulacao",
)

caminho_base_patrimonio_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="patrimonio_euforia",
)

caminho_base_patrimonio_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="patrimonio_aleatorias_controle",
)

caminho_base_patrimonio_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="patrimonio_mensais_controle",
)

caminho_base_patrimonio_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="base",
    nome="patrimonio_benchmark_ibovespa",
)

caminho_base_patrimonio_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="base",
    nome="patrimonio_cdi_only",
)

caminho_tbl_regras_consolidacao_curvas_patrimoniais = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="tbl",
    nome="regras_consolidacao_curvas_patrimoniais",
)

caminho_tbl_catalogo_curvas_patrimoniais = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="tbl",
    nome="catalogo_curvas_patrimoniais",
)

caminho_base_curvas_patrimoniais_consolidadas = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="base",
    nome="curvas_patrimoniais_consolidadas",
)

caminho_tbl_resumo_consolidacao_curvas_patrimoniais = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="tbl",
    nome="resumo_consolidacao_curvas_patrimoniais",
)

caminho_tbl_auditoria_consolidacao_curvas_patrimoniais = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="tbl",
    nome="auditoria_consolidacao_curvas_patrimoniais",
)

caminho_tbl_inconsistencias_consolidacao_curvas_patrimoniais = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="tbl",
    nome="inconsistencias_consolidacao_curvas_patrimoniais",
)

caminho_tbl_distribuicao_anual_curvas_patrimoniais = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_curvas_patrimoniais",
)

print(f"Entrada - patrimônio capitulação                : {caminho_base_patrimonio_capitulacao}")
print(f"Entrada - patrimônio euforia                    : {caminho_base_patrimonio_euforia}")
print(f"Entrada - patrimônio aleatórias controle        : {caminho_base_patrimonio_aleatorias_controle}")
print(f"Entrada - patrimônio mensais controle           : {caminho_base_patrimonio_mensais_controle}")
print(f"Entrada - patrimônio benchmark Ibovespa         : {caminho_base_patrimonio_benchmark_ibovespa}")
print(f"Entrada - patrimônio CDI-only                   : {caminho_base_patrimonio_cdi_only}")
print(f"Saída   - regras da consolidação                : {caminho_tbl_regras_consolidacao_curvas_patrimoniais}")
print(f"Saída   - catálogo das curvas                   : {caminho_tbl_catalogo_curvas_patrimoniais}")
print(f"Saída   - base mestra consolidada               : {caminho_base_curvas_patrimoniais_consolidadas}")
print(f"Saída   - resumo da consolidação                : {caminho_tbl_resumo_consolidacao_curvas_patrimoniais}")
print(f"Saída   - auditoria da consolidação             : {caminho_tbl_auditoria_consolidacao_curvas_patrimoniais}")
print(f"Saída   - inconsistências da consolidação       : {caminho_tbl_inconsistencias_consolidacao_curvas_patrimoniais}")
print(f"Saída   - distribuição anual consolidada        : {caminho_tbl_distribuicao_anual_curvas_patrimoniais}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_patrimonio_capitulacao = pd.read_parquet(caminho_base_patrimonio_capitulacao)
df_patrimonio_euforia = pd.read_parquet(caminho_base_patrimonio_euforia)
df_patrimonio_aleatorias_controle = pd.read_parquet(caminho_base_patrimonio_aleatorias_controle)
df_patrimonio_mensais_controle = pd.read_parquet(caminho_base_patrimonio_mensais_controle)
df_patrimonio_benchmark_ibovespa = pd.read_parquet(caminho_base_patrimonio_benchmark_ibovespa)
df_patrimonio_cdi_only = pd.read_parquet(caminho_base_patrimonio_cdi_only)

print(f"Patrimônio capitulação                    : {df_patrimonio_capitulacao.shape[0]:,} linhas x {df_patrimonio_capitulacao.shape[1]} colunas")
print(f"Patrimônio euforia                        : {df_patrimonio_euforia.shape[0]:,} linhas x {df_patrimonio_euforia.shape[1]} colunas")
print(f"Patrimônio aleatórias controle            : {df_patrimonio_aleatorias_controle.shape[0]:,} linhas x {df_patrimonio_aleatorias_controle.shape[1]} colunas")
print(f"Patrimônio mensais controle               : {df_patrimonio_mensais_controle.shape[0]:,} linhas x {df_patrimonio_mensais_controle.shape[1]} colunas")
print(f"Patrimônio benchmark Ibovespa             : {df_patrimonio_benchmark_ibovespa.shape[0]:,} linhas x {df_patrimonio_benchmark_ibovespa.shape[1]} colunas")
print(f"Patrimônio CDI-only                       : {df_patrimonio_cdi_only.shape[0]:,} linhas x {df_patrimonio_cdi_only.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_CONSOLIDACAO = 0.000001

COLUNAS_MESTRAS_CURVAS = [
    "data",
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "flag_estrategia_referencia_preenchida",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "origem_backtest",
    "ordem_exibicao",
    "capital_inicial_estrategia",
    "data_inicio_backtest",
    "data_fim_backtest",
    "flag_data_ativa_backtest",
    "flag_primeira_data_ativa",
    "flag_ultima_data_ativa",
    "n_posicoes_ativas",
    "n_tickers_unicos_ativos",
    "valor_carteira_investida",
    "saldo_caixa_fim_dia",
    "saldo_caixa",
    "patrimonio_total",
    "valor_total_aportado",
    "valor_aporte_evento",
    "max_patrimonio_acumulado",
    "retorno_diario",
    "retorno_acumulado",
    "drawdown_atual",
    "flag_em_drawdown",
    "flag_evento_aporte",
    "request_id",
    "ordem_aporte_estrategia",
    "valor_aporte_estrategia",
    "valor_investido_efetivo_aporte",
    "valor_caixa_residual_aporte",
    "valor_caixa_trazido_aporte",
    "rendimento_cdi_dia",
    "taxa_cdi_dia",
    "fonte_benchmark",
    "preco_benchmark_fechamento",
    "valor_cdi_percentual",
    "fator_cdi_dia",
    "fator_cdi_acumulado",
]

def detectar_coluna(df, candidatos, obrigatoria=True):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna

    if obrigatoria:
        raise ValueError(f"Não foi possível localizar nenhuma das colunas candidatas: {candidatos}")

    return None

def garantir_coluna(df, nome_coluna, valor_default=pd.NA):
    if nome_coluna not in df.columns:
        df[nome_coluna] = valor_default
    return df

def padronizar_texto(df, colunas_upper=None, colunas_lower=None):
    colunas_upper = [] if colunas_upper is None else colunas_upper
    colunas_lower = [] if colunas_lower is None else colunas_lower

    for coluna in colunas_upper:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.upper()

    for coluna in colunas_lower:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip().str.lower()

    return df

def normalizar_identificador_numero(valor):
    if pd.isna(valor):
        return "na"

    try:
        valor_float = float(valor)
        if np.isfinite(valor_float) and abs(valor_float - int(round(valor_float))) <= TOLERANCIA_CONSOLIDACAO:
            return str(int(round(valor_float)))
        return f"{valor_float:.10f}".rstrip("0").rstrip(".")
    except Exception:
        return str(valor).strip().lower()

def gerar_chave_estrategia(df):
    serie_estrategia = df["estrategia_id"].astype("string").str.strip().str.lower()
    serie_carteira = df["carteira_id"].apply(normalizar_identificador_numero)
    serie_replica = df["replica_id"].apply(normalizar_identificador_numero)

    return (
        serie_estrategia
        + "__carteira_"
        + serie_carteira.astype("string")
        + "__replica_"
        + serie_replica.astype("string")
    )

def coalescer_texto(serie_a, serie_b):
    a = serie_a.astype("string").str.strip()
    b = serie_b.astype("string").str.strip()

    mascara_a_vazia = a.isna() | (a == "") | (a.str.lower() == "<na>") | (a.str.lower() == "none")
    return a.mask(mascara_a_vazia, b)

def identificar_texto_preenchido(serie):
    serie_texto = serie.astype("string").str.strip()
    serie_lower = serie_texto.str.lower()

    return (
        serie_texto.notna()
        & (serie_texto != "")
        & (~serie_lower.isin(["<na>", "none", "nan", "null"]))
    )

def classificar_categoria(row):
    grupo_controle = str(row["grupo_controle"]).strip().lower()
    estrategia_id = str(row["estrategia_id"]).strip().lower()
    estrategia_ref = str(row["estrategia_referencia_padronizada"]).strip().lower()

    if estrategia_id == "cdi_only":
        return "benchmark"

    if estrategia_id == "ibovespa_buy_and_hold":
        return "benchmark"

    if grupo_controle == "sinal":
        return "estrategia_real"

    if grupo_controle == "aleatoria":
        return "controle_aleatorio"

    if grupo_controle == "mensal":
        return "controle_mensal"

    if grupo_controle == "benchmark":
        return "benchmark"

    if estrategia_ref in ["capitulacao", "euforia"]:
        return "controle_aleatorio"

    return "outros"

def classificar_subcategoria(row):
    categoria = row["categoria_estrategia"]
    estrategia_id = str(row["estrategia_id"]).strip().lower()
    estrategia_ref = str(row["estrategia_referencia_padronizada"]).strip().lower()

    if categoria == "estrategia_real":
        if estrategia_id == "capitulacao":
            return "capitulacao"
        if estrategia_id == "euforia":
            return "euforia"
        return "real_outros"

    if categoria == "controle_aleatorio":
        if estrategia_ref == "capitulacao":
            return "aleatoria_capitulacao"
        if estrategia_ref == "euforia":
            return "aleatoria_euforia"
        return "aleatoria_outras"

    if categoria == "controle_mensal":
        return "mensal"

    if categoria == "benchmark":
        if estrategia_id == "ibovespa_buy_and_hold":
            return "ibovespa_buy_and_hold"
        if estrategia_id == "cdi_only":
            return "cdi_only"
        return "benchmark_outros"

    return "outros"

def gerar_nome_exibicao(row):
    estrategia_id = str(row["estrategia_id"]).strip().lower()
    categoria = row["categoria_estrategia"]
    subcategoria = row["subcategoria_estrategia"]
    replica_id = normalizar_identificador_numero(row["replica_id"])
    carteira_id = normalizar_identificador_numero(row["carteira_id"])

    if estrategia_id == "capitulacao":
        return "Capitulação"

    if estrategia_id == "euforia":
        return "Euforia"

    if estrategia_id == "ibovespa_buy_and_hold":
        return "Ibovespa Buy and Hold"

    if estrategia_id == "cdi_only":
        return "CDI-Only"

    if categoria == "controle_aleatorio" and subcategoria == "aleatoria_capitulacao":
        return f"Aleatória da Capitulação #{replica_id}"

    if categoria == "controle_aleatorio" and subcategoria == "aleatoria_euforia":
        return f"Aleatória da Euforia #{replica_id}"

    if categoria == "controle_mensal":
        nome_base = estrategia_id.replace("_", " ").strip().title()
        if carteira_id != "na":
            return f"{nome_base} (Carteira {carteira_id})"
        return nome_base

    return estrategia_id.replace("_", " ").strip().title()

def gerar_ordem_exibicao(row):
    estrategia_id = str(row["estrategia_id"]).strip().lower()
    categoria = row["categoria_estrategia"]
    subcategoria = row["subcategoria_estrategia"]
    replica_id = normalizar_identificador_numero(row["replica_id"])
    carteira_id = normalizar_identificador_numero(row["carteira_id"])

    if estrategia_id == "capitulacao":
        return 10

    if estrategia_id == "euforia":
        return 20

    if categoria == "controle_aleatorio" and subcategoria == "aleatoria_capitulacao":
        return 100 + int(replica_id) if replica_id != "na" else 199

    if categoria == "controle_aleatorio" and subcategoria == "aleatoria_euforia":
        return 200 + int(replica_id) if replica_id != "na" else 299

    if categoria == "controle_mensal":
        nome_ordem = estrategia_id
        base = 300
        mapa_base = {
            "ibovespa_aportes_mensais": 310,
            "acoes_elegiveis_aportes_mensais": 320,
            "acoes_aleatorias_20_aportes_mensais_carteira_1": 331,
            "acoes_aleatorias_20_aportes_mensais_carteira_2": 332,
            "acoes_aleatorias_20_aportes_mensais_carteira_3": 333,
            "acoes_aleatorias_20_aportes_mensais_carteira_4": 334,
            "acoes_aleatorias_20_aportes_mensais_carteira_5": 335,
            "acoes_aleatorias_30_aportes_mensais_carteira_1": 341,
            "acoes_aleatorias_30_aportes_mensais_carteira_2": 342,
            "acoes_aleatorias_30_aportes_mensais_carteira_3": 343,
            "acoes_aleatorias_30_aportes_mensais_carteira_4": 344,
            "acoes_aleatorias_30_aportes_mensais_carteira_5": 345,
        }
        if nome_ordem in mapa_base:
            return mapa_base[nome_ordem]
        if carteira_id != "na":
            return base + int(carteira_id)
        return 399

    if estrategia_id == "ibovespa_buy_and_hold":
        return 900

    if estrategia_id == "cdi_only":
        return 910

    return 999

def preparar_base_patrimonial(df_patrimonio, origem_backtest):
    df_out = df_patrimonio.copy()

    coluna_data = detectar_coluna(df_out, ["data", "date"])
    coluna_patrimonio = detectar_coluna(df_out, ["patrimonio_total"])
    coluna_retorno_diario = detectar_coluna(df_out, ["retorno_diario"], obrigatoria=False)
    coluna_retorno_acumulado = detectar_coluna(df_out, ["retorno_acumulado"], obrigatoria=False)
    coluna_drawdown = detectar_coluna(df_out, ["drawdown_atual"], obrigatoria=False)
    coluna_max_patrimonio = detectar_coluna(df_out, ["max_patrimonio_acumulado"], obrigatoria=False)
    coluna_valor_carteira = detectar_coluna(df_out, ["valor_carteira_investida"], obrigatoria=False)
    coluna_saldo_caixa_fim = detectar_coluna(df_out, ["saldo_caixa_fim_dia"], obrigatoria=False)
    coluna_saldo_caixa = detectar_coluna(df_out, ["saldo_caixa"], obrigatoria=False)
    coluna_n_posicoes = detectar_coluna(df_out, ["n_posicoes_ativas"], obrigatoria=False)
    coluna_n_tickers = detectar_coluna(df_out, ["n_tickers_unicos_ativos"], obrigatoria=False)
    coluna_flag_drawdown = detectar_coluna(df_out, ["flag_em_drawdown"], obrigatoria=False)
    coluna_flag_aporte = detectar_coluna(df_out, ["flag_evento_aporte"], obrigatoria=False)
    coluna_request = detectar_coluna(df_out, ["request_id"], obrigatoria=False)
    coluna_ordem_aporte = detectar_coluna(df_out, ["ordem_aporte_estrategia"], obrigatoria=False)
    coluna_valor_total_aportado = detectar_coluna(df_out, ["valor_total_aportado"], obrigatoria=False)
    coluna_valor_aporte_evento = detectar_coluna(df_out, ["valor_aporte_evento"], obrigatoria=False)
    coluna_valor_aporte_estrategia = detectar_coluna(df_out, ["valor_aporte_estrategia"], obrigatoria=False)
    coluna_valor_investido_efetivo = detectar_coluna(df_out, ["valor_investido_efetivo_aporte"], obrigatoria=False)
    coluna_valor_caixa_residual = detectar_coluna(df_out, ["valor_caixa_residual_aporte"], obrigatoria=False)
    coluna_valor_caixa_trazido = detectar_coluna(df_out, ["valor_caixa_trazido_aporte"], obrigatoria=False)
    coluna_rendimento_cdi_dia = detectar_coluna(df_out, ["rendimento_cdi_dia"], obrigatoria=False)
    coluna_taxa_cdi_dia = detectar_coluna(df_out, ["taxa_cdi_dia"], obrigatoria=False)
    coluna_fonte_benchmark = detectar_coluna(df_out, ["fonte_benchmark"], obrigatoria=False)
    coluna_preco_benchmark = detectar_coluna(df_out, ["preco_benchmark_fechamento"], obrigatoria=False)
    coluna_valor_cdi_percentual = detectar_coluna(df_out, ["valor_cdi_percentual"], obrigatoria=False)
    coluna_fator_cdi_dia = detectar_coluna(df_out, ["fator_cdi_dia"], obrigatoria=False)
    coluna_fator_cdi_acum = detectar_coluna(df_out, ["fator_cdi_acumulado"], obrigatoria=False)

    df_out[coluna_data] = pd.to_datetime(df_out[coluna_data], errors="coerce")
    df_out[coluna_patrimonio] = pd.to_numeric(df_out[coluna_patrimonio], errors="coerce")

    for coluna in [
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "carteira_id",
        "replica_id",
        "capital_inicial_estrategia",
        "data_inicio_backtest",
        "data_fim_backtest",
    ]:
        df_out = garantir_coluna(df_out, coluna)

    df_out = padronizar_texto(
        df_out,
        colunas_lower=[
            "estrategia_id",
            "grupo_controle",
            "familia_estrategia",
            "estrategia_referencia",
            "tipo_estrategia",
        ],
    )

    df_out["capital_inicial_estrategia"] = pd.to_numeric(df_out["capital_inicial_estrategia"], errors="coerce")
    df_out["carteira_id"] = pd.to_numeric(df_out["carteira_id"], errors="coerce")
    df_out["replica_id"] = pd.to_numeric(df_out["replica_id"], errors="coerce")
    df_out["data_inicio_backtest"] = pd.to_datetime(df_out["data_inicio_backtest"], errors="coerce")
    df_out["data_fim_backtest"] = pd.to_datetime(df_out["data_fim_backtest"], errors="coerce")

    if coluna_valor_carteira is not None:
        df_out["valor_carteira_investida"] = pd.to_numeric(df_out[coluna_valor_carteira], errors="coerce")
    else:
        df_out["valor_carteira_investida"] = pd.to_numeric(df_out[coluna_patrimonio], errors="coerce")

    if coluna_saldo_caixa_fim is not None:
        df_out["saldo_caixa_fim_dia"] = pd.to_numeric(df_out[coluna_saldo_caixa_fim], errors="coerce").fillna(0.0)
    else:
        df_out["saldo_caixa_fim_dia"] = 0.0

    if coluna_saldo_caixa is not None:
        df_out["saldo_caixa"] = pd.to_numeric(df_out[coluna_saldo_caixa], errors="coerce").fillna(df_out["saldo_caixa_fim_dia"])
    else:
        df_out["saldo_caixa"] = df_out["saldo_caixa_fim_dia"]

    if coluna_retorno_diario is not None:
        df_out["retorno_diario"] = pd.to_numeric(df_out[coluna_retorno_diario], errors="coerce")
    else:
        df_out["retorno_diario"] = pd.NA

    if coluna_retorno_acumulado is not None:
        df_out["retorno_acumulado"] = pd.to_numeric(df_out[coluna_retorno_acumulado], errors="coerce")
    else:
        df_out["retorno_acumulado"] = pd.NA

    if coluna_drawdown is not None:
        df_out["drawdown_atual"] = pd.to_numeric(df_out[coluna_drawdown], errors="coerce")
    else:
        df_out["drawdown_atual"] = pd.NA

    if coluna_max_patrimonio is not None:
        df_out["max_patrimonio_acumulado"] = pd.to_numeric(df_out[coluna_max_patrimonio], errors="coerce")
    else:
        df_out["max_patrimonio_acumulado"] = pd.NA

    if coluna_n_posicoes is not None:
        df_out["n_posicoes_ativas"] = pd.to_numeric(df_out[coluna_n_posicoes], errors="coerce")
    else:
        df_out["n_posicoes_ativas"] = pd.NA

    if coluna_n_tickers is not None:
        df_out["n_tickers_unicos_ativos"] = pd.to_numeric(df_out[coluna_n_tickers], errors="coerce")
    else:
        df_out["n_tickers_unicos_ativos"] = pd.NA

    if coluna_flag_drawdown is not None:
        df_out["flag_em_drawdown"] = df_out[coluna_flag_drawdown].fillna(False)
    else:
        df_out["flag_em_drawdown"] = False

    if coluna_flag_aporte is not None:
        df_out["flag_evento_aporte"] = df_out[coluna_flag_aporte].fillna(False)
    else:
        df_out["flag_evento_aporte"] = False

    df_out["request_id"] = df_out[coluna_request].astype("string") if coluna_request is not None else pd.Series(pd.array([pd.NA] * len(df_out), dtype="string"))
    df_out["ordem_aporte_estrategia"] = pd.to_numeric(df_out[coluna_ordem_aporte], errors="coerce") if coluna_ordem_aporte is not None else pd.NA
    df_out["valor_total_aportado"] = pd.to_numeric(df_out[coluna_valor_total_aportado], errors="coerce") if coluna_valor_total_aportado is not None else pd.NA
    df_out["valor_aporte_evento"] = pd.to_numeric(df_out[coluna_valor_aporte_evento], errors="coerce") if coluna_valor_aporte_evento is not None else pd.NA
    df_out["valor_aporte_estrategia"] = pd.to_numeric(df_out[coluna_valor_aporte_estrategia], errors="coerce") if coluna_valor_aporte_estrategia is not None else pd.NA
    df_out["valor_investido_efetivo_aporte"] = pd.to_numeric(df_out[coluna_valor_investido_efetivo], errors="coerce") if coluna_valor_investido_efetivo is not None else pd.NA
    df_out["valor_caixa_residual_aporte"] = pd.to_numeric(df_out[coluna_valor_caixa_residual], errors="coerce") if coluna_valor_caixa_residual is not None else pd.NA
    df_out["valor_caixa_trazido_aporte"] = pd.to_numeric(df_out[coluna_valor_caixa_trazido], errors="coerce") if coluna_valor_caixa_trazido is not None else pd.NA
    df_out["rendimento_cdi_dia"] = pd.to_numeric(df_out[coluna_rendimento_cdi_dia], errors="coerce") if coluna_rendimento_cdi_dia is not None else pd.NA
    df_out["taxa_cdi_dia"] = pd.to_numeric(df_out[coluna_taxa_cdi_dia], errors="coerce") if coluna_taxa_cdi_dia is not None else pd.NA
    df_out["fonte_benchmark"] = df_out[coluna_fonte_benchmark].astype("string") if coluna_fonte_benchmark is not None else pd.NA
    df_out["preco_benchmark_fechamento"] = pd.to_numeric(df_out[coluna_preco_benchmark], errors="coerce") if coluna_preco_benchmark is not None else pd.NA
    df_out["valor_cdi_percentual"] = pd.to_numeric(df_out[coluna_valor_cdi_percentual], errors="coerce") if coluna_valor_cdi_percentual is not None else pd.NA
    df_out["fator_cdi_dia"] = pd.to_numeric(df_out[coluna_fator_cdi_dia], errors="coerce") if coluna_fator_cdi_dia is not None else pd.NA
    df_out["fator_cdi_acumulado"] = pd.to_numeric(df_out[coluna_fator_cdi_acum], errors="coerce") if coluna_fator_cdi_acum is not None else pd.NA

    df_out = (
        df_out
        .rename(columns={coluna_data: "data", coluna_patrimonio: "patrimonio_total"})
        .sort_values(["estrategia_id", "data"])
        .reset_index(drop=True)
    )

    df_out["flag_estrategia_referencia_preenchida"] = identificar_texto_preenchido(df_out["estrategia_referencia"])
    df_out["estrategia_referencia_padronizada"] = (
        df_out["estrategia_referencia"]
        .astype("string")
        .str.strip()
        .str.lower()
        .where(df_out["flag_estrategia_referencia_preenchida"], pd.NA)
    )

    df_out["chave_estrategia"] = gerar_chave_estrategia(df_out)

    df_out["origem_backtest"] = origem_backtest

    # Complementos defensivos para manter contrato estrutural uniforme.
    df_out["max_patrimonio_acumulado"] = (
        pd.to_numeric(df_out["max_patrimonio_acumulado"], errors="coerce")
        .groupby(df_out["chave_estrategia"])
        .transform(lambda s: s.ffill())
    )

    mascara_max_vazia = df_out["max_patrimonio_acumulado"].isna()
    if mascara_max_vazia.any():
        df_out.loc[mascara_max_vazia, "max_patrimonio_acumulado"] = (
            df_out.loc[mascara_max_vazia]
            .groupby("chave_estrategia")["patrimonio_total"]
            .cummax()
            .to_numpy()
        )

    if df_out["retorno_diario"].isna().any():
        df_out["retorno_diario"] = (
            df_out.groupby("chave_estrategia")["patrimonio_total"]
            .pct_change()
            .fillna(0.0)
        )

    if df_out["retorno_acumulado"].isna().any():
        df_out["retorno_acumulado"] = np.where(
            df_out["capital_inicial_estrategia"] > 0,
            (df_out["patrimonio_total"] / df_out["capital_inicial_estrategia"]) - 1.0,
            0.0,
        )

    if df_out["drawdown_atual"].isna().any():
        df_out["drawdown_atual"] = np.where(
            df_out["max_patrimonio_acumulado"] > 0,
            (df_out["patrimonio_total"] / df_out["max_patrimonio_acumulado"]) - 1.0,
            0.0,
        )

    df_out["categoria_estrategia"] = df_out.apply(classificar_categoria, axis=1)
    df_out["subcategoria_estrategia"] = df_out.apply(classificar_subcategoria, axis=1)
    df_out["nome_exibicao_estrategia"] = df_out.apply(gerar_nome_exibicao, axis=1)
    df_out["ordem_exibicao"] = df_out.apply(gerar_ordem_exibicao, axis=1)

    return df_out

print(f"Tolerância da consolidação                 : {TOLERANCIA_CONSOLIDACAO}")
print("OK")

# ============================================================
# 5) Preparação das bases e validações estruturais
# ============================================================

print("\n[5/10] Preparação das bases e validações estruturais...")

df_capitulacao_pad = preparar_base_patrimonial(df_patrimonio_capitulacao, "etapa_9_1")
df_euforia_pad = preparar_base_patrimonial(df_patrimonio_euforia, "etapa_9_2")
df_aleatorias_pad = preparar_base_patrimonial(df_patrimonio_aleatorias_controle, "etapa_9_3")
df_mensais_pad = preparar_base_patrimonial(df_patrimonio_mensais_controle, "etapa_9_4")
df_ibovespa_pad = preparar_base_patrimonial(df_patrimonio_benchmark_ibovespa, "etapa_9_5")
df_cdi_pad = preparar_base_patrimonial(df_patrimonio_cdi_only, "etapa_9_6")

df_curvas_ativas = pd.concat(
    [
        df_capitulacao_pad,
        df_euforia_pad,
        df_aleatorias_pad,
        df_mensais_pad,
        df_ibovespa_pad,
        df_cdi_pad,
    ],
    axis=0,
    ignore_index=True,
)

df_curvas_ativas = (
    df_curvas_ativas
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "data"])
    .reset_index(drop=True)
)

n_chaves_unicas = int(df_curvas_ativas["chave_estrategia"].nunique())
n_linhas_ativas = int(len(df_curvas_ativas))
n_estrategia_referencia_ausente_inputs = int((~df_curvas_ativas["flag_estrategia_referencia_preenchida"].fillna(False)).sum())
data_min_global = pd.Timestamp(df_curvas_ativas["data"].min())
data_max_global = pd.Timestamp(df_curvas_ativas["data"].max())

print(f"Estratégias consolidadas ativas            : {n_chaves_unicas:,}")
print(f"Linhas ativas consolidadas                 : {n_linhas_ativas:,}")
print(f"Estrategia_referencia ausente nos inputs   : {n_estrategia_referencia_ausente_inputs:,}")
print(f"Data inicial global                        : {data_min_global.date()}")
print(f"Data final global                          : {data_max_global.date()}")
print("OK")

# ============================================================
# 6) Construção da base mestra consolidada e do catálogo
# ============================================================

print("\n[6/10] Construção da base mestra consolidada e do catálogo...")

df_catalogo_curvas_patrimoniais = (
    df_curvas_ativas[
        [
            "chave_estrategia",
            "estrategia_id",
            "grupo_controle",
            "familia_estrategia",
            "estrategia_referencia",
            "estrategia_referencia_padronizada",
            "flag_estrategia_referencia_preenchida",
            "tipo_estrategia",
            "carteira_id",
            "replica_id",
            "nome_exibicao_estrategia",
            "categoria_estrategia",
            "subcategoria_estrategia",
            "origem_backtest",
            "ordem_exibicao",
            "capital_inicial_estrategia",
            "data_inicio_backtest",
            "data_fim_backtest",
            "fonte_benchmark",
        ]
    ]
    .drop_duplicates(subset=["chave_estrategia"])
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia"])
    .reset_index(drop=True)
)

df_intervalos = (
    df_curvas_ativas
    .groupby("chave_estrategia", as_index=False, dropna=False)
    .agg(
        data_inicio_serie=("data", "min"),
        data_fim_serie=("data", "max"),
        n_linhas_ativas=("data", "count"),
    )
)

df_catalogo_curvas_patrimoniais = (
    df_catalogo_curvas_patrimoniais
    .merge(df_intervalos, on="chave_estrategia", how="left")
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia"])
    .reset_index(drop=True)
)

df_calendario_mestre = pd.DataFrame(
    {"data": pd.Series(pd.Index(sorted(df_curvas_ativas["data"].dropna().unique())), dtype="datetime64[ns]")}
)

df_curvas_ativas["flag_data_observada_ativo"] = True

df_curvas_patrimoniais_consolidadas = (
    df_catalogo_curvas_patrimoniais.assign(_tmp=1)
    .merge(df_calendario_mestre.assign(_tmp=1), on="_tmp", how="inner")
    .drop(columns="_tmp")
    .merge(
        df_curvas_ativas[
            [
                "data",
                "chave_estrategia",
                "capital_inicial_estrategia",
                "data_inicio_backtest",
                "data_fim_backtest",
                "n_posicoes_ativas",
                "n_tickers_unicos_ativos",
                "valor_carteira_investida",
                "saldo_caixa_fim_dia",
                "saldo_caixa",
                "patrimonio_total",
                "valor_total_aportado",
                "valor_aporte_evento",
                "max_patrimonio_acumulado",
                "retorno_diario",
                "retorno_acumulado",
                "drawdown_atual",
                "flag_em_drawdown",
                "flag_evento_aporte",
                "request_id",
                "ordem_aporte_estrategia",
                "valor_aporte_estrategia",
                "valor_investido_efetivo_aporte",
                "valor_caixa_residual_aporte",
                "valor_caixa_trazido_aporte",
                "rendimento_cdi_dia",
                "taxa_cdi_dia",
                "fonte_benchmark",
                "preco_benchmark_fechamento",
                "valor_cdi_percentual",
                "fator_cdi_dia",
                "fator_cdi_acumulado",
                "flag_data_observada_ativo",
            ]
        ],
        on=["chave_estrategia", "data"],
        how="left",
        suffixes=("", "_ativo"),
    )
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "data"])
    .reset_index(drop=True)
)

df_curvas_patrimoniais_consolidadas["flag_data_observada_ativo"] = (
    df_curvas_patrimoniais_consolidadas["flag_data_observada_ativo"].fillna(False)
)

# A janela ativa consolidada deve refletir exatamente as datas realmente observadas na curva-fonte.
# Isso evita marcar como "ativas" datas intermediárias do calendário mestre onde a estratégia nunca teve patrimônio registrado.
df_curvas_patrimoniais_consolidadas["flag_data_ativa_backtest"] = (
    df_curvas_patrimoniais_consolidadas["flag_data_observada_ativo"]
)

# Propagação dos metadados de janela para todas as linhas da malha consolidada.
df_curvas_patrimoniais_consolidadas["data_inicio_backtest"] = (
    pd.to_datetime(df_curvas_patrimoniais_consolidadas["data_inicio_backtest"], errors="coerce")
    .fillna(pd.to_datetime(df_curvas_patrimoniais_consolidadas["data_inicio_serie"], errors="coerce"))
)

df_curvas_patrimoniais_consolidadas["data_fim_backtest"] = (
    pd.to_datetime(df_curvas_patrimoniais_consolidadas["data_fim_backtest"], errors="coerce")
    .fillna(pd.to_datetime(df_curvas_patrimoniais_consolidadas["data_fim_serie"], errors="coerce"))
)

df_curvas_patrimoniais_consolidadas["flag_primeira_data_ativa"] = (
    df_curvas_patrimoniais_consolidadas["flag_data_ativa_backtest"]
    & (df_curvas_patrimoniais_consolidadas["data"] == df_curvas_patrimoniais_consolidadas["data_inicio_serie"])
)

df_curvas_patrimoniais_consolidadas["flag_ultima_data_ativa"] = (
    df_curvas_patrimoniais_consolidadas["flag_data_ativa_backtest"]
    & (df_curvas_patrimoniais_consolidadas["data"] == df_curvas_patrimoniais_consolidadas["data_fim_serie"])
)

df_curvas_patrimoniais_consolidadas["fonte_benchmark"] = coalescer_texto(
    df_curvas_patrimoniais_consolidadas["fonte_benchmark"],
    pd.Series(pd.array([pd.NA] * len(df_curvas_patrimoniais_consolidadas), dtype="string")),
)

for coluna in COLUNAS_MESTRAS_CURVAS:
    df_curvas_patrimoniais_consolidadas = garantir_coluna(df_curvas_patrimoniais_consolidadas, coluna)

if "flag_data_observada_ativo" in df_curvas_patrimoniais_consolidadas.columns:
    df_curvas_patrimoniais_consolidadas = df_curvas_patrimoniais_consolidadas.drop(columns=["flag_data_observada_ativo"])

df_curvas_patrimoniais_consolidadas = (
    df_curvas_patrimoniais_consolidadas[COLUNAS_MESTRAS_CURVAS]
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "data"])
    .reset_index(drop=True)
)

print(f"Linhas da base mestra consolidada          : {len(df_curvas_patrimoniais_consolidadas):,}")
print(f"Estratégias no catálogo                    : {len(df_catalogo_curvas_patrimoniais):,}")
print("OK")

# ============================================================
# 7) Construção das tabelas formais, resumos e auditorias
# ============================================================

print("\n[7/10] Construção das tabelas formais, resumos e auditorias...")

df_regras_consolidacao_curvas_patrimoniais = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "BCC1",
            "escopo": "abrangencia",
            "regra_operacional": "reunir_as_curvas_patrimoniais_das_etapas_9_1_a_9_6_em_uma_base_unica",
            "detalhe": "A consolidação reúne as curvas patrimoniais de capitulação, euforia, controles aleatórios, controles mensais, benchmark Ibovespa e CDI-only.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "BCC2",
            "escopo": "padronizacao",
            "regra_operacional": "padronizar_nomenclatura_chaves_e_colunas_essenciais_em_todas_as_fontes",
            "detalhe": "Todas as fontes são convertidas para um contrato estrutural comum de colunas patrimoniais, identificadores e metadados, sem preenchimento defensivo de estrategia_referencia.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "BCC3",
            "escopo": "calendario",
            "regra_operacional": "alinhar_todas_as_estrategias_ao_calendario_mestre_por_data",
            "detalhe": "A base mestra utiliza o calendário união de todas as curvas, preservando os valores apenas no intervalo ativo de cada estratégia.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "BCC4",
            "escopo": "catalogo",
            "regra_operacional": "gerar_catalogo_unico_de_estrategias_com_chave_exibicao_e_classificacao",
            "detalhe": "Cada estratégia recebe uma chave consolidada, categoria analítica, nome de exibição e ordem de exibição padronizada.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "BCC5",
            "escopo": "auditoria",
            "regra_operacional": "validar_contiguidade_das_janelas_ativas_e_coerencia_dos_metadados_consolidados",
            "detalhe": "A auditoria compara a contagem observada de linhas ativas com a contagem esperada e exige estrategia_referencia preenchida na origem para todas as curvas.",
        },
    ]
)

df_resumo_consolidacao_curvas_patrimoniais = (
    df_curvas_ativas
    .groupby(
        [
            "chave_estrategia",
            "estrategia_id",
            "grupo_controle",
            "familia_estrategia",
            "estrategia_referencia",
            "estrategia_referencia_padronizada",
            "flag_estrategia_referencia_preenchida",
            "tipo_estrategia",
            "carteira_id",
            "replica_id",
            "nome_exibicao_estrategia",
            "categoria_estrategia",
            "subcategoria_estrategia",
            "origem_backtest",
            "ordem_exibicao",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        data_inicio_serie=("data", "min"),
        data_fim_serie=("data", "max"),
        n_linhas_ativas=("data", "count"),
        patrimonio_inicial_ativo=("patrimonio_total", "first"),
        patrimonio_final_ativo=("patrimonio_total", "last"),
        retorno_final_ativo=("retorno_acumulado", "last"),
        drawdown_maximo_ativo=("drawdown_atual", "min"),
        capital_inicial_estrategia=("capital_inicial_estrategia", "first"),
        n_eventos_aporte_ativos=("flag_evento_aporte", lambda s: int(pd.Series(s).fillna(False).sum())),
    )
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia"])
    .reset_index(drop=True)
)

df_auditoria_consolidacao_curvas_patrimoniais = (
    df_curvas_patrimoniais_consolidadas
    .groupby(
        [
            "chave_estrategia",
            "estrategia_id",
            "grupo_controle",
            "familia_estrategia",
            "estrategia_referencia",
            "estrategia_referencia_padronizada",
            "flag_estrategia_referencia_preenchida",
            "tipo_estrategia",
            "carteira_id",
            "replica_id",
            "nome_exibicao_estrategia",
            "categoria_estrategia",
            "subcategoria_estrategia",
            "origem_backtest",
            "ordem_exibicao",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        data_inicio_serie=("data_inicio_backtest", "first"),
        data_fim_serie=("data_fim_backtest", "first"),
        primeira_data_ativa_catalogo=("data", lambda s: pd.Series(s)[pd.Series(s).index].min()),
        ultima_data_ativa_catalogo=("data", lambda s: pd.Series(s)[pd.Series(s).index].max()),
        n_linhas_ativas_observadas=("flag_data_ativa_backtest", lambda s: int(pd.Series(s).fillna(False).sum())),
        n_linhas_com_patrimonio=("patrimonio_total", lambda s: int(pd.Series(s).notna().sum())),
        capital_inicial_estrategia=("capital_inicial_estrategia", "first"),
        patrimonio_final_ativo=("patrimonio_total", "last"),
        n_linhas_estrategia_referencia_ausente=("flag_estrategia_referencia_preenchida", lambda s: int((~pd.Series(s).fillna(False)).sum())),
    )
)

df_expectativas = (
    df_curvas_ativas
    .groupby("chave_estrategia", as_index=False)
    .agg(
        primeira_data_ativa_esperada=("data", "min"),
        ultima_data_ativa_esperada=("data", "max"),
        n_linhas_ativas_esperadas=("data", "count"),
    )
)

df_auditoria_consolidacao_curvas_patrimoniais = (
    df_auditoria_consolidacao_curvas_patrimoniais
    .merge(df_expectativas, on="chave_estrategia", how="left")
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia"])
    .reset_index(drop=True)
)

# Recalcular as primeiras/últimas datas ativas a partir das datas realmente observadas na curva-fonte.
df_primeira_ultima_ativas = (
    df_curvas_patrimoniais_consolidadas
    .loc[df_curvas_patrimoniais_consolidadas["flag_data_ativa_backtest"]]
    .groupby("chave_estrategia", as_index=False)
    .agg(
        primeira_data_ativa_catalogo=("data", "min"),
        ultima_data_ativa_catalogo=("data", "max"),
    )
)

df_auditoria_consolidacao_curvas_patrimoniais = (
    df_auditoria_consolidacao_curvas_patrimoniais
    .drop(columns=["primeira_data_ativa_catalogo", "ultima_data_ativa_catalogo"])
    .merge(df_primeira_ultima_ativas, on="chave_estrategia", how="left")
)

df_auditoria_consolidacao_curvas_patrimoniais["n_linhas_ativas_esperadas"] = pd.to_numeric(
    df_auditoria_consolidacao_curvas_patrimoniais["n_linhas_ativas_esperadas"],
    errors="coerce",
).fillna(0).astype(int)

df_auditoria_consolidacao_curvas_patrimoniais["flag_contagem_ativa_valida"] = (
    df_auditoria_consolidacao_curvas_patrimoniais["n_linhas_ativas_observadas"]
    == df_auditoria_consolidacao_curvas_patrimoniais["n_linhas_ativas_esperadas"]
)

df_auditoria_consolidacao_curvas_patrimoniais["flag_patrimonio_preenchido_nas_datas_ativas"] = (
    df_auditoria_consolidacao_curvas_patrimoniais["n_linhas_com_patrimonio"]
    == df_auditoria_consolidacao_curvas_patrimoniais["n_linhas_ativas_esperadas"]
)

df_auditoria_consolidacao_curvas_patrimoniais["flag_data_inicio_valida"] = pd.to_datetime(
    df_auditoria_consolidacao_curvas_patrimoniais["data_inicio_serie"],
    errors="coerce",
) == pd.to_datetime(
    df_auditoria_consolidacao_curvas_patrimoniais["primeira_data_ativa_esperada"],
    errors="coerce",
)

df_auditoria_consolidacao_curvas_patrimoniais["flag_data_fim_valida"] = pd.to_datetime(
    df_auditoria_consolidacao_curvas_patrimoniais["data_fim_serie"],
    errors="coerce",
) == pd.to_datetime(
    df_auditoria_consolidacao_curvas_patrimoniais["ultima_data_ativa_esperada"],
    errors="coerce",
)

df_auditoria_consolidacao_curvas_patrimoniais["flag_estrategia_referencia_preenchida"] = (
    pd.to_numeric(
        df_auditoria_consolidacao_curvas_patrimoniais["n_linhas_estrategia_referencia_ausente"],
        errors="coerce",
    ).fillna(0).astype(int)
    == 0
)

df_inconsistencias_consolidacao_curvas_patrimoniais = (
    df_auditoria_consolidacao_curvas_patrimoniais
    .loc[
        (~df_auditoria_consolidacao_curvas_patrimoniais["flag_contagem_ativa_valida"])
        | (~df_auditoria_consolidacao_curvas_patrimoniais["flag_patrimonio_preenchido_nas_datas_ativas"])
        | (~df_auditoria_consolidacao_curvas_patrimoniais["flag_data_inicio_valida"])
        | (~df_auditoria_consolidacao_curvas_patrimoniais["flag_data_fim_valida"])
        | (~df_auditoria_consolidacao_curvas_patrimoniais["flag_estrategia_referencia_preenchida"])
    ]
    .copy()
)

df_distribuicao_anual_curvas_patrimoniais = (
    df_curvas_ativas
    .assign(ano=lambda df: pd.to_datetime(df["data"], errors="coerce").dt.year)
    .groupby(
        [
            "chave_estrategia",
            "estrategia_id",
            "grupo_controle",
            "familia_estrategia",
            "estrategia_referencia",
            "estrategia_referencia_padronizada",
            "flag_estrategia_referencia_preenchida",
            "tipo_estrategia",
            "nome_exibicao_estrategia",
            "categoria_estrategia",
            "subcategoria_estrategia",
            "origem_backtest",
            "ordem_exibicao",
            "ano",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_datas_curva_patrimonial=("data", "count"),
        patrimonio_medio_ano=("patrimonio_total", "mean"),
        patrimonio_final_ano=("patrimonio_total", "last"),
        drawdown_minimo_ano=("drawdown_atual", "min"),
        retorno_ultimo_dia_ano=("retorno_acumulado", "last"),
    )
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "ano"])
    .reset_index(drop=True)
)

n_estrategia_referencia_ausente_auditoria = int(
    pd.to_numeric(
        df_auditoria_consolidacao_curvas_patrimoniais["n_linhas_estrategia_referencia_ausente"],
        errors="coerce",
    ).fillna(0).sum()
)

print(f"Inconsistências da consolidação            : {len(df_inconsistencias_consolidacao_curvas_patrimoniais):,}")
print(f"Estrategia_referencia ausente na auditoria : {n_estrategia_referencia_ausente_auditoria:,}")
print("OK")

# ============================================================
# 8) Salvamento dos outputs da subetapa
# ============================================================

print("\n[8/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_regras_consolidacao_curvas_patrimoniais, caminho_tbl_regras_consolidacao_curvas_patrimoniais, index=False)
salvar_dataframe(df_catalogo_curvas_patrimoniais, caminho_tbl_catalogo_curvas_patrimoniais, index=False)
salvar_dataframe(df_curvas_patrimoniais_consolidadas, caminho_base_curvas_patrimoniais_consolidadas, index=False)
salvar_dataframe(df_resumo_consolidacao_curvas_patrimoniais, caminho_tbl_resumo_consolidacao_curvas_patrimoniais, index=False)
salvar_dataframe(df_auditoria_consolidacao_curvas_patrimoniais, caminho_tbl_auditoria_consolidacao_curvas_patrimoniais, index=False)
salvar_dataframe(df_inconsistencias_consolidacao_curvas_patrimoniais, caminho_tbl_inconsistencias_consolidacao_curvas_patrimoniais, index=False)
salvar_dataframe(df_distribuicao_anual_curvas_patrimoniais, caminho_tbl_distribuicao_anual_curvas_patrimoniais, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 9) Preparação das amostras de validação
# ============================================================

print("\n[9/10] Preparação das amostras de validação...")

df_amostra_regras_consolidacao_curvas_patrimoniais = df_regras_consolidacao_curvas_patrimoniais.copy()
df_amostra_catalogo_curvas_patrimoniais = df_catalogo_curvas_patrimoniais.copy()
df_amostra_curvas_patrimoniais_consolidadas = (
    df_curvas_patrimoniais_consolidadas
    .groupby("chave_estrategia", as_index=False, dropna=False)
    .head(3)
    .copy()
)
df_amostra_resumo_consolidacao_curvas_patrimoniais = df_resumo_consolidacao_curvas_patrimoniais.copy()
df_amostra_auditoria_consolidacao_curvas_patrimoniais = df_auditoria_consolidacao_curvas_patrimoniais.copy()
df_amostra_inconsistencias_consolidacao_curvas_patrimoniais = df_inconsistencias_consolidacao_curvas_patrimoniais.head(20).copy()
df_amostra_distribuicao_anual_curvas_patrimoniais = df_distribuicao_anual_curvas_patrimoniais.head(40).copy()

print("Amostras de validação preparadas com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nRegras formais da consolidação das curvas patrimoniais:")
print(df_amostra_regras_consolidacao_curvas_patrimoniais.to_string(index=False))

print("\nCatálogo consolidado das curvas patrimoniais:")
print(df_amostra_catalogo_curvas_patrimoniais.to_string(index=False))

print("\nBase mestra consolidada das curvas patrimoniais - amostra:")
print(df_amostra_curvas_patrimoniais_consolidadas.to_string(index=False))

print("\nResumo da consolidação das curvas patrimoniais:")
print(df_amostra_resumo_consolidacao_curvas_patrimoniais.to_string(index=False))

print("\nAuditoria da consolidação das curvas patrimoniais:")
print(df_amostra_auditoria_consolidacao_curvas_patrimoniais.to_string(index=False))

print("\nInconsistências da consolidação das curvas patrimoniais - amostra:")
print(df_amostra_inconsistencias_consolidacao_curvas_patrimoniais.to_string(index=False))

print("\nDistribuição anual consolidada das curvas patrimoniais - amostra:")
print(df_amostra_distribuicao_anual_curvas_patrimoniais.to_string(index=False))

print("\nArquivos salvos na subetapa 9.7:")
print(f"- {caminho_tbl_regras_consolidacao_curvas_patrimoniais}")
print(f"- {caminho_tbl_catalogo_curvas_patrimoniais}")
print(f"- {caminho_base_curvas_patrimoniais_consolidadas}")
print(f"- {caminho_tbl_resumo_consolidacao_curvas_patrimoniais}")
print(f"- {caminho_tbl_auditoria_consolidacao_curvas_patrimoniais}")
print(f"- {caminho_tbl_inconsistencias_consolidacao_curvas_patrimoniais}")
print(f"- {caminho_tbl_distribuicao_anual_curvas_patrimoniais}")

print("\nETAPA 9.7 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 9.7 - CONSOLIDAÇÃO DAS CURVAS PATRIMONIAIS

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - patrimônio capitulação                : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_1_base_patrimonio_capitulacao.parquet
Entrada - patrimônio euforia                    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_2_base_patrimonio_euforia.parquet
Entrada - patrimônio aleatórias controle        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_3_base_patrimonio_aleatorias_controle.parquet
Entrada - patrimônio mensais controle           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resulta

# Etapa 10) Registro e Auditoria das Carteiras

## Etapa 10.1) Registro dos Aportes Realizados

In [47]:
%%time
# ============================================================
# Etapa 10.1) Registro dos Aportes Realizados
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 10.1 - REGISTRO DOS APORTES REALIZADOS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_caixa_residual_aporte = gerar_caminho_arquivo(
    etapa=8,
    subetapa=4,
    tipo_arquivo="base",
    nome="caixa_residual_aporte",
)

caminho_base_evento_inicial_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="base",
    nome="evento_inicial_benchmark_ibovespa",
)

caminho_base_evento_inicial_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="base",
    nome="evento_inicial_cdi_only",
)

caminho_tbl_catalogo_curvas_patrimoniais = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="tbl",
    nome="catalogo_curvas_patrimoniais",
)

caminho_tbl_resumo_consolidacao_curvas_patrimoniais = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="tbl",
    nome="resumo_consolidacao_curvas_patrimoniais",
)

caminho_tbl_regras_registro_aportes_realizados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="regras_registro_aportes_realizados",
)

caminho_base_aportes_realizados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="base",
    nome="aportes_realizados",
)

caminho_tbl_resumo_aportes_realizados_estrategia = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="resumo_aportes_realizados_estrategia",
)

caminho_tbl_distribuicao_anual_aportes_realizados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_aportes_realizados",
)

caminho_tbl_auditoria_aportes_realizados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="auditoria_aportes_realizados",
)

caminho_tbl_inconsistencias_aportes_realizados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="inconsistencias_aportes_realizados",
)

print(f"Entrada - eventos de caixa por aporte              : {caminho_base_caixa_residual_aporte}")
print(f"Entrada - evento inicial Ibovespa                  : {caminho_base_evento_inicial_benchmark_ibovespa}")
print(f"Entrada - evento inicial CDI-only                  : {caminho_base_evento_inicial_cdi_only}")
print(f"Entrada - catálogo de curvas patrimoniais          : {caminho_tbl_catalogo_curvas_patrimoniais}")
print(f"Entrada - resumo da consolidação patrimonial       : {caminho_tbl_resumo_consolidacao_curvas_patrimoniais}")
print(f"Saída   - regras do registro de aportes            : {caminho_tbl_regras_registro_aportes_realizados}")
print(f"Saída   - base de aportes realizados               : {caminho_base_aportes_realizados}")
print(f"Saída   - resumo por estratégia                    : {caminho_tbl_resumo_aportes_realizados_estrategia}")
print(f"Saída   - distribuição anual                       : {caminho_tbl_distribuicao_anual_aportes_realizados}")
print(f"Saída   - auditoria                                : {caminho_tbl_auditoria_aportes_realizados}")
print(f"Saída   - inconsistências                          : {caminho_tbl_inconsistencias_aportes_realizados}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_caixa_residual_aporte = pd.read_parquet(caminho_base_caixa_residual_aporte)
df_evento_inicial_benchmark_ibovespa = pd.read_parquet(caminho_base_evento_inicial_benchmark_ibovespa)
df_evento_inicial_cdi_only = pd.read_parquet(caminho_base_evento_inicial_cdi_only)
df_catalogo_curvas_patrimoniais = pd.read_parquet(caminho_tbl_catalogo_curvas_patrimoniais)
df_resumo_consolidacao_curvas_patrimoniais = pd.read_parquet(caminho_tbl_resumo_consolidacao_curvas_patrimoniais)

print(f"Eventos de caixa por aporte              : {df_caixa_residual_aporte.shape[0]:,} linhas x {df_caixa_residual_aporte.shape[1]} colunas")
print(f"Evento inicial Ibovespa                  : {df_evento_inicial_benchmark_ibovespa.shape[0]:,} linhas x {df_evento_inicial_benchmark_ibovespa.shape[1]} colunas")
print(f"Evento inicial CDI-only                  : {df_evento_inicial_cdi_only.shape[0]:,} linhas x {df_evento_inicial_cdi_only.shape[1]} colunas")
print(f"Catálogo de curvas patrimoniais          : {df_catalogo_curvas_patrimoniais.shape[0]:,} linhas x {df_catalogo_curvas_patrimoniais.shape[1]} colunas")
print(f"Resumo da consolidação patrimonial       : {df_resumo_consolidacao_curvas_patrimoniais.shape[0]:,} linhas x {df_resumo_consolidacao_curvas_patrimoniais.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_VALOR_APORTE = 0.01

COLUNAS_BASE_APORTES_REALIZADOS = [
    "chave_evento_aporte",
    "request_id",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_registro_aporte",
    "origem_calendario",
    "origem_backtest",
    "classe_evento_aporte",
    "data_sinal",
    "data_aporte_efetiva",
    "ano",
    "mes",
    "ordem_aporte_estrategia",
    "ordem_evento_aporte_global",
    "ordem_exibicao",
    "capital_inicial_estrategia",
    "valor_aporte_estrategia",
    "valor_investido_efetivo_aporte",
    "valor_caixa_residual_aporte",
    "valor_caixa_trazido_aporte",
    "saldo_caixa_antes_aporte",
    "saldo_caixa_apos_investimento_aporte",
    "saldo_caixa_final_periodo",
    "rendimento_cdi_periodo",
    "pct_orcamento_investido",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
    "ticker_referencia",
    "fonte_benchmark",
    "preco_execucao_referencia",
    "quantidade_referencia",
    "flag_evento_aporte",
    "flag_aporte_realizado",
    "flag_valor_aporte_positivo",
    "flag_data_aporte_valida",
    "flag_request_id_preenchido",
    "flag_chave_estrategia_preenchida",
    "flag_estrategia_referencia_preenchida",
    "flag_data_aporte_dentro_backtest",
    "flag_investimento_coberto_por_caixa",
    "flag_conservacao_orcamento",
    "flag_compra_minima_existe_no_aporte",
    "flag_benchmark_equivalente",
]

COLUNAS_INCONSISTENCIAS = [
    "tipo_inconsistencia",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "grupo_controle",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "valor_observado",
    "valor_referencia",
    "detalhe",
]

COLUNAS_CATALOGO_COMPLEMENTARES = [
    "estrategia_id",
    "chave_estrategia",
    "estrategia_referencia_padronizada",
    "flag_estrategia_referencia_preenchida",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "origem_backtest",
    "ordem_exibicao",
    "data_inicio_backtest",
    "data_fim_backtest",
]

COLUNAS_CHAVE_ESTRATEGIA = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_backtest",
    "ordem_exibicao",
]

def selecionar_colunas_com_padrao(df, colunas):
    df_saida = df.copy()
    for coluna in colunas:
        if coluna not in df_saida.columns:
            df_saida[coluna] = pd.NA
    return df_saida[colunas].copy()

def preencher_coluna_catalogo(df, coluna):
    coluna_catalogo = f"{coluna}_catalogo"
    if coluna in df.columns and coluna_catalogo in df.columns:
        df[coluna] = df[coluna].where(df[coluna].notna(), df[coluna_catalogo])
        df = df.drop(columns=[coluna_catalogo])
    elif coluna_catalogo in df.columns:
        df = df.rename(columns={coluna_catalogo: coluna})
    return df

def normalizar_texto(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()
    return df

def converter_numerico(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    return df

def converter_booleano(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = df[coluna].fillna(False).astype(bool)
    return df

def registrar_inconsistencia(lista, linha, tipo_inconsistencia, valor_observado, valor_referencia, detalhe):
    lista.append({
        "tipo_inconsistencia": tipo_inconsistencia,
        "chave_estrategia": linha.get("chave_estrategia"),
        "estrategia_id": linha.get("estrategia_id"),
        "nome_exibicao_estrategia": linha.get("nome_exibicao_estrategia"),
        "grupo_controle": linha.get("grupo_controle"),
        "categoria_estrategia": linha.get("categoria_estrategia"),
        "subcategoria_estrategia": linha.get("subcategoria_estrategia"),
        "valor_observado": valor_observado,
        "valor_referencia": valor_referencia,
        "detalhe": detalhe,
    })

print(f"Tolerância monetária da subetapa              : {TOLERANCIA_VALOR_APORTE}")
print("OK")

# ============================================================
# 5) Preparação dos aportes das carteiras com caixa completo
# ============================================================

print("\n[5/10] Preparação dos aportes das carteiras com caixa completo...")

colunas_aportes_carteiras = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "tipo_estrategia",
    "origem_calendario",
    "carteira_id",
    "replica_id",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "ano",
    "mes",
    "capital_inicial_estrategia",
    "valor_aporte_estrategia",
    "valor_total_investido",
    "caixa_residual_final_aporte",
    "valor_caixa_trazido_aporte",
    "saldo_caixa_antes_aporte",
    "saldo_caixa_apos_investimento_aporte",
    "saldo_caixa_final_periodo",
    "rendimento_cdi_periodo",
    "pct_orcamento_investido",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
    "flag_evento_aporte",
    "flag_investimento_coberto_por_caixa",
    "flag_conservacao_orcamento",
    "flag_compra_minima_existe_no_aporte",
    "flag_benchmark_equivalente",
]

df_aportes_carteiras = selecionar_colunas_com_padrao(df_caixa_residual_aporte, colunas_aportes_carteiras)

df_aportes_carteiras = df_aportes_carteiras.rename(columns={
    "valor_total_investido": "valor_investido_efetivo_aporte",
    "caixa_residual_final_aporte": "valor_caixa_residual_aporte",
})

df_aportes_carteiras["origem_registro_aporte"] = "etapa_8_4_caixa_completo"
df_aportes_carteiras["classe_evento_aporte"] = "aporte_carteira"
df_aportes_carteiras["ticker_referencia"] = pd.NA
df_aportes_carteiras["fonte_benchmark"] = pd.NA
df_aportes_carteiras["preco_execucao_referencia"] = np.nan
df_aportes_carteiras["quantidade_referencia"] = np.nan

print(f"Aportes de carteiras preparados             : {df_aportes_carteiras.shape[0]:,}")
print("OK")

# ============================================================
# 6) Preparação dos eventos iniciais dos benchmarks
# ============================================================

print("\n[6/10] Preparação dos eventos iniciais dos benchmarks...")

df_ibovespa_evento = df_evento_inicial_benchmark_ibovespa.copy()
df_ibovespa_evento["request_id"] = "ibovespa_buy_and_hold__1"
df_ibovespa_evento["origem_calendario"] = "9_5_evento_inicial_benchmark_ibovespa"
df_ibovespa_evento["origem_registro_aporte"] = "etapa_9_5_evento_inicial_benchmark_ibovespa"
df_ibovespa_evento["classe_evento_aporte"] = "aporte_benchmark"
df_ibovespa_evento["data_sinal"] = df_ibovespa_evento["data_execucao_inicial"]
df_ibovespa_evento["data_aporte_efetiva"] = df_ibovespa_evento["data_execucao_inicial"]
df_ibovespa_evento["ordem_aporte_estrategia"] = 1
df_ibovespa_evento["valor_aporte_estrategia"] = df_ibovespa_evento["capital_inicial_estrategia"]
df_ibovespa_evento["valor_investido_efetivo_aporte"] = df_ibovespa_evento["valor_investido_inicial"]
df_ibovespa_evento["valor_caixa_residual_aporte"] = 0.0
df_ibovespa_evento["valor_caixa_trazido_aporte"] = 0.0
df_ibovespa_evento["saldo_caixa_antes_aporte"] = df_ibovespa_evento["capital_inicial_estrategia"]
df_ibovespa_evento["saldo_caixa_apos_investimento_aporte"] = 0.0
df_ibovespa_evento["saldo_caixa_final_periodo"] = 0.0
df_ibovespa_evento["rendimento_cdi_periodo"] = 0.0
df_ibovespa_evento["pct_orcamento_investido"] = 1.0
df_ibovespa_evento["n_tickers_cesta_equal_weight"] = 1
df_ibovespa_evento["n_tickers_compra_efetivada"] = 1
df_ibovespa_evento["n_tickers_quantidade_zero"] = 0
df_ibovespa_evento["ticker_referencia"] = df_ibovespa_evento["ticker_benchmark"]
df_ibovespa_evento["preco_execucao_referencia"] = df_ibovespa_evento["preco_execucao_inicial"]
df_ibovespa_evento["quantidade_referencia"] = df_ibovespa_evento["quantidade_equivalente_benchmark"]
df_ibovespa_evento["flag_evento_aporte"] = True
df_ibovespa_evento["flag_investimento_coberto_por_caixa"] = True
df_ibovespa_evento["flag_conservacao_orcamento"] = True
df_ibovespa_evento["flag_compra_minima_existe_no_aporte"] = True
df_ibovespa_evento["flag_benchmark_equivalente"] = True

df_cdi_evento = df_evento_inicial_cdi_only.copy()
df_cdi_evento["request_id"] = "cdi_only__1"
df_cdi_evento["origem_calendario"] = "9_6_evento_inicial_cdi_only"
df_cdi_evento["origem_registro_aporte"] = "etapa_9_6_evento_inicial_cdi_only"
df_cdi_evento["classe_evento_aporte"] = "aporte_benchmark"
df_cdi_evento["data_sinal"] = df_cdi_evento["data_execucao_inicial"]
df_cdi_evento["data_aporte_efetiva"] = df_cdi_evento["data_execucao_inicial"]
df_cdi_evento["ordem_aporte_estrategia"] = 1
df_cdi_evento["valor_aporte_estrategia"] = df_cdi_evento["capital_inicial_estrategia"]
df_cdi_evento["valor_investido_efetivo_aporte"] = df_cdi_evento["capital_inicial_estrategia"]
df_cdi_evento["valor_caixa_residual_aporte"] = 0.0
df_cdi_evento["valor_caixa_trazido_aporte"] = 0.0
df_cdi_evento["saldo_caixa_antes_aporte"] = df_cdi_evento["capital_inicial_estrategia"]
df_cdi_evento["saldo_caixa_apos_investimento_aporte"] = 0.0
df_cdi_evento["saldo_caixa_final_periodo"] = 0.0
df_cdi_evento["rendimento_cdi_periodo"] = 0.0
df_cdi_evento["pct_orcamento_investido"] = 1.0
df_cdi_evento["n_tickers_cesta_equal_weight"] = 1
df_cdi_evento["n_tickers_compra_efetivada"] = 1
df_cdi_evento["n_tickers_quantidade_zero"] = 0
df_cdi_evento["ticker_referencia"] = df_cdi_evento["ticker_referencia"]
df_cdi_evento["fonte_benchmark"] = "cdi_padronizada"
df_cdi_evento["preco_execucao_referencia"] = np.nan
df_cdi_evento["quantidade_referencia"] = np.nan
df_cdi_evento["flag_evento_aporte"] = True
df_cdi_evento["flag_investimento_coberto_por_caixa"] = True
df_cdi_evento["flag_conservacao_orcamento"] = True
df_cdi_evento["flag_compra_minima_existe_no_aporte"] = True
df_cdi_evento["flag_benchmark_equivalente"] = True

colunas_evento_benchmark = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "tipo_estrategia",
    "origem_calendario",
    "carteira_id",
    "replica_id",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "capital_inicial_estrategia",
    "valor_aporte_estrategia",
    "valor_investido_efetivo_aporte",
    "valor_caixa_residual_aporte",
    "valor_caixa_trazido_aporte",
    "saldo_caixa_antes_aporte",
    "saldo_caixa_apos_investimento_aporte",
    "saldo_caixa_final_periodo",
    "rendimento_cdi_periodo",
    "pct_orcamento_investido",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
    "ticker_referencia",
    "fonte_benchmark",
    "preco_execucao_referencia",
    "quantidade_referencia",
    "flag_evento_aporte",
    "flag_investimento_coberto_por_caixa",
    "flag_conservacao_orcamento",
    "flag_compra_minima_existe_no_aporte",
    "flag_benchmark_equivalente",
    "origem_registro_aporte",
    "classe_evento_aporte",
]

df_benchmarks_eventos = pd.concat(
    [
        selecionar_colunas_com_padrao(df_ibovespa_evento, colunas_evento_benchmark),
        selecionar_colunas_com_padrao(df_cdi_evento, colunas_evento_benchmark),
    ],
    ignore_index=True,
)

df_benchmarks_eventos["ano"] = pd.to_datetime(df_benchmarks_eventos["data_aporte_efetiva"]).dt.year
df_benchmarks_eventos["mes"] = pd.to_datetime(df_benchmarks_eventos["data_aporte_efetiva"]).dt.month

print(f"Eventos iniciais de benchmarks preparados    : {df_benchmarks_eventos.shape[0]:,}")
print("OK")

# ============================================================
# 7) Consolidação da base mestra de aportes realizados
# ============================================================

print("\n[7/10] Consolidação da base mestra de aportes realizados...")

df_aportes_realizados = pd.concat(
    [
        df_aportes_carteiras,
        df_benchmarks_eventos,
    ],
    ignore_index=True,
)

df_catalogo_complementar = selecionar_colunas_com_padrao(
    df_catalogo_curvas_patrimoniais,
    COLUNAS_CATALOGO_COMPLEMENTARES,
)

df_aportes_realizados = df_aportes_realizados.merge(
    df_catalogo_complementar,
    on="estrategia_id",
    how="left",
    suffixes=("", "_catalogo"),
)

for coluna in COLUNAS_CATALOGO_COMPLEMENTARES:
    if coluna != "estrategia_id":
        df_aportes_realizados = preencher_coluna_catalogo(df_aportes_realizados, coluna)

colunas_texto = [
    "request_id",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "origem_registro_aporte",
    "origem_calendario",
    "origem_backtest",
    "classe_evento_aporte",
    "ticker_referencia",
    "fonte_benchmark",
]

df_aportes_realizados = normalizar_texto(df_aportes_realizados, colunas_texto)

df_aportes_realizados["data_sinal"] = pd.to_datetime(df_aportes_realizados["data_sinal"], errors="coerce")
df_aportes_realizados["data_aporte_efetiva"] = pd.to_datetime(df_aportes_realizados["data_aporte_efetiva"], errors="coerce")
df_aportes_realizados["data_inicio_backtest"] = pd.to_datetime(df_aportes_realizados["data_inicio_backtest"], errors="coerce")
df_aportes_realizados["data_fim_backtest"] = pd.to_datetime(df_aportes_realizados["data_fim_backtest"], errors="coerce")

df_aportes_realizados["ano"] = pd.to_numeric(df_aportes_realizados["ano"], errors="coerce")
df_aportes_realizados.loc[df_aportes_realizados["ano"].isna(), "ano"] = df_aportes_realizados.loc[df_aportes_realizados["ano"].isna(), "data_aporte_efetiva"].dt.year
df_aportes_realizados["mes"] = pd.to_numeric(df_aportes_realizados["mes"], errors="coerce")
df_aportes_realizados.loc[df_aportes_realizados["mes"].isna(), "mes"] = df_aportes_realizados.loc[df_aportes_realizados["mes"].isna(), "data_aporte_efetiva"].dt.month

df_aportes_realizados = converter_numerico(
    df_aportes_realizados,
    [
        "carteira_id",
        "replica_id",
        "ordem_aporte_estrategia",
        "ordem_exibicao",
        "capital_inicial_estrategia",
        "valor_aporte_estrategia",
        "valor_investido_efetivo_aporte",
        "valor_caixa_residual_aporte",
        "valor_caixa_trazido_aporte",
        "saldo_caixa_antes_aporte",
        "saldo_caixa_apos_investimento_aporte",
        "saldo_caixa_final_periodo",
        "rendimento_cdi_periodo",
        "pct_orcamento_investido",
        "n_tickers_cesta_equal_weight",
        "n_tickers_compra_efetivada",
        "n_tickers_quantidade_zero",
        "preco_execucao_referencia",
        "quantidade_referencia",
        "ano",
        "mes",
    ],
)

df_aportes_realizados = converter_booleano(
    df_aportes_realizados,
    [
        "flag_evento_aporte",
        "flag_estrategia_referencia_preenchida",
        "flag_investimento_coberto_por_caixa",
        "flag_conservacao_orcamento",
        "flag_compra_minima_existe_no_aporte",
        "flag_benchmark_equivalente",
    ],
)

df_aportes_realizados["flag_aporte_realizado"] = True
df_aportes_realizados["flag_valor_aporte_positivo"] = df_aportes_realizados["valor_aporte_estrategia"].fillna(0.0) > 0.0
df_aportes_realizados["flag_data_aporte_valida"] = df_aportes_realizados["data_aporte_efetiva"].notna()
df_aportes_realizados["flag_request_id_preenchido"] = df_aportes_realizados["request_id"].notna() & (df_aportes_realizados["request_id"].astype("string").str.len() > 0)
df_aportes_realizados["flag_chave_estrategia_preenchida"] = df_aportes_realizados["chave_estrategia"].notna() & (df_aportes_realizados["chave_estrategia"].astype("string").str.len() > 0)
df_aportes_realizados["flag_estrategia_referencia_preenchida"] = df_aportes_realizados["estrategia_referencia"].notna() & (df_aportes_realizados["estrategia_referencia"].astype("string").str.len() > 0)
df_aportes_realizados["flag_data_aporte_dentro_backtest"] = (
    df_aportes_realizados["data_aporte_efetiva"].notna()
    & df_aportes_realizados["data_inicio_backtest"].notna()
    & df_aportes_realizados["data_fim_backtest"].notna()
    & (df_aportes_realizados["data_aporte_efetiva"] >= df_aportes_realizados["data_inicio_backtest"])
    & (df_aportes_realizados["data_aporte_efetiva"] <= df_aportes_realizados["data_fim_backtest"])
)

df_aportes_realizados = df_aportes_realizados.sort_values(
    [
        "data_aporte_efetiva",
        "ordem_exibicao",
        "estrategia_id",
        "ordem_aporte_estrategia",
        "request_id",
    ],
    na_position="last",
).reset_index(drop=True)

df_aportes_realizados["ordem_evento_aporte_global"] = np.arange(1, len(df_aportes_realizados) + 1)

df_aportes_realizados["chave_evento_aporte"] = (
    df_aportes_realizados["chave_estrategia"].astype("string")
    + "__"
    + df_aportes_realizados["request_id"].astype("string")
)

df_aportes_realizados = selecionar_colunas_com_padrao(
    df_aportes_realizados,
    COLUNAS_BASE_APORTES_REALIZADOS,
)

print(f"Base mestra de aportes realizados          : {df_aportes_realizados.shape[0]:,} linhas x {df_aportes_realizados.shape[1]} colunas")
print(f"Estratégias com aportes registrados        : {df_aportes_realizados['estrategia_id'].nunique():,}")
print(f"Aportes de carteiras                       : {(df_aportes_realizados['classe_evento_aporte'] == 'aporte_carteira').sum():,}")
print(f"Aportes de benchmarks                      : {(df_aportes_realizados['classe_evento_aporte'] == 'aporte_benchmark').sum():,}")
print("OK")

# ============================================================
# 8) Construção das regras formais, resumos e distribuição anual
# ============================================================

print("\n[8/10] Construção das regras formais, resumos e distribuição anual...")

df_regras_registro_aportes_realizados = pd.DataFrame([
    {
        "ordem_regra": 1,
        "regra_id": "RAR1",
        "escopo": "fonte",
        "regra_operacional": "usar_eventos_de_caixa_completo_para_as_estrategias_de_carteira",
        "detalhe": "Os aportes das estratégias reais, aleatórias e mensais são consolidados a partir da base de caixa completo da etapa 8.4.",
    },
    {
        "ordem_regra": 2,
        "regra_id": "RAR2",
        "escopo": "benchmarks",
        "regra_operacional": "incluir_eventos_iniciais_dos_benchmarks_como_aportes_formais",
        "detalhe": "Ibovespa buy-and-hold e CDI-only entram com um evento inicial de alocação do capital para manter a auditoria comparável entre todas as curvas.",
    },
    {
        "ordem_regra": 3,
        "regra_id": "RAR3",
        "escopo": "nomenclatura",
        "regra_operacional": "usar_o_catalogo_da_etapa_9_7_como_fonte_de_classificacao",
        "detalhe": "Chave da estratégia, nome de exibição, categoria, subcategoria e ordem de exibição são herdados do catálogo consolidado das curvas patrimoniais.",
    },
    {
        "ordem_regra": 4,
        "regra_id": "RAR4",
        "escopo": "auditoria",
        "regra_operacional": "comparar_a_quantidade_de_aportes_com_os_eventos_de_aporte_das_curvas",
        "detalhe": "A auditoria exige que o número de aportes registrados por estratégia seja igual ao número de eventos de aporte ativos na consolidação patrimonial da etapa 9.7.",
    },
    {
        "ordem_regra": 5,
        "regra_id": "RAR5",
        "escopo": "capital",
        "regra_operacional": "validar_se_o_total_aportado_por_estrategia_reconcilia_com_o_capital_inicial",
        "detalhe": "A soma dos valores de aporte por estratégia deve reconciliar com o capital inicial, respeitando apenas tolerância monetária residual.",
    },
])

agregacoes_resumo = {
    "request_id": "nunique",
    "data_sinal": ["min", "max"],
    "data_aporte_efetiva": ["min", "max"],
    "valor_aporte_estrategia": ["sum", "mean", "min", "max"],
    "valor_investido_efetivo_aporte": "sum",
    "valor_caixa_residual_aporte": "sum",
    "valor_caixa_trazido_aporte": "max",
    "rendimento_cdi_periodo": "sum",
    "n_tickers_compra_efetivada": "sum",
    "n_tickers_quantidade_zero": "sum",
    "flag_data_aporte_valida": "sum",
    "flag_valor_aporte_positivo": "sum",
    "flag_data_aporte_dentro_backtest": "sum",
}

df_resumo_aportes_realizados_estrategia = (
    df_aportes_realizados
    .groupby(COLUNAS_CHAVE_ESTRATEGIA, dropna=False)
    .agg(agregacoes_resumo)
    .reset_index()
)

df_resumo_aportes_realizados_estrategia.columns = [
    "_".join([str(parte) for parte in coluna if str(parte) != ""]).strip("_")
    if isinstance(coluna, tuple) else coluna
    for coluna in df_resumo_aportes_realizados_estrategia.columns
]

df_resumo_aportes_realizados_estrategia = df_resumo_aportes_realizados_estrategia.rename(columns={
    "request_id_nunique": "n_aportes_realizados",
    "data_sinal_min": "primeira_data_sinal",
    "data_sinal_max": "ultima_data_sinal",
    "data_aporte_efetiva_min": "primeira_data_aporte_efetiva",
    "data_aporte_efetiva_max": "ultima_data_aporte_efetiva",
    "valor_aporte_estrategia_sum": "valor_total_aportado_registrado",
    "valor_aporte_estrategia_mean": "valor_medio_aporte_registrado",
    "valor_aporte_estrategia_min": "valor_minimo_aporte_registrado",
    "valor_aporte_estrategia_max": "valor_maximo_aporte_registrado",
    "valor_investido_efetivo_aporte_sum": "valor_total_investido_efetivo",
    "valor_caixa_residual_aporte_sum": "valor_caixa_residual_total_aportes",
    "valor_caixa_trazido_aporte_max": "maior_valor_caixa_trazido_aporte",
    "rendimento_cdi_periodo_sum": "rendimento_cdi_periodos_total",
    "n_tickers_compra_efetivada_sum": "n_compras_efetivadas_total",
    "n_tickers_quantidade_zero_sum": "n_linhas_quantidade_zero_total",
    "flag_data_aporte_valida_sum": "n_aportes_com_data_valida",
    "flag_valor_aporte_positivo_sum": "n_aportes_com_valor_positivo",
    "flag_data_aporte_dentro_backtest_sum": "n_aportes_dentro_backtest",
})

capital_por_estrategia = (
    df_aportes_realizados
    .groupby("estrategia_id", dropna=False)["capital_inicial_estrategia"]
    .max()
    .reset_index()
    .rename(columns={"capital_inicial_estrategia": "capital_inicial_estrategia_resumo"})
)

df_resumo_aportes_realizados_estrategia = df_resumo_aportes_realizados_estrategia.merge(
    capital_por_estrategia,
    on="estrategia_id",
    how="left",
)

df_resumo_aportes_realizados_estrategia["pct_capital_total_aportado"] = (
    df_resumo_aportes_realizados_estrategia["valor_total_aportado_registrado"]
    / df_resumo_aportes_realizados_estrategia["capital_inicial_estrategia_resumo"]
)

df_resumo_aportes_realizados_estrategia["pct_capital_total_investido_efetivo"] = (
    df_resumo_aportes_realizados_estrategia["valor_total_investido_efetivo"]
    / df_resumo_aportes_realizados_estrategia["capital_inicial_estrategia_resumo"]
)

df_distribuicao_anual_aportes_realizados = (
    df_aportes_realizados
    .groupby(COLUNAS_CHAVE_ESTRATEGIA + ["ano"], dropna=False)
    .agg(
        n_aportes_ano=("request_id", "nunique"),
        primeira_data_aporte_ano=("data_aporte_efetiva", "min"),
        ultima_data_aporte_ano=("data_aporte_efetiva", "max"),
        valor_total_aportado_ano=("valor_aporte_estrategia", "sum"),
        valor_total_investido_ano=("valor_investido_efetivo_aporte", "sum"),
        valor_medio_aporte_ano=("valor_aporte_estrategia", "mean"),
        n_compras_efetivadas_ano=("n_tickers_compra_efetivada", "sum"),
    )
    .reset_index()
    .sort_values(["ordem_exibicao", "estrategia_id", "ano"])
    .reset_index(drop=True)
)

print(f"Regras formais geradas                       : {df_regras_registro_aportes_realizados.shape[0]:,}")
print(f"Resumo por estratégia gerado                : {df_resumo_aportes_realizados_estrategia.shape[0]:,}")
print(f"Distribuição anual gerada                   : {df_distribuicao_anual_aportes_realizados.shape[0]:,}")
print("OK")

# ============================================================
# 9) Construção da auditoria e das inconsistências
# ============================================================

print("\n[9/10] Construção da auditoria e das inconsistências...")

metricas_integridade = (
    df_aportes_realizados
    .assign(
        flag_request_id_ausente=~df_aportes_realizados["flag_request_id_preenchido"],
        flag_chave_estrategia_ausente=~df_aportes_realizados["flag_chave_estrategia_preenchida"],
        flag_estrategia_referencia_ausente=~df_aportes_realizados["flag_estrategia_referencia_preenchida"],
        flag_data_aporte_ausente=~df_aportes_realizados["flag_data_aporte_valida"],
        flag_valor_aporte_nao_positivo=~df_aportes_realizados["flag_valor_aporte_positivo"],
        flag_valor_investido_negativo=df_aportes_realizados["valor_investido_efetivo_aporte"].fillna(0.0) < 0.0,
        flag_data_aporte_fora_intervalo=~df_aportes_realizados["flag_data_aporte_dentro_backtest"],
    )
    .groupby("estrategia_id", dropna=False)
    .agg(
        n_request_id_ausente=("flag_request_id_ausente", "sum"),
        n_chave_estrategia_ausente=("flag_chave_estrategia_ausente", "sum"),
        n_estrategia_referencia_ausente=("flag_estrategia_referencia_ausente", "sum"),
        n_data_aporte_ausente=("flag_data_aporte_ausente", "sum"),
        n_valor_aporte_nao_positivo=("flag_valor_aporte_nao_positivo", "sum"),
        n_valor_investido_negativo=("flag_valor_investido_negativo", "sum"),
        n_data_aporte_fora_intervalo=("flag_data_aporte_fora_intervalo", "sum"),
    )
    .reset_index()
)

df_request_ids_duplicados = (
    df_aportes_realizados[df_aportes_realizados.duplicated(["estrategia_id", "request_id"], keep=False)]
    .groupby("estrategia_id", dropna=False)["request_id"]
    .nunique()
    .reset_index()
    .rename(columns={"request_id": "n_request_id_duplicado"})
)

df_auditoria_aportes_realizados = df_resumo_aportes_realizados_estrategia.merge(
    metricas_integridade,
    on="estrategia_id",
    how="left",
)

df_auditoria_aportes_realizados = df_auditoria_aportes_realizados.merge(
    df_request_ids_duplicados,
    on="estrategia_id",
    how="left",
)

df_auditoria_aportes_realizados["n_request_id_duplicado"] = df_auditoria_aportes_realizados["n_request_id_duplicado"].fillna(0).astype(int)

colunas_resumo_curvas = [
    "estrategia_id",
    "n_eventos_aporte_ativos",
    "capital_inicial_estrategia",
]

df_resumo_curvas_para_auditoria = selecionar_colunas_com_padrao(
    df_resumo_consolidacao_curvas_patrimoniais,
    colunas_resumo_curvas,
).rename(columns={
    "capital_inicial_estrategia": "capital_inicial_estrategia_curva",
})

df_auditoria_aportes_realizados = df_auditoria_aportes_realizados.merge(
    df_resumo_curvas_para_auditoria,
    on="estrategia_id",
    how="left",
)

df_auditoria_aportes_realizados["desvio_n_aportes_vs_curva"] = (
    df_auditoria_aportes_realizados["n_aportes_realizados"]
    - df_auditoria_aportes_realizados["n_eventos_aporte_ativos"]
)

df_auditoria_aportes_realizados["desvio_valor_aportes_vs_capital"] = (
    df_auditoria_aportes_realizados["valor_total_aportado_registrado"]
    - df_auditoria_aportes_realizados["capital_inicial_estrategia_curva"]
)

df_auditoria_aportes_realizados["flag_contagem_aportes_coerente_com_curva"] = df_auditoria_aportes_realizados["desvio_n_aportes_vs_curva"].fillna(np.inf).abs() == 0

df_auditoria_aportes_realizados["flag_valor_total_aportes_coerente_com_capital"] = df_auditoria_aportes_realizados["desvio_valor_aportes_vs_capital"].fillna(np.inf).abs() <= TOLERANCIA_VALOR_APORTE

df_auditoria_aportes_realizados["flag_sem_inconsistencia_registro_aporte"] = (
    df_auditoria_aportes_realizados["flag_contagem_aportes_coerente_com_curva"]
    & df_auditoria_aportes_realizados["flag_valor_total_aportes_coerente_com_capital"]
    & (df_auditoria_aportes_realizados["n_request_id_ausente"].fillna(0) == 0)
    & (df_auditoria_aportes_realizados["n_request_id_duplicado"].fillna(0) == 0)
    & (df_auditoria_aportes_realizados["n_chave_estrategia_ausente"].fillna(0) == 0)
    & (df_auditoria_aportes_realizados["n_estrategia_referencia_ausente"].fillna(0) == 0)
    & (df_auditoria_aportes_realizados["n_data_aporte_ausente"].fillna(0) == 0)
    & (df_auditoria_aportes_realizados["n_valor_aporte_nao_positivo"].fillna(0) == 0)
    & (df_auditoria_aportes_realizados["n_valor_investido_negativo"].fillna(0) == 0)
    & (df_auditoria_aportes_realizados["n_data_aporte_fora_intervalo"].fillna(0) == 0)
)

lista_inconsistencias = []

for _, linha in df_auditoria_aportes_realizados.iterrows():
    if not bool(linha["flag_contagem_aportes_coerente_com_curva"]):
        registrar_inconsistencia(
            lista_inconsistencias,
            linha,
            "contagem_aportes_divergente_da_curva_patrimonial",
            linha["n_aportes_realizados"],
            linha["n_eventos_aporte_ativos"],
            "A quantidade de aportes registrados não coincide com a quantidade de eventos de aporte ativos na consolidação patrimonial.",
        )
    if not bool(linha["flag_valor_total_aportes_coerente_com_capital"]):
        registrar_inconsistencia(
            lista_inconsistencias,
            linha,
            "valor_total_aportes_divergente_do_capital_inicial",
            linha["valor_total_aportado_registrado"],
            linha["capital_inicial_estrategia_curva"],
            "A soma dos aportes registrados por estratégia não reconcilia com o capital inicial da curva patrimonial.",
        )
    if linha["n_request_id_ausente"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "request_id_ausente", linha["n_request_id_ausente"], 0, "Há aportes sem identificador operacional preenchido.")
    if linha["n_request_id_duplicado"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "request_id_duplicado", linha["n_request_id_duplicado"], 0, "Há request_id duplicado dentro da mesma estratégia.")
    if linha["n_chave_estrategia_ausente"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "chave_estrategia_ausente", linha["n_chave_estrategia_ausente"], 0, "Há aportes sem chave consolidada de estratégia.")
    if linha["n_estrategia_referencia_ausente"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "estrategia_referencia_ausente", linha["n_estrategia_referencia_ausente"], 0, "Há aportes sem referência analítica da estratégia.")
    if linha["n_data_aporte_ausente"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "data_aporte_ausente", linha["n_data_aporte_ausente"], 0, "Há aportes sem data efetiva de aporte.")
    if linha["n_valor_aporte_nao_positivo"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "valor_aporte_nao_positivo", linha["n_valor_aporte_nao_positivo"], 0, "Há aportes com valor nulo, ausente ou negativo.")
    if linha["n_valor_investido_negativo"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "valor_investido_negativo", linha["n_valor_investido_negativo"], 0, "Há aportes com valor investido efetivo negativo.")
    if linha["n_data_aporte_fora_intervalo"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "data_aporte_fora_intervalo_backtest", linha["n_data_aporte_fora_intervalo"], 0, "Há aportes fora da janela ativa do backtest consolidado.")

if len(lista_inconsistencias) == 0:
    df_inconsistencias_aportes_realizados = pd.DataFrame(columns=COLUNAS_INCONSISTENCIAS)
else:
    df_inconsistencias_aportes_realizados = pd.DataFrame(lista_inconsistencias)[COLUNAS_INCONSISTENCIAS]

n_inconsistencias_aportes_realizados = df_inconsistencias_aportes_realizados.shape[0]
n_divergencias_eventos_curva = int((~df_auditoria_aportes_realizados["flag_contagem_aportes_coerente_com_curva"]).sum())
n_divergencias_capital = int((~df_auditoria_aportes_realizados["flag_valor_total_aportes_coerente_com_capital"]).sum())

print(f"Inconsistências identificadas              : {n_inconsistencias_aportes_realizados:,}")
print(f"Divergências com eventos da 9.7            : {n_divergencias_eventos_curva:,}")
print(f"Divergências com capital inicial           : {n_divergencias_capital:,}")
print("OK")

# ============================================================
# 10) Salvamento dos outputs e validação final
# ============================================================

print("\n[10/10] Salvamento dos outputs e validação final...")

salvar_dataframe(df_regras_registro_aportes_realizados, caminho_tbl_regras_registro_aportes_realizados, index=False)
salvar_dataframe(df_aportes_realizados, caminho_base_aportes_realizados, index=False)
salvar_dataframe(df_resumo_aportes_realizados_estrategia, caminho_tbl_resumo_aportes_realizados_estrategia, index=False)
salvar_dataframe(df_distribuicao_anual_aportes_realizados, caminho_tbl_distribuicao_anual_aportes_realizados, index=False)
salvar_dataframe(df_auditoria_aportes_realizados, caminho_tbl_auditoria_aportes_realizados, index=False)
salvar_dataframe(df_inconsistencias_aportes_realizados, caminho_tbl_inconsistencias_aportes_realizados, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

print("\nRegras formais do registro de aportes realizados:")
print(df_regras_registro_aportes_realizados.to_string(index=False))

print("\nBase de aportes realizados - amostra:")
print(df_aportes_realizados.head(30).to_string(index=False))

print("\nResumo dos aportes realizados por estratégia:")
print(df_resumo_aportes_realizados_estrategia.to_string(index=False))

print("\nAuditoria dos aportes realizados:")
print(df_auditoria_aportes_realizados.to_string(index=False))

print("\nInconsistências dos aportes realizados - amostra:")
print(df_inconsistencias_aportes_realizados.head(30).to_string(index=False))

print("\nDistribuição anual dos aportes realizados - amostra:")
print(df_distribuicao_anual_aportes_realizados.head(50).to_string(index=False))

print("\nArquivos salvos na subetapa 10.1:")
print(f"- {caminho_tbl_regras_registro_aportes_realizados}")
print(f"- {caminho_base_aportes_realizados}")
print(f"- {caminho_tbl_resumo_aportes_realizados_estrategia}")
print(f"- {caminho_tbl_distribuicao_anual_aportes_realizados}")
print(f"- {caminho_tbl_auditoria_aportes_realizados}")
print(f"- {caminho_tbl_inconsistencias_aportes_realizados}")

print("\nETAPA 10.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 10.1 - REGISTRO DOS APORTES REALIZADOS

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - eventos de caixa por aporte              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_8\8_4_base_caixa_residual_aporte.parquet
Entrada - evento inicial Ibovespa                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_5_base_evento_inicial_benchmark_ibovespa.parquet
Entrada - evento inicial CDI-only                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_6_base_evento_inicial_cdi_only.parquet
Entrada - catálogo de curvas patrimoniais          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_bra

## Etapa 10.2) Registro dos Tickers Comprados por Aporte

In [48]:
%%time
# ============================================================
# Etapa 10.2) Registro dos Tickers Comprados por Aporte
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 10.2 - REGISTRO DOS TICKERS COMPRADOS POR APORTE")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_aportes_realizados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="base",
    nome="aportes_realizados",
)

caminho_base_compras_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="compras_capitulacao",
)

caminho_base_compras_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="compras_euforia",
)

caminho_base_compras_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="compras_aleatorias_controle",
)

caminho_base_compras_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="compras_mensais_controle",
)

caminho_tbl_regras_registro_tickers_comprados_aporte = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="regras_registro_tickers_comprados_aporte",
)

caminho_base_tickers_comprados_aporte = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="base",
    nome="tickers_comprados_aporte",
)

caminho_tbl_resumo_tickers_comprados_aporte = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="resumo_tickers_comprados_aporte",
)

caminho_tbl_resumo_tickers_comprados_estrategia = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="resumo_tickers_comprados_estrategia",
)

caminho_tbl_distribuicao_anual_tickers_comprados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_tickers_comprados",
)

caminho_tbl_distribuicao_setorial_tickers_comprados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="distribuicao_setorial_tickers_comprados",
)

caminho_tbl_auditoria_tickers_comprados_aporte = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="auditoria_tickers_comprados_aporte",
)

caminho_tbl_inconsistencias_tickers_comprados_aporte = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="inconsistencias_tickers_comprados_aporte",
)

print(f"Entrada - aportes realizados 10.1              : {caminho_base_aportes_realizados}")
print(f"Entrada - compras capitulação                  : {caminho_base_compras_capitulacao}")
print(f"Entrada - compras euforia                      : {caminho_base_compras_euforia}")
print(f"Entrada - compras aleatórias                   : {caminho_base_compras_aleatorias_controle}")
print(f"Entrada - compras mensais                      : {caminho_base_compras_mensais_controle}")
print(f"Saída   - regras do registro de tickers        : {caminho_tbl_regras_registro_tickers_comprados_aporte}")
print(f"Saída   - base de tickers comprados            : {caminho_base_tickers_comprados_aporte}")
print(f"Saída   - resumo por aporte                    : {caminho_tbl_resumo_tickers_comprados_aporte}")
print(f"Saída   - resumo por estratégia                : {caminho_tbl_resumo_tickers_comprados_estrategia}")
print(f"Saída   - distribuição anual                   : {caminho_tbl_distribuicao_anual_tickers_comprados}")
print(f"Saída   - distribuição setorial                : {caminho_tbl_distribuicao_setorial_tickers_comprados}")
print(f"Saída   - auditoria                            : {caminho_tbl_auditoria_tickers_comprados_aporte}")
print(f"Saída   - inconsistências                      : {caminho_tbl_inconsistencias_tickers_comprados_aporte}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_aportes_realizados = pd.read_parquet(caminho_base_aportes_realizados)
df_compras_capitulacao = pd.read_parquet(caminho_base_compras_capitulacao)
df_compras_euforia = pd.read_parquet(caminho_base_compras_euforia)
df_compras_aleatorias_controle = pd.read_parquet(caminho_base_compras_aleatorias_controle)
df_compras_mensais_controle = pd.read_parquet(caminho_base_compras_mensais_controle)

print(f"Aportes realizados 10.1                      : {df_aportes_realizados.shape[0]:,} linhas x {df_aportes_realizados.shape[1]} colunas")
print(f"Compras capitulação                          : {df_compras_capitulacao.shape[0]:,} linhas x {df_compras_capitulacao.shape[1]} colunas")
print(f"Compras euforia                              : {df_compras_euforia.shape[0]:,} linhas x {df_compras_euforia.shape[1]} colunas")
print(f"Compras aleatórias                           : {df_compras_aleatorias_controle.shape[0]:,} linhas x {df_compras_aleatorias_controle.shape[1]} colunas")
print(f"Compras mensais                              : {df_compras_mensais_controle.shape[0]:,} linhas x {df_compras_mensais_controle.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares, colunas e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares, colunas e parâmetros operacionais...")

TOLERANCIA_VALOR_COMPRA = 0.01

COLUNAS_COMPRAS_FONTE = [
    "request_id",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "tipo_estrategia",
    "origem_calendario",
    "grupo_registro_carteira",
    "benchmark_renda_variavel",
    "regra_selecao_cesta",
    "versao_cesta",
    "carteira_id",
    "replica_id",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
    "ordem_compra_no_aporte",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "preco_compra_planejado",
    "quantidade_comprada_planejada",
    "valor_investido_planejado",
    "peso_efetivo_compra_planejada",
    "valor_aporte_na_data",
    "valor_orcamento_alvo_ticker",
    "quantidade_inicial_inteira",
    "quantidade_extra_redistribuicao",
    "flag_recebeu_redistribuicao",
    "flag_compra_fracionaria_equivalente",
    "tipo_conversao_quantidade",
    "flag_evento_compra",
    "quantidade_unidades_extra_redistribuicao",
]

COLUNAS_APORTES_LOOKUP = [
    "chave_evento_aporte",
    "request_id",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_registro_aporte",
    "origem_calendario",
    "origem_backtest",
    "classe_evento_aporte",
    "data_sinal",
    "data_aporte_efetiva",
    "ano",
    "mes",
    "ordem_aporte_estrategia",
    "ordem_evento_aporte_global",
    "ordem_exibicao",
    "capital_inicial_estrategia",
    "valor_aporte_estrategia",
    "valor_investido_efetivo_aporte",
    "valor_caixa_residual_aporte",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
    "ticker_referencia",
    "fonte_benchmark",
    "preco_execucao_referencia",
    "quantidade_referencia",
]

COLUNAS_BASE_TICKERS_COMPRADOS = [
    "chave_evento_compra",
    "chave_evento_aporte",
    "request_id",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_registro_aporte",
    "origem_registro_compra",
    "origem_calendario",
    "origem_backtest",
    "classe_evento_aporte",
    "classe_instrumento",
    "grupo_registro_carteira",
    "regra_selecao_cesta",
    "versao_cesta",
    "data_sinal",
    "data_aporte_efetiva",
    "ano",
    "mes",
    "ordem_aporte_estrategia",
    "ordem_evento_aporte_global",
    "ordem_compra_no_aporte",
    "ordem_exibicao",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "fonte_benchmark",
    "preco_compra",
    "quantidade_comprada",
    "valor_investido_compra",
    "peso_efetivo_compra",
    "valor_aporte_estrategia",
    "valor_investido_efetivo_aporte",
    "valor_caixa_residual_aporte",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
    "valor_aporte_na_data",
    "valor_orcamento_alvo_ticker",
    "pct_valor_investido_no_aporte",
    "quantidade_inicial_inteira",
    "quantidade_extra_redistribuicao",
    "quantidade_unidades_extra_redistribuicao",
    "tipo_conversao_quantidade",
    "flag_evento_compra",
    "flag_compra_efetiva",
    "flag_ticker_preenchido",
    "flag_issuer_code_preenchido",
    "flag_data_aporte_valida",
    "flag_vinculo_aporte_10_1",
    "flag_preco_compra_valido",
    "flag_quantidade_comprada_valida",
    "flag_valor_investido_positivo",
    "flag_peso_efetivo_valido",
    "flag_recebeu_redistribuicao",
    "flag_compra_fracionaria_equivalente",
    "flag_benchmark_equivalente",
]

COLUNAS_CHAVE_ESTRATEGIA = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_backtest",
    "ordem_exibicao",
]

COLUNAS_INCONSISTENCIAS = [
    "tipo_inconsistencia",
    "chave_evento_aporte",
    "request_id",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "grupo_controle",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "valor_observado",
    "valor_referencia",
    "detalhe",
]

def selecionar_colunas_com_padrao(df, colunas):
    df_saida = df.copy()
    for coluna in colunas:
        if coluna not in df_saida.columns:
            df_saida[coluna] = pd.NA
    return df_saida[colunas].copy()

def validar_colunas_obrigatorias(df, colunas, nome_base):
    colunas_ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        raise KeyError(
            f"A base {nome_base} não possui as colunas obrigatórias: {colunas_ausentes}"
        )

def normalizar_texto(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()
    return df

def converter_numerico(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    return df

def converter_booleano(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = df[coluna].fillna(False).astype(bool)
    return df

def registrar_inconsistencia(lista, linha, tipo_inconsistencia, valor_observado, valor_referencia, detalhe):
    lista.append({
        "tipo_inconsistencia": tipo_inconsistencia,
        "chave_evento_aporte": linha.get("chave_evento_aporte"),
        "request_id": linha.get("request_id"),
        "chave_estrategia": linha.get("chave_estrategia"),
        "estrategia_id": linha.get("estrategia_id"),
        "nome_exibicao_estrategia": linha.get("nome_exibicao_estrategia"),
        "grupo_controle": linha.get("grupo_controle"),
        "categoria_estrategia": linha.get("categoria_estrategia"),
        "subcategoria_estrategia": linha.get("subcategoria_estrategia"),
        "valor_observado": valor_observado,
        "valor_referencia": valor_referencia,
        "detalhe": detalhe,
    })

print(f"Tolerância monetária da subetapa              : {TOLERANCIA_VALOR_COMPRA}")
print("OK")

# ============================================================
# 5) Preparação das compras efetivas de ações
# ============================================================

print("\n[5/10] Preparação das compras efetivas de ações...")

df_compras_capitulacao["origem_registro_compra"] = "etapa_9_1_backtest_capitulacao"
df_compras_euforia["origem_registro_compra"] = "etapa_9_2_backtest_euforia"
df_compras_aleatorias_controle["origem_registro_compra"] = "etapa_9_3_backtest_aleatorias_controle"
df_compras_mensais_controle["origem_registro_compra"] = "etapa_9_4_backtest_mensais_controle"

df_compras_acoes_fonte = pd.concat(
    [
        selecionar_colunas_com_padrao(df_compras_capitulacao, COLUNAS_COMPRAS_FONTE + ["origem_registro_compra"]),
        selecionar_colunas_com_padrao(df_compras_euforia, COLUNAS_COMPRAS_FONTE + ["origem_registro_compra"]),
        selecionar_colunas_com_padrao(df_compras_aleatorias_controle, COLUNAS_COMPRAS_FONTE + ["origem_registro_compra"]),
        selecionar_colunas_com_padrao(df_compras_mensais_controle, COLUNAS_COMPRAS_FONTE + ["origem_registro_compra"]),
    ],
    ignore_index=True,
)

df_compras_acoes_fonte = df_compras_acoes_fonte.rename(columns={
    "preco_compra_planejado": "preco_compra",
    "quantidade_comprada_planejada": "quantidade_comprada",
    "valor_investido_planejado": "valor_investido_compra",
    "peso_efetivo_compra_planejada": "peso_efetivo_compra",
})

df_compras_acoes_fonte = converter_numerico(
    df_compras_acoes_fonte,
    [
        "carteira_id",
        "replica_id",
        "ordem_aporte_estrategia",
        "ordem_compra_no_aporte",
        "preco_compra",
        "quantidade_comprada",
        "valor_investido_compra",
        "peso_efetivo_compra",
        "valor_aporte_na_data",
        "valor_orcamento_alvo_ticker",
        "quantidade_inicial_inteira",
        "quantidade_extra_redistribuicao",
        "quantidade_unidades_extra_redistribuicao",
    ],
)

df_compras_acoes_fonte = converter_booleano(
    df_compras_acoes_fonte,
    [
        "flag_evento_compra",
        "flag_recebeu_redistribuicao",
        "flag_compra_fracionaria_equivalente",
    ],
)

df_compras_acoes_fonte["data_sinal"] = pd.to_datetime(df_compras_acoes_fonte["data_sinal"], errors="coerce")
df_compras_acoes_fonte["data_aporte_efetiva"] = pd.to_datetime(df_compras_acoes_fonte["data_aporte_efetiva"], errors="coerce")

df_compras_acoes_fonte["flag_compra_efetiva"] = (
    df_compras_acoes_fonte["flag_evento_compra"]
    & (df_compras_acoes_fonte["quantidade_comprada"].fillna(0.0) > 0.0)
    & (df_compras_acoes_fonte["valor_investido_compra"].fillna(0.0) > 0.0)
)

n_linhas_compras_acoes_fonte = df_compras_acoes_fonte.shape[0]
n_linhas_compras_acoes_nao_efetivas = int((~df_compras_acoes_fonte["flag_compra_efetiva"]).sum())

df_compras_acoes = df_compras_acoes_fonte[df_compras_acoes_fonte["flag_compra_efetiva"]].copy()
df_compras_acoes["classe_instrumento"] = "acao"

print(f"Linhas de compras de ações na fonte          : {n_linhas_compras_acoes_fonte:,}")
print(f"Linhas de compras de ações efetivas          : {df_compras_acoes.shape[0]:,}")
print(f"Linhas de compras de ações não efetivas      : {n_linhas_compras_acoes_nao_efetivas:,}")
print("OK")

# ============================================================
# 6) Vinculação das compras de ações aos aportes registrados
# ============================================================

print("\n[6/10] Vinculação das compras de ações aos aportes registrados...")

df_aportes_lookup = selecionar_colunas_com_padrao(df_aportes_realizados, COLUNAS_APORTES_LOOKUP)

df_aportes_lookup = normalizar_texto(
    df_aportes_lookup,
    [
        "chave_evento_aporte",
        "request_id",
        "chave_estrategia",
        "estrategia_id",
        "nome_exibicao_estrategia",
        "categoria_estrategia",
        "subcategoria_estrategia",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "estrategia_referencia_padronizada",
        "tipo_estrategia",
        "origem_registro_aporte",
        "origem_calendario",
        "origem_backtest",
        "classe_evento_aporte",
        "ticker_referencia",
        "fonte_benchmark",
    ],
)

df_aportes_lookup["data_sinal"] = pd.to_datetime(df_aportes_lookup["data_sinal"], errors="coerce")
df_aportes_lookup["data_aporte_efetiva"] = pd.to_datetime(df_aportes_lookup["data_aporte_efetiva"], errors="coerce")

df_aportes_lookup = converter_numerico(
    df_aportes_lookup,
    [
        "carteira_id",
        "replica_id",
        "ano",
        "mes",
        "ordem_aporte_estrategia",
        "ordem_evento_aporte_global",
        "ordem_exibicao",
        "capital_inicial_estrategia",
        "valor_aporte_estrategia",
        "valor_investido_efetivo_aporte",
        "valor_caixa_residual_aporte",
        "n_tickers_cesta_equal_weight",
        "n_tickers_compra_efetivada",
        "n_tickers_quantidade_zero",
        "preco_execucao_referencia",
        "quantidade_referencia",
    ],
)

df_aportes_lookup_duplicados = df_aportes_lookup[df_aportes_lookup.duplicated(["estrategia_id", "request_id"], keep=False)]

if df_aportes_lookup_duplicados.shape[0] > 0:
    raise ValueError("A base de aportes da 10.1 possui duplicidades por estrategia_id + request_id.")

df_compras_acoes = normalizar_texto(
    df_compras_acoes,
    [
        "request_id",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "tipo_estrategia",
        "origem_calendario",
        "grupo_registro_carteira",
        "benchmark_renda_variavel",
        "regra_selecao_cesta",
        "versao_cesta",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "tipo_conversao_quantidade",
        "origem_registro_compra",
        "classe_instrumento",
    ],
)

df_compras_acoes_vinculadas = df_compras_acoes.merge(
    df_aportes_lookup,
    on=["estrategia_id", "request_id"],
    how="left",
    suffixes=("_compra", ""),
)

n_compras_acoes_sem_vinculo_10_1 = int(df_compras_acoes_vinculadas["chave_evento_aporte"].isna().sum())

for coluna in [
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "tipo_estrategia",
    "origem_calendario",
    "carteira_id",
    "replica_id",
    "data_sinal",
    "data_aporte_efetiva",
    "ordem_aporte_estrategia",
]:
    coluna_compra = f"{coluna}_compra"
    if coluna_compra in df_compras_acoes_vinculadas.columns:
        df_compras_acoes_vinculadas[coluna] = df_compras_acoes_vinculadas[coluna].where(
            df_compras_acoes_vinculadas[coluna].notna(),
            df_compras_acoes_vinculadas[coluna_compra],
        )
        df_compras_acoes_vinculadas = df_compras_acoes_vinculadas.drop(columns=[coluna_compra])

df_compras_acoes_vinculadas["fonte_benchmark"] = pd.NA
df_compras_acoes_vinculadas["flag_benchmark_equivalente"] = False

print(f"Compras de ações vinculadas à 10.1           : {df_compras_acoes_vinculadas.shape[0]:,}")
print(f"Compras de ações sem vínculo com a 10.1      : {n_compras_acoes_sem_vinculo_10_1:,}")
print("OK")

# ============================================================
# 7) Preparação dos instrumentos equivalentes de benchmark
# ============================================================

print("\n[7/10] Preparação dos instrumentos equivalentes de benchmark...")

df_benchmarks_aportes = df_aportes_lookup[
    df_aportes_lookup["classe_evento_aporte"].astype("string") == "aporte_benchmark"
].copy()

df_benchmarks_aportes["origem_registro_compra"] = df_benchmarks_aportes["origem_registro_aporte"]
df_benchmarks_aportes["classe_instrumento"] = np.where(
    df_benchmarks_aportes["estrategia_id"].astype("string") == "cdi_only",
    "taxa_referencia",
    "benchmark",
)
df_benchmarks_aportes["grupo_registro_carteira"] = np.where(
    df_benchmarks_aportes["classe_instrumento"] == "taxa_referencia",
    "renda_fixa",
    "benchmark",
)
df_benchmarks_aportes["regra_selecao_cesta"] = "instrumento_equivalente_benchmark"
df_benchmarks_aportes["versao_cesta"] = "benchmark_equivalente"
df_benchmarks_aportes["ordem_compra_no_aporte"] = 1
df_benchmarks_aportes["ticker"] = df_benchmarks_aportes["ticker_referencia"]
df_benchmarks_aportes["issuer_code"] = df_benchmarks_aportes["ticker_referencia"]
df_benchmarks_aportes["nome"] = df_benchmarks_aportes["nome_exibicao_estrategia"]
df_benchmarks_aportes["setor"] = np.where(
    df_benchmarks_aportes["classe_instrumento"] == "taxa_referencia",
    "Renda Fixa",
    "Benchmark",
)
df_benchmarks_aportes["subsetor"] = np.where(
    df_benchmarks_aportes["classe_instrumento"] == "taxa_referencia",
    "CDI",
    "Ibovespa",
)
df_benchmarks_aportes["segmento"] = df_benchmarks_aportes["subsetor"]
df_benchmarks_aportes["preco_compra"] = df_benchmarks_aportes["preco_execucao_referencia"]
df_benchmarks_aportes["quantidade_comprada"] = df_benchmarks_aportes["quantidade_referencia"]
df_benchmarks_aportes["valor_investido_compra"] = df_benchmarks_aportes["valor_investido_efetivo_aporte"]
df_benchmarks_aportes["peso_efetivo_compra"] = 1.0
df_benchmarks_aportes["valor_aporte_na_data"] = df_benchmarks_aportes["valor_aporte_estrategia"]
df_benchmarks_aportes["valor_orcamento_alvo_ticker"] = df_benchmarks_aportes["valor_aporte_estrategia"]
df_benchmarks_aportes["quantidade_inicial_inteira"] = df_benchmarks_aportes["quantidade_referencia"]
df_benchmarks_aportes["quantidade_extra_redistribuicao"] = 0.0
df_benchmarks_aportes["quantidade_unidades_extra_redistribuicao"] = 0.0
df_benchmarks_aportes["tipo_conversao_quantidade"] = np.where(
    df_benchmarks_aportes["classe_instrumento"] == "taxa_referencia",
    "aplicacao_cdi_integral",
    "benchmark_equivalente_fracionario",
)
df_benchmarks_aportes["flag_evento_compra"] = True
df_benchmarks_aportes["flag_compra_efetiva"] = True
df_benchmarks_aportes["flag_recebeu_redistribuicao"] = False
df_benchmarks_aportes["flag_compra_fracionaria_equivalente"] = True
df_benchmarks_aportes["flag_benchmark_equivalente"] = True

print(f"Instrumentos equivalentes de benchmark       : {df_benchmarks_aportes.shape[0]:,}")
print("OK")

# ============================================================
# 8) Consolidação da base de tickers comprados por aporte
# ============================================================

print("\n[8/10] Consolidação da base de tickers comprados por aporte...")

df_base_tickers_comprados_aporte = pd.concat(
    [
        selecionar_colunas_com_padrao(df_compras_acoes_vinculadas, COLUNAS_BASE_TICKERS_COMPRADOS),
        selecionar_colunas_com_padrao(df_benchmarks_aportes, COLUNAS_BASE_TICKERS_COMPRADOS),
    ],
    ignore_index=True,
)

df_base_tickers_comprados_aporte = normalizar_texto(
    df_base_tickers_comprados_aporte,
    [
        "chave_evento_aporte",
        "request_id",
        "chave_estrategia",
        "estrategia_id",
        "nome_exibicao_estrategia",
        "categoria_estrategia",
        "subcategoria_estrategia",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "estrategia_referencia_padronizada",
        "tipo_estrategia",
        "origem_registro_aporte",
        "origem_registro_compra",
        "origem_calendario",
        "origem_backtest",
        "classe_evento_aporte",
        "classe_instrumento",
        "grupo_registro_carteira",
        "regra_selecao_cesta",
        "versao_cesta",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "fonte_benchmark",
        "tipo_conversao_quantidade",
    ],
)

df_base_tickers_comprados_aporte["data_sinal"] = pd.to_datetime(df_base_tickers_comprados_aporte["data_sinal"], errors="coerce")
df_base_tickers_comprados_aporte["data_aporte_efetiva"] = pd.to_datetime(df_base_tickers_comprados_aporte["data_aporte_efetiva"], errors="coerce")

df_base_tickers_comprados_aporte = converter_numerico(
    df_base_tickers_comprados_aporte,
    [
        "carteira_id",
        "replica_id",
        "ano",
        "mes",
        "ordem_aporte_estrategia",
        "ordem_evento_aporte_global",
        "ordem_compra_no_aporte",
        "ordem_exibicao",
        "preco_compra",
        "quantidade_comprada",
        "valor_investido_compra",
        "peso_efetivo_compra",
        "valor_aporte_estrategia",
        "valor_investido_efetivo_aporte",
        "valor_caixa_residual_aporte",
        "n_tickers_cesta_equal_weight",
        "n_tickers_compra_efetivada",
        "n_tickers_quantidade_zero",
        "valor_aporte_na_data",
        "valor_orcamento_alvo_ticker",
        "quantidade_inicial_inteira",
        "quantidade_extra_redistribuicao",
        "quantidade_unidades_extra_redistribuicao",
    ],
)

df_base_tickers_comprados_aporte = converter_booleano(
    df_base_tickers_comprados_aporte,
    [
        "flag_evento_compra",
        "flag_compra_efetiva",
        "flag_recebeu_redistribuicao",
        "flag_compra_fracionaria_equivalente",
        "flag_benchmark_equivalente",
    ],
)

df_base_tickers_comprados_aporte["pct_valor_investido_no_aporte"] = (
    df_base_tickers_comprados_aporte["valor_investido_compra"]
    / df_base_tickers_comprados_aporte["valor_aporte_estrategia"]
)

df_base_tickers_comprados_aporte["flag_ticker_preenchido"] = (
    df_base_tickers_comprados_aporte["ticker"].notna()
    & (df_base_tickers_comprados_aporte["ticker"].astype("string").str.len() > 0)
)

df_base_tickers_comprados_aporte["flag_issuer_code_preenchido"] = (
    df_base_tickers_comprados_aporte["issuer_code"].notna()
    & (df_base_tickers_comprados_aporte["issuer_code"].astype("string").str.len() > 0)
)

df_base_tickers_comprados_aporte["flag_data_aporte_valida"] = df_base_tickers_comprados_aporte["data_aporte_efetiva"].notna()
df_base_tickers_comprados_aporte["flag_vinculo_aporte_10_1"] = df_base_tickers_comprados_aporte["chave_evento_aporte"].notna()

df_base_tickers_comprados_aporte["flag_preco_compra_valido"] = (
    (
        df_base_tickers_comprados_aporte["classe_instrumento"].isin(["acao", "benchmark"])
        & (df_base_tickers_comprados_aporte["preco_compra"].fillna(0.0) > 0.0)
    )
    | (df_base_tickers_comprados_aporte["classe_instrumento"] == "taxa_referencia")
)

df_base_tickers_comprados_aporte["flag_quantidade_comprada_valida"] = (
    (
        df_base_tickers_comprados_aporte["classe_instrumento"].isin(["acao", "benchmark"])
        & (df_base_tickers_comprados_aporte["quantidade_comprada"].fillna(0.0) > 0.0)
    )
    | (df_base_tickers_comprados_aporte["classe_instrumento"] == "taxa_referencia")
)

df_base_tickers_comprados_aporte["flag_valor_investido_positivo"] = df_base_tickers_comprados_aporte["valor_investido_compra"].fillna(0.0) > 0.0

df_base_tickers_comprados_aporte["flag_peso_efetivo_valido"] = (
    df_base_tickers_comprados_aporte["peso_efetivo_compra"].notna()
    & (df_base_tickers_comprados_aporte["peso_efetivo_compra"] >= 0.0)
    & (df_base_tickers_comprados_aporte["peso_efetivo_compra"] <= 1.0 + 0.00000001)
)

df_base_tickers_comprados_aporte["chave_evento_compra"] = (
    df_base_tickers_comprados_aporte["chave_evento_aporte"].astype("string")
    + "__compra_"
    + df_base_tickers_comprados_aporte["ordem_compra_no_aporte"].fillna(0).astype(int).astype("string")
    + "__"
    + df_base_tickers_comprados_aporte["ticker"].astype("string")
)

df_base_tickers_comprados_aporte = df_base_tickers_comprados_aporte.sort_values(
    [
        "data_aporte_efetiva",
        "ordem_exibicao",
        "estrategia_id",
        "ordem_aporte_estrategia",
        "ordem_compra_no_aporte",
        "ticker",
    ],
    na_position="last",
).reset_index(drop=True)

df_base_tickers_comprados_aporte = selecionar_colunas_com_padrao(
    df_base_tickers_comprados_aporte,
    COLUNAS_BASE_TICKERS_COMPRADOS,
)

validar_colunas_obrigatorias(
    df_base_tickers_comprados_aporte,
    COLUNAS_BASE_TICKERS_COMPRADOS,
    "10_2_base_tickers_comprados_aporte",
)

print(f"Base de tickers comprados por aporte        : {df_base_tickers_comprados_aporte.shape[0]:,} linhas x {df_base_tickers_comprados_aporte.shape[1]} colunas")
print(f"Instrumentos de ações registrados           : {(df_base_tickers_comprados_aporte['classe_instrumento'] == 'acao').sum():,}")
print(f"Instrumentos equivalentes de benchmark      : {(df_base_tickers_comprados_aporte['classe_instrumento'].isin(['benchmark', 'taxa_referencia'])).sum():,}")
print(f"Aportes com pelo menos um instrumento       : {df_base_tickers_comprados_aporte[['estrategia_id', 'request_id']].drop_duplicates().shape[0]:,}")
print(f"Estratégias com instrumentos registrados    : {df_base_tickers_comprados_aporte['estrategia_id'].nunique():,}")
print("OK")

# ============================================================
# 9) Construção das regras, resumos, auditoria e inconsistências
# ============================================================

print("\n[9/10] Construção das regras, resumos, auditoria e inconsistências...")

validar_colunas_obrigatorias(
    df_base_tickers_comprados_aporte,
    COLUNAS_BASE_TICKERS_COMPRADOS + ["n_tickers_compra_efetivada"],
    "base consolidada para resumos da 10.2",
)

validar_colunas_obrigatorias(
    df_aportes_lookup,
    COLUNAS_APORTES_LOOKUP,
    "10_1_base_aportes_realizados",
)

df_regras_registro_tickers_comprados_aporte = pd.DataFrame([
    {
        "ordem_regra": 1,
        "regra_id": "RTC1",
        "escopo": "fonte",
        "regra_operacional": "usar_as_bases_de_compras_dos_backtests_como_fonte_primaria",
        "detalhe": "As compras de ações são consolidadas a partir das bases efetivas de compras das etapas 9.1, 9.2, 9.3 e 9.4.",
    },
    {
        "ordem_regra": 2,
        "regra_id": "RTC2",
        "escopo": "vinculacao",
        "regra_operacional": "vincular_cada_compra_ao_aporte_registrado_na_10_1",
        "detalhe": "Cada linha de compra é vinculada à base de aportes realizados por estrategia_id e request_id.",
    },
    {
        "ordem_regra": 3,
        "regra_id": "RTC3",
        "escopo": "benchmarks",
        "regra_operacional": "registrar_instrumentos_equivalentes_para_benchmarks",
        "detalhe": "Ibovespa buy-and-hold e CDI-only são representados por um instrumento equivalente para manter a reconciliação com os aportes formais.",
    },
    {
        "ordem_regra": 4,
        "regra_id": "RTC4",
        "escopo": "efetividade",
        "regra_operacional": "manter_apenas_compras_efetivas_de_acoes",
        "detalhe": "Compras de ações entram na base quando possuem evento de compra, quantidade positiva e valor investido positivo.",
    },
    {
        "ordem_regra": 5,
        "regra_id": "RTC5",
        "escopo": "auditoria",
        "regra_operacional": "reconciliar_contagem_e_valor_com_a_base_de_aportes_realizados",
        "detalhe": "A auditoria compara, por aporte, o número de instrumentos e o valor investido registrado contra os campos consolidados na etapa 10.1.",
    },
])

df_resumo_tickers_comprados_aporte = (
    df_base_tickers_comprados_aporte
    .groupby(
        [
            "chave_evento_aporte",
            "request_id",
            "chave_estrategia",
            "estrategia_id",
            "nome_exibicao_estrategia",
            "categoria_estrategia",
            "subcategoria_estrategia",
            "grupo_controle",
            "familia_estrategia",
            "estrategia_referencia",
            "estrategia_referencia_padronizada",
            "tipo_estrategia",
            "carteira_id",
            "replica_id",
            "origem_backtest",
            "data_sinal",
            "data_aporte_efetiva",
            "ano",
            "mes",
            "ordem_aporte_estrategia",
            "ordem_evento_aporte_global",
            "ordem_exibicao",
            "valor_aporte_estrategia",
            "valor_investido_efetivo_aporte",
            "n_tickers_compra_efetivada",
        ],
        dropna=False,
    )
    .agg(
        n_linhas_instrumentos_aporte=("chave_evento_compra", "nunique"),
        n_tickers_comprados_aporte=("ticker", "nunique"),
        n_empresas_compradas_aporte=("issuer_code", "nunique"),
        n_setores_aporte=("setor", "nunique"),
        n_subsetores_aporte=("subsetor", "nunique"),
        n_segmentos_aporte=("segmento", "nunique"),
        valor_total_investido_aporte=("valor_investido_compra", "sum"),
        peso_total_efetivo_aporte=("peso_efetivo_compra", "sum"),
        menor_valor_compra_aporte=("valor_investido_compra", "min"),
        maior_valor_compra_aporte=("valor_investido_compra", "max"),
        n_linhas_sem_ticker=("flag_ticker_preenchido", lambda x: int((~x).sum())),
        n_linhas_sem_issuer_code=("flag_issuer_code_preenchido", lambda x: int((~x).sum())),
        n_linhas_preco_invalido=("flag_preco_compra_valido", lambda x: int((~x).sum())),
        n_linhas_quantidade_invalida=("flag_quantidade_comprada_valida", lambda x: int((~x).sum())),
        n_linhas_valor_investido_nao_positivo=("flag_valor_investido_positivo", lambda x: int((~x).sum())),
        n_linhas_peso_invalido=("flag_peso_efetivo_valido", lambda x: int((~x).sum())),
        n_linhas_sem_vinculo_10_1=("flag_vinculo_aporte_10_1", lambda x: int((~x).sum())),
    )
    .reset_index()
)

COLUNAS_CONTAGEM_RESUMO_APORTE = [
    "ano",
    "mes",
    "ordem_aporte_estrategia",
    "ordem_evento_aporte_global",
    "ordem_exibicao",
    "n_tickers_compra_efetivada",
    "n_linhas_instrumentos_aporte",
    "n_tickers_comprados_aporte",
    "n_empresas_compradas_aporte",
    "n_setores_aporte",
    "n_subsetores_aporte",
    "n_segmentos_aporte",
    "n_linhas_sem_ticker",
    "n_linhas_sem_issuer_code",
    "n_linhas_preco_invalido",
    "n_linhas_quantidade_invalida",
    "n_linhas_valor_investido_nao_positivo",
    "n_linhas_peso_invalido",
    "n_linhas_sem_vinculo_10_1",
]

for coluna in COLUNAS_CONTAGEM_RESUMO_APORTE:
    df_resumo_tickers_comprados_aporte[coluna] = pd.to_numeric(
        df_resumo_tickers_comprados_aporte[coluna],
        errors="coerce",
    ).fillna(0).astype(int)

df_resumo_tickers_comprados_aporte["desvio_n_tickers_vs_10_1"] = (
    df_resumo_tickers_comprados_aporte["n_tickers_comprados_aporte"]
    - df_resumo_tickers_comprados_aporte["n_tickers_compra_efetivada"]
)

df_resumo_tickers_comprados_aporte["desvio_valor_investido_vs_10_1"] = (
    df_resumo_tickers_comprados_aporte["valor_total_investido_aporte"]
    - df_resumo_tickers_comprados_aporte["valor_investido_efetivo_aporte"]
)

df_resumo_tickers_comprados_aporte["flag_contagem_tickers_coerente_10_1"] = (
    df_resumo_tickers_comprados_aporte["desvio_n_tickers_vs_10_1"].fillna(np.inf).abs() == 0
)

df_resumo_tickers_comprados_aporte["flag_valor_investido_coerente_10_1"] = (
    df_resumo_tickers_comprados_aporte["desvio_valor_investido_vs_10_1"].fillna(np.inf).abs() <= TOLERANCIA_VALOR_COMPRA
)

df_resumo_tickers_comprados_estrategia = (
    df_resumo_tickers_comprados_aporte
    .groupby(COLUNAS_CHAVE_ESTRATEGIA, dropna=False)
    .agg(
        n_aportes_com_instrumentos=("request_id", "nunique"),
        primeira_data_aporte=("data_aporte_efetiva", "min"),
        ultima_data_aporte=("data_aporte_efetiva", "max"),
        n_linhas_instrumentos=("n_linhas_instrumentos_aporte", "sum"),
        n_tickers_comprados_total=("n_tickers_comprados_aporte", "sum"),
        media_tickers_por_aporte=("n_tickers_comprados_aporte", "mean"),
        minimo_tickers_por_aporte=("n_tickers_comprados_aporte", "min"),
        maximo_tickers_por_aporte=("n_tickers_comprados_aporte", "max"),
        valor_total_investido=("valor_total_investido_aporte", "sum"),
        valor_medio_investido_por_aporte=("valor_total_investido_aporte", "mean"),
        n_aportes_contagem_coerente=("flag_contagem_tickers_coerente_10_1", "sum"),
        n_aportes_valor_coerente=("flag_valor_investido_coerente_10_1", "sum"),
    )
    .reset_index()
    .sort_values(["ordem_exibicao", "estrategia_id"])
    .reset_index(drop=True)
)

COLUNAS_CONTAGEM_RESUMO_ESTRATEGIA = [
    "n_aportes_com_instrumentos",
    "n_linhas_instrumentos",
    "n_tickers_comprados_total",
    "minimo_tickers_por_aporte",
    "maximo_tickers_por_aporte",
    "n_aportes_contagem_coerente",
    "n_aportes_valor_coerente",
]

for coluna in COLUNAS_CONTAGEM_RESUMO_ESTRATEGIA:
    df_resumo_tickers_comprados_estrategia[coluna] = pd.to_numeric(
        df_resumo_tickers_comprados_estrategia[coluna],
        errors="coerce",
    ).fillna(0).astype(int)

df_tickers_distintos_estrategia = (
    df_base_tickers_comprados_aporte
    .groupby("estrategia_id", dropna=False)
    .agg(
        n_tickers_distintos=("ticker", "nunique"),
        n_empresas_distintas=("issuer_code", "nunique"),
        n_setores_distintos=("setor", "nunique"),
        n_subsetores_distintos=("subsetor", "nunique"),
        n_segmentos_distintos=("segmento", "nunique"),
    )
    .reset_index()
)

df_resumo_tickers_comprados_estrategia = df_resumo_tickers_comprados_estrategia.merge(
    df_tickers_distintos_estrategia,
    on="estrategia_id",
    how="left",
)

df_distribuicao_anual_tickers_comprados = (
    df_base_tickers_comprados_aporte
    .groupby(COLUNAS_CHAVE_ESTRATEGIA + ["ano"], dropna=False)
    .agg(
        n_aportes_ano=("request_id", "nunique"),
        n_linhas_instrumentos_ano=("chave_evento_compra", "nunique"),
        n_tickers_distintos_ano=("ticker", "nunique"),
        n_empresas_distintas_ano=("issuer_code", "nunique"),
        valor_total_investido_ano=("valor_investido_compra", "sum"),
        primeira_data_aporte_ano=("data_aporte_efetiva", "min"),
        ultima_data_aporte_ano=("data_aporte_efetiva", "max"),
    )
    .reset_index()
    .sort_values(["ordem_exibicao", "estrategia_id", "ano"])
    .reset_index(drop=True)
)

df_distribuicao_setorial_tickers_comprados = (
    df_base_tickers_comprados_aporte
    .groupby(COLUNAS_CHAVE_ESTRATEGIA + ["setor", "subsetor", "segmento"], dropna=False)
    .agg(
        n_linhas_instrumentos=("chave_evento_compra", "nunique"),
        n_aportes_com_presenca=("request_id", "nunique"),
        n_tickers_distintos=("ticker", "nunique"),
        n_empresas_distintas=("issuer_code", "nunique"),
        valor_total_investido=("valor_investido_compra", "sum"),
    )
    .reset_index()
    .sort_values(["ordem_exibicao", "estrategia_id", "valor_total_investido"], ascending=[True, True, False])
    .reset_index(drop=True)
)

df_tickers_duplicados_aporte = (
    df_base_tickers_comprados_aporte[
        df_base_tickers_comprados_aporte.duplicated(["estrategia_id", "request_id", "ticker"], keep=False)
    ]
    .groupby(["estrategia_id", "request_id"], dropna=False)
    .agg(n_tickers_duplicados_no_aporte=("ticker", "nunique"))
    .reset_index()
)

df_auditoria_tickers_comprados_aporte = df_aportes_lookup.merge(
    df_resumo_tickers_comprados_aporte[
        [
            "estrategia_id",
            "request_id",
            "n_linhas_instrumentos_aporte",
            "n_tickers_comprados_aporte",
            "n_empresas_compradas_aporte",
            "valor_total_investido_aporte",
            "peso_total_efetivo_aporte",
            "n_linhas_sem_ticker",
            "n_linhas_sem_issuer_code",
            "n_linhas_preco_invalido",
            "n_linhas_quantidade_invalida",
            "n_linhas_valor_investido_nao_positivo",
            "n_linhas_peso_invalido",
            "n_linhas_sem_vinculo_10_1",
            "desvio_n_tickers_vs_10_1",
            "desvio_valor_investido_vs_10_1",
            "flag_contagem_tickers_coerente_10_1",
            "flag_valor_investido_coerente_10_1",
        ]
    ],
    on=["estrategia_id", "request_id"],
    how="left",
)

df_auditoria_tickers_comprados_aporte = df_auditoria_tickers_comprados_aporte.merge(
    df_tickers_duplicados_aporte,
    on=["estrategia_id", "request_id"],
    how="left",
)

for coluna in [
    "n_linhas_instrumentos_aporte",
    "n_tickers_comprados_aporte",
    "n_empresas_compradas_aporte",
    "valor_total_investido_aporte",
    "peso_total_efetivo_aporte",
    "n_linhas_sem_ticker",
    "n_linhas_sem_issuer_code",
    "n_linhas_preco_invalido",
    "n_linhas_quantidade_invalida",
    "n_linhas_valor_investido_nao_positivo",
    "n_linhas_peso_invalido",
    "n_linhas_sem_vinculo_10_1",
    "desvio_n_tickers_vs_10_1",
    "desvio_valor_investido_vs_10_1",
    "n_tickers_duplicados_no_aporte",
]:
    df_auditoria_tickers_comprados_aporte[coluna] = pd.to_numeric(
        df_auditoria_tickers_comprados_aporte[coluna],
        errors="coerce",
    ).fillna(0.0)

COLUNAS_CONTAGEM_AUDITORIA_TICKERS = [
    "ano",
    "mes",
    "ordem_aporte_estrategia",
    "ordem_evento_aporte_global",
    "ordem_exibicao",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
    "n_linhas_instrumentos_aporte",
    "n_tickers_comprados_aporte",
    "n_empresas_compradas_aporte",
    "n_linhas_sem_ticker",
    "n_linhas_sem_issuer_code",
    "n_linhas_preco_invalido",
    "n_linhas_quantidade_invalida",
    "n_linhas_valor_investido_nao_positivo",
    "n_linhas_peso_invalido",
    "n_linhas_sem_vinculo_10_1",
    "desvio_n_tickers_vs_10_1",
    "n_tickers_duplicados_no_aporte",
]

for coluna in COLUNAS_CONTAGEM_AUDITORIA_TICKERS:
    if coluna in df_auditoria_tickers_comprados_aporte.columns:
        df_auditoria_tickers_comprados_aporte[coluna] = pd.to_numeric(
            df_auditoria_tickers_comprados_aporte[coluna],
            errors="coerce",
        ).fillna(0).astype(int)

df_auditoria_tickers_comprados_aporte["flag_contagem_tickers_coerente_10_1"] = (
    df_auditoria_tickers_comprados_aporte["desvio_n_tickers_vs_10_1"].abs() == 0
)

df_auditoria_tickers_comprados_aporte["flag_valor_investido_coerente_10_1"] = (
    df_auditoria_tickers_comprados_aporte["desvio_valor_investido_vs_10_1"].abs() <= TOLERANCIA_VALOR_COMPRA
)

df_auditoria_tickers_comprados_aporte["flag_sem_inconsistencia_tickers_comprados"] = (
    df_auditoria_tickers_comprados_aporte["flag_contagem_tickers_coerente_10_1"]
    & df_auditoria_tickers_comprados_aporte["flag_valor_investido_coerente_10_1"]
    & (df_auditoria_tickers_comprados_aporte["n_linhas_instrumentos_aporte"] > 0)
    & (df_auditoria_tickers_comprados_aporte["n_linhas_sem_ticker"] == 0)
    & (df_auditoria_tickers_comprados_aporte["n_linhas_sem_issuer_code"] == 0)
    & (df_auditoria_tickers_comprados_aporte["n_linhas_preco_invalido"] == 0)
    & (df_auditoria_tickers_comprados_aporte["n_linhas_quantidade_invalida"] == 0)
    & (df_auditoria_tickers_comprados_aporte["n_linhas_valor_investido_nao_positivo"] == 0)
    & (df_auditoria_tickers_comprados_aporte["n_linhas_peso_invalido"] == 0)
    & (df_auditoria_tickers_comprados_aporte["n_linhas_sem_vinculo_10_1"] == 0)
    & (df_auditoria_tickers_comprados_aporte["n_tickers_duplicados_no_aporte"] == 0)
)

lista_inconsistencias = []

for _, linha in df_auditoria_tickers_comprados_aporte.iterrows():
    if linha["n_linhas_instrumentos_aporte"] <= 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "aporte_sem_instrumento_comprado", linha["n_linhas_instrumentos_aporte"], 1, "O aporte não possui instrumento comprado registrado na base 10.2.")
    if not bool(linha["flag_contagem_tickers_coerente_10_1"]):
        registrar_inconsistencia(lista_inconsistencias, linha, "contagem_tickers_divergente_da_10_1", linha["n_tickers_comprados_aporte"], linha["n_tickers_compra_efetivada"], "A quantidade de tickers comprados por aporte diverge da contagem registrada na etapa 10.1.")
    if not bool(linha["flag_valor_investido_coerente_10_1"]):
        registrar_inconsistencia(lista_inconsistencias, linha, "valor_investido_divergente_da_10_1", linha["valor_total_investido_aporte"], linha["valor_investido_efetivo_aporte"], "O valor investido por aporte diverge do valor consolidado na etapa 10.1.")
    if linha["n_linhas_sem_ticker"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "ticker_ausente", linha["n_linhas_sem_ticker"], 0, "Há linhas de compra sem ticker preenchido.")
    if linha["n_linhas_sem_issuer_code"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "issuer_code_ausente", linha["n_linhas_sem_issuer_code"], 0, "Há linhas de compra sem issuer_code preenchido.")
    if linha["n_linhas_preco_invalido"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "preco_compra_invalido", linha["n_linhas_preco_invalido"], 0, "Há linhas de compra com preço inválido para ações ou benchmark.")
    if linha["n_linhas_quantidade_invalida"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "quantidade_comprada_invalida", linha["n_linhas_quantidade_invalida"], 0, "Há linhas de compra com quantidade inválida para ações ou benchmark.")
    if linha["n_linhas_valor_investido_nao_positivo"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "valor_investido_nao_positivo", linha["n_linhas_valor_investido_nao_positivo"], 0, "Há linhas de compra com valor investido nulo, ausente ou negativo.")
    if linha["n_linhas_peso_invalido"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "peso_efetivo_invalido", linha["n_linhas_peso_invalido"], 0, "Há linhas de compra com peso efetivo fora do intervalo permitido.")
    if linha["n_linhas_sem_vinculo_10_1"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "vinculo_aporte_10_1_ausente", linha["n_linhas_sem_vinculo_10_1"], 0, "Há linhas de compra sem vínculo com a base de aportes realizados da 10.1.")
    if linha["n_tickers_duplicados_no_aporte"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "ticker_duplicado_no_mesmo_aporte", linha["n_tickers_duplicados_no_aporte"], 0, "Há ticker repetido dentro do mesmo aporte e estratégia.")

if len(lista_inconsistencias) == 0:
    df_inconsistencias_tickers_comprados_aporte = pd.DataFrame(columns=COLUNAS_INCONSISTENCIAS)
else:
    df_inconsistencias_tickers_comprados_aporte = pd.DataFrame(lista_inconsistencias)[COLUNAS_INCONSISTENCIAS]

n_inconsistencias_tickers_comprados = df_inconsistencias_tickers_comprados_aporte.shape[0]
n_aportes_sem_instrumentos = int((df_auditoria_tickers_comprados_aporte["n_linhas_instrumentos_aporte"] <= 0).sum())
n_divergencias_contagem = int((~df_auditoria_tickers_comprados_aporte["flag_contagem_tickers_coerente_10_1"]).sum())
n_divergencias_valor = int((~df_auditoria_tickers_comprados_aporte["flag_valor_investido_coerente_10_1"]).sum())

print(f"Regras formais geradas                       : {df_regras_registro_tickers_comprados_aporte.shape[0]:,}")
print(f"Resumo por aporte gerado                    : {df_resumo_tickers_comprados_aporte.shape[0]:,}")
print(f"Resumo por estratégia gerado                : {df_resumo_tickers_comprados_estrategia.shape[0]:,}")
print(f"Distribuição anual gerada                   : {df_distribuicao_anual_tickers_comprados.shape[0]:,}")
print(f"Distribuição setorial gerada                : {df_distribuicao_setorial_tickers_comprados.shape[0]:,}")
print(f"Inconsistências identificadas               : {n_inconsistencias_tickers_comprados:,}")
print(f"Aportes sem instrumentos registrados        : {n_aportes_sem_instrumentos:,}")
print(f"Divergências de contagem contra a 10.1      : {n_divergencias_contagem:,}")
print(f"Divergências de valor contra a 10.1         : {n_divergencias_valor:,}")
print("OK")

# ============================================================
# 10) Salvamento dos outputs e validação final
# ============================================================

print("\n[10/10] Salvamento dos outputs e validação final...")

salvar_dataframe(df_regras_registro_tickers_comprados_aporte, caminho_tbl_regras_registro_tickers_comprados_aporte, index=False)
salvar_dataframe(df_base_tickers_comprados_aporte, caminho_base_tickers_comprados_aporte, index=False)
salvar_dataframe(df_resumo_tickers_comprados_aporte, caminho_tbl_resumo_tickers_comprados_aporte, index=False)
salvar_dataframe(df_resumo_tickers_comprados_estrategia, caminho_tbl_resumo_tickers_comprados_estrategia, index=False)
salvar_dataframe(df_distribuicao_anual_tickers_comprados, caminho_tbl_distribuicao_anual_tickers_comprados, index=False)
salvar_dataframe(df_distribuicao_setorial_tickers_comprados, caminho_tbl_distribuicao_setorial_tickers_comprados, index=False)
salvar_dataframe(df_auditoria_tickers_comprados_aporte, caminho_tbl_auditoria_tickers_comprados_aporte, index=False)
salvar_dataframe(df_inconsistencias_tickers_comprados_aporte, caminho_tbl_inconsistencias_tickers_comprados_aporte, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

print("\nRegras formais do registro de tickers comprados por aporte:")
print(df_regras_registro_tickers_comprados_aporte.to_string(index=False))

print("\nBase de tickers comprados por aporte - amostra:")
print(df_base_tickers_comprados_aporte.head(30).to_string(index=False))

print("\nResumo dos tickers comprados por aporte - amostra:")
print(df_resumo_tickers_comprados_aporte.head(30).to_string(index=False))

print("\nResumo dos tickers comprados por estratégia:")
print(df_resumo_tickers_comprados_estrategia.to_string(index=False))

print("\nAuditoria dos tickers comprados por aporte - amostra:")
print(df_auditoria_tickers_comprados_aporte.head(40).to_string(index=False))

print("\nInconsistências dos tickers comprados por aporte - amostra:")
print(df_inconsistencias_tickers_comprados_aporte.head(30).to_string(index=False))

print("\nDistribuição anual dos tickers comprados - amostra:")
print(df_distribuicao_anual_tickers_comprados.head(50).to_string(index=False))

print("\nDistribuição setorial dos tickers comprados - amostra:")
print(df_distribuicao_setorial_tickers_comprados.head(50).to_string(index=False))

print("\nArquivos salvos na subetapa 10.2:")
print(f"- {caminho_tbl_regras_registro_tickers_comprados_aporte}")
print(f"- {caminho_base_tickers_comprados_aporte}")
print(f"- {caminho_tbl_resumo_tickers_comprados_aporte}")
print(f"- {caminho_tbl_resumo_tickers_comprados_estrategia}")
print(f"- {caminho_tbl_distribuicao_anual_tickers_comprados}")
print(f"- {caminho_tbl_distribuicao_setorial_tickers_comprados}")
print(f"- {caminho_tbl_auditoria_tickers_comprados_aporte}")
print(f"- {caminho_tbl_inconsistencias_tickers_comprados_aporte}")

print("\nETAPA 10.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 10.2 - REGISTRO DOS TICKERS COMPRADOS POR APORTE

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - aportes realizados 10.1              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_1_base_aportes_realizados.parquet
Entrada - compras capitulação                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_1_base_compras_capitulacao.parquet
Entrada - compras euforia                      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_2_base_compras_euforia.parquet
Entrada - compras aleatórias                   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_

## Etapa 10.3) Registro das Quantidades e Valores Comprados

In [49]:
%%time
# ============================================================
# Etapa 10.3) Registro das Quantidades e Valores Comprados
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 10.3 - REGISTRO DAS QUANTIDADES E VALORES COMPRADOS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_aportes_realizados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="base",
    nome="aportes_realizados",
)

caminho_base_tickers_comprados_aporte = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="base",
    nome="tickers_comprados_aporte",
)

caminho_tbl_resumo_tickers_comprados_aporte = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="resumo_tickers_comprados_aporte",
)

caminho_tbl_regras_registro_quantidades_valores_comprados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="regras_registro_quantidades_valores_comprados",
)

caminho_base_compras_finais = gerar_caminho_arquivo(
    etapa=10,
    subetapa=3,
    tipo_arquivo="base",
    nome="compras_finais",
)

caminho_tbl_resumo_quantidades_valores_aporte = gerar_caminho_arquivo(
    etapa=10,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="resumo_quantidades_valores_aporte",
)

caminho_tbl_resumo_quantidades_valores_estrategia = gerar_caminho_arquivo(
    etapa=10,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="resumo_quantidades_valores_estrategia",
)

caminho_tbl_distribuicao_valores_classe_instrumento = gerar_caminho_arquivo(
    etapa=10,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="distribuicao_valores_classe_instrumento",
)

caminho_tbl_auditoria_quantidades_valores_comprados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="auditoria_quantidades_valores_comprados",
)

caminho_tbl_inconsistencias_quantidades_valores_comprados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="inconsistencias_quantidades_valores_comprados",
)

print(f"Entrada - aportes realizados 10.1                    : {caminho_base_aportes_realizados}")
print(f"Entrada - base de tickers comprados 10.2             : {caminho_base_tickers_comprados_aporte}")
print(f"Entrada - resumo por aporte 10.2                     : {caminho_tbl_resumo_tickers_comprados_aporte}")
print(f"Saída   - regras do registro de quantidades e valores: {caminho_tbl_regras_registro_quantidades_valores_comprados}")
print(f"Saída   - base final de compras                      : {caminho_base_compras_finais}")
print(f"Saída   - resumo por aporte                          : {caminho_tbl_resumo_quantidades_valores_aporte}")
print(f"Saída   - resumo por estratégia                      : {caminho_tbl_resumo_quantidades_valores_estrategia}")
print(f"Saída   - distribuição por classe de instrumento      : {caminho_tbl_distribuicao_valores_classe_instrumento}")
print(f"Saída   - auditoria                                  : {caminho_tbl_auditoria_quantidades_valores_comprados}")
print(f"Saída   - inconsistências                            : {caminho_tbl_inconsistencias_quantidades_valores_comprados}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_aportes_realizados = pd.read_parquet(caminho_base_aportes_realizados)
df_tickers_comprados_aporte = pd.read_parquet(caminho_base_tickers_comprados_aporte)
df_resumo_tickers_comprados_aporte_10_2 = pd.read_parquet(caminho_tbl_resumo_tickers_comprados_aporte)

print(f"Aportes realizados 10.1                    : {df_aportes_realizados.shape[0]:,} linhas x {df_aportes_realizados.shape[1]} colunas")
print(f"Base de tickers comprados 10.2             : {df_tickers_comprados_aporte.shape[0]:,} linhas x {df_tickers_comprados_aporte.shape[1]} colunas")
print(f"Resumo por aporte 10.2                     : {df_resumo_tickers_comprados_aporte_10_2.shape[0]:,} linhas x {df_resumo_tickers_comprados_aporte_10_2.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares, colunas e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares, colunas e parâmetros operacionais...")

TOLERANCIA_MONETARIA_COMPRA = 0.01
TOLERANCIA_PESO_COMPRA = 0.000001

COLUNAS_OBRIGATORIAS_10_2 = [
    "chave_evento_compra",
    "chave_evento_aporte",
    "request_id",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_registro_aporte",
    "origem_registro_compra",
    "origem_calendario",
    "origem_backtest",
    "classe_evento_aporte",
    "classe_instrumento",
    "grupo_registro_carteira",
    "regra_selecao_cesta",
    "versao_cesta",
    "data_sinal",
    "data_aporte_efetiva",
    "ano",
    "mes",
    "ordem_aporte_estrategia",
    "ordem_evento_aporte_global",
    "ordem_compra_no_aporte",
    "ordem_exibicao",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "fonte_benchmark",
    "preco_compra",
    "quantidade_comprada",
    "valor_investido_compra",
    "peso_efetivo_compra",
    "valor_aporte_estrategia",
    "valor_investido_efetivo_aporte",
    "valor_caixa_residual_aporte",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
    "valor_aporte_na_data",
    "valor_orcamento_alvo_ticker",
    "pct_valor_investido_no_aporte",
    "quantidade_inicial_inteira",
    "quantidade_extra_redistribuicao",
    "quantidade_unidades_extra_redistribuicao",
    "tipo_conversao_quantidade",
    "flag_evento_compra",
    "flag_compra_efetiva",
    "flag_ticker_preenchido",
    "flag_issuer_code_preenchido",
    "flag_data_aporte_valida",
    "flag_vinculo_aporte_10_1",
    "flag_preco_compra_valido",
    "flag_quantidade_comprada_valida",
    "flag_valor_investido_positivo",
    "flag_peso_efetivo_valido",
    "flag_recebeu_redistribuicao",
    "flag_compra_fracionaria_equivalente",
    "flag_benchmark_equivalente",
]

COLUNAS_OBRIGATORIAS_APORTES_10_1 = [
    "chave_evento_aporte",
    "request_id",
    "estrategia_id",
    "valor_aporte_estrategia",
    "valor_investido_efetivo_aporte",
    "valor_caixa_residual_aporte",
    "n_tickers_compra_efetivada",
]

COLUNAS_OBRIGATORIAS_RESUMO_10_2 = [
    "chave_evento_aporte",
    "request_id",
    "estrategia_id",
    "n_linhas_instrumentos_aporte",
    "n_tickers_comprados_aporte",
    "valor_total_investido_aporte",
    "peso_total_efetivo_aporte",
    "desvio_n_tickers_vs_10_1",
    "desvio_valor_investido_vs_10_1",
    "flag_contagem_tickers_coerente_10_1",
    "flag_valor_investido_coerente_10_1",
]

COLUNAS_CHAVE_ESTRATEGIA = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_backtest",
    "ordem_exibicao",
]

COLUNAS_CHAVE_APORTE = [
    "chave_evento_aporte",
    "request_id",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_registro_aporte",
    "origem_calendario",
    "origem_backtest",
    "classe_evento_aporte",
    "data_sinal",
    "data_aporte_efetiva",
    "ano",
    "mes",
    "ordem_aporte_estrategia",
    "ordem_evento_aporte_global",
    "ordem_exibicao",
    "valor_aporte_estrategia",
    "valor_investido_efetivo_aporte",
    "valor_caixa_residual_aporte",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
]

COLUNAS_BASE_COMPRAS_FINAIS = COLUNAS_OBRIGATORIAS_10_2 + [
    "flag_preco_quantidade_relevante",
    "valor_calculado_preco_quantidade",
    "desvio_valor_preco_quantidade",
    "flag_valor_reconciliado_preco_quantidade",
    "peso_calculado_sobre_aporte",
    "desvio_peso_efetivo_vs_calculado",
    "flag_peso_reconciliado_com_valor",
    "flag_preco_final_valido",
    "flag_quantidade_final_valida",
    "flag_valor_final_positivo",
    "flag_peso_final_valido",
    "flag_compra_final_valida",
]

COLUNAS_INCONSISTENCIAS = [
    "tipo_inconsistencia",
    "chave_evento_aporte",
    "chave_evento_compra",
    "request_id",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "grupo_controle",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ticker",
    "classe_instrumento",
    "valor_observado",
    "valor_referencia",
    "detalhe",
]

COLUNAS_CONTAGEM_INTEIRA_BASE = [
    "ano",
    "mes",
    "ordem_aporte_estrategia",
    "ordem_evento_aporte_global",
    "ordem_compra_no_aporte",
    "ordem_exibicao",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
]

COLUNAS_CONTAGEM_INTEIRA_RESUMO = [
    "n_linhas_compras_finais_aporte",
    "n_tickers_comprados_aporte",
    "n_empresas_compradas_aporte",
    "n_linhas_acoes_aporte",
    "n_linhas_benchmark_aporte",
    "n_linhas_taxa_referencia_aporte",
    "n_linhas_preco_quantidade_relevante",
    "n_linhas_preco_quantidade_reconciliado",
    "n_linhas_peso_reconciliado",
    "n_linhas_compra_final_valida",
    "n_linhas_compra_final_invalida",
    "n_linhas_preco_final_invalido",
    "n_linhas_quantidade_final_invalida",
    "n_linhas_valor_final_nao_positivo",
    "n_linhas_peso_final_invalido",
    "n_linhas_ticker_ausente",
    "n_linhas_issuer_code_ausente",
    "n_linhas_sem_vinculo_10_1",
    "n_chaves_compra_duplicadas_aporte",
]

COLUNAS_NUMERICAS_BASE = [
    "carteira_id",
    "replica_id",
    "ano",
    "mes",
    "ordem_aporte_estrategia",
    "ordem_evento_aporte_global",
    "ordem_compra_no_aporte",
    "ordem_exibicao",
    "preco_compra",
    "quantidade_comprada",
    "valor_investido_compra",
    "peso_efetivo_compra",
    "valor_aporte_estrategia",
    "valor_investido_efetivo_aporte",
    "valor_caixa_residual_aporte",
    "n_tickers_cesta_equal_weight",
    "n_tickers_compra_efetivada",
    "n_tickers_quantidade_zero",
    "valor_aporte_na_data",
    "valor_orcamento_alvo_ticker",
    "pct_valor_investido_no_aporte",
    "quantidade_inicial_inteira",
    "quantidade_extra_redistribuicao",
    "quantidade_unidades_extra_redistribuicao",
]

COLUNAS_BOOLEANAS_BASE = [
    "flag_evento_compra",
    "flag_compra_efetiva",
    "flag_ticker_preenchido",
    "flag_issuer_code_preenchido",
    "flag_data_aporte_valida",
    "flag_vinculo_aporte_10_1",
    "flag_preco_compra_valido",
    "flag_quantidade_comprada_valida",
    "flag_valor_investido_positivo",
    "flag_peso_efetivo_valido",
    "flag_recebeu_redistribuicao",
    "flag_compra_fracionaria_equivalente",
    "flag_benchmark_equivalente",
]

COLUNAS_TEXTO_BASE = [
    "chave_evento_compra",
    "chave_evento_aporte",
    "request_id",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "origem_registro_aporte",
    "origem_registro_compra",
    "origem_calendario",
    "origem_backtest",
    "classe_evento_aporte",
    "classe_instrumento",
    "grupo_registro_carteira",
    "regra_selecao_cesta",
    "versao_cesta",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "fonte_benchmark",
    "tipo_conversao_quantidade",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias, nome_dataframe):
    colunas_ausentes = [coluna for coluna in colunas_obrigatorias if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(
            f"Colunas obrigatórias ausentes em {nome_dataframe}: {colunas_ausentes}"
        )

def selecionar_colunas_com_padrao(df, colunas):
    df_saida = df.copy()
    for coluna in colunas:
        if coluna not in df_saida.columns:
            df_saida[coluna] = pd.NA
    return df_saida[colunas].copy()

def normalizar_texto(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()
    return df

def converter_numerico(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    return df

def converter_booleano(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = df[coluna].fillna(False).astype(bool)
    return df

def converter_contagens_inteiras(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0).round(0).astype(int)
    return df

def registrar_inconsistencia(lista, linha, tipo_inconsistencia, valor_observado, valor_referencia, detalhe):
    lista.append({
        "tipo_inconsistencia": tipo_inconsistencia,
        "chave_evento_aporte": linha.get("chave_evento_aporte"),
        "chave_evento_compra": linha.get("chave_evento_compra"),
        "request_id": linha.get("request_id"),
        "chave_estrategia": linha.get("chave_estrategia"),
        "estrategia_id": linha.get("estrategia_id"),
        "nome_exibicao_estrategia": linha.get("nome_exibicao_estrategia"),
        "grupo_controle": linha.get("grupo_controle"),
        "categoria_estrategia": linha.get("categoria_estrategia"),
        "subcategoria_estrategia": linha.get("subcategoria_estrategia"),
        "ticker": linha.get("ticker"),
        "classe_instrumento": linha.get("classe_instrumento"),
        "valor_observado": valor_observado,
        "valor_referencia": valor_referencia,
        "detalhe": detalhe,
    })

def contar_false(serie):
    return int((~serie.fillna(False).astype(bool)).sum())

print(f"Tolerância monetária compra × quantidade      : {TOLERANCIA_MONETARIA_COMPRA}")
print(f"Tolerância de peso efetivo                    : {TOLERANCIA_PESO_COMPRA}")
print("OK")

# ============================================================
# 5) Preparação e validação estrutural dos inputs
# ============================================================

print("\n[5/10] Preparação e validação estrutural dos inputs...")

validar_colunas_obrigatorias(df_tickers_comprados_aporte, COLUNAS_OBRIGATORIAS_10_2, "10_2_base_tickers_comprados_aporte")
validar_colunas_obrigatorias(df_aportes_realizados, COLUNAS_OBRIGATORIAS_APORTES_10_1, "10_1_base_aportes_realizados")
validar_colunas_obrigatorias(df_resumo_tickers_comprados_aporte_10_2, COLUNAS_OBRIGATORIAS_RESUMO_10_2, "10_2_tbl_resumo_tickers_comprados_aporte")

if df_tickers_comprados_aporte["chave_evento_compra"].duplicated().any():
    n_chaves_duplicadas = int(df_tickers_comprados_aporte["chave_evento_compra"].duplicated(keep=False).sum())
    raise ValueError(f"A base 10.2 possui {n_chaves_duplicadas} linhas com chave_evento_compra duplicada.")

if df_aportes_realizados.duplicated(["estrategia_id", "request_id"], keep=False).any():
    n_aportes_duplicados = int(df_aportes_realizados.duplicated(["estrategia_id", "request_id"], keep=False).sum())
    raise ValueError(f"A base 10.1 possui {n_aportes_duplicados} linhas duplicadas por estrategia_id + request_id.")

if df_resumo_tickers_comprados_aporte_10_2.duplicated(["estrategia_id", "request_id"], keep=False).any():
    n_resumos_duplicados = int(df_resumo_tickers_comprados_aporte_10_2.duplicated(["estrategia_id", "request_id"], keep=False).sum())
    raise ValueError(f"O resumo 10.2 possui {n_resumos_duplicados} linhas duplicadas por estrategia_id + request_id.")

print(f"Colunas obrigatórias da 10.2 validadas       : {len(COLUNAS_OBRIGATORIAS_10_2):,}")
print(f"Chaves de compra duplicadas na 10.2          : 0")
print(f"Duplicidades estrategia_id + request_id 10.1 : 0")
print(f"Duplicidades estrategia_id + request_id 10.2 : 0")
print("OK")

# ============================================================
# 6) Construção da base final de compras
# ============================================================

print("\n[6/10] Construção da base final de compras...")

df_base_compras_finais = selecionar_colunas_com_padrao(
    df_tickers_comprados_aporte,
    COLUNAS_OBRIGATORIAS_10_2,
)

df_base_compras_finais = normalizar_texto(df_base_compras_finais, COLUNAS_TEXTO_BASE)
df_base_compras_finais = converter_numerico(df_base_compras_finais, COLUNAS_NUMERICAS_BASE)
df_base_compras_finais = converter_booleano(df_base_compras_finais, COLUNAS_BOOLEANAS_BASE)
df_base_compras_finais["data_sinal"] = pd.to_datetime(df_base_compras_finais["data_sinal"], errors="coerce")
df_base_compras_finais["data_aporte_efetiva"] = pd.to_datetime(df_base_compras_finais["data_aporte_efetiva"], errors="coerce")

mascara_indice_ibovespa = df_base_compras_finais["ticker"].astype("string").str.upper().eq("^BVSP")
n_instrumentos_indice_ibovespa = int(mascara_indice_ibovespa.sum())

df_base_compras_finais.loc[mascara_indice_ibovespa, "classe_instrumento"] = "benchmark"
df_base_compras_finais.loc[mascara_indice_ibovespa, "grupo_registro_carteira"] = "benchmark_equivalente"
df_base_compras_finais.loc[mascara_indice_ibovespa, "flag_benchmark_equivalente"] = True
df_base_compras_finais.loc[
    mascara_indice_ibovespa & df_base_compras_finais["fonte_benchmark"].isna(),
    "fonte_benchmark",
] = "ibov_padronizada"

df_base_compras_finais["flag_preco_quantidade_relevante"] = df_base_compras_finais["classe_instrumento"].isin(["acao", "benchmark"])

df_base_compras_finais["valor_calculado_preco_quantidade"] = np.where(
    df_base_compras_finais["flag_preco_quantidade_relevante"],
    df_base_compras_finais["preco_compra"] * df_base_compras_finais["quantidade_comprada"],
    df_base_compras_finais["valor_investido_compra"],
)

df_base_compras_finais["desvio_valor_preco_quantidade"] = (
    df_base_compras_finais["valor_investido_compra"]
    - df_base_compras_finais["valor_calculado_preco_quantidade"]
)

df_base_compras_finais["flag_valor_reconciliado_preco_quantidade"] = (
    (~df_base_compras_finais["flag_preco_quantidade_relevante"])
    | (df_base_compras_finais["desvio_valor_preco_quantidade"].abs() <= TOLERANCIA_MONETARIA_COMPRA)
)

df_base_compras_finais["peso_calculado_sobre_aporte"] = np.where(
    df_base_compras_finais["valor_aporte_estrategia"].fillna(0.0).abs() > 0.0,
    df_base_compras_finais["valor_investido_compra"] / df_base_compras_finais["valor_aporte_estrategia"],
    np.nan,
)

df_base_compras_finais["desvio_peso_efetivo_vs_calculado"] = (
    df_base_compras_finais["peso_efetivo_compra"]
    - df_base_compras_finais["peso_calculado_sobre_aporte"]
)

df_base_compras_finais["flag_peso_reconciliado_com_valor"] = (
    df_base_compras_finais["desvio_peso_efetivo_vs_calculado"].abs() <= TOLERANCIA_PESO_COMPRA
)

df_base_compras_finais["flag_preco_final_valido"] = (
    (
        df_base_compras_finais["flag_preco_quantidade_relevante"]
        & (df_base_compras_finais["preco_compra"].fillna(0.0) > 0.0)
    )
    | (df_base_compras_finais["classe_instrumento"] == "taxa_referencia")
)

df_base_compras_finais["flag_quantidade_final_valida"] = (
    (
        df_base_compras_finais["flag_preco_quantidade_relevante"]
        & (df_base_compras_finais["quantidade_comprada"].fillna(0.0) > 0.0)
    )
    | (df_base_compras_finais["classe_instrumento"] == "taxa_referencia")
)

df_base_compras_finais["flag_valor_final_positivo"] = (
    df_base_compras_finais["valor_investido_compra"].fillna(0.0) > 0.0
)

df_base_compras_finais["flag_peso_final_valido"] = (
    df_base_compras_finais["peso_efetivo_compra"].notna()
    & (df_base_compras_finais["peso_efetivo_compra"] >= 0.0)
    & (df_base_compras_finais["peso_efetivo_compra"] <= 1.0 + TOLERANCIA_PESO_COMPRA)
)

df_base_compras_finais["flag_compra_final_valida"] = (
    df_base_compras_finais["flag_compra_efetiva"]
    & df_base_compras_finais["flag_ticker_preenchido"]
    & df_base_compras_finais["flag_issuer_code_preenchido"]
    & df_base_compras_finais["flag_data_aporte_valida"]
    & df_base_compras_finais["flag_vinculo_aporte_10_1"]
    & df_base_compras_finais["flag_preco_final_valido"]
    & df_base_compras_finais["flag_quantidade_final_valida"]
    & df_base_compras_finais["flag_valor_final_positivo"]
    & df_base_compras_finais["flag_peso_final_valido"]
    & df_base_compras_finais["flag_valor_reconciliado_preco_quantidade"]
    & df_base_compras_finais["flag_peso_reconciliado_com_valor"]
)

df_base_compras_finais = converter_contagens_inteiras(df_base_compras_finais, COLUNAS_CONTAGEM_INTEIRA_BASE)

df_base_compras_finais = df_base_compras_finais.sort_values(
    [
        "data_aporte_efetiva",
        "ordem_exibicao",
        "estrategia_id",
        "ordem_aporte_estrategia",
        "ordem_compra_no_aporte",
        "ticker",
    ],
    na_position="last",
).reset_index(drop=True)

df_base_compras_finais = selecionar_colunas_com_padrao(
    df_base_compras_finais,
    COLUNAS_BASE_COMPRAS_FINAIS,
)

print(f"Base final de compras                         : {df_base_compras_finais.shape[0]:,} linhas x {df_base_compras_finais.shape[1]} colunas")
print(f"Instrumentos ^BVSP classificados como benchmark: {n_instrumentos_indice_ibovespa:,}")
print(f"Compras finais válidas                        : {int(df_base_compras_finais['flag_compra_final_valida'].sum()):,}")
print(f"Compras finais inválidas                      : {int((~df_base_compras_finais['flag_compra_final_valida']).sum()):,}")
print(f"Linhas com preço × quantidade reconciliado    : {int(df_base_compras_finais['flag_valor_reconciliado_preco_quantidade'].sum()):,}")
print(f"Linhas com peso efetivo reconciliado          : {int(df_base_compras_finais['flag_peso_reconciliado_com_valor'].sum()):,}")
print("OK")

# ============================================================
# 7) Construção das regras formais e resumos consolidados
# ============================================================

print("\n[7/10] Construção das regras formais e resumos consolidados...")

validar_colunas_obrigatorias(df_base_compras_finais, COLUNAS_CHAVE_APORTE, "10_3_base_compras_finais")
validar_colunas_obrigatorias(df_base_compras_finais, COLUNAS_CHAVE_ESTRATEGIA, "10_3_base_compras_finais")

df_regras_registro_quantidades_valores_comprados = pd.DataFrame([
    {
        "ordem_regra": 1,
        "regra_id": "RQV1",
        "escopo": "fonte",
        "regra_operacional": "usar_a_base_10_2_como_fonte_primaria_de_compras",
        "detalhe": "A base final de compras é derivada da composição validada na etapa 10.2, preservando vínculo com aporte, estratégia, ticker e classificação setorial.",
    },
    {
        "ordem_regra": 2,
        "regra_id": "RQV2",
        "escopo": "preco_quantidade_valor",
        "regra_operacional": "reconciliar_valor_investido_com_preco_vezes_quantidade",
        "detalhe": "Para ações e benchmark equivalente, o valor investido deve ser compatível com preço de compra multiplicado pela quantidade comprada.",
    },
    {
        "ordem_regra": 3,
        "regra_id": "RQV3",
        "escopo": "peso",
        "regra_operacional": "reconciliar_peso_efetivo_com_valor_investido_sobre_aporte",
        "detalhe": "O peso efetivo da compra deve ser compatível com a razão entre valor investido na linha e valor formal do aporte.",
    },
    {
        "ordem_regra": 4,
        "regra_id": "RQV4",
        "escopo": "indice",
        "regra_operacional": "classificar_exposicoes_ao_ibovespa_como_benchmark",
        "detalhe": "Linhas com ticker ^BVSP são tratadas como instrumentos equivalentes de benchmark, e não como ações individuais, preservando a leitura econômica da composição.",
    },
    {
        "ordem_regra": 5,
        "regra_id": "RQV5",
        "escopo": "benchmark_cdi",
        "regra_operacional": "tratar_cdi_only_como_taxa_referencia_sem_preco_e_quantidade",
        "detalhe": "A carteira CDI-only é mantida como instrumento de taxa de referência; preço e quantidade não são exigidos para esse caso.",
    },
    {
        "ordem_regra": 6,
        "regra_id": "RQV6",
        "escopo": "auditoria",
        "regra_operacional": "reconciliar_a_base_final_com_10_1_e_10_2",
        "detalhe": "A auditoria compara contagem de instrumentos, quantidade de tickers e valor investido contra as bases de aportes e composição já validadas.",
    },
])

df_resumo_quantidades_valores_aporte = (
    df_base_compras_finais
    .groupby(COLUNAS_CHAVE_APORTE, dropna=False)
    .agg(
        n_linhas_compras_finais_aporte=("chave_evento_compra", "nunique"),
        n_tickers_comprados_aporte=("ticker", "nunique"),
        n_empresas_compradas_aporte=("issuer_code", "nunique"),
        n_linhas_acoes_aporte=("classe_instrumento", lambda x: int((x == "acao").sum())),
        n_linhas_benchmark_aporte=("classe_instrumento", lambda x: int((x == "benchmark").sum())),
        n_linhas_taxa_referencia_aporte=("classe_instrumento", lambda x: int((x == "taxa_referencia").sum())),
        n_linhas_preco_quantidade_relevante=("flag_preco_quantidade_relevante", "sum"),
        n_linhas_preco_quantidade_reconciliado=("flag_valor_reconciliado_preco_quantidade", "sum"),
        n_linhas_peso_reconciliado=("flag_peso_reconciliado_com_valor", "sum"),
        n_linhas_compra_final_valida=("flag_compra_final_valida", "sum"),
        valor_total_investido_aporte=("valor_investido_compra", "sum"),
        valor_total_calculado_preco_quantidade=("valor_calculado_preco_quantidade", "sum"),
        desvio_total_preco_quantidade=("desvio_valor_preco_quantidade", "sum"),
        peso_total_efetivo_aporte=("peso_efetivo_compra", "sum"),
        peso_total_calculado_sobre_aporte=("peso_calculado_sobre_aporte", "sum"),
        desvio_total_peso_efetivo=("desvio_peso_efetivo_vs_calculado", "sum"),
        menor_preco_compra_aporte=("preco_compra", "min"),
        maior_preco_compra_aporte=("preco_compra", "max"),
        menor_quantidade_comprada_aporte=("quantidade_comprada", "min"),
        maior_quantidade_comprada_aporte=("quantidade_comprada", "max"),
        menor_valor_compra_aporte=("valor_investido_compra", "min"),
        maior_valor_compra_aporte=("valor_investido_compra", "max"),
        n_linhas_preco_final_invalido=("flag_preco_final_valido", contar_false),
        n_linhas_quantidade_final_invalida=("flag_quantidade_final_valida", contar_false),
        n_linhas_valor_final_nao_positivo=("flag_valor_final_positivo", contar_false),
        n_linhas_peso_final_invalido=("flag_peso_final_valido", contar_false),
        n_linhas_ticker_ausente=("flag_ticker_preenchido", contar_false),
        n_linhas_issuer_code_ausente=("flag_issuer_code_preenchido", contar_false),
        n_linhas_sem_vinculo_10_1=("flag_vinculo_aporte_10_1", contar_false),
    )
    .reset_index()
)

df_resumo_quantidades_valores_aporte["n_linhas_compra_final_invalida"] = (
    df_resumo_quantidades_valores_aporte["n_linhas_compras_finais_aporte"]
    - df_resumo_quantidades_valores_aporte["n_linhas_compra_final_valida"]
)

df_chaves_compra_duplicadas_aporte = (
    df_base_compras_finais[
        df_base_compras_finais.duplicated(["estrategia_id", "request_id", "chave_evento_compra"], keep=False)
    ]
    .groupby(["estrategia_id", "request_id"], dropna=False)
    .agg(n_chaves_compra_duplicadas_aporte=("chave_evento_compra", "nunique"))
    .reset_index()
)

df_resumo_quantidades_valores_aporte = df_resumo_quantidades_valores_aporte.merge(
    df_chaves_compra_duplicadas_aporte,
    on=["estrategia_id", "request_id"],
    how="left",
)

df_resumo_quantidades_valores_aporte["n_chaves_compra_duplicadas_aporte"] = pd.to_numeric(
    df_resumo_quantidades_valores_aporte["n_chaves_compra_duplicadas_aporte"],
    errors="coerce",
).fillna(0).astype(int)

df_resumo_quantidades_valores_aporte = converter_contagens_inteiras(
    df_resumo_quantidades_valores_aporte,
    COLUNAS_CONTAGEM_INTEIRA_RESUMO,
)

validar_colunas_obrigatorias(df_resumo_quantidades_valores_aporte, COLUNAS_CHAVE_ESTRATEGIA, "10_3_tbl_resumo_quantidades_valores_aporte")

df_resumo_quantidades_valores_estrategia = (
    df_resumo_quantidades_valores_aporte
    .groupby(COLUNAS_CHAVE_ESTRATEGIA, dropna=False)
    .agg(
        n_aportes_com_compras_finais=("request_id", "nunique"),
        primeira_data_aporte=("data_aporte_efetiva", "min"),
        ultima_data_aporte=("data_aporte_efetiva", "max"),
        n_linhas_compras_finais=("n_linhas_compras_finais_aporte", "sum"),
        n_tickers_comprados_total=("n_tickers_comprados_aporte", "sum"),
        media_tickers_por_aporte=("n_tickers_comprados_aporte", "mean"),
        minimo_tickers_por_aporte=("n_tickers_comprados_aporte", "min"),
        maximo_tickers_por_aporte=("n_tickers_comprados_aporte", "max"),
        n_linhas_acoes=("n_linhas_acoes_aporte", "sum"),
        n_linhas_benchmark=("n_linhas_benchmark_aporte", "sum"),
        n_linhas_taxa_referencia=("n_linhas_taxa_referencia_aporte", "sum"),
        valor_total_investido=("valor_total_investido_aporte", "sum"),
        valor_total_calculado_preco_quantidade=("valor_total_calculado_preco_quantidade", "sum"),
        desvio_total_preco_quantidade=("desvio_total_preco_quantidade", "sum"),
        peso_total_efetivo=("peso_total_efetivo_aporte", "sum"),
        peso_total_calculado=("peso_total_calculado_sobre_aporte", "sum"),
        desvio_total_peso_efetivo=("desvio_total_peso_efetivo", "sum"),
        n_linhas_compra_final_invalida=("n_linhas_compra_final_invalida", "sum"),
        n_linhas_preco_final_invalido=("n_linhas_preco_final_invalido", "sum"),
        n_linhas_quantidade_final_invalida=("n_linhas_quantidade_final_invalida", "sum"),
        n_linhas_valor_final_nao_positivo=("n_linhas_valor_final_nao_positivo", "sum"),
        n_linhas_peso_final_invalido=("n_linhas_peso_final_invalido", "sum"),
        n_linhas_sem_vinculo_10_1=("n_linhas_sem_vinculo_10_1", "sum"),
        n_chaves_compra_duplicadas=("n_chaves_compra_duplicadas_aporte", "sum"),
    )
    .reset_index()
    .sort_values(["ordem_exibicao", "estrategia_id"])
    .reset_index(drop=True)
)

df_distintos_estrategia = (
    df_base_compras_finais
    .groupby("estrategia_id", dropna=False)
    .agg(
        n_tickers_distintos=("ticker", "nunique"),
        n_empresas_distintas=("issuer_code", "nunique"),
        n_setores_distintos=("setor", "nunique"),
        n_subsetores_distintos=("subsetor", "nunique"),
        n_segmentos_distintos=("segmento", "nunique"),
    )
    .reset_index()
)

df_resumo_quantidades_valores_estrategia = df_resumo_quantidades_valores_estrategia.merge(
    df_distintos_estrategia,
    on="estrategia_id",
    how="left",
)

df_distribuicao_valores_classe_instrumento = (
    df_base_compras_finais
    .groupby(COLUNAS_CHAVE_ESTRATEGIA + ["classe_instrumento"], dropna=False)
    .agg(
        n_linhas_compras=("chave_evento_compra", "nunique"),
        n_aportes_com_presenca=("request_id", "nunique"),
        n_tickers_distintos=("ticker", "nunique"),
        valor_total_investido=("valor_investido_compra", "sum"),
        peso_total_efetivo=("peso_efetivo_compra", "sum"),
    )
    .reset_index()
    .sort_values(["ordem_exibicao", "estrategia_id", "classe_instrumento"])
    .reset_index(drop=True)
)

print(f"Regras formais geradas                       : {df_regras_registro_quantidades_valores_comprados.shape[0]:,}")
print(f"Resumo por aporte gerado                    : {df_resumo_quantidades_valores_aporte.shape[0]:,}")
print(f"Resumo por estratégia gerado                : {df_resumo_quantidades_valores_estrategia.shape[0]:,}")
print(f"Distribuição por classe gerada              : {df_distribuicao_valores_classe_instrumento.shape[0]:,}")
print("OK")

# ============================================================
# 8) Construção da auditoria e das inconsistências
# ============================================================

print("\n[8/10] Construção da auditoria e das inconsistências...")

df_aportes_lookup = selecionar_colunas_com_padrao(
    df_aportes_realizados,
    [
        "chave_evento_aporte",
        "request_id",
        "estrategia_id",
        "valor_aporte_estrategia",
        "valor_investido_efetivo_aporte",
        "valor_caixa_residual_aporte",
        "n_tickers_compra_efetivada",
    ],
)

df_resumo_10_2_lookup = selecionar_colunas_com_padrao(
    df_resumo_tickers_comprados_aporte_10_2,
    [
        "chave_evento_aporte",
        "request_id",
        "estrategia_id",
        "n_linhas_instrumentos_aporte",
        "n_tickers_comprados_aporte",
        "valor_total_investido_aporte",
        "peso_total_efetivo_aporte",
        "desvio_n_tickers_vs_10_1",
        "desvio_valor_investido_vs_10_1",
        "flag_contagem_tickers_coerente_10_1",
        "flag_valor_investido_coerente_10_1",
    ],
).rename(columns={
    "n_linhas_instrumentos_aporte": "n_linhas_instrumentos_aporte_10_2",
    "n_tickers_comprados_aporte": "n_tickers_comprados_aporte_10_2",
    "valor_total_investido_aporte": "valor_total_investido_aporte_10_2",
    "peso_total_efetivo_aporte": "peso_total_efetivo_aporte_10_2",
    "desvio_n_tickers_vs_10_1": "desvio_n_tickers_vs_10_1_10_2",
    "desvio_valor_investido_vs_10_1": "desvio_valor_investido_vs_10_1_10_2",
    "flag_contagem_tickers_coerente_10_1": "flag_contagem_tickers_coerente_10_1_10_2",
    "flag_valor_investido_coerente_10_1": "flag_valor_investido_coerente_10_1_10_2",
})

df_auditoria_quantidades_valores_comprados = df_resumo_quantidades_valores_aporte.merge(
    df_aportes_lookup,
    on=["estrategia_id", "request_id", "chave_evento_aporte"],
    how="left",
    suffixes=("", "_10_1"),
)

df_auditoria_quantidades_valores_comprados = df_auditoria_quantidades_valores_comprados.merge(
    df_resumo_10_2_lookup,
    on=["estrategia_id", "request_id", "chave_evento_aporte"],
    how="left",
)

for coluna in [
    "valor_aporte_estrategia_10_1",
    "valor_investido_efetivo_aporte_10_1",
    "valor_caixa_residual_aporte_10_1",
    "n_tickers_compra_efetivada_10_1",
    "n_linhas_instrumentos_aporte_10_2",
    "n_tickers_comprados_aporte_10_2",
    "valor_total_investido_aporte_10_2",
    "peso_total_efetivo_aporte_10_2",
]:
    if coluna in df_auditoria_quantidades_valores_comprados.columns:
        df_auditoria_quantidades_valores_comprados[coluna] = pd.to_numeric(
            df_auditoria_quantidades_valores_comprados[coluna],
            errors="coerce",
        )

for coluna in [
    "flag_contagem_tickers_coerente_10_1_10_2",
    "flag_valor_investido_coerente_10_1_10_2",
]:
    if coluna in df_auditoria_quantidades_valores_comprados.columns:
        df_auditoria_quantidades_valores_comprados[coluna] = (
            df_auditoria_quantidades_valores_comprados[coluna].fillna(False).astype(bool)
        )

df_auditoria_quantidades_valores_comprados["desvio_n_linhas_vs_10_2"] = (
    df_auditoria_quantidades_valores_comprados["n_linhas_compras_finais_aporte"]
    - df_auditoria_quantidades_valores_comprados["n_linhas_instrumentos_aporte_10_2"]
)

df_auditoria_quantidades_valores_comprados["desvio_n_tickers_vs_10_2"] = (
    df_auditoria_quantidades_valores_comprados["n_tickers_comprados_aporte"]
    - df_auditoria_quantidades_valores_comprados["n_tickers_comprados_aporte_10_2"]
)

df_auditoria_quantidades_valores_comprados["desvio_valor_investido_vs_10_2"] = (
    df_auditoria_quantidades_valores_comprados["valor_total_investido_aporte"]
    - df_auditoria_quantidades_valores_comprados["valor_total_investido_aporte_10_2"]
)

df_auditoria_quantidades_valores_comprados["desvio_valor_investido_vs_10_1"] = (
    df_auditoria_quantidades_valores_comprados["valor_total_investido_aporte"]
    - df_auditoria_quantidades_valores_comprados["valor_investido_efetivo_aporte_10_1"]
)

df_auditoria_quantidades_valores_comprados["desvio_n_tickers_vs_10_1"] = (
    df_auditoria_quantidades_valores_comprados["n_tickers_comprados_aporte"]
    - df_auditoria_quantidades_valores_comprados["n_tickers_compra_efetivada_10_1"]
)

df_auditoria_quantidades_valores_comprados["flag_contagem_linhas_coerente_10_2"] = (
    df_auditoria_quantidades_valores_comprados["desvio_n_linhas_vs_10_2"].fillna(np.inf).abs() == 0
)

df_auditoria_quantidades_valores_comprados["flag_contagem_tickers_coerente_10_2"] = (
    df_auditoria_quantidades_valores_comprados["desvio_n_tickers_vs_10_2"].fillna(np.inf).abs() == 0
)

df_auditoria_quantidades_valores_comprados["flag_valor_investido_coerente_10_2"] = (
    df_auditoria_quantidades_valores_comprados["desvio_valor_investido_vs_10_2"].fillna(np.inf).abs() <= TOLERANCIA_MONETARIA_COMPRA
)

df_auditoria_quantidades_valores_comprados["flag_valor_investido_coerente_10_1"] = (
    df_auditoria_quantidades_valores_comprados["desvio_valor_investido_vs_10_1"].fillna(np.inf).abs() <= TOLERANCIA_MONETARIA_COMPRA
)

df_auditoria_quantidades_valores_comprados["flag_contagem_tickers_coerente_10_1"] = (
    df_auditoria_quantidades_valores_comprados["desvio_n_tickers_vs_10_1"].fillna(np.inf).abs() == 0
)

df_auditoria_quantidades_valores_comprados["flag_soma_preco_quantidade_coerente"] = (
    df_auditoria_quantidades_valores_comprados["desvio_total_preco_quantidade"].fillna(np.inf).abs() <= TOLERANCIA_MONETARIA_COMPRA
)

df_auditoria_quantidades_valores_comprados["flag_soma_peso_coerente"] = (
    df_auditoria_quantidades_valores_comprados["desvio_total_peso_efetivo"].fillna(np.inf).abs() <= TOLERANCIA_PESO_COMPRA
)

df_auditoria_quantidades_valores_comprados["flag_sem_inconsistencia_quantidades_valores"] = (
    df_auditoria_quantidades_valores_comprados["flag_contagem_linhas_coerente_10_2"]
    & df_auditoria_quantidades_valores_comprados["flag_contagem_tickers_coerente_10_2"]
    & df_auditoria_quantidades_valores_comprados["flag_valor_investido_coerente_10_2"]
    & df_auditoria_quantidades_valores_comprados["flag_valor_investido_coerente_10_1"]
    & df_auditoria_quantidades_valores_comprados["flag_contagem_tickers_coerente_10_1"]
    & df_auditoria_quantidades_valores_comprados["flag_soma_preco_quantidade_coerente"]
    & df_auditoria_quantidades_valores_comprados["flag_soma_peso_coerente"]
    & (df_auditoria_quantidades_valores_comprados["n_linhas_compra_final_invalida"] == 0)
    & (df_auditoria_quantidades_valores_comprados["n_chaves_compra_duplicadas_aporte"] == 0)
)

lista_inconsistencias = []

for _, linha in df_auditoria_quantidades_valores_comprados.iterrows():
    if not bool(linha["flag_contagem_linhas_coerente_10_2"]):
        registrar_inconsistencia(lista_inconsistencias, linha, "contagem_linhas_divergente_da_10_2", linha["n_linhas_compras_finais_aporte"], linha["n_linhas_instrumentos_aporte_10_2"], "A contagem de linhas da base final diverge da composição validada na etapa 10.2.")
    if not bool(linha["flag_contagem_tickers_coerente_10_2"]):
        registrar_inconsistencia(lista_inconsistencias, linha, "contagem_tickers_divergente_da_10_2", linha["n_tickers_comprados_aporte"], linha["n_tickers_comprados_aporte_10_2"], "A contagem de tickers da base final diverge do resumo por aporte da etapa 10.2.")
    if not bool(linha["flag_valor_investido_coerente_10_2"]):
        registrar_inconsistencia(lista_inconsistencias, linha, "valor_investido_divergente_da_10_2", linha["valor_total_investido_aporte"], linha["valor_total_investido_aporte_10_2"], "O valor investido da base final diverge do valor consolidado na etapa 10.2.")
    if not bool(linha["flag_valor_investido_coerente_10_1"]):
        registrar_inconsistencia(lista_inconsistencias, linha, "valor_investido_divergente_da_10_1", linha["valor_total_investido_aporte"], linha["valor_investido_efetivo_aporte_10_1"], "O valor investido da base final diverge do valor consolidado na etapa 10.1.")
    if not bool(linha["flag_contagem_tickers_coerente_10_1"]):
        registrar_inconsistencia(lista_inconsistencias, linha, "contagem_tickers_divergente_da_10_1", linha["n_tickers_comprados_aporte"], linha["n_tickers_compra_efetivada_10_1"], "A contagem de tickers da base final diverge da contagem de compras efetivadas da etapa 10.1.")
    if not bool(linha["flag_soma_preco_quantidade_coerente"]):
        registrar_inconsistencia(lista_inconsistencias, linha, "preco_vezes_quantidade_divergente", linha["valor_total_investido_aporte"], linha["valor_total_calculado_preco_quantidade"], "A soma de preço vezes quantidade não reconcilia com o valor investido do aporte.")
    if not bool(linha["flag_soma_peso_coerente"]):
        registrar_inconsistencia(lista_inconsistencias, linha, "peso_efetivo_divergente_do_valor", linha["peso_total_efetivo_aporte"], linha["peso_total_calculado_sobre_aporte"], "A soma de pesos efetivos não reconcilia com valor investido dividido por valor do aporte.")
    if linha["n_linhas_compra_final_invalida"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "compra_final_invalida", linha["n_linhas_compra_final_invalida"], 0, "Há linhas de compra final com falha em preço, quantidade, valor, peso, vínculo ou identificação.")
    if linha["n_chaves_compra_duplicadas_aporte"] > 0:
        registrar_inconsistencia(lista_inconsistencias, linha, "chave_evento_compra_duplicada", linha["n_chaves_compra_duplicadas_aporte"], 0, "Há chaves de compra duplicadas dentro do mesmo aporte.")

if len(lista_inconsistencias) == 0:
    df_inconsistencias_quantidades_valores_comprados = pd.DataFrame(columns=COLUNAS_INCONSISTENCIAS)
else:
    df_inconsistencias_quantidades_valores_comprados = pd.DataFrame(lista_inconsistencias)[COLUNAS_INCONSISTENCIAS]

n_inconsistencias_quantidades_valores = df_inconsistencias_quantidades_valores_comprados.shape[0]
n_aportes_com_inconsistencia = int((~df_auditoria_quantidades_valores_comprados["flag_sem_inconsistencia_quantidades_valores"]).sum())
n_linhas_compra_final_invalida = int(df_resumo_quantidades_valores_aporte["n_linhas_compra_final_invalida"].sum())
n_divergencias_valor_10_1 = int((~df_auditoria_quantidades_valores_comprados["flag_valor_investido_coerente_10_1"]).sum())
n_divergencias_valor_10_2 = int((~df_auditoria_quantidades_valores_comprados["flag_valor_investido_coerente_10_2"]).sum())
n_divergencias_preco_quantidade = int((~df_auditoria_quantidades_valores_comprados["flag_soma_preco_quantidade_coerente"]).sum())
n_divergencias_peso = int((~df_auditoria_quantidades_valores_comprados["flag_soma_peso_coerente"]).sum())

print(f"Inconsistências identificadas               : {n_inconsistencias_quantidades_valores:,}")
print(f"Aportes com inconsistência                  : {n_aportes_com_inconsistencia:,}")
print(f"Linhas de compra final inválidas            : {n_linhas_compra_final_invalida:,}")
print(f"Divergências de valor contra a 10.1         : {n_divergencias_valor_10_1:,}")
print(f"Divergências de valor contra a 10.2         : {n_divergencias_valor_10_2:,}")
print(f"Divergências de preço × quantidade          : {n_divergencias_preco_quantidade:,}")
print(f"Divergências de peso efetivo                : {n_divergencias_peso:,}")
print("OK")

# ============================================================
# 9) Organização final dos outputs
# ============================================================

print("\n[9/10] Organização final dos outputs...")

df_resumo_quantidades_valores_estrategia = converter_contagens_inteiras(
    df_resumo_quantidades_valores_estrategia,
    [
        "n_aportes_com_compras_finais",
        "n_linhas_compras_finais",
        "n_tickers_comprados_total",
        "minimo_tickers_por_aporte",
        "maximo_tickers_por_aporte",
        "n_linhas_acoes",
        "n_linhas_benchmark",
        "n_linhas_taxa_referencia",
        "n_linhas_compra_final_invalida",
        "n_linhas_preco_final_invalido",
        "n_linhas_quantidade_final_invalida",
        "n_linhas_valor_final_nao_positivo",
        "n_linhas_peso_final_invalido",
        "n_linhas_sem_vinculo_10_1",
        "n_chaves_compra_duplicadas",
        "n_tickers_distintos",
        "n_empresas_distintas",
        "n_setores_distintos",
        "n_subsetores_distintos",
        "n_segmentos_distintos",
    ],
)

df_distribuicao_valores_classe_instrumento = converter_contagens_inteiras(
    df_distribuicao_valores_classe_instrumento,
    [
        "n_linhas_compras",
        "n_aportes_com_presenca",
        "n_tickers_distintos",
    ],
)

df_auditoria_quantidades_valores_comprados = converter_contagens_inteiras(
    df_auditoria_quantidades_valores_comprados,
    COLUNAS_CONTAGEM_INTEIRA_RESUMO
    + [
        "n_linhas_instrumentos_aporte_10_2",
        "n_tickers_comprados_aporte_10_2",
        "n_tickers_compra_efetivada_10_1",
    ],
)

print(f"Base final de compras organizada             : {df_base_compras_finais.shape[0]:,} linhas")
print(f"Auditoria organizada                         : {df_auditoria_quantidades_valores_comprados.shape[0]:,} linhas")
print(f"Tabela de inconsistências organizada         : {df_inconsistencias_quantidades_valores_comprados.shape[0]:,} linhas")
print("OK")

# ============================================================
# 10) Salvamento dos outputs e validação final
# ============================================================

print("\n[10/10] Salvamento dos outputs e validação final...")

salvar_dataframe(df_regras_registro_quantidades_valores_comprados, caminho_tbl_regras_registro_quantidades_valores_comprados, index=False)
salvar_dataframe(df_base_compras_finais, caminho_base_compras_finais, index=False)
salvar_dataframe(df_resumo_quantidades_valores_aporte, caminho_tbl_resumo_quantidades_valores_aporte, index=False)
salvar_dataframe(df_resumo_quantidades_valores_estrategia, caminho_tbl_resumo_quantidades_valores_estrategia, index=False)
salvar_dataframe(df_distribuicao_valores_classe_instrumento, caminho_tbl_distribuicao_valores_classe_instrumento, index=False)
salvar_dataframe(df_auditoria_quantidades_valores_comprados, caminho_tbl_auditoria_quantidades_valores_comprados, index=False)
salvar_dataframe(df_inconsistencias_quantidades_valores_comprados, caminho_tbl_inconsistencias_quantidades_valores_comprados, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

print("\nRegras formais do registro de quantidades e valores comprados:")
print(df_regras_registro_quantidades_valores_comprados.to_string(index=False))

print("\nBase final de compras - amostra:")
print(df_base_compras_finais.head(30).to_string(index=False))

print("\nResumo de quantidades e valores por aporte - amostra:")
print(df_resumo_quantidades_valores_aporte.head(30).to_string(index=False))

print("\nResumo de quantidades e valores por estratégia:")
print(df_resumo_quantidades_valores_estrategia.to_string(index=False))

print("\nDistribuição de valores por classe de instrumento:")
print(df_distribuicao_valores_classe_instrumento.to_string(index=False))

print("\nAuditoria de quantidades e valores comprados - amostra:")
print(df_auditoria_quantidades_valores_comprados.head(40).to_string(index=False))

print("\nInconsistências de quantidades e valores comprados - amostra:")
print(df_inconsistencias_quantidades_valores_comprados.head(30).to_string(index=False))

print("\nArquivos salvos na subetapa 10.3:")
print(f"- {caminho_tbl_regras_registro_quantidades_valores_comprados}")
print(f"- {caminho_base_compras_finais}")
print(f"- {caminho_tbl_resumo_quantidades_valores_aporte}")
print(f"- {caminho_tbl_resumo_quantidades_valores_estrategia}")
print(f"- {caminho_tbl_distribuicao_valores_classe_instrumento}")
print(f"- {caminho_tbl_auditoria_quantidades_valores_comprados}")
print(f"- {caminho_tbl_inconsistencias_quantidades_valores_comprados}")

print("\nETAPA 10.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 10.3 - REGISTRO DAS QUANTIDADES E VALORES COMPRADOS

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - aportes realizados 10.1                    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_1_base_aportes_realizados.parquet
Entrada - base de tickers comprados 10.2             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_2_base_tickers_comprados_aporte.parquet
Entrada - resumo por aporte 10.2                     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_2_tbl_resumo_tickers_comprados_aporte.parquet
Saída   - regras do registro de quantidades e valores: C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulac

## Etapa 10.4) Registro Histórico das Posições

In [50]:
%%time
# ============================================================
# Etapa 10.4) Registro Histórico das Posições
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 10.4 - REGISTRO HISTÓRICO DAS POSIÇÕES")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[2/10] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_compras_finais = gerar_caminho_arquivo(
    etapa=10,
    subetapa=3,
    tipo_arquivo="base",
    nome="compras_finais",
)

caminho_tbl_catalogo_curvas_patrimoniais = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="tbl",
    nome="catalogo_curvas_patrimoniais",
)

caminho_base_curvas_patrimoniais_consolidadas = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="base",
    nome="curvas_patrimoniais_consolidadas",
)

caminho_base_posicoes_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="posicoes_capitulacao",
)

caminho_base_posicoes_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="posicoes_euforia",
)

caminho_base_posicoes_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="posicoes_aleatorias_controle",
)

caminho_base_posicoes_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="posicoes_mensais_controle",
)

caminho_base_posicoes_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="base",
    nome="posicoes_benchmark_ibovespa",
)

caminho_base_posicoes_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="base",
    nome="posicoes_cdi_only",
)

caminho_base_patrimonio_capitulacao = gerar_caminho_arquivo(
    etapa=9,
    subetapa=1,
    tipo_arquivo="base",
    nome="patrimonio_capitulacao",
)

caminho_base_patrimonio_euforia = gerar_caminho_arquivo(
    etapa=9,
    subetapa=2,
    tipo_arquivo="base",
    nome="patrimonio_euforia",
)

caminho_base_patrimonio_aleatorias_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=3,
    tipo_arquivo="base",
    nome="patrimonio_aleatorias_controle",
)

caminho_base_patrimonio_mensais_controle = gerar_caminho_arquivo(
    etapa=9,
    subetapa=4,
    tipo_arquivo="base",
    nome="patrimonio_mensais_controle",
)

caminho_base_patrimonio_benchmark_ibovespa = gerar_caminho_arquivo(
    etapa=9,
    subetapa=5,
    tipo_arquivo="base",
    nome="patrimonio_benchmark_ibovespa",
)

caminho_base_patrimonio_cdi_only = gerar_caminho_arquivo(
    etapa=9,
    subetapa=6,
    tipo_arquivo="base",
    nome="patrimonio_cdi_only",
)

caminho_tbl_regras_registro_historico_posicoes = gerar_caminho_arquivo(
    etapa=10,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="regras_registro_historico_posicoes",
)

caminho_base_historico_posicoes = gerar_caminho_arquivo(
    etapa=10,
    subetapa=4,
    tipo_arquivo="base",
    nome="historico_posicoes",
)

caminho_tbl_resumo_historico_posicoes_estrategia = gerar_caminho_arquivo(
    etapa=10,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="resumo_historico_posicoes_estrategia",
)

caminho_tbl_resumo_historico_posicoes_ticker = gerar_caminho_arquivo(
    etapa=10,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="resumo_historico_posicoes_ticker",
)

caminho_tbl_distribuicao_anual_posicoes = gerar_caminho_arquivo(
    etapa=10,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_posicoes",
)

caminho_tbl_auditoria_historico_posicoes = gerar_caminho_arquivo(
    etapa=10,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="auditoria_historico_posicoes",
)

caminho_tbl_inconsistencias_historico_posicoes = gerar_caminho_arquivo(
    etapa=10,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="inconsistencias_historico_posicoes",
)

print(f"Entrada - compras finais 10.3                         : {caminho_base_compras_finais}")
print(f"Entrada - catálogo patrimonial 9.7                    : {caminho_tbl_catalogo_curvas_patrimoniais}")
print(f"Entrada - curvas patrimoniais consolidadas 9.7        : {caminho_base_curvas_patrimoniais_consolidadas}")
print(f"Entrada - posições capitulação 9.1                    : {caminho_base_posicoes_capitulacao}")
print(f"Entrada - posições euforia 9.2                        : {caminho_base_posicoes_euforia}")
print(f"Entrada - posições aleatórias 9.3                     : {caminho_base_posicoes_aleatorias_controle}")
print(f"Entrada - posições mensais 9.4                        : {caminho_base_posicoes_mensais_controle}")
print(f"Entrada - posições benchmark Ibovespa 9.5             : {caminho_base_posicoes_benchmark_ibovespa}")
print(f"Entrada - posições CDI-only 9.6                       : {caminho_base_posicoes_cdi_only}")
print(f"Entrada - patrimônio capitulação 9.1                  : {caminho_base_patrimonio_capitulacao}")
print(f"Entrada - patrimônio euforia 9.2                      : {caminho_base_patrimonio_euforia}")
print(f"Entrada - patrimônio aleatórias 9.3                   : {caminho_base_patrimonio_aleatorias_controle}")
print(f"Entrada - patrimônio mensais 9.4                      : {caminho_base_patrimonio_mensais_controle}")
print(f"Entrada - patrimônio benchmark Ibovespa 9.5           : {caminho_base_patrimonio_benchmark_ibovespa}")
print(f"Entrada - patrimônio CDI-only 9.6                     : {caminho_base_patrimonio_cdi_only}")
print(f"Saída   - regras do histórico de posições             : {caminho_tbl_regras_registro_historico_posicoes}")
print(f"Saída   - base histórica de posições                  : {caminho_base_historico_posicoes}")
print(f"Saída   - resumo por estratégia                       : {caminho_tbl_resumo_historico_posicoes_estrategia}")
print(f"Saída   - resumo por ticker                           : {caminho_tbl_resumo_historico_posicoes_ticker}")
print(f"Saída   - distribuição anual                          : {caminho_tbl_distribuicao_anual_posicoes}")
print(f"Saída   - auditoria                                   : {caminho_tbl_auditoria_historico_posicoes}")
print(f"Saída   - inconsistências                             : {caminho_tbl_inconsistencias_historico_posicoes}")
print("OK")

# ============================================================
# 3) Carga dos inputs da subetapa
# ============================================================

print("\n[3/10] Carga dos inputs da subetapa...")

df_compras_finais = pd.read_parquet(caminho_base_compras_finais)
df_catalogo_curvas_patrimoniais = pd.read_parquet(caminho_tbl_catalogo_curvas_patrimoniais)
df_curvas_patrimoniais_consolidadas = pd.read_parquet(caminho_base_curvas_patrimoniais_consolidadas)

df_posicoes_capitulacao = pd.read_parquet(caminho_base_posicoes_capitulacao)
df_posicoes_euforia = pd.read_parquet(caminho_base_posicoes_euforia)
df_posicoes_aleatorias_controle = pd.read_parquet(caminho_base_posicoes_aleatorias_controle)
df_posicoes_mensais_controle = pd.read_parquet(caminho_base_posicoes_mensais_controle)
df_posicoes_benchmark_ibovespa = pd.read_parquet(caminho_base_posicoes_benchmark_ibovespa)
df_posicoes_cdi_only = pd.read_parquet(caminho_base_posicoes_cdi_only)

df_patrimonio_capitulacao = pd.read_parquet(caminho_base_patrimonio_capitulacao)
df_patrimonio_euforia = pd.read_parquet(caminho_base_patrimonio_euforia)
df_patrimonio_aleatorias_controle = pd.read_parquet(caminho_base_patrimonio_aleatorias_controle)
df_patrimonio_mensais_controle = pd.read_parquet(caminho_base_patrimonio_mensais_controle)
df_patrimonio_benchmark_ibovespa = pd.read_parquet(caminho_base_patrimonio_benchmark_ibovespa)
df_patrimonio_cdi_only = pd.read_parquet(caminho_base_patrimonio_cdi_only)

print(f"Compras finais 10.3                         : {df_compras_finais.shape[0]:,} linhas x {df_compras_finais.shape[1]:,} colunas")
print(f"Catálogo patrimonial 9.7                    : {df_catalogo_curvas_patrimoniais.shape[0]:,} linhas x {df_catalogo_curvas_patrimoniais.shape[1]:,} colunas")
print(f"Curvas patrimoniais consolidadas 9.7        : {df_curvas_patrimoniais_consolidadas.shape[0]:,} linhas x {df_curvas_patrimoniais_consolidadas.shape[1]:,} colunas")
print(f"Posições capitulação 9.1                    : {df_posicoes_capitulacao.shape[0]:,} linhas x {df_posicoes_capitulacao.shape[1]:,} colunas")
print(f"Posições euforia 9.2                        : {df_posicoes_euforia.shape[0]:,} linhas x {df_posicoes_euforia.shape[1]:,} colunas")
print(f"Posições aleatórias 9.3                     : {df_posicoes_aleatorias_controle.shape[0]:,} linhas x {df_posicoes_aleatorias_controle.shape[1]:,} colunas")
print(f"Posições mensais 9.4                        : {df_posicoes_mensais_controle.shape[0]:,} linhas x {df_posicoes_mensais_controle.shape[1]:,} colunas")
print(f"Posições benchmark Ibovespa 9.5             : {df_posicoes_benchmark_ibovespa.shape[0]:,} linhas x {df_posicoes_benchmark_ibovespa.shape[1]:,} colunas")
print(f"Posições CDI-only 9.6                       : {df_posicoes_cdi_only.shape[0]:,} linhas x {df_posicoes_cdi_only.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[4/10] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_MONETARIA_POSICOES = 0.01
TOLERANCIA_PESO_POSICOES = 0.000001
TICKER_BENCHMARK_IBOVESPA = "^BVSP"
TICKER_CDI_ONLY = "CDI"

COLUNAS_CATALOGO_ESTRATEGIA = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_backtest",
    "ordem_exibicao",
]

COLUNAS_HISTORICO_POSICOES = [
    "chave_posicao_data",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_backtest",
    "origem_posicao",
    "classe_instrumento",
    "grupo_registro_carteira",
    "data",
    "ano",
    "mes",
    "ordem_exibicao",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "quantidade_em_carteira",
    "custo_medio_unitario",
    "preco_fechamento",
    "valor_mercado_posicao",
    "valor_investido_acumulado_ticker",
    "resultado_nao_realizado_posicao",
    "patrimonio_total",
    "valor_carteira_investida_total",
    "saldo_caixa",
    "saldo_caixa_fim_dia",
    "peso_posicao_no_patrimonio",
    "peso_posicao_na_carteira_investida",
    "flag_posicao_ativa",
    "flag_ticker_preenchido",
    "flag_issuer_code_preenchido",
    "flag_quantidade_valida",
    "flag_preco_fechamento_valido",
    "flag_valor_mercado_valido",
    "flag_peso_patrimonio_valido",
    "flag_posicao_historica_valida",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias, nome_base):
    colunas_ausentes = [coluna for coluna in colunas_obrigatorias if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"A base {nome_base} não contém as colunas obrigatórias: {colunas_ausentes}")

def validar_colunas_groupby(df, colunas_groupby, nome_base, nome_operacao):
    colunas_ausentes = [coluna for coluna in colunas_groupby if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        colunas_disponiveis = list(df.columns)
        raise ValueError(
            f"A operação {nome_operacao} na base {nome_base} não pode ser executada. "
            f"Colunas ausentes no agrupamento: {colunas_ausentes}. "
            f"Colunas disponíveis: {colunas_disponiveis}"
        )

def normalizar_string_serie(serie):
    return serie.astype("string").str.strip()

def primeira_coluna_disponivel(df, candidatos):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    return None

def assegurar_coluna(df, coluna, valor_default=pd.NA):
    if coluna not in df.columns:
        df[coluna] = valor_default
    return df

def preparar_catalogo_estrategias(df_catalogo):
    validar_colunas_obrigatorias(df_catalogo, ["estrategia_id"], "catalogo_curvas_patrimoniais")

    df_out = df_catalogo.copy()
    df_out["estrategia_id"] = normalizar_string_serie(df_out["estrategia_id"])

    for coluna in COLUNAS_CATALOGO_ESTRATEGIA:
        df_out = assegurar_coluna(df_out, coluna)

    df_out = (
        df_out[COLUNAS_CATALOGO_ESTRATEGIA]
        .drop_duplicates(subset=["estrategia_id"], keep="last")
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_patrimonial(df_patrimonio, origem_patrimonio):
    validar_colunas_obrigatorias(df_patrimonio, ["data", "estrategia_id", "patrimonio_total"], origem_patrimonio)

    df_out = df_patrimonio.copy()
    df_out["data"] = pd.to_datetime(df_out["data"], errors="coerce")
    df_out["estrategia_id"] = normalizar_string_serie(df_out["estrategia_id"])
    df_out["origem_patrimonio"] = origem_patrimonio

    for coluna in [
        "patrimonio_total",
        "valor_carteira_investida",
        "saldo_caixa",
        "saldo_caixa_fim_dia",
        "n_posicoes_ativas",
        "n_tickers_unicos_ativos",
    ]:
        df_out = assegurar_coluna(df_out, coluna, 0.0)

    for coluna in [
        "patrimonio_total",
        "valor_carteira_investida",
        "saldo_caixa",
        "saldo_caixa_fim_dia",
        "n_posicoes_ativas",
        "n_tickers_unicos_ativos",
    ]:
        df_out[coluna] = pd.to_numeric(df_out[coluna], errors="coerce")

    df_out["saldo_caixa"] = df_out["saldo_caixa"].fillna(df_out["saldo_caixa_fim_dia"])
    df_out["saldo_caixa_fim_dia"] = df_out["saldo_caixa_fim_dia"].fillna(df_out["saldo_caixa"])
    df_out["valor_carteira_investida"] = df_out["valor_carteira_investida"].fillna(0.0)

    df_out = (
        df_out[
            [
                "data",
                "estrategia_id",
                "patrimonio_total",
                "valor_carteira_investida",
                "saldo_caixa",
                "saldo_caixa_fim_dia",
                "n_posicoes_ativas",
                "n_tickers_unicos_ativos",
                "origem_patrimonio",
            ]
        ]
        .dropna(subset=["data", "estrategia_id"])
        .sort_values(["estrategia_id", "data"])
        .drop_duplicates(subset=["estrategia_id", "data"], keep="last")
        .reset_index(drop=True)
    )

    return df_out

def preparar_base_posicoes(df_posicoes, origem_posicao):
    validar_colunas_obrigatorias(df_posicoes, ["data", "estrategia_id", "ticker"], origem_posicao)

    df_out = df_posicoes.copy()
    df_out["origem_posicao"] = origem_posicao

    df_out["data"] = pd.to_datetime(df_out["data"], errors="coerce")
    df_out["estrategia_id"] = normalizar_string_serie(df_out["estrategia_id"])
    df_out["ticker"] = normalizar_string_serie(df_out["ticker"])

    df_out = assegurar_coluna(df_out, "issuer_code")
    df_out = assegurar_coluna(df_out, "nome")
    df_out = assegurar_coluna(df_out, "setor")
    df_out = assegurar_coluna(df_out, "subsetor")
    df_out = assegurar_coluna(df_out, "segmento")
    df_out = assegurar_coluna(df_out, "carteira_id")
    df_out = assegurar_coluna(df_out, "replica_id")
    df_out = assegurar_coluna(df_out, "capital_inicial_estrategia")

    df_out["issuer_code"] = normalizar_string_serie(df_out["issuer_code"])
    df_out["nome"] = normalizar_string_serie(df_out["nome"])
    df_out["setor"] = normalizar_string_serie(df_out["setor"])
    df_out["subsetor"] = normalizar_string_serie(df_out["subsetor"])
    df_out["segmento"] = normalizar_string_serie(df_out["segmento"])

    coluna_preco = primeira_coluna_disponivel(
        df_out,
        [
            "preco_fechamento",
            "close_adj",
            "preco_benchmark_fechamento",
            "preco_fechamento_posicao",
        ],
    )
    if coluna_preco is None:
        df_out["preco_fechamento"] = np.nan
    elif coluna_preco != "preco_fechamento":
        df_out["preco_fechamento"] = df_out[coluna_preco]

    for coluna in [
        "quantidade_em_carteira",
        "custo_medio_unitario",
        "preco_fechamento",
        "valor_mercado_posicao",
        "valor_investido_acumulado_ticker",
        "resultado_nao_realizado_posicao",
        "peso_posicao_no_patrimonio",
    ]:
        df_out = assegurar_coluna(df_out, coluna, np.nan)
        df_out[coluna] = pd.to_numeric(df_out[coluna], errors="coerce")

    df_out["valor_investido_acumulado_ticker"] = df_out["valor_investido_acumulado_ticker"].fillna(
        df_out["quantidade_em_carteira"] * df_out["custo_medio_unitario"]
    )
    df_out["resultado_nao_realizado_posicao"] = df_out["resultado_nao_realizado_posicao"].fillna(
        df_out["valor_mercado_posicao"] - df_out["valor_investido_acumulado_ticker"]
    )

    return df_out

def aplicar_catalogo(df_posicoes, df_catalogo):
    df_out = df_posicoes.copy()
    df_merge = df_catalogo.copy()

    df_out = df_out.merge(
        df_merge,
        on="estrategia_id",
        how="left",
        suffixes=("", "_catalogo"),
        validate="many_to_one",
    )

    for coluna in COLUNAS_CATALOGO_ESTRATEGIA:
        if coluna == "estrategia_id":
            continue
        coluna_catalogo = f"{coluna}_catalogo"
        if coluna_catalogo in df_out.columns:
            if coluna in df_out.columns:
                df_out[coluna] = df_out[coluna].where(df_out[coluna].notna(), df_out[coluna_catalogo])
                df_out = df_out.drop(columns=[coluna_catalogo])
            else:
                df_out = df_out.rename(columns={coluna_catalogo: coluna})
        else:
            df_out = assegurar_coluna(df_out, coluna)

    return df_out

def classificar_instrumentos(df_posicoes):
    df_out = df_posicoes.copy()

    df_out["classe_instrumento"] = np.select(
        [
            df_out["ticker"].astype("string") == TICKER_BENCHMARK_IBOVESPA,
            df_out["ticker"].astype("string") == TICKER_CDI_ONLY,
        ],
        [
            "benchmark",
            "taxa_referencia",
        ],
        default="acao",
    )

    df_out["grupo_registro_carteira"] = np.select(
        [
            df_out["classe_instrumento"] == "benchmark",
            df_out["classe_instrumento"] == "taxa_referencia",
        ],
        [
            "benchmark_equivalente",
            "renda_fixa",
        ],
        default="acoes",
    )

    df_out.loc[df_out["ticker"].astype("string") == TICKER_BENCHMARK_IBOVESPA, "issuer_code"] = TICKER_BENCHMARK_IBOVESPA
    df_out.loc[df_out["ticker"].astype("string") == TICKER_BENCHMARK_IBOVESPA, "nome"] = "IBOVESPA"
    df_out.loc[df_out["ticker"].astype("string") == TICKER_BENCHMARK_IBOVESPA, "setor"] = "ÍNDICE"
    df_out.loc[df_out["ticker"].astype("string") == TICKER_BENCHMARK_IBOVESPA, "subsetor"] = "ÍNDICE"
    df_out.loc[df_out["ticker"].astype("string") == TICKER_BENCHMARK_IBOVESPA, "segmento"] = "ÍNDICE"

    df_out.loc[df_out["ticker"].astype("string") == TICKER_CDI_ONLY, "issuer_code"] = TICKER_CDI_ONLY
    df_out.loc[df_out["ticker"].astype("string") == TICKER_CDI_ONLY, "nome"] = "Carteira CDI-Only"
    df_out.loc[df_out["ticker"].astype("string") == TICKER_CDI_ONLY, "setor"] = "Renda Fixa"
    df_out.loc[df_out["ticker"].astype("string") == TICKER_CDI_ONLY, "subsetor"] = "CDI"
    df_out.loc[df_out["ticker"].astype("string") == TICKER_CDI_ONLY, "segmento"] = "CDI"

    return df_out

def calcular_mediana_sem_erro(serie):
    serie_num = pd.to_numeric(serie, errors="coerce").dropna()
    if len(serie_num) == 0:
        return np.nan
    return float(serie_num.median())

def converter_serie_numerica(serie, valor_default=np.nan):
    serie_obj = serie.astype("object")
    return pd.to_numeric(serie_obj, errors="coerce").fillna(valor_default)

def converter_serie_inteira(serie, valor_default=0):
    serie_obj = serie.astype("object")
    return pd.to_numeric(serie_obj, errors="coerce").fillna(valor_default).astype(int)

print(f"Tolerância monetária das posições : {TOLERANCIA_MONETARIA_POSICOES}")
print(f"Tolerância de pesos das posições  : {TOLERANCIA_PESO_POSICOES}")
print("OK")

# ============================================================
# 5) Preparação e validação estrutural das bases
# ============================================================

print("\n[5/10] Preparação e validação estrutural das bases...")

validar_colunas_obrigatorias(
    df_compras_finais,
    ["chave_evento_compra", "estrategia_id", "ticker", "valor_investido_compra", "quantidade_comprada"],
    "10_3_base_compras_finais",
)
validar_colunas_obrigatorias(
    df_curvas_patrimoniais_consolidadas,
    ["data", "estrategia_id", "patrimonio_total"],
    "9_7_base_curvas_patrimoniais_consolidadas",
)

n_chaves_compra_duplicadas_10_3 = int(df_compras_finais["chave_evento_compra"].duplicated().sum())
n_estrategias_catalogo = int(df_catalogo_curvas_patrimoniais["estrategia_id"].nunique())

df_catalogo_estrategias = preparar_catalogo_estrategias(df_catalogo_curvas_patrimoniais)

lista_patrimonios = [
    preparar_base_patrimonial(df_patrimonio_capitulacao, "etapa_9_1"),
    preparar_base_patrimonial(df_patrimonio_euforia, "etapa_9_2"),
    preparar_base_patrimonial(df_patrimonio_aleatorias_controle, "etapa_9_3"),
    preparar_base_patrimonial(df_patrimonio_mensais_controle, "etapa_9_4"),
    preparar_base_patrimonial(df_patrimonio_benchmark_ibovespa, "etapa_9_5"),
    preparar_base_patrimonial(df_patrimonio_cdi_only, "etapa_9_6"),
]

df_patrimonio_unificado = (
    pd.concat(lista_patrimonios, axis=0, ignore_index=True)
    .sort_values(["estrategia_id", "data"])
    .drop_duplicates(subset=["estrategia_id", "data"], keep="last")
    .reset_index(drop=True)
)

n_duplicidades_patrimonio = int(df_patrimonio_unificado.duplicated(subset=["estrategia_id", "data"]).sum())

print(f"Estratégias no catálogo 9.7                    : {n_estrategias_catalogo:,}")
print(f"Chaves de compra duplicadas na 10.3            : {n_chaves_compra_duplicadas_10_3:,}")
print(f"Base patrimonial unificada                     : {df_patrimonio_unificado.shape[0]:,} linhas x {df_patrimonio_unificado.shape[1]:,} colunas")
print(f"Duplicidades estrategia_id + data no patrimônio: {n_duplicidades_patrimonio:,}")
print("OK")

# ============================================================
# 6) Consolidação da base histórica diária de posições
# ============================================================

print("\n[6/10] Consolidação da base histórica diária de posições...")

lista_posicoes = [
    preparar_base_posicoes(df_posicoes_capitulacao, "etapa_9_1"),
    preparar_base_posicoes(df_posicoes_euforia, "etapa_9_2"),
    preparar_base_posicoes(df_posicoes_aleatorias_controle, "etapa_9_3"),
    preparar_base_posicoes(df_posicoes_mensais_controle, "etapa_9_4"),
    preparar_base_posicoes(df_posicoes_benchmark_ibovespa, "etapa_9_5"),
    preparar_base_posicoes(df_posicoes_cdi_only, "etapa_9_6"),
]

df_historico_posicoes = (
    pd.concat(lista_posicoes, axis=0, ignore_index=True)
    .dropna(subset=["data", "estrategia_id", "ticker"])
    .reset_index(drop=True)
)

df_historico_posicoes = aplicar_catalogo(df_historico_posicoes, df_catalogo_estrategias)
df_historico_posicoes = classificar_instrumentos(df_historico_posicoes)

if "patrimonio_total" in df_historico_posicoes.columns:
    df_historico_posicoes = df_historico_posicoes.rename(columns={"patrimonio_total": "patrimonio_total_origem_posicao"})
if "valor_carteira_investida" in df_historico_posicoes.columns:
    df_historico_posicoes = df_historico_posicoes.rename(columns={"valor_carteira_investida": "valor_carteira_investida_origem_posicao"})
if "saldo_caixa" in df_historico_posicoes.columns:
    df_historico_posicoes = df_historico_posicoes.rename(columns={"saldo_caixa": "saldo_caixa_origem_posicao"})
if "saldo_caixa_fim_dia" in df_historico_posicoes.columns:
    df_historico_posicoes = df_historico_posicoes.rename(columns={"saldo_caixa_fim_dia": "saldo_caixa_fim_dia_origem_posicao"})

df_historico_posicoes = df_historico_posicoes.merge(
    df_patrimonio_unificado[
        [
            "data",
            "estrategia_id",
            "patrimonio_total",
            "valor_carteira_investida",
            "saldo_caixa",
            "saldo_caixa_fim_dia",
            "n_posicoes_ativas",
            "n_tickers_unicos_ativos",
        ]
    ],
    on=["data", "estrategia_id"],
    how="left",
    validate="many_to_one",
)

df_historico_posicoes = df_historico_posicoes.rename(
    columns={
        "valor_carteira_investida": "valor_carteira_investida_total",
    }
)

for coluna in [
    "quantidade_em_carteira",
    "custo_medio_unitario",
    "preco_fechamento",
    "valor_mercado_posicao",
    "valor_investido_acumulado_ticker",
    "resultado_nao_realizado_posicao",
    "patrimonio_total",
    "valor_carteira_investida_total",
    "saldo_caixa",
    "saldo_caixa_fim_dia",
]:
    df_historico_posicoes[coluna] = pd.to_numeric(df_historico_posicoes[coluna], errors="coerce")

for coluna in ["quantidade_em_carteira", "valor_mercado_posicao", "valor_investido_acumulado_ticker"]:
    df_historico_posicoes[coluna] = df_historico_posicoes[coluna].fillna(0.0)

df_historico_posicoes["peso_posicao_no_patrimonio"] = np.where(
    df_historico_posicoes["patrimonio_total"].abs() > TOLERANCIA_MONETARIA_POSICOES,
    df_historico_posicoes["valor_mercado_posicao"] / df_historico_posicoes["patrimonio_total"],
    0.0,
)

df_historico_posicoes["peso_posicao_na_carteira_investida"] = np.where(
    df_historico_posicoes["valor_carteira_investida_total"].abs() > TOLERANCIA_MONETARIA_POSICOES,
    df_historico_posicoes["valor_mercado_posicao"] / df_historico_posicoes["valor_carteira_investida_total"],
    np.where(df_historico_posicoes["classe_instrumento"] == "taxa_referencia", 1.0, 0.0),
)

df_historico_posicoes["ano"] = df_historico_posicoes["data"].dt.year.astype("Int64")
df_historico_posicoes["mes"] = df_historico_posicoes["data"].dt.month.astype("Int64")
df_historico_posicoes["chave_posicao_data"] = (
    df_historico_posicoes["chave_estrategia"].astype("string")
    + "__"
    + df_historico_posicoes["data"].dt.strftime("%Y-%m-%d").astype("string")
    + "__"
    + df_historico_posicoes["ticker"].astype("string")
)

df_historico_posicoes["flag_posicao_ativa"] = (
    (df_historico_posicoes["quantidade_em_carteira"] > 0)
    | (df_historico_posicoes["valor_mercado_posicao"] > 0)
)
df_historico_posicoes["flag_ticker_preenchido"] = df_historico_posicoes["ticker"].notna() & (df_historico_posicoes["ticker"].astype("string").str.len() > 0)
df_historico_posicoes["flag_issuer_code_preenchido"] = df_historico_posicoes["issuer_code"].notna() & (df_historico_posicoes["issuer_code"].astype("string").str.len() > 0)
df_historico_posicoes["flag_quantidade_valida"] = df_historico_posicoes["quantidade_em_carteira"] >= -TOLERANCIA_MONETARIA_POSICOES
df_historico_posicoes["flag_preco_fechamento_valido"] = (
    (df_historico_posicoes["classe_instrumento"] == "taxa_referencia")
    | (df_historico_posicoes["preco_fechamento"] > 0)
)
df_historico_posicoes["flag_valor_mercado_valido"] = df_historico_posicoes["valor_mercado_posicao"] >= -TOLERANCIA_MONETARIA_POSICOES
df_historico_posicoes["flag_peso_patrimonio_valido"] = df_historico_posicoes["peso_posicao_no_patrimonio"].between(-TOLERANCIA_PESO_POSICOES, 1.0 + TOLERANCIA_PESO_POSICOES)
df_historico_posicoes["flag_posicao_historica_valida"] = (
    df_historico_posicoes["flag_posicao_ativa"]
    & df_historico_posicoes["flag_ticker_preenchido"]
    & df_historico_posicoes["flag_issuer_code_preenchido"]
    & df_historico_posicoes["flag_quantidade_valida"]
    & df_historico_posicoes["flag_preco_fechamento_valido"]
    & df_historico_posicoes["flag_valor_mercado_valido"]
    & df_historico_posicoes["flag_peso_patrimonio_valido"]
)

for coluna in COLUNAS_HISTORICO_POSICOES:
    df_historico_posicoes = assegurar_coluna(df_historico_posicoes, coluna)

df_historico_posicoes = (
    df_historico_posicoes[COLUNAS_HISTORICO_POSICOES]
    .sort_values(["ordem_exibicao", "chave_estrategia", "data", "classe_instrumento", "ticker"])
    .reset_index(drop=True)
)

validar_colunas_obrigatorias(
    df_historico_posicoes,
    COLUNAS_HISTORICO_POSICOES,
    "10_4_base_historico_posicoes_em_memoria",
)

n_chaves_posicao_data_duplicadas = int(df_historico_posicoes["chave_posicao_data"].duplicated().sum())
n_linhas_posicoes_invalidas = int((~df_historico_posicoes["flag_posicao_historica_valida"]).sum())

print(f"Base histórica de posições consolidada       : {df_historico_posicoes.shape[0]:,} linhas x {df_historico_posicoes.shape[1]:,} colunas")
print(f"Estratégias com posições históricas          : {df_historico_posicoes['estrategia_id'].nunique():,}")
print(f"Tickers distintos na base histórica          : {df_historico_posicoes['ticker'].nunique():,}")
print(f"Chaves posição-data duplicadas               : {n_chaves_posicao_data_duplicadas:,}")
print(f"Linhas de posição histórica inválidas        : {n_linhas_posicoes_invalidas:,}")
print("OK")

# ============================================================
# 7) Construção das regras formais e resumos consolidados
# ============================================================

print("\n[7/10] Construção das regras formais e resumos consolidados...")

df_regras_registro_historico_posicoes = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "RHP1",
            "escopo": "fonte",
            "regra_operacional": "usar_as_bases_de_posicoes_diarias_das_etapas_9_1_a_9_6",
            "detalhe": "O histórico consolidado de posições parte das bases de posições já geradas pelos motores de backtest e benchmarks.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "RHP2",
            "escopo": "nomenclatura",
            "regra_operacional": "herdar_classificacao_do_catalogo_consolidado_da_etapa_9_7",
            "detalhe": "A classificação de estratégia, família, categoria e ordem de exibição é padronizada pelo catálogo patrimonial consolidado.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "RHP3",
            "escopo": "patrimonio",
            "regra_operacional": "recalcular_peso_da_posicao_com_base_no_patrimonio_total_diario",
            "detalhe": "O peso da posição é calculado como valor de mercado da posição dividido pelo patrimônio total diário da respectiva estratégia.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "RHP4",
            "escopo": "instrumentos",
            "regra_operacional": "classificar_acoes_benchmark_e_taxa_referencia_em_classes_distintas",
            "detalhe": "Ações, exposições ao Ibovespa e CDI-only são classificados separadamente para evitar mistura de instrumentos no HTML e nas análises de composição.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "RHP5",
            "escopo": "auditoria",
            "regra_operacional": "reconciliar_o_somatorio_das_posicoes_com_a_carteira_investida_patrimonial",
            "detalhe": "A auditoria compara o valor de mercado agregado das posições por estratégia e data com a carteira investida registrada na base patrimonial.",
        },
        {
            "ordem_regra": 6,
            "regra_id": "RHP6",
            "escopo": "html",
            "regra_operacional": "salvar_bases_analiticas_para_composicao_concentracao_e_paineis_finais",
            "detalhe": "A base histórica de posições alimenta as etapas de concentração, composição e as bases finais do HTML.",
        },
    ]
)

COLUNAS_GROUPBY_POSICOES_DATA = [
    "chave_estrategia",
    "estrategia_id",
    "data",
    "ano",
    "mes",
]

validar_colunas_groupby(
    df_historico_posicoes,
    COLUNAS_GROUPBY_POSICOES_DATA,
    "df_historico_posicoes",
    "df_posicoes_data",
)

df_posicoes_data = (
    df_historico_posicoes
    .groupby(
        COLUNAS_GROUPBY_POSICOES_DATA,
        as_index=False,
        dropna=False,
    )
    .agg(
        valor_mercado_posicoes_data=("valor_mercado_posicao", "sum"),
        valor_custo_posicoes_data=("valor_investido_acumulado_ticker", "sum"),
        resultado_nao_realizado_data=("resultado_nao_realizado_posicao", "sum"),
        peso_posicoes_patrimonio_data=("peso_posicao_no_patrimonio", "sum"),
        n_linhas_posicoes_data=("ticker", "count"),
        n_tickers_posicoes_data=("ticker", "nunique"),
        n_empresas_posicoes_data=("issuer_code", "nunique"),
    )
)

COLUNAS_GROUPBY_RESUMO_ESTRATEGIA = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_backtest",
    "ordem_exibicao",
]

validar_colunas_groupby(
    df_historico_posicoes,
    COLUNAS_GROUPBY_RESUMO_ESTRATEGIA,
    "df_historico_posicoes",
    "df_resumo_historico_posicoes_estrategia",
)

df_resumo_historico_posicoes_estrategia = (
    df_historico_posicoes
    .groupby(
        COLUNAS_GROUPBY_RESUMO_ESTRATEGIA,
        as_index=False,
        dropna=False,
    )
    .agg(
        primeira_data_posicao=("data", "min"),
        ultima_data_posicao=("data", "max"),
        n_linhas_historico_posicoes=("ticker", "count"),
        n_datas_com_posicoes=("data", "nunique"),
        n_tickers_distintos=("ticker", "nunique"),
        n_empresas_distintas=("issuer_code", "nunique"),
        n_setores_distintos=("setor", "nunique"),
        n_subsetores_distintos=("subsetor", "nunique"),
        n_segmentos_distintos=("segmento", "nunique"),
        valor_mercado_medio_posicao=("valor_mercado_posicao", "mean"),
        valor_mercado_maximo_posicao=("valor_mercado_posicao", "max"),
        peso_medio_posicao_patrimonio=("peso_posicao_no_patrimonio", "mean"),
        peso_maximo_posicao_patrimonio=("peso_posicao_no_patrimonio", "max"),
        n_linhas_posicao_invalida=("flag_posicao_historica_valida", lambda x: int((~x.astype(bool)).sum())),
    )
    .sort_values(["ordem_exibicao", "chave_estrategia"])
    .reset_index(drop=True)
)

COLUNAS_CONTAGEM_RESUMO_ESTRATEGIA = [
    "n_linhas_historico_posicoes",
    "n_datas_com_posicoes",
    "n_tickers_distintos",
    "n_empresas_distintas",
    "n_setores_distintos",
    "n_subsetores_distintos",
    "n_segmentos_distintos",
    "n_linhas_posicao_invalida",
]

for coluna in COLUNAS_CONTAGEM_RESUMO_ESTRATEGIA:
    df_resumo_historico_posicoes_estrategia[coluna] = converter_serie_inteira(
        df_resumo_historico_posicoes_estrategia[coluna],
        0,
    )

COLUNAS_GROUPBY_RESUMO_TICKER = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "classe_instrumento",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
]

validar_colunas_groupby(
    df_historico_posicoes,
    COLUNAS_GROUPBY_RESUMO_TICKER,
    "df_historico_posicoes",
    "df_resumo_historico_posicoes_ticker",
)

df_resumo_historico_posicoes_ticker = (
    df_historico_posicoes
    .groupby(
        COLUNAS_GROUPBY_RESUMO_TICKER,
        as_index=False,
        dropna=False,
    )
    .agg(
        primeira_data_posicao_ticker=("data", "min"),
        ultima_data_posicao_ticker=("data", "max"),
        n_datas_posicao_ticker=("data", "nunique"),
        quantidade_maxima_em_carteira=("quantidade_em_carteira", "max"),
        custo_medio_unitario_mediano=("custo_medio_unitario", calcular_mediana_sem_erro),
        valor_mercado_medio_ticker=("valor_mercado_posicao", "mean"),
        valor_mercado_maximo_ticker=("valor_mercado_posicao", "max"),
        peso_medio_ticker_patrimonio=("peso_posicao_no_patrimonio", "mean"),
        peso_maximo_ticker_patrimonio=("peso_posicao_no_patrimonio", "max"),
        resultado_nao_realizado_final_ticker=("resultado_nao_realizado_posicao", "last"),
    )
    .sort_values(["ordem_exibicao", "chave_estrategia", "classe_instrumento", "ticker"])
    .reset_index(drop=True)
)

df_catalogo_estrategias_distribuicao = df_catalogo_estrategias.drop(
    columns=["chave_estrategia"],
    errors="ignore",
)

df_posicoes_data_catalogo = df_posicoes_data.merge(
    df_catalogo_estrategias_distribuicao,
    on="estrategia_id",
    how="left",
    validate="many_to_one",
)

COLUNAS_GROUPBY_DISTRIBUICAO_ANUAL = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "origem_backtest",
    "ordem_exibicao",
    "ano",
]

validar_colunas_groupby(
    df_posicoes_data_catalogo,
    COLUNAS_GROUPBY_DISTRIBUICAO_ANUAL,
    "df_posicoes_data_catalogo",
    "df_distribuicao_anual_posicoes",
)

df_distribuicao_anual_posicoes = (
    df_posicoes_data_catalogo
    .groupby(
        COLUNAS_GROUPBY_DISTRIBUICAO_ANUAL,
        as_index=False,
        dropna=False,
    )
    .agg(
        n_datas_com_posicoes_ano=("data", "nunique"),
        valor_mercado_medio_posicoes_ano=("valor_mercado_posicoes_data", "mean"),
        valor_mercado_final_posicoes_ano=("valor_mercado_posicoes_data", "last"),
        valor_custo_final_posicoes_ano=("valor_custo_posicoes_data", "last"),
        peso_medio_posicoes_patrimonio_ano=("peso_posicoes_patrimonio_data", "mean"),
        peso_final_posicoes_patrimonio_ano=("peso_posicoes_patrimonio_data", "last"),
        media_tickers_posicoes_ano=("n_tickers_posicoes_data", "mean"),
        max_tickers_posicoes_ano=("n_tickers_posicoes_data", "max"),
        media_empresas_posicoes_ano=("n_empresas_posicoes_data", "mean"),
        max_empresas_posicoes_ano=("n_empresas_posicoes_data", "max"),
    )
    .sort_values(["ordem_exibicao", "chave_estrategia", "ano"])
    .reset_index(drop=True)
)

print(f"Regras formais geradas                      : {len(df_regras_registro_historico_posicoes):,}")
print(f"Resumo por estratégia gerado                : {len(df_resumo_historico_posicoes_estrategia):,}")
print(f"Resumo por ticker gerado                    : {len(df_resumo_historico_posicoes_ticker):,}")
print(f"Distribuição anual gerada                   : {len(df_distribuicao_anual_posicoes):,}")
print("OK")

# ============================================================
# 8) Construção da auditoria e das inconsistências
# ============================================================

print("\n[8/10] Construção da auditoria e das inconsistências...")

df_posicoes_agregadas_data = (
    df_historico_posicoes
    .groupby(["estrategia_id", "data"], as_index=False, dropna=False)
    .agg(
        valor_mercado_posicoes=("valor_mercado_posicao", "sum"),
        peso_posicoes_patrimonio=("peso_posicao_no_patrimonio", "sum"),
        n_linhas_posicoes_data=("ticker", "count"),
        n_tickers_posicoes_data=("ticker", "nunique"),
        n_linhas_invalidas_data=("flag_posicao_historica_valida", lambda x: int((~x.astype(bool)).sum())),
    )
)

df_auditoria_data = (
    df_patrimonio_unificado
    .merge(
        df_posicoes_agregadas_data,
        on=["estrategia_id", "data"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        df_catalogo_estrategias,
        on="estrategia_id",
        how="left",
        validate="many_to_one",
    )
)

df_auditoria_data["valor_mercado_posicoes"] = converter_serie_numerica(df_auditoria_data["valor_mercado_posicoes"], 0.0)
df_auditoria_data["peso_posicoes_patrimonio"] = converter_serie_numerica(df_auditoria_data["peso_posicoes_patrimonio"], 0.0)
df_auditoria_data["n_linhas_posicoes_data"] = converter_serie_inteira(df_auditoria_data["n_linhas_posicoes_data"], 0)
df_auditoria_data["n_tickers_posicoes_data"] = converter_serie_inteira(df_auditoria_data["n_tickers_posicoes_data"], 0)
df_auditoria_data["n_linhas_invalidas_data"] = converter_serie_inteira(df_auditoria_data["n_linhas_invalidas_data"], 0)

df_auditoria_data["valor_carteira_investida"] = converter_serie_numerica(df_auditoria_data["valor_carteira_investida"], 0.0)
df_auditoria_data["patrimonio_total"] = converter_serie_numerica(df_auditoria_data["patrimonio_total"], np.nan)

df_auditoria_data["peso_carteira_investida_patrimonio"] = np.where(
    df_auditoria_data["patrimonio_total"].abs() > TOLERANCIA_MONETARIA_POSICOES,
    df_auditoria_data["valor_carteira_investida"] / df_auditoria_data["patrimonio_total"],
    0.0,
)

df_auditoria_data["desvio_valor_posicoes_vs_patrimonio"] = (
    df_auditoria_data["valor_mercado_posicoes"] - df_auditoria_data["valor_carteira_investida"]
)
df_auditoria_data["desvio_peso_posicoes_vs_patrimonio"] = (
    df_auditoria_data["peso_posicoes_patrimonio"] - df_auditoria_data["peso_carteira_investida_patrimonio"]
)

df_auditoria_data["flag_valor_posicoes_reconciliado"] = df_auditoria_data["desvio_valor_posicoes_vs_patrimonio"].abs() <= TOLERANCIA_MONETARIA_POSICOES
df_auditoria_data["flag_peso_posicoes_reconciliado"] = df_auditoria_data["desvio_peso_posicoes_vs_patrimonio"].abs() <= TOLERANCIA_PESO_POSICOES
df_auditoria_data["flag_sem_linhas_invalidas_data"] = df_auditoria_data["n_linhas_invalidas_data"] == 0

COLUNAS_GROUPBY_AUDITORIA = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_backtest",
    "ordem_exibicao",
]

validar_colunas_groupby(
    df_auditoria_data,
    COLUNAS_GROUPBY_AUDITORIA,
    "df_auditoria_data",
    "df_auditoria_historico_posicoes",
)

df_auditoria_historico_posicoes = (
    df_auditoria_data
    .groupby(
        COLUNAS_GROUPBY_AUDITORIA,
        as_index=False,
        dropna=False,
    )
    .agg(
        primeira_data_patrimonial=("data", "min"),
        ultima_data_patrimonial=("data", "max"),
        n_datas_patrimoniais=("data", "nunique"),
        n_datas_com_posicoes=("n_linhas_posicoes_data", lambda x: int((pd.to_numeric(x, errors="coerce") > 0).sum())),
        n_linhas_posicoes_total=("n_linhas_posicoes_data", "sum"),
        n_tickers_posicoes_maximo=("n_tickers_posicoes_data", "max"),
        n_linhas_invalidas_total=("n_linhas_invalidas_data", "sum"),
        desvio_maximo_valor_posicoes_vs_patrimonio=("desvio_valor_posicoes_vs_patrimonio", lambda x: float(np.nanmax(np.abs(pd.to_numeric(x, errors="coerce"))))),
        desvio_maximo_peso_posicoes_vs_patrimonio=("desvio_peso_posicoes_vs_patrimonio", lambda x: float(np.nanmax(np.abs(pd.to_numeric(x, errors="coerce"))))),
        n_datas_valor_posicoes_nao_reconciliado=("flag_valor_posicoes_reconciliado", lambda x: int((~x.astype(bool)).sum())),
        n_datas_peso_posicoes_nao_reconciliado=("flag_peso_posicoes_reconciliado", lambda x: int((~x.astype(bool)).sum())),
        n_datas_com_linhas_invalidas=("flag_sem_linhas_invalidas_data", lambda x: int((~x.astype(bool)).sum())),
    )
    .sort_values(["ordem_exibicao", "chave_estrategia"])
    .reset_index(drop=True)
)

df_auditoria_historico_posicoes["flag_valor_posicoes_reconciliado"] = df_auditoria_historico_posicoes["n_datas_valor_posicoes_nao_reconciliado"] == 0
df_auditoria_historico_posicoes["flag_peso_posicoes_reconciliado"] = df_auditoria_historico_posicoes["n_datas_peso_posicoes_nao_reconciliado"] == 0
df_auditoria_historico_posicoes["flag_sem_linhas_invalidas"] = df_auditoria_historico_posicoes["n_linhas_invalidas_total"] == 0
df_auditoria_historico_posicoes["flag_historico_posicoes_valido"] = (
    df_auditoria_historico_posicoes["flag_valor_posicoes_reconciliado"]
    & df_auditoria_historico_posicoes["flag_peso_posicoes_reconciliado"]
    & df_auditoria_historico_posicoes["flag_sem_linhas_invalidas"]
)

df_inconsistencias_historico_posicoes = (
    df_auditoria_historico_posicoes
    .loc[~df_auditoria_historico_posicoes["flag_historico_posicoes_valido"]]
    .copy()
    .reset_index(drop=True)
)

n_inconsistencias = int(len(df_inconsistencias_historico_posicoes))
n_estrategias_com_inconsistencia = int(df_inconsistencias_historico_posicoes["estrategia_id"].nunique()) if n_inconsistencias > 0 else 0
n_datas_valor_posicoes_nao_reconciliado = int(df_auditoria_historico_posicoes["n_datas_valor_posicoes_nao_reconciliado"].sum())
n_datas_peso_posicoes_nao_reconciliado = int(df_auditoria_historico_posicoes["n_datas_peso_posicoes_nao_reconciliado"].sum())
n_linhas_invalidas_total = int(df_auditoria_historico_posicoes["n_linhas_invalidas_total"].sum())

print(f"Inconsistências identificadas               : {n_inconsistencias:,}")
print(f"Estratégias com inconsistência              : {n_estrategias_com_inconsistencia:,}")
print(f"Datas com valor de posições divergente      : {n_datas_valor_posicoes_nao_reconciliado:,}")
print(f"Datas com peso de posições divergente       : {n_datas_peso_posicoes_nao_reconciliado:,}")
print(f"Linhas de posição histórica inválidas       : {n_linhas_invalidas_total:,}")
print("OK")

# ============================================================
# 9) Organização final dos outputs
# ============================================================

print("\n[9/10] Organização final dos outputs...")

df_regras_registro_historico_posicoes = df_regras_registro_historico_posicoes.sort_values("ordem_regra").reset_index(drop=True)

df_resumo_historico_posicoes_estrategia = df_resumo_historico_posicoes_estrategia.sort_values(
    ["ordem_exibicao", "chave_estrategia"]
).reset_index(drop=True)

df_resumo_historico_posicoes_ticker = df_resumo_historico_posicoes_ticker.sort_values(
    ["ordem_exibicao", "chave_estrategia", "classe_instrumento", "ticker"]
).reset_index(drop=True)

df_distribuicao_anual_posicoes = df_distribuicao_anual_posicoes.sort_values(
    ["ordem_exibicao", "chave_estrategia", "ano"]
).reset_index(drop=True)

df_auditoria_historico_posicoes = df_auditoria_historico_posicoes.sort_values(
    ["ordem_exibicao", "chave_estrategia"]
).reset_index(drop=True)

df_inconsistencias_historico_posicoes = df_inconsistencias_historico_posicoes.sort_values(
    ["ordem_exibicao", "chave_estrategia"]
).reset_index(drop=True)

print(f"Base histórica de posições organizada       : {len(df_historico_posicoes):,} linhas")
print(f"Resumo por estratégia organizado            : {len(df_resumo_historico_posicoes_estrategia):,} linhas")
print(f"Resumo por ticker organizado                : {len(df_resumo_historico_posicoes_ticker):,} linhas")
print(f"Auditoria organizada                        : {len(df_auditoria_historico_posicoes):,} linhas")
print(f"Tabela de inconsistências organizada        : {len(df_inconsistencias_historico_posicoes):,} linhas")
print("OK")

# ============================================================
# 10) Salvamento dos outputs e validação final
# ============================================================

print("\n[10/10] Salvamento dos outputs e validação final...")

salvar_dataframe(df_regras_registro_historico_posicoes, caminho_tbl_regras_registro_historico_posicoes, index=False)
salvar_dataframe(df_historico_posicoes, caminho_base_historico_posicoes, index=False)
salvar_dataframe(df_resumo_historico_posicoes_estrategia, caminho_tbl_resumo_historico_posicoes_estrategia, index=False)
salvar_dataframe(df_resumo_historico_posicoes_ticker, caminho_tbl_resumo_historico_posicoes_ticker, index=False)
salvar_dataframe(df_distribuicao_anual_posicoes, caminho_tbl_distribuicao_anual_posicoes, index=False)
salvar_dataframe(df_auditoria_historico_posicoes, caminho_tbl_auditoria_historico_posicoes, index=False)
salvar_dataframe(df_inconsistencias_historico_posicoes, caminho_tbl_inconsistencias_historico_posicoes, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

print("\nRegras formais do registro histórico de posições:")
print(df_regras_registro_historico_posicoes.to_string(index=False))

print("\nBase histórica de posições - amostra:")
print(df_historico_posicoes.head(30).to_string(index=False))

print("\nResumo histórico de posições por estratégia:")
print(df_resumo_historico_posicoes_estrategia.to_string(index=False))

print("\nResumo histórico de posições por ticker - amostra:")
print(df_resumo_historico_posicoes_ticker.head(40).to_string(index=False))

print("\nDistribuição anual das posições - amostra:")
print(df_distribuicao_anual_posicoes.head(40).to_string(index=False))

print("\nAuditoria do histórico de posições:")
print(df_auditoria_historico_posicoes.to_string(index=False))

print("\nInconsistências do histórico de posições - amostra:")
print(df_inconsistencias_historico_posicoes.head(30).to_string(index=False))

print("\nArquivos salvos na subetapa 10.4:")
print(f"- {caminho_tbl_regras_registro_historico_posicoes}")
print(f"- {caminho_base_historico_posicoes}")
print(f"- {caminho_tbl_resumo_historico_posicoes_estrategia}")
print(f"- {caminho_tbl_resumo_historico_posicoes_ticker}")
print(f"- {caminho_tbl_distribuicao_anual_posicoes}")
print(f"- {caminho_tbl_auditoria_historico_posicoes}")
print(f"- {caminho_tbl_inconsistencias_historico_posicoes}")

print("\nETAPA 10.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 10.4 - REGISTRO HISTÓRICO DAS POSIÇÕES

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição dos caminhos de entrada e saída da subetapa...
Entrada - compras finais 10.3                         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_3_base_compras_finais.parquet
Entrada - catálogo patrimonial 9.7                    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_7_tbl_catalogo_curvas_patrimoniais.parquet
Entrada - curvas patrimoniais consolidadas 9.7        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_7_base_curvas_patrimoniais_consolidadas.parquet
Entrada - posições capitulação 9.1                    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_m

## Etapa 10.5) Registro de Concentração por Empresa, Setor, Subsetor e Segmento

In [51]:
%%time
# ============================================================
# Etapa 10.5) Registro de Concentração por Empresa, Setor, Subsetor e Segmento
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 10.5 - REGISTRO DE CONCENTRAÇÃO POR EMPRESA, SETOR, SUBSETOR E SEGMENTO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/11] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Liberação de memória das etapas anteriores
# ============================================================

print("\n[2/11] Liberação de memória das etapas anteriores...")

nomes_preservados_memoria = {
    "pd",
    "np",
    "gerar_caminho_arquivo",
    "salvar_dataframe",
}

tipos_removiveis_memoria = (
    pd.DataFrame,
    pd.Series,
    pd.Index,
    np.ndarray,
)

objetos_globais_antes = list(globals().items())
variaveis_removidas_memoria = []
memoria_estimativa_removida_bytes = 0

for nome_variavel, objeto_variavel in objetos_globais_antes:
    if nome_variavel in nomes_preservados_memoria:
        continue
    if nome_variavel.startswith("_"):
        continue
    if isinstance(objeto_variavel, tipos_removiveis_memoria):
        try:
            if isinstance(objeto_variavel, pd.DataFrame):
                memoria_estimativa_removida_bytes += int(objeto_variavel.memory_usage(deep=True).sum())
            elif isinstance(objeto_variavel, pd.Series):
                memoria_estimativa_removida_bytes += int(objeto_variavel.memory_usage(deep=True))
            elif isinstance(objeto_variavel, pd.Index):
                memoria_estimativa_removida_bytes += int(objeto_variavel.memory_usage(deep=True))
            elif isinstance(objeto_variavel, np.ndarray):
                memoria_estimativa_removida_bytes += int(objeto_variavel.nbytes)
        except Exception:
            pass
        try:
            del globals()[nome_variavel]
            variaveis_removidas_memoria.append(nome_variavel)
        except Exception:
            pass

gc_modulo = globals().get("gc", None)
if gc_modulo is None:
    try:
        gc_modulo = __import__("gc")
    except Exception:
        gc_modulo = None

objetos_liberados_gc = pd.NA
if gc_modulo is not None:
    objetos_liberados_gc = gc_modulo.collect()

memoria_pyarrow_liberada = pd.NA
try:
    pyarrow_modulo = __import__("pyarrow")
    memoria_pyarrow_antes = int(pyarrow_modulo.default_memory_pool().bytes_allocated())
    pyarrow_modulo.default_memory_pool().release_unused()
    memoria_pyarrow_depois = int(pyarrow_modulo.default_memory_pool().bytes_allocated())
    memoria_pyarrow_liberada = memoria_pyarrow_antes - memoria_pyarrow_depois
except Exception:
    pass

print(f"Variáveis globais removidas da memória        : {len(variaveis_removidas_memoria):,}")
print(f"Memória estimada removida                    : {memoria_estimativa_removida_bytes / (1024 ** 2):,.2f} MB")
print(f"Objetos liberados pelo garbage collector     : {objetos_liberados_gc}")
print(f"Memória liberada no pool do PyArrow          : {memoria_pyarrow_liberada}")
print("OK")

# ============================================================
# 3) Definição dos caminhos de entrada e saída da subetapa
# ============================================================

print("\n[3/11] Definição dos caminhos de entrada e saída da subetapa...")

caminho_base_historico_posicoes = gerar_caminho_arquivo(
    etapa=10,
    subetapa=4,
    tipo_arquivo="base",
    nome="historico_posicoes",
)

caminho_tbl_auditoria_historico_posicoes = gerar_caminho_arquivo(
    etapa=10,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="auditoria_historico_posicoes",
)

caminho_tbl_regras_registro_concentracao_carteiras = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="regras_registro_concentracao_carteiras",
)

caminho_base_concentracao_empresa = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="base",
    nome="concentracao_empresa",
)

caminho_base_concentracao_setor = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="base",
    nome="concentracao_setor",
)

caminho_base_concentracao_subsetor = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="base",
    nome="concentracao_subsetor",
)

caminho_base_concentracao_segmento = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="base",
    nome="concentracao_segmento",
)

caminho_base_concentracao_classe_instrumento = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="base",
    nome="concentracao_classe_instrumento",
)

caminho_tbl_metricas_concentracao_diaria = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="metricas_concentracao_diaria",
)

caminho_tbl_resumo_concentracao_estrategia = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="resumo_concentracao_estrategia",
)

caminho_tbl_distribuicao_anual_concentracao = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_concentracao",
)

caminho_tbl_auditoria_concentracao_carteiras = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="auditoria_concentracao_carteiras",
)

caminho_tbl_inconsistencias_concentracao_carteiras = gerar_caminho_arquivo(
    etapa=10,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="inconsistencias_concentracao_carteiras",
)

print(f"Entrada - histórico de posições 10.4                 : {caminho_base_historico_posicoes}")
print(f"Entrada - auditoria do histórico de posições 10.4     : {caminho_tbl_auditoria_historico_posicoes}")
print(f"Saída   - regras de concentração                      : {caminho_tbl_regras_registro_concentracao_carteiras}")
print(f"Saída   - concentração por empresa                    : {caminho_base_concentracao_empresa}")
print(f"Saída   - concentração por setor                      : {caminho_base_concentracao_setor}")
print(f"Saída   - concentração por subsetor                   : {caminho_base_concentracao_subsetor}")
print(f"Saída   - concentração por segmento                   : {caminho_base_concentracao_segmento}")
print(f"Saída   - concentração por classe de instrumento      : {caminho_base_concentracao_classe_instrumento}")
print(f"Saída   - métricas diárias de concentração            : {caminho_tbl_metricas_concentracao_diaria}")
print(f"Saída   - resumo por estratégia                       : {caminho_tbl_resumo_concentracao_estrategia}")
print(f"Saída   - distribuição anual                          : {caminho_tbl_distribuicao_anual_concentracao}")
print(f"Saída   - auditoria                                   : {caminho_tbl_auditoria_concentracao_carteiras}")
print(f"Saída   - inconsistências                             : {caminho_tbl_inconsistencias_concentracao_carteiras}")
print("OK")

# ============================================================
# 4) Carga dos inputs da subetapa
# ============================================================

print("\n[4/11] Carga dos inputs da subetapa...")

COLUNAS_LEITURA_HISTORICO_POSICOES_10_5 = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_backtest",
    "ordem_exibicao",
    "data",
    "ano",
    "mes",
    "classe_instrumento",
    "grupo_registro_carteira",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "valor_mercado_posicao",
    "valor_investido_acumulado_ticker",
    "resultado_nao_realizado_posicao",
    "patrimonio_total",
    "valor_carteira_investida_total",
    "saldo_caixa",
    "saldo_caixa_fim_dia",
    "peso_posicao_no_patrimonio",
    "peso_posicao_na_carteira_investida",
    "flag_posicao_historica_valida",
]

def ler_parquet_colunas_eficiente(caminho_arquivo, colunas=None, nome_base="base"):
    try:
        parquet_modulo = __import__("pyarrow.parquet", fromlist=["read_table"])
        tabela_arrow = parquet_modulo.read_table(caminho_arquivo, columns=colunas)
        df_lido = tabela_arrow.to_pandas(
            ignore_metadata=True,
            split_blocks=True,
            self_destruct=True,
        )
        del tabela_arrow
        if gc_modulo is not None:
            gc_modulo.collect()
        return df_lido, f"pyarrow_colunas_necessarias_{nome_base}"
    except MemoryError:
        raise
    except Exception:
        df_lido = pd.read_parquet(caminho_arquivo, columns=colunas)
        return df_lido, f"pandas_read_parquet_colunas_necessarias_{nome_base}"

df_historico_posicoes, fonte_historico_posicoes = ler_parquet_colunas_eficiente(
    caminho_base_historico_posicoes,
    colunas=COLUNAS_LEITURA_HISTORICO_POSICOES_10_5,
    nome_base="historico_posicoes_10_4",
)

df_auditoria_historico_posicoes, fonte_auditoria_historico_posicoes = ler_parquet_colunas_eficiente(
    caminho_tbl_auditoria_historico_posicoes,
    colunas=None,
    nome_base="auditoria_historico_posicoes_10_4",
)

print(f"Histórico de posições 10.4                 : {df_historico_posicoes.shape[0]:,} linhas x {df_historico_posicoes.shape[1]:,} colunas")
print(f"Fonte do histórico de posições             : {fonte_historico_posicoes}")
print(f"Auditoria do histórico de posições 10.4    : {df_auditoria_historico_posicoes.shape[0]:,} linhas x {df_auditoria_historico_posicoes.shape[1]:,} colunas")
print(f"Fonte da auditoria 10.4                    : {fonte_auditoria_historico_posicoes}")
print("OK")

# ============================================================
# 5) Funções auxiliares e parâmetros operacionais
# ============================================================

print("\n[5/11] Funções auxiliares e parâmetros operacionais...")

TOLERANCIA_MONETARIA_CONCENTRACAO = 0.01
TOLERANCIA_PESO_CONCENTRACAO = 0.000001
VALOR_NAO_CLASSIFICADO = "Não Classificado"

COLUNAS_CHAVE_ESTRATEGIA = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "origem_backtest",
    "ordem_exibicao",
]

COLUNAS_CHAVE_DATA = COLUNAS_CHAVE_ESTRATEGIA + [
    "data",
    "ano",
    "mes",
]

COLUNAS_POSICOES_OBRIGATORIAS = COLUNAS_CHAVE_DATA + [
    "classe_instrumento",
    "grupo_registro_carteira",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "valor_mercado_posicao",
    "valor_investido_acumulado_ticker",
    "resultado_nao_realizado_posicao",
    "patrimonio_total",
    "valor_carteira_investida_total",
    "peso_posicao_no_patrimonio",
    "peso_posicao_na_carteira_investida",
    "flag_posicao_historica_valida",
]

COLUNAS_BASE_CONCENTRACAO = COLUNAS_CHAVE_DATA + [
    "nivel_concentracao",
    "grupo_concentracao",
    "classe_instrumento",
    "grupo_registro_carteira",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "valor_mercado_concentracao",
    "valor_custo_concentracao",
    "resultado_nao_realizado_concentracao",
    "patrimonio_total",
    "valor_carteira_investida_total",
    "saldo_caixa",
    "saldo_caixa_fim_dia",
    "peso_concentracao_no_patrimonio",
    "peso_concentracao_na_carteira_investida",
    "n_linhas_posicao_concentracao",
    "n_tickers_concentracao",
    "n_empresas_concentracao",
    "flag_valor_concentracao_valido",
    "flag_peso_patrimonio_concentracao_valido",
    "flag_peso_carteira_investida_concentracao_valido",
    "flag_concentracao_valida",
]

def validar_colunas_obrigatorias(df, colunas_obrigatorias, nome_base):
    colunas_ausentes = [coluna for coluna in colunas_obrigatorias if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"A base {nome_base} não contém as colunas obrigatórias: {colunas_ausentes}")

def validar_colunas_groupby(df, colunas_groupby, nome_base, nome_operacao):
    colunas_ausentes = [coluna for coluna in colunas_groupby if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        colunas_disponiveis = list(df.columns)
        raise ValueError(
            f"A operação {nome_operacao} na base {nome_base} não pode ser executada. "
            f"Colunas ausentes no agrupamento: {colunas_ausentes}. "
            f"Colunas disponíveis: {colunas_disponiveis}"
        )

def validar_colunas_selecao(df, colunas_selecao, nome_base, nome_operacao):
    colunas_ausentes = [coluna for coluna in colunas_selecao if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        colunas_disponiveis = list(df.columns)
        raise ValueError(
            f"A operação {nome_operacao} na base {nome_base} não pode ser executada. "
            f"Colunas ausentes na seleção: {colunas_ausentes}. "
            f"Colunas disponíveis: {colunas_disponiveis}"
        )

def converter_serie_numerica(serie, valor_default=0.0):
    if pd.api.types.is_numeric_dtype(serie):
        return serie.fillna(valor_default)
    return pd.to_numeric(serie, errors="coerce").fillna(valor_default)

def converter_serie_inteira(serie, valor_default=0):
    return pd.to_numeric(serie, errors="coerce").fillna(valor_default).astype(int)

def normalizar_coluna_categoria(df, coluna, valor_default=VALOR_NAO_CLASSIFICADO):
    serie = df[coluna]
    if isinstance(serie.dtype, pd.CategoricalDtype):
        if valor_default not in serie.cat.categories:
            serie = serie.cat.add_categories([valor_default])
        df[coluna] = serie.fillna(valor_default)
        return df

    serie = serie.astype("object")
    serie = serie.where(serie.notna(), valor_default)
    serie = serie.replace({"": valor_default, "<NA>": valor_default, "nan": valor_default, "None": valor_default})
    df[coluna] = serie.astype("category")
    return df

def assegurar_coluna(df, coluna, valor_default=pd.NA):
    if coluna not in df.columns:
        df[coluna] = valor_default
    return df

def compactar_colunas_texto_para_categoria(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            df = normalizar_coluna_categoria(df, coluna)
    return df

def grupo_empresa(df):
    return df["issuer_code"].astype("object").astype(str) + " - " + df["nome"].astype("object").astype(str)

def grupo_subsetor_hierarquico(df):
    return df["setor"].astype("object").astype(str) + " > " + df["subsetor"].astype("object").astype(str)

def grupo_segmento_hierarquico(df):
    return (
        df["setor"].astype("object").astype(str)
        + " > "
        + df["subsetor"].astype("object").astype(str)
        + " > "
        + df["segmento"].astype("object").astype(str)
    )

def preparar_dataframe_para_parquet_incremental(df):
    df_out = df.reset_index(drop=True)
    for coluna in df_out.columns:
        if isinstance(df_out[coluna].dtype, pd.CategoricalDtype):
            df_out[coluna] = df_out[coluna].astype("string")
        elif df_out[coluna].dtype == "object":
            df_out[coluna] = df_out[coluna].astype("string")
    return df_out

def escrever_parquet_incremental(df, caminho_arquivo, escritores_parquet):
    pyarrow_modulo = __import__("pyarrow")
    parquet_modulo = __import__("pyarrow.parquet", fromlist=["ParquetWriter"])

    chave_caminho = str(caminho_arquivo)
    df_escrita = preparar_dataframe_para_parquet_incremental(df)
    tabela_arrow = pyarrow_modulo.Table.from_pandas(df_escrita, preserve_index=False)

    if chave_caminho not in escritores_parquet:
        if hasattr(caminho_arquivo, "parent"):
            caminho_arquivo.parent.mkdir(parents=True, exist_ok=True)
        escritores_parquet[chave_caminho] = parquet_modulo.ParquetWriter(chave_caminho, tabela_arrow.schema)

    escritores_parquet[chave_caminho].write_table(tabela_arrow)

    del df_escrita, tabela_arrow
    if gc_modulo is not None:
        gc_modulo.collect()

def fechar_escritores_parquet(escritores_parquet):
    for escritor in escritores_parquet.values():
        escritor.close()
    escritores_parquet.clear()

def montar_base_concentracao_chunk(df_base, nivel_concentracao, colunas_dimensao, coluna_grupo_concentracao):
    colunas_groupby = COLUNAS_CHAVE_DATA + colunas_dimensao
    validar_colunas_groupby(
        df_base,
        colunas_groupby,
        "df_posicoes_validas_chunk",
        f"base_concentracao_{nivel_concentracao}",
    )

    df_out = (
        df_base
        .groupby(colunas_groupby, as_index=False, dropna=False, observed=True, sort=False)
        .agg(
            valor_mercado_concentracao=("valor_mercado_posicao", "sum"),
            valor_custo_concentracao=("valor_investido_acumulado_ticker", "sum"),
            resultado_nao_realizado_concentracao=("resultado_nao_realizado_posicao", "sum"),
            patrimonio_total=("patrimonio_total", "last"),
            valor_carteira_investida_total=("valor_carteira_investida_total", "last"),
            saldo_caixa=("saldo_caixa", "last"),
            saldo_caixa_fim_dia=("saldo_caixa_fim_dia", "last"),
            n_linhas_posicao_concentracao=("ticker", "count"),
            n_tickers_concentracao=("ticker", "nunique"),
            n_empresas_concentracao=("issuer_code", "nunique"),
        )
    )

    df_out["nivel_concentracao"] = nivel_concentracao

    if callable(coluna_grupo_concentracao):
        df_out["grupo_concentracao"] = coluna_grupo_concentracao(df_out)
    else:
        df_out["grupo_concentracao"] = df_out[coluna_grupo_concentracao].astype("object").astype(str)

    for coluna in ["classe_instrumento", "grupo_registro_carteira", "issuer_code", "nome", "setor", "subsetor", "segmento"]:
        df_out = assegurar_coluna(df_out, coluna, VALOR_NAO_CLASSIFICADO)
        df_out = normalizar_coluna_categoria(df_out, coluna)

    df_out["grupo_concentracao"] = df_out["grupo_concentracao"].astype("object").replace({"": VALOR_NAO_CLASSIFICADO})

    for coluna in [
        "valor_mercado_concentracao",
        "valor_custo_concentracao",
        "resultado_nao_realizado_concentracao",
        "patrimonio_total",
        "valor_carteira_investida_total",
        "saldo_caixa",
        "saldo_caixa_fim_dia",
    ]:
        df_out[coluna] = converter_serie_numerica(df_out[coluna], 0.0)

    for coluna in ["n_linhas_posicao_concentracao", "n_tickers_concentracao", "n_empresas_concentracao"]:
        df_out[coluna] = converter_serie_inteira(df_out[coluna], 0)

    df_out["peso_concentracao_no_patrimonio"] = np.where(
        df_out["patrimonio_total"].abs() > TOLERANCIA_MONETARIA_CONCENTRACAO,
        df_out["valor_mercado_concentracao"] / df_out["patrimonio_total"],
        0.0,
    )

    df_out["peso_concentracao_na_carteira_investida"] = np.where(
        df_out["valor_carteira_investida_total"].abs() > TOLERANCIA_MONETARIA_CONCENTRACAO,
        df_out["valor_mercado_concentracao"] / df_out["valor_carteira_investida_total"],
        0.0,
    )

    df_out["flag_valor_concentracao_valido"] = df_out["valor_mercado_concentracao"] >= -TOLERANCIA_MONETARIA_CONCENTRACAO
    df_out["flag_peso_patrimonio_concentracao_valido"] = df_out["peso_concentracao_no_patrimonio"].between(
        -TOLERANCIA_PESO_CONCENTRACAO,
        1.0 + TOLERANCIA_PESO_CONCENTRACAO,
    )
    df_out["flag_peso_carteira_investida_concentracao_valido"] = df_out["peso_concentracao_na_carteira_investida"].between(
        -TOLERANCIA_PESO_CONCENTRACAO,
        1.0 + TOLERANCIA_PESO_CONCENTRACAO,
    )
    df_out["flag_concentracao_valida"] = (
        df_out["flag_valor_concentracao_valido"]
        & df_out["flag_peso_patrimonio_concentracao_valido"]
        & df_out["flag_peso_carteira_investida_concentracao_valido"]
    )

    for coluna in COLUNAS_BASE_CONCENTRACAO:
        df_out = assegurar_coluna(df_out, coluna)

    df_out = df_out.reindex(columns=COLUNAS_BASE_CONCENTRACAO, copy=False)
    validar_colunas_obrigatorias(df_out, COLUNAS_BASE_CONCENTRACAO, f"base_concentracao_{nivel_concentracao}")

    return df_out

def calcular_metricas_diarias(df_base, prefixo):
    colunas_necessarias = COLUNAS_CHAVE_DATA + [
        "peso_concentracao_na_carteira_investida",
        "peso_concentracao_no_patrimonio",
    ]
    validar_colunas_selecao(
        df_base,
        colunas_necessarias,
        f"base_concentracao_{prefixo}",
        f"metricas_diarias_{prefixo}",
    )

    df_tmp = df_base[colunas_necessarias].copy()
    df_tmp["peso_concentracao_na_carteira_investida"] = converter_serie_numerica(df_tmp["peso_concentracao_na_carteira_investida"], 0.0)
    df_tmp["peso_concentracao_no_patrimonio"] = converter_serie_numerica(df_tmp["peso_concentracao_no_patrimonio"], 0.0)
    df_tmp["peso_carteira_investida_quadrado"] = df_tmp["peso_concentracao_na_carteira_investida"] ** 2
    df_tmp["peso_patrimonio_quadrado"] = df_tmp["peso_concentracao_no_patrimonio"] ** 2

    df_metricas = (
        df_tmp
        .groupby(COLUNAS_CHAVE_DATA, as_index=False, dropna=False, observed=True, sort=False)
        .agg(
            n_grupos_concentracao=("peso_concentracao_na_carteira_investida", "count"),
            maior_peso_carteira_investida=("peso_concentracao_na_carteira_investida", "max"),
            top_5_peso_carteira_investida=("peso_concentracao_na_carteira_investida", lambda x: float(x.nlargest(5).sum())),
            hhi_carteira_investida=("peso_carteira_investida_quadrado", "sum"),
            maior_peso_patrimonio=("peso_concentracao_no_patrimonio", "max"),
            top_5_peso_patrimonio=("peso_concentracao_no_patrimonio", lambda x: float(x.nlargest(5).sum())),
            hhi_patrimonio=("peso_patrimonio_quadrado", "sum"),
        )
    )

    df_metricas = df_metricas.rename(
        columns={
            "n_grupos_concentracao": f"n_{prefixo}_em_carteira",
            "maior_peso_carteira_investida": f"maior_peso_{prefixo}_carteira_investida",
            "top_5_peso_carteira_investida": f"top_5_peso_{prefixo}_carteira_investida",
            "hhi_carteira_investida": f"hhi_{prefixo}_carteira_investida",
            "maior_peso_patrimonio": f"maior_peso_{prefixo}_patrimonio",
            "top_5_peso_patrimonio": f"top_5_peso_{prefixo}_patrimonio",
            "hhi_patrimonio": f"hhi_{prefixo}_patrimonio",
        }
    )

    df_metricas[f"n_{prefixo}_em_carteira"] = converter_serie_inteira(df_metricas[f"n_{prefixo}_em_carteira"], 0)

    for coluna in [coluna for coluna in df_metricas.columns if coluna not in COLUNAS_CHAVE_DATA and coluna != f"n_{prefixo}_em_carteira"]:
        df_metricas[coluna] = converter_serie_numerica(df_metricas[coluna], 0.0)

    del df_tmp
    return df_metricas

def montar_pesos_por_classe(df_classe):
    colunas_necessarias = COLUNAS_CHAVE_DATA + [
        "classe_instrumento",
        "peso_concentracao_no_patrimonio",
        "peso_concentracao_na_carteira_investida",
    ]
    validar_colunas_selecao(
        df_classe,
        colunas_necessarias,
        "base_concentracao_classe_instrumento",
        "pesos_por_classe",
    )

    df_tmp = df_classe[colunas_necessarias].copy()
    df_tmp["classe_instrumento_normalizada"] = (
        df_tmp["classe_instrumento"]
        .astype("object")
        .astype(str)
        .str.strip()
        .str.lower()
    )
    df_tmp["peso_concentracao_no_patrimonio"] = converter_serie_numerica(
        df_tmp["peso_concentracao_no_patrimonio"],
        0.0,
    )
    df_tmp["peso_concentracao_na_carteira_investida"] = converter_serie_numerica(
        df_tmp["peso_concentracao_na_carteira_investida"],
        0.0,
    )

    df_tmp["peso_acoes_no_patrimonio_linha"] = np.where(
        df_tmp["classe_instrumento_normalizada"].eq("acao"),
        df_tmp["peso_concentracao_no_patrimonio"],
        0.0,
    )
    df_tmp["peso_benchmark_no_patrimonio_linha"] = np.where(
        df_tmp["classe_instrumento_normalizada"].eq("benchmark"),
        df_tmp["peso_concentracao_no_patrimonio"],
        0.0,
    )
    df_tmp["peso_taxa_referencia_no_patrimonio_linha"] = np.where(
        df_tmp["classe_instrumento_normalizada"].eq("taxa_referencia"),
        df_tmp["peso_concentracao_no_patrimonio"],
        0.0,
    )
    df_tmp["peso_acoes_na_carteira_investida_linha"] = np.where(
        df_tmp["classe_instrumento_normalizada"].eq("acao"),
        df_tmp["peso_concentracao_na_carteira_investida"],
        0.0,
    )
    df_tmp["peso_benchmark_na_carteira_investida_linha"] = np.where(
        df_tmp["classe_instrumento_normalizada"].eq("benchmark"),
        df_tmp["peso_concentracao_na_carteira_investida"],
        0.0,
    )
    df_tmp["peso_taxa_referencia_na_carteira_investida_linha"] = np.where(
        df_tmp["classe_instrumento_normalizada"].eq("taxa_referencia"),
        df_tmp["peso_concentracao_na_carteira_investida"],
        0.0,
    )

    df_out = (
        df_tmp
        .groupby(COLUNAS_CHAVE_DATA, as_index=False, dropna=False, observed=True, sort=False)
        .agg(
            peso_acoes_no_patrimonio=("peso_acoes_no_patrimonio_linha", "sum"),
            peso_benchmark_no_patrimonio=("peso_benchmark_no_patrimonio_linha", "sum"),
            peso_taxa_referencia_no_patrimonio=("peso_taxa_referencia_no_patrimonio_linha", "sum"),
            peso_acoes_na_carteira_investida=("peso_acoes_na_carteira_investida_linha", "sum"),
            peso_benchmark_na_carteira_investida=("peso_benchmark_na_carteira_investida_linha", "sum"),
            peso_taxa_referencia_na_carteira_investida=("peso_taxa_referencia_na_carteira_investida_linha", "sum"),
        )
        .reset_index(drop=True)
    )

    for coluna in [
        "peso_acoes_no_patrimonio",
        "peso_benchmark_no_patrimonio",
        "peso_taxa_referencia_no_patrimonio",
        "peso_acoes_na_carteira_investida",
        "peso_benchmark_na_carteira_investida",
        "peso_taxa_referencia_na_carteira_investida",
    ]:
        df_out[coluna] = converter_serie_numerica(df_out[coluna], 0.0)

    del df_tmp
    return df_out

def auditar_base_concentracao(df_concentracao, df_referencia_data, nivel_concentracao):
    colunas_necessarias = COLUNAS_CHAVE_DATA + [
        "valor_mercado_concentracao",
        "peso_concentracao_no_patrimonio",
        "peso_concentracao_na_carteira_investida",
        "flag_concentracao_valida",
    ]
    validar_colunas_selecao(
        df_concentracao,
        colunas_necessarias,
        f"base_concentracao_{nivel_concentracao}",
        f"auditoria_{nivel_concentracao}",
    )

    df_agregado = (
        df_concentracao
        .groupby(COLUNAS_CHAVE_DATA, as_index=False, dropna=False, observed=True, sort=False)
        .agg(
            valor_mercado_concentracao_total=("valor_mercado_concentracao", "sum"),
            peso_concentracao_patrimonio_total=("peso_concentracao_no_patrimonio", "sum"),
            peso_concentracao_carteira_investida_total=("peso_concentracao_na_carteira_investida", "sum"),
            n_linhas_concentracao_total=("grupo_concentracao", "count"),
            n_linhas_concentracao_invalida=("flag_concentracao_valida", lambda x: int((~x.fillna(False).astype(bool)).sum())),
        )
    )

    df_data = df_referencia_data.merge(
        df_agregado,
        on=COLUNAS_CHAVE_DATA,
        how="left",
        validate="one_to_one",
    )

    for coluna in [
        "valor_mercado_concentracao_total",
        "peso_concentracao_patrimonio_total",
        "peso_concentracao_carteira_investida_total",
    ]:
        df_data[coluna] = converter_serie_numerica(df_data[coluna], 0.0)

    for coluna in ["n_linhas_concentracao_total", "n_linhas_concentracao_invalida"]:
        df_data[coluna] = converter_serie_inteira(df_data[coluna], 0)

    df_data["desvio_valor_concentracao"] = df_data["valor_mercado_concentracao_total"] - df_data["valor_mercado_posicoes_data"]
    df_data["desvio_peso_patrimonio_concentracao"] = df_data["peso_concentracao_patrimonio_total"] - df_data["peso_posicoes_patrimonio_data"]
    df_data["desvio_peso_carteira_investida_concentracao"] = df_data["peso_concentracao_carteira_investida_total"] - df_data["peso_posicoes_carteira_investida_data"]

    df_data["flag_valor_concentracao_reconciliado"] = df_data["desvio_valor_concentracao"].abs() <= TOLERANCIA_MONETARIA_CONCENTRACAO
    df_data["flag_peso_patrimonio_concentracao_reconciliado"] = df_data["desvio_peso_patrimonio_concentracao"].abs() <= TOLERANCIA_PESO_CONCENTRACAO
    df_data["flag_peso_carteira_investida_concentracao_reconciliado"] = df_data["desvio_peso_carteira_investida_concentracao"].abs() <= TOLERANCIA_PESO_CONCENTRACAO
    df_data["flag_sem_linhas_concentracao_invalidas"] = df_data["n_linhas_concentracao_invalida"] == 0

    df_auditoria = (
        df_data
        .groupby(COLUNAS_CHAVE_ESTRATEGIA, as_index=False, dropna=False, observed=True, sort=False)
        .agg(
            primeira_data_auditada=("data", "min"),
            ultima_data_auditada=("data", "max"),
            n_datas_auditadas=("data", "nunique"),
            n_linhas_concentracao_total=("n_linhas_concentracao_total", "sum"),
            n_linhas_concentracao_invalida=("n_linhas_concentracao_invalida", "sum"),
            desvio_maximo_valor_concentracao=("desvio_valor_concentracao", lambda x: float(np.nanmax(np.abs(converter_serie_numerica(x, 0.0))))),
            desvio_maximo_peso_patrimonio_concentracao=("desvio_peso_patrimonio_concentracao", lambda x: float(np.nanmax(np.abs(converter_serie_numerica(x, 0.0))))),
            desvio_maximo_peso_carteira_investida_concentracao=("desvio_peso_carteira_investida_concentracao", lambda x: float(np.nanmax(np.abs(converter_serie_numerica(x, 0.0))))),
            n_datas_valor_concentracao_nao_reconciliado=("flag_valor_concentracao_reconciliado", lambda x: int((~x.fillna(False).astype(bool)).sum())),
            n_datas_peso_patrimonio_concentracao_nao_reconciliado=("flag_peso_patrimonio_concentracao_reconciliado", lambda x: int((~x.fillna(False).astype(bool)).sum())),
            n_datas_peso_carteira_investida_concentracao_nao_reconciliado=("flag_peso_carteira_investida_concentracao_reconciliado", lambda x: int((~x.fillna(False).astype(bool)).sum())),
            n_datas_com_linhas_concentracao_invalidas=("flag_sem_linhas_concentracao_invalidas", lambda x: int((~x.fillna(False).astype(bool)).sum())),
        )
    )

    df_auditoria["nivel_concentracao"] = nivel_concentracao

    for coluna in [
        "n_datas_auditadas",
        "n_linhas_concentracao_total",
        "n_linhas_concentracao_invalida",
        "n_datas_valor_concentracao_nao_reconciliado",
        "n_datas_peso_patrimonio_concentracao_nao_reconciliado",
        "n_datas_peso_carteira_investida_concentracao_nao_reconciliado",
        "n_datas_com_linhas_concentracao_invalidas",
    ]:
        df_auditoria[coluna] = converter_serie_inteira(df_auditoria[coluna], 0)

    df_auditoria["flag_valor_concentracao_reconciliado"] = df_auditoria["n_datas_valor_concentracao_nao_reconciliado"] == 0
    df_auditoria["flag_peso_patrimonio_concentracao_reconciliado"] = df_auditoria["n_datas_peso_patrimonio_concentracao_nao_reconciliado"] == 0
    df_auditoria["flag_peso_carteira_investida_concentracao_reconciliado"] = df_auditoria["n_datas_peso_carteira_investida_concentracao_nao_reconciliado"] == 0
    df_auditoria["flag_sem_linhas_concentracao_invalidas"] = df_auditoria["n_linhas_concentracao_invalida"] == 0
    df_auditoria["flag_concentracao_reconciliada"] = (
        df_auditoria["flag_valor_concentracao_reconciliado"]
        & df_auditoria["flag_peso_patrimonio_concentracao_reconciliado"]
        & df_auditoria["flag_peso_carteira_investida_concentracao_reconciliado"]
        & df_auditoria["flag_sem_linhas_concentracao_invalidas"]
    )

    del df_agregado, df_data
    return df_auditoria

print(f"Tolerância monetária da concentração : {TOLERANCIA_MONETARIA_CONCENTRACAO}")
print(f"Tolerância de pesos da concentração  : {TOLERANCIA_PESO_CONCENTRACAO}")
print("OK")

# ============================================================
# 6) Preparação e validação estrutural da base de posições
# ============================================================

print("\n[6/11] Preparação e validação estrutural da base de posições...")

validar_colunas_obrigatorias(
    df_historico_posicoes,
    COLUNAS_POSICOES_OBRIGATORIAS,
    "10_4_base_historico_posicoes",
)

validar_colunas_obrigatorias(
    df_auditoria_historico_posicoes,
    ["estrategia_id", "flag_historico_posicoes_valido"],
    "10_4_tbl_auditoria_historico_posicoes",
)

colunas_texto_posicoes = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "origem_backtest",
    "classe_instrumento",
    "grupo_registro_carteira",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
]

df_historico_posicoes["data"] = pd.to_datetime(df_historico_posicoes["data"], errors="coerce")
df_historico_posicoes["ano"] = converter_serie_inteira(df_historico_posicoes["ano"], 0)
df_historico_posicoes["mes"] = converter_serie_inteira(df_historico_posicoes["mes"], 0)
df_historico_posicoes = compactar_colunas_texto_para_categoria(df_historico_posicoes, colunas_texto_posicoes)

for coluna in [
    "valor_mercado_posicao",
    "valor_investido_acumulado_ticker",
    "resultado_nao_realizado_posicao",
    "patrimonio_total",
    "valor_carteira_investida_total",
    "peso_posicao_no_patrimonio",
    "peso_posicao_na_carteira_investida",
]:
    df_historico_posicoes[coluna] = converter_serie_numerica(df_historico_posicoes[coluna], 0.0)

for coluna in ["saldo_caixa", "saldo_caixa_fim_dia"]:
    df_historico_posicoes = assegurar_coluna(df_historico_posicoes, coluna, 0.0)
    df_historico_posicoes[coluna] = converter_serie_numerica(df_historico_posicoes[coluna], 0.0)

flag_posicao_historica_valida = df_historico_posicoes["flag_posicao_historica_valida"].fillna(False).astype(bool)
flag_valor_positivo = df_historico_posicoes["valor_mercado_posicao"] > TOLERANCIA_MONETARIA_CONCENTRACAO
flag_posicoes_validas = flag_posicao_historica_valida & flag_valor_positivo

if bool(flag_posicoes_validas.all()):
    df_posicoes_validas = df_historico_posicoes
else:
    df_posicoes_validas = df_historico_posicoes.loc[flag_posicoes_validas].reset_index(drop=True)

n_posicoes_entrada = int(len(df_historico_posicoes))
n_posicoes_validas = int(len(df_posicoes_validas))
n_posicoes_descartadas = n_posicoes_entrada - n_posicoes_validas
n_estrategias_posicoes_validas = int(df_posicoes_validas["estrategia_id"].nunique())
n_datas_posicoes_validas = int(df_posicoes_validas[["estrategia_id", "data"]].drop_duplicates().shape[0])
n_estrategias_auditoria_10_4_invalidas = int((~df_auditoria_historico_posicoes["flag_historico_posicoes_valido"].fillna(False).astype(bool)).sum())

validar_colunas_groupby(
    df_posicoes_validas,
    COLUNAS_CHAVE_DATA,
    "df_posicoes_validas",
    "df_referencia_posicoes_data",
)

df_referencia_posicoes_data = (
    df_posicoes_validas
    .groupby(COLUNAS_CHAVE_DATA, as_index=False, dropna=False, observed=True, sort=False)
    .agg(
        valor_mercado_posicoes_data=("valor_mercado_posicao", "sum"),
        peso_posicoes_patrimonio_data=("peso_posicao_no_patrimonio", "sum"),
        peso_posicoes_carteira_investida_data=("peso_posicao_na_carteira_investida", "sum"),
        n_linhas_posicoes_data=("ticker", "count"),
    )
)

n_referencias_posicoes_data = int(len(df_referencia_posicoes_data))

print(f"Posições históricas na entrada                    : {n_posicoes_entrada:,}")
print(f"Posições válidas para concentração                : {n_posicoes_validas:,}")
print(f"Posições descartadas da concentração              : {n_posicoes_descartadas:,}")
print(f"Estratégias com posições válidas                  : {n_estrategias_posicoes_validas:,}")
print(f"Pares estratégia-data com posições válidas        : {n_datas_posicoes_validas:,}")
print(f"Referências estratégia-data para auditoria        : {n_referencias_posicoes_data:,}")
print(f"Estratégias inválidas na auditoria 10.4           : {n_estrategias_auditoria_10_4_invalidas:,}")
print("OK")

# ============================================================
# 7) Construção incremental das bases e métricas de concentração
# ============================================================

print("\n[7/11] Construção incremental das bases e métricas de concentração...")

especificacoes_concentracao = [
    {
        "nivel": "empresa",
        "prefixo": "empresas",
        "caminho": caminho_base_concentracao_empresa,
        "colunas_dimensao": ["classe_instrumento", "grupo_registro_carteira", "issuer_code", "nome", "setor", "subsetor", "segmento"],
        "grupo": grupo_empresa,
    },
    {
        "nivel": "setor",
        "prefixo": "setores",
        "caminho": caminho_base_concentracao_setor,
        "colunas_dimensao": ["classe_instrumento", "grupo_registro_carteira", "setor"],
        "grupo": "setor",
    },
    {
        "nivel": "subsetor",
        "prefixo": "subsetores",
        "caminho": caminho_base_concentracao_subsetor,
        "colunas_dimensao": ["classe_instrumento", "grupo_registro_carteira", "setor", "subsetor"],
        "grupo": grupo_subsetor_hierarquico,
    },
    {
        "nivel": "segmento",
        "prefixo": "segmentos",
        "caminho": caminho_base_concentracao_segmento,
        "colunas_dimensao": ["classe_instrumento", "grupo_registro_carteira", "setor", "subsetor", "segmento"],
        "grupo": grupo_segmento_hierarquico,
    },
    {
        "nivel": "classe_instrumento",
        "prefixo": "classes_instrumento",
        "caminho": caminho_base_concentracao_classe_instrumento,
        "colunas_dimensao": ["classe_instrumento", "grupo_registro_carteira"],
        "grupo": "classe_instrumento",
    },
]

metricas_por_nivel = []
auditorias_por_nivel = []
amostras_por_nivel = {}
contagens_por_nivel = []
df_pesos_classe = None
escritores_parquet = {}

try:
    chaves_estrategia_processamento = (
        df_posicoes_validas[COLUNAS_CHAVE_ESTRATEGIA]
        .drop_duplicates()
        .sort_values(["ordem_exibicao", "chave_estrategia"])
        .reset_index(drop=True)
    )

    for especificacao in especificacoes_concentracao:
        nivel = especificacao["nivel"]
        prefixo = especificacao["prefixo"]
        caminho_saida = especificacao["caminho"]

        print(f"\n[7/11] Processando concentração por {nivel} em blocos por estratégia...")

        metricas_chunks = []
        auditorias_chunks = []
        n_linhas_nivel = 0
        n_invalidas_nivel = 0
        n_colunas_nivel = len(COLUNAS_BASE_CONCENTRACAO)
        amostra_nivel = []

        for idx_estrategia, linha_estrategia in chaves_estrategia_processamento.iterrows():
            chave_estrategia_atual = linha_estrategia["chave_estrategia"]
            mascara_estrategia = df_posicoes_validas["chave_estrategia"] == chave_estrategia_atual
            df_posicoes_estrategia = df_posicoes_validas.loc[mascara_estrategia]

            df_referencia_estrategia = df_referencia_posicoes_data.loc[
                df_referencia_posicoes_data["chave_estrategia"] == chave_estrategia_atual
            ]

            df_concentracao_chunk = montar_base_concentracao_chunk(
                df_posicoes_estrategia,
                nivel_concentracao=nivel,
                colunas_dimensao=especificacao["colunas_dimensao"],
                coluna_grupo_concentracao=especificacao["grupo"],
            )

            n_linhas_nivel += int(len(df_concentracao_chunk))
            n_invalidas_nivel += int((~df_concentracao_chunk["flag_concentracao_valida"].fillna(False).astype(bool)).sum())
            n_colunas_nivel = int(df_concentracao_chunk.shape[1])

            if sum(len(amostra) for amostra in amostra_nivel) < 30:
                linhas_faltantes_amostra = 30 - sum(len(amostra) for amostra in amostra_nivel)
                amostra_nivel.append(df_concentracao_chunk.head(linhas_faltantes_amostra).copy())

            df_metricas_chunk = calcular_metricas_diarias(df_concentracao_chunk, prefixo)
            metricas_chunks.append(df_metricas_chunk)

            if nivel == "classe_instrumento":
                df_pesos_classe_chunk = montar_pesos_por_classe(df_concentracao_chunk)
                if df_pesos_classe is None:
                    df_pesos_classe = df_pesos_classe_chunk.copy()
                else:
                    df_pesos_classe = pd.concat([df_pesos_classe, df_pesos_classe_chunk], axis=0, ignore_index=True)
                del df_pesos_classe_chunk

            df_auditoria_chunk = auditar_base_concentracao(df_concentracao_chunk, df_referencia_estrategia, nivel)
            auditorias_chunks.append(df_auditoria_chunk)

            escrever_parquet_incremental(df_concentracao_chunk, caminho_saida, escritores_parquet)

            if (idx_estrategia + 1) % 5 == 0 or (idx_estrategia + 1) == len(chaves_estrategia_processamento):
                print(f"  - {nivel}: {idx_estrategia + 1:,}/{len(chaves_estrategia_processamento):,} estratégias processadas")

            del df_posicoes_estrategia, df_referencia_estrategia, df_concentracao_chunk, df_metricas_chunk, df_auditoria_chunk
            if gc_modulo is not None:
                gc_modulo.collect()

        df_metricas_nivel = pd.concat(metricas_chunks, axis=0, ignore_index=True)
        df_auditoria_nivel = pd.concat(auditorias_chunks, axis=0, ignore_index=True)

        metricas_por_nivel.append(df_metricas_nivel)
        auditorias_por_nivel.append(df_auditoria_nivel)

        if len(amostra_nivel) > 0:
            amostras_por_nivel[nivel] = pd.concat(amostra_nivel, axis=0, ignore_index=True).head(30)
        else:
            amostras_por_nivel[nivel] = pd.DataFrame(columns=COLUNAS_BASE_CONCENTRACAO)

        contagens_por_nivel.append(
            {
                "nivel_concentracao": nivel,
                "linhas": n_linhas_nivel,
                "colunas": n_colunas_nivel,
                "linhas_invalidas": n_invalidas_nivel,
            }
        )

        print(f"Concentração por {nivel:<21} : {n_linhas_nivel:,} linhas x {n_colunas_nivel:,} colunas")
        print(f"Linhas inválidas - {nivel:<21} : {n_invalidas_nivel:,}")
        print(f"Arquivo salvo - {nivel:<22}: {caminho_saida}")

        del metricas_chunks, auditorias_chunks, amostra_nivel
        if gc_modulo is not None:
            gc_modulo.collect()

finally:
    fechar_escritores_parquet(escritores_parquet)

print("\nResumo das bases de concentração processadas:")
print(pd.DataFrame(contagens_por_nivel).to_string(index=False))
print("OK")

del df_posicoes_validas
if gc_modulo is not None:
    gc_modulo.collect()

# ============================================================
# 8) Construção das métricas diárias consolidadas
# ============================================================

print("\n[8/11] Construção das métricas diárias consolidadas...")

df_metricas_concentracao_diaria = metricas_por_nivel[0].copy()

for df_metricas_nivel in metricas_por_nivel[1:]:
    df_metricas_concentracao_diaria = df_metricas_concentracao_diaria.merge(
        df_metricas_nivel,
        on=COLUNAS_CHAVE_DATA,
        how="outer",
        validate="one_to_one",
    )

if df_pesos_classe is not None:
    df_metricas_concentracao_diaria = df_metricas_concentracao_diaria.merge(
        df_pesos_classe,
        on=COLUNAS_CHAVE_DATA,
        how="left",
        validate="one_to_one",
    )

for coluna in [coluna for coluna in df_metricas_concentracao_diaria.columns if coluna.startswith("n_")]:
    df_metricas_concentracao_diaria[coluna] = converter_serie_inteira(df_metricas_concentracao_diaria[coluna], 0)

for coluna in [coluna for coluna in df_metricas_concentracao_diaria.columns if coluna not in COLUNAS_CHAVE_DATA and not coluna.startswith("n_")]:
    df_metricas_concentracao_diaria[coluna] = converter_serie_numerica(df_metricas_concentracao_diaria[coluna], 0.0)

df_metricas_concentracao_diaria = df_metricas_concentracao_diaria.sort_values(
    ["ordem_exibicao", "chave_estrategia", "data"]
).reset_index(drop=True)

for objeto in metricas_por_nivel:
    del objeto
metricas_por_nivel = None
if df_pesos_classe is not None:
    del df_pesos_classe

if gc_modulo is not None:
    gc_modulo.collect()

n_linhas_com_peso_acoes_patrimonio = int((df_metricas_concentracao_diaria["peso_acoes_no_patrimonio"].abs() > TOLERANCIA_PESO_CONCENTRACAO).sum())
n_linhas_com_peso_acoes_carteira = int((df_metricas_concentracao_diaria["peso_acoes_na_carteira_investida"].abs() > TOLERANCIA_PESO_CONCENTRACAO).sum())
n_linhas_com_peso_benchmark_patrimonio = int((df_metricas_concentracao_diaria["peso_benchmark_no_patrimonio"].abs() > TOLERANCIA_PESO_CONCENTRACAO).sum())
n_linhas_com_peso_taxa_referencia_patrimonio = int((df_metricas_concentracao_diaria["peso_taxa_referencia_no_patrimonio"].abs() > TOLERANCIA_PESO_CONCENTRACAO).sum())

print(f"Métricas diárias de concentração            : {df_metricas_concentracao_diaria.shape[0]:,} linhas x {df_metricas_concentracao_diaria.shape[1]:,} colunas")
print(f"Estratégias com métricas diárias             : {df_metricas_concentracao_diaria['estrategia_id'].nunique():,}")
print(f"Pares estratégia-data nas métricas           : {df_metricas_concentracao_diaria[['estrategia_id', 'data']].drop_duplicates().shape[0]:,}")
print(f"Linhas com peso em ações no patrimônio       : {n_linhas_com_peso_acoes_patrimonio:,}")
print(f"Linhas com peso em ações na carteira         : {n_linhas_com_peso_acoes_carteira:,}")
print(f"Linhas com peso em benchmark no patrimônio   : {n_linhas_com_peso_benchmark_patrimonio:,}")
print(f"Linhas com peso em taxa ref. no patrimônio   : {n_linhas_com_peso_taxa_referencia_patrimonio:,}")
print("OK")

# ============================================================
# 9) Construção das regras, resumos e distribuição anual
# ============================================================

print("\n[9/11] Construção das regras, resumos e distribuição anual...")

df_regras_registro_concentracao_carteiras = pd.DataFrame(
    [
        {
            "ordem_regra": 1,
            "regra_id": "RCC1",
            "escopo": "fonte",
            "regra_operacional": "usar_a_base_historica_de_posicoes_validada_na_etapa_10_4",
            "detalhe": "A concentração é calculada a partir das posições históricas diárias já reconciliadas contra o patrimônio.",
        },
        {
            "ordem_regra": 2,
            "regra_id": "RCC2",
            "escopo": "dimensoes",
            "regra_operacional": "agregar_posicoes_por_empresa_setor_subsetor_segmento_e_classe_de_instrumento",
            "detalhe": "As posições são agregadas em dimensões complementares. Subsetor e segmento são tratados por hierarquia: setor > subsetor e setor > subsetor > segmento.",
        },
        {
            "ordem_regra": 3,
            "regra_id": "RCC3",
            "escopo": "pesos",
            "regra_operacional": "calcular_pesos_sobre_o_patrimonio_total_e_sobre_a_carteira_investida",
            "detalhe": "Os pesos sobre patrimônio medem exposição total incluindo caixa; os pesos sobre carteira investida medem concentração apenas do bloco alocado em instrumentos.",
        },
        {
            "ordem_regra": 4,
            "regra_id": "RCC4",
            "escopo": "metricas",
            "regra_operacional": "calcular_maior_peso_top_5_hhi_e_quantidade_de_grupos_por_dimensao",
            "detalhe": "As métricas diárias registram concentração máxima, concentração dos cinco maiores grupos, HHI e número de grupos ativos por dimensão.",
        },
        {
            "ordem_regra": 5,
            "regra_id": "RCC5",
            "escopo": "auditoria",
            "regra_operacional": "reconciliar_cada_dimensao_de_concentracao_com_a_base_de_posicoes_da_etapa_10_4",
            "detalhe": "A soma dos valores e pesos de cada dimensão deve reproduzir os totais de posições por estratégia e data.",
        },
        {
            "ordem_regra": 6,
            "regra_id": "RCC6",
            "escopo": "memoria",
            "regra_operacional": "processar_cada_dimensao_em_blocos_por_estrategia_e_salvar_incrementalmente",
            "detalhe": "As bases de concentração são processadas por estratégia e escritas em parquet incremental para evitar cópias integrais de bases com milhões de linhas.",
        },
        {
            "ordem_regra": 7,
            "regra_id": "RCC7",
            "escopo": "html",
            "regra_operacional": "salvar_bases_analiticas_para_tabelas_graficos_e_paineis_de_concentracao",
            "detalhe": "Os outputs da subetapa alimentam análises de composição, concentração e exposição setorial no HTML final.",
        },
    ]
)

colunas_metricas_resumo = [coluna for coluna in df_metricas_concentracao_diaria.columns if coluna not in COLUNAS_CHAVE_DATA]

agg_resumo_estrategia = {}
for coluna in colunas_metricas_resumo:
    agg_resumo_estrategia[f"media_{coluna}"] = (coluna, "mean")
    agg_resumo_estrategia[f"max_{coluna}"] = (coluna, "max")
    agg_resumo_estrategia[f"ultima_{coluna}"] = (coluna, "last")

df_resumo_concentracao_estrategia = (
    df_metricas_concentracao_diaria
    .groupby(COLUNAS_CHAVE_ESTRATEGIA, as_index=False, dropna=False, observed=True, sort=False)
    .agg(
        primeira_data_concentracao=("data", "min"),
        ultima_data_concentracao=("data", "max"),
        n_datas_concentracao=("data", "nunique"),
        **agg_resumo_estrategia,
    )
    .sort_values(["ordem_exibicao", "chave_estrategia"])
    .reset_index(drop=True)
)

colunas_chave_anual = COLUNAS_CHAVE_ESTRATEGIA + ["ano"]
agg_anual = {}
for coluna in colunas_metricas_resumo:
    agg_anual[f"media_{coluna}_ano"] = (coluna, "mean")
    agg_anual[f"max_{coluna}_ano"] = (coluna, "max")
    agg_anual[f"final_{coluna}_ano"] = (coluna, "last")

df_distribuicao_anual_concentracao = (
    df_metricas_concentracao_diaria
    .groupby(colunas_chave_anual, as_index=False, dropna=False, observed=True, sort=False)
    .agg(
        n_datas_concentracao_ano=("data", "nunique"),
        **agg_anual,
    )
    .sort_values(["ordem_exibicao", "chave_estrategia", "ano"])
    .reset_index(drop=True)
)

for df_contagem in [df_resumo_concentracao_estrategia, df_distribuicao_anual_concentracao]:
    for coluna in [col for col in df_contagem.columns if col.startswith("n_")]:
        df_contagem[coluna] = converter_serie_inteira(df_contagem[coluna], 0)

print(f"Regras formais geradas                       : {len(df_regras_registro_concentracao_carteiras):,}")
print(f"Resumo por estratégia gerado                 : {len(df_resumo_concentracao_estrategia):,}")
print(f"Distribuição anual gerada                    : {len(df_distribuicao_anual_concentracao):,}")
print("OK")

# ============================================================
# 10) Construção da auditoria e das inconsistências
# ============================================================

print("\n[10/11] Construção da auditoria e das inconsistências...")

df_auditoria_concentracao_carteiras = (
    pd.concat(auditorias_por_nivel, axis=0, ignore_index=True)
    .sort_values(["ordem_exibicao", "chave_estrategia", "nivel_concentracao"])
    .reset_index(drop=True)
)

for objeto in auditorias_por_nivel:
    del objeto
auditorias_por_nivel = None

if gc_modulo is not None:
    gc_modulo.collect()

df_inconsistencias_concentracao_carteiras = (
    df_auditoria_concentracao_carteiras
    .loc[~df_auditoria_concentracao_carteiras["flag_concentracao_reconciliada"].fillna(False).astype(bool)]
    .copy()
    .reset_index(drop=True)
)

n_inconsistencias = int(len(df_inconsistencias_concentracao_carteiras))
n_estrategias_com_inconsistencia = int(df_inconsistencias_concentracao_carteiras["estrategia_id"].nunique()) if n_inconsistencias > 0 else 0
n_niveis_com_inconsistencia = int(df_inconsistencias_concentracao_carteiras["nivel_concentracao"].nunique()) if n_inconsistencias > 0 else 0
n_datas_valor_nao_reconciliado = int(df_auditoria_concentracao_carteiras["n_datas_valor_concentracao_nao_reconciliado"].sum())
n_datas_peso_patrimonio_nao_reconciliado = int(df_auditoria_concentracao_carteiras["n_datas_peso_patrimonio_concentracao_nao_reconciliado"].sum())
n_datas_peso_carteira_nao_reconciliado = int(df_auditoria_concentracao_carteiras["n_datas_peso_carteira_investida_concentracao_nao_reconciliado"].sum())
n_linhas_concentracao_invalida_total = int(df_auditoria_concentracao_carteiras["n_linhas_concentracao_invalida"].sum())

print(f"Auditoria de concentração gerada             : {len(df_auditoria_concentracao_carteiras):,} linhas")
print(f"Inconsistências identificadas                : {n_inconsistencias:,}")
print(f"Estratégias com inconsistência               : {n_estrategias_com_inconsistencia:,}")
print(f"Níveis com inconsistência                    : {n_niveis_com_inconsistencia:,}")
print(f"Datas com valor não reconciliado             : {n_datas_valor_nao_reconciliado:,}")
print(f"Datas com peso patrimônio não reconciliado   : {n_datas_peso_patrimonio_nao_reconciliado:,}")
print(f"Datas com peso carteira não reconciliado     : {n_datas_peso_carteira_nao_reconciliado:,}")
print(f"Linhas de concentração inválidas             : {n_linhas_concentracao_invalida_total:,}")
print("OK")

# ============================================================
# 11) Organização final, salvamento dos outputs e validação final
# ============================================================

print("\n[11/11] Organização final, salvamento dos outputs e validação final...")

df_regras_registro_concentracao_carteiras = df_regras_registro_concentracao_carteiras.sort_values("ordem_regra").reset_index(drop=True)

df_metricas_concentracao_diaria = df_metricas_concentracao_diaria.sort_values(
    ["ordem_exibicao", "chave_estrategia", "data"]
).reset_index(drop=True)

df_resumo_concentracao_estrategia = df_resumo_concentracao_estrategia.sort_values(
    ["ordem_exibicao", "chave_estrategia"]
).reset_index(drop=True)

df_distribuicao_anual_concentracao = df_distribuicao_anual_concentracao.sort_values(
    ["ordem_exibicao", "chave_estrategia", "ano"]
).reset_index(drop=True)

df_auditoria_concentracao_carteiras = df_auditoria_concentracao_carteiras.sort_values(
    ["ordem_exibicao", "chave_estrategia", "nivel_concentracao"]
).reset_index(drop=True)

df_inconsistencias_concentracao_carteiras = df_inconsistencias_concentracao_carteiras.sort_values(
    ["ordem_exibicao", "chave_estrategia", "nivel_concentracao"]
).reset_index(drop=True)

salvar_dataframe(df_regras_registro_concentracao_carteiras, caminho_tbl_regras_registro_concentracao_carteiras, index=False)
salvar_dataframe(df_metricas_concentracao_diaria, caminho_tbl_metricas_concentracao_diaria, index=False)
salvar_dataframe(df_resumo_concentracao_estrategia, caminho_tbl_resumo_concentracao_estrategia, index=False)
salvar_dataframe(df_distribuicao_anual_concentracao, caminho_tbl_distribuicao_anual_concentracao, index=False)
salvar_dataframe(df_auditoria_concentracao_carteiras, caminho_tbl_auditoria_concentracao_carteiras, index=False)
salvar_dataframe(df_inconsistencias_concentracao_carteiras, caminho_tbl_inconsistencias_concentracao_carteiras, index=False)

print("Outputs finais da subetapa salvos com sucesso.")
print("OK")

print("\nRegras formais do registro de concentração:")
print(df_regras_registro_concentracao_carteiras.to_string(index=False))

print("\nResumo das bases de concentração salvas incrementalmente:")
print(pd.DataFrame(contagens_por_nivel).to_string(index=False))

print("\nAmostras das bases de concentração salvas incrementalmente:")
for nivel in ["empresa", "setor", "subsetor", "segmento", "classe_instrumento"]:
    print(f"\nConcentração por {nivel} - amostra:")
    print(amostras_por_nivel[nivel].to_string(index=False))

print("\nMétricas diárias de concentração - amostra:")
print(df_metricas_concentracao_diaria.head(30).to_string(index=False))

print("\nResumo de concentração por estratégia:")
print(df_resumo_concentracao_estrategia.to_string(index=False))

print("\nDistribuição anual de concentração - amostra:")
print(df_distribuicao_anual_concentracao.head(40).to_string(index=False))

print("\nAuditoria de concentração:")
print(df_auditoria_concentracao_carteiras.to_string(index=False))

print("\nInconsistências de concentração - amostra:")
print(df_inconsistencias_concentracao_carteiras.head(30).to_string(index=False))

print("\nArquivos salvos na subetapa 10.5:")
print(f"- {caminho_tbl_regras_registro_concentracao_carteiras}")
print(f"- {caminho_base_concentracao_empresa}")
print(f"- {caminho_base_concentracao_setor}")
print(f"- {caminho_base_concentracao_subsetor}")
print(f"- {caminho_base_concentracao_segmento}")
print(f"- {caminho_base_concentracao_classe_instrumento}")
print(f"- {caminho_tbl_metricas_concentracao_diaria}")
print(f"- {caminho_tbl_resumo_concentracao_estrategia}")
print(f"- {caminho_tbl_distribuicao_anual_concentracao}")
print(f"- {caminho_tbl_auditoria_concentracao_carteiras}")
print(f"- {caminho_tbl_inconsistencias_concentracao_carteiras}")

print("\nETAPA 10.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 10.5 - REGISTRO DE CONCENTRAÇÃO POR EMPRESA, SETOR, SUBSETOR E SEGMENTO

[1/11] Validação inicial do ambiente...
OK

[2/11] Liberação de memória das etapas anteriores...
Variáveis globais removidas da memória        : 630
Memória estimada removida                    : 49,294.94 MB
Objetos liberados pelo garbage collector     : 31
Memória liberada no pool do PyArrow          : 0
OK

[3/11] Definição dos caminhos de entrada e saída da subetapa...
Entrada - histórico de posições 10.4                 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_4_base_historico_posicoes.parquet
Entrada - auditoria do histórico de posições 10.4     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_4_tbl_auditoria_historico_posicoes.parquet
Saída   - regras de concentração                      : C:\Users\mht-1\OneDrive

# Etapa 11) Métricas de Performance e Risco

## Etapa 11.1) Retornos e Rentabilidade

In [52]:
%%time
# ============================================================
# Etapa 11.1) Retornos e Rentabilidade
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 11.1 - RETORNOS E RENTABILIDADE")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/10] Definição determinística dos caminhos de entrada e saída...")

caminho_base_curvas_patrimoniais_consolidadas = gerar_caminho_arquivo(
    etapa=9,
    subetapa=7,
    tipo_arquivo="base",
    nome="curvas_patrimoniais_consolidadas",
)

caminho_base_retornos_diarios = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="base",
    nome="retornos_diarios",
)

caminho_tbl_retornos_diarios = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_diarios",
)

caminho_tbl_retornos_mensais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_mensais",
)

caminho_tbl_retornos_anuais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_anuais",
)

caminho_tbl_resumo_rentabilidade_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="resumo_rentabilidade_multifrequencia",
)

caminho_tbl_auditoria_validacao_retornos = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_retornos",
)

print(f"Entrada - curvas patrimoniais consolidadas      : {caminho_base_curvas_patrimoniais_consolidadas}")
print(f"Saída   - base de retornos diários              : {caminho_base_retornos_diarios}")
print(f"Saída   - tabela de retornos diários            : {caminho_tbl_retornos_diarios}")
print(f"Saída   - tabela de retornos mensais            : {caminho_tbl_retornos_mensais}")
print(f"Saída   - tabela de retornos anuais             : {caminho_tbl_retornos_anuais}")
print(f"Saída   - resumo multifrequência                : {caminho_tbl_resumo_rentabilidade_multifrequencia}")
print(f"Saída   - auditoria de validação                : {caminho_tbl_auditoria_validacao_retornos}")
print("OK")

# ============================================================
# 3) Carga da base oficial da etapa 9.7
# ============================================================

print("\n[3/10] Carga da base oficial de curvas patrimoniais consolidadas...")

df_curvas_patrimoniais_consolidadas = pd.read_parquet(caminho_base_curvas_patrimoniais_consolidadas)

print(f"Curvas patrimoniais consolidadas        : {df_curvas_patrimoniais_consolidadas.shape[0]:,} linhas x {df_curvas_patrimoniais_consolidadas.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Validação estrutural da base de entrada
# ============================================================

print("\n[4/10] Validação estrutural da base de entrada...")

colunas_obrigatorias_entrada = [
    "data",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "flag_data_ativa_backtest",
    "patrimonio_total",
    "retorno_diario",
]

colunas_ausentes_entrada = [
    coluna for coluna in colunas_obrigatorias_entrada
    if coluna not in df_curvas_patrimoniais_consolidadas.columns
]

if len(colunas_ausentes_entrada) > 0:
    raise KeyError(
        "A base oficial da 9.7 não possui todas as colunas necessárias para a 11.1. "
        f"Colunas ausentes: {colunas_ausentes_entrada}"
    )

df_curvas_patrimoniais_consolidadas["data"] = pd.to_datetime(
    df_curvas_patrimoniais_consolidadas["data"],
    errors="coerce",
)

df_curvas_patrimoniais_consolidadas["flag_data_ativa_backtest"] = (
    df_curvas_patrimoniais_consolidadas["flag_data_ativa_backtest"]
    .fillna(False)
    .astype(bool)
)

duplicatas_entrada = int(
    df_curvas_patrimoniais_consolidadas.duplicated(["chave_estrategia", "data"]).sum()
)

if duplicatas_entrada > 0:
    raise ValueError(
        "A base oficial da 9.7 possui duplicidades por chave_estrategia e data. "
        f"Duplicidades identificadas: {duplicatas_entrada:,}"
    )

n_datas_invalidas = int(df_curvas_patrimoniais_consolidadas["data"].isna().sum())

if n_datas_invalidas > 0:
    raise ValueError(
        "A base oficial da 9.7 possui datas inválidas. "
        f"Datas inválidas identificadas: {n_datas_invalidas:,}"
    )

print(f"Colunas obrigatórias verificadas       : {len(colunas_obrigatorias_entrada)}")
print(f"Duplicidades por estratégia e data     : {duplicatas_entrada:,}")
print(f"Datas inválidas                         : {n_datas_invalidas:,}")
print("OK")

# ============================================================
# 5) Construção da base diária de retornos
# ============================================================

print("\n[5/10] Construção da base diária de retornos...")

colunas_identificacao = [
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "origem_backtest",
    "ordem_exibicao",
    "capital_inicial_estrategia",
    "data_inicio_backtest",
    "data_fim_backtest",
]

colunas_identificacao = [
    coluna for coluna in colunas_identificacao
    if coluna in df_curvas_patrimoniais_consolidadas.columns
]

colunas_valores_diarios = [
    "data",
    "flag_data_ativa_backtest",
    "patrimonio_total",
    "valor_total_aportado",
    "valor_carteira_investida",
    "saldo_caixa_fim_dia",
    "saldo_caixa",
    "n_posicoes_ativas",
    "n_tickers_unicos_ativos",
    "flag_evento_aporte",
    "valor_aporte_evento",
    "retorno_diario",
    "retorno_acumulado",
    "drawdown_atual",
]

colunas_valores_diarios = [
    coluna for coluna in colunas_valores_diarios
    if coluna in df_curvas_patrimoniais_consolidadas.columns
]

df_base_retornos_diarios = (
    df_curvas_patrimoniais_consolidadas
    .loc[df_curvas_patrimoniais_consolidadas["flag_data_ativa_backtest"], colunas_identificacao + colunas_valores_diarios]
    .copy()
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", "data"])
    .reset_index(drop=True)
)

colunas_numericas_base = [
    "patrimonio_total",
    "valor_total_aportado",
    "valor_carteira_investida",
    "saldo_caixa_fim_dia",
    "saldo_caixa",
    "valor_aporte_evento",
    "retorno_diario",
    "retorno_acumulado",
    "drawdown_atual",
]

for coluna in colunas_numericas_base:
    if coluna in df_base_retornos_diarios.columns:
        df_base_retornos_diarios[coluna] = pd.to_numeric(df_base_retornos_diarios[coluna], errors="coerce")

if "patrimonio_total" in df_base_retornos_diarios.columns:
    df_base_retornos_diarios["retorno_diario_calculado_por_patrimonio"] = (
        df_base_retornos_diarios
        .groupby("chave_estrategia", dropna=False)["patrimonio_total"]
        .pct_change()
    )
else:
    df_base_retornos_diarios["retorno_diario_calculado_por_patrimonio"] = np.nan

df_base_retornos_diarios["retorno_diario_origem"] = df_base_retornos_diarios["retorno_diario"]

df_base_retornos_diarios["retorno_diario"] = df_base_retornos_diarios["retorno_diario_origem"].where(
    df_base_retornos_diarios["retorno_diario_origem"].notna(),
    df_base_retornos_diarios["retorno_diario_calculado_por_patrimonio"],
)

mascara_primeiro_registro = (
    df_base_retornos_diarios
    .groupby("chave_estrategia", dropna=False)
    .cumcount()
    .eq(0)
)

df_base_retornos_diarios.loc[
    mascara_primeiro_registro & df_base_retornos_diarios["retorno_diario"].isna(),
    "retorno_diario",
] = 0.0

df_base_retornos_diarios["retorno_diario"] = (
    df_base_retornos_diarios["retorno_diario"]
    .replace([np.inf, -np.inf], np.nan)
)

df_base_retornos_diarios["retorno_acumulado_calculado"] = (
    df_base_retornos_diarios
    .assign(fator_retorno=(1 + df_base_retornos_diarios["retorno_diario"].fillna(0)))
    .groupby("chave_estrategia", dropna=False)["fator_retorno"]
    .cumprod()
    .sub(1)
)

if "retorno_acumulado" in df_base_retornos_diarios.columns:
    df_base_retornos_diarios["retorno_acumulado_origem"] = df_base_retornos_diarios["retorno_acumulado"]
    df_base_retornos_diarios["retorno_acumulado"] = df_base_retornos_diarios["retorno_acumulado_origem"].where(
        df_base_retornos_diarios["retorno_acumulado_origem"].notna(),
        df_base_retornos_diarios["retorno_acumulado_calculado"],
    )
else:
    df_base_retornos_diarios["retorno_acumulado"] = df_base_retornos_diarios["retorno_acumulado_calculado"]
    df_base_retornos_diarios["retorno_acumulado_origem"] = np.nan

df_base_retornos_diarios["ano"] = df_base_retornos_diarios["data"].dt.year.astype(int)
df_base_retornos_diarios["mes"] = df_base_retornos_diarios["data"].dt.month.astype(int)
df_base_retornos_diarios["ano_mes"] = df_base_retornos_diarios["data"].dt.to_period("M").astype(str)
df_base_retornos_diarios["frequencia"] = "diaria"

df_tbl_retornos_diarios = df_base_retornos_diarios.copy()
df_tbl_retornos_diarios["retorno_periodo"] = df_tbl_retornos_diarios["retorno_diario"]
df_tbl_retornos_diarios["retorno_acumulado_ate_periodo"] = df_tbl_retornos_diarios["retorno_acumulado"]

print(f"Linhas ativas na base diária           : {len(df_base_retornos_diarios):,}")
print(f"Estratégias na base diária             : {df_base_retornos_diarios['chave_estrategia'].nunique():,}")
print(f"Data inicial                           : {df_base_retornos_diarios['data'].min().date()}")
print(f"Data final                             : {df_base_retornos_diarios['data'].max().date()}")
print(f"Retornos diários ausentes              : {int(df_base_retornos_diarios['retorno_diario'].isna().sum()):,}")
print("OK")

# ============================================================
# 6) Agregação mensal e anual dos retornos
# ============================================================

print("\n[6/10] Agregação mensal e anual dos retornos...")

def calcular_retorno_composto(serie):
    serie_valida = pd.to_numeric(serie, errors="coerce").dropna()
    if len(serie_valida) == 0:
        return np.nan
    return float((1 + serie_valida).prod() - 1)

def agregar_retornos_periodo(df_diario, coluna_periodo, frequencia):
    df_periodo_base = df_diario.copy()
    df_periodo_base["periodo_agrupamento"] = df_periodo_base[coluna_periodo]

    colunas_grupo = colunas_identificacao + ["periodo_agrupamento"]

    agregacoes = {
        "data_inicio_periodo": ("data", "min"),
        "data_fim_periodo": ("data", "max"),
        "n_dias_observados": ("data", "nunique"),
        "n_retornos_validos": ("retorno_diario", lambda x: int(pd.to_numeric(x, errors="coerce").notna().sum())),
        "retorno_periodo": ("retorno_diario", calcular_retorno_composto),
        "patrimonio_inicio_observado": ("patrimonio_total", "first"),
        "patrimonio_fim_observado": ("patrimonio_total", "last"),
    }

    if "valor_total_aportado" in df_periodo_base.columns:
        agregacoes["valor_total_aportado_inicio"] = ("valor_total_aportado", "first")
        agregacoes["valor_total_aportado_fim"] = ("valor_total_aportado", "last")

    if "flag_evento_aporte" in df_periodo_base.columns:
        agregacoes["n_eventos_aporte"] = ("flag_evento_aporte", lambda x: int(pd.Series(x).fillna(False).astype(bool).sum()))

    if "valor_aporte_evento" in df_periodo_base.columns:
        agregacoes["valor_aportado_no_periodo"] = ("valor_aporte_evento", "sum")

    df_periodo = (
        df_periodo_base
        .groupby(colunas_grupo, as_index=False, dropna=False)
        .agg(**agregacoes)
        .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", "periodo_agrupamento"])
        .reset_index(drop=True)
    )

    df_periodo["frequencia"] = frequencia
    df_periodo["retorno_acumulado_ate_periodo"] = (
        df_periodo
        .assign(fator_retorno=(1 + df_periodo["retorno_periodo"].fillna(0)))
        .groupby("chave_estrategia", dropna=False)["fator_retorno"]
        .cumprod()
        .sub(1)
    )

    return df_periodo

df_tbl_retornos_mensais = agregar_retornos_periodo(
    df_diario=df_base_retornos_diarios,
    coluna_periodo="ano_mes",
    frequencia="mensal",
)

df_tbl_retornos_mensais = df_tbl_retornos_mensais.rename(columns={"periodo_agrupamento": "ano_mes"})
df_tbl_retornos_mensais["ano"] = pd.to_datetime(df_tbl_retornos_mensais["ano_mes"] + "-01", errors="coerce").dt.year.astype(int)
df_tbl_retornos_mensais["mes"] = pd.to_datetime(df_tbl_retornos_mensais["ano_mes"] + "-01", errors="coerce").dt.month.astype(int)

df_tbl_retornos_anuais = agregar_retornos_periodo(
    df_diario=df_base_retornos_diarios,
    coluna_periodo="ano",
    frequencia="anual",
)

df_tbl_retornos_anuais = df_tbl_retornos_anuais.rename(columns={"periodo_agrupamento": "ano"})
df_tbl_retornos_anuais["ano"] = df_tbl_retornos_anuais["ano"].astype(int)

print(f"Linhas da tabela mensal                : {len(df_tbl_retornos_mensais):,}")
print(f"Linhas da tabela anual                 : {len(df_tbl_retornos_anuais):,}")
print(f"Meses únicos                           : {df_tbl_retornos_mensais['ano_mes'].nunique():,}")
print(f"Anos únicos                            : {df_tbl_retornos_anuais['ano'].nunique():,}")
print("OK")

# ============================================================
# 7) Resumo de rentabilidade multifrequência
# ============================================================

print("\n[7/10] Construção do resumo de rentabilidade multifrequência...")

def resumir_rentabilidade(df_periodo, coluna_retorno, frequencia, fator_anualizacao):
    registros = []

    for chave, grupo in df_periodo.groupby("chave_estrategia", dropna=False):
        grupo = grupo.sort_values("data_fim_periodo" if "data_fim_periodo" in grupo.columns else "data")
        retornos = pd.to_numeric(grupo[coluna_retorno], errors="coerce").dropna()

        if len(retornos) == 0:
            retorno_acumulado_total = np.nan
            retorno_anualizado = np.nan
            retorno_medio_periodo = np.nan
            retorno_mediano_periodo = np.nan
            melhor_periodo = np.nan
            pior_periodo = np.nan
            pct_periodos_positivos = np.nan
        else:
            retorno_acumulado_total = float((1 + retornos).prod() - 1)
            retorno_anualizado = float((1 + retorno_acumulado_total) ** (fator_anualizacao / len(retornos)) - 1) if retorno_acumulado_total > -1 else np.nan
            retorno_medio_periodo = float(retornos.mean())
            retorno_mediano_periodo = float(retornos.median())
            melhor_periodo = float(retornos.max())
            pior_periodo = float(retornos.min())
            pct_periodos_positivos = float((retornos > 0).mean())

        primeiro = grupo.iloc[0]
        ultimo = grupo.iloc[-1]

        registro = {
            "chave_estrategia": chave,
            "estrategia_id": primeiro.get("estrategia_id", pd.NA),
            "grupo_controle": primeiro.get("grupo_controle", pd.NA),
            "familia_estrategia": primeiro.get("familia_estrategia", pd.NA),
            "estrategia_referencia": primeiro.get("estrategia_referencia", pd.NA),
            "estrategia_referencia_padronizada": primeiro.get("estrategia_referencia_padronizada", pd.NA),
            "tipo_estrategia": primeiro.get("tipo_estrategia", pd.NA),
            "nome_exibicao_estrategia": primeiro.get("nome_exibicao_estrategia", pd.NA),
            "categoria_estrategia": primeiro.get("categoria_estrategia", pd.NA),
            "subcategoria_estrategia": primeiro.get("subcategoria_estrategia", pd.NA),
            "origem_backtest": primeiro.get("origem_backtest", pd.NA),
            "ordem_exibicao": primeiro.get("ordem_exibicao", pd.NA),
            "frequencia": frequencia,
            "n_periodos": int(len(grupo)),
            "n_retornos_validos": int(len(retornos)),
            "data_inicio": grupo["data"].min() if "data" in grupo.columns else grupo["data_inicio_periodo"].min(),
            "data_fim": grupo["data"].max() if "data" in grupo.columns else grupo["data_fim_periodo"].max(),
            "retorno_acumulado_total": retorno_acumulado_total,
            "retorno_anualizado": retorno_anualizado,
            "retorno_medio_periodo": retorno_medio_periodo,
            "retorno_mediano_periodo": retorno_mediano_periodo,
            "melhor_periodo": melhor_periodo,
            "pior_periodo": pior_periodo,
            "pct_periodos_positivos": pct_periodos_positivos,
            "patrimonio_inicio_observado": primeiro.get("patrimonio_total", primeiro.get("patrimonio_inicio_observado", np.nan)),
            "patrimonio_fim_observado": ultimo.get("patrimonio_total", ultimo.get("patrimonio_fim_observado", np.nan)),
        }

        registros.append(registro)

    return pd.DataFrame(registros)

df_resumo_diario = resumir_rentabilidade(
    df_periodo=df_base_retornos_diarios,
    coluna_retorno="retorno_diario",
    frequencia="diaria",
    fator_anualizacao=252,
)

df_resumo_mensal = resumir_rentabilidade(
    df_periodo=df_tbl_retornos_mensais,
    coluna_retorno="retorno_periodo",
    frequencia="mensal",
    fator_anualizacao=12,
)

df_resumo_anual = resumir_rentabilidade(
    df_periodo=df_tbl_retornos_anuais,
    coluna_retorno="retorno_periodo",
    frequencia="anual",
    fator_anualizacao=1,
)

df_resumo_rentabilidade_multifrequencia = pd.concat(
    [df_resumo_diario, df_resumo_mensal, df_resumo_anual],
    ignore_index=True,
)

df_resumo_rentabilidade_multifrequencia = (
    df_resumo_rentabilidade_multifrequencia
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "frequencia"])
    .reset_index(drop=True)
)

print(f"Linhas do resumo multifrequência       : {len(df_resumo_rentabilidade_multifrequencia):,}")
print(f"Frequências no resumo                  : {', '.join(sorted(df_resumo_rentabilidade_multifrequencia['frequencia'].dropna().unique()))}")
print("OK")

# ============================================================
# 8) Auditoria de validação dos retornos
# ============================================================

print("\n[8/10] Construção da auditoria de validação dos retornos...")

n_linhas_ativas_entrada = int(df_curvas_patrimoniais_consolidadas["flag_data_ativa_backtest"].sum())
n_linhas_base_diaria = int(len(df_base_retornos_diarios))
n_estrategias_diarias = int(df_base_retornos_diarios["chave_estrategia"].nunique())
n_estrategias_entrada = int(
    df_curvas_patrimoniais_consolidadas
    .loc[df_curvas_patrimoniais_consolidadas["flag_data_ativa_backtest"], "chave_estrategia"]
    .nunique()
)

duplicatas_base_diaria = int(df_base_retornos_diarios.duplicated(["chave_estrategia", "data"]).sum())
retornos_diarios_ausentes = int(df_base_retornos_diarios["retorno_diario"].isna().sum())
patrimonio_diario_ausente = int(df_base_retornos_diarios["patrimonio_total"].isna().sum())
linhas_mensais_esperadas = int(df_base_retornos_diarios[["chave_estrategia", "ano_mes"]].drop_duplicates().shape[0])
linhas_anuais_esperadas = int(df_base_retornos_diarios[["chave_estrategia", "ano"]].drop_duplicates().shape[0])

registros_auditoria = [
    {
        "item": "linhas_ativas_entrada_9_7",
        "valor": n_linhas_ativas_entrada,
        "valor_referencia": n_linhas_base_diaria,
        "status": "OK" if n_linhas_ativas_entrada == n_linhas_base_diaria else "ERRO",
        "observacao": "A base diária da 11.1 deve preservar as linhas ativas da base consolidada da 9.7.",
    },
    {
        "item": "estrategias_ativas_entrada_9_7",
        "valor": n_estrategias_entrada,
        "valor_referencia": n_estrategias_diarias,
        "status": "OK" if n_estrategias_entrada == n_estrategias_diarias else "ERRO",
        "observacao": "A quantidade de estratégias ativas deve ser preservada entre a 9.7 e a base diária da 11.1.",
    },
    {
        "item": "duplicatas_base_diaria",
        "valor": duplicatas_base_diaria,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_diaria == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por chave_estrategia e data na base diária.",
    },
    {
        "item": "retornos_diarios_ausentes",
        "valor": retornos_diarios_ausentes,
        "valor_referencia": 0,
        "status": "OK" if retornos_diarios_ausentes == 0 else "ALERTA",
        "observacao": "Retornos ausentes devem ser avaliados antes das métricas de risco e eficiência.",
    },
    {
        "item": "patrimonio_diario_ausente",
        "valor": patrimonio_diario_ausente,
        "valor_referencia": 0,
        "status": "OK" if patrimonio_diario_ausente == 0 else "ALERTA",
        "observacao": "Patrimônio ausente limita validações por curva patrimonial.",
    },
    {
        "item": "linhas_mensais",
        "valor": int(len(df_tbl_retornos_mensais)),
        "valor_referencia": linhas_mensais_esperadas,
        "status": "OK" if len(df_tbl_retornos_mensais) == linhas_mensais_esperadas else "ERRO",
        "observacao": "A tabela mensal deve ter uma linha por estratégia e mês observado.",
    },
    {
        "item": "linhas_anuais",
        "valor": int(len(df_tbl_retornos_anuais)),
        "valor_referencia": linhas_anuais_esperadas,
        "status": "OK" if len(df_tbl_retornos_anuais) == linhas_anuais_esperadas else "ERRO",
        "observacao": "A tabela anual deve ter uma linha por estratégia e ano observado.",
    },
]

df_auditoria_validacao_retornos = pd.DataFrame(registros_auditoria)

n_erros_auditoria = int((df_auditoria_validacao_retornos["status"] == "ERRO").sum())

if n_erros_auditoria > 0:
    print(df_auditoria_validacao_retornos.to_string(index=False))
    raise ValueError("A auditoria da 11.1 identificou inconsistências bloqueantes.")

print(f"Itens de auditoria                     : {len(df_auditoria_validacao_retornos):,}")
print(f"Erros bloqueantes                      : {n_erros_auditoria:,}")
print("OK")

# ============================================================
# 9) Salvamento dos outputs da subetapa
# ============================================================

print("\n[9/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_base_retornos_diarios, caminho_base_retornos_diarios, index=False)
salvar_dataframe(df_tbl_retornos_diarios, caminho_tbl_retornos_diarios, index=False)
salvar_dataframe(df_tbl_retornos_mensais, caminho_tbl_retornos_mensais, index=False)
salvar_dataframe(df_tbl_retornos_anuais, caminho_tbl_retornos_anuais, index=False)
salvar_dataframe(df_resumo_rentabilidade_multifrequencia, caminho_tbl_resumo_rentabilidade_multifrequencia, index=False)
salvar_dataframe(df_auditoria_validacao_retornos, caminho_tbl_auditoria_validacao_retornos, index=False)

PARAMS_RETORNOS_RENTABILIDADE = {
    "fonte_oficial": "9_7_base_curvas_patrimoniais_consolidadas",
    "frequencias_calculadas": ["diaria", "mensal", "anual"],
    "metodo_retorno_periodo": "composicao_geometrica_dos_retornos_diarios",
    "fator_anualizacao_diaria": 252,
    "fator_anualizacao_mensal": 12,
    "fator_anualizacao_anual": 1,
    "n_linhas_base_diaria": int(len(df_base_retornos_diarios)),
    "n_estrategias": int(df_base_retornos_diarios["chave_estrategia"].nunique()),
    "data_inicio": df_base_retornos_diarios["data"].min(),
    "data_fim": df_base_retornos_diarios["data"].max(),
}

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nAuditoria de validação dos retornos:")
print(df_auditoria_validacao_retornos.to_string(index=False))

print("\nResumo multifrequência - amostra:")
print(df_resumo_rentabilidade_multifrequencia.head(30).to_string(index=False))

print("\nRetornos mensais - amostra:")
print(df_tbl_retornos_mensais.head(20).to_string(index=False))

print("\nRetornos anuais - amostra:")
print(df_tbl_retornos_anuais.head(20).to_string(index=False))

print("\nArquivos salvos na subetapa 11.1:")
print(f"- {caminho_base_retornos_diarios}")
print(f"- {caminho_tbl_retornos_diarios}")
print(f"- {caminho_tbl_retornos_mensais}")
print(f"- {caminho_tbl_retornos_anuais}")
print(f"- {caminho_tbl_resumo_rentabilidade_multifrequencia}")
print(f"- {caminho_tbl_auditoria_validacao_retornos}")

print("\nETAPA 11.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 11.1 - RETORNOS E RENTABILIDADE

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição determinística dos caminhos de entrada e saída...
Entrada - curvas patrimoniais consolidadas      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9\9_7_base_curvas_patrimoniais_consolidadas.parquet
Saída   - base de retornos diários              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_base_retornos_diarios.parquet
Saída   - tabela de retornos diários            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_tbl_retornos_diarios.parquet
Saída   - tabela de retornos mensais            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1

## Etapa 11.2) Volatilidade e Métricas de Risco

In [53]:
%%time
# ============================================================
# Etapa 11.2) Volatilidade e Métricas de Risco
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 11.2 - VOLATILIDADE E MÉTRICAS DE RISCO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/10] Definição determinística dos caminhos de entrada e saída...")

caminho_base_retornos_diarios = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="base",
    nome="retornos_diarios",
)

caminho_tbl_retornos_mensais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_mensais",
)

caminho_tbl_retornos_anuais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_anuais",
)

caminho_base_risco_diaria = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="base",
    nome="risco_diaria",
)

caminho_tbl_risco_diario = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="risco_diario",
)

caminho_tbl_risco_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="risco_mensal",
)

caminho_tbl_risco_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="risco_anual",
)

caminho_tbl_resumo_risco_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="resumo_risco_multifrequencia",
)

caminho_tbl_auditoria_validacao_risco = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_risco",
)

caminho_tbl_volatilidade_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="volatilidade_mensal",
)

caminho_tbl_volatilidade_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="volatilidade_anual",
)

print(f"Entrada - retornos diários da 11.1          : {caminho_base_retornos_diarios}")
print(f"Entrada - retornos mensais da 11.1          : {caminho_tbl_retornos_mensais}")
print(f"Entrada - retornos anuais da 11.1           : {caminho_tbl_retornos_anuais}")
print(f"Saída   - base de risco diária              : {caminho_base_risco_diaria}")
print(f"Saída   - tabela de risco diário            : {caminho_tbl_risco_diario}")
print(f"Saída   - tabela de risco mensal            : {caminho_tbl_risco_mensal}")
print(f"Saída   - tabela de risco anual             : {caminho_tbl_risco_anual}")
print(f"Saída   - resumo multifrequência            : {caminho_tbl_resumo_risco_multifrequencia}")
print(f"Saída   - auditoria de validação            : {caminho_tbl_auditoria_validacao_risco}")
print(f"Saída   - tabela volatilidade mensal        : {caminho_tbl_volatilidade_mensal}")
print(f"Saída   - tabela volatilidade anual         : {caminho_tbl_volatilidade_anual}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da 11.1
# ============================================================

print("\n[3/10] Carga das bases oficiais de retornos da 11.1...")

df_retornos_diarios = pd.read_parquet(caminho_base_retornos_diarios)
df_retornos_mensais = pd.read_parquet(caminho_tbl_retornos_mensais)
df_retornos_anuais = pd.read_parquet(caminho_tbl_retornos_anuais)

print(f"Base de retornos diários              : {df_retornos_diarios.shape[0]:,} linhas x {df_retornos_diarios.shape[1]} colunas")
print(f"Tabela de retornos mensais            : {df_retornos_mensais.shape[0]:,} linhas x {df_retornos_mensais.shape[1]} colunas")
print(f"Tabela de retornos anuais             : {df_retornos_anuais.shape[0]:,} linhas x {df_retornos_anuais.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Validação estrutural das bases de entrada
# ============================================================

print("\n[4/10] Validação estrutural das bases de entrada...")

colunas_obrigatorias_diarias = [
    "data",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "patrimonio_total",
    "retorno_diario",
]

colunas_obrigatorias_mensais = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "ano_mes",
    "data_inicio_periodo",
    "data_fim_periodo",
    "retorno_periodo",
]

colunas_obrigatorias_anuais = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "ano",
    "data_inicio_periodo",
    "data_fim_periodo",
    "retorno_periodo",
]

colunas_ausentes_diarias = [coluna for coluna in colunas_obrigatorias_diarias if coluna not in df_retornos_diarios.columns]
colunas_ausentes_mensais = [coluna for coluna in colunas_obrigatorias_mensais if coluna not in df_retornos_mensais.columns]
colunas_ausentes_anuais = [coluna for coluna in colunas_obrigatorias_anuais if coluna not in df_retornos_anuais.columns]

if len(colunas_ausentes_diarias) > 0:
    raise KeyError(f"A base diária da 11.1 não possui colunas necessárias para a 11.2: {colunas_ausentes_diarias}")
if len(colunas_ausentes_mensais) > 0:
    raise KeyError(f"A tabela mensal da 11.1 não possui colunas necessárias para a 11.2: {colunas_ausentes_mensais}")
if len(colunas_ausentes_anuais) > 0:
    raise KeyError(f"A tabela anual da 11.1 não possui colunas necessárias para a 11.2: {colunas_ausentes_anuais}")

for df_base, colunas_data in [
    (df_retornos_diarios, ["data"]),
    (df_retornos_mensais, ["data_inicio_periodo", "data_fim_periodo"]),
    (df_retornos_anuais, ["data_inicio_periodo", "data_fim_periodo"]),
]:
    for coluna_data in colunas_data:
        df_base[coluna_data] = pd.to_datetime(df_base[coluna_data], errors="coerce")

for coluna in ["retorno_diario", "patrimonio_total"]:
    df_retornos_diarios[coluna] = pd.to_numeric(df_retornos_diarios[coluna], errors="coerce")

df_retornos_mensais["retorno_periodo"] = pd.to_numeric(df_retornos_mensais["retorno_periodo"], errors="coerce")
df_retornos_anuais["retorno_periodo"] = pd.to_numeric(df_retornos_anuais["retorno_periodo"], errors="coerce")

duplicatas_diarias = int(df_retornos_diarios.duplicated(["chave_estrategia", "data"]).sum())
duplicatas_mensais = int(df_retornos_mensais.duplicated(["chave_estrategia", "ano_mes"]).sum())
duplicatas_anuais = int(df_retornos_anuais.duplicated(["chave_estrategia", "ano"]).sum())

datas_invalidas_diarias = int(df_retornos_diarios["data"].isna().sum())
datas_invalidas_mensais = int(df_retornos_mensais[["data_inicio_periodo", "data_fim_periodo"]].isna().sum().sum())
datas_invalidas_anuais = int(df_retornos_anuais[["data_inicio_periodo", "data_fim_periodo"]].isna().sum().sum())

if duplicatas_diarias > 0:
    raise ValueError(f"A base diária da 11.1 possui duplicidades por chave_estrategia e data: {duplicatas_diarias:,}")
if duplicatas_mensais > 0:
    raise ValueError(f"A tabela mensal da 11.1 possui duplicidades por chave_estrategia e ano_mes: {duplicatas_mensais:,}")
if duplicatas_anuais > 0:
    raise ValueError(f"A tabela anual da 11.1 possui duplicidades por chave_estrategia e ano: {duplicatas_anuais:,}")
if datas_invalidas_diarias > 0 or datas_invalidas_mensais > 0 or datas_invalidas_anuais > 0:
    raise ValueError(
        "Foram identificadas datas inválidas nas bases de retorno. "
        f"Diárias: {datas_invalidas_diarias:,}; mensais: {datas_invalidas_mensais:,}; anuais: {datas_invalidas_anuais:,}."
    )

print(f"Duplicidades diárias                  : {duplicatas_diarias:,}")
print(f"Duplicidades mensais                  : {duplicatas_mensais:,}")
print(f"Duplicidades anuais                   : {duplicatas_anuais:,}")
print(f"Datas inválidas diárias               : {datas_invalidas_diarias:,}")
print(f"Datas inválidas mensais               : {datas_invalidas_mensais:,}")
print(f"Datas inválidas anuais                : {datas_invalidas_anuais:,}")
print("OK")

# ============================================================
# 5) Construção da base diária de risco
# ============================================================

print("\n[5/10] Construção da base diária de risco...")

colunas_identificacao = [
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "origem_backtest",
    "ordem_exibicao",
    "capital_inicial_estrategia",
    "data_inicio_backtest",
    "data_fim_backtest",
]

colunas_identificacao = [coluna for coluna in colunas_identificacao if coluna in df_retornos_diarios.columns]

colunas_base_risco = colunas_identificacao + [
    "data",
    "ano",
    "mes",
    "ano_mes",
    "frequencia",
    "patrimonio_total",
    "retorno_diario",
    "retorno_acumulado",
    "retorno_periodo",
    "retorno_acumulado_ate_periodo",
]

colunas_base_risco = [coluna for coluna in colunas_base_risco if coluna in df_retornos_diarios.columns]

df_base_risco_diaria = (
    df_retornos_diarios[colunas_base_risco]
    .copy()
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", "data"])
    .reset_index(drop=True)
)

if "retorno_periodo" not in df_base_risco_diaria.columns:
    df_base_risco_diaria["retorno_periodo"] = df_base_risco_diaria["retorno_diario"]

if "retorno_acumulado_ate_periodo" not in df_base_risco_diaria.columns:
    if "retorno_acumulado" in df_base_risco_diaria.columns:
        df_base_risco_diaria["retorno_acumulado_ate_periodo"] = df_base_risco_diaria["retorno_acumulado"]
    else:
        df_base_risco_diaria["retorno_acumulado_ate_periodo"] = np.nan

df_base_risco_diaria["retorno_negativo"] = df_base_risco_diaria["retorno_diario"].where(
    df_base_risco_diaria["retorno_diario"] < 0,
    0.0,
)

df_base_risco_diaria["perda_diaria"] = df_base_risco_diaria["retorno_negativo"].abs()
df_base_risco_diaria["retorno_negativo_quadrado"] = df_base_risco_diaria["retorno_negativo"] ** 2

for janela in [21, 63, 252]:
    min_periodos = max(5, int(janela * 0.5))
    coluna_vol = f"volatilidade_rolling_{janela}d"
    coluna_vol_anual = f"volatilidade_rolling_{janela}d_anualizada"
    coluna_downside = f"downside_rolling_{janela}d"
    coluna_downside_anual = f"downside_rolling_{janela}d_anualizada"

    df_base_risco_diaria[coluna_vol] = (
        df_base_risco_diaria
        .groupby("chave_estrategia", dropna=False)["retorno_diario"]
        .transform(lambda serie: serie.rolling(window=janela, min_periods=min_periodos).std(ddof=1))
    )

    media_quadrado_negativo = (
        df_base_risco_diaria
        .groupby("chave_estrategia", dropna=False)["retorno_negativo_quadrado"]
        .transform(lambda serie: serie.rolling(window=janela, min_periods=min_periodos).mean())
    )

    df_base_risco_diaria[coluna_downside] = np.sqrt(media_quadrado_negativo)
    df_base_risco_diaria[coluna_vol_anual] = df_base_risco_diaria[coluna_vol] * np.sqrt(252)
    df_base_risco_diaria[coluna_downside_anual] = df_base_risco_diaria[coluna_downside] * np.sqrt(252)

df_base_risco_diaria["media_rolling_252d"] = (
    df_base_risco_diaria
    .groupby("chave_estrategia", dropna=False)["retorno_diario"]
    .transform(lambda serie: serie.rolling(window=252, min_periods=126).mean())
)

df_base_risco_diaria["desvio_rolling_252d"] = (
    df_base_risco_diaria
    .groupby("chave_estrategia", dropna=False)["retorno_diario"]
    .transform(lambda serie: serie.rolling(window=252, min_periods=126).std(ddof=1))
)

df_base_risco_diaria["zscore_retorno_rolling_252d"] = np.where(
    df_base_risco_diaria["desvio_rolling_252d"].abs() > 0,
    (df_base_risco_diaria["retorno_diario"] - df_base_risco_diaria["media_rolling_252d"]) / df_base_risco_diaria["desvio_rolling_252d"],
    np.nan,
)

print(f"Linhas da base diária de risco         : {len(df_base_risco_diaria):,}")
print(f"Estratégias na base diária de risco    : {df_base_risco_diaria['chave_estrategia'].nunique():,}")
print(f"Retornos diários ausentes              : {int(df_base_risco_diaria['retorno_diario'].isna().sum()):,}")
print("OK")

# ============================================================
# 6) Cálculo das métricas de risco por frequência
# ============================================================

print("\n[6/10] Cálculo das métricas de risco por frequência...")

def calcular_cvar(retornos, nivel):
    retornos = pd.to_numeric(pd.Series(retornos), errors="coerce").dropna()
    if len(retornos) == 0:
        return np.nan
    var = retornos.quantile(nivel)
    cauda = retornos.loc[retornos <= var]
    if len(cauda) == 0:
        return np.nan
    return float(cauda.mean())

def calcular_downside(retornos):
    retornos = pd.to_numeric(pd.Series(retornos), errors="coerce").dropna()
    if len(retornos) == 0:
        return np.nan
    perdas = np.minimum(retornos.to_numpy(dtype=float), 0.0)
    return float(np.sqrt(np.mean(perdas ** 2)))

def extrair_valor_linha(linha, coluna):
    if coluna in linha.index:
        return linha.get(coluna, pd.NA)
    return pd.NA

def resumir_risco(df_periodo, coluna_retorno, frequencia, fator_anualizacao):
    registros = []

    for chave, grupo in df_periodo.groupby("chave_estrategia", dropna=False):
        if frequencia == "diaria":
            coluna_ordenacao = "data"
            data_inicio = grupo["data"].min()
            data_fim = grupo["data"].max()
        else:
            coluna_ordenacao = "data_fim_periodo"
            data_inicio = grupo["data_inicio_periodo"].min()
            data_fim = grupo["data_fim_periodo"].max()

        grupo = grupo.sort_values(coluna_ordenacao).copy()
        retornos = pd.to_numeric(grupo[coluna_retorno], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        primeiro = grupo.iloc[0]
        ultimo = grupo.iloc[-1]

        if len(retornos) >= 2:
            volatilidade_periodo = float(retornos.std(ddof=1))
            assimetria = float(retornos.skew())
            curtose = float(retornos.kurt())
        else:
            volatilidade_periodo = np.nan
            assimetria = np.nan
            curtose = np.nan

        if len(retornos) > 0:
            downside_deviation_periodo = calcular_downside(retornos)
            retorno_medio_periodo = float(retornos.mean())
            retorno_mediano_periodo = float(retornos.median())
            melhor_retorno = float(retornos.max())
            pior_retorno = float(retornos.min())
            amplitude_retorno = float(melhor_retorno - pior_retorno)
            pct_retornos_negativos = float((retornos < 0).mean())
            pct_retornos_positivos = float((retornos > 0).mean())
            perda_media_periodos_negativos = float(retornos.loc[retornos < 0].mean()) if int((retornos < 0).sum()) > 0 else 0.0
            var_historico_5pct = float(retornos.quantile(0.05))
            var_historico_1pct = float(retornos.quantile(0.01))
            cvar_historico_5pct = calcular_cvar(retornos, 0.05)
            cvar_historico_1pct = calcular_cvar(retornos, 0.01)
        else:
            downside_deviation_periodo = np.nan
            retorno_medio_periodo = np.nan
            retorno_mediano_periodo = np.nan
            melhor_retorno = np.nan
            pior_retorno = np.nan
            amplitude_retorno = np.nan
            pct_retornos_negativos = np.nan
            pct_retornos_positivos = np.nan
            perda_media_periodos_negativos = np.nan
            var_historico_5pct = np.nan
            var_historico_1pct = np.nan
            cvar_historico_5pct = np.nan
            cvar_historico_1pct = np.nan

        volatilidade_anualizada_equivalente = (
            float(volatilidade_periodo * np.sqrt(fator_anualizacao))
            if pd.notna(volatilidade_periodo)
            else np.nan
        )

        downside_deviation_anualizada_equivalente = (
            float(downside_deviation_periodo * np.sqrt(fator_anualizacao))
            if pd.notna(downside_deviation_periodo)
            else np.nan
        )

        registro = {
            "chave_estrategia": chave,
            "estrategia_id": extrair_valor_linha(primeiro, "estrategia_id"),
            "grupo_controle": extrair_valor_linha(primeiro, "grupo_controle"),
            "familia_estrategia": extrair_valor_linha(primeiro, "familia_estrategia"),
            "estrategia_referencia": extrair_valor_linha(primeiro, "estrategia_referencia"),
            "estrategia_referencia_padronizada": extrair_valor_linha(primeiro, "estrategia_referencia_padronizada"),
            "tipo_estrategia": extrair_valor_linha(primeiro, "tipo_estrategia"),
            "carteira_id": extrair_valor_linha(primeiro, "carteira_id"),
            "replica_id": extrair_valor_linha(primeiro, "replica_id"),
            "nome_exibicao_estrategia": extrair_valor_linha(primeiro, "nome_exibicao_estrategia"),
            "categoria_estrategia": extrair_valor_linha(primeiro, "categoria_estrategia"),
            "subcategoria_estrategia": extrair_valor_linha(primeiro, "subcategoria_estrategia"),
            "origem_backtest": extrair_valor_linha(primeiro, "origem_backtest"),
            "ordem_exibicao": extrair_valor_linha(primeiro, "ordem_exibicao"),
            "frequencia": frequencia,
            "fator_anualizacao": fator_anualizacao,
            "n_periodos": int(len(grupo)),
            "n_retornos_validos": int(len(retornos)),
            "data_inicio": data_inicio,
            "data_fim": data_fim,
            "retorno_medio_periodo": retorno_medio_periodo,
            "retorno_mediano_periodo": retorno_mediano_periodo,
            "volatilidade_periodo": volatilidade_periodo,
            "volatilidade_anualizada_equivalente": volatilidade_anualizada_equivalente,
            "downside_deviation_periodo": downside_deviation_periodo,
            "downside_deviation_anualizada_equivalente": downside_deviation_anualizada_equivalente,
            "melhor_retorno": melhor_retorno,
            "pior_retorno": pior_retorno,
            "amplitude_retorno": amplitude_retorno,
            "pct_retornos_negativos": pct_retornos_negativos,
            "pct_retornos_positivos": pct_retornos_positivos,
            "perda_media_periodos_negativos": perda_media_periodos_negativos,
            "var_historico_5pct": var_historico_5pct,
            "var_historico_1pct": var_historico_1pct,
            "cvar_historico_5pct": cvar_historico_5pct,
            "cvar_historico_1pct": cvar_historico_1pct,
            "assimetria": assimetria,
            "curtose": curtose,
            "patrimonio_inicio_observado": extrair_valor_linha(primeiro, "patrimonio_total") if frequencia == "diaria" else extrair_valor_linha(primeiro, "patrimonio_inicio_observado"),
            "patrimonio_fim_observado": extrair_valor_linha(ultimo, "patrimonio_total") if frequencia == "diaria" else extrair_valor_linha(ultimo, "patrimonio_fim_observado"),
        }

        registros.append(registro)

    df_resumo = pd.DataFrame(registros)
    df_resumo = (
        df_resumo
        .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia"])
        .reset_index(drop=True)
    )

    return df_resumo

df_tbl_risco_diario = resumir_risco(
    df_periodo=df_retornos_diarios,
    coluna_retorno="retorno_diario",
    frequencia="diaria",
    fator_anualizacao=252,
)

df_tbl_risco_mensal = resumir_risco(
    df_periodo=df_retornos_mensais,
    coluna_retorno="retorno_periodo",
    frequencia="mensal",
    fator_anualizacao=12,
)

df_tbl_risco_anual = resumir_risco(
    df_periodo=df_retornos_anuais,
    coluna_retorno="retorno_periodo",
    frequencia="anual",
    fator_anualizacao=1,
)

print(f"Linhas da tabela de risco diário       : {len(df_tbl_risco_diario):,}")
print(f"Linhas da tabela de risco mensal       : {len(df_tbl_risco_mensal):,}")
print(f"Linhas da tabela de risco anual        : {len(df_tbl_risco_anual):,}")
print("OK")

# ============================================================
# 7) Construção do resumo de risco multifrequência
# ============================================================

print("\n[7/10] Construção do resumo de risco multifrequência...")

df_resumo_risco_multifrequencia = pd.concat(
    [df_tbl_risco_diario, df_tbl_risco_mensal, df_tbl_risco_anual],
    ignore_index=True,
)

df_resumo_risco_multifrequencia = (
    df_resumo_risco_multifrequencia
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", "frequencia"])
    .reset_index(drop=True)
)

frequencias_resumo = sorted(df_resumo_risco_multifrequencia["frequencia"].dropna().unique())

print(f"Linhas do resumo multifrequência       : {len(df_resumo_risco_multifrequencia):,}")
print(f"Frequências no resumo                  : {', '.join(frequencias_resumo)}")
print("OK")

# ============================================================
# 8) Auditoria de validação das métricas de risco
# ============================================================

print("\n[8/10] Construção da auditoria de validação das métricas de risco...")

n_estrategias_diarias = int(df_retornos_diarios["chave_estrategia"].nunique())
n_estrategias_mensais = int(df_retornos_mensais["chave_estrategia"].nunique())
n_estrategias_anuais = int(df_retornos_anuais["chave_estrategia"].nunique())
linhas_resumo_esperadas = int(n_estrategias_diarias * 3)

volatilidade_ausente_diaria = int(df_tbl_risco_diario["volatilidade_periodo"].isna().sum())
volatilidade_ausente_mensal = int(df_tbl_risco_mensal["volatilidade_periodo"].isna().sum())
volatilidade_ausente_anual = int(df_tbl_risco_anual["volatilidade_periodo"].isna().sum())
downside_ausente_resumo = int(df_resumo_risco_multifrequencia["downside_deviation_periodo"].isna().sum())
retornos_diarios_ausentes = int(df_retornos_diarios["retorno_diario"].isna().sum())
retornos_mensais_ausentes = int(df_retornos_mensais["retorno_periodo"].isna().sum())
retornos_anuais_ausentes = int(df_retornos_anuais["retorno_periodo"].isna().sum())

registros_auditoria = [
    {
        "item": "estrategias_diarias_vs_mensais",
        "valor": n_estrategias_diarias,
        "valor_referencia": n_estrategias_mensais,
        "status": "OK" if n_estrategias_diarias == n_estrategias_mensais else "ERRO",
        "observacao": "A quantidade de estratégias deve ser igual entre as visões diária e mensal.",
    },
    {
        "item": "estrategias_diarias_vs_anuais",
        "valor": n_estrategias_diarias,
        "valor_referencia": n_estrategias_anuais,
        "status": "OK" if n_estrategias_diarias == n_estrategias_anuais else "ERRO",
        "observacao": "A quantidade de estratégias deve ser igual entre as visões diária e anual.",
    },
    {
        "item": "duplicatas_base_risco_diaria",
        "valor": int(df_base_risco_diaria.duplicated(["chave_estrategia", "data"]).sum()),
        "valor_referencia": 0,
        "status": "OK" if int(df_base_risco_diaria.duplicated(["chave_estrategia", "data"]).sum()) == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por chave_estrategia e data na base diária de risco.",
    },
    {
        "item": "linhas_tabela_risco_diario",
        "valor": int(len(df_tbl_risco_diario)),
        "valor_referencia": n_estrategias_diarias,
        "status": "OK" if len(df_tbl_risco_diario) == n_estrategias_diarias else "ERRO",
        "observacao": "A tabela de risco diário deve ter uma linha por estratégia.",
    },
    {
        "item": "linhas_tabela_risco_mensal",
        "valor": int(len(df_tbl_risco_mensal)),
        "valor_referencia": n_estrategias_mensais,
        "status": "OK" if len(df_tbl_risco_mensal) == n_estrategias_mensais else "ERRO",
        "observacao": "A tabela de risco mensal deve ter uma linha por estratégia.",
    },
    {
        "item": "linhas_tabela_risco_anual",
        "valor": int(len(df_tbl_risco_anual)),
        "valor_referencia": n_estrategias_anuais,
        "status": "OK" if len(df_tbl_risco_anual) == n_estrategias_anuais else "ERRO",
        "observacao": "A tabela de risco anual deve ter uma linha por estratégia.",
    },
    {
        "item": "linhas_resumo_multifrequencia",
        "valor": int(len(df_resumo_risco_multifrequencia)),
        "valor_referencia": linhas_resumo_esperadas,
        "status": "OK" if len(df_resumo_risco_multifrequencia) == linhas_resumo_esperadas else "ERRO",
        "observacao": "O resumo multifrequência deve ter uma linha por estratégia e frequência.",
    },
    {
        "item": "frequencias_resumo_multifrequencia",
        "valor": int(len(frequencias_resumo)),
        "valor_referencia": 3,
        "status": "OK" if frequencias_resumo == ["anual", "diaria", "mensal"] else "ERRO",
        "observacao": "O resumo deve conter exatamente as frequências anual, diária e mensal.",
    },
    {
        "item": "retornos_ausentes_entradas",
        "valor": int(retornos_diarios_ausentes + retornos_mensais_ausentes + retornos_anuais_ausentes),
        "valor_referencia": 0,
        "status": "OK" if int(retornos_diarios_ausentes + retornos_mensais_ausentes + retornos_anuais_ausentes) == 0 else "ALERTA",
        "observacao": "Retornos ausentes nas entradas reduzem a comparabilidade das métricas de risco.",
    },
    {
        "item": "volatilidade_ausente_resumo",
        "valor": int(volatilidade_ausente_diaria + volatilidade_ausente_mensal + volatilidade_ausente_anual),
        "valor_referencia": 0,
        "status": "OK" if int(volatilidade_ausente_diaria + volatilidade_ausente_mensal + volatilidade_ausente_anual) == 0 else "ALERTA",
        "observacao": "Volatilidades ausentes podem ocorrer se uma estratégia tiver menos de dois retornos válidos na frequência.",
    },
    {
        "item": "downside_ausente_resumo",
        "valor": downside_ausente_resumo,
        "valor_referencia": 0,
        "status": "OK" if downside_ausente_resumo == 0 else "ALERTA",
        "observacao": "Downside ausente deve ser avaliado antes da etapa de Sortino.",
    },
]

df_auditoria_validacao_risco = pd.DataFrame(registros_auditoria)

n_erros_auditoria = int((df_auditoria_validacao_risco["status"] == "ERRO").sum())

if n_erros_auditoria > 0:
    print(df_auditoria_validacao_risco.to_string(index=False))
    raise ValueError("A auditoria da 11.2 identificou inconsistências bloqueantes.")

print(f"Itens de auditoria                     : {len(df_auditoria_validacao_risco):,}")
print(f"Erros bloqueantes                      : {n_erros_auditoria:,}")
print("OK")

# ============================================================
# 9) Salvamento dos outputs da subetapa
# ============================================================

print("\n[9/10] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_base_risco_diaria, caminho_base_risco_diaria, index=False)
salvar_dataframe(df_tbl_risco_diario, caminho_tbl_risco_diario, index=False)
salvar_dataframe(df_tbl_risco_mensal, caminho_tbl_risco_mensal, index=False)
salvar_dataframe(df_tbl_risco_anual, caminho_tbl_risco_anual, index=False)
salvar_dataframe(df_resumo_risco_multifrequencia, caminho_tbl_resumo_risco_multifrequencia, index=False)
salvar_dataframe(df_auditoria_validacao_risco, caminho_tbl_auditoria_validacao_risco, index=False)

salvar_dataframe(df_tbl_risco_mensal, caminho_tbl_volatilidade_mensal, index=False)
salvar_dataframe(df_tbl_risco_anual, caminho_tbl_volatilidade_anual, index=False)

PARAMS_VOLATILIDADE_METRICAS_RISCO = {
    "fontes_oficiais": [
        "11_1_base_retornos_diarios",
        "11_1_tbl_retornos_mensais",
        "11_1_tbl_retornos_anuais",
    ],
    "frequencias_calculadas": ["diaria", "mensal", "anual"],
    "metodo_volatilidade": "desvio_padrao_amostral_dos_retornos_por_frequencia",
    "metodo_downside": "raiz_da_media_dos_quadrados_dos_retornos_abaixo_de_zero",
    "fator_anualizacao_diaria": 252,
    "fator_anualizacao_mensal": 12,
    "fator_anualizacao_anual": 1,
    "janelas_rolling_diarias": [21, 63, 252],
    "n_linhas_base_risco_diaria": int(len(df_base_risco_diaria)),
    "n_estrategias": int(n_estrategias_diarias),
    "data_inicio": df_retornos_diarios["data"].min(),
    "data_fim": df_retornos_diarios["data"].max(),
}

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 10) Validação final da subetapa
# ============================================================

print("\n[10/10] Validação final da subetapa...")

print("\nAuditoria de validação das métricas de risco:")
print(df_auditoria_validacao_risco.to_string(index=False))

print("\nResumo de risco multifrequência - amostra:")
print(df_resumo_risco_multifrequencia.head(30).to_string(index=False))

print("\nRisco mensal - amostra:")
print(df_tbl_risco_mensal.head(20).to_string(index=False))

print("\nRisco anual - amostra:")
print(df_tbl_risco_anual.head(20).to_string(index=False))

print("\nBase diária de risco - amostra:")
print(df_base_risco_diaria.head(20).to_string(index=False))

print("\nArquivos salvos na subetapa 11.2:")
print(f"- {caminho_base_risco_diaria}")
print(f"- {caminho_tbl_risco_diario}")
print(f"- {caminho_tbl_risco_mensal}")
print(f"- {caminho_tbl_risco_anual}")
print(f"- {caminho_tbl_resumo_risco_multifrequencia}")
print(f"- {caminho_tbl_auditoria_validacao_risco}")
print(f"- {caminho_tbl_volatilidade_mensal}")
print(f"- {caminho_tbl_volatilidade_anual}")

print("\nETAPA 11.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 11.2 - VOLATILIDADE E MÉTRICAS DE RISCO

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição determinística dos caminhos de entrada e saída...
Entrada - retornos diários da 11.1          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_base_retornos_diarios.parquet
Entrada - retornos mensais da 11.1          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_tbl_retornos_mensais.parquet
Entrada - retornos anuais da 11.1           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_tbl_retornos_anuais.parquet
Saída   - base de risco diária              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_2_base_risco_diaria

## Etapa 11.3) Sharpe, Sortino e Calmar

In [54]:
%%time
# ============================================================
# Etapa 11.3) Sharpe, Sortino e Calmar
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 11.3 - SHARPE, SORTINO E CALMAR")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/11] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/11] Definição determinística dos caminhos de entrada e saída...")

caminho_base_retornos_diarios = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="base",
    nome="retornos_diarios",
)

caminho_tbl_retornos_mensais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_mensais",
)

caminho_tbl_retornos_anuais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_anuais",
)

caminho_tbl_risco_diario = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="risco_diario",
)

caminho_tbl_risco_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="risco_mensal",
)

caminho_tbl_risco_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="risco_anual",
)

caminho_base_selic_padronizada = gerar_caminho_arquivo(
    etapa=1,
    subetapa=3,
    tipo_arquivo="base",
    nome="selic_padronizada",
)

caminho_base_retornos_excesso_diarios = gerar_caminho_arquivo(
    etapa=11,
    subetapa=3,
    tipo_arquivo="base",
    nome="retornos_excesso_diarios",
)

caminho_tbl_sharpe_sortino_calmar_diario = gerar_caminho_arquivo(
    etapa=11,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="sharpe_sortino_calmar_diario",
)

caminho_tbl_sharpe_sortino_calmar_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="sharpe_sortino_calmar_mensal",
)

caminho_tbl_sharpe_sortino_calmar_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="sharpe_sortino_calmar_anual",
)

caminho_tbl_metricas_eficiencia_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="metricas_eficiencia_multifrequencia",
)

caminho_tbl_auditoria_validacao_eficiencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_eficiencia",
)

print(f"Entrada - retornos diários da 11.1          : {caminho_base_retornos_diarios}")
print(f"Entrada - retornos mensais da 11.1          : {caminho_tbl_retornos_mensais}")
print(f"Entrada - retornos anuais da 11.1           : {caminho_tbl_retornos_anuais}")
print(f"Entrada - risco diário da 11.2              : {caminho_tbl_risco_diario}")
print(f"Entrada - risco mensal da 11.2              : {caminho_tbl_risco_mensal}")
print(f"Entrada - risco anual da 11.2               : {caminho_tbl_risco_anual}")
print(f"Entrada - Selic padronizada                 : {caminho_base_selic_padronizada}")
print(f"Saída   - base diária de retornos em excesso: {caminho_base_retornos_excesso_diarios}")
print(f"Saída   - métricas diárias                  : {caminho_tbl_sharpe_sortino_calmar_diario}")
print(f"Saída   - métricas mensais                  : {caminho_tbl_sharpe_sortino_calmar_mensal}")
print(f"Saída   - métricas anuais                   : {caminho_tbl_sharpe_sortino_calmar_anual}")
print(f"Saída   - resumo multifrequência            : {caminho_tbl_metricas_eficiencia_multifrequencia}")
print(f"Saída   - auditoria de validação            : {caminho_tbl_auditoria_validacao_eficiencia}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais de retornos, risco e Selic
# ============================================================

print("\n[3/11] Carga das bases oficiais de retornos, risco e Selic...")

df_retornos_diarios = pd.read_parquet(caminho_base_retornos_diarios)
df_retornos_mensais = pd.read_parquet(caminho_tbl_retornos_mensais)
df_retornos_anuais = pd.read_parquet(caminho_tbl_retornos_anuais)

df_risco_diario = pd.read_parquet(caminho_tbl_risco_diario)
df_risco_mensal = pd.read_parquet(caminho_tbl_risco_mensal)
df_risco_anual = pd.read_parquet(caminho_tbl_risco_anual)

df_selic_original = pd.read_parquet(caminho_base_selic_padronizada)

print(f"Base de retornos diários              : {df_retornos_diarios.shape[0]:,} linhas x {df_retornos_diarios.shape[1]} colunas")
print(f"Tabela de retornos mensais            : {df_retornos_mensais.shape[0]:,} linhas x {df_retornos_mensais.shape[1]} colunas")
print(f"Tabela de retornos anuais             : {df_retornos_anuais.shape[0]:,} linhas x {df_retornos_anuais.shape[1]} colunas")
print(f"Tabela de risco diário                : {df_risco_diario.shape[0]:,} linhas x {df_risco_diario.shape[1]} colunas")
print(f"Tabela de risco mensal                : {df_risco_mensal.shape[0]:,} linhas x {df_risco_mensal.shape[1]} colunas")
print(f"Tabela de risco anual                 : {df_risco_anual.shape[0]:,} linhas x {df_risco_anual.shape[1]} colunas")
print(f"Base Selic padronizada                : {df_selic_original.shape[0]:,} linhas x {df_selic_original.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Validação estrutural das bases de entrada
# ============================================================

print("\n[4/11] Validação estrutural das bases de entrada...")

colunas_obrigatorias_diarias = [
    "data",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "patrimonio_total",
    "retorno_diario",
]

colunas_obrigatorias_mensais = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "ano_mes",
    "data_inicio_periodo",
    "data_fim_periodo",
    "retorno_periodo",
]

colunas_obrigatorias_anuais = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "ano",
    "data_inicio_periodo",
    "data_fim_periodo",
    "retorno_periodo",
]

colunas_obrigatorias_risco = [
    "chave_estrategia",
    "frequencia",
    "volatilidade_periodo",
    "volatilidade_anualizada_equivalente",
    "downside_deviation_periodo",
    "downside_deviation_anualizada_equivalente",
]

colunas_ausentes_diarias = [coluna for coluna in colunas_obrigatorias_diarias if coluna not in df_retornos_diarios.columns]
colunas_ausentes_mensais = [coluna for coluna in colunas_obrigatorias_mensais if coluna not in df_retornos_mensais.columns]
colunas_ausentes_anuais = [coluna for coluna in colunas_obrigatorias_anuais if coluna not in df_retornos_anuais.columns]
colunas_ausentes_risco_diario = [coluna for coluna in colunas_obrigatorias_risco if coluna not in df_risco_diario.columns]
colunas_ausentes_risco_mensal = [coluna for coluna in colunas_obrigatorias_risco if coluna not in df_risco_mensal.columns]
colunas_ausentes_risco_anual = [coluna for coluna in colunas_obrigatorias_risco if coluna not in df_risco_anual.columns]

if len(colunas_ausentes_diarias) > 0:
    raise KeyError(f"A base diária da 11.1 não possui colunas necessárias para a 11.3: {colunas_ausentes_diarias}")
if len(colunas_ausentes_mensais) > 0:
    raise KeyError(f"A tabela mensal da 11.1 não possui colunas necessárias para a 11.3: {colunas_ausentes_mensais}")
if len(colunas_ausentes_anuais) > 0:
    raise KeyError(f"A tabela anual da 11.1 não possui colunas necessárias para a 11.3: {colunas_ausentes_anuais}")
if len(colunas_ausentes_risco_diario) > 0:
    raise KeyError(f"A tabela diária da 11.2 não possui colunas necessárias para a 11.3: {colunas_ausentes_risco_diario}")
if len(colunas_ausentes_risco_mensal) > 0:
    raise KeyError(f"A tabela mensal da 11.2 não possui colunas necessárias para a 11.3: {colunas_ausentes_risco_mensal}")
if len(colunas_ausentes_risco_anual) > 0:
    raise KeyError(f"A tabela anual da 11.2 não possui colunas necessárias para a 11.3: {colunas_ausentes_risco_anual}")

for df_base, colunas_data in [
    (df_retornos_diarios, ["data"]),
    (df_retornos_mensais, ["data_inicio_periodo", "data_fim_periodo"]),
    (df_retornos_anuais, ["data_inicio_periodo", "data_fim_periodo"]),
]:
    for coluna_data in colunas_data:
        df_base[coluna_data] = pd.to_datetime(df_base[coluna_data], errors="coerce")

for coluna in ["retorno_diario", "patrimonio_total"]:
    df_retornos_diarios[coluna] = pd.to_numeric(df_retornos_diarios[coluna], errors="coerce")

df_retornos_mensais["retorno_periodo"] = pd.to_numeric(df_retornos_mensais["retorno_periodo"], errors="coerce")
df_retornos_anuais["retorno_periodo"] = pd.to_numeric(df_retornos_anuais["retorno_periodo"], errors="coerce")

duplicatas_diarias = int(df_retornos_diarios.duplicated(["chave_estrategia", "data"]).sum())
duplicatas_mensais = int(df_retornos_mensais.duplicated(["chave_estrategia", "ano_mes"]).sum())
duplicatas_anuais = int(df_retornos_anuais.duplicated(["chave_estrategia", "ano"]).sum())

datas_invalidas_diarias = int(df_retornos_diarios["data"].isna().sum())
datas_invalidas_mensais = int(df_retornos_mensais[["data_inicio_periodo", "data_fim_periodo"]].isna().sum().sum())
datas_invalidas_anuais = int(df_retornos_anuais[["data_inicio_periodo", "data_fim_periodo"]].isna().sum().sum())

if duplicatas_diarias > 0:
    raise ValueError(f"A base diária da 11.1 possui duplicidades por chave_estrategia e data: {duplicatas_diarias:,}")
if duplicatas_mensais > 0:
    raise ValueError(f"A tabela mensal da 11.1 possui duplicidades por chave_estrategia e ano_mes: {duplicatas_mensais:,}")
if duplicatas_anuais > 0:
    raise ValueError(f"A tabela anual da 11.1 possui duplicidades por chave_estrategia e ano: {duplicatas_anuais:,}")
if datas_invalidas_diarias > 0 or datas_invalidas_mensais > 0 or datas_invalidas_anuais > 0:
    raise ValueError(
        "Foram identificadas datas inválidas nas bases de retorno. "
        f"Diárias: {datas_invalidas_diarias:,}; mensais: {datas_invalidas_mensais:,}; anuais: {datas_invalidas_anuais:,}."
    )

print(f"Duplicidades diárias                  : {duplicatas_diarias:,}")
print(f"Duplicidades mensais                  : {duplicatas_mensais:,}")
print(f"Duplicidades anuais                   : {duplicatas_anuais:,}")
print(f"Datas inválidas diárias               : {datas_invalidas_diarias:,}")
print(f"Datas inválidas mensais               : {datas_invalidas_mensais:,}")
print(f"Datas inválidas anuais                : {datas_invalidas_anuais:,}")
print("OK")

# ============================================================
# 5) Padronização da Selic diária como taxa livre de risco
# ============================================================

print("\n[5/11] Padronização da Selic diária como taxa livre de risco...")

def padronizar_selic_diaria(df_selic):
    df = df_selic.copy()

    if "data" not in df.columns:
        nome_indice = df.index.name if df.index.name is not None else "index"
        df = df.reset_index().rename(columns={nome_indice: "data"})

    mapa_colunas = {str(coluna).strip().lower(): coluna for coluna in df.columns}

    if "data" not in df.columns:
        if "data" in mapa_colunas:
            df = df.rename(columns={mapa_colunas["data"]: "data"})
        elif "date" in mapa_colunas:
            df = df.rename(columns={mapa_colunas["date"]: "data"})
        else:
            colunas_datetime = [coluna for coluna in df.columns if pd.api.types.is_datetime64_any_dtype(df[coluna])]
            if len(colunas_datetime) == 0:
                raise KeyError("A base Selic padronizada não possui coluna de data identificável.")
            df = df.rename(columns={colunas_datetime[0]: "data"})

    mapa_colunas = {str(coluna).strip().lower(): coluna for coluna in df.columns}
    candidatos_valor = ["valor", "selic", "taxa_selic", "taxa_selic_diaria", "taxa"]
    coluna_valor = None

    for candidato in candidatos_valor:
        if candidato in mapa_colunas:
            coluna_valor = mapa_colunas[candidato]
            break

    if coluna_valor is None:
        colunas_numericas = [coluna for coluna in df.columns if coluna != "data" and pd.api.types.is_numeric_dtype(df[coluna])]
        if len(colunas_numericas) == 0:
            raise KeyError("A base Selic padronizada não possui coluna numérica identificável para a taxa.")
        coluna_valor = colunas_numericas[0]

    df = df[["data", coluna_valor]].copy()
    df = df.rename(columns={coluna_valor: "valor_selic_original"})
    df["data"] = pd.to_datetime(df["data"], errors="coerce")
    df["valor_selic_original"] = pd.to_numeric(
        df["valor_selic_original"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    df = df.dropna(subset=["data", "valor_selic_original"]).sort_values("data")
    df = df.drop_duplicates("data", keep="last").reset_index(drop=True)

    mediana_abs = float(df["valor_selic_original"].abs().median()) if len(df) > 0 else np.nan

    if pd.isna(mediana_abs):
        raise ValueError("A base Selic padronizada não possui valores válidos após a padronização.")

    if mediana_abs > 1:
        metodo_conversao = "percentual_anual_para_decimal_diario_equivalente"
        df["taxa_livre_risco_diaria"] = (1 + df["valor_selic_original"] / 100) ** (1 / 252) - 1
    elif mediana_abs > 0.01:
        metodo_conversao = "percentual_diario_para_decimal_diario"
        df["taxa_livre_risco_diaria"] = df["valor_selic_original"] / 100
    else:
        metodo_conversao = "decimal_diario"
        df["taxa_livre_risco_diaria"] = df["valor_selic_original"]

    df["taxa_livre_risco_diaria"] = pd.to_numeric(df["taxa_livre_risco_diaria"], errors="coerce")
    df = df.dropna(subset=["taxa_livre_risco_diaria"]).reset_index(drop=True)

    return df, metodo_conversao

df_selic, metodo_conversao_selic = padronizar_selic_diaria(df_selic_original)

if len(df_selic) == 0:
    raise ValueError("A base Selic padronizada ficou vazia após o tratamento.")

print(f"Linhas válidas da Selic                : {len(df_selic):,}")
print(f"Data inicial da Selic                  : {df_selic['data'].min().date()}")
print(f"Data final da Selic                    : {df_selic['data'].max().date()}")
print(f"Método de conversão da Selic           : {metodo_conversao_selic}")
print(f"Mediana da taxa livre diária decimal   : {df_selic['taxa_livre_risco_diaria'].median():.10f}")
print("OK")

# ============================================================
# 6) Construção das taxas livres de risco por frequência
# ============================================================

print("\n[6/11] Construção das taxas livres de risco por frequência...")

serie_selic = df_selic[["data", "taxa_livre_risco_diaria"]].copy().sort_values("data")

datas_diarias = df_retornos_diarios[["data"]].drop_duplicates().sort_values("data").reset_index(drop=True)
df_taxa_livre_risco_diaria = pd.merge_asof(
    datas_diarias,
    serie_selic,
    on="data",
    direction="backward",
)

df_taxa_livre_risco_diaria["taxa_livre_risco_diaria"] = (
    df_taxa_livre_risco_diaria["taxa_livre_risco_diaria"]
    .ffill()
    .bfill()
)

def calcular_retorno_livre_risco_periodo(data_inicio, data_fim):
    data_inicio = pd.to_datetime(data_inicio, errors="coerce")
    data_fim = pd.to_datetime(data_fim, errors="coerce")

    if pd.isna(data_inicio) or pd.isna(data_fim):
        return np.nan

    taxas = serie_selic.loc[
        (serie_selic["data"] >= data_inicio) &
        (serie_selic["data"] <= data_fim),
        "taxa_livre_risco_diaria",
    ]

    if len(taxas) == 0:
        return np.nan

    return float((1 + taxas).prod() - 1)

df_taxa_livre_risco_mensal = (
    df_retornos_mensais[["ano_mes", "data_inicio_periodo", "data_fim_periodo"]]
    .drop_duplicates()
    .copy()
)

df_taxa_livre_risco_mensal["taxa_livre_risco_periodo"] = [
    calcular_retorno_livre_risco_periodo(data_inicio, data_fim)
    for data_inicio, data_fim in zip(
        df_taxa_livre_risco_mensal["data_inicio_periodo"],
        df_taxa_livre_risco_mensal["data_fim_periodo"],
    )
]

df_taxa_livre_risco_anual = (
    df_retornos_anuais[["ano", "data_inicio_periodo", "data_fim_periodo"]]
    .drop_duplicates()
    .copy()
)

df_taxa_livre_risco_anual["taxa_livre_risco_periodo"] = [
    calcular_retorno_livre_risco_periodo(data_inicio, data_fim)
    for data_inicio, data_fim in zip(
        df_taxa_livre_risco_anual["data_inicio_periodo"],
        df_taxa_livre_risco_anual["data_fim_periodo"],
    )
]

ausentes_taxa_diaria = int(df_taxa_livre_risco_diaria["taxa_livre_risco_diaria"].isna().sum())
ausentes_taxa_mensal = int(df_taxa_livre_risco_mensal["taxa_livre_risco_periodo"].isna().sum())
ausentes_taxa_anual = int(df_taxa_livre_risco_anual["taxa_livre_risco_periodo"].isna().sum())

if ausentes_taxa_diaria > 0 or ausentes_taxa_mensal > 0 or ausentes_taxa_anual > 0:
    raise ValueError(
        "Foram identificadas taxas livres de risco ausentes após a composição por frequência. "
        f"Diária: {ausentes_taxa_diaria:,}; mensal: {ausentes_taxa_mensal:,}; anual: {ausentes_taxa_anual:,}."
    )

print(f"Datas diárias com Selic associada      : {len(df_taxa_livre_risco_diaria):,}")
print(f"Meses com Selic composta               : {len(df_taxa_livre_risco_mensal):,}")
print(f"Anos com Selic composta                : {len(df_taxa_livre_risco_anual):,}")
print(f"Taxas ausentes diárias                 : {ausentes_taxa_diaria:,}")
print(f"Taxas ausentes mensais                 : {ausentes_taxa_mensal:,}")
print(f"Taxas ausentes anuais                  : {ausentes_taxa_anual:,}")
print("OK")

# ============================================================
# 7) Construção das bases de retornos em excesso
# ============================================================

print("\n[7/11] Construção das bases de retornos em excesso...")

df_excesso_diario = df_retornos_diarios.copy()
df_excesso_diario = df_excesso_diario.merge(
    df_taxa_livre_risco_diaria.rename(columns={"taxa_livre_risco_diaria": "taxa_livre_risco_periodo"}),
    on="data",
    how="left",
)
df_excesso_diario["retorno_periodo"] = df_excesso_diario["retorno_diario"]
df_excesso_diario["retorno_excesso_periodo"] = df_excesso_diario["retorno_periodo"] - df_excesso_diario["taxa_livre_risco_periodo"]
df_excesso_diario["frequencia"] = "diaria"

df_excesso_mensal = df_retornos_mensais.copy()
df_excesso_mensal = df_excesso_mensal.merge(
    df_taxa_livre_risco_mensal[["ano_mes", "data_inicio_periodo", "data_fim_periodo", "taxa_livre_risco_periodo"]],
    on=["ano_mes", "data_inicio_periodo", "data_fim_periodo"],
    how="left",
)
df_excesso_mensal["retorno_excesso_periodo"] = df_excesso_mensal["retorno_periodo"] - df_excesso_mensal["taxa_livre_risco_periodo"]
df_excesso_mensal["frequencia"] = "mensal"

df_excesso_anual = df_retornos_anuais.copy()
df_excesso_anual = df_excesso_anual.merge(
    df_taxa_livre_risco_anual[["ano", "data_inicio_periodo", "data_fim_periodo", "taxa_livre_risco_periodo"]],
    on=["ano", "data_inicio_periodo", "data_fim_periodo"],
    how="left",
)
df_excesso_anual["retorno_excesso_periodo"] = df_excesso_anual["retorno_periodo"] - df_excesso_anual["taxa_livre_risco_periodo"]
df_excesso_anual["frequencia"] = "anual"

for df_base, coluna_ordenacao in [
    (df_excesso_diario, "data"),
    (df_excesso_mensal, "data_fim_periodo"),
    (df_excesso_anual, "data_fim_periodo"),
]:
    df_base.sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", coluna_ordenacao], inplace=True)
    df_base["taxa_livre_risco_acumulada_ate_periodo"] = (
        df_base
        .groupby("chave_estrategia", dropna=False)["taxa_livre_risco_periodo"]
        .transform(lambda serie: (1 + serie).cumprod() - 1)
    )
    df_base["retorno_excesso_acumulado_composto_ate_periodo"] = (
        df_base
        .groupby("chave_estrategia", dropna=False)["retorno_excesso_periodo"]
        .transform(lambda serie: (1 + serie).cumprod() - 1)
    )
    df_base.reset_index(drop=True, inplace=True)

retornos_excesso_ausentes_diarios = int(df_excesso_diario["retorno_excesso_periodo"].isna().sum())
retornos_excesso_ausentes_mensais = int(df_excesso_mensal["retorno_excesso_periodo"].isna().sum())
retornos_excesso_ausentes_anuais = int(df_excesso_anual["retorno_excesso_periodo"].isna().sum())

if retornos_excesso_ausentes_diarios > 0 or retornos_excesso_ausentes_mensais > 0 or retornos_excesso_ausentes_anuais > 0:
    raise ValueError(
        "Foram identificados retornos em excesso ausentes. "
        f"Diários: {retornos_excesso_ausentes_diarios:,}; mensais: {retornos_excesso_ausentes_mensais:,}; anuais: {retornos_excesso_ausentes_anuais:,}."
    )

expansao_linhas_mensais = int(len(df_excesso_mensal) - len(df_retornos_mensais))
expansao_linhas_anuais = int(len(df_excesso_anual) - len(df_retornos_anuais))

if expansao_linhas_mensais != 0 or expansao_linhas_anuais != 0:
    raise ValueError(
        "A associação da taxa livre de risco não deve alterar a quantidade de linhas das bases da 11.1. "
        f"Expansão mensal: {expansao_linhas_mensais:,}; expansão anual: {expansao_linhas_anuais:,}."
    )

print(f"Linhas da base diária de excesso       : {len(df_excesso_diario):,}")
print(f"Linhas da base mensal de excesso       : {len(df_excesso_mensal):,}")
print(f"Linhas da base anual de excesso        : {len(df_excesso_anual):,}")
print(f"Retornos em excesso ausentes diários   : {retornos_excesso_ausentes_diarios:,}")
print(f"Retornos em excesso ausentes mensais   : {retornos_excesso_ausentes_mensais:,}")
print(f"Retornos em excesso ausentes anuais    : {retornos_excesso_ausentes_anuais:,}")
print(f"Expansão de linhas mensais             : {expansao_linhas_mensais:,}")
print(f"Expansão de linhas anuais              : {expansao_linhas_anuais:,}")
print("OK")

# ============================================================
# 8) Cálculo de Sharpe, Sortino e Calmar por frequência
# ============================================================

print("\n[8/11] Cálculo de Sharpe, Sortino e Calmar por frequência...")

def extrair_valor_linha(linha, coluna):
    if coluna in linha.index:
        return linha.get(coluna, pd.NA)
    return pd.NA

def dividir_seguro(numerador, denominador):
    if pd.isna(numerador) or pd.isna(denominador):
        return np.nan
    if abs(float(denominador)) <= 0:
        return np.nan
    return float(numerador) / float(denominador)

def calcular_downside_excesso(retornos_excesso):
    retornos_excesso = pd.to_numeric(pd.Series(retornos_excesso), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if len(retornos_excesso) == 0:
        return np.nan
    perdas = np.minimum(retornos_excesso.to_numpy(dtype=float), 0.0)
    return float(np.sqrt(np.mean(perdas ** 2)))

def calcular_retorno_anualizado(retorno_acumulado_total, data_inicio, data_fim):
    data_inicio = pd.to_datetime(data_inicio, errors="coerce")
    data_fim = pd.to_datetime(data_fim, errors="coerce")

    if pd.isna(retorno_acumulado_total) or pd.isna(data_inicio) or pd.isna(data_fim):
        return np.nan

    n_anos = (data_fim - data_inicio).days / 365.25

    if n_anos <= 0:
        return np.nan
    if (1 + retorno_acumulado_total) <= 0:
        return np.nan

    return float((1 + retorno_acumulado_total) ** (1 / n_anos) - 1)

def calcular_drawdown_por_retornos(grupo, coluna_retorno):
    retornos = pd.to_numeric(grupo[coluna_retorno], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    curva = (1 + retornos).cumprod()
    pico = curva.cummax()
    drawdown = curva / pico - 1

    if len(drawdown) == 0:
        return np.nan, pd.NaT

    max_drawdown = float(drawdown.min())
    posicao_max_drawdown = drawdown.idxmin()

    if "data" in grupo.columns:
        data_max_drawdown = grupo.loc[posicao_max_drawdown, "data"]
    else:
        data_max_drawdown = grupo.loc[posicao_max_drawdown, "data_fim_periodo"]

    return max_drawdown, data_max_drawdown

def resumir_metricas_eficiencia(df_periodo, df_risco, frequencia, fator_anualizacao, coluna_data_ordenacao):
    registros = []

    colunas_risco = [
        "chave_estrategia",
        "volatilidade_periodo",
        "volatilidade_anualizada_equivalente",
        "downside_deviation_periodo",
        "downside_deviation_anualizada_equivalente",
        "var_historico_5pct",
        "var_historico_1pct",
        "cvar_historico_5pct",
        "cvar_historico_1pct",
    ]
    colunas_risco = [coluna for coluna in colunas_risco if coluna in df_risco.columns]
    df_risco_aux = df_risco[colunas_risco].drop_duplicates("chave_estrategia").copy()

    for chave, grupo in df_periodo.groupby("chave_estrategia", dropna=False):
        grupo = grupo.sort_values(coluna_data_ordenacao).copy()
        primeiro = grupo.iloc[0]
        ultimo = grupo.iloc[-1]

        risco_linha_df = df_risco_aux.loc[df_risco_aux["chave_estrategia"] == chave]
        risco_linha = risco_linha_df.iloc[0] if len(risco_linha_df) > 0 else pd.Series(dtype="object")

        def extrair_risco_numerico(coluna):
            if coluna not in risco_linha.index:
                return np.nan
            valor = pd.to_numeric(pd.Series([risco_linha[coluna]]), errors="coerce").iloc[0]
            return float(valor) if pd.notna(valor) else np.nan

        volatilidade_sharpe_periodo = extrair_risco_numerico("volatilidade_periodo")
        volatilidade_sharpe_anualizada = extrair_risco_numerico("volatilidade_anualizada_equivalente")
        downside_risco_periodo = extrair_risco_numerico("downside_deviation_periodo")
        downside_risco_anualizado = extrair_risco_numerico("downside_deviation_anualizada_equivalente")

        retornos = pd.to_numeric(grupo["retorno_periodo"], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        retornos_excesso = pd.to_numeric(grupo["retorno_excesso_periodo"], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        taxas_livres = pd.to_numeric(grupo["taxa_livre_risco_periodo"], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

        data_inicio = grupo["data"].min() if frequencia == "diaria" else grupo["data_inicio_periodo"].min()
        data_fim = grupo["data"].max() if frequencia == "diaria" else grupo["data_fim_periodo"].max()
        n_dias_corridos = int((pd.to_datetime(data_fim) - pd.to_datetime(data_inicio)).days) if pd.notna(data_inicio) and pd.notna(data_fim) else pd.NA
        n_anos = float(n_dias_corridos / 365.25) if pd.notna(n_dias_corridos) and n_dias_corridos > 0 else np.nan

        if len(retornos) > 0:
            retorno_acumulado_total = float((1 + retornos).prod() - 1)
            retorno_medio_periodo = float(retornos.mean())
            retorno_mediano_periodo = float(retornos.median())
            melhor_retorno_periodo = float(retornos.max())
            pior_retorno_periodo = float(retornos.min())
        else:
            retorno_acumulado_total = np.nan
            retorno_medio_periodo = np.nan
            retorno_mediano_periodo = np.nan
            melhor_retorno_periodo = np.nan
            pior_retorno_periodo = np.nan

        if len(taxas_livres) > 0:
            taxa_livre_risco_acumulada_total = float((1 + taxas_livres).prod() - 1)
            taxa_livre_risco_media_periodo = float(taxas_livres.mean())
        else:
            taxa_livre_risco_acumulada_total = np.nan
            taxa_livre_risco_media_periodo = np.nan

        retorno_anualizado = calcular_retorno_anualizado(retorno_acumulado_total, data_inicio, data_fim)
        taxa_livre_risco_anualizada = calcular_retorno_anualizado(taxa_livre_risco_acumulada_total, data_inicio, data_fim)
        excesso_retorno_anualizado = retorno_anualizado - taxa_livre_risco_anualizada if pd.notna(retorno_anualizado) and pd.notna(taxa_livre_risco_anualizada) else np.nan

        if len(retornos_excesso) >= 2:
            retorno_excesso_medio_periodo = float(retornos_excesso.mean())
            retorno_excesso_mediano_periodo = float(retornos_excesso.median())
            volatilidade_excesso_periodo = float(retornos_excesso.std(ddof=1))
            volatilidade_excesso_anualizada = float(volatilidade_excesso_periodo * np.sqrt(fator_anualizacao))
            melhor_excesso_periodo = float(retornos_excesso.max())
            pior_excesso_periodo = float(retornos_excesso.min())
            pct_periodos_excesso_positivo = float((retornos_excesso > 0).mean())
            pct_periodos_excesso_negativo = float((retornos_excesso < 0).mean())
        elif len(retornos_excesso) == 1:
            retorno_excesso_medio_periodo = float(retornos_excesso.mean())
            retorno_excesso_mediano_periodo = float(retornos_excesso.median())
            volatilidade_excesso_periodo = np.nan
            volatilidade_excesso_anualizada = np.nan
            melhor_excesso_periodo = float(retornos_excesso.max())
            pior_excesso_periodo = float(retornos_excesso.min())
            pct_periodos_excesso_positivo = float((retornos_excesso > 0).mean())
            pct_periodos_excesso_negativo = float((retornos_excesso < 0).mean())
        else:
            retorno_excesso_medio_periodo = np.nan
            retorno_excesso_mediano_periodo = np.nan
            volatilidade_excesso_periodo = np.nan
            volatilidade_excesso_anualizada = np.nan
            melhor_excesso_periodo = np.nan
            pior_excesso_periodo = np.nan
            pct_periodos_excesso_positivo = np.nan
            pct_periodos_excesso_negativo = np.nan

        downside_excesso_periodo = calcular_downside_excesso(retornos_excesso)
        downside_excesso_anualizado = downside_excesso_periodo * np.sqrt(fator_anualizacao) if pd.notna(downside_excesso_periodo) else np.nan

        sharpe_periodo = dividir_seguro(retorno_excesso_medio_periodo, volatilidade_sharpe_periodo)
        sharpe_anualizado = dividir_seguro(excesso_retorno_anualizado, volatilidade_sharpe_anualizada)

        sortino_periodo = dividir_seguro(retorno_excesso_medio_periodo, downside_excesso_periodo)
        sortino_anualizado = dividir_seguro(excesso_retorno_anualizado, downside_excesso_anualizado)

        max_drawdown, data_max_drawdown = calcular_drawdown_por_retornos(grupo, "retorno_periodo")
        calmar = dividir_seguro(retorno_anualizado, abs(max_drawdown)) if pd.notna(max_drawdown) and max_drawdown < 0 else np.nan

        registro = {
            "chave_estrategia": chave,
            "estrategia_id": extrair_valor_linha(primeiro, "estrategia_id"),
            "grupo_controle": extrair_valor_linha(primeiro, "grupo_controle"),
            "familia_estrategia": extrair_valor_linha(primeiro, "familia_estrategia"),
            "estrategia_referencia": extrair_valor_linha(primeiro, "estrategia_referencia"),
            "estrategia_referencia_padronizada": extrair_valor_linha(primeiro, "estrategia_referencia_padronizada"),
            "tipo_estrategia": extrair_valor_linha(primeiro, "tipo_estrategia"),
            "carteira_id": extrair_valor_linha(primeiro, "carteira_id"),
            "replica_id": extrair_valor_linha(primeiro, "replica_id"),
            "nome_exibicao_estrategia": extrair_valor_linha(primeiro, "nome_exibicao_estrategia"),
            "categoria_estrategia": extrair_valor_linha(primeiro, "categoria_estrategia"),
            "subcategoria_estrategia": extrair_valor_linha(primeiro, "subcategoria_estrategia"),
            "origem_backtest": extrair_valor_linha(primeiro, "origem_backtest"),
            "ordem_exibicao": extrair_valor_linha(primeiro, "ordem_exibicao"),
            "frequencia": frequencia,
            "fator_anualizacao": fator_anualizacao,
            "n_periodos": int(len(grupo)),
            "n_retornos_validos": int(len(retornos)),
            "n_retornos_excesso_validos": int(len(retornos_excesso)),
            "data_inicio": data_inicio,
            "data_fim": data_fim,
            "n_dias_corridos": n_dias_corridos,
            "n_anos": n_anos,
            "retorno_acumulado_total": retorno_acumulado_total,
            "retorno_anualizado": retorno_anualizado,
            "retorno_medio_periodo": retorno_medio_periodo,
            "retorno_mediano_periodo": retorno_mediano_periodo,
            "melhor_retorno_periodo": melhor_retorno_periodo,
            "pior_retorno_periodo": pior_retorno_periodo,
            "taxa_livre_risco_acumulada_total": taxa_livre_risco_acumulada_total,
            "taxa_livre_risco_anualizada": taxa_livre_risco_anualizada,
            "taxa_livre_risco_media_periodo": taxa_livre_risco_media_periodo,
            "excesso_retorno_anualizado": excesso_retorno_anualizado,
            "retorno_excesso_medio_periodo": retorno_excesso_medio_periodo,
            "retorno_excesso_mediano_periodo": retorno_excesso_mediano_periodo,
            "volatilidade_excesso_periodo": volatilidade_excesso_periodo,
            "volatilidade_excesso_anualizada": volatilidade_excesso_anualizada,
            "volatilidade_sharpe_periodo": volatilidade_sharpe_periodo,
            "volatilidade_sharpe_anualizada": volatilidade_sharpe_anualizada,
            "downside_risco_periodo": downside_risco_periodo,
            "downside_risco_anualizado": downside_risco_anualizado,
            "downside_excesso_periodo": downside_excesso_periodo,
            "downside_excesso_anualizado": downside_excesso_anualizado,
            "melhor_excesso_periodo": melhor_excesso_periodo,
            "pior_excesso_periodo": pior_excesso_periodo,
            "pct_periodos_excesso_positivo": pct_periodos_excesso_positivo,
            "pct_periodos_excesso_negativo": pct_periodos_excesso_negativo,
            "sharpe_periodo": sharpe_periodo,
            "sharpe_anualizado": sharpe_anualizado,
            "sortino_periodo": sortino_periodo,
            "sortino_anualizado": sortino_anualizado,
            "max_drawdown": max_drawdown,
            "data_max_drawdown": data_max_drawdown,
            "calmar": calmar,
        }

        if len(risco_linha_df) > 0:
            for coluna in colunas_risco:
                if coluna != "chave_estrategia":
                    registro[f"risco_{coluna}"] = risco_linha[coluna]

        registros.append(registro)

    df_metricas = pd.DataFrame(registros)
    df_metricas = (
        df_metricas
        .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia"])
        .reset_index(drop=True)
    )

    return df_metricas

df_tbl_sharpe_sortino_calmar_diario = resumir_metricas_eficiencia(
    df_periodo=df_excesso_diario,
    df_risco=df_risco_diario,
    frequencia="diaria",
    fator_anualizacao=252,
    coluna_data_ordenacao="data",
)

df_tbl_sharpe_sortino_calmar_mensal = resumir_metricas_eficiencia(
    df_periodo=df_excesso_mensal,
    df_risco=df_risco_mensal,
    frequencia="mensal",
    fator_anualizacao=12,
    coluna_data_ordenacao="data_fim_periodo",
)

df_tbl_sharpe_sortino_calmar_anual = resumir_metricas_eficiencia(
    df_periodo=df_excesso_anual,
    df_risco=df_risco_anual,
    frequencia="anual",
    fator_anualizacao=1,
    coluna_data_ordenacao="data_fim_periodo",
)

print(f"Linhas da tabela diária de eficiência  : {len(df_tbl_sharpe_sortino_calmar_diario):,}")
print(f"Linhas da tabela mensal de eficiência  : {len(df_tbl_sharpe_sortino_calmar_mensal):,}")
print(f"Linhas da tabela anual de eficiência   : {len(df_tbl_sharpe_sortino_calmar_anual):,}")
print("OK")

# ============================================================
# 9) Consolidação do resumo multifrequência
# ============================================================

print("\n[9/11] Consolidação do resumo de eficiência multifrequência...")

df_metricas_eficiencia_multifrequencia = pd.concat(
    [
        df_tbl_sharpe_sortino_calmar_diario,
        df_tbl_sharpe_sortino_calmar_mensal,
        df_tbl_sharpe_sortino_calmar_anual,
    ],
    ignore_index=True,
)

df_metricas_eficiencia_multifrequencia = (
    df_metricas_eficiencia_multifrequencia
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", "frequencia"])
    .reset_index(drop=True)
)

frequencias_resumo = sorted(df_metricas_eficiencia_multifrequencia["frequencia"].dropna().unique())

print(f"Linhas do resumo multifrequência       : {len(df_metricas_eficiencia_multifrequencia):,}")
print(f"Frequências no resumo                  : {', '.join(frequencias_resumo)}")
print("OK")

# ============================================================
# 10) Auditoria de validação das métricas de eficiência
# ============================================================

print("\n[10/11] Construção da auditoria de validação das métricas de eficiência...")

n_estrategias_diarias = int(df_excesso_diario["chave_estrategia"].nunique())
n_estrategias_mensais = int(df_excesso_mensal["chave_estrategia"].nunique())
n_estrategias_anuais = int(df_excesso_anual["chave_estrategia"].nunique())
linhas_resumo_esperadas = int(n_estrategias_diarias * 3)

metricas_centrales = [
    "sharpe_anualizado",
    "sortino_anualizado",
    "calmar",
    "max_drawdown",
    "retorno_anualizado",
    "taxa_livre_risco_anualizada",
]

valores_infinitos = 0
for coluna in metricas_centrales:
    valores = pd.to_numeric(df_metricas_eficiencia_multifrequencia[coluna], errors="coerce")
    valores_infinitos += int(np.isinf(valores).sum())

sharpe_ausente = int(df_metricas_eficiencia_multifrequencia["sharpe_anualizado"].isna().sum())
denominador_sharpe_ausente = int(df_metricas_eficiencia_multifrequencia["volatilidade_sharpe_anualizada"].isna().sum())
sortino_ausente = int(df_metricas_eficiencia_multifrequencia["sortino_anualizado"].isna().sum())
calmar_ausente = int(df_metricas_eficiencia_multifrequencia["calmar"].isna().sum())
max_drawdown_ausente = int(df_metricas_eficiencia_multifrequencia["max_drawdown"].isna().sum())
retorno_anualizado_ausente = int(df_metricas_eficiencia_multifrequencia["retorno_anualizado"].isna().sum())
taxa_livre_risco_anualizada_ausente = int(df_metricas_eficiencia_multifrequencia["taxa_livre_risco_anualizada"].isna().sum())

registros_auditoria = [
    {
        "item": "estrategias_diarias_vs_mensais",
        "valor": n_estrategias_diarias,
        "valor_referencia": n_estrategias_mensais,
        "status": "OK" if n_estrategias_diarias == n_estrategias_mensais else "ERRO",
        "observacao": "A quantidade de estratégias deve ser igual entre as visões diária e mensal.",
    },
    {
        "item": "estrategias_diarias_vs_anuais",
        "valor": n_estrategias_diarias,
        "valor_referencia": n_estrategias_anuais,
        "status": "OK" if n_estrategias_diarias == n_estrategias_anuais else "ERRO",
        "observacao": "A quantidade de estratégias deve ser igual entre as visões diária e anual.",
    },
    {
        "item": "linhas_tabela_eficiencia_diaria",
        "valor": int(len(df_tbl_sharpe_sortino_calmar_diario)),
        "valor_referencia": n_estrategias_diarias,
        "status": "OK" if len(df_tbl_sharpe_sortino_calmar_diario) == n_estrategias_diarias else "ERRO",
        "observacao": "A tabela diária de eficiência deve ter uma linha por estratégia.",
    },
    {
        "item": "linhas_tabela_eficiencia_mensal",
        "valor": int(len(df_tbl_sharpe_sortino_calmar_mensal)),
        "valor_referencia": n_estrategias_mensais,
        "status": "OK" if len(df_tbl_sharpe_sortino_calmar_mensal) == n_estrategias_mensais else "ERRO",
        "observacao": "A tabela mensal de eficiência deve ter uma linha por estratégia.",
    },
    {
        "item": "linhas_tabela_eficiencia_anual",
        "valor": int(len(df_tbl_sharpe_sortino_calmar_anual)),
        "valor_referencia": n_estrategias_anuais,
        "status": "OK" if len(df_tbl_sharpe_sortino_calmar_anual) == n_estrategias_anuais else "ERRO",
        "observacao": "A tabela anual de eficiência deve ter uma linha por estratégia.",
    },
    {
        "item": "linhas_resumo_multifrequencia",
        "valor": int(len(df_metricas_eficiencia_multifrequencia)),
        "valor_referencia": linhas_resumo_esperadas,
        "status": "OK" if len(df_metricas_eficiencia_multifrequencia) == linhas_resumo_esperadas else "ERRO",
        "observacao": "O resumo multifrequência deve ter uma linha por estratégia e frequência.",
    },
    {
        "item": "frequencias_resumo_multifrequencia",
        "valor": int(len(frequencias_resumo)),
        "valor_referencia": 3,
        "status": "OK" if frequencias_resumo == ["anual", "diaria", "mensal"] else "ERRO",
        "observacao": "O resumo deve conter exatamente as frequências anual, diária e mensal.",
    },
    {
        "item": "retornos_excesso_ausentes",
        "valor": int(retornos_excesso_ausentes_diarios + retornos_excesso_ausentes_mensais + retornos_excesso_ausentes_anuais),
        "valor_referencia": 0,
        "status": "OK" if int(retornos_excesso_ausentes_diarios + retornos_excesso_ausentes_mensais + retornos_excesso_ausentes_anuais) == 0 else "ERRO",
        "observacao": "Retornos em excesso ausentes inviabilizam Sharpe e Sortino.",
    },
    {
        "item": "metricas_infinitas",
        "valor": int(valores_infinitos),
        "valor_referencia": 0,
        "status": "OK" if valores_infinitos == 0 else "ERRO",
        "observacao": "As métricas centrais não devem conter valores infinitos.",
    },
    {
        "item": "denominador_sharpe_ausente",
        "valor": denominador_sharpe_ausente,
        "valor_referencia": 0,
        "status": "OK" if denominador_sharpe_ausente == 0 else "ALERTA",
        "observacao": "O denominador principal do Sharpe deve vir da volatilidade dos retornos da etapa 11.2.",
    },
    {
        "item": "sharpe_anualizado_ausente",
        "valor": sharpe_ausente,
        "valor_referencia": 0,
        "status": "OK" if sharpe_ausente == 0 else "ALERTA",
        "observacao": "Sharpe ausente pode ocorrer se a volatilidade dos retornos for nula ou insuficiente.",
    },
    {
        "item": "sortino_anualizado_ausente",
        "valor": sortino_ausente,
        "valor_referencia": 0,
        "status": "OK" if sortino_ausente == 0 else "ALERTA",
        "observacao": "Sortino ausente pode ocorrer se não houver downside do retorno em excesso.",
    },
    {
        "item": "calmar_ausente",
        "valor": calmar_ausente,
        "valor_referencia": 0,
        "status": "OK" if calmar_ausente == 0 else "ALERTA",
        "observacao": "Calmar ausente pode ocorrer quando não há drawdown negativo na frequência analisada.",
    },
    {
        "item": "max_drawdown_ausente",
        "valor": max_drawdown_ausente,
        "valor_referencia": 0,
        "status": "OK" if max_drawdown_ausente == 0 else "ALERTA",
        "observacao": "Max drawdown ausente deve ser avaliado antes da subetapa específica de drawdown.",
    },
    {
        "item": "retorno_anualizado_ausente",
        "valor": retorno_anualizado_ausente,
        "valor_referencia": 0,
        "status": "OK" if retorno_anualizado_ausente == 0 else "ALERTA",
        "observacao": "Retorno anualizado ausente reduz a comparabilidade de Sharpe, Sortino e Calmar.",
    },
    {
        "item": "taxa_livre_risco_anualizada_ausente",
        "valor": taxa_livre_risco_anualizada_ausente,
        "valor_referencia": 0,
        "status": "OK" if taxa_livre_risco_anualizada_ausente == 0 else "ALERTA",
        "observacao": "Taxa livre de risco anualizada ausente indica falha na associação da Selic.",
    },
]

df_auditoria_validacao_eficiencia = pd.DataFrame(registros_auditoria)

n_erros_auditoria = int((df_auditoria_validacao_eficiencia["status"] == "ERRO").sum())

if n_erros_auditoria > 0:
    print(df_auditoria_validacao_eficiencia.to_string(index=False))
    raise ValueError("A auditoria da 11.3 identificou inconsistências bloqueantes.")

print(f"Itens de auditoria                     : {len(df_auditoria_validacao_eficiencia):,}")
print(f"Erros bloqueantes                      : {n_erros_auditoria:,}")
print("OK")

# ============================================================
# 11) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[11/11] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(df_excesso_diario, caminho_base_retornos_excesso_diarios, index=False)
salvar_dataframe(df_tbl_sharpe_sortino_calmar_diario, caminho_tbl_sharpe_sortino_calmar_diario, index=False)
salvar_dataframe(df_tbl_sharpe_sortino_calmar_mensal, caminho_tbl_sharpe_sortino_calmar_mensal, index=False)
salvar_dataframe(df_tbl_sharpe_sortino_calmar_anual, caminho_tbl_sharpe_sortino_calmar_anual, index=False)
salvar_dataframe(df_metricas_eficiencia_multifrequencia, caminho_tbl_metricas_eficiencia_multifrequencia, index=False)
salvar_dataframe(df_auditoria_validacao_eficiencia, caminho_tbl_auditoria_validacao_eficiencia, index=False)

PARAMS_SHARPE_SORTINO_CALMAR = {
    "fontes_oficiais": [
        "11_1_base_retornos_diarios",
        "11_1_tbl_retornos_mensais",
        "11_1_tbl_retornos_anuais",
        "11_2_tbl_risco_diario",
        "11_2_tbl_risco_mensal",
        "11_2_tbl_risco_anual",
        "1_3_base_selic_padronizada",
    ],
    "frequencias_calculadas": ["diaria", "mensal", "anual"],
    "taxa_livre_risco": "Selic diária padronizada da etapa 1.3",
    "metodo_sharpe": "retorno_excesso_dividido_pela_volatilidade_dos_retornos_da_etapa_11_2",
    "metodo_sortino": "retorno_excesso_dividido_pelo_downside_do_retorno_excesso_em_relacao_a_selic",
    "metodo_calmar": "retorno_anualizado_dividido_pelo_modulo_do_max_drawdown_da_frequencia",
    "fator_anualizacao_diaria": 252,
    "fator_anualizacao_mensal": 12,
    "fator_anualizacao_anual": 1,
    "metodo_conversao_selic": metodo_conversao_selic,
    "n_estrategias": int(n_estrategias_diarias),
    "data_inicio": df_retornos_diarios["data"].min(),
    "data_fim": df_retornos_diarios["data"].max(),
}

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação das métricas de eficiência:")
print(df_auditoria_validacao_eficiencia.to_string(index=False))

print("\nResumo multifrequência - amostra:")
print(df_metricas_eficiencia_multifrequencia.head(30).to_string(index=False))

print("\nMétricas diárias - amostra:")
print(df_tbl_sharpe_sortino_calmar_diario.head(20).to_string(index=False))

print("\nMétricas mensais - amostra:")
print(df_tbl_sharpe_sortino_calmar_mensal.head(20).to_string(index=False))

print("\nMétricas anuais - amostra:")
print(df_tbl_sharpe_sortino_calmar_anual.head(20).to_string(index=False))

print("\nBase diária de retornos em excesso - amostra:")
colunas_amostra_excesso = [
    "data",
    "chave_estrategia",
    "nome_exibicao_estrategia",
    "retorno_periodo",
    "taxa_livre_risco_periodo",
    "retorno_excesso_periodo",
    "retorno_excesso_acumulado_composto_ate_periodo",
]
colunas_amostra_excesso = [coluna for coluna in colunas_amostra_excesso if coluna in df_excesso_diario.columns]
print(df_excesso_diario[colunas_amostra_excesso].head(20).to_string(index=False))

print("\nArquivos salvos na subetapa 11.3:")
print(f"- {caminho_base_retornos_excesso_diarios}")
print(f"- {caminho_tbl_sharpe_sortino_calmar_diario}")
print(f"- {caminho_tbl_sharpe_sortino_calmar_mensal}")
print(f"- {caminho_tbl_sharpe_sortino_calmar_anual}")
print(f"- {caminho_tbl_metricas_eficiencia_multifrequencia}")
print(f"- {caminho_tbl_auditoria_validacao_eficiencia}")

print("\nETAPA 11.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 11.3 - SHARPE, SORTINO E CALMAR

[1/11] Validação inicial do ambiente...
OK

[2/11] Definição determinística dos caminhos de entrada e saída...
Entrada - retornos diários da 11.1          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_base_retornos_diarios.parquet
Entrada - retornos mensais da 11.1          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_tbl_retornos_mensais.parquet
Entrada - retornos anuais da 11.1           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_tbl_retornos_anuais.parquet
Entrada - risco diário da 11.2              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_2_tbl_risco_diario.parquet


## Etapa 11.4) Drawdown e Recuperação

In [55]:
%%time
# ============================================================
# Etapa 11.4) Drawdown e Recuperação
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 11.4 - DRAWDOWN E RECUPERAÇÃO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/11] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/11] Definição determinística dos caminhos de entrada e saída...")

caminho_base_retornos_diarios = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="base",
    nome="retornos_diarios",
)

caminho_tbl_retornos_mensais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_mensais",
)

caminho_tbl_retornos_anuais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_anuais",
)

caminho_base_drawdown_diario = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="base",
    nome="drawdown_diario",
)

caminho_base_drawdown_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="base",
    nome="drawdown_mensal",
)

caminho_base_drawdown_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="base",
    nome="drawdown_anual",
)

caminho_tbl_eventos_drawdown_diario = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="eventos_drawdown_diario",
)

caminho_tbl_eventos_drawdown_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="eventos_drawdown_mensal",
)

caminho_tbl_eventos_drawdown_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="eventos_drawdown_anual",
)

caminho_tbl_drawdown_recuperacao_diario = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="drawdown_recuperacao_diario",
)

caminho_tbl_drawdown_recuperacao_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="drawdown_recuperacao_mensal",
)

caminho_tbl_drawdown_recuperacao_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="drawdown_recuperacao_anual",
)

caminho_tbl_resumo_drawdown_recuperacao_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="resumo_drawdown_recuperacao_multifrequencia",
)

caminho_tbl_auditoria_validacao_drawdown = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_drawdown",
)

print(f"Entrada - retornos diários da 11.1          : {caminho_base_retornos_diarios}")
print(f"Entrada - retornos mensais da 11.1          : {caminho_tbl_retornos_mensais}")
print(f"Entrada - retornos anuais da 11.1           : {caminho_tbl_retornos_anuais}")
print(f"Saída   - base de drawdown diária           : {caminho_base_drawdown_diario}")
print(f"Saída   - base de drawdown mensal           : {caminho_base_drawdown_mensal}")
print(f"Saída   - base de drawdown anual            : {caminho_base_drawdown_anual}")
print(f"Saída   - eventos de drawdown diário        : {caminho_tbl_eventos_drawdown_diario}")
print(f"Saída   - eventos de drawdown mensal        : {caminho_tbl_eventos_drawdown_mensal}")
print(f"Saída   - eventos de drawdown anual         : {caminho_tbl_eventos_drawdown_anual}")
print(f"Saída   - resumo diário                     : {caminho_tbl_drawdown_recuperacao_diario}")
print(f"Saída   - resumo mensal                     : {caminho_tbl_drawdown_recuperacao_mensal}")
print(f"Saída   - resumo anual                      : {caminho_tbl_drawdown_recuperacao_anual}")
print(f"Saída   - resumo multifrequência            : {caminho_tbl_resumo_drawdown_recuperacao_multifrequencia}")
print(f"Saída   - auditoria de validação            : {caminho_tbl_auditoria_validacao_drawdown}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais de retornos da 11.1
# ============================================================

print("\n[3/11] Carga das bases oficiais de retornos da 11.1...")

df_retornos_diarios = pd.read_parquet(caminho_base_retornos_diarios)
df_retornos_mensais = pd.read_parquet(caminho_tbl_retornos_mensais)
df_retornos_anuais = pd.read_parquet(caminho_tbl_retornos_anuais)

print(f"Base de retornos diários              : {df_retornos_diarios.shape[0]:,} linhas x {df_retornos_diarios.shape[1]} colunas")
print(f"Tabela de retornos mensais            : {df_retornos_mensais.shape[0]:,} linhas x {df_retornos_mensais.shape[1]} colunas")
print(f"Tabela de retornos anuais             : {df_retornos_anuais.shape[0]:,} linhas x {df_retornos_anuais.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Validação estrutural das bases de entrada
# ============================================================

print("\n[4/11] Validação estrutural das bases de entrada...")

colunas_obrigatorias_diarias = [
    "data",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "patrimonio_total",
]

colunas_obrigatorias_mensais = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "ano_mes",
    "data_inicio_periodo",
    "data_fim_periodo",
    "patrimonio_fim_observado",
]

colunas_obrigatorias_anuais = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "ano",
    "data_inicio_periodo",
    "data_fim_periodo",
    "patrimonio_fim_observado",
]

colunas_ausentes_diarias = [coluna for coluna in colunas_obrigatorias_diarias if coluna not in df_retornos_diarios.columns]
colunas_ausentes_mensais = [coluna for coluna in colunas_obrigatorias_mensais if coluna not in df_retornos_mensais.columns]
colunas_ausentes_anuais = [coluna for coluna in colunas_obrigatorias_anuais if coluna not in df_retornos_anuais.columns]

if len(colunas_ausentes_diarias) > 0:
    raise ValueError(f"Colunas ausentes na base diária da 11.1: {colunas_ausentes_diarias}")
if len(colunas_ausentes_mensais) > 0:
    raise ValueError(f"Colunas ausentes na tabela mensal da 11.1: {colunas_ausentes_mensais}")
if len(colunas_ausentes_anuais) > 0:
    raise ValueError(f"Colunas ausentes na tabela anual da 11.1: {colunas_ausentes_anuais}")

df_retornos_diarios = df_retornos_diarios.copy()
df_retornos_mensais = df_retornos_mensais.copy()
df_retornos_anuais = df_retornos_anuais.copy()

df_retornos_diarios["data"] = pd.to_datetime(df_retornos_diarios["data"], errors="coerce")
df_retornos_mensais["data_inicio_periodo"] = pd.to_datetime(df_retornos_mensais["data_inicio_periodo"], errors="coerce")
df_retornos_mensais["data_fim_periodo"] = pd.to_datetime(df_retornos_mensais["data_fim_periodo"], errors="coerce")
df_retornos_anuais["data_inicio_periodo"] = pd.to_datetime(df_retornos_anuais["data_inicio_periodo"], errors="coerce")
df_retornos_anuais["data_fim_periodo"] = pd.to_datetime(df_retornos_anuais["data_fim_periodo"], errors="coerce")

duplicidades_diarias = int(df_retornos_diarios.duplicated(["chave_estrategia", "data"]).sum())
duplicidades_mensais = int(df_retornos_mensais.duplicated(["chave_estrategia", "ano_mes"]).sum())
duplicidades_anuais = int(df_retornos_anuais.duplicated(["chave_estrategia", "ano"]).sum())

datas_invalidas_diarias = int(df_retornos_diarios["data"].isna().sum())
datas_invalidas_mensais = int(df_retornos_mensais["data_fim_periodo"].isna().sum())
datas_invalidas_anuais = int(df_retornos_anuais["data_fim_periodo"].isna().sum())

patrimonio_ausente_diario = int(pd.to_numeric(df_retornos_diarios["patrimonio_total"], errors="coerce").isna().sum())
patrimonio_ausente_mensal = int(pd.to_numeric(df_retornos_mensais["patrimonio_fim_observado"], errors="coerce").isna().sum())
patrimonio_ausente_anual = int(pd.to_numeric(df_retornos_anuais["patrimonio_fim_observado"], errors="coerce").isna().sum())

print(f"Duplicidades diárias                  : {duplicidades_diarias:,}")
print(f"Duplicidades mensais                  : {duplicidades_mensais:,}")
print(f"Duplicidades anuais                   : {duplicidades_anuais:,}")
print(f"Datas inválidas diárias               : {datas_invalidas_diarias:,}")
print(f"Datas inválidas mensais               : {datas_invalidas_mensais:,}")
print(f"Datas inválidas anuais                : {datas_invalidas_anuais:,}")
print(f"Patrimônios ausentes diários          : {patrimonio_ausente_diario:,}")
print(f"Patrimônios ausentes mensais          : {patrimonio_ausente_mensal:,}")
print(f"Patrimônios ausentes anuais           : {patrimonio_ausente_anual:,}")

if duplicidades_diarias > 0:
    raise ValueError("A base diária possui duplicidades por chave_estrategia e data.")
if duplicidades_mensais > 0:
    raise ValueError("A tabela mensal possui duplicidades por chave_estrategia e ano_mes.")
if duplicidades_anuais > 0:
    raise ValueError("A tabela anual possui duplicidades por chave_estrategia e ano.")
if datas_invalidas_diarias > 0 or datas_invalidas_mensais > 0 or datas_invalidas_anuais > 0:
    raise ValueError("Existem datas inválidas nas bases de entrada da 11.1.")
if patrimonio_ausente_diario > 0 or patrimonio_ausente_mensal > 0 or patrimonio_ausente_anual > 0:
    raise ValueError("Existem patrimônios ausentes nas bases de entrada da 11.1.")

print("OK")

# ============================================================
# 5) Definição das funções auxiliares de drawdown e recuperação
# ============================================================

print("\n[5/11] Definição das funções auxiliares de drawdown e recuperação...")

colunas_identificacao = [
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "origem_backtest",
    "ordem_exibicao",
]

def extrair_valor_linha(linha, coluna, valor_padrao=np.nan):
    if coluna in linha.index:
        return linha[coluna]
    return valor_padrao

def ordenar_colunas_existentes(df, colunas_prioritarias):
    colunas_inicio = [coluna for coluna in colunas_prioritarias if coluna in df.columns]
    colunas_restantes = [coluna for coluna in df.columns if coluna not in colunas_inicio]
    return df[colunas_inicio + colunas_restantes]

def preparar_base_periodica(df_periodo, frequencia):
    df = df_periodo.copy()

    if frequencia == "diaria":
        coluna_data = "data"
        coluna_patrimonio = "patrimonio_total"
        coluna_periodo = None
    elif frequencia == "mensal":
        coluna_data = "data_fim_periodo"
        coluna_patrimonio = "patrimonio_fim_observado"
        coluna_periodo = "ano_mes"
    elif frequencia == "anual":
        coluna_data = "data_fim_periodo"
        coluna_patrimonio = "patrimonio_fim_observado"
        coluna_periodo = "ano"
    else:
        raise ValueError(f"Frequência não reconhecida: {frequencia}")

    df[coluna_data] = pd.to_datetime(df[coluna_data], errors="coerce")
    df[coluna_patrimonio] = pd.to_numeric(df[coluna_patrimonio], errors="coerce")

    df = df.dropna(subset=["chave_estrategia", coluna_data, coluna_patrimonio]).copy()
    df = df.sort_values(["chave_estrategia", coluna_data]).reset_index(drop=True)

    if frequencia != "diaria":
        df["data"] = df[coluna_data]
    if coluna_patrimonio != "patrimonio_total":
        df["patrimonio_total"] = df[coluna_patrimonio]
    if coluna_periodo is not None:
        df["periodo_referencia"] = df[coluna_periodo]
    else:
        df["periodo_referencia"] = df["data"].dt.strftime("%Y-%m-%d")

    df["frequencia"] = frequencia

    return df

def calcular_base_drawdown(df_periodo, frequencia):
    df = preparar_base_periodica(df_periodo=df_periodo, frequencia=frequencia)
    registros = []

    for chave, grupo in df.groupby("chave_estrategia", sort=False):
        grupo = grupo.sort_values("data").reset_index(drop=True).copy()
        patrimonio = pd.to_numeric(grupo["patrimonio_total"], errors="coerce")

        pico_acumulado = patrimonio.cummax()
        drawdown = np.where(pico_acumulado > 0, patrimonio / pico_acumulado - 1.0, np.nan)

        datas = pd.to_datetime(grupo["data"], errors="coerce")
        posicoes_pico = []
        datas_pico = []
        patrimonio_pico_corrente = -np.inf
        posicao_pico_corrente = 0
        data_pico_corrente = pd.NaT

        for posicao, valor_patrimonio in enumerate(patrimonio):
            if pd.notna(valor_patrimonio) and (valor_patrimonio > patrimonio_pico_corrente):
                patrimonio_pico_corrente = float(valor_patrimonio)
                posicao_pico_corrente = int(posicao)
                data_pico_corrente = datas.iloc[posicao]
            posicoes_pico.append(posicao_pico_corrente)
            datas_pico.append(data_pico_corrente)

        grupo["posicao_periodo"] = np.arange(len(grupo), dtype=int)
        grupo["patrimonio_pico_acumulado"] = pico_acumulado.astype(float)
        grupo["data_pico_acumulado"] = pd.to_datetime(datas_pico, errors="coerce")
        grupo["posicao_pico_acumulado"] = posicoes_pico
        grupo["drawdown"] = pd.Series(drawdown, index=grupo.index).astype(float)
        grupo["drawdown_pct"] = grupo["drawdown"]
        grupo["drawdown_abs"] = grupo["patrimonio_total"] - grupo["patrimonio_pico_acumulado"]
        grupo["em_drawdown"] = grupo["drawdown"] < 0
        grupo["novo_pico_historico"] = grupo["drawdown"].fillna(0).eq(0)
        grupo["periodos_desde_pico"] = grupo["posicao_periodo"] - grupo["posicao_pico_acumulado"]

        registros.append(grupo)

    if len(registros) == 0:
        return pd.DataFrame()

    df_drawdown = pd.concat(registros, ignore_index=True)

    colunas_prioritarias = [
        *colunas_identificacao,
        "frequencia",
        "periodo_referencia",
        "data",
        "data_inicio_periodo",
        "data_fim_periodo",
        "ano",
        "mes",
        "ano_mes",
        "posicao_periodo",
        "patrimonio_total",
        "patrimonio_pico_acumulado",
        "data_pico_acumulado",
        "posicao_pico_acumulado",
        "drawdown",
        "drawdown_pct",
        "drawdown_abs",
        "em_drawdown",
        "novo_pico_historico",
        "periodos_desde_pico",
    ]

    df_drawdown = ordenar_colunas_existentes(df_drawdown, colunas_prioritarias)

    return df_drawdown

def identificar_eventos_drawdown(df_drawdown, frequencia):
    eventos = []

    for chave, grupo in df_drawdown.groupby("chave_estrategia", sort=False):
        grupo = grupo.sort_values("data").reset_index(drop=True).copy()

        em_evento = False
        evento = None
        contador_evento = 0

        for posicao, linha in grupo.iterrows():
            drawdown_linha = linha["drawdown"]
            em_drawdown_linha = bool(pd.notna(drawdown_linha) and drawdown_linha < 0)

            if em_drawdown_linha and not em_evento:
                contador_evento += 1
                em_evento = True
                evento = {
                    "chave_estrategia": chave,
                    "frequencia": frequencia,
                    "evento_drawdown_id": contador_evento,
                    "data_pico_pre_drawdown": linha["data_pico_acumulado"],
                    "posicao_pico_pre_drawdown": int(linha["posicao_pico_acumulado"]),
                    "patrimonio_pico_pre_drawdown": float(linha["patrimonio_pico_acumulado"]),
                    "data_inicio_drawdown": linha["data"],
                    "posicao_inicio_drawdown": int(linha["posicao_periodo"]),
                    "data_vale_drawdown": linha["data"],
                    "posicao_vale_drawdown": int(linha["posicao_periodo"]),
                    "patrimonio_vale_drawdown": float(linha["patrimonio_total"]),
                    "max_drawdown_evento": float(drawdown_linha),
                    "data_recuperacao": pd.NaT,
                    "posicao_recuperacao": np.nan,
                    "patrimonio_recuperacao": np.nan,
                    "recuperado": False,
                }

            if em_evento and evento is not None:
                if pd.notna(drawdown_linha) and drawdown_linha < evento["max_drawdown_evento"]:
                    evento["max_drawdown_evento"] = float(drawdown_linha)
                    evento["data_vale_drawdown"] = linha["data"]
                    evento["posicao_vale_drawdown"] = int(linha["posicao_periodo"])
                    evento["patrimonio_vale_drawdown"] = float(linha["patrimonio_total"])

                if pd.notna(drawdown_linha) and drawdown_linha >= 0:
                    evento["data_recuperacao"] = linha["data"]
                    evento["posicao_recuperacao"] = int(linha["posicao_periodo"])
                    evento["patrimonio_recuperacao"] = float(linha["patrimonio_total"])
                    evento["recuperado"] = True
                    eventos.append(evento)
                    em_evento = False
                    evento = None

        if em_evento and evento is not None:
            ultima_linha = grupo.iloc[-1]
            evento["data_fim_observacao"] = ultima_linha["data"]
            evento["posicao_fim_observacao"] = int(ultima_linha["posicao_periodo"])
            eventos.append(evento)

    if len(eventos) == 0:
        colunas_eventos = [
            "chave_estrategia",
            "frequencia",
            "evento_drawdown_id",
            "data_pico_pre_drawdown",
            "posicao_pico_pre_drawdown",
            "patrimonio_pico_pre_drawdown",
            "data_inicio_drawdown",
            "posicao_inicio_drawdown",
            "data_vale_drawdown",
            "posicao_vale_drawdown",
            "patrimonio_vale_drawdown",
            "max_drawdown_evento",
            "data_recuperacao",
            "posicao_recuperacao",
            "patrimonio_recuperacao",
            "recuperado",
            "data_fim_observacao",
            "posicao_fim_observacao",
        ]
        return pd.DataFrame(columns=colunas_eventos)

    df_eventos = pd.DataFrame(eventos)

    if "data_fim_observacao" not in df_eventos.columns:
        df_eventos["data_fim_observacao"] = pd.NaT
    if "posicao_fim_observacao" not in df_eventos.columns:
        df_eventos["posicao_fim_observacao"] = np.nan

    for coluna in ["data_pico_pre_drawdown", "data_inicio_drawdown", "data_vale_drawdown", "data_recuperacao", "data_fim_observacao"]:
        df_eventos[coluna] = pd.to_datetime(df_eventos[coluna], errors="coerce")

    df_eventos["amplitude_drawdown_evento_abs"] = df_eventos["max_drawdown_evento"].abs()
    df_eventos["periodos_pico_ao_vale"] = df_eventos["posicao_vale_drawdown"] - df_eventos["posicao_pico_pre_drawdown"]
    df_eventos["periodos_inicio_ao_vale"] = df_eventos["posicao_vale_drawdown"] - df_eventos["posicao_inicio_drawdown"] + 1
    df_eventos["periodos_pico_a_recuperacao"] = np.where(
        df_eventos["recuperado"],
        df_eventos["posicao_recuperacao"] - df_eventos["posicao_pico_pre_drawdown"],
        np.nan,
    )
    df_eventos["periodos_vale_a_recuperacao"] = np.where(
        df_eventos["recuperado"],
        df_eventos["posicao_recuperacao"] - df_eventos["posicao_vale_drawdown"],
        np.nan,
    )
    df_eventos["periodos_em_aberto"] = np.where(
        ~df_eventos["recuperado"],
        df_eventos["posicao_fim_observacao"] - df_eventos["posicao_pico_pre_drawdown"],
        np.nan,
    )

    metadados = (
        df_drawdown
        .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", "data"])
        .groupby("chave_estrategia", as_index=False)
        .first()[[coluna for coluna in colunas_identificacao if coluna in df_drawdown.columns]]
    )

    df_eventos = df_eventos.merge(metadados, on="chave_estrategia", how="left")

    colunas_prioritarias = [
        *colunas_identificacao,
        "frequencia",
        "evento_drawdown_id",
        "data_pico_pre_drawdown",
        "data_inicio_drawdown",
        "data_vale_drawdown",
        "data_recuperacao",
        "data_fim_observacao",
        "recuperado",
        "max_drawdown_evento",
        "amplitude_drawdown_evento_abs",
        "patrimonio_pico_pre_drawdown",
        "patrimonio_vale_drawdown",
        "patrimonio_recuperacao",
        "periodos_pico_ao_vale",
        "periodos_inicio_ao_vale",
        "periodos_pico_a_recuperacao",
        "periodos_vale_a_recuperacao",
        "periodos_em_aberto",
    ]

    df_eventos = ordenar_colunas_existentes(df_eventos, colunas_prioritarias)
    df_eventos = df_eventos.sort_values(["ordem_exibicao", "chave_estrategia", "evento_drawdown_id"]).reset_index(drop=True)

    return df_eventos

def resumir_drawdown_recuperacao(df_drawdown, df_eventos, frequencia):
    registros = []

    for chave, grupo in df_drawdown.groupby("chave_estrategia", sort=False):
        grupo = grupo.sort_values("data").reset_index(drop=True).copy()
        primeiro = grupo.iloc[0]
        ultimo = grupo.iloc[-1]
        grupo_eventos = df_eventos[df_eventos["chave_estrategia"].eq(chave)].copy() if len(df_eventos) > 0 else pd.DataFrame()

        serie_drawdown = pd.to_numeric(grupo["drawdown"], errors="coerce")
        serie_drawdown_negativo = serie_drawdown[serie_drawdown < 0]
        idx_max_drawdown = serie_drawdown.idxmin() if serie_drawdown.notna().any() else None

        if idx_max_drawdown is not None and pd.notna(idx_max_drawdown):
            linha_max_drawdown = grupo.loc[idx_max_drawdown]
            max_drawdown = float(linha_max_drawdown["drawdown"])
            data_max_drawdown = linha_max_drawdown["data"]
            data_pico_max_drawdown = linha_max_drawdown["data_pico_acumulado"]
            patrimonio_pico_max_drawdown = linha_max_drawdown["patrimonio_pico_acumulado"]
            patrimonio_vale_max_drawdown = linha_max_drawdown["patrimonio_total"]
            periodos_pico_ao_vale_max_drawdown = linha_max_drawdown["posicao_periodo"] - linha_max_drawdown["posicao_pico_acumulado"]
        else:
            max_drawdown = np.nan
            data_max_drawdown = pd.NaT
            data_pico_max_drawdown = pd.NaT
            patrimonio_pico_max_drawdown = np.nan
            patrimonio_vale_max_drawdown = np.nan
            periodos_pico_ao_vale_max_drawdown = np.nan

        n_eventos_drawdown = int(len(grupo_eventos))
        n_eventos_recuperados = int(grupo_eventos["recuperado"].sum()) if n_eventos_drawdown > 0 else 0
        n_eventos_nao_recuperados = int(n_eventos_drawdown - n_eventos_recuperados)

        if n_eventos_drawdown > 0:
            pior_evento = grupo_eventos.sort_values("max_drawdown_evento").iloc[0]
            data_recuperacao_max_drawdown = pior_evento["data_recuperacao"]
            recuperado_max_drawdown = bool(pior_evento["recuperado"])
            periodos_pico_a_recuperacao_max_drawdown = pior_evento["periodos_pico_a_recuperacao"]
            periodos_vale_a_recuperacao_max_drawdown = pior_evento["periodos_vale_a_recuperacao"]
            amplitude_media_eventos = float(pd.to_numeric(grupo_eventos["amplitude_drawdown_evento_abs"], errors="coerce").mean())
            amplitude_mediana_eventos = float(pd.to_numeric(grupo_eventos["amplitude_drawdown_evento_abs"], errors="coerce").median())
            duracao_media_eventos_recuperados = float(pd.to_numeric(grupo_eventos.loc[grupo_eventos["recuperado"], "periodos_pico_a_recuperacao"], errors="coerce").mean())
            duracao_mediana_eventos_recuperados = float(pd.to_numeric(grupo_eventos.loc[grupo_eventos["recuperado"], "periodos_pico_a_recuperacao"], errors="coerce").median())
            duracao_max_eventos_recuperados = float(pd.to_numeric(grupo_eventos.loc[grupo_eventos["recuperado"], "periodos_pico_a_recuperacao"], errors="coerce").max())
            tempo_medio_vale_a_recuperacao = float(pd.to_numeric(grupo_eventos.loc[grupo_eventos["recuperado"], "periodos_vale_a_recuperacao"], errors="coerce").mean())
            max_periodos_em_aberto = float(pd.to_numeric(grupo_eventos.loc[~grupo_eventos["recuperado"], "periodos_em_aberto"], errors="coerce").max())
        else:
            data_recuperacao_max_drawdown = pd.NaT
            recuperado_max_drawdown = False
            periodos_pico_a_recuperacao_max_drawdown = np.nan
            periodos_vale_a_recuperacao_max_drawdown = np.nan
            amplitude_media_eventos = np.nan
            amplitude_mediana_eventos = np.nan
            duracao_media_eventos_recuperados = np.nan
            duracao_mediana_eventos_recuperados = np.nan
            duracao_max_eventos_recuperados = np.nan
            tempo_medio_vale_a_recuperacao = np.nan
            max_periodos_em_aberto = np.nan

        registro = {
            "chave_estrategia": chave,
            "estrategia_id": extrair_valor_linha(primeiro, "estrategia_id"),
            "grupo_controle": extrair_valor_linha(primeiro, "grupo_controle"),
            "familia_estrategia": extrair_valor_linha(primeiro, "familia_estrategia"),
            "estrategia_referencia": extrair_valor_linha(primeiro, "estrategia_referencia"),
            "estrategia_referencia_padronizada": extrair_valor_linha(primeiro, "estrategia_referencia_padronizada"),
            "tipo_estrategia": extrair_valor_linha(primeiro, "tipo_estrategia"),
            "carteira_id": extrair_valor_linha(primeiro, "carteira_id"),
            "replica_id": extrair_valor_linha(primeiro, "replica_id"),
            "nome_exibicao_estrategia": extrair_valor_linha(primeiro, "nome_exibicao_estrategia"),
            "categoria_estrategia": extrair_valor_linha(primeiro, "categoria_estrategia"),
            "subcategoria_estrategia": extrair_valor_linha(primeiro, "subcategoria_estrategia"),
            "origem_backtest": extrair_valor_linha(primeiro, "origem_backtest"),
            "ordem_exibicao": extrair_valor_linha(primeiro, "ordem_exibicao"),
            "frequencia": frequencia,
            "n_periodos": int(len(grupo)),
            "data_inicio": primeiro["data"],
            "data_fim": ultimo["data"],
            "patrimonio_inicio_observado": float(primeiro["patrimonio_total"]),
            "patrimonio_fim_observado": float(ultimo["patrimonio_total"]),
            "patrimonio_pico_final": float(ultimo["patrimonio_pico_acumulado"]),
            "data_pico_final": ultimo["data_pico_acumulado"],
            "drawdown_atual": float(ultimo["drawdown"]),
            "em_drawdown_atual": bool(ultimo["em_drawdown"]),
            "periodos_desde_pico_atual": int(ultimo["periodos_desde_pico"]),
            "max_drawdown": max_drawdown,
            "max_drawdown_abs": abs(max_drawdown) if pd.notna(max_drawdown) else np.nan,
            "data_pico_max_drawdown": data_pico_max_drawdown,
            "data_max_drawdown": data_max_drawdown,
            "data_recuperacao_max_drawdown": data_recuperacao_max_drawdown,
            "recuperado_max_drawdown": recuperado_max_drawdown,
            "patrimonio_pico_max_drawdown": patrimonio_pico_max_drawdown,
            "patrimonio_vale_max_drawdown": patrimonio_vale_max_drawdown,
            "periodos_pico_ao_vale_max_drawdown": periodos_pico_ao_vale_max_drawdown,
            "periodos_pico_a_recuperacao_max_drawdown": periodos_pico_a_recuperacao_max_drawdown,
            "periodos_vale_a_recuperacao_max_drawdown": periodos_vale_a_recuperacao_max_drawdown,
            "pct_periodos_em_drawdown": float(grupo["em_drawdown"].mean()),
            "drawdown_medio_periodos": float(serie_drawdown.mean()) if serie_drawdown.notna().any() else np.nan,
            "drawdown_medio_periodos_negativos": float(serie_drawdown_negativo.mean()) if len(serie_drawdown_negativo) > 0 else np.nan,
            "n_eventos_drawdown": n_eventos_drawdown,
            "n_eventos_recuperados": n_eventos_recuperados,
            "n_eventos_nao_recuperados": n_eventos_nao_recuperados,
            "pct_eventos_recuperados": n_eventos_recuperados / n_eventos_drawdown if n_eventos_drawdown > 0 else np.nan,
            "amplitude_media_eventos": amplitude_media_eventos,
            "amplitude_mediana_eventos": amplitude_mediana_eventos,
            "duracao_media_eventos_recuperados_periodos": duracao_media_eventos_recuperados,
            "duracao_mediana_eventos_recuperados_periodos": duracao_mediana_eventos_recuperados,
            "duracao_max_eventos_recuperados_periodos": duracao_max_eventos_recuperados,
            "tempo_medio_vale_a_recuperacao_periodos": tempo_medio_vale_a_recuperacao,
            "max_periodos_em_aberto": max_periodos_em_aberto,
        }

        registros.append(registro)

    df_resumo = pd.DataFrame(registros)
    df_resumo = df_resumo.sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia"]).reset_index(drop=True)

    return df_resumo

print("OK")

# ============================================================
# 6) Construção das bases de drawdown por frequência
# ============================================================

print("\n[6/11] Construção das bases de drawdown por frequência...")

df_base_drawdown_diario = calcular_base_drawdown(df_periodo=df_retornos_diarios, frequencia="diaria")
df_base_drawdown_mensal = calcular_base_drawdown(df_periodo=df_retornos_mensais, frequencia="mensal")
df_base_drawdown_anual = calcular_base_drawdown(df_periodo=df_retornos_anuais, frequencia="anual")

print(f"Linhas da base de drawdown diária      : {len(df_base_drawdown_diario):,}")
print(f"Linhas da base de drawdown mensal      : {len(df_base_drawdown_mensal):,}")
print(f"Linhas da base de drawdown anual       : {len(df_base_drawdown_anual):,}")
print(f"Estratégias na base diária             : {df_base_drawdown_diario['chave_estrategia'].nunique():,}")
print(f"Estratégias na base mensal             : {df_base_drawdown_mensal['chave_estrategia'].nunique():,}")
print(f"Estratégias na base anual              : {df_base_drawdown_anual['chave_estrategia'].nunique():,}")
print("OK")

# ============================================================
# 7) Identificação dos eventos de drawdown e recuperação
# ============================================================

print("\n[7/11] Identificação dos eventos de drawdown e recuperação...")

df_eventos_drawdown_diario = identificar_eventos_drawdown(df_base_drawdown_diario, frequencia="diaria")
df_eventos_drawdown_mensal = identificar_eventos_drawdown(df_base_drawdown_mensal, frequencia="mensal")
df_eventos_drawdown_anual = identificar_eventos_drawdown(df_base_drawdown_anual, frequencia="anual")

print(f"Eventos de drawdown diários            : {len(df_eventos_drawdown_diario):,}")
print(f"Eventos de drawdown mensais            : {len(df_eventos_drawdown_mensal):,}")
print(f"Eventos de drawdown anuais             : {len(df_eventos_drawdown_anual):,}")
print(f"Eventos diários recuperados            : {int(df_eventos_drawdown_diario['recuperado'].sum()) if len(df_eventos_drawdown_diario) > 0 else 0:,}")
print(f"Eventos mensais recuperados            : {int(df_eventos_drawdown_mensal['recuperado'].sum()) if len(df_eventos_drawdown_mensal) > 0 else 0:,}")
print(f"Eventos anuais recuperados             : {int(df_eventos_drawdown_anual['recuperado'].sum()) if len(df_eventos_drawdown_anual) > 0 else 0:,}")
print("OK")

# ============================================================
# 8) Construção dos resumos de drawdown e recuperação
# ============================================================

print("\n[8/11] Construção dos resumos de drawdown e recuperação...")

df_tbl_drawdown_recuperacao_diario = resumir_drawdown_recuperacao(
    df_drawdown=df_base_drawdown_diario,
    df_eventos=df_eventos_drawdown_diario,
    frequencia="diaria",
)

df_tbl_drawdown_recuperacao_mensal = resumir_drawdown_recuperacao(
    df_drawdown=df_base_drawdown_mensal,
    df_eventos=df_eventos_drawdown_mensal,
    frequencia="mensal",
)

df_tbl_drawdown_recuperacao_anual = resumir_drawdown_recuperacao(
    df_drawdown=df_base_drawdown_anual,
    df_eventos=df_eventos_drawdown_anual,
    frequencia="anual",
)

df_resumo_drawdown_recuperacao_multifrequencia = pd.concat(
    [
        df_tbl_drawdown_recuperacao_diario,
        df_tbl_drawdown_recuperacao_mensal,
        df_tbl_drawdown_recuperacao_anual,
    ],
    ignore_index=True,
)

df_resumo_drawdown_recuperacao_multifrequencia["ordem_frequencia"] = df_resumo_drawdown_recuperacao_multifrequencia["frequencia"].map({
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
})

df_resumo_drawdown_recuperacao_multifrequencia = (
    df_resumo_drawdown_recuperacao_multifrequencia
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", "ordem_frequencia"])
    .drop(columns=["ordem_frequencia"])
    .reset_index(drop=True)
)

print(f"Linhas do resumo diário                : {len(df_tbl_drawdown_recuperacao_diario):,}")
print(f"Linhas do resumo mensal                : {len(df_tbl_drawdown_recuperacao_mensal):,}")
print(f"Linhas do resumo anual                 : {len(df_tbl_drawdown_recuperacao_anual):,}")
print(f"Linhas do resumo multifrequência       : {len(df_resumo_drawdown_recuperacao_multifrequencia):,}")
print(f"Frequências no resumo                  : {', '.join(sorted(df_resumo_drawdown_recuperacao_multifrequencia['frequencia'].dropna().unique()))}")
print("OK")

# ============================================================
# 9) Construção da auditoria de validação das métricas de drawdown
# ============================================================

print("\n[9/11] Construção da auditoria de validação das métricas de drawdown...")

qtd_estrategias_diario = int(df_base_drawdown_diario["chave_estrategia"].nunique())
qtd_estrategias_mensal = int(df_base_drawdown_mensal["chave_estrategia"].nunique())
qtd_estrategias_anual = int(df_base_drawdown_anual["chave_estrategia"].nunique())

linhas_esperadas_resumo = qtd_estrategias_diario * 3

drawdowns_positivos_diario = int((pd.to_numeric(df_base_drawdown_diario["drawdown"], errors="coerce") > 0).sum())
drawdowns_positivos_mensal = int((pd.to_numeric(df_base_drawdown_mensal["drawdown"], errors="coerce") > 0).sum())
drawdowns_positivos_anual = int((pd.to_numeric(df_base_drawdown_anual["drawdown"], errors="coerce") > 0).sum())

drawdowns_ausentes_resumo = int(df_resumo_drawdown_recuperacao_multifrequencia["max_drawdown"].isna().sum())

duplicatas_base_drawdown_diaria = int(df_base_drawdown_diario.duplicated(["chave_estrategia", "data"]).sum())
duplicatas_base_drawdown_mensal = int(df_base_drawdown_mensal.duplicated(["chave_estrategia", "periodo_referencia"]).sum())
duplicatas_base_drawdown_anual = int(df_base_drawdown_anual.duplicated(["chave_estrategia", "periodo_referencia"]).sum())

frequencias_resumo = set(df_resumo_drawdown_recuperacao_multifrequencia["frequencia"].dropna().unique())
frequencias_esperadas = {"diaria", "mensal", "anual"}

auditoria = [
    {
        "item": "estrategias_diarias_vs_mensais",
        "valor": qtd_estrategias_diario,
        "valor_referencia": qtd_estrategias_mensal,
        "status": "OK" if qtd_estrategias_diario == qtd_estrategias_mensal else "ERRO",
        "observacao": "A quantidade de estratégias deve ser igual entre as visões diária e mensal.",
    },
    {
        "item": "estrategias_diarias_vs_anuais",
        "valor": qtd_estrategias_diario,
        "valor_referencia": qtd_estrategias_anual,
        "status": "OK" if qtd_estrategias_diario == qtd_estrategias_anual else "ERRO",
        "observacao": "A quantidade de estratégias deve ser igual entre as visões diária e anual.",
    },
    {
        "item": "linhas_base_drawdown_diaria",
        "valor": len(df_base_drawdown_diario),
        "valor_referencia": len(df_retornos_diarios),
        "status": "OK" if len(df_base_drawdown_diario) == len(df_retornos_diarios) else "ERRO",
        "observacao": "A base de drawdown diária deve preservar o número de linhas da base diária da 11.1.",
    },
    {
        "item": "linhas_base_drawdown_mensal",
        "valor": len(df_base_drawdown_mensal),
        "valor_referencia": len(df_retornos_mensais),
        "status": "OK" if len(df_base_drawdown_mensal) == len(df_retornos_mensais) else "ERRO",
        "observacao": "A base de drawdown mensal deve preservar o número de linhas da tabela mensal da 11.1.",
    },
    {
        "item": "linhas_base_drawdown_anual",
        "valor": len(df_base_drawdown_anual),
        "valor_referencia": len(df_retornos_anuais),
        "status": "OK" if len(df_base_drawdown_anual) == len(df_retornos_anuais) else "ERRO",
        "observacao": "A base de drawdown anual deve preservar o número de linhas da tabela anual da 11.1.",
    },
    {
        "item": "duplicatas_base_drawdown_diaria",
        "valor": duplicatas_base_drawdown_diaria,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_drawdown_diaria == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por chave_estrategia e data na base diária de drawdown.",
    },
    {
        "item": "duplicatas_base_drawdown_mensal",
        "valor": duplicatas_base_drawdown_mensal,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_drawdown_mensal == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por chave_estrategia e período mensal na base mensal de drawdown.",
    },
    {
        "item": "duplicatas_base_drawdown_anual",
        "valor": duplicatas_base_drawdown_anual,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_drawdown_anual == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por chave_estrategia e período anual na base anual de drawdown.",
    },
    {
        "item": "linhas_resumo_multifrequencia",
        "valor": len(df_resumo_drawdown_recuperacao_multifrequencia),
        "valor_referencia": linhas_esperadas_resumo,
        "status": "OK" if len(df_resumo_drawdown_recuperacao_multifrequencia) == linhas_esperadas_resumo else "ERRO",
        "observacao": "O resumo multifrequência deve ter uma linha por estratégia e frequência.",
    },
    {
        "item": "frequencias_resumo_multifrequencia",
        "valor": len(frequencias_resumo),
        "valor_referencia": len(frequencias_esperadas),
        "status": "OK" if frequencias_resumo == frequencias_esperadas else "ERRO",
        "observacao": "O resumo deve conter exatamente as frequências anual, diária e mensal.",
    },
    {
        "item": "drawdowns_positivos_diarios",
        "valor": drawdowns_positivos_diario,
        "valor_referencia": 0,
        "status": "OK" if drawdowns_positivos_diario == 0 else "ERRO",
        "observacao": "Drawdowns devem ser menores ou iguais a zero por definição.",
    },
    {
        "item": "drawdowns_positivos_mensais",
        "valor": drawdowns_positivos_mensal,
        "valor_referencia": 0,
        "status": "OK" if drawdowns_positivos_mensal == 0 else "ERRO",
        "observacao": "Drawdowns devem ser menores ou iguais a zero por definição.",
    },
    {
        "item": "drawdowns_positivos_anuais",
        "valor": drawdowns_positivos_anual,
        "valor_referencia": 0,
        "status": "OK" if drawdowns_positivos_anual == 0 else "ERRO",
        "observacao": "Drawdowns devem ser menores ou iguais a zero por definição.",
    },
    {
        "item": "max_drawdown_ausente_resumo",
        "valor": drawdowns_ausentes_resumo,
        "valor_referencia": 0,
        "status": "OK" if drawdowns_ausentes_resumo == 0 else "ERRO",
        "observacao": "Toda estratégia deve possuir max drawdown calculado em cada frequência.",
    },
]

df_auditoria_validacao_drawdown = pd.DataFrame(auditoria)
erros_bloqueantes = int(df_auditoria_validacao_drawdown["status"].eq("ERRO").sum())

print(f"Itens de auditoria                     : {len(df_auditoria_validacao_drawdown):,}")
print(f"Erros bloqueantes                      : {erros_bloqueantes:,}")

if erros_bloqueantes > 0:
    print("\nAuditoria de validação das métricas de drawdown:")
    print(df_auditoria_validacao_drawdown.to_string(index=False))
    raise ValueError("A auditoria da etapa 11.4 identificou erros bloqueantes.")

print("OK")

# ============================================================
# 10) Salvamento dos outputs da subetapa
# ============================================================

print("\n[10/11] Salvamento dos outputs da subetapa...")

salvar_dataframe(df_base_drawdown_diario, caminho_base_drawdown_diario, index=False)
salvar_dataframe(df_base_drawdown_mensal, caminho_base_drawdown_mensal, index=False)
salvar_dataframe(df_base_drawdown_anual, caminho_base_drawdown_anual, index=False)
salvar_dataframe(df_eventos_drawdown_diario, caminho_tbl_eventos_drawdown_diario, index=False)
salvar_dataframe(df_eventos_drawdown_mensal, caminho_tbl_eventos_drawdown_mensal, index=False)
salvar_dataframe(df_eventos_drawdown_anual, caminho_tbl_eventos_drawdown_anual, index=False)
salvar_dataframe(df_tbl_drawdown_recuperacao_diario, caminho_tbl_drawdown_recuperacao_diario, index=False)
salvar_dataframe(df_tbl_drawdown_recuperacao_mensal, caminho_tbl_drawdown_recuperacao_mensal, index=False)
salvar_dataframe(df_tbl_drawdown_recuperacao_anual, caminho_tbl_drawdown_recuperacao_anual, index=False)
salvar_dataframe(df_resumo_drawdown_recuperacao_multifrequencia, caminho_tbl_resumo_drawdown_recuperacao_multifrequencia, index=False)
salvar_dataframe(df_auditoria_validacao_drawdown, caminho_tbl_auditoria_validacao_drawdown, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 11) Validação final da subetapa
# ============================================================

print("\n[11/11] Validação final da subetapa...")

print("\nAuditoria de validação das métricas de drawdown:")
print(df_auditoria_validacao_drawdown.to_string(index=False))

print("\nResumo multifrequência - amostra:")
print(df_resumo_drawdown_recuperacao_multifrequencia.head(30).to_string(index=False))

print("\nEventos de drawdown diários - amostra:")
if len(df_eventos_drawdown_diario) > 0:
    print(df_eventos_drawdown_diario.head(20).to_string(index=False))
else:
    print("Sem eventos de drawdown diário identificados.")

print("\nBase diária de drawdown - amostra:")
print(df_base_drawdown_diario.head(20).to_string(index=False))

print("\nArquivos salvos na subetapa 11.4:")
print(f"- {caminho_base_drawdown_diario}")
print(f"- {caminho_base_drawdown_mensal}")
print(f"- {caminho_base_drawdown_anual}")
print(f"- {caminho_tbl_eventos_drawdown_diario}")
print(f"- {caminho_tbl_eventos_drawdown_mensal}")
print(f"- {caminho_tbl_eventos_drawdown_anual}")
print(f"- {caminho_tbl_drawdown_recuperacao_diario}")
print(f"- {caminho_tbl_drawdown_recuperacao_mensal}")
print(f"- {caminho_tbl_drawdown_recuperacao_anual}")
print(f"- {caminho_tbl_resumo_drawdown_recuperacao_multifrequencia}")
print(f"- {caminho_tbl_auditoria_validacao_drawdown}")

print("\nETAPA 11.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 11.4 - DRAWDOWN E RECUPERAÇÃO

[1/11] Validação inicial do ambiente...
OK

[2/11] Definição determinística dos caminhos de entrada e saída...
Entrada - retornos diários da 11.1          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_base_retornos_diarios.parquet
Entrada - retornos mensais da 11.1          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_tbl_retornos_mensais.parquet
Entrada - retornos anuais da 11.1           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_tbl_retornos_anuais.parquet
Saída   - base de drawdown diária           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_4_base_drawdown_diario.parque

## Etapa 11.5) Comparação com Ibovespa e CDI

In [56]:
%%time
# ============================================================
# Etapa 11.5) Comparação com Ibovespa e CDI
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 11.5 - COMPARAÇÃO COM IBOVESPA E CDI")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

caminho_base_retornos_diarios = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="base",
    nome="retornos_diarios",
)

caminho_tbl_retornos_mensais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_mensais",
)

caminho_tbl_retornos_anuais = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="retornos_anuais",
)

caminho_tbl_risco_diario = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="risco_diario",
)

caminho_tbl_risco_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="risco_mensal",
)

caminho_tbl_risco_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="risco_anual",
)

caminho_tbl_metricas_eficiencia_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="metricas_eficiencia_multifrequencia",
)

caminho_tbl_drawdown_recuperacao_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="resumo_drawdown_recuperacao_multifrequencia",
)

caminho_base_comparacao_benchmarks_diaria = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="base",
    nome="comparacao_benchmarks_diaria",
)

caminho_base_comparacao_benchmarks_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="base",
    nome="comparacao_benchmarks_mensal",
)

caminho_base_comparacao_benchmarks_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="base",
    nome="comparacao_benchmarks_anual",
)

caminho_tbl_comparacao_ibovespa_cdi_diario = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="comparacao_ibovespa_cdi_diario",
)

caminho_tbl_comparacao_ibovespa_cdi_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="comparacao_ibovespa_cdi_mensal",
)

caminho_tbl_comparacao_ibovespa_cdi_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="comparacao_ibovespa_cdi_anual",
)

caminho_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="resumo_comparacao_ibovespa_cdi_multifrequencia",
)

caminho_tbl_ranking_superacao_benchmarks = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="ranking_superacao_benchmarks",
)

caminho_tbl_auditoria_validacao_comparacao_benchmarks = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_comparacao_benchmarks",
)

print(f"Entrada - retornos diários da 11.1             : {caminho_base_retornos_diarios}")
print(f"Entrada - retornos mensais da 11.1             : {caminho_tbl_retornos_mensais}")
print(f"Entrada - retornos anuais da 11.1              : {caminho_tbl_retornos_anuais}")
print(f"Entrada - risco diário da 11.2                 : {caminho_tbl_risco_diario}")
print(f"Entrada - risco mensal da 11.2                 : {caminho_tbl_risco_mensal}")
print(f"Entrada - risco anual da 11.2                  : {caminho_tbl_risco_anual}")
print(f"Entrada - eficiência multifrequência da 11.3   : {caminho_tbl_metricas_eficiencia_multifrequencia}")
print(f"Entrada - drawdown multifrequência da 11.4     : {caminho_tbl_drawdown_recuperacao_multifrequencia}")
print(f"Saída   - base comparação diária               : {caminho_base_comparacao_benchmarks_diaria}")
print(f"Saída   - base comparação mensal               : {caminho_base_comparacao_benchmarks_mensal}")
print(f"Saída   - base comparação anual                : {caminho_base_comparacao_benchmarks_anual}")
print(f"Saída   - tabela comparação diária             : {caminho_tbl_comparacao_ibovespa_cdi_diario}")
print(f"Saída   - tabela comparação mensal             : {caminho_tbl_comparacao_ibovespa_cdi_mensal}")
print(f"Saída   - tabela comparação anual              : {caminho_tbl_comparacao_ibovespa_cdi_anual}")
print(f"Saída   - resumo multifrequência               : {caminho_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia}")
print(f"Saída   - ranking de superação                 : {caminho_tbl_ranking_superacao_benchmarks}")
print(f"Saída   - auditoria de validação               : {caminho_tbl_auditoria_validacao_comparacao_benchmarks}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais de retorno, risco, eficiência e drawdown
# ============================================================

print("\n[3/12] Carga das bases oficiais de retorno, risco, eficiência e drawdown...")

df_retornos_diarios = pd.read_parquet(caminho_base_retornos_diarios)
df_retornos_mensais = pd.read_parquet(caminho_tbl_retornos_mensais)
df_retornos_anuais = pd.read_parquet(caminho_tbl_retornos_anuais)

df_risco_diario = pd.read_parquet(caminho_tbl_risco_diario)
df_risco_mensal = pd.read_parquet(caminho_tbl_risco_mensal)
df_risco_anual = pd.read_parquet(caminho_tbl_risco_anual)

df_eficiencia_multifrequencia = pd.read_parquet(caminho_tbl_metricas_eficiencia_multifrequencia)
df_drawdown_multifrequencia = pd.read_parquet(caminho_tbl_drawdown_recuperacao_multifrequencia)

print(f"Base de retornos diários                 : {df_retornos_diarios.shape[0]:,} linhas x {df_retornos_diarios.shape[1]} colunas")
print(f"Tabela de retornos mensais               : {df_retornos_mensais.shape[0]:,} linhas x {df_retornos_mensais.shape[1]} colunas")
print(f"Tabela de retornos anuais                : {df_retornos_anuais.shape[0]:,} linhas x {df_retornos_anuais.shape[1]} colunas")
print(f"Tabela de risco diário                   : {df_risco_diario.shape[0]:,} linhas x {df_risco_diario.shape[1]} colunas")
print(f"Tabela de risco mensal                   : {df_risco_mensal.shape[0]:,} linhas x {df_risco_mensal.shape[1]} colunas")
print(f"Tabela de risco anual                    : {df_risco_anual.shape[0]:,} linhas x {df_risco_anual.shape[1]} colunas")
print(f"Tabela de eficiência multifrequência     : {df_eficiencia_multifrequencia.shape[0]:,} linhas x {df_eficiencia_multifrequencia.shape[1]} colunas")
print(f"Tabela de drawdown multifrequência       : {df_drawdown_multifrequencia.shape[0]:,} linhas x {df_drawdown_multifrequencia.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Validação estrutural das bases de entrada
# ============================================================

print("\n[4/12] Validação estrutural das bases de entrada...")

colunas_obrigatorias_diarias = [
    "data",
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "retorno_diario",
    "patrimonio_total",
]

colunas_obrigatorias_mensais = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "ano_mes",
    "data_inicio_periodo",
    "data_fim_periodo",
    "retorno_periodo",
    "patrimonio_fim_observado",
]

colunas_obrigatorias_anuais = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "familia_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "ordem_exibicao",
    "ano",
    "data_inicio_periodo",
    "data_fim_periodo",
    "retorno_periodo",
    "patrimonio_fim_observado",
]

for nome_base, df_base, colunas_obrigatorias in [
    ("retornos_diarios", df_retornos_diarios, colunas_obrigatorias_diarias),
    ("retornos_mensais", df_retornos_mensais, colunas_obrigatorias_mensais),
    ("retornos_anuais", df_retornos_anuais, colunas_obrigatorias_anuais),
]:
    colunas_ausentes = [coluna for coluna in colunas_obrigatorias if coluna not in df_base.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"Colunas ausentes na base {nome_base}: {colunas_ausentes}")

for nome_base, df_base in [
    ("risco_diario", df_risco_diario),
    ("risco_mensal", df_risco_mensal),
    ("risco_anual", df_risco_anual),
    ("eficiencia_multifrequencia", df_eficiencia_multifrequencia),
    ("drawdown_multifrequencia", df_drawdown_multifrequencia),
]:
    colunas_ausentes = [coluna for coluna in ["chave_estrategia", "frequencia"] if coluna not in df_base.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"Colunas ausentes na base {nome_base}: {colunas_ausentes}")

df_retornos_diarios = df_retornos_diarios.copy()
df_retornos_mensais = df_retornos_mensais.copy()
df_retornos_anuais = df_retornos_anuais.copy()

df_retornos_diarios["data"] = pd.to_datetime(df_retornos_diarios["data"], errors="coerce")
df_retornos_mensais["data_inicio_periodo"] = pd.to_datetime(df_retornos_mensais["data_inicio_periodo"], errors="coerce")
df_retornos_mensais["data_fim_periodo"] = pd.to_datetime(df_retornos_mensais["data_fim_periodo"], errors="coerce")
df_retornos_anuais["data_inicio_periodo"] = pd.to_datetime(df_retornos_anuais["data_inicio_periodo"], errors="coerce")
df_retornos_anuais["data_fim_periodo"] = pd.to_datetime(df_retornos_anuais["data_fim_periodo"], errors="coerce")

duplicatas_diarias = int(df_retornos_diarios.duplicated(["chave_estrategia", "data"]).sum())
duplicatas_mensais = int(df_retornos_mensais.duplicated(["chave_estrategia", "ano_mes"]).sum())
duplicatas_anuais = int(df_retornos_anuais.duplicated(["chave_estrategia", "ano"]).sum())

datas_invalidas_diarias = int(df_retornos_diarios["data"].isna().sum())
datas_invalidas_mensais = int(df_retornos_mensais[["data_inicio_periodo", "data_fim_periodo"]].isna().any(axis=1).sum())
datas_invalidas_anuais = int(df_retornos_anuais[["data_inicio_periodo", "data_fim_periodo"]].isna().any(axis=1).sum())

if duplicatas_diarias > 0:
    raise ValueError("Há duplicidades por chave_estrategia e data na base diária da 11.1.")
if duplicatas_mensais > 0:
    raise ValueError("Há duplicidades por chave_estrategia e ano_mes na tabela mensal da 11.1.")
if duplicatas_anuais > 0:
    raise ValueError("Há duplicidades por chave_estrategia e ano na tabela anual da 11.1.")
if datas_invalidas_diarias > 0 or datas_invalidas_mensais > 0 or datas_invalidas_anuais > 0:
    raise ValueError("Há datas inválidas em ao menos uma base de retornos da 11.1.")

print(f"Duplicidades diárias                     : {duplicatas_diarias:,}")
print(f"Duplicidades mensais                     : {duplicatas_mensais:,}")
print(f"Duplicidades anuais                      : {duplicatas_anuais:,}")
print(f"Datas inválidas diárias                  : {datas_invalidas_diarias:,}")
print(f"Datas inválidas mensais                  : {datas_invalidas_mensais:,}")
print(f"Datas inválidas anuais                   : {datas_invalidas_anuais:,}")
print("OK")

# ============================================================
# 5) Identificação determinística dos benchmarks oficiais
# ============================================================

print("\n[5/12] Identificação determinística dos benchmarks Ibovespa e CDI...")

chave_benchmark_ibovespa = "ibovespa_buy_and_hold__carteira_na__replica_na"
chave_benchmark_cdi = "cdi_only__carteira_na__replica_na"


def validar_chave_benchmark_exata(df_base, chave_benchmark, rotulo_benchmark, termo_diagnostico):
    chaves_disponiveis = set(df_base["chave_estrategia"].dropna().astype(str).unique().tolist())

    if chave_benchmark not in chaves_disponiveis:
        chaves_relacionadas = sorted(
            [
                chave
                for chave in chaves_disponiveis
                if termo_diagnostico.lower() in chave.lower()
            ]
        )

        raise ValueError(
            f"A chave oficial do benchmark {rotulo_benchmark} não foi encontrada na base diária da 11.1. "
            f"Chave esperada: {chave_benchmark}. "
            f"Chaves relacionadas disponíveis para diagnóstico: {chaves_relacionadas}"
        )

    return chave_benchmark


chave_benchmark_ibovespa = validar_chave_benchmark_exata(
    df_base=df_retornos_diarios,
    chave_benchmark=chave_benchmark_ibovespa,
    rotulo_benchmark="Ibovespa Buy and Hold",
    termo_diagnostico="ibovespa",
)

chave_benchmark_cdi = validar_chave_benchmark_exata(
    df_base=df_retornos_diarios,
    chave_benchmark=chave_benchmark_cdi,
    rotulo_benchmark="CDI-only",
    termo_diagnostico="cdi",
)

nome_benchmark_ibovespa = df_retornos_diarios.loc[
    df_retornos_diarios["chave_estrategia"].astype(str).eq(chave_benchmark_ibovespa),
    "nome_exibicao_estrategia",
].dropna().astype(str).iloc[0]

nome_benchmark_cdi = df_retornos_diarios.loc[
    df_retornos_diarios["chave_estrategia"].astype(str).eq(chave_benchmark_cdi),
    "nome_exibicao_estrategia",
].dropna().astype(str).iloc[0]

print(f"Benchmark Ibovespa oficial - chave      : {chave_benchmark_ibovespa}")
print(f"Benchmark Ibovespa oficial - nome       : {nome_benchmark_ibovespa}")
print(f"Benchmark CDI-only - chave              : {chave_benchmark_cdi}")
print(f"Benchmark CDI-only - nome               : {nome_benchmark_cdi}")
print("OK")

# ============================================================
# 6) Padronização das bases por frequência para comparação
# ============================================================

print("\n[6/12] Padronização das bases por frequência para comparação...")

colunas_identificacao_preferenciais = [
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "origem_backtest",
    "ordem_exibicao",
]


def padronizar_periodos_comparacao(df_base, frequencia):
    df = df_base.copy()

    if frequencia == "diaria":
        df["periodo_comparacao"] = pd.to_datetime(df["data"], errors="coerce")
        df["data_inicio_periodo"] = df["periodo_comparacao"]
        df["data_fim_periodo"] = df["periodo_comparacao"]
        df["retorno_periodo"] = pd.to_numeric(df["retorno_diario"], errors="coerce")
        df["patrimonio_fim_observado"] = pd.to_numeric(df["patrimonio_total"], errors="coerce")
    elif frequencia == "mensal":
        df["periodo_comparacao"] = df["ano_mes"].astype(str)
        df["data_inicio_periodo"] = pd.to_datetime(df["data_inicio_periodo"], errors="coerce")
        df["data_fim_periodo"] = pd.to_datetime(df["data_fim_periodo"], errors="coerce")
        df["retorno_periodo"] = pd.to_numeric(df["retorno_periodo"], errors="coerce")
        df["patrimonio_fim_observado"] = pd.to_numeric(df["patrimonio_fim_observado"], errors="coerce")
    elif frequencia == "anual":
        df["periodo_comparacao"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64").astype(str)
        df["data_inicio_periodo"] = pd.to_datetime(df["data_inicio_periodo"], errors="coerce")
        df["data_fim_periodo"] = pd.to_datetime(df["data_fim_periodo"], errors="coerce")
        df["retorno_periodo"] = pd.to_numeric(df["retorno_periodo"], errors="coerce")
        df["patrimonio_fim_observado"] = pd.to_numeric(df["patrimonio_fim_observado"], errors="coerce")
    else:
        raise ValueError(f"Frequência não reconhecida: {frequencia}")

    df["frequencia"] = frequencia

    colunas_identificacao = [coluna for coluna in colunas_identificacao_preferenciais if coluna in df.columns]
    colunas_saida = colunas_identificacao + [
        "frequencia",
        "periodo_comparacao",
        "data_inicio_periodo",
        "data_fim_periodo",
        "retorno_periodo",
        "patrimonio_fim_observado",
    ]

    return df[colunas_saida].copy()


df_periodos_diarios = padronizar_periodos_comparacao(df_retornos_diarios, "diaria")
df_periodos_mensais = padronizar_periodos_comparacao(df_retornos_mensais, "mensal")
df_periodos_anuais = padronizar_periodos_comparacao(df_retornos_anuais, "anual")

print(f"Períodos diários padronizados            : {df_periodos_diarios.shape[0]:,} linhas")
print(f"Períodos mensais padronizados            : {df_periodos_mensais.shape[0]:,} linhas")
print(f"Períodos anuais padronizados             : {df_periodos_anuais.shape[0]:,} linhas")
print("OK")

# ============================================================
# 7) Construção das bases de comparação período a período
# ============================================================

print("\n[7/12] Construção das bases de comparação período a período...")


def construir_base_comparacao_periodica(df_periodos, frequencia):
    df = df_periodos.copy()

    df_benchmark_ibovespa = (
        df.loc[df["chave_estrategia"].astype(str).eq(chave_benchmark_ibovespa), ["periodo_comparacao", "retorno_periodo", "patrimonio_fim_observado"]]
        .rename(
            columns={
                "retorno_periodo": "retorno_periodo_ibovespa",
                "patrimonio_fim_observado": "patrimonio_fim_observado_ibovespa",
            }
        )
        .drop_duplicates("periodo_comparacao")
    )

    df_benchmark_cdi = (
        df.loc[df["chave_estrategia"].astype(str).eq(chave_benchmark_cdi), ["periodo_comparacao", "retorno_periodo", "patrimonio_fim_observado"]]
        .rename(
            columns={
                "retorno_periodo": "retorno_periodo_cdi",
                "patrimonio_fim_observado": "patrimonio_fim_observado_cdi",
            }
        )
        .drop_duplicates("periodo_comparacao")
    )

    linhas_antes = len(df)

    df = df.merge(df_benchmark_ibovespa, on="periodo_comparacao", how="left")
    df = df.merge(df_benchmark_cdi, on="periodo_comparacao", how="left")

    linhas_depois = len(df)
    if linhas_depois != linhas_antes:
        raise ValueError(f"A base de comparação {frequencia} expandiu linhas após o merge dos benchmarks.")

    df["retorno_excesso_vs_ibovespa"] = df["retorno_periodo"] - df["retorno_periodo_ibovespa"]
    df["retorno_excesso_vs_cdi"] = df["retorno_periodo"] - df["retorno_periodo_cdi"]

    df["retorno_relativo_composto_vs_ibovespa"] = (1.0 + df["retorno_periodo"]) / (1.0 + df["retorno_periodo_ibovespa"]) - 1.0
    df["retorno_relativo_composto_vs_cdi"] = (1.0 + df["retorno_periodo"]) / (1.0 + df["retorno_periodo_cdi"]) - 1.0

    df["flag_superou_ibovespa"] = df["retorno_excesso_vs_ibovespa"] > 0.0
    df["flag_superou_cdi"] = df["retorno_excesso_vs_cdi"] > 0.0
    df["flag_empatou_ibovespa"] = np.isclose(df["retorno_excesso_vs_ibovespa"].fillna(np.nan), 0.0, atol=0.0000000001, rtol=0.0)
    df["flag_empatou_cdi"] = np.isclose(df["retorno_excesso_vs_cdi"].fillna(np.nan), 0.0, atol=0.0000000001, rtol=0.0)
    df["flag_perdeu_ibovespa"] = df["retorno_excesso_vs_ibovespa"] < 0.0
    df["flag_perdeu_cdi"] = df["retorno_excesso_vs_cdi"] < 0.0

    df["benchmark_ibovespa_chave"] = chave_benchmark_ibovespa
    df["benchmark_ibovespa_nome"] = nome_benchmark_ibovespa
    df["benchmark_cdi_chave"] = chave_benchmark_cdi
    df["benchmark_cdi_nome"] = nome_benchmark_cdi

    df = (
        df
        .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", "periodo_comparacao"])
        .reset_index(drop=True)
    )

    return df


df_base_comparacao_diaria = construir_base_comparacao_periodica(df_periodos_diarios, "diaria")
df_base_comparacao_mensal = construir_base_comparacao_periodica(df_periodos_mensais, "mensal")
df_base_comparacao_anual = construir_base_comparacao_periodica(df_periodos_anuais, "anual")

print(f"Linhas da base de comparação diária      : {len(df_base_comparacao_diaria):,}")
print(f"Linhas da base de comparação mensal      : {len(df_base_comparacao_mensal):,}")
print(f"Linhas da base de comparação anual       : {len(df_base_comparacao_anual):,}")
print(f"Retornos Ibovespa ausentes - diária      : {int(df_base_comparacao_diaria['retorno_periodo_ibovespa'].isna().sum()):,}")
print(f"Retornos CDI ausentes - diária           : {int(df_base_comparacao_diaria['retorno_periodo_cdi'].isna().sum()):,}")
print(f"Retornos Ibovespa ausentes - mensal      : {int(df_base_comparacao_mensal['retorno_periodo_ibovespa'].isna().sum()):,}")
print(f"Retornos CDI ausentes - mensal           : {int(df_base_comparacao_mensal['retorno_periodo_cdi'].isna().sum()):,}")
print(f"Retornos Ibovespa ausentes - anual       : {int(df_base_comparacao_anual['retorno_periodo_ibovespa'].isna().sum()):,}")
print(f"Retornos CDI ausentes - anual            : {int(df_base_comparacao_anual['retorno_periodo_cdi'].isna().sum()):,}")
print("OK")

# ============================================================
# 8) Construção das métricas complementares de risco e eficiência
# ============================================================

print("\n[8/12] Construção das métricas complementares de risco e eficiência...")


def selecionar_colunas_existentes(df_base, colunas):
    return [coluna for coluna in colunas if coluna in df_base.columns]


def montar_metricas_complementares(frequencia, df_risco):
    df_risco_freq = df_risco.copy()
    if "frequencia" in df_risco_freq.columns:
        df_risco_freq = df_risco_freq.loc[df_risco_freq["frequencia"].astype(str).eq(frequencia)].copy()

    df_eficiencia_freq = df_eficiencia_multifrequencia.loc[
        df_eficiencia_multifrequencia["frequencia"].astype(str).eq(frequencia)
    ].copy()

    df_drawdown_freq = df_drawdown_multifrequencia.loc[
        df_drawdown_multifrequencia["frequencia"].astype(str).eq(frequencia)
    ].copy()

    colunas_risco = selecionar_colunas_existentes(
        df_risco_freq,
        [
            "chave_estrategia",
            "volatilidade_periodo",
            "volatilidade_anualizada_equivalente",
            "downside_deviation_periodo",
            "downside_deviation_anualizada_equivalente",
            "var_historico_5pct",
            "var_historico_1pct",
            "cvar_historico_5pct",
            "cvar_historico_1pct",
        ],
    )

    colunas_eficiencia = selecionar_colunas_existentes(
        df_eficiencia_freq,
        [
            "chave_estrategia",
            "sharpe_anualizado",
            "sortino_anualizado",
            "calmar",
        ],
    )

    colunas_drawdown = selecionar_colunas_existentes(
        df_drawdown_freq,
        [
            "chave_estrategia",
            "max_drawdown",
            "max_drawdown_abs",
            "pct_periodos_em_drawdown",
            "periodos_pico_a_recuperacao_max_drawdown",
            "n_eventos_drawdown",
            "n_eventos_recuperados",
            "pct_eventos_recuperados",
        ],
    )

    df_metricas = df_risco_freq[colunas_risco].drop_duplicates("chave_estrategia") if len(colunas_risco) > 1 else pd.DataFrame({"chave_estrategia": []})

    if len(colunas_eficiencia) > 1:
        df_metricas = df_metricas.merge(
            df_eficiencia_freq[colunas_eficiencia].drop_duplicates("chave_estrategia"),
            on="chave_estrategia",
            how="outer",
        )

    if len(colunas_drawdown) > 1:
        df_metricas = df_metricas.merge(
            df_drawdown_freq[colunas_drawdown].drop_duplicates("chave_estrategia"),
            on="chave_estrategia",
            how="outer",
        )

    df_metricas["frequencia"] = frequencia
    return df_metricas


df_metricas_complementares_diarias = montar_metricas_complementares("diaria", df_risco_diario)
df_metricas_complementares_mensais = montar_metricas_complementares("mensal", df_risco_mensal)
df_metricas_complementares_anuais = montar_metricas_complementares("anual", df_risco_anual)

print(f"Métricas complementares diárias          : {len(df_metricas_complementares_diarias):,} linhas")
print(f"Métricas complementares mensais          : {len(df_metricas_complementares_mensais):,} linhas")
print(f"Métricas complementares anuais           : {len(df_metricas_complementares_anuais):,} linhas")
print("OK")

# ============================================================
# 9) Consolidação das tabelas comparativas por estratégia
# ============================================================

print("\n[9/12] Consolidação das tabelas comparativas por estratégia...")


def produto_retorno_acumulado(serie_retornos):
    retornos = pd.to_numeric(serie_retornos, errors="coerce").dropna()
    if len(retornos) == 0:
        return np.nan
    return float((1.0 + retornos).prod() - 1.0)


def calcular_retorno_anualizado(retorno_acumulado, n_anos):
    if pd.isna(retorno_acumulado) or pd.isna(n_anos) or n_anos <= 0:
        return np.nan
    if (1.0 + retorno_acumulado) <= 0:
        return np.nan
    return float((1.0 + retorno_acumulado) ** (1.0 / n_anos) - 1.0)


def extrair_primeiro_valor(grupo, coluna):
    if coluna not in grupo.columns:
        return np.nan
    serie = grupo[coluna].dropna()
    if len(serie) == 0:
        return np.nan
    return serie.iloc[0]


def montar_resumo_comparacao(df_comparacao, df_metricas_complementares, frequencia):
    registros = []

    for chave, grupo_original in df_comparacao.groupby("chave_estrategia", dropna=False):
        grupo = grupo_original.sort_values("data_fim_periodo").copy()
        primeiro = grupo.iloc[0]
        ultimo = grupo.iloc[-1]

        grupo_ibovespa = grupo.dropna(subset=["retorno_periodo", "retorno_periodo_ibovespa"])
        grupo_cdi = grupo.dropna(subset=["retorno_periodo", "retorno_periodo_cdi"])

        n_periodos = int(len(grupo))
        n_periodos_ibovespa = int(len(grupo_ibovespa))
        n_periodos_cdi = int(len(grupo_cdi))

        data_inicio = pd.to_datetime(grupo["data_inicio_periodo"].min(), errors="coerce")
        data_fim = pd.to_datetime(grupo["data_fim_periodo"].max(), errors="coerce")
        n_dias_corridos = int((data_fim - data_inicio).days) if pd.notna(data_inicio) and pd.notna(data_fim) else np.nan
        n_anos = float(n_dias_corridos / 365.25) if pd.notna(n_dias_corridos) and n_dias_corridos > 0 else np.nan

        retorno_acumulado_estrategia = produto_retorno_acumulado(grupo["retorno_periodo"])
        retorno_acumulado_ibovespa = produto_retorno_acumulado(grupo_ibovespa["retorno_periodo_ibovespa"])
        retorno_acumulado_cdi = produto_retorno_acumulado(grupo_cdi["retorno_periodo_cdi"])

        retorno_anualizado_estrategia = calcular_retorno_anualizado(retorno_acumulado_estrategia, n_anos)
        retorno_anualizado_ibovespa = calcular_retorno_anualizado(retorno_acumulado_ibovespa, n_anos)
        retorno_anualizado_cdi = calcular_retorno_anualizado(retorno_acumulado_cdi, n_anos)

        excesso_acumulado_composto_vs_ibovespa = (
            float((1.0 + retorno_acumulado_estrategia) / (1.0 + retorno_acumulado_ibovespa) - 1.0)
            if pd.notna(retorno_acumulado_estrategia) and pd.notna(retorno_acumulado_ibovespa) and (1.0 + retorno_acumulado_ibovespa) != 0
            else np.nan
        )
        excesso_acumulado_composto_vs_cdi = (
            float((1.0 + retorno_acumulado_estrategia) / (1.0 + retorno_acumulado_cdi) - 1.0)
            if pd.notna(retorno_acumulado_estrategia) and pd.notna(retorno_acumulado_cdi) and (1.0 + retorno_acumulado_cdi) != 0
            else np.nan
        )

        excesso_anualizado_vs_ibovespa = retorno_anualizado_estrategia - retorno_anualizado_ibovespa if pd.notna(retorno_anualizado_estrategia) and pd.notna(retorno_anualizado_ibovespa) else np.nan
        excesso_anualizado_vs_cdi = retorno_anualizado_estrategia - retorno_anualizado_cdi if pd.notna(retorno_anualizado_estrategia) and pd.notna(retorno_anualizado_cdi) else np.nan

        n_superou_ibovespa = int(grupo_ibovespa["flag_superou_ibovespa"].fillna(False).sum())
        n_superou_cdi = int(grupo_cdi["flag_superou_cdi"].fillna(False).sum())
        n_empatou_ibovespa = int(grupo_ibovespa["flag_empatou_ibovespa"].fillna(False).sum())
        n_empatou_cdi = int(grupo_cdi["flag_empatou_cdi"].fillna(False).sum())
        n_perdeu_ibovespa = int(grupo_ibovespa["flag_perdeu_ibovespa"].fillna(False).sum())
        n_perdeu_cdi = int(grupo_cdi["flag_perdeu_cdi"].fillna(False).sum())

        pct_superou_ibovespa = float(n_superou_ibovespa / n_periodos_ibovespa) if n_periodos_ibovespa > 0 else np.nan
        pct_superou_cdi = float(n_superou_cdi / n_periodos_cdi) if n_periodos_cdi > 0 else np.nan
        pct_empatou_ibovespa = float(n_empatou_ibovespa / n_periodos_ibovespa) if n_periodos_ibovespa > 0 else np.nan
        pct_empatou_cdi = float(n_empatou_cdi / n_periodos_cdi) if n_periodos_cdi > 0 else np.nan
        pct_perdeu_ibovespa = float(n_perdeu_ibovespa / n_periodos_ibovespa) if n_periodos_ibovespa > 0 else np.nan
        pct_perdeu_cdi = float(n_perdeu_cdi / n_periodos_cdi) if n_periodos_cdi > 0 else np.nan

        registro = {
            "chave_estrategia": chave,
            "estrategia_id": extrair_primeiro_valor(grupo, "estrategia_id"),
            "grupo_controle": extrair_primeiro_valor(grupo, "grupo_controle"),
            "familia_estrategia": extrair_primeiro_valor(grupo, "familia_estrategia"),
            "estrategia_referencia": extrair_primeiro_valor(grupo, "estrategia_referencia"),
            "estrategia_referencia_padronizada": extrair_primeiro_valor(grupo, "estrategia_referencia_padronizada"),
            "tipo_estrategia": extrair_primeiro_valor(grupo, "tipo_estrategia"),
            "carteira_id": extrair_primeiro_valor(grupo, "carteira_id"),
            "replica_id": extrair_primeiro_valor(grupo, "replica_id"),
            "nome_exibicao_estrategia": extrair_primeiro_valor(grupo, "nome_exibicao_estrategia"),
            "categoria_estrategia": extrair_primeiro_valor(grupo, "categoria_estrategia"),
            "subcategoria_estrategia": extrair_primeiro_valor(grupo, "subcategoria_estrategia"),
            "origem_backtest": extrair_primeiro_valor(grupo, "origem_backtest"),
            "ordem_exibicao": extrair_primeiro_valor(grupo, "ordem_exibicao"),
            "frequencia": frequencia,
            "benchmark_ibovespa_chave": chave_benchmark_ibovespa,
            "benchmark_ibovespa_nome": nome_benchmark_ibovespa,
            "benchmark_cdi_chave": chave_benchmark_cdi,
            "benchmark_cdi_nome": nome_benchmark_cdi,
            "n_periodos": n_periodos,
            "n_periodos_comparaveis_ibovespa": n_periodos_ibovespa,
            "n_periodos_comparaveis_cdi": n_periodos_cdi,
            "data_inicio": data_inicio,
            "data_fim": data_fim,
            "n_dias_corridos": n_dias_corridos,
            "n_anos": n_anos,
            "patrimonio_inicio_observado": extrair_primeiro_valor(grupo, "patrimonio_fim_observado"),
            "patrimonio_fim_observado": extrair_primeiro_valor(grupo.iloc[[-1]], "patrimonio_fim_observado"),
            "retorno_acumulado_estrategia": retorno_acumulado_estrategia,
            "retorno_acumulado_ibovespa": retorno_acumulado_ibovespa,
            "retorno_acumulado_cdi": retorno_acumulado_cdi,
            "excesso_acumulado_composto_vs_ibovespa": excesso_acumulado_composto_vs_ibovespa,
            "excesso_acumulado_composto_vs_cdi": excesso_acumulado_composto_vs_cdi,
            "retorno_anualizado_estrategia": retorno_anualizado_estrategia,
            "retorno_anualizado_ibovespa": retorno_anualizado_ibovespa,
            "retorno_anualizado_cdi": retorno_anualizado_cdi,
            "excesso_anualizado_vs_ibovespa": excesso_anualizado_vs_ibovespa,
            "excesso_anualizado_vs_cdi": excesso_anualizado_vs_cdi,
            "n_vezes_superou_ibovespa": n_superou_ibovespa,
            "n_vezes_superou_cdi": n_superou_cdi,
            "n_vezes_empatou_ibovespa": n_empatou_ibovespa,
            "n_vezes_empatou_cdi": n_empatou_cdi,
            "n_vezes_perdeu_ibovespa": n_perdeu_ibovespa,
            "n_vezes_perdeu_cdi": n_perdeu_cdi,
            "pct_periodos_superou_ibovespa": pct_superou_ibovespa,
            "pct_periodos_superou_cdi": pct_superou_cdi,
            "pct_periodos_empatou_ibovespa": pct_empatou_ibovespa,
            "pct_periodos_empatou_cdi": pct_empatou_cdi,
            "pct_periodos_perdeu_ibovespa": pct_perdeu_ibovespa,
            "pct_periodos_perdeu_cdi": pct_perdeu_cdi,
            "excesso_medio_periodo_vs_ibovespa": float(grupo_ibovespa["retorno_excesso_vs_ibovespa"].mean()) if n_periodos_ibovespa > 0 else np.nan,
            "excesso_medio_periodo_vs_cdi": float(grupo_cdi["retorno_excesso_vs_cdi"].mean()) if n_periodos_cdi > 0 else np.nan,
            "excesso_mediano_periodo_vs_ibovespa": float(grupo_ibovespa["retorno_excesso_vs_ibovespa"].median()) if n_periodos_ibovespa > 0 else np.nan,
            "excesso_mediano_periodo_vs_cdi": float(grupo_cdi["retorno_excesso_vs_cdi"].median()) if n_periodos_cdi > 0 else np.nan,
            "melhor_excesso_periodo_vs_ibovespa": float(grupo_ibovespa["retorno_excesso_vs_ibovespa"].max()) if n_periodos_ibovespa > 0 else np.nan,
            "melhor_excesso_periodo_vs_cdi": float(grupo_cdi["retorno_excesso_vs_cdi"].max()) if n_periodos_cdi > 0 else np.nan,
            "pior_excesso_periodo_vs_ibovespa": float(grupo_ibovespa["retorno_excesso_vs_ibovespa"].min()) if n_periodos_ibovespa > 0 else np.nan,
            "pior_excesso_periodo_vs_cdi": float(grupo_cdi["retorno_excesso_vs_cdi"].min()) if n_periodos_cdi > 0 else np.nan,
        }

        registros.append(registro)

    df_resumo = pd.DataFrame(registros)

    df_resumo = df_resumo.merge(
        df_metricas_complementares.drop(columns=["frequencia"], errors="ignore"),
        on="chave_estrategia",
        how="left",
    )

    metricas_para_diferenca = [
        "volatilidade_periodo",
        "volatilidade_anualizada_equivalente",
        "downside_deviation_periodo",
        "downside_deviation_anualizada_equivalente",
        "var_historico_5pct",
        "cvar_historico_5pct",
        "sharpe_anualizado",
        "sortino_anualizado",
        "calmar",
        "max_drawdown",
        "max_drawdown_abs",
        "pct_periodos_em_drawdown",
        "periodos_pico_a_recuperacao_max_drawdown",
    ]

    metricas_para_diferenca = [coluna for coluna in metricas_para_diferenca if coluna in df_resumo.columns]

    for metrica in metricas_para_diferenca:
        serie_ibovespa = df_resumo.loc[df_resumo["chave_estrategia"].astype(str).eq(chave_benchmark_ibovespa), metrica].dropna()
        serie_cdi = df_resumo.loc[df_resumo["chave_estrategia"].astype(str).eq(chave_benchmark_cdi), metrica].dropna()

        valor_ibovespa = serie_ibovespa.iloc[0] if len(serie_ibovespa) > 0 else np.nan
        valor_cdi = serie_cdi.iloc[0] if len(serie_cdi) > 0 else np.nan

        df_resumo[f"{metrica}_ibovespa"] = valor_ibovespa
        df_resumo[f"{metrica}_cdi"] = valor_cdi
        df_resumo[f"dif_{metrica}_vs_ibovespa"] = df_resumo[metrica] - valor_ibovespa
        df_resumo[f"dif_{metrica}_vs_cdi"] = df_resumo[metrica] - valor_cdi

    df_resumo["flag_retorno_acumulado_superou_ibovespa"] = df_resumo["excesso_acumulado_composto_vs_ibovespa"] > 0.0
    df_resumo["flag_retorno_acumulado_superou_cdi"] = df_resumo["excesso_acumulado_composto_vs_cdi"] > 0.0
    df_resumo["flag_retorno_anualizado_superou_ibovespa"] = df_resumo["excesso_anualizado_vs_ibovespa"] > 0.0
    df_resumo["flag_retorno_anualizado_superou_cdi"] = df_resumo["excesso_anualizado_vs_cdi"] > 0.0

    df_resumo = (
        df_resumo
        .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia"])
        .reset_index(drop=True)
    )

    return df_resumo


df_tbl_comparacao_ibovespa_cdi_diario = montar_resumo_comparacao(
    df_base_comparacao_diaria,
    df_metricas_complementares_diarias,
    "diaria",
)

df_tbl_comparacao_ibovespa_cdi_mensal = montar_resumo_comparacao(
    df_base_comparacao_mensal,
    df_metricas_complementares_mensais,
    "mensal",
)

df_tbl_comparacao_ibovespa_cdi_anual = montar_resumo_comparacao(
    df_base_comparacao_anual,
    df_metricas_complementares_anuais,
    "anual",
)

print(f"Linhas da tabela comparativa diária      : {len(df_tbl_comparacao_ibovespa_cdi_diario):,}")
print(f"Linhas da tabela comparativa mensal      : {len(df_tbl_comparacao_ibovespa_cdi_mensal):,}")
print(f"Linhas da tabela comparativa anual       : {len(df_tbl_comparacao_ibovespa_cdi_anual):,}")
print("OK")

# ============================================================
# 10) Consolidação do resumo multifrequência e ranking de superação
# ============================================================

print("\n[10/12] Consolidação do resumo multifrequência e ranking de superação...")

df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia = pd.concat(
    [
        df_tbl_comparacao_ibovespa_cdi_diario,
        df_tbl_comparacao_ibovespa_cdi_mensal,
        df_tbl_comparacao_ibovespa_cdi_anual,
    ],
    ignore_index=True,
)

df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia = (
    df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia
    .sort_values(["ordem_exibicao", "nome_exibicao_estrategia", "chave_estrategia", "frequencia"])
    .reset_index(drop=True)
)

colunas_ranking = [
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "origem_backtest",
    "ordem_exibicao",
    "frequencia",
    "n_periodos_comparaveis_ibovespa",
    "n_periodos_comparaveis_cdi",
    "n_vezes_superou_ibovespa",
    "n_vezes_superou_cdi",
    "pct_periodos_superou_ibovespa",
    "pct_periodos_superou_cdi",
    "excesso_acumulado_composto_vs_ibovespa",
    "excesso_acumulado_composto_vs_cdi",
    "excesso_anualizado_vs_ibovespa",
    "excesso_anualizado_vs_cdi",
    "retorno_anualizado_estrategia",
    "retorno_anualizado_ibovespa",
    "retorno_anualizado_cdi",
    "flag_retorno_acumulado_superou_ibovespa",
    "flag_retorno_acumulado_superou_cdi",
]

colunas_ranking = [coluna for coluna in colunas_ranking if coluna in df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia.columns]

df_tbl_ranking_superacao_benchmarks = df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia[colunas_ranking].copy()

df_tbl_ranking_superacao_benchmarks["ranking_pct_superacao_ibovespa"] = (
    df_tbl_ranking_superacao_benchmarks
    .groupby("frequencia")["pct_periodos_superou_ibovespa"]
    .rank(ascending=False, method="min")
)

df_tbl_ranking_superacao_benchmarks["ranking_pct_superacao_cdi"] = (
    df_tbl_ranking_superacao_benchmarks
    .groupby("frequencia")["pct_periodos_superou_cdi"]
    .rank(ascending=False, method="min")
)

df_tbl_ranking_superacao_benchmarks["ranking_excesso_acumulado_vs_ibovespa"] = (
    df_tbl_ranking_superacao_benchmarks
    .groupby("frequencia")["excesso_acumulado_composto_vs_ibovespa"]
    .rank(ascending=False, method="min")
)

df_tbl_ranking_superacao_benchmarks["ranking_excesso_acumulado_vs_cdi"] = (
    df_tbl_ranking_superacao_benchmarks
    .groupby("frequencia")["excesso_acumulado_composto_vs_cdi"]
    .rank(ascending=False, method="min")
)

print(f"Linhas do resumo multifrequência         : {len(df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia):,}")
print(f"Frequências no resumo                    : {', '.join(sorted(df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia['frequencia'].dropna().unique()))}")
print(f"Linhas do ranking de superação           : {len(df_tbl_ranking_superacao_benchmarks):,}")
print("OK")

# ============================================================
# 11) Construção da auditoria de validação
# ============================================================

print("\n[11/12] Construção da auditoria de validação...")

n_estrategias_diarias = int(df_base_comparacao_diaria["chave_estrategia"].nunique())
n_estrategias_mensais = int(df_base_comparacao_mensal["chave_estrategia"].nunique())
n_estrategias_anuais = int(df_base_comparacao_anual["chave_estrategia"].nunique())

linhas_esperadas_resumo = n_estrategias_diarias * 3

retornos_benchmark_ausentes = int(
    df_base_comparacao_diaria[["retorno_periodo_ibovespa", "retorno_periodo_cdi"]].isna().sum().sum()
    + df_base_comparacao_mensal[["retorno_periodo_ibovespa", "retorno_periodo_cdi"]].isna().sum().sum()
    + df_base_comparacao_anual[["retorno_periodo_ibovespa", "retorno_periodo_cdi"]].isna().sum().sum()
)

duplicatas_base_diaria = int(df_base_comparacao_diaria.duplicated(["chave_estrategia", "periodo_comparacao"]).sum())
duplicatas_base_mensal = int(df_base_comparacao_mensal.duplicated(["chave_estrategia", "periodo_comparacao"]).sum())
duplicatas_base_anual = int(df_base_comparacao_anual.duplicated(["chave_estrategia", "periodo_comparacao"]).sum())

metricas_infinitas = int(
    np.isinf(
        df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia
        .select_dtypes(include=[np.number])
        .to_numpy()
    ).sum()
)

itens_auditoria = [
    {
        "item": "benchmark_ibovespa_identificado",
        "valor": 1 if isinstance(chave_benchmark_ibovespa, str) and len(chave_benchmark_ibovespa) > 0 else 0,
        "valor_referencia": 1,
        "status": "OK" if isinstance(chave_benchmark_ibovespa, str) and len(chave_benchmark_ibovespa) > 0 else "ERRO",
        "observacao": "A comparação exige uma chave única para o benchmark Ibovespa.",
    },
    {
        "item": "benchmark_cdi_identificado",
        "valor": 1 if isinstance(chave_benchmark_cdi, str) and len(chave_benchmark_cdi) > 0 else 0,
        "valor_referencia": 1,
        "status": "OK" if isinstance(chave_benchmark_cdi, str) and len(chave_benchmark_cdi) > 0 else "ERRO",
        "observacao": "A comparação exige uma chave única para a carteira CDI-only.",
    },
    {
        "item": "estrategias_diarias_vs_mensais",
        "valor": n_estrategias_diarias,
        "valor_referencia": n_estrategias_mensais,
        "status": "OK" if n_estrategias_diarias == n_estrategias_mensais else "ERRO",
        "observacao": "A quantidade de estratégias deve ser igual entre as visões diária e mensal.",
    },
    {
        "item": "estrategias_diarias_vs_anuais",
        "valor": n_estrategias_diarias,
        "valor_referencia": n_estrategias_anuais,
        "status": "OK" if n_estrategias_diarias == n_estrategias_anuais else "ERRO",
        "observacao": "A quantidade de estratégias deve ser igual entre as visões diária e anual.",
    },
    {
        "item": "linhas_base_comparacao_diaria",
        "valor": int(len(df_base_comparacao_diaria)),
        "valor_referencia": int(len(df_periodos_diarios)),
        "status": "OK" if len(df_base_comparacao_diaria) == len(df_periodos_diarios) else "ERRO",
        "observacao": "A base de comparação diária deve preservar o número de linhas da base diária da 11.1.",
    },
    {
        "item": "linhas_base_comparacao_mensal",
        "valor": int(len(df_base_comparacao_mensal)),
        "valor_referencia": int(len(df_periodos_mensais)),
        "status": "OK" if len(df_base_comparacao_mensal) == len(df_periodos_mensais) else "ERRO",
        "observacao": "A base de comparação mensal deve preservar o número de linhas da tabela mensal da 11.1.",
    },
    {
        "item": "linhas_base_comparacao_anual",
        "valor": int(len(df_base_comparacao_anual)),
        "valor_referencia": int(len(df_periodos_anuais)),
        "status": "OK" if len(df_base_comparacao_anual) == len(df_periodos_anuais) else "ERRO",
        "observacao": "A base de comparação anual deve preservar o número de linhas da tabela anual da 11.1.",
    },
    {
        "item": "duplicatas_base_comparacao_diaria",
        "valor": duplicatas_base_diaria,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_diaria == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por chave_estrategia e período na base diária de comparação.",
    },
    {
        "item": "duplicatas_base_comparacao_mensal",
        "valor": duplicatas_base_mensal,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_mensal == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por chave_estrategia e período na base mensal de comparação.",
    },
    {
        "item": "duplicatas_base_comparacao_anual",
        "valor": duplicatas_base_anual,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_anual == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por chave_estrategia e período na base anual de comparação.",
    },
    {
        "item": "retornos_benchmarks_ausentes",
        "valor": retornos_benchmark_ausentes,
        "valor_referencia": 0,
        "status": "OK" if retornos_benchmark_ausentes == 0 else "ERRO",
        "observacao": "Retornos ausentes de Ibovespa ou CDI inviabilizam a contagem de superação.",
    },
    {
        "item": "linhas_resumo_multifrequencia",
        "valor": int(len(df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia)),
        "valor_referencia": int(linhas_esperadas_resumo),
        "status": "OK" if len(df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia) == linhas_esperadas_resumo else "ERRO",
        "observacao": "O resumo multifrequência deve ter uma linha por estratégia e frequência.",
    },
    {
        "item": "frequencias_resumo_multifrequencia",
        "valor": int(df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia["frequencia"].nunique()),
        "valor_referencia": 3,
        "status": "OK" if df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia["frequencia"].nunique() == 3 else "ERRO",
        "observacao": "O resumo deve conter exatamente as frequências anual, diária e mensal.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": 0,
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_tbl_auditoria_validacao_comparacao_benchmarks = pd.DataFrame(itens_auditoria)

erros_bloqueantes = int((df_tbl_auditoria_validacao_comparacao_benchmarks["status"] == "ERRO").sum())

print(f"Itens de auditoria                        : {len(df_tbl_auditoria_validacao_comparacao_benchmarks):,}")
print(f"Erros bloqueantes                         : {erros_bloqueantes:,}")

if erros_bloqueantes > 0:
    print(df_tbl_auditoria_validacao_comparacao_benchmarks.to_string(index=False))
    raise ValueError("A auditoria da comparação com Ibovespa e CDI encontrou erros bloqueantes.")

print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(df_base_comparacao_diaria, caminho_base_comparacao_benchmarks_diaria, index=False)
salvar_dataframe(df_base_comparacao_mensal, caminho_base_comparacao_benchmarks_mensal, index=False)
salvar_dataframe(df_base_comparacao_anual, caminho_base_comparacao_benchmarks_anual, index=False)

salvar_dataframe(df_tbl_comparacao_ibovespa_cdi_diario, caminho_tbl_comparacao_ibovespa_cdi_diario, index=False)
salvar_dataframe(df_tbl_comparacao_ibovespa_cdi_mensal, caminho_tbl_comparacao_ibovespa_cdi_mensal, index=False)
salvar_dataframe(df_tbl_comparacao_ibovespa_cdi_anual, caminho_tbl_comparacao_ibovespa_cdi_anual, index=False)

salvar_dataframe(df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia, caminho_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia, index=False)
salvar_dataframe(df_tbl_ranking_superacao_benchmarks, caminho_tbl_ranking_superacao_benchmarks, index=False)
salvar_dataframe(df_tbl_auditoria_validacao_comparacao_benchmarks, caminho_tbl_auditoria_validacao_comparacao_benchmarks, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da comparação com Ibovespa e CDI:")
print(df_tbl_auditoria_validacao_comparacao_benchmarks.to_string(index=False))

print("\nResumo multifrequência - amostra:")
print(df_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia.head(30).to_string(index=False))

print("\nRanking de superação - amostra:")
print(df_tbl_ranking_superacao_benchmarks.head(30).to_string(index=False))

print("\nBase diária de comparação - amostra:")
colunas_amostra_base = [
    "data_fim_periodo",
    "periodo_comparacao",
    "chave_estrategia",
    "nome_exibicao_estrategia",
    "retorno_periodo",
    "retorno_periodo_ibovespa",
    "retorno_periodo_cdi",
    "retorno_excesso_vs_ibovespa",
    "retorno_excesso_vs_cdi",
    "flag_superou_ibovespa",
    "flag_superou_cdi",
]
colunas_amostra_base = [coluna for coluna in colunas_amostra_base if coluna in df_base_comparacao_diaria.columns]
print(df_base_comparacao_diaria[colunas_amostra_base].head(20).to_string(index=False))

print("\nArquivos salvos na subetapa 11.5:")
print(f"- {caminho_base_comparacao_benchmarks_diaria}")
print(f"- {caminho_base_comparacao_benchmarks_mensal}")
print(f"- {caminho_base_comparacao_benchmarks_anual}")
print(f"- {caminho_tbl_comparacao_ibovespa_cdi_diario}")
print(f"- {caminho_tbl_comparacao_ibovespa_cdi_mensal}")
print(f"- {caminho_tbl_comparacao_ibovespa_cdi_anual}")
print(f"- {caminho_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia}")
print(f"- {caminho_tbl_ranking_superacao_benchmarks}")
print(f"- {caminho_tbl_auditoria_validacao_comparacao_benchmarks}")

print("\nETAPA 11.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 11.5 - COMPARAÇÃO COM IBOVESPA E CDI

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - retornos diários da 11.1             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_base_retornos_diarios.parquet
Entrada - retornos mensais da 11.1             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_tbl_retornos_mensais.parquet
Entrada - retornos anuais da 11.1              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_tbl_retornos_anuais.parquet
Entrada - risco diário da 11.2                 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_2_tbl_risc

## Etapa 11.6) Comparação com Estratégias Aleatórias e Mensais

In [57]:
%%time
# ============================================================
# Etapa 11.6) Comparação com Estratégias Aleatórias e Mensais
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 11.6 - COMPARAÇÃO COM ESTRATÉGIAS ALEATÓRIAS E MENSAIS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

caminho_base_comparacao_benchmarks_diaria = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="base",
    nome="comparacao_benchmarks_diaria",
)

caminho_base_comparacao_benchmarks_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="base",
    nome="comparacao_benchmarks_mensal",
)

caminho_base_comparacao_benchmarks_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="base",
    nome="comparacao_benchmarks_anual",
)

caminho_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=5,
    tipo_arquivo="tbl",
    nome="resumo_comparacao_ibovespa_cdi_multifrequencia",
)

caminho_base_comparacao_controles_diaria = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="base",
    nome="comparacao_controles_diaria",
)

caminho_base_comparacao_controles_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="base",
    nome="comparacao_controles_mensal",
)

caminho_base_comparacao_controles_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="base",
    nome="comparacao_controles_anual",
)

caminho_tbl_pares_comparacao_controles = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="pares_comparacao_controles",
)

caminho_tbl_comparacao_controles_diario = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="comparacao_controles_diario",
)

caminho_tbl_comparacao_controles_mensal = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="comparacao_controles_mensal",
)

caminho_tbl_comparacao_controles_anual = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="comparacao_controles_anual",
)

caminho_tbl_resumo_comparacao_controles_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="resumo_comparacao_controles_multifrequencia",
)

caminho_tbl_posicionamento_relativo_multifrequencia = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="posicionamento_relativo_multifrequencia",
)

caminho_tbl_ranking_estrategias_reais_vs_controles = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="ranking_estrategias_reais_vs_controles",
)

caminho_tbl_auditoria_validacao_controles = gerar_caminho_arquivo(
    etapa=11,
    subetapa=6,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_controles",
)

print(f"Entrada - base comparação diária da 11.5        : {caminho_base_comparacao_benchmarks_diaria}")
print(f"Entrada - base comparação mensal da 11.5        : {caminho_base_comparacao_benchmarks_mensal}")
print(f"Entrada - base comparação anual da 11.5         : {caminho_base_comparacao_benchmarks_anual}")
print(f"Entrada - resumo multifrequência da 11.5        : {caminho_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia}")
print(f"Saída   - base comparação controles diária      : {caminho_base_comparacao_controles_diaria}")
print(f"Saída   - base comparação controles mensal      : {caminho_base_comparacao_controles_mensal}")
print(f"Saída   - base comparação controles anual       : {caminho_base_comparacao_controles_anual}")
print(f"Saída   - pares de comparação                   : {caminho_tbl_pares_comparacao_controles}")
print(f"Saída   - tabela controles diária               : {caminho_tbl_comparacao_controles_diario}")
print(f"Saída   - tabela controles mensal               : {caminho_tbl_comparacao_controles_mensal}")
print(f"Saída   - tabela controles anual                : {caminho_tbl_comparacao_controles_anual}")
print(f"Saída   - resumo multifrequência                : {caminho_tbl_resumo_comparacao_controles_multifrequencia}")
print(f"Saída   - posicionamento relativo               : {caminho_tbl_posicionamento_relativo_multifrequencia}")
print(f"Saída   - ranking estratégias reais vs controles: {caminho_tbl_ranking_estrategias_reais_vs_controles}")
print(f"Saída   - auditoria de validação                : {caminho_tbl_auditoria_validacao_controles}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da 11.5
# ============================================================

print("\n[3/12] Carga das bases oficiais da 11.5...")

df_base_benchmarks_diaria = pd.read_parquet(caminho_base_comparacao_benchmarks_diaria)
df_base_benchmarks_mensal = pd.read_parquet(caminho_base_comparacao_benchmarks_mensal)
df_base_benchmarks_anual = pd.read_parquet(caminho_base_comparacao_benchmarks_anual)
df_metricas_multifrequencia = pd.read_parquet(caminho_tbl_resumo_comparacao_ibovespa_cdi_multifrequencia)

print(f"Base comparação benchmarks diária        : {df_base_benchmarks_diaria.shape[0]:,} linhas x {df_base_benchmarks_diaria.shape[1]} colunas")
print(f"Base comparação benchmarks mensal        : {df_base_benchmarks_mensal.shape[0]:,} linhas x {df_base_benchmarks_mensal.shape[1]} colunas")
print(f"Base comparação benchmarks anual         : {df_base_benchmarks_anual.shape[0]:,} linhas x {df_base_benchmarks_anual.shape[1]} colunas")
print(f"Resumo multifrequência da 11.5           : {df_metricas_multifrequencia.shape[0]:,} linhas x {df_metricas_multifrequencia.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Validação estrutural das bases de entrada
# ============================================================

print("\n[4/12] Validação estrutural das bases de entrada...")

colunas_obrigatorias_base_periodica = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "familia_estrategia",
    "ordem_exibicao",
    "frequencia",
    "periodo_comparacao",
    "data_inicio_periodo",
    "data_fim_periodo",
    "retorno_periodo",
    "patrimonio_fim_observado",
]

colunas_obrigatorias_metricas = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "familia_estrategia",
    "ordem_exibicao",
    "frequencia",
    "retorno_acumulado_estrategia",
    "retorno_anualizado_estrategia",
]

for nome_base, df_base in [
    ("comparacao_benchmarks_diaria", df_base_benchmarks_diaria),
    ("comparacao_benchmarks_mensal", df_base_benchmarks_mensal),
    ("comparacao_benchmarks_anual", df_base_benchmarks_anual),
]:
    colunas_ausentes = [coluna for coluna in colunas_obrigatorias_base_periodica if coluna not in df_base.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"Colunas ausentes na base {nome_base}: {colunas_ausentes}")

colunas_ausentes_metricas = [coluna for coluna in colunas_obrigatorias_metricas if coluna not in df_metricas_multifrequencia.columns]
if len(colunas_ausentes_metricas) > 0:
    raise ValueError(f"Colunas ausentes no resumo multifrequência da 11.5: {colunas_ausentes_metricas}")

df_base_benchmarks_diaria = df_base_benchmarks_diaria.copy()
df_base_benchmarks_mensal = df_base_benchmarks_mensal.copy()
df_base_benchmarks_anual = df_base_benchmarks_anual.copy()
df_metricas_multifrequencia = df_metricas_multifrequencia.copy()

for df_base in [df_base_benchmarks_diaria, df_base_benchmarks_mensal, df_base_benchmarks_anual]:
    df_base["data_inicio_periodo"] = pd.to_datetime(df_base["data_inicio_periodo"], errors="coerce")
    df_base["data_fim_periodo"] = pd.to_datetime(df_base["data_fim_periodo"], errors="coerce")
    df_base["retorno_periodo"] = pd.to_numeric(df_base["retorno_periodo"], errors="coerce")
    df_base["patrimonio_fim_observado"] = pd.to_numeric(df_base["patrimonio_fim_observado"], errors="coerce")

for coluna in ["data_inicio", "data_fim"]:
    if coluna in df_metricas_multifrequencia.columns:
        df_metricas_multifrequencia[coluna] = pd.to_datetime(df_metricas_multifrequencia[coluna], errors="coerce")

duplicatas_diarias = int(df_base_benchmarks_diaria.duplicated(["chave_estrategia", "periodo_comparacao"]).sum())
duplicatas_mensais = int(df_base_benchmarks_mensal.duplicated(["chave_estrategia", "periodo_comparacao"]).sum())
duplicatas_anuais = int(df_base_benchmarks_anual.duplicated(["chave_estrategia", "periodo_comparacao"]).sum())
duplicatas_metricas = int(df_metricas_multifrequencia.duplicated(["chave_estrategia", "frequencia"]).sum())

if duplicatas_diarias > 0:
    raise ValueError("Há duplicidades por chave_estrategia e período na base diária da 11.5.")
if duplicatas_mensais > 0:
    raise ValueError("Há duplicidades por chave_estrategia e período na base mensal da 11.5.")
if duplicatas_anuais > 0:
    raise ValueError("Há duplicidades por chave_estrategia e período na base anual da 11.5.")
if duplicatas_metricas > 0:
    raise ValueError("Há duplicidades por chave_estrategia e frequência no resumo multifrequência da 11.5.")

print(f"Duplicidades diárias                     : {duplicatas_diarias:,}")
print(f"Duplicidades mensais                     : {duplicatas_mensais:,}")
print(f"Duplicidades anuais                      : {duplicatas_anuais:,}")
print(f"Duplicidades no resumo multifrequência   : {duplicatas_metricas:,}")
print("OK")

# ============================================================
# 5) Definição das estratégias reais e dos controles comparáveis
# ============================================================

print("\n[5/12] Definição das estratégias reais e dos controles comparáveis...")

estrategias_reais_deterministicas = {
    "capitulacao": "capitulacao__carteira_na__replica_na",
    "euforia": "euforia__carteira_na__replica_na",
}

frequencias_analise = ["diaria", "mensal", "anual"]

colunas_catalogo = [
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "estrategia_referencia",
    "estrategia_referencia_padronizada",
    "tipo_estrategia",
    "carteira_id",
    "replica_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "origem_backtest",
    "ordem_exibicao",
]

colunas_catalogo = [coluna for coluna in colunas_catalogo if coluna in df_metricas_multifrequencia.columns]

df_catalogo_estrategias = (
    df_metricas_multifrequencia[colunas_catalogo]
    .drop_duplicates("chave_estrategia")
    .reset_index(drop=True)
)

chaves_disponiveis = set(df_catalogo_estrategias["chave_estrategia"].dropna().astype(str).tolist())
chaves_reais_ausentes = [chave for chave in estrategias_reais_deterministicas.values() if chave not in chaves_disponiveis]
if len(chaves_reais_ausentes) > 0:
    raise ValueError(f"Estratégias reais não encontradas no catálogo consolidado: {chaves_reais_ausentes}")

colunas_texto_catalogo = [
    coluna
    for coluna in [
        "chave_estrategia",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "estrategia_referencia",
        "estrategia_referencia_padronizada",
        "tipo_estrategia",
        "nome_exibicao_estrategia",
        "categoria_estrategia",
        "subcategoria_estrategia",
    ]
    if coluna in df_catalogo_estrategias.columns
]

texto_catalogo = (
    df_catalogo_estrategias[colunas_texto_catalogo]
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

categoria = df_catalogo_estrategias.get("categoria_estrategia", pd.Series("", index=df_catalogo_estrategias.index)).astype(str).str.lower()
grupo_controle = df_catalogo_estrategias.get("grupo_controle", pd.Series("", index=df_catalogo_estrategias.index)).astype(str).str.lower()
referencia_padronizada = df_catalogo_estrategias.get("estrategia_referencia_padronizada", pd.Series("", index=df_catalogo_estrategias.index)).astype(str).str.lower()
subcategoria = df_catalogo_estrategias.get("subcategoria_estrategia", pd.Series("", index=df_catalogo_estrategias.index)).astype(str).str.lower()
chave_catalogo = df_catalogo_estrategias["chave_estrategia"].astype(str)

mascara_controles_mensais = (
    categoria.eq("controle_mensal")
    | grupo_controle.eq("mensal")
    | subcategoria.eq("mensal")
    | texto_catalogo.str.contains("aportes mensais", regex=False, na=False)
)

mascara_exclusao_mensal = (
    chave_catalogo.isin(list(estrategias_reais_deterministicas.values()))
    | chave_catalogo.str.contains("buy_and_hold", regex=False, na=False)
    | chave_catalogo.str.contains("cdi_only", regex=False, na=False)
)

chaves_controles_mensais = sorted(
    df_catalogo_estrategias.loc[mascara_controles_mensais & (~mascara_exclusao_mensal), "chave_estrategia"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

registros_pares = []

for estrategia_referencia, chave_real in estrategias_reais_deterministicas.items():
    mascara_controles_aleatorios = (
        (categoria.eq("controle_aleatorio") | grupo_controle.eq("aleatoria") | texto_catalogo.str.contains("aleatoria", regex=False, na=False))
        & (
            referencia_padronizada.eq(estrategia_referencia)
            | subcategoria.str.contains(f"aleatoria_{estrategia_referencia}", regex=False, na=False)
            | texto_catalogo.str.contains(f"aleatoria {estrategia_referencia}", regex=False, na=False)
            | texto_catalogo.str.contains(f"aleatoria_{estrategia_referencia}", regex=False, na=False)
            | texto_catalogo.str.contains(f"{estrategia_referencia}_aleatoria", regex=False, na=False)
        )
        & (~chave_catalogo.eq(chave_real))
    )

    chaves_controles_aleatorios = sorted(
        df_catalogo_estrategias.loc[mascara_controles_aleatorios, "chave_estrategia"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for chave_controle in chaves_controles_aleatorios:
        registros_pares.append(
            {
                "estrategia_referencia": estrategia_referencia,
                "chave_estrategia_real": chave_real,
                "chave_estrategia_controle": chave_controle,
                "grupo_comparacao_controle": "aleatorio_mesma_referencia",
                "tipo_comparacao_controle": "controle_aleatorio",
            }
        )

    for chave_controle in chaves_controles_mensais:
        registros_pares.append(
            {
                "estrategia_referencia": estrategia_referencia,
                "chave_estrategia_real": chave_real,
                "chave_estrategia_controle": chave_controle,
                "grupo_comparacao_controle": "controle_mensal",
                "tipo_comparacao_controle": "controle_mensal",
            }
        )

df_tbl_pares_comparacao_controles = pd.DataFrame(registros_pares).drop_duplicates().reset_index(drop=True)

if len(df_tbl_pares_comparacao_controles) == 0:
    raise ValueError("Nenhum par de comparação foi construído para a Etapa 11.6.")

mapa_nomes = df_catalogo_estrategias.set_index("chave_estrategia").to_dict(orient="index")

def obter_valor_catalogo(chave, coluna):
    if chave not in mapa_nomes:
        return np.nan
    return mapa_nomes[chave].get(coluna, np.nan)

for prefixo, coluna_chave in [
    ("real", "chave_estrategia_real"),
    ("controle", "chave_estrategia_controle"),
]:
    df_tbl_pares_comparacao_controles[f"estrategia_id_{prefixo}"] = df_tbl_pares_comparacao_controles[coluna_chave].map(lambda x: obter_valor_catalogo(x, "estrategia_id"))
    df_tbl_pares_comparacao_controles[f"nome_exibicao_estrategia_{prefixo}"] = df_tbl_pares_comparacao_controles[coluna_chave].map(lambda x: obter_valor_catalogo(x, "nome_exibicao_estrategia"))
    df_tbl_pares_comparacao_controles[f"categoria_estrategia_{prefixo}"] = df_tbl_pares_comparacao_controles[coluna_chave].map(lambda x: obter_valor_catalogo(x, "categoria_estrategia"))
    df_tbl_pares_comparacao_controles[f"subcategoria_estrategia_{prefixo}"] = df_tbl_pares_comparacao_controles[coluna_chave].map(lambda x: obter_valor_catalogo(x, "subcategoria_estrategia"))
    df_tbl_pares_comparacao_controles[f"familia_estrategia_{prefixo}"] = df_tbl_pares_comparacao_controles[coluna_chave].map(lambda x: obter_valor_catalogo(x, "familia_estrategia"))
    df_tbl_pares_comparacao_controles[f"ordem_exibicao_{prefixo}"] = df_tbl_pares_comparacao_controles[coluna_chave].map(lambda x: obter_valor_catalogo(x, "ordem_exibicao"))

ordem_pares = np.arange(1, len(df_tbl_pares_comparacao_controles) + 1)
df_tbl_pares_comparacao_controles["comparacao_id"] = (
    df_tbl_pares_comparacao_controles["estrategia_referencia"].astype(str)
    + "__"
    + df_tbl_pares_comparacao_controles["grupo_comparacao_controle"].astype(str)
    + "__par_"
    + pd.Series(ordem_pares, index=df_tbl_pares_comparacao_controles.index).astype(str)
)

n_pares_capitulacao = int((df_tbl_pares_comparacao_controles["estrategia_referencia"] == "capitulacao").sum())
n_pares_euforia = int((df_tbl_pares_comparacao_controles["estrategia_referencia"] == "euforia").sum())
n_controles_aleatorios_capitulacao = int(
    (
        (df_tbl_pares_comparacao_controles["estrategia_referencia"] == "capitulacao")
        & (df_tbl_pares_comparacao_controles["tipo_comparacao_controle"] == "controle_aleatorio")
    ).sum()
)
n_controles_aleatorios_euforia = int(
    (
        (df_tbl_pares_comparacao_controles["estrategia_referencia"] == "euforia")
        & (df_tbl_pares_comparacao_controles["tipo_comparacao_controle"] == "controle_aleatorio")
    ).sum()
)
n_controles_mensais_unicos = int(len(chaves_controles_mensais))

print(f"Estratégias no catálogo                  : {df_catalogo_estrategias['chave_estrategia'].nunique():,}")
print(f"Controles aleatórios da capitulação      : {n_controles_aleatorios_capitulacao:,}")
print(f"Controles aleatórios da euforia          : {n_controles_aleatorios_euforia:,}")
print(f"Controles mensais únicos                 : {n_controles_mensais_unicos:,}")
print(f"Pares de comparação da capitulação       : {n_pares_capitulacao:,}")
print(f"Pares de comparação da euforia           : {n_pares_euforia:,}")
print(f"Pares de comparação totais               : {len(df_tbl_pares_comparacao_controles):,}")
print("OK")

# ============================================================
# 6) Construção das bases período a período contra os controles
# ============================================================

print("\n[6/12] Construção das bases período a período contra os controles...")

colunas_base_periodica = [
    "chave_estrategia",
    "estrategia_id",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "familia_estrategia",
    "ordem_exibicao",
    "frequencia",
    "periodo_comparacao",
    "data_inicio_periodo",
    "data_fim_periodo",
    "retorno_periodo",
    "patrimonio_fim_observado",
]

colunas_base_periodica = [coluna for coluna in colunas_base_periodica if coluna in df_base_benchmarks_diaria.columns]


def construir_base_periodica_controles(df_base_periodica, frequencia):
    df_periodica = df_base_periodica.copy()
    df_periodica = df_periodica.loc[df_periodica["frequencia"].astype(str).eq(frequencia)].copy()

    registros = []

    for linha_par in df_tbl_pares_comparacao_controles.itertuples(index=False):
        chave_real = linha_par.chave_estrategia_real
        chave_controle = linha_par.chave_estrategia_controle

        df_real = df_periodica.loc[
            df_periodica["chave_estrategia"].astype(str).eq(chave_real),
            colunas_base_periodica,
        ].copy()

        df_controle = df_periodica.loc[
            df_periodica["chave_estrategia"].astype(str).eq(chave_controle),
            colunas_base_periodica,
        ].copy()

        if len(df_real) == 0 or len(df_controle) == 0:
            continue

        df_real = df_real.rename(
            columns={
                "chave_estrategia": "chave_estrategia_real",
                "estrategia_id": "estrategia_id_real",
                "nome_exibicao_estrategia": "nome_exibicao_estrategia_real",
                "categoria_estrategia": "categoria_estrategia_real",
                "subcategoria_estrategia": "subcategoria_estrategia_real",
                "familia_estrategia": "familia_estrategia_real",
                "ordem_exibicao": "ordem_exibicao_real",
                "retorno_periodo": "retorno_periodo_real",
                "patrimonio_fim_observado": "patrimonio_fim_observado_real",
            }
        )

        df_controle = df_controle.rename(
            columns={
                "chave_estrategia": "chave_estrategia_controle",
                "estrategia_id": "estrategia_id_controle",
                "nome_exibicao_estrategia": "nome_exibicao_estrategia_controle",
                "categoria_estrategia": "categoria_estrategia_controle",
                "subcategoria_estrategia": "subcategoria_estrategia_controle",
                "familia_estrategia": "familia_estrategia_controle",
                "ordem_exibicao": "ordem_exibicao_controle",
                "retorno_periodo": "retorno_periodo_controle",
                "patrimonio_fim_observado": "patrimonio_fim_observado_controle",
            }
        )

        colunas_chave_merge = ["frequencia", "periodo_comparacao", "data_inicio_periodo", "data_fim_periodo"]

        df_comp = df_real.merge(
            df_controle,
            on=colunas_chave_merge,
            how="inner",
        )

        if len(df_comp) == 0:
            continue

        df_comp["comparacao_id"] = linha_par.comparacao_id
        df_comp["estrategia_referencia"] = linha_par.estrategia_referencia
        df_comp["grupo_comparacao_controle"] = linha_par.grupo_comparacao_controle
        df_comp["tipo_comparacao_controle"] = linha_par.tipo_comparacao_controle

        df_comp["retorno_excesso_real_vs_controle"] = df_comp["retorno_periodo_real"] - df_comp["retorno_periodo_controle"]
        df_comp["retorno_relativo_composto_real_vs_controle"] = (1.0 + df_comp["retorno_periodo_real"]) / (1.0 + df_comp["retorno_periodo_controle"]) - 1.0
        df_comp["flag_real_superou_controle"] = df_comp["retorno_excesso_real_vs_controle"] > 0.0
        df_comp["flag_real_empatou_controle"] = np.isclose(
            df_comp["retorno_excesso_real_vs_controle"].fillna(np.nan),
            0.0,
            atol=0.0000000001,
            rtol=0.0,
        )
        df_comp["flag_real_perdeu_controle"] = df_comp["retorno_excesso_real_vs_controle"] < 0.0

        registros.append(df_comp)

    if len(registros) == 0:
        return pd.DataFrame()

    df_saida = pd.concat(registros, ignore_index=True)

    colunas_ordem = [
        "comparacao_id",
        "estrategia_referencia",
        "grupo_comparacao_controle",
        "tipo_comparacao_controle",
        "frequencia",
        "periodo_comparacao",
        "data_inicio_periodo",
        "data_fim_periodo",
        "chave_estrategia_real",
        "estrategia_id_real",
        "nome_exibicao_estrategia_real",
        "categoria_estrategia_real",
        "subcategoria_estrategia_real",
        "familia_estrategia_real",
        "ordem_exibicao_real",
        "chave_estrategia_controle",
        "estrategia_id_controle",
        "nome_exibicao_estrategia_controle",
        "categoria_estrategia_controle",
        "subcategoria_estrategia_controle",
        "familia_estrategia_controle",
        "ordem_exibicao_controle",
        "retorno_periodo_real",
        "retorno_periodo_controle",
        "retorno_excesso_real_vs_controle",
        "retorno_relativo_composto_real_vs_controle",
        "flag_real_superou_controle",
        "flag_real_empatou_controle",
        "flag_real_perdeu_controle",
        "patrimonio_fim_observado_real",
        "patrimonio_fim_observado_controle",
    ]

    colunas_ordem = [coluna for coluna in colunas_ordem if coluna in df_saida.columns]
    colunas_restantes = [coluna for coluna in df_saida.columns if coluna not in colunas_ordem]

    df_saida = df_saida[colunas_ordem + colunas_restantes].copy()
    df_saida = (
        df_saida
        .sort_values(["estrategia_referencia", "grupo_comparacao_controle", "comparacao_id", "periodo_comparacao"])
        .reset_index(drop=True)
    )

    return df_saida


df_base_comparacao_controles_diaria = construir_base_periodica_controles(df_base_benchmarks_diaria, "diaria")
df_base_comparacao_controles_mensal = construir_base_periodica_controles(df_base_benchmarks_mensal, "mensal")
df_base_comparacao_controles_anual = construir_base_periodica_controles(df_base_benchmarks_anual, "anual")

print(f"Linhas da base controles diária          : {len(df_base_comparacao_controles_diaria):,}")
print(f"Linhas da base controles mensal          : {len(df_base_comparacao_controles_mensal):,}")
print(f"Linhas da base controles anual           : {len(df_base_comparacao_controles_anual):,}")
print(f"Pares com dados diários                  : {df_base_comparacao_controles_diaria['comparacao_id'].nunique() if len(df_base_comparacao_controles_diaria) > 0 else 0:,}")
print(f"Pares com dados mensais                  : {df_base_comparacao_controles_mensal['comparacao_id'].nunique() if len(df_base_comparacao_controles_mensal) > 0 else 0:,}")
print(f"Pares com dados anuais                   : {df_base_comparacao_controles_anual['comparacao_id'].nunique() if len(df_base_comparacao_controles_anual) > 0 else 0:,}")
print("OK")

# ============================================================
# 7) Consolidação das tabelas de comparação por par
# ============================================================

print("\n[7/12] Consolidação das tabelas de comparação por par...")

metricas_complementares = [
    "retorno_acumulado_estrategia",
    "retorno_anualizado_estrategia",
    "volatilidade_periodo",
    "volatilidade_anualizada_equivalente",
    "downside_deviation_periodo",
    "downside_deviation_anualizada_equivalente",
    "var_historico_5pct",
    "cvar_historico_5pct",
    "sharpe_anualizado",
    "sortino_anualizado",
    "calmar",
    "max_drawdown",
    "max_drawdown_abs",
    "pct_periodos_em_drawdown",
    "periodos_pico_a_recuperacao_max_drawdown",
    "n_eventos_drawdown",
    "n_eventos_recuperados",
    "pct_eventos_recuperados",
    "pct_periodos_superou_ibovespa",
    "pct_periodos_superou_cdi",
    "excesso_acumulado_composto_vs_ibovespa",
    "excesso_acumulado_composto_vs_cdi",
    "excesso_anualizado_vs_ibovespa",
    "excesso_anualizado_vs_cdi",
]

metricas_complementares = [coluna for coluna in metricas_complementares if coluna in df_metricas_multifrequencia.columns]

df_metricas_indice = df_metricas_multifrequencia.set_index(["chave_estrategia", "frequencia"])


def produto_retorno_acumulado(serie_retornos):
    retornos = pd.to_numeric(serie_retornos, errors="coerce").dropna()
    if len(retornos) == 0:
        return np.nan
    return float((1.0 + retornos).prod() - 1.0)


def calcular_retorno_anualizado(retorno_acumulado, n_anos):
    if pd.isna(retorno_acumulado) or pd.isna(n_anos) or n_anos <= 0:
        return np.nan
    if (1.0 + retorno_acumulado) <= 0:
        return np.nan
    return float((1.0 + retorno_acumulado) ** (1.0 / n_anos) - 1.0)


def obter_metrica(chave, frequencia, coluna):
    if coluna not in df_metricas_multifrequencia.columns:
        return np.nan
    try:
        valor = df_metricas_indice.loc[(chave, frequencia), coluna]
    except KeyError:
        return np.nan
    if isinstance(valor, pd.Series):
        valor = valor.dropna().iloc[0] if len(valor.dropna()) > 0 else np.nan
    return valor


def montar_tabela_comparacao_controles(df_base_controles, frequencia):
    if len(df_base_controles) == 0:
        return pd.DataFrame()

    registros = []

    for comparacao_id, grupo_original in df_base_controles.groupby("comparacao_id", dropna=False):
        grupo = grupo_original.sort_values("data_fim_periodo").copy()
        primeiro = grupo.iloc[0]

        n_periodos = int(len(grupo))
        data_inicio = pd.to_datetime(grupo["data_inicio_periodo"].min(), errors="coerce")
        data_fim = pd.to_datetime(grupo["data_fim_periodo"].max(), errors="coerce")
        n_dias_corridos = int((data_fim - data_inicio).days) if pd.notna(data_inicio) and pd.notna(data_fim) else np.nan
        n_anos = float(n_dias_corridos / 365.25) if pd.notna(n_dias_corridos) and n_dias_corridos > 0 else np.nan

        retorno_acumulado_real_periodico = produto_retorno_acumulado(grupo["retorno_periodo_real"])
        retorno_acumulado_controle_periodico = produto_retorno_acumulado(grupo["retorno_periodo_controle"])
        retorno_acumulado_relativo_composto = (
            float((1.0 + retorno_acumulado_real_periodico) / (1.0 + retorno_acumulado_controle_periodico) - 1.0)
            if pd.notna(retorno_acumulado_real_periodico) and pd.notna(retorno_acumulado_controle_periodico) and (1.0 + retorno_acumulado_controle_periodico) != 0
            else np.nan
        )

        retorno_anualizado_real_periodico = calcular_retorno_anualizado(retorno_acumulado_real_periodico, n_anos)
        retorno_anualizado_controle_periodico = calcular_retorno_anualizado(retorno_acumulado_controle_periodico, n_anos)
        excesso_anualizado_real_vs_controle_periodico = (
            retorno_anualizado_real_periodico - retorno_anualizado_controle_periodico
            if pd.notna(retorno_anualizado_real_periodico) and pd.notna(retorno_anualizado_controle_periodico)
            else np.nan
        )

        n_superou_controle = int(grupo["flag_real_superou_controle"].fillna(False).sum())
        n_empatou_controle = int(grupo["flag_real_empatou_controle"].fillna(False).sum())
        n_perdeu_controle = int(grupo["flag_real_perdeu_controle"].fillna(False).sum())

        chave_real = str(primeiro["chave_estrategia_real"])
        chave_controle = str(primeiro["chave_estrategia_controle"])

        registro = {
            "comparacao_id": comparacao_id,
            "estrategia_referencia": primeiro["estrategia_referencia"],
            "grupo_comparacao_controle": primeiro["grupo_comparacao_controle"],
            "tipo_comparacao_controle": primeiro["tipo_comparacao_controle"],
            "frequencia": frequencia,
            "chave_estrategia_real": chave_real,
            "estrategia_id_real": primeiro.get("estrategia_id_real", np.nan),
            "nome_exibicao_estrategia_real": primeiro.get("nome_exibicao_estrategia_real", np.nan),
            "categoria_estrategia_real": primeiro.get("categoria_estrategia_real", np.nan),
            "subcategoria_estrategia_real": primeiro.get("subcategoria_estrategia_real", np.nan),
            "familia_estrategia_real": primeiro.get("familia_estrategia_real", np.nan),
            "ordem_exibicao_real": primeiro.get("ordem_exibicao_real", np.nan),
            "chave_estrategia_controle": chave_controle,
            "estrategia_id_controle": primeiro.get("estrategia_id_controle", np.nan),
            "nome_exibicao_estrategia_controle": primeiro.get("nome_exibicao_estrategia_controle", np.nan),
            "categoria_estrategia_controle": primeiro.get("categoria_estrategia_controle", np.nan),
            "subcategoria_estrategia_controle": primeiro.get("subcategoria_estrategia_controle", np.nan),
            "familia_estrategia_controle": primeiro.get("familia_estrategia_controle", np.nan),
            "ordem_exibicao_controle": primeiro.get("ordem_exibicao_controle", np.nan),
            "n_periodos_comparaveis": n_periodos,
            "data_inicio": data_inicio,
            "data_fim": data_fim,
            "n_dias_corridos": n_dias_corridos,
            "n_anos": n_anos,
            "n_vezes_real_superou_controle": n_superou_controle,
            "n_vezes_real_empatou_controle": n_empatou_controle,
            "n_vezes_real_perdeu_controle": n_perdeu_controle,
            "pct_periodos_real_superou_controle": float(n_superou_controle / n_periodos) if n_periodos > 0 else np.nan,
            "pct_periodos_real_empatou_controle": float(n_empatou_controle / n_periodos) if n_periodos > 0 else np.nan,
            "pct_periodos_real_perdeu_controle": float(n_perdeu_controle / n_periodos) if n_periodos > 0 else np.nan,
            "excesso_medio_periodo_real_vs_controle": float(grupo["retorno_excesso_real_vs_controle"].mean()) if n_periodos > 0 else np.nan,
            "excesso_mediano_periodo_real_vs_controle": float(grupo["retorno_excesso_real_vs_controle"].median()) if n_periodos > 0 else np.nan,
            "melhor_excesso_periodo_real_vs_controle": float(grupo["retorno_excesso_real_vs_controle"].max()) if n_periodos > 0 else np.nan,
            "pior_excesso_periodo_real_vs_controle": float(grupo["retorno_excesso_real_vs_controle"].min()) if n_periodos > 0 else np.nan,
            "retorno_acumulado_real_periodico": retorno_acumulado_real_periodico,
            "retorno_acumulado_controle_periodico": retorno_acumulado_controle_periodico,
            "retorno_acumulado_relativo_composto_real_vs_controle": retorno_acumulado_relativo_composto,
            "retorno_anualizado_real_periodico": retorno_anualizado_real_periodico,
            "retorno_anualizado_controle_periodico": retorno_anualizado_controle_periodico,
            "excesso_anualizado_real_vs_controle_periodico": excesso_anualizado_real_vs_controle_periodico,
            "flag_retorno_acumulado_real_superou_controle": retorno_acumulado_relativo_composto > 0.0 if pd.notna(retorno_acumulado_relativo_composto) else False,
            "flag_retorno_anualizado_real_superou_controle": excesso_anualizado_real_vs_controle_periodico > 0.0 if pd.notna(excesso_anualizado_real_vs_controle_periodico) else False,
        }

        for metrica in metricas_complementares:
            valor_real = obter_metrica(chave_real, frequencia, metrica)
            valor_controle = obter_metrica(chave_controle, frequencia, metrica)
            registro[f"{metrica}_real"] = valor_real
            registro[f"{metrica}_controle"] = valor_controle
            registro[f"dif_{metrica}_real_vs_controle"] = valor_real - valor_controle if pd.notna(valor_real) and pd.notna(valor_controle) else np.nan

        registro["flag_sharpe_real_superou_controle"] = registro.get("dif_sharpe_anualizado_real_vs_controle", np.nan) > 0.0 if pd.notna(registro.get("dif_sharpe_anualizado_real_vs_controle", np.nan)) else False
        registro["flag_sortino_real_superou_controle"] = registro.get("dif_sortino_anualizado_real_vs_controle", np.nan) > 0.0 if pd.notna(registro.get("dif_sortino_anualizado_real_vs_controle", np.nan)) else False
        registro["flag_calmar_real_superou_controle"] = registro.get("dif_calmar_real_vs_controle", np.nan) > 0.0 if pd.notna(registro.get("dif_calmar_real_vs_controle", np.nan)) else False
        registro["flag_volatilidade_real_menor_controle"] = registro.get("dif_volatilidade_anualizada_equivalente_real_vs_controle", np.nan) < 0.0 if pd.notna(registro.get("dif_volatilidade_anualizada_equivalente_real_vs_controle", np.nan)) else False
        registro["flag_drawdown_abs_real_menor_controle"] = registro.get("dif_max_drawdown_abs_real_vs_controle", np.nan) < 0.0 if pd.notna(registro.get("dif_max_drawdown_abs_real_vs_controle", np.nan)) else False

        flags_score = [
            "flag_retorno_acumulado_real_superou_controle",
            "flag_retorno_anualizado_real_superou_controle",
            "flag_sharpe_real_superou_controle",
            "flag_sortino_real_superou_controle",
            "flag_calmar_real_superou_controle",
            "flag_volatilidade_real_menor_controle",
            "flag_drawdown_abs_real_menor_controle",
        ]

        registro["score_relativo_real_vs_controle"] = int(sum(bool(registro.get(flag, False)) for flag in flags_score))
        registro["score_relativo_maximo"] = int(len(flags_score))
        registro["pct_score_relativo_real_vs_controle"] = float(registro["score_relativo_real_vs_controle"] / registro["score_relativo_maximo"])

        registros.append(registro)

    df_saida = pd.DataFrame(registros)

    if len(df_saida) > 0:
        df_saida = (
            df_saida
            .sort_values([
                "estrategia_referencia",
                "grupo_comparacao_controle",
                "score_relativo_real_vs_controle",
                "retorno_acumulado_relativo_composto_real_vs_controle",
            ], ascending=[True, True, False, False])
            .reset_index(drop=True)
        )

    return df_saida


df_tbl_comparacao_controles_diario = montar_tabela_comparacao_controles(df_base_comparacao_controles_diaria, "diaria")
df_tbl_comparacao_controles_mensal = montar_tabela_comparacao_controles(df_base_comparacao_controles_mensal, "mensal")
df_tbl_comparacao_controles_anual = montar_tabela_comparacao_controles(df_base_comparacao_controles_anual, "anual")

print(f"Linhas da tabela controles diária        : {len(df_tbl_comparacao_controles_diario):,}")
print(f"Linhas da tabela controles mensal        : {len(df_tbl_comparacao_controles_mensal):,}")
print(f"Linhas da tabela controles anual         : {len(df_tbl_comparacao_controles_anual):,}")
print("OK")

# ============================================================
# 8) Consolidação do resumo multifrequência
# ============================================================

print("\n[8/12] Consolidação do resumo multifrequência...")

df_tbl_resumo_comparacao_controles_multifrequencia = pd.concat(
    [
        df_tbl_comparacao_controles_diario,
        df_tbl_comparacao_controles_mensal,
        df_tbl_comparacao_controles_anual,
    ],
    ignore_index=True,
)

if len(df_tbl_resumo_comparacao_controles_multifrequencia) > 0:
    df_tbl_resumo_comparacao_controles_multifrequencia = (
        df_tbl_resumo_comparacao_controles_multifrequencia
        .sort_values([
            "estrategia_referencia",
            "grupo_comparacao_controle",
            "frequencia",
            "score_relativo_real_vs_controle",
            "retorno_acumulado_relativo_composto_real_vs_controle",
        ], ascending=[True, True, True, False, False])
        .reset_index(drop=True)
    )

print(f"Linhas do resumo multifrequência         : {len(df_tbl_resumo_comparacao_controles_multifrequencia):,}")
print(f"Frequências no resumo                    : {', '.join(sorted(df_tbl_resumo_comparacao_controles_multifrequencia['frequencia'].dropna().unique())) if len(df_tbl_resumo_comparacao_controles_multifrequencia) > 0 else 'sem dados'}")
print("OK")

# ============================================================
# 9) Consolidação do posicionamento relativo das estratégias reais
# ============================================================

print("\n[9/12] Consolidação do posicionamento relativo das estratégias reais...")

registros_posicionamento = []

if len(df_tbl_resumo_comparacao_controles_multifrequencia) > 0:
    colunas_flags_posicionamento = [
        "flag_retorno_acumulado_real_superou_controle",
        "flag_retorno_anualizado_real_superou_controle",
        "flag_sharpe_real_superou_controle",
        "flag_sortino_real_superou_controle",
        "flag_calmar_real_superou_controle",
        "flag_volatilidade_real_menor_controle",
        "flag_drawdown_abs_real_menor_controle",
    ]

    for chaves_grupo, grupo in df_tbl_resumo_comparacao_controles_multifrequencia.groupby(
        ["estrategia_referencia", "chave_estrategia_real", "nome_exibicao_estrategia_real", "grupo_comparacao_controle", "tipo_comparacao_controle", "frequencia"],
        dropna=False,
    ):
        estrategia_referencia, chave_real, nome_real, grupo_comparacao, tipo_comparacao, frequencia = chaves_grupo
        n_controles = int(grupo["chave_estrategia_controle"].nunique())

        registro = {
            "estrategia_referencia": estrategia_referencia,
            "chave_estrategia_real": chave_real,
            "nome_exibicao_estrategia_real": nome_real,
            "grupo_comparacao_controle": grupo_comparacao,
            "tipo_comparacao_controle": tipo_comparacao,
            "frequencia": frequencia,
            "n_controles_comparados": n_controles,
            "n_pares_comparados": int(len(grupo)),
            "score_relativo_medio": float(grupo["score_relativo_real_vs_controle"].mean()) if len(grupo) > 0 else np.nan,
            "pct_score_relativo_medio": float(grupo["pct_score_relativo_real_vs_controle"].mean()) if len(grupo) > 0 else np.nan,
            "retorno_acumulado_relativo_composto_medio": float(grupo["retorno_acumulado_relativo_composto_real_vs_controle"].mean()) if len(grupo) > 0 else np.nan,
            "retorno_acumulado_relativo_composto_mediano": float(grupo["retorno_acumulado_relativo_composto_real_vs_controle"].median()) if len(grupo) > 0 else np.nan,
            "excesso_anualizado_medio_real_vs_controle": float(grupo["excesso_anualizado_real_vs_controle_periodico"].mean()) if len(grupo) > 0 else np.nan,
            "excesso_anualizado_mediano_real_vs_controle": float(grupo["excesso_anualizado_real_vs_controle_periodico"].median()) if len(grupo) > 0 else np.nan,
            "pct_periodos_superacao_medio": float(grupo["pct_periodos_real_superou_controle"].mean()) if len(grupo) > 0 else np.nan,
            "pct_periodos_superacao_mediano": float(grupo["pct_periodos_real_superou_controle"].median()) if len(grupo) > 0 else np.nan,
        }

        for coluna_flag in colunas_flags_posicionamento:
            if coluna_flag in grupo.columns:
                nome_saida = coluna_flag.replace("flag_", "pct_controles_")
                registro[nome_saida] = float(grupo[coluna_flag].fillna(False).mean()) if len(grupo) > 0 else np.nan
                registro[nome_saida.replace("pct_controles_", "n_controles_")] = int(grupo[coluna_flag].fillna(False).sum()) if len(grupo) > 0 else 0

        registros_posicionamento.append(registro)

df_tbl_posicionamento_relativo_multifrequencia = pd.DataFrame(registros_posicionamento)

if len(df_tbl_posicionamento_relativo_multifrequencia) > 0:
    df_tbl_posicionamento_relativo_multifrequencia = (
        df_tbl_posicionamento_relativo_multifrequencia
        .sort_values(["estrategia_referencia", "grupo_comparacao_controle", "frequencia"])
        .reset_index(drop=True)
    )

print(f"Linhas do posicionamento relativo        : {len(df_tbl_posicionamento_relativo_multifrequencia):,}")
print("OK")

# ============================================================
# 10) Construção do ranking de estratégias reais versus controles
# ============================================================

print("\n[10/12] Construção do ranking de estratégias reais versus controles...")

colunas_ranking = [
    "comparacao_id",
    "estrategia_referencia",
    "grupo_comparacao_controle",
    "tipo_comparacao_controle",
    "frequencia",
    "chave_estrategia_real",
    "nome_exibicao_estrategia_real",
    "chave_estrategia_controle",
    "nome_exibicao_estrategia_controle",
    "n_periodos_comparaveis",
    "n_vezes_real_superou_controle",
    "pct_periodos_real_superou_controle",
    "retorno_acumulado_relativo_composto_real_vs_controle",
    "excesso_anualizado_real_vs_controle_periodico",
    "dif_retorno_anualizado_estrategia_real_vs_controle",
    "dif_sharpe_anualizado_real_vs_controle",
    "dif_sortino_anualizado_real_vs_controle",
    "dif_calmar_real_vs_controle",
    "dif_volatilidade_anualizada_equivalente_real_vs_controle",
    "dif_max_drawdown_abs_real_vs_controle",
    "score_relativo_real_vs_controle",
    "score_relativo_maximo",
    "pct_score_relativo_real_vs_controle",
]

colunas_ranking = [coluna for coluna in colunas_ranking if coluna in df_tbl_resumo_comparacao_controles_multifrequencia.columns]

df_tbl_ranking_estrategias_reais_vs_controles = df_tbl_resumo_comparacao_controles_multifrequencia[colunas_ranking].copy()

if len(df_tbl_ranking_estrategias_reais_vs_controles) > 0:
    df_tbl_ranking_estrategias_reais_vs_controles["ranking_score_relativo"] = (
        df_tbl_ranking_estrategias_reais_vs_controles
        .groupby(["estrategia_referencia", "grupo_comparacao_controle", "frequencia"])["pct_score_relativo_real_vs_controle"]
        .rank(ascending=False, method="min")
    )

    if "retorno_acumulado_relativo_composto_real_vs_controle" in df_tbl_ranking_estrategias_reais_vs_controles.columns:
        df_tbl_ranking_estrategias_reais_vs_controles["ranking_retorno_relativo"] = (
            df_tbl_ranking_estrategias_reais_vs_controles
            .groupby(["estrategia_referencia", "grupo_comparacao_controle", "frequencia"])["retorno_acumulado_relativo_composto_real_vs_controle"]
            .rank(ascending=False, method="min")
        )

    if "pct_periodos_real_superou_controle" in df_tbl_ranking_estrategias_reais_vs_controles.columns:
        df_tbl_ranking_estrategias_reais_vs_controles["ranking_pct_periodos_superacao"] = (
            df_tbl_ranking_estrategias_reais_vs_controles
            .groupby(["estrategia_referencia", "grupo_comparacao_controle", "frequencia"])["pct_periodos_real_superou_controle"]
            .rank(ascending=False, method="min")
        )

    df_tbl_ranking_estrategias_reais_vs_controles = (
        df_tbl_ranking_estrategias_reais_vs_controles
        .sort_values([
            "estrategia_referencia",
            "grupo_comparacao_controle",
            "frequencia",
            "ranking_score_relativo",
            "ranking_retorno_relativo",
        ])
        .reset_index(drop=True)
    )

print(f"Linhas do ranking                         : {len(df_tbl_ranking_estrategias_reais_vs_controles):,}")
print("OK")

# ============================================================
# 11) Construção da auditoria de validação
# ============================================================

print("\n[11/12] Construção da auditoria de validação...")

n_pares_esperados = int(len(df_tbl_pares_comparacao_controles))

n_pares_diarios = int(df_base_comparacao_controles_diaria["comparacao_id"].nunique()) if len(df_base_comparacao_controles_diaria) > 0 else 0
n_pares_mensais = int(df_base_comparacao_controles_mensal["comparacao_id"].nunique()) if len(df_base_comparacao_controles_mensal) > 0 else 0
n_pares_anuais = int(df_base_comparacao_controles_anual["comparacao_id"].nunique()) if len(df_base_comparacao_controles_anual) > 0 else 0

duplicatas_base_diaria = int(df_base_comparacao_controles_diaria.duplicated(["comparacao_id", "periodo_comparacao"]).sum()) if len(df_base_comparacao_controles_diaria) > 0 else 0
duplicatas_base_mensal = int(df_base_comparacao_controles_mensal.duplicated(["comparacao_id", "periodo_comparacao"]).sum()) if len(df_base_comparacao_controles_mensal) > 0 else 0
duplicatas_base_anual = int(df_base_comparacao_controles_anual.duplicated(["comparacao_id", "periodo_comparacao"]).sum()) if len(df_base_comparacao_controles_anual) > 0 else 0

linhas_resumo_esperadas = n_pares_esperados * 3

metricas_infinitas = int(
    np.isinf(
        df_tbl_resumo_comparacao_controles_multifrequencia
        .select_dtypes(include=[np.number])
        .to_numpy()
    ).sum()
) if len(df_tbl_resumo_comparacao_controles_multifrequencia) > 0 else 0

retornos_ausentes_bases = int(
    df_base_comparacao_controles_diaria[["retorno_periodo_real", "retorno_periodo_controle"]].isna().sum().sum()
    + df_base_comparacao_controles_mensal[["retorno_periodo_real", "retorno_periodo_controle"]].isna().sum().sum()
    + df_base_comparacao_controles_anual[["retorno_periodo_real", "retorno_periodo_controle"]].isna().sum().sum()
) if all(len(df) > 0 for df in [df_base_comparacao_controles_diaria, df_base_comparacao_controles_mensal, df_base_comparacao_controles_anual]) else 0

itens_auditoria = [
    {
        "item": "estrategia_real_capitulacao_identificada",
        "valor": 1 if estrategias_reais_deterministicas["capitulacao"] in chaves_disponiveis else 0,
        "valor_referencia": 1,
        "status": "OK" if estrategias_reais_deterministicas["capitulacao"] in chaves_disponiveis else "ERRO",
        "observacao": "A estratégia real de capitulação deve estar presente no catálogo consolidado.",
    },
    {
        "item": "estrategia_real_euforia_identificada",
        "valor": 1 if estrategias_reais_deterministicas["euforia"] in chaves_disponiveis else 0,
        "valor_referencia": 1,
        "status": "OK" if estrategias_reais_deterministicas["euforia"] in chaves_disponiveis else "ERRO",
        "observacao": "A estratégia real de euforia deve estar presente no catálogo consolidado.",
    },
    {
        "item": "controles_aleatorios_capitulacao",
        "valor": n_controles_aleatorios_capitulacao,
        "valor_referencia": 1,
        "status": "OK" if n_controles_aleatorios_capitulacao >= 1 else "ERRO",
        "observacao": "A capitulação deve ter ao menos uma réplica aleatória comparável.",
    },
    {
        "item": "controles_aleatorios_euforia",
        "valor": n_controles_aleatorios_euforia,
        "valor_referencia": 1,
        "status": "OK" if n_controles_aleatorios_euforia >= 1 else "ERRO",
        "observacao": "A euforia deve ter ao menos uma réplica aleatória comparável.",
    },
    {
        "item": "controles_mensais_unicos",
        "valor": n_controles_mensais_unicos,
        "valor_referencia": 1,
        "status": "OK" if n_controles_mensais_unicos >= 1 else "ERRO",
        "observacao": "A comparação deve ter ao menos um controle mensal.",
    },
    {
        "item": "pares_diarios_vs_pares_esperados",
        "valor": n_pares_diarios,
        "valor_referencia": n_pares_esperados,
        "status": "OK" if n_pares_diarios == n_pares_esperados else "ERRO",
        "observacao": "Todos os pares de comparação devem estar presentes na frequência diária.",
    },
    {
        "item": "pares_mensais_vs_pares_esperados",
        "valor": n_pares_mensais,
        "valor_referencia": n_pares_esperados,
        "status": "OK" if n_pares_mensais == n_pares_esperados else "ERRO",
        "observacao": "Todos os pares de comparação devem estar presentes na frequência mensal.",
    },
    {
        "item": "pares_anuais_vs_pares_esperados",
        "valor": n_pares_anuais,
        "valor_referencia": n_pares_esperados,
        "status": "OK" if n_pares_anuais == n_pares_esperados else "ERRO",
        "observacao": "Todos os pares de comparação devem estar presentes na frequência anual.",
    },
    {
        "item": "duplicatas_base_controles_diaria",
        "valor": duplicatas_base_diaria,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_diaria == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por comparação e período na base diária.",
    },
    {
        "item": "duplicatas_base_controles_mensal",
        "valor": duplicatas_base_mensal,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_mensal == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por comparação e período na base mensal.",
    },
    {
        "item": "duplicatas_base_controles_anual",
        "valor": duplicatas_base_anual,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_base_anual == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por comparação e período na base anual.",
    },
    {
        "item": "linhas_resumo_multifrequencia",
        "valor": int(len(df_tbl_resumo_comparacao_controles_multifrequencia)),
        "valor_referencia": int(linhas_resumo_esperadas),
        "status": "OK" if len(df_tbl_resumo_comparacao_controles_multifrequencia) == linhas_resumo_esperadas else "ERRO",
        "observacao": "O resumo multifrequência deve ter uma linha por par de comparação e frequência.",
    },
    {
        "item": "frequencias_resumo_multifrequencia",
        "valor": int(df_tbl_resumo_comparacao_controles_multifrequencia["frequencia"].nunique()) if len(df_tbl_resumo_comparacao_controles_multifrequencia) > 0 else 0,
        "valor_referencia": 3,
        "status": "OK" if len(df_tbl_resumo_comparacao_controles_multifrequencia) > 0 and df_tbl_resumo_comparacao_controles_multifrequencia["frequencia"].nunique() == 3 else "ERRO",
        "observacao": "O resumo deve conter exatamente as frequências anual, diária e mensal.",
    },
    {
        "item": "retornos_ausentes_bases_controles",
        "valor": retornos_ausentes_bases,
        "valor_referencia": 0,
        "status": "OK" if retornos_ausentes_bases == 0 else "ERRO",
        "observacao": "Retornos ausentes inviabilizam a comparação período a período.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": 0,
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_tbl_auditoria_validacao_controles = pd.DataFrame(itens_auditoria)

erros_bloqueantes = int((df_tbl_auditoria_validacao_controles["status"] == "ERRO").sum())

print(f"Itens de auditoria                        : {len(df_tbl_auditoria_validacao_controles):,}")
print(f"Erros bloqueantes                         : {erros_bloqueantes:,}")

if erros_bloqueantes > 0:
    print(df_tbl_auditoria_validacao_controles.to_string(index=False))
    raise ValueError("A auditoria da comparação com estratégias aleatórias e mensais encontrou erros bloqueantes.")

print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(df_base_comparacao_controles_diaria, caminho_base_comparacao_controles_diaria, index=False)
salvar_dataframe(df_base_comparacao_controles_mensal, caminho_base_comparacao_controles_mensal, index=False)
salvar_dataframe(df_base_comparacao_controles_anual, caminho_base_comparacao_controles_anual, index=False)

salvar_dataframe(df_tbl_pares_comparacao_controles, caminho_tbl_pares_comparacao_controles, index=False)
salvar_dataframe(df_tbl_comparacao_controles_diario, caminho_tbl_comparacao_controles_diario, index=False)
salvar_dataframe(df_tbl_comparacao_controles_mensal, caminho_tbl_comparacao_controles_mensal, index=False)
salvar_dataframe(df_tbl_comparacao_controles_anual, caminho_tbl_comparacao_controles_anual, index=False)
salvar_dataframe(df_tbl_resumo_comparacao_controles_multifrequencia, caminho_tbl_resumo_comparacao_controles_multifrequencia, index=False)
salvar_dataframe(df_tbl_posicionamento_relativo_multifrequencia, caminho_tbl_posicionamento_relativo_multifrequencia, index=False)
salvar_dataframe(df_tbl_ranking_estrategias_reais_vs_controles, caminho_tbl_ranking_estrategias_reais_vs_controles, index=False)
salvar_dataframe(df_tbl_auditoria_validacao_controles, caminho_tbl_auditoria_validacao_controles, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da comparação com estratégias aleatórias e mensais:")
print(df_tbl_auditoria_validacao_controles.to_string(index=False))

print("\nPares de comparação - amostra:")
print(df_tbl_pares_comparacao_controles.head(30).to_string(index=False))

print("\nResumo multifrequência - amostra:")
colunas_amostra_resumo = [
    "estrategia_referencia",
    "grupo_comparacao_controle",
    "frequencia",
    "nome_exibicao_estrategia_real",
    "nome_exibicao_estrategia_controle",
    "n_periodos_comparaveis",
    "n_vezes_real_superou_controle",
    "pct_periodos_real_superou_controle",
    "retorno_acumulado_relativo_composto_real_vs_controle",
    "excesso_anualizado_real_vs_controle_periodico",
    "dif_sharpe_anualizado_real_vs_controle",
    "dif_calmar_real_vs_controle",
    "score_relativo_real_vs_controle",
    "pct_score_relativo_real_vs_controle",
]
colunas_amostra_resumo = [coluna for coluna in colunas_amostra_resumo if coluna in df_tbl_resumo_comparacao_controles_multifrequencia.columns]
print(df_tbl_resumo_comparacao_controles_multifrequencia[colunas_amostra_resumo].head(30).to_string(index=False))

print("\nPosicionamento relativo - amostra:")
print(df_tbl_posicionamento_relativo_multifrequencia.head(30).to_string(index=False))

print("\nRanking estratégias reais vs controles - amostra:")
print(df_tbl_ranking_estrategias_reais_vs_controles.head(30).to_string(index=False))

print("\nBase diária de comparação com controles - amostra:")
colunas_amostra_base = [
    "data_fim_periodo",
    "periodo_comparacao",
    "estrategia_referencia",
    "grupo_comparacao_controle",
    "nome_exibicao_estrategia_real",
    "nome_exibicao_estrategia_controle",
    "retorno_periodo_real",
    "retorno_periodo_controle",
    "retorno_excesso_real_vs_controle",
    "flag_real_superou_controle",
]
colunas_amostra_base = [coluna for coluna in colunas_amostra_base if coluna in df_base_comparacao_controles_diaria.columns]
print(df_base_comparacao_controles_diaria[colunas_amostra_base].head(20).to_string(index=False))

print("\nArquivos salvos na subetapa 11.6:")
print(f"- {caminho_base_comparacao_controles_diaria}")
print(f"- {caminho_base_comparacao_controles_mensal}")
print(f"- {caminho_base_comparacao_controles_anual}")
print(f"- {caminho_tbl_pares_comparacao_controles}")
print(f"- {caminho_tbl_comparacao_controles_diario}")
print(f"- {caminho_tbl_comparacao_controles_mensal}")
print(f"- {caminho_tbl_comparacao_controles_anual}")
print(f"- {caminho_tbl_resumo_comparacao_controles_multifrequencia}")
print(f"- {caminho_tbl_posicionamento_relativo_multifrequencia}")
print(f"- {caminho_tbl_ranking_estrategias_reais_vs_controles}")
print(f"- {caminho_tbl_auditoria_validacao_controles}")

print("\nETAPA 11.6 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 11.6 - COMPARAÇÃO COM ESTRATÉGIAS ALEATÓRIAS E MENSAIS

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - base comparação diária da 11.5        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_5_base_comparacao_benchmarks_diaria.parquet
Entrada - base comparação mensal da 11.5        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_5_base_comparacao_benchmarks_mensal.parquet
Entrada - base comparação anual da 11.5         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_5_base_comparacao_benchmarks_anual.parquet
Entrada - resumo multifrequência da 11.5        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_

## Etapa 11.7) Comparação Direta entre Capitulação e Euforia

In [58]:
%%time
# ============================================================
# Etapa 11.7) Comparação Direta entre Capitulação e Euforia
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 11.7 - COMPARAÇÃO DIRETA ENTRE CAPITULAÇÃO E EUFORIA")
print("=" * 100)

# Esta subetapa compara diretamente as estratégias reais de Capitulação e Euforia.
# A diferença principal é definida como Capitulação menos Euforia.
# Valores positivos indicam vantagem de Capitulação no período; valores negativos indicam vantagem de Euforia.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/11] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/11] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])
DIR_ETAPA_11.mkdir(parents=True, exist_ok=True)

CAMINHO_COMPARACAO_BENCHMARKS_DIARIA_11_5 = DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_diaria.parquet"
CAMINHO_COMPARACAO_BENCHMARKS_MENSAL_11_5 = DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_mensal.parquet"
CAMINHO_COMPARACAO_BENCHMARKS_ANUAL_11_5 = DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_anual.parquet"

CAMINHO_BASE_DIARIA = DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_diaria.parquet"
CAMINHO_BASE_MENSAL = DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_mensal.parquet"
CAMINHO_BASE_ANUAL = DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_anual.parquet"
CAMINHO_BASE_CONSOLIDADA = DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_consolidada.parquet"
CAMINHO_RESUMO = DIR_ETAPA_11 / "11_7_tbl_resumo_comparacao_capitulacao_euforia.parquet"
CAMINHO_RANKING = DIR_ETAPA_11 / "11_7_tbl_ranking_capitulacao_vs_euforia.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_11 / "11_7_tbl_base_grafico_capitulacao_vs_euforia.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_11 / "11_7_tbl_parametros_comparacao_capitulacao_euforia.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_11 / "11_7_tbl_auditoria_validacao_comparacao_capitulacao_euforia.parquet"

CAMINHOS_OBRIGATORIOS = [
    CAMINHO_COMPARACAO_BENCHMARKS_DIARIA_11_5,
    CAMINHO_COMPARACAO_BENCHMARKS_MENSAL_11_5,
    CAMINHO_COMPARACAO_BENCHMARKS_ANUAL_11_5,
]

for caminho in CAMINHOS_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - comparação benchmarks diária da 11.5 : {CAMINHO_COMPARACAO_BENCHMARKS_DIARIA_11_5}")
print(f"Entrada - comparação benchmarks mensal da 11.5 : {CAMINHO_COMPARACAO_BENCHMARKS_MENSAL_11_5}")
print(f"Entrada - comparação benchmarks anual da 11.5  : {CAMINHO_COMPARACAO_BENCHMARKS_ANUAL_11_5}")
print(f"Saída   - base diária cap. vs euforia          : {CAMINHO_BASE_DIARIA}")
print(f"Saída   - base mensal cap. vs euforia          : {CAMINHO_BASE_MENSAL}")
print(f"Saída   - base anual cap. vs euforia           : {CAMINHO_BASE_ANUAL}")
print(f"Saída   - base consolidada                     : {CAMINHO_BASE_CONSOLIDADA}")
print(f"Saída   - resumo comparativo                   : {CAMINHO_RESUMO}")
print(f"Saída   - ranking comparativo                  : {CAMINHO_RANKING}")
print(f"Saída   - base gráfico                         : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - parâmetros                           : {CAMINHO_PARAMETROS}")
print(f"Saída   - auditoria                            : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/11] Carga das bases oficiais da subetapa...")

df_benchmarks_diaria = pd.read_parquet(CAMINHO_COMPARACAO_BENCHMARKS_DIARIA_11_5)
df_benchmarks_mensal = pd.read_parquet(CAMINHO_COMPARACAO_BENCHMARKS_MENSAL_11_5)
df_benchmarks_anual = pd.read_parquet(CAMINHO_COMPARACAO_BENCHMARKS_ANUAL_11_5)

print(f"Comparação benchmarks diária da 11.5 : {len(df_benchmarks_diaria):,} linhas x {df_benchmarks_diaria.shape[1]:,} colunas")
print(f"Comparação benchmarks mensal da 11.5 : {len(df_benchmarks_mensal):,} linhas x {df_benchmarks_mensal.shape[1]:,} colunas")
print(f"Comparação benchmarks anual da 11.5  : {len(df_benchmarks_anual):,} linhas x {df_benchmarks_anual.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/11] Funções auxiliares da subetapa...")

CHAVE_CAPITULACAO = "capitulacao__carteira_na__replica_na"
CHAVE_EUFORIA = "euforia__carteira_na__replica_na"

FREQUENCIAS_ANALISE = {
    "diaria": {
        "periodos_por_ano": 252.0,
        "visao": "diaria",
        "aplicacao_metodologica": "visao_diaria_complementar",
    },
    "mensal": {
        "periodos_por_ano": 12.0,
        "visao": "mensal",
        "aplicacao_metodologica": "visao_mensal_principal",
    },
    "anual": {
        "periodos_por_ano": 1.0,
        "visao": "anual",
        "aplicacao_metodologica": "visao_anual_exploratoria",
    },
}

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}


def validar_colunas_obrigatorias(df, colunas, nome_base):
    colunas_ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        raise KeyError(
            f"Colunas obrigatórias ausentes na base {nome_base}: {colunas_ausentes}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )


def obter_coluna_opcional(df, coluna, valor_padrao=np.nan):
    if coluna in df.columns:
        return df[coluna]
    return pd.Series(valor_padrao, index=df.index)


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def converter_data_opcional(df, coluna):
    if coluna in df.columns:
        return pd.to_datetime(df[coluna], errors="coerce").dt.normalize()
    return pd.Series(pd.NaT, index=df.index)


def dividir_seguro(numerador, denominador):
    numerador = pd.to_numeric(pd.Series(numerador), errors="coerce")
    denominador = pd.to_numeric(pd.Series(denominador), errors="coerce")
    resultado = np.where(
        np.isclose(denominador, 0.0, atol=0.0000000001, rtol=0.0),
        np.nan,
        numerador / denominador,
    )
    return resultado


def calcular_retorno_relativo_composto(retorno_base, retorno_comparador):
    retorno_base = pd.to_numeric(retorno_base, errors="coerce")
    retorno_comparador = pd.to_numeric(retorno_comparador, errors="coerce")
    denominador = 1.0 + retorno_comparador
    resultado = np.where(
        np.isclose(denominador, 0.0, atol=0.0000000001, rtol=0.0),
        np.nan,
        (1.0 + retorno_base) / denominador - 1.0,
    )
    return pd.Series(resultado, index=retorno_base.index)


def calcular_max_drawdown(retornos):
    serie = pd.to_numeric(pd.Series(retornos), errors="coerce").dropna()
    if len(serie) == 0:
        return np.nan
    curva = (1.0 + serie).cumprod()
    pico = curva.cummax()
    drawdown = curva / pico - 1.0
    return float(drawdown.min())


def calcular_metricas_serie(retornos, periodos_por_ano):
    serie = pd.to_numeric(pd.Series(retornos), errors="coerce").dropna()
    n_periodos = int(len(serie))

    if n_periodos == 0:
        return {
            "n_periodos": 0,
            "retorno_acumulado": np.nan,
            "retorno_periodico_medio": np.nan,
            "retorno_periodico_mediano": np.nan,
            "retorno_anualizado": np.nan,
            "volatilidade_periodica": np.nan,
            "volatilidade_anualizada": np.nan,
            "downside_vol_periodica": np.nan,
            "downside_vol_anualizada": np.nan,
            "sharpe_simplificado": np.nan,
            "sortino_simplificado": np.nan,
            "max_drawdown": np.nan,
            "calmar_simplificado": np.nan,
            "pct_periodos_positivos": np.nan,
        }

    curva_final = float((1.0 + serie).prod())
    retorno_acumulado = curva_final - 1.0
    retorno_periodico_medio = float(serie.mean())
    retorno_periodico_mediano = float(serie.median())

    if curva_final > 0.0:
        retorno_anualizado = curva_final ** (periodos_por_ano / n_periodos) - 1.0
    else:
        retorno_anualizado = np.nan

    volatilidade_periodica = float(serie.std(ddof=1)) if n_periodos > 1 else np.nan
    volatilidade_anualizada = volatilidade_periodica * np.sqrt(periodos_por_ano) if pd.notna(volatilidade_periodica) else np.nan

    perdas = np.minimum(serie, 0.0)
    downside_vol_periodica = float(np.sqrt(np.mean(np.square(perdas)))) if n_periodos > 0 else np.nan
    downside_vol_anualizada = downside_vol_periodica * np.sqrt(periodos_por_ano) if pd.notna(downside_vol_periodica) else np.nan

    if pd.notna(volatilidade_periodica) and volatilidade_periodica > 0.0:
        sharpe_simplificado = retorno_periodico_medio / volatilidade_periodica * np.sqrt(periodos_por_ano)
    else:
        sharpe_simplificado = np.nan

    if pd.notna(downside_vol_periodica) and downside_vol_periodica > 0.0:
        sortino_simplificado = retorno_periodico_medio / downside_vol_periodica * np.sqrt(periodos_por_ano)
    else:
        sortino_simplificado = np.nan

    max_drawdown = calcular_max_drawdown(serie)
    if pd.notna(max_drawdown) and max_drawdown < 0.0 and pd.notna(retorno_anualizado):
        calmar_simplificado = retorno_anualizado / abs(max_drawdown)
    else:
        calmar_simplificado = np.nan

    pct_periodos_positivos = float((serie > 0.0).mean())

    return {
        "n_periodos": n_periodos,
        "retorno_acumulado": retorno_acumulado,
        "retorno_periodico_medio": retorno_periodico_medio,
        "retorno_periodico_mediano": retorno_periodico_mediano,
        "retorno_anualizado": retorno_anualizado,
        "volatilidade_periodica": volatilidade_periodica,
        "volatilidade_anualizada": volatilidade_anualizada,
        "downside_vol_periodica": downside_vol_periodica,
        "downside_vol_anualizada": downside_vol_anualizada,
        "sharpe_simplificado": sharpe_simplificado,
        "sortino_simplificado": sortino_simplificado,
        "max_drawdown": max_drawdown,
        "calmar_simplificado": calmar_simplificado,
        "pct_periodos_positivos": pct_periodos_positivos,
    }


def classificar_vencedor(valor_capitulacao, valor_euforia, maior_melhor=True, tolerancia=0.0000000001):
    if pd.isna(valor_capitulacao) or pd.isna(valor_euforia):
        return "nao_classificado"
    diferenca = valor_capitulacao - valor_euforia
    if abs(diferenca) <= tolerancia:
        return "empate_tecnico"
    if maior_melhor:
        return "capitulacao" if diferenca > 0 else "euforia"
    return "capitulacao" if diferenca < 0 else "euforia"


def padronizar_dataframe_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def preparar_auditoria(linhas):
    df = pd.DataFrame(linhas)
    if len(df) == 0:
        return pd.DataFrame(columns=["item", "valor", "valor_referencia", "status", "observacao"])
    df["status"] = df["status"].astype(str)
    return df[["item", "valor", "valor_referencia", "status", "observacao"]].copy()

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Construção das bases pareadas por frequência
# ============================================================

print("\n[5/11] Construção das bases pareadas por frequência...")

BASES_ENTRADA = {
    "diaria": df_benchmarks_diaria,
    "mensal": df_benchmarks_mensal,
    "anual": df_benchmarks_anual,
}

bases_pareadas = {}
registros_auditoria_interna = []

for frequencia, df_base in BASES_ENTRADA.items():
    validar_colunas_obrigatorias(
        df_base,
        ["chave_estrategia", "periodo_comparacao", "retorno_periodo"],
        f"11.5 comparação benchmarks {frequencia}",
    )

    df_temp = df_base.copy()
    df_temp["chave_estrategia"] = df_temp["chave_estrategia"].astype(str)
    df_temp["periodo_comparacao"] = df_temp["periodo_comparacao"].astype(str)
    df_temp["retorno_periodo"] = converter_numero(df_temp["retorno_periodo"])

    df_cap = df_temp.loc[df_temp["chave_estrategia"].eq(CHAVE_CAPITULACAO)].copy()
    df_euf = df_temp.loc[df_temp["chave_estrategia"].eq(CHAVE_EUFORIA)].copy()

    duplicatas_cap = int(df_cap.duplicated(["periodo_comparacao"]).sum())
    duplicatas_euf = int(df_euf.duplicated(["periodo_comparacao"]).sum())

    registros_auditoria_interna.append({
        "frequencia": frequencia,
        "linhas_capitulacao": len(df_cap),
        "linhas_euforia": len(df_euf),
        "duplicatas_capitulacao": duplicatas_cap,
        "duplicatas_euforia": duplicatas_euf,
    })

    colunas_base = [
        "periodo_comparacao",
        "retorno_periodo",
    ]

    for coluna in [
        "data_inicio_periodo",
        "data_fim_periodo",
        "nome_exibicao_estrategia",
        "estrategia_id",
        "categoria_estrategia",
        "subcategoria_estrategia",
        "familia_estrategia",
    ]:
        if coluna in df_temp.columns:
            colunas_base.append(coluna)

    df_cap = df_cap[colunas_base].drop_duplicates(["periodo_comparacao"], keep="first").copy()
    df_euf = df_euf[colunas_base].drop_duplicates(["periodo_comparacao"], keep="first").copy()

    df_cap = df_cap.rename(columns={
        "retorno_periodo": "retorno_periodo_capitulacao",
        "data_inicio_periodo": "data_inicio_periodo_capitulacao",
        "data_fim_periodo": "data_fim_periodo_capitulacao",
        "nome_exibicao_estrategia": "nome_exibicao_capitulacao",
        "estrategia_id": "estrategia_id_capitulacao",
        "categoria_estrategia": "categoria_capitulacao",
        "subcategoria_estrategia": "subcategoria_capitulacao",
        "familia_estrategia": "familia_capitulacao",
    })
    df_euf = df_euf.rename(columns={
        "retorno_periodo": "retorno_periodo_euforia",
        "data_inicio_periodo": "data_inicio_periodo_euforia",
        "data_fim_periodo": "data_fim_periodo_euforia",
        "nome_exibicao_estrategia": "nome_exibicao_euforia",
        "estrategia_id": "estrategia_id_euforia",
        "categoria_estrategia": "categoria_euforia",
        "subcategoria_estrategia": "subcategoria_euforia",
        "familia_estrategia": "familia_euforia",
    })

    df_pareada = df_cap.merge(df_euf, on="periodo_comparacao", how="inner")
    df_pareada["frequencia"] = frequencia
    df_pareada["visao_comparacao"] = FREQUENCIAS_ANALISE[frequencia]["visao"]
    df_pareada["aplicacao_metodologica"] = FREQUENCIAS_ANALISE[frequencia]["aplicacao_metodologica"]
    df_pareada["ordem_frequencia"] = MAPA_ORDEM_FREQUENCIA[frequencia]

    if "data_inicio_periodo_capitulacao" in df_pareada.columns:
        df_pareada["data_inicio_periodo"] = pd.to_datetime(df_pareada["data_inicio_periodo_capitulacao"], errors="coerce").dt.normalize()
    else:
        df_pareada["data_inicio_periodo"] = pd.NaT

    if "data_fim_periodo_capitulacao" in df_pareada.columns:
        df_pareada["data_fim_periodo"] = pd.to_datetime(df_pareada["data_fim_periodo_capitulacao"], errors="coerce").dt.normalize()
    else:
        df_pareada["data_fim_periodo"] = pd.NaT

    df_pareada["chave_capitulacao"] = CHAVE_CAPITULACAO
    df_pareada["chave_euforia"] = CHAVE_EUFORIA
    df_pareada["retorno_periodo_capitulacao"] = converter_numero(df_pareada["retorno_periodo_capitulacao"])
    df_pareada["retorno_periodo_euforia"] = converter_numero(df_pareada["retorno_periodo_euforia"])
    df_pareada["excesso_capitulacao_vs_euforia"] = df_pareada["retorno_periodo_capitulacao"] - df_pareada["retorno_periodo_euforia"]
    df_pareada["excesso_euforia_vs_capitulacao"] = -df_pareada["excesso_capitulacao_vs_euforia"]
    df_pareada["retorno_relativo_composto_capitulacao_vs_euforia"] = calcular_retorno_relativo_composto(
        df_pareada["retorno_periodo_capitulacao"],
        df_pareada["retorno_periodo_euforia"],
    )
    df_pareada["retorno_relativo_composto_euforia_vs_capitulacao"] = calcular_retorno_relativo_composto(
        df_pareada["retorno_periodo_euforia"],
        df_pareada["retorno_periodo_capitulacao"],
    )

    df_pareada["flag_capitulacao_superou_euforia"] = df_pareada["excesso_capitulacao_vs_euforia"] > 0.0
    df_pareada["flag_euforia_superou_capitulacao"] = df_pareada["excesso_capitulacao_vs_euforia"] < 0.0
    df_pareada["flag_empate_capitulacao_euforia"] = np.isclose(
        df_pareada["excesso_capitulacao_vs_euforia"].fillna(np.nan),
        0.0,
        atol=0.0000000001,
        rtol=0.0,
    )

    df_pareada = df_pareada.sort_values(["data_fim_periodo", "periodo_comparacao"]).reset_index(drop=True)
    df_pareada["patrimonio_relativo_capitulacao"] = (1.0 + df_pareada["retorno_periodo_capitulacao"]).cumprod()
    df_pareada["patrimonio_relativo_euforia"] = (1.0 + df_pareada["retorno_periodo_euforia"]).cumprod()
    df_pareada["diferenca_patrimonio_relativo_capitulacao_menos_euforia"] = df_pareada["patrimonio_relativo_capitulacao"] - df_pareada["patrimonio_relativo_euforia"]
    df_pareada["razao_patrimonio_capitulacao_sobre_euforia"] = np.where(
        np.isclose(df_pareada["patrimonio_relativo_euforia"], 0.0, atol=0.0000000001, rtol=0.0),
        np.nan,
        df_pareada["patrimonio_relativo_capitulacao"] / df_pareada["patrimonio_relativo_euforia"],
    )

    colunas_saida = [
        "frequencia",
        "visao_comparacao",
        "aplicacao_metodologica",
        "ordem_frequencia",
        "periodo_comparacao",
        "data_inicio_periodo",
        "data_fim_periodo",
        "chave_capitulacao",
        "chave_euforia",
        "retorno_periodo_capitulacao",
        "retorno_periodo_euforia",
        "excesso_capitulacao_vs_euforia",
        "excesso_euforia_vs_capitulacao",
        "retorno_relativo_composto_capitulacao_vs_euforia",
        "retorno_relativo_composto_euforia_vs_capitulacao",
        "flag_capitulacao_superou_euforia",
        "flag_euforia_superou_capitulacao",
        "flag_empate_capitulacao_euforia",
        "patrimonio_relativo_capitulacao",
        "patrimonio_relativo_euforia",
        "diferenca_patrimonio_relativo_capitulacao_menos_euforia",
        "razao_patrimonio_capitulacao_sobre_euforia",
    ]

    for coluna in [
        "nome_exibicao_capitulacao",
        "nome_exibicao_euforia",
        "estrategia_id_capitulacao",
        "estrategia_id_euforia",
        "categoria_capitulacao",
        "categoria_euforia",
        "subcategoria_capitulacao",
        "subcategoria_euforia",
        "familia_capitulacao",
        "familia_euforia",
    ]:
        if coluna in df_pareada.columns:
            colunas_saida.append(coluna)

    df_pareada = df_pareada[colunas_saida].copy()
    bases_pareadas[frequencia] = df_pareada

    print(f"Base pareada {frequencia:<6}: {len(df_pareada):,} linhas")

print("OK")

# ============================================================
# 6) Consolidação das bases pareadas
# ============================================================

print("\n[6/11] Consolidação das bases pareadas...")

df_base_diaria = bases_pareadas["diaria"].copy()
df_base_mensal = bases_pareadas["mensal"].copy()
df_base_anual = bases_pareadas["anual"].copy()

df_base_consolidada = pd.concat(
    [df_base_diaria, df_base_mensal, df_base_anual],
    ignore_index=True,
)

df_base_consolidada = df_base_consolidada.sort_values(
    ["ordem_frequencia", "data_fim_periodo", "periodo_comparacao"]
).reset_index(drop=True)

print(f"Base diária                         : {len(df_base_diaria):,} linhas")
print(f"Base mensal                         : {len(df_base_mensal):,} linhas")
print(f"Base anual                          : {len(df_base_anual):,} linhas")
print(f"Base consolidada                    : {len(df_base_consolidada):,} linhas")
print("OK")

# ============================================================
# 7) Cálculo do resumo comparativo por frequência
# ============================================================

print("\n[7/11] Cálculo do resumo comparativo por frequência...")

registros_resumo = []

for frequencia, df_freq in bases_pareadas.items():
    periodos_por_ano = FREQUENCIAS_ANALISE[frequencia]["periodos_por_ano"]

    metricas_cap = calcular_metricas_serie(df_freq["retorno_periodo_capitulacao"], periodos_por_ano)
    metricas_euf = calcular_metricas_serie(df_freq["retorno_periodo_euforia"], periodos_por_ano)
    metricas_excesso = calcular_metricas_serie(df_freq["excesso_capitulacao_vs_euforia"], periodos_por_ano)

    retorno_relativo_acumulado_cap_vs_euf = (
        float(df_freq["patrimonio_relativo_capitulacao"].iloc[-1] / df_freq["patrimonio_relativo_euforia"].iloc[-1] - 1.0)
        if len(df_freq) > 0 and df_freq["patrimonio_relativo_euforia"].iloc[-1] != 0
        else np.nan
    )

    registros_resumo.append({
        "frequencia": frequencia,
        "visao_comparacao": FREQUENCIAS_ANALISE[frequencia]["visao"],
        "aplicacao_metodologica": FREQUENCIAS_ANALISE[frequencia]["aplicacao_metodologica"],
        "ordem_frequencia": MAPA_ORDEM_FREQUENCIA[frequencia],
        "n_periodos": int(len(df_freq)),
        "data_inicio": df_freq["data_inicio_periodo"].min(),
        "data_fim": df_freq["data_fim_periodo"].max(),
        "retorno_acumulado_capitulacao": metricas_cap["retorno_acumulado"],
        "retorno_acumulado_euforia": metricas_euf["retorno_acumulado"],
        "diferenca_retorno_acumulado_capitulacao_menos_euforia": metricas_cap["retorno_acumulado"] - metricas_euf["retorno_acumulado"],
        "retorno_relativo_acumulado_capitulacao_vs_euforia": retorno_relativo_acumulado_cap_vs_euf,
        "retorno_anualizado_capitulacao": metricas_cap["retorno_anualizado"],
        "retorno_anualizado_euforia": metricas_euf["retorno_anualizado"],
        "diferenca_retorno_anualizado_capitulacao_menos_euforia": metricas_cap["retorno_anualizado"] - metricas_euf["retorno_anualizado"],
        "volatilidade_anualizada_capitulacao": metricas_cap["volatilidade_anualizada"],
        "volatilidade_anualizada_euforia": metricas_euf["volatilidade_anualizada"],
        "diferenca_volatilidade_anualizada_capitulacao_menos_euforia": metricas_cap["volatilidade_anualizada"] - metricas_euf["volatilidade_anualizada"],
        "downside_vol_anualizada_capitulacao": metricas_cap["downside_vol_anualizada"],
        "downside_vol_anualizada_euforia": metricas_euf["downside_vol_anualizada"],
        "diferenca_downside_vol_anualizada_capitulacao_menos_euforia": metricas_cap["downside_vol_anualizada"] - metricas_euf["downside_vol_anualizada"],
        "sharpe_simplificado_capitulacao": metricas_cap["sharpe_simplificado"],
        "sharpe_simplificado_euforia": metricas_euf["sharpe_simplificado"],
        "diferenca_sharpe_simplificado_capitulacao_menos_euforia": metricas_cap["sharpe_simplificado"] - metricas_euf["sharpe_simplificado"],
        "sortino_simplificado_capitulacao": metricas_cap["sortino_simplificado"],
        "sortino_simplificado_euforia": metricas_euf["sortino_simplificado"],
        "diferenca_sortino_simplificado_capitulacao_menos_euforia": metricas_cap["sortino_simplificado"] - metricas_euf["sortino_simplificado"],
        "max_drawdown_capitulacao": metricas_cap["max_drawdown"],
        "max_drawdown_euforia": metricas_euf["max_drawdown"],
        "diferenca_max_drawdown_capitulacao_menos_euforia": metricas_cap["max_drawdown"] - metricas_euf["max_drawdown"],
        "calmar_simplificado_capitulacao": metricas_cap["calmar_simplificado"],
        "calmar_simplificado_euforia": metricas_euf["calmar_simplificado"],
        "diferenca_calmar_simplificado_capitulacao_menos_euforia": metricas_cap["calmar_simplificado"] - metricas_euf["calmar_simplificado"],
        "media_excesso_capitulacao_vs_euforia": metricas_excesso["retorno_periodico_medio"],
        "mediana_excesso_capitulacao_vs_euforia": metricas_excesso["retorno_periodico_mediano"],
        "volatilidade_excesso_capitulacao_vs_euforia": metricas_excesso["volatilidade_periodica"],
        "pct_periodos_capitulacao_superou_euforia": float(df_freq["flag_capitulacao_superou_euforia"].mean()),
        "pct_periodos_euforia_superou_capitulacao": float(df_freq["flag_euforia_superou_capitulacao"].mean()),
        "pct_periodos_empate": float(df_freq["flag_empate_capitulacao_euforia"].mean()),
        "vencedor_retorno_acumulado": classificar_vencedor(metricas_cap["retorno_acumulado"], metricas_euf["retorno_acumulado"], maior_melhor=True),
        "vencedor_retorno_anualizado": classificar_vencedor(metricas_cap["retorno_anualizado"], metricas_euf["retorno_anualizado"], maior_melhor=True),
        "vencedor_volatilidade_anualizada": classificar_vencedor(metricas_cap["volatilidade_anualizada"], metricas_euf["volatilidade_anualizada"], maior_melhor=False),
        "vencedor_sharpe_simplificado": classificar_vencedor(metricas_cap["sharpe_simplificado"], metricas_euf["sharpe_simplificado"], maior_melhor=True),
        "vencedor_sortino_simplificado": classificar_vencedor(metricas_cap["sortino_simplificado"], metricas_euf["sortino_simplificado"], maior_melhor=True),
        "vencedor_calmar_simplificado": classificar_vencedor(metricas_cap["calmar_simplificado"], metricas_euf["calmar_simplificado"], maior_melhor=True),
        "vencedor_max_drawdown": classificar_vencedor(abs(metricas_cap["max_drawdown"]), abs(metricas_euf["max_drawdown"]), maior_melhor=False),
        "vencedor_frequencia_superacao": "capitulacao" if float(df_freq["flag_capitulacao_superou_euforia"].mean()) > float(df_freq["flag_euforia_superou_capitulacao"].mean()) else "euforia" if float(df_freq["flag_euforia_superou_capitulacao"].mean()) > float(df_freq["flag_capitulacao_superou_euforia"].mean()) else "empate_tecnico",
    })

df_resumo = pd.DataFrame(registros_resumo).sort_values("ordem_frequencia").reset_index(drop=True)

print(f"Resumo comparativo por frequência       : {len(df_resumo):,} linhas")
print("OK")

# ============================================================
# 8) Construção do ranking comparativo
# ============================================================

print("\n[8/11] Construção do ranking comparativo...")

METRICAS_RANKING = [
    ("retorno_acumulado", "Retorno acumulado", True),
    ("retorno_anualizado", "Retorno anualizado", True),
    ("volatilidade_anualizada", "Volatilidade anualizada", False),
    ("downside_vol_anualizada", "Downside volatility anualizada", False),
    ("sharpe_simplificado", "Sharpe simplificado", True),
    ("sortino_simplificado", "Sortino simplificado", True),
    ("calmar_simplificado", "Calmar simplificado", True),
    ("max_drawdown_abs", "Drawdown máximo absoluto", False),
]

registros_ranking = []

for _, linha in df_resumo.iterrows():
    frequencia = linha["frequencia"]
    for metrica, nome_metrica, maior_melhor in METRICAS_RANKING:
        if metrica == "max_drawdown_abs":
            valor_cap = abs(linha["max_drawdown_capitulacao"])
            valor_euf = abs(linha["max_drawdown_euforia"])
        else:
            valor_cap = linha.get(f"{metrica}_capitulacao", np.nan)
            valor_euf = linha.get(f"{metrica}_euforia", np.nan)

        vencedor = classificar_vencedor(valor_cap, valor_euf, maior_melhor=maior_melhor)

        registros_ranking.append({
            "frequencia": frequencia,
            "visao_comparacao": linha["visao_comparacao"],
            "aplicacao_metodologica": linha["aplicacao_metodologica"],
            "ordem_frequencia": linha["ordem_frequencia"],
            "metrica": metrica,
            "nome_metrica": nome_metrica,
            "criterio": "maior_melhor" if maior_melhor else "menor_melhor",
            "valor_capitulacao": valor_cap,
            "valor_euforia": valor_euf,
            "diferenca_capitulacao_menos_euforia": valor_cap - valor_euf if pd.notna(valor_cap) and pd.notna(valor_euf) else np.nan,
            "vencedor": vencedor,
        })

df_ranking = pd.DataFrame(registros_ranking).sort_values(["ordem_frequencia", "metrica"]).reset_index(drop=True)

print(f"Ranking comparativo                     : {len(df_ranking):,} linhas")
print("OK")

# ============================================================
# 9) Construção da base de gráficos
# ============================================================

print("\n[9/11] Construção da base de gráficos...")

registros_grafico = []

for frequencia, df_freq in bases_pareadas.items():
    df_curvas = df_freq[[
        "frequencia",
        "visao_comparacao",
        "aplicacao_metodologica",
        "ordem_frequencia",
        "periodo_comparacao",
        "data_fim_periodo",
        "patrimonio_relativo_capitulacao",
        "patrimonio_relativo_euforia",
        "excesso_capitulacao_vs_euforia",
        "retorno_periodo_capitulacao",
        "retorno_periodo_euforia",
    ]].copy()

    df_long_curvas = df_curvas.melt(
        id_vars=[
            "frequencia",
            "visao_comparacao",
            "aplicacao_metodologica",
            "ordem_frequencia",
            "periodo_comparacao",
            "data_fim_periodo",
        ],
        value_vars=[
            "patrimonio_relativo_capitulacao",
            "patrimonio_relativo_euforia",
        ],
        var_name="metrica_origem",
        value_name="valor",
    )
    df_long_curvas["tipo_grafico"] = "curva_patrimonial_relativa"
    df_long_curvas["estrategia"] = np.where(
        df_long_curvas["metrica_origem"].str.contains("capitulacao", case=False, na=False),
        "capitulacao",
        "euforia",
    )
    df_long_curvas["metrica"] = "patrimonio_relativo"
    registros_grafico.append(df_long_curvas)

    df_excesso = df_curvas[[
        "frequencia",
        "visao_comparacao",
        "aplicacao_metodologica",
        "ordem_frequencia",
        "periodo_comparacao",
        "data_fim_periodo",
        "excesso_capitulacao_vs_euforia",
    ]].copy()
    df_excesso = df_excesso.rename(columns={"excesso_capitulacao_vs_euforia": "valor"})
    df_excesso["tipo_grafico"] = "excesso_capitulacao_vs_euforia"
    df_excesso["estrategia"] = "capitulacao_menos_euforia"
    df_excesso["metrica"] = "excesso_retorno_periodico"
    df_excesso["metrica_origem"] = "excesso_capitulacao_vs_euforia"
    registros_grafico.append(df_excesso)

for _, linha in df_ranking.iterrows():
    registros_grafico.append(pd.DataFrame([
        {
            "frequencia": linha["frequencia"],
            "visao_comparacao": linha["visao_comparacao"],
            "aplicacao_metodologica": linha["aplicacao_metodologica"],
            "ordem_frequencia": linha["ordem_frequencia"],
            "periodo_comparacao": "resumo",
            "data_fim_periodo": pd.NaT,
            "tipo_grafico": "ranking_metricas",
            "estrategia": "capitulacao",
            "metrica": linha["metrica"],
            "metrica_origem": linha["nome_metrica"],
            "valor": linha["valor_capitulacao"],
            "vencedor": linha["vencedor"],
        },
        {
            "frequencia": linha["frequencia"],
            "visao_comparacao": linha["visao_comparacao"],
            "aplicacao_metodologica": linha["aplicacao_metodologica"],
            "ordem_frequencia": linha["ordem_frequencia"],
            "periodo_comparacao": "resumo",
            "data_fim_periodo": pd.NaT,
            "tipo_grafico": "ranking_metricas",
            "estrategia": "euforia",
            "metrica": linha["metrica"],
            "metrica_origem": linha["nome_metrica"],
            "valor": linha["valor_euforia"],
            "vencedor": linha["vencedor"],
        },
    ]))

if len(registros_grafico) > 0:
    df_base_grafico = pd.concat(registros_grafico, ignore_index=True)
else:
    df_base_grafico = pd.DataFrame()

print(f"Base de gráficos                        : {len(df_base_grafico):,} linhas")
print("OK")

# ============================================================
# 10) Parâmetros e auditoria de validação
# ============================================================

print("\n[10/11] Parâmetros e auditoria de validação...")

PARAMETROS_11_7 = {
    "etapa": "11",
    "subetapa": "11.7",
    "nome_subetapa": "Comparação Direta entre Capitulação e Euforia",
    "fonte_principal": "bases de comparação contra benchmarks da Etapa 11.5",
    "estrategia_base_diferenca": "capitulacao",
    "estrategia_comparada": "euforia",
    "diferenca_principal": "excesso_capitulacao_vs_euforia",
    "interpretacao_diferenca_positiva": "Capitulação superou Euforia no período",
    "interpretacao_diferenca_negativa": "Euforia superou Capitulação no período",
    "frequencias_preservadas": "diaria, mensal e anual",
    "observacao_metricas_risco": "Sharpe, Sortino e Calmar são calculados de forma simplificada a partir das séries pareadas de retorno da própria subetapa.",
}

df_parametros = pd.DataFrame([{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_11_7.items()])

df_auditoria_interna = pd.DataFrame(registros_auditoria_interna)

linhas_diaria = len(df_base_diaria)
linhas_mensal = len(df_base_mensal)
linhas_anual = len(df_base_anual)
frequencias_identificadas = int(df_base_consolidada["frequencia"].nunique()) if len(df_base_consolidada) > 0 else 0
periodos_duplicados = int(df_base_consolidada.duplicated(["frequencia", "periodo_comparacao"]).sum()) if len(df_base_consolidada) > 0 else 0
retornos_ausentes = int(df_base_consolidada[["retorno_periodo_capitulacao", "retorno_periodo_euforia", "excesso_capitulacao_vs_euforia"]].isna().sum().sum()) if len(df_base_consolidada) > 0 else 0
duplicatas_origem_cap = int(df_auditoria_interna["duplicatas_capitulacao"].sum()) if len(df_auditoria_interna) > 0 else 0
duplicatas_origem_euf = int(df_auditoria_interna["duplicatas_euforia"].sum()) if len(df_auditoria_interna) > 0 else 0
ranking_vazio = int(len(df_ranking) == 0)

metricas_infinitas = 0
for df_verificacao in [df_base_diaria, df_base_mensal, df_base_anual, df_base_consolidada, df_resumo, df_ranking, df_base_grafico]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

linhas_auditoria = [
    {
        "item": "linhas_base_diaria",
        "valor": linhas_diaria,
        "valor_referencia": "> 0",
        "status": "OK" if linhas_diaria > 0 else "ERRO",
        "observacao": "A comparação direta deve preservar a visão diária.",
    },
    {
        "item": "linhas_base_mensal",
        "valor": linhas_mensal,
        "valor_referencia": "> 0",
        "status": "OK" if linhas_mensal > 0 else "ERRO",
        "observacao": "A comparação direta deve preservar a visão mensal.",
    },
    {
        "item": "linhas_base_anual",
        "valor": linhas_anual,
        "valor_referencia": "> 0",
        "status": "OK" if linhas_anual > 0 else "ERRO",
        "observacao": "A comparação direta deve preservar a visão anual.",
    },
    {
        "item": "frequencias_identificadas",
        "valor": frequencias_identificadas,
        "valor_referencia": ">= 3",
        "status": "OK" if frequencias_identificadas >= 3 else "ERRO",
        "observacao": "Devem existir as visões diária, mensal e anual.",
    },
    {
        "item": "periodos_duplicados_base_consolidada",
        "valor": periodos_duplicados,
        "valor_referencia": "0",
        "status": "OK" if periodos_duplicados == 0 else "ERRO",
        "observacao": "Cada frequência deve ter no máximo uma comparação por período.",
    },
    {
        "item": "retornos_ausentes_base_consolidada",
        "valor": retornos_ausentes,
        "valor_referencia": "0",
        "status": "OK" if retornos_ausentes == 0 else "ERRO",
        "observacao": "A base pareada não deve conter retornos ausentes nas colunas centrais.",
    },
    {
        "item": "duplicatas_origem_capitulacao",
        "valor": duplicatas_origem_cap,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_origem_cap == 0 else "ERRO",
        "observacao": "A origem da Capitulação não deve ter duplicatas por período.",
    },
    {
        "item": "duplicatas_origem_euforia",
        "valor": duplicatas_origem_euf,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_origem_euf == 0 else "ERRO",
        "observacao": "A origem da Euforia não deve ter duplicatas por período.",
    },
    {
        "item": "linhas_resumo_comparativo",
        "valor": len(df_resumo),
        "valor_referencia": ">= 3",
        "status": "OK" if len(df_resumo) >= 3 else "ERRO",
        "observacao": "O resumo deve ter uma linha por frequência.",
    },
    {
        "item": "ranking_comparativo_vazio",
        "valor": ranking_vazio,
        "valor_referencia": "0",
        "status": "OK" if ranking_vazio == 0 else "ERRO",
        "observacao": "O ranking comparativo deve ser gerado para o HTML final.",
    },
    {
        "item": "base_grafico_linhas",
        "valor": len(df_base_grafico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Parâmetros metodológicos                : {len(df_parametros):,} linhas")
print(f"Itens de auditoria                      : {len(df_auditoria):,}")
print(f"Erros bloqueantes                       : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 11) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[11/11] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(padronizar_dataframe_parquet(df_base_diaria), CAMINHO_BASE_DIARIA, index=False)
salvar_dataframe(padronizar_dataframe_parquet(df_base_mensal), CAMINHO_BASE_MENSAL, index=False)
salvar_dataframe(padronizar_dataframe_parquet(df_base_anual), CAMINHO_BASE_ANUAL, index=False)
salvar_dataframe(padronizar_dataframe_parquet(df_base_consolidada), CAMINHO_BASE_CONSOLIDADA, index=False)
salvar_dataframe(padronizar_dataframe_parquet(df_resumo), CAMINHO_RESUMO, index=False)
salvar_dataframe(padronizar_dataframe_parquet(df_ranking), CAMINHO_RANKING, index=False)
salvar_dataframe(padronizar_dataframe_parquet(df_base_grafico), CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe(padronizar_dataframe_parquet(df_parametros), CAMINHO_PARAMETROS, index=False)
salvar_dataframe(padronizar_dataframe_parquet(df_auditoria), CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da comparação direta entre Capitulação e Euforia:")
print(df_auditoria.to_string(index=False))

print("\nResumo comparativo Capitulação vs Euforia:")
colunas_resumo_print = [
    "frequencia",
    "n_periodos",
    "retorno_acumulado_capitulacao",
    "retorno_acumulado_euforia",
    "retorno_relativo_acumulado_capitulacao_vs_euforia",
    "retorno_anualizado_capitulacao",
    "retorno_anualizado_euforia",
    "volatilidade_anualizada_capitulacao",
    "volatilidade_anualizada_euforia",
    "sharpe_simplificado_capitulacao",
    "sharpe_simplificado_euforia",
    "sortino_simplificado_capitulacao",
    "sortino_simplificado_euforia",
    "calmar_simplificado_capitulacao",
    "calmar_simplificado_euforia",
    "pct_periodos_capitulacao_superou_euforia",
    "pct_periodos_euforia_superou_capitulacao",
    "vencedor_retorno_acumulado",
    "vencedor_sharpe_simplificado",
    "vencedor_calmar_simplificado",
]
print(df_resumo[colunas_resumo_print].to_string(index=False))

print("\nRanking comparativo - amostra:")
print(df_ranking.head(50).to_string(index=False))

print("\nArquivos salvos na subetapa 11.7:")
print(f"- {CAMINHO_BASE_DIARIA}")
print(f"- {CAMINHO_BASE_MENSAL}")
print(f"- {CAMINHO_BASE_ANUAL}")
print(f"- {CAMINHO_BASE_CONSOLIDADA}")
print(f"- {CAMINHO_RESUMO}")
print(f"- {CAMINHO_RANKING}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 11.7 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 11.7 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 11.7 - COMPARAÇÃO DIRETA ENTRE CAPITULAÇÃO E EUFORIA

[1/11] Validação inicial do ambiente...
OK

[2/11] Definição determinística dos caminhos de entrada e saída...
Entrada - comparação benchmarks diária da 11.5 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_5_base_comparacao_benchmarks_diaria.parquet
Entrada - comparação benchmarks mensal da 11.5 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_5_base_comparacao_benchmarks_mensal.parquet
Entrada - comparação benchmarks anual da 11.5  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_5_base_comparacao_benchmarks_anual.parquet
Saída   - base diária cap. vs euforia          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_eufori

# Etapa 12) Análise dos Sinais e da Composição das Carteiras

## Etapa 12.1) Frequência e Distribuição Temporal dos Sinais

In [59]:
%%time
# ============================================================
# Etapa 12.1) Frequência e Distribuição Temporal dos Sinais
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 12.1 - FREQUÊNCIA E DISTRIBUIÇÃO TEMPORAL DOS SINAIS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

caminho_base_calendario_final_sinais = gerar_caminho_arquivo(
    etapa=6,
    subetapa=4,
    tipo_arquivo="base",
    nome="calendario_final_sinais",
)

caminho_base_aportes_realizados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="base",
    nome="aportes_realizados",
)

caminho_base_retornos_diarios = gerar_caminho_arquivo(
    etapa=11,
    subetapa=1,
    tipo_arquivo="base",
    nome="retornos_diarios",
)

caminho_base_sinais_temporal = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="base",
    nome="sinais_temporal",
)

caminho_base_regimes_ibovespa = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="base",
    nome="regimes_ibovespa",
)

caminho_tbl_distribuicao_anual_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="distribuicao_anual_sinais",
)

caminho_tbl_distribuicao_mensal_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="distribuicao_mensal_sinais",
)

caminho_tbl_distribuicao_mes_calendario_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="distribuicao_mes_calendario_sinais",
)

caminho_tbl_distribuicao_regime_mercado_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="distribuicao_regime_mercado_sinais",
)

caminho_tbl_intervalos_entre_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="intervalos_entre_sinais",
)

caminho_tbl_base_grafico_timeline_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="base_grafico_timeline_sinais",
)

caminho_tbl_base_grafico_distribuicao_anual_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="base_grafico_distribuicao_anual_sinais",
)

caminho_tbl_auditoria_validacao_frequencia_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_frequencia_sinais",
)

print(f"Entrada - calendário final de sinais da 6.4 : {caminho_base_calendario_final_sinais}")
print(f"Entrada - aportes realizados da 10.1        : {caminho_base_aportes_realizados}")
print(f"Entrada - retornos diários da 11.1          : {caminho_base_retornos_diarios}")
print(f"Saída   - base temporal de sinais           : {caminho_base_sinais_temporal}")
print(f"Saída   - base de regimes do Ibovespa       : {caminho_base_regimes_ibovespa}")
print(f"Saída   - distribuição anual                : {caminho_tbl_distribuicao_anual_sinais}")
print(f"Saída   - distribuição mensal               : {caminho_tbl_distribuicao_mensal_sinais}")
print(f"Saída   - distribuição por mês calendário   : {caminho_tbl_distribuicao_mes_calendario_sinais}")
print(f"Saída   - distribuição por regime de mercado: {caminho_tbl_distribuicao_regime_mercado_sinais}")
print(f"Saída   - intervalos entre sinais           : {caminho_tbl_intervalos_entre_sinais}")
print(f"Saída   - base gráfico timeline             : {caminho_tbl_base_grafico_timeline_sinais}")
print(f"Saída   - base gráfico anual                : {caminho_tbl_base_grafico_distribuicao_anual_sinais}")
print(f"Saída   - auditoria de validação            : {caminho_tbl_auditoria_validacao_frequencia_sinais}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais de sinais, aportes e retornos
# ============================================================

print("\n[3/12] Carga das bases oficiais de sinais, aportes e retornos...")

df_calendario_sinais = pd.read_parquet(caminho_base_calendario_final_sinais)
df_aportes_realizados = pd.read_parquet(caminho_base_aportes_realizados)
df_retornos_diarios = pd.read_parquet(caminho_base_retornos_diarios)

print(f"Calendário final de sinais               : {df_calendario_sinais.shape[0]:,} linhas x {df_calendario_sinais.shape[1]} colunas")
print(f"Aportes realizados                       : {df_aportes_realizados.shape[0]:,} linhas x {df_aportes_realizados.shape[1]} colunas")
print(f"Retornos diários                         : {df_retornos_diarios.shape[0]:,} linhas x {df_retornos_diarios.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

def selecionar_primeira_coluna_existente(df, candidatos, nome_logico, obrigatoria=True):
    colunas_existentes = [coluna for coluna in candidatos if coluna in df.columns]
    if len(colunas_existentes) == 0:
        if obrigatoria:
            raise ValueError(
                f"Não foi possível identificar a coluna lógica '{nome_logico}'. "
                f"Candidatas avaliadas: {candidatos}. Colunas disponíveis: {list(df.columns)}"
            )
        return None
    return colunas_existentes[0]


def classificar_referencia_sinal(valor):
    texto = str(valor).strip().lower()
    if "capitul" in texto:
        return "capitulacao"
    if "euforia" in texto:
        return "euforia"
    return pd.NA


def extrair_primeiro_valor_nao_nulo(grupo, coluna):
    if coluna not in grupo.columns:
        return pd.NA
    serie = grupo[coluna].dropna()
    if len(serie) == 0:
        return pd.NA
    return serie.iloc[0]


def calcular_intervalo_dias_uteis(data_atual, data_anterior):
    if pd.isna(data_atual) or pd.isna(data_anterior):
        return np.nan
    data_atual_np = np.datetime64(pd.Timestamp(data_atual).date())
    data_anterior_np = np.datetime64(pd.Timestamp(data_anterior).date())
    if data_atual_np < data_anterior_np:
        return np.nan
    return int(np.busday_count(data_anterior_np, data_atual_np))


def produto_composto(serie_retornos):
    retornos = pd.to_numeric(serie_retornos, errors="coerce").dropna()
    if len(retornos) == 0:
        return np.nan
    return float((1.0 + retornos).prod() - 1.0)


def calcular_pct(parte, total):
    if pd.isna(parte) or pd.isna(total) or total == 0:
        return np.nan
    return float(parte / total)

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização da base de sinais finais
# ============================================================

print("\n[5/12] Padronização da base de sinais finais...")

df_sinais = df_calendario_sinais.copy()

coluna_data_sinal = selecionar_primeira_coluna_existente(
    df_sinais,
    ["data_sinal", "data_sinal_final", "data", "data_evento", "data_referencia"],
    "data_sinal",
    obrigatoria=True,
)

coluna_tipo_sinal = selecionar_primeira_coluna_existente(
    df_sinais,
    ["tipo_sinal", "sinal", "estrategia_referencia", "estrategia", "tipo_estrategia", "regra_sinal"],
    "tipo_sinal",
    obrigatoria=True,
)

coluna_ordem_sinal = selecionar_primeira_coluna_existente(
    df_sinais,
    ["ordem_sinal_final", "ordem_sinal", "numero_sinal", "id_sinal", "sequencia_sinal"],
    "ordem_sinal",
    obrigatoria=False,
)

coluna_score_sinal = selecionar_primeira_coluna_existente(
    df_sinais,
    ["score_sinal", "score", "intensidade_sinal", "intensidade", "score_final"],
    "score_sinal",
    obrigatoria=False,
)

renomeacoes_sinais = {
    coluna_data_sinal: "data_sinal",
    coluna_tipo_sinal: "tipo_sinal_origem",
}

if coluna_ordem_sinal is not None:
    renomeacoes_sinais[coluna_ordem_sinal] = "ordem_sinal_origem"
if coluna_score_sinal is not None:
    renomeacoes_sinais[coluna_score_sinal] = "score_sinal_origem"

df_sinais = df_sinais.rename(columns=renomeacoes_sinais).copy()
df_sinais["data_sinal"] = pd.to_datetime(df_sinais["data_sinal"], errors="coerce")
df_sinais["estrategia_referencia"] = df_sinais["tipo_sinal_origem"].apply(classificar_referencia_sinal)

df_sinais = df_sinais.loc[df_sinais["estrategia_referencia"].isin(["capitulacao", "euforia"])].copy()

df_sinais["nome_exibicao_sinal"] = np.select(
    [
        df_sinais["estrategia_referencia"].eq("capitulacao"),
        df_sinais["estrategia_referencia"].eq("euforia"),
    ],
    ["Capitulação", "Euforia"],
    default="Não classificado",
)

df_sinais = df_sinais.sort_values(["estrategia_referencia", "data_sinal"]).reset_index(drop=True)

if "ordem_sinal_origem" in df_sinais.columns:
    df_sinais["ordem_sinal"] = pd.to_numeric(df_sinais["ordem_sinal_origem"], errors="coerce")
else:
    df_sinais["ordem_sinal"] = df_sinais.groupby("estrategia_referencia").cumcount() + 1

if "score_sinal_origem" in df_sinais.columns:
    df_sinais["score_sinal"] = pd.to_numeric(df_sinais["score_sinal_origem"], errors="coerce")
else:
    df_sinais["score_sinal"] = np.nan

df_sinais["ano_sinal"] = df_sinais["data_sinal"].dt.year.astype("Int64")
df_sinais["mes_sinal"] = df_sinais["data_sinal"].dt.month.astype("Int64")
df_sinais["ano_mes_sinal"] = df_sinais["data_sinal"].dt.to_period("M").astype(str)
df_sinais["trimestre_sinal"] = df_sinais["data_sinal"].dt.to_period("Q").astype(str)
df_sinais["semestre_sinal"] = np.where(
    df_sinais["mes_sinal"].astype("Int64") <= 6,
    df_sinais["ano_sinal"].astype(str) + "S1",
    df_sinais["ano_sinal"].astype(str) + "S2",
)
df_sinais["dia_semana_sinal"] = df_sinais["data_sinal"].dt.day_name()
df_sinais["dia_mes_sinal"] = df_sinais["data_sinal"].dt.day.astype("Int64")

df_sinais["data_sinal_anterior"] = df_sinais.groupby("estrategia_referencia")["data_sinal"].shift(1)
df_sinais["intervalo_dias_corridos_desde_sinal_anterior"] = (
    df_sinais["data_sinal"] - df_sinais["data_sinal_anterior"]
).dt.days

df_sinais["intervalo_dias_uteis_aproximado_desde_sinal_anterior"] = [
    calcular_intervalo_dias_uteis(data_atual, data_anterior)
    for data_atual, data_anterior in zip(df_sinais["data_sinal"], df_sinais["data_sinal_anterior"])
]

datas_invalidas_sinais = int(df_sinais["data_sinal"].isna().sum())
referencias_invalidas_sinais = int(df_sinais["estrategia_referencia"].isna().sum())
duplicatas_sinais = int(df_sinais.duplicated(["estrategia_referencia", "data_sinal"]).sum())

if datas_invalidas_sinais > 0:
    raise ValueError("Há datas inválidas na base final de sinais após a padronização.")
if referencias_invalidas_sinais > 0:
    raise ValueError("Há sinais sem estratégia de referência classificada após a padronização.")
if duplicatas_sinais > 0:
    raise ValueError("Há duplicidades por estratégia de referência e data de sinal na base final de sinais.")

print(f"Coluna original de data do sinal         : {coluna_data_sinal}")
print(f"Coluna original de tipo do sinal         : {coluna_tipo_sinal}")
print(f"Coluna original de ordem do sinal        : {coluna_ordem_sinal}")
print(f"Coluna original de score/intensidade     : {coluna_score_sinal}")
print(f"Sinais finais padronizados               : {len(df_sinais):,}")
print(f"Estratégias de sinal                     : {', '.join(sorted(df_sinais['estrategia_referencia'].dropna().unique()))}")
print(f"Duplicidades por estratégia e data       : {duplicatas_sinais:,}")
print("OK")

# ============================================================
# 6) Integração dos aportes realizados às datas de sinal
# ============================================================

print("\n[6/12] Integração dos aportes realizados às datas de sinal...")

df_aportes = df_aportes_realizados.copy()

coluna_data_sinal_aportes = selecionar_primeira_coluna_existente(
    df_aportes,
    ["data_sinal", "data_sinal_referencia", "data_evento", "data_referencia"],
    "data_sinal_aportes",
    obrigatoria=False,
)

coluna_data_aporte = selecionar_primeira_coluna_existente(
    df_aportes,
    ["data_aporte", "data_aporte_efetiva", "data_execucao", "data_operacional", "data_compra"],
    "data_aporte",
    obrigatoria=False,
)

coluna_estrategia_aportes = selecionar_primeira_coluna_existente(
    df_aportes,
    ["estrategia_referencia", "estrategia_referencia_padronizada", "tipo_sinal", "tipo_estrategia", "estrategia_id", "chave_estrategia"],
    "estrategia_referencia_aportes",
    obrigatoria=False,
)

coluna_valor_aporte = selecionar_primeira_coluna_existente(
    df_aportes,
    ["valor_aporte", "valor_aporte_executado", "valor_alocado", "valor_investido", "orcamento_aporte"],
    "valor_aporte",
    obrigatoria=False,
)

if coluna_data_sinal_aportes is not None and coluna_data_aporte is not None and coluna_estrategia_aportes is not None:
    df_aportes = df_aportes.rename(
        columns={
            coluna_data_sinal_aportes: "data_sinal",
            coluna_data_aporte: "data_aporte_realizada",
            coluna_estrategia_aportes: "estrategia_referencia_aporte_origem",
        }
    ).copy()

    df_aportes["data_sinal"] = pd.to_datetime(df_aportes["data_sinal"], errors="coerce")
    df_aportes["data_aporte_realizada"] = pd.to_datetime(df_aportes["data_aporte_realizada"], errors="coerce")
    df_aportes["estrategia_referencia"] = df_aportes["estrategia_referencia_aporte_origem"].apply(classificar_referencia_sinal)

    if coluna_valor_aporte is not None:
        if coluna_valor_aporte != "valor_aporte":
            df_aportes = df_aportes.rename(columns={coluna_valor_aporte: "valor_aporte"}).copy()
        df_aportes["valor_aporte"] = pd.to_numeric(df_aportes["valor_aporte"], errors="coerce")
    else:
        df_aportes["valor_aporte"] = np.nan

    df_aportes_sinais = (
        df_aportes.loc[df_aportes["estrategia_referencia"].isin(["capitulacao", "euforia"])]
        .groupby(["estrategia_referencia", "data_sinal"], dropna=False)
        .agg(
            n_aportes_realizados=("data_aporte_realizada", "count"),
            primeira_data_aporte_realizada=("data_aporte_realizada", "min"),
            ultima_data_aporte_realizada=("data_aporte_realizada", "max"),
            valor_aporte_total_realizado=("valor_aporte", "sum"),
            valor_aporte_medio_realizado=("valor_aporte", "mean"),
        )
        .reset_index()
    )

    linhas_antes_merge_aportes = len(df_sinais)
    df_sinais = df_sinais.merge(
        df_aportes_sinais,
        on=["estrategia_referencia", "data_sinal"],
        how="left",
    )
    linhas_depois_merge_aportes = len(df_sinais)

    if linhas_depois_merge_aportes != linhas_antes_merge_aportes:
        raise ValueError("A integração dos aportes realizados expandiu a base de sinais.")

    df_sinais["n_aportes_realizados"] = df_sinais["n_aportes_realizados"].fillna(0).astype(int)
else:
    df_sinais["n_aportes_realizados"] = 0
    df_sinais["primeira_data_aporte_realizada"] = pd.NaT
    df_sinais["ultima_data_aporte_realizada"] = pd.NaT
    df_sinais["valor_aporte_total_realizado"] = np.nan
    df_sinais["valor_aporte_medio_realizado"] = np.nan

sinais_com_aporte = int((df_sinais["n_aportes_realizados"] > 0).sum())

print(f"Coluna de data do sinal nos aportes       : {coluna_data_sinal_aportes}")
print(f"Coluna de data efetiva do aporte          : {coluna_data_aporte}")
print(f"Coluna de estratégia nos aportes          : {coluna_estrategia_aportes}")
print(f"Coluna de valor do aporte                 : {coluna_valor_aporte}")
print(f"Sinais com aporte realizado associado     : {sinais_com_aporte:,}")
print("OK")

# ============================================================
# 7) Construção da base de regimes de mercado do Ibovespa
# ============================================================

print("\n[7/12] Construção da base de regimes de mercado do Ibovespa...")

df_retornos = df_retornos_diarios.copy()

colunas_obrigatorias_retornos = ["data", "chave_estrategia", "retorno_diario", "patrimonio_total"]
colunas_ausentes_retornos = [coluna for coluna in colunas_obrigatorias_retornos if coluna not in df_retornos.columns]
if len(colunas_ausentes_retornos) > 0:
    raise ValueError(f"Colunas ausentes na base de retornos diários da 11.1: {colunas_ausentes_retornos}")

df_retornos["data"] = pd.to_datetime(df_retornos["data"], errors="coerce")
df_retornos["retorno_diario"] = pd.to_numeric(df_retornos["retorno_diario"], errors="coerce")
df_retornos["patrimonio_total"] = pd.to_numeric(df_retornos["patrimonio_total"], errors="coerce")

chave_ibovespa_oficial = "ibovespa_buy_and_hold__carteira_na__replica_na"

if chave_ibovespa_oficial in df_retornos["chave_estrategia"].astype(str).unique().tolist():
    df_ibovespa = df_retornos.loc[df_retornos["chave_estrategia"].astype(str).eq(chave_ibovespa_oficial)].copy()
else:
    if "nome_exibicao_estrategia" not in df_retornos.columns:
        raise ValueError("Não foi possível localizar o Ibovespa oficial por chave e a coluna nome_exibicao_estrategia não existe.")
    mascara_ibovespa = df_retornos["nome_exibicao_estrategia"].astype(str).str.lower().str.contains("ibovespa buy and hold", regex=False, na=False)
    chaves_ibovespa = sorted(df_retornos.loc[mascara_ibovespa, "chave_estrategia"].dropna().astype(str).unique().tolist())
    if len(chaves_ibovespa) != 1:
        raise ValueError(f"Não foi possível identificar uma única chave para o Ibovespa oficial. Chaves encontradas: {chaves_ibovespa}")
    chave_ibovespa_oficial = chaves_ibovespa[0]
    df_ibovespa = df_retornos.loc[df_retornos["chave_estrategia"].astype(str).eq(chave_ibovespa_oficial)].copy()

df_ibovespa = df_ibovespa.sort_values("data").reset_index(drop=True)
df_ibovespa["patrimonio_pico_acumulado"] = df_ibovespa["patrimonio_total"].cummax()
df_ibovespa["drawdown_ibovespa"] = df_ibovespa["patrimonio_total"] / df_ibovespa["patrimonio_pico_acumulado"] - 1.0
df_ibovespa["retorno_ibovespa_63p"] = df_ibovespa["patrimonio_total"] / df_ibovespa["patrimonio_total"].shift(63) - 1.0
df_ibovespa["retorno_ibovespa_126p"] = df_ibovespa["patrimonio_total"] / df_ibovespa["patrimonio_total"].shift(126) - 1.0
df_ibovespa["retorno_ibovespa_252p"] = df_ibovespa["patrimonio_total"] / df_ibovespa["patrimonio_total"].shift(252) - 1.0
df_ibovespa["volatilidade_ibovespa_63p_anualizada"] = df_ibovespa["retorno_diario"].rolling(63, min_periods=21).std() * np.sqrt(252)

def classificar_regime_mercado(linha):
    retorno_252 = linha["retorno_ibovespa_252p"]
    retorno_126 = linha["retorno_ibovespa_126p"]
    drawdown = linha["drawdown_ibovespa"]

    if pd.notna(drawdown) and drawdown <= -0.20:
        return "bear_market_drawdown_maior_20pct"
    if pd.notna(retorno_252) and retorno_252 >= 0.20 and (pd.isna(drawdown) or drawdown > -0.10):
        return "alta_12m_maior_20pct"
    if pd.notna(retorno_252) and retorno_252 <= -0.10:
        return "queda_12m_maior_10pct"
    if pd.notna(retorno_126) and retorno_126 >= 0.10 and (pd.isna(drawdown) or drawdown > -0.10):
        return "alta_6m_maior_10pct"
    if pd.notna(retorno_126) and retorno_126 <= -0.10:
        return "queda_6m_maior_10pct"
    if pd.isna(retorno_252):
        return "aquecimento_sem_252_pregoes"
    return "lateral_ou_indefinido"


df_ibovespa["regime_mercado"] = df_ibovespa.apply(classificar_regime_mercado, axis=1)

df_base_regimes_ibovespa = df_ibovespa[
    [
        "data",
        "chave_estrategia",
        "patrimonio_total",
        "retorno_diario",
        "retorno_ibovespa_63p",
        "retorno_ibovespa_126p",
        "retorno_ibovespa_252p",
        "drawdown_ibovespa",
        "volatilidade_ibovespa_63p_anualizada",
        "regime_mercado",
    ]
].copy()

df_base_regimes_ibovespa = df_base_regimes_ibovespa.rename(
    columns={
        "chave_estrategia": "chave_benchmark_regime",
        "patrimonio_total": "patrimonio_ibovespa",
        "retorno_diario": "retorno_diario_ibovespa",
    }
)

print(f"Chave do Ibovespa usada para regimes      : {chave_ibovespa_oficial}")
print(f"Base de regimes do Ibovespa               : {len(df_base_regimes_ibovespa):,} linhas")
print(f"Regimes identificados                     : {', '.join(sorted(df_base_regimes_ibovespa['regime_mercado'].dropna().unique()))}")
print("OK")

# ============================================================
# 8) Integração dos regimes de mercado à base de sinais
# ============================================================

print("\n[8/12] Integração dos regimes de mercado à base de sinais...")

df_sinais = df_sinais.sort_values("data_sinal").reset_index(drop=True)
df_regimes_merge = df_base_regimes_ibovespa.sort_values("data").reset_index(drop=True)

linhas_antes_merge_regimes = len(df_sinais)

df_sinais = pd.merge_asof(
    df_sinais,
    df_regimes_merge,
    left_on="data_sinal",
    right_on="data",
    direction="backward",
)

df_sinais = df_sinais.rename(columns={"data": "data_regime_mercado"})
linhas_depois_merge_regimes = len(df_sinais)

if linhas_depois_merge_regimes != linhas_antes_merge_regimes:
    raise ValueError("A integração dos regimes de mercado expandiu a base de sinais.")

df_sinais["defasagem_dias_regime_mercado"] = (df_sinais["data_sinal"] - df_sinais["data_regime_mercado"]).dt.days
regimes_ausentes_sinais = int(df_sinais["regime_mercado"].isna().sum())

print(f"Linhas antes do merge de regimes          : {linhas_antes_merge_regimes:,}")
print(f"Linhas após o merge de regimes            : {linhas_depois_merge_regimes:,}")
print(f"Sinais sem regime de mercado associado    : {regimes_ausentes_sinais:,}")
print("OK")

# ============================================================
# 9) Construção das distribuições temporais dos sinais
# ============================================================

print("\n[9/12] Construção das distribuições temporais dos sinais...")

total_sinais_por_estrategia = (
    df_sinais.groupby("estrategia_referencia", dropna=False)
    .size()
    .rename("total_sinais_estrategia")
    .reset_index()
)

anos_observados = sorted(df_sinais["ano_sinal"].dropna().astype(int).unique().tolist())
estrategias_observadas = sorted(df_sinais["estrategia_referencia"].dropna().astype(str).unique().tolist())

idx_anual = pd.MultiIndex.from_product(
    [estrategias_observadas, anos_observados],
    names=["estrategia_referencia", "ano_sinal"],
).to_frame(index=False)

df_tbl_distribuicao_anual_sinais = (
    df_sinais.groupby(["estrategia_referencia", "ano_sinal"], dropna=False)
    .agg(
        n_sinais=("data_sinal", "count"),
        primeira_data_sinal=("data_sinal", "min"),
        ultima_data_sinal=("data_sinal", "max"),
        n_sinais_com_aporte_realizado=("n_aportes_realizados", lambda x: int((pd.to_numeric(x, errors="coerce") > 0).sum())),
        valor_aporte_total_realizado=("valor_aporte_total_realizado", "sum"),
        score_sinal_medio=("score_sinal", "mean"),
        score_sinal_maximo=("score_sinal", "max"),
    )
    .reset_index()
)

df_tbl_distribuicao_anual_sinais = idx_anual.merge(
    df_tbl_distribuicao_anual_sinais,
    on=["estrategia_referencia", "ano_sinal"],
    how="left",
)

df_tbl_distribuicao_anual_sinais["n_sinais"] = df_tbl_distribuicao_anual_sinais["n_sinais"].fillna(0).astype(int)
df_tbl_distribuicao_anual_sinais["n_sinais_com_aporte_realizado"] = df_tbl_distribuicao_anual_sinais["n_sinais_com_aporte_realizado"].fillna(0).astype(int)
df_tbl_distribuicao_anual_sinais["valor_aporte_total_realizado"] = df_tbl_distribuicao_anual_sinais["valor_aporte_total_realizado"].fillna(0.0)
df_tbl_distribuicao_anual_sinais["ano_com_sinal"] = df_tbl_distribuicao_anual_sinais["n_sinais"] > 0

df_tbl_distribuicao_anual_sinais = df_tbl_distribuicao_anual_sinais.merge(total_sinais_por_estrategia, on="estrategia_referencia", how="left")
df_tbl_distribuicao_anual_sinais["pct_sinais_estrategia"] = df_tbl_distribuicao_anual_sinais.apply(
    lambda linha: calcular_pct(linha["n_sinais"], linha["total_sinais_estrategia"]),
    axis=1,
)

df_tbl_distribuicao_anual_sinais["nome_exibicao_sinal"] = np.where(
    df_tbl_distribuicao_anual_sinais["estrategia_referencia"].eq("capitulacao"),
    "Capitulação",
    "Euforia",
)

df_tbl_distribuicao_anual_sinais = df_tbl_distribuicao_anual_sinais.sort_values(
    ["estrategia_referencia", "ano_sinal"]
).reset_index(drop=True)

idx_mensal = pd.MultiIndex.from_product(
    [estrategias_observadas, sorted(df_sinais["ano_mes_sinal"].dropna().astype(str).unique().tolist())],
    names=["estrategia_referencia", "ano_mes_sinal"],
).to_frame(index=False)

idx_mensal["data_inicio_mes"] = pd.to_datetime(idx_mensal["ano_mes_sinal"] + "-01", errors="coerce")
idx_mensal["ano_sinal"] = idx_mensal["data_inicio_mes"].dt.year.astype("Int64")
idx_mensal["mes_sinal"] = idx_mensal["data_inicio_mes"].dt.month.astype("Int64")

df_tbl_distribuicao_mensal_sinais = (
    df_sinais.groupby(["estrategia_referencia", "ano_mes_sinal"], dropna=False)
    .agg(
        n_sinais=("data_sinal", "count"),
        primeira_data_sinal=("data_sinal", "min"),
        ultima_data_sinal=("data_sinal", "max"),
        n_sinais_com_aporte_realizado=("n_aportes_realizados", lambda x: int((pd.to_numeric(x, errors="coerce") > 0).sum())),
        valor_aporte_total_realizado=("valor_aporte_total_realizado", "sum"),
        score_sinal_medio=("score_sinal", "mean"),
        score_sinal_maximo=("score_sinal", "max"),
    )
    .reset_index()
)

df_tbl_distribuicao_mensal_sinais = idx_mensal.merge(
    df_tbl_distribuicao_mensal_sinais,
    on=["estrategia_referencia", "ano_mes_sinal"],
    how="left",
)

df_tbl_distribuicao_mensal_sinais["n_sinais"] = df_tbl_distribuicao_mensal_sinais["n_sinais"].fillna(0).astype(int)
df_tbl_distribuicao_mensal_sinais["n_sinais_com_aporte_realizado"] = df_tbl_distribuicao_mensal_sinais["n_sinais_com_aporte_realizado"].fillna(0).astype(int)
df_tbl_distribuicao_mensal_sinais["valor_aporte_total_realizado"] = df_tbl_distribuicao_mensal_sinais["valor_aporte_total_realizado"].fillna(0.0)
df_tbl_distribuicao_mensal_sinais["mes_com_sinal"] = df_tbl_distribuicao_mensal_sinais["n_sinais"] > 0

df_tbl_distribuicao_mensal_sinais = df_tbl_distribuicao_mensal_sinais.merge(total_sinais_por_estrategia, on="estrategia_referencia", how="left")
df_tbl_distribuicao_mensal_sinais["pct_sinais_estrategia"] = df_tbl_distribuicao_mensal_sinais.apply(
    lambda linha: calcular_pct(linha["n_sinais"], linha["total_sinais_estrategia"]),
    axis=1,
)

df_tbl_distribuicao_mensal_sinais["nome_exibicao_sinal"] = np.where(
    df_tbl_distribuicao_mensal_sinais["estrategia_referencia"].eq("capitulacao"),
    "Capitulação",
    "Euforia",
)

df_tbl_distribuicao_mensal_sinais = df_tbl_distribuicao_mensal_sinais.sort_values(
    ["estrategia_referencia", "ano_mes_sinal"]
).reset_index(drop=True)

print(f"Distribuição anual de sinais              : {len(df_tbl_distribuicao_anual_sinais):,} linhas")
print(f"Distribuição mensal de sinais             : {len(df_tbl_distribuicao_mensal_sinais):,} linhas")
print("OK")

# ============================================================
# 10) Distribuições por mês calendário, regime e intervalos
# ============================================================

print("\n[10/12] Distribuições por mês calendário, regime e intervalos...")

meses_calendario = pd.DataFrame({"mes_sinal": list(range(1, 13))})
idx_mes_calendario = pd.MultiIndex.from_product(
    [estrategias_observadas, list(range(1, 13))],
    names=["estrategia_referencia", "mes_sinal"],
).to_frame(index=False)

df_tbl_distribuicao_mes_calendario_sinais = (
    df_sinais.groupby(["estrategia_referencia", "mes_sinal"], dropna=False)
    .agg(
        n_sinais=("data_sinal", "count"),
        n_anos_com_sinal=("ano_sinal", lambda x: int(pd.Series(x).dropna().nunique())),
        primeira_data_sinal=("data_sinal", "min"),
        ultima_data_sinal=("data_sinal", "max"),
        score_sinal_medio=("score_sinal", "mean"),
    )
    .reset_index()
)

df_tbl_distribuicao_mes_calendario_sinais = idx_mes_calendario.merge(
    df_tbl_distribuicao_mes_calendario_sinais,
    on=["estrategia_referencia", "mes_sinal"],
    how="left",
)

df_tbl_distribuicao_mes_calendario_sinais["n_sinais"] = df_tbl_distribuicao_mes_calendario_sinais["n_sinais"].fillna(0).astype(int)
df_tbl_distribuicao_mes_calendario_sinais["n_anos_com_sinal"] = df_tbl_distribuicao_mes_calendario_sinais["n_anos_com_sinal"].fillna(0).astype(int)
df_tbl_distribuicao_mes_calendario_sinais = df_tbl_distribuicao_mes_calendario_sinais.merge(total_sinais_por_estrategia, on="estrategia_referencia", how="left")
df_tbl_distribuicao_mes_calendario_sinais["pct_sinais_estrategia"] = df_tbl_distribuicao_mes_calendario_sinais.apply(
    lambda linha: calcular_pct(linha["n_sinais"], linha["total_sinais_estrategia"]),
    axis=1,
)

df_tbl_distribuicao_mes_calendario_sinais["nome_mes"] = pd.to_datetime(
    "2020-" + df_tbl_distribuicao_mes_calendario_sinais["mes_sinal"].astype(str).str.zfill(2) + "-01"
).dt.month_name()

df_tbl_distribuicao_mes_calendario_sinais["nome_exibicao_sinal"] = np.where(
    df_tbl_distribuicao_mes_calendario_sinais["estrategia_referencia"].eq("capitulacao"),
    "Capitulação",
    "Euforia",
)

df_tbl_distribuicao_mes_calendario_sinais = df_tbl_distribuicao_mes_calendario_sinais.sort_values(
    ["estrategia_referencia", "mes_sinal"]
).reset_index(drop=True)

regimes_observados = sorted(df_sinais["regime_mercado"].dropna().astype(str).unique().tolist())
idx_regime = pd.MultiIndex.from_product(
    [estrategias_observadas, regimes_observados],
    names=["estrategia_referencia", "regime_mercado"],
).to_frame(index=False)

df_tbl_distribuicao_regime_mercado_sinais = (
    df_sinais.groupby(["estrategia_referencia", "regime_mercado"], dropna=False)
    .agg(
        n_sinais=("data_sinal", "count"),
        primeira_data_sinal=("data_sinal", "min"),
        ultima_data_sinal=("data_sinal", "max"),
        retorno_ibovespa_252p_medio=("retorno_ibovespa_252p", "mean"),
        drawdown_ibovespa_medio=("drawdown_ibovespa", "mean"),
        volatilidade_ibovespa_63p_anualizada_media=("volatilidade_ibovespa_63p_anualizada", "mean"),
        score_sinal_medio=("score_sinal", "mean"),
    )
    .reset_index()
)

df_tbl_distribuicao_regime_mercado_sinais = idx_regime.merge(
    df_tbl_distribuicao_regime_mercado_sinais,
    on=["estrategia_referencia", "regime_mercado"],
    how="left",
)

df_tbl_distribuicao_regime_mercado_sinais["n_sinais"] = df_tbl_distribuicao_regime_mercado_sinais["n_sinais"].fillna(0).astype(int)
df_tbl_distribuicao_regime_mercado_sinais = df_tbl_distribuicao_regime_mercado_sinais.merge(total_sinais_por_estrategia, on="estrategia_referencia", how="left")
df_tbl_distribuicao_regime_mercado_sinais["pct_sinais_estrategia"] = df_tbl_distribuicao_regime_mercado_sinais.apply(
    lambda linha: calcular_pct(linha["n_sinais"], linha["total_sinais_estrategia"]),
    axis=1,
)

df_tbl_distribuicao_regime_mercado_sinais["nome_exibicao_sinal"] = np.where(
    df_tbl_distribuicao_regime_mercado_sinais["estrategia_referencia"].eq("capitulacao"),
    "Capitulação",
    "Euforia",
)

df_tbl_intervalos_entre_sinais = (
    df_sinais.groupby("estrategia_referencia", dropna=False)
    .agg(
        n_sinais=("data_sinal", "count"),
        primeira_data_sinal=("data_sinal", "min"),
        ultima_data_sinal=("data_sinal", "max"),
        intervalo_medio_dias_corridos=("intervalo_dias_corridos_desde_sinal_anterior", "mean"),
        intervalo_mediano_dias_corridos=("intervalo_dias_corridos_desde_sinal_anterior", "median"),
        intervalo_minimo_dias_corridos=("intervalo_dias_corridos_desde_sinal_anterior", "min"),
        intervalo_maximo_dias_corridos=("intervalo_dias_corridos_desde_sinal_anterior", "max"),
        intervalo_medio_dias_uteis_aproximado=("intervalo_dias_uteis_aproximado_desde_sinal_anterior", "mean"),
        intervalo_mediano_dias_uteis_aproximado=("intervalo_dias_uteis_aproximado_desde_sinal_anterior", "median"),
        intervalo_minimo_dias_uteis_aproximado=("intervalo_dias_uteis_aproximado_desde_sinal_anterior", "min"),
        intervalo_maximo_dias_uteis_aproximado=("intervalo_dias_uteis_aproximado_desde_sinal_anterior", "max"),
    )
    .reset_index()
)

df_tbl_intervalos_entre_sinais["nome_exibicao_sinal"] = np.where(
    df_tbl_intervalos_entre_sinais["estrategia_referencia"].eq("capitulacao"),
    "Capitulação",
    "Euforia",
)

df_tbl_intervalos_entre_sinais["n_intervalos_validos"] = df_tbl_intervalos_entre_sinais["n_sinais"] - 1

df_tbl_intervalos_entre_sinais = df_tbl_intervalos_entre_sinais.sort_values("estrategia_referencia").reset_index(drop=True)

print(f"Distribuição por mês calendário          : {len(df_tbl_distribuicao_mes_calendario_sinais):,} linhas")
print(f"Distribuição por regime de mercado       : {len(df_tbl_distribuicao_regime_mercado_sinais):,} linhas")
print(f"Tabela de intervalos entre sinais        : {len(df_tbl_intervalos_entre_sinais):,} linhas")
print("OK")

# ============================================================
# 11) Bases finais para gráficos e auditoria de validação
# ============================================================

print("\n[11/12] Bases finais para gráficos e auditoria de validação...")

df_tbl_base_grafico_timeline_sinais = df_sinais[
    [
        "estrategia_referencia",
        "nome_exibicao_sinal",
        "data_sinal",
        "ano_sinal",
        "mes_sinal",
        "ano_mes_sinal",
        "ordem_sinal",
        "score_sinal",
        "regime_mercado",
        "retorno_ibovespa_252p",
        "drawdown_ibovespa",
        "n_aportes_realizados",
        "valor_aporte_total_realizado",
    ]
].copy()

df_tbl_base_grafico_timeline_sinais = df_tbl_base_grafico_timeline_sinais.sort_values(
    ["estrategia_referencia", "data_sinal"]
).reset_index(drop=True)

df_tbl_base_grafico_distribuicao_anual_sinais = df_tbl_distribuicao_anual_sinais[
    [
        "estrategia_referencia",
        "nome_exibicao_sinal",
        "ano_sinal",
        "n_sinais",
        "pct_sinais_estrategia",
        "n_sinais_com_aporte_realizado",
        "valor_aporte_total_realizado",
        "ano_com_sinal",
    ]
].copy()

n_sinais_capitulacao = int((df_sinais["estrategia_referencia"] == "capitulacao").sum())
n_sinais_euforia = int((df_sinais["estrategia_referencia"] == "euforia").sum())
n_anos_observados = int(df_sinais["ano_sinal"].nunique(dropna=True))
n_regimes_observados = int(df_sinais["regime_mercado"].nunique(dropna=True))
linhas_esperadas_anual = int(len(estrategias_observadas) * len(anos_observados))
linhas_esperadas_mensal = int(len(estrategias_observadas) * df_sinais["ano_mes_sinal"].nunique(dropna=True))
linhas_esperadas_mes_calendario = int(len(estrategias_observadas) * 12)
linhas_esperadas_regime = int(len(estrategias_observadas) * len(regimes_observados))

itens_auditoria = [
    {
        "item": "base_sinais_temporal_linhas",
        "valor": int(len(df_sinais)),
        "valor_referencia": int(len(df_sinais)),
        "status": "OK" if len(df_sinais) > 0 else "ERRO",
        "observacao": "A base temporal de sinais deve conter ao menos um registro.",
    },
    {
        "item": "sinais_capitulacao",
        "valor": n_sinais_capitulacao,
        "valor_referencia": 1,
        "status": "OK" if n_sinais_capitulacao > 0 else "ERRO",
        "observacao": "A estratégia de capitulação deve ter sinais finais registrados.",
    },
    {
        "item": "sinais_euforia",
        "valor": n_sinais_euforia,
        "valor_referencia": 1,
        "status": "OK" if n_sinais_euforia > 0 else "ERRO",
        "observacao": "A estratégia de euforia deve ter sinais finais registrados.",
    },
    {
        "item": "datas_invalidas_sinais",
        "valor": datas_invalidas_sinais,
        "valor_referencia": 0,
        "status": "OK" if datas_invalidas_sinais == 0 else "ERRO",
        "observacao": "Não deve haver datas inválidas na base temporal de sinais.",
    },
    {
        "item": "duplicatas_sinais",
        "valor": duplicatas_sinais,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_sinais == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade por estratégia de referência e data de sinal.",
    },
    {
        "item": "regimes_ausentes_sinais",
        "valor": regimes_ausentes_sinais,
        "valor_referencia": 0,
        "status": "OK" if regimes_ausentes_sinais == 0 else "ALERTA",
        "observacao": "Sinais sem regime de mercado associado devem ser avaliados manualmente.",
    },
    {
        "item": "anos_observados",
        "valor": n_anos_observados,
        "valor_referencia": 1,
        "status": "OK" if n_anos_observados > 0 else "ERRO",
        "observacao": "A distribuição temporal deve cobrir ao menos um ano de análise.",
    },
    {
        "item": "regimes_observados",
        "valor": n_regimes_observados,
        "valor_referencia": 1,
        "status": "OK" if n_regimes_observados > 0 else "ALERTA",
        "observacao": "A classificação de regime deve identificar ao menos um regime de mercado.",
    },
    {
        "item": "linhas_distribuicao_anual",
        "valor": int(len(df_tbl_distribuicao_anual_sinais)),
        "valor_referencia": linhas_esperadas_anual,
        "status": "OK" if len(df_tbl_distribuicao_anual_sinais) == linhas_esperadas_anual else "ERRO",
        "observacao": "A distribuição anual deve conter uma linha por estratégia e ano observado.",
    },
    {
        "item": "linhas_distribuicao_mensal",
        "valor": int(len(df_tbl_distribuicao_mensal_sinais)),
        "valor_referencia": linhas_esperadas_mensal,
        "status": "OK" if len(df_tbl_distribuicao_mensal_sinais) == linhas_esperadas_mensal else "ERRO",
        "observacao": "A distribuição mensal deve conter uma linha por estratégia e mês observado.",
    },
    {
        "item": "linhas_distribuicao_mes_calendario",
        "valor": int(len(df_tbl_distribuicao_mes_calendario_sinais)),
        "valor_referencia": linhas_esperadas_mes_calendario,
        "status": "OK" if len(df_tbl_distribuicao_mes_calendario_sinais) == linhas_esperadas_mes_calendario else "ERRO",
        "observacao": "A distribuição por mês calendário deve conter 12 linhas por estratégia.",
    },
    {
        "item": "linhas_distribuicao_regime",
        "valor": int(len(df_tbl_distribuicao_regime_mercado_sinais)),
        "valor_referencia": linhas_esperadas_regime,
        "status": "OK" if len(df_tbl_distribuicao_regime_mercado_sinais) == linhas_esperadas_regime else "ERRO",
        "observacao": "A distribuição por regime deve conter uma linha por estratégia e regime observado.",
    },
    {
        "item": "linhas_timeline",
        "valor": int(len(df_tbl_base_grafico_timeline_sinais)),
        "valor_referencia": int(len(df_sinais)),
        "status": "OK" if len(df_tbl_base_grafico_timeline_sinais) == len(df_sinais) else "ERRO",
        "observacao": "A base de timeline deve preservar uma linha por sinal.",
    },
]

df_tbl_auditoria_validacao_frequencia_sinais = pd.DataFrame(itens_auditoria)

erros_bloqueantes = int((df_tbl_auditoria_validacao_frequencia_sinais["status"] == "ERRO").sum())

print(f"Base gráfico timeline                    : {len(df_tbl_base_grafico_timeline_sinais):,} linhas")
print(f"Base gráfico distribuição anual          : {len(df_tbl_base_grafico_distribuicao_anual_sinais):,} linhas")
print(f"Itens de auditoria                       : {len(df_tbl_auditoria_validacao_frequencia_sinais):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")

if erros_bloqueantes > 0:
    print(df_tbl_auditoria_validacao_frequencia_sinais.to_string(index=False))
    raise ValueError("A auditoria da frequência e distribuição temporal dos sinais encontrou erros bloqueantes.")

print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(df_sinais, caminho_base_sinais_temporal, index=False)
salvar_dataframe(df_base_regimes_ibovespa, caminho_base_regimes_ibovespa, index=False)
salvar_dataframe(df_tbl_distribuicao_anual_sinais, caminho_tbl_distribuicao_anual_sinais, index=False)
salvar_dataframe(df_tbl_distribuicao_mensal_sinais, caminho_tbl_distribuicao_mensal_sinais, index=False)
salvar_dataframe(df_tbl_distribuicao_mes_calendario_sinais, caminho_tbl_distribuicao_mes_calendario_sinais, index=False)
salvar_dataframe(df_tbl_distribuicao_regime_mercado_sinais, caminho_tbl_distribuicao_regime_mercado_sinais, index=False)
salvar_dataframe(df_tbl_intervalos_entre_sinais, caminho_tbl_intervalos_entre_sinais, index=False)
salvar_dataframe(df_tbl_base_grafico_timeline_sinais, caminho_tbl_base_grafico_timeline_sinais, index=False)
salvar_dataframe(df_tbl_base_grafico_distribuicao_anual_sinais, caminho_tbl_base_grafico_distribuicao_anual_sinais, index=False)
salvar_dataframe(df_tbl_auditoria_validacao_frequencia_sinais, caminho_tbl_auditoria_validacao_frequencia_sinais, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da frequência e distribuição temporal dos sinais:")
print(df_tbl_auditoria_validacao_frequencia_sinais.to_string(index=False))

print("\nDistribuição anual de sinais:")
print(df_tbl_distribuicao_anual_sinais.to_string(index=False))

print("\nDistribuição por regime de mercado:")
print(df_tbl_distribuicao_regime_mercado_sinais.to_string(index=False))

print("\nIntervalos entre sinais:")
print(df_tbl_intervalos_entre_sinais.to_string(index=False))

print("\nBase temporal de sinais - amostra:")
colunas_amostra_sinais = [
    "estrategia_referencia",
    "nome_exibicao_sinal",
    "ordem_sinal",
    "data_sinal",
    "ano_sinal",
    "ano_mes_sinal",
    "regime_mercado",
    "retorno_ibovespa_252p",
    "drawdown_ibovespa",
    "n_aportes_realizados",
    "valor_aporte_total_realizado",
]
colunas_amostra_sinais = [coluna for coluna in colunas_amostra_sinais if coluna in df_sinais.columns]
print(df_sinais[colunas_amostra_sinais].head(30).to_string(index=False))

print("\nArquivos salvos na subetapa 12.1:")
print(f"- {caminho_base_sinais_temporal}")
print(f"- {caminho_base_regimes_ibovespa}")
print(f"- {caminho_tbl_distribuicao_anual_sinais}")
print(f"- {caminho_tbl_distribuicao_mensal_sinais}")
print(f"- {caminho_tbl_distribuicao_mes_calendario_sinais}")
print(f"- {caminho_tbl_distribuicao_regime_mercado_sinais}")
print(f"- {caminho_tbl_intervalos_entre_sinais}")
print(f"- {caminho_tbl_base_grafico_timeline_sinais}")
print(f"- {caminho_tbl_base_grafico_distribuicao_anual_sinais}")
print(f"- {caminho_tbl_auditoria_validacao_frequencia_sinais}")

print("\nETAPA 12.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 12.1 - FREQUÊNCIA E DISTRIBUIÇÃO TEMPORAL DOS SINAIS

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - calendário final de sinais da 6.4 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_4_base_calendario_final_sinais.parquet
Entrada - aportes realizados da 10.1        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_1_base_aportes_realizados.parquet
Entrada - retornos diários da 11.1          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_base_retornos_diarios.parquet
Saída   - base temporal de sinais           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12

## Etapa 12.2) Intensidade dos Sinais de Capitulação e Euforia

In [60]:
%%time
# ============================================================
# Etapa 12.2) Intensidade dos Sinais de Capitulação e Euforia
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 12.2 - INTENSIDADE DOS SINAIS DE CAPITULAÇÃO E EUFORIA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/11] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/11] Definição determinística dos caminhos de entrada e saída...")

caminho_base_sinais_temporal = gerar_caminho_arquivo(
    etapa=12,
    subetapa=1,
    tipo_arquivo="base",
    nome="sinais_temporal",
)

caminho_base_indicadores_agregados_amplitude = gerar_caminho_arquivo(
    etapa=5,
    subetapa=5,
    tipo_arquivo="base",
    nome="indicadores_agregados_amplitude_pregao",
)

caminho_base_intensidade_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=2,
    tipo_arquivo="base",
    nome="intensidade_sinais",
)

caminho_base_amplitude_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=2,
    tipo_arquivo="base",
    nome="amplitude_sinais",
)

caminho_tbl_resumo_intensidade_estrategia = gerar_caminho_arquivo(
    etapa=12,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="resumo_intensidade_estrategia",
)

caminho_tbl_intensidade_por_ano = gerar_caminho_arquivo(
    etapa=12,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="intensidade_por_ano",
)

caminho_tbl_intensidade_por_regime = gerar_caminho_arquivo(
    etapa=12,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="intensidade_por_regime",
)

caminho_tbl_ranking_intensidade_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="ranking_intensidade_sinais",
)

caminho_tbl_base_grafico_intensidade_timeline = gerar_caminho_arquivo(
    etapa=12,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="base_grafico_intensidade_timeline",
)

caminho_tbl_auditoria_validacao_intensidade_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=2,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_intensidade_sinais",
)

print(f"Entrada - base temporal de sinais da 12.1           : {caminho_base_sinais_temporal}")
print(f"Entrada - indicadores agregados de amplitude da 5.5 : {caminho_base_indicadores_agregados_amplitude}")
print(f"Saída   - base intensidade dos sinais               : {caminho_base_intensidade_sinais}")
print(f"Saída   - base amplitude dos sinais                 : {caminho_base_amplitude_sinais}")
print(f"Saída   - resumo por estratégia                     : {caminho_tbl_resumo_intensidade_estrategia}")
print(f"Saída   - intensidade por ano                       : {caminho_tbl_intensidade_por_ano}")
print(f"Saída   - intensidade por regime                    : {caminho_tbl_intensidade_por_regime}")
print(f"Saída   - ranking de intensidade                    : {caminho_tbl_ranking_intensidade_sinais}")
print(f"Saída   - base gráfico timeline                     : {caminho_tbl_base_grafico_intensidade_timeline}")
print(f"Saída   - auditoria de validação                    : {caminho_tbl_auditoria_validacao_intensidade_sinais}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais
# ============================================================

print("\n[3/11] Carga das bases oficiais de sinais e amplitude...")

df_sinais_temporal = pd.read_parquet(caminho_base_sinais_temporal)
df_amplitude = pd.read_parquet(caminho_base_indicadores_agregados_amplitude)

print(f"Base temporal de sinais               : {len(df_sinais_temporal):,} linhas x {df_sinais_temporal.shape[1]:,} colunas")
print(f"Base agregada de amplitude            : {len(df_amplitude):,} linhas x {df_amplitude.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/11] Funções auxiliares da subetapa...")

def normalizar_nome_coluna(nome_coluna):
    texto = str(nome_coluna).strip().lower()
    texto = texto.translate(str.maketrans("áàãâäéèêëíìîïóòõôöúùûüç", "aaaaaeeeeiiiiooooouuuuc"))
    texto = texto.replace("%", "pct")
    texto = texto.replace("/", "_")
    texto = texto.replace("-", "_")
    texto = texto.replace(" ", "_")
    while "__" in texto:
        texto = texto.replace("__", "_")
    return texto

def localizar_coluna_por_prioridade(df, candidatos_exatos, termos_obrigatorios=None, termos_qualquer=None, termos_excluir=None):
    mapa_normalizado = {normalizar_nome_coluna(coluna): coluna for coluna in df.columns}

    for candidato in candidatos_exatos:
        candidato_norm = normalizar_nome_coluna(candidato)
        if candidato_norm in mapa_normalizado:
            return mapa_normalizado[candidato_norm]

    termos_obrigatorios = termos_obrigatorios or []
    termos_qualquer = termos_qualquer or []
    termos_excluir = termos_excluir or []

    for coluna in df.columns:
        coluna_norm = normalizar_nome_coluna(coluna)
        atende_obrigatorios = all(termo in coluna_norm for termo in termos_obrigatorios)
        atende_qualquer = True if len(termos_qualquer) == 0 else any(termo in coluna_norm for termo in termos_qualquer)
        contem_excluido = any(termo in coluna_norm for termo in termos_excluir)
        if atende_obrigatorios and atende_qualquer and not contem_excluido:
            return coluna

    return None

def converter_percentual_para_fracao(serie):
    serie_num = pd.to_numeric(serie, errors="coerce")
    maximo = serie_num.dropna().max() if serie_num.notna().any() else np.nan
    if pd.notna(maximo) and maximo > 1.5:
        return serie_num / 100.0
    return serie_num

def calcular_zscore(serie):
    serie_num = pd.to_numeric(serie, errors="coerce")
    media = serie_num.mean(skipna=True)
    desvio = serie_num.std(skipna=True, ddof=0)
    if pd.isna(desvio) or desvio == 0:
        return pd.Series(np.nan, index=serie_num.index)
    return (serie_num - media) / desvio

def classificar_percentil_intensidade(percentil):
    if pd.isna(percentil):
        return "sem_classificacao"
    if percentil >= 0.90:
        return "muito_alta"
    if percentil >= 0.75:
        return "alta"
    if percentil >= 0.50:
        return "moderada"
    return "baixa"

def primeira_coluna_existente(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            return coluna
    return None

def montar_resumo_por_grupo(df, colunas_grupo):
    base = df.copy()
    resumo = (
        base.groupby(colunas_grupo, dropna=False)
        .agg(
            n_sinais=("data_sinal", "count"),
            primeira_data_sinal=("data_sinal", "min"),
            ultima_data_sinal=("data_sinal", "max"),
            pct_minima_52s_medio=("pct_minima_52s", "mean"),
            pct_maxima_52s_medio=("pct_maxima_52s", "mean"),
            pct_acima_mm200_medio=("pct_acima_mm200", "mean"),
            pct_abaixo_mm200_medio=("pct_abaixo_mm200", "mean"),
            score_intensidade_medio=("score_intensidade_sinal", "mean"),
            score_intensidade_mediano=("score_intensidade_sinal", "median"),
            score_intensidade_minimo=("score_intensidade_sinal", "min"),
            score_intensidade_maximo=("score_intensidade_sinal", "max"),
            percentil_intensidade_medio=("percentil_intensidade_sinal", "mean"),
            percentil_intensidade_mediano=("percentil_intensidade_sinal", "median"),
            n_sinais_intensos=("flag_sinal_intenso", "sum"),
        )
        .reset_index()
    )
    resumo["pct_sinais_intensos"] = np.where(
        resumo["n_sinais"] > 0,
        resumo["n_sinais_intensos"] / resumo["n_sinais"],
        np.nan,
    )
    return resumo

def adicionar_auditoria(lista, item, valor, valor_referencia, status, observacao):
    lista.append(
        {
            "item": item,
            "valor": valor,
            "valor_referencia": valor_referencia,
            "status": status,
            "observacao": observacao,
        }
    )

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Identificação determinística das colunas de amplitude
# ============================================================

print("\n[5/11] Identificação das colunas de indicadores agregados de amplitude...")

coluna_data_amplitude = localizar_coluna_por_prioridade(
    df_amplitude,
    candidatos_exatos=["data", "data_pregao", "data_referencia"],
    termos_obrigatorios=["data"],
)

coluna_pct_minima_52s = localizar_coluna_por_prioridade(
    df_amplitude,
    candidatos_exatos=[
        "pct_tickers_minima_52s",
        "pct_tickers_minima_52_semanas",
        "pct_minima_52s",
        "pct_minima_52_semanas",
        "percentual_tickers_minima_52s",
        "percentual_minima_52_semanas",
        "pct_em_minima_52_semanas",
    ],
    termos_obrigatorios=["minima"],
    termos_qualquer=["52", "252"],
    termos_excluir=["qtd", "quantidade", "n_", "flag", "maxima"],
)

coluna_pct_maxima_52s = localizar_coluna_por_prioridade(
    df_amplitude,
    candidatos_exatos=[
        "pct_tickers_maxima_52s",
        "pct_tickers_maxima_52_semanas",
        "pct_maxima_52s",
        "pct_maxima_52_semanas",
        "percentual_tickers_maxima_52s",
        "percentual_maxima_52_semanas",
        "pct_em_maxima_52_semanas",
    ],
    termos_obrigatorios=["maxima"],
    termos_qualquer=["52", "252"],
    termos_excluir=["qtd", "quantidade", "n_", "flag", "minima"],
)

coluna_pct_acima_mm200 = localizar_coluna_por_prioridade(
    df_amplitude,
    candidatos_exatos=[
        "pct_tickers_acima_mm200",
        "pct_acima_mm200",
        "percentual_tickers_acima_mm200",
        "percentual_acima_mm200",
        "pct_acima_media_movel_200",
        "pct_tickers_acima_media_movel_200",
    ],
    termos_obrigatorios=["acima"],
    termos_qualquer=["mm200", "media_movel_200", "movel_200"],
    termos_excluir=["qtd", "quantidade", "n_", "flag", "abaixo"],
)

coluna_pct_abaixo_mm200 = localizar_coluna_por_prioridade(
    df_amplitude,
    candidatos_exatos=[
        "pct_tickers_abaixo_mm200",
        "pct_abaixo_mm200",
        "percentual_tickers_abaixo_mm200",
        "percentual_abaixo_mm200",
        "pct_abaixo_media_movel_200",
        "pct_tickers_abaixo_media_movel_200",
    ],
    termos_obrigatorios=["abaixo"],
    termos_qualquer=["mm200", "media_movel_200", "movel_200"],
    termos_excluir=["qtd", "quantidade", "n_", "flag", "acima"],
)

colunas_metricas_detectadas = {
    "data_amplitude": coluna_data_amplitude,
    "pct_minima_52s": coluna_pct_minima_52s,
    "pct_maxima_52s": coluna_pct_maxima_52s,
    "pct_acima_mm200": coluna_pct_acima_mm200,
    "pct_abaixo_mm200": coluna_pct_abaixo_mm200,
}

for nome_metrica, coluna_detectada in colunas_metricas_detectadas.items():
    print(f"{nome_metrica:<35}: {coluna_detectada}")

colunas_obrigatorias_detectadas = [
    coluna_data_amplitude,
    coluna_pct_minima_52s,
    coluna_pct_maxima_52s,
    coluna_pct_acima_mm200,
    coluna_pct_abaixo_mm200,
]

if any(coluna is None for coluna in colunas_obrigatorias_detectadas):
    colunas_disponiveis = ", ".join([str(coluna) for coluna in df_amplitude.columns])
    raise ValueError(
        "Não foi possível identificar todas as colunas obrigatórias de amplitude na base 5.5. "
        f"Colunas disponíveis: {colunas_disponiveis}"
    )

print("OK")

# ============================================================
# 6) Padronização da base de amplitude e construção dos scores diários
# ============================================================

print("\n[6/11] Padronização da base de amplitude e construção dos scores diários...")

base_amplitude_scores = df_amplitude.copy()
base_amplitude_scores["data"] = pd.to_datetime(base_amplitude_scores[coluna_data_amplitude], errors="coerce")
base_amplitude_scores = base_amplitude_scores.sort_values("data").drop_duplicates(subset=["data"], keep="last").reset_index(drop=True)

base_amplitude_scores["pct_minima_52s"] = converter_percentual_para_fracao(base_amplitude_scores[coluna_pct_minima_52s])
base_amplitude_scores["pct_maxima_52s"] = converter_percentual_para_fracao(base_amplitude_scores[coluna_pct_maxima_52s])
base_amplitude_scores["pct_acima_mm200"] = converter_percentual_para_fracao(base_amplitude_scores[coluna_pct_acima_mm200])
base_amplitude_scores["pct_abaixo_mm200"] = converter_percentual_para_fracao(base_amplitude_scores[coluna_pct_abaixo_mm200])

base_amplitude_scores["z_pct_minima_52s"] = calcular_zscore(base_amplitude_scores["pct_minima_52s"])
base_amplitude_scores["z_pct_maxima_52s"] = calcular_zscore(base_amplitude_scores["pct_maxima_52s"])
base_amplitude_scores["z_pct_acima_mm200"] = calcular_zscore(base_amplitude_scores["pct_acima_mm200"])
base_amplitude_scores["z_pct_abaixo_mm200"] = calcular_zscore(base_amplitude_scores["pct_abaixo_mm200"])

base_amplitude_scores["score_capitulacao_intensidade"] = np.nanmean(
    np.vstack(
        [
            base_amplitude_scores["z_pct_minima_52s"].to_numpy(dtype=float),
            base_amplitude_scores["z_pct_abaixo_mm200"].to_numpy(dtype=float),
            -base_amplitude_scores["z_pct_maxima_52s"].to_numpy(dtype=float),
            -base_amplitude_scores["z_pct_acima_mm200"].to_numpy(dtype=float),
        ]
    ),
    axis=0,
)

base_amplitude_scores["score_euforia_intensidade"] = np.nanmean(
    np.vstack(
        [
            base_amplitude_scores["z_pct_maxima_52s"].to_numpy(dtype=float),
            base_amplitude_scores["z_pct_acima_mm200"].to_numpy(dtype=float),
            -base_amplitude_scores["z_pct_minima_52s"].to_numpy(dtype=float),
            -base_amplitude_scores["z_pct_abaixo_mm200"].to_numpy(dtype=float),
        ]
    ),
    axis=0,
)

base_amplitude_scores["percentil_score_capitulacao"] = base_amplitude_scores["score_capitulacao_intensidade"].rank(pct=True)
base_amplitude_scores["percentil_score_euforia"] = base_amplitude_scores["score_euforia_intensidade"].rank(pct=True)

colunas_base_amplitude_sinais = [
    "data",
    "pct_minima_52s",
    "pct_maxima_52s",
    "pct_acima_mm200",
    "pct_abaixo_mm200",
    "z_pct_minima_52s",
    "z_pct_maxima_52s",
    "z_pct_acima_mm200",
    "z_pct_abaixo_mm200",
    "score_capitulacao_intensidade",
    "score_euforia_intensidade",
    "percentil_score_capitulacao",
    "percentil_score_euforia",
]

base_amplitude_sinais = base_amplitude_scores[colunas_base_amplitude_sinais].copy()

print(f"Base de amplitude padronizada          : {len(base_amplitude_sinais):,} linhas")
print(f"Datas inválidas na amplitude           : {int(base_amplitude_sinais['data'].isna().sum()):,}")
print(f"Score capitulação ausente              : {int(base_amplitude_sinais['score_capitulacao_intensidade'].isna().sum()):,}")
print(f"Score euforia ausente                  : {int(base_amplitude_sinais['score_euforia_intensidade'].isna().sum()):,}")
print("OK")

# ============================================================
# 7) Integração dos scores de amplitude à base temporal de sinais
# ============================================================

print("\n[7/11] Integração dos scores de amplitude à base temporal de sinais...")

base_sinais = df_sinais_temporal.copy()

if "data_sinal" not in base_sinais.columns:
    raise ValueError("A base temporal da 12.1 deve conter a coluna 'data_sinal'.")
if "estrategia_referencia" not in base_sinais.columns:
    raise ValueError("A base temporal da 12.1 deve conter a coluna 'estrategia_referencia'.")

base_sinais["data_sinal"] = pd.to_datetime(base_sinais["data_sinal"], errors="coerce")
base_sinais["estrategia_referencia"] = base_sinais["estrategia_referencia"].astype(str).str.strip().str.lower()

linhas_sinais_antes_merge = len(base_sinais)

base_intensidade_sinais = base_sinais.merge(
    base_amplitude_sinais,
    left_on="data_sinal",
    right_on="data",
    how="left",
    suffixes=("", "_amplitude"),
)

base_intensidade_sinais["tipo_sinal_intensidade"] = np.select(
    [
        base_intensidade_sinais["estrategia_referencia"].eq("capitulacao"),
        base_intensidade_sinais["estrategia_referencia"].eq("euforia"),
    ],
    ["capitulacao", "euforia"],
    default="outro",
)

base_intensidade_sinais["score_intensidade_sinal"] = np.select(
    [
        base_intensidade_sinais["tipo_sinal_intensidade"].eq("capitulacao"),
        base_intensidade_sinais["tipo_sinal_intensidade"].eq("euforia"),
    ],
    [
        base_intensidade_sinais["score_capitulacao_intensidade"],
        base_intensidade_sinais["score_euforia_intensidade"],
    ],
    default=np.nan,
)

base_intensidade_sinais["percentil_intensidade_sinal"] = np.select(
    [
        base_intensidade_sinais["tipo_sinal_intensidade"].eq("capitulacao"),
        base_intensidade_sinais["tipo_sinal_intensidade"].eq("euforia"),
    ],
    [
        base_intensidade_sinais["percentil_score_capitulacao"],
        base_intensidade_sinais["percentil_score_euforia"],
    ],
    default=np.nan,
)

base_intensidade_sinais["classe_intensidade_sinal"] = base_intensidade_sinais["percentil_intensidade_sinal"].apply(classificar_percentil_intensidade)
base_intensidade_sinais["flag_sinal_intenso"] = base_intensidade_sinais["percentil_intensidade_sinal"] >= 0.75
base_intensidade_sinais["flag_sinal_muito_intenso"] = base_intensidade_sinais["percentil_intensidade_sinal"] >= 0.90

base_intensidade_sinais = base_intensidade_sinais.sort_values(
    ["estrategia_referencia", "data_sinal"]
).reset_index(drop=True)

print(f"Linhas antes do merge de amplitude      : {linhas_sinais_antes_merge:,}")
print(f"Linhas após o merge de amplitude        : {len(base_intensidade_sinais):,}")
print(f"Sinais sem amplitude associada          : {int(base_intensidade_sinais['score_intensidade_sinal'].isna().sum()):,}")
print(f"Sinais classificados como intensos      : {int(base_intensidade_sinais['flag_sinal_intenso'].sum()):,}")
print("OK")

# ============================================================
# 8) Construção dos resumos de intensidade por estratégia, ano e regime
# ============================================================

print("\n[8/11] Construção dos resumos de intensidade por estratégia, ano e regime...")

if "nome_exibicao_sinal" not in base_intensidade_sinais.columns:
    base_intensidade_sinais["nome_exibicao_sinal"] = np.where(
        base_intensidade_sinais["estrategia_referencia"].eq("capitulacao"),
        "Capitulação",
        np.where(base_intensidade_sinais["estrategia_referencia"].eq("euforia"), "Euforia", base_intensidade_sinais["estrategia_referencia"]),
    )

if "ano_sinal" not in base_intensidade_sinais.columns:
    base_intensidade_sinais["ano_sinal"] = base_intensidade_sinais["data_sinal"].dt.year

if "regime_mercado" not in base_intensidade_sinais.columns:
    base_intensidade_sinais["regime_mercado"] = "sem_regime"

resumo_intensidade_estrategia = montar_resumo_por_grupo(
    base_intensidade_sinais,
    ["estrategia_referencia", "nome_exibicao_sinal"],
)

intensidade_por_ano = montar_resumo_por_grupo(
    base_intensidade_sinais,
    ["estrategia_referencia", "nome_exibicao_sinal", "ano_sinal"],
)

intensidade_por_regime = montar_resumo_por_grupo(
    base_intensidade_sinais,
    ["estrategia_referencia", "nome_exibicao_sinal", "regime_mercado"],
)

print(f"Resumo por estratégia                  : {len(resumo_intensidade_estrategia):,} linhas")
print(f"Intensidade por ano                    : {len(intensidade_por_ano):,} linhas")
print(f"Intensidade por regime                 : {len(intensidade_por_regime):,} linhas")
print("OK")

# ============================================================
# 9) Construção do ranking de intensidade e bases para gráficos
# ============================================================

print("\n[9/11] Construção do ranking de intensidade e bases para gráficos...")

ranking_intensidade_sinais = base_intensidade_sinais.copy()
ranking_intensidade_sinais["ranking_intensidade_estrategia"] = (
    ranking_intensidade_sinais.groupby("estrategia_referencia")["score_intensidade_sinal"]
    .rank(method="first", ascending=False)
)
ranking_intensidade_sinais = ranking_intensidade_sinais.sort_values(
    ["estrategia_referencia", "ranking_intensidade_estrategia"]
).reset_index(drop=True)

colunas_ranking_preferenciais = [
    "estrategia_referencia",
    "nome_exibicao_sinal",
    "ranking_intensidade_estrategia",
    "ordem_sinal",
    "data_sinal",
    "ano_sinal",
    "regime_mercado",
    "pct_minima_52s",
    "pct_maxima_52s",
    "pct_acima_mm200",
    "pct_abaixo_mm200",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
]
colunas_ranking = [coluna for coluna in colunas_ranking_preferenciais if coluna in ranking_intensidade_sinais.columns]
ranking_intensidade_sinais = ranking_intensidade_sinais[colunas_ranking].copy()

base_grafico_intensidade_timeline = base_intensidade_sinais.copy()
colunas_timeline_preferenciais = [
    "data_sinal",
    "estrategia_referencia",
    "nome_exibicao_sinal",
    "ano_sinal",
    "ano_mes_sinal",
    "regime_mercado",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "pct_minima_52s",
    "pct_maxima_52s",
    "pct_acima_mm200",
    "pct_abaixo_mm200",
]
colunas_timeline = [coluna for coluna in colunas_timeline_preferenciais if coluna in base_grafico_intensidade_timeline.columns]
base_grafico_intensidade_timeline = base_grafico_intensidade_timeline[colunas_timeline].copy()

print(f"Ranking de intensidade dos sinais      : {len(ranking_intensidade_sinais):,} linhas")
print(f"Base gráfico timeline de intensidade   : {len(base_grafico_intensidade_timeline):,} linhas")
print("OK")

# ============================================================
# 10) Auditoria de validação
# ============================================================

print("\n[10/11] Construção da auditoria de validação...")

auditoria = []

n_sinais = len(base_intensidade_sinais)
n_sinais_origem = len(df_sinais_temporal)
n_estrategias = base_intensidade_sinais["estrategia_referencia"].nunique(dropna=True)

duplicatas_sinais = int(
    base_intensidade_sinais.duplicated(subset=["estrategia_referencia", "data_sinal"]).sum()
)

datas_invalidas_sinais = int(base_intensidade_sinais["data_sinal"].isna().sum())
sinais_sem_score = int(base_intensidade_sinais["score_intensidade_sinal"].isna().sum())
sinais_sem_percentil = int(base_intensidade_sinais["percentil_intensidade_sinal"].isna().sum())
metricas_amplitude_ausentes = int(
    base_intensidade_sinais[["pct_minima_52s", "pct_maxima_52s", "pct_acima_mm200", "pct_abaixo_mm200"]]
    .isna()
    .any(axis=1)
    .sum()
)

colunas_numericas = base_intensidade_sinais.select_dtypes(include=[np.number]).columns.tolist()
metricas_infinitas = int(np.isinf(base_intensidade_sinais[colunas_numericas].to_numpy(dtype=float)).sum()) if len(colunas_numericas) > 0 else 0

adicionar_auditoria(
    auditoria,
    "linhas_base_intensidade_sinais",
    n_sinais,
    n_sinais_origem,
    "OK" if n_sinais == n_sinais_origem else "ERRO",
    "A base de intensidade deve preservar uma linha por sinal final da 12.1.",
)

adicionar_auditoria(
    auditoria,
    "estrategias_com_sinais",
    n_estrategias,
    2,
    "OK" if n_estrategias == 2 else "ERRO",
    "A base deve conter sinais de capitulação e euforia.",
)

adicionar_auditoria(
    auditoria,
    "datas_invalidas_sinais",
    datas_invalidas_sinais,
    0,
    "OK" if datas_invalidas_sinais == 0 else "ERRO",
    "Não deve haver datas inválidas nos sinais finais.",
)

adicionar_auditoria(
    auditoria,
    "duplicatas_sinais",
    duplicatas_sinais,
    0,
    "OK" if duplicatas_sinais == 0 else "ERRO",
    "Não deve haver duplicidade por estratégia de referência e data de sinal.",
)

adicionar_auditoria(
    auditoria,
    "colunas_amplitude_detectadas",
    int(all(coluna is not None for coluna in colunas_obrigatorias_detectadas)),
    1,
    "OK" if all(coluna is not None for coluna in colunas_obrigatorias_detectadas) else "ERRO",
    "As quatro métricas centrais de amplitude devem ser identificadas na base 5.5.",
)

adicionar_auditoria(
    auditoria,
    "sinais_sem_metricas_amplitude",
    metricas_amplitude_ausentes,
    0,
    "OK" if metricas_amplitude_ausentes == 0 else "ERRO",
    "Todos os sinais finais devem receber os indicadores de amplitude da respectiva data.",
)

adicionar_auditoria(
    auditoria,
    "sinais_sem_score_intensidade",
    sinais_sem_score,
    0,
    "OK" if sinais_sem_score == 0 else "ERRO",
    "Todos os sinais devem receber score de intensidade coerente com o tipo de sinal.",
)

adicionar_auditoria(
    auditoria,
    "sinais_sem_percentil_intensidade",
    sinais_sem_percentil,
    0,
    "OK" if sinais_sem_percentil == 0 else "ERRO",
    "Todos os sinais devem receber percentil de intensidade.",
)

adicionar_auditoria(
    auditoria,
    "linhas_resumo_estrategia",
    len(resumo_intensidade_estrategia),
    n_estrategias,
    "OK" if len(resumo_intensidade_estrategia) == n_estrategias else "ERRO",
    "O resumo por estratégia deve conter uma linha por estratégia de sinal.",
)

adicionar_auditoria(
    auditoria,
    "linhas_ranking_intensidade",
    len(ranking_intensidade_sinais),
    n_sinais,
    "OK" if len(ranking_intensidade_sinais) == n_sinais else "ERRO",
    "O ranking de intensidade deve preservar uma linha por sinal.",
)

adicionar_auditoria(
    auditoria,
    "metricas_infinitas",
    metricas_infinitas,
    0,
    "OK" if metricas_infinitas == 0 else "ERRO",
    "As métricas numéricas consolidadas não devem conter valores infinitos.",
)

tbl_auditoria_validacao_intensidade_sinais = pd.DataFrame(auditoria)
erros_bloqueantes = int((tbl_auditoria_validacao_intensidade_sinais["status"] == "ERRO").sum())

print(f"Itens de auditoria                       : {len(tbl_auditoria_validacao_intensidade_sinais):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 11) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[11/11] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(base_intensidade_sinais, caminho_base_intensidade_sinais, index=False)
salvar_dataframe(base_amplitude_sinais, caminho_base_amplitude_sinais, index=False)
salvar_dataframe(resumo_intensidade_estrategia, caminho_tbl_resumo_intensidade_estrategia, index=False)
salvar_dataframe(intensidade_por_ano, caminho_tbl_intensidade_por_ano, index=False)
salvar_dataframe(intensidade_por_regime, caminho_tbl_intensidade_por_regime, index=False)
salvar_dataframe(ranking_intensidade_sinais, caminho_tbl_ranking_intensidade_sinais, index=False)
salvar_dataframe(base_grafico_intensidade_timeline, caminho_tbl_base_grafico_intensidade_timeline, index=False)
salvar_dataframe(tbl_auditoria_validacao_intensidade_sinais, caminho_tbl_auditoria_validacao_intensidade_sinais, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da intensidade dos sinais:")
print(tbl_auditoria_validacao_intensidade_sinais.to_string(index=False))

print("\nResumo de intensidade por estratégia:")
print(resumo_intensidade_estrategia.to_string(index=False))

print("\nIntensidade por regime de mercado:")
print(intensidade_por_regime.to_string(index=False))

print("\nRanking de intensidade dos sinais - amostra:")
print(ranking_intensidade_sinais.head(30).to_string(index=False))

print("\nBase de intensidade dos sinais - amostra:")
colunas_amostra = [
    "estrategia_referencia",
    "nome_exibicao_sinal",
    "data_sinal",
    "ano_sinal",
    "regime_mercado",
    "pct_minima_52s",
    "pct_maxima_52s",
    "pct_acima_mm200",
    "pct_abaixo_mm200",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
]
colunas_amostra = [coluna for coluna in colunas_amostra if coluna in base_intensidade_sinais.columns]
print(base_intensidade_sinais[colunas_amostra].head(30).to_string(index=False))

print("\nArquivos salvos na subetapa 12.2:")
for caminho in [
    caminho_base_intensidade_sinais,
    caminho_base_amplitude_sinais,
    caminho_tbl_resumo_intensidade_estrategia,
    caminho_tbl_intensidade_por_ano,
    caminho_tbl_intensidade_por_regime,
    caminho_tbl_ranking_intensidade_sinais,
    caminho_tbl_base_grafico_intensidade_timeline,
    caminho_tbl_auditoria_validacao_intensidade_sinais,
]:
    print(f"- {caminho}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 12.2 encontrou erros bloqueantes. Verifique a tabela de auditoria impressa acima.")

print("\nETAPA 12.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 12.2 - INTENSIDADE DOS SINAIS DE CAPITULAÇÃO E EUFORIA

[1/11] Validação inicial do ambiente...
OK

[2/11] Definição determinística dos caminhos de entrada e saída...
Entrada - base temporal de sinais da 12.1           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12\12_1_base_sinais_temporal.parquet
Entrada - indicadores agregados de amplitude da 5.5 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_5_base_indicadores_agregados_amplitude_pregao.parquet
Saída   - base intensidade dos sinais               : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12\12_2_base_intensidade_sinais.parquet
Saída   - base amplitude dos sinais                 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulaca

## Etapa 12.3) Perfil das Compras Realizadas

In [61]:
%%time
# ============================================================
# Etapa 12.3) Perfil das Compras Realizadas
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 12.3 - PERFIL DAS COMPRAS REALIZADAS")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

caminho_base_aportes_realizados = gerar_caminho_arquivo(
    etapa=10,
    subetapa=1,
    tipo_arquivo="base",
    nome="aportes_realizados",
)

caminho_base_tickers_comprados_aporte = gerar_caminho_arquivo(
    etapa=10,
    subetapa=2,
    tipo_arquivo="base",
    nome="tickers_comprados_aporte",
)

caminho_base_compras_finais = gerar_caminho_arquivo(
    etapa=10,
    subetapa=3,
    tipo_arquivo="base",
    nome="compras_finais",
)

caminho_base_intensidade_sinais = gerar_caminho_arquivo(
    etapa=12,
    subetapa=2,
    tipo_arquivo="base",
    nome="intensidade_sinais",
)

caminho_base_perfil_compras_aporte = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="base",
    nome="perfil_compras_aporte",
)

caminho_base_perfil_compras_ticker = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="base",
    nome="perfil_compras_ticker",
)

caminho_tbl_resumo_perfil_compras_estrategia = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="resumo_perfil_compras_estrategia",
)

caminho_tbl_perfil_compras_por_ano = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="perfil_compras_por_ano",
)

caminho_tbl_perfil_compras_por_regime = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="perfil_compras_por_regime",
)

caminho_tbl_ranking_aportes_por_cesta = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="ranking_aportes_por_cesta",
)

caminho_tbl_top_tickers_comprados = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="top_tickers_comprados",
)

caminho_tbl_top_empresas_compradas = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="top_empresas_compradas",
)

caminho_tbl_top_setores_comprados = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="top_setores_comprados",
)

caminho_tbl_base_grafico_perfil_compras_timeline = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="base_grafico_perfil_compras_timeline",
)

caminho_tbl_auditoria_validacao_perfil_compras = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_perfil_compras",
)

print(f"Entrada - aportes realizados da 10.1            : {caminho_base_aportes_realizados}")
print(f"Entrada - tickers comprados por aporte da 10.2  : {caminho_base_tickers_comprados_aporte}")
print(f"Entrada - compras finais da 10.3                : {caminho_base_compras_finais}")
print(f"Entrada - intensidade dos sinais da 12.2        : {caminho_base_intensidade_sinais}")
print(f"Saída   - perfil por aporte                     : {caminho_base_perfil_compras_aporte}")
print(f"Saída   - perfil por ticker comprado            : {caminho_base_perfil_compras_ticker}")
print(f"Saída   - resumo por estratégia                 : {caminho_tbl_resumo_perfil_compras_estrategia}")
print(f"Saída   - perfil por ano                        : {caminho_tbl_perfil_compras_por_ano}")
print(f"Saída   - perfil por regime                     : {caminho_tbl_perfil_compras_por_regime}")
print(f"Saída   - ranking de aportes por cesta          : {caminho_tbl_ranking_aportes_por_cesta}")
print(f"Saída   - top tickers comprados                 : {caminho_tbl_top_tickers_comprados}")
print(f"Saída   - top empresas compradas                : {caminho_tbl_top_empresas_compradas}")
print(f"Saída   - top setores comprados                 : {caminho_tbl_top_setores_comprados}")
print(f"Saída   - base gráfico timeline                 : {caminho_tbl_base_grafico_perfil_compras_timeline}")
print(f"Saída   - auditoria de validação                : {caminho_tbl_auditoria_validacao_perfil_compras}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais de aportes, compras e sinais
# ============================================================

print("\n[3/12] Carga das bases oficiais de aportes, compras e sinais...")

df_aportes = pd.read_parquet(caminho_base_aportes_realizados)
df_tickers = pd.read_parquet(caminho_base_tickers_comprados_aporte)
df_compras = pd.read_parquet(caminho_base_compras_finais)
df_intensidade = pd.read_parquet(caminho_base_intensidade_sinais)

print(f"Base de aportes realizados             : {len(df_aportes):,} linhas x {df_aportes.shape[1]:,} colunas")
print(f"Base de tickers comprados por aporte   : {len(df_tickers):,} linhas x {df_tickers.shape[1]:,} colunas")
print(f"Base de compras finais                 : {len(df_compras):,} linhas x {df_compras.shape[1]:,} colunas")
print(f"Base de intensidade dos sinais         : {len(df_intensidade):,} linhas x {df_intensidade.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

def normalizar_nome_coluna(nome_coluna):
    texto = str(nome_coluna).strip().lower()
    texto = texto.translate(str.maketrans("áàãâäéèêëíìîïóòõôöúùûüç", "aaaaaeeeeiiiiooooouuuuc"))
    texto = texto.replace("%", "pct")
    texto = texto.replace("/", "_")
    texto = texto.replace("-", "_")
    texto = texto.replace(" ", "_")
    while "__" in texto:
        texto = texto.replace("__", "_")
    return texto

def localizar_coluna_por_prioridade(df, candidatos_exatos, termos_obrigatorios=None, termos_qualquer=None, termos_excluir=None):
    mapa_normalizado = {normalizar_nome_coluna(coluna): coluna for coluna in df.columns}

    for candidato in candidatos_exatos:
        candidato_norm = normalizar_nome_coluna(candidato)
        if candidato_norm in mapa_normalizado:
            return mapa_normalizado[candidato_norm]

    termos_obrigatorios = termos_obrigatorios or []
    termos_qualquer = termos_qualquer or []
    termos_excluir = termos_excluir or []

    for coluna in df.columns:
        coluna_norm = normalizar_nome_coluna(coluna)
        atende_obrigatorios = all(termo in coluna_norm for termo in termos_obrigatorios)
        atende_qualquer = True if len(termos_qualquer) == 0 else any(termo in coluna_norm for termo in termos_qualquer)
        contem_excluido = any(termo in coluna_norm for termo in termos_excluir)
        if atende_obrigatorios and atende_qualquer and not contem_excluido:
            return coluna

    return None

def primeira_coluna_existente(df, colunas):
    for coluna in colunas:
        if coluna in df.columns:
            return coluna
    return None

def padronizar_texto(serie):
    return serie.astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan, "<NA>": np.nan})

def padronizar_estrategia_referencia(serie):
    serie_txt = padronizar_texto(serie).str.lower()
    serie_txt = serie_txt.str.replace("ç", "c", regex=False)
    serie_txt = serie_txt.str.replace("ã", "a", regex=False)
    serie_txt = serie_txt.str.replace("á", "a", regex=False)
    serie_txt = serie_txt.str.replace("à", "a", regex=False)
    serie_txt = serie_txt.str.replace("â", "a", regex=False)
    serie_txt = serie_txt.str.replace("é", "e", regex=False)
    serie_txt = serie_txt.str.replace("ê", "e", regex=False)
    serie_txt = serie_txt.str.replace("í", "i", regex=False)
    serie_txt = serie_txt.str.replace("ó", "o", regex=False)
    serie_txt = serie_txt.str.replace("ô", "o", regex=False)
    serie_txt = serie_txt.str.replace("ú", "u", regex=False)
    return serie_txt

def obter_serie_texto(df, coluna, valor_padrao=np.nan):
    if coluna is not None and coluna in df.columns:
        return padronizar_texto(df[coluna])
    return pd.Series(valor_padrao, index=df.index, dtype="object")

def obter_serie_numerica(df, coluna, valor_padrao=np.nan):
    if coluna is not None and coluna in df.columns:
        return pd.to_numeric(df[coluna], errors="coerce")
    return pd.Series(valor_padrao, index=df.index, dtype="float64")

def obter_serie_data(df, coluna):
    if coluna is not None and coluna in df.columns:
        return pd.to_datetime(df[coluna], errors="coerce")
    return pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

def detectar_colunas_base(df):
    return {
        "aporte_id": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["aporte_id", "id_aporte", "codigo_aporte", "chave_aporte", "id_ordem_aporte"],
            termos_obrigatorios=["aporte"],
            termos_qualquer=["id", "chave", "codigo"],
        ),
        "chave_estrategia": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["chave_estrategia", "estrategia_chave"],
            termos_obrigatorios=["chave", "estrategia"],
        ),
        "estrategia_id": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["estrategia_id", "id_estrategia"],
            termos_obrigatorios=["estrategia"],
            termos_qualquer=["id"],
        ),
        "estrategia_referencia": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["estrategia_referencia", "estrategia_referencia_padronizada", "tipo_sinal", "tipo_estrategia"],
            termos_obrigatorios=["estrategia"],
            termos_qualquer=["referencia", "tipo"],
        ),
        "grupo_controle": primeira_coluna_existente(df, ["grupo_controle", "categoria_estrategia", "tipo_estrategia"]),
        "familia_estrategia": primeira_coluna_existente(df, ["familia_estrategia", "subcategoria_estrategia", "estrategia_familia"]),
        "nome_exibicao": primeira_coluna_existente(df, ["nome_exibicao_estrategia", "nome_exibicao_sinal", "nome_exibicao", "estrategia_nome"]),
        "carteira_id": primeira_coluna_existente(df, ["carteira_id", "id_carteira"]),
        "replica_id": primeira_coluna_existente(df, ["replica_id", "id_replica"]),
        "ordem_aporte": primeira_coluna_existente(df, ["ordem_aporte", "ordem_sinal", "numero_aporte", "n_aporte"]),
        "data_sinal": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["data_sinal", "data_do_sinal"],
            termos_obrigatorios=["data"],
            termos_qualquer=["sinal"],
        ),
        "data_aporte": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["data_aporte", "data_efetiva_aporte", "data_compra", "data_execucao", "data_ordem"],
            termos_obrigatorios=["data"],
            termos_qualquer=["aporte", "compra", "execucao", "ordem"],
        ),
        "valor_aporte": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["valor_aporte", "valor_aporte_realizado", "valor_aporte_planejado", "valor_disponivel_aporte", "orcamento_aporte"],
            termos_obrigatorios=[],
            termos_qualquer=["valor_aporte", "orcamento_aporte"],
        ),
        "valor_investido": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["valor_investido", "valor_compra", "valor_total_compra", "valor_executado", "valor_alocado", "valor_financeiro", "valor_mercado_compra"],
            termos_obrigatorios=["valor"],
            termos_qualquer=["investido", "compra", "executado", "alocado", "financeiro"],
            termos_excluir=["aporte", "caixa", "residual", "peso"],
        ),
        "preco_compra": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["preco_compra", "preco_execucao", "preco", "close_adj", "preco_referencia"],
            termos_obrigatorios=[],
            termos_qualquer=["preco", "close_adj"],
            termos_excluir=["medio"],
        ),
        "quantidade": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["quantidade", "quantidade_comprada", "qtd_comprada", "qtd", "qtde"],
            termos_obrigatorios=[],
            termos_qualquer=["quantidade", "qtd", "qtde"],
            termos_excluir=["tickers", "ativos", "empresas"],
        ),
        "peso": localizar_coluna_por_prioridade(
            df,
            candidatos_exatos=["peso_efetivo", "peso_efetivo_ticker", "peso_compra", "peso_alocado"],
            termos_obrigatorios=["peso"],
            termos_qualquer=["efetivo", "compra", "alocado"],
        ),
        "ticker": primeira_coluna_existente(df, ["ticker", "ativo", "codigo_negociacao"]),
        "issuer_code": primeira_coluna_existente(df, ["issuer_code", "codigo_emissor", "empresa_id"]),
        "nome": primeira_coluna_existente(df, ["nome", "nome_empresa", "nom_res", "empresa"]),
        "setor": primeira_coluna_existente(df, ["setor"]),
        "subsetor": primeira_coluna_existente(df, ["subsetor", "sub_setor"]),
        "segmento": primeira_coluna_existente(df, ["segmento"]),
        "classe_instrumento": primeira_coluna_existente(df, ["classe_instrumento", "classe_ativo", "tipo_ativo"]),
    }

def construir_chave_aporte(df):
    if "aporte_id" in df.columns and df["aporte_id"].notna().any():
        return padronizar_texto(df["aporte_id"])

    partes = []
    for coluna in [
        "estrategia_referencia",
        "chave_estrategia",
        "estrategia_id",
        "carteira_id",
        "replica_id",
        "data_sinal",
        "data_aporte",
        "ordem_aporte",
    ]:
        if coluna in df.columns:
            if pd.api.types.is_datetime64_any_dtype(df[coluna]):
                partes.append(df[coluna].dt.strftime("%Y-%m-%d").fillna("na"))
            else:
                partes.append(padronizar_texto(df[coluna]).fillna("na"))

    if len(partes) == 0:
        return pd.Series([str(i) for i in df.index], index=df.index, dtype="object")

    chave = partes[0].copy()
    for parte in partes[1:]:
        chave = chave.astype(str) + "__" + parte.astype(str)
    return chave

def preparar_base_generica(df, nome_base):
    colunas = detectar_colunas_base(df)
    base = pd.DataFrame(index=df.index)

    base["origem_base"] = nome_base
    base["aporte_id"] = obter_serie_texto(df, colunas["aporte_id"])
    base["chave_estrategia"] = obter_serie_texto(df, colunas["chave_estrategia"])
    base["estrategia_id"] = obter_serie_texto(df, colunas["estrategia_id"])
    base["estrategia_referencia"] = padronizar_estrategia_referencia(obter_serie_texto(df, colunas["estrategia_referencia"]))
    base["grupo_controle"] = obter_serie_texto(df, colunas["grupo_controle"])
    base["familia_estrategia"] = obter_serie_texto(df, colunas["familia_estrategia"])
    base["nome_exibicao_estrategia"] = obter_serie_texto(df, colunas["nome_exibicao"])
    base["carteira_id"] = obter_serie_texto(df, colunas["carteira_id"])
    base["replica_id"] = obter_serie_texto(df, colunas["replica_id"])
    base["ordem_aporte"] = obter_serie_numerica(df, colunas["ordem_aporte"])
    base["data_sinal"] = obter_serie_data(df, colunas["data_sinal"])
    base["data_aporte"] = obter_serie_data(df, colunas["data_aporte"])
    base["valor_aporte"] = obter_serie_numerica(df, colunas["valor_aporte"])
    base["valor_investido_ticker"] = obter_serie_numerica(df, colunas["valor_investido"])
    base["preco_compra"] = obter_serie_numerica(df, colunas["preco_compra"])
    base["quantidade_comprada"] = obter_serie_numerica(df, colunas["quantidade"])
    base["peso_efetivo_ticker_origem"] = obter_serie_numerica(df, colunas["peso"])
    base["ticker"] = obter_serie_texto(df, colunas["ticker"])
    base["issuer_code"] = obter_serie_texto(df, colunas["issuer_code"])
    base["nome"] = obter_serie_texto(df, colunas["nome"])
    base["setor"] = obter_serie_texto(df, colunas["setor"])
    base["subsetor"] = obter_serie_texto(df, colunas["subsetor"])
    base["segmento"] = obter_serie_texto(df, colunas["segmento"])
    base["classe_instrumento"] = obter_serie_texto(df, colunas["classe_instrumento"])

    if base["estrategia_referencia"].isna().all():
        texto_busca = (
            base["chave_estrategia"].fillna("") + " " +
            base["estrategia_id"].fillna("") + " " +
            base["familia_estrategia"].fillna("") + " " +
            base["nome_exibicao_estrategia"].fillna("")
        )
        texto_busca = padronizar_estrategia_referencia(texto_busca)
        base["estrategia_referencia"] = np.select(
            [texto_busca.str.contains("capitulacao", na=False), texto_busca.str.contains("euforia", na=False)],
            ["capitulacao", "euforia"],
            default=np.nan,
        )
    else:
        texto_busca = (
            base["estrategia_referencia"].fillna("") + " " +
            base["chave_estrategia"].fillna("") + " " +
            base["estrategia_id"].fillna("") + " " +
            base["familia_estrategia"].fillna("") + " " +
            base["nome_exibicao_estrategia"].fillna("")
        )
        texto_busca = padronizar_estrategia_referencia(texto_busca)
        base["estrategia_referencia"] = np.select(
            [texto_busca.str.contains("capitulacao", na=False), texto_busca.str.contains("euforia", na=False)],
            ["capitulacao", "euforia"],
            default=base["estrategia_referencia"],
        )

    base["chave_aporte"] = construir_chave_aporte(base)

    return base, colunas

def preencher_coluna_por_prioridade(df, coluna_destino, colunas_origem):
    if coluna_destino not in df.columns:
        df[coluna_destino] = np.nan
    for coluna in colunas_origem:
        if coluna in df.columns:
            df[coluna_destino] = df[coluna_destino].where(df[coluna_destino].notna(), df[coluna])
    return df

def primeiro_valido(serie):
    valores = serie.dropna()
    if len(valores) == 0:
        return np.nan
    return valores.iloc[0]

def garantir_coluna(df, coluna, valor_padrao=np.nan):
    if coluna not in df.columns:
        df[coluna] = valor_padrao
    return df

def garantir_colunas(df, colunas, valor_padrao=np.nan):
    df = df.copy()
    for coluna in colunas:
        if coluna not in df.columns:
            df[coluna] = valor_padrao
    return df

def remover_colunas_auxiliares(df, colunas_auxiliares):
    colunas_remover = [coluna for coluna in colunas_auxiliares if coluna in df.columns]
    if len(colunas_remover) > 0:
        df = df.drop(columns=colunas_remover)
    return df

def garantir_chave_aporte(df, nome_base):
    df = df.copy()
    if "chave_aporte" not in df.columns:
        df["chave_aporte"] = construir_chave_aporte(df)
    else:
        df["chave_aporte"] = padronizar_texto(df["chave_aporte"])

    if df["chave_aporte"].isna().all():
        df["chave_aporte"] = pd.Series(
            [f"{nome_base}__linha_{i}" for i in range(len(df))],
            index=df.index,
            dtype="object",
        )
    return df

def selecionar_colunas_existentes(df, colunas_desejadas):
    return [coluna for coluna in colunas_desejadas if coluna in df.columns]

def validar_chaves_merge(df_esquerda, df_direita, chaves, nome_merge):
    chaves_ausentes_esquerda = [chave for chave in chaves if chave not in df_esquerda.columns]
    chaves_ausentes_direita = [chave for chave in chaves if chave not in df_direita.columns]

    if len(chaves_ausentes_esquerda) > 0 or len(chaves_ausentes_direita) > 0:
        print(f"ALERTA - merge não executado em {nome_merge}.")
        print(f"Chaves ausentes na base principal : {chaves_ausentes_esquerda}")
        print(f"Chaves ausentes na base auxiliar  : {chaves_ausentes_direita}")
        return False

    return True

def merge_auxiliar_sem_keyerror(df_principal, df_auxiliar, chaves, nome_merge, suffixes=("", "_aux")):
    if len(df_auxiliar) == 0:
        print(f"ALERTA - merge não executado em {nome_merge}: base auxiliar vazia.")
        return df_principal.copy(), 0

    if not validar_chaves_merge(df_principal, df_auxiliar, chaves, nome_merge):
        return df_principal.copy(), 0

    linhas_antes = len(df_principal)
    resultado = df_principal.merge(
        df_auxiliar,
        on=chaves,
        how="left",
        suffixes=suffixes,
    )
    expansao = len(resultado) - linhas_antes

    print(f"Merge {nome_merge:<32}: linhas antes={linhas_antes:,} | depois={len(resultado):,} | expansão={expansao:,}")

    if expansao != 0:
        raise ValueError(f"O merge {nome_merge} alterou a quantidade de linhas da base principal.")

    return resultado, 1

def calcular_hhi(pesos):
    pesos_num = pd.to_numeric(pesos, errors="coerce").dropna()
    if len(pesos_num) == 0:
        return np.nan
    return float((pesos_num ** 2).sum())

def calcular_top_n(pesos, n):
    pesos_num = pd.to_numeric(pesos, errors="coerce").dropna().sort_values(ascending=False)
    if len(pesos_num) == 0:
        return np.nan
    return float(pesos_num.head(n).sum())

def label_exibicao_estrategia(serie):
    return np.where(
        serie.eq("capitulacao"),
        "Capitulação",
        np.where(serie.eq("euforia"), "Euforia", serie),
    )

def montar_resumo_por_grupo(df, colunas_grupo):
    base = df.copy()
    resumo = (
        base.groupby(colunas_grupo, dropna=False)
        .agg(
            n_aportes=("chave_aporte", "nunique"),
            primeira_data_aporte=("data_aporte", "min"),
            ultima_data_aporte=("data_aporte", "max"),
            n_tickers_medio=("n_tickers_comprados", "mean"),
            n_tickers_mediano=("n_tickers_comprados", "median"),
            n_tickers_minimo=("n_tickers_comprados", "min"),
            n_tickers_maximo=("n_tickers_comprados", "max"),
            n_empresas_medio=("n_empresas_compradas", "mean"),
            n_setores_medio=("n_setores_comprados", "mean"),
            valor_aporte_medio=("valor_aporte_referencia", "mean"),
            valor_investido_medio=("valor_investido_total", "mean"),
            valor_investido_total=("valor_investido_total", "sum"),
            valor_medio_por_ticker_medio=("valor_medio_por_ticker", "mean"),
            pct_aporte_investido_medio=("pct_aporte_investido", "mean"),
            caixa_residual_medio=("caixa_residual_aporte", "mean"),
            peso_maximo_ticker_medio=("peso_maximo_ticker", "mean"),
            hhi_compras_medio=("hhi_compras_ticker", "mean"),
            top5_peso_medio=("top5_peso_ticker", "mean"),
            score_intensidade_medio=("score_intensidade_sinal", "mean"),
            percentil_intensidade_medio=("percentil_intensidade_sinal", "mean"),
        )
        .reset_index()
    )
    return resumo

def adicionar_auditoria(lista, item, valor, valor_referencia, status, observacao):
    lista.append(
        {
            "item": item,
            "valor": valor,
            "valor_referencia": valor_referencia,
            "status": status,
            "observacao": observacao,
        }
    )

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Identificação das colunas centrais nas bases de origem
# ============================================================

print("\n[5/12] Identificação das colunas centrais nas bases de origem...")

base_aportes_padronizada, colunas_aportes_detectadas = preparar_base_generica(df_aportes, "10_1_base_aportes_realizados")
base_tickers_padronizada, colunas_tickers_detectadas = preparar_base_generica(df_tickers, "10_2_base_tickers_comprados_aporte")
base_compras_padronizada, colunas_compras_detectadas = preparar_base_generica(df_compras, "10_3_base_compras_finais")

print("Colunas detectadas na base 10.1:")
for chave, coluna in colunas_aportes_detectadas.items():
    if chave in ["aporte_id", "chave_estrategia", "estrategia_referencia", "data_sinal", "data_aporte", "valor_aporte"]:
        print(f"{chave:<28}: {coluna}")

print("\nColunas detectadas na base 10.2:")
for chave, coluna in colunas_tickers_detectadas.items():
    if chave in ["ticker", "issuer_code", "nome", "setor", "subsetor", "segmento", "data_aporte"]:
        print(f"{chave:<28}: {coluna}")

print("\nColunas detectadas na base 10.3:")
for chave, coluna in colunas_compras_detectadas.items():
    if chave in ["ticker", "issuer_code", "data_aporte", "valor_aporte", "valor_investido", "preco_compra", "quantidade", "peso"]:
        print(f"{chave:<28}: {coluna}")

if base_compras_padronizada["ticker"].isna().all():
    raise ValueError("Não foi possível identificar a coluna de ticker na base 10.3 de compras finais.")
if base_compras_padronizada["valor_investido_ticker"].isna().all() and not (
    base_compras_padronizada["quantidade_comprada"].notna().any() and base_compras_padronizada["preco_compra"].notna().any()
):
    raise ValueError("Não foi possível identificar valor investido nem calcular valor por quantidade e preço na base 10.3.")

print("OK")

# ============================================================
# 6) Padronização das bases de compras e filtragem das estratégias de sinais
# ============================================================

print("\n[6/12] Padronização das bases de compras e filtragem das estratégias de sinais...")

estrategias_sinais = ["capitulacao", "euforia"]

base_aportes_sinais = base_aportes_padronizada[
    base_aportes_padronizada["estrategia_referencia"].isin(estrategias_sinais)
].copy()

base_tickers_sinais = base_tickers_padronizada[
    base_tickers_padronizada["estrategia_referencia"].isin(estrategias_sinais)
].copy()

base_compras_sinais = base_compras_padronizada[
    base_compras_padronizada["estrategia_referencia"].isin(estrategias_sinais)
].copy()

if len(base_compras_sinais) == 0:
    raise ValueError("A base 10.3 não retornou compras para as estratégias de capitulação e euforia.")

if base_compras_sinais["valor_investido_ticker"].isna().any():
    valor_calculado = base_compras_sinais["quantidade_comprada"] * base_compras_sinais["preco_compra"]
    base_compras_sinais["valor_investido_ticker"] = base_compras_sinais["valor_investido_ticker"].where(
        base_compras_sinais["valor_investido_ticker"].notna(),
        valor_calculado,
    )

if base_compras_sinais["data_aporte"].isna().all() and base_compras_sinais["data_sinal"].notna().any():
    base_compras_sinais["data_aporte"] = base_compras_sinais["data_sinal"]

base_compras_sinais["ano_aporte"] = base_compras_sinais["data_aporte"].dt.year
base_compras_sinais["ano_sinal"] = base_compras_sinais["data_sinal"].dt.year
base_compras_sinais["nome_exibicao_estrategia"] = np.where(
    base_compras_sinais["nome_exibicao_estrategia"].notna(),
    base_compras_sinais["nome_exibicao_estrategia"],
    label_exibicao_estrategia(base_compras_sinais["estrategia_referencia"]),
)

print(f"Aportes de sinais na base 10.1          : {len(base_aportes_sinais):,}")
print(f"Tickers de sinais na base 10.2          : {len(base_tickers_sinais):,}")
print(f"Compras de sinais na base 10.3          : {len(base_compras_sinais):,}")
print(f"Estratégias encontradas                 : {base_compras_sinais['estrategia_referencia'].nunique(dropna=True):,}")
print(f"Valor investido ausente após tratamento : {int(base_compras_sinais['valor_investido_ticker'].isna().sum()):,}")
print(f"Coluna chave_aporte em 10.1             : {'chave_aporte' in base_aportes_sinais.columns}")
print(f"Coluna chave_aporte em 10.2             : {'chave_aporte' in base_tickers_sinais.columns}")
print(f"Coluna chave_aporte em 10.3             : {'chave_aporte' in base_compras_sinais.columns}")
print("OK")

# ============================================================
# 7) Enriquecimento das compras com metadados dos aportes e dos sinais
# ============================================================

print("\n[7/12] Enriquecimento das compras com metadados dos aportes e dos sinais...")

base_compras_sinais = garantir_chave_aporte(base_compras_sinais, "10_3_base_compras_finais")
base_aportes_sinais = garantir_chave_aporte(base_aportes_sinais, "10_1_base_aportes_realizados")
base_tickers_sinais = garantir_chave_aporte(base_tickers_sinais, "10_2_base_tickers_comprados_aporte")

base_compras_enriquecida = base_compras_sinais.copy()

colunas_aportes_para_merge = [
    "chave_aporte",
    "estrategia_referencia",
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "nome_exibicao_estrategia",
    "carteira_id",
    "replica_id",
    "ordem_aporte",
    "data_sinal",
    "data_aporte",
    "valor_aporte",
]
colunas_aportes_para_merge = selecionar_colunas_existentes(base_aportes_sinais, colunas_aportes_para_merge)
base_aportes_merge = base_aportes_sinais[colunas_aportes_para_merge].copy()

if "chave_aporte" in base_aportes_merge.columns:
    base_aportes_merge = base_aportes_merge.drop_duplicates(subset=["chave_aporte"], keep="first")

base_compras_enriquecida, flag_merge_aportes = merge_auxiliar_sem_keyerror(
    base_compras_enriquecida,
    base_aportes_merge,
    ["chave_aporte"],
    "metadados dos aportes",
    suffixes=("", "_aporte"),
)

for coluna in [
    "estrategia_referencia",
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "nome_exibicao_estrategia",
    "carteira_id",
    "replica_id",
    "ordem_aporte",
    "data_sinal",
    "data_aporte",
    "valor_aporte",
]:
    base_compras_enriquecida = preencher_coluna_por_prioridade(
        base_compras_enriquecida,
        coluna,
        [f"{coluna}_aporte"],
    )

colunas_auxiliares_aporte = [
    f"{coluna}_aporte"
    for coluna in [
        "estrategia_referencia",
        "chave_estrategia",
        "estrategia_id",
        "grupo_controle",
        "familia_estrategia",
        "nome_exibicao_estrategia",
        "carteira_id",
        "replica_id",
        "ordem_aporte",
        "data_sinal",
        "data_aporte",
        "valor_aporte",
    ]
]
base_compras_enriquecida = remover_colunas_auxiliares(base_compras_enriquecida, colunas_auxiliares_aporte)

base_compras_enriquecida = garantir_colunas(
    base_compras_enriquecida,
    [
        "chave_aporte",
        "estrategia_referencia",
        "data_sinal",
        "data_aporte",
        "valor_aporte",
        "valor_investido_ticker",
        "quantidade_comprada",
        "preco_compra",
    ],
)

colunas_tickers_para_merge = [
    "chave_aporte",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "classe_instrumento",
]
colunas_tickers_para_merge = selecionar_colunas_existentes(base_tickers_sinais, colunas_tickers_para_merge)
base_tickers_merge = base_tickers_sinais[colunas_tickers_para_merge].copy()

if all(coluna in base_tickers_merge.columns for coluna in ["chave_aporte", "ticker"]):
    base_tickers_merge = base_tickers_merge.drop_duplicates(subset=["chave_aporte", "ticker"], keep="first")

base_compras_enriquecida, flag_merge_tickers = merge_auxiliar_sem_keyerror(
    base_compras_enriquecida,
    base_tickers_merge,
    ["chave_aporte", "ticker"],
    "metadados dos tickers",
    suffixes=("", "_ticker"),
)

for coluna in ["issuer_code", "nome", "setor", "subsetor", "segmento", "classe_instrumento"]:
    base_compras_enriquecida = preencher_coluna_por_prioridade(
        base_compras_enriquecida,
        coluna,
        [f"{coluna}_ticker"],
    )

colunas_auxiliares_ticker = [
    f"{coluna}_ticker"
    for coluna in ["issuer_code", "nome", "setor", "subsetor", "segmento", "classe_instrumento"]
]
base_compras_enriquecida = remover_colunas_auxiliares(base_compras_enriquecida, colunas_auxiliares_ticker)

base_compras_enriquecida = garantir_colunas(
    base_compras_enriquecida,
    [
        "chave_aporte",
        "ticker",
        "issuer_code",
        "nome",
        "setor",
        "subsetor",
        "segmento",
        "classe_instrumento",
        "data_sinal",
        "data_aporte",
        "ano_sinal",
        "ano_aporte",
    ],
)

base_intensidade_merge = df_intensidade.copy()
if "data_sinal" not in base_intensidade_merge.columns:
    raise ValueError("A base 12.2 deve conter a coluna 'data_sinal'.")
if "estrategia_referencia" not in base_intensidade_merge.columns:
    raise ValueError("A base 12.2 deve conter a coluna 'estrategia_referencia'.")

base_intensidade_merge["data_sinal"] = pd.to_datetime(base_intensidade_merge["data_sinal"], errors="coerce")
base_intensidade_merge["estrategia_referencia"] = padronizar_estrategia_referencia(base_intensidade_merge["estrategia_referencia"])

colunas_intensidade_preferenciais = [
    "estrategia_referencia",
    "data_sinal",
    "ano_sinal",
    "ano_mes_sinal",
    "regime_mercado",
    "pct_minima_52s",
    "pct_maxima_52s",
    "pct_acima_mm200",
    "pct_abaixo_mm200",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
]
colunas_intensidade_disponiveis = [coluna for coluna in colunas_intensidade_preferenciais if coluna in base_intensidade_merge.columns]
base_intensidade_merge = base_intensidade_merge[colunas_intensidade_disponiveis].drop_duplicates(
    subset=["estrategia_referencia", "data_sinal"],
    keep="first",
)

base_compras_enriquecida = garantir_colunas(
    base_compras_enriquecida,
    ["estrategia_referencia", "data_sinal", "data_aporte", "ano_sinal", "ano_aporte"],
)
base_compras_enriquecida["data_sinal"] = pd.to_datetime(base_compras_enriquecida["data_sinal"], errors="coerce")
base_compras_enriquecida["data_aporte"] = pd.to_datetime(base_compras_enriquecida["data_aporte"], errors="coerce")

# Identificação explícita das estratégias reais de sinal.
# As carteiras aleatórias herdam a referência capitulação/euforia, mas não possuem score de intensidade próprio.
texto_nome_exibicao = padronizar_estrategia_referencia(base_compras_enriquecida["nome_exibicao_estrategia"].fillna(""))
base_compras_enriquecida["tipo_perfil_estrategia"] = np.where(
    texto_nome_exibicao.isin(["capitulacao", "euforia"]),
    "estrategia_real_sinal",
    "controle_aleatorio",
)
base_compras_enriquecida["flag_estrategia_real_sinal"] = base_compras_enriquecida["tipo_perfil_estrategia"].eq("estrategia_real_sinal")

base_compras_enriquecida, flag_merge_intensidade = merge_auxiliar_sem_keyerror(
    base_compras_enriquecida,
    base_intensidade_merge,
    ["estrategia_referencia", "data_sinal"],
    "intensidade dos sinais",
    suffixes=("", "_sinal"),
)

base_compras_enriquecida = garantir_colunas(
    base_compras_enriquecida,
    [
        "regime_mercado",
        "score_intensidade_sinal",
        "percentil_intensidade_sinal",
        "classe_intensidade_sinal",
        "flag_sinal_intenso",
        "flag_sinal_muito_intenso",
        "ano_sinal",
        "ano_aporte",
    ],
)
colunas_intensidade_para_controles = [
    "pct_minima_52s",
    "pct_maxima_52s",
    "pct_acima_mm200",
    "pct_abaixo_mm200",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
]
mascara_controle_aleatorio = ~base_compras_enriquecida["flag_estrategia_real_sinal"].astype(bool)
for coluna_intensidade_controle in colunas_intensidade_para_controles:
    if coluna_intensidade_controle in base_compras_enriquecida.columns:
        base_compras_enriquecida.loc[mascara_controle_aleatorio, coluna_intensidade_controle] = np.nan

base_compras_enriquecida["regime_mercado"] = base_compras_enriquecida["regime_mercado"].fillna("sem_regime")
base_compras_enriquecida.loc[mascara_controle_aleatorio, "regime_mercado"] = "sem_regime"
base_compras_enriquecida["data_aporte"] = pd.to_datetime(base_compras_enriquecida["data_aporte"], errors="coerce")
base_compras_enriquecida["data_sinal"] = pd.to_datetime(base_compras_enriquecida["data_sinal"], errors="coerce")
base_compras_enriquecida["ano_aporte"] = pd.to_numeric(base_compras_enriquecida["ano_aporte"], errors="coerce")
base_compras_enriquecida["ano_aporte"] = base_compras_enriquecida["ano_aporte"].where(
    base_compras_enriquecida["ano_aporte"].notna(),
    base_compras_enriquecida["data_aporte"].dt.year,
)
base_compras_enriquecida["ano_sinal"] = pd.to_numeric(base_compras_enriquecida["ano_sinal"], errors="coerce")
base_compras_enriquecida["ano_sinal"] = base_compras_enriquecida["ano_sinal"].where(
    base_compras_enriquecida["ano_sinal"].notna(),
    base_compras_enriquecida["data_sinal"].dt.year,
)

print(f"Compras enriquecidas                    : {len(base_compras_enriquecida):,} linhas")
print(f"Compras sem dados de intensidade        : {int(base_compras_enriquecida['score_intensidade_sinal'].isna().sum()) if 'score_intensidade_sinal' in base_compras_enriquecida.columns else len(base_compras_enriquecida):,}")
print(f"Compras sem issuer_code                 : {int(base_compras_enriquecida['issuer_code'].isna().sum()):,}")
print(f"Compras de estratégias reais de sinal   : {int(base_compras_enriquecida['flag_estrategia_real_sinal'].sum()):,}")
print(f"Compras de controles aleatórios         : {int((~base_compras_enriquecida['flag_estrategia_real_sinal']).sum()):,}")
print("OK")

# ============================================================
# 8) Consolidação da base de compras por ticker e aporte
# ============================================================

print("\n[8/12] Consolidação da base de compras por ticker e aporte...")

for coluna in ["issuer_code", "nome", "setor", "subsetor", "segmento", "classe_instrumento"]:
    if coluna in base_compras_enriquecida.columns:
        base_compras_enriquecida[coluna] = base_compras_enriquecida[coluna].fillna("não_classificado")

colunas_identificacao_ticker = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "carteira_id",
    "replica_id",
    "chave_aporte",
    "ordem_aporte",
    "data_sinal",
    "data_aporte",
    "ano_sinal",
    "ano_aporte",
    "regime_mercado",
    "tipo_perfil_estrategia",
    "flag_estrategia_real_sinal",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
    "ticker",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "classe_instrumento",
]
colunas_identificacao_ticker = [coluna for coluna in colunas_identificacao_ticker if coluna in base_compras_enriquecida.columns]

base_perfil_compras_ticker = (
    base_compras_enriquecida.groupby(colunas_identificacao_ticker, dropna=False)
    .agg(
        quantidade_comprada=("quantidade_comprada", "sum"),
        valor_investido_ticker=("valor_investido_ticker", "sum"),
        valor_aporte_referencia=("valor_aporte", primeiro_valido),
        preco_compra_medio_origem=("preco_compra", "mean"),
    )
    .reset_index()
)

base_perfil_compras_ticker["preco_medio_ponderado_compra"] = np.where(
    base_perfil_compras_ticker["quantidade_comprada"].abs() > 0,
    base_perfil_compras_ticker["valor_investido_ticker"] / base_perfil_compras_ticker["quantidade_comprada"],
    base_perfil_compras_ticker["preco_compra_medio_origem"],
)

base_perfil_compras_ticker["valor_total_aporte_calculado"] = base_perfil_compras_ticker.groupby("chave_aporte")["valor_investido_ticker"].transform("sum")
base_perfil_compras_ticker["peso_efetivo_ticker"] = np.where(
    base_perfil_compras_ticker["valor_total_aporte_calculado"] > 0,
    base_perfil_compras_ticker["valor_investido_ticker"] / base_perfil_compras_ticker["valor_total_aporte_calculado"],
    np.nan,
)

base_perfil_compras_ticker = base_perfil_compras_ticker.sort_values(
    ["estrategia_referencia", "data_aporte", "chave_aporte", "peso_efetivo_ticker", "ticker"],
    ascending=[True, True, True, False, True],
).reset_index(drop=True)

print(f"Base de perfil por ticker               : {len(base_perfil_compras_ticker):,} linhas")
print(f"Aportes distintos com compras           : {base_perfil_compras_ticker['chave_aporte'].nunique(dropna=True):,}")
print(f"Tickers distintos comprados             : {base_perfil_compras_ticker['ticker'].nunique(dropna=True):,}")
print("OK")

# ============================================================
# 9) Construção do perfil consolidado por aporte
# ============================================================

print("\n[9/12] Construção do perfil consolidado por aporte...")

colunas_identificacao_aporte = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "carteira_id",
    "replica_id",
    "chave_aporte",
    "ordem_aporte",
    "data_sinal",
    "data_aporte",
    "ano_sinal",
    "ano_aporte",
    "regime_mercado",
    "tipo_perfil_estrategia",
    "flag_estrategia_real_sinal",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
]
colunas_identificacao_aporte = [coluna for coluna in colunas_identificacao_aporte if coluna in base_perfil_compras_ticker.columns]

base_perfil_compras_aporte = (
    base_perfil_compras_ticker.groupby(colunas_identificacao_aporte, dropna=False)
    .agg(
        n_linhas_compra=("ticker", "size"),
        n_tickers_comprados=("ticker", "nunique"),
        n_empresas_compradas=("issuer_code", "nunique"),
        n_setores_comprados=("setor", "nunique"),
        n_subsetores_comprados=("subsetor", "nunique"),
        n_segmentos_comprados=("segmento", "nunique"),
        valor_aporte_referencia=("valor_aporte_referencia", primeiro_valido),
        valor_investido_total=("valor_investido_ticker", "sum"),
        valor_medio_por_ticker=("valor_investido_ticker", "mean"),
        valor_mediano_por_ticker=("valor_investido_ticker", "median"),
        valor_minimo_ticker=("valor_investido_ticker", "min"),
        valor_maximo_ticker=("valor_investido_ticker", "max"),
        quantidade_total_comprada=("quantidade_comprada", "sum"),
        preco_medio_simples=("preco_medio_ponderado_compra", "mean"),
        preco_mediano=("preco_medio_ponderado_compra", "median"),
        peso_maximo_ticker=("peso_efetivo_ticker", "max"),
        peso_minimo_ticker=("peso_efetivo_ticker", "min"),
        peso_medio_ticker=("peso_efetivo_ticker", "mean"),
        hhi_compras_ticker=("peso_efetivo_ticker", calcular_hhi),
        top3_peso_ticker=("peso_efetivo_ticker", lambda s: calcular_top_n(s, 3)),
        top5_peso_ticker=("peso_efetivo_ticker", lambda s: calcular_top_n(s, 5)),
    )
    .reset_index()
)

base_perfil_compras_aporte["valor_aporte_referencia"] = base_perfil_compras_aporte["valor_aporte_referencia"].where(
    base_perfil_compras_aporte["valor_aporte_referencia"].notna(),
    base_perfil_compras_aporte["valor_investido_total"],
)
base_perfil_compras_aporte["caixa_residual_aporte"] = base_perfil_compras_aporte["valor_aporte_referencia"] - base_perfil_compras_aporte["valor_investido_total"]
base_perfil_compras_aporte["pct_aporte_investido"] = np.where(
    base_perfil_compras_aporte["valor_aporte_referencia"] > 0,
    base_perfil_compras_aporte["valor_investido_total"] / base_perfil_compras_aporte["valor_aporte_referencia"],
    np.nan,
)
base_perfil_compras_aporte["pct_caixa_residual"] = np.where(
    base_perfil_compras_aporte["valor_aporte_referencia"] > 0,
    base_perfil_compras_aporte["caixa_residual_aporte"] / base_perfil_compras_aporte["valor_aporte_referencia"],
    np.nan,
)

idx_maior_ticker = base_perfil_compras_ticker.groupby("chave_aporte")["valor_investido_ticker"].idxmax()
base_maior_ticker = base_perfil_compras_ticker.loc[
    idx_maior_ticker,
    ["chave_aporte", "ticker", "issuer_code", "nome", "setor", "valor_investido_ticker", "peso_efetivo_ticker"],
].rename(
    columns={
        "ticker": "ticker_maior_compra",
        "issuer_code": "issuer_code_maior_compra",
        "nome": "nome_maior_compra",
        "setor": "setor_ticker_maior_compra",
        "valor_investido_ticker": "valor_ticker_maior_compra",
        "peso_efetivo_ticker": "peso_ticker_maior_compra",
    }
)

base_setor_aporte = (
    base_perfil_compras_ticker.groupby(["chave_aporte", "setor"], dropna=False)["valor_investido_ticker"]
    .sum()
    .reset_index()
)
base_setor_aporte["valor_total_aporte"] = base_setor_aporte.groupby("chave_aporte")["valor_investido_ticker"].transform("sum")
base_setor_aporte["peso_setor"] = np.where(
    base_setor_aporte["valor_total_aporte"] > 0,
    base_setor_aporte["valor_investido_ticker"] / base_setor_aporte["valor_total_aporte"],
    np.nan,
)
idx_maior_setor = base_setor_aporte.groupby("chave_aporte")["peso_setor"].idxmax()
base_maior_setor = base_setor_aporte.loc[
    idx_maior_setor,
    ["chave_aporte", "setor", "valor_investido_ticker", "peso_setor"],
].rename(
    columns={
        "setor": "setor_maior_peso",
        "valor_investido_ticker": "valor_setor_maior_peso",
        "peso_setor": "peso_setor_maior",
    }
)

base_perfil_compras_aporte = base_perfil_compras_aporte.merge(base_maior_ticker, on="chave_aporte", how="left")
base_perfil_compras_aporte = base_perfil_compras_aporte.merge(base_maior_setor, on="chave_aporte", how="left")

base_perfil_compras_aporte = base_perfil_compras_aporte.sort_values(
    ["estrategia_referencia", "data_aporte", "chave_aporte"]
).reset_index(drop=True)

print(f"Base de perfil por aporte               : {len(base_perfil_compras_aporte):,} linhas")
print(f"Média de tickers por aporte             : {base_perfil_compras_aporte['n_tickers_comprados'].mean():.2f}")
print(f"Média de empresas por aporte            : {base_perfil_compras_aporte['n_empresas_compradas'].mean():.2f}")
print(f"Percentual médio investido por aporte   : {base_perfil_compras_aporte['pct_aporte_investido'].mean():.6f}")
print("OK")

# ============================================================
# 10) Construção dos resumos, rankings e bases para gráficos
# ============================================================

print("\n[10/12] Construção dos resumos, rankings e bases para gráficos...")

resumo_perfil_compras_estrategia = montar_resumo_por_grupo(
    base_perfil_compras_aporte,
    ["estrategia_referencia", "nome_exibicao_estrategia"],
)

perfil_compras_por_ano = montar_resumo_por_grupo(
    base_perfil_compras_aporte,
    ["estrategia_referencia", "nome_exibicao_estrategia", "ano_aporte"],
)

perfil_compras_por_regime = montar_resumo_por_grupo(
    base_perfil_compras_aporte,
    ["estrategia_referencia", "nome_exibicao_estrategia", "regime_mercado"],
)

ranking_aportes_por_cesta = base_perfil_compras_aporte.copy()
ranking_aportes_por_cesta["ranking_tamanho_cesta_estrategia"] = (
    ranking_aportes_por_cesta.groupby("estrategia_referencia")["n_tickers_comprados"]
    .rank(method="first", ascending=False)
)
ranking_aportes_por_cesta["ranking_valor_investido_estrategia"] = (
    ranking_aportes_por_cesta.groupby("estrategia_referencia")["valor_investido_total"]
    .rank(method="first", ascending=False)
)
ranking_aportes_por_cesta = ranking_aportes_por_cesta.sort_values(
    ["estrategia_referencia", "ranking_tamanho_cesta_estrategia", "data_aporte"]
).reset_index(drop=True)

colunas_ranking_aportes = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "ranking_tamanho_cesta_estrategia",
    "ranking_valor_investido_estrategia",
    "ordem_aporte",
    "data_sinal",
    "data_aporte",
    "ano_aporte",
    "regime_mercado",
    "n_tickers_comprados",
    "n_empresas_compradas",
    "n_setores_comprados",
    "valor_aporte_referencia",
    "valor_investido_total",
    "valor_medio_por_ticker",
    "pct_aporte_investido",
    "peso_maximo_ticker",
    "hhi_compras_ticker",
    "ticker_maior_compra",
    "setor_maior_peso",
    "score_intensidade_sinal",
    "classe_intensidade_sinal",
]
colunas_ranking_aportes = [coluna for coluna in colunas_ranking_aportes if coluna in ranking_aportes_por_cesta.columns]
ranking_aportes_por_cesta = ranking_aportes_por_cesta[colunas_ranking_aportes].copy()

top_tickers_comprados = (
    base_perfil_compras_ticker.groupby(
        ["estrategia_referencia", "nome_exibicao_estrategia", "ticker", "issuer_code", "nome", "setor", "subsetor", "segmento"],
        dropna=False,
    )
    .agg(
        n_aportes_com_compra=("chave_aporte", "nunique"),
        valor_investido_total=("valor_investido_ticker", "sum"),
        quantidade_total=("quantidade_comprada", "sum"),
        valor_medio_por_aporte=("valor_investido_ticker", "mean"),
        peso_medio_no_aporte=("peso_efetivo_ticker", "mean"),
        primeira_data_compra=("data_aporte", "min"),
        ultima_data_compra=("data_aporte", "max"),
    )
    .reset_index()
)
top_tickers_comprados["ranking_ticker_estrategia"] = (
    top_tickers_comprados.groupby("estrategia_referencia")["valor_investido_total"]
    .rank(method="first", ascending=False)
)
top_tickers_comprados = top_tickers_comprados.sort_values(
    ["estrategia_referencia", "ranking_ticker_estrategia"]
).reset_index(drop=True)

top_empresas_compradas = (
    base_perfil_compras_ticker.groupby(
        ["estrategia_referencia", "nome_exibicao_estrategia", "issuer_code", "nome", "setor", "subsetor", "segmento"],
        dropna=False,
    )
    .agg(
        n_tickers_comprados=("ticker", "nunique"),
        n_aportes_com_compra=("chave_aporte", "nunique"),
        valor_investido_total=("valor_investido_ticker", "sum"),
        quantidade_total=("quantidade_comprada", "sum"),
        valor_medio_por_aporte=("valor_investido_ticker", "mean"),
        peso_medio_no_aporte=("peso_efetivo_ticker", "mean"),
        primeira_data_compra=("data_aporte", "min"),
        ultima_data_compra=("data_aporte", "max"),
    )
    .reset_index()
)
top_empresas_compradas["ranking_empresa_estrategia"] = (
    top_empresas_compradas.groupby("estrategia_referencia")["valor_investido_total"]
    .rank(method="first", ascending=False)
)
top_empresas_compradas = top_empresas_compradas.sort_values(
    ["estrategia_referencia", "ranking_empresa_estrategia"]
).reset_index(drop=True)

top_setores_comprados = (
    base_perfil_compras_ticker.groupby(
        ["estrategia_referencia", "nome_exibicao_estrategia", "setor"],
        dropna=False,
    )
    .agg(
        n_aportes_com_compra=("chave_aporte", "nunique"),
        n_empresas_compradas=("issuer_code", "nunique"),
        n_tickers_comprados=("ticker", "nunique"),
        valor_investido_total=("valor_investido_ticker", "sum"),
        valor_medio_por_compra=("valor_investido_ticker", "mean"),
        peso_medio_no_aporte=("peso_efetivo_ticker", "mean"),
    )
    .reset_index()
)
top_setores_comprados["ranking_setor_estrategia"] = (
    top_setores_comprados.groupby("estrategia_referencia")["valor_investido_total"]
    .rank(method="first", ascending=False)
)
top_setores_comprados = top_setores_comprados.sort_values(
    ["estrategia_referencia", "ranking_setor_estrategia"]
).reset_index(drop=True)

base_grafico_perfil_compras_timeline = base_perfil_compras_aporte.copy()
colunas_timeline = [
    "data_sinal",
    "data_aporte",
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "ano_aporte",
    "regime_mercado",
    "n_tickers_comprados",
    "n_empresas_compradas",
    "n_setores_comprados",
    "valor_aporte_referencia",
    "valor_investido_total",
    "valor_medio_por_ticker",
    "pct_aporte_investido",
    "caixa_residual_aporte",
    "peso_maximo_ticker",
    "hhi_compras_ticker",
    "top5_peso_ticker",
    "ticker_maior_compra",
    "setor_maior_peso",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
]
colunas_timeline = [coluna for coluna in colunas_timeline if coluna in base_grafico_perfil_compras_timeline.columns]
base_grafico_perfil_compras_timeline = base_grafico_perfil_compras_timeline[colunas_timeline].copy()

print(f"Resumo por estratégia                  : {len(resumo_perfil_compras_estrategia):,} linhas")
print(f"Perfil por ano                         : {len(perfil_compras_por_ano):,} linhas")
print(f"Perfil por regime                      : {len(perfil_compras_por_regime):,} linhas")
print(f"Ranking de aportes por cesta           : {len(ranking_aportes_por_cesta):,} linhas")
print(f"Top tickers comprados                  : {len(top_tickers_comprados):,} linhas")
print(f"Top empresas compradas                 : {len(top_empresas_compradas):,} linhas")
print(f"Top setores comprados                  : {len(top_setores_comprados):,} linhas")
print("OK")

# ============================================================
# 11) Auditoria de validação
# ============================================================

print("\n[11/12] Construção da auditoria de validação...")

auditoria = []

n_aportes_sinais_origem = base_aportes_sinais["chave_aporte"].nunique(dropna=True)
n_aportes_com_compras = base_perfil_compras_aporte["chave_aporte"].nunique(dropna=True)
n_estrategias = base_perfil_compras_aporte["estrategia_referencia"].nunique(dropna=True)
compras_sem_valor = int(base_perfil_compras_ticker["valor_investido_ticker"].isna().sum())
compras_valor_nao_positivo = int((base_perfil_compras_ticker["valor_investido_ticker"].fillna(0) <= 0).sum())
compras_sem_ticker = int(base_perfil_compras_ticker["ticker"].isna().sum())
compras_sem_empresa = int(base_perfil_compras_ticker["issuer_code"].isna().sum())
duplicatas_ticker_aporte = int(base_perfil_compras_ticker.duplicated(subset=["chave_aporte", "ticker"]).sum())
aportes_sem_compras = int(max(n_aportes_sinais_origem - n_aportes_com_compras, 0))
aportes_sem_data = int(base_perfil_compras_aporte["data_aporte"].isna().sum())
if "flag_estrategia_real_sinal" in base_perfil_compras_aporte.columns:
    mascara_aportes_reais_sinal = base_perfil_compras_aporte["flag_estrategia_real_sinal"].astype(bool)
else:
    texto_nome_auditoria = padronizar_estrategia_referencia(base_perfil_compras_aporte["nome_exibicao_estrategia"].fillna(""))
    mascara_aportes_reais_sinal = texto_nome_auditoria.isin(["capitulacao", "euforia"])

n_aportes_reais_sinal = int(mascara_aportes_reais_sinal.sum())
aportes_reais_sem_intensidade = int(base_perfil_compras_aporte.loc[mascara_aportes_reais_sinal, "score_intensidade_sinal"].isna().sum()) if "score_intensidade_sinal" in base_perfil_compras_aporte.columns else n_aportes_reais_sinal
aportes_controles_sem_intensidade = int(base_perfil_compras_aporte.loc[~mascara_aportes_reais_sinal, "score_intensidade_sinal"].isna().sum()) if "score_intensidade_sinal" in base_perfil_compras_aporte.columns else int((~mascara_aportes_reais_sinal).sum())
aportes_valor_investido_acima_aporte = int(
    (
        base_perfil_compras_aporte["valor_investido_total"] >
        base_perfil_compras_aporte["valor_aporte_referencia"] * 1.000001
    ).sum()
)

colunas_numericas_aporte = base_perfil_compras_aporte.select_dtypes(include=[np.number]).columns.tolist()
metricas_infinitas = int(np.isinf(base_perfil_compras_aporte[colunas_numericas_aporte].to_numpy(dtype=float)).sum()) if len(colunas_numericas_aporte) > 0 else 0

adicionar_auditoria(
    auditoria,
    "estrategias_com_compras",
    n_estrategias,
    2,
    "OK" if n_estrategias == 2 else "ERRO",
    "A base deve conter compras das estratégias de capitulação e euforia.",
)

adicionar_auditoria(
    auditoria,
    "aportes_sinais_origem_10_1",
    n_aportes_sinais_origem,
    "informativo",
    "OK",
    "Quantidade de aportes de capitulação e euforia identificados na base 10.1.",
)

adicionar_auditoria(
    auditoria,
    "aportes_com_compras_12_3",
    n_aportes_com_compras,
    n_aportes_sinais_origem,
    "OK" if n_aportes_com_compras == n_aportes_sinais_origem else "ERRO",
    "Cada aporte de sinal deve possuir pelo menos uma compra efetiva associada.",
)

adicionar_auditoria(
    auditoria,
    "aportes_sem_compras",
    aportes_sem_compras,
    0,
    "OK" if aportes_sem_compras == 0 else "ERRO",
    "Não deve haver aporte de sinal sem compras finais registradas.",
)

adicionar_auditoria(
    auditoria,
    "compras_sem_ticker",
    compras_sem_ticker,
    0,
    "OK" if compras_sem_ticker == 0 else "ERRO",
    "Toda compra deve estar vinculada a um ticker.",
)

adicionar_auditoria(
    auditoria,
    "compras_sem_empresa",
    compras_sem_empresa,
    0,
    "OK" if compras_sem_empresa == 0 else "ERRO",
    "Toda compra deve estar vinculada a um issuer_code.",
)

adicionar_auditoria(
    auditoria,
    "compras_sem_valor_investido",
    compras_sem_valor,
    0,
    "OK" if compras_sem_valor == 0 else "ERRO",
    "Toda compra deve possuir valor investido calculável.",
)

adicionar_auditoria(
    auditoria,
    "compras_com_valor_nao_positivo",
    compras_valor_nao_positivo,
    0,
    "OK" if compras_valor_nao_positivo == 0 else "ERRO",
    "Compras finais devem ter valor investido positivo.",
)

adicionar_auditoria(
    auditoria,
    "duplicatas_ticker_aporte",
    duplicatas_ticker_aporte,
    0,
    "OK" if duplicatas_ticker_aporte == 0 else "ERRO",
    "Após a consolidação, cada combinação de aporte e ticker deve aparecer uma única vez.",
)

adicionar_auditoria(
    auditoria,
    "aportes_sem_data_aporte",
    aportes_sem_data,
    0,
    "OK" if aportes_sem_data == 0 else "ERRO",
    "Todo aporte consolidado deve possuir data efetiva de aporte.",
)

adicionar_auditoria(
    auditoria,
    "aportes_reais_sinal_sem_intensidade",
    aportes_reais_sem_intensidade,
    0,
    "OK" if aportes_reais_sem_intensidade == 0 else "ERRO",
    "Apenas os aportes reais de Capitulação e Euforia devem herdar a intensidade calculada na 12.2.",
)

adicionar_auditoria(
    auditoria,
    "aportes_controles_sem_intensidade",
    aportes_controles_sem_intensidade,
    "informativo",
    "OK",
    "Controles aleatórios não possuem intensidade própria de sinal e são mantidos como informação complementar.",
)

adicionar_auditoria(
    auditoria,
    "aportes_com_valor_investido_acima_aporte",
    aportes_valor_investido_acima_aporte,
    0,
    "OK" if aportes_valor_investido_acima_aporte == 0 else "ERRO",
    "O valor investido total por aporte não deve superar o valor de referência do aporte, salvo diferença residual irrelevante.",
)

adicionar_auditoria(
    auditoria,
    "metricas_infinitas",
    metricas_infinitas,
    0,
    "OK" if metricas_infinitas == 0 else "ERRO",
    "As métricas numéricas consolidadas não devem conter valores infinitos.",
)

tbl_auditoria_validacao_perfil_compras = pd.DataFrame(auditoria)

# Padronização explícita da auditoria para evitar tipos mistos no salvamento em parquet.
# A coluna valor_referencia pode combinar números e rótulos informativos.
for coluna_auditoria_texto in ["item", "valor", "valor_referencia", "status", "observacao"]:
    if coluna_auditoria_texto in tbl_auditoria_validacao_perfil_compras.columns:
        tbl_auditoria_validacao_perfil_compras[coluna_auditoria_texto] = (
            tbl_auditoria_validacao_perfil_compras[coluna_auditoria_texto]
            .astype("object")
            .where(tbl_auditoria_validacao_perfil_compras[coluna_auditoria_texto].notna(), np.nan)
        )
        tbl_auditoria_validacao_perfil_compras[coluna_auditoria_texto] = (
            tbl_auditoria_validacao_perfil_compras[coluna_auditoria_texto]
            .map(lambda valor: str(valor) if pd.notna(valor) else np.nan)
        )

erros_bloqueantes = int((tbl_auditoria_validacao_perfil_compras["status"] == "ERRO").sum())

print(f"Itens de auditoria                       : {len(tbl_auditoria_validacao_perfil_compras):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(base_perfil_compras_aporte, caminho_base_perfil_compras_aporte, index=False)
salvar_dataframe(base_perfil_compras_ticker, caminho_base_perfil_compras_ticker, index=False)
salvar_dataframe(resumo_perfil_compras_estrategia, caminho_tbl_resumo_perfil_compras_estrategia, index=False)
salvar_dataframe(perfil_compras_por_ano, caminho_tbl_perfil_compras_por_ano, index=False)
salvar_dataframe(perfil_compras_por_regime, caminho_tbl_perfil_compras_por_regime, index=False)
salvar_dataframe(ranking_aportes_por_cesta, caminho_tbl_ranking_aportes_por_cesta, index=False)
salvar_dataframe(top_tickers_comprados, caminho_tbl_top_tickers_comprados, index=False)
salvar_dataframe(top_empresas_compradas, caminho_tbl_top_empresas_compradas, index=False)
salvar_dataframe(top_setores_comprados, caminho_tbl_top_setores_comprados, index=False)
salvar_dataframe(base_grafico_perfil_compras_timeline, caminho_tbl_base_grafico_perfil_compras_timeline, index=False)
salvar_dataframe(tbl_auditoria_validacao_perfil_compras, caminho_tbl_auditoria_validacao_perfil_compras, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação do perfil das compras:")
print(tbl_auditoria_validacao_perfil_compras.to_string(index=False))

print("\nResumo do perfil de compras por estratégia:")
print(resumo_perfil_compras_estrategia.to_string(index=False))

print("\nPerfil de compras por regime de mercado:")
print(perfil_compras_por_regime.to_string(index=False))

print("\nRanking de aportes por tamanho de cesta - amostra:")
print(ranking_aportes_por_cesta.head(30).to_string(index=False))

print("\nTop tickers comprados - amostra:")
print(top_tickers_comprados.head(30).to_string(index=False))

print("\nTop empresas compradas - amostra:")
print(top_empresas_compradas.head(30).to_string(index=False))

print("\nBase de perfil por aporte - amostra:")
colunas_amostra_aporte = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "data_sinal",
    "data_aporte",
    "regime_mercado",
    "n_tickers_comprados",
    "n_empresas_compradas",
    "n_setores_comprados",
    "valor_aporte_referencia",
    "valor_investido_total",
    "valor_medio_por_ticker",
    "pct_aporte_investido",
    "caixa_residual_aporte",
    "peso_maximo_ticker",
    "hhi_compras_ticker",
    "ticker_maior_compra",
    "setor_maior_peso",
    "score_intensidade_sinal",
    "classe_intensidade_sinal",
]
colunas_amostra_aporte = [coluna for coluna in colunas_amostra_aporte if coluna in base_perfil_compras_aporte.columns]
print(base_perfil_compras_aporte[colunas_amostra_aporte].head(30).to_string(index=False))

print("\nArquivos salvos na subetapa 12.3:")
for caminho in [
    caminho_base_perfil_compras_aporte,
    caminho_base_perfil_compras_ticker,
    caminho_tbl_resumo_perfil_compras_estrategia,
    caminho_tbl_perfil_compras_por_ano,
    caminho_tbl_perfil_compras_por_regime,
    caminho_tbl_ranking_aportes_por_cesta,
    caminho_tbl_top_tickers_comprados,
    caminho_tbl_top_empresas_compradas,
    caminho_tbl_top_setores_comprados,
    caminho_tbl_base_grafico_perfil_compras_timeline,
    caminho_tbl_auditoria_validacao_perfil_compras,
]:
    print(f"- {caminho}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 12.3 encontrou erros bloqueantes. Verifique a tabela de auditoria impressa acima.")

print("\nETAPA 12.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 12.3 - PERFIL DAS COMPRAS REALIZADAS

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - aportes realizados da 10.1            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_1_base_aportes_realizados.parquet
Entrada - tickers comprados por aporte da 10.2  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_2_base_tickers_comprados_aporte.parquet
Entrada - compras finais da 10.3                : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10\10_3_base_compras_finais.parquet
Entrada - intensidade dos sinais da 12.2        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_1

## Etapa 12.4) Concentração por Empresa

In [62]:
%%time
# ============================================================
# Etapa 12.4) Concentração por Empresa
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 12.4 - CONCENTRAÇÃO POR EMPRESA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/10] Definição determinística dos caminhos de entrada e saída...")

caminho_base_perfil_compras_ticker = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="base",
    nome="perfil_compras_ticker",
)

caminho_base_perfil_compras_aporte = gerar_caminho_arquivo(
    etapa=12,
    subetapa=3,
    tipo_arquivo="base",
    nome="perfil_compras_aporte",
)

caminho_base_concentracao_empresa_aporte = gerar_caminho_arquivo(
    etapa=12,
    subetapa=4,
    tipo_arquivo="base",
    nome="concentracao_empresa_aporte",
)

caminho_base_concentracao_empresa_resumo_aporte = gerar_caminho_arquivo(
    etapa=12,
    subetapa=4,
    tipo_arquivo="base",
    nome="concentracao_empresa_resumo_aporte",
)

caminho_tbl_resumo_concentracao_empresa_estrategia = gerar_caminho_arquivo(
    etapa=12,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="resumo_concentracao_empresa_estrategia",
)

caminho_tbl_concentracao_empresa_por_ano = gerar_caminho_arquivo(
    etapa=12,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="concentracao_empresa_por_ano",
)

caminho_tbl_concentracao_empresa_por_regime = gerar_caminho_arquivo(
    etapa=12,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="concentracao_empresa_por_regime",
)

caminho_tbl_top_empresas_concentracao = gerar_caminho_arquivo(
    etapa=12,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="top_empresas_concentracao",
)

caminho_tbl_base_grafico_concentracao_empresa_timeline = gerar_caminho_arquivo(
    etapa=12,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="base_grafico_concentracao_empresa_timeline",
)

caminho_tbl_base_grafico_top_empresas = gerar_caminho_arquivo(
    etapa=12,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="base_grafico_top_empresas",
)

caminho_tbl_auditoria_validacao_concentracao_empresa = gerar_caminho_arquivo(
    etapa=12,
    subetapa=4,
    tipo_arquivo="tbl",
    nome="auditoria_validacao_concentracao_empresa",
)

print(f"Entrada - perfil por ticker da 12.3             : {caminho_base_perfil_compras_ticker}")
print(f"Entrada - perfil por aporte da 12.3             : {caminho_base_perfil_compras_aporte}")
print(f"Saída   - concentração empresa por aporte       : {caminho_base_concentracao_empresa_aporte}")
print(f"Saída   - resumo concentração por aporte        : {caminho_base_concentracao_empresa_resumo_aporte}")
print(f"Saída   - resumo por estratégia                 : {caminho_tbl_resumo_concentracao_empresa_estrategia}")
print(f"Saída   - concentração por ano                  : {caminho_tbl_concentracao_empresa_por_ano}")
print(f"Saída   - concentração por regime               : {caminho_tbl_concentracao_empresa_por_regime}")
print(f"Saída   - top empresas                          : {caminho_tbl_top_empresas_concentracao}")
print(f"Saída   - base gráfico timeline                 : {caminho_tbl_base_grafico_concentracao_empresa_timeline}")
print(f"Saída   - base gráfico top empresas             : {caminho_tbl_base_grafico_top_empresas}")
print(f"Saída   - auditoria de validação                : {caminho_tbl_auditoria_validacao_concentracao_empresa}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da 12.3
# ============================================================

print("\n[3/10] Carga das bases oficiais da 12.3...")

df_perfil_ticker = pd.read_parquet(caminho_base_perfil_compras_ticker)
df_perfil_aporte = pd.read_parquet(caminho_base_perfil_compras_aporte)

print(f"Base de perfil por ticker da 12.3       : {len(df_perfil_ticker):,} linhas x {df_perfil_ticker.shape[1]:,} colunas")
print(f"Base de perfil por aporte da 12.3       : {len(df_perfil_aporte):,} linhas x {df_perfil_aporte.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/10] Funções auxiliares da subetapa...")

def garantir_colunas_obrigatorias(df, colunas, nome_base):
    colunas_ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"A base {nome_base} não contém as colunas obrigatórias: {colunas_ausentes}")


def garantir_coluna(df, coluna, valor_padrao=np.nan):
    if coluna not in df.columns:
        df[coluna] = valor_padrao
    return df


def selecionar_colunas_existentes(df, colunas):
    return [coluna for coluna in colunas if coluna in df.columns]


def primeiro_valido(serie):
    valores = serie.dropna()
    if len(valores) == 0:
        return np.nan
    return valores.iloc[0]


def soma_top_n(serie, n):
    valores = pd.to_numeric(serie, errors="coerce").dropna().sort_values(ascending=False)
    if len(valores) == 0:
        return np.nan
    return float(valores.head(n).sum())


def calcular_hhi(serie):
    valores = pd.to_numeric(serie, errors="coerce").dropna()
    if len(valores) == 0:
        return np.nan
    return float((valores ** 2).sum())


def calcular_entropia_normalizada(serie):
    pesos = pd.to_numeric(serie, errors="coerce").dropna()
    pesos = pesos[pesos > 0]
    if len(pesos) <= 1:
        return 0.0 if len(pesos) == 1 else np.nan
    entropia = float(-(pesos * np.log(pesos)).sum())
    return float(entropia / np.log(len(pesos)))


def adicionar_auditoria(lista, item, valor, valor_referencia, status, observacao):
    lista.append(
        {
            "item": item,
            "valor": valor,
            "valor_referencia": valor_referencia,
            "status": status,
            "observacao": observacao,
        }
    )


def padronizar_auditoria_para_parquet(df):
    df = df.copy()
    for coluna in ["item", "valor", "valor_referencia", "status", "observacao"]:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("object").where(df[coluna].notna(), np.nan)
            df[coluna] = df[coluna].map(lambda valor: str(valor) if pd.notna(valor) else np.nan)
    return df


def montar_resumo_metricas_concentracao(df, colunas_grupo):
    resumo = (
        df.groupby(colunas_grupo, dropna=False)
        .agg(
            n_aportes=("chave_aporte", "nunique"),
            primeira_data_aporte=("data_aporte", "min"),
            ultima_data_aporte=("data_aporte", "max"),
            n_empresas_medio=("n_empresas", "mean"),
            n_empresas_mediano=("n_empresas", "median"),
            n_empresas_minimo=("n_empresas", "min"),
            n_empresas_maximo=("n_empresas", "max"),
            peso_top1_empresa_medio=("peso_top1_empresa", "mean"),
            peso_top1_empresa_mediano=("peso_top1_empresa", "median"),
            peso_top1_empresa_maximo=("peso_top1_empresa", "max"),
            peso_top3_empresas_medio=("peso_top3_empresas", "mean"),
            peso_top5_empresas_medio=("peso_top5_empresas", "mean"),
            hhi_empresa_medio=("hhi_empresa", "mean"),
            hhi_empresa_mediano=("hhi_empresa", "median"),
            entropia_empresa_media=("entropia_empresa_normalizada", "mean"),
            valor_investido_total=("valor_investido_total_aporte", "sum"),
            score_intensidade_medio=("score_intensidade_sinal", "mean"),
            percentil_intensidade_medio=("percentil_intensidade_sinal", "mean"),
        )
        .reset_index()
    )
    return resumo

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Validação e padronização das colunas de entrada
# ============================================================

print("\n[5/10] Validação e padronização das colunas de entrada...")

colunas_obrigatorias_ticker = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "chave_aporte",
    "data_aporte",
    "issuer_code",
    "ticker",
    "valor_investido_ticker",
]

garantir_colunas_obrigatorias(
    df_perfil_ticker,
    colunas_obrigatorias_ticker,
    "12_3_base_perfil_compras_ticker",
)

garantir_colunas_obrigatorias(
    df_perfil_aporte,
    ["chave_aporte"],
    "12_3_base_perfil_compras_aporte",
)

base_ticker = df_perfil_ticker.copy()
base_aporte = df_perfil_aporte.copy()

colunas_opcionais_ticker = [
    "data_sinal",
    "ano_sinal",
    "ano_aporte",
    "regime_mercado",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "classe_instrumento",
    "valor_aporte_referencia",
    "valor_total_aporte_calculado",
    "peso_efetivo_ticker",
]

for coluna in colunas_opcionais_ticker:
    base_ticker = garantir_coluna(base_ticker, coluna)

base_ticker["data_aporte"] = pd.to_datetime(base_ticker["data_aporte"], errors="coerce")
base_ticker["data_sinal"] = pd.to_datetime(base_ticker["data_sinal"], errors="coerce")
base_ticker["ano_aporte"] = pd.to_numeric(base_ticker["ano_aporte"], errors="coerce")
base_ticker["ano_aporte"] = base_ticker["ano_aporte"].where(
    base_ticker["ano_aporte"].notna(),
    base_ticker["data_aporte"].dt.year,
)
base_ticker["ano_sinal"] = pd.to_numeric(base_ticker["ano_sinal"], errors="coerce")
base_ticker["ano_sinal"] = base_ticker["ano_sinal"].where(
    base_ticker["ano_sinal"].notna(),
    base_ticker["data_sinal"].dt.year,
)

for coluna in ["valor_investido_ticker", "valor_aporte_referencia", "valor_total_aporte_calculado", "peso_efetivo_ticker", "score_intensidade_sinal", "percentil_intensidade_sinal"]:
    base_ticker[coluna] = pd.to_numeric(base_ticker[coluna], errors="coerce")

for coluna in ["issuer_code", "ticker", "nome", "setor", "subsetor", "segmento", "classe_instrumento", "regime_mercado"]:
    base_ticker[coluna] = base_ticker[coluna].astype("object").where(base_ticker[coluna].notna(), "não_classificado")

base_ticker["estrategia_referencia"] = base_ticker["estrategia_referencia"].astype(str).str.strip()
base_ticker["nome_exibicao_estrategia"] = base_ticker["nome_exibicao_estrategia"].astype(str).str.strip()
base_ticker["chave_aporte"] = base_ticker["chave_aporte"].astype(str).str.strip()

if base_ticker["peso_efetivo_ticker"].isna().all():
    base_ticker["valor_total_aporte_calculado"] = base_ticker.groupby("chave_aporte")["valor_investido_ticker"].transform("sum")
    base_ticker["peso_efetivo_ticker"] = np.where(
        base_ticker["valor_total_aporte_calculado"] > 0,
        base_ticker["valor_investido_ticker"] / base_ticker["valor_total_aporte_calculado"],
        np.nan,
    )

base_ticker = base_ticker[base_ticker["valor_investido_ticker"].notna()].copy()

print(f"Linhas válidas para concentração        : {len(base_ticker):,}")
print(f"Aportes distintos                       : {base_ticker['chave_aporte'].nunique(dropna=True):,}")
print(f"Empresas distintas                      : {base_ticker['issuer_code'].nunique(dropna=True):,}")
print(f"Estratégias de referência               : {base_ticker['estrategia_referencia'].nunique(dropna=True):,}")
print(f"Nomes de estratégia                     : {base_ticker['nome_exibicao_estrategia'].nunique(dropna=True):,}")
print("OK")

# ============================================================
# 6) Construção da base de concentração por empresa e aporte
# ============================================================

print("\n[6/10] Construção da base de concentração por empresa e aporte...")

colunas_identificacao_empresa = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "chave_aporte",
    "data_sinal",
    "data_aporte",
    "ano_sinal",
    "ano_aporte",
    "regime_mercado",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
    "issuer_code",
    "nome",
    "setor",
    "subsetor",
    "segmento",
    "classe_instrumento",
]
colunas_identificacao_empresa = selecionar_colunas_existentes(base_ticker, colunas_identificacao_empresa)

base_concentracao_empresa_aporte = (
    base_ticker.groupby(colunas_identificacao_empresa, dropna=False)
    .agg(
        n_tickers_empresa=("ticker", "nunique"),
        tickers_empresa=("ticker", lambda s: ", ".join(sorted(s.dropna().astype(str).unique()))),
        valor_investido_empresa=("valor_investido_ticker", "sum"),
        peso_empresa_origem=("peso_efetivo_ticker", "sum"),
        valor_aporte_referencia=("valor_aporte_referencia", primeiro_valido),
        valor_total_aporte_calculado=("valor_total_aporte_calculado", primeiro_valido),
    )
    .reset_index()
)

base_concentracao_empresa_aporte["valor_total_aporte_calculado"] = base_concentracao_empresa_aporte["valor_total_aporte_calculado"].where(
    base_concentracao_empresa_aporte["valor_total_aporte_calculado"].notna(),
    base_concentracao_empresa_aporte.groupby("chave_aporte")["valor_investido_empresa"].transform("sum"),
)
base_concentracao_empresa_aporte["valor_aporte_referencia"] = base_concentracao_empresa_aporte["valor_aporte_referencia"].where(
    base_concentracao_empresa_aporte["valor_aporte_referencia"].notna(),
    base_concentracao_empresa_aporte["valor_total_aporte_calculado"],
)
base_concentracao_empresa_aporte["peso_empresa_aporte"] = np.where(
    base_concentracao_empresa_aporte["valor_total_aporte_calculado"] > 0,
    base_concentracao_empresa_aporte["valor_investido_empresa"] / base_concentracao_empresa_aporte["valor_total_aporte_calculado"],
    base_concentracao_empresa_aporte["peso_empresa_origem"],
)

base_concentracao_empresa_aporte["ranking_empresa_no_aporte"] = (
    base_concentracao_empresa_aporte.groupby("chave_aporte")["peso_empresa_aporte"]
    .rank(method="first", ascending=False)
)

base_concentracao_empresa_aporte = base_concentracao_empresa_aporte.sort_values(
    ["estrategia_referencia", "nome_exibicao_estrategia", "data_aporte", "chave_aporte", "ranking_empresa_no_aporte", "issuer_code"]
).reset_index(drop=True)

print(f"Base empresa por aporte                 : {len(base_concentracao_empresa_aporte):,} linhas")
print(f"Aportes distintos                       : {base_concentracao_empresa_aporte['chave_aporte'].nunique(dropna=True):,}")
print(f"Empresas distintas                      : {base_concentracao_empresa_aporte['issuer_code'].nunique(dropna=True):,}")
print("OK")

# ============================================================
# 7) Construção dos indicadores de concentração por aporte
# ============================================================

print("\n[7/10] Construção dos indicadores de concentração por aporte...")

colunas_identificacao_aporte = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "chave_aporte",
    "data_sinal",
    "data_aporte",
    "ano_sinal",
    "ano_aporte",
    "regime_mercado",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
]
colunas_identificacao_aporte = selecionar_colunas_existentes(base_concentracao_empresa_aporte, colunas_identificacao_aporte)

base_concentracao_empresa_resumo_aporte = (
    base_concentracao_empresa_aporte.groupby(colunas_identificacao_aporte, dropna=False)
    .agg(
        n_empresas=("issuer_code", "nunique"),
        n_linhas_empresa=("issuer_code", "size"),
        valor_investido_total_aporte=("valor_investido_empresa", "sum"),
        valor_aporte_referencia=("valor_aporte_referencia", primeiro_valido),
        peso_top1_empresa=("peso_empresa_aporte", "max"),
        peso_top3_empresas=("peso_empresa_aporte", lambda s: soma_top_n(s, 3)),
        peso_top5_empresas=("peso_empresa_aporte", lambda s: soma_top_n(s, 5)),
        hhi_empresa=("peso_empresa_aporte", calcular_hhi),
        entropia_empresa_normalizada=("peso_empresa_aporte", calcular_entropia_normalizada),
    )
    .reset_index()
)

base_top_empresa = (
    base_concentracao_empresa_aporte.sort_values(
        ["chave_aporte", "peso_empresa_aporte", "issuer_code"],
        ascending=[True, False, True],
    )
    .drop_duplicates(subset=["chave_aporte"], keep="first")
    [["chave_aporte", "issuer_code", "nome", "setor", "peso_empresa_aporte", "valor_investido_empresa"]]
    .rename(
        columns={
            "issuer_code": "issuer_code_maior_peso_empresa",
            "nome": "nome_maior_peso_empresa",
            "setor": "setor_maior_peso_empresa",
            "peso_empresa_aporte": "peso_maior_empresa",
            "valor_investido_empresa": "valor_maior_empresa",
        }
    )
)

base_concentracao_empresa_resumo_aporte = base_concentracao_empresa_resumo_aporte.merge(
    base_top_empresa,
    on="chave_aporte",
    how="left",
)

base_concentracao_empresa_resumo_aporte["pct_aporte_investido"] = np.where(
    base_concentracao_empresa_resumo_aporte["valor_aporte_referencia"] > 0,
    base_concentracao_empresa_resumo_aporte["valor_investido_total_aporte"] / base_concentracao_empresa_resumo_aporte["valor_aporte_referencia"],
    np.nan,
)

print(f"Resumo empresa por aporte               : {len(base_concentracao_empresa_resumo_aporte):,} linhas")
print(f"Média de empresas por aporte            : {base_concentracao_empresa_resumo_aporte['n_empresas'].mean():.2f}")
print(f"Peso médio da maior empresa             : {base_concentracao_empresa_resumo_aporte['peso_top1_empresa'].mean():.6f}")
print(f"HHI médio por empresa                   : {base_concentracao_empresa_resumo_aporte['hhi_empresa'].mean():.6f}")
print("OK")

# ============================================================
# 8) Construção dos resumos, rankings e bases para gráficos
# ============================================================

print("\n[8/10] Construção dos resumos, rankings e bases para gráficos...")

resumo_concentracao_empresa_estrategia = montar_resumo_metricas_concentracao(
    base_concentracao_empresa_resumo_aporte,
    ["estrategia_referencia", "nome_exibicao_estrategia"],
)

n_empresas_distintas_estrategia = (
    base_concentracao_empresa_aporte.groupby(["estrategia_referencia", "nome_exibicao_estrategia"], dropna=False)["issuer_code"]
    .nunique()
    .reset_index(name="n_empresas_distintas")
)
resumo_concentracao_empresa_estrategia = resumo_concentracao_empresa_estrategia.merge(
    n_empresas_distintas_estrategia,
    on=["estrategia_referencia", "nome_exibicao_estrategia"],
    how="left",
)

concentracao_empresa_por_ano = montar_resumo_metricas_concentracao(
    base_concentracao_empresa_resumo_aporte,
    ["estrategia_referencia", "nome_exibicao_estrategia", "ano_aporte"],
)

n_empresas_distintas_ano = (
    base_concentracao_empresa_aporte.groupby(["estrategia_referencia", "nome_exibicao_estrategia", "ano_aporte"], dropna=False)["issuer_code"]
    .nunique()
    .reset_index(name="n_empresas_distintas")
)
concentracao_empresa_por_ano = concentracao_empresa_por_ano.merge(
    n_empresas_distintas_ano,
    on=["estrategia_referencia", "nome_exibicao_estrategia", "ano_aporte"],
    how="left",
)

concentracao_empresa_por_regime = montar_resumo_metricas_concentracao(
    base_concentracao_empresa_resumo_aporte,
    ["estrategia_referencia", "nome_exibicao_estrategia", "regime_mercado"],
)

n_empresas_distintas_regime = (
    base_concentracao_empresa_aporte.groupby(["estrategia_referencia", "nome_exibicao_estrategia", "regime_mercado"], dropna=False)["issuer_code"]
    .nunique()
    .reset_index(name="n_empresas_distintas")
)
concentracao_empresa_por_regime = concentracao_empresa_por_regime.merge(
    n_empresas_distintas_regime,
    on=["estrategia_referencia", "nome_exibicao_estrategia", "regime_mercado"],
    how="left",
)

totais_aportes_por_estrategia = (
    base_concentracao_empresa_resumo_aporte.groupby(["estrategia_referencia", "nome_exibicao_estrategia"], dropna=False)["chave_aporte"]
    .nunique()
    .reset_index(name="total_aportes_estrategia")
)

top_empresas_concentracao = (
    base_concentracao_empresa_aporte.groupby(
        ["estrategia_referencia", "nome_exibicao_estrategia", "issuer_code", "nome", "setor", "subsetor", "segmento"],
        dropna=False,
    )
    .agg(
        n_aportes_com_exposicao=("chave_aporte", "nunique"),
        primeira_data_aporte=("data_aporte", "min"),
        ultima_data_aporte=("data_aporte", "max"),
        valor_investido_total=("valor_investido_empresa", "sum"),
        peso_medio_quando_presente=("peso_empresa_aporte", "mean"),
        peso_mediano_quando_presente=("peso_empresa_aporte", "median"),
        peso_maximo=("peso_empresa_aporte", "max"),
        ranking_medio_no_aporte=("ranking_empresa_no_aporte", "mean"),
        n_tickers_empresa=("n_tickers_empresa", "max"),
    )
    .reset_index()
)

top_empresas_concentracao = top_empresas_concentracao.merge(
    totais_aportes_por_estrategia,
    on=["estrategia_referencia", "nome_exibicao_estrategia"],
    how="left",
)
top_empresas_concentracao["pct_aportes_com_exposicao"] = np.where(
    top_empresas_concentracao["total_aportes_estrategia"] > 0,
    top_empresas_concentracao["n_aportes_com_exposicao"] / top_empresas_concentracao["total_aportes_estrategia"],
    np.nan,
)
top_empresas_concentracao["score_ranking_empresa"] = (
    top_empresas_concentracao["valor_investido_total"].rank(method="average", ascending=False, pct=True) +
    top_empresas_concentracao["n_aportes_com_exposicao"].rank(method="average", ascending=False, pct=True) +
    top_empresas_concentracao["peso_medio_quando_presente"].rank(method="average", ascending=False, pct=True)
) / 3

top_empresas_concentracao["ranking_valor_estrategia"] = (
    top_empresas_concentracao.groupby(["estrategia_referencia", "nome_exibicao_estrategia"])["valor_investido_total"]
    .rank(method="first", ascending=False)
)
top_empresas_concentracao["ranking_frequencia_estrategia"] = (
    top_empresas_concentracao.groupby(["estrategia_referencia", "nome_exibicao_estrategia"])["n_aportes_com_exposicao"]
    .rank(method="first", ascending=False)
)
top_empresas_concentracao["ranking_peso_medio_estrategia"] = (
    top_empresas_concentracao.groupby(["estrategia_referencia", "nome_exibicao_estrategia"])["peso_medio_quando_presente"]
    .rank(method="first", ascending=False)
)

top_empresas_concentracao = top_empresas_concentracao.sort_values(
    ["estrategia_referencia", "nome_exibicao_estrategia", "ranking_valor_estrategia", "ranking_frequencia_estrategia"]
).reset_index(drop=True)

base_grafico_concentracao_empresa_timeline = base_concentracao_empresa_resumo_aporte.copy()
colunas_timeline = [
    "data_sinal",
    "data_aporte",
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "ano_aporte",
    "regime_mercado",
    "n_empresas",
    "peso_top1_empresa",
    "peso_top3_empresas",
    "peso_top5_empresas",
    "hhi_empresa",
    "entropia_empresa_normalizada",
    "issuer_code_maior_peso_empresa",
    "nome_maior_peso_empresa",
    "setor_maior_peso_empresa",
    "peso_maior_empresa",
    "valor_maior_empresa",
    "valor_investido_total_aporte",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
]
base_grafico_concentracao_empresa_timeline = base_grafico_concentracao_empresa_timeline[selecionar_colunas_existentes(base_grafico_concentracao_empresa_timeline, colunas_timeline)].copy()

base_grafico_top_empresas = top_empresas_concentracao.copy()
colunas_top_empresas_grafico = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "issuer_code",
    "nome",
    "setor",
    "n_aportes_com_exposicao",
    "pct_aportes_com_exposicao",
    "valor_investido_total",
    "peso_medio_quando_presente",
    "peso_maximo",
    "ranking_valor_estrategia",
    "ranking_frequencia_estrategia",
    "ranking_peso_medio_estrategia",
]
base_grafico_top_empresas = base_grafico_top_empresas[selecionar_colunas_existentes(base_grafico_top_empresas, colunas_top_empresas_grafico)].copy()

print(f"Resumo por estratégia                  : {len(resumo_concentracao_empresa_estrategia):,} linhas")
print(f"Concentração por ano                   : {len(concentracao_empresa_por_ano):,} linhas")
print(f"Concentração por regime                : {len(concentracao_empresa_por_regime):,} linhas")
print(f"Top empresas concentração              : {len(top_empresas_concentracao):,} linhas")
print(f"Base gráfico timeline                  : {len(base_grafico_concentracao_empresa_timeline):,} linhas")
print(f"Base gráfico top empresas              : {len(base_grafico_top_empresas):,} linhas")
print("OK")

# ============================================================
# 9) Auditoria de validação
# ============================================================

print("\n[9/10] Construção da auditoria de validação...")

auditoria = []

n_aportes_origem = int(base_aporte["chave_aporte"].nunique(dropna=True))
n_aportes_concentracao = int(base_concentracao_empresa_resumo_aporte["chave_aporte"].nunique(dropna=True))
n_estrategias_referencia = int(base_concentracao_empresa_resumo_aporte["estrategia_referencia"].nunique(dropna=True))
n_nomes_estrategia = int(base_concentracao_empresa_resumo_aporte["nome_exibicao_estrategia"].nunique(dropna=True))
empresas_ausentes = int(base_concentracao_empresa_aporte["issuer_code"].isna().sum())
empresas_nao_classificadas = int((base_concentracao_empresa_aporte["issuer_code"].astype(str) == "não_classificado").sum())
duplicatas_empresa_aporte = int(base_concentracao_empresa_aporte.duplicated(subset=["chave_aporte", "issuer_code"]).sum())
aportes_sem_data = int(base_concentracao_empresa_resumo_aporte["data_aporte"].isna().sum())
aportes_sem_empresas = int((base_concentracao_empresa_resumo_aporte["n_empresas"].fillna(0) <= 0).sum())
valores_nao_positivos = int((base_concentracao_empresa_aporte["valor_investido_empresa"].fillna(0) <= 0).sum())

soma_pesos_empresa = (
    base_concentracao_empresa_aporte.groupby("chave_aporte", dropna=False)["peso_empresa_aporte"]
    .sum()
    .reset_index(name="soma_peso_empresa")
)
maior_desvio_soma_pesos = float((soma_pesos_empresa["soma_peso_empresa"] - 1).abs().max()) if len(soma_pesos_empresa) > 0 else np.nan
aportes_soma_peso_inconsistente = int(((soma_pesos_empresa["soma_peso_empresa"] - 1).abs() > 0.0001).sum()) if len(soma_pesos_empresa) > 0 else 0

colunas_numericas_resumo = base_concentracao_empresa_resumo_aporte.select_dtypes(include=[np.number]).columns.tolist()
metricas_infinitas = int(np.isinf(base_concentracao_empresa_resumo_aporte[colunas_numericas_resumo].to_numpy(dtype=float)).sum()) if len(colunas_numericas_resumo) > 0 else 0

adicionar_auditoria(
    auditoria,
    "aportes_origem_12_3",
    n_aportes_origem,
    "informativo",
    "OK",
    "Quantidade de aportes identificados na base 12.3 de perfil por aporte.",
)

adicionar_auditoria(
    auditoria,
    "aportes_concentracao_empresa_12_4",
    n_aportes_concentracao,
    n_aportes_origem,
    "OK" if n_aportes_concentracao == n_aportes_origem else "ERRO",
    "A concentração por empresa deve preservar os aportes consolidados na 12.3.",
)

adicionar_auditoria(
    auditoria,
    "estrategias_referencia",
    n_estrategias_referencia,
    2,
    "OK" if n_estrategias_referencia == 2 else "ERRO",
    "A base deve manter as estratégias de referência de capitulação e euforia.",
)

adicionar_auditoria(
    auditoria,
    "nomes_estrategia",
    n_nomes_estrategia,
    "informativo",
    "OK",
    "Quantidade de nomes de estratégia incluindo estratégias reais e controles aleatórios.",
)

adicionar_auditoria(
    auditoria,
    "empresas_ausentes",
    empresas_ausentes,
    0,
    "OK" if empresas_ausentes == 0 else "ERRO",
    "Toda exposição por empresa deve possuir issuer_code.",
)

adicionar_auditoria(
    auditoria,
    "empresas_nao_classificadas",
    empresas_nao_classificadas,
    0,
    "OK" if empresas_nao_classificadas == 0 else "ERRO",
    "A concentração empresarial não deve depender de empresas não classificadas.",
)

adicionar_auditoria(
    auditoria,
    "duplicatas_empresa_aporte",
    duplicatas_empresa_aporte,
    0,
    "OK" if duplicatas_empresa_aporte == 0 else "ERRO",
    "Cada combinação de aporte e empresa deve aparecer uma única vez após agregação.",
)

adicionar_auditoria(
    auditoria,
    "aportes_sem_data_aporte",
    aportes_sem_data,
    0,
    "OK" if aportes_sem_data == 0 else "ERRO",
    "Todo aporte consolidado deve possuir data efetiva de aporte.",
)

adicionar_auditoria(
    auditoria,
    "aportes_sem_empresas",
    aportes_sem_empresas,
    0,
    "OK" if aportes_sem_empresas == 0 else "ERRO",
    "Todo aporte deve possuir pelo menos uma empresa com exposição.",
)

adicionar_auditoria(
    auditoria,
    "valores_empresa_nao_positivos",
    valores_nao_positivos,
    0,
    "OK" if valores_nao_positivos == 0 else "ERRO",
    "As exposições por empresa devem ter valor investido positivo.",
)

adicionar_auditoria(
    auditoria,
    "aportes_soma_peso_empresa_inconsistente",
    aportes_soma_peso_inconsistente,
    0,
    "OK" if aportes_soma_peso_inconsistente == 0 else "ERRO",
    "A soma dos pesos por empresa dentro de cada aporte deve ser próxima de 100%.",
)

adicionar_auditoria(
    auditoria,
    "maior_desvio_soma_pesos_empresa",
    maior_desvio_soma_pesos,
    "<= 0.0001",
    "OK" if pd.notna(maior_desvio_soma_pesos) and maior_desvio_soma_pesos <= 0.0001 else "ERRO",
    "Maior desvio absoluto da soma dos pesos empresariais por aporte.",
)

adicionar_auditoria(
    auditoria,
    "metricas_infinitas",
    metricas_infinitas,
    0,
    "OK" if metricas_infinitas == 0 else "ERRO",
    "As métricas numéricas consolidadas não devem conter valores infinitos.",
)

tbl_auditoria_validacao_concentracao_empresa = padronizar_auditoria_para_parquet(pd.DataFrame(auditoria))
erros_bloqueantes = int((tbl_auditoria_validacao_concentracao_empresa["status"] == "ERRO").sum())

print(f"Itens de auditoria                       : {len(tbl_auditoria_validacao_concentracao_empresa):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 10) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[10/10] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(base_concentracao_empresa_aporte, caminho_base_concentracao_empresa_aporte, index=False)
salvar_dataframe(base_concentracao_empresa_resumo_aporte, caminho_base_concentracao_empresa_resumo_aporte, index=False)
salvar_dataframe(resumo_concentracao_empresa_estrategia, caminho_tbl_resumo_concentracao_empresa_estrategia, index=False)
salvar_dataframe(concentracao_empresa_por_ano, caminho_tbl_concentracao_empresa_por_ano, index=False)
salvar_dataframe(concentracao_empresa_por_regime, caminho_tbl_concentracao_empresa_por_regime, index=False)
salvar_dataframe(top_empresas_concentracao, caminho_tbl_top_empresas_concentracao, index=False)
salvar_dataframe(base_grafico_concentracao_empresa_timeline, caminho_tbl_base_grafico_concentracao_empresa_timeline, index=False)
salvar_dataframe(base_grafico_top_empresas, caminho_tbl_base_grafico_top_empresas, index=False)
salvar_dataframe(tbl_auditoria_validacao_concentracao_empresa, caminho_tbl_auditoria_validacao_concentracao_empresa, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da concentração por empresa:")
print(tbl_auditoria_validacao_concentracao_empresa.to_string(index=False))

print("\nResumo de concentração por empresa e estratégia:")
print(resumo_concentracao_empresa_estrategia.to_string(index=False))

print("\nConcentração por empresa e ano - amostra:")
print(concentracao_empresa_por_ano.head(40).to_string(index=False))

print("\nConcentração por empresa e regime de mercado:")
print(concentracao_empresa_por_regime.to_string(index=False))

print("\nTop empresas por concentração - amostra:")
colunas_top_amostra = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "issuer_code",
    "nome",
    "setor",
    "n_aportes_com_exposicao",
    "pct_aportes_com_exposicao",
    "valor_investido_total",
    "peso_medio_quando_presente",
    "peso_maximo",
    "ranking_valor_estrategia",
    "ranking_frequencia_estrategia",
]
colunas_top_amostra = selecionar_colunas_existentes(top_empresas_concentracao, colunas_top_amostra)
print(top_empresas_concentracao[colunas_top_amostra].head(40).to_string(index=False))

print("\nBase de concentração por aporte - amostra:")
colunas_aporte_amostra = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "data_aporte",
    "regime_mercado",
    "n_empresas",
    "peso_top1_empresa",
    "peso_top3_empresas",
    "peso_top5_empresas",
    "hhi_empresa",
    "entropia_empresa_normalizada",
    "issuer_code_maior_peso_empresa",
    "nome_maior_peso_empresa",
    "peso_maior_empresa",
    "score_intensidade_sinal",
    "classe_intensidade_sinal",
]
colunas_aporte_amostra = selecionar_colunas_existentes(base_concentracao_empresa_resumo_aporte, colunas_aporte_amostra)
print(base_concentracao_empresa_resumo_aporte[colunas_aporte_amostra].head(40).to_string(index=False))

print("\nArquivos salvos na subetapa 12.4:")
for caminho in [
    caminho_base_concentracao_empresa_aporte,
    caminho_base_concentracao_empresa_resumo_aporte,
    caminho_tbl_resumo_concentracao_empresa_estrategia,
    caminho_tbl_concentracao_empresa_por_ano,
    caminho_tbl_concentracao_empresa_por_regime,
    caminho_tbl_top_empresas_concentracao,
    caminho_tbl_base_grafico_concentracao_empresa_timeline,
    caminho_tbl_base_grafico_top_empresas,
    caminho_tbl_auditoria_validacao_concentracao_empresa,
]:
    print(f"- {caminho}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 12.4 encontrou erros bloqueantes. Verifique a tabela de auditoria impressa acima.")

print("\nETAPA 12.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 12.4 - CONCENTRAÇÃO POR EMPRESA

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição determinística dos caminhos de entrada e saída...
Entrada - perfil por ticker da 12.3             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12\12_3_base_perfil_compras_ticker.parquet
Entrada - perfil por aporte da 12.3             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12\12_3_base_perfil_compras_aporte.parquet
Saída   - concentração empresa por aporte       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12\12_4_base_concentracao_empresa_aporte.parquet
Saída   - resumo concentração por aporte        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados

## Etapa 12.5) Concentração por Setor, Subsetor e Segmento

In [63]:
%%time
# ============================================================
# Etapa 12.5) Concentração por Setor, Subsetor e Segmento
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 12.5 - CONCENTRAÇÃO POR SETOR, SUBSETOR E SEGMENTO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/10] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "gerar_caminho_arquivo" not in globals():
    raise NameError("A função 'gerar_caminho_arquivo' não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função 'salvar_dataframe' não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/10] Definição determinística dos caminhos de entrada e saída...")

caminho_base_perfil_compras_ticker = gerar_caminho_arquivo(etapa=12, subetapa=3, tipo_arquivo="base", nome="perfil_compras_ticker")
caminho_base_perfil_compras_aporte = gerar_caminho_arquivo(etapa=12, subetapa=3, tipo_arquivo="base", nome="perfil_compras_aporte")

caminho_base_concentracao_setorial_aporte = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="base", nome="concentracao_setorial_aporte")
caminho_base_concentracao_setorial_resumo_aporte = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="base", nome="concentracao_setorial_resumo_aporte")
caminho_tbl_resumo_concentracao_setorial_estrategia = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="tbl", nome="resumo_concentracao_setorial_estrategia")
caminho_tbl_concentracao_setorial_por_ano = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="tbl", nome="concentracao_setorial_por_ano")
caminho_tbl_concentracao_setorial_por_regime = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="tbl", nome="concentracao_setorial_por_regime")
caminho_tbl_top_setores_concentracao = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="tbl", nome="top_setores_concentracao")
caminho_tbl_top_subsetores_concentracao = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="tbl", nome="top_subsetores_concentracao")
caminho_tbl_top_segmentos_concentracao = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="tbl", nome="top_segmentos_concentracao")
caminho_tbl_base_grafico_concentracao_setorial_timeline = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="tbl", nome="base_grafico_concentracao_setorial_timeline")
caminho_tbl_base_grafico_top_classificacoes = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="tbl", nome="base_grafico_top_classificacoes_setoriais")
caminho_tbl_auditoria_validacao_concentracao_setorial = gerar_caminho_arquivo(etapa=12, subetapa=5, tipo_arquivo="tbl", nome="auditoria_validacao_concentracao_setorial")

print(f"Entrada - perfil por ticker da 12.3             : {caminho_base_perfil_compras_ticker}")
print(f"Entrada - perfil por aporte da 12.3             : {caminho_base_perfil_compras_aporte}")
print(f"Saída   - concentração setorial por aporte      : {caminho_base_concentracao_setorial_aporte}")
print(f"Saída   - resumo concentração por aporte        : {caminho_base_concentracao_setorial_resumo_aporte}")
print(f"Saída   - resumo por estratégia                 : {caminho_tbl_resumo_concentracao_setorial_estrategia}")
print(f"Saída   - concentração por ano                  : {caminho_tbl_concentracao_setorial_por_ano}")
print(f"Saída   - concentração por regime               : {caminho_tbl_concentracao_setorial_por_regime}")
print(f"Saída   - top setores                           : {caminho_tbl_top_setores_concentracao}")
print(f"Saída   - top subsetores                        : {caminho_tbl_top_subsetores_concentracao}")
print(f"Saída   - top segmentos                         : {caminho_tbl_top_segmentos_concentracao}")
print(f"Saída   - base gráfico timeline                 : {caminho_tbl_base_grafico_concentracao_setorial_timeline}")
print(f"Saída   - base gráfico top classificações       : {caminho_tbl_base_grafico_top_classificacoes}")
print(f"Saída   - auditoria de validação                : {caminho_tbl_auditoria_validacao_concentracao_setorial}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da 12.3
# ============================================================

print("\n[3/10] Carga das bases oficiais da 12.3...")

df_perfil_ticker = pd.read_parquet(caminho_base_perfil_compras_ticker)
df_perfil_aporte = pd.read_parquet(caminho_base_perfil_compras_aporte)

print(f"Base de perfil por ticker da 12.3       : {len(df_perfil_ticker):,} linhas x {df_perfil_ticker.shape[1]:,} colunas")
print(f"Base de perfil por aporte da 12.3       : {len(df_perfil_aporte):,} linhas x {df_perfil_aporte.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/10] Funções auxiliares da subetapa...")

def garantir_colunas_obrigatorias(df, colunas, nome_base):
    colunas_ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(colunas_ausentes) > 0:
        raise ValueError(f"A base {nome_base} não contém as colunas obrigatórias: {colunas_ausentes}")


def garantir_coluna(df, coluna, valor_padrao=np.nan):
    if coluna not in df.columns:
        df[coluna] = valor_padrao
    return df


def selecionar_colunas_existentes(df, colunas):
    return [coluna for coluna in colunas if coluna in df.columns]


def padronizar_texto_classificacao(serie):
    serie_txt = serie.astype("object").where(serie.notna(), "não_classificado")
    serie_txt = serie_txt.astype(str).str.strip()
    serie_txt = serie_txt.replace(
        {
            "": "não_classificado",
            "nan": "não_classificado",
            "None": "não_classificado",
            "<NA>": "não_classificado",
            "NaT": "não_classificado",
        }
    )
    return serie_txt


def soma_top_n(serie, n):
    valores = pd.to_numeric(serie, errors="coerce").dropna().sort_values(ascending=False)
    if len(valores) == 0:
        return np.nan
    return float(valores.head(n).sum())


def calcular_hhi(serie):
    valores = pd.to_numeric(serie, errors="coerce").dropna()
    if len(valores) == 0:
        return np.nan
    return float((valores ** 2).sum())


def calcular_entropia_normalizada(serie):
    pesos = pd.to_numeric(serie, errors="coerce").dropna()
    pesos = pesos[pesos > 0]
    if len(pesos) <= 1:
        return 0.0 if len(pesos) == 1 else np.nan
    entropia = float(-(pesos * np.log(pesos)).sum())
    return float(entropia / np.log(len(pesos)))


def adicionar_auditoria(lista, item, valor, valor_referencia, status, observacao):
    lista.append(
        {
            "item": item,
            "valor": valor,
            "valor_referencia": valor_referencia,
            "status": status,
            "observacao": observacao,
        }
    )


def padronizar_auditoria_para_parquet(df):
    df = df.copy()
    for coluna in ["item", "valor", "valor_referencia", "status", "observacao"]:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("object").where(df[coluna].notna(), np.nan)
            df[coluna] = df[coluna].map(lambda valor: str(valor) if pd.notna(valor) else np.nan)
    return df


def construir_base_nivel(df, nivel_classificacao, colunas_identificacao):
    base = df.copy()

    if nivel_classificacao == "setor":
        base["nivel_classificacao"] = "setor"
        base["setor_hierarquico"] = base["setor"]
        base["subsetor_hierarquico"] = "todos_os_subsetores"
        base["segmento_hierarquico"] = "todos_os_segmentos"
        base["chave_classificacao"] = base["setor"]
        base["nome_classificacao"] = base["setor"]
        base["rotulo_classificacao_hierarquica"] = base["setor"]

    elif nivel_classificacao == "subsetor":
        base["nivel_classificacao"] = "subsetor"
        base["setor_hierarquico"] = base["setor"]
        base["subsetor_hierarquico"] = base["subsetor"]
        base["segmento_hierarquico"] = "todos_os_segmentos"
        base["chave_classificacao"] = base["setor"] + " > " + base["subsetor"]
        base["nome_classificacao"] = base["subsetor"]
        base["rotulo_classificacao_hierarquica"] = base["setor"] + " > " + base["subsetor"]

    elif nivel_classificacao == "segmento":
        base["nivel_classificacao"] = "segmento"
        base["setor_hierarquico"] = base["setor"]
        base["subsetor_hierarquico"] = base["subsetor"]
        base["segmento_hierarquico"] = base["segmento"]
        base["chave_classificacao"] = base["setor"] + " > " + base["subsetor"] + " > " + base["segmento"]
        base["nome_classificacao"] = base["segmento"]
        base["rotulo_classificacao_hierarquica"] = base["setor"] + " > " + base["subsetor"] + " > " + base["segmento"]

    else:
        raise ValueError(f"Nível de classificação não reconhecido: {nivel_classificacao}")

    colunas_grupo = colunas_identificacao + [
        "nivel_classificacao",
        "chave_classificacao",
        "nome_classificacao",
        "rotulo_classificacao_hierarquica",
        "setor_hierarquico",
        "subsetor_hierarquico",
        "segmento_hierarquico",
    ]
    colunas_grupo = selecionar_colunas_existentes(base, colunas_grupo)

    base_nivel = (
        base.groupby(colunas_grupo, dropna=False)
        .agg(
            valor_investido_classificacao=("valor_investido_ticker", "sum"),
            n_tickers_classificacao=("ticker", "nunique"),
            n_empresas_classificacao=("issuer_code", "nunique"),
        )
        .reset_index()
    )

    base_nivel["valor_total_aporte_nivel"] = base_nivel.groupby(
        ["nivel_classificacao", "chave_aporte"],
        dropna=False,
    )["valor_investido_classificacao"].transform("sum")

    base_nivel["peso_classificacao"] = np.where(
        base_nivel["valor_total_aporte_nivel"] > 0,
        base_nivel["valor_investido_classificacao"] / base_nivel["valor_total_aporte_nivel"],
        np.nan,
    )

    return base_nivel


def montar_resumo_metricas_concentracao(df, colunas_grupo):
    resumo = (
        df.groupby(colunas_grupo, dropna=False)
        .agg(
            n_aportes=("chave_aporte", "nunique"),
            primeira_data_aporte=("data_aporte", "min"),
            ultima_data_aporte=("data_aporte", "max"),
            n_classificacoes_medio=("n_classificacoes", "mean"),
            n_classificacoes_mediano=("n_classificacoes", "median"),
            n_classificacoes_minimo=("n_classificacoes", "min"),
            n_classificacoes_maximo=("n_classificacoes", "max"),
            peso_top1_classificacao_medio=("peso_top1_classificacao", "mean"),
            peso_top1_classificacao_mediano=("peso_top1_classificacao", "median"),
            peso_top1_classificacao_maximo=("peso_top1_classificacao", "max"),
            peso_top3_classificacoes_medio=("peso_top3_classificacoes", "mean"),
            peso_top5_classificacoes_medio=("peso_top5_classificacoes", "mean"),
            hhi_classificacao_medio=("hhi_classificacao", "mean"),
            hhi_classificacao_mediano=("hhi_classificacao", "median"),
            entropia_classificacao_media=("entropia_classificacao_normalizada", "mean"),
            valor_investido_total=("valor_total_aporte_nivel", "sum"),
            score_intensidade_medio=("score_intensidade_sinal", "mean"),
            percentil_intensidade_medio=("percentil_intensidade_sinal", "mean"),
        )
        .reset_index()
    )
    return resumo

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Validação e padronização das colunas de entrada
# ============================================================

print("\n[5/10] Validação e padronização das colunas de entrada...")

colunas_obrigatorias_ticker = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "chave_aporte",
    "data_aporte",
    "issuer_code",
    "ticker",
    "setor",
    "subsetor",
    "segmento",
    "valor_investido_ticker",
]

garantir_colunas_obrigatorias(df_perfil_ticker, colunas_obrigatorias_ticker, "12_3_base_perfil_compras_ticker")
garantir_colunas_obrigatorias(df_perfil_aporte, ["chave_aporte", "data_aporte"], "12_3_base_perfil_compras_aporte")

base_ticker = df_perfil_ticker.copy()
base_aporte = df_perfil_aporte.copy()

colunas_opcionais = [
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "carteira_id",
    "replica_id",
    "ordem_aporte",
    "data_sinal",
    "ano_sinal",
    "ano_aporte",
    "regime_mercado",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
]

for coluna in colunas_opcionais:
    base_ticker = garantir_coluna(base_ticker, coluna, np.nan)
    base_aporte = garantir_coluna(base_aporte, coluna, np.nan)

for coluna in ["estrategia_referencia", "nome_exibicao_estrategia", "chave_aporte", "ticker", "issuer_code"]:
    base_ticker[coluna] = padronizar_texto_classificacao(base_ticker[coluna])

for coluna in ["setor", "subsetor", "segmento"]:
    base_ticker[coluna] = padronizar_texto_classificacao(base_ticker[coluna])

base_ticker["data_aporte"] = pd.to_datetime(base_ticker["data_aporte"], errors="coerce")
base_ticker["data_sinal"] = pd.to_datetime(base_ticker["data_sinal"], errors="coerce")
base_ticker["valor_investido_ticker"] = pd.to_numeric(base_ticker["valor_investido_ticker"], errors="coerce")
base_ticker["ano_aporte"] = pd.to_numeric(base_ticker["ano_aporte"], errors="coerce")
base_ticker["ano_aporte"] = base_ticker["ano_aporte"].where(base_ticker["ano_aporte"].notna(), base_ticker["data_aporte"].dt.year)
base_ticker["ano_sinal"] = pd.to_numeric(base_ticker["ano_sinal"], errors="coerce")
base_ticker["ano_sinal"] = base_ticker["ano_sinal"].where(base_ticker["ano_sinal"].notna(), base_ticker["data_sinal"].dt.year)
base_ticker["regime_mercado"] = padronizar_texto_classificacao(base_ticker["regime_mercado"])
base_ticker["score_intensidade_sinal"] = pd.to_numeric(base_ticker["score_intensidade_sinal"], errors="coerce")
base_ticker["percentil_intensidade_sinal"] = pd.to_numeric(base_ticker["percentil_intensidade_sinal"], errors="coerce")

base_ticker = base_ticker[
    base_ticker["valor_investido_ticker"].notna() &
    (base_ticker["valor_investido_ticker"] > 0) &
    base_ticker["data_aporte"].notna() &
    base_ticker["chave_aporte"].notna()
].copy()

base_ticker["chave_subsetor_hierarquica"] = base_ticker["setor"] + " > " + base_ticker["subsetor"]
base_ticker["chave_segmento_hierarquica"] = base_ticker["setor"] + " > " + base_ticker["subsetor"] + " > " + base_ticker["segmento"]

print(f"Linhas válidas para concentração        : {len(base_ticker):,}")
print(f"Aportes distintos                       : {base_ticker['chave_aporte'].nunique(dropna=True):,}")
print(f"Setores distintos                       : {base_ticker['setor'].nunique(dropna=True):,}")
print(f"Subsetores hierárquicos distintos       : {base_ticker['chave_subsetor_hierarquica'].nunique(dropna=True):,}")
print(f"Segmentos hierárquicos distintos        : {base_ticker['chave_segmento_hierarquica'].nunique(dropna=True):,}")
print(f"Estratégias de referência               : {base_ticker['estrategia_referencia'].nunique(dropna=True):,}")
print(f"Nomes de estratégia                     : {base_ticker['nome_exibicao_estrategia'].nunique(dropna=True):,}")
print("OK")

# ============================================================
# 6) Construção das bases hierárquicas por setor, subsetor e segmento
# ============================================================

print("\n[6/10] Construção das bases hierárquicas por setor, subsetor e segmento...")

colunas_identificacao_aporte = [
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "chave_estrategia",
    "estrategia_id",
    "grupo_controle",
    "familia_estrategia",
    "carteira_id",
    "replica_id",
    "chave_aporte",
    "ordem_aporte",
    "data_sinal",
    "data_aporte",
    "ano_sinal",
    "ano_aporte",
    "regime_mercado",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
    "flag_sinal_intenso",
    "flag_sinal_muito_intenso",
]
colunas_identificacao_aporte = selecionar_colunas_existentes(base_ticker, colunas_identificacao_aporte)

bases_nivel = []
for nivel in ["setor", "subsetor", "segmento"]:
    base_nivel = construir_base_nivel(base_ticker, nivel, colunas_identificacao_aporte)
    bases_nivel.append(base_nivel)
    print(
        f"Nível {nivel:<8}: "
        f"{len(base_nivel):,} linhas | "
        f"{base_nivel['chave_aporte'].nunique(dropna=True):,} aportes | "
        f"{base_nivel['chave_classificacao'].nunique(dropna=True):,} classificações"
    )

base_concentracao_setorial_aporte = pd.concat(bases_nivel, ignore_index=True)
base_concentracao_setorial_aporte = base_concentracao_setorial_aporte.sort_values(
    ["nivel_classificacao", "estrategia_referencia", "data_aporte", "chave_aporte", "peso_classificacao", "chave_classificacao"],
    ascending=[True, True, True, True, False, True],
).reset_index(drop=True)

print(f"Base setorial por aporte                : {len(base_concentracao_setorial_aporte):,} linhas")
print(f"Aportes distintos                       : {base_concentracao_setorial_aporte['chave_aporte'].nunique(dropna=True):,}")
print("OK")

# ============================================================
# 7) Construção dos indicadores de concentração por aporte e nível
# ============================================================

print("\n[7/10] Construção dos indicadores de concentração por aporte e nível...")

colunas_identificacao_resumo = [coluna for coluna in colunas_identificacao_aporte if coluna in base_concentracao_setorial_aporte.columns]
colunas_resumo_grupo = ["nivel_classificacao"] + colunas_identificacao_resumo

base_concentracao_setorial_resumo_aporte = (
    base_concentracao_setorial_aporte.groupby(colunas_resumo_grupo, dropna=False)
    .agg(
        n_classificacoes=("chave_classificacao", "nunique"),
        valor_total_aporte_nivel=("valor_investido_classificacao", "sum"),
        peso_top1_classificacao=("peso_classificacao", "max"),
        peso_top3_classificacoes=("peso_classificacao", lambda s: soma_top_n(s, 3)),
        peso_top5_classificacoes=("peso_classificacao", lambda s: soma_top_n(s, 5)),
        hhi_classificacao=("peso_classificacao", calcular_hhi),
        entropia_classificacao_normalizada=("peso_classificacao", calcular_entropia_normalizada),
    )
    .reset_index()
)

idx_maior_classificacao = base_concentracao_setorial_aporte.groupby(
    ["nivel_classificacao", "chave_aporte"],
    dropna=False,
)["peso_classificacao"].idxmax()

base_maior_classificacao = base_concentracao_setorial_aporte.loc[
    idx_maior_classificacao,
    [
        "nivel_classificacao",
        "chave_aporte",
        "chave_classificacao",
        "nome_classificacao",
        "rotulo_classificacao_hierarquica",
        "setor_hierarquico",
        "subsetor_hierarquico",
        "segmento_hierarquico",
        "valor_investido_classificacao",
        "peso_classificacao",
    ],
].rename(
    columns={
        "chave_classificacao": "chave_maior_peso_classificacao",
        "nome_classificacao": "nome_maior_peso_classificacao",
        "rotulo_classificacao_hierarquica": "rotulo_maior_peso_classificacao",
        "setor_hierarquico": "setor_maior_peso_classificacao",
        "subsetor_hierarquico": "subsetor_maior_peso_classificacao",
        "segmento_hierarquico": "segmento_maior_peso_classificacao",
        "valor_investido_classificacao": "valor_maior_peso_classificacao",
        "peso_classificacao": "peso_maior_classificacao",
    }
)

base_concentracao_setorial_resumo_aporte = base_concentracao_setorial_resumo_aporte.merge(
    base_maior_classificacao,
    on=["nivel_classificacao", "chave_aporte"],
    how="left",
)

base_concentracao_setorial_resumo_aporte = base_concentracao_setorial_resumo_aporte.sort_values(
    ["nivel_classificacao", "estrategia_referencia", "data_aporte", "chave_aporte"],
).reset_index(drop=True)

print(f"Resumo setorial por aporte              : {len(base_concentracao_setorial_resumo_aporte):,} linhas")
for nivel in ["setor", "subsetor", "segmento"]:
    base_tmp = base_concentracao_setorial_resumo_aporte[base_concentracao_setorial_resumo_aporte["nivel_classificacao"] == nivel]
    print(
        f"Nível {nivel:<8}: "
        f"n médio classificações={base_tmp['n_classificacoes'].mean():.2f} | "
        f"peso médio top1={base_tmp['peso_top1_classificacao'].mean():.6f} | "
        f"HHI médio={base_tmp['hhi_classificacao'].mean():.6f}"
    )
print("OK")

# ============================================================
# 8) Construção dos resumos, rankings e bases para gráficos
# ============================================================

print("\n[8/10] Construção dos resumos, rankings e bases para gráficos...")

resumo_concentracao_setorial_estrategia = montar_resumo_metricas_concentracao(
    base_concentracao_setorial_resumo_aporte,
    ["nivel_classificacao", "estrategia_referencia", "nome_exibicao_estrategia"],
)

concentracao_setorial_por_ano = montar_resumo_metricas_concentracao(
    base_concentracao_setorial_resumo_aporte,
    ["nivel_classificacao", "estrategia_referencia", "nome_exibicao_estrategia", "ano_aporte"],
)

concentracao_setorial_por_regime = montar_resumo_metricas_concentracao(
    base_concentracao_setorial_resumo_aporte,
    ["nivel_classificacao", "estrategia_referencia", "nome_exibicao_estrategia", "regime_mercado"],
)

aportes_por_estrategia = (
    base_concentracao_setorial_resumo_aporte[["estrategia_referencia", "nome_exibicao_estrategia", "chave_aporte"]]
    .drop_duplicates()
    .groupby(["estrategia_referencia", "nome_exibicao_estrategia"], dropna=False)["chave_aporte"]
    .nunique()
    .reset_index(name="n_aportes_estrategia")
)

top_classificacoes = (
    base_concentracao_setorial_aporte.groupby(
        [
            "nivel_classificacao",
            "estrategia_referencia",
            "nome_exibicao_estrategia",
            "chave_classificacao",
            "nome_classificacao",
            "rotulo_classificacao_hierarquica",
            "setor_hierarquico",
            "subsetor_hierarquico",
            "segmento_hierarquico",
        ],
        dropna=False,
    )
    .agg(
        n_aportes_com_exposicao=("chave_aporte", "nunique"),
        valor_investido_total=("valor_investido_classificacao", "sum"),
        peso_medio_quando_presente=("peso_classificacao", "mean"),
        peso_mediano_quando_presente=("peso_classificacao", "median"),
        peso_maximo=("peso_classificacao", "max"),
        primeira_data_exposicao=("data_aporte", "min"),
        ultima_data_exposicao=("data_aporte", "max"),
        n_tickers_distintos=("n_tickers_classificacao", "sum"),
        n_empresas_distintas=("n_empresas_classificacao", "sum"),
    )
    .reset_index()
)

top_classificacoes = top_classificacoes.merge(
    aportes_por_estrategia,
    on=["estrategia_referencia", "nome_exibicao_estrategia"],
    how="left",
)

top_classificacoes["pct_aportes_com_exposicao"] = np.where(
    top_classificacoes["n_aportes_estrategia"] > 0,
    top_classificacoes["n_aportes_com_exposicao"] / top_classificacoes["n_aportes_estrategia"],
    np.nan,
)

top_classificacoes["ranking_valor_estrategia_nivel"] = (
    top_classificacoes.groupby(["nivel_classificacao", "estrategia_referencia", "nome_exibicao_estrategia"], dropna=False)["valor_investido_total"]
    .rank(method="first", ascending=False)
)
top_classificacoes["ranking_frequencia_estrategia_nivel"] = (
    top_classificacoes.groupby(["nivel_classificacao", "estrategia_referencia", "nome_exibicao_estrategia"], dropna=False)["n_aportes_com_exposicao"]
    .rank(method="first", ascending=False)
)
top_classificacoes = top_classificacoes.sort_values(
    ["nivel_classificacao", "estrategia_referencia", "nome_exibicao_estrategia", "ranking_valor_estrategia_nivel"],
).reset_index(drop=True)

top_setores_concentracao = top_classificacoes[top_classificacoes["nivel_classificacao"] == "setor"].copy()
top_subsetores_concentracao = top_classificacoes[top_classificacoes["nivel_classificacao"] == "subsetor"].copy()
top_segmentos_concentracao = top_classificacoes[top_classificacoes["nivel_classificacao"] == "segmento"].copy()

base_grafico_concentracao_setorial_timeline = base_concentracao_setorial_resumo_aporte.copy()
colunas_timeline = [
    "nivel_classificacao",
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "data_sinal",
    "data_aporte",
    "ano_aporte",
    "regime_mercado",
    "n_classificacoes",
    "peso_top1_classificacao",
    "peso_top3_classificacoes",
    "peso_top5_classificacoes",
    "hhi_classificacao",
    "entropia_classificacao_normalizada",
    "chave_maior_peso_classificacao",
    "rotulo_maior_peso_classificacao",
    "peso_maior_classificacao",
    "score_intensidade_sinal",
    "percentil_intensidade_sinal",
    "classe_intensidade_sinal",
]
colunas_timeline = selecionar_colunas_existentes(base_grafico_concentracao_setorial_timeline, colunas_timeline)
base_grafico_concentracao_setorial_timeline = base_grafico_concentracao_setorial_timeline[colunas_timeline].copy()

base_grafico_top_classificacoes = top_classificacoes.copy()

print(f"Resumo por estratégia                  : {len(resumo_concentracao_setorial_estrategia):,} linhas")
print(f"Concentração por ano                   : {len(concentracao_setorial_por_ano):,} linhas")
print(f"Concentração por regime                : {len(concentracao_setorial_por_regime):,} linhas")
print(f"Top setores concentração               : {len(top_setores_concentracao):,} linhas")
print(f"Top subsetores concentração            : {len(top_subsetores_concentracao):,} linhas")
print(f"Top segmentos concentração             : {len(top_segmentos_concentracao):,} linhas")
print(f"Base gráfico timeline                  : {len(base_grafico_concentracao_setorial_timeline):,} linhas")
print(f"Base gráfico top classificações        : {len(base_grafico_top_classificacoes):,} linhas")
print("OK")

# ============================================================
# 9) Construção da auditoria de validação
# ============================================================

print("\n[9/10] Construção da auditoria de validação...")

auditoria = []

n_aportes_origem = base_aporte["chave_aporte"].nunique(dropna=True)
n_aportes_base = base_ticker["chave_aporte"].nunique(dropna=True)
n_estrategias = base_ticker["estrategia_referencia"].nunique(dropna=True)
n_setores = base_ticker["setor"].nunique(dropna=True)
n_subsetores_hierarquicos = base_ticker["chave_subsetor_hierarquica"].nunique(dropna=True)
n_segmentos_hierarquicos = base_ticker["chave_segmento_hierarquica"].nunique(dropna=True)
setores_nao_classificados = int((base_ticker["setor"] == "não_classificado").sum())
subsetores_nao_classificados = int((base_ticker["subsetor"] == "não_classificado").sum())
segmentos_nao_classificados = int((base_ticker["segmento"] == "não_classificado").sum())
valores_nao_positivos = int((base_concentracao_setorial_aporte["valor_investido_classificacao"].fillna(0) <= 0).sum())
duplicatas_classificacao_aporte = int(
    base_concentracao_setorial_aporte.duplicated(
        subset=["nivel_classificacao", "chave_aporte", "chave_classificacao"]
    ).sum()
)

base_soma_pesos = (
    base_concentracao_setorial_aporte.groupby(["nivel_classificacao", "chave_aporte"], dropna=False)["peso_classificacao"]
    .sum()
    .reset_index(name="soma_pesos")
)
base_soma_pesos["desvio_abs"] = (base_soma_pesos["soma_pesos"] - 1.0).abs()
maior_desvio_pesos = float(base_soma_pesos["desvio_abs"].max()) if len(base_soma_pesos) > 0 else np.nan
aportes_soma_pesos_inconsistente = int((base_soma_pesos["desvio_abs"] > 0.0001).sum())

aportes_por_nivel = (
    base_concentracao_setorial_resumo_aporte.groupby("nivel_classificacao", dropna=False)["chave_aporte"]
    .nunique()
    .to_dict()
)
aportes_setor = int(aportes_por_nivel.get("setor", 0))
aportes_subsetor = int(aportes_por_nivel.get("subsetor", 0))
aportes_segmento = int(aportes_por_nivel.get("segmento", 0))

subsetores_com_mesmo_nome_em_multiplos_setores = int(
    (
        base_ticker[["setor", "subsetor"]]
        .drop_duplicates()
        .groupby("subsetor", dropna=False)["setor"]
        .nunique()
        .gt(1)
        .sum()
    )
)

segmentos_com_mesmo_nome_em_multiplos_ramos = int(
    (
        base_ticker[["setor", "subsetor", "segmento"]]
        .drop_duplicates()
        .assign(ramo=lambda df: df["setor"] + " > " + df["subsetor"])
        .groupby("segmento", dropna=False)["ramo"]
        .nunique()
        .gt(1)
        .sum()
    )
)

chaves_subsetor_invalidas = int((~base_ticker["chave_subsetor_hierarquica"].str.contains(" > ", regex=False, na=False)).sum())
chaves_segmento_invalidas = int((base_ticker["chave_segmento_hierarquica"].str.count(" > ") < 2).sum())

colunas_numericas_resumo = base_concentracao_setorial_resumo_aporte.select_dtypes(include=[np.number]).columns.tolist()
metricas_infinitas = int(np.isinf(base_concentracao_setorial_resumo_aporte[colunas_numericas_resumo].to_numpy(dtype=float)).sum()) if len(colunas_numericas_resumo) > 0 else 0

adicionar_auditoria(auditoria, "aportes_origem_12_3", n_aportes_origem, "informativo", "OK", "Quantidade de aportes identificados na base 12.3 de perfil por aporte.")
adicionar_auditoria(auditoria, "aportes_validos_base_ticker", n_aportes_base, n_aportes_origem, "OK" if n_aportes_base == n_aportes_origem else "ERRO", "A base setorial deve preservar os aportes consolidados na 12.3.")
adicionar_auditoria(auditoria, "estrategias_referencia", n_estrategias, 2, "OK" if n_estrategias == 2 else "ERRO", "A base deve manter as estratégias de referência de capitulação e euforia.")
adicionar_auditoria(auditoria, "aportes_setor", aportes_setor, n_aportes_base, "OK" if aportes_setor == n_aportes_base else "ERRO", "O nível setor deve preservar todos os aportes válidos.")
adicionar_auditoria(auditoria, "aportes_subsetor", aportes_subsetor, n_aportes_base, "OK" if aportes_subsetor == n_aportes_base else "ERRO", "O nível subsetor deve preservar todos os aportes válidos.")
adicionar_auditoria(auditoria, "aportes_segmento", aportes_segmento, n_aportes_base, "OK" if aportes_segmento == n_aportes_base else "ERRO", "O nível segmento deve preservar todos os aportes válidos.")
adicionar_auditoria(auditoria, "setores_distintos", n_setores, "informativo", "OK", "Quantidade de setores distintos após padronização.")
adicionar_auditoria(auditoria, "subsetores_hierarquicos_distintos", n_subsetores_hierarquicos, "informativo", "OK", "Subsetores são identificados pela chave hierárquica setor > subsetor.")
adicionar_auditoria(auditoria, "segmentos_hierarquicos_distintos", n_segmentos_hierarquicos, "informativo", "OK", "Segmentos são identificados pela chave hierárquica setor > subsetor > segmento.")
adicionar_auditoria(auditoria, "subsetores_com_mesmo_nome_em_multiplos_setores", subsetores_com_mesmo_nome_em_multiplos_setores, "informativo", "OK", "Nomes repetidos de subsetor são tratados pela chave hierárquica completa.")
adicionar_auditoria(auditoria, "segmentos_com_mesmo_nome_em_multiplos_ramos", segmentos_com_mesmo_nome_em_multiplos_ramos, "informativo", "OK", "Nomes repetidos de segmento são tratados pela chave hierárquica completa.")
adicionar_auditoria(auditoria, "chaves_subsetor_invalidas", chaves_subsetor_invalidas, 0, "OK" if chaves_subsetor_invalidas == 0 else "ERRO", "Toda chave de subsetor deve seguir o padrão setor > subsetor.")
adicionar_auditoria(auditoria, "chaves_segmento_invalidas", chaves_segmento_invalidas, 0, "OK" if chaves_segmento_invalidas == 0 else "ERRO", "Toda chave de segmento deve seguir o padrão setor > subsetor > segmento.")
adicionar_auditoria(auditoria, "linhas_setor_nao_classificado", setores_nao_classificados, "informativo", "OK", "Quantidade de linhas com setor não classificado na base de compras.")
adicionar_auditoria(auditoria, "linhas_subsetor_nao_classificado", subsetores_nao_classificados, "informativo", "OK", "Quantidade de linhas com subsetor não classificado na base de compras.")
adicionar_auditoria(auditoria, "linhas_segmento_nao_classificado", segmentos_nao_classificados, "informativo", "OK", "Quantidade de linhas com segmento não classificado na base de compras.")
adicionar_auditoria(auditoria, "duplicatas_classificacao_aporte", duplicatas_classificacao_aporte, 0, "OK" if duplicatas_classificacao_aporte == 0 else "ERRO", "Cada combinação de nível, aporte e classificação hierárquica deve aparecer uma única vez após agregação.")
adicionar_auditoria(auditoria, "valores_classificacao_nao_positivos", valores_nao_positivos, 0, "OK" if valores_nao_positivos == 0 else "ERRO", "As exposições por classificação setorial devem ter valor investido positivo.")
adicionar_auditoria(auditoria, "aportes_soma_peso_classificacao_inconsistente", aportes_soma_pesos_inconsistente, 0, "OK" if aportes_soma_pesos_inconsistente == 0 else "ERRO", "A soma dos pesos por nível de classificação dentro de cada aporte deve ser próxima de 100%.")
adicionar_auditoria(auditoria, "maior_desvio_soma_pesos_classificacao", maior_desvio_pesos, "<= 0.0001", "OK" if pd.notna(maior_desvio_pesos) and maior_desvio_pesos <= 0.0001 else "ERRO", "Maior desvio absoluto da soma dos pesos setoriais por nível e aporte.")
adicionar_auditoria(auditoria, "metricas_infinitas", metricas_infinitas, 0, "OK" if metricas_infinitas == 0 else "ERRO", "As métricas numéricas consolidadas não devem conter valores infinitos.")

tbl_auditoria_validacao_concentracao_setorial = padronizar_auditoria_para_parquet(pd.DataFrame(auditoria))
erros_bloqueantes = int((tbl_auditoria_validacao_concentracao_setorial["status"] == "ERRO").sum())

print(f"Itens de auditoria                       : {len(tbl_auditoria_validacao_concentracao_setorial):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 10) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[10/10] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(base_concentracao_setorial_aporte, caminho_base_concentracao_setorial_aporte, index=False)
salvar_dataframe(base_concentracao_setorial_resumo_aporte, caminho_base_concentracao_setorial_resumo_aporte, index=False)
salvar_dataframe(resumo_concentracao_setorial_estrategia, caminho_tbl_resumo_concentracao_setorial_estrategia, index=False)
salvar_dataframe(concentracao_setorial_por_ano, caminho_tbl_concentracao_setorial_por_ano, index=False)
salvar_dataframe(concentracao_setorial_por_regime, caminho_tbl_concentracao_setorial_por_regime, index=False)
salvar_dataframe(top_setores_concentracao, caminho_tbl_top_setores_concentracao, index=False)
salvar_dataframe(top_subsetores_concentracao, caminho_tbl_top_subsetores_concentracao, index=False)
salvar_dataframe(top_segmentos_concentracao, caminho_tbl_top_segmentos_concentracao, index=False)
salvar_dataframe(base_grafico_concentracao_setorial_timeline, caminho_tbl_base_grafico_concentracao_setorial_timeline, index=False)
salvar_dataframe(base_grafico_top_classificacoes, caminho_tbl_base_grafico_top_classificacoes, index=False)
salvar_dataframe(tbl_auditoria_validacao_concentracao_setorial, caminho_tbl_auditoria_validacao_concentracao_setorial, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da concentração setorial:")
print(tbl_auditoria_validacao_concentracao_setorial.to_string(index=False))

print("\nResumo de concentração setorial por estratégia:")
print(resumo_concentracao_setorial_estrategia.to_string(index=False))

print("\nConcentração setorial por regime de mercado:")
print(concentracao_setorial_por_regime.to_string(index=False))

print("\nTop setores por concentração - amostra:")
print(top_setores_concentracao.head(40).to_string(index=False))

print("\nTop subsetores por concentração - amostra:")
colunas_amostra_top = [
    "nivel_classificacao",
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "rotulo_classificacao_hierarquica",
    "n_aportes_com_exposicao",
    "pct_aportes_com_exposicao",
    "valor_investido_total",
    "peso_medio_quando_presente",
    "peso_maximo",
    "ranking_valor_estrategia_nivel",
    "ranking_frequencia_estrategia_nivel",
]
colunas_amostra_top = selecionar_colunas_existentes(top_subsetores_concentracao, colunas_amostra_top)
print(top_subsetores_concentracao[colunas_amostra_top].head(40).to_string(index=False))

print("\nTop segmentos por concentração - amostra:")
colunas_amostra_top_segmento = selecionar_colunas_existentes(top_segmentos_concentracao, colunas_amostra_top)
print(top_segmentos_concentracao[colunas_amostra_top_segmento].head(40).to_string(index=False))

print("\nBase de concentração setorial por aporte - amostra:")
colunas_amostra_resumo = [
    "nivel_classificacao",
    "estrategia_referencia",
    "nome_exibicao_estrategia",
    "data_aporte",
    "regime_mercado",
    "n_classificacoes",
    "peso_top1_classificacao",
    "peso_top3_classificacoes",
    "peso_top5_classificacoes",
    "hhi_classificacao",
    "entropia_classificacao_normalizada",
    "rotulo_maior_peso_classificacao",
    "peso_maior_classificacao",
    "score_intensidade_sinal",
    "classe_intensidade_sinal",
]
colunas_amostra_resumo = selecionar_colunas_existentes(base_concentracao_setorial_resumo_aporte, colunas_amostra_resumo)
print(base_concentracao_setorial_resumo_aporte[colunas_amostra_resumo].head(40).to_string(index=False))

print("\nArquivos salvos na subetapa 12.5:")
for caminho in [
    caminho_base_concentracao_setorial_aporte,
    caminho_base_concentracao_setorial_resumo_aporte,
    caminho_tbl_resumo_concentracao_setorial_estrategia,
    caminho_tbl_concentracao_setorial_por_ano,
    caminho_tbl_concentracao_setorial_por_regime,
    caminho_tbl_top_setores_concentracao,
    caminho_tbl_top_subsetores_concentracao,
    caminho_tbl_top_segmentos_concentracao,
    caminho_tbl_base_grafico_concentracao_setorial_timeline,
    caminho_tbl_base_grafico_top_classificacoes,
    caminho_tbl_auditoria_validacao_concentracao_setorial,
]:
    print(f"- {caminho}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 12.5 encontrou erros bloqueantes. Verifique a tabela de auditoria impressa acima.")

print("\nETAPA 12.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 12.5 - CONCENTRAÇÃO POR SETOR, SUBSETOR E SEGMENTO

[1/10] Validação inicial do ambiente...
OK

[2/10] Definição determinística dos caminhos de entrada e saída...
Entrada - perfil por ticker da 12.3             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12\12_3_base_perfil_compras_ticker.parquet
Entrada - perfil por aporte da 12.3             : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12\12_3_base_perfil_compras_aporte.parquet
Saída   - concentração setorial por aporte      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12\12_5_base_concentracao_setorial_aporte.parquet
Saída   - resumo concentração por aporte        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_b

# Etapa 13) Robustez e Sensibilidades

## Etapa 13.1) Sensibilidade do Cooldown

In [64]:
%%time
# ============================================================
# Etapa 13.1) Sensibilidade do Cooldown
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 13.1 - SENSIBILIDADE DO COOLDOWN")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/11] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/11] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_6 = Path(DIRETORIOS_PROJETO["etapa_6"])
DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])
DIR_ETAPA_13 = Path(DIRETORIOS_PROJETO["etapa_13"])
DIR_ETAPA_13.mkdir(parents=True, exist_ok=True)

CAMINHO_DATAS_BRUTAS_CAPITULACAO = DIR_ETAPA_6 / "6_1_tbl_datas_capitulacao_bruta.parquet"
CAMINHO_DATAS_BRUTAS_EUFORIA = DIR_ETAPA_6 / "6_2_tbl_datas_euforia_bruta.parquet"
CAMINHO_CALENDARIO_FINAL_SINAIS = DIR_ETAPA_6 / "6_4_base_calendario_final_sinais.parquet"
CAMINHO_RETORNOS_DIARIOS = DIR_ETAPA_11 / "11_1_base_retornos_diarios.parquet"

CAMINHO_BASE_SINAIS_COOLDOWN = DIR_ETAPA_13 / "13_1_base_sinais_sensibilidade_cooldown.parquet"
CAMINHO_TBL_RESUMO_COOLDOWN = DIR_ETAPA_13 / "13_1_tbl_resumo_sensibilidade_cooldown.parquet"
CAMINHO_TBL_COMPARACAO_REFERENCIA = DIR_ETAPA_13 / "13_1_tbl_comparacao_cooldown_referencia.parquet"
CAMINHO_TBL_IMPACTO_TEMPORAL = DIR_ETAPA_13 / "13_1_tbl_impacto_temporal_cooldown.parquet"
CAMINHO_BASE_PERFORMANCE_PROXY = DIR_ETAPA_13 / "13_1_base_performance_proxy_pos_sinal.parquet"
CAMINHO_TBL_PERFORMANCE_PROXY = DIR_ETAPA_13 / "13_1_tbl_performance_proxy_pos_sinal_cooldown.parquet"
CAMINHO_TBL_GRAFICO_SINAIS = DIR_ETAPA_13 / "13_1_tbl_base_grafico_sinais_cooldown.parquet"
CAMINHO_TBL_AUDITORIA = DIR_ETAPA_13 / "13_1_tbl_auditoria_validacao_sensibilidade_cooldown.parquet"

print(f"Entrada - datas brutas de capitulação      : {CAMINHO_DATAS_BRUTAS_CAPITULACAO}")
print(f"Entrada - datas brutas de euforia          : {CAMINHO_DATAS_BRUTAS_EUFORIA}")
print(f"Entrada - calendário final de sinais       : {CAMINHO_CALENDARIO_FINAL_SINAIS}")
print(f"Entrada - retornos diários da 11.1         : {CAMINHO_RETORNOS_DIARIOS}")
print(f"Saída   - base sinais por cooldown         : {CAMINHO_BASE_SINAIS_COOLDOWN}")
print(f"Saída   - resumo por cooldown              : {CAMINHO_TBL_RESUMO_COOLDOWN}")
print(f"Saída   - comparação com referência        : {CAMINHO_TBL_COMPARACAO_REFERENCIA}")
print(f"Saída   - impacto temporal                 : {CAMINHO_TBL_IMPACTO_TEMPORAL}")
print(f"Saída   - base performance proxy           : {CAMINHO_BASE_PERFORMANCE_PROXY}")
print(f"Saída   - tabela performance proxy         : {CAMINHO_TBL_PERFORMANCE_PROXY}")
print(f"Saída   - base gráfico sinais              : {CAMINHO_TBL_GRAFICO_SINAIS}")
print(f"Saída   - auditoria de validação           : {CAMINHO_TBL_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais de sinais e retornos
# ============================================================

print("\n[3/11] Carga das bases oficiais de sinais e retornos...")

arquivos_obrigatorios = [
    CAMINHO_DATAS_BRUTAS_CAPITULACAO,
    CAMINHO_DATAS_BRUTAS_EUFORIA,
    CAMINHO_CALENDARIO_FINAL_SINAIS,
]

arquivos_ausentes = [str(caminho) for caminho in arquivos_obrigatorios if not caminho.exists()]
if len(arquivos_ausentes) > 0:
    raise FileNotFoundError("Arquivos obrigatórios ausentes: " + " | ".join(arquivos_ausentes))

df_datas_capitulacao_bruta = pd.read_parquet(CAMINHO_DATAS_BRUTAS_CAPITULACAO)
df_datas_euforia_bruta = pd.read_parquet(CAMINHO_DATAS_BRUTAS_EUFORIA)
df_calendario_final_sinais = pd.read_parquet(CAMINHO_CALENDARIO_FINAL_SINAIS)

if CAMINHO_RETORNOS_DIARIOS.exists():
    df_retornos_diarios = pd.read_parquet(CAMINHO_RETORNOS_DIARIOS)
    retornos_disponiveis = True
else:
    df_retornos_diarios = pd.DataFrame()
    retornos_disponiveis = False

print(f"Datas brutas de capitulação        : {len(df_datas_capitulacao_bruta):,} linhas x {df_datas_capitulacao_bruta.shape[1]} colunas")
print(f"Datas brutas de euforia            : {len(df_datas_euforia_bruta):,} linhas x {df_datas_euforia_bruta.shape[1]} colunas")
print(f"Calendário final de sinais oficial : {len(df_calendario_final_sinais):,} linhas x {df_calendario_final_sinais.shape[1]} colunas")
print(f"Retornos diários disponíveis       : {retornos_disponiveis}")
if retornos_disponiveis:
    print(f"Base de retornos diários           : {len(df_retornos_diarios):,} linhas x {df_retornos_diarios.shape[1]} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/11] Funções auxiliares da subetapa...")

def normalizar_colunas(df):
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    return df


def detectar_coluna(df, candidatos, obrigatoria=True, nome_logico="coluna"):
    colunas = list(df.columns)
    mapa = {str(col).strip().lower(): col for col in colunas}
    for candidato in candidatos:
        chave = str(candidato).strip().lower()
        if chave in mapa:
            return mapa[chave]
    if obrigatoria:
        raise KeyError(f"Não foi possível detectar {nome_logico}. Candidatas: {candidatos}. Colunas disponíveis: {colunas}")
    return None


def detectar_coluna_data(df, nome_base):
    candidatos = ["data_sinal", "data_evento", "data_pregao", "data_aporte", "data", "date"]
    for candidato in candidatos:
        col = detectar_coluna(df, [candidato], obrigatoria=False)
        if col is not None:
            serie = pd.to_datetime(df[col], errors="coerce")
            if serie.notna().sum() > 0:
                return col
    for col in df.columns:
        serie = pd.to_datetime(df[col], errors="coerce")
        if serie.notna().sum() > 0:
            return col
    raise KeyError(f"Nenhuma coluna de data válida foi encontrada na base {nome_base}. Colunas disponíveis: {list(df.columns)}")


def preparar_datas_brutas(df, estrategia_referencia, nome_exibicao_estrategia, nome_base):
    df = normalizar_colunas(df)
    coluna_data = detectar_coluna_data(df, nome_base)
    datas = pd.DataFrame({"data_sinal": pd.to_datetime(df[coluna_data], errors="coerce")})
    datas = datas.dropna(subset=["data_sinal"]).drop_duplicates().sort_values("data_sinal").reset_index(drop=True)
    datas["estrategia_referencia"] = estrategia_referencia
    datas["nome_exibicao_estrategia"] = nome_exibicao_estrategia
    datas["origem_datas_brutas"] = nome_base
    datas["ordem_sinal_bruto"] = datas.groupby("estrategia_referencia").cumcount() + 1
    return datas[["estrategia_referencia", "nome_exibicao_estrategia", "data_sinal", "ordem_sinal_bruto", "origem_datas_brutas"]]


def preparar_calendario_oficial(df):
    df = normalizar_colunas(df)
    coluna_data = detectar_coluna_data(df, "calendario_final_sinais")
    coluna_estrategia = detectar_coluna(df, ["estrategia_referencia", "tipo_sinal", "estrategia", "nome_estrategia"], obrigatoria=False)
    saida = df.copy()
    saida["data_sinal"] = pd.to_datetime(saida[coluna_data], errors="coerce")
    if coluna_estrategia is None:
        saida["estrategia_referencia"] = "nao_identificada"
    else:
        saida["estrategia_referencia"] = saida[coluna_estrategia].astype(str).str.strip().str.lower()
    saida["estrategia_referencia"] = saida["estrategia_referencia"].replace({"capitulação": "capitulacao", "capitulacao": "capitulacao", "euforia": "euforia"})
    saida = saida[saida["estrategia_referencia"].isin(["capitulacao", "euforia"])].copy()
    saida = saida.dropna(subset=["data_sinal"]).drop_duplicates(subset=["estrategia_referencia", "data_sinal"])
    return saida[["estrategia_referencia", "data_sinal"]].sort_values(["estrategia_referencia", "data_sinal"]).reset_index(drop=True)


def obter_calendario_pregoes(df_retornos, datas_brutas):
    datas = []
    if not df_retornos.empty:
        df_temp = normalizar_colunas(df_retornos)
        try:
            coluna_data = detectar_coluna_data(df_temp, "retornos_diarios")
            datas.extend(list(pd.to_datetime(df_temp[coluna_data], errors="coerce").dropna().drop_duplicates()))
        except Exception:
            pass
    datas.extend(list(pd.to_datetime(datas_brutas["data_sinal"], errors="coerce").dropna().drop_duplicates()))
    if len(datas) == 0:
        raise ValueError("Não foi possível construir calendário de pregões para aplicação do cooldown.")
    return pd.Series(pd.to_datetime(datas)).dropna().drop_duplicates().sort_values().reset_index(drop=True)


def alinhar_data_ao_calendario(data, calendario):
    data = pd.Timestamp(data)
    pos = calendario.searchsorted(data)
    if pos >= len(calendario):
        return pd.NaT
    return calendario.iloc[pos]


def aplicar_cooldown_pregoes(df_datas, cooldown_pregoes, calendario):
    df_datas = df_datas.copy().sort_values("data_sinal").reset_index(drop=True)
    mapa_ordem = {pd.Timestamp(data): ordem for ordem, data in enumerate(calendario)}
    registros = []
    ultima_ordem_aceita = None
    ultima_data_aceita = pd.NaT
    ordem_aceita = 0
    for _, linha in df_datas.iterrows():
        data_original = pd.Timestamp(linha["data_sinal"])
        data_sinal = alinhar_data_ao_calendario(data_original, calendario)
        if pd.isna(data_sinal):
            continue
        ordem_atual = mapa_ordem.get(pd.Timestamp(data_sinal), None)
        if ordem_atual is None:
            continue
        if ultima_ordem_aceita is None:
            aceitar = True
            distancia_pregoes = np.nan
        else:
            distancia_pregoes = ordem_atual - ultima_ordem_aceita
            aceitar = distancia_pregoes >= cooldown_pregoes
        if aceitar:
            ordem_aceita += 1
            registros.append({
                "estrategia_referencia": linha["estrategia_referencia"],
                "nome_exibicao_estrategia": linha["nome_exibicao_estrategia"],
                "cooldown_pregoes": int(cooldown_pregoes),
                "data_sinal": data_sinal,
                "data_sinal_original": data_original,
                "ordem_sinal_bruto": int(linha["ordem_sinal_bruto"]),
                "ordem_sinal_pos_cooldown": ordem_aceita,
                "pregoes_desde_sinal_anterior_aceito": distancia_pregoes,
                "data_sinal_anterior_aceito": ultima_data_aceita,
            })
            ultima_ordem_aceita = ordem_atual
            ultima_data_aceita = data_sinal
    return pd.DataFrame(registros)


def construir_resumo_cooldown(df_base, df_brutos, capital_total_referencia):
    registros = []
    chaves = ["estrategia_referencia", "nome_exibicao_estrategia", "cooldown_pregoes"]
    for (estrategia, nome, cooldown), grupo in df_base.groupby(chaves, dropna=False):
        n_brutos = int(df_brutos.loc[df_brutos["estrategia_referencia"].eq(estrategia), "data_sinal"].nunique())
        n_aceitos = int(grupo["data_sinal"].nunique())
        intervalos = pd.to_numeric(grupo["pregoes_desde_sinal_anterior_aceito"], errors="coerce").dropna()
        valor_aporte = capital_total_referencia / n_aceitos if n_aceitos > 0 else np.nan
        registros.append({
            "estrategia_referencia": estrategia,
            "nome_exibicao_estrategia": nome,
            "cooldown_pregoes": int(cooldown),
            "n_sinais_brutos": n_brutos,
            "n_sinais_pos_cooldown": n_aceitos,
            "n_sinais_rejeitados": n_brutos - n_aceitos,
            "pct_sinais_aceitos": n_aceitos / n_brutos if n_brutos > 0 else np.nan,
            "primeira_data_sinal": grupo["data_sinal"].min(),
            "ultima_data_sinal": grupo["data_sinal"].max(),
            "intervalo_medio_pregoes": intervalos.mean() if len(intervalos) > 0 else np.nan,
            "intervalo_mediano_pregoes": intervalos.median() if len(intervalos) > 0 else np.nan,
            "intervalo_minimo_pregoes": intervalos.min() if len(intervalos) > 0 else np.nan,
            "intervalo_maximo_pregoes": intervalos.max() if len(intervalos) > 0 else np.nan,
            "valor_aporte_teorico_capital_constante": valor_aporte,
        })
    return pd.DataFrame(registros).sort_values(["estrategia_referencia", "cooldown_pregoes"]).reset_index(drop=True)


def preparar_serie_benchmark(df_retornos):
    if df_retornos.empty:
        return pd.DataFrame(), "sem_base_retornos", False
    df = normalizar_colunas(df_retornos)
    coluna_data = detectar_coluna_data(df, "retornos_diarios")
    coluna_chave = detectar_coluna(df, ["chave_estrategia", "estrategia_id", "nome_exibicao_estrategia", "estrategia_referencia"], obrigatoria=False)
    coluna_retorno = detectar_coluna(df, ["retorno_diario", "retorno", "retorno_periodo", "retorno_carteira", "retorno_simples"], obrigatoria=False)
    coluna_patrimonio = detectar_coluna(df, ["patrimonio", "patrimonio_total", "valor_carteira", "valor_patrimonial", "valor_patrimonio"], obrigatoria=False)
    df["data"] = pd.to_datetime(df[coluna_data], errors="coerce")
    origem = "base_sem_coluna_estrategia"
    if coluna_chave is not None:
        df["chave_tmp"] = df[coluna_chave].astype(str).str.strip()
        mascara_exata = df["chave_tmp"].eq("ibovespa_buy_and_hold__carteira_na__replica_na")
        if mascara_exata.sum() > 0:
            df = df.loc[mascara_exata].copy()
            origem = "ibovespa_buy_and_hold__carteira_na__replica_na"
        else:
            mascara_flex = df["chave_tmp"].str.lower().str.contains("ibov|ibovespa", regex=True, na=False)
            if mascara_flex.sum() > 0:
                df = df.loc[mascara_flex].copy()
                origem = "ibovespa_identificado_por_busca_flexivel"
            else:
                return pd.DataFrame(), "sem_ibovespa_identificado", False
    df = df.dropna(subset=["data"]).sort_values("data").drop_duplicates(subset=["data"], keep="last")
    if coluna_retorno is not None:
        df["retorno_diario_benchmark"] = pd.to_numeric(df[coluna_retorno], errors="coerce")
    elif coluna_patrimonio is not None:
        df["patrimonio_tmp"] = pd.to_numeric(df[coluna_patrimonio], errors="coerce")
        df["retorno_diario_benchmark"] = df["patrimonio_tmp"].pct_change()
    else:
        return pd.DataFrame(), "sem_retorno_ou_patrimonio", False
    df = df[["data", "retorno_diario_benchmark"]].dropna(subset=["retorno_diario_benchmark"]).reset_index(drop=True)
    return df, origem, len(df) > 0


def calcular_retorno_forward(serie_retorno, data_sinal, horizonte_pregoes):
    datas = serie_retorno["data"].to_numpy(dtype="datetime64[ns]")
    pos = np.searchsorted(datas, np.datetime64(pd.Timestamp(data_sinal)))
    inicio = pos + 1
    fim = inicio + int(horizonte_pregoes)
    if inicio >= len(serie_retorno) or fim > len(serie_retorno):
        return np.nan
    retornos = pd.to_numeric(serie_retorno.iloc[inicio:fim]["retorno_diario_benchmark"], errors="coerce").dropna()
    if len(retornos) < int(horizonte_pregoes):
        return np.nan
    return float((1.0 + retornos).prod() - 1.0)

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Preparação das datas brutas e do calendário oficial
# ============================================================

print("\n[5/11] Preparação das datas brutas e do calendário oficial...")

df_capitulacao_bruta = preparar_datas_brutas(df_datas_capitulacao_bruta, "capitulacao", "Capitulação", "6_1_tbl_datas_capitulacao_bruta")
df_euforia_bruta = preparar_datas_brutas(df_datas_euforia_bruta, "euforia", "Euforia", "6_2_tbl_datas_euforia_bruta")
df_sinais_brutos = pd.concat([df_capitulacao_bruta, df_euforia_bruta], ignore_index=True)
df_calendario_oficial = preparar_calendario_oficial(df_calendario_final_sinais)

n_oficial_capitulacao = df_calendario_oficial.loc[df_calendario_oficial["estrategia_referencia"].eq("capitulacao"), "data_sinal"].nunique()
n_oficial_euforia = df_calendario_oficial.loc[df_calendario_oficial["estrategia_referencia"].eq("euforia"), "data_sinal"].nunique()

print(f"Sinais brutos de Capitulação          : {df_capitulacao_bruta['data_sinal'].nunique():,}")
print(f"Sinais brutos de Euforia              : {df_euforia_bruta['data_sinal'].nunique():,}")
print(f"Sinais finais oficiais                : {len(df_calendario_oficial):,}")
print(f"Sinais finais oficiais - Capitulação  : {n_oficial_capitulacao:,}")
print(f"Sinais finais oficiais - Euforia      : {n_oficial_euforia:,}")
print("OK")

# ============================================================
# 6) Definição dos cenários de cooldown e calendário de pregões
# ============================================================

print("\n[6/11] Definição dos cenários de cooldown e calendário de pregões...")

COOLDOWN_REFERENCIA = int(PARAMETROS_GLOBAIS.get("cooldown_sinais_pregoes", 63)) if "PARAMETROS_GLOBAIS" in globals() else 63
COOLDOWNS_TESTADOS = sorted(set([0, 21, 42, COOLDOWN_REFERENCIA, 84, 126]))
CAPITAL_TOTAL_REFERENCIA = float(PARAMETROS_GLOBAIS.get("capital_inicial_estrategia", 100000)) if "PARAMETROS_GLOBAIS" in globals() else 100000.00
HORIZONTES_FORWARD = [21, 63, 126, 252]

calendario_pregoes = obter_calendario_pregoes(df_retornos_diarios, df_sinais_brutos)

print(f"Cooldown de referência                : {COOLDOWN_REFERENCIA} pregões")
print(f"Cooldowns testados                    : {COOLDOWNS_TESTADOS}")
print(f"Capital total de referência           : {CAPITAL_TOTAL_REFERENCIA:.2f}")
print(f"Horizontes forward para proxy         : {HORIZONTES_FORWARD}")
print(f"Pregões no calendário operacional     : {len(calendario_pregoes):,}")
print("OK")

# ============================================================
# 7) Aplicação dos cooldowns alternativos
# ============================================================

print("\n[7/11] Aplicação dos cooldowns alternativos...")

bases_cooldown = []
for cooldown in COOLDOWNS_TESTADOS:
    for estrategia in ["capitulacao", "euforia"]:
        df_datas_estrategia = df_sinais_brutos.loc[df_sinais_brutos["estrategia_referencia"].eq(estrategia)].copy()
        bases_cooldown.append(aplicar_cooldown_pregoes(df_datas_estrategia, cooldown, calendario_pregoes))

df_base_sinais_cooldown = pd.concat(bases_cooldown, ignore_index=True) if len(bases_cooldown) > 0 else pd.DataFrame()

if not df_base_sinais_cooldown.empty:
    df_base_sinais_cooldown["ano_sinal"] = df_base_sinais_cooldown["data_sinal"].dt.year
    df_base_sinais_cooldown["mes_sinal"] = df_base_sinais_cooldown["data_sinal"].dt.to_period("M").astype(str)
    df_base_sinais_cooldown["cenario_cooldown"] = np.where(df_base_sinais_cooldown["cooldown_pregoes"].eq(COOLDOWN_REFERENCIA), "referencia", "alternativo")

print(f"Base de sinais por cooldown           : {len(df_base_sinais_cooldown):,} linhas")
print(f"Estratégias analisadas                : {df_base_sinais_cooldown['estrategia_referencia'].nunique() if not df_base_sinais_cooldown.empty else 0}")
print(f"Cenários de cooldown gerados          : {df_base_sinais_cooldown['cooldown_pregoes'].nunique() if not df_base_sinais_cooldown.empty else 0}")
print("OK")

# ============================================================
# 8) Construção dos resumos de sensibilidade e comparação com a referência
# ============================================================

print("\n[8/11] Construção dos resumos de sensibilidade e comparação com a referência...")

df_resumo_cooldown = construir_resumo_cooldown(df_base_sinais_cooldown, df_sinais_brutos, CAPITAL_TOTAL_REFERENCIA)

df_ref = df_resumo_cooldown.loc[
    df_resumo_cooldown["cooldown_pregoes"].eq(COOLDOWN_REFERENCIA),
    ["estrategia_referencia", "n_sinais_pos_cooldown", "valor_aporte_teorico_capital_constante", "primeira_data_sinal", "ultima_data_sinal"],
].copy()
df_ref = df_ref.rename(columns={
    "n_sinais_pos_cooldown": "n_sinais_referencia",
    "valor_aporte_teorico_capital_constante": "valor_aporte_referencia",
    "primeira_data_sinal": "primeira_data_referencia",
    "ultima_data_sinal": "ultima_data_referencia",
})

df_comparacao_referencia = df_resumo_cooldown.merge(df_ref, on="estrategia_referencia", how="left")
df_comparacao_referencia["delta_n_sinais_vs_referencia"] = df_comparacao_referencia["n_sinais_pos_cooldown"] - df_comparacao_referencia["n_sinais_referencia"]
df_comparacao_referencia["pct_delta_n_sinais_vs_referencia"] = np.where(
    df_comparacao_referencia["n_sinais_referencia"].gt(0),
    df_comparacao_referencia["delta_n_sinais_vs_referencia"] / df_comparacao_referencia["n_sinais_referencia"],
    np.nan,
)
df_comparacao_referencia["delta_valor_aporte_vs_referencia"] = df_comparacao_referencia["valor_aporte_teorico_capital_constante"] - df_comparacao_referencia["valor_aporte_referencia"]
df_comparacao_referencia["pct_delta_valor_aporte_vs_referencia"] = np.where(
    df_comparacao_referencia["valor_aporte_referencia"].gt(0),
    df_comparacao_referencia["delta_valor_aporte_vs_referencia"] / df_comparacao_referencia["valor_aporte_referencia"],
    np.nan,
)

df_impacto_temporal = (
    df_base_sinais_cooldown
    .groupby(["estrategia_referencia", "nome_exibicao_estrategia", "cooldown_pregoes", "ano_sinal"], dropna=False)
    .agg(n_sinais=("data_sinal", "nunique"))
    .reset_index()
    .sort_values(["estrategia_referencia", "cooldown_pregoes", "ano_sinal"])
    .reset_index(drop=True)
) if not df_base_sinais_cooldown.empty else pd.DataFrame()

print(f"Resumo por cooldown                   : {len(df_resumo_cooldown):,} linhas")
print(f"Comparação com referência             : {len(df_comparacao_referencia):,} linhas")
print(f"Impacto temporal anual                : {len(df_impacto_temporal):,} linhas")
print("OK")

# ============================================================
# 9) Cálculo de performance proxy pós-sinal com Ibovespa
# ============================================================

print("\n[9/11] Cálculo de performance proxy pós-sinal com Ibovespa...")

serie_benchmark, origem_benchmark, benchmark_disponivel = preparar_serie_benchmark(df_retornos_diarios)
registros_proxy = []

if benchmark_disponivel and not df_base_sinais_cooldown.empty:
    for _, linha in df_base_sinais_cooldown.iterrows():
        for horizonte in HORIZONTES_FORWARD:
            retorno_forward = calcular_retorno_forward(serie_benchmark, linha["data_sinal"], horizonte)
            registros_proxy.append({
                "estrategia_referencia": linha["estrategia_referencia"],
                "nome_exibicao_estrategia": linha["nome_exibicao_estrategia"],
                "cooldown_pregoes": int(linha["cooldown_pregoes"]),
                "data_sinal": linha["data_sinal"],
                "horizonte_pregoes": int(horizonte),
                "benchmark_proxy": origem_benchmark,
                "retorno_ibovespa_pos_sinal": retorno_forward,
            })

df_base_performance_proxy = pd.DataFrame(registros_proxy)

if not df_base_performance_proxy.empty:
    df_tbl_performance_proxy = (
        df_base_performance_proxy
        .dropna(subset=["retorno_ibovespa_pos_sinal"])
        .groupby(["estrategia_referencia", "nome_exibicao_estrategia", "cooldown_pregoes", "horizonte_pregoes"], dropna=False)
        .agg(
            n_sinais_com_retorno=("retorno_ibovespa_pos_sinal", "count"),
            retorno_medio_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "mean"),
            retorno_mediano_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "median"),
            retorno_minimo_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "min"),
            retorno_maximo_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "max"),
            pct_sinais_retorno_positivo=("retorno_ibovespa_pos_sinal", lambda x: float((pd.Series(x) > 0).mean()) if len(x) > 0 else np.nan),
        )
        .reset_index()
        .sort_values(["estrategia_referencia", "cooldown_pregoes", "horizonte_pregoes"])
        .reset_index(drop=True)
    )
else:
    df_tbl_performance_proxy = pd.DataFrame()

print(f"Benchmark proxy identificado           : {origem_benchmark}")
print(f"Benchmark proxy disponível             : {benchmark_disponivel}")
print(f"Base de performance proxy              : {len(df_base_performance_proxy):,} linhas")
print(f"Tabela de performance proxy            : {len(df_tbl_performance_proxy):,} linhas")
print("OK")

# ============================================================
# 10) Construção da base de gráficos e auditoria
# ============================================================

print("\n[10/11] Construção da base de gráficos e auditoria...")

if not df_impacto_temporal.empty:
    df_base_grafico_sinais = df_impacto_temporal.copy()
    df_base_grafico_sinais["rotulo_cooldown"] = "Cooldown " + df_base_grafico_sinais["cooldown_pregoes"].astype(str) + " pregões"
else:
    df_base_grafico_sinais = pd.DataFrame()

n_oficial = df_calendario_oficial.groupby("estrategia_referencia")["data_sinal"].nunique().to_dict()
n_recalc = (
    df_base_sinais_cooldown.loc[df_base_sinais_cooldown["cooldown_pregoes"].eq(COOLDOWN_REFERENCIA)]
    .groupby("estrategia_referencia")["data_sinal"].nunique().to_dict()
)

df_validacao_oficial = pd.DataFrame([
    {"estrategia_referencia": estrategia, "n_sinais_oficial": int(n_oficial.get(estrategia, 0)), "n_sinais_recalculado_cooldown_referencia": int(n_recalc.get(estrategia, 0))}
    for estrategia in ["capitulacao", "euforia"]
])
df_validacao_oficial["diferenca"] = df_validacao_oficial["n_sinais_recalculado_cooldown_referencia"] - df_validacao_oficial["n_sinais_oficial"]

sinais_ref_diferem_oficial = int((df_validacao_oficial["diferenca"].abs() > 0).sum())
duplicatas_base_cooldown = int(df_base_sinais_cooldown.duplicated(subset=["estrategia_referencia", "cooldown_pregoes", "data_sinal"]).sum()) if not df_base_sinais_cooldown.empty else 0
cooldowns_gerados = int(df_base_sinais_cooldown["cooldown_pregoes"].nunique()) if not df_base_sinais_cooldown.empty else 0
estrategias_geradas = int(df_base_sinais_cooldown["estrategia_referencia"].nunique()) if not df_base_sinais_cooldown.empty else 0
sinais_sem_data = int(df_base_sinais_cooldown["data_sinal"].isna().sum()) if not df_base_sinais_cooldown.empty else 0
metricas_infinitas = 0
for df_teste in [df_resumo_cooldown, df_comparacao_referencia, df_tbl_performance_proxy]:
    if not df_teste.empty:
        colunas_numericas = df_teste.select_dtypes(include=[np.number]).columns
        if len(colunas_numericas) > 0:
            metricas_infinitas += int(np.isinf(df_teste[colunas_numericas].to_numpy()).sum())

df_auditoria = pd.DataFrame([
    {"item": "estrategias_geradas", "valor": estrategias_geradas, "valor_referencia": 2, "status": "OK" if estrategias_geradas == 2 else "ERRO", "observacao": "A sensibilidade deve contemplar Capitulação e Euforia."},
    {"item": "cooldowns_gerados", "valor": cooldowns_gerados, "valor_referencia": len(COOLDOWNS_TESTADOS), "status": "OK" if cooldowns_gerados == len(COOLDOWNS_TESTADOS) else "ERRO", "observacao": "Todos os cenários de cooldown definidos para a subetapa devem ser gerados."},
    {"item": "sinais_brutos_capitulacao", "valor": int(df_capitulacao_bruta["data_sinal"].nunique()), "valor_referencia": "informativo", "status": "OK", "observacao": "Quantidade de candidatos brutos antes do cooldown para Capitulação."},
    {"item": "sinais_brutos_euforia", "valor": int(df_euforia_bruta["data_sinal"].nunique()), "valor_referencia": "informativo", "status": "OK", "observacao": "Quantidade de candidatos brutos antes do cooldown para Euforia."},
    {"item": "sinais_sem_data", "valor": sinais_sem_data, "valor_referencia": 0, "status": "OK" if sinais_sem_data == 0 else "ERRO", "observacao": "Todos os sinais aceitos por cenário de cooldown devem possuir data válida."},
    {"item": "duplicatas_estrategia_cooldown_data", "valor": duplicatas_base_cooldown, "valor_referencia": 0, "status": "OK" if duplicatas_base_cooldown == 0 else "ERRO", "observacao": "Cada combinação de estratégia, cooldown e data deve aparecer uma única vez."},
    {"item": "sinais_referencia_diferem_oficial", "valor": sinais_ref_diferem_oficial, "valor_referencia": 0, "status": "OK" if sinais_ref_diferem_oficial == 0 else "ALERTA", "observacao": "Compara a quantidade de sinais recalculada no cooldown de referência com o calendário oficial da 6.4."},
    {"item": "benchmark_proxy_disponivel", "valor": int(benchmark_disponivel), "valor_referencia": "informativo", "status": "OK", "observacao": "Indica se foi possível calcular retorno forward do Ibovespa como proxy de timing pós-sinal."},
    {"item": "origem_benchmark_proxy", "valor": origem_benchmark, "valor_referencia": "informativo", "status": "OK", "observacao": "Origem da série usada no cálculo de retorno forward pós-sinal."},
    {"item": "metricas_infinitas", "valor": metricas_infinitas, "valor_referencia": 0, "status": "OK" if metricas_infinitas == 0 else "ERRO", "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos."},
])

for col in ["item", "valor", "valor_referencia", "status", "observacao"]:
    df_auditoria[col] = df_auditoria[col].astype(str)

erros_bloqueantes = int(df_auditoria["status"].eq("ERRO").sum())

print(f"Base gráfico sinais por cooldown       : {len(df_base_grafico_sinais):,} linhas")
print(f"Itens de auditoria                     : {len(df_auditoria):,}")
print(f"Erros bloqueantes                      : {erros_bloqueantes}")
print("OK")

# ============================================================
# 11) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[11/11] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(df_base_sinais_cooldown, CAMINHO_BASE_SINAIS_COOLDOWN, index=False)
salvar_dataframe(df_resumo_cooldown, CAMINHO_TBL_RESUMO_COOLDOWN, index=False)
salvar_dataframe(df_comparacao_referencia, CAMINHO_TBL_COMPARACAO_REFERENCIA, index=False)
salvar_dataframe(df_impacto_temporal, CAMINHO_TBL_IMPACTO_TEMPORAL, index=False)
salvar_dataframe(df_base_performance_proxy, CAMINHO_BASE_PERFORMANCE_PROXY, index=False)
salvar_dataframe(df_tbl_performance_proxy, CAMINHO_TBL_PERFORMANCE_PROXY, index=False)
salvar_dataframe(df_base_grafico_sinais, CAMINHO_TBL_GRAFICO_SINAIS, index=False)
salvar_dataframe(df_auditoria, CAMINHO_TBL_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da sensibilidade do cooldown:")
print(df_auditoria.to_string(index=False))

print("\nValidação da referência oficial do cooldown:")
print(df_validacao_oficial.to_string(index=False))

print("\nResumo de sensibilidade por cooldown:")
print(df_resumo_cooldown.to_string(index=False))

if not df_tbl_performance_proxy.empty:
    print("\nPerformance proxy pós-sinal - amostra:")
    print(df_tbl_performance_proxy.head(30).to_string(index=False))
else:
    print("\nPerformance proxy pós-sinal não calculada por ausência de série diária compatível do Ibovespa.")

print("\nArquivos salvos na subetapa 13.1:")
print(f"- {CAMINHO_BASE_SINAIS_COOLDOWN}")
print(f"- {CAMINHO_TBL_RESUMO_COOLDOWN}")
print(f"- {CAMINHO_TBL_COMPARACAO_REFERENCIA}")
print(f"- {CAMINHO_TBL_IMPACTO_TEMPORAL}")
print(f"- {CAMINHO_BASE_PERFORMANCE_PROXY}")
print(f"- {CAMINHO_TBL_PERFORMANCE_PROXY}")
print(f"- {CAMINHO_TBL_GRAFICO_SINAIS}")
print(f"- {CAMINHO_TBL_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 13.1 identificou erros bloqueantes. Verifique a tabela de auditoria antes de prosseguir.")

print("\nETAPA 13.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 13.1 - SENSIBILIDADE DO COOLDOWN

[1/11] Validação inicial do ambiente...
OK

[2/11] Definição determinística dos caminhos de entrada e saída...
Entrada - datas brutas de capitulação      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_1_tbl_datas_capitulacao_bruta.parquet
Entrada - datas brutas de euforia          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_2_tbl_datas_euforia_bruta.parquet
Entrada - calendário final de sinais       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_4_base_calendario_final_sinais.parquet
Entrada - retornos diários da 11.1         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_1_base_retornos_di

## Etapa 13.2) Sensibilidade dos Critérios de Liquidez

In [65]:
%%time
# ============================================================
# Etapa 13.2) Sensibilidade dos Critérios de Liquidez
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 13.2 - SENSIBILIDADE DOS CRITÉRIOS DE LIQUIDEZ")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

for chave_diretorio in ["etapa_3", "etapa_6", "etapa_12", "etapa_13"]:
    if chave_diretorio not in DIRETORIOS_PROJETO:
        raise KeyError(f"O diretório obrigatório {chave_diretorio} não foi encontrado em DIRETORIOS_PROJETO.")

Path(DIRETORIOS_PROJETO["etapa_13"]).mkdir(parents=True, exist_ok=True)

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

CAMINHO_PARAMETROS_LIQUIDEZ_3_1 = Path(DIRETORIOS_PROJETO["etapa_3"]) / "3_1_tbl_parametros_liquidez.parquet"
CAMINHO_LIQUIDEZ_TICKER_DATA_3_2 = Path(DIRETORIOS_PROJETO["etapa_3"]) / "3_2_base_liquidez_ticker_data.parquet"
CAMINHO_ELEGIBILIDADE_OPERACIONAL_3_4 = Path(DIRETORIOS_PROJETO["etapa_3"]) / "3_4_base_elegibilidade_operacional_data.parquet"
CAMINHO_CALENDARIO_FINAL_SINAIS_6_4 = Path(DIRETORIOS_PROJETO["etapa_6"]) / "6_4_base_calendario_final_sinais.parquet"
CAMINHO_PERFIL_COMPRAS_TICKER_12_3 = Path(DIRETORIOS_PROJETO["etapa_12"]) / "12_3_base_perfil_compras_ticker.parquet"

CAMINHO_BASE_CENARIOS_TICKER_DATA = Path(DIRETORIOS_PROJETO["etapa_13"]) / "13_2_base_liquidez_cenarios_ticker_data.parquet"
CAMINHO_TBL_PARAMETROS_CENARIOS = Path(DIRETORIOS_PROJETO["etapa_13"]) / "13_2_tbl_parametros_cenarios_liquidez.parquet"
CAMINHO_TBL_RESUMO_UNIVERSO = Path(DIRETORIOS_PROJETO["etapa_13"]) / "13_2_tbl_resumo_universo_liquidez_cenarios.parquet"
CAMINHO_TBL_COBERTURA_DIARIA = Path(DIRETORIOS_PROJETO["etapa_13"]) / "13_2_tbl_cobertura_diaria_liquidez_cenarios.parquet"
CAMINHO_TBL_COMPARACAO_REFERENCIA = Path(DIRETORIOS_PROJETO["etapa_13"]) / "13_2_tbl_comparacao_referencia_liquidez.parquet"
CAMINHO_TBL_IMPACTO_SINAIS = Path(DIRETORIOS_PROJETO["etapa_13"]) / "13_2_tbl_impacto_sinais_liquidez.parquet"
CAMINHO_TBL_IMPACTO_CESTAS = Path(DIRETORIOS_PROJETO["etapa_13"]) / "13_2_tbl_impacto_cestas_compras_liquidez.parquet"
CAMINHO_TBL_BASE_GRAFICO = Path(DIRETORIOS_PROJETO["etapa_13"]) / "13_2_tbl_base_grafico_liquidez_cenarios.parquet"
CAMINHO_TBL_AUDITORIA = Path(DIRETORIOS_PROJETO["etapa_13"]) / "13_2_tbl_auditoria_validacao_sensibilidade_liquidez.parquet"

caminhos_entrada_obrigatorios = [
    CAMINHO_LIQUIDEZ_TICKER_DATA_3_2,
    CAMINHO_ELEGIBILIDADE_OPERACIONAL_3_4,
    CAMINHO_CALENDARIO_FINAL_SINAIS_6_4,
    CAMINHO_PERFIL_COMPRAS_TICKER_12_3,
]

for caminho in caminhos_entrada_obrigatorios:
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - parâmetros de liquidez da 3.1      : {CAMINHO_PARAMETROS_LIQUIDEZ_3_1}")
print(f"Entrada - liquidez ticker/data da 3.2        : {CAMINHO_LIQUIDEZ_TICKER_DATA_3_2}")
print(f"Entrada - elegibilidade operacional da 3.4   : {CAMINHO_ELEGIBILIDADE_OPERACIONAL_3_4}")
print(f"Entrada - calendário final de sinais da 6.4  : {CAMINHO_CALENDARIO_FINAL_SINAIS_6_4}")
print(f"Entrada - perfil compras ticker da 12.3      : {CAMINHO_PERFIL_COMPRAS_TICKER_12_3}")
print(f"Saída   - base cenários ticker/data          : {CAMINHO_BASE_CENARIOS_TICKER_DATA}")
print(f"Saída   - parâmetros dos cenários            : {CAMINHO_TBL_PARAMETROS_CENARIOS}")
print(f"Saída   - resumo universo                    : {CAMINHO_TBL_RESUMO_UNIVERSO}")
print(f"Saída   - cobertura diária                   : {CAMINHO_TBL_COBERTURA_DIARIA}")
print(f"Saída   - comparação com referência          : {CAMINHO_TBL_COMPARACAO_REFERENCIA}")
print(f"Saída   - impacto em sinais                  : {CAMINHO_TBL_IMPACTO_SINAIS}")
print(f"Saída   - impacto em cestas de compras       : {CAMINHO_TBL_IMPACTO_CESTAS}")
print(f"Saída   - base gráfico                       : {CAMINHO_TBL_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação             : {CAMINHO_TBL_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/12] Carga das bases oficiais da subetapa...")

df_parametros_liquidez_3_1 = pd.read_parquet(CAMINHO_PARAMETROS_LIQUIDEZ_3_1) if CAMINHO_PARAMETROS_LIQUIDEZ_3_1.exists() else pd.DataFrame()
df_liquidez_ticker_data = pd.read_parquet(CAMINHO_LIQUIDEZ_TICKER_DATA_3_2)
df_elegibilidade_operacional = pd.read_parquet(CAMINHO_ELEGIBILIDADE_OPERACIONAL_3_4)
df_calendario_sinais = pd.read_parquet(CAMINHO_CALENDARIO_FINAL_SINAIS_6_4)
df_perfil_compras_ticker = pd.read_parquet(CAMINHO_PERFIL_COMPRAS_TICKER_12_3)

print(f"Parâmetros de liquidez da 3.1      : {len(df_parametros_liquidez_3_1):,} linhas x {df_parametros_liquidez_3_1.shape[1]:,} colunas")
print(f"Base liquidez ticker/data da 3.2   : {len(df_liquidez_ticker_data):,} linhas x {df_liquidez_ticker_data.shape[1]:,} colunas")
print(f"Elegibilidade operacional da 3.4   : {len(df_elegibilidade_operacional):,} linhas x {df_elegibilidade_operacional.shape[1]:,} colunas")
print(f"Calendário final de sinais da 6.4  : {len(df_calendario_sinais):,} linhas x {df_calendario_sinais.shape[1]:,} colunas")
print(f"Perfil compras ticker da 12.3      : {len(df_perfil_compras_ticker):,} linhas x {df_perfil_compras_ticker.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

def selecionar_coluna_existente(df, candidatos, nome_logico, obrigatoria=True):
    """
    Seleciona a primeira coluna existente entre uma lista de candidatos.
    Retorna None quando a coluna não for obrigatória e nenhum candidato existir.
    """
    colunas_disponiveis = list(df.columns)
    for coluna in candidatos:
        if coluna in colunas_disponiveis:
            return coluna

    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar a coluna lógica '{nome_logico}'. "
            f"Candidatos avaliados: {candidatos}. "
            f"Colunas disponíveis: {colunas_disponiveis}"
        )

    return None

def selecionar_coluna_por_tokens(df, tokens_obrigatorios, tokens_exclusao=None, nome_logico="coluna", obrigatoria=False):
    """
    Seleciona uma coluna por presença de tokens no nome.
    Evita KeyError ao retornar None quando a coluna não for obrigatória.
    """
    if tokens_exclusao is None:
        tokens_exclusao = []

    for coluna in df.columns:
        nome_coluna = str(coluna).lower()
        contem_todos = all(token.lower() in nome_coluna for token in tokens_obrigatorios)
        contem_exclusao = any(token.lower() in nome_coluna for token in tokens_exclusao)
        if contem_todos and not contem_exclusao:
            return coluna

    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar a coluna lógica '{nome_logico}' "
            f"pelos tokens obrigatórios {tokens_obrigatorios}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    return None

def converter_data_segura(df, coluna_data, nome_base):
    """
    Converte uma coluna de data para datetime com validação controlada.
    """
    df = df.copy()
    if coluna_data not in df.columns:
        raise KeyError(f"A coluna de data '{coluna_data}' não existe na base {nome_base}.")
    df[coluna_data] = pd.to_datetime(df[coluna_data], errors="coerce")
    return df

def converter_numerico_seguro(df, colunas):
    """
    Converte colunas para numérico apenas quando existirem no DataFrame.
    """
    df = df.copy()
    for coluna in colunas:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    return df

def criar_chave_data_ticker(df, coluna_data, coluna_ticker):
    """
    Cria chave textual data/ticker para comparação de elegibilidade.
    """
    df = df.copy()
    if coluna_data not in df.columns or coluna_ticker not in df.columns:
        raise KeyError(f"Não foi possível criar chave data/ticker. Colunas disponíveis: {list(df.columns)}")
    df["chave_data_ticker"] = (
        pd.to_datetime(df[coluna_data], errors="coerce").dt.strftime("%Y-%m-%d")
        + "__"
        + df[coluna_ticker].astype(str).str.strip().str.upper()
    )
    return df

def criar_chave_data_issuer(df, coluna_data, coluna_issuer):
    """
    Cria chave textual data/issuer_code para comparação complementar.
    """
    df = df.copy()
    if coluna_data not in df.columns or coluna_issuer not in df.columns:
        return df
    df["chave_data_issuer"] = (
        pd.to_datetime(df[coluna_data], errors="coerce").dt.strftime("%Y-%m-%d")
        + "__"
        + df[coluna_issuer].astype(str).str.strip().str.upper()
    )
    return df

def inferir_limiar_referencia(serie, mascara_referencia, nome_metrica):
    """
    Infere limiar operacional a partir da cauda inferior dos ativos elegíveis de referência.
    Usa o percentil 1 para reduzir sensibilidade a eventuais ruídos sem depender de nomes internos do filtro original.
    """
    valores_ref = pd.to_numeric(serie[mascara_referencia], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

    if valores_ref.empty:
        return np.nan

    valores_ref = valores_ref[valores_ref > 0]

    if valores_ref.empty:
        return np.nan

    limiar = float(valores_ref.quantile(0.01))

    if not np.isfinite(limiar):
        return np.nan

    return limiar

def construir_tabela_auditoria(itens):
    """
    Constrói tabela de auditoria com colunas padronizadas em texto.
    """
    df = pd.DataFrame(itens)
    for coluna in ["item", "valor", "valor_referencia", "status", "observacao"]:
        if coluna not in df.columns:
            df[coluna] = ""
        df[coluna] = df[coluna].astype(str)
    return df[["item", "valor", "valor_referencia", "status", "observacao"]]

def status_ok_alerta_erro(condicao_ok, condicao_alerta=False):
    """
    Padroniza status de auditoria.
    """
    if condicao_ok:
        return "OK"
    if condicao_alerta:
        return "ALERTA"
    return "ERRO"

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Validação e padronização das colunas centrais
# ============================================================

print("\n[5/12] Validação e padronização das colunas centrais...")

col_data_liquidez = selecionar_coluna_existente(
    df_liquidez_ticker_data,
    ["data", "dt_pregao", "data_pregao", "data_referencia"],
    "data da base de liquidez",
    obrigatoria=True,
)

col_ticker_liquidez = selecionar_coluna_existente(
    df_liquidez_ticker_data,
    ["ticker", "codigo_negociacao", "ativo"],
    "ticker da base de liquidez",
    obrigatoria=True,
)

col_issuer_liquidez = selecionar_coluna_existente(
    df_liquidez_ticker_data,
    ["issuer_code", "codigo_emissor", "empresa", "codigo_empresa"],
    "issuer_code da base de liquidez",
    obrigatoria=False,
)

col_data_elegibilidade = selecionar_coluna_existente(
    df_elegibilidade_operacional,
    ["data", "dt_pregao", "data_pregao", "data_referencia"],
    "data da elegibilidade operacional",
    obrigatoria=True,
)

col_ticker_elegibilidade = selecionar_coluna_existente(
    df_elegibilidade_operacional,
    ["ticker", "codigo_negociacao", "ativo"],
    "ticker da elegibilidade operacional",
    obrigatoria=True,
)

col_issuer_elegibilidade = selecionar_coluna_existente(
    df_elegibilidade_operacional,
    ["issuer_code", "codigo_emissor", "empresa", "codigo_empresa"],
    "issuer_code da elegibilidade operacional",
    obrigatoria=False,
)

col_data_sinal = selecionar_coluna_existente(
    df_calendario_sinais,
    ["data_sinal", "data", "data_aporte", "data_referencia"],
    "data do calendário final de sinais",
    obrigatoria=True,
)

col_estrategia_sinal = selecionar_coluna_existente(
    df_calendario_sinais,
    ["estrategia_referencia", "estrategia", "tipo_sinal", "nome_estrategia"],
    "estratégia do calendário final de sinais",
    obrigatoria=True,
)

col_data_compra = selecionar_coluna_existente(
    df_perfil_compras_ticker,
    ["data_aporte", "data_compra", "data", "data_referencia"],
    "data da base de compras por ticker",
    obrigatoria=True,
)

col_ticker_compra = selecionar_coluna_existente(
    df_perfil_compras_ticker,
    ["ticker", "codigo_negociacao", "ativo"],
    "ticker da base de compras por ticker",
    obrigatoria=True,
)

col_issuer_compra = selecionar_coluna_existente(
    df_perfil_compras_ticker,
    ["issuer_code", "codigo_emissor", "empresa", "codigo_empresa"],
    "issuer_code da base de compras por ticker",
    obrigatoria=False,
)

col_estrategia_compra = selecionar_coluna_existente(
    df_perfil_compras_ticker,
    ["estrategia_referencia", "estrategia", "nome_estrategia", "nome_exibicao_estrategia"],
    "estratégia da base de compras por ticker",
    obrigatoria=True,
)

col_nome_estrategia_compra = selecionar_coluna_existente(
    df_perfil_compras_ticker,
    ["nome_exibicao_estrategia", "nome_estrategia", "estrategia_nome"],
    "nome de exibição da estratégia",
    obrigatoria=False,
)

col_chave_aporte_compra = selecionar_coluna_existente(
    df_perfil_compras_ticker,
    ["chave_aporte", "id_aporte", "codigo_aporte"],
    "chave de aporte da base de compras por ticker",
    obrigatoria=False,
)

col_valor_compra = selecionar_coluna_existente(
    df_perfil_compras_ticker,
    ["valor_investido_ticker", "valor_investido", "valor_compra", "valor_total_compra", "valor_alocado_ticker"],
    "valor investido por ticker",
    obrigatoria=False,
)

col_volume = selecionar_coluna_existente(
    df_liquidez_ticker_data,
    [
        "volume_fin_medio_252",
        "volume_fin_media_252",
        "volume_fin_mm252",
        "volume_fin_medio_126",
        "volume_fin_media_126",
        "volume_fin_mm126",
        "volume_fin_medio_63",
        "volume_fin_media_63",
        "volume_fin_mm63",
        "media_volume_fin",
        "volume_fin_medio",
        "volume_fin_media",
        "volume_fin",
    ],
    "métrica de volume financeiro",
    obrigatoria=False,
)

if col_volume is None:
    col_volume = selecionar_coluna_por_tokens(
        df_liquidez_ticker_data,
        tokens_obrigatorios=["volume", "fin"],
        tokens_exclusao=["flag"],
        nome_logico="métrica de volume financeiro",
        obrigatoria=False,
    )

col_frequencia = selecionar_coluna_existente(
    df_liquidez_ticker_data,
    [
        "frequencia_negociacao_252",
        "freq_negociacao_252",
        "pct_pregoes_negociados_252",
        "percentual_pregoes_negociados_252",
        "frequencia_negociacao",
        "freq_negociacao",
        "pct_pregoes_negociados",
        "percentual_pregoes_negociados",
    ],
    "métrica de frequência de negociação",
    obrigatoria=False,
)

if col_frequencia is None:
    col_frequencia = selecionar_coluna_por_tokens(
        df_liquidez_ticker_data,
        tokens_obrigatorios=["freq"],
        tokens_exclusao=["flag"],
        nome_logico="métrica de frequência de negociação",
        obrigatoria=False,
    )

col_pregoes = selecionar_coluna_existente(
    df_liquidez_ticker_data,
    [
        "n_pregoes_negociados_252",
        "qtd_pregoes_negociados_252",
        "pregoes_negociados_252",
        "n_dias_negociados_252",
        "n_pregoes_negociados",
        "qtd_pregoes_negociados",
        "pregoes_negociados",
        "n_dias_negociados",
    ],
    "métrica de quantidade de pregões negociados",
    obrigatoria=False,
)

if col_pregoes is None:
    col_pregoes = selecionar_coluna_por_tokens(
        df_liquidez_ticker_data,
        tokens_obrigatorios=["pregoes", "negoci"],
        tokens_exclusao=["flag"],
        nome_logico="métrica de quantidade de pregões negociados",
        obrigatoria=False,
    )

col_preco = selecionar_coluna_existente(
    df_liquidez_ticker_data,
    ["close_adj", "preco_referencia", "preco_fechamento", "preco", "close"],
    "métrica de preço",
    obrigatoria=False,
)

col_flag_liquidez_ref = selecionar_coluna_existente(
    df_liquidez_ticker_data,
    [
        "elegivel_liquidez",
        "flag_elegivel_liquidez",
        "liquidez_elegivel",
        "flag_liquidez_ok",
        "aprovado_liquidez",
        "flag_aprovado_liquidez",
        "elegivel_operacional",
        "flag_elegivel_operacional",
    ],
    "flag de elegibilidade de liquidez",
    obrigatoria=False,
)

metricas_liquidez = [col for col in [col_volume, col_frequencia, col_pregoes, col_preco] if col is not None]

if len(metricas_liquidez) == 0:
    raise ValueError(
        "Nenhuma métrica de liquidez foi identificada na base 3.2. "
        "A subetapa exige ao menos uma métrica operacional para testar cenários alternativos."
    )

df_liquidez_ticker_data = converter_data_segura(df_liquidez_ticker_data, col_data_liquidez, "3_2_base_liquidez_ticker_data")
df_elegibilidade_operacional = converter_data_segura(df_elegibilidade_operacional, col_data_elegibilidade, "3_4_base_elegibilidade_operacional_data")
df_calendario_sinais = converter_data_segura(df_calendario_sinais, col_data_sinal, "6_4_base_calendario_final_sinais")
df_perfil_compras_ticker = converter_data_segura(df_perfil_compras_ticker, col_data_compra, "12_3_base_perfil_compras_ticker")

df_liquidez_ticker_data[col_ticker_liquidez] = df_liquidez_ticker_data[col_ticker_liquidez].astype(str).str.strip().str.upper()
df_elegibilidade_operacional[col_ticker_elegibilidade] = df_elegibilidade_operacional[col_ticker_elegibilidade].astype(str).str.strip().str.upper()
df_perfil_compras_ticker[col_ticker_compra] = df_perfil_compras_ticker[col_ticker_compra].astype(str).str.strip().str.upper()

if col_issuer_liquidez is not None:
    df_liquidez_ticker_data[col_issuer_liquidez] = df_liquidez_ticker_data[col_issuer_liquidez].astype(str).str.strip().str.upper()
if col_issuer_elegibilidade is not None:
    df_elegibilidade_operacional[col_issuer_elegibilidade] = df_elegibilidade_operacional[col_issuer_elegibilidade].astype(str).str.strip().str.upper()
if col_issuer_compra is not None:
    df_perfil_compras_ticker[col_issuer_compra] = df_perfil_compras_ticker[col_issuer_compra].astype(str).str.strip().str.upper()

df_liquidez_ticker_data = converter_numerico_seguro(df_liquidez_ticker_data, metricas_liquidez)
if col_valor_compra is not None:
    df_perfil_compras_ticker = converter_numerico_seguro(df_perfil_compras_ticker, [col_valor_compra])

print(f"Coluna data liquidez                  : {col_data_liquidez}")
print(f"Coluna ticker liquidez                : {col_ticker_liquidez}")
print(f"Coluna issuer liquidez                : {col_issuer_liquidez}")
print(f"Coluna flag referência liquidez        : {col_flag_liquidez_ref}")
print(f"Métrica volume financeiro              : {col_volume}")
print(f"Métrica frequência de negociação       : {col_frequencia}")
print(f"Métrica pregões negociados             : {col_pregoes}")
print(f"Métrica preço                          : {col_preco}")
print(f"Data do calendário de sinais           : {col_data_sinal}")
print(f"Data da base de compras                : {col_data_compra}")
print(f"Valor por compra                       : {col_valor_compra}")
print("OK")

# ============================================================
# 6) Construção da elegibilidade de referência
# ============================================================

print("\n[6/12] Construção da elegibilidade de referência...")

df_liquidez = df_liquidez_ticker_data.copy()
df_liquidez = criar_chave_data_ticker(df_liquidez, col_data_liquidez, col_ticker_liquidez)

df_eleg_ref = df_elegibilidade_operacional.copy()
df_eleg_ref = criar_chave_data_ticker(df_eleg_ref, col_data_elegibilidade, col_ticker_elegibilidade)

if col_issuer_liquidez is not None:
    df_liquidez = criar_chave_data_issuer(df_liquidez, col_data_liquidez, col_issuer_liquidez)
if col_issuer_elegibilidade is not None:
    df_eleg_ref = criar_chave_data_issuer(df_eleg_ref, col_data_elegibilidade, col_issuer_elegibilidade)

if col_flag_liquidez_ref is not None:
    serie_flag = df_liquidez[col_flag_liquidez_ref]
    if str(serie_flag.dtype) == "bool":
        df_liquidez["elegivel_referencia"] = serie_flag.fillna(False).astype(bool)
    else:
        df_liquidez["elegivel_referencia"] = serie_flag.astype(str).str.strip().str.lower().isin(
            ["1", "true", "t", "sim", "s", "yes", "y", "ok"]
        )
    origem_referencia = f"flag_3_2:{col_flag_liquidez_ref}"
else:
    chaves_ref_ticker = set(df_eleg_ref["chave_data_ticker"].dropna().astype(str).unique())
    df_liquidez["elegivel_referencia"] = df_liquidez["chave_data_ticker"].astype(str).isin(chaves_ref_ticker)
    origem_referencia = "membership_3_4:data_ticker"

n_linhas_liquidez = len(df_liquidez)
n_elegiveis_referencia = int(df_liquidez["elegivel_referencia"].sum())
n_datas_liquidez = int(df_liquidez[col_data_liquidez].nunique())
n_tickers_liquidez = int(df_liquidez[col_ticker_liquidez].nunique())

print(f"Origem da referência de liquidez       : {origem_referencia}")
print(f"Linhas na base de liquidez             : {n_linhas_liquidez:,}")
print(f"Linhas elegíveis na referência         : {n_elegiveis_referencia:,}")
print(f"Datas com liquidez                     : {n_datas_liquidez:,}")
print(f"Tickers na base de liquidez            : {n_tickers_liquidez:,}")
print("OK")

# ============================================================
# 7) Definição dos cenários alternativos de liquidez
# ============================================================

print("\n[7/12] Definição dos cenários alternativos de liquidez...")

cenarios_liquidez = [
    {
        "cenario_liquidez": "muito_flexivel",
        "tipo_limiar": "multiplicativo_inferior",
        "multiplicador_limiar": 0.50,
        "quantil_limiar": np.nan,
        "descricao_cenario": "Critérios 50% abaixo da referência inferida.",
        "ordem_cenario": 1,
    },
    {
        "cenario_liquidez": "flexivel",
        "tipo_limiar": "multiplicativo_inferior",
        "multiplicador_limiar": 0.75,
        "quantil_limiar": np.nan,
        "descricao_cenario": "Critérios 25% abaixo da referência inferida.",
        "ordem_cenario": 2,
    },
    {
        "cenario_liquidez": "referencia_oficial",
        "tipo_limiar": "referencia_oficial",
        "multiplicador_limiar": 1.00,
        "quantil_limiar": 0.01,
        "descricao_cenario": "Elegibilidade oficial já validada nas etapas 3.2 a 3.4.",
        "ordem_cenario": 3,
    },
    {
        "cenario_liquidez": "rigoroso",
        "tipo_limiar": "quantil_superior_elegiveis_referencia",
        "multiplicador_limiar": np.nan,
        "quantil_limiar": 0.10,
        "descricao_cenario": "Critérios mais restritivos pelo percentil 10 dos elegíveis oficiais.",
        "ordem_cenario": 4,
    },
    {
        "cenario_liquidez": "muito_rigoroso",
        "tipo_limiar": "quantil_superior_elegiveis_referencia",
        "multiplicador_limiar": np.nan,
        "quantil_limiar": 0.25,
        "descricao_cenario": "Critérios mais restritivos pelo percentil 25 dos elegíveis oficiais.",
        "ordem_cenario": 5,
    },
]

limiares_referencia = {}
limiares_cenarios = {}
valores_validos_por_metrica = {}

for metrica in metricas_liquidez:
    serie_metrica = pd.to_numeric(df_liquidez[metrica], errors="coerce").replace([np.inf, -np.inf], np.nan)
    valores_validos = serie_metrica.dropna()
    valores_validos = valores_validos[valores_validos > 0]
    valores_validos_por_metrica[metrica] = valores_validos
    limiares_referencia[metrica] = inferir_limiar_referencia(
        df_liquidez[metrica],
        df_liquidez["elegivel_referencia"],
        metrica,
    )

registros_parametros_cenarios = []

for cenario in cenarios_liquidez:
    for metrica in metricas_liquidez:
        limiar_base = limiares_referencia.get(metrica, np.nan)
        serie_metrica = pd.to_numeric(df_liquidez[metrica], errors="coerce").replace([np.inf, -np.inf], np.nan)
        valores_ref = serie_metrica[df_liquidez["elegivel_referencia"]].dropna()
        valores_ref = valores_ref[valores_ref > 0]

        if cenario["tipo_limiar"] == "referencia_oficial":
            limiar_cenario = limiar_base
            origem_limiar = "referencia_oficial"
        elif cenario["tipo_limiar"] == "multiplicativo_inferior":
            limiar_cenario = limiar_base * cenario["multiplicador_limiar"] if pd.notna(limiar_base) else np.nan
            origem_limiar = "multiplicador_sobre_percentil_1_elegiveis_referencia"
        elif cenario["tipo_limiar"] == "quantil_superior_elegiveis_referencia":
            if len(valores_ref) > 0:
                limiar_cenario = float(valores_ref.quantile(cenario["quantil_limiar"]))
            else:
                limiar_cenario = np.nan
            origem_limiar = f"percentil_{int(cenario['quantil_limiar'] * 100)}_elegiveis_referencia"
        else:
            limiar_cenario = np.nan
            origem_limiar = "indefinido"

        if pd.notna(limiar_cenario) and len(valores_validos_por_metrica.get(metrica, pd.Series(dtype=float))) > 0:
            limiar_cenario = min(float(limiar_cenario), float(valores_validos_por_metrica[metrica].max()))

        limiares_cenarios[(cenario["cenario_liquidez"], metrica)] = limiar_cenario

        registros_parametros_cenarios.append(
            {
                "cenario_liquidez": cenario["cenario_liquidez"],
                "ordem_cenario": cenario["ordem_cenario"],
                "descricao_cenario": cenario["descricao_cenario"],
                "tipo_limiar": cenario["tipo_limiar"],
                "metrica_liquidez": metrica,
                "limiar_referencia_inferido": limiar_base,
                "multiplicador_limiar": cenario["multiplicador_limiar"],
                "quantil_limiar": cenario["quantil_limiar"],
                "limiar_cenario": limiar_cenario,
                "origem_limiar": origem_limiar,
            }
        )

df_parametros_cenarios = pd.DataFrame(registros_parametros_cenarios).sort_values(
    ["ordem_cenario", "metrica_liquidez"]
).reset_index(drop=True)

print(f"Cenários definidos                    : {len(cenarios_liquidez)}")
print(f"Métricas usadas nos cenários           : {len(metricas_liquidez)}")
print("\nParâmetros dos cenários de liquidez:")
print(df_parametros_cenarios.to_string(index=False))
print("OK")

# ============================================================
# 8) Aplicação dos cenários alternativos de liquidez
# ============================================================

print("\n[8/12] Aplicação dos cenários alternativos de liquidez...")

df_base_cenarios = df_liquidez[
    [col_data_liquidez, col_ticker_liquidez]
    + ([col_issuer_liquidez] if col_issuer_liquidez is not None else [])
    + ["chave_data_ticker", "elegivel_referencia"]
    + metricas_liquidez
].copy()

df_base_cenarios = df_base_cenarios.rename(
    columns={
        col_data_liquidez: "data",
        col_ticker_liquidez: "ticker",
    }
)

if col_issuer_liquidez is not None:
    df_base_cenarios = df_base_cenarios.rename(columns={col_issuer_liquidez: "issuer_code"})
else:
    df_base_cenarios["issuer_code"] = "sem_issuer_code"

for cenario in cenarios_liquidez:
    nome_coluna_cenario = f"elegivel_liquidez_{cenario['cenario_liquidez']}"

    if cenario["cenario_liquidez"] == "referencia_oficial":
        df_base_cenarios[nome_coluna_cenario] = df_base_cenarios["elegivel_referencia"].fillna(False).astype(bool)
    else:
        mascara = pd.Series(True, index=df_base_cenarios.index)
        for metrica in metricas_liquidez:
            limiar_cenario = limiares_cenarios.get((cenario["cenario_liquidez"], metrica), np.nan)

            if pd.notna(limiar_cenario):
                mascara = mascara & (pd.to_numeric(df_base_cenarios[metrica], errors="coerce") >= limiar_cenario)
            else:
                mascara = mascara & False

        df_base_cenarios[nome_coluna_cenario] = mascara.fillna(False).astype(bool)

colunas_flags_cenarios = [f"elegivel_liquidez_{c['cenario_liquidez']}" for c in cenarios_liquidez]

print(f"Base de cenários ticker/data           : {len(df_base_cenarios):,} linhas")
print(f"Flags de cenários criadas              : {len(colunas_flags_cenarios)}")
print("OK")

# ============================================================
# 9) Construção dos resumos de universo e comparação com referência
# ============================================================

print("\n[9/12] Construção dos resumos de universo e comparação com referência...")

registros_resumo_universo = []
registros_comparacao_referencia = []

flag_ref = "elegivel_liquidez_referencia_oficial"

for cenario in cenarios_liquidez:
    nome_cenario = cenario["cenario_liquidez"]
    flag_cenario = f"elegivel_liquidez_{nome_cenario}"

    df_eleg_cenario = df_base_cenarios[df_base_cenarios[flag_cenario]].copy()

    registros_resumo_universo.append(
        {
            "cenario_liquidez": nome_cenario,
            "ordem_cenario": cenario["ordem_cenario"],
            "n_linhas_ticker_data": int(len(df_base_cenarios)),
            "n_linhas_elegiveis": int(df_base_cenarios[flag_cenario].sum()),
            "pct_linhas_elegiveis": float(df_base_cenarios[flag_cenario].mean()) if len(df_base_cenarios) > 0 else np.nan,
            "n_datas_com_elegiveis": int(df_eleg_cenario["data"].nunique()),
            "n_tickers_distintos_elegiveis": int(df_eleg_cenario["ticker"].nunique()),
            "n_empresas_distintas_elegiveis": int(df_eleg_cenario["issuer_code"].nunique()),
            "media_tickers_elegiveis_por_data": float(df_eleg_cenario.groupby("data")["ticker"].nunique().mean()) if len(df_eleg_cenario) > 0 else np.nan,
            "mediana_tickers_elegiveis_por_data": float(df_eleg_cenario.groupby("data")["ticker"].nunique().median()) if len(df_eleg_cenario) > 0 else np.nan,
            "media_empresas_elegiveis_por_data": float(df_eleg_cenario.groupby("data")["issuer_code"].nunique().mean()) if len(df_eleg_cenario) > 0 else np.nan,
            "mediana_empresas_elegiveis_por_data": float(df_eleg_cenario.groupby("data")["issuer_code"].nunique().median()) if len(df_eleg_cenario) > 0 else np.nan,
        }
    )

    comuns = (df_base_cenarios[flag_cenario] & df_base_cenarios[flag_ref]).sum()
    adicionados = (df_base_cenarios[flag_cenario] & ~df_base_cenarios[flag_ref]).sum()
    removidos = (~df_base_cenarios[flag_cenario] & df_base_cenarios[flag_ref]).sum()
    uniao = (df_base_cenarios[flag_cenario] | df_base_cenarios[flag_ref]).sum()

    registros_comparacao_referencia.append(
        {
            "cenario_liquidez": nome_cenario,
            "ordem_cenario": cenario["ordem_cenario"],
            "n_elegiveis_cenario": int(df_base_cenarios[flag_cenario].sum()),
            "n_elegiveis_referencia": int(df_base_cenarios[flag_ref].sum()),
            "n_comuns_com_referencia": int(comuns),
            "n_adicionados_vs_referencia": int(adicionados),
            "n_removidos_vs_referencia": int(removidos),
            "delta_liquido_elegiveis_vs_referencia": int(df_base_cenarios[flag_cenario].sum() - df_base_cenarios[flag_ref].sum()),
            "pct_jaccard_vs_referencia": float(comuns / uniao) if uniao > 0 else np.nan,
        }
    )

df_resumo_universo = pd.DataFrame(registros_resumo_universo).sort_values("ordem_cenario").reset_index(drop=True)
df_comparacao_referencia = pd.DataFrame(registros_comparacao_referencia).sort_values("ordem_cenario").reset_index(drop=True)

registros_cobertura_diaria = []

for cenario in cenarios_liquidez:
    nome_cenario = cenario["cenario_liquidez"]
    flag_cenario = f"elegivel_liquidez_{nome_cenario}"

    df_tmp = (
        df_base_cenarios[df_base_cenarios[flag_cenario]]
        .groupby("data")
        .agg(
            n_tickers_elegiveis=("ticker", "nunique"),
            n_empresas_elegiveis=("issuer_code", "nunique"),
        )
        .reset_index()
    )

    df_tmp["cenario_liquidez"] = nome_cenario
    df_tmp["ordem_cenario"] = cenario["ordem_cenario"]
    registros_cobertura_diaria.append(df_tmp)

df_cobertura_diaria = pd.concat(registros_cobertura_diaria, ignore_index=True) if registros_cobertura_diaria else pd.DataFrame()
df_cobertura_diaria["ano"] = pd.to_datetime(df_cobertura_diaria["data"], errors="coerce").dt.year

print(f"Resumo de universo por cenário         : {len(df_resumo_universo):,} linhas")
print(f"Comparação com referência              : {len(df_comparacao_referencia):,} linhas")
print(f"Cobertura diária por cenário           : {len(df_cobertura_diaria):,} linhas")
print("OK")

# ============================================================
# 10) Impacto sobre datas de sinais e cestas de compras
# ============================================================

print("\n[10/12] Impacto sobre datas de sinais e cestas de compras...")

df_sinais = df_calendario_sinais[[col_data_sinal, col_estrategia_sinal]].copy()
df_sinais = df_sinais.rename(columns={col_data_sinal: "data_sinal", col_estrategia_sinal: "estrategia_referencia"})
df_sinais["data_sinal"] = pd.to_datetime(df_sinais["data_sinal"], errors="coerce")
df_sinais["estrategia_referencia"] = df_sinais["estrategia_referencia"].astype(str).str.strip().str.lower()
df_sinais = df_sinais.dropna(subset=["data_sinal"]).drop_duplicates().reset_index(drop=True)

df_cobertura_sinais = df_cobertura_diaria.rename(columns={"data": "data_sinal"}).copy()
df_cobertura_sinais["data_sinal"] = pd.to_datetime(df_cobertura_sinais["data_sinal"], errors="coerce")

df_impacto_sinais = df_sinais.merge(
    df_cobertura_sinais,
    how="left",
    on="data_sinal",
)

df_impacto_sinais["n_tickers_elegiveis"] = pd.to_numeric(df_impacto_sinais["n_tickers_elegiveis"], errors="coerce")
df_impacto_sinais["n_empresas_elegiveis"] = pd.to_numeric(df_impacto_sinais["n_empresas_elegiveis"], errors="coerce")

df_impacto_sinais = (
    df_impacto_sinais
    .groupby(["estrategia_referencia", "cenario_liquidez", "ordem_cenario"], dropna=False)
    .agg(
        n_sinais=("data_sinal", "nunique"),
        n_sinais_sem_cobertura_liquidez=("n_tickers_elegiveis", lambda x: int(x.isna().sum())),
        media_tickers_elegiveis_datas_sinal=("n_tickers_elegiveis", "mean"),
        mediana_tickers_elegiveis_datas_sinal=("n_tickers_elegiveis", "median"),
        minimo_tickers_elegiveis_datas_sinal=("n_tickers_elegiveis", "min"),
        maximo_tickers_elegiveis_datas_sinal=("n_tickers_elegiveis", "max"),
        media_empresas_elegiveis_datas_sinal=("n_empresas_elegiveis", "mean"),
        mediana_empresas_elegiveis_datas_sinal=("n_empresas_elegiveis", "median"),
    )
    .reset_index()
    .sort_values(["estrategia_referencia", "ordem_cenario"])
    .reset_index(drop=True)
)

df_compras = df_perfil_compras_ticker.copy()
df_compras = df_compras.rename(columns={col_data_compra: "data_aporte", col_ticker_compra: "ticker", col_estrategia_compra: "estrategia_referencia"})

if col_nome_estrategia_compra is not None and col_nome_estrategia_compra in df_compras.columns:
    df_compras = df_compras.rename(columns={col_nome_estrategia_compra: "nome_exibicao_estrategia"})
else:
    df_compras["nome_exibicao_estrategia"] = df_compras["estrategia_referencia"].astype(str)

if col_issuer_compra is not None and col_issuer_compra in df_compras.columns:
    df_compras = df_compras.rename(columns={col_issuer_compra: "issuer_code"})
else:
    df_compras["issuer_code"] = "sem_issuer_code"

if col_chave_aporte_compra is not None and col_chave_aporte_compra in df_compras.columns:
    df_compras = df_compras.rename(columns={col_chave_aporte_compra: "chave_aporte"})
else:
    df_compras["chave_aporte"] = (
        df_compras["estrategia_referencia"].astype(str).str.strip()
        + "__"
        + pd.to_datetime(df_compras["data_aporte"], errors="coerce").dt.strftime("%Y-%m-%d")
    )

if col_valor_compra is not None and col_valor_compra in df_compras.columns:
    df_compras = df_compras.rename(columns={col_valor_compra: "valor_investido_ticker"})
else:
    df_compras["valor_investido_ticker"] = 1.0

df_compras["data_aporte"] = pd.to_datetime(df_compras["data_aporte"], errors="coerce")
df_compras["ticker"] = df_compras["ticker"].astype(str).str.strip().str.upper()
df_compras["issuer_code"] = df_compras["issuer_code"].astype(str).str.strip().str.upper()
df_compras["estrategia_referencia"] = df_compras["estrategia_referencia"].astype(str).str.strip().str.lower()
df_compras["valor_investido_ticker"] = pd.to_numeric(df_compras["valor_investido_ticker"], errors="coerce").fillna(0.0)

colunas_lookup_liquidez = ["data", "ticker"] + colunas_flags_cenarios
df_lookup_cenarios = df_base_cenarios[colunas_lookup_liquidez].copy()
df_lookup_cenarios["data"] = pd.to_datetime(df_lookup_cenarios["data"], errors="coerce")
df_lookup_cenarios["ticker"] = df_lookup_cenarios["ticker"].astype(str).str.strip().str.upper()

df_compras = df_compras.merge(
    df_lookup_cenarios,
    how="left",
    left_on=["data_aporte", "ticker"],
    right_on=["data", "ticker"],
)

registros_impacto_cestas = []

for cenario in cenarios_liquidez:
    nome_cenario = cenario["cenario_liquidez"]
    flag_cenario = f"elegivel_liquidez_{nome_cenario}"

    if flag_cenario not in df_compras.columns:
        df_compras[flag_cenario] = False

    df_compras[flag_cenario] = df_compras[flag_cenario].fillna(False).astype(bool)

    df_tmp = df_compras.copy()
    df_tmp["ticker_mantido_cenario"] = df_tmp[flag_cenario].astype(int)
    df_tmp["valor_mantido_cenario"] = np.where(df_tmp[flag_cenario], df_tmp["valor_investido_ticker"], 0.0)

    df_tmp_aporte = (
        df_tmp
        .groupby(["estrategia_referencia", "nome_exibicao_estrategia", "chave_aporte", "data_aporte"], dropna=False)
        .agg(
            n_tickers_comprados=("ticker", "nunique"),
            n_tickers_mantidos_liquidez=("ticker_mantido_cenario", "sum"),
            valor_investido_total=("valor_investido_ticker", "sum"),
            valor_investido_mantido_liquidez=("valor_mantido_cenario", "sum"),
        )
        .reset_index()
    )

    df_tmp_aporte["pct_tickers_mantidos_liquidez"] = np.where(
        df_tmp_aporte["n_tickers_comprados"] > 0,
        df_tmp_aporte["n_tickers_mantidos_liquidez"] / df_tmp_aporte["n_tickers_comprados"],
        np.nan,
    )
    df_tmp_aporte["pct_valor_mantido_liquidez"] = np.where(
        df_tmp_aporte["valor_investido_total"] > 0,
        df_tmp_aporte["valor_investido_mantido_liquidez"] / df_tmp_aporte["valor_investido_total"],
        np.nan,
    )
    df_tmp_aporte["cenario_liquidez"] = nome_cenario
    df_tmp_aporte["ordem_cenario"] = cenario["ordem_cenario"]

    registros_impacto_cestas.append(df_tmp_aporte)

df_impacto_cestas_aporte = pd.concat(registros_impacto_cestas, ignore_index=True) if registros_impacto_cestas else pd.DataFrame()

df_impacto_cestas = (
    df_impacto_cestas_aporte
    .groupby(["estrategia_referencia", "nome_exibicao_estrategia", "cenario_liquidez", "ordem_cenario"], dropna=False)
    .agg(
        n_aportes=("chave_aporte", "nunique"),
        primeira_data_aporte=("data_aporte", "min"),
        ultima_data_aporte=("data_aporte", "max"),
        n_tickers_comprados_total=("n_tickers_comprados", "sum"),
        n_tickers_mantidos_liquidez_total=("n_tickers_mantidos_liquidez", "sum"),
        pct_tickers_mantidos_liquidez_medio=("pct_tickers_mantidos_liquidez", "mean"),
        pct_tickers_mantidos_liquidez_mediano=("pct_tickers_mantidos_liquidez", "median"),
        valor_investido_total=("valor_investido_total", "sum"),
        valor_investido_mantido_liquidez=("valor_investido_mantido_liquidez", "sum"),
        pct_valor_mantido_liquidez_medio=("pct_valor_mantido_liquidez", "mean"),
        pct_valor_mantido_liquidez_mediano=("pct_valor_mantido_liquidez", "median"),
    )
    .reset_index()
    .sort_values(["estrategia_referencia", "nome_exibicao_estrategia", "ordem_cenario"])
    .reset_index(drop=True)
)

df_impacto_cestas["pct_tickers_mantidos_liquidez_total"] = np.where(
    df_impacto_cestas["n_tickers_comprados_total"] > 0,
    df_impacto_cestas["n_tickers_mantidos_liquidez_total"] / df_impacto_cestas["n_tickers_comprados_total"],
    np.nan,
)

df_impacto_cestas["pct_valor_mantido_liquidez_total"] = np.where(
    df_impacto_cestas["valor_investido_total"] > 0,
    df_impacto_cestas["valor_investido_mantido_liquidez"] / df_impacto_cestas["valor_investido_total"],
    np.nan,
)

print(f"Impacto sobre sinais                   : {len(df_impacto_sinais):,} linhas")
print(f"Impacto sobre cestas de compras        : {len(df_impacto_cestas):,} linhas")
print("OK")

# ============================================================
# 11) Construção da base de gráficos e auditoria
# ============================================================

print("\n[11/12] Construção da base de gráficos e auditoria...")

df_base_grafico = (
    df_cobertura_diaria
    .groupby(["cenario_liquidez", "ordem_cenario", "ano"], dropna=False)
    .agg(
        n_datas=("data", "nunique"),
        media_tickers_elegiveis=("n_tickers_elegiveis", "mean"),
        mediana_tickers_elegiveis=("n_tickers_elegiveis", "median"),
        media_empresas_elegiveis=("n_empresas_elegiveis", "mean"),
        mediana_empresas_elegiveis=("n_empresas_elegiveis", "median"),
    )
    .reset_index()
    .sort_values(["ordem_cenario", "ano"])
    .reset_index(drop=True)
)

n_cenarios_gerados = len(cenarios_liquidez)
n_flags_cenarios = len([col for col in colunas_flags_cenarios if col in df_base_cenarios.columns])
n_cenarios_alternativos_sem_elegiveis = int(
    df_resumo_universo[
        (df_resumo_universo["cenario_liquidez"] != "referencia_oficial")
        & (pd.to_numeric(df_resumo_universo["n_linhas_elegiveis"], errors="coerce").fillna(0) == 0)
    ]["cenario_liquidez"].nunique()
)
n_datas_sem_sinal_cobertura = int(df_impacto_sinais["n_sinais_sem_cobertura_liquidez"].fillna(0).sum()) if "n_sinais_sem_cobertura_liquidez" in df_impacto_sinais.columns else 0
n_compras_sem_lookup_liquidez = int(df_compras[flag_ref].isna().sum()) if flag_ref in df_compras.columns else 0
n_metricas_infinitas = int(
    df_base_cenarios.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).isna().sum().sum()
    - df_base_cenarios.select_dtypes(include=[np.number]).isna().sum().sum()
)

itens_auditoria = [
    {
        "item": "linhas_liquidez_origem_3_2",
        "valor": n_linhas_liquidez,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Quantidade de linhas na base de liquidez por ticker/data da 3.2.",
    },
    {
        "item": "datas_liquidez_origem_3_2",
        "valor": n_datas_liquidez,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Quantidade de datas disponíveis na base de liquidez por ticker/data.",
    },
    {
        "item": "tickers_liquidez_origem_3_2",
        "valor": n_tickers_liquidez,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Quantidade de tickers disponíveis na base de liquidez por ticker/data.",
    },
    {
        "item": "metricas_liquidez_identificadas",
        "valor": len(metricas_liquidez),
        "valor_referencia": ">= 1",
        "status": status_ok_alerta_erro(len(metricas_liquidez) >= 1),
        "observacao": "A sensibilidade exige ao menos uma métrica operacional de liquidez.",
    },
    {
        "item": "cenarios_liquidez_gerados",
        "valor": n_cenarios_gerados,
        "valor_referencia": 5,
        "status": status_ok_alerta_erro(n_cenarios_gerados == 5),
        "observacao": "A subetapa deve gerar cenários flexíveis, referência e rigorosos.",
    },
    {
        "item": "flags_cenarios_criadas",
        "valor": n_flags_cenarios,
        "valor_referencia": n_cenarios_gerados,
        "status": status_ok_alerta_erro(n_flags_cenarios == n_cenarios_gerados),
        "observacao": "Cada cenário de liquidez deve possuir uma flag na base ticker/data.",
    },
    {
        "item": "cenarios_alternativos_sem_elegiveis",
        "valor": n_cenarios_alternativos_sem_elegiveis,
        "valor_referencia": 0,
        "status": status_ok_alerta_erro(n_cenarios_alternativos_sem_elegiveis == 0),
        "observacao": "Cenários alternativos de liquidez não devem ficar vazios por limiar operacional inviável.",
    },
    {
        "item": "elegiveis_referencia",
        "valor": n_elegiveis_referencia,
        "valor_referencia": "> 0",
        "status": status_ok_alerta_erro(n_elegiveis_referencia > 0),
        "observacao": "A referência oficial de liquidez deve possuir ativos elegíveis.",
    },
    {
        "item": "origem_referencia_liquidez",
        "valor": origem_referencia,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Origem usada para construir a elegibilidade de referência.",
    },
    {
        "item": "sinais_sem_cobertura_liquidez",
        "valor": n_datas_sem_sinal_cobertura,
        "valor_referencia": 0,
        "status": status_ok_alerta_erro(n_datas_sem_sinal_cobertura == 0),
        "observacao": "Datas de sinais deveriam possuir cobertura de liquidez para cálculo de impacto.",
    },
    {
        "item": "compras_sem_lookup_liquidez_referencia",
        "valor": n_compras_sem_lookup_liquidez,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Compras sem lookup exato de liquidez na data/ticker; item informativo para análise operacional.",
    },
    {
        "item": "metricas_infinitas",
        "valor": n_metricas_infinitas,
        "valor_referencia": 0,
        "status": status_ok_alerta_erro(n_metricas_infinitas == 0),
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = construir_tabela_auditoria(itens_auditoria)
n_erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Base gráfico liquidez cenários         : {len(df_base_grafico):,} linhas")
print(f"Itens de auditoria                     : {len(df_auditoria):,}")
print(f"Erros bloqueantes                      : {n_erros_bloqueantes:,}")
print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(df_base_cenarios, CAMINHO_BASE_CENARIOS_TICKER_DATA, index=False)
salvar_dataframe(df_parametros_cenarios, CAMINHO_TBL_PARAMETROS_CENARIOS, index=False)
salvar_dataframe(df_resumo_universo, CAMINHO_TBL_RESUMO_UNIVERSO, index=False)
salvar_dataframe(df_cobertura_diaria, CAMINHO_TBL_COBERTURA_DIARIA, index=False)
salvar_dataframe(df_comparacao_referencia, CAMINHO_TBL_COMPARACAO_REFERENCIA, index=False)
salvar_dataframe(df_impacto_sinais, CAMINHO_TBL_IMPACTO_SINAIS, index=False)
salvar_dataframe(df_impacto_cestas, CAMINHO_TBL_IMPACTO_CESTAS, index=False)
salvar_dataframe(df_base_grafico, CAMINHO_TBL_BASE_GRAFICO, index=False)
salvar_dataframe(df_auditoria, CAMINHO_TBL_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da sensibilidade dos critérios de liquidez:")
print(df_auditoria.to_string(index=False))

print("\nResumo do universo elegível por cenário:")
print(df_resumo_universo.to_string(index=False))

print("\nComparação com a referência oficial:")
print(df_comparacao_referencia.to_string(index=False))

print("\nImpacto dos cenários nas datas de sinais:")
print(df_impacto_sinais.to_string(index=False))

print("\nImpacto dos cenários nas cestas de compras - amostra:")
print(df_impacto_cestas.head(40).to_string(index=False))

print("\nBase anual para gráficos - amostra:")
print(df_base_grafico.head(40).to_string(index=False))

print("\nArquivos salvos na subetapa 13.2:")
print(f"- {CAMINHO_BASE_CENARIOS_TICKER_DATA}")
print(f"- {CAMINHO_TBL_PARAMETROS_CENARIOS}")
print(f"- {CAMINHO_TBL_RESUMO_UNIVERSO}")
print(f"- {CAMINHO_TBL_COBERTURA_DIARIA}")
print(f"- {CAMINHO_TBL_COMPARACAO_REFERENCIA}")
print(f"- {CAMINHO_TBL_IMPACTO_SINAIS}")
print(f"- {CAMINHO_TBL_IMPACTO_CESTAS}")
print(f"- {CAMINHO_TBL_BASE_GRAFICO}")
print(f"- {CAMINHO_TBL_AUDITORIA}")

if n_erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 13.2 identificou erros bloqueantes. Verifique a tabela de auditoria.")

print("\nETAPA 13.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 13.2 - SENSIBILIDADE DOS CRITÉRIOS DE LIQUIDEZ

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - parâmetros de liquidez da 3.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_1_tbl_parametros_liquidez.parquet
Entrada - liquidez ticker/data da 3.2        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_2_base_liquidez_ticker_data.parquet
Entrada - elegibilidade operacional da 3.4   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_4_base_elegibilidade_operacional_data.parquet
Entrada - calendário final de sinais da 6.4  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\eta

## Etapa 13.3) Sensibilidade do Filtro Contábil

In [66]:
%%time
# ============================================================
# Etapa 13.3) Sensibilidade do Filtro Contábil
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 13.3 - SENSIBILIDADE DO FILTRO CONTÁBIL")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_3 = Path(DIRETORIOS_PROJETO["etapa_3"])
DIR_ETAPA_4 = Path(DIRETORIOS_PROJETO["etapa_4"])
DIR_ETAPA_6 = Path(DIRETORIOS_PROJETO["etapa_6"])
DIR_ETAPA_12 = Path(DIRETORIOS_PROJETO["etapa_12"])
DIR_ETAPA_13 = Path(DIRETORIOS_PROJETO["etapa_13"])
DIR_ETAPA_13.mkdir(parents=True, exist_ok=True)

CAMINHO_ELEGIBILIDADE_OPERACIONAL_3_4 = DIR_ETAPA_3 / "3_4_base_elegibilidade_operacional_data.parquet"
CAMINHO_REGRAS_FILTRO_CONTABIL_4_3 = DIR_ETAPA_4 / "4_3_tbl_regras_filtro_contabil.parquet"
CAMINHO_PARAMETROS_FILTRO_CONTABIL_4_3 = DIR_ETAPA_4 / "4_3_tbl_parametros_filtro_contabil.parquet"
CAMINHO_ELEGIBILIDADE_CONTABIL_4_4 = DIR_ETAPA_4 / "4_4_base_elegibilidade_contabil_janela.parquet"
CAMINHO_TICKERS_APTOS_4_5 = DIR_ETAPA_4 / "4_5_base_tickers_aptos_janela_contabil.parquet"
CAMINHO_CALENDARIO_SINAIS_6_4 = DIR_ETAPA_6 / "6_4_base_calendario_final_sinais.parquet"
CAMINHO_PERFIL_COMPRAS_TICKER_12_3 = DIR_ETAPA_12 / "12_3_base_perfil_compras_ticker.parquet"

CAMINHO_BASE_CENARIOS_EMPRESA_JANELA = DIR_ETAPA_13 / "13_3_base_filtro_contabil_cenarios_empresa_janela.parquet"
CAMINHO_PARAMETROS_CENARIOS = DIR_ETAPA_13 / "13_3_tbl_parametros_cenarios_filtro_contabil.parquet"
CAMINHO_RESUMO_UNIVERSO = DIR_ETAPA_13 / "13_3_tbl_resumo_universo_contabil_cenarios.parquet"
CAMINHO_COMPARACAO_REFERENCIA = DIR_ETAPA_13 / "13_3_tbl_comparacao_referencia_filtro_contabil.parquet"
CAMINHO_IMPACTO_SINAIS = DIR_ETAPA_13 / "13_3_tbl_impacto_sinais_filtro_contabil.parquet"
CAMINHO_IMPACTO_CESTAS = DIR_ETAPA_13 / "13_3_tbl_impacto_cestas_compras_filtro_contabil.parquet"
CAMINHO_MOTIVOS_REPROVACAO = DIR_ETAPA_13 / "13_3_tbl_motivos_reprovacao_cenarios.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_13 / "13_3_tbl_base_grafico_filtro_contabil_cenarios.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_13 / "13_3_tbl_auditoria_validacao_sensibilidade_filtro_contabil.parquet"

CAMINHOS_ENTRADA_OBRIGATORIOS = [
    CAMINHO_ELEGIBILIDADE_OPERACIONAL_3_4,
    CAMINHO_ELEGIBILIDADE_CONTABIL_4_4,
    CAMINHO_TICKERS_APTOS_4_5,
    CAMINHO_CALENDARIO_SINAIS_6_4,
    CAMINHO_PERFIL_COMPRAS_TICKER_12_3,
]

for caminho in CAMINHOS_ENTRADA_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - elegibilidade operacional da 3.4   : {CAMINHO_ELEGIBILIDADE_OPERACIONAL_3_4}")
print(f"Entrada - regras filtro contábil da 4.3      : {CAMINHO_REGRAS_FILTRO_CONTABIL_4_3}")
print(f"Entrada - parâmetros filtro contábil da 4.3  : {CAMINHO_PARAMETROS_FILTRO_CONTABIL_4_3}")
print(f"Entrada - elegibilidade contábil da 4.4      : {CAMINHO_ELEGIBILIDADE_CONTABIL_4_4}")
print(f"Entrada - tickers aptos da 4.5               : {CAMINHO_TICKERS_APTOS_4_5}")
print(f"Entrada - calendário final de sinais da 6.4  : {CAMINHO_CALENDARIO_SINAIS_6_4}")
print(f"Entrada - perfil compras ticker da 12.3      : {CAMINHO_PERFIL_COMPRAS_TICKER_12_3}")
print(f"Saída   - base cenários empresa/janela       : {CAMINHO_BASE_CENARIOS_EMPRESA_JANELA}")
print(f"Saída   - parâmetros dos cenários            : {CAMINHO_PARAMETROS_CENARIOS}")
print(f"Saída   - resumo universo                    : {CAMINHO_RESUMO_UNIVERSO}")
print(f"Saída   - comparação com referência          : {CAMINHO_COMPARACAO_REFERENCIA}")
print(f"Saída   - impacto em sinais                  : {CAMINHO_IMPACTO_SINAIS}")
print(f"Saída   - impacto em cestas de compras       : {CAMINHO_IMPACTO_CESTAS}")
print(f"Saída   - motivos de reprovação              : {CAMINHO_MOTIVOS_REPROVACAO}")
print(f"Saída   - base gráfico                       : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação             : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/12] Carga das bases oficiais da subetapa...")

df_elegibilidade_operacional = pd.read_parquet(CAMINHO_ELEGIBILIDADE_OPERACIONAL_3_4)
df_elegibilidade_contabil = pd.read_parquet(CAMINHO_ELEGIBILIDADE_CONTABIL_4_4)
df_tickers_aptos = pd.read_parquet(CAMINHO_TICKERS_APTOS_4_5)
df_calendario_sinais = pd.read_parquet(CAMINHO_CALENDARIO_SINAIS_6_4)
df_perfil_compras_ticker = pd.read_parquet(CAMINHO_PERFIL_COMPRAS_TICKER_12_3)

if Path(CAMINHO_REGRAS_FILTRO_CONTABIL_4_3).exists():
    df_regras_filtro_contabil = pd.read_parquet(CAMINHO_REGRAS_FILTRO_CONTABIL_4_3)
else:
    df_regras_filtro_contabil = pd.DataFrame()

if Path(CAMINHO_PARAMETROS_FILTRO_CONTABIL_4_3).exists():
    df_parametros_filtro_contabil = pd.read_parquet(CAMINHO_PARAMETROS_FILTRO_CONTABIL_4_3)
else:
    df_parametros_filtro_contabil = pd.DataFrame()

print(f"Elegibilidade operacional da 3.4   : {len(df_elegibilidade_operacional):,} linhas x {df_elegibilidade_operacional.shape[1]:,} colunas")
print(f"Regras filtro contábil da 4.3      : {len(df_regras_filtro_contabil):,} linhas x {df_regras_filtro_contabil.shape[1]:,} colunas")
print(f"Parâmetros filtro contábil da 4.3  : {len(df_parametros_filtro_contabil):,} linhas x {df_parametros_filtro_contabil.shape[1]:,} colunas")
print(f"Elegibilidade contábil da 4.4      : {len(df_elegibilidade_contabil):,} linhas x {df_elegibilidade_contabil.shape[1]:,} colunas")
print(f"Tickers aptos da 4.5               : {len(df_tickers_aptos):,} linhas x {df_tickers_aptos.shape[1]:,} colunas")
print(f"Calendário final de sinais da 6.4  : {len(df_calendario_sinais):,} linhas x {df_calendario_sinais.shape[1]:,} colunas")
print(f"Perfil compras ticker da 12.3      : {len(df_perfil_compras_ticker):,} linhas x {df_perfil_compras_ticker.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

def escolher_coluna(df, candidatos, obrigatoria=False, nome_logico="coluna"):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar {nome_logico}. "
            f"Candidatas avaliadas: {candidatos}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

def escolher_coluna_por_termos(df, termos_obrigatorios, termos_exclusao=None, obrigatoria=False, nome_logico="coluna"):
    termos_exclusao = termos_exclusao or []
    for coluna in df.columns:
        nome = str(coluna).lower()
        if all(termo in nome for termo in termos_obrigatorios) and not any(termo in nome for termo in termos_exclusao):
            return coluna
    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar {nome_logico} pelos termos {termos_obrigatorios}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

def para_datetime_seguro(serie):
    return pd.to_datetime(serie, errors="coerce").dt.normalize()

def para_numero_seguro(serie):
    return pd.to_numeric(serie, errors="coerce")

def para_booleano_seguro(serie):
    if serie.dtype == bool:
        return serie.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(serie):
        return serie.fillna(0).astype(float).ne(0)
    serie_texto = serie.astype(str).str.strip().str.lower()
    valores_verdadeiros = {"1", "true", "t", "sim", "s", "yes", "y", "ok", "apto", "apta", "elegivel", "elegível", "aprovado", "aprovada"}
    return serie_texto.isin(valores_verdadeiros)

def construir_chave_linha(df, colunas):
    if not colunas:
        return pd.Series(np.arange(len(df)).astype(str), index=df.index)
    chave = pd.Series("", index=df.index, dtype="object")
    for coluna in colunas:
        if coluna in df.columns:
            chave = chave + "|" + df[coluna].astype(str).fillna("NA")
    return chave.str.strip("|")

def flag_numerica(df, coluna, operador, limiar, ausente_neutro=True):
    if coluna is None or coluna not in df.columns:
        valor_padrao = True if ausente_neutro else False
        return pd.Series(valor_padrao, index=df.index, dtype="bool")
    serie = para_numero_seguro(df[coluna])
    if operador == ">":
        return serie.gt(limiar).fillna(False)
    if operador == ">=":
        return serie.ge(limiar).fillna(False)
    if operador == "<":
        return serie.lt(limiar).fillna(False)
    if operador == "<=":
        return serie.le(limiar).fillna(False)
    raise ValueError(f"Operador não suportado: {operador}")

def localizar_flag_referencia_contabil(df):
    candidatos_exatos = [
        "elegivel_contabil",
        "flag_elegivel_contabil",
        "apto_contabil",
        "flag_apto_contabil",
        "apta_contabil",
        "flag_apta_contabil",
        "aprovado_filtro_contabil",
        "flag_aprovado_filtro_contabil",
        "passou_filtro_contabil",
        "flag_passou_filtro_contabil",
        "elegivel_filtro_contabil",
        "flag_elegivel_filtro_contabil",
    ]
    coluna = escolher_coluna(df, candidatos_exatos, obrigatoria=False)
    if coluna is not None:
        return coluna
    termos_aceitos = [
        ["eleg", "contabil"],
        ["eleg", "contábil"],
        ["apto", "contabil"],
        ["apta", "contabil"],
        ["aprov", "contabil"],
        ["pass", "contabil"],
    ]
    for termos in termos_aceitos:
        coluna = escolher_coluna_por_termos(
            df,
            termos_obrigatorios=termos,
            termos_exclusao=["motivo", "descricao", "texto", "nome"],
            obrigatoria=False,
            nome_logico="flag de referência contábil"
        )
        if coluna is not None:
            return coluna
    return None

def agregar_flags_por_data_issuer(df_flags, coluna_data, coluna_issuer, colunas_flags):
    if df_flags.empty:
        return pd.DataFrame(columns=[coluna_data, coluna_issuer] + colunas_flags)
    agregacoes = {coluna: "max" for coluna in colunas_flags if coluna in df_flags.columns}
    return (
        df_flags.groupby([coluna_data, coluna_issuer], as_index=False)
        .agg(agregacoes)
    )

def construir_lookup_cenarios_por_datas(df_datas, coluna_data_base, coluna_issuer_base, df_cenarios, coluna_issuer_cenario, coluna_inicio, coluna_fim, colunas_flags):
    datas_validas = sorted(df_datas[coluna_data_base].dropna().unique())
    partes = []
    for data_ref in datas_validas:
        base_data = df_datas.loc[df_datas[coluna_data_base].eq(data_ref), [coluna_data_base, coluna_issuer_base]].drop_duplicates().copy()
        if base_data.empty:
            continue
        ativos_data = df_cenarios.loc[
            df_cenarios[coluna_inicio].le(data_ref) & df_cenarios[coluna_fim].ge(data_ref),
            [coluna_issuer_cenario] + colunas_flags
        ].copy()
        if ativos_data.empty:
            for coluna_flag in colunas_flags:
                base_data[coluna_flag] = False
            partes.append(base_data)
            continue
        ativos_data = (
            ativos_data.groupby(coluna_issuer_cenario, as_index=False)[colunas_flags]
            .max()
        )
        base_data = base_data.merge(
            ativos_data,
            left_on=coluna_issuer_base,
            right_on=coluna_issuer_cenario,
            how="left"
        )
        if coluna_issuer_cenario != coluna_issuer_base and coluna_issuer_cenario in base_data.columns:
            base_data = base_data.drop(columns=[coluna_issuer_cenario])
        for coluna_flag in colunas_flags:
            base_data[coluna_flag] = base_data[coluna_flag].fillna(False).astype(bool)
        partes.append(base_data)
    if not partes:
        return pd.DataFrame(columns=[coluna_data_base, coluna_issuer_base] + colunas_flags)
    return pd.concat(partes, ignore_index=True)

def preparar_auditoria(linhas):
    df_aud = pd.DataFrame(linhas)
    for coluna in ["item", "valor", "valor_referencia", "status", "observacao"]:
        if coluna not in df_aud.columns:
            df_aud[coluna] = ""
        df_aud[coluna] = df_aud[coluna].astype(str)
    return df_aud

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Validação e padronização das colunas centrais
# ============================================================

print("\n[5/12] Validação e padronização das colunas centrais...")

COL_ISSUER_CONTABIL = escolher_coluna(
    df_elegibilidade_contabil,
    ["issuer_code", "codigo_emissor", "emissor", "empresa", "company_id"],
    obrigatoria=True,
    nome_logico="issuer_code na base contábil da 4.4"
)
COL_DATA_INICIO_CONTABIL = escolher_coluna(
    df_elegibilidade_contabil,
    [
        "data_inicio_vigencia_efetiva_projeto",
        "data_inicio_vigencia_operacional",
        "data_inicio_vigencia_contabil",
        "data_inicio_vigencia",
        "inicio_vigencia_contabil",
        "data_inicio_janela_contabil",
        "data_inicio_janela",
        "data_inicio_operacional",
        "vigencia_inicio",
        "data_inicio",
    ],
    obrigatoria=True,
    nome_logico="data inicial da vigência contábil"
)
COL_DATA_FIM_CONTABIL = escolher_coluna(
    df_elegibilidade_contabil,
    [
        "data_fim_vigencia_efetiva_projeto",
        "data_fim_vigencia_operacional_ajustada",
        "data_fim_vigencia_operacional",
        "data_fim_vigencia_limite_tecnico",
        "data_fim_vigencia_contabil",
        "data_fim_vigencia",
        "fim_vigencia_contabil",
        "data_fim_janela_contabil",
        "data_fim_janela",
        "data_fim_operacional",
        "vigencia_fim",
        "data_fim",
    ],
    obrigatoria=True,
    nome_logico="data final da vigência contábil"
)

COL_ISSUER_TICKERS_APTOS = escolher_coluna(
    df_tickers_aptos,
    ["issuer_code", "codigo_emissor", "emissor", "empresa", "company_id"],
    obrigatoria=True,
    nome_logico="issuer_code na base de tickers aptos da 4.5"
)
COL_TICKER_TICKERS_APTOS = escolher_coluna(
    df_tickers_aptos,
    ["ticker", "codigo_negociacao", "ativo"],
    obrigatoria=False,
    nome_logico="ticker na base de tickers aptos da 4.5"
)
COL_DATA_INICIO_TICKERS_APTOS = escolher_coluna(
    df_tickers_aptos,
    [
        "data_inicio_vigencia_efetiva_projeto",
        "data_inicio_vigencia_operacional",
        "data_inicio_vigencia_contabil",
        "data_inicio_vigencia",
        "inicio_vigencia_contabil",
        "data_inicio_janela_contabil",
        "data_inicio_janela",
        "data_inicio_operacional",
        "vigencia_inicio",
        "data_inicio",
    ],
    obrigatoria=False,
    nome_logico="data inicial da vigência na base 4.5"
)
COL_DATA_FIM_TICKERS_APTOS = escolher_coluna(
    df_tickers_aptos,
    [
        "data_fim_vigencia_efetiva_projeto",
        "data_fim_vigencia_operacional_ajustada",
        "data_fim_vigencia_operacional",
        "data_fim_vigencia_limite_tecnico",
        "data_fim_vigencia_contabil",
        "data_fim_vigencia",
        "fim_vigencia_contabil",
        "data_fim_janela_contabil",
        "data_fim_janela",
        "data_fim_operacional",
        "vigencia_fim",
        "data_fim",
    ],
    obrigatoria=False,
    nome_logico="data final da vigência na base 4.5"
)

COL_DATA_OPERACIONAL = escolher_coluna(
    df_elegibilidade_operacional,
    ["data", "data_pregao", "dt_pregao"],
    obrigatoria=True,
    nome_logico="data na elegibilidade operacional da 3.4"
)
COL_ISSUER_OPERACIONAL = escolher_coluna(
    df_elegibilidade_operacional,
    ["issuer_code", "codigo_emissor", "emissor", "empresa", "company_id"],
    obrigatoria=True,
    nome_logico="issuer_code na elegibilidade operacional da 3.4"
)
COL_TICKER_OPERACIONAL = escolher_coluna(
    df_elegibilidade_operacional,
    ["ticker", "codigo_negociacao", "ativo"],
    obrigatoria=False,
    nome_logico="ticker na elegibilidade operacional da 3.4"
)

COL_DATA_SINAL = escolher_coluna(
    df_calendario_sinais,
    ["data", "data_sinal", "data_aporte", "data_referencia"],
    obrigatoria=True,
    nome_logico="data no calendário de sinais da 6.4"
)
COL_ESTRATEGIA_SINAL = escolher_coluna(
    df_calendario_sinais,
    ["estrategia_referencia", "estrategia", "tipo_sinal"],
    obrigatoria=True,
    nome_logico="estratégia no calendário de sinais da 6.4"
)

COL_DATA_COMPRA = escolher_coluna(
    df_perfil_compras_ticker,
    ["data_aporte", "data_compra", "data", "data_sinal"],
    obrigatoria=True,
    nome_logico="data de compra/aporte na base 12.3"
)
COL_ISSUER_COMPRA = escolher_coluna(
    df_perfil_compras_ticker,
    ["issuer_code", "codigo_emissor", "emissor", "empresa", "company_id"],
    obrigatoria=True,
    nome_logico="issuer_code na base 12.3"
)
COL_TICKER_COMPRA = escolher_coluna(
    df_perfil_compras_ticker,
    ["ticker", "codigo_negociacao", "ativo"],
    obrigatoria=False,
    nome_logico="ticker na base 12.3"
)
COL_ESTRATEGIA_COMPRA = escolher_coluna(
    df_perfil_compras_ticker,
    ["estrategia_referencia", "estrategia", "tipo_sinal"],
    obrigatoria=True,
    nome_logico="estratégia de referência na base 12.3"
)
COL_NOME_EXIBICAO_COMPRA = escolher_coluna(
    df_perfil_compras_ticker,
    ["nome_exibicao_estrategia", "nome_estrategia", "estrategia_nome"],
    obrigatoria=False,
    nome_logico="nome de exibição da estratégia na base 12.3"
)
COL_VALOR_COMPRA = escolher_coluna(
    df_perfil_compras_ticker,
    ["valor_investido_ticker", "valor_investido", "valor_compra", "valor_alocado", "valor_aporte_ticker"],
    obrigatoria=False,
    nome_logico="valor investido por compra na base 12.3"
)

COL_FLAG_REFERENCIA_CONTABIL = localizar_flag_referencia_contabil(df_elegibilidade_contabil)

COL_LUCRO = escolher_coluna(df_elegibilidade_contabil, ["lucro_liquido", "lucro_liquido_controlador", "lucro_controlador"], obrigatoria=False)
COL_PATRIMONIO = escolher_coluna(df_elegibilidade_contabil, ["patrimonio_liquido", "patrimonio_liquido_final", "pl"], obrigatoria=False)
COL_RECEITA = escolher_coluna(df_elegibilidade_contabil, ["receita_liquida", "receita_liquida_total", "receita"], obrigatoria=False)
COL_EBIT = escolher_coluna(df_elegibilidade_contabil, ["ebit", "resultado_operacional"], obrigatoria=False)
COL_LIQ_CORRENTE = escolher_coluna(df_elegibilidade_contabil, ["liquidez_corrente", "liq_corrente"], obrigatoria=False)
COL_LIQ_GERAL = escolher_coluna(df_elegibilidade_contabil, ["liquidez_geral", "liq_geral"], obrigatoria=False)
COL_LIQ_IMEDIATA = escolher_coluna(df_elegibilidade_contabil, ["liquidez_imediata", "liq_imediata"], obrigatoria=False)

for coluna_data in [COL_DATA_INICIO_CONTABIL, COL_DATA_FIM_CONTABIL]:
    df_elegibilidade_contabil[coluna_data] = para_datetime_seguro(df_elegibilidade_contabil[coluna_data])

if COL_DATA_INICIO_TICKERS_APTOS is not None:
    df_tickers_aptos[COL_DATA_INICIO_TICKERS_APTOS] = para_datetime_seguro(df_tickers_aptos[COL_DATA_INICIO_TICKERS_APTOS])
if COL_DATA_FIM_TICKERS_APTOS is not None:
    df_tickers_aptos[COL_DATA_FIM_TICKERS_APTOS] = para_datetime_seguro(df_tickers_aptos[COL_DATA_FIM_TICKERS_APTOS])

df_elegibilidade_operacional[COL_DATA_OPERACIONAL] = para_datetime_seguro(df_elegibilidade_operacional[COL_DATA_OPERACIONAL])
df_calendario_sinais[COL_DATA_SINAL] = para_datetime_seguro(df_calendario_sinais[COL_DATA_SINAL])
df_perfil_compras_ticker[COL_DATA_COMPRA] = para_datetime_seguro(df_perfil_compras_ticker[COL_DATA_COMPRA])

METRICAS_CONTABEIS = {
    "lucro_liquido": COL_LUCRO,
    "patrimonio_liquido": COL_PATRIMONIO,
    "receita_liquida": COL_RECEITA,
    "ebit": COL_EBIT,
    "liquidez_corrente": COL_LIQ_CORRENTE,
    "liquidez_geral": COL_LIQ_GERAL,
    "liquidez_imediata": COL_LIQ_IMEDIATA,
}
METRICAS_IDENTIFICADAS = {nome: coluna for nome, coluna in METRICAS_CONTABEIS.items() if coluna is not None}

print(f"Issuer contábil                         : {COL_ISSUER_CONTABIL}")
print(f"Início vigência contábil                : {COL_DATA_INICIO_CONTABIL}")
print(f"Fim vigência contábil                   : {COL_DATA_FIM_CONTABIL}")
print(f"Início vigência tickers aptos           : {COL_DATA_INICIO_TICKERS_APTOS}")
print(f"Fim vigência tickers aptos              : {COL_DATA_FIM_TICKERS_APTOS}")
print(f"Flag referência contábil                : {COL_FLAG_REFERENCIA_CONTABIL}")
print(f"Issuer compras                          : {COL_ISSUER_COMPRA}")
print(f"Data compras                            : {COL_DATA_COMPRA}")
print(f"Valor por compra                        : {COL_VALOR_COMPRA}")
print(f"Data sinais                             : {COL_DATA_SINAL}")
print(f"Métricas contábeis identificadas         : {len(METRICAS_IDENTIFICADAS)}")
for nome_metrica, coluna_metrica in METRICAS_IDENTIFICADAS.items():
    print(f"- {nome_metrica:<24}: {coluna_metrica}")
print("OK")

# ============================================================
# 6) Construção da base contábil de cenários por empresa e janela
# ============================================================

print("\n[6/12] Construção da base contábil de cenários por empresa e janela...")

colunas_base_contabil = [COL_ISSUER_CONTABIL, COL_DATA_INICIO_CONTABIL, COL_DATA_FIM_CONTABIL]
for coluna in METRICAS_IDENTIFICADAS.values():
    if coluna not in colunas_base_contabil:
        colunas_base_contabil.append(coluna)
if COL_FLAG_REFERENCIA_CONTABIL is not None and COL_FLAG_REFERENCIA_CONTABIL not in colunas_base_contabil:
    colunas_base_contabil.append(COL_FLAG_REFERENCIA_CONTABIL)

colunas_auxiliares_contexto = [
    "ticker", "nome", "setor", "subsetor", "segmento", "periodo", "ano", "trimestre",
    "tipo_periodo", "data_referencia_contabil"
]
for coluna in colunas_auxiliares_contexto:
    if coluna in df_elegibilidade_contabil.columns and coluna not in colunas_base_contabil:
        colunas_base_contabil.append(coluna)

df_cenarios_contabil = df_elegibilidade_contabil[colunas_base_contabil].copy()
df_cenarios_contabil = df_cenarios_contabil.dropna(subset=[COL_ISSUER_CONTABIL, COL_DATA_INICIO_CONTABIL, COL_DATA_FIM_CONTABIL]).copy()
df_cenarios_contabil[COL_ISSUER_CONTABIL] = df_cenarios_contabil[COL_ISSUER_CONTABIL].astype(str).str.strip()
df_cenarios_contabil = df_cenarios_contabil.loc[df_cenarios_contabil[COL_ISSUER_CONTABIL].ne("")].copy()
df_cenarios_contabil = df_cenarios_contabil.loc[df_cenarios_contabil[COL_DATA_INICIO_CONTABIL].le(df_cenarios_contabil[COL_DATA_FIM_CONTABIL])].copy()

# Referência oficial: prioriza flag da 4.4; se ausente, usa membership da base 4.5.
if COL_FLAG_REFERENCIA_CONTABIL is not None:
    df_cenarios_contabil["flag_referencia_oficial_contabil"] = para_booleano_seguro(df_cenarios_contabil[COL_FLAG_REFERENCIA_CONTABIL])
    ORIGEM_REFERENCIA_CONTABIL = f"flag_4_4:{COL_FLAG_REFERENCIA_CONTABIL}"
else:
    chaves_contabil = [COL_ISSUER_CONTABIL, COL_DATA_INICIO_CONTABIL, COL_DATA_FIM_CONTABIL]
    df_cenarios_contabil["chave_empresa_janela"] = construir_chave_linha(df_cenarios_contabil, chaves_contabil)
    if COL_DATA_INICIO_TICKERS_APTOS is not None and COL_DATA_FIM_TICKERS_APTOS is not None:
        df_tickers_aptos_tmp = df_tickers_aptos[[COL_ISSUER_TICKERS_APTOS, COL_DATA_INICIO_TICKERS_APTOS, COL_DATA_FIM_TICKERS_APTOS]].drop_duplicates().copy()
        df_tickers_aptos_tmp = df_tickers_aptos_tmp.rename(columns={
            COL_ISSUER_TICKERS_APTOS: COL_ISSUER_CONTABIL,
            COL_DATA_INICIO_TICKERS_APTOS: COL_DATA_INICIO_CONTABIL,
            COL_DATA_FIM_TICKERS_APTOS: COL_DATA_FIM_CONTABIL,
        })
        df_tickers_aptos_tmp["chave_empresa_janela"] = construir_chave_linha(df_tickers_aptos_tmp, chaves_contabil)
        chaves_aptas = set(df_tickers_aptos_tmp["chave_empresa_janela"].dropna().unique())
        df_cenarios_contabil["flag_referencia_oficial_contabil"] = df_cenarios_contabil["chave_empresa_janela"].isin(chaves_aptas)
        ORIGEM_REFERENCIA_CONTABIL = "membership_4_5:issuer_janela"
    else:
        df_cenarios_contabil["flag_referencia_oficial_contabil"] = False
        ORIGEM_REFERENCIA_CONTABIL = "indisponivel"

# Flags fundamentais de indicadores contábeis.
df_cenarios_contabil["flag_lucro_liquido_positivo_sens"] = flag_numerica(df_cenarios_contabil, COL_LUCRO, ">", 0, ausente_neutro=False)
df_cenarios_contabil["flag_patrimonio_liquido_positivo_sens"] = flag_numerica(df_cenarios_contabil, COL_PATRIMONIO, ">", 0, ausente_neutro=True)
df_cenarios_contabil["flag_receita_liquida_positiva_sens"] = flag_numerica(df_cenarios_contabil, COL_RECEITA, ">", 0, ausente_neutro=True)
df_cenarios_contabil["flag_ebit_positivo_sens"] = flag_numerica(df_cenarios_contabil, COL_EBIT, ">", 0, ausente_neutro=False)
df_cenarios_contabil["flag_liquidez_corrente_min_1_sens"] = flag_numerica(df_cenarios_contabil, COL_LIQ_CORRENTE, ">=", 1.0, ausente_neutro=True)
df_cenarios_contabil["flag_liquidez_corrente_min_1_2_sens"] = flag_numerica(df_cenarios_contabil, COL_LIQ_CORRENTE, ">=", 1.2, ausente_neutro=True)
df_cenarios_contabil["flag_liquidez_geral_min_1_sens"] = flag_numerica(df_cenarios_contabil, COL_LIQ_GERAL, ">=", 1.0, ausente_neutro=True)

flag_resultado_operacional_ou_lucro = (
    df_cenarios_contabil["flag_lucro_liquido_positivo_sens"] |
    df_cenarios_contabil["flag_ebit_positivo_sens"]
)

# Cenários de sensibilidade.
# Os cenários flexíveis ampliam a referência oficial.
# Os cenários rigorosos são definidos como subconjuntos da referência oficial,
# preservando a interpretação ordinal da sensibilidade.
df_cenarios_contabil["flag_contabil_muito_flexivel"] = (
    df_cenarios_contabil["flag_patrimonio_liquido_positivo_sens"] &
    (
        df_cenarios_contabil["flag_receita_liquida_positiva_sens"] |
        flag_resultado_operacional_ou_lucro
    )
)
df_cenarios_contabil["flag_contabil_flexivel"] = (
    df_cenarios_contabil["flag_patrimonio_liquido_positivo_sens"] &
    df_cenarios_contabil["flag_receita_liquida_positiva_sens"] &
    flag_resultado_operacional_ou_lucro
)
df_cenarios_contabil["flag_contabil_referencia_oficial"] = df_cenarios_contabil["flag_referencia_oficial_contabil"].astype(bool)

flag_criterio_rigoroso_adicional = (
    df_cenarios_contabil["flag_patrimonio_liquido_positivo_sens"] &
    df_cenarios_contabil["flag_receita_liquida_positiva_sens"] &
    df_cenarios_contabil["flag_lucro_liquido_positivo_sens"] &
    df_cenarios_contabil["flag_ebit_positivo_sens"] &
    df_cenarios_contabil["flag_liquidez_corrente_min_1_sens"]
)
flag_criterio_muito_rigoroso_adicional = (
    flag_criterio_rigoroso_adicional &
    df_cenarios_contabil["flag_liquidez_corrente_min_1_2_sens"] &
    df_cenarios_contabil["flag_liquidez_geral_min_1_sens"]
)

df_cenarios_contabil["flag_contabil_rigoroso"] = (
    df_cenarios_contabil["flag_contabil_referencia_oficial"] &
    flag_criterio_rigoroso_adicional
)
df_cenarios_contabil["flag_contabil_muito_rigoroso"] = (
    df_cenarios_contabil["flag_contabil_referencia_oficial"] &
    flag_criterio_muito_rigoroso_adicional
)

COLUNAS_FLAGS_CENARIOS = [
    "flag_contabil_muito_flexivel",
    "flag_contabil_flexivel",
    "flag_contabil_referencia_oficial",
    "flag_contabil_rigoroso",
    "flag_contabil_muito_rigoroso",
]

CENARIOS_CONTABEIS = [
    {"cenario_filtro_contabil": "muito_flexivel", "ordem_cenario": 1, "flag_coluna": "flag_contabil_muito_flexivel", "descricao_cenario": "Filtro contábil 50% mais flexível em espírito: exige patrimônio positivo e alguma evidência de operação/resultado."},
    {"cenario_filtro_contabil": "flexivel", "ordem_cenario": 2, "flag_coluna": "flag_contabil_flexivel", "descricao_cenario": "Filtro contábil flexível: exige patrimônio positivo, receita positiva e lucro líquido ou EBIT positivo."},
    {"cenario_filtro_contabil": "referencia_oficial", "ordem_cenario": 3, "flag_coluna": "flag_contabil_referencia_oficial", "descricao_cenario": "Filtro contábil oficial validado nas etapas 4.3 a 4.5."},
    {"cenario_filtro_contabil": "rigoroso", "ordem_cenario": 4, "flag_coluna": "flag_contabil_rigoroso", "descricao_cenario": "Filtro contábil rigoroso: subconjunto da referência oficial com lucro líquido, EBIT e liquidez corrente mínima positivos quando disponíveis."},
    {"cenario_filtro_contabil": "muito_rigoroso", "ordem_cenario": 5, "flag_coluna": "flag_contabil_muito_rigoroso", "descricao_cenario": "Filtro contábil muito rigoroso: subconjunto da referência oficial com liquidez corrente mais alta e liquidez geral mínima quando disponíveis."},
]

df_parametros_cenarios = pd.DataFrame(CENARIOS_CONTABEIS)
df_parametros_cenarios["origem_referencia_contabil"] = ORIGEM_REFERENCIA_CONTABIL
df_parametros_cenarios["metricas_contabeis_identificadas"] = ", ".join([f"{k}:{v}" for k, v in METRICAS_IDENTIFICADAS.items()])

print(f"Base contábil de cenários            : {len(df_cenarios_contabil):,} linhas")
print(f"Empresas distintas                   : {df_cenarios_contabil[COL_ISSUER_CONTABIL].nunique():,}")
print(f"Janelas distintas                    : {df_cenarios_contabil[[COL_DATA_INICIO_CONTABIL, COL_DATA_FIM_CONTABIL]].drop_duplicates().shape[0]:,}")
print(f"Origem da referência contábil         : {ORIGEM_REFERENCIA_CONTABIL}")
print(f"Cenários contábeis definidos          : {len(CENARIOS_CONTABEIS):,}")
print("OK")

# ============================================================
# 7) Resumo do universo contábil por cenário
# ============================================================

print("\n[7/12] Resumo do universo contábil por cenário...")

linhas_resumo_universo = []
for cenario in CENARIOS_CONTABEIS:
    flag_coluna = cenario["flag_coluna"]
    df_filtrado = df_cenarios_contabil.loc[df_cenarios_contabil[flag_coluna]].copy()
    linhas_resumo_universo.append({
        "cenario_filtro_contabil": cenario["cenario_filtro_contabil"],
        "ordem_cenario": cenario["ordem_cenario"],
        "n_linhas_empresa_janela": len(df_cenarios_contabil),
        "n_linhas_elegiveis": len(df_filtrado),
        "pct_linhas_elegiveis": len(df_filtrado) / len(df_cenarios_contabil) if len(df_cenarios_contabil) > 0 else np.nan,
        "n_empresas_distintas_elegiveis": df_filtrado[COL_ISSUER_CONTABIL].nunique(),
        "n_janelas_distintas_elegiveis": df_filtrado[[COL_DATA_INICIO_CONTABIL, COL_DATA_FIM_CONTABIL]].drop_duplicates().shape[0] if len(df_filtrado) > 0 else 0,
        "primeira_vigencia_elegivel": df_filtrado[COL_DATA_INICIO_CONTABIL].min() if len(df_filtrado) > 0 else pd.NaT,
        "ultima_vigencia_elegivel": df_filtrado[COL_DATA_FIM_CONTABIL].max() if len(df_filtrado) > 0 else pd.NaT,
    })

df_resumo_universo = pd.DataFrame(linhas_resumo_universo)

chave_referencia = "flag_contabil_referencia_oficial"
linhas_comparacao = []
for cenario in CENARIOS_CONTABEIS:
    flag_coluna = cenario["flag_coluna"]
    atual = df_cenarios_contabil[flag_coluna].astype(bool)
    referencia = df_cenarios_contabil[chave_referencia].astype(bool)
    n_atual = int(atual.sum())
    n_referencia = int(referencia.sum())
    n_comuns = int((atual & referencia).sum())
    n_adicionados = int((atual & ~referencia).sum())
    n_removidos = int((~atual & referencia).sum())
    denominador_jaccard = int((atual | referencia).sum())
    linhas_comparacao.append({
        "cenario_filtro_contabil": cenario["cenario_filtro_contabil"],
        "ordem_cenario": cenario["ordem_cenario"],
        "n_elegiveis_cenario": n_atual,
        "n_elegiveis_referencia": n_referencia,
        "n_comuns_com_referencia": n_comuns,
        "n_adicionados_vs_referencia": n_adicionados,
        "n_removidos_vs_referencia": n_removidos,
        "delta_liquido_elegiveis_vs_referencia": n_atual - n_referencia,
        "pct_jaccard_vs_referencia": n_comuns / denominador_jaccard if denominador_jaccard > 0 else np.nan,
    })

df_comparacao_referencia = pd.DataFrame(linhas_comparacao)

print(f"Resumo universo contábil             : {len(df_resumo_universo):,} linhas")
print(f"Comparação com referência             : {len(df_comparacao_referencia):,} linhas")
print("OK")

# ============================================================
# 8) Impacto dos cenários nas datas de sinais
# ============================================================

print("\n[8/12] Impacto dos cenários nas datas de sinais...")

df_sinais_base = df_calendario_sinais[[COL_DATA_SINAL, COL_ESTRATEGIA_SINAL]].dropna(subset=[COL_DATA_SINAL]).drop_duplicates().copy()
df_sinais_base[COL_DATA_SINAL] = para_datetime_seguro(df_sinais_base[COL_DATA_SINAL])

linhas_impacto_sinais = []
for _, linha_sinal in df_sinais_base.iterrows():
    data_sinal = linha_sinal[COL_DATA_SINAL]
    estrategia = linha_sinal[COL_ESTRATEGIA_SINAL]
    df_janelas_ativas = df_cenarios_contabil.loc[
        df_cenarios_contabil[COL_DATA_INICIO_CONTABIL].le(data_sinal) &
        df_cenarios_contabil[COL_DATA_FIM_CONTABIL].ge(data_sinal)
    ].copy()
    for cenario in CENARIOS_CONTABEIS:
        flag_coluna = cenario["flag_coluna"]
        df_filtrado = df_janelas_ativas.loc[df_janelas_ativas[flag_coluna]].copy()
        linhas_impacto_sinais.append({
            "data_sinal": data_sinal,
            "estrategia_referencia": estrategia,
            "cenario_filtro_contabil": cenario["cenario_filtro_contabil"],
            "ordem_cenario": cenario["ordem_cenario"],
            "n_empresas_contabeis_elegiveis_data_sinal": df_filtrado[COL_ISSUER_CONTABIL].nunique(),
            "n_janelas_contabeis_ativas_data_sinal": len(df_janelas_ativas),
            "sinal_sem_cobertura_contabil": int(len(df_janelas_ativas) == 0),
        })

df_impacto_sinais_detalhe = pd.DataFrame(linhas_impacto_sinais)
if df_impacto_sinais_detalhe.empty:
    df_impacto_sinais = pd.DataFrame(columns=[
        "estrategia_referencia", "cenario_filtro_contabil", "ordem_cenario", "n_sinais",
        "n_sinais_sem_cobertura_contabil", "media_empresas_elegiveis_datas_sinal",
        "mediana_empresas_elegiveis_datas_sinal", "minimo_empresas_elegiveis_datas_sinal",
        "maximo_empresas_elegiveis_datas_sinal"
    ])
else:
    df_impacto_sinais = (
        df_impacto_sinais_detalhe
        .groupby(["estrategia_referencia", "cenario_filtro_contabil", "ordem_cenario"], as_index=False)
        .agg(
            n_sinais=("data_sinal", "nunique"),
            n_sinais_sem_cobertura_contabil=("sinal_sem_cobertura_contabil", "sum"),
            media_empresas_elegiveis_datas_sinal=("n_empresas_contabeis_elegiveis_data_sinal", "mean"),
            mediana_empresas_elegiveis_datas_sinal=("n_empresas_contabeis_elegiveis_data_sinal", "median"),
            minimo_empresas_elegiveis_datas_sinal=("n_empresas_contabeis_elegiveis_data_sinal", "min"),
            maximo_empresas_elegiveis_datas_sinal=("n_empresas_contabeis_elegiveis_data_sinal", "max"),
        )
        .sort_values(["estrategia_referencia", "ordem_cenario"])
        .reset_index(drop=True)
    )

print(f"Impacto sobre sinais                  : {len(df_impacto_sinais):,} linhas")
print("OK")

# ============================================================
# 9) Impacto dos cenários nas cestas de compras
# ============================================================

print("\n[9/12] Impacto dos cenários nas cestas de compras...")

df_compras_base = df_perfil_compras_ticker.copy()
df_compras_base[COL_DATA_COMPRA] = para_datetime_seguro(df_compras_base[COL_DATA_COMPRA])
df_compras_base[COL_ISSUER_COMPRA] = df_compras_base[COL_ISSUER_COMPRA].astype(str).str.strip()
if COL_VALOR_COMPRA is not None:
    df_compras_base[COL_VALOR_COMPRA] = para_numero_seguro(df_compras_base[COL_VALOR_COMPRA]).fillna(0)
else:
    COL_VALOR_COMPRA = "valor_investido_ticker_estimado_unitario"
    df_compras_base[COL_VALOR_COMPRA] = 1.0

colunas_lookup_base = [COL_DATA_COMPRA, COL_ISSUER_COMPRA]
df_datas_issuers_compras = df_compras_base[colunas_lookup_base].dropna(subset=[COL_DATA_COMPRA, COL_ISSUER_COMPRA]).drop_duplicates().copy()

df_lookup_compras = construir_lookup_cenarios_por_datas(
    df_datas=df_datas_issuers_compras,
    coluna_data_base=COL_DATA_COMPRA,
    coluna_issuer_base=COL_ISSUER_COMPRA,
    df_cenarios=df_cenarios_contabil,
    coluna_issuer_cenario=COL_ISSUER_CONTABIL,
    coluna_inicio=COL_DATA_INICIO_CONTABIL,
    coluna_fim=COL_DATA_FIM_CONTABIL,
    colunas_flags=COLUNAS_FLAGS_CENARIOS,
)

df_compras_enriquecida = df_compras_base.merge(
    df_lookup_compras,
    on=[COL_DATA_COMPRA, COL_ISSUER_COMPRA],
    how="left"
)
for coluna_flag in COLUNAS_FLAGS_CENARIOS:
    if coluna_flag not in df_compras_enriquecida.columns:
        df_compras_enriquecida[coluna_flag] = False
    df_compras_enriquecida[coluna_flag] = df_compras_enriquecida[coluna_flag].fillna(False).astype(bool)

linhas_impacto_cestas = []
for cenario in CENARIOS_CONTABEIS:
    flag_coluna = cenario["flag_coluna"]
    df_tmp = df_compras_enriquecida.copy()
    df_tmp["flag_mantido_filtro_contabil"] = df_tmp[flag_coluna].astype(bool)
    df_tmp["valor_mantido_filtro_contabil"] = np.where(
        df_tmp["flag_mantido_filtro_contabil"],
        df_tmp[COL_VALOR_COMPRA],
        0.0
    )
    colunas_grupo = [COL_ESTRATEGIA_COMPRA]
    if COL_NOME_EXIBICAO_COMPRA is not None:
        colunas_grupo.append(COL_NOME_EXIBICAO_COMPRA)
    agrupado = (
        df_tmp.groupby(colunas_grupo, as_index=False)
        .agg(
            n_linhas_compras_total=(COL_ISSUER_COMPRA, "size"),
            n_empresas_compradas_total=(COL_ISSUER_COMPRA, "nunique"),
            n_linhas_mantidas_filtro_contabil=("flag_mantido_filtro_contabil", "sum"),
            valor_investido_total=(COL_VALOR_COMPRA, "sum"),
            valor_investido_mantido_filtro_contabil=("valor_mantido_filtro_contabil", "sum"),
            primeira_data_aporte=(COL_DATA_COMPRA, "min"),
            ultima_data_aporte=(COL_DATA_COMPRA, "max"),
        )
    )
    agrupado["cenario_filtro_contabil"] = cenario["cenario_filtro_contabil"]
    agrupado["ordem_cenario"] = cenario["ordem_cenario"]
    agrupado["pct_linhas_mantidas_filtro_contabil"] = np.where(
        agrupado["n_linhas_compras_total"].gt(0),
        agrupado["n_linhas_mantidas_filtro_contabil"] / agrupado["n_linhas_compras_total"],
        np.nan
    )
    agrupado["pct_valor_mantido_filtro_contabil"] = np.where(
        agrupado["valor_investido_total"].gt(0),
        agrupado["valor_investido_mantido_filtro_contabil"] / agrupado["valor_investido_total"],
        np.nan
    )
    linhas_impacto_cestas.append(agrupado)

if linhas_impacto_cestas:
    df_impacto_cestas = pd.concat(linhas_impacto_cestas, ignore_index=True)
    renomear_grupo = {COL_ESTRATEGIA_COMPRA: "estrategia_referencia"}
    if COL_NOME_EXIBICAO_COMPRA is not None:
        renomear_grupo[COL_NOME_EXIBICAO_COMPRA] = "nome_exibicao_estrategia"
    else:
        df_impacto_cestas["nome_exibicao_estrategia"] = df_impacto_cestas[COL_ESTRATEGIA_COMPRA].astype(str)
    df_impacto_cestas = df_impacto_cestas.rename(columns=renomear_grupo)
    df_impacto_cestas = df_impacto_cestas.sort_values(["estrategia_referencia", "nome_exibicao_estrategia", "ordem_cenario"]).reset_index(drop=True)
else:
    df_impacto_cestas = pd.DataFrame()

print(f"Lookup contábil compras               : {len(df_lookup_compras):,} linhas")
print(f"Impacto sobre cestas de compras        : {len(df_impacto_cestas):,} linhas")
print("OK")

# ============================================================
# 10) Motivos de reprovação e base de gráficos
# ============================================================

print("\n[10/12] Motivos de reprovação e base de gráficos...")

criterios_reprovacao = [
    ("patrimonio_liquido_nao_positivo", "flag_patrimonio_liquido_positivo_sens"),
    ("receita_liquida_nao_positiva", "flag_receita_liquida_positiva_sens"),
    ("lucro_liquido_nao_positivo", "flag_lucro_liquido_positivo_sens"),
    ("ebit_nao_positivo", "flag_ebit_positivo_sens"),
    ("liquidez_corrente_menor_1", "flag_liquidez_corrente_min_1_sens"),
    ("liquidez_corrente_menor_1_2", "flag_liquidez_corrente_min_1_2_sens"),
    ("liquidez_geral_menor_1", "flag_liquidez_geral_min_1_sens"),
]

linhas_motivos = []
for cenario in CENARIOS_CONTABEIS:
    flag_coluna = cenario["flag_coluna"]
    df_reprovados = df_cenarios_contabil.loc[~df_cenarios_contabil[flag_coluna]].copy()
    for motivo, coluna_flag_criterio in criterios_reprovacao:
        if coluna_flag_criterio not in df_reprovados.columns:
            continue
        n_reprovados_motivo = int((~df_reprovados[coluna_flag_criterio].astype(bool)).sum())
        linhas_motivos.append({
            "cenario_filtro_contabil": cenario["cenario_filtro_contabil"],
            "ordem_cenario": cenario["ordem_cenario"],
            "motivo_reprovacao": motivo,
            "n_linhas_reprovadas_motivo": n_reprovados_motivo,
            "n_linhas_reprovadas_cenario": len(df_reprovados),
            "pct_reprovadas_motivo_sobre_reprovadas_cenario": n_reprovados_motivo / len(df_reprovados) if len(df_reprovados) > 0 else np.nan,
        })

df_motivos_reprovacao = pd.DataFrame(linhas_motivos)
if not df_motivos_reprovacao.empty:
    df_motivos_reprovacao = df_motivos_reprovacao.sort_values(
        ["ordem_cenario", "n_linhas_reprovadas_motivo"],
        ascending=[True, False]
    ).reset_index(drop=True)

base_grafico_universo = df_resumo_universo.copy()
base_grafico_universo["tipo_grafico"] = "universo_contabil"
base_grafico_universo["metrica"] = "n_empresas_distintas_elegiveis"
base_grafico_universo["valor"] = base_grafico_universo["n_empresas_distintas_elegiveis"]

base_grafico_compras = df_impacto_cestas.copy()
if not base_grafico_compras.empty:
    base_grafico_compras["tipo_grafico"] = "cestas_compras"
    base_grafico_compras["metrica"] = "pct_valor_mantido_filtro_contabil"
    base_grafico_compras["valor"] = base_grafico_compras["pct_valor_mantido_filtro_contabil"]
    colunas_grafico_compras = [
        "tipo_grafico", "metrica", "valor", "cenario_filtro_contabil", "ordem_cenario",
        "estrategia_referencia", "nome_exibicao_estrategia"
    ]
    base_grafico_compras = base_grafico_compras[colunas_grafico_compras]
else:
    base_grafico_compras = pd.DataFrame(columns=["tipo_grafico", "metrica", "valor", "cenario_filtro_contabil", "ordem_cenario", "estrategia_referencia", "nome_exibicao_estrategia"])

base_grafico_universo = base_grafico_universo[[
    "tipo_grafico", "metrica", "valor", "cenario_filtro_contabil", "ordem_cenario"
]].copy()
base_grafico_universo["estrategia_referencia"] = "universo_contabil"
base_grafico_universo["nome_exibicao_estrategia"] = "Universo Contábil"

df_base_grafico = pd.concat([base_grafico_universo, base_grafico_compras], ignore_index=True)

print(f"Motivos de reprovação                 : {len(df_motivos_reprovacao):,} linhas")
print(f"Base gráfico filtro contábil           : {len(df_base_grafico):,} linhas")
print("OK")

# ============================================================
# 11) Auditoria de validação
# ============================================================

print("\n[11/12] Auditoria de validação...")

cenarios_alternativos = [c for c in CENARIOS_CONTABEIS if c["cenario_filtro_contabil"] != "referencia_oficial"]
cenarios_alternativos_sem_elegiveis = 0
for cenario in cenarios_alternativos:
    n_elegiveis = int(df_cenarios_contabil[cenario["flag_coluna"]].sum())
    if n_elegiveis == 0:
        cenarios_alternativos_sem_elegiveis += 1

n_referencia_elegiveis = int(df_cenarios_contabil["flag_contabil_referencia_oficial"].sum())
n_rigoroso_elegiveis = int(df_cenarios_contabil["flag_contabil_rigoroso"].sum())
n_muito_rigoroso_elegiveis = int(df_cenarios_contabil["flag_contabil_muito_rigoroso"].sum())
cenarios_rigorosos_acima_referencia = int(n_rigoroso_elegiveis > n_referencia_elegiveis) + int(n_muito_rigoroso_elegiveis > n_referencia_elegiveis)
n_sinais_sem_cobertura = int(df_impacto_sinais_detalhe["sinal_sem_cobertura_contabil"].sum()) if not df_impacto_sinais_detalhe.empty else len(df_sinais_base)
compras_sem_lookup_referencia = int((~df_compras_enriquecida["flag_contabil_referencia_oficial"].astype(bool)).sum()) if "flag_contabil_referencia_oficial" in df_compras_enriquecida.columns else len(df_compras_enriquecida)

df_numericos_validacao = pd.concat([
    df_resumo_universo.select_dtypes(include=[np.number]),
    df_comparacao_referencia.select_dtypes(include=[np.number]),
    df_impacto_sinais.select_dtypes(include=[np.number]),
    df_impacto_cestas.select_dtypes(include=[np.number]) if not df_impacto_cestas.empty else pd.DataFrame(),
], axis=0, ignore_index=True)
metricas_infinitas = int(np.isinf(df_numericos_validacao.to_numpy(dtype=float)).sum()) if not df_numericos_validacao.empty else 0

duplicatas_empresa_janela = int(df_cenarios_contabil.duplicated(subset=[COL_ISSUER_CONTABIL, COL_DATA_INICIO_CONTABIL, COL_DATA_FIM_CONTABIL]).sum())

linhas_auditoria = [
    {
        "item": "linhas_elegibilidade_contabil_4_4",
        "valor": len(df_elegibilidade_contabil),
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Quantidade de linhas na base de elegibilidade contábil por janela da 4.4.",
    },
    {
        "item": "linhas_base_cenarios_contabeis",
        "valor": len(df_cenarios_contabil),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_cenarios_contabil) > 0 else "ERRO",
        "observacao": "A base de cenários contábeis deve possuir linhas válidas.",
    },
    {
        "item": "metricas_contabeis_identificadas",
        "valor": len(METRICAS_IDENTIFICADAS),
        "valor_referencia": ">= 3",
        "status": "OK" if len(METRICAS_IDENTIFICADAS) >= 3 else "ERRO",
        "observacao": "A sensibilidade do filtro contábil exige métricas contábeis suficientes para diferenciar cenários.",
    },
    {
        "item": "cenarios_contabeis_gerados",
        "valor": len(CENARIOS_CONTABEIS),
        "valor_referencia": 5,
        "status": "OK" if len(CENARIOS_CONTABEIS) == 5 else "ERRO",
        "observacao": "A subetapa deve gerar cenários flexíveis, referência e rigorosos.",
    },
    {
        "item": "cenarios_alternativos_sem_elegiveis",
        "valor": cenarios_alternativos_sem_elegiveis,
        "valor_referencia": 0,
        "status": "OK" if cenarios_alternativos_sem_elegiveis == 0 else "ERRO",
        "observacao": "Cenários alternativos do filtro contábil não devem ficar vazios por desenho inviável.",
    },
    {
        "item": "cenarios_rigorosos_acima_referencia",
        "valor": cenarios_rigorosos_acima_referencia,
        "valor_referencia": 0,
        "status": "OK" if cenarios_rigorosos_acima_referencia == 0 else "ERRO",
        "observacao": "Cenários rigorosos devem ser subconjuntos da referência oficial para preservar a interpretação ordinal.",
    },
    {
        "item": "elegiveis_referencia_contabil",
        "valor": n_referencia_elegiveis,
        "valor_referencia": "> 0",
        "status": "OK" if n_referencia_elegiveis > 0 else "ERRO",
        "observacao": "A referência oficial do filtro contábil deve possuir empresas elegíveis.",
    },
    {
        "item": "origem_referencia_contabil",
        "valor": ORIGEM_REFERENCIA_CONTABIL,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Origem usada para construir a referência oficial do filtro contábil.",
    },
    {
        "item": "sinais_sem_cobertura_contabil",
        "valor": n_sinais_sem_cobertura,
        "valor_referencia": 0,
        "status": "OK" if n_sinais_sem_cobertura == 0 else "ERRO",
        "observacao": "Datas de sinais deveriam possuir cobertura contábil para cálculo de impacto.",
    },
    {
        "item": "compras_referencia_nao_mantidas_filtro_contabil",
        "valor": compras_sem_lookup_referencia,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Compras não mantidas pela referência contábil; item informativo, pois a cesta também depende de regras operacionais e calendário.",
    },
    {
        "item": "duplicatas_empresa_janela_cenarios",
        "valor": duplicatas_empresa_janela,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Duplicidades por empresa/janela são informativas, pois podem refletir diferentes tickers ou períodos contábeis equivalentes.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": 0,
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
n_erros_bloqueantes = int(df_auditoria["status"].eq("ERRO").sum())

print(f"Itens de auditoria                     : {len(df_auditoria):,}")
print(f"Erros bloqueantes                      : {n_erros_bloqueantes:,}")
print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(df_cenarios_contabil, CAMINHO_BASE_CENARIOS_EMPRESA_JANELA, index=False)
salvar_dataframe(df_parametros_cenarios, CAMINHO_PARAMETROS_CENARIOS, index=False)
salvar_dataframe(df_resumo_universo, CAMINHO_RESUMO_UNIVERSO, index=False)
salvar_dataframe(df_comparacao_referencia, CAMINHO_COMPARACAO_REFERENCIA, index=False)
salvar_dataframe(df_impacto_sinais, CAMINHO_IMPACTO_SINAIS, index=False)
salvar_dataframe(df_impacto_cestas, CAMINHO_IMPACTO_CESTAS, index=False)
salvar_dataframe(df_motivos_reprovacao, CAMINHO_MOTIVOS_REPROVACAO, index=False)
salvar_dataframe(df_base_grafico, CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da sensibilidade do filtro contábil:")
print(df_auditoria.to_string(index=False))

print("\nResumo do universo contábil por cenário:")
print(df_resumo_universo.to_string(index=False))

print("\nComparação com a referência oficial:")
print(df_comparacao_referencia.to_string(index=False))

print("\nImpacto dos cenários nas datas de sinais:")
print(df_impacto_sinais.to_string(index=False))

print("\nImpacto dos cenários nas cestas de compras - amostra:")
print(df_impacto_cestas.head(40).to_string(index=False))

print("\nMotivos de reprovação - amostra:")
print(df_motivos_reprovacao.head(40).to_string(index=False))

print("\nArquivos salvos na subetapa 13.3:")
print(f"- {CAMINHO_BASE_CENARIOS_EMPRESA_JANELA}")
print(f"- {CAMINHO_PARAMETROS_CENARIOS}")
print(f"- {CAMINHO_RESUMO_UNIVERSO}")
print(f"- {CAMINHO_COMPARACAO_REFERENCIA}")
print(f"- {CAMINHO_IMPACTO_SINAIS}")
print(f"- {CAMINHO_IMPACTO_CESTAS}")
print(f"- {CAMINHO_MOTIVOS_REPROVACAO}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_AUDITORIA}")

if n_erros_bloqueantes > 0:
    raise RuntimeError("A subetapa 13.3 encontrou erros bloqueantes na auditoria. Verifique a tabela de validação.")

print("\nETAPA 13.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 13.3 - SENSIBILIDADE DO FILTRO CONTÁBIL

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - elegibilidade operacional da 3.4   : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_3\3_4_base_elegibilidade_operacional_data.parquet
Entrada - regras filtro contábil da 4.3      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_3_tbl_regras_filtro_contabil.parquet
Entrada - parâmetros filtro contábil da 4.3  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_4\4_3_tbl_parametros_filtro_contabil.parquet
Entrada - elegibilidade contábil da 4.4      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\et

## Etapa 13.4) Sensibilidade dos Limiares de Capitulação

In [67]:
%%time
# ============================================================
# Etapa 13.4) Sensibilidade dos Limiares de Capitulação
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 13.4 - SENSIBILIDADE DOS LIMIARES DE CAPITULAÇÃO")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_5 = Path(DIRETORIOS_PROJETO["etapa_5"])
DIR_ETAPA_6 = Path(DIRETORIOS_PROJETO["etapa_6"])
DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])
DIR_ETAPA_13 = Path(DIRETORIOS_PROJETO["etapa_13"])
DIR_ETAPA_13.mkdir(parents=True, exist_ok=True)

CAMINHO_AMPLITUDE_5_5 = DIR_ETAPA_5 / "5_5_base_indicadores_agregados_amplitude_pregao.parquet"
CAMINHO_PARAMETROS_CAPITULACAO_6_1 = DIR_ETAPA_6 / "6_1_tbl_parametros_capitulacao.parquet"
CAMINHO_LIMIARES_CAPITULACAO_6_1 = DIR_ETAPA_6 / "6_1_tbl_limiares_capitulacao_dinamicos.parquet"
CAMINHO_BASE_BRUTA_CAPITULACAO_6_1 = DIR_ETAPA_6 / "6_1_base_capitulacao_bruta_pregao.parquet"
CAMINHO_DATAS_BRUTAS_CAPITULACAO_6_1 = DIR_ETAPA_6 / "6_1_tbl_datas_capitulacao_bruta.parquet"
CAMINHO_CALENDARIO_FINAL_SINAIS_6_4 = DIR_ETAPA_6 / "6_4_base_calendario_final_sinais.parquet"
CAMINHO_RETORNOS_DIARIOS_11_1 = DIR_ETAPA_11 / "11_1_base_retornos_diarios.parquet"

CAMINHO_BASE_SINAIS = DIR_ETAPA_13 / "13_4_base_sinais_sensibilidade_limiares_capitulacao.parquet"
CAMINHO_PARAMETROS_CENARIOS = DIR_ETAPA_13 / "13_4_tbl_parametros_cenarios_limiares_capitulacao.parquet"
CAMINHO_RESUMO = DIR_ETAPA_13 / "13_4_tbl_resumo_sensibilidade_limiares_capitulacao.parquet"
CAMINHO_COMPARACAO_REFERENCIA = DIR_ETAPA_13 / "13_4_tbl_comparacao_limiar_capitulacao_referencia.parquet"
CAMINHO_IMPACTO_TEMPORAL = DIR_ETAPA_13 / "13_4_tbl_impacto_temporal_limiares_capitulacao.parquet"
CAMINHO_BASE_PROXY = DIR_ETAPA_13 / "13_4_base_performance_proxy_pos_sinal_capitulacao.parquet"
CAMINHO_TBL_PROXY = DIR_ETAPA_13 / "13_4_tbl_performance_proxy_pos_sinal_limiares_capitulacao.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_13 / "13_4_tbl_base_grafico_limiares_capitulacao.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_13 / "13_4_tbl_auditoria_validacao_sensibilidade_limiares_capitulacao.parquet"

CAMINHOS_ENTRADA_OBRIGATORIOS = [
    CAMINHO_AMPLITUDE_5_5,
    CAMINHO_BASE_BRUTA_CAPITULACAO_6_1,
    CAMINHO_DATAS_BRUTAS_CAPITULACAO_6_1,
    CAMINHO_CALENDARIO_FINAL_SINAIS_6_4,
]

for caminho in CAMINHOS_ENTRADA_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - amplitude agregada da 5.5          : {CAMINHO_AMPLITUDE_5_5}")
print(f"Entrada - parâmetros capitulação da 6.1      : {CAMINHO_PARAMETROS_CAPITULACAO_6_1}")
print(f"Entrada - limiares capitulação da 6.1        : {CAMINHO_LIMIARES_CAPITULACAO_6_1}")
print(f"Entrada - base bruta capitulação da 6.1      : {CAMINHO_BASE_BRUTA_CAPITULACAO_6_1}")
print(f"Entrada - datas brutas capitulação da 6.1    : {CAMINHO_DATAS_BRUTAS_CAPITULACAO_6_1}")
print(f"Entrada - calendário final de sinais da 6.4  : {CAMINHO_CALENDARIO_FINAL_SINAIS_6_4}")
print(f"Entrada - retornos diários da 11.1           : {CAMINHO_RETORNOS_DIARIOS_11_1}")
print(f"Saída   - base sinais por limiar             : {CAMINHO_BASE_SINAIS}")
print(f"Saída   - parâmetros dos cenários            : {CAMINHO_PARAMETROS_CENARIOS}")
print(f"Saída   - resumo sensibilidade               : {CAMINHO_RESUMO}")
print(f"Saída   - comparação com referência          : {CAMINHO_COMPARACAO_REFERENCIA}")
print(f"Saída   - impacto temporal                   : {CAMINHO_IMPACTO_TEMPORAL}")
print(f"Saída   - base performance proxy             : {CAMINHO_BASE_PROXY}")
print(f"Saída   - tabela performance proxy           : {CAMINHO_TBL_PROXY}")
print(f"Saída   - base gráfico                       : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação             : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/12] Carga das bases oficiais da subetapa...")

df_amplitude = pd.read_parquet(CAMINHO_AMPLITUDE_5_5)
df_base_bruta_capitulacao = pd.read_parquet(CAMINHO_BASE_BRUTA_CAPITULACAO_6_1)
df_datas_brutas_capitulacao = pd.read_parquet(CAMINHO_DATAS_BRUTAS_CAPITULACAO_6_1)
df_calendario_final_sinais = pd.read_parquet(CAMINHO_CALENDARIO_FINAL_SINAIS_6_4)

if Path(CAMINHO_PARAMETROS_CAPITULACAO_6_1).exists():
    df_parametros_capitulacao = pd.read_parquet(CAMINHO_PARAMETROS_CAPITULACAO_6_1)
else:
    df_parametros_capitulacao = pd.DataFrame()

if Path(CAMINHO_LIMIARES_CAPITULACAO_6_1).exists():
    df_limiares_capitulacao = pd.read_parquet(CAMINHO_LIMIARES_CAPITULACAO_6_1)
else:
    df_limiares_capitulacao = pd.DataFrame()

if Path(CAMINHO_RETORNOS_DIARIOS_11_1).exists():
    df_retornos_diarios = pd.read_parquet(CAMINHO_RETORNOS_DIARIOS_11_1)
    RETORNOS_DIARIOS_DISPONIVEIS = True
else:
    df_retornos_diarios = pd.DataFrame()
    RETORNOS_DIARIOS_DISPONIVEIS = False

print(f"Amplitude agregada da 5.5          : {len(df_amplitude):,} linhas x {df_amplitude.shape[1]:,} colunas")
print(f"Parâmetros capitulação da 6.1      : {len(df_parametros_capitulacao):,} linhas x {df_parametros_capitulacao.shape[1]:,} colunas")
print(f"Limiares capitulação da 6.1        : {len(df_limiares_capitulacao):,} linhas x {df_limiares_capitulacao.shape[1]:,} colunas")
print(f"Base bruta capitulação da 6.1      : {len(df_base_bruta_capitulacao):,} linhas x {df_base_bruta_capitulacao.shape[1]:,} colunas")
print(f"Datas brutas capitulação da 6.1    : {len(df_datas_brutas_capitulacao):,} linhas x {df_datas_brutas_capitulacao.shape[1]:,} colunas")
print(f"Calendário final de sinais da 6.4  : {len(df_calendario_final_sinais):,} linhas x {df_calendario_final_sinais.shape[1]:,} colunas")
print(f"Retornos diários disponíveis       : {RETORNOS_DIARIOS_DISPONIVEIS}")
if RETORNOS_DIARIOS_DISPONIVEIS:
    print(f"Base de retornos diários           : {len(df_retornos_diarios):,} linhas x {df_retornos_diarios.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

def escolher_coluna(df, candidatos, obrigatoria=False, nome_logico="coluna"):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar {nome_logico}. "
            f"Candidatas avaliadas: {candidatos}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

def escolher_coluna_por_termos(df, listas_termos, termos_exclusao=None, obrigatoria=False, nome_logico="coluna"):
    termos_exclusao = termos_exclusao or []
    for termos in listas_termos:
        for coluna in df.columns:
            nome = str(coluna).lower()
            if all(termo in nome for termo in termos) and not any(termo in nome for termo in termos_exclusao):
                return coluna
    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar {nome_logico} pelos termos {listas_termos}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

def para_datetime_seguro(serie):
    return pd.to_datetime(serie, errors="coerce").dt.normalize()

def para_numero_seguro(serie):
    return pd.to_numeric(serie, errors="coerce")

def normalizar_percentual(serie):
    serie_num = para_numero_seguro(serie)
    p95 = serie_num.dropna().quantile(0.95) if serie_num.notna().any() else np.nan
    if pd.notna(p95) and p95 > 1.5:
        serie_num = serie_num / 100.0
    return serie_num.clip(lower=0, upper=1)

def aplicar_cooldown_datas(datas, calendario_pregoes, cooldown_pregoes):
    datas_validas = sorted(pd.Series(datas).dropna().unique())
    mapa_ordem = {data: i for i, data in enumerate(calendario_pregoes)}
    datas_aceitas = []
    ultima_ordem_aceita = None
    for data in datas_validas:
        if data not in mapa_ordem:
            continue
        ordem_data = mapa_ordem[data]
        if ultima_ordem_aceita is None or (ordem_data - ultima_ordem_aceita) >= cooldown_pregoes:
            datas_aceitas.append(data)
            ultima_ordem_aceita = ordem_data
    return datas_aceitas

def calcular_retorno_forward(df_serie_retorno, data_sinal, horizonte_pregoes, coluna_data, coluna_retorno):
    if df_serie_retorno.empty:
        return np.nan
    datas = df_serie_retorno[coluna_data].to_numpy()
    idx = np.searchsorted(datas, np.datetime64(data_sinal), side="left")
    idx_inicio = idx + 1
    idx_fim = idx_inicio + horizonte_pregoes
    if idx_inicio >= len(df_serie_retorno):
        return np.nan
    janela = df_serie_retorno.iloc[idx_inicio:min(idx_fim, len(df_serie_retorno))][coluna_retorno].dropna()
    if len(janela) < max(1, int(horizonte_pregoes * 0.8)):
        return np.nan
    return float((1.0 + janela).prod() - 1.0)

def identificar_serie_benchmark_retornos(df):
    if df.empty:
        return pd.DataFrame(), None, None, "indisponivel"
    col_data = escolher_coluna(df, ["data", "data_retorno", "data_pregao", "dt_pregao"], obrigatoria=False)
    if col_data is None:
        return pd.DataFrame(), None, None, "sem_coluna_data"
    col_retorno = escolher_coluna(
        df,
        ["retorno_diario", "retorno", "retorno_carteira", "retorno_periodo", "retorno_1d"],
        obrigatoria=False
    )
    if col_retorno is None:
        col_retorno = escolher_coluna_por_termos(
            df,
            listas_termos=[["retorno"]],
            termos_exclusao=["acumul", "anual", "mensal", "excesso"],
            obrigatoria=False,
            nome_logico="retorno diário"
        )
    if col_retorno is None:
        return pd.DataFrame(), col_data, None, "sem_coluna_retorno"
    colunas_texto = [col for col in df.columns if df[col].dtype == "object" or str(df[col].dtype).startswith("string")]
    mascara_benchmark = pd.Series(False, index=df.index)
    for col in colunas_texto:
        serie_texto = df[col].astype(str).str.lower()
        mascara_benchmark = mascara_benchmark | serie_texto.str.contains("ibovespa", na=False) | serie_texto.str.contains("ibov", na=False)
    df_bench = df.loc[mascara_benchmark, [col_data, col_retorno] + colunas_texto].copy()
    if df_bench.empty:
        col_wide = escolher_coluna_por_termos(
            df,
            listas_termos=[["ibov"], ["ibovespa"]],
            termos_exclusao=["patrimonio", "valor"],
            obrigatoria=False,
            nome_logico="coluna wide do Ibovespa"
        )
        if col_wide is not None:
            df_bench = df[[col_data, col_wide]].copy().rename(columns={col_wide: col_retorno})
            origem = f"wide:{col_wide}"
        else:
            return pd.DataFrame(), col_data, col_retorno, "benchmark_nao_identificado"
    else:
        origem = "filtro_textual_ibovespa"
    df_bench[col_data] = para_datetime_seguro(df_bench[col_data])
    df_bench[col_retorno] = para_numero_seguro(df_bench[col_retorno])
    df_bench = (
        df_bench[[col_data, col_retorno]]
        .dropna(subset=[col_data, col_retorno])
        .groupby(col_data, as_index=False)[col_retorno]
        .mean()
        .sort_values(col_data)
        .reset_index(drop=True)
    )
    return df_bench, col_data, col_retorno, origem

def preparar_auditoria(linhas):
    df_aud = pd.DataFrame(linhas)
    for coluna in ["item", "valor", "valor_referencia", "status", "observacao"]:
        if coluna not in df_aud.columns:
            df_aud[coluna] = ""
        df_aud[coluna] = df_aud[coluna].astype(str)
    return df_aud

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização das bases e identificação das colunas de amplitude
# ============================================================

print("\n[5/12] Padronização das bases e identificação das colunas de amplitude...")

COL_DATA_AMPLITUDE = escolher_coluna(
    df_amplitude,
    ["data", "data_pregao", "dt_pregao"],
    obrigatoria=True,
    nome_logico="data na base agregada de amplitude da 5.5"
)
COL_DATA_BRUTA = escolher_coluna(
    df_datas_brutas_capitulacao,
    ["data", "data_sinal", "data_pregao", "data_referencia"],
    obrigatoria=True,
    nome_logico="data na tabela de datas brutas de capitulação da 6.1"
)
COL_DATA_FINAL = escolher_coluna(
    df_calendario_final_sinais,
    ["data", "data_sinal", "data_aporte", "data_referencia"],
    obrigatoria=True,
    nome_logico="data no calendário final de sinais da 6.4"
)
COL_ESTRATEGIA_FINAL = escolher_coluna(
    df_calendario_final_sinais,
    ["estrategia_referencia", "estrategia", "tipo_sinal"],
    obrigatoria=True,
    nome_logico="estratégia no calendário final de sinais da 6.4"
)

COL_PCT_MINIMA_52S = escolher_coluna(
    df_amplitude,
    [
        "pct_tickers_minima_52s", "pct_tickers_minima_52_semanas", "pct_minima_52s", "pct_minimas_52s",
        "perc_tickers_minima_52s", "proporcao_tickers_minima_52s", "pct_ativos_minima_52s",
        "pct_em_minima_52s", "percentual_minima_52s", "percentual_minimas_52s"
    ],
    obrigatoria=False,
    nome_logico="percentual de ativos em mínima de 52 semanas"
)
COL_PCT_ABAIXO_MM200 = escolher_coluna(
    df_amplitude,
    [
        "pct_tickers_abaixo_mm200", "pct_abaixo_mm200", "pct_ativos_abaixo_mm200",
        "perc_tickers_abaixo_mm200", "proporcao_tickers_abaixo_mm200", "percentual_abaixo_mm200",
        "pct_tickers_abaixo_media_200", "pct_abaixo_media_200"
    ],
    obrigatoria=False,
    nome_logico="percentual de ativos abaixo da MM200"
)
COL_PCT_MAXIMA_52S = escolher_coluna(
    df_amplitude,
    [
        "pct_tickers_maxima_52s", "pct_tickers_maxima_52_semanas", "pct_maxima_52s", "pct_maximas_52s",
        "perc_tickers_maxima_52s", "proporcao_tickers_maxima_52s", "pct_ativos_maxima_52s",
        "pct_em_maxima_52s", "percentual_maxima_52s", "percentual_maximas_52s"
    ],
    obrigatoria=False,
    nome_logico="percentual de ativos em máxima de 52 semanas"
)
COL_PCT_ACIMA_MM200 = escolher_coluna(
    df_amplitude,
    [
        "pct_tickers_acima_mm200", "pct_acima_mm200", "pct_ativos_acima_mm200",
        "perc_tickers_acima_mm200", "proporcao_tickers_acima_mm200", "percentual_acima_mm200",
        "pct_tickers_acima_media_200", "pct_acima_media_200"
    ],
    obrigatoria=False,
    nome_logico="percentual de ativos acima da MM200"
)

if COL_PCT_MINIMA_52S is None:
    COL_PCT_MINIMA_52S = escolher_coluna_por_termos(
        df_amplitude,
        listas_termos=[["pct", "minima"], ["pct", "mínima"], ["percentual", "minima"], ["proporcao", "minima"]],
        termos_exclusao=["max", "maior", "euforia"],
        obrigatoria=False,
        nome_logico="percentual de ativos em mínima de 52 semanas"
    )
if COL_PCT_ABAIXO_MM200 is None:
    COL_PCT_ABAIXO_MM200 = escolher_coluna_por_termos(
        df_amplitude,
        listas_termos=[["pct", "abaixo", "200"], ["percentual", "abaixo", "200"], ["proporcao", "abaixo", "200"]],
        termos_exclusao=["acima", "euforia"],
        obrigatoria=False,
        nome_logico="percentual de ativos abaixo da MM200"
    )
if COL_PCT_MAXIMA_52S is None:
    COL_PCT_MAXIMA_52S = escolher_coluna_por_termos(
        df_amplitude,
        listas_termos=[["pct", "maxima"], ["pct", "máxima"], ["percentual", "maxima"], ["proporcao", "maxima"]],
        termos_exclusao=["min", "menor", "capitulacao"],
        obrigatoria=False,
        nome_logico="percentual de ativos em máxima de 52 semanas"
    )
if COL_PCT_ACIMA_MM200 is None:
    COL_PCT_ACIMA_MM200 = escolher_coluna_por_termos(
        df_amplitude,
        listas_termos=[["pct", "acima", "200"], ["percentual", "acima", "200"], ["proporcao", "acima", "200"]],
        termos_exclusao=["abaixo", "capitulacao"],
        obrigatoria=False,
        nome_logico="percentual de ativos acima da MM200"
    )

metricas_centrais_disponiveis = [col for col in [COL_PCT_MINIMA_52S, COL_PCT_ABAIXO_MM200] if col is not None]
if len(metricas_centrais_disponiveis) == 0:
    raise KeyError(
        "Não foi possível identificar métricas centrais de capitulação na base 5.5. "
        "É necessário encontrar ao menos percentual de mínimas de 52 semanas ou percentual abaixo da MM200. "
        f"Colunas disponíveis: {list(df_amplitude.columns)}"
    )

df_amplitude[COL_DATA_AMPLITUDE] = para_datetime_seguro(df_amplitude[COL_DATA_AMPLITUDE])
df_datas_brutas_capitulacao[COL_DATA_BRUTA] = para_datetime_seguro(df_datas_brutas_capitulacao[COL_DATA_BRUTA])
df_calendario_final_sinais[COL_DATA_FINAL] = para_datetime_seguro(df_calendario_final_sinais[COL_DATA_FINAL])

df_amp = df_amplitude.dropna(subset=[COL_DATA_AMPLITUDE]).copy().sort_values(COL_DATA_AMPLITUDE).reset_index(drop=True)

df_amp["pct_minima_52s_sens"] = normalizar_percentual(df_amp[COL_PCT_MINIMA_52S]) if COL_PCT_MINIMA_52S is not None else 0.0
df_amp["pct_abaixo_mm200_sens"] = normalizar_percentual(df_amp[COL_PCT_ABAIXO_MM200]) if COL_PCT_ABAIXO_MM200 is not None else 0.0
df_amp["pct_maxima_52s_sens"] = normalizar_percentual(df_amp[COL_PCT_MAXIMA_52S]) if COL_PCT_MAXIMA_52S is not None else 0.0
df_amp["pct_acima_mm200_sens"] = normalizar_percentual(df_amp[COL_PCT_ACIMA_MM200]) if COL_PCT_ACIMA_MM200 is not None else 0.0

n_metricas_pro_capitulacao = int(COL_PCT_MINIMA_52S is not None) + int(COL_PCT_ABAIXO_MM200 is not None)
n_metricas_anti_capitulacao = int(COL_PCT_MAXIMA_52S is not None) + int(COL_PCT_ACIMA_MM200 is not None)

df_amp["score_capitulacao_sens"] = (
    df_amp["pct_minima_52s_sens"] +
    df_amp["pct_abaixo_mm200_sens"] -
    0.50 * df_amp["pct_maxima_52s_sens"] -
    0.50 * df_amp["pct_acima_mm200_sens"]
)

df_datas_brutas_ref = (
    df_datas_brutas_capitulacao[[COL_DATA_BRUTA]]
    .dropna()
    .drop_duplicates()
    .rename(columns={COL_DATA_BRUTA: "data_sinal"})
    .sort_values("data_sinal")
    .reset_index(drop=True)
)

df_sinais_finais_cap = (
    df_calendario_final_sinais.loc[
        df_calendario_final_sinais[COL_ESTRATEGIA_FINAL].astype(str).str.lower().str.contains("capitul", na=False),
        [COL_DATA_FINAL]
    ]
    .dropna()
    .drop_duplicates()
    .rename(columns={COL_DATA_FINAL: "data_sinal"})
    .sort_values("data_sinal")
    .reset_index(drop=True)
)

print(f"Coluna data amplitude                  : {COL_DATA_AMPLITUDE}")
print(f"Coluna % mínima 52 semanas             : {COL_PCT_MINIMA_52S}")
print(f"Coluna % abaixo MM200                  : {COL_PCT_ABAIXO_MM200}")
print(f"Coluna % máxima 52 semanas             : {COL_PCT_MAXIMA_52S}")
print(f"Coluna % acima MM200                   : {COL_PCT_ACIMA_MM200}")
print(f"Métricas pró-capitulação disponíveis   : {n_metricas_pro_capitulacao}")
print(f"Métricas anti-capitulação disponíveis  : {n_metricas_anti_capitulacao}")
print(f"Datas brutas oficiais de capitulação   : {len(df_datas_brutas_ref):,}")
print(f"Datas finais oficiais de capitulação   : {len(df_sinais_finais_cap):,}")
print("OK")

# ============================================================
# 6) Definição dos cenários de limiares e calendário de pregões
# ============================================================

print("\n[6/12] Definição dos cenários de limiares e calendário de pregões...")

COOLDOWN_REFERENCIA = int(globals().get("PARAMETROS_GLOBAIS", {}).get("cooldown_sinais_pregoes", 63))
HORIZONTES_FORWARD = [21, 63, 126, 252]

calendario_pregoes = sorted(df_amp[COL_DATA_AMPLITUDE].dropna().unique())

df_scores_ref = df_amp.merge(df_datas_brutas_ref, left_on=COL_DATA_AMPLITUDE, right_on="data_sinal", how="inner")
if df_scores_ref.empty:
    raise ValueError("Não foi possível cruzar as datas brutas de capitulação da 6.1 com a base de amplitude da 5.5.")

limiar_referencia_score = float(df_scores_ref["score_capitulacao_sens"].min())
quantil_25_score_ref = float(df_scores_ref["score_capitulacao_sens"].quantile(0.25))
quantil_50_score_ref = float(df_scores_ref["score_capitulacao_sens"].quantile(0.50))

if pd.isna(limiar_referencia_score):
    raise ValueError("O score de capitulação de referência não pôde ser calculado.")

if limiar_referencia_score > 0:
    limiar_muito_flexivel = limiar_referencia_score * 0.75
    limiar_flexivel = limiar_referencia_score * 0.90
else:
    limiar_muito_flexivel = float(df_amp["score_capitulacao_sens"].quantile(0.85))
    limiar_flexivel = float(df_amp["score_capitulacao_sens"].quantile(0.90))

limiar_rigoroso = max(limiar_referencia_score, quantil_25_score_ref)
limiar_muito_rigoroso = max(limiar_rigoroso, quantil_50_score_ref)

CENARIOS_LIMIARES_CAPITULACAO = [
    {
        "cenario_limiar_capitulacao": "muito_flexivel",
        "ordem_cenario": 1,
        "descricao_cenario": "Limiar 25% abaixo da referência proxy inferida pelo score de capitulação.",
        "tipo_cenario": "score_proxy",
        "limiar_score_capitulacao": limiar_muito_flexivel,
    },
    {
        "cenario_limiar_capitulacao": "flexivel",
        "ordem_cenario": 2,
        "descricao_cenario": "Limiar 10% abaixo da referência proxy inferida pelo score de capitulação.",
        "tipo_cenario": "score_proxy",
        "limiar_score_capitulacao": limiar_flexivel,
    },
    {
        "cenario_limiar_capitulacao": "referencia_oficial",
        "ordem_cenario": 3,
        "descricao_cenario": "Datas brutas oficiais de capitulação da 6.1, com cooldown oficial de 63 pregões.",
        "tipo_cenario": "referencia_oficial_6_1",
        "limiar_score_capitulacao": limiar_referencia_score,
    },
    {
        "cenario_limiar_capitulacao": "rigoroso",
        "ordem_cenario": 4,
        "descricao_cenario": "Limiar mais restritivo usando o percentil 25 do score nas datas brutas oficiais.",
        "tipo_cenario": "score_proxy",
        "limiar_score_capitulacao": limiar_rigoroso,
    },
    {
        "cenario_limiar_capitulacao": "muito_rigoroso",
        "ordem_cenario": 5,
        "descricao_cenario": "Limiar mais restritivo usando o percentil 50 do score nas datas brutas oficiais.",
        "tipo_cenario": "score_proxy",
        "limiar_score_capitulacao": limiar_muito_rigoroso,
    },
]

df_parametros_cenarios = pd.DataFrame(CENARIOS_LIMIARES_CAPITULACAO)
df_parametros_cenarios["cooldown_pregoes"] = COOLDOWN_REFERENCIA
df_parametros_cenarios["coluna_pct_minima_52s"] = str(COL_PCT_MINIMA_52S)
df_parametros_cenarios["coluna_pct_abaixo_mm200"] = str(COL_PCT_ABAIXO_MM200)
df_parametros_cenarios["coluna_pct_maxima_52s"] = str(COL_PCT_MAXIMA_52S)
df_parametros_cenarios["coluna_pct_acima_mm200"] = str(COL_PCT_ACIMA_MM200)
df_parametros_cenarios["limiar_referencia_score"] = limiar_referencia_score
df_parametros_cenarios["quantil_25_score_datas_brutas_referencia"] = quantil_25_score_ref
df_parametros_cenarios["quantil_50_score_datas_brutas_referencia"] = quantil_50_score_ref

print(f"Cooldown de referência                : {COOLDOWN_REFERENCIA} pregões")
print(f"Cenários de limiar definidos           : {len(CENARIOS_LIMIARES_CAPITULACAO):,}")
print(f"Horizontes forward para proxy          : {HORIZONTES_FORWARD}")
print(f"Pregões no calendário operacional      : {len(calendario_pregoes):,}")
print(f"Score mínimo das datas brutas oficiais : {limiar_referencia_score:.10f}")
print(f"Score P25 das datas brutas oficiais    : {quantil_25_score_ref:.10f}")
print(f"Score P50 das datas brutas oficiais    : {quantil_50_score_ref:.10f}")
print("\nParâmetros dos cenários:")
print(df_parametros_cenarios.to_string(index=False))
print("OK")

# ============================================================
# 7) Aplicação dos limiares alternativos e cooldown
# ============================================================

print("\n[7/12] Aplicação dos limiares alternativos e cooldown...")

partes_sinais = []
for cenario in CENARIOS_LIMIARES_CAPITULACAO:
    nome_cenario = cenario["cenario_limiar_capitulacao"]
    ordem_cenario = cenario["ordem_cenario"]
    limiar_score = cenario["limiar_score_capitulacao"]
    tipo_cenario = cenario["tipo_cenario"]
    if nome_cenario == "referencia_oficial":
        df_candidatos = df_amp.merge(df_datas_brutas_ref, left_on=COL_DATA_AMPLITUDE, right_on="data_sinal", how="inner").copy()
        df_candidatos["origem_candidato"] = "datas_brutas_oficiais_6_1"
    else:
        df_candidatos = df_amp.loc[df_amp["score_capitulacao_sens"].ge(limiar_score)].copy()
        df_candidatos["data_sinal"] = df_candidatos[COL_DATA_AMPLITUDE]
        df_candidatos["origem_candidato"] = "score_proxy_capitulacao"
    datas_aceitas = aplicar_cooldown_datas(df_candidatos["data_sinal"], calendario_pregoes, COOLDOWN_REFERENCIA)
    set_aceitas = set(datas_aceitas)
    df_candidatos = df_candidatos.copy()
    df_candidatos["cenario_limiar_capitulacao"] = nome_cenario
    df_candidatos["ordem_cenario"] = ordem_cenario
    df_candidatos["tipo_cenario"] = tipo_cenario
    df_candidatos["limiar_score_capitulacao"] = limiar_score
    df_candidatos["estrategia_referencia"] = "capitulacao"
    df_candidatos["nome_exibicao_estrategia"] = "Capitulação"
    df_candidatos["flag_sinal_bruto_cenario"] = True
    df_candidatos["flag_sinal_pos_cooldown"] = df_candidatos["data_sinal"].isin(set_aceitas)
    df_candidatos["cooldown_pregoes"] = COOLDOWN_REFERENCIA
    colunas_saida = [
        "data_sinal", "estrategia_referencia", "nome_exibicao_estrategia", "cenario_limiar_capitulacao",
        "ordem_cenario", "tipo_cenario", "limiar_score_capitulacao", "cooldown_pregoes",
        "flag_sinal_bruto_cenario", "flag_sinal_pos_cooldown", "origem_candidato",
        "score_capitulacao_sens", "pct_minima_52s_sens", "pct_abaixo_mm200_sens", "pct_maxima_52s_sens", "pct_acima_mm200_sens"
    ]
    partes_sinais.append(df_candidatos[colunas_saida].copy())

if partes_sinais:
    df_sinais_limiares = pd.concat(partes_sinais, ignore_index=True)
else:
    df_sinais_limiares = pd.DataFrame()

df_sinais_limiares = df_sinais_limiares.sort_values(["ordem_cenario", "data_sinal"]).reset_index(drop=True)

duplicatas_cenario_data = int(df_sinais_limiares.duplicated(subset=["cenario_limiar_capitulacao", "data_sinal"]).sum()) if not df_sinais_limiares.empty else 0
if duplicatas_cenario_data > 0:
    df_sinais_limiares = df_sinais_limiares.drop_duplicates(subset=["cenario_limiar_capitulacao", "data_sinal"], keep="first").reset_index(drop=True)

print(f"Base de sinais por limiar             : {len(df_sinais_limiares):,} linhas")
print(f"Cenários de limiar gerados             : {df_sinais_limiares['cenario_limiar_capitulacao'].nunique() if not df_sinais_limiares.empty else 0:,}")
print(f"Duplicatas removidas por cenário/data  : {duplicatas_cenario_data:,}")
print("OK")

# ============================================================
# 8) Construção dos resumos de sensibilidade e comparação com a referência
# ============================================================

print("\n[8/12] Construção dos resumos de sensibilidade e comparação com a referência...")

linhas_resumo = []
for cenario in CENARIOS_LIMIARES_CAPITULACAO:
    nome_cenario = cenario["cenario_limiar_capitulacao"]
    ordem_cenario = cenario["ordem_cenario"]
    df_cenario = df_sinais_limiares.loc[df_sinais_limiares["cenario_limiar_capitulacao"].eq(nome_cenario)].copy()
    df_aceitos = df_cenario.loc[df_cenario["flag_sinal_pos_cooldown"]].copy().sort_values("data_sinal")
    datas_aceitas = list(df_aceitos["data_sinal"])
    intervalos = []
    if len(datas_aceitas) > 1:
        mapa_ordem = {data: i for i, data in enumerate(calendario_pregoes)}
        ordens = [mapa_ordem[data] for data in datas_aceitas if data in mapa_ordem]
        intervalos = list(np.diff(ordens))
    linhas_resumo.append({
        "estrategia_referencia": "capitulacao",
        "nome_exibicao_estrategia": "Capitulação",
        "cenario_limiar_capitulacao": nome_cenario,
        "ordem_cenario": ordem_cenario,
        "tipo_cenario": cenario["tipo_cenario"],
        "limiar_score_capitulacao": cenario["limiar_score_capitulacao"],
        "cooldown_pregoes": COOLDOWN_REFERENCIA,
        "n_sinais_brutos": len(df_cenario),
        "n_sinais_pos_cooldown": len(df_aceitos),
        "n_sinais_rejeitados_cooldown": len(df_cenario) - len(df_aceitos),
        "pct_sinais_aceitos": len(df_aceitos) / len(df_cenario) if len(df_cenario) > 0 else np.nan,
        "primeira_data_sinal": df_aceitos["data_sinal"].min() if len(df_aceitos) > 0 else pd.NaT,
        "ultima_data_sinal": df_aceitos["data_sinal"].max() if len(df_aceitos) > 0 else pd.NaT,
        "score_medio_sinais_aceitos": df_aceitos["score_capitulacao_sens"].mean() if len(df_aceitos) > 0 else np.nan,
        "score_mediano_sinais_aceitos": df_aceitos["score_capitulacao_sens"].median() if len(df_aceitos) > 0 else np.nan,
        "intervalo_medio_pregoes": float(np.mean(intervalos)) if intervalos else np.nan,
        "intervalo_mediano_pregoes": float(np.median(intervalos)) if intervalos else np.nan,
        "intervalo_minimo_pregoes": float(np.min(intervalos)) if intervalos else np.nan,
        "intervalo_maximo_pregoes": float(np.max(intervalos)) if intervalos else np.nan,
    })

df_resumo = pd.DataFrame(linhas_resumo).sort_values("ordem_cenario").reset_index(drop=True)

n_referencia_oficial = int(df_sinais_finais_cap["data_sinal"].nunique())
n_referencia_recalculada = int(df_resumo.loc[df_resumo["cenario_limiar_capitulacao"].eq("referencia_oficial"), "n_sinais_pos_cooldown"].iloc[0]) if not df_resumo.empty else 0

linhas_comparacao = []
set_ref = set(df_sinais_limiares.loc[
    df_sinais_limiares["cenario_limiar_capitulacao"].eq("referencia_oficial") & df_sinais_limiares["flag_sinal_pos_cooldown"],
    "data_sinal"
])
for cenario in CENARIOS_LIMIARES_CAPITULACAO:
    nome_cenario = cenario["cenario_limiar_capitulacao"]
    set_atual = set(df_sinais_limiares.loc[
        df_sinais_limiares["cenario_limiar_capitulacao"].eq(nome_cenario) & df_sinais_limiares["flag_sinal_pos_cooldown"],
        "data_sinal"
    ])
    n_comuns = len(set_atual.intersection(set_ref))
    n_uniao = len(set_atual.union(set_ref))
    linhas_comparacao.append({
        "estrategia_referencia": "capitulacao",
        "nome_exibicao_estrategia": "Capitulação",
        "cenario_limiar_capitulacao": nome_cenario,
        "ordem_cenario": cenario["ordem_cenario"],
        "n_sinais_cenario": len(set_atual),
        "n_sinais_referencia": len(set_ref),
        "n_sinais_comuns_com_referencia": n_comuns,
        "n_sinais_adicionados_vs_referencia": len(set_atual - set_ref),
        "n_sinais_removidos_vs_referencia": len(set_ref - set_atual),
        "delta_liquido_sinais_vs_referencia": len(set_atual) - len(set_ref),
        "pct_jaccard_vs_referencia": n_comuns / n_uniao if n_uniao > 0 else np.nan,
    })

df_comparacao_referencia = pd.DataFrame(linhas_comparacao).sort_values("ordem_cenario").reset_index(drop=True)

if not df_sinais_limiares.empty:
    df_impacto_temporal = (
        df_sinais_limiares.loc[df_sinais_limiares["flag_sinal_pos_cooldown"]]
        .assign(ano=lambda x: x["data_sinal"].dt.year)
        .groupby(["cenario_limiar_capitulacao", "ordem_cenario", "ano"], as_index=False)
        .agg(
            n_sinais=("data_sinal", "nunique"),
            score_medio=("score_capitulacao_sens", "mean"),
            score_mediano=("score_capitulacao_sens", "median"),
        )
        .sort_values(["ordem_cenario", "ano"])
        .reset_index(drop=True)
    )
else:
    df_impacto_temporal = pd.DataFrame()

print(f"Resumo de sensibilidade              : {len(df_resumo):,} linhas")
print(f"Comparação com referência             : {len(df_comparacao_referencia):,} linhas")
print(f"Impacto temporal anual                : {len(df_impacto_temporal):,} linhas")
print("OK")

# ============================================================
# 9) Cálculo de performance proxy pós-sinal com Ibovespa
# ============================================================

print("\n[9/12] Cálculo de performance proxy pós-sinal com Ibovespa...")

df_benchmark, COL_DATA_BENCHMARK, COL_RETORNO_BENCHMARK, ORIGEM_BENCHMARK = identificar_serie_benchmark_retornos(df_retornos_diarios)
BENCHMARK_DISPONIVEL = not df_benchmark.empty and COL_DATA_BENCHMARK is not None and COL_RETORNO_BENCHMARK is not None

linhas_proxy = []
if BENCHMARK_DISPONIVEL:
    for _, linha_sinal in df_sinais_limiares.loc[df_sinais_limiares["flag_sinal_pos_cooldown"]].iterrows():
        for horizonte in HORIZONTES_FORWARD:
            retorno_forward = calcular_retorno_forward(
                df_serie_retorno=df_benchmark,
                data_sinal=linha_sinal["data_sinal"],
                horizonte_pregoes=horizonte,
                coluna_data=COL_DATA_BENCHMARK,
                coluna_retorno=COL_RETORNO_BENCHMARK,
            )
            linhas_proxy.append({
                "data_sinal": linha_sinal["data_sinal"],
                "estrategia_referencia": "capitulacao",
                "nome_exibicao_estrategia": "Capitulação",
                "cenario_limiar_capitulacao": linha_sinal["cenario_limiar_capitulacao"],
                "ordem_cenario": linha_sinal["ordem_cenario"],
                "horizonte_pregoes": horizonte,
                "retorno_ibovespa_pos_sinal": retorno_forward,
                "score_capitulacao_sens": linha_sinal["score_capitulacao_sens"],
                "origem_benchmark_proxy": ORIGEM_BENCHMARK,
            })

df_base_proxy = pd.DataFrame(linhas_proxy)
if not df_base_proxy.empty:
    df_tbl_proxy = (
        df_base_proxy.dropna(subset=["retorno_ibovespa_pos_sinal"])
        .groupby(["estrategia_referencia", "nome_exibicao_estrategia", "cenario_limiar_capitulacao", "ordem_cenario", "horizonte_pregoes"], as_index=False)
        .agg(
            n_sinais_com_retorno=("data_sinal", "nunique"),
            retorno_medio_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "mean"),
            retorno_mediano_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "median"),
            retorno_minimo_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "min"),
            retorno_maximo_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "max"),
            pct_sinais_retorno_positivo=("retorno_ibovespa_pos_sinal", lambda x: float((x > 0).mean()) if len(x) > 0 else np.nan),
            score_medio_sinais=("score_capitulacao_sens", "mean"),
        )
        .sort_values(["ordem_cenario", "horizonte_pregoes"])
        .reset_index(drop=True)
    )
else:
    df_tbl_proxy = pd.DataFrame(columns=[
        "estrategia_referencia", "nome_exibicao_estrategia", "cenario_limiar_capitulacao", "ordem_cenario",
        "horizonte_pregoes", "n_sinais_com_retorno", "retorno_medio_ibovespa_pos_sinal",
        "retorno_mediano_ibovespa_pos_sinal", "retorno_minimo_ibovespa_pos_sinal",
        "retorno_maximo_ibovespa_pos_sinal", "pct_sinais_retorno_positivo", "score_medio_sinais"
    ])

print(f"Benchmark proxy identificado           : {ORIGEM_BENCHMARK}")
print(f"Benchmark proxy disponível             : {BENCHMARK_DISPONIVEL}")
print(f"Base de performance proxy              : {len(df_base_proxy):,} linhas")
print(f"Tabela de performance proxy            : {len(df_tbl_proxy):,} linhas")
print("OK")

# ============================================================
# 10) Construção da base de gráficos e auditoria
# ============================================================

print("\n[10/12] Construção da base de gráficos e auditoria...")

base_grafico_sinais = df_resumo[[
    "cenario_limiar_capitulacao", "ordem_cenario", "n_sinais_brutos", "n_sinais_pos_cooldown",
    "limiar_score_capitulacao", "score_medio_sinais_aceitos", "score_mediano_sinais_aceitos"
]].copy()
base_grafico_sinais["tipo_grafico"] = "resumo_sinais_limiar_capitulacao"
base_grafico_sinais["metrica_principal"] = "n_sinais_pos_cooldown"
base_grafico_sinais["valor_principal"] = base_grafico_sinais["n_sinais_pos_cooldown"]

base_grafico_temporal = df_impacto_temporal.copy()
if not base_grafico_temporal.empty:
    base_grafico_temporal["tipo_grafico"] = "sinais_anuais_limiar_capitulacao"
    base_grafico_temporal["metrica_principal"] = "n_sinais"
    base_grafico_temporal["valor_principal"] = base_grafico_temporal["n_sinais"]
else:
    base_grafico_temporal = pd.DataFrame(columns=base_grafico_sinais.columns)

for coluna in base_grafico_sinais.columns:
    if coluna not in base_grafico_temporal.columns:
        base_grafico_temporal[coluna] = np.nan
for coluna in base_grafico_temporal.columns:
    if coluna not in base_grafico_sinais.columns:
        base_grafico_sinais[coluna] = np.nan

df_base_grafico = pd.concat([
    base_grafico_sinais[sorted(base_grafico_sinais.columns)],
    base_grafico_temporal[sorted(base_grafico_sinais.columns)]
], ignore_index=True)

sinais_sem_data = int(df_sinais_limiares["data_sinal"].isna().sum()) if not df_sinais_limiares.empty else 0
sinais_referencia_diferem_oficial = int(abs(n_referencia_recalculada - n_referencia_oficial))
cenarios_sem_sinal = int((df_resumo["n_sinais_pos_cooldown"] == 0).sum()) if not df_resumo.empty else len(CENARIOS_LIMIARES_CAPITULACAO)
metricas_infinitas = 0
frames_validacao_numerica = [df_resumo, df_comparacao_referencia, df_impacto_temporal, df_tbl_proxy]
for frame in frames_validacao_numerica:
    if not frame.empty:
        arr = frame.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
        metricas_infinitas += int(np.isinf(arr).sum())

linhas_auditoria = [
    {
        "item": "metricas_pro_capitulacao_identificadas",
        "valor": n_metricas_pro_capitulacao,
        "valor_referencia": ">= 1",
        "status": "OK" if n_metricas_pro_capitulacao >= 1 else "ERRO",
        "observacao": "A sensibilidade exige ao menos uma métrica central de capitulação identificada na base 5.5.",
    },
    {
        "item": "datas_brutas_capitulacao_6_1",
        "valor": len(df_datas_brutas_ref),
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Quantidade de candidatos brutos oficiais antes do cooldown para Capitulação.",
    },
    {
        "item": "sinais_finais_capitulacao_6_4",
        "valor": n_referencia_oficial,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Quantidade de sinais finais oficiais de Capitulação no calendário da 6.4.",
    },
    {
        "item": "cenarios_limiares_gerados",
        "valor": len(CENARIOS_LIMIARES_CAPITULACAO),
        "valor_referencia": 5,
        "status": "OK" if len(CENARIOS_LIMIARES_CAPITULACAO) == 5 else "ERRO",
        "observacao": "A subetapa deve gerar cenários flexíveis, referência e rigorosos.",
    },
    {
        "item": "cenarios_sem_sinal_pos_cooldown",
        "valor": cenarios_sem_sinal,
        "valor_referencia": 0,
        "status": "OK" if cenarios_sem_sinal == 0 else "ERRO",
        "observacao": "Cenários alternativos de limiar não devem ficar sem sinais após cooldown.",
    },
    {
        "item": "sinais_sem_data",
        "valor": sinais_sem_data,
        "valor_referencia": 0,
        "status": "OK" if sinais_sem_data == 0 else "ERRO",
        "observacao": "Todos os sinais aceitos por cenário de limiar devem possuir data válida.",
    },
    {
        "item": "duplicatas_cenario_data",
        "valor": duplicatas_cenario_data,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_cenario_data == 0 else "ERRO",
        "observacao": "Cada combinação de cenário e data deve aparecer uma única vez após tratamento.",
    },
    {
        "item": "sinais_referencia_diferem_oficial",
        "valor": sinais_referencia_diferem_oficial,
        "valor_referencia": 0,
        "status": "OK" if sinais_referencia_diferem_oficial == 0 else "ERRO",
        "observacao": "Compara a quantidade recalculada no cenário de referência com o calendário oficial da 6.4.",
    },
    {
        "item": "benchmark_proxy_disponivel",
        "valor": int(BENCHMARK_DISPONIVEL),
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Indica se foi possível calcular retorno forward do Ibovespa como proxy de timing pós-sinal.",
    },
    {
        "item": "origem_benchmark_proxy",
        "valor": ORIGEM_BENCHMARK,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Origem da série usada no cálculo de retorno forward pós-sinal.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": 0,
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
n_erros_bloqueantes = int(df_auditoria["status"].eq("ERRO").sum())

print(f"Base gráfico limiares capitulação      : {len(df_base_grafico):,} linhas")
print(f"Itens de auditoria                     : {len(df_auditoria):,}")
print(f"Erros bloqueantes                      : {n_erros_bloqueantes:,}")
print("OK")

# ============================================================
# 11) Salvamento dos outputs
# ============================================================

print("\n[11/12] Salvamento dos outputs...")

salvar_dataframe(df_sinais_limiares, CAMINHO_BASE_SINAIS, index=False)
salvar_dataframe(df_parametros_cenarios, CAMINHO_PARAMETROS_CENARIOS, index=False)
salvar_dataframe(df_resumo, CAMINHO_RESUMO, index=False)
salvar_dataframe(df_comparacao_referencia, CAMINHO_COMPARACAO_REFERENCIA, index=False)
salvar_dataframe(df_impacto_temporal, CAMINHO_IMPACTO_TEMPORAL, index=False)
salvar_dataframe(df_base_proxy, CAMINHO_BASE_PROXY, index=False)
salvar_dataframe(df_tbl_proxy, CAMINHO_TBL_PROXY, index=False)
salvar_dataframe(df_base_grafico, CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 12) Validação final da subetapa
# ============================================================

print("\n[12/12] Validação final da subetapa...")

print("\nAuditoria de validação da sensibilidade dos limiares de capitulação:")
print(df_auditoria.to_string(index=False))

print("\nResumo de sensibilidade por cenário:")
print(df_resumo.to_string(index=False))

print("\nComparação com a referência oficial:")
print(df_comparacao_referencia.to_string(index=False))

print("\nImpacto temporal anual - amostra:")
print(df_impacto_temporal.head(40).to_string(index=False))

print("\nPerformance proxy pós-sinal - amostra:")
print(df_tbl_proxy.head(40).to_string(index=False))

print("\nArquivos salvos na subetapa 13.4:")
print(f"- {CAMINHO_BASE_SINAIS}")
print(f"- {CAMINHO_PARAMETROS_CENARIOS}")
print(f"- {CAMINHO_RESUMO}")
print(f"- {CAMINHO_COMPARACAO_REFERENCIA}")
print(f"- {CAMINHO_IMPACTO_TEMPORAL}")
print(f"- {CAMINHO_BASE_PROXY}")
print(f"- {CAMINHO_TBL_PROXY}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_AUDITORIA}")

if n_erros_bloqueantes > 0:
    raise RuntimeError("A subetapa 13.4 encontrou erros bloqueantes na auditoria. Verifique a tabela de validação.")

print("\nETAPA 13.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 13.4 - SENSIBILIDADE DOS LIMIARES DE CAPITULAÇÃO

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - amplitude agregada da 5.5          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_5_base_indicadores_agregados_amplitude_pregao.parquet
Entrada - parâmetros capitulação da 6.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_1_tbl_parametros_capitulacao.parquet
Entrada - limiares capitulação da 6.1        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_1_tbl_limiares_capitulacao_dinamicos.parquet
Entrada - base bruta capitulação da 6.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_bra

## Etapa 13.5) Sensibilidade dos Limiares de Euforia

In [68]:
%%time
# ============================================================
# Etapa 13.5) Sensibilidade dos Limiares de Euforia
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 13.5 - SENSIBILIDADE DOS LIMIARES DE EUFORIA")
print("=" * 100)

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_5 = Path(DIRETORIOS_PROJETO["etapa_5"])
DIR_ETAPA_6 = Path(DIRETORIOS_PROJETO["etapa_6"])
DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])
DIR_ETAPA_13 = Path(DIRETORIOS_PROJETO["etapa_13"])
DIR_ETAPA_13.mkdir(parents=True, exist_ok=True)

CAMINHO_AMPLITUDE_5_5 = DIR_ETAPA_5 / "5_5_base_indicadores_agregados_amplitude_pregao.parquet"
CAMINHO_PARAMETROS_EUFORIA_6_2 = DIR_ETAPA_6 / "6_2_tbl_parametros_euforia.parquet"
CAMINHO_LIMIARES_EUFORIA_6_2 = DIR_ETAPA_6 / "6_2_tbl_limiares_euforia_dinamicos.parquet"
CAMINHO_BASE_BRUTA_EUFORIA_6_2 = DIR_ETAPA_6 / "6_2_base_euforia_bruta_pregao.parquet"
CAMINHO_DATAS_BRUTAS_EUFORIA_6_2 = DIR_ETAPA_6 / "6_2_tbl_datas_euforia_bruta.parquet"
CAMINHO_CALENDARIO_FINAL_SINAIS_6_4 = DIR_ETAPA_6 / "6_4_base_calendario_final_sinais.parquet"
CAMINHO_RETORNOS_DIARIOS_11_1 = DIR_ETAPA_11 / "11_1_base_retornos_diarios.parquet"

CAMINHO_BASE_SINAIS = DIR_ETAPA_13 / "13_5_base_sinais_sensibilidade_limiares_euforia.parquet"
CAMINHO_PARAMETROS_CENARIOS = DIR_ETAPA_13 / "13_5_tbl_parametros_cenarios_limiares_euforia.parquet"
CAMINHO_RESUMO = DIR_ETAPA_13 / "13_5_tbl_resumo_sensibilidade_limiares_euforia.parquet"
CAMINHO_COMPARACAO_REFERENCIA = DIR_ETAPA_13 / "13_5_tbl_comparacao_limiar_euforia_referencia.parquet"
CAMINHO_IMPACTO_TEMPORAL = DIR_ETAPA_13 / "13_5_tbl_impacto_temporal_limiares_euforia.parquet"
CAMINHO_BASE_PROXY = DIR_ETAPA_13 / "13_5_base_performance_proxy_pos_sinal_euforia.parquet"
CAMINHO_TBL_PROXY = DIR_ETAPA_13 / "13_5_tbl_performance_proxy_pos_sinal_limiares_euforia.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_13 / "13_5_tbl_base_grafico_limiares_euforia.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_13 / "13_5_tbl_auditoria_validacao_sensibilidade_limiares_euforia.parquet"

CAMINHOS_ENTRADA_OBRIGATORIOS = [
    CAMINHO_AMPLITUDE_5_5,
    CAMINHO_BASE_BRUTA_EUFORIA_6_2,
    CAMINHO_DATAS_BRUTAS_EUFORIA_6_2,
    CAMINHO_CALENDARIO_FINAL_SINAIS_6_4,
]

for caminho in CAMINHOS_ENTRADA_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - amplitude agregada da 5.5          : {CAMINHO_AMPLITUDE_5_5}")
print(f"Entrada - parâmetros euforia da 6.2      : {CAMINHO_PARAMETROS_EUFORIA_6_2}")
print(f"Entrada - limiares euforia da 6.2        : {CAMINHO_LIMIARES_EUFORIA_6_2}")
print(f"Entrada - base bruta euforia da 6.2      : {CAMINHO_BASE_BRUTA_EUFORIA_6_2}")
print(f"Entrada - datas brutas euforia da 6.2    : {CAMINHO_DATAS_BRUTAS_EUFORIA_6_2}")
print(f"Entrada - calendário final de sinais da 6.4  : {CAMINHO_CALENDARIO_FINAL_SINAIS_6_4}")
print(f"Entrada - retornos diários da 11.1           : {CAMINHO_RETORNOS_DIARIOS_11_1}")
print(f"Saída   - base sinais por limiar             : {CAMINHO_BASE_SINAIS}")
print(f"Saída   - parâmetros dos cenários            : {CAMINHO_PARAMETROS_CENARIOS}")
print(f"Saída   - resumo sensibilidade               : {CAMINHO_RESUMO}")
print(f"Saída   - comparação com referência          : {CAMINHO_COMPARACAO_REFERENCIA}")
print(f"Saída   - impacto temporal                   : {CAMINHO_IMPACTO_TEMPORAL}")
print(f"Saída   - base performance proxy             : {CAMINHO_BASE_PROXY}")
print(f"Saída   - tabela performance proxy           : {CAMINHO_TBL_PROXY}")
print(f"Saída   - base gráfico                       : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação             : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/12] Carga das bases oficiais da subetapa...")

df_amplitude = pd.read_parquet(CAMINHO_AMPLITUDE_5_5)
df_base_bruta_euforia = pd.read_parquet(CAMINHO_BASE_BRUTA_EUFORIA_6_2)
df_datas_brutas_euforia = pd.read_parquet(CAMINHO_DATAS_BRUTAS_EUFORIA_6_2)
df_calendario_final_sinais = pd.read_parquet(CAMINHO_CALENDARIO_FINAL_SINAIS_6_4)

if Path(CAMINHO_PARAMETROS_EUFORIA_6_2).exists():
    df_parametros_euforia = pd.read_parquet(CAMINHO_PARAMETROS_EUFORIA_6_2)
else:
    df_parametros_euforia = pd.DataFrame()

if Path(CAMINHO_LIMIARES_EUFORIA_6_2).exists():
    df_limiares_euforia = pd.read_parquet(CAMINHO_LIMIARES_EUFORIA_6_2)
else:
    df_limiares_euforia = pd.DataFrame()

if Path(CAMINHO_RETORNOS_DIARIOS_11_1).exists():
    df_retornos_diarios = pd.read_parquet(CAMINHO_RETORNOS_DIARIOS_11_1)
    RETORNOS_DIARIOS_DISPONIVEIS = True
else:
    df_retornos_diarios = pd.DataFrame()
    RETORNOS_DIARIOS_DISPONIVEIS = False

print(f"Amplitude agregada da 5.5          : {len(df_amplitude):,} linhas x {df_amplitude.shape[1]:,} colunas")
print(f"Parâmetros euforia da 6.2      : {len(df_parametros_euforia):,} linhas x {df_parametros_euforia.shape[1]:,} colunas")
print(f"Limiares euforia da 6.2        : {len(df_limiares_euforia):,} linhas x {df_limiares_euforia.shape[1]:,} colunas")
print(f"Base bruta euforia da 6.2      : {len(df_base_bruta_euforia):,} linhas x {df_base_bruta_euforia.shape[1]:,} colunas")
print(f"Datas brutas euforia da 6.2    : {len(df_datas_brutas_euforia):,} linhas x {df_datas_brutas_euforia.shape[1]:,} colunas")
print(f"Calendário final de sinais da 6.4  : {len(df_calendario_final_sinais):,} linhas x {df_calendario_final_sinais.shape[1]:,} colunas")
print(f"Retornos diários disponíveis       : {RETORNOS_DIARIOS_DISPONIVEIS}")
if RETORNOS_DIARIOS_DISPONIVEIS:
    print(f"Base de retornos diários           : {len(df_retornos_diarios):,} linhas x {df_retornos_diarios.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

def escolher_coluna(df, candidatos, obrigatoria=False, nome_logico="coluna"):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar {nome_logico}. "
            f"Candidatas avaliadas: {candidatos}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

def escolher_coluna_por_termos(df, listas_termos, termos_exclusao=None, obrigatoria=False, nome_logico="coluna"):
    termos_exclusao = termos_exclusao or []
    for termos in listas_termos:
        for coluna in df.columns:
            nome = str(coluna).lower()
            if all(termo in nome for termo in termos) and not any(termo in nome for termo in termos_exclusao):
                return coluna
    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar {nome_logico} pelos termos {listas_termos}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

def para_datetime_seguro(serie):
    return pd.to_datetime(serie, errors="coerce").dt.normalize()

def para_numero_seguro(serie):
    return pd.to_numeric(serie, errors="coerce")

def normalizar_percentual(serie):
    serie_num = para_numero_seguro(serie)
    p95 = serie_num.dropna().quantile(0.95) if serie_num.notna().any() else np.nan
    if pd.notna(p95) and p95 > 1.5:
        serie_num = serie_num / 100.0
    return serie_num.clip(lower=0, upper=1)

def aplicar_cooldown_datas(datas, calendario_pregoes, cooldown_pregoes):
    datas_validas = sorted(pd.Series(datas).dropna().unique())
    mapa_ordem = {data: i for i, data in enumerate(calendario_pregoes)}
    datas_aceitas = []
    ultima_ordem_aceita = None
    for data in datas_validas:
        if data not in mapa_ordem:
            continue
        ordem_data = mapa_ordem[data]
        if ultima_ordem_aceita is None or (ordem_data - ultima_ordem_aceita) >= cooldown_pregoes:
            datas_aceitas.append(data)
            ultima_ordem_aceita = ordem_data
    return datas_aceitas

def calcular_retorno_forward(df_serie_retorno, data_sinal, horizonte_pregoes, coluna_data, coluna_retorno):
    if df_serie_retorno.empty:
        return np.nan
    datas = df_serie_retorno[coluna_data].to_numpy()
    idx = np.searchsorted(datas, np.datetime64(data_sinal), side="left")
    idx_inicio = idx + 1
    idx_fim = idx_inicio + horizonte_pregoes
    if idx_inicio >= len(df_serie_retorno):
        return np.nan
    janela = df_serie_retorno.iloc[idx_inicio:min(idx_fim, len(df_serie_retorno))][coluna_retorno].dropna()
    if len(janela) < max(1, int(horizonte_pregoes * 0.8)):
        return np.nan
    return float((1.0 + janela).prod() - 1.0)

def identificar_serie_benchmark_retornos(df):
    if df.empty:
        return pd.DataFrame(), None, None, "indisponivel"
    col_data = escolher_coluna(df, ["data", "data_retorno", "data_pregao", "dt_pregao"], obrigatoria=False)
    if col_data is None:
        return pd.DataFrame(), None, None, "sem_coluna_data"
    col_retorno = escolher_coluna(
        df,
        ["retorno_diario", "retorno", "retorno_carteira", "retorno_periodo", "retorno_1d"],
        obrigatoria=False
    )
    if col_retorno is None:
        col_retorno = escolher_coluna_por_termos(
            df,
            listas_termos=[["retorno"]],
            termos_exclusao=["acumul", "anual", "mensal", "excesso"],
            obrigatoria=False,
            nome_logico="retorno diário"
        )
    if col_retorno is None:
        return pd.DataFrame(), col_data, None, "sem_coluna_retorno"
    colunas_texto = [col for col in df.columns if df[col].dtype == "object" or str(df[col].dtype).startswith("string")]
    mascara_benchmark = pd.Series(False, index=df.index)
    for col in colunas_texto:
        serie_texto = df[col].astype(str).str.lower()
        mascara_benchmark = mascara_benchmark | serie_texto.str.contains("ibovespa", na=False) | serie_texto.str.contains("ibov", na=False)
    df_bench = df.loc[mascara_benchmark, [col_data, col_retorno] + colunas_texto].copy()
    if df_bench.empty:
        col_wide = escolher_coluna_por_termos(
            df,
            listas_termos=[["ibov"], ["ibovespa"]],
            termos_exclusao=["patrimonio", "valor"],
            obrigatoria=False,
            nome_logico="coluna wide do Ibovespa"
        )
        if col_wide is not None:
            df_bench = df[[col_data, col_wide]].copy().rename(columns={col_wide: col_retorno})
            origem = f"wide:{col_wide}"
        else:
            return pd.DataFrame(), col_data, col_retorno, "benchmark_nao_identificado"
    else:
        origem = "filtro_textual_ibovespa"
    df_bench[col_data] = para_datetime_seguro(df_bench[col_data])
    df_bench[col_retorno] = para_numero_seguro(df_bench[col_retorno])
    df_bench = (
        df_bench[[col_data, col_retorno]]
        .dropna(subset=[col_data, col_retorno])
        .groupby(col_data, as_index=False)[col_retorno]
        .mean()
        .sort_values(col_data)
        .reset_index(drop=True)
    )
    return df_bench, col_data, col_retorno, origem

def preparar_auditoria(linhas):
    df_aud = pd.DataFrame(linhas)
    for coluna in ["item", "valor", "valor_referencia", "status", "observacao"]:
        if coluna not in df_aud.columns:
            df_aud[coluna] = ""
        df_aud[coluna] = df_aud[coluna].astype(str)
    return df_aud

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização das bases e identificação das colunas de amplitude
# ============================================================

print("\n[5/12] Padronização das bases e identificação das colunas de amplitude...")

COL_DATA_AMPLITUDE = escolher_coluna(
    df_amplitude,
    ["data", "data_pregao", "dt_pregao"],
    obrigatoria=True,
    nome_logico="data na base agregada de amplitude da 5.5"
)
COL_DATA_BRUTA = escolher_coluna(
    df_datas_brutas_euforia,
    ["data", "data_sinal", "data_pregao", "data_referencia"],
    obrigatoria=True,
    nome_logico="data na tabela de datas brutas de euforia da 6.2"
)
COL_DATA_FINAL = escolher_coluna(
    df_calendario_final_sinais,
    ["data", "data_sinal", "data_aporte", "data_referencia"],
    obrigatoria=True,
    nome_logico="data no calendário final de sinais da 6.4"
)
COL_ESTRATEGIA_FINAL = escolher_coluna(
    df_calendario_final_sinais,
    ["estrategia_referencia", "estrategia", "tipo_sinal"],
    obrigatoria=True,
    nome_logico="estratégia no calendário final de sinais da 6.4"
)

COL_PCT_MINIMA_52S = escolher_coluna(
    df_amplitude,
    [
        "pct_tickers_minima_52s", "pct_tickers_minima_52_semanas", "pct_minima_52s", "pct_minimas_52s",
        "perc_tickers_minima_52s", "proporcao_tickers_minima_52s", "pct_ativos_minima_52s",
        "pct_em_minima_52s", "percentual_minima_52s", "percentual_minimas_52s"
    ],
    obrigatoria=False,
    nome_logico="percentual de ativos em mínima de 52 semanas"
)
COL_PCT_ABAIXO_MM200 = escolher_coluna(
    df_amplitude,
    [
        "pct_tickers_abaixo_mm200", "pct_abaixo_mm200", "pct_ativos_abaixo_mm200",
        "perc_tickers_abaixo_mm200", "proporcao_tickers_abaixo_mm200", "percentual_abaixo_mm200",
        "pct_tickers_abaixo_media_200", "pct_abaixo_media_200"
    ],
    obrigatoria=False,
    nome_logico="percentual de ativos abaixo da MM200"
)
COL_PCT_MAXIMA_52S = escolher_coluna(
    df_amplitude,
    [
        "pct_tickers_maxima_52s", "pct_tickers_maxima_52_semanas", "pct_maxima_52s", "pct_maximas_52s",
        "perc_tickers_maxima_52s", "proporcao_tickers_maxima_52s", "pct_ativos_maxima_52s",
        "pct_em_maxima_52s", "percentual_maxima_52s", "percentual_maximas_52s"
    ],
    obrigatoria=False,
    nome_logico="percentual de ativos em máxima de 52 semanas"
)
COL_PCT_ACIMA_MM200 = escolher_coluna(
    df_amplitude,
    [
        "pct_tickers_acima_mm200", "pct_acima_mm200", "pct_ativos_acima_mm200",
        "perc_tickers_acima_mm200", "proporcao_tickers_acima_mm200", "percentual_acima_mm200",
        "pct_tickers_acima_media_200", "pct_acima_media_200"
    ],
    obrigatoria=False,
    nome_logico="percentual de ativos acima da MM200"
)

if COL_PCT_MINIMA_52S is None:
    COL_PCT_MINIMA_52S = escolher_coluna_por_termos(
        df_amplitude,
        listas_termos=[["pct", "minima"], ["pct", "mínima"], ["percentual", "minima"], ["proporcao", "minima"]],
        termos_exclusao=["max", "maior", "capitulacao"],
        obrigatoria=False,
        nome_logico="percentual de ativos em mínima de 52 semanas"
    )
if COL_PCT_ABAIXO_MM200 is None:
    COL_PCT_ABAIXO_MM200 = escolher_coluna_por_termos(
        df_amplitude,
        listas_termos=[["pct", "abaixo", "200"], ["percentual", "abaixo", "200"], ["proporcao", "abaixo", "200"]],
        termos_exclusao=["acima", "capitulacao"],
        obrigatoria=False,
        nome_logico="percentual de ativos abaixo da MM200"
    )
if COL_PCT_MAXIMA_52S is None:
    COL_PCT_MAXIMA_52S = escolher_coluna_por_termos(
        df_amplitude,
        listas_termos=[["pct", "maxima"], ["pct", "máxima"], ["percentual", "maxima"], ["proporcao", "maxima"]],
        termos_exclusao=["min", "menor", "capitulacao"],
        obrigatoria=False,
        nome_logico="percentual de ativos em máxima de 52 semanas"
    )
if COL_PCT_ACIMA_MM200 is None:
    COL_PCT_ACIMA_MM200 = escolher_coluna_por_termos(
        df_amplitude,
        listas_termos=[["pct", "acima", "200"], ["percentual", "acima", "200"], ["proporcao", "acima", "200"]],
        termos_exclusao=["abaixo", "capitulacao"],
        obrigatoria=False,
        nome_logico="percentual de ativos acima da MM200"
    )

metricas_centrais_disponiveis = [col for col in [COL_PCT_MAXIMA_52S, COL_PCT_ACIMA_MM200] if col is not None]
if len(metricas_centrais_disponiveis) == 0:
    raise KeyError(
        "Não foi possível identificar métricas centrais de euforia na base 5.5. "
        "É necessário encontrar ao menos percentual de máximas de 52 semanas ou percentual acima da MM200. "
        f"Colunas disponíveis: {list(df_amplitude.columns)}"
    )

df_amplitude[COL_DATA_AMPLITUDE] = para_datetime_seguro(df_amplitude[COL_DATA_AMPLITUDE])
df_datas_brutas_euforia[COL_DATA_BRUTA] = para_datetime_seguro(df_datas_brutas_euforia[COL_DATA_BRUTA])
df_calendario_final_sinais[COL_DATA_FINAL] = para_datetime_seguro(df_calendario_final_sinais[COL_DATA_FINAL])

df_amp = df_amplitude.dropna(subset=[COL_DATA_AMPLITUDE]).copy().sort_values(COL_DATA_AMPLITUDE).reset_index(drop=True)

df_amp["pct_minima_52s_sens"] = normalizar_percentual(df_amp[COL_PCT_MINIMA_52S]) if COL_PCT_MINIMA_52S is not None else 0.0
df_amp["pct_abaixo_mm200_sens"] = normalizar_percentual(df_amp[COL_PCT_ABAIXO_MM200]) if COL_PCT_ABAIXO_MM200 is not None else 0.0
df_amp["pct_maxima_52s_sens"] = normalizar_percentual(df_amp[COL_PCT_MAXIMA_52S]) if COL_PCT_MAXIMA_52S is not None else 0.0
df_amp["pct_acima_mm200_sens"] = normalizar_percentual(df_amp[COL_PCT_ACIMA_MM200]) if COL_PCT_ACIMA_MM200 is not None else 0.0

n_metricas_pro_euforia = int(COL_PCT_MAXIMA_52S is not None) + int(COL_PCT_ACIMA_MM200 is not None)
n_metricas_anti_euforia = int(COL_PCT_MINIMA_52S is not None) + int(COL_PCT_ABAIXO_MM200 is not None)

df_amp["score_euforia_sens"] = (
    df_amp["pct_maxima_52s_sens"] +
    df_amp["pct_acima_mm200_sens"] -
    0.50 * df_amp["pct_minima_52s_sens"] -
    0.50 * df_amp["pct_abaixo_mm200_sens"]
)

df_datas_brutas_ref = (
    df_datas_brutas_euforia[[COL_DATA_BRUTA]]
    .dropna()
    .drop_duplicates()
    .rename(columns={COL_DATA_BRUTA: "data_sinal"})
    .sort_values("data_sinal")
    .reset_index(drop=True)
)

df_sinais_finais_euforia = (
    df_calendario_final_sinais.loc[
        df_calendario_final_sinais[COL_ESTRATEGIA_FINAL].astype(str).str.lower().str.contains("euforia", na=False),
        [COL_DATA_FINAL]
    ]
    .dropna()
    .drop_duplicates()
    .rename(columns={COL_DATA_FINAL: "data_sinal"})
    .sort_values("data_sinal")
    .reset_index(drop=True)
)

print(f"Coluna data amplitude                  : {COL_DATA_AMPLITUDE}")
print(f"Coluna % mínima 52 semanas             : {COL_PCT_MINIMA_52S}")
print(f"Coluna % abaixo MM200                  : {COL_PCT_ABAIXO_MM200}")
print(f"Coluna % máxima 52 semanas             : {COL_PCT_MAXIMA_52S}")
print(f"Coluna % acima MM200                   : {COL_PCT_ACIMA_MM200}")
print(f"Métricas pró-euforia disponíveis   : {n_metricas_pro_euforia}")
print(f"Métricas anti-euforia disponíveis  : {n_metricas_anti_euforia}")
print(f"Datas brutas oficiais de euforia   : {len(df_datas_brutas_ref):,}")
print(f"Datas finais oficiais de euforia   : {len(df_sinais_finais_euforia):,}")
print("OK")

# ============================================================
# 6) Definição dos cenários de limiares e calendário de pregões
# ============================================================

print("\n[6/12] Definição dos cenários de limiares e calendário de pregões...")

COOLDOWN_REFERENCIA = int(globals().get("PARAMETROS_GLOBAIS", {}).get("cooldown_sinais_pregoes", 63))
HORIZONTES_FORWARD = [21, 63, 126, 252]

calendario_pregoes = sorted(df_amp[COL_DATA_AMPLITUDE].dropna().unique())

df_scores_ref = df_amp.merge(df_datas_brutas_ref, left_on=COL_DATA_AMPLITUDE, right_on="data_sinal", how="inner")
if df_scores_ref.empty:
    raise ValueError("Não foi possível cruzar as datas brutas de euforia da 6.2 com a base de amplitude da 5.5.")

limiar_referencia_score = float(df_scores_ref["score_euforia_sens"].min())
quantil_25_score_ref = float(df_scores_ref["score_euforia_sens"].quantile(0.25))
quantil_50_score_ref = float(df_scores_ref["score_euforia_sens"].quantile(0.50))

if pd.isna(limiar_referencia_score):
    raise ValueError("O score de euforia de referência não pôde ser calculado.")

if limiar_referencia_score > 0:
    limiar_muito_flexivel = limiar_referencia_score * 0.75
    limiar_flexivel = limiar_referencia_score * 0.90
else:
    limiar_muito_flexivel = float(df_amp["score_euforia_sens"].quantile(0.85))
    limiar_flexivel = float(df_amp["score_euforia_sens"].quantile(0.90))

limiar_rigoroso = max(limiar_referencia_score, quantil_25_score_ref)
limiar_muito_rigoroso = max(limiar_rigoroso, quantil_50_score_ref)

CENARIOS_LIMIARES_EUFORIA = [
    {
        "cenario_limiar_euforia": "muito_flexivel",
        "ordem_cenario": 1,
        "descricao_cenario": "Limiar 25% abaixo da referência proxy inferida pelo score de euforia.",
        "tipo_cenario": "score_proxy",
        "limiar_score_euforia": limiar_muito_flexivel,
    },
    {
        "cenario_limiar_euforia": "flexivel",
        "ordem_cenario": 2,
        "descricao_cenario": "Limiar 10% abaixo da referência proxy inferida pelo score de euforia.",
        "tipo_cenario": "score_proxy",
        "limiar_score_euforia": limiar_flexivel,
    },
    {
        "cenario_limiar_euforia": "referencia_oficial",
        "ordem_cenario": 3,
        "descricao_cenario": "Datas brutas oficiais de euforia da 6.2, com cooldown oficial de 63 pregões.",
        "tipo_cenario": "referencia_oficial_6_2",
        "limiar_score_euforia": limiar_referencia_score,
    },
    {
        "cenario_limiar_euforia": "rigoroso",
        "ordem_cenario": 4,
        "descricao_cenario": "Limiar mais restritivo usando o percentil 25 do score nas datas brutas oficiais.",
        "tipo_cenario": "score_proxy",
        "limiar_score_euforia": limiar_rigoroso,
    },
    {
        "cenario_limiar_euforia": "muito_rigoroso",
        "ordem_cenario": 5,
        "descricao_cenario": "Limiar mais restritivo usando o percentil 50 do score nas datas brutas oficiais.",
        "tipo_cenario": "score_proxy",
        "limiar_score_euforia": limiar_muito_rigoroso,
    },
]

df_parametros_cenarios = pd.DataFrame(CENARIOS_LIMIARES_EUFORIA)
df_parametros_cenarios["cooldown_pregoes"] = COOLDOWN_REFERENCIA
df_parametros_cenarios["coluna_pct_minima_52s"] = str(COL_PCT_MINIMA_52S)
df_parametros_cenarios["coluna_pct_abaixo_mm200"] = str(COL_PCT_ABAIXO_MM200)
df_parametros_cenarios["coluna_pct_maxima_52s"] = str(COL_PCT_MAXIMA_52S)
df_parametros_cenarios["coluna_pct_acima_mm200"] = str(COL_PCT_ACIMA_MM200)
df_parametros_cenarios["limiar_referencia_score"] = limiar_referencia_score
df_parametros_cenarios["quantil_25_score_datas_brutas_referencia"] = quantil_25_score_ref
df_parametros_cenarios["quantil_50_score_datas_brutas_referencia"] = quantil_50_score_ref

print(f"Cooldown de referência                : {COOLDOWN_REFERENCIA} pregões")
print(f"Cenários de limiar definidos           : {len(CENARIOS_LIMIARES_EUFORIA):,}")
print(f"Horizontes forward para proxy          : {HORIZONTES_FORWARD}")
print(f"Pregões no calendário operacional      : {len(calendario_pregoes):,}")
print(f"Score mínimo das datas brutas oficiais : {limiar_referencia_score:.10f}")
print(f"Score P25 das datas brutas oficiais    : {quantil_25_score_ref:.10f}")
print(f"Score P50 das datas brutas oficiais    : {quantil_50_score_ref:.10f}")
print("\nParâmetros dos cenários:")
print(df_parametros_cenarios.to_string(index=False))
print("OK")

# ============================================================
# 7) Aplicação dos limiares alternativos e cooldown
# ============================================================

print("\n[7/12] Aplicação dos limiares alternativos e cooldown...")

partes_sinais = []
for cenario in CENARIOS_LIMIARES_EUFORIA:
    nome_cenario = cenario["cenario_limiar_euforia"]
    ordem_cenario = cenario["ordem_cenario"]
    limiar_score = cenario["limiar_score_euforia"]
    tipo_cenario = cenario["tipo_cenario"]
    if nome_cenario == "referencia_oficial":
        df_candidatos = df_amp.merge(df_datas_brutas_ref, left_on=COL_DATA_AMPLITUDE, right_on="data_sinal", how="inner").copy()
        df_candidatos["origem_candidato"] = "datas_brutas_oficiais_6_2"
    else:
        df_candidatos = df_amp.loc[df_amp["score_euforia_sens"].ge(limiar_score)].copy()
        df_candidatos["data_sinal"] = df_candidatos[COL_DATA_AMPLITUDE]
        df_candidatos["origem_candidato"] = "score_proxy_euforia"
    datas_aceitas = aplicar_cooldown_datas(df_candidatos["data_sinal"], calendario_pregoes, COOLDOWN_REFERENCIA)
    set_aceitas = set(datas_aceitas)
    df_candidatos = df_candidatos.copy()
    df_candidatos["cenario_limiar_euforia"] = nome_cenario
    df_candidatos["ordem_cenario"] = ordem_cenario
    df_candidatos["tipo_cenario"] = tipo_cenario
    df_candidatos["limiar_score_euforia"] = limiar_score
    df_candidatos["estrategia_referencia"] = "euforia"
    df_candidatos["nome_exibicao_estrategia"] = "Euforia"
    df_candidatos["flag_sinal_bruto_cenario"] = True
    df_candidatos["flag_sinal_pos_cooldown"] = df_candidatos["data_sinal"].isin(set_aceitas)
    df_candidatos["cooldown_pregoes"] = COOLDOWN_REFERENCIA
    colunas_saida = [
        "data_sinal", "estrategia_referencia", "nome_exibicao_estrategia", "cenario_limiar_euforia",
        "ordem_cenario", "tipo_cenario", "limiar_score_euforia", "cooldown_pregoes",
        "flag_sinal_bruto_cenario", "flag_sinal_pos_cooldown", "origem_candidato",
        "score_euforia_sens", "pct_minima_52s_sens", "pct_abaixo_mm200_sens", "pct_maxima_52s_sens", "pct_acima_mm200_sens"
    ]
    partes_sinais.append(df_candidatos[colunas_saida].copy())

if partes_sinais:
    df_sinais_limiares = pd.concat(partes_sinais, ignore_index=True)
else:
    df_sinais_limiares = pd.DataFrame()

df_sinais_limiares = df_sinais_limiares.sort_values(["ordem_cenario", "data_sinal"]).reset_index(drop=True)

duplicatas_cenario_data = int(df_sinais_limiares.duplicated(subset=["cenario_limiar_euforia", "data_sinal"]).sum()) if not df_sinais_limiares.empty else 0
if duplicatas_cenario_data > 0:
    df_sinais_limiares = df_sinais_limiares.drop_duplicates(subset=["cenario_limiar_euforia", "data_sinal"], keep="first").reset_index(drop=True)

print(f"Base de sinais por limiar             : {len(df_sinais_limiares):,} linhas")
print(f"Cenários de limiar gerados             : {df_sinais_limiares['cenario_limiar_euforia'].nunique() if not df_sinais_limiares.empty else 0:,}")
print(f"Duplicatas removidas por cenário/data  : {duplicatas_cenario_data:,}")
print("OK")

# ============================================================
# 8) Construção dos resumos de sensibilidade e comparação com a referência
# ============================================================

print("\n[8/12] Construção dos resumos de sensibilidade e comparação com a referência...")

linhas_resumo = []
for cenario in CENARIOS_LIMIARES_EUFORIA:
    nome_cenario = cenario["cenario_limiar_euforia"]
    ordem_cenario = cenario["ordem_cenario"]
    df_cenario = df_sinais_limiares.loc[df_sinais_limiares["cenario_limiar_euforia"].eq(nome_cenario)].copy()
    df_aceitos = df_cenario.loc[df_cenario["flag_sinal_pos_cooldown"]].copy().sort_values("data_sinal")
    datas_aceitas = list(df_aceitos["data_sinal"])
    intervalos = []
    if len(datas_aceitas) > 1:
        mapa_ordem = {data: i for i, data in enumerate(calendario_pregoes)}
        ordens = [mapa_ordem[data] for data in datas_aceitas if data in mapa_ordem]
        intervalos = list(np.diff(ordens))
    linhas_resumo.append({
        "estrategia_referencia": "euforia",
        "nome_exibicao_estrategia": "Euforia",
        "cenario_limiar_euforia": nome_cenario,
        "ordem_cenario": ordem_cenario,
        "tipo_cenario": cenario["tipo_cenario"],
        "limiar_score_euforia": cenario["limiar_score_euforia"],
        "cooldown_pregoes": COOLDOWN_REFERENCIA,
        "n_sinais_brutos": len(df_cenario),
        "n_sinais_pos_cooldown": len(df_aceitos),
        "n_sinais_rejeitados_cooldown": len(df_cenario) - len(df_aceitos),
        "pct_sinais_aceitos": len(df_aceitos) / len(df_cenario) if len(df_cenario) > 0 else np.nan,
        "primeira_data_sinal": df_aceitos["data_sinal"].min() if len(df_aceitos) > 0 else pd.NaT,
        "ultima_data_sinal": df_aceitos["data_sinal"].max() if len(df_aceitos) > 0 else pd.NaT,
        "score_medio_sinais_aceitos": df_aceitos["score_euforia_sens"].mean() if len(df_aceitos) > 0 else np.nan,
        "score_mediano_sinais_aceitos": df_aceitos["score_euforia_sens"].median() if len(df_aceitos) > 0 else np.nan,
        "intervalo_medio_pregoes": float(np.mean(intervalos)) if intervalos else np.nan,
        "intervalo_mediano_pregoes": float(np.median(intervalos)) if intervalos else np.nan,
        "intervalo_minimo_pregoes": float(np.min(intervalos)) if intervalos else np.nan,
        "intervalo_maximo_pregoes": float(np.max(intervalos)) if intervalos else np.nan,
    })

df_resumo = pd.DataFrame(linhas_resumo).sort_values("ordem_cenario").reset_index(drop=True)

n_referencia_oficial = int(df_sinais_finais_euforia["data_sinal"].nunique())
n_referencia_recalculada = int(df_resumo.loc[df_resumo["cenario_limiar_euforia"].eq("referencia_oficial"), "n_sinais_pos_cooldown"].iloc[0]) if not df_resumo.empty else 0

linhas_comparacao = []
set_ref = set(df_sinais_limiares.loc[
    df_sinais_limiares["cenario_limiar_euforia"].eq("referencia_oficial") & df_sinais_limiares["flag_sinal_pos_cooldown"],
    "data_sinal"
])
for cenario in CENARIOS_LIMIARES_EUFORIA:
    nome_cenario = cenario["cenario_limiar_euforia"]
    set_atual = set(df_sinais_limiares.loc[
        df_sinais_limiares["cenario_limiar_euforia"].eq(nome_cenario) & df_sinais_limiares["flag_sinal_pos_cooldown"],
        "data_sinal"
    ])
    n_comuns = len(set_atual.intersection(set_ref))
    n_uniao = len(set_atual.union(set_ref))
    linhas_comparacao.append({
        "estrategia_referencia": "euforia",
        "nome_exibicao_estrategia": "Euforia",
        "cenario_limiar_euforia": nome_cenario,
        "ordem_cenario": cenario["ordem_cenario"],
        "n_sinais_cenario": len(set_atual),
        "n_sinais_referencia": len(set_ref),
        "n_sinais_comuns_com_referencia": n_comuns,
        "n_sinais_adicionados_vs_referencia": len(set_atual - set_ref),
        "n_sinais_removidos_vs_referencia": len(set_ref - set_atual),
        "delta_liquido_sinais_vs_referencia": len(set_atual) - len(set_ref),
        "pct_jaccard_vs_referencia": n_comuns / n_uniao if n_uniao > 0 else np.nan,
    })

df_comparacao_referencia = pd.DataFrame(linhas_comparacao).sort_values("ordem_cenario").reset_index(drop=True)

if not df_sinais_limiares.empty:
    df_impacto_temporal = (
        df_sinais_limiares.loc[df_sinais_limiares["flag_sinal_pos_cooldown"]]
        .assign(ano=lambda x: x["data_sinal"].dt.year)
        .groupby(["cenario_limiar_euforia", "ordem_cenario", "ano"], as_index=False)
        .agg(
            n_sinais=("data_sinal", "nunique"),
            score_medio=("score_euforia_sens", "mean"),
            score_mediano=("score_euforia_sens", "median"),
        )
        .sort_values(["ordem_cenario", "ano"])
        .reset_index(drop=True)
    )
else:
    df_impacto_temporal = pd.DataFrame()

print(f"Resumo de sensibilidade              : {len(df_resumo):,} linhas")
print(f"Comparação com referência             : {len(df_comparacao_referencia):,} linhas")
print(f"Impacto temporal anual                : {len(df_impacto_temporal):,} linhas")
print("OK")

# ============================================================
# 9) Cálculo de performance proxy pós-sinal com Ibovespa
# ============================================================

print("\n[9/12] Cálculo de performance proxy pós-sinal com Ibovespa...")

df_benchmark, COL_DATA_BENCHMARK, COL_RETORNO_BENCHMARK, ORIGEM_BENCHMARK = identificar_serie_benchmark_retornos(df_retornos_diarios)
BENCHMARK_DISPONIVEL = not df_benchmark.empty and COL_DATA_BENCHMARK is not None and COL_RETORNO_BENCHMARK is not None

linhas_proxy = []
if BENCHMARK_DISPONIVEL:
    for _, linha_sinal in df_sinais_limiares.loc[df_sinais_limiares["flag_sinal_pos_cooldown"]].iterrows():
        for horizonte in HORIZONTES_FORWARD:
            retorno_forward = calcular_retorno_forward(
                df_serie_retorno=df_benchmark,
                data_sinal=linha_sinal["data_sinal"],
                horizonte_pregoes=horizonte,
                coluna_data=COL_DATA_BENCHMARK,
                coluna_retorno=COL_RETORNO_BENCHMARK,
            )
            linhas_proxy.append({
                "data_sinal": linha_sinal["data_sinal"],
                "estrategia_referencia": "euforia",
                "nome_exibicao_estrategia": "Euforia",
                "cenario_limiar_euforia": linha_sinal["cenario_limiar_euforia"],
                "ordem_cenario": linha_sinal["ordem_cenario"],
                "horizonte_pregoes": horizonte,
                "retorno_ibovespa_pos_sinal": retorno_forward,
                "score_euforia_sens": linha_sinal["score_euforia_sens"],
                "origem_benchmark_proxy": ORIGEM_BENCHMARK,
            })

df_base_proxy = pd.DataFrame(linhas_proxy)
if not df_base_proxy.empty:
    df_tbl_proxy = (
        df_base_proxy.dropna(subset=["retorno_ibovespa_pos_sinal"])
        .groupby(["estrategia_referencia", "nome_exibicao_estrategia", "cenario_limiar_euforia", "ordem_cenario", "horizonte_pregoes"], as_index=False)
        .agg(
            n_sinais_com_retorno=("data_sinal", "nunique"),
            retorno_medio_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "mean"),
            retorno_mediano_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "median"),
            retorno_minimo_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "min"),
            retorno_maximo_ibovespa_pos_sinal=("retorno_ibovespa_pos_sinal", "max"),
            pct_sinais_retorno_positivo=("retorno_ibovespa_pos_sinal", lambda x: float((x > 0).mean()) if len(x) > 0 else np.nan),
            score_medio_sinais=("score_euforia_sens", "mean"),
        )
        .sort_values(["ordem_cenario", "horizonte_pregoes"])
        .reset_index(drop=True)
    )
else:
    df_tbl_proxy = pd.DataFrame(columns=[
        "estrategia_referencia", "nome_exibicao_estrategia", "cenario_limiar_euforia", "ordem_cenario",
        "horizonte_pregoes", "n_sinais_com_retorno", "retorno_medio_ibovespa_pos_sinal",
        "retorno_mediano_ibovespa_pos_sinal", "retorno_minimo_ibovespa_pos_sinal",
        "retorno_maximo_ibovespa_pos_sinal", "pct_sinais_retorno_positivo", "score_medio_sinais"
    ])

print(f"Benchmark proxy identificado           : {ORIGEM_BENCHMARK}")
print(f"Benchmark proxy disponível             : {BENCHMARK_DISPONIVEL}")
print(f"Base de performance proxy              : {len(df_base_proxy):,} linhas")
print(f"Tabela de performance proxy            : {len(df_tbl_proxy):,} linhas")
print("OK")

# ============================================================
# 10) Construção da base de gráficos e auditoria
# ============================================================

print("\n[10/12] Construção da base de gráficos e auditoria...")

base_grafico_sinais = df_resumo[[
    "cenario_limiar_euforia", "ordem_cenario", "n_sinais_brutos", "n_sinais_pos_cooldown",
    "limiar_score_euforia", "score_medio_sinais_aceitos", "score_mediano_sinais_aceitos"
]].copy()
base_grafico_sinais["tipo_grafico"] = "resumo_sinais_limiar_euforia"
base_grafico_sinais["metrica_principal"] = "n_sinais_pos_cooldown"
base_grafico_sinais["valor_principal"] = base_grafico_sinais["n_sinais_pos_cooldown"]

base_grafico_temporal = df_impacto_temporal.copy()
if not base_grafico_temporal.empty:
    base_grafico_temporal["tipo_grafico"] = "sinais_anuais_limiar_euforia"
    base_grafico_temporal["metrica_principal"] = "n_sinais"
    base_grafico_temporal["valor_principal"] = base_grafico_temporal["n_sinais"]
else:
    base_grafico_temporal = pd.DataFrame(columns=base_grafico_sinais.columns)

for coluna in base_grafico_sinais.columns:
    if coluna not in base_grafico_temporal.columns:
        base_grafico_temporal[coluna] = np.nan
for coluna in base_grafico_temporal.columns:
    if coluna not in base_grafico_sinais.columns:
        base_grafico_sinais[coluna] = np.nan

df_base_grafico = pd.concat([
    base_grafico_sinais[sorted(base_grafico_sinais.columns)],
    base_grafico_temporal[sorted(base_grafico_sinais.columns)]
], ignore_index=True)

sinais_sem_data = int(df_sinais_limiares["data_sinal"].isna().sum()) if not df_sinais_limiares.empty else 0
sinais_referencia_diferem_oficial = int(abs(n_referencia_recalculada - n_referencia_oficial))
cenarios_sem_sinal = int((df_resumo["n_sinais_pos_cooldown"] == 0).sum()) if not df_resumo.empty else len(CENARIOS_LIMIARES_EUFORIA)
metricas_infinitas = 0
frames_validacao_numerica = [df_resumo, df_comparacao_referencia, df_impacto_temporal, df_tbl_proxy]
for frame in frames_validacao_numerica:
    if not frame.empty:
        arr = frame.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
        metricas_infinitas += int(np.isinf(arr).sum())

linhas_auditoria = [
    {
        "item": "metricas_pro_euforia_identificadas",
        "valor": n_metricas_pro_euforia,
        "valor_referencia": ">= 1",
        "status": "OK" if n_metricas_pro_euforia >= 1 else "ERRO",
        "observacao": "A sensibilidade exige ao menos uma métrica central de euforia identificada na base 5.5.",
    },
    {
        "item": "datas_brutas_euforia_6_2",
        "valor": len(df_datas_brutas_ref),
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Quantidade de candidatos brutos oficiais antes do cooldown para Euforia.",
    },
    {
        "item": "sinais_finais_euforia_6_4",
        "valor": n_referencia_oficial,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Quantidade de sinais finais oficiais de Euforia no calendário da 6.4.",
    },
    {
        "item": "cenarios_limiares_gerados",
        "valor": len(CENARIOS_LIMIARES_EUFORIA),
        "valor_referencia": 5,
        "status": "OK" if len(CENARIOS_LIMIARES_EUFORIA) == 5 else "ERRO",
        "observacao": "A subetapa deve gerar cenários flexíveis, referência e rigorosos.",
    },
    {
        "item": "cenarios_sem_sinal_pos_cooldown",
        "valor": cenarios_sem_sinal,
        "valor_referencia": 0,
        "status": "OK" if cenarios_sem_sinal == 0 else "ERRO",
        "observacao": "Cenários alternativos de limiar não devem ficar sem sinais após cooldown.",
    },
    {
        "item": "sinais_sem_data",
        "valor": sinais_sem_data,
        "valor_referencia": 0,
        "status": "OK" if sinais_sem_data == 0 else "ERRO",
        "observacao": "Todos os sinais aceitos por cenário de limiar devem possuir data válida.",
    },
    {
        "item": "duplicatas_cenario_data",
        "valor": duplicatas_cenario_data,
        "valor_referencia": 0,
        "status": "OK" if duplicatas_cenario_data == 0 else "ERRO",
        "observacao": "Cada combinação de cenário e data deve aparecer uma única vez após tratamento.",
    },
    {
        "item": "sinais_referencia_diferem_oficial",
        "valor": sinais_referencia_diferem_oficial,
        "valor_referencia": 0,
        "status": "OK" if sinais_referencia_diferem_oficial == 0 else "ERRO",
        "observacao": "Compara a quantidade recalculada no cenário de referência com o calendário oficial da 6.4.",
    },
    {
        "item": "benchmark_proxy_disponivel",
        "valor": int(BENCHMARK_DISPONIVEL),
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Indica se foi possível calcular retorno forward do Ibovespa como proxy de timing pós-sinal.",
    },
    {
        "item": "origem_benchmark_proxy",
        "valor": ORIGEM_BENCHMARK,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Origem da série usada no cálculo de retorno forward pós-sinal.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": 0,
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
n_erros_bloqueantes = int(df_auditoria["status"].eq("ERRO").sum())

print(f"Base gráfico limiares euforia      : {len(df_base_grafico):,} linhas")
print(f"Itens de auditoria                     : {len(df_auditoria):,}")
print(f"Erros bloqueantes                      : {n_erros_bloqueantes:,}")
print("OK")

# ============================================================
# 11) Salvamento dos outputs
# ============================================================

print("\n[11/12] Salvamento dos outputs...")

salvar_dataframe(df_sinais_limiares, CAMINHO_BASE_SINAIS, index=False)
salvar_dataframe(df_parametros_cenarios, CAMINHO_PARAMETROS_CENARIOS, index=False)
salvar_dataframe(df_resumo, CAMINHO_RESUMO, index=False)
salvar_dataframe(df_comparacao_referencia, CAMINHO_COMPARACAO_REFERENCIA, index=False)
salvar_dataframe(df_impacto_temporal, CAMINHO_IMPACTO_TEMPORAL, index=False)
salvar_dataframe(df_base_proxy, CAMINHO_BASE_PROXY, index=False)
salvar_dataframe(df_tbl_proxy, CAMINHO_TBL_PROXY, index=False)
salvar_dataframe(df_base_grafico, CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 12) Validação final da subetapa
# ============================================================

print("\n[12/12] Validação final da subetapa...")

print("\nAuditoria de validação da sensibilidade dos limiares de euforia:")
print(df_auditoria.to_string(index=False))

print("\nResumo de sensibilidade por cenário:")
print(df_resumo.to_string(index=False))

print("\nComparação com a referência oficial:")
print(df_comparacao_referencia.to_string(index=False))

print("\nImpacto temporal anual - amostra:")
print(df_impacto_temporal.head(40).to_string(index=False))

print("\nPerformance proxy pós-sinal - amostra:")
print(df_tbl_proxy.head(40).to_string(index=False))

print("\nArquivos salvos na subetapa 13.5:")
print(f"- {CAMINHO_BASE_SINAIS}")
print(f"- {CAMINHO_PARAMETROS_CENARIOS}")
print(f"- {CAMINHO_RESUMO}")
print(f"- {CAMINHO_COMPARACAO_REFERENCIA}")
print(f"- {CAMINHO_IMPACTO_TEMPORAL}")
print(f"- {CAMINHO_BASE_PROXY}")
print(f"- {CAMINHO_TBL_PROXY}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_AUDITORIA}")

if n_erros_bloqueantes > 0:
    raise RuntimeError("A subetapa 13.5 encontrou erros bloqueantes na auditoria. Verifique a tabela de validação.")

print("\nETAPA 13.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 13.5 - SENSIBILIDADE DOS LIMIARES DE EUFORIA

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - amplitude agregada da 5.5          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_5\5_5_base_indicadores_agregados_amplitude_pregao.parquet
Entrada - parâmetros euforia da 6.2      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_2_tbl_parametros_euforia.parquet
Entrada - limiares euforia da 6.2        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_6\6_2_tbl_limiares_euforia_dinamicos.parquet
Entrada - base bruta euforia da 6.2      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa

## Etapa 13.6) Sensibilidade das Estratégias Aleatórias

In [69]:
%%time
# ============================================================
# Etapa 13.6) Sensibilidade das Estratégias Aleatórias
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 13.6 - SENSIBILIDADE DAS ESTRATÉGIAS ALEATÓRIAS")
print("=" * 100)

# Esta subetapa não reexecuta o backtest nem altera limiares dos sinais.
# O objetivo é medir a estabilidade das conclusões frente às réplicas aleatórias já geradas e validadas.
# A análise preserva as visões diária, mensal e anual produzidas na etapa 11.6.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "re" not in globals():
    raise NameError("A biblioteca re deve estar importada no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_7 = Path(DIRETORIOS_PROJETO["etapa_7"])
DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])
DIR_ETAPA_13 = Path(DIRETORIOS_PROJETO["etapa_13"])
DIR_ETAPA_13.mkdir(parents=True, exist_ok=True)

CAMINHO_CALENDARIOS_ALEATORIOS_7_5 = DIR_ETAPA_7 / "7_5_base_calendarios_aleatorios_controle.parquet"
CAMINHO_RESUMO_CALENDARIOS_ALEATORIOS_7_5 = DIR_ETAPA_7 / "7_5_tbl_resumo_calendarios_aleatorios_controle.parquet"
CAMINHO_DISTRIBUICAO_ANUAL_ALEATORIOS_7_5 = DIR_ETAPA_7 / "7_5_tbl_distribuicao_anual_calendarios_aleatorios_controle.parquet"
CAMINHO_PARES_COMPARACAO_CONTROLES_11_6 = DIR_ETAPA_11 / "11_6_tbl_pares_comparacao_controles.parquet"
CAMINHO_TBL_CONTROLES_DIARIO_11_6 = DIR_ETAPA_11 / "11_6_tbl_comparacao_controles_diario.parquet"
CAMINHO_TBL_CONTROLES_MENSAL_11_6 = DIR_ETAPA_11 / "11_6_tbl_comparacao_controles_mensal.parquet"
CAMINHO_TBL_CONTROLES_ANUAL_11_6 = DIR_ETAPA_11 / "11_6_tbl_comparacao_controles_anual.parquet"
CAMINHO_POSICIONAMENTO_11_6 = DIR_ETAPA_11 / "11_6_tbl_posicionamento_relativo_multifrequencia.parquet"
CAMINHO_RANKING_11_6 = DIR_ETAPA_11 / "11_6_tbl_ranking_estrategias_reais_vs_controles.parquet"

CAMINHO_BASE_ALEATORIAS = DIR_ETAPA_13 / "13_6_base_sensibilidade_estrategias_aleatorias.parquet"
CAMINHO_PARAMETROS_CENARIOS = DIR_ETAPA_13 / "13_6_tbl_parametros_cenarios_estrategias_aleatorias.parquet"
CAMINHO_RESUMO_REPLICAS = DIR_ETAPA_13 / "13_6_tbl_resumo_sensibilidade_replicas_aleatorias.parquet"
CAMINHO_DISTRIBUICAO_REPLICAS = DIR_ETAPA_13 / "13_6_tbl_distribuicao_metricas_replicas_aleatorias.parquet"
CAMINHO_LEAVE_ONE_OUT = DIR_ETAPA_13 / "13_6_tbl_estabilidade_leave_one_out_aleatorias.parquet"
CAMINHO_POSICIONAMENTO_FINAL = DIR_ETAPA_13 / "13_6_tbl_posicionamento_final_estrategias_reais_vs_aleatorias.parquet"
CAMINHO_IMPACTO_TEMPORAL = DIR_ETAPA_13 / "13_6_tbl_impacto_temporal_replicas_aleatorias.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_13 / "13_6_tbl_base_grafico_sensibilidade_aleatorias.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_13 / "13_6_tbl_auditoria_validacao_sensibilidade_aleatorias.parquet"

CAMINHOS_ENTRADA_OBRIGATORIOS = [
    CAMINHO_TBL_CONTROLES_DIARIO_11_6,
    CAMINHO_TBL_CONTROLES_MENSAL_11_6,
    CAMINHO_TBL_CONTROLES_ANUAL_11_6,
]

for caminho in CAMINHOS_ENTRADA_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - calendários aleatórios da 7.5       : {CAMINHO_CALENDARIOS_ALEATORIOS_7_5}")
print(f"Entrada - resumo aleatórios da 7.5            : {CAMINHO_RESUMO_CALENDARIOS_ALEATORIOS_7_5}")
print(f"Entrada - distribuição anual aleatórios da 7.5: {CAMINHO_DISTRIBUICAO_ANUAL_ALEATORIOS_7_5}")
print(f"Entrada - pares de comparação da 11.6         : {CAMINHO_PARES_COMPARACAO_CONTROLES_11_6}")
print(f"Entrada - comparação controles diária da 11.6 : {CAMINHO_TBL_CONTROLES_DIARIO_11_6}")
print(f"Entrada - comparação controles mensal da 11.6 : {CAMINHO_TBL_CONTROLES_MENSAL_11_6}")
print(f"Entrada - comparação controles anual da 11.6  : {CAMINHO_TBL_CONTROLES_ANUAL_11_6}")
print(f"Entrada - posicionamento relativo da 11.6     : {CAMINHO_POSICIONAMENTO_11_6}")
print(f"Entrada - ranking final da 11.6               : {CAMINHO_RANKING_11_6}")
print(f"Saída   - base sensibilidade aleatórias       : {CAMINHO_BASE_ALEATORIAS}")
print(f"Saída   - parâmetros dos cenários             : {CAMINHO_PARAMETROS_CENARIOS}")
print(f"Saída   - resumo por número de réplicas       : {CAMINHO_RESUMO_REPLICAS}")
print(f"Saída   - distribuição por réplica            : {CAMINHO_DISTRIBUICAO_REPLICAS}")
print(f"Saída   - estabilidade leave-one-out          : {CAMINHO_LEAVE_ONE_OUT}")
print(f"Saída   - posicionamento final                : {CAMINHO_POSICIONAMENTO_FINAL}")
print(f"Saída   - impacto temporal                    : {CAMINHO_IMPACTO_TEMPORAL}")
print(f"Saída   - base gráfico                        : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação              : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/12] Carga das bases oficiais da subetapa...")

df_controles_diario = pd.read_parquet(CAMINHO_TBL_CONTROLES_DIARIO_11_6)
df_controles_mensal = pd.read_parquet(CAMINHO_TBL_CONTROLES_MENSAL_11_6)
df_controles_anual = pd.read_parquet(CAMINHO_TBL_CONTROLES_ANUAL_11_6)

if Path(CAMINHO_CALENDARIOS_ALEATORIOS_7_5).exists():
    df_calendarios_aleatorios = pd.read_parquet(CAMINHO_CALENDARIOS_ALEATORIOS_7_5)
else:
    df_calendarios_aleatorios = pd.DataFrame()

if Path(CAMINHO_RESUMO_CALENDARIOS_ALEATORIOS_7_5).exists():
    df_resumo_calendarios_aleatorios = pd.read_parquet(CAMINHO_RESUMO_CALENDARIOS_ALEATORIOS_7_5)
else:
    df_resumo_calendarios_aleatorios = pd.DataFrame()

if Path(CAMINHO_DISTRIBUICAO_ANUAL_ALEATORIOS_7_5).exists():
    df_distribuicao_anual_aleatorios = pd.read_parquet(CAMINHO_DISTRIBUICAO_ANUAL_ALEATORIOS_7_5)
else:
    df_distribuicao_anual_aleatorios = pd.DataFrame()

if Path(CAMINHO_PARES_COMPARACAO_CONTROLES_11_6).exists():
    df_pares_controles = pd.read_parquet(CAMINHO_PARES_COMPARACAO_CONTROLES_11_6)
else:
    df_pares_controles = pd.DataFrame()

if Path(CAMINHO_POSICIONAMENTO_11_6).exists():
    df_posicionamento_11_6 = pd.read_parquet(CAMINHO_POSICIONAMENTO_11_6)
else:
    df_posicionamento_11_6 = pd.DataFrame()

if Path(CAMINHO_RANKING_11_6).exists():
    df_ranking_11_6 = pd.read_parquet(CAMINHO_RANKING_11_6)
else:
    df_ranking_11_6 = pd.DataFrame()

print(f"Comparação controles diária da 11.6 : {len(df_controles_diario):,} linhas x {df_controles_diario.shape[1]:,} colunas")
print(f"Comparação controles mensal da 11.6 : {len(df_controles_mensal):,} linhas x {df_controles_mensal.shape[1]:,} colunas")
print(f"Comparação controles anual da 11.6  : {len(df_controles_anual):,} linhas x {df_controles_anual.shape[1]:,} colunas")
print(f"Calendários aleatórios da 7.5       : {len(df_calendarios_aleatorios):,} linhas x {df_calendarios_aleatorios.shape[1]:,} colunas")
print(f"Resumo calendários aleatórios da 7.5: {len(df_resumo_calendarios_aleatorios):,} linhas x {df_resumo_calendarios_aleatorios.shape[1]:,} colunas")
print(f"Pares de comparação da 11.6         : {len(df_pares_controles):,} linhas x {df_pares_controles.shape[1]:,} colunas")
print(f"Posicionamento relativo da 11.6     : {len(df_posicionamento_11_6):,} linhas x {df_posicionamento_11_6.shape[1]:,} colunas")
print(f"Ranking final da 11.6               : {len(df_ranking_11_6):,} linhas x {df_ranking_11_6.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

def escolher_coluna(df, candidatos, obrigatoria=False, nome_logico="coluna"):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar {nome_logico}. "
            f"Candidatas avaliadas: {candidatos}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

def para_numero_seguro(serie):
    return pd.to_numeric(serie, errors="coerce")

def para_datetime_seguro(serie):
    return pd.to_datetime(serie, errors="coerce").dt.normalize()

def para_booleano_seguro(serie):
    if serie.dtype == bool:
        return serie.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(serie):
        return serie.fillna(0).astype(float).ne(0)
    serie_texto = serie.astype(str).str.strip().str.lower()
    valores_verdadeiros = {"1", "true", "t", "sim", "s", "yes", "y", "ok", "v", "verdadeiro"}
    return serie_texto.isin(valores_verdadeiros)

def normalizar_texto_serie(serie):
    return serie.astype(str).str.strip().str.lower()

def extrair_numero_replica(linha):
    textos = []
    for coluna in [
        "nome_exibicao_estrategia_controle",
        "estrategia_id_controle",
        "chave_estrategia_controle",
        "comparacao_id",
    ]:
        if coluna in linha.index:
            textos.append(str(linha.get(coluna, "")))
    texto = " ".join(textos).lower()
    padroes = [
        r"#\s*(\d+)",
        r"replica[_\s-]*(\d+)",
        r"réplica[_\s-]*(\d+)",
        r"aleatoria[_\s-]*(\d+)",
        r"aleatória[_\s-]*(\d+)",
        r"random[_\s-]*(\d+)",
    ]
    for padrao in padroes:
        match = re.search(padrao, texto)
        if match:
            return int(match.group(1))
    numeros = re.findall(r"\d+", texto)
    if numeros:
        return int(numeros[-1])
    return np.nan

def classificar_robustez(pct_superacao, excesso_medio):
    if pd.isna(pct_superacao) or pd.isna(excesso_medio):
        return "indefinida"
    if pct_superacao >= 0.80 and excesso_medio > 0:
        return "favoravel_forte"
    if pct_superacao >= 0.60 and excesso_medio > 0:
        return "favoravel_moderada"
    if pct_superacao <= 0.40 and excesso_medio < 0:
        return "desfavoravel"
    return "neutra"

def produto_retorno_acumulado(serie_retornos):
    retornos = para_numero_seguro(serie_retornos).dropna()
    if len(retornos) == 0:
        return np.nan
    return float((1.0 + retornos).prod() - 1.0)

def resumo_subconjunto(grupo, contexto):
    n_pares = int(len(grupo))
    if "COL_CHAVE_CONTROLE" in globals() and COL_CHAVE_CONTROLE in grupo.columns:
        n_replicas = int(grupo[COL_CHAVE_CONTROLE].astype(str).nunique())
    elif "replica_aleatoria_numero" in grupo.columns:
        n_replicas = int(grupo["replica_aleatoria_numero"].nunique())
    else:
        n_replicas = n_pares
    n_sementes = int(grupo["semente_aleatoria_numero"].nunique()) if "semente_aleatoria_numero" in grupo.columns else np.nan
    registro = dict(contexto)
    registro["n_replicas_aleatorias"] = n_replicas
    registro["n_sementes_aleatorias_distintas"] = n_sementes
    registro["n_pares_comparacao"] = n_pares
    registro["n_periodos_comparaveis_total"] = int(para_numero_seguro(grupo.get("n_periodos_comparaveis", pd.Series(dtype=float))).sum()) if "n_periodos_comparaveis" in grupo.columns else np.nan
    registro["pct_replicas_retorno_acumulado_real_superou_controle"] = float(para_booleano_seguro(grupo["flag_retorno_acumulado_real_superou_controle"]).mean()) if "flag_retorno_acumulado_real_superou_controle" in grupo.columns and n_pares > 0 else np.nan
    registro["pct_replicas_retorno_anualizado_real_superou_controle"] = float(para_booleano_seguro(grupo["flag_retorno_anualizado_real_superou_controle"]).mean()) if "flag_retorno_anualizado_real_superou_controle" in grupo.columns and n_pares > 0 else np.nan
    registro["pct_replicas_sharpe_real_superou_controle"] = float(para_booleano_seguro(grupo["flag_sharpe_real_superou_controle"]).mean()) if "flag_sharpe_real_superou_controle" in grupo.columns and n_pares > 0 else np.nan
    registro["pct_replicas_sortino_real_superou_controle"] = float(para_booleano_seguro(grupo["flag_sortino_real_superou_controle"]).mean()) if "flag_sortino_real_superou_controle" in grupo.columns and n_pares > 0 else np.nan
    registro["pct_replicas_calmar_real_superou_controle"] = float(para_booleano_seguro(grupo["flag_calmar_real_superou_controle"]).mean()) if "flag_calmar_real_superou_controle" in grupo.columns and n_pares > 0 else np.nan
    registro["pct_replicas_volatilidade_real_menor_controle"] = float(para_booleano_seguro(grupo["flag_volatilidade_real_menor_controle"]).mean()) if "flag_volatilidade_real_menor_controle" in grupo.columns and n_pares > 0 else np.nan
    registro["pct_replicas_drawdown_abs_real_menor_controle"] = float(para_booleano_seguro(grupo["flag_drawdown_abs_real_menor_controle"]).mean()) if "flag_drawdown_abs_real_menor_controle" in grupo.columns and n_pares > 0 else np.nan

    metricas_numericas = {
        "pct_periodos_real_superou_controle": "pct_periodos_real_superou_controle",
        "excesso_medio_periodo_real_vs_controle": "excesso_medio_periodo_real_vs_controle",
        "excesso_mediano_periodo_real_vs_controle": "excesso_mediano_periodo_real_vs_controle",
        "retorno_acumulado_relativo_composto_real_vs_controle": "retorno_acumulado_relativo_composto_real_vs_controle",
        "excesso_anualizado_real_vs_controle_periodico": "excesso_anualizado_real_vs_controle_periodico",
        "dif_sharpe_anualizado_real_vs_controle": "dif_sharpe_anualizado_real_vs_controle",
        "dif_sortino_anualizado_real_vs_controle": "dif_sortino_anualizado_real_vs_controle",
        "dif_calmar_real_vs_controle": "dif_calmar_real_vs_controle",
        "dif_volatilidade_anualizada_equivalente_real_vs_controle": "dif_volatilidade_anualizada_equivalente_real_vs_controle",
        "dif_max_drawdown_abs_real_vs_controle": "dif_max_drawdown_abs_real_vs_controle",
    }

    for nome_saida, coluna in metricas_numericas.items():
        if coluna in grupo.columns:
            valores = para_numero_seguro(grupo[coluna]).dropna()
            registro[f"media_{nome_saida}"] = float(valores.mean()) if len(valores) > 0 else np.nan
            registro[f"mediana_{nome_saida}"] = float(valores.median()) if len(valores) > 0 else np.nan
            registro[f"min_{nome_saida}"] = float(valores.min()) if len(valores) > 0 else np.nan
            registro[f"max_{nome_saida}"] = float(valores.max()) if len(valores) > 0 else np.nan
        else:
            registro[f"media_{nome_saida}"] = np.nan
            registro[f"mediana_{nome_saida}"] = np.nan
            registro[f"min_{nome_saida}"] = np.nan
            registro[f"max_{nome_saida}"] = np.nan

    registro["classificacao_robustez_retorno_acumulado"] = classificar_robustez(
        registro.get("pct_replicas_retorno_acumulado_real_superou_controle", np.nan),
        registro.get("media_excesso_anualizado_real_vs_controle_periodico", np.nan),
    )
    registro["classificacao_robustez_periodos"] = classificar_robustez(
        registro.get("media_pct_periodos_real_superou_controle", np.nan),
        registro.get("media_excesso_medio_periodo_real_vs_controle", np.nan),
    )
    return registro

def preparar_auditoria(linhas):
    df_aud = pd.DataFrame(linhas)
    for coluna in ["item", "valor", "valor_referencia", "status", "observacao"]:
        if coluna not in df_aud.columns:
            df_aud[coluna] = ""
        df_aud[coluna] = df_aud[coluna].astype(str)
    return df_aud

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização da base de comparações aleatórias
# ============================================================

print("\n[5/12] Padronização da base de comparações aleatórias...")

def empilhar_base_controle(df, frequencia):
    df_saida = df.copy()
    if "frequencia" not in df_saida.columns:
        df_saida["frequencia"] = frequencia
    else:
        df_saida["frequencia"] = df_saida["frequencia"].fillna(frequencia).astype(str)
    return df_saida

df_controles_multifrequencia = pd.concat(
    [
        empilhar_base_controle(df_controles_diario, "diaria"),
        empilhar_base_controle(df_controles_mensal, "mensal"),
        empilhar_base_controle(df_controles_anual, "anual"),
    ],
    ignore_index=True,
    sort=False,
)

COL_ESTRATEGIA_REFERENCIA = escolher_coluna(
    df_controles_multifrequencia,
    ["estrategia_referencia", "estrategia", "tipo_sinal"],
    obrigatoria=True,
    nome_logico="estratégia de referência na base 11.6",
)
COL_GRUPO_CONTROLE = escolher_coluna(
    df_controles_multifrequencia,
    ["grupo_comparacao_controle", "grupo_controle", "tipo_controle"],
    obrigatoria=False,
    nome_logico="grupo de comparação do controle",
)
COL_TIPO_CONTROLE = escolher_coluna(
    df_controles_multifrequencia,
    ["tipo_comparacao_controle", "tipo_controle", "subtipo_controle"],
    obrigatoria=False,
    nome_logico="tipo de comparação do controle",
)
COL_COMPARACAO_ID = escolher_coluna(
    df_controles_multifrequencia,
    ["comparacao_id", "id_comparacao", "par_comparacao"],
    obrigatoria=False,
    nome_logico="comparação id",
)
COL_CHAVE_CONTROLE = escolher_coluna(
    df_controles_multifrequencia,
    ["chave_estrategia_controle", "estrategia_id_controle", "chave_controle", "id_controle"],
    obrigatoria=False,
    nome_logico="chave da estratégia de controle",
)
COL_NOME_CONTROLE = escolher_coluna(
    df_controles_multifrequencia,
    ["nome_exibicao_estrategia_controle", "nome_estrategia_controle", "nome_controle"],
    obrigatoria=False,
    nome_logico="nome da estratégia de controle",
)
COL_CHAVE_REAL = escolher_coluna(
    df_controles_multifrequencia,
    ["chave_estrategia_real", "estrategia_id_real", "chave_real", "id_real"],
    obrigatoria=False,
    nome_logico="chave da estratégia real",
)
COL_NOME_REAL = escolher_coluna(
    df_controles_multifrequencia,
    ["nome_exibicao_estrategia_real", "nome_estrategia_real", "nome_real"],
    obrigatoria=False,
    nome_logico="nome da estratégia real",
)

if COL_COMPARACAO_ID is None:
    df_controles_multifrequencia["comparacao_id"] = (
        df_controles_multifrequencia[COL_ESTRATEGIA_REFERENCIA].astype(str) + "__" +
        df_controles_multifrequencia.get(COL_NOME_CONTROLE, pd.Series("controle", index=df_controles_multifrequencia.index)).astype(str) + "__" +
        df_controles_multifrequencia["frequencia"].astype(str)
    )
    COL_COMPARACAO_ID = "comparacao_id"

if COL_CHAVE_CONTROLE is None:
    df_controles_multifrequencia["chave_estrategia_controle"] = df_controles_multifrequencia[COL_COMPARACAO_ID].astype(str)
    COL_CHAVE_CONTROLE = "chave_estrategia_controle"

if COL_NOME_CONTROLE is None:
    df_controles_multifrequencia["nome_exibicao_estrategia_controle"] = df_controles_multifrequencia[COL_CHAVE_CONTROLE].astype(str)
    COL_NOME_CONTROLE = "nome_exibicao_estrategia_controle"

if COL_CHAVE_REAL is None:
    df_controles_multifrequencia["chave_estrategia_real"] = df_controles_multifrequencia[COL_ESTRATEGIA_REFERENCIA].astype(str)
    COL_CHAVE_REAL = "chave_estrategia_real"

if COL_NOME_REAL is None:
    df_controles_multifrequencia["nome_exibicao_estrategia_real"] = df_controles_multifrequencia[COL_ESTRATEGIA_REFERENCIA].astype(str)
    COL_NOME_REAL = "nome_exibicao_estrategia_real"

texto_filtro_aleatorio = pd.Series("", index=df_controles_multifrequencia.index, dtype="object")
for coluna in [COL_GRUPO_CONTROLE, COL_TIPO_CONTROLE, COL_NOME_CONTROLE, COL_CHAVE_CONTROLE, COL_COMPARACAO_ID]:
    if coluna is not None and coluna in df_controles_multifrequencia.columns:
        texto_filtro_aleatorio = texto_filtro_aleatorio + " " + df_controles_multifrequencia[coluna].astype(str)

flag_aleatorio = normalizar_texto_serie(texto_filtro_aleatorio).str.contains("aleat|random", regex=True, na=False)
df_base_aleatorias = df_controles_multifrequencia.loc[flag_aleatorio].copy()

for coluna in [COL_ESTRATEGIA_REFERENCIA, COL_CHAVE_CONTROLE, COL_NOME_CONTROLE, COL_CHAVE_REAL, COL_NOME_REAL, "frequencia"]:
    if coluna in df_base_aleatorias.columns:
        df_base_aleatorias[coluna] = df_base_aleatorias[coluna].astype(str).str.strip()

df_base_aleatorias["semente_aleatoria_numero"] = df_base_aleatorias.apply(extrair_numero_replica, axis=1)
df_base_aleatorias["semente_aleatoria_numero"] = pd.to_numeric(df_base_aleatorias["semente_aleatoria_numero"], errors="coerce")
df_base_aleatorias["replica_aleatoria_id"] = df_base_aleatorias[COL_CHAVE_CONTROLE].astype(str)

def atribuir_ordem_replica_controle(grupo):
    grupo = grupo.copy()
    chaves_ordenadas = (
        grupo[[COL_CHAVE_CONTROLE, "semente_aleatoria_numero"]]
        .drop_duplicates(subset=[COL_CHAVE_CONTROLE])
        .sort_values(["semente_aleatoria_numero", COL_CHAVE_CONTROLE], na_position="last")
        .reset_index(drop=True)
    )
    mapa_ordem = {str(linha[COL_CHAVE_CONTROLE]): pos + 1 for pos, linha in chaves_ordenadas.iterrows()}
    grupo["replica_aleatoria_ordem"] = grupo[COL_CHAVE_CONTROLE].astype(str).map(mapa_ordem)
    grupo["replica_aleatoria_numero"] = grupo["replica_aleatoria_ordem"]
    return grupo

df_base_aleatorias = (
    df_base_aleatorias
    .groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"], group_keys=False, dropna=False)
    .apply(atribuir_ordem_replica_controle)
    .reset_index(drop=True)
)

df_base_aleatorias["replica_aleatoria_ordem"] = pd.to_numeric(df_base_aleatorias["replica_aleatoria_ordem"], errors="coerce")
df_base_aleatorias["replica_aleatoria_numero"] = pd.to_numeric(df_base_aleatorias["replica_aleatoria_numero"], errors="coerce")

colunas_booleanas = [
    "flag_retorno_acumulado_real_superou_controle",
    "flag_retorno_anualizado_real_superou_controle",
    "flag_sharpe_real_superou_controle",
    "flag_sortino_real_superou_controle",
    "flag_calmar_real_superou_controle",
    "flag_volatilidade_real_menor_controle",
    "flag_drawdown_abs_real_menor_controle",
]
for coluna in colunas_booleanas:
    if coluna in df_base_aleatorias.columns:
        df_base_aleatorias[coluna] = para_booleano_seguro(df_base_aleatorias[coluna])

colunas_numericas = [
    "n_periodos_comparaveis",
    "pct_periodos_real_superou_controle",
    "excesso_medio_periodo_real_vs_controle",
    "excesso_mediano_periodo_real_vs_controle",
    "retorno_acumulado_relativo_composto_real_vs_controle",
    "excesso_anualizado_real_vs_controle_periodico",
    "dif_sharpe_anualizado_real_vs_controle",
    "dif_sortino_anualizado_real_vs_controle",
    "dif_calmar_real_vs_controle",
    "dif_volatilidade_anualizada_equivalente_real_vs_controle",
    "dif_max_drawdown_abs_real_vs_controle",
]
for coluna in colunas_numericas:
    if coluna in df_base_aleatorias.columns:
        df_base_aleatorias[coluna] = para_numero_seguro(df_base_aleatorias[coluna])

print(f"Base multifrequência de controles         : {len(df_controles_multifrequencia):,} linhas")
print(f"Base filtrada para controles aleatórios   : {len(df_base_aleatorias):,} linhas")
print(f"Estratégias de referência identificadas    : {df_base_aleatorias[COL_ESTRATEGIA_REFERENCIA].nunique() if len(df_base_aleatorias) > 0 else 0:,}")
print(f"Frequências identificadas                  : {df_base_aleatorias['frequencia'].nunique() if len(df_base_aleatorias) > 0 else 0:,}")
print(f"Réplicas aleatórias distintas              : {df_base_aleatorias[COL_CHAVE_CONTROLE].nunique() if len(df_base_aleatorias) > 0 else 0:,}")
print("OK")

# ============================================================
# 6) Definição do motor de robustez por réplicas aleatórias
# ============================================================

print("\n[6/12] Definição do motor de robustez por réplicas aleatórias...")

linhas_parametros = []
if len(df_base_aleatorias) > 0:
    for (estrategia, frequencia), grupo in df_base_aleatorias.groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"], dropna=False):
        replicas_ordenadas = (
            grupo[[COL_CHAVE_CONTROLE, COL_NOME_CONTROLE, "replica_aleatoria_numero", "replica_aleatoria_ordem", "semente_aleatoria_numero"]]
            .drop_duplicates(subset=[COL_CHAVE_CONTROLE])
            .sort_values(["replica_aleatoria_ordem", COL_CHAVE_CONTROLE])
            .reset_index(drop=True)
        )
        n_replicas_disponiveis = int(len(replicas_ordenadas))
        for n_replicas in range(1, n_replicas_disponiveis + 1):
            linhas_parametros.append({
                "estrategia_referencia": estrategia,
                "frequencia": frequencia,
                "cenario_aleatorio": f"primeiras_{n_replicas}_replicas",
                "ordem_cenario": n_replicas,
                "n_replicas_cenario": n_replicas,
                "n_replicas_disponiveis": n_replicas_disponiveis,
                "tipo_cenario": "robustez_subamostra_progressiva_replicas",
                "descricao_cenario": f"Usa as primeiras {n_replicas} réplicas aleatórias em ordem determinística para medir estabilidade da comparação.",
                "replicas_incluidas": ", ".join(replicas_ordenadas.head(n_replicas)[COL_NOME_CONTROLE].astype(str).tolist()),
            })

df_parametros_cenarios = pd.DataFrame(linhas_parametros)

print(f"Cenários de robustez por réplicas gerados : {len(df_parametros_cenarios):,}")
print(f"Máximo de réplicas por estratégia/freq.    : {int(df_parametros_cenarios['n_replicas_disponiveis'].max()) if len(df_parametros_cenarios) > 0 else 0:,}")
print("OK")

# ============================================================
# 7) Aplicação da análise progressiva por quantidade de réplicas
# ============================================================

print("\n[7/12] Aplicação da análise progressiva por quantidade de réplicas...")

linhas_resumo_replicas = []
if len(df_base_aleatorias) > 0:
    for (estrategia, frequencia), grupo_original in df_base_aleatorias.groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"], dropna=False):
        grupo_original = grupo_original.sort_values(["replica_aleatoria_ordem", COL_CHAVE_CONTROLE]).copy()
        replicas_ordenadas = (
            grupo_original[[COL_CHAVE_CONTROLE, "replica_aleatoria_ordem"]]
            .drop_duplicates(subset=[COL_CHAVE_CONTROLE])
            .sort_values(["replica_aleatoria_ordem", COL_CHAVE_CONTROLE])
            .reset_index(drop=True)
        )
        chaves_replicas = replicas_ordenadas[COL_CHAVE_CONTROLE].astype(str).tolist()
        for n_replicas in range(1, len(chaves_replicas) + 1):
            chaves_incluidas = set(chaves_replicas[:n_replicas])
            grupo = grupo_original.loc[grupo_original[COL_CHAVE_CONTROLE].astype(str).isin(chaves_incluidas)].copy()
            contexto = {
                "estrategia_referencia": estrategia,
                "frequencia": frequencia,
                "cenario_aleatorio": f"primeiras_{n_replicas}_replicas",
                "ordem_cenario": n_replicas,
                "tipo_cenario": "robustez_subamostra_progressiva_replicas",
                "n_replicas_disponiveis": len(chaves_replicas),
            }
            linhas_resumo_replicas.append(resumo_subconjunto(grupo, contexto))

df_resumo_replicas = pd.DataFrame(linhas_resumo_replicas)
if not df_resumo_replicas.empty:
    df_resumo_replicas = df_resumo_replicas.sort_values(["estrategia_referencia", "frequencia", "ordem_cenario"]).reset_index(drop=True)

print(f"Resumo de robustez por número de réplicas: {len(df_resumo_replicas):,} linhas")
print("OK")

# ============================================================
# 8) Distribuição das métricas por réplica aleatória
# ============================================================

print("\n[8/12] Distribuição das métricas por réplica aleatória...")

colunas_agrupar_replica = [
    COL_ESTRATEGIA_REFERENCIA,
    "frequencia",
    COL_CHAVE_CONTROLE,
    COL_NOME_CONTROLE,
    "replica_aleatoria_numero",
    "replica_aleatoria_ordem",
]
colunas_agrupar_replica = [coluna for coluna in colunas_agrupar_replica if coluna in df_base_aleatorias.columns]

if len(df_base_aleatorias) > 0:
    df_distribuicao_replicas = df_base_aleatorias.copy()
    df_distribuicao_replicas = df_distribuicao_replicas.rename(columns={
        COL_ESTRATEGIA_REFERENCIA: "estrategia_referencia",
        COL_CHAVE_CONTROLE: "chave_estrategia_controle",
        COL_NOME_CONTROLE: "nome_exibicao_estrategia_controle",
        COL_CHAVE_REAL: "chave_estrategia_real",
        COL_NOME_REAL: "nome_exibicao_estrategia_real",
    })
    colunas_saida_distribuicao = [
        "estrategia_referencia",
        "frequencia",
        "comparacao_id",
        "chave_estrategia_real",
        "nome_exibicao_estrategia_real",
        "chave_estrategia_controle",
        "nome_exibicao_estrategia_controle",
        "replica_aleatoria_id",
        "replica_aleatoria_numero",
        "replica_aleatoria_ordem",
        "semente_aleatoria_numero",
        "n_periodos_comparaveis",
        "pct_periodos_real_superou_controle",
        "retorno_acumulado_relativo_composto_real_vs_controle",
        "excesso_anualizado_real_vs_controle_periodico",
        "excesso_medio_periodo_real_vs_controle",
        "dif_sharpe_anualizado_real_vs_controle",
        "dif_sortino_anualizado_real_vs_controle",
        "dif_calmar_real_vs_controle",
        "dif_volatilidade_anualizada_equivalente_real_vs_controle",
        "dif_max_drawdown_abs_real_vs_controle",
        "flag_retorno_acumulado_real_superou_controle",
        "flag_retorno_anualizado_real_superou_controle",
        "flag_sharpe_real_superou_controle",
        "flag_sortino_real_superou_controle",
        "flag_calmar_real_superou_controle",
        "flag_volatilidade_real_menor_controle",
        "flag_drawdown_abs_real_menor_controle",
    ]
    colunas_saida_distribuicao = [coluna for coluna in colunas_saida_distribuicao if coluna in df_distribuicao_replicas.columns]
    df_distribuicao_replicas = df_distribuicao_replicas[colunas_saida_distribuicao].copy()
    df_distribuicao_replicas = df_distribuicao_replicas.sort_values(["estrategia_referencia", "frequencia", "replica_aleatoria_ordem"]).reset_index(drop=True)
else:
    df_distribuicao_replicas = pd.DataFrame()

print(f"Distribuição por réplica aleatória       : {len(df_distribuicao_replicas):,} linhas")
print("OK")

# ============================================================
# 9) Estabilidade leave-one-out das réplicas aleatórias
# ============================================================

print("\n[9/12] Estabilidade leave-one-out das réplicas aleatórias...")

linhas_leave_one_out = []
if len(df_base_aleatorias) > 0:
    for (estrategia, frequencia), grupo_original in df_base_aleatorias.groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"], dropna=False):
        grupo_original = grupo_original.sort_values(["replica_aleatoria_ordem", COL_CHAVE_CONTROLE]).copy()
        replicas = (
            grupo_original[[COL_CHAVE_CONTROLE, COL_NOME_CONTROLE, "replica_aleatoria_numero", "replica_aleatoria_ordem", "semente_aleatoria_numero"]]
            .drop_duplicates(subset=[COL_CHAVE_CONTROLE])
            .sort_values(["replica_aleatoria_ordem", COL_CHAVE_CONTROLE])
            .reset_index(drop=True)
        )
        if len(replicas) <= 1:
            continue
        for _, linha_replica in replicas.iterrows():
            chave_excluida = str(linha_replica[COL_CHAVE_CONTROLE])
            grupo = grupo_original.loc[~grupo_original[COL_CHAVE_CONTROLE].astype(str).eq(chave_excluida)].copy()
            contexto = {
                "estrategia_referencia": estrategia,
                "frequencia": frequencia,
                "cenario_aleatorio": "leave_one_out",
                "ordem_cenario": int(linha_replica["replica_aleatoria_ordem"]) if pd.notna(linha_replica["replica_aleatoria_ordem"]) else np.nan,
                "tipo_cenario": "robustez_exclusao_unitaria_replica",
                "n_replicas_disponiveis": len(replicas),
                "replica_excluida_chave": chave_excluida,
                "replica_excluida_nome": str(linha_replica[COL_NOME_CONTROLE]),
                "replica_excluida_numero": linha_replica["replica_aleatoria_numero"],
                "semente_excluida_numero": linha_replica["semente_aleatoria_numero"] if "semente_aleatoria_numero" in linha_replica.index else np.nan,
            }
            linhas_leave_one_out.append(resumo_subconjunto(grupo, contexto))

df_leave_one_out = pd.DataFrame(linhas_leave_one_out)
if not df_leave_one_out.empty:
    df_leave_one_out = df_leave_one_out.sort_values(["estrategia_referencia", "frequencia", "ordem_cenario"]).reset_index(drop=True)

print(f"Cenários leave-one-out                  : {len(df_leave_one_out):,} linhas")
print("OK")

# ============================================================
# 10) Posicionamento final e impacto temporal das aleatórias
# ============================================================

print("\n[10/12] Posicionamento final e impacto temporal das aleatórias...")

if not df_resumo_replicas.empty:
    df_posicionamento_final = df_resumo_replicas.loc[
        df_resumo_replicas["ordem_cenario"].eq(df_resumo_replicas["n_replicas_disponiveis"])
    ].copy()
    df_posicionamento_final["cenario_aleatorio"] = "todas_replicas_disponiveis"
    df_posicionamento_final = df_posicionamento_final.sort_values(["estrategia_referencia", "frequencia"]).reset_index(drop=True)
else:
    df_posicionamento_final = pd.DataFrame()

# Impacto temporal anual dos calendários aleatórios, quando disponível.
if not df_distribuicao_anual_aleatorios.empty:
    df_impacto_temporal = df_distribuicao_anual_aleatorios.copy()
    COL_ANO_TEMPORAL = escolher_coluna(df_impacto_temporal, ["ano", "ano_aporte", "ano_sinal"], obrigatoria=False)
    COL_ESTRATEGIA_TEMPORAL = escolher_coluna(df_impacto_temporal, ["estrategia_referencia", "estrategia", "tipo_sinal"], obrigatoria=False)
    COL_REPLICA_TEMPORAL = escolher_coluna(df_impacto_temporal, ["replica_aleatoria", "replica", "numero_replica", "replica_controle"], obrigatoria=False)
    if COL_ANO_TEMPORAL is None:
        df_impacto_temporal["ano"] = np.nan
    if COL_ESTRATEGIA_TEMPORAL is None:
        df_impacto_temporal["estrategia_referencia"] = "indefinida"
    if COL_REPLICA_TEMPORAL is None:
        df_impacto_temporal["replica_aleatoria"] = np.nan
    renomear_temporal = {}
    if COL_ANO_TEMPORAL is not None and COL_ANO_TEMPORAL != "ano":
        renomear_temporal[COL_ANO_TEMPORAL] = "ano"
    if COL_ESTRATEGIA_TEMPORAL is not None and COL_ESTRATEGIA_TEMPORAL != "estrategia_referencia":
        renomear_temporal[COL_ESTRATEGIA_TEMPORAL] = "estrategia_referencia"
    if COL_REPLICA_TEMPORAL is not None and COL_REPLICA_TEMPORAL != "replica_aleatoria":
        renomear_temporal[COL_REPLICA_TEMPORAL] = "replica_aleatoria"
    df_impacto_temporal = df_impacto_temporal.rename(columns=renomear_temporal)
else:
    df_impacto_temporal = pd.DataFrame(columns=["estrategia_referencia", "replica_aleatoria", "ano"])

print(f"Posicionamento final com todas réplicas  : {len(df_posicionamento_final):,} linhas")
print(f"Impacto temporal das aleatórias          : {len(df_impacto_temporal):,} linhas")
print("OK")

# ============================================================
# 11) Base de gráficos e auditoria
# ============================================================

print("\n[11/12] Base de gráficos e auditoria...")

bases_grafico = []
if not df_resumo_replicas.empty:
    metricas_grafico_resumo = [
        "pct_replicas_retorno_acumulado_real_superou_controle",
        "media_pct_periodos_real_superou_controle",
        "media_excesso_anualizado_real_vs_controle_periodico",
        "media_retorno_acumulado_relativo_composto_real_vs_controle",
        "media_dif_sharpe_anualizado_real_vs_controle",
        "media_dif_sortino_anualizado_real_vs_controle",
        "media_dif_calmar_real_vs_controle",
        "media_dif_max_drawdown_abs_real_vs_controle",
    ]
    for metrica in metricas_grafico_resumo:
        if metrica in df_resumo_replicas.columns:
            tmp = df_resumo_replicas[["estrategia_referencia", "frequencia", "cenario_aleatorio", "ordem_cenario", "n_replicas_aleatorias", metrica]].copy()
            tmp = tmp.rename(columns={metrica: "valor"})
            tmp["tipo_grafico"] = "sensibilidade_numero_replicas"
            tmp["metrica"] = metrica
            bases_grafico.append(tmp)

if not df_distribuicao_replicas.empty:
    metricas_grafico_distribuicao = [
        "excesso_anualizado_real_vs_controle_periodico",
        "retorno_acumulado_relativo_composto_real_vs_controle",
        "pct_periodos_real_superou_controle",
        "dif_sharpe_anualizado_real_vs_controle",
    ]
    for metrica in metricas_grafico_distribuicao:
        if metrica in df_distribuicao_replicas.columns:
            tmp = df_distribuicao_replicas[["estrategia_referencia", "frequencia", "nome_exibicao_estrategia_controle", "replica_aleatoria_ordem", metrica]].copy()
            tmp = tmp.rename(columns={
                "nome_exibicao_estrategia_controle": "cenario_aleatorio",
                "replica_aleatoria_ordem": "ordem_cenario",
                metrica: "valor",
            })
            tmp["n_replicas_aleatorias"] = 1
            tmp["tipo_grafico"] = "distribuicao_replicas"
            tmp["metrica"] = metrica
            bases_grafico.append(tmp)

if bases_grafico:
    df_base_grafico = pd.concat(bases_grafico, ignore_index=True, sort=False)
    df_base_grafico["valor"] = para_numero_seguro(df_base_grafico["valor"])
    df_base_grafico = df_base_grafico.sort_values(["tipo_grafico", "estrategia_referencia", "frequencia", "metrica", "ordem_cenario"]).reset_index(drop=True)
else:
    df_base_grafico = pd.DataFrame(columns=["tipo_grafico", "metrica", "valor", "estrategia_referencia", "frequencia", "cenario_aleatorio", "ordem_cenario", "n_replicas_aleatorias"])

n_linhas_base_aleatorias = len(df_base_aleatorias)
n_estrategias_referencia = int(df_base_aleatorias[COL_ESTRATEGIA_REFERENCIA].nunique()) if len(df_base_aleatorias) > 0 else 0
n_frequencias = int(df_base_aleatorias["frequencia"].nunique()) if len(df_base_aleatorias) > 0 else 0
n_replicas_distintas = int(df_base_aleatorias[COL_CHAVE_CONTROLE].nunique()) if len(df_base_aleatorias) > 0 else 0
if len(df_base_aleatorias) > 0:
    min_replicas_por_grupo = int(df_base_aleatorias.groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"])[COL_CHAVE_CONTROLE].nunique().min())
else:
    min_replicas_por_grupo = 0
n_cenarios_progressivos = len(df_resumo_replicas)
n_leave_one_out = len(df_leave_one_out)
n_posicionamento_final = len(df_posicionamento_final)

if len(df_base_aleatorias) > 0:
    duplicatas_chave = int(df_base_aleatorias.duplicated(subset=[COL_ESTRATEGIA_REFERENCIA, "frequencia", COL_CHAVE_CONTROLE]).sum())
else:
    duplicatas_chave = 0

numeric_validacao = pd.concat(
    [
        df_resumo_replicas.select_dtypes(include=[np.number]) if not df_resumo_replicas.empty else pd.DataFrame(),
        df_distribuicao_replicas.select_dtypes(include=[np.number]) if not df_distribuicao_replicas.empty else pd.DataFrame(),
        df_leave_one_out.select_dtypes(include=[np.number]) if not df_leave_one_out.empty else pd.DataFrame(),
        df_posicionamento_final.select_dtypes(include=[np.number]) if not df_posicionamento_final.empty else pd.DataFrame(),
    ],
    axis=0,
    ignore_index=True,
)
metricas_infinitas = int(np.isinf(numeric_validacao.to_numpy(dtype=float)).sum()) if not numeric_validacao.empty else 0

linhas_auditoria = [
    {
        "item": "linhas_base_aleatorias",
        "valor": n_linhas_base_aleatorias,
        "valor_referencia": "> 0",
        "status": "OK" if n_linhas_base_aleatorias > 0 else "ERRO",
        "observacao": "A subetapa deve identificar comparações com estratégias aleatórias a partir das bases multifrequência da 11.6.",
    },
    {
        "item": "estrategias_referencia_identificadas",
        "valor": n_estrategias_referencia,
        "valor_referencia": ">= 2",
        "status": "OK" if n_estrategias_referencia >= 2 else "ERRO",
        "observacao": "Espera-se encontrar Capitulação e Euforia nas comparações aleatórias.",
    },
    {
        "item": "frequencias_identificadas",
        "valor": n_frequencias,
        "valor_referencia": ">= 3",
        "status": "OK" if n_frequencias >= 3 else "ERRO",
        "observacao": "A sensibilidade deve preservar as visões diária, mensal e anual geradas na 11.6.",
    },
    {
        "item": "replicas_aleatorias_distintas",
        "valor": n_replicas_distintas,
        "valor_referencia": ">= 2",
        "status": "OK" if n_replicas_distintas >= 2 else "ERRO",
        "observacao": "A análise de robustez exige mais de uma réplica aleatória por família de controle.",
    },
    {
        "item": "replicas_minimas_por_estrategia_frequencia",
        "valor": min_replicas_por_grupo,
        "valor_referencia": ">= 5",
        "status": "OK" if min_replicas_por_grupo >= 5 else "ERRO",
        "observacao": "Cada combinação de estratégia e frequência deve ter réplicas suficientes para avaliar estabilidade.",
    },
    {
        "item": "cenarios_progressivos_gerados",
        "valor": n_cenarios_progressivos,
        "valor_referencia": "> 0",
        "status": "OK" if n_cenarios_progressivos > 0 else "ERRO",
        "observacao": "A análise deve gerar cenários por número de réplicas incluídas para medir estabilidade dos controles aleatórios.",
    },
    {
        "item": "cenarios_leave_one_out_gerados",
        "valor": n_leave_one_out,
        "valor_referencia": "> 0",
        "status": "OK" if n_leave_one_out > 0 else "ERRO",
        "observacao": "A análise leave-one-out deve medir a influência de cada réplica aleatória.",
    },
    {
        "item": "duplicatas_estrategia_frequencia_replica",
        "valor": duplicatas_chave,
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Duplicidades aqui podem ocorrer se uma réplica aparecer em múltiplos grupos de comparação; item informativo.",
    },
    {
        "item": "calendarios_aleatorios_7_5_disponiveis",
        "valor": int(len(df_calendarios_aleatorios) > 0),
        "valor_referencia": "informativo",
        "status": "OK",
        "observacao": "Indica se a base original de calendários aleatórios da 7.5 foi encontrada para auditoria complementar.",
    },
    {
        "item": "posicionamento_final_linhas",
        "valor": n_posicionamento_final,
        "valor_referencia": "> 0",
        "status": "OK" if n_posicionamento_final > 0 else "ERRO",
        "observacao": "A subetapa deve gerar um posicionamento final com todas as réplicas disponíveis por estratégia e frequência.",
    },
    {
        "item": "base_grafico_linhas",
        "valor": len(df_base_grafico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": 0,
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
n_erros_bloqueantes = int(df_auditoria["status"].eq("ERRO").sum())

print(f"Base gráfico sensibilidade aleatórias   : {len(df_base_grafico):,} linhas")
print(f"Itens de auditoria                      : {len(df_auditoria):,}")
print(f"Erros bloqueantes                       : {n_erros_bloqueantes:,}")
print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(df_base_aleatorias, CAMINHO_BASE_ALEATORIAS, index=False)
salvar_dataframe(df_parametros_cenarios, CAMINHO_PARAMETROS_CENARIOS, index=False)
salvar_dataframe(df_resumo_replicas, CAMINHO_RESUMO_REPLICAS, index=False)
salvar_dataframe(df_distribuicao_replicas, CAMINHO_DISTRIBUICAO_REPLICAS, index=False)
salvar_dataframe(df_leave_one_out, CAMINHO_LEAVE_ONE_OUT, index=False)
salvar_dataframe(df_posicionamento_final, CAMINHO_POSICIONAMENTO_FINAL, index=False)
salvar_dataframe(df_impacto_temporal, CAMINHO_IMPACTO_TEMPORAL, index=False)
salvar_dataframe(df_base_grafico, CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da sensibilidade das estratégias aleatórias:")
print(df_auditoria.to_string(index=False))

print("\nResumo por número de réplicas - amostra:")
if len(df_resumo_replicas) > 0:
    colunas_print_resumo = [
        "estrategia_referencia",
        "frequencia",
        "cenario_aleatorio",
        "n_replicas_aleatorias",
        "n_sementes_aleatorias_distintas",
        "pct_replicas_retorno_acumulado_real_superou_controle",
        "media_pct_periodos_real_superou_controle",
        "media_excesso_anualizado_real_vs_controle_periodico",
        "classificacao_robustez_retorno_acumulado",
    ]
    colunas_print_resumo = [coluna for coluna in colunas_print_resumo if coluna in df_resumo_replicas.columns]
    print(df_resumo_replicas[colunas_print_resumo].head(60).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nDistribuição por réplica aleatória - amostra:")
if len(df_distribuicao_replicas) > 0:
    colunas_print_distribuicao = [
        "estrategia_referencia",
        "frequencia",
        "nome_exibicao_estrategia_controle",
        "replica_aleatoria_numero",
        "semente_aleatoria_numero",
        "pct_periodos_real_superou_controle",
        "retorno_acumulado_relativo_composto_real_vs_controle",
        "excesso_anualizado_real_vs_controle_periodico",
    ]
    colunas_print_distribuicao = [coluna for coluna in colunas_print_distribuicao if coluna in df_distribuicao_replicas.columns]
    print(df_distribuicao_replicas[colunas_print_distribuicao].head(60).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nEstabilidade leave-one-out - amostra:")
if len(df_leave_one_out) > 0:
    colunas_print_loo = [
        "estrategia_referencia",
        "frequencia",
        "replica_excluida_nome",
        "n_replicas_aleatorias",
        "n_sementes_aleatorias_distintas",
        "pct_replicas_retorno_acumulado_real_superou_controle",
        "media_excesso_anualizado_real_vs_controle_periodico",
        "classificacao_robustez_retorno_acumulado",
    ]
    colunas_print_loo = [coluna for coluna in colunas_print_loo if coluna in df_leave_one_out.columns]
    print(df_leave_one_out[colunas_print_loo].head(60).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nPosicionamento final com todas as réplicas:")
if len(df_posicionamento_final) > 0:
    colunas_print_final = [
        "estrategia_referencia",
        "frequencia",
        "n_replicas_aleatorias",
        "n_sementes_aleatorias_distintas",
        "pct_replicas_retorno_acumulado_real_superou_controle",
        "pct_replicas_sharpe_real_superou_controle",
        "pct_replicas_sortino_real_superou_controle",
        "pct_replicas_calmar_real_superou_controle",
        "media_excesso_anualizado_real_vs_controle_periodico",
        "classificacao_robustez_retorno_acumulado",
    ]
    colunas_print_final = [coluna for coluna in colunas_print_final if coluna in df_posicionamento_final.columns]
    print(df_posicionamento_final[colunas_print_final].to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nArquivos salvos na subetapa 13.6:")
print(f"- {CAMINHO_BASE_ALEATORIAS}")
print(f"- {CAMINHO_PARAMETROS_CENARIOS}")
print(f"- {CAMINHO_RESUMO_REPLICAS}")
print(f"- {CAMINHO_DISTRIBUICAO_REPLICAS}")
print(f"- {CAMINHO_LEAVE_ONE_OUT}")
print(f"- {CAMINHO_POSICIONAMENTO_FINAL}")
print(f"- {CAMINHO_IMPACTO_TEMPORAL}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_AUDITORIA}")

if n_erros_bloqueantes > 0:
    raise RuntimeError("A subetapa 13.6 encontrou erros bloqueantes na auditoria. Verifique a tabela de validação.")

print("\nETAPA 13.6 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 13.6 - SENSIBILIDADE DAS ESTRATÉGIAS ALEATÓRIAS

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - calendários aleatórios da 7.5       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_5_base_calendarios_aleatorios_controle.parquet
Entrada - resumo aleatórios da 7.5            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_5_tbl_resumo_calendarios_aleatorios_controle.parquet
Entrada - distribuição anual aleatórios da 7.5: C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_5_tbl_distribuicao_anual_calendarios_aleatorios_controle.parquet
Entrada - pares de comparação da 11.6         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento

## Etapa 13.7) Sensibilidade das Estratégias Mensais de Controle

In [70]:
%%time
# ============================================================
# Etapa 13.7) Sensibilidade das Estratégias Mensais de Controle
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 13.7 - SENSIBILIDADE DAS ESTRATÉGIAS MENSAIS DE CONTROLE")
print("=" * 100)

# Esta subetapa não reexecuta o backtest.
# O objetivo é medir a robustez das conclusões frente aos controles sistemáticos de aportes mensais já gerados.
# A análise preserva as visões diária, mensal e anual produzidas na etapa 11.6.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "re" not in globals():
    raise NameError("A biblioteca re deve estar importada no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_7 = Path(DIRETORIOS_PROJETO["etapa_7"])
DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])
DIR_ETAPA_13 = Path(DIRETORIOS_PROJETO["etapa_13"])
DIR_ETAPA_13.mkdir(parents=True, exist_ok=True)

CAMINHO_CALENDARIO_MENSAIS_7_6 = DIR_ETAPA_7 / "7_6_base_calendario_aportes_estrategias_mensais_controle.parquet"
CAMINHO_RESUMO_MENSAIS_7_6 = DIR_ETAPA_7 / "7_6_tbl_resumo_estrategias_mensais_controle.parquet"
CAMINHO_DISTRIBUICAO_ANUAL_MENSAIS_7_6 = DIR_ETAPA_7 / "7_6_tbl_distribuicao_anual_estrategias_mensais_controle.parquet"
CAMINHO_ESTRATEGIAS_MENSAIS_7_6 = DIR_ETAPA_7 / "7_6_tbl_estrategias_mensais_controle.parquet"
CAMINHO_PARES_COMPARACAO_CONTROLES_11_6 = DIR_ETAPA_11 / "11_6_tbl_pares_comparacao_controles.parquet"
CAMINHO_TBL_CONTROLES_DIARIO_11_6 = DIR_ETAPA_11 / "11_6_tbl_comparacao_controles_diario.parquet"
CAMINHO_TBL_CONTROLES_MENSAL_11_6 = DIR_ETAPA_11 / "11_6_tbl_comparacao_controles_mensal.parquet"
CAMINHO_TBL_CONTROLES_ANUAL_11_6 = DIR_ETAPA_11 / "11_6_tbl_comparacao_controles_anual.parquet"
CAMINHO_POSICIONAMENTO_11_6 = DIR_ETAPA_11 / "11_6_tbl_posicionamento_relativo_multifrequencia.parquet"
CAMINHO_RANKING_11_6 = DIR_ETAPA_11 / "11_6_tbl_ranking_estrategias_reais_vs_controles.parquet"

CAMINHO_BASE_MENSAIS = DIR_ETAPA_13 / "13_7_base_sensibilidade_estrategias_mensais_controle.parquet"
CAMINHO_PARAMETROS_CENARIOS = DIR_ETAPA_13 / "13_7_tbl_parametros_cenarios_estrategias_mensais_controle.parquet"
CAMINHO_RESUMO_MENSAIS = DIR_ETAPA_13 / "13_7_tbl_resumo_sensibilidade_controles_mensais.parquet"
CAMINHO_DISTRIBUICAO_MENSAIS = DIR_ETAPA_13 / "13_7_tbl_distribuicao_metricas_controles_mensais.parquet"
CAMINHO_ESTABILIDADE_MENSAIS = DIR_ETAPA_13 / "13_7_tbl_estabilidade_controles_mensais.parquet"
CAMINHO_POSICIONAMENTO_FINAL = DIR_ETAPA_13 / "13_7_tbl_posicionamento_final_estrategias_reais_vs_mensais.parquet"
CAMINHO_IMPACTO_TEMPORAL = DIR_ETAPA_13 / "13_7_tbl_impacto_temporal_controles_mensais.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_13 / "13_7_tbl_base_grafico_sensibilidade_mensais.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_13 / "13_7_tbl_auditoria_validacao_sensibilidade_mensais.parquet"

CAMINHOS_ENTRADA_OBRIGATORIOS = [
    CAMINHO_TBL_CONTROLES_DIARIO_11_6,
    CAMINHO_TBL_CONTROLES_MENSAL_11_6,
    CAMINHO_TBL_CONTROLES_ANUAL_11_6,
]

for caminho in CAMINHOS_ENTRADA_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - calendário mensais da 7.6          : {CAMINHO_CALENDARIO_MENSAIS_7_6}")
print(f"Entrada - resumo mensais da 7.6              : {CAMINHO_RESUMO_MENSAIS_7_6}")
print(f"Entrada - distribuição anual mensais da 7.6  : {CAMINHO_DISTRIBUICAO_ANUAL_MENSAIS_7_6}")
print(f"Entrada - estratégias mensais da 7.6         : {CAMINHO_ESTRATEGIAS_MENSAIS_7_6}")
print(f"Entrada - pares de comparação da 11.6        : {CAMINHO_PARES_COMPARACAO_CONTROLES_11_6}")
print(f"Entrada - comparação controles diária da 11.6: {CAMINHO_TBL_CONTROLES_DIARIO_11_6}")
print(f"Entrada - comparação controles mensal da 11.6: {CAMINHO_TBL_CONTROLES_MENSAL_11_6}")
print(f"Entrada - comparação controles anual da 11.6 : {CAMINHO_TBL_CONTROLES_ANUAL_11_6}")
print(f"Entrada - posicionamento relativo da 11.6    : {CAMINHO_POSICIONAMENTO_11_6}")
print(f"Entrada - ranking final da 11.6              : {CAMINHO_RANKING_11_6}")
print(f"Saída   - base sensibilidade mensais         : {CAMINHO_BASE_MENSAIS}")
print(f"Saída   - parâmetros dos cenários            : {CAMINHO_PARAMETROS_CENARIOS}")
print(f"Saída   - resumo controles mensais           : {CAMINHO_RESUMO_MENSAIS}")
print(f"Saída   - distribuição controles mensais     : {CAMINHO_DISTRIBUICAO_MENSAIS}")
print(f"Saída   - estabilidade controles mensais     : {CAMINHO_ESTABILIDADE_MENSAIS}")
print(f"Saída   - posicionamento final               : {CAMINHO_POSICIONAMENTO_FINAL}")
print(f"Saída   - impacto temporal                   : {CAMINHO_IMPACTO_TEMPORAL}")
print(f"Saída   - base gráfico                       : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação             : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/12] Carga das bases oficiais da subetapa...")

df_controles_diario = pd.read_parquet(CAMINHO_TBL_CONTROLES_DIARIO_11_6)
df_controles_mensal = pd.read_parquet(CAMINHO_TBL_CONTROLES_MENSAL_11_6)
df_controles_anual = pd.read_parquet(CAMINHO_TBL_CONTROLES_ANUAL_11_6)

if Path(CAMINHO_CALENDARIO_MENSAIS_7_6).exists():
    df_calendario_mensais = pd.read_parquet(CAMINHO_CALENDARIO_MENSAIS_7_6)
else:
    df_calendario_mensais = pd.DataFrame()

if Path(CAMINHO_RESUMO_MENSAIS_7_6).exists():
    df_resumo_mensais_7_6 = pd.read_parquet(CAMINHO_RESUMO_MENSAIS_7_6)
else:
    df_resumo_mensais_7_6 = pd.DataFrame()

if Path(CAMINHO_DISTRIBUICAO_ANUAL_MENSAIS_7_6).exists():
    df_distribuicao_anual_mensais_7_6 = pd.read_parquet(CAMINHO_DISTRIBUICAO_ANUAL_MENSAIS_7_6)
else:
    df_distribuicao_anual_mensais_7_6 = pd.DataFrame()

if Path(CAMINHO_ESTRATEGIAS_MENSAIS_7_6).exists():
    df_estrategias_mensais_7_6 = pd.read_parquet(CAMINHO_ESTRATEGIAS_MENSAIS_7_6)
else:
    df_estrategias_mensais_7_6 = pd.DataFrame()

if Path(CAMINHO_PARES_COMPARACAO_CONTROLES_11_6).exists():
    df_pares_controles = pd.read_parquet(CAMINHO_PARES_COMPARACAO_CONTROLES_11_6)
else:
    df_pares_controles = pd.DataFrame()

if Path(CAMINHO_POSICIONAMENTO_11_6).exists():
    df_posicionamento_11_6 = pd.read_parquet(CAMINHO_POSICIONAMENTO_11_6)
else:
    df_posicionamento_11_6 = pd.DataFrame()

if Path(CAMINHO_RANKING_11_6).exists():
    df_ranking_11_6 = pd.read_parquet(CAMINHO_RANKING_11_6)
else:
    df_ranking_11_6 = pd.DataFrame()

print(f"Comparação controles diária da 11.6 : {len(df_controles_diario):,} linhas x {df_controles_diario.shape[1]:,} colunas")
print(f"Comparação controles mensal da 11.6 : {len(df_controles_mensal):,} linhas x {df_controles_mensal.shape[1]:,} colunas")
print(f"Comparação controles anual da 11.6  : {len(df_controles_anual):,} linhas x {df_controles_anual.shape[1]:,} colunas")
print(f"Calendário mensais da 7.6           : {len(df_calendario_mensais):,} linhas x {df_calendario_mensais.shape[1]:,} colunas")
print(f"Resumo mensais da 7.6               : {len(df_resumo_mensais_7_6):,} linhas x {df_resumo_mensais_7_6.shape[1]:,} colunas")
print(f"Distribuição anual mensais da 7.6   : {len(df_distribuicao_anual_mensais_7_6):,} linhas x {df_distribuicao_anual_mensais_7_6.shape[1]:,} colunas")
print(f"Estratégias mensais da 7.6          : {len(df_estrategias_mensais_7_6):,} linhas x {df_estrategias_mensais_7_6.shape[1]:,} colunas")
print(f"Pares de comparação da 11.6         : {len(df_pares_controles):,} linhas x {df_pares_controles.shape[1]:,} colunas")
print(f"Posicionamento relativo da 11.6     : {len(df_posicionamento_11_6):,} linhas x {df_posicionamento_11_6.shape[1]:,} colunas")
print(f"Ranking final da 11.6               : {len(df_ranking_11_6):,} linhas x {df_ranking_11_6.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

def escolher_coluna(df, candidatos, obrigatoria=False, nome_logico="coluna"):
    for coluna in candidatos:
        if coluna in df.columns:
            return coluna
    if obrigatoria:
        raise KeyError(
            f"Não foi possível identificar {nome_logico}. "
            f"Candidatas avaliadas: {candidatos}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

def para_numero_seguro(serie):
    return pd.to_numeric(serie, errors="coerce")

def para_datetime_seguro(serie):
    return pd.to_datetime(serie, errors="coerce").dt.normalize()

def para_booleano_seguro(serie):
    if serie.dtype == bool:
        return serie.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(serie):
        return serie.fillna(0).astype(float).ne(0)
    serie_texto = serie.astype(str).str.strip().str.lower()
    valores_verdadeiros = {"1", "true", "t", "sim", "s", "yes", "y", "ok", "v", "verdadeiro"}
    return serie_texto.isin(valores_verdadeiros)

def normalizar_texto_serie(serie):
    return serie.astype(str).str.strip().str.lower()

def concatenar_texto_linha(linha, colunas):
    textos = []
    for coluna in colunas:
        if coluna is not None and coluna in linha.index:
            textos.append(str(linha.get(coluna, "")))
    return " ".join(textos)

def extrair_numero_carteira(linha):
    texto = concatenar_texto_linha(
        linha,
        [
            "nome_exibicao_estrategia_controle",
            "estrategia_id_controle",
            "chave_estrategia_controle",
            "comparacao_id",
        ],
    ).lower()
    padroes = [
        r"carteira\s*(\d+)",
        r"#\s*(\d+)",
        r"replica[_\s-]*(\d+)",
        r"réplica[_\s-]*(\d+)",
        r"mensal[_\s-]*(\d+)",
        r"controle[_\s-]*(\d+)",
    ]
    for padrao in padroes:
        match = re.search(padrao, texto)
        if match:
            return int(match.group(1))
    numeros = re.findall(r"\d+", texto)
    if numeros:
        return int(numeros[-1])
    return np.nan

def extrair_n_aportes_mensais_texto(texto):
    texto = str(texto).lower()
    padroes = [
        r"(\d+)\s*aportes?\s*mensais?",
        r"aportes?\s*mensais?\s*(\d+)",
        r"(\d+)\s*compras?\s*mensais?",
    ]
    for padrao in padroes:
        match = re.search(padrao, texto)
        if match:
            return int(match.group(1))
    return np.nan

def extrair_n_aportes_mensais_linha(linha):
    candidatos_colunas = [
        "n_aportes_mensais",
        "quantidade_aportes_mensais",
        "num_aportes_mensais",
        "n_aportes",
        "qtd_aportes",
    ]
    for coluna in candidatos_colunas:
        if coluna in linha.index and pd.notna(linha.get(coluna)):
            valor = pd.to_numeric(pd.Series([linha.get(coluna)]), errors="coerce").iloc[0]
            if pd.notna(valor):
                return int(valor)
    texto = concatenar_texto_linha(
        linha,
        [
            "nome_exibicao_estrategia_controle",
            "nome_exibicao_estrategia",
            "estrategia_id_controle",
            "estrategia_id",
            "chave_estrategia_controle",
            "chave_estrategia",
            "comparacao_id",
        ],
    )
    return extrair_n_aportes_mensais_texto(texto)

def inferir_estrategia_referencia_texto(texto):
    texto = str(texto).lower()
    if "capitul" in texto:
        return "capitulacao"
    if "euforia" in texto:
        return "euforia"
    return "nao_identificado"

def classificar_robustez(pct_superacao, excesso_medio):
    if pd.isna(pct_superacao) or pd.isna(excesso_medio):
        return "indefinida"
    if pct_superacao >= 0.80 and excesso_medio > 0:
        return "favoravel_forte"
    if pct_superacao >= 0.60 and excesso_medio > 0:
        return "favoravel_moderada"
    if pct_superacao <= 0.40 and excesso_medio < 0:
        return "desfavoravel"
    return "neutra"

def resumo_subconjunto(grupo, contexto, coluna_chave_controle):
    n_pares = int(len(grupo))
    if coluna_chave_controle in grupo.columns:
        n_controles = int(grupo[coluna_chave_controle].astype(str).nunique())
    else:
        n_controles = n_pares
    n_tipos = int(grupo["tipo_controle_mensal"].nunique()) if "tipo_controle_mensal" in grupo.columns else np.nan
    registro = dict(contexto)
    registro["n_controles_mensais"] = n_controles
    registro["n_tipos_controle_mensal"] = n_tipos
    registro["n_pares_comparacao"] = n_pares
    registro["n_periodos_comparaveis_total"] = int(para_numero_seguro(grupo.get("n_periodos_comparaveis", pd.Series(dtype=float))).sum()) if "n_periodos_comparaveis" in grupo.columns else np.nan

    flags = {
        "pct_controles_retorno_acumulado_real_superou_controle": "flag_retorno_acumulado_real_superou_controle",
        "pct_controles_retorno_anualizado_real_superou_controle": "flag_retorno_anualizado_real_superou_controle",
        "pct_controles_sharpe_real_superou_controle": "flag_sharpe_real_superou_controle",
        "pct_controles_sortino_real_superou_controle": "flag_sortino_real_superou_controle",
        "pct_controles_calmar_real_superou_controle": "flag_calmar_real_superou_controle",
        "pct_controles_volatilidade_real_menor_controle": "flag_volatilidade_real_menor_controle",
        "pct_controles_drawdown_abs_real_menor_controle": "flag_drawdown_abs_real_menor_controle",
    }
    for nome_saida, coluna in flags.items():
        if coluna in grupo.columns and n_pares > 0:
            registro[nome_saida] = float(para_booleano_seguro(grupo[coluna]).mean())
        else:
            registro[nome_saida] = np.nan

    metricas_numericas = {
        "pct_periodos_real_superou_controle": "pct_periodos_real_superou_controle",
        "excesso_medio_periodo_real_vs_controle": "excesso_medio_periodo_real_vs_controle",
        "excesso_mediano_periodo_real_vs_controle": "excesso_mediano_periodo_real_vs_controle",
        "retorno_acumulado_relativo_composto_real_vs_controle": "retorno_acumulado_relativo_composto_real_vs_controle",
        "excesso_anualizado_real_vs_controle_periodico": "excesso_anualizado_real_vs_controle_periodico",
        "dif_sharpe_anualizado_real_vs_controle": "dif_sharpe_anualizado_real_vs_controle",
        "dif_sortino_anualizado_real_vs_controle": "dif_sortino_anualizado_real_vs_controle",
        "dif_calmar_real_vs_controle": "dif_calmar_real_vs_controle",
        "dif_volatilidade_anualizada_equivalente_real_vs_controle": "dif_volatilidade_anualizada_equivalente_real_vs_controle",
        "dif_max_drawdown_abs_real_vs_controle": "dif_max_drawdown_abs_real_vs_controle",
    }
    for nome_saida, coluna in metricas_numericas.items():
        if coluna in grupo.columns:
            valores = para_numero_seguro(grupo[coluna]).dropna()
            registro[f"media_{nome_saida}"] = float(valores.mean()) if len(valores) > 0 else np.nan
            registro[f"mediana_{nome_saida}"] = float(valores.median()) if len(valores) > 0 else np.nan
            registro[f"min_{nome_saida}"] = float(valores.min()) if len(valores) > 0 else np.nan
            registro[f"max_{nome_saida}"] = float(valores.max()) if len(valores) > 0 else np.nan
        else:
            registro[f"media_{nome_saida}"] = np.nan
            registro[f"mediana_{nome_saida}"] = np.nan
            registro[f"min_{nome_saida}"] = np.nan
            registro[f"max_{nome_saida}"] = np.nan

    registro["classificacao_robustez_retorno_acumulado"] = classificar_robustez(
        registro.get("pct_controles_retorno_acumulado_real_superou_controle", np.nan),
        registro.get("media_excesso_anualizado_real_vs_controle_periodico", np.nan),
    )
    registro["classificacao_robustez_periodos"] = classificar_robustez(
        registro.get("media_pct_periodos_real_superou_controle", np.nan),
        registro.get("media_excesso_medio_periodo_real_vs_controle", np.nan),
    )
    return registro

def preparar_auditoria(linhas):
    df_aud = pd.DataFrame(linhas)
    for coluna in ["item", "valor", "valor_referencia", "status", "observacao"]:
        if coluna not in df_aud.columns:
            df_aud[coluna] = ""
        df_aud[coluna] = df_aud[coluna].astype(str)
    return df_aud

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização da base de comparações dos controles mensais
# ============================================================

print("\n[5/12] Padronização da base de comparações dos controles mensais...")

def empilhar_base_controle(df, frequencia):
    df_saida = df.copy()
    if "frequencia" not in df_saida.columns:
        df_saida["frequencia"] = frequencia
    else:
        df_saida["frequencia"] = df_saida["frequencia"].fillna(frequencia).astype(str)
    return df_saida

df_controles_multifrequencia = pd.concat(
    [
        empilhar_base_controle(df_controles_diario, "diaria"),
        empilhar_base_controle(df_controles_mensal, "mensal"),
        empilhar_base_controle(df_controles_anual, "anual"),
    ],
    ignore_index=True,
    sort=False,
)

COL_ESTRATEGIA_REFERENCIA = escolher_coluna(
    df_controles_multifrequencia,
    ["estrategia_referencia", "estrategia", "tipo_sinal"],
    obrigatoria=True,
    nome_logico="estratégia de referência na base 11.6",
)
COL_GRUPO_CONTROLE = escolher_coluna(
    df_controles_multifrequencia,
    ["grupo_comparacao_controle", "grupo_controle", "tipo_controle"],
    obrigatoria=False,
    nome_logico="grupo de comparação do controle",
)
COL_TIPO_CONTROLE = escolher_coluna(
    df_controles_multifrequencia,
    ["tipo_comparacao_controle", "tipo_controle", "subtipo_controle"],
    obrigatoria=False,
    nome_logico="tipo de comparação do controle",
)
COL_COMPARACAO_ID = escolher_coluna(
    df_controles_multifrequencia,
    ["comparacao_id", "id_comparacao", "par_comparacao"],
    obrigatoria=False,
    nome_logico="comparação id",
)
COL_CHAVE_CONTROLE = escolher_coluna(
    df_controles_multifrequencia,
    ["chave_estrategia_controle", "estrategia_id_controle", "chave_controle", "id_controle"],
    obrigatoria=False,
    nome_logico="chave da estratégia de controle",
)
COL_NOME_CONTROLE = escolher_coluna(
    df_controles_multifrequencia,
    ["nome_exibicao_estrategia_controle", "nome_estrategia_controle", "nome_controle"],
    obrigatoria=False,
    nome_logico="nome da estratégia de controle",
)
COL_CHAVE_REAL = escolher_coluna(
    df_controles_multifrequencia,
    ["chave_estrategia_real", "estrategia_id_real", "chave_real", "id_real"],
    obrigatoria=False,
    nome_logico="chave da estratégia real",
)
COL_NOME_REAL = escolher_coluna(
    df_controles_multifrequencia,
    ["nome_exibicao_estrategia_real", "nome_estrategia_real", "nome_real"],
    obrigatoria=False,
    nome_logico="nome da estratégia real",
)

if COL_COMPARACAO_ID is None:
    df_controles_multifrequencia["comparacao_id"] = (
        df_controles_multifrequencia[COL_ESTRATEGIA_REFERENCIA].astype(str) + "__" +
        df_controles_multifrequencia.get(COL_NOME_CONTROLE, pd.Series("controle", index=df_controles_multifrequencia.index)).astype(str) + "__" +
        df_controles_multifrequencia["frequencia"].astype(str)
    )
    COL_COMPARACAO_ID = "comparacao_id"

if COL_CHAVE_CONTROLE is None:
    df_controles_multifrequencia["chave_estrategia_controle"] = df_controles_multifrequencia[COL_COMPARACAO_ID].astype(str)
    COL_CHAVE_CONTROLE = "chave_estrategia_controle"

if COL_NOME_CONTROLE is None:
    df_controles_multifrequencia["nome_exibicao_estrategia_controle"] = df_controles_multifrequencia[COL_CHAVE_CONTROLE].astype(str)
    COL_NOME_CONTROLE = "nome_exibicao_estrategia_controle"

if COL_CHAVE_REAL is None:
    df_controles_multifrequencia["chave_estrategia_real"] = df_controles_multifrequencia[COL_ESTRATEGIA_REFERENCIA].astype(str)
    COL_CHAVE_REAL = "chave_estrategia_real"

if COL_NOME_REAL is None:
    df_controles_multifrequencia["nome_exibicao_estrategia_real"] = df_controles_multifrequencia[COL_ESTRATEGIA_REFERENCIA].astype(str)
    COL_NOME_REAL = "nome_exibicao_estrategia_real"

texto_filtro_mensal = pd.Series("", index=df_controles_multifrequencia.index, dtype="object")
for coluna in [COL_GRUPO_CONTROLE, COL_TIPO_CONTROLE, COL_NOME_CONTROLE, COL_CHAVE_CONTROLE, COL_COMPARACAO_ID]:
    if coluna is not None and coluna in df_controles_multifrequencia.columns:
        texto_filtro_mensal = texto_filtro_mensal + " " + df_controles_multifrequencia[coluna].astype(str)

texto_filtro_mensal_norm = normalizar_texto_serie(texto_filtro_mensal)
flag_mensal = texto_filtro_mensal_norm.str.contains("mensal|mensais|monthly", regex=True, na=False)
flag_excluir_benchmarks = texto_filtro_mensal_norm.str.contains("ibovespa|cdi|benchmark", regex=True, na=False)
df_base_mensais = df_controles_multifrequencia.loc[flag_mensal & (~flag_excluir_benchmarks)].copy()

for coluna in [COL_ESTRATEGIA_REFERENCIA, COL_CHAVE_CONTROLE, COL_NOME_CONTROLE, COL_CHAVE_REAL, COL_NOME_REAL, "frequencia"]:
    if coluna in df_base_mensais.columns:
        df_base_mensais[coluna] = df_base_mensais[coluna].astype(str).str.strip()

df_base_mensais["n_aportes_mensais"] = df_base_mensais.apply(extrair_n_aportes_mensais_linha, axis=1)
df_base_mensais["n_aportes_mensais"] = pd.to_numeric(df_base_mensais["n_aportes_mensais"], errors="coerce")
df_base_mensais["tipo_controle_mensal"] = np.where(
    df_base_mensais["n_aportes_mensais"].notna(),
    df_base_mensais["n_aportes_mensais"].astype("Int64").astype(str) + "_aportes_mensais",
    "mensal_sem_numero_aportes",
)
df_base_mensais["carteira_controle_mensal_numero"] = df_base_mensais.apply(extrair_numero_carteira, axis=1)
df_base_mensais["controle_mensal_id"] = df_base_mensais[COL_CHAVE_CONTROLE].astype(str)

def atribuir_ordem_controle_mensal(grupo):
    grupo = grupo.copy()
    chaves_ordenadas = (
        grupo[[COL_CHAVE_CONTROLE, COL_NOME_CONTROLE, "tipo_controle_mensal", "n_aportes_mensais", "carteira_controle_mensal_numero"]]
        .drop_duplicates(subset=[COL_CHAVE_CONTROLE])
        .sort_values(["n_aportes_mensais", "carteira_controle_mensal_numero", COL_CHAVE_CONTROLE], na_position="last")
        .reset_index(drop=True)
    )
    mapa_ordem = {str(linha[COL_CHAVE_CONTROLE]): pos + 1 for pos, linha in chaves_ordenadas.iterrows()}
    grupo["controle_mensal_ordem"] = grupo[COL_CHAVE_CONTROLE].astype(str).map(mapa_ordem)
    grupo["controle_mensal_numero"] = grupo["controle_mensal_ordem"]
    return grupo

if len(df_base_mensais) > 0:
    df_base_mensais = (
        df_base_mensais
        .groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"], group_keys=False, dropna=False)
        .apply(atribuir_ordem_controle_mensal)
        .reset_index(drop=True)
    )
else:
    df_base_mensais["controle_mensal_ordem"] = pd.Series(dtype=float)
    df_base_mensais["controle_mensal_numero"] = pd.Series(dtype=float)

df_base_mensais["controle_mensal_ordem"] = pd.to_numeric(df_base_mensais["controle_mensal_ordem"], errors="coerce")
df_base_mensais["controle_mensal_numero"] = pd.to_numeric(df_base_mensais["controle_mensal_numero"], errors="coerce")

colunas_booleanas = [
    "flag_retorno_acumulado_real_superou_controle",
    "flag_retorno_anualizado_real_superou_controle",
    "flag_sharpe_real_superou_controle",
    "flag_sortino_real_superou_controle",
    "flag_calmar_real_superou_controle",
    "flag_volatilidade_real_menor_controle",
    "flag_drawdown_abs_real_menor_controle",
]
for coluna in colunas_booleanas:
    if coluna in df_base_mensais.columns:
        df_base_mensais[coluna] = para_booleano_seguro(df_base_mensais[coluna])

colunas_numericas = [
    "n_periodos_comparaveis",
    "pct_periodos_real_superou_controle",
    "excesso_medio_periodo_real_vs_controle",
    "excesso_mediano_periodo_real_vs_controle",
    "retorno_acumulado_relativo_composto_real_vs_controle",
    "excesso_anualizado_real_vs_controle_periodico",
    "dif_sharpe_anualizado_real_vs_controle",
    "dif_sortino_anualizado_real_vs_controle",
    "dif_calmar_real_vs_controle",
    "dif_volatilidade_anualizada_equivalente_real_vs_controle",
    "dif_max_drawdown_abs_real_vs_controle",
]
for coluna in colunas_numericas:
    if coluna in df_base_mensais.columns:
        df_base_mensais[coluna] = para_numero_seguro(df_base_mensais[coluna])

print(f"Base multifrequência de controles          : {len(df_controles_multifrequencia):,} linhas")
print(f"Base filtrada para controles mensais       : {len(df_base_mensais):,} linhas")
print(f"Estratégias de referência identificadas     : {df_base_mensais[COL_ESTRATEGIA_REFERENCIA].nunique() if len(df_base_mensais) > 0 else 0:,}")
print(f"Frequências identificadas                   : {df_base_mensais['frequencia'].nunique() if len(df_base_mensais) > 0 else 0:,}")
print(f"Controles mensais distintos                 : {df_base_mensais[COL_CHAVE_CONTROLE].nunique() if len(df_base_mensais) > 0 else 0:,}")
print(f"Tipos de controles mensais identificados    : {df_base_mensais['tipo_controle_mensal'].nunique() if len(df_base_mensais) > 0 else 0:,}")
print("OK")

# ============================================================
# 6) Definição dos cenários de sensibilidade dos controles mensais
# ============================================================

print("\n[6/12] Definição dos cenários de sensibilidade dos controles mensais...")

linhas_parametros = []
if len(df_base_mensais) > 0:
    for (estrategia, frequencia), grupo in df_base_mensais.groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"], dropna=False):
        controles_ordenados = (
            grupo[[COL_CHAVE_CONTROLE, COL_NOME_CONTROLE, "tipo_controle_mensal", "n_aportes_mensais", "controle_mensal_ordem"]]
            .drop_duplicates(subset=[COL_CHAVE_CONTROLE])
            .sort_values(["controle_mensal_ordem", COL_CHAVE_CONTROLE])
            .reset_index(drop=True)
        )
        n_controles_disponiveis = int(len(controles_ordenados))
        linhas_parametros.append({
            "estrategia_referencia": estrategia,
            "frequencia": frequencia,
            "cenario_mensal": "todos_controles_mensais",
            "ordem_cenario": 0,
            "tipo_cenario": "todos_controles_mensais",
            "n_controles_cenario": n_controles_disponiveis,
            "n_controles_disponiveis": n_controles_disponiveis,
            "tipo_controle_mensal": "todos",
            "descricao_cenario": "Usa todos os controles mensais disponíveis para a estratégia e frequência.",
            "controles_incluidos": ", ".join(controles_ordenados[COL_NOME_CONTROLE].astype(str).tolist()),
        })
        for tipo_controle, grupo_tipo in controles_ordenados.groupby("tipo_controle_mensal", dropna=False):
            linhas_parametros.append({
                "estrategia_referencia": estrategia,
                "frequencia": frequencia,
                "cenario_mensal": f"somente_{tipo_controle}",
                "ordem_cenario": 100 + len(linhas_parametros),
                "tipo_cenario": "tipo_aporte_mensal",
                "n_controles_cenario": int(len(grupo_tipo)),
                "n_controles_disponiveis": n_controles_disponiveis,
                "tipo_controle_mensal": tipo_controle,
                "descricao_cenario": f"Usa apenas controles mensais do tipo {tipo_controle}.",
                "controles_incluidos": ", ".join(grupo_tipo[COL_NOME_CONTROLE].astype(str).tolist()),
            })
        for n_controles in range(1, n_controles_disponiveis + 1):
            linhas_parametros.append({
                "estrategia_referencia": estrategia,
                "frequencia": frequencia,
                "cenario_mensal": f"primeiros_{n_controles}_controles_mensais",
                "ordem_cenario": 1000 + n_controles,
                "tipo_cenario": "subamostra_progressiva_controles_mensais",
                "n_controles_cenario": n_controles,
                "n_controles_disponiveis": n_controles_disponiveis,
                "tipo_controle_mensal": "progressivo",
                "descricao_cenario": f"Usa os primeiros {n_controles} controles mensais em ordem determinística para medir estabilidade da comparação.",
                "controles_incluidos": ", ".join(controles_ordenados.head(n_controles)[COL_NOME_CONTROLE].astype(str).tolist()),
            })

df_parametros_cenarios = pd.DataFrame(linhas_parametros)

print(f"Cenários de sensibilidade mensais gerados : {len(df_parametros_cenarios):,}")
print(f"Máximo de controles por estratégia/freq.  : {int(df_parametros_cenarios['n_controles_disponiveis'].max()) if len(df_parametros_cenarios) > 0 else 0:,}")
print("OK")

# ============================================================
# 7) Aplicação dos cenários de sensibilidade dos controles mensais
# ============================================================

print("\n[7/12] Aplicação dos cenários de sensibilidade dos controles mensais...")

linhas_resumo_mensais = []
if len(df_base_mensais) > 0:
    for (estrategia, frequencia), grupo_original in df_base_mensais.groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"], dropna=False):
        grupo_original = grupo_original.sort_values(["controle_mensal_ordem", COL_CHAVE_CONTROLE]).copy()
        controles_ordenados = (
            grupo_original[[COL_CHAVE_CONTROLE, "controle_mensal_ordem"]]
            .drop_duplicates(subset=[COL_CHAVE_CONTROLE])
            .sort_values(["controle_mensal_ordem", COL_CHAVE_CONTROLE])
            .reset_index(drop=True)
        )
        chaves_controles = controles_ordenados[COL_CHAVE_CONTROLE].astype(str).tolist()

        contexto_todos = {
            "estrategia_referencia": estrategia,
            "frequencia": frequencia,
            "cenario_mensal": "todos_controles_mensais",
            "tipo_cenario": "todos_controles_mensais",
            "ordem_cenario": 0,
            "tipo_controle_mensal": "todos",
        }
        linhas_resumo_mensais.append(resumo_subconjunto(grupo_original, contexto_todos, COL_CHAVE_CONTROLE))

        for tipo_controle, grupo_tipo in grupo_original.groupby("tipo_controle_mensal", dropna=False):
            contexto_tipo = {
                "estrategia_referencia": estrategia,
                "frequencia": frequencia,
                "cenario_mensal": f"somente_{tipo_controle}",
                "tipo_cenario": "tipo_aporte_mensal",
                "ordem_cenario": 100 + len(linhas_resumo_mensais),
                "tipo_controle_mensal": tipo_controle,
            }
            linhas_resumo_mensais.append(resumo_subconjunto(grupo_tipo, contexto_tipo, COL_CHAVE_CONTROLE))

        for n_controles in range(1, len(chaves_controles) + 1):
            chaves_incluidas = chaves_controles[:n_controles]
            grupo_cenario = grupo_original.loc[grupo_original[COL_CHAVE_CONTROLE].astype(str).isin(chaves_incluidas)].copy()
            contexto_progressivo = {
                "estrategia_referencia": estrategia,
                "frequencia": frequencia,
                "cenario_mensal": f"primeiros_{n_controles}_controles_mensais",
                "tipo_cenario": "subamostra_progressiva_controles_mensais",
                "ordem_cenario": 1000 + n_controles,
                "tipo_controle_mensal": "progressivo",
            }
            linhas_resumo_mensais.append(resumo_subconjunto(grupo_cenario, contexto_progressivo, COL_CHAVE_CONTROLE))

df_resumo_mensais = pd.DataFrame(linhas_resumo_mensais)
if len(df_resumo_mensais) > 0:
    df_resumo_mensais = df_resumo_mensais.sort_values(
        ["estrategia_referencia", "frequencia", "ordem_cenario", "cenario_mensal"]
    ).reset_index(drop=True)

print(f"Resumo de sensibilidade dos controles mensais: {len(df_resumo_mensais):,} linhas")
print("OK")

# ============================================================
# 8) Distribuição das métricas por controle mensal
# ============================================================

print("\n[8/12] Distribuição das métricas por controle mensal...")

linhas_distribuicao = []
if len(df_base_mensais) > 0:
    for chaves, grupo in df_base_mensais.groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia", "tipo_controle_mensal", COL_CHAVE_CONTROLE], dropna=False):
        estrategia, frequencia, tipo_controle, chave_controle = chaves
        nome_controle = grupo[COL_NOME_CONTROLE].iloc[0] if COL_NOME_CONTROLE in grupo.columns and len(grupo) > 0 else str(chave_controle)
        contexto = {
            "estrategia_referencia": estrategia,
            "frequencia": frequencia,
            "tipo_controle_mensal": tipo_controle,
            "chave_estrategia_controle": chave_controle,
            "nome_exibicao_estrategia_controle": nome_controle,
            "n_aportes_mensais": grupo["n_aportes_mensais"].dropna().iloc[0] if grupo["n_aportes_mensais"].notna().any() else np.nan,
            "carteira_controle_mensal_numero": grupo["carteira_controle_mensal_numero"].dropna().iloc[0] if grupo["carteira_controle_mensal_numero"].notna().any() else np.nan,
            "controle_mensal_ordem": grupo["controle_mensal_ordem"].dropna().iloc[0] if grupo["controle_mensal_ordem"].notna().any() else np.nan,
        }
        linhas_distribuicao.append(resumo_subconjunto(grupo, contexto, COL_CHAVE_CONTROLE))

df_distribuicao_mensais = pd.DataFrame(linhas_distribuicao)
if len(df_distribuicao_mensais) > 0:
    df_distribuicao_mensais = df_distribuicao_mensais.sort_values(
        ["estrategia_referencia", "frequencia", "tipo_controle_mensal", "controle_mensal_ordem", "nome_exibicao_estrategia_controle"]
    ).reset_index(drop=True)

print(f"Distribuição por controle mensal             : {len(df_distribuicao_mensais):,} linhas")
print("OK")

# ============================================================
# 9) Estabilidade por tipo e leave-one-out dos controles mensais
# ============================================================

print("\n[9/12] Estabilidade por tipo e leave-one-out dos controles mensais...")

linhas_estabilidade = []
if len(df_base_mensais) > 0:
    for (estrategia, frequencia), grupo_original in df_base_mensais.groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"], dropna=False):
        tipos_controle = sorted(grupo_original["tipo_controle_mensal"].dropna().astype(str).unique().tolist())
        for tipo_excluido in tipos_controle:
            grupo_restante = grupo_original.loc[grupo_original["tipo_controle_mensal"].astype(str).ne(tipo_excluido)].copy()
            if len(grupo_restante) == 0:
                continue
            contexto_tipo = {
                "estrategia_referencia": estrategia,
                "frequencia": frequencia,
                "tipo_estabilidade": "leave_one_type_out",
                "controle_excluido": tipo_excluido,
                "tipo_controle_mensal_excluido": tipo_excluido,
            }
            linhas_estabilidade.append(resumo_subconjunto(grupo_restante, contexto_tipo, COL_CHAVE_CONTROLE))

        controles = (
            grupo_original[[COL_CHAVE_CONTROLE, COL_NOME_CONTROLE, "tipo_controle_mensal"]]
            .drop_duplicates(subset=[COL_CHAVE_CONTROLE])
            .sort_values(["tipo_controle_mensal", COL_CHAVE_CONTROLE])
            .reset_index(drop=True)
        )
        for _, linha_controle in controles.iterrows():
            chave_excluida = str(linha_controle[COL_CHAVE_CONTROLE])
            nome_excluido = str(linha_controle[COL_NOME_CONTROLE])
            tipo_excluido = str(linha_controle["tipo_controle_mensal"])
            grupo_restante = grupo_original.loc[grupo_original[COL_CHAVE_CONTROLE].astype(str).ne(chave_excluida)].copy()
            if len(grupo_restante) == 0:
                continue
            contexto_controle = {
                "estrategia_referencia": estrategia,
                "frequencia": frequencia,
                "tipo_estabilidade": "leave_one_control_out",
                "controle_excluido": nome_excluido,
                "tipo_controle_mensal_excluido": tipo_excluido,
            }
            linhas_estabilidade.append(resumo_subconjunto(grupo_restante, contexto_controle, COL_CHAVE_CONTROLE))

df_estabilidade_mensais = pd.DataFrame(linhas_estabilidade)
if len(df_estabilidade_mensais) > 0:
    df_estabilidade_mensais = df_estabilidade_mensais.sort_values(
        ["estrategia_referencia", "frequencia", "tipo_estabilidade", "tipo_controle_mensal_excluido", "controle_excluido"]
    ).reset_index(drop=True)

print(f"Cenários de estabilidade dos controles mensais: {len(df_estabilidade_mensais):,} linhas")
print("OK")

# ============================================================
# 10) Posicionamento final e impacto temporal dos controles mensais
# ============================================================

print("\n[10/12] Posicionamento final e impacto temporal dos controles mensais...")

if len(df_resumo_mensais) > 0:
    df_posicionamento_final_mensais = df_resumo_mensais.loc[
        df_resumo_mensais["tipo_cenario"].astype(str).eq("todos_controles_mensais")
    ].copy().reset_index(drop=True)
else:
    df_posicionamento_final_mensais = pd.DataFrame()

# Impacto temporal a partir do calendário da 7.6; fallback para a distribuição anual já salva na 7.6.
linhas_temporais = []
if len(df_calendario_mensais) > 0:
    df_tmp_cal = df_calendario_mensais.copy()
    COL_DATA_CAL = escolher_coluna(
        df_tmp_cal,
        ["data_aporte", "data", "data_referencia", "data_execucao", "data_sinal"],
        obrigatoria=False,
        nome_logico="data do calendário mensal",
    )
    COL_NOME_CAL = escolher_coluna(
        df_tmp_cal,
        ["nome_exibicao_estrategia", "nome_exibicao_estrategia_controle", "nome_estrategia", "estrategia_id", "chave_estrategia"],
        obrigatoria=False,
        nome_logico="nome da estratégia mensal no calendário",
    )
    COL_CHAVE_CAL = escolher_coluna(
        df_tmp_cal,
        ["chave_estrategia", "estrategia_id", "chave_estrategia_controle", "id_estrategia"],
        obrigatoria=False,
        nome_logico="chave da estratégia mensal no calendário",
    )
    COL_VALOR_CAL = escolher_coluna(
        df_tmp_cal,
        ["valor_aporte", "valor_aporte_realizado", "valor_aporte_total", "valor_aporte_teorico"],
        obrigatoria=False,
        nome_logico="valor do aporte no calendário mensal",
    )

    if COL_DATA_CAL is not None:
        df_tmp_cal["data_aporte_padronizada"] = para_datetime_seguro(df_tmp_cal[COL_DATA_CAL])
        df_tmp_cal["ano"] = df_tmp_cal["data_aporte_padronizada"].dt.year
    elif "ano" in df_tmp_cal.columns:
        df_tmp_cal["ano"] = pd.to_numeric(df_tmp_cal["ano"], errors="coerce")
    else:
        df_tmp_cal["ano"] = np.nan

    if COL_NOME_CAL is not None:
        df_tmp_cal["nome_estrategia_mensal"] = df_tmp_cal[COL_NOME_CAL].astype(str)
    elif COL_CHAVE_CAL is not None:
        df_tmp_cal["nome_estrategia_mensal"] = df_tmp_cal[COL_CHAVE_CAL].astype(str)
    else:
        df_tmp_cal["nome_estrategia_mensal"] = "controle_mensal"

    texto_cal = df_tmp_cal["nome_estrategia_mensal"].astype(str)
    if COL_CHAVE_CAL is not None:
        texto_cal = texto_cal + " " + df_tmp_cal[COL_CHAVE_CAL].astype(str)
    df_tmp_cal["estrategia_referencia"] = texto_cal.apply(inferir_estrategia_referencia_texto)
    df_tmp_cal["n_aportes_mensais"] = df_tmp_cal.apply(extrair_n_aportes_mensais_linha, axis=1)
    df_tmp_cal["n_aportes_mensais"] = pd.to_numeric(df_tmp_cal["n_aportes_mensais"], errors="coerce")
    df_tmp_cal["tipo_controle_mensal"] = np.where(
        df_tmp_cal["n_aportes_mensais"].notna(),
        df_tmp_cal["n_aportes_mensais"].astype("Int64").astype(str) + "_aportes_mensais",
        "mensal_sem_numero_aportes",
    )
    if COL_CHAVE_CAL is not None:
        df_tmp_cal["chave_estrategia_mensal"] = df_tmp_cal[COL_CHAVE_CAL].astype(str)
    else:
        df_tmp_cal["chave_estrategia_mensal"] = df_tmp_cal["nome_estrategia_mensal"].astype(str)

    if COL_VALOR_CAL is not None:
        df_tmp_cal["valor_aporte_mensal"] = para_numero_seguro(df_tmp_cal[COL_VALOR_CAL])
    else:
        df_tmp_cal["valor_aporte_mensal"] = np.nan

    df_tmp_cal = df_tmp_cal.loc[df_tmp_cal["ano"].notna()].copy()
    if len(df_tmp_cal) > 0:
        df_impacto_temporal = (
            df_tmp_cal
            .groupby(["estrategia_referencia", "tipo_controle_mensal", "ano"], dropna=False)
            .agg(
                n_aportes_calendario=("ano", "size"),
                n_estrategias_mensais_distintas=("chave_estrategia_mensal", "nunique"),
                valor_total_aportes_mensais=("valor_aporte_mensal", "sum"),
                valor_medio_aporte_mensal=("valor_aporte_mensal", "mean"),
            )
            .reset_index()
            .sort_values(["estrategia_referencia", "tipo_controle_mensal", "ano"])
            .reset_index(drop=True)
        )
    else:
        df_impacto_temporal = pd.DataFrame()
elif len(df_distribuicao_anual_mensais_7_6) > 0:
    df_impacto_temporal = df_distribuicao_anual_mensais_7_6.copy()
    if "fonte_impacto_temporal" not in df_impacto_temporal.columns:
        df_impacto_temporal["fonte_impacto_temporal"] = "7_6_tbl_distribuicao_anual_estrategias_mensais_controle"
else:
    df_impacto_temporal = pd.DataFrame()

print(f"Posicionamento final com todos controles mensais: {len(df_posicionamento_final_mensais):,} linhas")
print(f"Impacto temporal dos controles mensais           : {len(df_impacto_temporal):,} linhas")
print("OK")

# ============================================================
# 11) Base de gráficos e auditoria
# ============================================================

print("\n[11/12] Base de gráficos e auditoria...")

metricas_grafico = [
    "pct_controles_retorno_acumulado_real_superou_controle",
    "pct_controles_sharpe_real_superou_controle",
    "pct_controles_sortino_real_superou_controle",
    "pct_controles_calmar_real_superou_controle",
    "media_pct_periodos_real_superou_controle",
    "media_excesso_anualizado_real_vs_controle_periodico",
    "media_dif_sharpe_anualizado_real_vs_controle",
    "media_dif_sortino_anualizado_real_vs_controle",
    "media_dif_calmar_real_vs_controle",
]
colunas_id_grafico = [
    "estrategia_referencia",
    "frequencia",
    "cenario_mensal",
    "tipo_cenario",
    "tipo_controle_mensal",
    "ordem_cenario",
    "n_controles_mensais",
    "n_tipos_controle_mensal",
    "classificacao_robustez_retorno_acumulado",
]
colunas_id_grafico = [coluna for coluna in colunas_id_grafico if coluna in df_resumo_mensais.columns]
metricas_grafico_existentes = [coluna for coluna in metricas_grafico if coluna in df_resumo_mensais.columns]

if len(df_resumo_mensais) > 0 and len(metricas_grafico_existentes) > 0:
    df_base_grafico = df_resumo_mensais[colunas_id_grafico + metricas_grafico_existentes].melt(
        id_vars=colunas_id_grafico,
        value_vars=metricas_grafico_existentes,
        var_name="metrica",
        value_name="valor",
    )
    df_base_grafico["valor"] = para_numero_seguro(df_base_grafico["valor"])
else:
    df_base_grafico = pd.DataFrame(columns=colunas_id_grafico + ["metrica", "valor"])

min_controles_por_estrategia_frequencia = 0
if len(df_base_mensais) > 0:
    min_controles_por_estrategia_frequencia = int(
        df_base_mensais.groupby([COL_ESTRATEGIA_REFERENCIA, "frequencia"])[COL_CHAVE_CONTROLE].nunique().min()
    )

metricas_infinitas = 0
for df_verificacao in [df_base_mensais, df_resumo_mensais, df_distribuicao_mensais, df_estabilidade_mensais, df_posicionamento_final_mensais, df_base_grafico]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

linhas_auditoria = [
    {
        "item": "linhas_base_mensais",
        "valor": len(df_base_mensais),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_mensais) > 0 else "ERRO",
        "observacao": "A subetapa deve identificar controles mensais nas bases multifrequência da 11.6.",
    },
    {
        "item": "estrategias_referencia_identificadas",
        "valor": df_base_mensais[COL_ESTRATEGIA_REFERENCIA].nunique() if len(df_base_mensais) > 0 else 0,
        "valor_referencia": ">= 2",
        "status": "OK" if (df_base_mensais[COL_ESTRATEGIA_REFERENCIA].nunique() if len(df_base_mensais) > 0 else 0) >= 2 else "ERRO",
        "observacao": "Espera-se encontrar Capitulação e Euforia nas comparações contra controles mensais.",
    },
    {
        "item": "frequencias_identificadas",
        "valor": df_base_mensais["frequencia"].nunique() if len(df_base_mensais) > 0 else 0,
        "valor_referencia": ">= 3",
        "status": "OK" if (df_base_mensais["frequencia"].nunique() if len(df_base_mensais) > 0 else 0) >= 3 else "ERRO",
        "observacao": "A sensibilidade deve preservar as visões diária, mensal e anual geradas na 11.6.",
    },
    {
        "item": "controles_mensais_distintos",
        "valor": df_base_mensais[COL_CHAVE_CONTROLE].nunique() if len(df_base_mensais) > 0 else 0,
        "valor_referencia": ">= 2",
        "status": "OK" if (df_base_mensais[COL_CHAVE_CONTROLE].nunique() if len(df_base_mensais) > 0 else 0) >= 2 else "ERRO",
        "observacao": "A análise exige mais de um controle mensal para avaliar estabilidade.",
    },
    {
        "item": "tipos_controle_mensal_identificados",
        "valor": df_base_mensais["tipo_controle_mensal"].nunique() if len(df_base_mensais) > 0 else 0,
        "valor_referencia": ">= 1",
        "status": "OK" if (df_base_mensais["tipo_controle_mensal"].nunique() if len(df_base_mensais) > 0 else 0) >= 1 else "ERRO",
        "observacao": "A subetapa deve identificar ao menos um tipo de controle mensal.",
    },
    {
        "item": "controles_minimos_por_estrategia_frequencia",
        "valor": min_controles_por_estrategia_frequencia,
        "valor_referencia": ">= 2",
        "status": "OK" if min_controles_por_estrategia_frequencia >= 2 else "ERRO",
        "observacao": "Cada combinação de estratégia e frequência deve ter controles mensais suficientes para avaliação comparativa.",
    },
    {
        "item": "cenarios_mensais_gerados",
        "valor": len(df_resumo_mensais),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_resumo_mensais) > 0 else "ERRO",
        "observacao": "A análise deve gerar cenários por tipo e por subamostra progressiva de controles mensais.",
    },
    {
        "item": "cenarios_estabilidade_gerados",
        "valor": len(df_estabilidade_mensais),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_estabilidade_mensais) > 0 else "ERRO",
        "observacao": "A análise deve gerar cenários de estabilidade excluindo tipos e controles individuais.",
    },
    {
        "item": "posicionamento_final_linhas",
        "valor": len(df_posicionamento_final_mensais),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_posicionamento_final_mensais) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar posicionamento final com todos os controles mensais disponíveis por estratégia e frequência.",
    },
    {
        "item": "impacto_temporal_linhas",
        "valor": len(df_impacto_temporal),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_impacto_temporal) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar uma visão temporal dos controles mensais para o HTML final.",
    },
    {
        "item": "base_grafico_linhas",
        "valor": len(df_base_grafico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Base gráfico sensibilidade mensais : {len(df_base_grafico):,} linhas")
print(f"Itens de auditoria                 : {len(df_auditoria):,}")
print(f"Erros bloqueantes                  : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(df_base_mensais, CAMINHO_BASE_MENSAIS, index=False)
salvar_dataframe(df_parametros_cenarios, CAMINHO_PARAMETROS_CENARIOS, index=False)
salvar_dataframe(df_resumo_mensais, CAMINHO_RESUMO_MENSAIS, index=False)
salvar_dataframe(df_distribuicao_mensais, CAMINHO_DISTRIBUICAO_MENSAIS, index=False)
salvar_dataframe(df_estabilidade_mensais, CAMINHO_ESTABILIDADE_MENSAIS, index=False)
salvar_dataframe(df_posicionamento_final_mensais, CAMINHO_POSICIONAMENTO_FINAL, index=False)
salvar_dataframe(df_impacto_temporal, CAMINHO_IMPACTO_TEMPORAL, index=False)
salvar_dataframe(df_base_grafico, CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da sensibilidade das estratégias mensais de controle:")
print(df_auditoria.to_string(index=False))

print("\nResumo de sensibilidade dos controles mensais - amostra:")
if len(df_resumo_mensais) > 0:
    colunas_resumo_print = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "cenario_mensal",
            "tipo_cenario",
            "tipo_controle_mensal",
            "n_controles_mensais",
            "pct_controles_retorno_acumulado_real_superou_controle",
            "pct_controles_sharpe_real_superou_controle",
            "pct_controles_sortino_real_superou_controle",
            "pct_controles_calmar_real_superou_controle",
            "media_excesso_anualizado_real_vs_controle_periodico",
            "classificacao_robustez_retorno_acumulado",
        ] if coluna in df_resumo_mensais.columns
    ]
    print(df_resumo_mensais[colunas_resumo_print].head(80).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nDistribuição por controle mensal - amostra:")
if len(df_distribuicao_mensais) > 0:
    colunas_dist_print = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "tipo_controle_mensal",
            "nome_exibicao_estrategia_controle",
            "n_aportes_mensais",
            "carteira_controle_mensal_numero",
            "pct_controles_retorno_acumulado_real_superou_controle",
            "media_pct_periodos_real_superou_controle",
            "media_excesso_anualizado_real_vs_controle_periodico",
        ] if coluna in df_distribuicao_mensais.columns
    ]
    print(df_distribuicao_mensais[colunas_dist_print].head(80).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nEstabilidade dos controles mensais - amostra:")
if len(df_estabilidade_mensais) > 0:
    colunas_est_print = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "tipo_estabilidade",
            "controle_excluido",
            "tipo_controle_mensal_excluido",
            "n_controles_mensais",
            "pct_controles_retorno_acumulado_real_superou_controle",
            "media_excesso_anualizado_real_vs_controle_periodico",
            "classificacao_robustez_retorno_acumulado",
        ] if coluna in df_estabilidade_mensais.columns
    ]
    print(df_estabilidade_mensais[colunas_est_print].head(80).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nPosicionamento final com todos os controles mensais:")
if len(df_posicionamento_final_mensais) > 0:
    colunas_pos_print = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "n_controles_mensais",
            "n_tipos_controle_mensal",
            "pct_controles_retorno_acumulado_real_superou_controle",
            "pct_controles_sharpe_real_superou_controle",
            "pct_controles_sortino_real_superou_controle",
            "pct_controles_calmar_real_superou_controle",
            "media_excesso_anualizado_real_vs_controle_periodico",
            "classificacao_robustez_retorno_acumulado",
        ] if coluna in df_posicionamento_final_mensais.columns
    ]
    print(df_posicionamento_final_mensais[colunas_pos_print].to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nArquivos salvos na subetapa 13.7:")
print(f"- {CAMINHO_BASE_MENSAIS}")
print(f"- {CAMINHO_PARAMETROS_CENARIOS}")
print(f"- {CAMINHO_RESUMO_MENSAIS}")
print(f"- {CAMINHO_DISTRIBUICAO_MENSAIS}")
print(f"- {CAMINHO_ESTABILIDADE_MENSAIS}")
print(f"- {CAMINHO_POSICIONAMENTO_FINAL}")
print(f"- {CAMINHO_IMPACTO_TEMPORAL}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 13.7 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 13.7 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 13.7 - SENSIBILIDADE DAS ESTRATÉGIAS MENSAIS DE CONTROLE

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - calendário mensais da 7.6          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_6_base_calendario_aportes_estrategias_mensais_controle.parquet
Entrada - resumo mensais da 7.6              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_6_tbl_resumo_estrategias_mensais_controle.parquet
Entrada - distribuição anual mensais da 7.6  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_7\7_6_tbl_distribuicao_anual_estrategias_mensais_controle.parquet
Entrada - estratégias mensais da 7.6         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_proje

# Etapa 14) Inferência Estatística

## Etapa 14.1) Preparação das Bases para Inferência Estatística

In [71]:
%%time
# ============================================================
# Etapa 14.1) Preparação das Bases para Inferência Estatística
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 14.1 - PREPARAÇÃO DAS BASES PARA INFERÊNCIA ESTATÍSTICA")
print("=" * 100)

# Esta subetapa não executa testes estatísticos.
# O objetivo é padronizar as séries pareadas de excesso de retorno que serão consumidas nas subetapas 14.2 a 14.8.
# A visão mensal é marcada como principal para inferência, a visão diária como complementar e a visão anual como exploratória.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/13] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/13] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])

if "etapa_14" in DIRETORIOS_PROJETO:
    DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
else:
    DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["resultados"]) / "etapa_14"
    DIRETORIOS_PROJETO["etapa_14"] = DIR_ETAPA_14

DIR_ETAPA_14.mkdir(parents=True, exist_ok=True)

CAMINHO_BENCHMARKS_DIARIA_11_5 = DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_diaria.parquet"
CAMINHO_BENCHMARKS_MENSAL_11_5 = DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_mensal.parquet"
CAMINHO_BENCHMARKS_ANUAL_11_5 = DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_anual.parquet"

CAMINHO_CONTROLES_DIARIA_11_6 = DIR_ETAPA_11 / "11_6_base_comparacao_controles_diaria.parquet"
CAMINHO_CONTROLES_MENSAL_11_6 = DIR_ETAPA_11 / "11_6_base_comparacao_controles_mensal.parquet"
CAMINHO_CONTROLES_ANUAL_11_6 = DIR_ETAPA_11 / "11_6_base_comparacao_controles_anual.parquet"
CAMINHO_PARES_CONTROLES_11_6 = DIR_ETAPA_11 / "11_6_tbl_pares_comparacao_controles.parquet"

CAMINHO_CAP_EUF_DIARIA_11_7 = DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_diaria.parquet"
CAMINHO_CAP_EUF_MENSAL_11_7 = DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_mensal.parquet"
CAMINHO_CAP_EUF_ANUAL_11_7 = DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_anual.parquet"

CAMINHO_BASE_BENCHMARKS = DIR_ETAPA_14 / "14_1_base_inferencia_benchmarks_pareada.parquet"
CAMINHO_BASE_CONTROLES = DIR_ETAPA_14 / "14_1_base_inferencia_controles_pareada.parquet"
CAMINHO_BASE_COMPARACAO_DIRETA = DIR_ETAPA_14 / "14_1_base_inferencia_comparacao_direta_pareada.parquet"
CAMINHO_BASE_CONSOLIDADA = DIR_ETAPA_14 / "14_1_base_inferencia_comparacoes_pareadas.parquet"
CAMINHO_MAPA_COMPARACOES = DIR_ETAPA_14 / "14_1_tbl_mapa_comparacoes_inferencia.parquet"
CAMINHO_RESUMO_BASES = DIR_ETAPA_14 / "14_1_tbl_resumo_bases_inferencia.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_14 / "14_1_tbl_parametros_inferencia_estatistica.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_14 / "14_1_tbl_auditoria_validacao_bases_inferencia.parquet"

CAMINHOS_OBRIGATORIOS = [
    CAMINHO_BENCHMARKS_DIARIA_11_5,
    CAMINHO_BENCHMARKS_MENSAL_11_5,
    CAMINHO_BENCHMARKS_ANUAL_11_5,
    CAMINHO_CONTROLES_DIARIA_11_6,
    CAMINHO_CONTROLES_MENSAL_11_6,
    CAMINHO_CONTROLES_ANUAL_11_6,
    CAMINHO_CAP_EUF_DIARIA_11_7,
    CAMINHO_CAP_EUF_MENSAL_11_7,
    CAMINHO_CAP_EUF_ANUAL_11_7,
]

for caminho in CAMINHOS_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - benchmarks diária da 11.5        : {CAMINHO_BENCHMARKS_DIARIA_11_5}")
print(f"Entrada - benchmarks mensal da 11.5        : {CAMINHO_BENCHMARKS_MENSAL_11_5}")
print(f"Entrada - benchmarks anual da 11.5         : {CAMINHO_BENCHMARKS_ANUAL_11_5}")
print(f"Entrada - controles diária da 11.6         : {CAMINHO_CONTROLES_DIARIA_11_6}")
print(f"Entrada - controles mensal da 11.6         : {CAMINHO_CONTROLES_MENSAL_11_6}")
print(f"Entrada - controles anual da 11.6          : {CAMINHO_CONTROLES_ANUAL_11_6}")
print(f"Entrada - pares de controles da 11.6       : {CAMINHO_PARES_CONTROLES_11_6}")
print(f"Entrada - cap. vs euforia diária da 11.7   : {CAMINHO_CAP_EUF_DIARIA_11_7}")
print(f"Entrada - cap. vs euforia mensal da 11.7   : {CAMINHO_CAP_EUF_MENSAL_11_7}")
print(f"Entrada - cap. vs euforia anual da 11.7    : {CAMINHO_CAP_EUF_ANUAL_11_7}")
print(f"Saída   - base benchmarks pareada          : {CAMINHO_BASE_BENCHMARKS}")
print(f"Saída   - base controles pareada           : {CAMINHO_BASE_CONTROLES}")
print(f"Saída   - base comparação direta pareada   : {CAMINHO_BASE_COMPARACAO_DIRETA}")
print(f"Saída   - base consolidada de inferência   : {CAMINHO_BASE_CONSOLIDADA}")
print(f"Saída   - mapa de comparações              : {CAMINHO_MAPA_COMPARACOES}")
print(f"Saída   - resumo das bases                 : {CAMINHO_RESUMO_BASES}")
print(f"Saída   - parâmetros de inferência         : {CAMINHO_PARAMETROS}")
print(f"Saída   - auditoria de validação           : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/13] Carga das bases oficiais da subetapa...")

df_benchmarks_diaria = pd.read_parquet(CAMINHO_BENCHMARKS_DIARIA_11_5)
df_benchmarks_mensal = pd.read_parquet(CAMINHO_BENCHMARKS_MENSAL_11_5)
df_benchmarks_anual = pd.read_parquet(CAMINHO_BENCHMARKS_ANUAL_11_5)

df_controles_diaria = pd.read_parquet(CAMINHO_CONTROLES_DIARIA_11_6)
df_controles_mensal = pd.read_parquet(CAMINHO_CONTROLES_MENSAL_11_6)
df_controles_anual = pd.read_parquet(CAMINHO_CONTROLES_ANUAL_11_6)

df_cap_euf_diaria = pd.read_parquet(CAMINHO_CAP_EUF_DIARIA_11_7)
df_cap_euf_mensal = pd.read_parquet(CAMINHO_CAP_EUF_MENSAL_11_7)
df_cap_euf_anual = pd.read_parquet(CAMINHO_CAP_EUF_ANUAL_11_7)

if CAMINHO_PARES_CONTROLES_11_6.exists():
    df_pares_controles = pd.read_parquet(CAMINHO_PARES_CONTROLES_11_6)
else:
    df_pares_controles = pd.DataFrame()

print(f"Benchmarks diária da 11.5       : {len(df_benchmarks_diaria):,} linhas x {df_benchmarks_diaria.shape[1]:,} colunas")
print(f"Benchmarks mensal da 11.5       : {len(df_benchmarks_mensal):,} linhas x {df_benchmarks_mensal.shape[1]:,} colunas")
print(f"Benchmarks anual da 11.5        : {len(df_benchmarks_anual):,} linhas x {df_benchmarks_anual.shape[1]:,} colunas")
print(f"Controles diária da 11.6        : {len(df_controles_diaria):,} linhas x {df_controles_diaria.shape[1]:,} colunas")
print(f"Controles mensal da 11.6        : {len(df_controles_mensal):,} linhas x {df_controles_mensal.shape[1]:,} colunas")
print(f"Controles anual da 11.6         : {len(df_controles_anual):,} linhas x {df_controles_anual.shape[1]:,} colunas")
print(f"Pares de controles da 11.6      : {len(df_pares_controles):,} linhas x {df_pares_controles.shape[1]:,} colunas")
print(f"Cap. vs euforia diária da 11.7  : {len(df_cap_euf_diaria):,} linhas x {df_cap_euf_diaria.shape[1]:,} colunas")
print(f"Cap. vs euforia mensal da 11.7  : {len(df_cap_euf_mensal):,} linhas x {df_cap_euf_mensal.shape[1]:,} colunas")
print(f"Cap. vs euforia anual da 11.7   : {len(df_cap_euf_anual):,} linhas x {df_cap_euf_anual.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/13] Funções auxiliares da subetapa...")

ESTRATEGIAS_REAIS_DETERMINISTICAS = {
    "capitulacao": "capitulacao__carteira_na__replica_na",
    "euforia": "euforia__carteira_na__replica_na",
}

FREQUENCIAS_ANALISE = ["diaria", "mensal", "anual"]

MAPA_VISAO_INFERENCIA = {
    "diaria": "complementar",
    "mensal": "principal",
    "anual": "exploratoria",
}

MAPA_DESCRICAO_VISAO = {
    "diaria": "visão diária complementar, sensível a autocorrelação e alta frequência",
    "mensal": "visão mensal principal para inferência estatística",
    "anual": "visão anual descritiva e exploratória, com baixa quantidade de observações",
}

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}


def validar_colunas_obrigatorias(df, colunas, nome_base):
    ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(ausentes) > 0:
        raise KeyError(
            f"Colunas obrigatórias ausentes na base {nome_base}: {ausentes}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )


def obter_coluna_opcional(df, coluna, valor_padrao=np.nan):
    if coluna in df.columns:
        return df[coluna]
    return pd.Series(valor_padrao, index=df.index)


def converter_data_opcional(df, coluna):
    if coluna in df.columns:
        return pd.to_datetime(df[coluna], errors="coerce").dt.normalize()
    return pd.Series(pd.NaT, index=df.index)


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def converter_booleano(serie):
    if pd.api.types.is_bool_dtype(serie):
        return serie.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce").fillna(0.0).ne(0.0)
    texto = serie.astype(str).str.strip().str.lower()
    return texto.isin(["true", "1", "sim", "s", "yes", "y", "ok", "verdadeiro"])


def calcular_retorno_relativo_composto(retorno_real, retorno_comparador):
    retorno_real = converter_numero(retorno_real)
    retorno_comparador = converter_numero(retorno_comparador)
    denominador = 1.0 + retorno_comparador
    resultado = np.where(
        np.isclose(denominador, 0.0, atol=0.0000000001, rtol=0.0),
        np.nan,
        (1.0 + retorno_real) / denominador - 1.0,
    )
    return pd.Series(resultado, index=retorno_real.index)


def obter_visao_inferencia(frequencia):
    return MAPA_VISAO_INFERENCIA.get(str(frequencia), "nao_classificada")


def obter_descricao_visao(frequencia):
    return MAPA_DESCRICAO_VISAO.get(str(frequencia), "frequência não classificada")


def obter_ordem_frequencia(frequencia):
    return MAPA_ORDEM_FREQUENCIA.get(str(frequencia), 99)


def ordenar_base_inferencia(df):
    colunas_ordenacao = [
        "ordem_frequencia_inferencia",
        "estrategia_referencia",
        "grupo_comparacao",
        "tipo_comparacao",
        "comparacao_id",
        "data_fim_periodo",
        "periodo_comparacao",
    ]
    colunas_ordenacao = [coluna for coluna in colunas_ordenacao if coluna in df.columns]
    if len(colunas_ordenacao) == 0:
        return df.reset_index(drop=True)
    return df.sort_values(colunas_ordenacao).reset_index(drop=True)


def preparar_auditoria(linhas):
    df = pd.DataFrame(linhas)
    if len(df) == 0:
        return pd.DataFrame(columns=["item", "valor", "valor_referencia", "status", "observacao"])
    df["status"] = df["status"].astype(str)
    return df[["item", "valor", "valor_referencia", "status", "observacao"]].copy()

def preparar_dataframe_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if df_saida[coluna].dtype == "object":
            serie_nao_nula = df_saida[coluna].dropna()
            if len(serie_nao_nula) == 0:
                df_saida[coluna] = df_saida[coluna].astype("string")
            elif serie_nao_nula.map(lambda valor: isinstance(valor, (bool, np.bool_))).all():
                df_saida[coluna] = df_saida[coluna].astype("boolean")
            else:
                df_saida[coluna] = df_saida[coluna].where(df_saida[coluna].notna(), None).astype("string")
    return df_saida

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Preparação das comparações pareadas contra Ibovespa e CDI
# ============================================================

print("\n[5/13] Preparação das comparações pareadas contra Ibovespa e CDI...")

BASES_BENCHMARKS = {
    "diaria": df_benchmarks_diaria,
    "mensal": df_benchmarks_mensal,
    "anual": df_benchmarks_anual,
}

BENCHMARKS_ANALISE = [
    {
        "benchmark_id": "ibovespa",
        "tipo_comparacao": "benchmark_ibovespa",
        "nome_comparador_padrao": "Ibovespa",
        "coluna_retorno": "retorno_periodo_ibovespa",
        "coluna_relativo": "retorno_relativo_composto_vs_ibovespa",
        "coluna_superou": "flag_superou_ibovespa",
        "coluna_empatou": "flag_empatou_ibovespa",
        "coluna_perdeu": "flag_perdeu_ibovespa",
        "coluna_chave": "benchmark_ibovespa_chave",
        "coluna_nome": "benchmark_ibovespa_nome",
    },
    {
        "benchmark_id": "cdi",
        "tipo_comparacao": "benchmark_cdi",
        "nome_comparador_padrao": "CDI",
        "coluna_retorno": "retorno_periodo_cdi",
        "coluna_relativo": "retorno_relativo_composto_vs_cdi",
        "coluna_superou": "flag_superou_cdi",
        "coluna_empatou": "flag_empatou_cdi",
        "coluna_perdeu": "flag_perdeu_cdi",
        "coluna_chave": "benchmark_cdi_chave",
        "coluna_nome": "benchmark_cdi_nome",
    },
]

registros_benchmarks = []
linhas_benchmarks_descartadas = []

for frequencia, df_base in BASES_BENCHMARKS.items():
    validar_colunas_obrigatorias(
        df_base,
        ["chave_estrategia", "periodo_comparacao", "retorno_periodo"],
        f"11.5 benchmarks {frequencia}",
    )

    for benchmark in BENCHMARKS_ANALISE:
        validar_colunas_obrigatorias(
            df_base,
            [benchmark["coluna_retorno"]],
            f"11.5 benchmarks {frequencia} - {benchmark['benchmark_id']}",
        )

    for estrategia_referencia, chave_real in ESTRATEGIAS_REAIS_DETERMINISTICAS.items():
        df_real = df_base.loc[df_base["chave_estrategia"].astype(str).eq(chave_real)].copy()

        for benchmark in BENCHMARKS_ANALISE:
            if len(df_real) == 0:
                continue

            df_temp = pd.DataFrame(index=df_real.index)
            df_temp["origem_comparacao"] = "etapa_11_5"
            df_temp["grupo_comparacao"] = "benchmark"
            df_temp["tipo_comparacao"] = benchmark["tipo_comparacao"]
            df_temp["subtipo_comparacao"] = benchmark["benchmark_id"]
            df_temp["comparacao_id"] = (
                estrategia_referencia
                + "__"
                + benchmark["tipo_comparacao"]
                + "__"
                + frequencia
            )
            df_temp["estrategia_referencia"] = estrategia_referencia
            df_temp["frequencia"] = frequencia
            df_temp["visao_inferencia"] = obter_visao_inferencia(frequencia)
            df_temp["descricao_visao_inferencia"] = obter_descricao_visao(frequencia)
            df_temp["ordem_frequencia_inferencia"] = obter_ordem_frequencia(frequencia)
            df_temp["periodo_comparacao"] = df_real["periodo_comparacao"].astype(str)
            df_temp["data_inicio_periodo"] = converter_data_opcional(df_real, "data_inicio_periodo")
            df_temp["data_fim_periodo"] = converter_data_opcional(df_real, "data_fim_periodo")
            df_temp["chave_estrategia_real"] = df_real["chave_estrategia"].astype(str)
            df_temp["estrategia_id_real"] = obter_coluna_opcional(df_real, "estrategia_id")
            df_temp["nome_exibicao_estrategia_real"] = obter_coluna_opcional(df_real, "nome_exibicao_estrategia")
            df_temp["categoria_estrategia_real"] = obter_coluna_opcional(df_real, "categoria_estrategia")
            df_temp["subcategoria_estrategia_real"] = obter_coluna_opcional(df_real, "subcategoria_estrategia")
            df_temp["familia_estrategia_real"] = obter_coluna_opcional(df_real, "familia_estrategia")
            df_temp["chave_comparador"] = obter_coluna_opcional(df_real, benchmark["coluna_chave"], benchmark["benchmark_id"])
            df_temp["nome_exibicao_comparador"] = obter_coluna_opcional(df_real, benchmark["coluna_nome"], benchmark["nome_comparador_padrao"])
            df_temp["retorno_periodo_real"] = converter_numero(df_real["retorno_periodo"])
            df_temp["retorno_periodo_comparador"] = converter_numero(df_real[benchmark["coluna_retorno"]])
            df_temp["retorno_excesso_real_vs_comparador"] = df_temp["retorno_periodo_real"] - df_temp["retorno_periodo_comparador"]

            if benchmark["coluna_relativo"] in df_real.columns:
                df_temp["retorno_relativo_composto_real_vs_comparador"] = converter_numero(df_real[benchmark["coluna_relativo"]])
            else:
                df_temp["retorno_relativo_composto_real_vs_comparador"] = calcular_retorno_relativo_composto(
                    df_temp["retorno_periodo_real"],
                    df_temp["retorno_periodo_comparador"],
                )

            if benchmark["coluna_superou"] in df_real.columns:
                df_temp["flag_real_superou_comparador"] = converter_booleano(df_real[benchmark["coluna_superou"]])
            else:
                df_temp["flag_real_superou_comparador"] = df_temp["retorno_excesso_real_vs_comparador"] > 0.0

            if benchmark["coluna_empatou"] in df_real.columns:
                df_temp["flag_real_empatou_comparador"] = converter_booleano(df_real[benchmark["coluna_empatou"]])
            else:
                df_temp["flag_real_empatou_comparador"] = np.isclose(
                    df_temp["retorno_excesso_real_vs_comparador"].fillna(np.nan),
                    0.0,
                    atol=0.0000000001,
                    rtol=0.0,
                )

            if benchmark["coluna_perdeu"] in df_real.columns:
                df_temp["flag_real_perdeu_comparador"] = converter_booleano(df_real[benchmark["coluna_perdeu"]])
            else:
                df_temp["flag_real_perdeu_comparador"] = df_temp["retorno_excesso_real_vs_comparador"] < 0.0

            df_temp["flag_serie_principal_inferencia"] = df_temp["frequencia"].eq("mensal")
            df_temp["flag_serie_complementar_inferencia"] = df_temp["frequencia"].eq("diaria")
            df_temp["flag_serie_exploratoria_inferencia"] = df_temp["frequencia"].eq("anual")

            linhas_antes = len(df_temp)
            df_temp = df_temp.dropna(
                subset=[
                    "retorno_periodo_real",
                    "retorno_periodo_comparador",
                    "retorno_excesso_real_vs_comparador",
                ]
            ).copy()
            linhas_depois = len(df_temp)
            linhas_benchmarks_descartadas.append(linhas_antes - linhas_depois)

            registros_benchmarks.append(df_temp)

if len(registros_benchmarks) > 0:
    df_base_benchmarks = pd.concat(registros_benchmarks, ignore_index=True)
else:
    df_base_benchmarks = pd.DataFrame()

df_base_benchmarks = ordenar_base_inferencia(df_base_benchmarks)

print(f"Base pareada contra benchmarks           : {len(df_base_benchmarks):,} linhas")
print(f"Comparações contra benchmarks            : {df_base_benchmarks['comparacao_id'].nunique() if len(df_base_benchmarks) > 0 else 0:,}")
print(f"Linhas descartadas por ausência de retorno: {int(np.sum(linhas_benchmarks_descartadas)):,}")
print("OK")

# ============================================================
# 6) Preparação das comparações pareadas contra controles
# ============================================================

print("\n[6/13] Preparação das comparações pareadas contra controles...")

BASES_CONTROLES = {
    "diaria": df_controles_diaria,
    "mensal": df_controles_mensal,
    "anual": df_controles_anual,
}

registros_controles = []
linhas_controles_descartadas = []

for frequencia, df_base in BASES_CONTROLES.items():
    validar_colunas_obrigatorias(
        df_base,
        [
            "comparacao_id",
            "estrategia_referencia",
            "periodo_comparacao",
            "retorno_periodo_real",
            "retorno_periodo_controle",
            "retorno_excesso_real_vs_controle",
        ],
        f"11.6 controles {frequencia}",
    )

    df_temp = pd.DataFrame(index=df_base.index)
    df_temp["origem_comparacao"] = "etapa_11_6"
    df_temp["grupo_comparacao"] = obter_coluna_opcional(df_base, "grupo_comparacao_controle", "controle")
    df_temp["tipo_comparacao"] = obter_coluna_opcional(df_base, "tipo_comparacao_controle", "controle")
    df_temp["subtipo_comparacao"] = df_temp["tipo_comparacao"].astype(str)
    df_temp["comparacao_id"] = df_base["comparacao_id"].astype(str) + "__" + frequencia
    df_temp["comparacao_id_original"] = df_base["comparacao_id"].astype(str)
    df_temp["estrategia_referencia"] = df_base["estrategia_referencia"].astype(str)
    df_temp["frequencia"] = frequencia
    df_temp["visao_inferencia"] = obter_visao_inferencia(frequencia)
    df_temp["descricao_visao_inferencia"] = obter_descricao_visao(frequencia)
    df_temp["ordem_frequencia_inferencia"] = obter_ordem_frequencia(frequencia)
    df_temp["periodo_comparacao"] = df_base["periodo_comparacao"].astype(str)
    df_temp["data_inicio_periodo"] = converter_data_opcional(df_base, "data_inicio_periodo")
    df_temp["data_fim_periodo"] = converter_data_opcional(df_base, "data_fim_periodo")
    df_temp["chave_estrategia_real"] = obter_coluna_opcional(df_base, "chave_estrategia_real")
    df_temp["estrategia_id_real"] = obter_coluna_opcional(df_base, "estrategia_id_real")
    df_temp["nome_exibicao_estrategia_real"] = obter_coluna_opcional(df_base, "nome_exibicao_estrategia_real")
    df_temp["categoria_estrategia_real"] = obter_coluna_opcional(df_base, "categoria_estrategia_real")
    df_temp["subcategoria_estrategia_real"] = obter_coluna_opcional(df_base, "subcategoria_estrategia_real")
    df_temp["familia_estrategia_real"] = obter_coluna_opcional(df_base, "familia_estrategia_real")
    df_temp["chave_comparador"] = obter_coluna_opcional(df_base, "chave_estrategia_controle")
    df_temp["estrategia_id_comparador"] = obter_coluna_opcional(df_base, "estrategia_id_controle")
    df_temp["nome_exibicao_comparador"] = obter_coluna_opcional(df_base, "nome_exibicao_estrategia_controle")
    df_temp["categoria_comparador"] = obter_coluna_opcional(df_base, "categoria_estrategia_controle")
    df_temp["subcategoria_comparador"] = obter_coluna_opcional(df_base, "subcategoria_estrategia_controle")
    df_temp["familia_comparador"] = obter_coluna_opcional(df_base, "familia_estrategia_controle")
    df_temp["retorno_periodo_real"] = converter_numero(df_base["retorno_periodo_real"])
    df_temp["retorno_periodo_comparador"] = converter_numero(df_base["retorno_periodo_controle"])
    df_temp["retorno_excesso_real_vs_comparador"] = converter_numero(df_base["retorno_excesso_real_vs_controle"])

    if "retorno_relativo_composto_real_vs_controle" in df_base.columns:
        df_temp["retorno_relativo_composto_real_vs_comparador"] = converter_numero(df_base["retorno_relativo_composto_real_vs_controle"])
    else:
        df_temp["retorno_relativo_composto_real_vs_comparador"] = calcular_retorno_relativo_composto(
            df_temp["retorno_periodo_real"],
            df_temp["retorno_periodo_comparador"],
        )

    if "flag_real_superou_controle" in df_base.columns:
        df_temp["flag_real_superou_comparador"] = converter_booleano(df_base["flag_real_superou_controle"])
    else:
        df_temp["flag_real_superou_comparador"] = df_temp["retorno_excesso_real_vs_comparador"] > 0.0

    if "flag_real_empatou_controle" in df_base.columns:
        df_temp["flag_real_empatou_comparador"] = converter_booleano(df_base["flag_real_empatou_controle"])
    else:
        df_temp["flag_real_empatou_comparador"] = np.isclose(
            df_temp["retorno_excesso_real_vs_comparador"].fillna(np.nan),
            0.0,
            atol=0.0000000001,
            rtol=0.0,
        )

    if "flag_real_perdeu_controle" in df_base.columns:
        df_temp["flag_real_perdeu_comparador"] = converter_booleano(df_base["flag_real_perdeu_controle"])
    else:
        df_temp["flag_real_perdeu_comparador"] = df_temp["retorno_excesso_real_vs_comparador"] < 0.0

    df_temp["flag_serie_principal_inferencia"] = df_temp["frequencia"].eq("mensal")
    df_temp["flag_serie_complementar_inferencia"] = df_temp["frequencia"].eq("diaria")
    df_temp["flag_serie_exploratoria_inferencia"] = df_temp["frequencia"].eq("anual")

    linhas_antes = len(df_temp)
    df_temp = df_temp.dropna(
        subset=[
            "retorno_periodo_real",
            "retorno_periodo_comparador",
            "retorno_excesso_real_vs_comparador",
        ]
    ).copy()
    linhas_depois = len(df_temp)
    linhas_controles_descartadas.append(linhas_antes - linhas_depois)

    registros_controles.append(df_temp)

if len(registros_controles) > 0:
    df_base_controles = pd.concat(registros_controles, ignore_index=True)
else:
    df_base_controles = pd.DataFrame()

df_base_controles = ordenar_base_inferencia(df_base_controles)

print(f"Base pareada contra controles            : {len(df_base_controles):,} linhas")
print(f"Comparações contra controles             : {df_base_controles['comparacao_id'].nunique() if len(df_base_controles) > 0 else 0:,}")
print(f"Linhas descartadas por ausência de retorno: {int(np.sum(linhas_controles_descartadas)):,}")
print("OK")

# ============================================================
# 7) Preparação da comparação direta entre Capitulação e Euforia
# ============================================================

print("\n[7/13] Preparação da comparação direta entre Capitulação e Euforia...")

BASES_COMPARACAO_DIRETA = {
    "diaria": df_cap_euf_diaria,
    "mensal": df_cap_euf_mensal,
    "anual": df_cap_euf_anual,
}

registros_comparacao_direta = []
linhas_diretas_descartadas = []

for frequencia, df_base in BASES_COMPARACAO_DIRETA.items():
    validar_colunas_obrigatorias(
        df_base,
        [
            "frequencia",
            "periodo_comparacao",
            "retorno_periodo_capitulacao",
            "retorno_periodo_euforia",
            "excesso_capitulacao_vs_euforia",
        ],
        f"11.7 comparação direta {frequencia}",
    )

    df_temp = pd.DataFrame(index=df_base.index)
    df_temp["origem_comparacao"] = "etapa_11_7"
    df_temp["grupo_comparacao"] = "comparacao_direta"
    df_temp["tipo_comparacao"] = "capitulacao_vs_euforia"
    df_temp["subtipo_comparacao"] = "capitulacao_vs_euforia"
    df_temp["comparacao_id"] = "capitulacao__vs__euforia__" + frequencia
    df_temp["comparacao_id_original"] = "capitulacao__vs__euforia"
    df_temp["estrategia_referencia"] = "capitulacao"
    df_temp["frequencia"] = frequencia
    df_temp["visao_inferencia"] = obter_visao_inferencia(frequencia)
    df_temp["descricao_visao_inferencia"] = obter_descricao_visao(frequencia)
    df_temp["ordem_frequencia_inferencia"] = obter_ordem_frequencia(frequencia)
    df_temp["periodo_comparacao"] = df_base["periodo_comparacao"].astype(str)
    df_temp["data_inicio_periodo"] = converter_data_opcional(df_base, "data_inicio_periodo")
    df_temp["data_fim_periodo"] = converter_data_opcional(df_base, "data_fim_periodo")
    df_temp["chave_estrategia_real"] = obter_coluna_opcional(df_base, "chave_capitulacao", ESTRATEGIAS_REAIS_DETERMINISTICAS["capitulacao"])
    df_temp["estrategia_id_real"] = obter_coluna_opcional(df_base, "estrategia_id_capitulacao", "capitulacao")
    df_temp["nome_exibicao_estrategia_real"] = obter_coluna_opcional(df_base, "nome_exibicao_capitulacao", "Capitulação")
    df_temp["categoria_estrategia_real"] = obter_coluna_opcional(df_base, "categoria_capitulacao", "estrategia_real")
    df_temp["subcategoria_estrategia_real"] = obter_coluna_opcional(df_base, "subcategoria_capitulacao", "capitulacao")
    df_temp["familia_estrategia_real"] = obter_coluna_opcional(df_base, "familia_capitulacao", "capitulacao")
    df_temp["chave_comparador"] = obter_coluna_opcional(df_base, "chave_euforia", ESTRATEGIAS_REAIS_DETERMINISTICAS["euforia"])
    df_temp["estrategia_id_comparador"] = obter_coluna_opcional(df_base, "estrategia_id_euforia", "euforia")
    df_temp["nome_exibicao_comparador"] = obter_coluna_opcional(df_base, "nome_exibicao_euforia", "Euforia")
    df_temp["categoria_comparador"] = obter_coluna_opcional(df_base, "categoria_euforia", "estrategia_real")
    df_temp["subcategoria_comparador"] = obter_coluna_opcional(df_base, "subcategoria_euforia", "euforia")
    df_temp["familia_comparador"] = obter_coluna_opcional(df_base, "familia_euforia", "euforia")
    df_temp["retorno_periodo_real"] = converter_numero(df_base["retorno_periodo_capitulacao"])
    df_temp["retorno_periodo_comparador"] = converter_numero(df_base["retorno_periodo_euforia"])
    df_temp["retorno_excesso_real_vs_comparador"] = converter_numero(df_base["excesso_capitulacao_vs_euforia"])

    if "retorno_relativo_composto_capitulacao_vs_euforia" in df_base.columns:
        df_temp["retorno_relativo_composto_real_vs_comparador"] = converter_numero(df_base["retorno_relativo_composto_capitulacao_vs_euforia"])
    else:
        df_temp["retorno_relativo_composto_real_vs_comparador"] = calcular_retorno_relativo_composto(
            df_temp["retorno_periodo_real"],
            df_temp["retorno_periodo_comparador"],
        )

    if "flag_capitulacao_superou_euforia" in df_base.columns:
        df_temp["flag_real_superou_comparador"] = converter_booleano(df_base["flag_capitulacao_superou_euforia"])
    else:
        df_temp["flag_real_superou_comparador"] = df_temp["retorno_excesso_real_vs_comparador"] > 0.0

    if "flag_empate_capitulacao_euforia" in df_base.columns:
        df_temp["flag_real_empatou_comparador"] = converter_booleano(df_base["flag_empate_capitulacao_euforia"])
    else:
        df_temp["flag_real_empatou_comparador"] = np.isclose(
            df_temp["retorno_excesso_real_vs_comparador"].fillna(np.nan),
            0.0,
            atol=0.0000000001,
            rtol=0.0,
        )

    if "flag_euforia_superou_capitulacao" in df_base.columns:
        df_temp["flag_real_perdeu_comparador"] = converter_booleano(df_base["flag_euforia_superou_capitulacao"])
    else:
        df_temp["flag_real_perdeu_comparador"] = df_temp["retorno_excesso_real_vs_comparador"] < 0.0

    df_temp["flag_serie_principal_inferencia"] = df_temp["frequencia"].eq("mensal")
    df_temp["flag_serie_complementar_inferencia"] = df_temp["frequencia"].eq("diaria")
    df_temp["flag_serie_exploratoria_inferencia"] = df_temp["frequencia"].eq("anual")

    linhas_antes = len(df_temp)
    df_temp = df_temp.dropna(
        subset=[
            "retorno_periodo_real",
            "retorno_periodo_comparador",
            "retorno_excesso_real_vs_comparador",
        ]
    ).copy()
    linhas_depois = len(df_temp)
    linhas_diretas_descartadas.append(linhas_antes - linhas_depois)

    registros_comparacao_direta.append(df_temp)

if len(registros_comparacao_direta) > 0:
    df_base_comparacao_direta = pd.concat(registros_comparacao_direta, ignore_index=True)
else:
    df_base_comparacao_direta = pd.DataFrame()

df_base_comparacao_direta = ordenar_base_inferencia(df_base_comparacao_direta)

print(f"Base pareada Cap. vs Euforia             : {len(df_base_comparacao_direta):,} linhas")
print(f"Comparações Cap. vs Euforia              : {df_base_comparacao_direta['comparacao_id'].nunique() if len(df_base_comparacao_direta) > 0 else 0:,}")
print(f"Linhas descartadas por ausência de retorno: {int(np.sum(linhas_diretas_descartadas)):,}")
print("OK")

# ============================================================
# 8) Consolidação da base única de inferência
# ============================================================

print("\n[8/13] Consolidação da base única de inferência...")

colunas_padrao = [
    "origem_comparacao",
    "grupo_comparacao",
    "tipo_comparacao",
    "subtipo_comparacao",
    "comparacao_id",
    "comparacao_id_original",
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "descricao_visao_inferencia",
    "ordem_frequencia_inferencia",
    "periodo_comparacao",
    "data_inicio_periodo",
    "data_fim_periodo",
    "chave_estrategia_real",
    "estrategia_id_real",
    "nome_exibicao_estrategia_real",
    "categoria_estrategia_real",
    "subcategoria_estrategia_real",
    "familia_estrategia_real",
    "chave_comparador",
    "estrategia_id_comparador",
    "nome_exibicao_comparador",
    "categoria_comparador",
    "subcategoria_comparador",
    "familia_comparador",
    "retorno_periodo_real",
    "retorno_periodo_comparador",
    "retorno_excesso_real_vs_comparador",
    "retorno_relativo_composto_real_vs_comparador",
    "flag_real_superou_comparador",
    "flag_real_empatou_comparador",
    "flag_real_perdeu_comparador",
    "flag_serie_principal_inferencia",
    "flag_serie_complementar_inferencia",
    "flag_serie_exploratoria_inferencia",
]

for df_nome, df_ref in [
    ("benchmarks", df_base_benchmarks),
    ("controles", df_base_controles),
    ("comparacao_direta", df_base_comparacao_direta),
]:
    for coluna in colunas_padrao:
        if coluna not in df_ref.columns:
            df_ref[coluna] = np.nan

    if df_nome == "benchmarks":
        df_base_benchmarks = df_ref.copy()
    elif df_nome == "controles":
        df_base_controles = df_ref.copy()
    else:
        df_base_comparacao_direta = df_ref.copy()

df_base_inferencia = pd.concat(
    [
        df_base_benchmarks[colunas_padrao],
        df_base_controles[colunas_padrao],
        df_base_comparacao_direta[colunas_padrao],
    ],
    ignore_index=True,
)

df_base_inferencia = ordenar_base_inferencia(df_base_inferencia)

print(f"Base consolidada de inferência           : {len(df_base_inferencia):,} linhas")
print(f"Comparações consolidadas                 : {df_base_inferencia['comparacao_id'].nunique() if len(df_base_inferencia) > 0 else 0:,}")
print(f"Estratégias de referência                : {df_base_inferencia['estrategia_referencia'].nunique() if len(df_base_inferencia) > 0 else 0:,}")
print(f"Frequências preservadas                  : {df_base_inferencia['frequencia'].nunique() if len(df_base_inferencia) > 0 else 0:,}")
print("OK")

# ============================================================
# 9) Construção do mapa de comparações para inferência
# ============================================================

print("\n[9/13] Construção do mapa de comparações para inferência...")

if len(df_base_inferencia) > 0:
    agrupadores_mapa = [
        "comparacao_id",
        "origem_comparacao",
        "grupo_comparacao",
        "tipo_comparacao",
        "subtipo_comparacao",
        "estrategia_referencia",
        "frequencia",
        "visao_inferencia",
        "descricao_visao_inferencia",
        "chave_estrategia_real",
        "nome_exibicao_estrategia_real",
        "chave_comparador",
        "nome_exibicao_comparador",
    ]

    df_mapa_comparacoes = (
        df_base_inferencia
        .groupby(agrupadores_mapa, dropna=False)
        .agg(
            n_periodos=("periodo_comparacao", "count"),
            n_periodos_validos_excesso=("retorno_excesso_real_vs_comparador", lambda x: int(pd.to_numeric(x, errors="coerce").notna().sum())),
            data_inicio_min=("data_inicio_periodo", "min"),
            data_fim_max=("data_fim_periodo", "max"),
            media_excesso_periodico=("retorno_excesso_real_vs_comparador", "mean"),
            mediana_excesso_periodico=("retorno_excesso_real_vs_comparador", "median"),
            desvio_excesso_periodico=("retorno_excesso_real_vs_comparador", "std"),
            pct_periodos_real_superou_comparador=("flag_real_superou_comparador", "mean"),
        )
        .reset_index()
    )
else:
    df_mapa_comparacoes = pd.DataFrame()

if len(df_mapa_comparacoes) > 0:
    df_mapa_comparacoes["flag_comparacao_principal_inferencia"] = df_mapa_comparacoes["frequencia"].eq("mensal")
    df_mapa_comparacoes["flag_comparacao_complementar_inferencia"] = df_mapa_comparacoes["frequencia"].eq("diaria")
    df_mapa_comparacoes["flag_comparacao_exploratoria_inferencia"] = df_mapa_comparacoes["frequencia"].eq("anual")
    df_mapa_comparacoes["ordem_frequencia_inferencia"] = df_mapa_comparacoes["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
    df_mapa_comparacoes["aplicacao_metodologica"] = np.select(
        [
            df_mapa_comparacoes["frequencia"].eq("mensal"),
            df_mapa_comparacoes["frequencia"].eq("diaria"),
            df_mapa_comparacoes["frequencia"].eq("anual"),
        ],
        [
            "principal_para_testes_formais",
            "complementar_para_testes_de_robustez",
            "descritiva_exploratoria",
        ],
        default="nao_classificada",
    )
    df_mapa_comparacoes = ordenar_base_inferencia(df_mapa_comparacoes)

print(f"Mapa de comparações de inferência        : {len(df_mapa_comparacoes):,} linhas")
print(f"Comparações mensais principais           : {int((df_mapa_comparacoes['frequencia'] == 'mensal').sum()) if len(df_mapa_comparacoes) > 0 else 0:,}")
print(f"Comparações diárias complementares       : {int((df_mapa_comparacoes['frequencia'] == 'diaria').sum()) if len(df_mapa_comparacoes) > 0 else 0:,}")
print(f"Comparações anuais exploratórias         : {int((df_mapa_comparacoes['frequencia'] == 'anual').sum()) if len(df_mapa_comparacoes) > 0 else 0:,}")
print("OK")

# ============================================================
# 10) Resumo das bases de inferência
# ============================================================

print("\n[10/13] Resumo das bases de inferência...")

if len(df_base_inferencia) > 0:
    df_resumo_bases = (
        df_base_inferencia
        .groupby(["grupo_comparacao", "tipo_comparacao", "estrategia_referencia", "frequencia", "visao_inferencia"], dropna=False)
        .agg(
            n_comparacoes=("comparacao_id", "nunique"),
            n_linhas=("comparacao_id", "size"),
            n_periodos_unicos=("periodo_comparacao", "nunique"),
            n_periodos_validos_excesso=("retorno_excesso_real_vs_comparador", lambda x: int(pd.to_numeric(x, errors="coerce").notna().sum())),
            media_excesso_periodico=("retorno_excesso_real_vs_comparador", "mean"),
            mediana_excesso_periodico=("retorno_excesso_real_vs_comparador", "median"),
            pct_periodos_real_superou_comparador=("flag_real_superou_comparador", "mean"),
            data_inicio_min=("data_inicio_periodo", "min"),
            data_fim_max=("data_fim_periodo", "max"),
        )
        .reset_index()
    )
    df_resumo_bases["ordem_frequencia_inferencia"] = df_resumo_bases["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
    df_resumo_bases = ordenar_base_inferencia(df_resumo_bases)
else:
    df_resumo_bases = pd.DataFrame()

print(f"Resumo das bases de inferência           : {len(df_resumo_bases):,} linhas")
if len(df_resumo_bases) > 0:
    print(df_resumo_bases.head(20).to_string(index=False))
else:
    print("Sem linhas para exibir.")
print("OK")

# ============================================================
# 11) Parâmetros metodológicos para as subetapas estatísticas
# ============================================================

print("\n[11/13] Parâmetros metodológicos para as subetapas estatísticas...")

PARAMETROS_INFERENCIA_14_1 = {
    "etapa": "14",
    "subetapa": "14.1",
    "nome_subetapa": "Preparação das Bases para Inferência Estatística",
    "frequencia_principal_inferencia": "mensal",
    "frequencia_complementar_inferencia": "diaria",
    "frequencia_exploratoria": "anual",
    "comparacoes_benchmark": "Ibovespa e CDI",
    "comparacoes_controle": "estratégias aleatórias e estratégias mensais de controle",
    "comparacoes_diretas": "Capitulação contra Euforia",
    "metrica_pareada_principal": "retorno_excesso_real_vs_comparador",
    "hipotese_nula_teste_sinal": "probabilidade de superação igual a 50%",
    "min_observacoes_recomendado_testes_pareados": 12,
    "min_observacoes_recomendado_bootstrap": 12,
    "alpha_referencia": 0.05,
    "alpha_indicativo": 0.10,
    "ajuste_multiplas_comparacoes_planejado": "Holm na Etapa 14.8",
    "observacao_metodologica": "Esta subetapa prepara bases e não executa testes estatísticos.",
}

df_parametros_inferencia = pd.DataFrame(
    [{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_INFERENCIA_14_1.items()]
)

print(df_parametros_inferencia.to_string(index=False))
print("OK")

# ============================================================
# 12) Auditoria de validação das bases preparadas
# ============================================================

print("\n[12/13] Auditoria de validação das bases preparadas...")

if len(df_base_inferencia) > 0:
    frequencias_identificadas = int(df_base_inferencia["frequencia"].nunique())
    estrategias_identificadas = int(df_base_inferencia["estrategia_referencia"].nunique())
    comparacoes_mensais = int((df_mapa_comparacoes["frequencia"] == "mensal").sum()) if len(df_mapa_comparacoes) > 0 else 0
    comparacoes_diarias = int((df_mapa_comparacoes["frequencia"] == "diaria").sum()) if len(df_mapa_comparacoes) > 0 else 0
    comparacoes_anuais = int((df_mapa_comparacoes["frequencia"] == "anual").sum()) if len(df_mapa_comparacoes) > 0 else 0
    n_benchmarks = int(df_base_inferencia.loc[df_base_inferencia["grupo_comparacao"].astype(str).eq("benchmark"), "tipo_comparacao"].nunique())
    n_controles_aleatorios = int((df_base_inferencia["tipo_comparacao"].astype(str) == "controle_aleatorio").sum())
    n_controles_mensais = int((df_base_inferencia["tipo_comparacao"].astype(str) == "controle_mensal").sum())
    n_comparacao_direta = int((df_base_inferencia["tipo_comparacao"].astype(str) == "capitulacao_vs_euforia").sum())
    duplicatas_comparacao_periodo = int(df_base_inferencia.duplicated(["comparacao_id", "frequencia", "periodo_comparacao"]).sum())
    valores_excesso_ausentes = int(df_base_inferencia["retorno_excesso_real_vs_comparador"].isna().sum())
else:
    frequencias_identificadas = 0
    estrategias_identificadas = 0
    comparacoes_mensais = 0
    comparacoes_diarias = 0
    comparacoes_anuais = 0
    n_benchmarks = 0
    n_controles_aleatorios = 0
    n_controles_mensais = 0
    n_comparacao_direta = 0
    duplicatas_comparacao_periodo = 0
    valores_excesso_ausentes = 0

metricas_infinitas = 0
for df_verificacao in [df_base_benchmarks, df_base_controles, df_base_comparacao_direta, df_base_inferencia, df_mapa_comparacoes, df_resumo_bases]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

linhas_auditoria = [
    {
        "item": "linhas_base_benchmarks_pareada",
        "valor": len(df_base_benchmarks),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_benchmarks) > 0 else "ERRO",
        "observacao": "A preparação deve gerar base pareada contra Ibovespa e CDI a partir da Etapa 11.5.",
    },
    {
        "item": "linhas_base_controles_pareada",
        "valor": len(df_base_controles),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_controles) > 0 else "ERRO",
        "observacao": "A preparação deve gerar base pareada contra controles aleatórios e mensais a partir da Etapa 11.6.",
    },
    {
        "item": "linhas_base_consolidada",
        "valor": len(df_base_inferencia),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_inferencia) > 0 else "ERRO",
        "observacao": "A base consolidada é o insumo principal das subetapas 14.2 a 14.8.",
    },
    {
        "item": "estrategias_referencia_identificadas",
        "valor": estrategias_identificadas,
        "valor_referencia": ">= 2",
        "status": "OK" if estrategias_identificadas >= 2 else "ERRO",
        "observacao": "Espera-se encontrar Capitulação e Euforia.",
    },
    {
        "item": "frequencias_identificadas",
        "valor": frequencias_identificadas,
        "valor_referencia": ">= 3",
        "status": "OK" if frequencias_identificadas >= 3 else "ERRO",
        "observacao": "A preparação deve preservar as visões diária, mensal e anual.",
    },
    {
        "item": "comparacoes_mensais_principais",
        "valor": comparacoes_mensais,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_mensais > 0 else "ERRO",
        "observacao": "A visão mensal deve existir porque será a principal para inferência estatística.",
    },
    {
        "item": "comparacoes_diarias_complementares",
        "valor": comparacoes_diarias,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_diarias > 0 else "ERRO",
        "observacao": "A visão diária deve ser preservada como análise complementar.",
    },
    {
        "item": "comparacoes_anuais_exploratorias",
        "valor": comparacoes_anuais,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_anuais > 0 else "ERRO",
        "observacao": "A visão anual deve ser preservada como análise descritiva e exploratória.",
    },
    {
        "item": "benchmarks_identificados",
        "valor": n_benchmarks,
        "valor_referencia": ">= 2",
        "status": "OK" if n_benchmarks >= 2 else "ERRO",
        "observacao": "Devem existir comparações contra Ibovespa e CDI.",
    },
    {
        "item": "linhas_controle_aleatorio",
        "valor": n_controles_aleatorios,
        "valor_referencia": "> 0",
        "status": "OK" if n_controles_aleatorios > 0 else "ERRO",
        "observacao": "A base deve preservar comparações contra estratégias aleatórias.",
    },
    {
        "item": "linhas_controle_mensal",
        "valor": n_controles_mensais,
        "valor_referencia": "> 0",
        "status": "OK" if n_controles_mensais > 0 else "ERRO",
        "observacao": "A base deve preservar comparações contra estratégias mensais de controle.",
    },
    {
        "item": "linhas_comparacao_direta_capitulacao_euforia",
        "valor": n_comparacao_direta,
        "valor_referencia": "> 0",
        "status": "OK" if n_comparacao_direta > 0 else "ERRO",
        "observacao": "A base deve preservar a comparação direta entre Capitulação e Euforia preparada na 11.7.",
    },
    {
        "item": "duplicatas_comparacao_frequencia_periodo",
        "valor": duplicatas_comparacao_periodo,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_comparacao_periodo == 0 else "ERRO",
        "observacao": "Cada comparação deve ter no máximo uma linha por frequência e período.",
    },
    {
        "item": "valores_excesso_ausentes",
        "valor": valores_excesso_ausentes,
        "valor_referencia": "0",
        "status": "OK" if valores_excesso_ausentes == 0 else "ERRO",
        "observacao": "As séries preparadas para inferência não devem conter excesso de retorno ausente.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Itens de auditoria                       : {len(df_auditoria):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 13) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[13/13] Salvamento dos outputs e validação final da subetapa...")

df_base_benchmarks_saida = preparar_dataframe_para_parquet(df_base_benchmarks)
df_base_controles_saida = preparar_dataframe_para_parquet(df_base_controles)
df_base_comparacao_direta_saida = preparar_dataframe_para_parquet(df_base_comparacao_direta)
df_base_inferencia_saida = preparar_dataframe_para_parquet(df_base_inferencia)
df_mapa_comparacoes_saida = preparar_dataframe_para_parquet(df_mapa_comparacoes)
df_resumo_bases_saida = preparar_dataframe_para_parquet(df_resumo_bases)
df_parametros_inferencia_saida = preparar_dataframe_para_parquet(df_parametros_inferencia)
df_auditoria_saida = preparar_dataframe_para_parquet(df_auditoria)

salvar_dataframe(df_base_benchmarks_saida, CAMINHO_BASE_BENCHMARKS, index=False)
salvar_dataframe(df_base_controles_saida, CAMINHO_BASE_CONTROLES, index=False)
salvar_dataframe(df_base_comparacao_direta_saida, CAMINHO_BASE_COMPARACAO_DIRETA, index=False)
salvar_dataframe(df_base_inferencia_saida, CAMINHO_BASE_CONSOLIDADA, index=False)
salvar_dataframe(df_mapa_comparacoes_saida, CAMINHO_MAPA_COMPARACOES, index=False)
salvar_dataframe(df_resumo_bases_saida, CAMINHO_RESUMO_BASES, index=False)
salvar_dataframe(df_parametros_inferencia_saida, CAMINHO_PARAMETROS, index=False)
salvar_dataframe(df_auditoria_saida, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação das bases de inferência:")
print(df_auditoria.to_string(index=False))

print("\nMapa de comparações de inferência - amostra:")
if len(df_mapa_comparacoes) > 0:
    colunas_mapa_print = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "visao_inferencia",
            "grupo_comparacao",
            "tipo_comparacao",
            "nome_exibicao_comparador",
            "n_periodos",
            "media_excesso_periodico",
            "pct_periodos_real_superou_comparador",
            "aplicacao_metodologica",
        ] if coluna in df_mapa_comparacoes.columns
    ]
    print(df_mapa_comparacoes[colunas_mapa_print].head(80).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nResumo das bases de inferência:")
if len(df_resumo_bases) > 0:
    colunas_resumo_print = [
        coluna for coluna in [
            "grupo_comparacao",
            "tipo_comparacao",
            "estrategia_referencia",
            "frequencia",
            "visao_inferencia",
            "n_comparacoes",
            "n_linhas",
            "n_periodos_unicos",
            "media_excesso_periodico",
            "pct_periodos_real_superou_comparador",
        ] if coluna in df_resumo_bases.columns
    ]
    print(df_resumo_bases[colunas_resumo_print].to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nArquivos salvos na subetapa 14.1:")
print(f"- {CAMINHO_BASE_BENCHMARKS}")
print(f"- {CAMINHO_BASE_CONTROLES}")
print(f"- {CAMINHO_BASE_COMPARACAO_DIRETA}")
print(f"- {CAMINHO_BASE_CONSOLIDADA}")
print(f"- {CAMINHO_MAPA_COMPARACOES}")
print(f"- {CAMINHO_RESUMO_BASES}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 14.1 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 14.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 14.1 - PREPARAÇÃO DAS BASES PARA INFERÊNCIA ESTATÍSTICA

[1/13] Validação inicial do ambiente...
OK

[2/13] Definição determinística dos caminhos de entrada e saída...
Entrada - benchmarks diária da 11.5        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_5_base_comparacao_benchmarks_diaria.parquet
Entrada - benchmarks mensal da 11.5        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_5_base_comparacao_benchmarks_mensal.parquet
Entrada - benchmarks anual da 11.5         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_5_base_comparacao_benchmarks_anual.parquet
Entrada - controles diária da 11.6         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_bra

## Etapa 14.2) Testes de Superação contra Ibovespa e CDI

In [72]:
%%time
# ============================================================
# Etapa 14.2) Testes de Superação contra Ibovespa e CDI
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 14.2 - TESTES DE SUPERAÇÃO CONTRA IBOVESPA E CDI")
print("=" * 100)

# Esta subetapa executa os testes estatísticos contra os benchmarks Ibovespa e CDI.
# A base de entrada é a base pareada de benchmarks preparada na Etapa 14.1.
# A visão mensal é tratada como principal para inferência, a visão diária como complementar e a visão anual como exploratória.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/13] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "stats" not in globals():
    raise NameError("A biblioteca scipy.stats deve estar importada como stats antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/13] Definição determinística dos caminhos de entrada e saída...")

if "etapa_14" in DIRETORIOS_PROJETO:
    DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
else:
    DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["resultados"]) / "etapa_14"
    DIRETORIOS_PROJETO["etapa_14"] = DIR_ETAPA_14

DIR_ETAPA_14.mkdir(parents=True, exist_ok=True)

CAMINHO_BASE_BENCHMARKS_14_1 = DIR_ETAPA_14 / "14_1_base_inferencia_benchmarks_pareada.parquet"
CAMINHO_MAPA_COMPARACOES_14_1 = DIR_ETAPA_14 / "14_1_tbl_mapa_comparacoes_inferencia.parquet"
CAMINHO_PARAMETROS_14_1 = DIR_ETAPA_14 / "14_1_tbl_parametros_inferencia_estatistica.parquet"

CAMINHO_BASE_TESTES = DIR_ETAPA_14 / "14_2_base_benchmarks_testes_superacao.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_14 / "14_2_tbl_parametros_testes_benchmarks.parquet"
CAMINHO_TESTE_SINAL = DIR_ETAPA_14 / "14_2_tbl_teste_sinal_benchmarks.parquet"
CAMINHO_TESTE_WILCOXON = DIR_ETAPA_14 / "14_2_tbl_teste_wilcoxon_benchmarks.parquet"
CAMINHO_TESTE_PERMUTACAO = DIR_ETAPA_14 / "14_2_tbl_teste_permutacao_benchmarks.parquet"
CAMINHO_RESUMO = DIR_ETAPA_14 / "14_2_tbl_resumo_testes_benchmarks.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_14 / "14_2_tbl_base_grafico_pvalues_benchmarks.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_14 / "14_2_tbl_auditoria_validacao_testes_benchmarks.parquet"

CAMINHOS_OBRIGATORIOS = [
    CAMINHO_BASE_BENCHMARKS_14_1,
    CAMINHO_MAPA_COMPARACOES_14_1,
]

for caminho in CAMINHOS_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - base benchmarks da 14.1       : {CAMINHO_BASE_BENCHMARKS_14_1}")
print(f"Entrada - mapa comparações da 14.1      : {CAMINHO_MAPA_COMPARACOES_14_1}")
print(f"Entrada - parâmetros da 14.1            : {CAMINHO_PARAMETROS_14_1}")
print(f"Saída   - base de testes benchmarks     : {CAMINHO_BASE_TESTES}")
print(f"Saída   - parâmetros da 14.2            : {CAMINHO_PARAMETROS}")
print(f"Saída   - teste de sinal benchmarks     : {CAMINHO_TESTE_SINAL}")
print(f"Saída   - teste Wilcoxon benchmarks     : {CAMINHO_TESTE_WILCOXON}")
print(f"Saída   - teste permutação benchmarks   : {CAMINHO_TESTE_PERMUTACAO}")
print(f"Saída   - resumo dos testes             : {CAMINHO_RESUMO}")
print(f"Saída   - base gráfico p-valores        : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação        : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/13] Carga das bases oficiais da subetapa...")

df_base_benchmarks = pd.read_parquet(CAMINHO_BASE_BENCHMARKS_14_1)
df_mapa_comparacoes = pd.read_parquet(CAMINHO_MAPA_COMPARACOES_14_1)

if CAMINHO_PARAMETROS_14_1.exists():
    df_parametros_14_1 = pd.read_parquet(CAMINHO_PARAMETROS_14_1)
else:
    df_parametros_14_1 = pd.DataFrame()

print(f"Base benchmarks da 14.1       : {len(df_base_benchmarks):,} linhas x {df_base_benchmarks.shape[1]:,} colunas")
print(f"Mapa comparações da 14.1      : {len(df_mapa_comparacoes):,} linhas x {df_mapa_comparacoes.shape[1]:,} colunas")
print(f"Parâmetros da 14.1            : {len(df_parametros_14_1):,} linhas x {df_parametros_14_1.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/13] Funções auxiliares da subetapa...")

ALPHA_REFERENCIA = 0.05
ALPHA_INDICATIVO = 0.10
N_PERMUTACOES = 5000
SEMENTE_PERMUTACAO = 1402
MIN_OBSERVACOES_TESTE = 12
MIN_OBSERVACOES_ANUAL_EXPLORATORIA = 8

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}

MAPA_ORDEM_BENCHMARK = {
    "benchmark_ibovespa": 1,
    "benchmark_cdi": 2,
}


def validar_colunas_obrigatorias(df, colunas, nome_base):
    ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(ausentes) > 0:
        raise KeyError(
            f"Colunas obrigatórias ausentes na base {nome_base}: {ausentes}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def converter_booleano(serie):
    if pd.api.types.is_bool_dtype(serie):
        return serie.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce").fillna(0.0).ne(0.0)
    texto = serie.astype(str).str.strip().str.lower()
    return texto.isin(["true", "1", "sim", "s", "yes", "y", "ok", "verdadeiro"])


def preparar_dataframe_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def aplicar_holm(df, coluna_pvalor, coluna_saida, colunas_grupo=None):
    df_saida = df.copy()
    df_saida[coluna_saida] = np.nan

    if coluna_pvalor not in df_saida.columns:
        return df_saida

    if colunas_grupo is None or len(colunas_grupo) == 0:
        grupos = [(None, df_saida.index)]
    else:
        grupos = []
        for _, parte in df_saida.groupby(colunas_grupo, dropna=False):
            grupos.append((None, parte.index))

    for _, indices in grupos:
        pvalores = pd.to_numeric(df_saida.loc[indices, coluna_pvalor], errors="coerce")
        validos = pvalores.dropna().sort_values().index.tolist()
        m = len(validos)
        if m == 0:
            continue

        maximo_acumulado = 0.0
        for posicao, indice in enumerate(validos, start=1):
            pvalor = float(df_saida.loc[indice, coluna_pvalor])
            p_ajustado = min(1.0, (m - posicao + 1) * pvalor)
            maximo_acumulado = max(maximo_acumulado, p_ajustado)
            df_saida.loc[indice, coluna_saida] = min(1.0, maximo_acumulado)

    return df_saida


def teste_sinal_binomial(n_vitorias, n_derrotas):
    n_validos = int(n_vitorias + n_derrotas)
    if n_validos <= 0:
        return {
            "estatistica_teste": np.nan,
            "p_valor": np.nan,
            "status_teste": "nao_aplicavel_sem_observacoes",
        }

    try:
        resultado = stats.binomtest(int(n_vitorias), n=n_validos, p=0.5, alternative="greater")
        p_valor = float(resultado.pvalue)
    except AttributeError:
        p_valor = float(stats.binom_test(int(n_vitorias), n=n_validos, p=0.5, alternative="greater"))

    return {
        "estatistica_teste": int(n_vitorias),
        "p_valor": p_valor,
        "status_teste": "calculado",
    }


def teste_wilcoxon_pareado(excessos):
    serie = pd.Series(excessos).dropna().astype(float)
    serie = serie[np.isfinite(serie)]
    n_obs = int(len(serie))
    n_obs_nao_zero = int((~np.isclose(serie, 0.0, atol=0.0000000001, rtol=0.0)).sum())

    if n_obs < MIN_OBSERVACOES_TESTE:
        return {
            "n_observacoes_teste": n_obs,
            "n_observacoes_nao_zero": n_obs_nao_zero,
            "estatistica_teste": np.nan,
            "p_valor": np.nan,
            "status_teste": "nao_aplicavel_amostra_insuficiente",
        }

    if n_obs_nao_zero <= 0:
        return {
            "n_observacoes_teste": n_obs,
            "n_observacoes_nao_zero": n_obs_nao_zero,
            "estatistica_teste": np.nan,
            "p_valor": np.nan,
            "status_teste": "nao_aplicavel_excessos_iguais_a_zero",
        }

    try:
        resultado = stats.wilcoxon(
            serie,
            zero_method="wilcox",
            alternative="greater",
            mode="auto",
        )
        estatistica = float(resultado.statistic)
        p_valor = float(resultado.pvalue)
        status = "calculado"
    except Exception as erro:
        estatistica = np.nan
        p_valor = np.nan
        status = "erro_calculo_wilcoxon: " + str(erro)

    return {
        "n_observacoes_teste": n_obs,
        "n_observacoes_nao_zero": n_obs_nao_zero,
        "estatistica_teste": estatistica,
        "p_valor": p_valor,
        "status_teste": status,
    }


def teste_permutacao_pareada_media(excessos, n_permutacoes=N_PERMUTACOES, semente=SEMENTE_PERMUTACAO):
    serie = pd.Series(excessos).dropna().astype(float)
    serie = serie[np.isfinite(serie)]
    n_obs = int(len(serie))

    if n_obs < MIN_OBSERVACOES_TESTE:
        return {
            "n_observacoes_teste": n_obs,
            "media_observada": np.nan,
            "p_valor": np.nan,
            "n_permutacoes_efetivas": 0,
            "status_teste": "nao_aplicavel_amostra_insuficiente",
        }

    valores = serie.to_numpy(dtype=float)
    media_observada = float(np.mean(valores))

    rng = np.random.default_rng(semente + n_obs)
    contagem_extrema = 0
    bloco = 500
    permutacoes_executadas = 0

    while permutacoes_executadas < n_permutacoes:
        tamanho_bloco = min(bloco, n_permutacoes - permutacoes_executadas)
        sinais = rng.choice(np.array([-1.0, 1.0]), size=(tamanho_bloco, n_obs), replace=True)
        medias_permutadas = (sinais * valores).mean(axis=1)
        contagem_extrema += int((medias_permutadas >= media_observada).sum())
        permutacoes_executadas += tamanho_bloco

    p_valor = (contagem_extrema + 1.0) / (permutacoes_executadas + 1.0)

    return {
        "n_observacoes_teste": n_obs,
        "media_observada": media_observada,
        "p_valor": float(p_valor),
        "n_permutacoes_efetivas": int(permutacoes_executadas),
        "status_teste": "calculado",
    }


def classificar_resultado(media_excesso, pct_superacao, p_wilcoxon_holm, p_permutacao_holm, p_sinal_holm, visao_inferencia):
    pvalores = pd.Series([p_wilcoxon_holm, p_permutacao_holm, p_sinal_holm], dtype="float64").dropna()
    n_sig_5 = int((pvalores <= ALPHA_REFERENCIA).sum())
    n_sig_10 = int((pvalores <= ALPHA_INDICATIVO).sum())

    if pd.isna(media_excesso) or pd.isna(pct_superacao):
        return "sem_classificacao"

    if media_excesso > 0.0 and pct_superacao > 0.5 and n_sig_5 >= 2:
        return "favoravel_forte"
    if media_excesso > 0.0 and pct_superacao > 0.5 and n_sig_5 >= 1:
        return "favoravel_moderada"
    if media_excesso > 0.0 and pct_superacao > 0.5 and n_sig_10 >= 1:
        return "favoravel_indicativa"
    if media_excesso > 0.0 or pct_superacao > 0.5:
        if visao_inferencia == "anual":
            return "favoravel_descritiva"
        return "favoravel_economica_sem_significancia"
    if media_excesso < 0.0 and pct_superacao < 0.5 and n_sig_5 >= 1:
        return "desfavoravel_com_significancia"
    if media_excesso < 0.0 and pct_superacao < 0.5:
        return "desfavoravel_economica_sem_significancia"
    return "neutra"


def preparar_auditoria(linhas):
    df = pd.DataFrame(linhas)
    if len(df) == 0:
        return pd.DataFrame(columns=["item", "valor", "valor_referencia", "status", "observacao"])
    df["status"] = df["status"].astype(str)
    return df[["item", "valor", "valor_referencia", "status", "observacao"]].copy()

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização da base de benchmarks para testes
# ============================================================

print("\n[5/13] Padronização da base de benchmarks para testes...")

COLUNAS_OBRIGATORIAS_BASE = [
    "comparacao_id",
    "grupo_comparacao",
    "tipo_comparacao",
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "periodo_comparacao",
    "retorno_periodo_real",
    "retorno_periodo_comparador",
    "retorno_excesso_real_vs_comparador",
    "flag_real_superou_comparador",
]

validar_colunas_obrigatorias(df_base_benchmarks, COLUNAS_OBRIGATORIAS_BASE, "14.1 base benchmarks pareada")

df_base_testes = df_base_benchmarks.copy()
df_base_testes = df_base_testes.loc[df_base_testes["grupo_comparacao"].astype(str).eq("benchmark")].copy()
df_base_testes = df_base_testes.loc[df_base_testes["tipo_comparacao"].astype(str).isin(["benchmark_ibovespa", "benchmark_cdi"])].copy()

df_base_testes["retorno_periodo_real"] = converter_numero(df_base_testes["retorno_periodo_real"])
df_base_testes["retorno_periodo_comparador"] = converter_numero(df_base_testes["retorno_periodo_comparador"])
df_base_testes["retorno_excesso_real_vs_comparador"] = converter_numero(df_base_testes["retorno_excesso_real_vs_comparador"])
df_base_testes["flag_real_superou_comparador"] = converter_booleano(df_base_testes["flag_real_superou_comparador"])

if "flag_real_empatou_comparador" in df_base_testes.columns:
    df_base_testes["flag_real_empatou_comparador"] = converter_booleano(df_base_testes["flag_real_empatou_comparador"])
else:
    df_base_testes["flag_real_empatou_comparador"] = np.isclose(
        df_base_testes["retorno_excesso_real_vs_comparador"].fillna(np.nan),
        0.0,
        atol=0.0000000001,
        rtol=0.0,
    )

if "flag_real_perdeu_comparador" in df_base_testes.columns:
    df_base_testes["flag_real_perdeu_comparador"] = converter_booleano(df_base_testes["flag_real_perdeu_comparador"])
else:
    df_base_testes["flag_real_perdeu_comparador"] = df_base_testes["retorno_excesso_real_vs_comparador"] < 0.0

df_base_testes = df_base_testes.dropna(
    subset=[
        "retorno_periodo_real",
        "retorno_periodo_comparador",
        "retorno_excesso_real_vs_comparador",
    ]
).copy()

df_base_testes["ordem_frequencia_inferencia"] = df_base_testes["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_base_testes["ordem_benchmark"] = df_base_testes["tipo_comparacao"].map(MAPA_ORDEM_BENCHMARK).fillna(99).astype(int)

df_base_testes = df_base_testes.sort_values(
    [
        "ordem_frequencia_inferencia",
        "estrategia_referencia",
        "ordem_benchmark",
        "comparacao_id",
        "periodo_comparacao",
    ]
).reset_index(drop=True)

print(f"Base de testes contra benchmarks         : {len(df_base_testes):,} linhas")
print(f"Comparações contra benchmarks            : {df_base_testes['comparacao_id'].nunique():,}")
print(f"Estratégias de referência                : {df_base_testes['estrategia_referencia'].nunique():,}")
print(f"Frequências preservadas                  : {df_base_testes['frequencia'].nunique():,}")
print("OK")

# ============================================================
# 6) Construção das estatísticas descritivas por comparação
# ============================================================

print("\n[6/13] Construção das estatísticas descritivas por comparação...")

AGRUPADORES_COMPARACAO = [
    "comparacao_id",
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "grupo_comparacao",
    "tipo_comparacao",
    "subtipo_comparacao",
    "chave_comparador",
    "nome_exibicao_comparador",
]

for coluna in AGRUPADORES_COMPARACAO:
    if coluna not in df_base_testes.columns:
        df_base_testes[coluna] = np.nan

registros_descritivos = []

for chaves, grupo in df_base_testes.groupby(AGRUPADORES_COMPARACAO, dropna=False):
    registro = dict(zip(AGRUPADORES_COMPARACAO, chaves))
    excessos = converter_numero(grupo["retorno_excesso_real_vs_comparador"]).dropna()
    retornos_real = converter_numero(grupo["retorno_periodo_real"]).dropna()
    retornos_comparador = converter_numero(grupo["retorno_periodo_comparador"]).dropna()

    n_periodos = int(len(grupo))
    n_vitorias = int(converter_booleano(grupo["flag_real_superou_comparador"]).sum())
    n_empates = int(converter_booleano(grupo["flag_real_empatou_comparador"]).sum())
    n_derrotas = int(converter_booleano(grupo["flag_real_perdeu_comparador"]).sum())
    n_validos_sinal = int(n_vitorias + n_derrotas)

    registro["n_periodos"] = n_periodos
    registro["n_periodos_validos_excesso"] = int(excessos.notna().sum())
    registro["n_vitorias_real"] = n_vitorias
    registro["n_empates_real"] = n_empates
    registro["n_derrotas_real"] = n_derrotas
    registro["n_validos_teste_sinal"] = n_validos_sinal
    registro["pct_periodos_real_superou_comparador"] = n_vitorias / n_periodos if n_periodos > 0 else np.nan
    registro["pct_periodos_real_superou_sem_empates"] = n_vitorias / n_validos_sinal if n_validos_sinal > 0 else np.nan
    registro["media_retorno_real_periodico"] = float(retornos_real.mean()) if len(retornos_real) > 0 else np.nan
    registro["media_retorno_comparador_periodico"] = float(retornos_comparador.mean()) if len(retornos_comparador) > 0 else np.nan
    registro["media_excesso_periodico"] = float(excessos.mean()) if len(excessos) > 0 else np.nan
    registro["mediana_excesso_periodico"] = float(excessos.median()) if len(excessos) > 0 else np.nan
    registro["desvio_excesso_periodico"] = float(excessos.std(ddof=1)) if len(excessos) > 1 else np.nan
    registro["min_excesso_periodico"] = float(excessos.min()) if len(excessos) > 0 else np.nan
    registro["max_excesso_periodico"] = float(excessos.max()) if len(excessos) > 0 else np.nan
    registro["q05_excesso_periodico"] = float(excessos.quantile(0.05)) if len(excessos) > 0 else np.nan
    registro["q25_excesso_periodico"] = float(excessos.quantile(0.25)) if len(excessos) > 0 else np.nan
    registro["q75_excesso_periodico"] = float(excessos.quantile(0.75)) if len(excessos) > 0 else np.nan
    registro["q95_excesso_periodico"] = float(excessos.quantile(0.95)) if len(excessos) > 0 else np.nan

    frequencia = str(registro["frequencia"])
    fator_anualizacao = 252.0 if frequencia == "diaria" else 12.0 if frequencia == "mensal" else 1.0
    registro["media_excesso_anualizada_aproximada"] = registro["media_excesso_periodico"] * fator_anualizacao if pd.notna(registro["media_excesso_periodico"]) else np.nan
    registro["erro_padrao_media_excesso"] = registro["desvio_excesso_periodico"] / np.sqrt(len(excessos)) if len(excessos) > 1 and pd.notna(registro["desvio_excesso_periodico"]) else np.nan
    registro["efeito_padronizado_media"] = registro["media_excesso_periodico"] / registro["desvio_excesso_periodico"] if pd.notna(registro["media_excesso_periodico"]) and pd.notna(registro["desvio_excesso_periodico"]) and not np.isclose(registro["desvio_excesso_periodico"], 0.0) else np.nan
    registro["flag_amostra_minima_testes"] = int(len(excessos) >= MIN_OBSERVACOES_TESTE)
    registro["flag_visao_principal_inferencia"] = int(frequencia == "mensal")
    registro["flag_visao_complementar_inferencia"] = int(frequencia == "diaria")
    registro["flag_visao_exploratoria_inferencia"] = int(frequencia == "anual")

    registros_descritivos.append(registro)

df_descritivo = pd.DataFrame(registros_descritivos)

print(f"Estatísticas descritivas por comparação  : {len(df_descritivo):,} linhas")
print("OK")

# ============================================================
# 7) Teste binomial de sinal
# ============================================================

print("\n[7/13] Execução do teste binomial de sinal...")

registros_sinal = []

for _, linha in df_descritivo.iterrows():
    resultado = teste_sinal_binomial(
        int(linha["n_vitorias_real"]),
        int(linha["n_derrotas_real"]),
    )

    registro = {coluna: linha[coluna] for coluna in AGRUPADORES_COMPARACAO if coluna in linha.index}
    registro["n_periodos"] = int(linha["n_periodos"])
    registro["n_vitorias_real"] = int(linha["n_vitorias_real"])
    registro["n_derrotas_real"] = int(linha["n_derrotas_real"])
    registro["n_empates_real"] = int(linha["n_empates_real"])
    registro["n_validos_teste_sinal"] = int(linha["n_validos_teste_sinal"])
    registro["pct_periodos_real_superou_comparador"] = linha["pct_periodos_real_superou_comparador"]
    registro["pct_periodos_real_superou_sem_empates"] = linha["pct_periodos_real_superou_sem_empates"]
    registro["hipotese_nula"] = "probabilidade_superacao_igual_50pct"
    registro["hipotese_alternativa"] = "probabilidade_superacao_maior_50pct"
    registro["estatistica_teste_sinal"] = resultado["estatistica_teste"]
    registro["p_valor_teste_sinal"] = resultado["p_valor"]
    registro["status_teste_sinal"] = resultado["status_teste"]
    registros_sinal.append(registro)

df_teste_sinal = pd.DataFrame(registros_sinal)
df_teste_sinal = aplicar_holm(df_teste_sinal, "p_valor_teste_sinal", "p_valor_holm_teste_sinal", ["frequencia"])
df_teste_sinal["significativo_5pct_teste_sinal"] = pd.to_numeric(df_teste_sinal["p_valor_holm_teste_sinal"], errors="coerce") <= ALPHA_REFERENCIA
df_teste_sinal["significativo_10pct_teste_sinal"] = pd.to_numeric(df_teste_sinal["p_valor_holm_teste_sinal"], errors="coerce") <= ALPHA_INDICATIVO

print(f"Testes binomiais de sinal calculados     : {len(df_teste_sinal):,} linhas")
print("OK")

# ============================================================
# 8) Teste de Wilcoxon pareado dos excessos de retorno
# ============================================================

print("\n[8/13] Execução do teste de Wilcoxon pareado...")

registros_wilcoxon = []

for chaves, grupo in df_base_testes.groupby(AGRUPADORES_COMPARACAO, dropna=False):
    registro = dict(zip(AGRUPADORES_COMPARACAO, chaves))
    excessos = converter_numero(grupo["retorno_excesso_real_vs_comparador"])
    resultado = teste_wilcoxon_pareado(excessos)

    registro["n_observacoes_teste"] = resultado["n_observacoes_teste"]
    registro["n_observacoes_nao_zero"] = resultado["n_observacoes_nao_zero"]
    registro["hipotese_nula"] = "mediana_excesso_igual_zero"
    registro["hipotese_alternativa"] = "mediana_excesso_maior_zero"
    registro["estatistica_wilcoxon"] = resultado["estatistica_teste"]
    registro["p_valor_wilcoxon"] = resultado["p_valor"]
    registro["status_teste_wilcoxon"] = resultado["status_teste"]
    registros_wilcoxon.append(registro)

df_teste_wilcoxon = pd.DataFrame(registros_wilcoxon)
df_teste_wilcoxon = aplicar_holm(df_teste_wilcoxon, "p_valor_wilcoxon", "p_valor_holm_wilcoxon", ["frequencia"])
df_teste_wilcoxon["significativo_5pct_wilcoxon"] = pd.to_numeric(df_teste_wilcoxon["p_valor_holm_wilcoxon"], errors="coerce") <= ALPHA_REFERENCIA
df_teste_wilcoxon["significativo_10pct_wilcoxon"] = pd.to_numeric(df_teste_wilcoxon["p_valor_holm_wilcoxon"], errors="coerce") <= ALPHA_INDICATIVO

print(f"Testes de Wilcoxon calculados            : {len(df_teste_wilcoxon):,} linhas")
print("OK")

# ============================================================
# 9) Teste de permutação pareado dos excessos médios
# ============================================================

print("\n[9/13] Execução do teste de permutação pareado...")

registros_permutacao = []

for chaves, grupo in df_base_testes.groupby(AGRUPADORES_COMPARACAO, dropna=False):
    registro = dict(zip(AGRUPADORES_COMPARACAO, chaves))
    excessos = converter_numero(grupo["retorno_excesso_real_vs_comparador"])
    resultado = teste_permutacao_pareada_media(excessos)

    registro["n_observacoes_teste"] = resultado["n_observacoes_teste"]
    registro["media_observada_excesso"] = resultado["media_observada"]
    registro["n_permutacoes_planejadas"] = N_PERMUTACOES
    registro["n_permutacoes_efetivas"] = resultado["n_permutacoes_efetivas"]
    registro["semente_permutacao"] = SEMENTE_PERMUTACAO
    registro["hipotese_nula"] = "media_excesso_igual_zero_com_inversao_de_sinal"
    registro["hipotese_alternativa"] = "media_excesso_maior_zero"
    registro["p_valor_permutacao"] = resultado["p_valor"]
    registro["status_teste_permutacao"] = resultado["status_teste"]
    registros_permutacao.append(registro)

df_teste_permutacao = pd.DataFrame(registros_permutacao)
df_teste_permutacao = aplicar_holm(df_teste_permutacao, "p_valor_permutacao", "p_valor_holm_permutacao", ["frequencia"])
df_teste_permutacao["significativo_5pct_permutacao"] = pd.to_numeric(df_teste_permutacao["p_valor_holm_permutacao"], errors="coerce") <= ALPHA_REFERENCIA
df_teste_permutacao["significativo_10pct_permutacao"] = pd.to_numeric(df_teste_permutacao["p_valor_holm_permutacao"], errors="coerce") <= ALPHA_INDICATIVO

print(f"Testes de permutação calculados          : {len(df_teste_permutacao):,} linhas")
print("OK")

# ============================================================
# 10) Consolidação do resumo dos testes contra benchmarks
# ============================================================

print("\n[10/13] Consolidação do resumo dos testes contra benchmarks...")

colunas_sinal_resumo = AGRUPADORES_COMPARACAO + [
    "p_valor_teste_sinal",
    "p_valor_holm_teste_sinal",
    "significativo_5pct_teste_sinal",
    "significativo_10pct_teste_sinal",
    "status_teste_sinal",
]

colunas_wilcoxon_resumo = AGRUPADORES_COMPARACAO + [
    "estatistica_wilcoxon",
    "p_valor_wilcoxon",
    "p_valor_holm_wilcoxon",
    "significativo_5pct_wilcoxon",
    "significativo_10pct_wilcoxon",
    "status_teste_wilcoxon",
]

colunas_permutacao_resumo = AGRUPADORES_COMPARACAO + [
    "media_observada_excesso",
    "p_valor_permutacao",
    "p_valor_holm_permutacao",
    "significativo_5pct_permutacao",
    "significativo_10pct_permutacao",
    "status_teste_permutacao",
]

df_resumo = df_descritivo.merge(
    df_teste_sinal[colunas_sinal_resumo],
    on=AGRUPADORES_COMPARACAO,
    how="left",
)

df_resumo = df_resumo.merge(
    df_teste_wilcoxon[colunas_wilcoxon_resumo],
    on=AGRUPADORES_COMPARACAO,
    how="left",
)

df_resumo = df_resumo.merge(
    df_teste_permutacao[colunas_permutacao_resumo],
    on=AGRUPADORES_COMPARACAO,
    how="left",
)

for coluna in [
    "significativo_5pct_teste_sinal",
    "significativo_5pct_wilcoxon",
    "significativo_5pct_permutacao",
    "significativo_10pct_teste_sinal",
    "significativo_10pct_wilcoxon",
    "significativo_10pct_permutacao",
]:
    if coluna in df_resumo.columns:
        df_resumo[coluna] = converter_booleano(df_resumo[coluna])

cols_sig_5 = [
    "significativo_5pct_teste_sinal",
    "significativo_5pct_wilcoxon",
    "significativo_5pct_permutacao",
]
cols_sig_10 = [
    "significativo_10pct_teste_sinal",
    "significativo_10pct_wilcoxon",
    "significativo_10pct_permutacao",
]

df_resumo["n_testes_significativos_5pct"] = df_resumo[cols_sig_5].sum(axis=1).astype(int)
df_resumo["n_testes_significativos_10pct"] = df_resumo[cols_sig_10].sum(axis=1).astype(int)

df_resumo["classificacao_evidencia_benchmark"] = df_resumo.apply(
    lambda linha: classificar_resultado(
        linha.get("media_excesso_periodico", np.nan),
        linha.get("pct_periodos_real_superou_comparador", np.nan),
        linha.get("p_valor_holm_wilcoxon", np.nan),
        linha.get("p_valor_holm_permutacao", np.nan),
        linha.get("p_valor_holm_teste_sinal", np.nan),
        linha.get("visao_inferencia", ""),
    ),
    axis=1,
)

df_resumo["ordem_frequencia_inferencia"] = df_resumo["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_resumo["ordem_benchmark"] = df_resumo["tipo_comparacao"].map(MAPA_ORDEM_BENCHMARK).fillna(99).astype(int)
df_resumo = df_resumo.sort_values(
    [
        "ordem_frequencia_inferencia",
        "estrategia_referencia",
        "ordem_benchmark",
        "comparacao_id",
    ]
).reset_index(drop=True)

print(f"Resumo consolidado dos testes            : {len(df_resumo):,} linhas")
print("OK")

# ============================================================
# 11) Construção da base de gráficos de p-valores
# ============================================================

print("\n[11/13] Construção da base de gráficos de p-valores...")

registros_grafico = []

for _, linha in df_resumo.iterrows():
    testes = [
        {
            "teste_estatistico": "teste_sinal_binomial",
            "p_valor": linha.get("p_valor_teste_sinal", np.nan),
            "p_valor_holm_14_2": linha.get("p_valor_holm_teste_sinal", np.nan),
            "status_teste": linha.get("status_teste_sinal", ""),
        },
        {
            "teste_estatistico": "wilcoxon_pareado",
            "p_valor": linha.get("p_valor_wilcoxon", np.nan),
            "p_valor_holm_14_2": linha.get("p_valor_holm_wilcoxon", np.nan),
            "status_teste": linha.get("status_teste_wilcoxon", ""),
        },
        {
            "teste_estatistico": "permutacao_pareada_media",
            "p_valor": linha.get("p_valor_permutacao", np.nan),
            "p_valor_holm_14_2": linha.get("p_valor_holm_permutacao", np.nan),
            "status_teste": linha.get("status_teste_permutacao", ""),
        },
    ]

    for teste in testes:
        registros_grafico.append(
            {
                "comparacao_id": linha.get("comparacao_id", ""),
                "estrategia_referencia": linha.get("estrategia_referencia", ""),
                "frequencia": linha.get("frequencia", ""),
                "visao_inferencia": linha.get("visao_inferencia", ""),
                "tipo_comparacao": linha.get("tipo_comparacao", ""),
                "nome_exibicao_comparador": linha.get("nome_exibicao_comparador", ""),
                "teste_estatistico": teste["teste_estatistico"],
                "p_valor": teste["p_valor"],
                "p_valor_holm_14_2": teste["p_valor_holm_14_2"],
                "menos_log10_p_valor_holm": -np.log10(teste["p_valor_holm_14_2"]) if pd.notna(teste["p_valor_holm_14_2"]) and teste["p_valor_holm_14_2"] > 0 else np.nan,
                "status_teste": teste["status_teste"],
                "media_excesso_periodico": linha.get("media_excesso_periodico", np.nan),
                "pct_periodos_real_superou_comparador": linha.get("pct_periodos_real_superou_comparador", np.nan),
                "classificacao_evidencia_benchmark": linha.get("classificacao_evidencia_benchmark", ""),
                "alpha_referencia": ALPHA_REFERENCIA,
                "alpha_indicativo": ALPHA_INDICATIVO,
            }
        )

df_base_grafico = pd.DataFrame(registros_grafico)

print(f"Base de gráficos de p-valores            : {len(df_base_grafico):,} linhas")
print("OK")

# ============================================================
# 12) Parâmetros e auditoria de validação
# ============================================================

print("\n[12/13] Parâmetros e auditoria de validação...")

PARAMETROS_14_2 = {
    "etapa": "14",
    "subetapa": "14.2",
    "nome_subetapa": "Testes de Superação contra Ibovespa e CDI",
    "base_entrada_principal": "14_1_base_inferencia_benchmarks_pareada.parquet",
    "frequencia_principal_inferencia": "mensal",
    "frequencia_complementar_inferencia": "diaria",
    "frequencia_exploratoria": "anual",
    "comparadores": "Ibovespa e CDI",
    "teste_sinal": "binomial unilateral com H0 p=0.5 e H1 p>0.5",
    "teste_pareado_mediana": "Wilcoxon pareado unilateral com H1 mediana do excesso > 0",
    "teste_pareado_media": "permutação pareada por inversão de sinal com H1 média do excesso > 0",
    "n_permutacoes": N_PERMUTACOES,
    "semente_permutacao": SEMENTE_PERMUTACAO,
    "min_observacoes_testes": MIN_OBSERVACOES_TESTE,
    "alpha_referencia": ALPHA_REFERENCIA,
    "alpha_indicativo": ALPHA_INDICATIVO,
    "ajuste_pvalor_subetapa": "Holm dentro da frequência e do tipo de teste",
    "ajuste_global_planejado": "Consolidação global pelo método de Holm na Etapa 14.7",
}

df_parametros = pd.DataFrame(
    [{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_14_2.items()]
)

n_comparacoes = int(df_resumo["comparacao_id"].nunique()) if len(df_resumo) > 0 else 0
n_frequencias = int(df_resumo["frequencia"].nunique()) if len(df_resumo) > 0 else 0
n_estrategias = int(df_resumo["estrategia_referencia"].nunique()) if len(df_resumo) > 0 else 0
n_benchmarks = int(df_resumo["tipo_comparacao"].nunique()) if len(df_resumo) > 0 else 0
n_testes_sinal_calculados = int((df_teste_sinal["status_teste_sinal"] == "calculado").sum()) if len(df_teste_sinal) > 0 else 0
n_testes_wilcoxon_calculados = int((df_teste_wilcoxon["status_teste_wilcoxon"] == "calculado").sum()) if len(df_teste_wilcoxon) > 0 else 0
n_testes_permutacao_calculados = int((df_teste_permutacao["status_teste_permutacao"] == "calculado").sum()) if len(df_teste_permutacao) > 0 else 0
n_mensais_principais = int((df_resumo["frequencia"] == "mensal").sum()) if len(df_resumo) > 0 else 0
n_diarias_complementares = int((df_resumo["frequencia"] == "diaria").sum()) if len(df_resumo) > 0 else 0
n_anuais_exploratorias = int((df_resumo["frequencia"] == "anual").sum()) if len(df_resumo) > 0 else 0
valores_excesso_ausentes = int(df_base_testes["retorno_excesso_real_vs_comparador"].isna().sum()) if len(df_base_testes) > 0 else 0
duplicatas_base = int(df_base_testes.duplicated(["comparacao_id", "frequencia", "periodo_comparacao"]).sum()) if len(df_base_testes) > 0 else 0

metricas_infinitas = 0
for df_verificacao in [df_base_testes, df_teste_sinal, df_teste_wilcoxon, df_teste_permutacao, df_resumo, df_base_grafico]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

linhas_auditoria = [
    {
        "item": "linhas_base_testes_benchmarks",
        "valor": len(df_base_testes),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_testes) > 0 else "ERRO",
        "observacao": "A subetapa deve carregar a base pareada de benchmarks preparada na 14.1.",
    },
    {
        "item": "comparacoes_benchmark_identificadas",
        "valor": n_comparacoes,
        "valor_referencia": ">= 12",
        "status": "OK" if n_comparacoes >= 12 else "ERRO",
        "observacao": "Espera-se 2 estratégias x 2 benchmarks x 3 frequências.",
    },
    {
        "item": "estrategias_referencia_identificadas",
        "valor": n_estrategias,
        "valor_referencia": ">= 2",
        "status": "OK" if n_estrategias >= 2 else "ERRO",
        "observacao": "Espera-se encontrar Capitulação e Euforia.",
    },
    {
        "item": "benchmarks_identificados",
        "valor": n_benchmarks,
        "valor_referencia": ">= 2",
        "status": "OK" if n_benchmarks >= 2 else "ERRO",
        "observacao": "Espera-se encontrar Ibovespa e CDI.",
    },
    {
        "item": "frequencias_identificadas",
        "valor": n_frequencias,
        "valor_referencia": ">= 3",
        "status": "OK" if n_frequencias >= 3 else "ERRO",
        "observacao": "A subetapa deve preservar as visões diária, mensal e anual.",
    },
    {
        "item": "comparacoes_mensais_principais",
        "valor": n_mensais_principais,
        "valor_referencia": "> 0",
        "status": "OK" if n_mensais_principais > 0 else "ERRO",
        "observacao": "A visão mensal deve existir como referência principal dos testes formais.",
    },
    {
        "item": "comparacoes_diarias_complementares",
        "valor": n_diarias_complementares,
        "valor_referencia": "> 0",
        "status": "OK" if n_diarias_complementares > 0 else "ERRO",
        "observacao": "A visão diária deve ser preservada como análise complementar.",
    },
    {
        "item": "comparacoes_anuais_exploratorias",
        "valor": n_anuais_exploratorias,
        "valor_referencia": "> 0",
        "status": "OK" if n_anuais_exploratorias > 0 else "ERRO",
        "observacao": "A visão anual deve ser preservada como análise exploratória.",
    },
    {
        "item": "testes_sinal_calculados",
        "valor": n_testes_sinal_calculados,
        "valor_referencia": "> 0",
        "status": "OK" if n_testes_sinal_calculados > 0 else "ERRO",
        "observacao": "O teste binomial de sinal deve ser calculado para as comparações aplicáveis.",
    },
    {
        "item": "testes_wilcoxon_calculados",
        "valor": n_testes_wilcoxon_calculados,
        "valor_referencia": "> 0",
        "status": "OK" if n_testes_wilcoxon_calculados > 0 else "ERRO",
        "observacao": "O teste de Wilcoxon pareado deve ser calculado para as comparações aplicáveis.",
    },
    {
        "item": "testes_permutacao_calculados",
        "valor": n_testes_permutacao_calculados,
        "valor_referencia": "> 0",
        "status": "OK" if n_testes_permutacao_calculados > 0 else "ERRO",
        "observacao": "O teste de permutação pareado deve ser calculado para as comparações aplicáveis.",
    },
    {
        "item": "valores_excesso_ausentes",
        "valor": valores_excesso_ausentes,
        "valor_referencia": "0",
        "status": "OK" if valores_excesso_ausentes == 0 else "ERRO",
        "observacao": "A base de testes não deve conter excesso de retorno ausente.",
    },
    {
        "item": "duplicatas_comparacao_frequencia_periodo",
        "valor": duplicatas_base,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_base == 0 else "ERRO",
        "observacao": "Cada comparação deve ter no máximo uma linha por frequência e período.",
    },
    {
        "item": "base_grafico_linhas",
        "valor": len(df_base_grafico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Parâmetros metodológicos                 : {len(df_parametros):,} linhas")
print(f"Itens de auditoria                       : {len(df_auditoria):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 13) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[13/13] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(preparar_dataframe_parquet(df_base_testes), CAMINHO_BASE_TESTES, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_parametros), CAMINHO_PARAMETROS, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_teste_sinal), CAMINHO_TESTE_SINAL, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_teste_wilcoxon), CAMINHO_TESTE_WILCOXON, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_teste_permutacao), CAMINHO_TESTE_PERMUTACAO, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_resumo), CAMINHO_RESUMO, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_base_grafico), CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_auditoria), CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação dos testes contra benchmarks:")
print(df_auditoria.to_string(index=False))

print("\nResumo dos testes contra Ibovespa e CDI:")
if len(df_resumo) > 0:
    colunas_print = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "visao_inferencia",
            "tipo_comparacao",
            "nome_exibicao_comparador",
            "n_periodos",
            "media_excesso_periodico",
            "media_excesso_anualizada_aproximada",
            "pct_periodos_real_superou_comparador",
            "p_valor_holm_teste_sinal",
            "p_valor_holm_wilcoxon",
            "p_valor_holm_permutacao",
            "n_testes_significativos_5pct",
            "classificacao_evidencia_benchmark",
        ] if coluna in df_resumo.columns
    ]
    print(df_resumo[colunas_print].to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nBase gráfica de p-valores - amostra:")
if len(df_base_grafico) > 0:
    colunas_grafico_print = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "tipo_comparacao",
            "teste_estatistico",
            "p_valor_holm_14_2",
            "media_excesso_periodico",
            "pct_periodos_real_superou_comparador",
            "classificacao_evidencia_benchmark",
        ] if coluna in df_base_grafico.columns
    ]
    print(df_base_grafico[colunas_grafico_print].head(60).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nArquivos salvos na subetapa 14.2:")
print(f"- {CAMINHO_BASE_TESTES}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_TESTE_SINAL}")
print(f"- {CAMINHO_TESTE_WILCOXON}")
print(f"- {CAMINHO_TESTE_PERMUTACAO}")
print(f"- {CAMINHO_RESUMO}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 14.2 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 14.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 14.2 - TESTES DE SUPERAÇÃO CONTRA IBOVESPA E CDI

[1/13] Validação inicial do ambiente...
OK

[2/13] Definição determinística dos caminhos de entrada e saída...
Entrada - base benchmarks da 14.1       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_base_inferencia_benchmarks_pareada.parquet
Entrada - mapa comparações da 14.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_mapa_comparacoes_inferencia.parquet
Entrada - parâmetros da 14.1            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_parametros_inferencia_estatistica.parquet
Saída   - base de testes benchmarks     : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resulta

## Etapa 14.3) Testes contra Estratégias Aleatórias de Controle

In [73]:
%%time
# ============================================================
# Etapa 14.3) Testes contra Estratégias Aleatórias de Controle
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 14.3 - TESTES CONTRA ESTRATÉGIAS ALEATÓRIAS DE CONTROLE")
print("=" * 100)

# Esta subetapa executa testes estatísticos contra as réplicas aleatórias de controle.
# A base de entrada é a base pareada de controles preparada na Etapa 14.1.
# A visão mensal é tratada como principal para inferência, a visão diária como complementar e a visão anual como exploratória.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/14] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "stats" not in globals():
    raise NameError("A biblioteca scipy.stats deve estar importada como stats antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/14] Definição determinística dos caminhos de entrada e saída...")

if "etapa_14" in DIRETORIOS_PROJETO:
    DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
else:
    DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["resultados"]) / "etapa_14"
    DIRETORIOS_PROJETO["etapa_14"] = DIR_ETAPA_14

DIR_ETAPA_14.mkdir(parents=True, exist_ok=True)

CAMINHO_BASE_CONTROLES_14_1 = DIR_ETAPA_14 / "14_1_base_inferencia_controles_pareada.parquet"
CAMINHO_BASE_CONSOLIDADA_14_1 = DIR_ETAPA_14 / "14_1_base_inferencia_comparacoes_pareadas.parquet"
CAMINHO_MAPA_COMPARACOES_14_1 = DIR_ETAPA_14 / "14_1_tbl_mapa_comparacoes_inferencia.parquet"
CAMINHO_PARAMETROS_14_1 = DIR_ETAPA_14 / "14_1_tbl_parametros_inferencia_estatistica.parquet"

CAMINHO_BASE_TESTES = DIR_ETAPA_14 / "14_3_base_aleatorios_testes_superacao.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_14 / "14_3_tbl_parametros_testes_aleatorios.parquet"
CAMINHO_TESTE_SINAL = DIR_ETAPA_14 / "14_3_tbl_teste_sinal_aleatorios.parquet"
CAMINHO_TESTE_WILCOXON = DIR_ETAPA_14 / "14_3_tbl_teste_wilcoxon_aleatorios.parquet"
CAMINHO_TESTE_PERMUTACAO = DIR_ETAPA_14 / "14_3_tbl_teste_permutacao_aleatorios.parquet"
CAMINHO_POSICIONAMENTO = DIR_ETAPA_14 / "14_3_tbl_posicionamento_empirico_aleatorios.parquet"
CAMINHO_RESUMO_PARES = DIR_ETAPA_14 / "14_3_tbl_resumo_testes_aleatorios_pares.parquet"
CAMINHO_RESUMO_GRUPOS = DIR_ETAPA_14 / "14_3_tbl_resumo_testes_aleatorios_grupos.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_14 / "14_3_tbl_base_grafico_pvalues_aleatorios.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_14 / "14_3_tbl_auditoria_validacao_testes_aleatorios.parquet"

CAMINHOS_OBRIGATORIOS = [
    CAMINHO_BASE_CONTROLES_14_1,
    CAMINHO_MAPA_COMPARACOES_14_1,
]

for caminho in CAMINHOS_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - base controles da 14.1        : {CAMINHO_BASE_CONTROLES_14_1}")
print(f"Entrada - base consolidada da 14.1      : {CAMINHO_BASE_CONSOLIDADA_14_1}")
print(f"Entrada - mapa comparações da 14.1      : {CAMINHO_MAPA_COMPARACOES_14_1}")
print(f"Entrada - parâmetros da 14.1            : {CAMINHO_PARAMETROS_14_1}")
print(f"Saída   - base de testes aleatórios     : {CAMINHO_BASE_TESTES}")
print(f"Saída   - parâmetros da 14.3            : {CAMINHO_PARAMETROS}")
print(f"Saída   - teste de sinal aleatórios     : {CAMINHO_TESTE_SINAL}")
print(f"Saída   - teste Wilcoxon aleatórios     : {CAMINHO_TESTE_WILCOXON}")
print(f"Saída   - teste permutação aleatórios   : {CAMINHO_TESTE_PERMUTACAO}")
print(f"Saída   - posicionamento empírico       : {CAMINHO_POSICIONAMENTO}")
print(f"Saída   - resumo dos pares              : {CAMINHO_RESUMO_PARES}")
print(f"Saída   - resumo dos grupos             : {CAMINHO_RESUMO_GRUPOS}")
print(f"Saída   - base gráfico p-valores        : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação        : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/14] Carga das bases oficiais da subetapa...")

df_base_controles = pd.read_parquet(CAMINHO_BASE_CONTROLES_14_1)
df_mapa_comparacoes = pd.read_parquet(CAMINHO_MAPA_COMPARACOES_14_1)

if CAMINHO_BASE_CONSOLIDADA_14_1.exists():
    df_base_consolidada_14_1 = pd.read_parquet(CAMINHO_BASE_CONSOLIDADA_14_1)
else:
    df_base_consolidada_14_1 = pd.DataFrame()

if CAMINHO_PARAMETROS_14_1.exists():
    df_parametros_14_1 = pd.read_parquet(CAMINHO_PARAMETROS_14_1)
else:
    df_parametros_14_1 = pd.DataFrame()

print(f"Base controles da 14.1        : {len(df_base_controles):,} linhas x {df_base_controles.shape[1]:,} colunas")
print(f"Base consolidada da 14.1      : {len(df_base_consolidada_14_1):,} linhas x {df_base_consolidada_14_1.shape[1]:,} colunas")
print(f"Mapa comparações da 14.1      : {len(df_mapa_comparacoes):,} linhas x {df_mapa_comparacoes.shape[1]:,} colunas")
print(f"Parâmetros da 14.1            : {len(df_parametros_14_1):,} linhas x {df_parametros_14_1.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/14] Funções auxiliares da subetapa...")

ALPHA_REFERENCIA = 0.05
ALPHA_INDICATIVO = 0.10
N_PERMUTACOES = 5000
SEMENTE_PERMUTACAO = 1403
MIN_OBSERVACOES_TESTE = 12
MIN_REPLICAS_EMPIRICAS = 5

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}


def validar_colunas_obrigatorias(df, colunas, nome_base):
    ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(ausentes) > 0:
        raise KeyError(
            f"Colunas obrigatórias ausentes na base {nome_base}: {ausentes}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def converter_booleano(serie):
    if pd.api.types.is_bool_dtype(serie):
        return serie.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce").fillna(0.0).ne(0.0)
    texto = serie.astype(str).str.strip().str.lower()
    return texto.isin(["true", "1", "sim", "s", "yes", "y", "ok", "verdadeiro"])


def preparar_dataframe_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def aplicar_holm(df, coluna_pvalor, coluna_saida, colunas_grupo=None):
    df_saida = df.copy()
    df_saida[coluna_saida] = np.nan

    if coluna_pvalor not in df_saida.columns:
        return df_saida

    if colunas_grupo is None or len(colunas_grupo) == 0:
        grupos = [(None, df_saida.index)]
    else:
        grupos = []
        for _, parte in df_saida.groupby(colunas_grupo, dropna=False):
            grupos.append((None, parte.index))

    for _, indices in grupos:
        pvalores = pd.to_numeric(df_saida.loc[indices, coluna_pvalor], errors="coerce")
        validos = pvalores.dropna().sort_values().index.tolist()
        m = len(validos)
        if m == 0:
            continue

        maximo_acumulado = 0.0
        for posicao, indice in enumerate(validos, start=1):
            pvalor = float(df_saida.loc[indice, coluna_pvalor])
            p_ajustado = min(1.0, (m - posicao + 1) * pvalor)
            maximo_acumulado = max(maximo_acumulado, p_ajustado)
            df_saida.loc[indice, coluna_saida] = min(1.0, maximo_acumulado)

    return df_saida


def teste_sinal_binomial(n_vitorias, n_derrotas):
    n_validos = int(n_vitorias + n_derrotas)
    if n_validos <= 0:
        return {
            "estatistica_teste": np.nan,
            "p_valor": np.nan,
            "status_teste": "nao_aplicavel_sem_observacoes",
        }

    try:
        resultado = stats.binomtest(int(n_vitorias), n=n_validos, p=0.5, alternative="greater")
        p_valor = float(resultado.pvalue)
    except AttributeError:
        p_valor = float(stats.binom_test(int(n_vitorias), n=n_validos, p=0.5, alternative="greater"))

    return {
        "estatistica_teste": int(n_vitorias),
        "p_valor": p_valor,
        "status_teste": "calculado",
    }


def teste_wilcoxon_pareado(excessos):
    serie = pd.Series(excessos).dropna().astype(float)
    serie = serie[np.isfinite(serie)]
    n_obs = int(len(serie))
    n_obs_nao_zero = int((~np.isclose(serie, 0.0, atol=0.0000000001, rtol=0.0)).sum())

    if n_obs < MIN_OBSERVACOES_TESTE:
        return {
            "n_observacoes_teste": n_obs,
            "n_observacoes_nao_zero": n_obs_nao_zero,
            "estatistica_teste": np.nan,
            "p_valor": np.nan,
            "status_teste": "nao_aplicavel_observacoes_insuficientes",
        }

    if n_obs_nao_zero <= 0:
        return {
            "n_observacoes_teste": n_obs,
            "n_observacoes_nao_zero": n_obs_nao_zero,
            "estatistica_teste": np.nan,
            "p_valor": np.nan,
            "status_teste": "nao_aplicavel_excessos_zero",
        }

    try:
        resultado = stats.wilcoxon(
            serie,
            zero_method="wilcox",
            alternative="greater",
            correction=False,
            mode="auto",
        )
        estatistica = float(resultado.statistic)
        p_valor = float(resultado.pvalue)
        status = "calculado"
    except Exception as erro:
        estatistica = np.nan
        p_valor = np.nan
        status = f"erro_calculo_wilcoxon: {erro}"

    return {
        "n_observacoes_teste": n_obs,
        "n_observacoes_nao_zero": n_obs_nao_zero,
        "estatistica_teste": estatistica,
        "p_valor": p_valor,
        "status_teste": status,
    }


def teste_permutacao_pareado_media(excessos, n_permutacoes, semente):
    serie = pd.Series(excessos).dropna().astype(float)
    serie = serie[np.isfinite(serie)].to_numpy(dtype=float)
    n_obs = int(len(serie))

    if n_obs < MIN_OBSERVACOES_TESTE:
        return {
            "n_observacoes_teste": n_obs,
            "estatistica_observada": np.nan,
            "p_valor": np.nan,
            "n_permutacoes": int(n_permutacoes),
            "status_teste": "nao_aplicavel_observacoes_insuficientes",
        }

    estatistica_observada = float(np.mean(serie))
    rng = np.random.default_rng(int(semente))
    contador_extremos = 0
    tamanho_lote = 500
    permutacoes_processadas = 0

    while permutacoes_processadas < n_permutacoes:
        lote = min(tamanho_lote, n_permutacoes - permutacoes_processadas)
        sinais = rng.choice(np.array([-1.0, 1.0]), size=(lote, n_obs), replace=True)
        medias_perm = np.mean(sinais * serie.reshape(1, -1), axis=1)
        contador_extremos += int(np.sum(medias_perm >= estatistica_observada))
        permutacoes_processadas += lote

    p_valor = (contador_extremos + 1.0) / (n_permutacoes + 1.0)

    return {
        "n_observacoes_teste": n_obs,
        "estatistica_observada": estatistica_observada,
        "p_valor": float(p_valor),
        "n_permutacoes": int(n_permutacoes),
        "status_teste": "calculado",
    }


def classificar_evidencia(p_sinal, p_wilcoxon, p_permutacao, media_excesso):
    pvalores = [p_sinal, p_wilcoxon, p_permutacao]
    pvalores = [float(p) for p in pvalores if pd.notna(p)]
    n_significativos_5 = int(sum(p <= ALPHA_REFERENCIA for p in pvalores))
    n_significativos_10 = int(sum(p <= ALPHA_INDICATIVO for p in pvalores))

    if pd.isna(media_excesso):
        return "sem_classificacao", n_significativos_5, n_significativos_10
    if media_excesso > 0 and n_significativos_5 >= 2:
        return "favoravel_estatistica_forte", n_significativos_5, n_significativos_10
    if media_excesso > 0 and n_significativos_5 >= 1:
        return "favoravel_estatistica_moderada", n_significativos_5, n_significativos_10
    if media_excesso > 0 and n_significativos_10 >= 1:
        return "favoravel_indicativa", n_significativos_5, n_significativos_10
    if media_excesso > 0:
        return "favoravel_economica_sem_significancia", n_significativos_5, n_significativos_10
    if media_excesso < 0 and n_significativos_5 >= 1:
        return "desfavoravel_com_sinal_estatistico", n_significativos_5, n_significativos_10
    if media_excesso < 0:
        return "desfavoravel_economica_sem_significancia", n_significativos_5, n_significativos_10
    return "neutra", n_significativos_5, n_significativos_10


def preparar_auditoria(linhas):
    df = pd.DataFrame(linhas)
    if len(df) == 0:
        return pd.DataFrame(columns=["item", "valor", "valor_referencia", "status", "observacao"])
    df["status"] = df["status"].astype(str)
    return df[["item", "valor", "valor_referencia", "status", "observacao"]].copy()

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização da base de estratégias aleatórias
# ============================================================

print("\n[5/14] Padronização da base de estratégias aleatórias...")

COLUNAS_OBRIGATORIAS = [
    "comparacao_id",
    "grupo_comparacao",
    "tipo_comparacao",
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "periodo_comparacao",
    "nome_exibicao_comparador",
    "retorno_periodo_real",
    "retorno_periodo_comparador",
    "retorno_excesso_real_vs_comparador",
    "flag_real_superou_comparador",
    "flag_real_empatou_comparador",
    "flag_real_perdeu_comparador",
]

validar_colunas_obrigatorias(df_base_controles, COLUNAS_OBRIGATORIAS, "14.1 base controles pareada")

df_base_aleatorios = df_base_controles.loc[
    df_base_controles["tipo_comparacao"].astype(str).eq("controle_aleatorio")
].copy()

if len(df_base_aleatorios) == 0:
    raise ValueError("A base da 14.1 não contém linhas com tipo_comparacao = controle_aleatorio.")

for coluna in [
    "retorno_periodo_real",
    "retorno_periodo_comparador",
    "retorno_excesso_real_vs_comparador",
    "retorno_relativo_composto_real_vs_comparador",
]:
    if coluna in df_base_aleatorios.columns:
        df_base_aleatorios[coluna] = converter_numero(df_base_aleatorios[coluna])

for coluna in [
    "flag_real_superou_comparador",
    "flag_real_empatou_comparador",
    "flag_real_perdeu_comparador",
]:
    if coluna in df_base_aleatorios.columns:
        df_base_aleatorios[coluna] = converter_booleano(df_base_aleatorios[coluna])

df_base_aleatorios = df_base_aleatorios.dropna(
    subset=[
        "retorno_periodo_real",
        "retorno_periodo_comparador",
        "retorno_excesso_real_vs_comparador",
    ]
).copy()

df_base_aleatorios["ordem_frequencia_inferencia"] = df_base_aleatorios["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_base_aleatorios = df_base_aleatorios.sort_values(
    [
        "ordem_frequencia_inferencia",
        "estrategia_referencia",
        "comparacao_id",
        "periodo_comparacao",
    ]
).reset_index(drop=True)

print(f"Base de testes contra aleatórias          : {len(df_base_aleatorios):,} linhas")
print(f"Comparações contra aleatórias             : {df_base_aleatorios['comparacao_id'].nunique():,}")
print(f"Estratégias de referência                 : {df_base_aleatorios['estrategia_referencia'].nunique():,}")
print(f"Frequências preservadas                   : {df_base_aleatorios['frequencia'].nunique():,}")
print("OK")

# ============================================================
# 6) Construção das estatísticas descritivas por par de comparação
# ============================================================

print("\n[6/14] Construção das estatísticas descritivas por par de comparação...")

AGRUPADORES_PAR = [
    "comparacao_id",
    "comparacao_id_original",
    "grupo_comparacao",
    "tipo_comparacao",
    "subtipo_comparacao",
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "nome_exibicao_estrategia_real",
    "chave_comparador",
    "nome_exibicao_comparador",
]
AGRUPADORES_PAR = [coluna for coluna in AGRUPADORES_PAR if coluna in df_base_aleatorios.columns]

df_estatisticas = (
    df_base_aleatorios
    .groupby(AGRUPADORES_PAR, dropna=False)
    .agg(
        n_periodos=("periodo_comparacao", "count"),
        n_vitorias=("flag_real_superou_comparador", "sum"),
        n_empates=("flag_real_empatou_comparador", "sum"),
        n_derrotas=("flag_real_perdeu_comparador", "sum"),
        media_retorno_real=("retorno_periodo_real", "mean"),
        media_retorno_comparador=("retorno_periodo_comparador", "mean"),
        media_excesso_periodico=("retorno_excesso_real_vs_comparador", "mean"),
        mediana_excesso_periodico=("retorno_excesso_real_vs_comparador", "median"),
        desvio_excesso_periodico=("retorno_excesso_real_vs_comparador", "std"),
        min_excesso_periodico=("retorno_excesso_real_vs_comparador", "min"),
        max_excesso_periodico=("retorno_excesso_real_vs_comparador", "max"),
        pct_periodos_real_superou_comparador=("flag_real_superou_comparador", "mean"),
    )
    .reset_index()
)

df_estatisticas["n_vitorias"] = df_estatisticas["n_vitorias"].astype(int)
df_estatisticas["n_empates"] = df_estatisticas["n_empates"].astype(int)
df_estatisticas["n_derrotas"] = df_estatisticas["n_derrotas"].astype(int)
df_estatisticas["n_periodos_sem_empate"] = df_estatisticas["n_vitorias"] + df_estatisticas["n_derrotas"]
df_estatisticas["ordem_frequencia_inferencia"] = df_estatisticas["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_estatisticas["media_excesso_anualizada_aproximada"] = np.select(
    [
        df_estatisticas["frequencia"].eq("diaria"),
        df_estatisticas["frequencia"].eq("mensal"),
        df_estatisticas["frequencia"].eq("anual"),
    ],
    [
        df_estatisticas["media_excesso_periodico"] * 252.0,
        df_estatisticas["media_excesso_periodico"] * 12.0,
        df_estatisticas["media_excesso_periodico"],
    ],
    default=np.nan,
)

df_estatisticas = df_estatisticas.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia", "comparacao_id"]
).reset_index(drop=True)

print(f"Estatísticas descritivas por par          : {len(df_estatisticas):,} linhas")
print("OK")

# ============================================================
# 7) Execução do teste binomial de sinal por par
# ============================================================

print("\n[7/14] Execução do teste binomial de sinal por par...")

registros_sinal = []
for _, linha in df_estatisticas.iterrows():
    resultado = teste_sinal_binomial(linha["n_vitorias"], linha["n_derrotas"])
    registro = linha[AGRUPADORES_PAR].to_dict()
    registro.update({
        "n_periodos": int(linha["n_periodos"]),
        "n_vitorias": int(linha["n_vitorias"]),
        "n_empates": int(linha["n_empates"]),
        "n_derrotas": int(linha["n_derrotas"]),
        "n_periodos_sem_empate": int(linha["n_periodos_sem_empate"]),
        "pct_periodos_real_superou_comparador": float(linha["pct_periodos_real_superou_comparador"]),
        "estatistica_teste_sinal": resultado["estatistica_teste"],
        "p_valor_teste_sinal": resultado["p_valor"],
        "status_teste_sinal": resultado["status_teste"],
    })
    registros_sinal.append(registro)

df_teste_sinal = pd.DataFrame(registros_sinal)
df_teste_sinal = aplicar_holm(df_teste_sinal, "p_valor_teste_sinal", "p_valor_holm_teste_sinal", ["frequencia"])

print(f"Testes binomiais de sinal calculados      : {len(df_teste_sinal):,} linhas")
print("OK")

# ============================================================
# 8) Execução do teste de Wilcoxon pareado por par
# ============================================================

print("\n[8/14] Execução do teste de Wilcoxon pareado por par...")

registros_wilcoxon = []
for chave, df_par in df_base_aleatorios.groupby(AGRUPADORES_PAR, dropna=False):
    if not isinstance(chave, tuple):
        chave = (chave,)
    metadados = dict(zip(AGRUPADORES_PAR, chave))
    resultado = teste_wilcoxon_pareado(df_par["retorno_excesso_real_vs_comparador"])
    metadados.update({
        "n_observacoes_teste": resultado["n_observacoes_teste"],
        "n_observacoes_nao_zero": resultado["n_observacoes_nao_zero"],
        "estatistica_wilcoxon": resultado["estatistica_teste"],
        "p_valor_wilcoxon": resultado["p_valor"],
        "status_teste_wilcoxon": resultado["status_teste"],
    })
    registros_wilcoxon.append(metadados)

df_teste_wilcoxon = pd.DataFrame(registros_wilcoxon)
df_teste_wilcoxon = aplicar_holm(df_teste_wilcoxon, "p_valor_wilcoxon", "p_valor_holm_wilcoxon", ["frequencia"])

df_teste_wilcoxon["ordem_frequencia_inferencia"] = df_teste_wilcoxon["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_teste_wilcoxon = df_teste_wilcoxon.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia", "comparacao_id"]
).reset_index(drop=True)

print(f"Testes de Wilcoxon calculados             : {len(df_teste_wilcoxon):,} linhas")
print("OK")

# ============================================================
# 9) Execução do teste de permutação pareado por par
# ============================================================

print("\n[9/14] Execução do teste de permutação pareado por par...")

registros_permutacao = []
for indice_grupo, (chave, df_par) in enumerate(df_base_aleatorios.groupby(AGRUPADORES_PAR, dropna=False)):
    if not isinstance(chave, tuple):
        chave = (chave,)
    metadados = dict(zip(AGRUPADORES_PAR, chave))
    semente_teste = SEMENTE_PERMUTACAO + indice_grupo
    resultado = teste_permutacao_pareado_media(
        df_par["retorno_excesso_real_vs_comparador"],
        N_PERMUTACOES,
        semente_teste,
    )
    metadados.update({
        "n_observacoes_teste": resultado["n_observacoes_teste"],
        "estatistica_observada_media_excesso": resultado["estatistica_observada"],
        "p_valor_permutacao": resultado["p_valor"],
        "n_permutacoes": resultado["n_permutacoes"],
        "semente_permutacao": int(semente_teste),
        "status_teste_permutacao": resultado["status_teste"],
    })
    registros_permutacao.append(metadados)

df_teste_permutacao = pd.DataFrame(registros_permutacao)
df_teste_permutacao = aplicar_holm(df_teste_permutacao, "p_valor_permutacao", "p_valor_holm_permutacao", ["frequencia"])

df_teste_permutacao["ordem_frequencia_inferencia"] = df_teste_permutacao["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_teste_permutacao = df_teste_permutacao.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia", "comparacao_id"]
).reset_index(drop=True)

print(f"Testes de permutação calculados           : {len(df_teste_permutacao):,} linhas")
print("OK")

# ============================================================
# 10) Consolidação do resumo por par de comparação
# ============================================================

print("\n[10/14] Consolidação do resumo por par de comparação...")

df_resumo_pares = df_estatisticas.copy()

df_resumo_pares = df_resumo_pares.merge(
    df_teste_sinal[AGRUPADORES_PAR + ["p_valor_teste_sinal", "p_valor_holm_teste_sinal", "status_teste_sinal"]],
    on=AGRUPADORES_PAR,
    how="left",
)
df_resumo_pares = df_resumo_pares.merge(
    df_teste_wilcoxon[AGRUPADORES_PAR + ["p_valor_wilcoxon", "p_valor_holm_wilcoxon", "status_teste_wilcoxon"]],
    on=AGRUPADORES_PAR,
    how="left",
)
df_resumo_pares = df_resumo_pares.merge(
    df_teste_permutacao[AGRUPADORES_PAR + ["p_valor_permutacao", "p_valor_holm_permutacao", "status_teste_permutacao"]],
    on=AGRUPADORES_PAR,
    how="left",
)

classificacoes = df_resumo_pares.apply(
    lambda linha: classificar_evidencia(
        linha.get("p_valor_holm_teste_sinal", np.nan),
        linha.get("p_valor_holm_wilcoxon", np.nan),
        linha.get("p_valor_holm_permutacao", np.nan),
        linha.get("media_excesso_periodico", np.nan),
    ),
    axis=1,
)

df_resumo_pares["classificacao_evidencia_aleatorio"] = [x[0] for x in classificacoes]
df_resumo_pares["n_testes_significativos_5pct"] = [x[1] for x in classificacoes]
df_resumo_pares["n_testes_significativos_10pct"] = [x[2] for x in classificacoes]
df_resumo_pares["observacao_potencia_estatistica"] = np.where(
    df_resumo_pares["frequencia"].eq("anual"),
    "visao anual exploratória com baixa quantidade de observações",
    np.where(
        df_resumo_pares["frequencia"].eq("mensal"),
        "visão mensal principal para inferência",
        "visão diária complementar sujeita a dependência serial",
    ),
)

df_resumo_pares = df_resumo_pares.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia", "comparacao_id"]
).reset_index(drop=True)

print(f"Resumo consolidado por par                : {len(df_resumo_pares):,} linhas")
print("OK")

# ============================================================
# 11) Posicionamento empírico frente às réplicas aleatórias
# ============================================================

print("\n[11/14] Posicionamento empírico frente às réplicas aleatórias...")

AGRUPADORES_GRUPO = ["estrategia_referencia", "frequencia", "visao_inferencia"]

registros_posicionamento = []
for chave, df_grupo in df_resumo_pares.groupby(AGRUPADORES_GRUPO, dropna=False):
    estrategia, frequencia, visao = chave
    n_replicas = int(df_grupo["comparacao_id"].nunique())
    n_replicas_media_pos = int((df_grupo["media_excesso_periodico"] > 0).sum())
    n_replicas_mediana_pos = int((df_grupo["mediana_excesso_periodico"] > 0).sum())
    n_replicas_sinal_sig = int((df_grupo["p_valor_holm_teste_sinal"] <= ALPHA_REFERENCIA).sum())
    n_replicas_wilcoxon_sig = int((df_grupo["p_valor_holm_wilcoxon"] <= ALPHA_REFERENCIA).sum())
    n_replicas_perm_sig = int((df_grupo["p_valor_holm_permutacao"] <= ALPHA_REFERENCIA).sum())

    pct_media_pos = n_replicas_media_pos / n_replicas if n_replicas > 0 else np.nan
    pct_mediana_pos = n_replicas_mediana_pos / n_replicas if n_replicas > 0 else np.nan
    p_empirico_media = (1.0 + (n_replicas - n_replicas_media_pos)) / (n_replicas + 1.0) if n_replicas > 0 else np.nan

    if pd.notna(pct_media_pos) and pct_media_pos >= 0.80:
        classificacao_empirica = "favoravel_forte"
    elif pd.notna(pct_media_pos) and pct_media_pos >= 0.60:
        classificacao_empirica = "favoravel_moderada"
    elif pd.notna(pct_media_pos) and pct_media_pos > 0.50:
        classificacao_empirica = "favoravel_fraca"
    elif pd.notna(pct_media_pos) and pct_media_pos == 0.50:
        classificacao_empirica = "neutra"
    else:
        classificacao_empirica = "desfavoravel"

    registros_posicionamento.append({
        "estrategia_referencia": estrategia,
        "frequencia": frequencia,
        "visao_inferencia": visao,
        "n_replicas_aleatorias": n_replicas,
        "n_replicas_media_excesso_positiva": n_replicas_media_pos,
        "n_replicas_mediana_excesso_positiva": n_replicas_mediana_pos,
        "pct_replicas_media_excesso_positiva": pct_media_pos,
        "pct_replicas_mediana_excesso_positiva": pct_mediana_pos,
        "p_empirico_superioridade_media_excesso": p_empirico_media,
        "media_das_medias_excesso_periodico": df_grupo["media_excesso_periodico"].mean(),
        "mediana_das_medias_excesso_periodico": df_grupo["media_excesso_periodico"].median(),
        "min_media_excesso_periodico": df_grupo["media_excesso_periodico"].min(),
        "max_media_excesso_periodico": df_grupo["media_excesso_periodico"].max(),
        "pct_periodos_vencedores_medio": df_grupo["pct_periodos_real_superou_comparador"].mean(),
        "n_replicas_teste_sinal_significativo_5pct": n_replicas_sinal_sig,
        "n_replicas_wilcoxon_significativo_5pct": n_replicas_wilcoxon_sig,
        "n_replicas_permutacao_significativo_5pct": n_replicas_perm_sig,
        "classificacao_empirica_vs_aleatorias": classificacao_empirica,
        "observacao_potencia_replicas": "baixa potência empírica por quantidade limitada de réplicas" if n_replicas <= MIN_REPLICAS_EMPIRICAS else "quantidade de réplicas acima do mínimo operacional",
        "ordem_frequencia_inferencia": MAPA_ORDEM_FREQUENCIA.get(str(frequencia), 99),
    })

df_posicionamento_empirico = pd.DataFrame(registros_posicionamento)
df_posicionamento_empirico = df_posicionamento_empirico.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia"]
).reset_index(drop=True)

print(f"Posicionamento empírico por grupo         : {len(df_posicionamento_empirico):,} linhas")
print("OK")

# ============================================================
# 12) Consolidação do resumo por grupo de controle aleatório
# ============================================================

print("\n[12/14] Consolidação do resumo por grupo de controle aleatório...")

df_resumo_grupos = (
    df_resumo_pares
    .groupby(AGRUPADORES_GRUPO, dropna=False)
    .agg(
        n_pares_aleatorios=("comparacao_id", "nunique"),
        media_excesso_periodico_media_pares=("media_excesso_periodico", "mean"),
        mediana_excesso_periodico_media_pares=("media_excesso_periodico", "median"),
        pct_pares_com_media_excesso_positiva=("media_excesso_periodico", lambda x: float((pd.to_numeric(x, errors="coerce") > 0).mean())),
        pct_pares_com_wilcoxon_significativo_5pct=("p_valor_holm_wilcoxon", lambda x: float((pd.to_numeric(x, errors="coerce") <= ALPHA_REFERENCIA).mean())),
        pct_pares_com_permutacao_significativa_5pct=("p_valor_holm_permutacao", lambda x: float((pd.to_numeric(x, errors="coerce") <= ALPHA_REFERENCIA).mean())),
        pct_periodos_real_superou_comparador_medio=("pct_periodos_real_superou_comparador", "mean"),
        n_testes_significativos_5pct_total=("n_testes_significativos_5pct", "sum"),
    )
    .reset_index()
)

df_resumo_grupos = df_resumo_grupos.merge(
    df_posicionamento_empirico,
    on=AGRUPADORES_GRUPO,
    how="left",
    suffixes=("", "_posicionamento"),
)

df_resumo_grupos["ordem_frequencia_inferencia"] = df_resumo_grupos["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_resumo_grupos = df_resumo_grupos.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia"]
).reset_index(drop=True)

print(f"Resumo consolidado por grupo              : {len(df_resumo_grupos):,} linhas")
print("OK")

# ============================================================
# 13) Base de gráficos de p-valores, parâmetros e auditoria
# ============================================================

print("\n[13/14] Base de gráficos de p-valores, parâmetros e auditoria...")

registros_grafico = []
for _, linha in df_resumo_pares.iterrows():
    metadados = {
        "estrategia_referencia": linha.get("estrategia_referencia"),
        "frequencia": linha.get("frequencia"),
        "visao_inferencia": linha.get("visao_inferencia"),
        "comparacao_id": linha.get("comparacao_id"),
        "nome_exibicao_comparador": linha.get("nome_exibicao_comparador"),
        "media_excesso_periodico": linha.get("media_excesso_periodico"),
        "pct_periodos_real_superou_comparador": linha.get("pct_periodos_real_superou_comparador"),
        "classificacao_evidencia_aleatorio": linha.get("classificacao_evidencia_aleatorio"),
        "ordem_frequencia_inferencia": linha.get("ordem_frequencia_inferencia"),
    }
    for teste, coluna in [
        ("teste_sinal_binomial", "p_valor_holm_teste_sinal"),
        ("wilcoxon_pareado", "p_valor_holm_wilcoxon"),
        ("permutacao_pareada_media", "p_valor_holm_permutacao"),
    ]:
        registro = metadados.copy()
        registro["teste_estatistico"] = teste
        registro["p_valor_holm_14_3"] = linha.get(coluna)
        registros_grafico.append(registro)

df_base_grafico_pvalues = pd.DataFrame(registros_grafico)
if len(df_base_grafico_pvalues) > 0:
    df_base_grafico_pvalues = df_base_grafico_pvalues.sort_values(
        ["ordem_frequencia_inferencia", "estrategia_referencia", "comparacao_id", "teste_estatistico"]
    ).reset_index(drop=True)

PARAMETROS_TESTES_ALEATORIOS = {
    "etapa": "14",
    "subetapa": "14.3",
    "nome_subetapa": "Testes contra Estratégias Aleatórias de Controle",
    "base_entrada": "14_1_base_inferencia_controles_pareada.parquet",
    "tipo_comparacao_filtrado": "controle_aleatorio",
    "frequencia_principal_inferencia": "mensal",
    "frequencia_complementar_inferencia": "diaria",
    "frequencia_exploratoria": "anual",
    "teste_sinal": "binomial unilateral para probabilidade de superação maior que 50%",
    "teste_wilcoxon": "Wilcoxon pareado unilateral para mediana do excesso maior que zero",
    "teste_permutacao": "permutação pareada por inversão de sinal para média do excesso maior que zero",
    "n_permutacoes": N_PERMUTACOES,
    "semente_base_permutacao": SEMENTE_PERMUTACAO,
    "min_observacoes_teste": MIN_OBSERVACOES_TESTE,
    "alpha_referencia": ALPHA_REFERENCIA,
    "alpha_indicativo": ALPHA_INDICATIVO,
    "ajuste_multiplas_comparacoes_local": "Holm por frequência dentro da subetapa 14.3",
    "observacao_metodologica": "A posição empírica frente às réplicas aleatórias tem baixa potência porque há poucas réplicas por estratégia e frequência.",
}

df_parametros = pd.DataFrame(
    [{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_TESTES_ALEATORIOS.items()]
)

frequencias_identificadas = int(df_base_aleatorios["frequencia"].nunique()) if len(df_base_aleatorios) > 0 else 0
estrategias_identificadas = int(df_base_aleatorios["estrategia_referencia"].nunique()) if len(df_base_aleatorios) > 0 else 0
comparacoes_identificadas = int(df_base_aleatorios["comparacao_id"].nunique()) if len(df_base_aleatorios) > 0 else 0
comparacoes_mensais = int((df_resumo_pares["frequencia"] == "mensal").sum()) if len(df_resumo_pares) > 0 else 0
comparacoes_diarias = int((df_resumo_pares["frequencia"] == "diaria").sum()) if len(df_resumo_pares) > 0 else 0
comparacoes_anuais = int((df_resumo_pares["frequencia"] == "anual").sum()) if len(df_resumo_pares) > 0 else 0
duplicatas_comparacao_periodo = int(df_base_aleatorios.duplicated(["comparacao_id", "frequencia", "periodo_comparacao"]).sum())
valores_excesso_ausentes = int(df_base_aleatorios["retorno_excesso_real_vs_comparador"].isna().sum())
min_replicas_grupo = int(df_posicionamento_empirico["n_replicas_aleatorias"].min()) if len(df_posicionamento_empirico) > 0 else 0

metricas_infinitas = 0
for df_verificacao in [
    df_base_aleatorios,
    df_estatisticas,
    df_teste_sinal,
    df_teste_wilcoxon,
    df_teste_permutacao,
    df_resumo_pares,
    df_posicionamento_empirico,
    df_resumo_grupos,
    df_base_grafico_pvalues,
]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number]).apply(pd.to_numeric, errors="coerce")
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy(dtype=float)).sum())

linhas_auditoria = [
    {
        "item": "linhas_base_testes_aleatorios",
        "valor": len(df_base_aleatorios),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_aleatorios) > 0 else "ERRO",
        "observacao": "A subetapa deve carregar comparações pareadas contra controles aleatórios preparadas na 14.1.",
    },
    {
        "item": "comparacoes_aleatorias_identificadas",
        "valor": comparacoes_identificadas,
        "valor_referencia": ">= 30",
        "status": "OK" if comparacoes_identificadas >= 30 else "ERRO",
        "observacao": "Espera-se 2 estratégias x 5 réplicas aleatórias x 3 frequências.",
    },
    {
        "item": "estrategias_referencia_identificadas",
        "valor": estrategias_identificadas,
        "valor_referencia": ">= 2",
        "status": "OK" if estrategias_identificadas >= 2 else "ERRO",
        "observacao": "Espera-se encontrar Capitulação e Euforia.",
    },
    {
        "item": "frequencias_identificadas",
        "valor": frequencias_identificadas,
        "valor_referencia": ">= 3",
        "status": "OK" if frequencias_identificadas >= 3 else "ERRO",
        "observacao": "A subetapa deve preservar as visões diária, mensal e anual.",
    },
    {
        "item": "comparacoes_mensais_principais",
        "valor": comparacoes_mensais,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_mensais > 0 else "ERRO",
        "observacao": "A visão mensal deve existir como referência principal dos testes formais.",
    },
    {
        "item": "comparacoes_diarias_complementares",
        "valor": comparacoes_diarias,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_diarias > 0 else "ERRO",
        "observacao": "A visão diária deve ser preservada como análise complementar.",
    },
    {
        "item": "comparacoes_anuais_exploratorias",
        "valor": comparacoes_anuais,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_anuais > 0 else "ERRO",
        "observacao": "A visão anual deve ser preservada como análise exploratória.",
    },
    {
        "item": "replicas_minimas_por_grupo",
        "valor": min_replicas_grupo,
        "valor_referencia": ">= 5",
        "status": "OK" if min_replicas_grupo >= 5 else "ERRO",
        "observacao": "Cada estratégia e frequência deve preservar ao menos cinco réplicas aleatórias.",
    },
    {
        "item": "testes_sinal_calculados",
        "valor": len(df_teste_sinal),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_sinal) > 0 else "ERRO",
        "observacao": "O teste binomial de sinal deve ser calculado para os pares aplicáveis.",
    },
    {
        "item": "testes_wilcoxon_calculados",
        "valor": len(df_teste_wilcoxon),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_wilcoxon) > 0 else "ERRO",
        "observacao": "O teste de Wilcoxon pareado deve ser calculado para os pares aplicáveis.",
    },
    {
        "item": "testes_permutacao_calculados",
        "valor": len(df_teste_permutacao),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_permutacao) > 0 else "ERRO",
        "observacao": "O teste de permutação pareado deve ser calculado para os pares aplicáveis.",
    },
    {
        "item": "posicionamento_empirico_linhas",
        "valor": len(df_posicionamento_empirico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_posicionamento_empirico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar posicionamento empírico das estratégias reais frente às réplicas aleatórias.",
    },
    {
        "item": "duplicatas_comparacao_frequencia_periodo",
        "valor": duplicatas_comparacao_periodo,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_comparacao_periodo == 0 else "ERRO",
        "observacao": "Cada comparação deve ter no máximo uma linha por frequência e período.",
    },
    {
        "item": "valores_excesso_ausentes",
        "valor": valores_excesso_ausentes,
        "valor_referencia": "0",
        "status": "OK" if valores_excesso_ausentes == 0 else "ERRO",
        "observacao": "A base de testes não deve conter excesso de retorno ausente.",
    },
    {
        "item": "base_grafico_linhas",
        "valor": len(df_base_grafico_pvalues),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico_pvalues) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Base gráfica de p-valores                 : {len(df_base_grafico_pvalues):,} linhas")
print(f"Parâmetros metodológicos                  : {len(df_parametros):,} linhas")
print(f"Itens de auditoria                        : {len(df_auditoria):,}")
print(f"Erros bloqueantes                         : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 14) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[14/14] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(preparar_dataframe_parquet(df_base_aleatorios), CAMINHO_BASE_TESTES, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_parametros), CAMINHO_PARAMETROS, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_teste_sinal), CAMINHO_TESTE_SINAL, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_teste_wilcoxon), CAMINHO_TESTE_WILCOXON, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_teste_permutacao), CAMINHO_TESTE_PERMUTACAO, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_posicionamento_empirico), CAMINHO_POSICIONAMENTO, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_resumo_pares), CAMINHO_RESUMO_PARES, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_resumo_grupos), CAMINHO_RESUMO_GRUPOS, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_base_grafico_pvalues), CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe(preparar_dataframe_parquet(df_auditoria), CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação dos testes contra estratégias aleatórias:")
print(df_auditoria.to_string(index=False))

print("\nResumo dos testes contra estratégias aleatórias - pares:")
colunas_resumo_pares_print = [
    coluna for coluna in [
        "estrategia_referencia",
        "frequencia",
        "visao_inferencia",
        "nome_exibicao_comparador",
        "n_periodos",
        "media_excesso_periodico",
        "media_excesso_anualizada_aproximada",
        "pct_periodos_real_superou_comparador",
        "p_valor_holm_teste_sinal",
        "p_valor_holm_wilcoxon",
        "p_valor_holm_permutacao",
        "n_testes_significativos_5pct",
        "classificacao_evidencia_aleatorio",
    ] if coluna in df_resumo_pares.columns
]
print(df_resumo_pares[colunas_resumo_pares_print].to_string(index=False))

print("\nPosicionamento empírico frente às réplicas aleatórias:")
colunas_pos_print = [
    coluna for coluna in [
        "estrategia_referencia",
        "frequencia",
        "visao_inferencia",
        "n_replicas_aleatorias",
        "pct_replicas_media_excesso_positiva",
        "p_empirico_superioridade_media_excesso",
        "media_das_medias_excesso_periodico",
        "pct_periodos_vencedores_medio",
        "classificacao_empirica_vs_aleatorias",
        "observacao_potencia_replicas",
    ] if coluna in df_posicionamento_empirico.columns
]
print(df_posicionamento_empirico[colunas_pos_print].to_string(index=False))

print("\nResumo consolidado por grupo de controles aleatórios:")
colunas_grupo_print = [
    coluna for coluna in [
        "estrategia_referencia",
        "frequencia",
        "visao_inferencia",
        "n_pares_aleatorios",
        "media_excesso_periodico_media_pares",
        "pct_pares_com_media_excesso_positiva",
        "pct_periodos_real_superou_comparador_medio",
        "n_testes_significativos_5pct_total",
        "classificacao_empirica_vs_aleatorias",
    ] if coluna in df_resumo_grupos.columns
]
print(df_resumo_grupos[colunas_grupo_print].to_string(index=False))

print("\nArquivos salvos na subetapa 14.3:")
print(f"- {CAMINHO_BASE_TESTES}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_TESTE_SINAL}")
print(f"- {CAMINHO_TESTE_WILCOXON}")
print(f"- {CAMINHO_TESTE_PERMUTACAO}")
print(f"- {CAMINHO_POSICIONAMENTO}")
print(f"- {CAMINHO_RESUMO_PARES}")
print(f"- {CAMINHO_RESUMO_GRUPOS}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 14.3 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 14.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 14.3 - TESTES CONTRA ESTRATÉGIAS ALEATÓRIAS DE CONTROLE

[1/14] Validação inicial do ambiente...
OK

[2/14] Definição determinística dos caminhos de entrada e saída...
Entrada - base controles da 14.1        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_base_inferencia_controles_pareada.parquet
Entrada - base consolidada da 14.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_base_inferencia_comparacoes_pareadas.parquet
Entrada - mapa comparações da 14.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_mapa_comparacoes_inferencia.parquet
Entrada - parâmetros da 14.1            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\re

## Etapa 14.4) Testes contra Estratégias Mensais de Controle

In [74]:
%%time
# ============================================================
# Etapa 14.4) Testes contra Estratégias Mensais de Controle
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 14.4 - TESTES CONTRA ESTRATÉGIAS MENSAIS DE CONTROLE")
print("=" * 100)

# Esta subetapa executa testes estatísticos pareados contra os controles mensais.
# A visão mensal permanece como referência principal de inferência, a diária como complementar e a anual como exploratória.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/14] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "stats" not in globals():
    raise NameError("O módulo scipy.stats deve estar importado como stats antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/14] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
DIR_ETAPA_14.mkdir(parents=True, exist_ok=True)

CAMINHO_BASE_CONTROLES_14_1 = DIR_ETAPA_14 / "14_1_base_inferencia_controles_pareada.parquet"
CAMINHO_BASE_CONSOLIDADA_14_1 = DIR_ETAPA_14 / "14_1_base_inferencia_comparacoes_pareadas.parquet"
CAMINHO_MAPA_COMPARACOES_14_1 = DIR_ETAPA_14 / "14_1_tbl_mapa_comparacoes_inferencia.parquet"
CAMINHO_PARAMETROS_14_1 = DIR_ETAPA_14 / "14_1_tbl_parametros_inferencia_estatistica.parquet"

CAMINHO_BASE_TESTES = DIR_ETAPA_14 / "14_4_base_mensais_testes_superacao.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_14 / "14_4_tbl_parametros_testes_mensais.parquet"
CAMINHO_TESTE_SINAL = DIR_ETAPA_14 / "14_4_tbl_teste_sinal_mensais.parquet"
CAMINHO_TESTE_WILCOXON = DIR_ETAPA_14 / "14_4_tbl_teste_wilcoxon_mensais.parquet"
CAMINHO_TESTE_PERMUTACAO = DIR_ETAPA_14 / "14_4_tbl_teste_permutacao_mensais.parquet"
CAMINHO_TESTE_GRUPOS_COMPLEMENTAR = DIR_ETAPA_14 / "14_4_tbl_teste_grupos_mensais_complementar.parquet"
CAMINHO_POSICIONAMENTO = DIR_ETAPA_14 / "14_4_tbl_posicionamento_empirico_mensais.parquet"
CAMINHO_RESUMO_PARES = DIR_ETAPA_14 / "14_4_tbl_resumo_testes_mensais_pares.parquet"
CAMINHO_RESUMO_GRUPOS = DIR_ETAPA_14 / "14_4_tbl_resumo_testes_mensais_grupos.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_14 / "14_4_tbl_base_grafico_pvalues_mensais.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_14 / "14_4_tbl_auditoria_validacao_testes_mensais.parquet"

CAMINHOS_OBRIGATORIOS = [
    CAMINHO_BASE_CONTROLES_14_1,
    CAMINHO_BASE_CONSOLIDADA_14_1,
    CAMINHO_MAPA_COMPARACOES_14_1,
    CAMINHO_PARAMETROS_14_1,
]

for caminho in CAMINHOS_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - base controles da 14.1        : {CAMINHO_BASE_CONTROLES_14_1}")
print(f"Entrada - base consolidada da 14.1      : {CAMINHO_BASE_CONSOLIDADA_14_1}")
print(f"Entrada - mapa comparações da 14.1      : {CAMINHO_MAPA_COMPARACOES_14_1}")
print(f"Entrada - parâmetros da 14.1            : {CAMINHO_PARAMETROS_14_1}")
print(f"Saída   - base de testes mensais        : {CAMINHO_BASE_TESTES}")
print(f"Saída   - parâmetros da 14.4            : {CAMINHO_PARAMETROS}")
print(f"Saída   - teste de sinal mensais        : {CAMINHO_TESTE_SINAL}")
print(f"Saída   - teste Wilcoxon mensais        : {CAMINHO_TESTE_WILCOXON}")
print(f"Saída   - teste permutação mensais      : {CAMINHO_TESTE_PERMUTACAO}")
print(f"Saída   - teste complementar por grupos : {CAMINHO_TESTE_GRUPOS_COMPLEMENTAR}")
print(f"Saída   - posicionamento empírico       : {CAMINHO_POSICIONAMENTO}")
print(f"Saída   - resumo dos pares              : {CAMINHO_RESUMO_PARES}")
print(f"Saída   - resumo dos grupos             : {CAMINHO_RESUMO_GRUPOS}")
print(f"Saída   - base gráfico p-valores        : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação        : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/14] Carga das bases oficiais da subetapa...")

df_controles_14_1 = pd.read_parquet(CAMINHO_BASE_CONTROLES_14_1)
df_consolidada_14_1 = pd.read_parquet(CAMINHO_BASE_CONSOLIDADA_14_1)
df_mapa_14_1 = pd.read_parquet(CAMINHO_MAPA_COMPARACOES_14_1)
df_parametros_14_1 = pd.read_parquet(CAMINHO_PARAMETROS_14_1)

print(f"Base controles da 14.1        : {len(df_controles_14_1):,} linhas x {df_controles_14_1.shape[1]:,} colunas")
print(f"Base consolidada da 14.1      : {len(df_consolidada_14_1):,} linhas x {df_consolidada_14_1.shape[1]:,} colunas")
print(f"Mapa comparações da 14.1      : {len(df_mapa_14_1):,} linhas x {df_mapa_14_1.shape[1]:,} colunas")
print(f"Parâmetros da 14.1            : {len(df_parametros_14_1):,} linhas x {df_parametros_14_1.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/14] Funções auxiliares da subetapa...")

ALPHA_REFERENCIA = 0.05
ALPHA_INDICATIVO = 0.10
N_PERMUTACOES = 2000
SEMENTE_PERMUTACAO = 20260426
MIN_OBSERVACOES_TESTES = 12
TOLERANCIA_EMPATE = 0.0000000001

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}


def validar_colunas_obrigatorias(df, colunas, nome_base):
    ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(ausentes) > 0:
        raise KeyError(
            f"Colunas obrigatórias ausentes na base {nome_base}: {ausentes}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )


def obter_primeiro_valor_valido(serie, valor_padrao=np.nan):
    valores = serie.dropna()
    if len(valores) == 0:
        return valor_padrao
    return valores.iloc[0]


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def converter_booleano(serie):
    if pd.api.types.is_bool_dtype(serie):
        return serie.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce").fillna(0.0).ne(0.0)
    texto = serie.astype(str).str.strip().str.lower()
    return texto.isin(["true", "1", "sim", "s", "yes", "y", "ok", "verdadeiro"])


def identificar_tipo_controle_mensal(nome_comparador, chave_comparador=None, subcategoria=None, familia=None):
    texto = " ".join([
        "" if pd.isna(nome_comparador) else str(nome_comparador),
        "" if pd.isna(chave_comparador) else str(chave_comparador),
        "" if pd.isna(subcategoria) else str(subcategoria),
        "" if pd.isna(familia) else str(familia),
    ]).strip().lower()

    texto_ascii = (
        texto
        .replace("ações", "acoes")
        .replace("ação", "acao")
        .replace("aleatórias", "aleatorias")
        .replace("aleatória", "aleatoria")
        .replace("elegíveis", "elegiveis")
        .replace("elegível", "elegivel")
        .replace("mensáis", "mensais")
    )

    if "ibovespa" in texto_ascii and "aportes mensais" in texto_ascii:
        return "ibovespa_aportes_mensais"
    if "elegiveis" in texto_ascii and "aportes mensais" in texto_ascii:
        return "acoes_elegiveis_aportes_mensais"
    if "20 aportes mensais" in texto_ascii:
        return "acoes_aleatorias_20_aportes_mensais"
    if "30 aportes mensais" in texto_ascii:
        return "acoes_aleatorias_30_aportes_mensais"
    if "aleatorias" in texto_ascii and "mensais" in texto_ascii:
        return "acoes_aleatorias_aportes_mensais"
    if "mensal" in texto_ascii or "mensais" in texto_ascii:
        return "outros_controles_mensais"
    return "controle_mensal_nao_classificado"


def nome_exibicao_tipo_controle(tipo_controle):
    mapa = {
        "ibovespa_aportes_mensais": "Ibovespa com aportes mensais",
        "acoes_elegiveis_aportes_mensais": "Ações elegíveis com aportes mensais",
        "acoes_aleatorias_20_aportes_mensais": "Ações aleatórias com 20 aportes mensais",
        "acoes_aleatorias_30_aportes_mensais": "Ações aleatórias com 30 aportes mensais",
        "acoes_aleatorias_aportes_mensais": "Ações aleatórias com aportes mensais",
        "outros_controles_mensais": "Outros controles mensais",
        "controle_mensal_nao_classificado": "Controle mensal não classificado",
    }
    return mapa.get(str(tipo_controle), str(tipo_controle))


def ajustar_pvalores_holm(pvalores):
    serie = pd.Series(pvalores, dtype="float64")
    ajustado = pd.Series(np.nan, index=serie.index, dtype="float64")
    validos = serie.dropna()

    if len(validos) == 0:
        return ajustado

    ordenados = validos.sort_values()
    m = len(ordenados)
    acumulado = 0.0

    for posicao, (idx, pvalor) in enumerate(ordenados.items(), start=1):
        valor_ajustado = min((m - posicao + 1) * float(pvalor), 1.0)
        acumulado = max(acumulado, valor_ajustado)
        ajustado.loc[idx] = min(acumulado, 1.0)

    return ajustado


def teste_sinal_binomial(valores, tolerancia=TOLERANCIA_EMPATE):
    valores = pd.Series(valores).dropna().astype(float)
    valores = valores[np.isfinite(valores)]
    n_vitorias = int((valores > tolerancia).sum())
    n_derrotas = int((valores < -tolerancia).sum())
    n_empates = int((valores.abs() <= tolerancia).sum())
    n_validos = n_vitorias + n_derrotas

    if n_validos < MIN_OBSERVACOES_TESTES:
        return {
            "n_validos_teste_sinal": n_validos,
            "n_vitorias": n_vitorias,
            "n_derrotas": n_derrotas,
            "n_empates": n_empates,
            "estatistica_teste_sinal": np.nan,
            "p_valor_teste_sinal": np.nan,
            "status_teste_sinal": "observacoes_insuficientes",
        }

    resultado = stats.binomtest(n_vitorias, n=n_validos, p=0.5, alternative="greater")

    return {
        "n_validos_teste_sinal": n_validos,
        "n_vitorias": n_vitorias,
        "n_derrotas": n_derrotas,
        "n_empates": n_empates,
        "estatistica_teste_sinal": n_vitorias,
        "p_valor_teste_sinal": float(resultado.pvalue),
        "status_teste_sinal": "calculado",
    }


def teste_wilcoxon_pareado(valores, tolerancia=TOLERANCIA_EMPATE):
    valores = pd.Series(valores).dropna().astype(float)
    valores = valores[np.isfinite(valores)]
    valores_nao_zero = valores[~np.isclose(valores, 0.0, atol=tolerancia, rtol=0.0)]

    if len(valores_nao_zero) < MIN_OBSERVACOES_TESTES:
        return {
            "n_validos_wilcoxon": int(len(valores_nao_zero)),
            "estatistica_wilcoxon": np.nan,
            "p_valor_wilcoxon": np.nan,
            "status_wilcoxon": "observacoes_insuficientes",
        }

    try:
        resultado = stats.wilcoxon(
            valores_nao_zero,
            alternative="greater",
            zero_method="wilcox",
            correction=False,
            mode="auto",
        )
        estatistica = float(resultado.statistic)
        pvalor = float(resultado.pvalue)
        status = "calculado"
    except ValueError as erro:
        estatistica = np.nan
        pvalor = np.nan
        status = f"erro_wilcoxon: {erro}"

    return {
        "n_validos_wilcoxon": int(len(valores_nao_zero)),
        "estatistica_wilcoxon": estatistica,
        "p_valor_wilcoxon": pvalor,
        "status_wilcoxon": status,
    }


def teste_permutacao_pareado_media(valores, n_permutacoes=N_PERMUTACOES, semente=SEMENTE_PERMUTACAO, tolerancia=TOLERANCIA_EMPATE):
    valores = pd.Series(valores).dropna().astype(float)
    valores = valores[np.isfinite(valores)]
    valores = valores[~np.isclose(valores, 0.0, atol=tolerancia, rtol=0.0)]

    if len(valores) < MIN_OBSERVACOES_TESTES:
        return {
            "n_validos_permutacao": int(len(valores)),
            "estatistica_observada_permutacao": np.nan,
            "p_valor_permutacao": np.nan,
            "n_permutacoes": int(n_permutacoes),
            "status_permutacao": "observacoes_insuficientes",
        }

    valores_np = valores.to_numpy(dtype=float)
    estatistica_observada = float(np.mean(valores_np))

    if np.isclose(np.std(valores_np), 0.0, atol=tolerancia, rtol=0.0):
        return {
            "n_validos_permutacao": int(len(valores_np)),
            "estatistica_observada_permutacao": estatistica_observada,
            "p_valor_permutacao": 1.0,
            "n_permutacoes": int(n_permutacoes),
            "status_permutacao": "sem_variabilidade",
        }

    rng = np.random.default_rng(semente)
    sinais = rng.choice(np.array([-1.0, 1.0]), size=(int(n_permutacoes), len(valores_np)), replace=True)
    medias_permutadas = (sinais * valores_np).mean(axis=1)
    pvalor = (1.0 + np.sum(medias_permutadas >= estatistica_observada)) / (float(n_permutacoes) + 1.0)

    return {
        "n_validos_permutacao": int(len(valores_np)),
        "estatistica_observada_permutacao": estatistica_observada,
        "p_valor_permutacao": float(pvalor),
        "n_permutacoes": int(n_permutacoes),
        "status_permutacao": "calculado",
    }


def classificar_evidencia(media_excesso, n_testes_significativos_5pct, n_testes_significativos_10pct):
    if pd.isna(media_excesso):
        return "nao_classificada"
    if media_excesso > 0 and n_testes_significativos_5pct >= 2:
        return "favoravel_estatistica_forte"
    if media_excesso > 0 and n_testes_significativos_5pct >= 1:
        return "favoravel_estatistica_indicativa"
    if media_excesso > 0 and n_testes_significativos_10pct >= 1:
        return "favoravel_indicativa_10pct"
    if media_excesso > 0:
        return "favoravel_economica_sem_significancia"
    if media_excesso < 0:
        return "desfavoravel_economica_sem_significancia"
    return "neutra_sem_diferenca"


def preparar_dataframe_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def salvar_dataframe_seguro(df, caminho, index=False):
    salvar_dataframe(preparar_dataframe_para_parquet(df), caminho, index=index)


def preparar_auditoria(linhas):
    df = pd.DataFrame(linhas)
    if len(df) == 0:
        return pd.DataFrame(columns=["item", "valor", "valor_referencia", "status", "observacao"])
    df["status"] = df["status"].astype(str)
    return df[["item", "valor", "valor_referencia", "status", "observacao"]].copy()

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização da base de estratégias mensais de controle
# ============================================================

print("\n[5/14] Padronização da base de estratégias mensais de controle...")

COLUNAS_OBRIGATORIAS_BASE = [
    "comparacao_id",
    "grupo_comparacao",
    "tipo_comparacao",
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "periodo_comparacao",
    "nome_exibicao_comparador",
    "retorno_periodo_real",
    "retorno_periodo_comparador",
    "retorno_excesso_real_vs_comparador",
    "flag_real_superou_comparador",
]

validar_colunas_obrigatorias(df_controles_14_1, COLUNAS_OBRIGATORIAS_BASE, "14.1 base controles pareada")

mascara_mensais = (
    df_controles_14_1["tipo_comparacao"].astype(str).eq("controle_mensal")
    | df_controles_14_1["grupo_comparacao"].astype(str).eq("controle_mensal")
)

df_base_mensais = df_controles_14_1.loc[mascara_mensais].copy()

if len(df_base_mensais) == 0:
    raise ValueError("A base de controles mensais ficou vazia após o filtro tipo_comparacao/grupo_comparacao = controle_mensal.")

df_base_mensais["retorno_periodo_real"] = converter_numero(df_base_mensais["retorno_periodo_real"])
df_base_mensais["retorno_periodo_comparador"] = converter_numero(df_base_mensais["retorno_periodo_comparador"])
df_base_mensais["retorno_excesso_real_vs_comparador"] = converter_numero(df_base_mensais["retorno_excesso_real_vs_comparador"])
df_base_mensais["flag_real_superou_comparador"] = converter_booleano(df_base_mensais["flag_real_superou_comparador"])

if "flag_real_empatou_comparador" in df_base_mensais.columns:
    df_base_mensais["flag_real_empatou_comparador"] = converter_booleano(df_base_mensais["flag_real_empatou_comparador"])
else:
    df_base_mensais["flag_real_empatou_comparador"] = np.isclose(
        df_base_mensais["retorno_excesso_real_vs_comparador"],
        0.0,
        atol=TOLERANCIA_EMPATE,
        rtol=0.0,
    )

if "flag_real_perdeu_comparador" in df_base_mensais.columns:
    df_base_mensais["flag_real_perdeu_comparador"] = converter_booleano(df_base_mensais["flag_real_perdeu_comparador"])
else:
    df_base_mensais["flag_real_perdeu_comparador"] = df_base_mensais["retorno_excesso_real_vs_comparador"] < -TOLERANCIA_EMPATE

for coluna_opcional in ["subcategoria_comparador", "familia_comparador", "chave_comparador"]:
    if coluna_opcional not in df_base_mensais.columns:
        df_base_mensais[coluna_opcional] = np.nan

df_base_mensais["tipo_controle_mensal"] = df_base_mensais.apply(
    lambda row: identificar_tipo_controle_mensal(
        row.get("nome_exibicao_comparador", np.nan),
        row.get("chave_comparador", np.nan),
        row.get("subcategoria_comparador", np.nan),
        row.get("familia_comparador", np.nan),
    ),
    axis=1,
)
df_base_mensais["nome_tipo_controle_mensal"] = df_base_mensais["tipo_controle_mensal"].map(nome_exibicao_tipo_controle)
df_base_mensais["ordem_frequencia_inferencia"] = df_base_mensais["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)

df_base_mensais = df_base_mensais.dropna(
    subset=[
        "retorno_periodo_real",
        "retorno_periodo_comparador",
        "retorno_excesso_real_vs_comparador",
    ]
).copy()

df_base_mensais = df_base_mensais.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia", "tipo_controle_mensal", "comparacao_id", "periodo_comparacao"]
).reset_index(drop=True)

print(f"Base de testes contra mensais             : {len(df_base_mensais):,} linhas")
print(f"Comparações contra mensais                : {df_base_mensais['comparacao_id'].nunique():,}")
print(f"Estratégias de referência                 : {df_base_mensais['estrategia_referencia'].nunique():,}")
print(f"Frequências preservadas                   : {df_base_mensais['frequencia'].nunique():,}")
print(f"Tipos de controle mensal identificados    : {df_base_mensais['tipo_controle_mensal'].nunique():,}")
print("OK")

# ============================================================
# 6) Construção das estatísticas descritivas por par de comparação
# ============================================================

print("\n[6/14] Construção das estatísticas descritivas por par de comparação...")

AGRUPADORES_PAR = [
    "comparacao_id",
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "grupo_comparacao",
    "tipo_comparacao",
    "tipo_controle_mensal",
    "nome_tipo_controle_mensal",
    "chave_comparador",
    "nome_exibicao_comparador",
]

df_estatisticas_pares = (
    df_base_mensais
    .groupby(AGRUPADORES_PAR, dropna=False)
    .agg(
        n_periodos=("periodo_comparacao", "count"),
        n_periodos_unicos=("periodo_comparacao", "nunique"),
        data_inicio_min=("data_inicio_periodo", "min"),
        data_fim_max=("data_fim_periodo", "max"),
        media_retorno_real=("retorno_periodo_real", "mean"),
        media_retorno_comparador=("retorno_periodo_comparador", "mean"),
        media_excesso_periodico=("retorno_excesso_real_vs_comparador", "mean"),
        mediana_excesso_periodico=("retorno_excesso_real_vs_comparador", "median"),
        desvio_excesso_periodico=("retorno_excesso_real_vs_comparador", "std"),
        minimo_excesso_periodico=("retorno_excesso_real_vs_comparador", "min"),
        maximo_excesso_periodico=("retorno_excesso_real_vs_comparador", "max"),
        pct_periodos_real_superou_comparador=("flag_real_superou_comparador", "mean"),
        pct_periodos_real_empatou_comparador=("flag_real_empatou_comparador", "mean"),
        pct_periodos_real_perdeu_comparador=("flag_real_perdeu_comparador", "mean"),
    )
    .reset_index()
)

df_estatisticas_pares["media_excesso_anualizada_aproximada"] = np.select(
    [
        df_estatisticas_pares["frequencia"].eq("diaria"),
        df_estatisticas_pares["frequencia"].eq("mensal"),
        df_estatisticas_pares["frequencia"].eq("anual"),
    ],
    [
        df_estatisticas_pares["media_excesso_periodico"] * 252.0,
        df_estatisticas_pares["media_excesso_periodico"] * 12.0,
        df_estatisticas_pares["media_excesso_periodico"],
    ],
    default=np.nan,
)

df_estatisticas_pares["ordem_frequencia_inferencia"] = df_estatisticas_pares["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_estatisticas_pares = df_estatisticas_pares.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia", "tipo_controle_mensal", "comparacao_id"]
).reset_index(drop=True)

print(f"Estatísticas descritivas por par          : {len(df_estatisticas_pares):,} linhas")
print("OK")

# ============================================================
# 7) Execução do teste binomial de sinal por par
# ============================================================

print("\n[7/14] Execução do teste binomial de sinal por par...")

registros_sinal = []
for chaves, df_grupo in df_base_mensais.groupby(AGRUPADORES_PAR, dropna=False):
    registro = dict(zip(AGRUPADORES_PAR, chaves))
    resultado = teste_sinal_binomial(df_grupo["retorno_excesso_real_vs_comparador"])
    registro.update(resultado)
    registros_sinal.append(registro)

df_teste_sinal = pd.DataFrame(registros_sinal)
if len(df_teste_sinal) > 0:
    df_teste_sinal["p_valor_holm_teste_sinal"] = ajustar_pvalores_holm(df_teste_sinal["p_valor_teste_sinal"])
    df_teste_sinal["flag_significativo_5pct_teste_sinal"] = df_teste_sinal["p_valor_holm_teste_sinal"] < ALPHA_REFERENCIA
    df_teste_sinal["flag_significativo_10pct_teste_sinal"] = df_teste_sinal["p_valor_holm_teste_sinal"] < ALPHA_INDICATIVO

print(f"Testes binomiais de sinal calculados      : {len(df_teste_sinal):,} linhas")
print("OK")

# ============================================================
# 8) Execução do teste de Wilcoxon pareado por par
# ============================================================

print("\n[8/14] Execução do teste de Wilcoxon pareado por par...")

registros_wilcoxon = []
for chaves, df_grupo in df_base_mensais.groupby(AGRUPADORES_PAR, dropna=False):
    registro = dict(zip(AGRUPADORES_PAR, chaves))
    resultado = teste_wilcoxon_pareado(df_grupo["retorno_excesso_real_vs_comparador"])
    registro.update(resultado)
    registros_wilcoxon.append(registro)

df_teste_wilcoxon = pd.DataFrame(registros_wilcoxon)
if len(df_teste_wilcoxon) > 0:
    df_teste_wilcoxon["p_valor_holm_wilcoxon"] = ajustar_pvalores_holm(df_teste_wilcoxon["p_valor_wilcoxon"])
    df_teste_wilcoxon["flag_significativo_5pct_wilcoxon"] = df_teste_wilcoxon["p_valor_holm_wilcoxon"] < ALPHA_REFERENCIA
    df_teste_wilcoxon["flag_significativo_10pct_wilcoxon"] = df_teste_wilcoxon["p_valor_holm_wilcoxon"] < ALPHA_INDICATIVO

print(f"Testes de Wilcoxon calculados             : {len(df_teste_wilcoxon):,} linhas")
print("OK")

# ============================================================
# 9) Execução do teste de permutação pareado por par
# ============================================================

print("\n[9/14] Execução do teste de permutação pareado por par...")

registros_permutacao = []
for indice_grupo, (chaves, df_grupo) in enumerate(df_base_mensais.groupby(AGRUPADORES_PAR, dropna=False), start=1):
    registro = dict(zip(AGRUPADORES_PAR, chaves))
    resultado = teste_permutacao_pareado_media(
        df_grupo["retorno_excesso_real_vs_comparador"],
        n_permutacoes=N_PERMUTACOES,
        semente=SEMENTE_PERMUTACAO + indice_grupo,
    )
    registro.update(resultado)
    registros_permutacao.append(registro)

df_teste_permutacao = pd.DataFrame(registros_permutacao)
if len(df_teste_permutacao) > 0:
    df_teste_permutacao["p_valor_holm_permutacao"] = ajustar_pvalores_holm(df_teste_permutacao["p_valor_permutacao"])
    df_teste_permutacao["flag_significativo_5pct_permutacao"] = df_teste_permutacao["p_valor_holm_permutacao"] < ALPHA_REFERENCIA
    df_teste_permutacao["flag_significativo_10pct_permutacao"] = df_teste_permutacao["p_valor_holm_permutacao"] < ALPHA_INDICATIVO

print(f"Testes de permutação calculados           : {len(df_teste_permutacao):,} linhas")
print("OK")

# ============================================================
# 10) Consolidação do resumo por par de comparação
# ============================================================

print("\n[10/14] Consolidação do resumo por par de comparação...")

CHAVES_RESUMO = AGRUPADORES_PAR

df_resumo_pares = df_estatisticas_pares.merge(
    df_teste_sinal[
        CHAVES_RESUMO
        + [
            "n_validos_teste_sinal",
            "n_vitorias",
            "n_derrotas",
            "n_empates",
            "p_valor_teste_sinal",
            "p_valor_holm_teste_sinal",
            "flag_significativo_5pct_teste_sinal",
            "flag_significativo_10pct_teste_sinal",
            "status_teste_sinal",
        ]
    ],
    on=CHAVES_RESUMO,
    how="left",
)

df_resumo_pares = df_resumo_pares.merge(
    df_teste_wilcoxon[
        CHAVES_RESUMO
        + [
            "n_validos_wilcoxon",
            "estatistica_wilcoxon",
            "p_valor_wilcoxon",
            "p_valor_holm_wilcoxon",
            "flag_significativo_5pct_wilcoxon",
            "flag_significativo_10pct_wilcoxon",
            "status_wilcoxon",
        ]
    ],
    on=CHAVES_RESUMO,
    how="left",
)

df_resumo_pares = df_resumo_pares.merge(
    df_teste_permutacao[
        CHAVES_RESUMO
        + [
            "n_validos_permutacao",
            "estatistica_observada_permutacao",
            "p_valor_permutacao",
            "p_valor_holm_permutacao",
            "n_permutacoes",
            "flag_significativo_5pct_permutacao",
            "flag_significativo_10pct_permutacao",
            "status_permutacao",
        ]
    ],
    on=CHAVES_RESUMO,
    how="left",
)

colunas_sig_5 = [
    "flag_significativo_5pct_teste_sinal",
    "flag_significativo_5pct_wilcoxon",
    "flag_significativo_5pct_permutacao",
]
colunas_sig_10 = [
    "flag_significativo_10pct_teste_sinal",
    "flag_significativo_10pct_wilcoxon",
    "flag_significativo_10pct_permutacao",
]

for coluna in colunas_sig_5 + colunas_sig_10:
    df_resumo_pares[coluna] = df_resumo_pares[coluna].fillna(False).astype(bool)

df_resumo_pares["n_testes_significativos_5pct"] = df_resumo_pares[colunas_sig_5].sum(axis=1).astype(int)
df_resumo_pares["n_testes_significativos_10pct"] = df_resumo_pares[colunas_sig_10].sum(axis=1).astype(int)
df_resumo_pares["classificacao_evidencia_mensal"] = df_resumo_pares.apply(
    lambda row: classificar_evidencia(
        row["media_excesso_periodico"],
        row["n_testes_significativos_5pct"],
        row["n_testes_significativos_10pct"],
    ),
    axis=1,
)

df_resumo_pares = df_resumo_pares.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia", "tipo_controle_mensal", "comparacao_id"]
).reset_index(drop=True)

print(f"Resumo consolidado por par                : {len(df_resumo_pares):,} linhas")
print("OK")

# ============================================================
# 11) Teste complementar por grupos de controles mensais
# ============================================================

print("\n[11/14] Teste complementar por grupos de controles mensais...")

registros_grupos_complementar = []

for (estrategia, frequencia, visao), df_grupo in df_base_mensais.groupby(["estrategia_referencia", "frequencia", "visao_inferencia"], dropna=False):
    df_contingencia = (
        df_grupo
        .assign(
            resultado_sinal=np.select(
                [
                    df_grupo["retorno_excesso_real_vs_comparador"] > TOLERANCIA_EMPATE,
                    df_grupo["retorno_excesso_real_vs_comparador"] < -TOLERANCIA_EMPATE,
                ],
                ["vitoria", "derrota"],
                default="empate",
            )
        )
        .loc[lambda x: x["resultado_sinal"].isin(["vitoria", "derrota"])]
        .groupby(["tipo_controle_mensal", "resultado_sinal"], dropna=False)
        .size()
        .unstack(fill_value=0)
    )

    for coluna_resultado in ["vitoria", "derrota"]:
        if coluna_resultado not in df_contingencia.columns:
            df_contingencia[coluna_resultado] = 0

    df_contingencia = df_contingencia[["vitoria", "derrota"]]
    n_tipos = int(df_contingencia.shape[0])
    n_total = int(df_contingencia.to_numpy().sum())

    estatistica = np.nan
    pvalor = np.nan
    gl = np.nan
    teste = "nao_aplicavel"
    status = "tipos_insuficientes"
    min_esperado = np.nan

    if n_tipos >= 2 and n_total > 0:
        try:
            chi2, p_chi2, dof, expected = stats.chi2_contingency(df_contingencia.to_numpy())
            estatistica = float(chi2)
            pvalor = float(p_chi2)
            gl = float(dof)
            teste = "qui_quadrado_independencia"
            status = "calculado"
            min_esperado = float(np.min(expected))
        except ValueError as erro:
            status = f"erro_qui_quadrado: {erro}"

        if df_contingencia.shape == (2, 2):
            try:
                oddsratio, p_fisher = stats.fisher_exact(df_contingencia.to_numpy(), alternative="two-sided")
                if pd.isna(pvalor) or min_esperado < 5.0:
                    estatistica = float(oddsratio)
                    pvalor = float(p_fisher)
                    gl = np.nan
                    teste = "fisher_exato_2x2"
                    status = "calculado"
            except ValueError:
                pass

    registros_grupos_complementar.append({
        "estrategia_referencia": estrategia,
        "frequencia": frequencia,
        "visao_inferencia": visao,
        "n_tipos_controle_mensal": n_tipos,
        "n_observacoes_sinal": n_total,
        "teste_complementar": teste,
        "estatistica_teste_grupos": estatistica,
        "p_valor_teste_grupos": pvalor,
        "graus_liberdade": gl,
        "min_frequencia_esperada": min_esperado,
        "status_teste_grupos": status,
        "observacao": "Teste complementar para comparar distribuição de vitórias e derrotas entre tipos de controles mensais.",
    })

df_teste_grupos_complementar = pd.DataFrame(registros_grupos_complementar)
if len(df_teste_grupos_complementar) > 0:
    df_teste_grupos_complementar["p_valor_holm_teste_grupos"] = ajustar_pvalores_holm(df_teste_grupos_complementar["p_valor_teste_grupos"])
    df_teste_grupos_complementar["flag_significativo_5pct_teste_grupos"] = df_teste_grupos_complementar["p_valor_holm_teste_grupos"] < ALPHA_REFERENCIA
    df_teste_grupos_complementar["flag_significativo_10pct_teste_grupos"] = df_teste_grupos_complementar["p_valor_holm_teste_grupos"] < ALPHA_INDICATIVO
    df_teste_grupos_complementar["ordem_frequencia_inferencia"] = df_teste_grupos_complementar["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
    df_teste_grupos_complementar = df_teste_grupos_complementar.sort_values(
        ["ordem_frequencia_inferencia", "estrategia_referencia"]
    ).reset_index(drop=True)

print(f"Testes complementares por grupos          : {len(df_teste_grupos_complementar):,} linhas")
print("OK")

# ============================================================
# 12) Posicionamento empírico e resumo por grupos de controle mensal
# ============================================================

print("\n[12/14] Posicionamento empírico e resumo por grupos de controle mensal...")

registros_posicionamento = []
for (estrategia, frequencia, visao), df_grupo in df_resumo_pares.groupby(["estrategia_referencia", "frequencia", "visao_inferencia"], dropna=False):
    n_controles = int(df_grupo["comparacao_id"].nunique())
    pct_medias_positivas = float((df_grupo["media_excesso_periodico"] > 0).mean()) if n_controles > 0 else np.nan
    pct_testes_sig_5 = float((df_grupo["n_testes_significativos_5pct"] > 0).mean()) if n_controles > 0 else np.nan
    pct_testes_sig_10 = float((df_grupo["n_testes_significativos_10pct"] > 0).mean()) if n_controles > 0 else np.nan
    media_das_medias = float(df_grupo["media_excesso_periodico"].mean()) if n_controles > 0 else np.nan
    pct_periodos_vencedores_medio = float(df_grupo["pct_periodos_real_superou_comparador"].mean()) if n_controles > 0 else np.nan

    if pd.isna(pct_medias_positivas):
        classificacao = "nao_classificada"
    elif pct_medias_positivas >= 0.75 and pct_testes_sig_5 > 0:
        classificacao = "favoravel_estatistica"
    elif pct_medias_positivas >= 0.75:
        classificacao = "favoravel_economica"
    elif pct_medias_positivas >= 0.50:
        classificacao = "neutra_a_favoravel"
    else:
        classificacao = "desfavoravel"

    registros_posicionamento.append({
        "estrategia_referencia": estrategia,
        "frequencia": frequencia,
        "visao_inferencia": visao,
        "n_controles_mensais": n_controles,
        "n_tipos_controle_mensal": int(df_grupo["tipo_controle_mensal"].nunique()),
        "pct_controles_media_excesso_positiva": pct_medias_positivas,
        "pct_controles_com_algum_teste_significativo_5pct": pct_testes_sig_5,
        "pct_controles_com_algum_teste_significativo_10pct": pct_testes_sig_10,
        "media_das_medias_excesso_periodico": media_das_medias,
        "pct_periodos_vencedores_medio": pct_periodos_vencedores_medio,
        "classificacao_empirica_vs_mensais": classificacao,
    })

df_posicionamento = pd.DataFrame(registros_posicionamento)
if len(df_posicionamento) > 0:
    df_posicionamento["ordem_frequencia_inferencia"] = df_posicionamento["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
    df_posicionamento = df_posicionamento.sort_values(
        ["ordem_frequencia_inferencia", "estrategia_referencia"]
    ).reset_index(drop=True)

AGRUPADORES_GRUPO = [
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "tipo_controle_mensal",
    "nome_tipo_controle_mensal",
]

df_resumo_grupos = (
    df_resumo_pares
    .groupby(AGRUPADORES_GRUPO, dropna=False)
    .agg(
        n_pares_mensais=("comparacao_id", "nunique"),
        media_excesso_periodico_media_pares=("media_excesso_periodico", "mean"),
        mediana_excesso_periodico_media_pares=("media_excesso_periodico", "median"),
        pct_pares_com_media_excesso_positiva=("media_excesso_periodico", lambda x: float((pd.to_numeric(x, errors="coerce") > 0).mean())),
        pct_periodos_real_superou_comparador_medio=("pct_periodos_real_superou_comparador", "mean"),
        n_testes_significativos_5pct_total=("n_testes_significativos_5pct", "sum"),
        n_testes_significativos_10pct_total=("n_testes_significativos_10pct", "sum"),
    )
    .reset_index()
)

df_resumo_grupos["classificacao_empirica_vs_mensais"] = np.select(
    [
        (df_resumo_grupos["pct_pares_com_media_excesso_positiva"] >= 0.75) & (df_resumo_grupos["n_testes_significativos_5pct_total"] > 0),
        df_resumo_grupos["pct_pares_com_media_excesso_positiva"] >= 0.75,
        df_resumo_grupos["pct_pares_com_media_excesso_positiva"] >= 0.50,
    ],
    [
        "favoravel_estatistica",
        "favoravel_economica",
        "neutra_a_favoravel",
    ],
    default="desfavoravel",
)

df_resumo_grupos["ordem_frequencia_inferencia"] = df_resumo_grupos["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_resumo_grupos = df_resumo_grupos.sort_values(
    ["ordem_frequencia_inferencia", "estrategia_referencia", "tipo_controle_mensal"]
).reset_index(drop=True)

print(f"Posicionamento empírico por frequência    : {len(df_posicionamento):,} linhas")
print(f"Resumo consolidado por grupo              : {len(df_resumo_grupos):,} linhas")
print("OK")

# ============================================================
# 13) Base de gráficos de p-valores, parâmetros e auditoria
# ============================================================

print("\n[13/14] Base de gráficos de p-valores, parâmetros e auditoria...")

registros_grafico = []
for _, row in df_resumo_pares.iterrows():
    base_registro = {
        "comparacao_id": row["comparacao_id"],
        "estrategia_referencia": row["estrategia_referencia"],
        "frequencia": row["frequencia"],
        "visao_inferencia": row["visao_inferencia"],
        "tipo_controle_mensal": row["tipo_controle_mensal"],
        "nome_tipo_controle_mensal": row["nome_tipo_controle_mensal"],
        "nome_exibicao_comparador": row["nome_exibicao_comparador"],
        "n_periodos": row["n_periodos"],
        "media_excesso_periodico": row["media_excesso_periodico"],
        "pct_periodos_real_superou_comparador": row["pct_periodos_real_superou_comparador"],
        "classificacao_evidencia_mensal": row["classificacao_evidencia_mensal"],
    }

    registros_grafico.append({
        **base_registro,
        "teste_estatistico": "teste_sinal_binomial",
        "p_valor_original": row["p_valor_teste_sinal"],
        "p_valor_holm_14_4": row["p_valor_holm_teste_sinal"],
    })
    registros_grafico.append({
        **base_registro,
        "teste_estatistico": "wilcoxon_pareado",
        "p_valor_original": row["p_valor_wilcoxon"],
        "p_valor_holm_14_4": row["p_valor_holm_wilcoxon"],
    })
    registros_grafico.append({
        **base_registro,
        "teste_estatistico": "permutacao_pareada_media",
        "p_valor_original": row["p_valor_permutacao"],
        "p_valor_holm_14_4": row["p_valor_holm_permutacao"],
    })

df_base_grafico = pd.DataFrame(registros_grafico)
if len(df_base_grafico) > 0:
    df_base_grafico["flag_significativo_5pct"] = df_base_grafico["p_valor_holm_14_4"] < ALPHA_REFERENCIA
    df_base_grafico["flag_significativo_10pct"] = df_base_grafico["p_valor_holm_14_4"] < ALPHA_INDICATIVO
    df_base_grafico["ordem_frequencia_inferencia"] = df_base_grafico["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
    df_base_grafico = df_base_grafico.sort_values(
        ["ordem_frequencia_inferencia", "estrategia_referencia", "tipo_controle_mensal", "comparacao_id", "teste_estatistico"]
    ).reset_index(drop=True)

PARAMETROS_14_4 = {
    "etapa": "14",
    "subetapa": "14.4",
    "nome_subetapa": "Testes contra Estratégias Mensais de Controle",
    "comparacao": "Capitulação e Euforia contra controles mensais",
    "frequencia_principal_inferencia": "mensal",
    "frequencia_complementar_inferencia": "diaria",
    "frequencia_exploratoria": "anual",
    "metrica_testada": "retorno_excesso_real_vs_comparador",
    "hipotese_nula_teste_sinal": "probabilidade de superação igual a 50%",
    "hipotese_alternativa_teste_sinal": "probabilidade de superação superior a 50%",
    "hipotese_nula_wilcoxon": "mediana do excesso de retorno igual a zero",
    "hipotese_alternativa_wilcoxon": "mediana do excesso de retorno superior a zero",
    "hipotese_nula_permutacao": "média do excesso de retorno igual a zero por inversão de sinal",
    "hipotese_alternativa_permutacao": "média do excesso de retorno superior a zero",
    "n_permutacoes": N_PERMUTACOES,
    "semente_permutacao": SEMENTE_PERMUTACAO,
    "alpha_referencia": ALPHA_REFERENCIA,
    "alpha_indicativo": ALPHA_INDICATIVO,
    "ajuste_multiplas_comparacoes": "Holm dentro da subetapa 14.4 por família de teste",
    "observacao_metodologica": "O teste complementar por grupos compara distribuições de vitória e derrota entre tipos de controles mensais.",
}

df_parametros = pd.DataFrame(
    [{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_14_4.items()]
)

duplicatas_comparacao_periodo = int(df_base_mensais.duplicated(["comparacao_id", "frequencia", "periodo_comparacao"]).sum())
valores_excesso_ausentes = int(df_base_mensais["retorno_excesso_real_vs_comparador"].isna().sum())
replicas_minimas_por_grupo = int(
    df_base_mensais.groupby(["estrategia_referencia", "frequencia"], dropna=False)["comparacao_id"].nunique().min()
)

metricas_infinitas = 0
for df_verificacao in [
    df_base_mensais,
    df_estatisticas_pares,
    df_teste_sinal,
    df_teste_wilcoxon,
    df_teste_permutacao,
    df_teste_grupos_complementar,
    df_posicionamento,
    df_resumo_pares,
    df_resumo_grupos,
    df_base_grafico,
]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

linhas_auditoria = [
    {
        "item": "linhas_base_testes_mensais",
        "valor": len(df_base_mensais),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_mensais) > 0 else "ERRO",
        "observacao": "A subetapa deve carregar comparações pareadas contra controles mensais preparadas na 14.1.",
    },
    {
        "item": "comparacoes_mensais_identificadas",
        "valor": int(df_base_mensais["comparacao_id"].nunique()),
        "valor_referencia": ">= 60",
        "status": "OK" if df_base_mensais["comparacao_id"].nunique() >= 60 else "ERRO",
        "observacao": "Espera-se múltiplos controles mensais para 2 estratégias e 3 frequências.",
    },
    {
        "item": "estrategias_referencia_identificadas",
        "valor": int(df_base_mensais["estrategia_referencia"].nunique()),
        "valor_referencia": ">= 2",
        "status": "OK" if df_base_mensais["estrategia_referencia"].nunique() >= 2 else "ERRO",
        "observacao": "Espera-se encontrar Capitulação e Euforia.",
    },
    {
        "item": "frequencias_identificadas",
        "valor": int(df_base_mensais["frequencia"].nunique()),
        "valor_referencia": ">= 3",
        "status": "OK" if df_base_mensais["frequencia"].nunique() >= 3 else "ERRO",
        "observacao": "A subetapa deve preservar as visões diária, mensal e anual.",
    },
    {
        "item": "tipos_controle_mensal_identificados",
        "valor": int(df_base_mensais["tipo_controle_mensal"].nunique()),
        "valor_referencia": ">= 3",
        "status": "OK" if df_base_mensais["tipo_controle_mensal"].nunique() >= 3 else "ERRO",
        "observacao": "A subetapa deve separar os diferentes tipos de controle mensal.",
    },
    {
        "item": "comparacoes_mensais_principais",
        "valor": int(df_resumo_pares["frequencia"].eq("mensal").sum()),
        "valor_referencia": "> 0",
        "status": "OK" if int(df_resumo_pares["frequencia"].eq("mensal").sum()) > 0 else "ERRO",
        "observacao": "A visão mensal deve existir como referência principal dos testes formais.",
    },
    {
        "item": "comparacoes_diarias_complementares",
        "valor": int(df_resumo_pares["frequencia"].eq("diaria").sum()),
        "valor_referencia": "> 0",
        "status": "OK" if int(df_resumo_pares["frequencia"].eq("diaria").sum()) > 0 else "ERRO",
        "observacao": "A visão diária deve ser preservada como análise complementar.",
    },
    {
        "item": "comparacoes_anuais_exploratorias",
        "valor": int(df_resumo_pares["frequencia"].eq("anual").sum()),
        "valor_referencia": "> 0",
        "status": "OK" if int(df_resumo_pares["frequencia"].eq("anual").sum()) > 0 else "ERRO",
        "observacao": "A visão anual deve ser preservada como análise exploratória.",
    },
    {
        "item": "controles_minimos_por_grupo",
        "valor": replicas_minimas_por_grupo,
        "valor_referencia": ">= 10",
        "status": "OK" if replicas_minimas_por_grupo >= 10 else "ERRO",
        "observacao": "Cada estratégia e frequência deve preservar múltiplos controles mensais.",
    },
    {
        "item": "testes_sinal_calculados",
        "valor": int(len(df_teste_sinal)),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_sinal) > 0 else "ERRO",
        "observacao": "O teste binomial de sinal deve ser calculado para os pares aplicáveis.",
    },
    {
        "item": "testes_wilcoxon_calculados",
        "valor": int(len(df_teste_wilcoxon)),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_wilcoxon) > 0 else "ERRO",
        "observacao": "O teste de Wilcoxon pareado deve ser calculado para os pares aplicáveis.",
    },
    {
        "item": "testes_permutacao_calculados",
        "valor": int(len(df_teste_permutacao)),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_permutacao) > 0 else "ERRO",
        "observacao": "O teste de permutação pareado deve ser calculado para os pares aplicáveis.",
    },
    {
        "item": "teste_complementar_grupos_linhas",
        "valor": int(len(df_teste_grupos_complementar)),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_grupos_complementar) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar teste complementar por grupos de controles mensais.",
    },
    {
        "item": "posicionamento_empirico_linhas",
        "valor": int(len(df_posicionamento)),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_posicionamento) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar posicionamento empírico das estratégias reais frente aos controles mensais.",
    },
    {
        "item": "duplicatas_comparacao_frequencia_periodo",
        "valor": duplicatas_comparacao_periodo,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_comparacao_periodo == 0 else "ERRO",
        "observacao": "Cada comparação deve ter no máximo uma linha por frequência e período.",
    },
    {
        "item": "valores_excesso_ausentes",
        "valor": valores_excesso_ausentes,
        "valor_referencia": "0",
        "status": "OK" if valores_excesso_ausentes == 0 else "ERRO",
        "observacao": "A base de testes não deve conter excesso de retorno ausente.",
    },
    {
        "item": "base_grafico_linhas",
        "valor": int(len(df_base_grafico)),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Base gráfica de p-valores                 : {len(df_base_grafico):,} linhas")
print(f"Parâmetros metodológicos                  : {len(df_parametros):,} linhas")
print(f"Itens de auditoria                        : {len(df_auditoria):,}")
print(f"Erros bloqueantes                         : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 14) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[14/14] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe_seguro(df_base_mensais, CAMINHO_BASE_TESTES, index=False)
salvar_dataframe_seguro(df_parametros, CAMINHO_PARAMETROS, index=False)
salvar_dataframe_seguro(df_teste_sinal, CAMINHO_TESTE_SINAL, index=False)
salvar_dataframe_seguro(df_teste_wilcoxon, CAMINHO_TESTE_WILCOXON, index=False)
salvar_dataframe_seguro(df_teste_permutacao, CAMINHO_TESTE_PERMUTACAO, index=False)
salvar_dataframe_seguro(df_teste_grupos_complementar, CAMINHO_TESTE_GRUPOS_COMPLEMENTAR, index=False)
salvar_dataframe_seguro(df_posicionamento, CAMINHO_POSICIONAMENTO, index=False)
salvar_dataframe_seguro(df_resumo_pares, CAMINHO_RESUMO_PARES, index=False)
salvar_dataframe_seguro(df_resumo_grupos, CAMINHO_RESUMO_GRUPOS, index=False)
salvar_dataframe_seguro(df_base_grafico, CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe_seguro(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação dos testes contra estratégias mensais:")
print(df_auditoria.to_string(index=False))

print("\nResumo dos testes contra estratégias mensais - pares:")
colunas_print_pares = [
    coluna for coluna in [
        "estrategia_referencia",
        "frequencia",
        "visao_inferencia",
        "tipo_controle_mensal",
        "nome_exibicao_comparador",
        "n_periodos",
        "media_excesso_periodico",
        "media_excesso_anualizada_aproximada",
        "pct_periodos_real_superou_comparador",
        "p_valor_holm_teste_sinal",
        "p_valor_holm_wilcoxon",
        "p_valor_holm_permutacao",
        "n_testes_significativos_5pct",
        "classificacao_evidencia_mensal",
    ] if coluna in df_resumo_pares.columns
]
print(df_resumo_pares[colunas_print_pares].to_string(index=False))

print("\nPosicionamento empírico frente aos controles mensais:")
if len(df_posicionamento) > 0:
    print(df_posicionamento.to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nResumo consolidado por tipo de controle mensal:")
if len(df_resumo_grupos) > 0:
    colunas_print_grupos = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "visao_inferencia",
            "tipo_controle_mensal",
            "n_pares_mensais",
            "media_excesso_periodico_media_pares",
            "pct_pares_com_media_excesso_positiva",
            "pct_periodos_real_superou_comparador_medio",
            "n_testes_significativos_5pct_total",
            "classificacao_empirica_vs_mensais",
        ] if coluna in df_resumo_grupos.columns
    ]
    print(df_resumo_grupos[colunas_print_grupos].to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nTeste complementar por grupos de controles mensais:")
if len(df_teste_grupos_complementar) > 0:
    colunas_print_grupos_teste = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "visao_inferencia",
            "n_tipos_controle_mensal",
            "n_observacoes_sinal",
            "teste_complementar",
            "p_valor_holm_teste_grupos",
            "status_teste_grupos",
        ] if coluna in df_teste_grupos_complementar.columns
    ]
    print(df_teste_grupos_complementar[colunas_print_grupos_teste].to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nArquivos salvos na subetapa 14.4:")
print(f"- {CAMINHO_BASE_TESTES}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_TESTE_SINAL}")
print(f"- {CAMINHO_TESTE_WILCOXON}")
print(f"- {CAMINHO_TESTE_PERMUTACAO}")
print(f"- {CAMINHO_TESTE_GRUPOS_COMPLEMENTAR}")
print(f"- {CAMINHO_POSICIONAMENTO}")
print(f"- {CAMINHO_RESUMO_PARES}")
print(f"- {CAMINHO_RESUMO_GRUPOS}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 14.4 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 14.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 14.4 - TESTES CONTRA ESTRATÉGIAS MENSAIS DE CONTROLE

[1/14] Validação inicial do ambiente...
OK

[2/14] Definição determinística dos caminhos de entrada e saída...
Entrada - base controles da 14.1        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_base_inferencia_controles_pareada.parquet
Entrada - base consolidada da 14.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_base_inferencia_comparacoes_pareadas.parquet
Entrada - mapa comparações da 14.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_mapa_comparacoes_inferencia.parquet
Entrada - parâmetros da 14.1            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resul

## Etapa 14.5) Testes entre Capitulação e Euforia

In [75]:
%%time
# ============================================================
# Etapa 14.5) Testes entre Capitulação e Euforia
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 14.5 - TESTES ENTRE CAPITULAÇÃO E EUFORIA")
print("=" * 100)

# Esta subetapa executa testes estatísticos pareados entre Capitulação e Euforia.
# A diferença principal é definida como Capitulação menos Euforia.
# Valores positivos favorecem Capitulação; valores negativos favorecem Euforia.
# A visão mensal permanece como referência principal de inferência, a diária como complementar e a anual como exploratória.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/13] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "stats" not in globals():
    raise NameError("O módulo scipy.stats deve estar importado como stats antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/13] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
DIR_ETAPA_14.mkdir(parents=True, exist_ok=True)

CAMINHO_BASE_DIRETA_14_1 = DIR_ETAPA_14 / "14_1_base_inferencia_comparacao_direta_pareada.parquet"
CAMINHO_BASE_CONSOLIDADA_14_1 = DIR_ETAPA_14 / "14_1_base_inferencia_comparacoes_pareadas.parquet"
CAMINHO_MAPA_COMPARACOES_14_1 = DIR_ETAPA_14 / "14_1_tbl_mapa_comparacoes_inferencia.parquet"
CAMINHO_PARAMETROS_14_1 = DIR_ETAPA_14 / "14_1_tbl_parametros_inferencia_estatistica.parquet"

CAMINHO_BASE_TESTES = DIR_ETAPA_14 / "14_5_base_capitulacao_euforia_testes_comparacao.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_14 / "14_5_tbl_parametros_testes_capitulacao_euforia.parquet"
CAMINHO_TESTE_SINAL = DIR_ETAPA_14 / "14_5_tbl_teste_sinal_capitulacao_euforia.parquet"
CAMINHO_TESTE_WILCOXON = DIR_ETAPA_14 / "14_5_tbl_teste_wilcoxon_capitulacao_euforia.parquet"
CAMINHO_TESTE_PERMUTACAO = DIR_ETAPA_14 / "14_5_tbl_teste_permutacao_capitulacao_euforia.parquet"
CAMINHO_RESUMO_TESTES = DIR_ETAPA_14 / "14_5_tbl_resumo_testes_capitulacao_euforia.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_14 / "14_5_tbl_base_grafico_pvalues_capitulacao_euforia.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_14 / "14_5_tbl_auditoria_validacao_testes_capitulacao_euforia.parquet"

CAMINHOS_OBRIGATORIOS = [
    CAMINHO_BASE_DIRETA_14_1,
    CAMINHO_BASE_CONSOLIDADA_14_1,
    CAMINHO_MAPA_COMPARACOES_14_1,
    CAMINHO_PARAMETROS_14_1,
]

for caminho in CAMINHOS_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - base direta da 14.1           : {CAMINHO_BASE_DIRETA_14_1}")
print(f"Entrada - base consolidada da 14.1      : {CAMINHO_BASE_CONSOLIDADA_14_1}")
print(f"Entrada - mapa comparações da 14.1      : {CAMINHO_MAPA_COMPARACOES_14_1}")
print(f"Entrada - parâmetros da 14.1            : {CAMINHO_PARAMETROS_14_1}")
print(f"Saída   - base de testes Cap. vs Euforia: {CAMINHO_BASE_TESTES}")
print(f"Saída   - parâmetros da 14.5            : {CAMINHO_PARAMETROS}")
print(f"Saída   - teste de sinal Cap. vs Euforia: {CAMINHO_TESTE_SINAL}")
print(f"Saída   - teste Wilcoxon Cap. vs Euforia: {CAMINHO_TESTE_WILCOXON}")
print(f"Saída   - teste permutação Cap. vs Euf. : {CAMINHO_TESTE_PERMUTACAO}")
print(f"Saída   - resumo dos testes             : {CAMINHO_RESUMO_TESTES}")
print(f"Saída   - base gráfico p-valores        : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - auditoria de validação        : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/13] Carga das bases oficiais da subetapa...")

df_direta_14_1 = pd.read_parquet(CAMINHO_BASE_DIRETA_14_1)
df_consolidada_14_1 = pd.read_parquet(CAMINHO_BASE_CONSOLIDADA_14_1)
df_mapa_14_1 = pd.read_parquet(CAMINHO_MAPA_COMPARACOES_14_1)
df_parametros_14_1 = pd.read_parquet(CAMINHO_PARAMETROS_14_1)

print(f"Base direta da 14.1           : {len(df_direta_14_1):,} linhas x {df_direta_14_1.shape[1]:,} colunas")
print(f"Base consolidada da 14.1      : {len(df_consolidada_14_1):,} linhas x {df_consolidada_14_1.shape[1]:,} colunas")
print(f"Mapa comparações da 14.1      : {len(df_mapa_14_1):,} linhas x {df_mapa_14_1.shape[1]:,} colunas")
print(f"Parâmetros da 14.1            : {len(df_parametros_14_1):,} linhas x {df_parametros_14_1.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/13] Funções auxiliares da subetapa...")

ALPHA_REFERENCIA = 0.05
ALPHA_INDICATIVO = 0.10
N_PERMUTACOES = 2000
SEMENTE_PERMUTACAO = 20260426
MIN_OBSERVACOES_TESTES = 12
TOLERANCIA_EMPATE = 0.0000000001

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}


def validar_colunas_obrigatorias(df, colunas, nome_base):
    ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(ausentes) > 0:
        raise KeyError(
            f"Colunas obrigatórias ausentes na base {nome_base}: {ausentes}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )


def obter_primeiro_valor_valido(serie, valor_padrao=np.nan):
    valores = serie.dropna()
    if len(valores) == 0:
        return valor_padrao
    return valores.iloc[0]


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def ajustar_pvalores_holm(pvalores):
    serie = pd.Series(pvalores, dtype="float64")
    ajustado = pd.Series(np.nan, index=serie.index, dtype="float64")
    validos = serie.dropna()

    if len(validos) == 0:
        return ajustado

    ordenados = validos.sort_values()
    m = len(ordenados)
    acumulado = 0.0

    for posicao, (idx, pvalor) in enumerate(ordenados.items(), start=1):
        valor_ajustado = min((m - posicao + 1) * float(pvalor), 1.0)
        acumulado = max(acumulado, valor_ajustado)
        ajustado.loc[idx] = min(acumulado, 1.0)

    return ajustado


def calcular_binomial_sinal(diferencas):
    serie = pd.Series(diferencas).dropna().astype(float)
    positivos = int((serie > TOLERANCIA_EMPATE).sum())
    negativos = int((serie < -TOLERANCIA_EMPATE).sum())
    empates = int((serie.abs() <= TOLERANCIA_EMPATE).sum())
    n_validos = positivos + negativos

    if n_validos < MIN_OBSERVACOES_TESTES:
        return {
            "n_vitorias_capitulacao": positivos,
            "n_vitorias_euforia": negativos,
            "n_empates": empates,
            "n_observacoes_teste": n_validos,
            "estatistica_teste_sinal": np.nan,
            "p_valor_teste_sinal": np.nan,
            "status_teste_sinal": "observacoes_insuficientes",
        }

    try:
        resultado = stats.binomtest(positivos, n_validos, p=0.5, alternative="two-sided")
        p_valor = float(resultado.pvalue)
    except AttributeError:
        p_valor = float(stats.binom_test(positivos, n_validos, p=0.5, alternative="two-sided"))

    return {
        "n_vitorias_capitulacao": positivos,
        "n_vitorias_euforia": negativos,
        "n_empates": empates,
        "n_observacoes_teste": n_validos,
        "estatistica_teste_sinal": positivos,
        "p_valor_teste_sinal": p_valor,
        "status_teste_sinal": "calculado",
    }


def calcular_wilcoxon_pareado(diferencas):
    serie = pd.Series(diferencas).dropna().astype(float)
    serie = serie.loc[serie.abs() > TOLERANCIA_EMPATE]

    if len(serie) < MIN_OBSERVACOES_TESTES:
        return {
            "n_observacoes_teste": int(len(serie)),
            "estatistica_wilcoxon": np.nan,
            "p_valor_wilcoxon": np.nan,
            "status_wilcoxon": "observacoes_insuficientes",
        }

    try:
        resultado = stats.wilcoxon(serie, alternative="two-sided", zero_method="wilcox")
        estatistica = float(resultado.statistic)
        p_valor = float(resultado.pvalue)
        status = "calculado"
    except ValueError:
        estatistica = np.nan
        p_valor = np.nan
        status = "erro_calculo"

    return {
        "n_observacoes_teste": int(len(serie)),
        "estatistica_wilcoxon": estatistica,
        "p_valor_wilcoxon": p_valor,
        "status_wilcoxon": status,
    }


def calcular_permutacao_pareada(diferencas, n_permutacoes=N_PERMUTACOES, semente=SEMENTE_PERMUTACAO):
    serie = pd.Series(diferencas).dropna().astype(float)
    serie = serie.loc[serie.abs() > TOLERANCIA_EMPATE]

    if len(serie) < MIN_OBSERVACOES_TESTES:
        return {
            "n_observacoes_teste": int(len(serie)),
            "estatistica_observada_media": np.nan,
            "p_valor_permutacao": np.nan,
            "n_permutacoes": int(n_permutacoes),
            "status_permutacao": "observacoes_insuficientes",
        }

    valores = serie.to_numpy(dtype=float)
    estatistica_observada = float(np.mean(valores))
    rng = np.random.default_rng(semente + int(len(valores)))
    estatisticas_permutadas = np.empty(n_permutacoes, dtype=float)

    for i in range(n_permutacoes):
        sinais = rng.choice(np.array([-1.0, 1.0]), size=len(valores), replace=True)
        estatisticas_permutadas[i] = np.mean(valores * sinais)

    p_valor = float((np.sum(np.abs(estatisticas_permutadas) >= abs(estatistica_observada)) + 1.0) / (n_permutacoes + 1.0))

    return {
        "n_observacoes_teste": int(len(serie)),
        "estatistica_observada_media": estatistica_observada,
        "p_valor_permutacao": p_valor,
        "n_permutacoes": int(n_permutacoes),
        "status_permutacao": "calculado",
    }


def classificar_direcao(media_excesso, mediana_excesso, pct_capitulacao_superou):
    media = float(media_excesso) if pd.notna(media_excesso) else np.nan
    mediana = float(mediana_excesso) if pd.notna(mediana_excesso) else np.nan
    pct = float(pct_capitulacao_superou) if pd.notna(pct_capitulacao_superou) else np.nan

    pontos_capitulacao = 0
    pontos_euforia = 0

    if pd.notna(media):
        if media > TOLERANCIA_EMPATE:
            pontos_capitulacao += 1
        elif media < -TOLERANCIA_EMPATE:
            pontos_euforia += 1

    if pd.notna(mediana):
        if mediana > TOLERANCIA_EMPATE:
            pontos_capitulacao += 1
        elif mediana < -TOLERANCIA_EMPATE:
            pontos_euforia += 1

    if pd.notna(pct):
        if pct > 0.5:
            pontos_capitulacao += 1
        elif pct < 0.5:
            pontos_euforia += 1

    if pontos_capitulacao > pontos_euforia:
        return "direcao_capitulacao"
    if pontos_euforia > pontos_capitulacao:
        return "direcao_euforia"
    return "direcao_mista_ou_neutra"


def classificar_evidencia(row):
    direcao = str(row.get("direcao_economica", "direcao_mista_ou_neutra"))
    n_sig_5 = int(row.get("n_testes_significativos_5pct", 0)) if pd.notna(row.get("n_testes_significativos_5pct", np.nan)) else 0
    n_sig_10 = int(row.get("n_testes_significativos_10pct", 0)) if pd.notna(row.get("n_testes_significativos_10pct", np.nan)) else 0

    if direcao == "direcao_capitulacao":
        if n_sig_5 > 0:
            return "evidencia_estatistica_capitulacao_superior"
        if n_sig_10 > 0:
            return "evidencia_indicativa_capitulacao_superior"
        return "favoravel_economica_capitulacao_sem_significancia"

    if direcao == "direcao_euforia":
        if n_sig_5 > 0:
            return "evidencia_estatistica_euforia_superior"
        if n_sig_10 > 0:
            return "evidencia_indicativa_euforia_superior"
        return "favoravel_economica_euforia_sem_significancia"

    return "neutra_ou_mista_sem_significancia"


def padronizar_dataframe_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def salvar_parquet_seguro(df, caminho, index=False):
    salvar_dataframe(padronizar_dataframe_para_parquet(df), caminho, index=index)


def preparar_auditoria(linhas):
    df = pd.DataFrame(linhas)
    if len(df) == 0:
        return pd.DataFrame(columns=["item", "valor", "valor_referencia", "status", "observacao"])
    df["status"] = df["status"].astype(str)
    return df[["item", "valor", "valor_referencia", "status", "observacao"]].copy()

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização da base de comparação direta
# ============================================================

print("\n[5/13] Padronização da base de comparação direta...")

validar_colunas_obrigatorias(
    df_direta_14_1,
    [
        "grupo_comparacao",
        "tipo_comparacao",
        "comparacao_id",
        "estrategia_referencia",
        "frequencia",
        "visao_inferencia",
        "periodo_comparacao",
        "retorno_periodo_real",
        "retorno_periodo_comparador",
        "retorno_excesso_real_vs_comparador",
        "flag_real_superou_comparador",
        "flag_real_empatou_comparador",
        "flag_real_perdeu_comparador",
    ],
    "14.1 comparação direta",
)

df_base_testes = df_direta_14_1.loc[
    df_direta_14_1["grupo_comparacao"].astype(str).eq("comparacao_direta")
    & df_direta_14_1["tipo_comparacao"].astype(str).eq("capitulacao_vs_euforia")
].copy()

if len(df_base_testes) == 0:
    raise ValueError("A base de comparação direta entre Capitulação e Euforia está vazia.")

for coluna in [
    "retorno_periodo_real",
    "retorno_periodo_comparador",
    "retorno_excesso_real_vs_comparador",
    "retorno_relativo_composto_real_vs_comparador",
]:
    if coluna in df_base_testes.columns:
        df_base_testes[coluna] = converter_numero(df_base_testes[coluna])

df_base_testes = df_base_testes.rename(
    columns={
        "retorno_periodo_real": "retorno_periodo_capitulacao",
        "retorno_periodo_comparador": "retorno_periodo_euforia",
        "retorno_excesso_real_vs_comparador": "retorno_excesso_capitulacao_vs_euforia",
        "retorno_relativo_composto_real_vs_comparador": "retorno_relativo_composto_capitulacao_vs_euforia",
        "flag_real_superou_comparador": "flag_capitulacao_superou_euforia",
        "flag_real_empatou_comparador": "flag_capitulacao_empatou_euforia",
        "flag_real_perdeu_comparador": "flag_euforia_superou_capitulacao",
        "chave_estrategia_real": "chave_capitulacao",
        "chave_comparador": "chave_euforia",
        "nome_exibicao_estrategia_real": "nome_exibicao_capitulacao",
        "nome_exibicao_comparador": "nome_exibicao_euforia",
    }
)

for coluna_booleana in [
    "flag_capitulacao_superou_euforia",
    "flag_capitulacao_empatou_euforia",
    "flag_euforia_superou_capitulacao",
]:
    if coluna_booleana in df_base_testes.columns:
        df_base_testes[coluna_booleana] = df_base_testes[coluna_booleana].fillna(False).astype(bool)

linhas_antes = len(df_base_testes)
df_base_testes = df_base_testes.dropna(
    subset=[
        "retorno_periodo_capitulacao",
        "retorno_periodo_euforia",
        "retorno_excesso_capitulacao_vs_euforia",
    ]
).copy()
linhas_descartadas = linhas_antes - len(df_base_testes)

df_base_testes["direcao_excesso"] = np.select(
    [
        df_base_testes["retorno_excesso_capitulacao_vs_euforia"] > TOLERANCIA_EMPATE,
        df_base_testes["retorno_excesso_capitulacao_vs_euforia"] < -TOLERANCIA_EMPATE,
    ],
    ["capitulacao_superou", "euforia_superou"],
    default="empate_tecnico",
)

df_base_testes["flag_teste_formal_principal"] = df_base_testes["frequencia"].astype(str).eq("mensal")
df_base_testes["flag_teste_complementar"] = df_base_testes["frequencia"].astype(str).eq("diaria")
df_base_testes["flag_teste_exploratorio"] = df_base_testes["frequencia"].astype(str).eq("anual")

df_base_testes = df_base_testes.sort_values(
    ["ordem_frequencia_inferencia", "frequencia", "periodo_comparacao"]
).reset_index(drop=True)

print(f"Base de testes Capitulação vs Euforia    : {len(df_base_testes):,} linhas")
print(f"Comparações diretas identificadas        : {df_base_testes['comparacao_id'].nunique():,}")
print(f"Frequências preservadas                  : {df_base_testes['frequencia'].nunique():,}")
print(f"Linhas descartadas por ausência de retorno: {linhas_descartadas:,}")
print("OK")

# ============================================================
# 6) Construção das estatísticas descritivas por frequência
# ============================================================

print("\n[6/13] Construção das estatísticas descritivas por frequência...")

agrupadores = [
    "comparacao_id",
    "grupo_comparacao",
    "tipo_comparacao",
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "descricao_visao_inferencia",
    "ordem_frequencia_inferencia",
]

df_descritivas = (
    df_base_testes
    .groupby(agrupadores, dropna=False)
    .agg(
        n_periodos=("periodo_comparacao", "count"),
        data_inicio_min=("data_inicio_periodo", "min"),
        data_fim_max=("data_fim_periodo", "max"),
        retorno_medio_capitulacao=("retorno_periodo_capitulacao", "mean"),
        retorno_medio_euforia=("retorno_periodo_euforia", "mean"),
        media_excesso_periodico=("retorno_excesso_capitulacao_vs_euforia", "mean"),
        mediana_excesso_periodico=("retorno_excesso_capitulacao_vs_euforia", "median"),
        desvio_excesso_periodico=("retorno_excesso_capitulacao_vs_euforia", "std"),
        minimo_excesso_periodico=("retorno_excesso_capitulacao_vs_euforia", "min"),
        maximo_excesso_periodico=("retorno_excesso_capitulacao_vs_euforia", "max"),
        pct_periodos_capitulacao_superou_euforia=("flag_capitulacao_superou_euforia", "mean"),
        pct_periodos_euforia_superou_capitulacao=("flag_euforia_superou_capitulacao", "mean"),
        pct_periodos_empate_tecnico=("flag_capitulacao_empatou_euforia", "mean"),
    )
    .reset_index()
)

df_descritivas["media_excesso_anualizada_aproximada"] = np.select(
    [
        df_descritivas["frequencia"].astype(str).eq("diaria"),
        df_descritivas["frequencia"].astype(str).eq("mensal"),
        df_descritivas["frequencia"].astype(str).eq("anual"),
    ],
    [
        df_descritivas["media_excesso_periodico"] * 252.0,
        df_descritivas["media_excesso_periodico"] * 12.0,
        df_descritivas["media_excesso_periodico"],
    ],
    default=np.nan,
)

df_descritivas["direcao_economica"] = df_descritivas.apply(
    lambda row: classificar_direcao(
        row["media_excesso_periodico"],
        row["mediana_excesso_periodico"],
        row["pct_periodos_capitulacao_superou_euforia"],
    ),
    axis=1,
)

df_descritivas = df_descritivas.sort_values(["ordem_frequencia_inferencia", "frequencia"]).reset_index(drop=True)

print(f"Estatísticas descritivas por frequência  : {len(df_descritivas):,} linhas")
print("OK")

# ============================================================
# 7) Execução do teste binomial de sinal
# ============================================================

print("\n[7/13] Execução do teste binomial de sinal...")

registros_sinal = []

for chaves, df_grupo in df_base_testes.groupby(agrupadores, dropna=False):
    registro = dict(zip(agrupadores, chaves))
    resultado = calcular_binomial_sinal(df_grupo["retorno_excesso_capitulacao_vs_euforia"])
    registro.update(resultado)
    registros_sinal.append(registro)

df_teste_sinal = pd.DataFrame(registros_sinal)

if len(df_teste_sinal) > 0:
    df_teste_sinal["pct_capitulacao_superou_euforia_teste"] = np.where(
        df_teste_sinal["n_observacoes_teste"] > 0,
        df_teste_sinal["n_vitorias_capitulacao"] / df_teste_sinal["n_observacoes_teste"],
        np.nan,
    )
    df_teste_sinal["p_valor_holm_teste_sinal"] = ajustar_pvalores_holm(df_teste_sinal["p_valor_teste_sinal"])
    df_teste_sinal["significativo_5pct_teste_sinal"] = df_teste_sinal["p_valor_holm_teste_sinal"].le(ALPHA_REFERENCIA)
    df_teste_sinal["significativo_10pct_teste_sinal"] = df_teste_sinal["p_valor_holm_teste_sinal"].le(ALPHA_INDICATIVO)
    df_teste_sinal = df_teste_sinal.sort_values(["ordem_frequencia_inferencia", "frequencia"]).reset_index(drop=True)

print(f"Testes binomiais de sinal calculados     : {len(df_teste_sinal):,} linhas")
print("OK")

# ============================================================
# 8) Execução do teste de Wilcoxon pareado
# ============================================================

print("\n[8/13] Execução do teste de Wilcoxon pareado...")

registros_wilcoxon = []

for chaves, df_grupo in df_base_testes.groupby(agrupadores, dropna=False):
    registro = dict(zip(agrupadores, chaves))
    resultado = calcular_wilcoxon_pareado(df_grupo["retorno_excesso_capitulacao_vs_euforia"])
    registro.update(resultado)
    registros_wilcoxon.append(registro)

df_teste_wilcoxon = pd.DataFrame(registros_wilcoxon)

if len(df_teste_wilcoxon) > 0:
    df_teste_wilcoxon["p_valor_holm_wilcoxon"] = ajustar_pvalores_holm(df_teste_wilcoxon["p_valor_wilcoxon"])
    df_teste_wilcoxon["significativo_5pct_wilcoxon"] = df_teste_wilcoxon["p_valor_holm_wilcoxon"].le(ALPHA_REFERENCIA)
    df_teste_wilcoxon["significativo_10pct_wilcoxon"] = df_teste_wilcoxon["p_valor_holm_wilcoxon"].le(ALPHA_INDICATIVO)
    df_teste_wilcoxon = df_teste_wilcoxon.sort_values(["ordem_frequencia_inferencia", "frequencia"]).reset_index(drop=True)

print(f"Testes de Wilcoxon calculados            : {len(df_teste_wilcoxon):,} linhas")
print("OK")

# ============================================================
# 9) Execução do teste de permutação pareado
# ============================================================

print("\n[9/13] Execução do teste de permutação pareado...")

registros_permutacao = []

for chaves, df_grupo in df_base_testes.groupby(agrupadores, dropna=False):
    registro = dict(zip(agrupadores, chaves))
    resultado = calcular_permutacao_pareada(df_grupo["retorno_excesso_capitulacao_vs_euforia"])
    registro.update(resultado)
    registros_permutacao.append(registro)

df_teste_permutacao = pd.DataFrame(registros_permutacao)

if len(df_teste_permutacao) > 0:
    df_teste_permutacao["p_valor_holm_permutacao"] = ajustar_pvalores_holm(df_teste_permutacao["p_valor_permutacao"])
    df_teste_permutacao["significativo_5pct_permutacao"] = df_teste_permutacao["p_valor_holm_permutacao"].le(ALPHA_REFERENCIA)
    df_teste_permutacao["significativo_10pct_permutacao"] = df_teste_permutacao["p_valor_holm_permutacao"].le(ALPHA_INDICATIVO)
    df_teste_permutacao = df_teste_permutacao.sort_values(["ordem_frequencia_inferencia", "frequencia"]).reset_index(drop=True)

print(f"Testes de permutação calculados          : {len(df_teste_permutacao):,} linhas")
print("OK")

# ============================================================
# 10) Consolidação do resumo dos testes
# ============================================================

print("\n[10/13] Consolidação do resumo dos testes...")

colunas_merge = agrupadores

df_resumo_testes = df_descritivas.copy()

colunas_sinal = colunas_merge + [
    "n_vitorias_capitulacao",
    "n_vitorias_euforia",
    "n_empates",
    "n_observacoes_teste",
    "pct_capitulacao_superou_euforia_teste",
    "p_valor_teste_sinal",
    "p_valor_holm_teste_sinal",
    "significativo_5pct_teste_sinal",
    "significativo_10pct_teste_sinal",
    "status_teste_sinal",
]

df_resumo_testes = df_resumo_testes.merge(
    df_teste_sinal[colunas_sinal],
    on=colunas_merge,
    how="left",
)

df_resumo_testes = df_resumo_testes.merge(
    df_teste_wilcoxon[
        colunas_merge + [
            "estatistica_wilcoxon",
            "p_valor_wilcoxon",
            "p_valor_holm_wilcoxon",
            "significativo_5pct_wilcoxon",
            "significativo_10pct_wilcoxon",
            "status_wilcoxon",
        ]
    ],
    on=colunas_merge,
    how="left",
)

df_resumo_testes = df_resumo_testes.merge(
    df_teste_permutacao[
        colunas_merge + [
            "estatistica_observada_media",
            "p_valor_permutacao",
            "p_valor_holm_permutacao",
            "significativo_5pct_permutacao",
            "significativo_10pct_permutacao",
            "n_permutacoes",
            "status_permutacao",
        ]
    ],
    on=colunas_merge,
    how="left",
)

for coluna in [
    "significativo_5pct_teste_sinal",
    "significativo_5pct_wilcoxon",
    "significativo_5pct_permutacao",
    "significativo_10pct_teste_sinal",
    "significativo_10pct_wilcoxon",
    "significativo_10pct_permutacao",
]:
    if coluna in df_resumo_testes.columns:
        df_resumo_testes[coluna] = df_resumo_testes[coluna].fillna(False).astype(bool)

df_resumo_testes["n_testes_significativos_5pct"] = (
    df_resumo_testes["significativo_5pct_teste_sinal"].astype(int)
    + df_resumo_testes["significativo_5pct_wilcoxon"].astype(int)
    + df_resumo_testes["significativo_5pct_permutacao"].astype(int)
)

df_resumo_testes["n_testes_significativos_10pct"] = (
    df_resumo_testes["significativo_10pct_teste_sinal"].astype(int)
    + df_resumo_testes["significativo_10pct_wilcoxon"].astype(int)
    + df_resumo_testes["significativo_10pct_permutacao"].astype(int)
)

df_resumo_testes["classificacao_evidencia_capitulacao_vs_euforia"] = df_resumo_testes.apply(classificar_evidencia, axis=1)

df_resumo_testes["interpretacao_direcional"] = np.select(
    [
        df_resumo_testes["direcao_economica"].eq("direcao_capitulacao"),
        df_resumo_testes["direcao_economica"].eq("direcao_euforia"),
    ],
    [
        "excesso_medio_ou_frequencia_favorece_capitulacao",
        "excesso_medio_ou_frequencia_favorece_euforia",
    ],
    default="resultado_misto_ou_neutro",
)

df_resumo_testes = df_resumo_testes.sort_values(["ordem_frequencia_inferencia", "frequencia"]).reset_index(drop=True)

print(f"Resumo consolidado dos testes            : {len(df_resumo_testes):,} linhas")
print("OK")

# ============================================================
# 11) Construção da base de gráficos de p-valores
# ============================================================

print("\n[11/13] Construção da base de gráficos de p-valores...")

registros_grafico = []

for _, row in df_resumo_testes.iterrows():
    base_registro = {
        "estrategia_referencia": row["estrategia_referencia"],
        "frequencia": row["frequencia"],
        "visao_inferencia": row["visao_inferencia"],
        "ordem_frequencia_inferencia": row["ordem_frequencia_inferencia"],
        "grupo_comparacao": row["grupo_comparacao"],
        "tipo_comparacao": row["tipo_comparacao"],
        "comparacao_id": row["comparacao_id"],
        "media_excesso_periodico": row["media_excesso_periodico"],
        "pct_periodos_capitulacao_superou_euforia": row["pct_periodos_capitulacao_superou_euforia"],
        "direcao_economica": row["direcao_economica"],
        "classificacao_evidencia_capitulacao_vs_euforia": row["classificacao_evidencia_capitulacao_vs_euforia"],
    }

    registros_grafico.append({
        **base_registro,
        "teste_estatistico": "teste_sinal_binomial",
        "p_valor_original": row.get("p_valor_teste_sinal", np.nan),
        "p_valor_holm_14_5": row.get("p_valor_holm_teste_sinal", np.nan),
        "significativo_5pct": row.get("significativo_5pct_teste_sinal", False),
        "significativo_10pct": row.get("significativo_10pct_teste_sinal", False),
    })

    registros_grafico.append({
        **base_registro,
        "teste_estatistico": "wilcoxon_pareado",
        "p_valor_original": row.get("p_valor_wilcoxon", np.nan),
        "p_valor_holm_14_5": row.get("p_valor_holm_wilcoxon", np.nan),
        "significativo_5pct": row.get("significativo_5pct_wilcoxon", False),
        "significativo_10pct": row.get("significativo_10pct_wilcoxon", False),
    })

    registros_grafico.append({
        **base_registro,
        "teste_estatistico": "permutacao_pareada_media",
        "p_valor_original": row.get("p_valor_permutacao", np.nan),
        "p_valor_holm_14_5": row.get("p_valor_holm_permutacao", np.nan),
        "significativo_5pct": row.get("significativo_5pct_permutacao", False),
        "significativo_10pct": row.get("significativo_10pct_permutacao", False),
    })

df_base_grafico = pd.DataFrame(registros_grafico)

if len(df_base_grafico) > 0:
    df_base_grafico = df_base_grafico.sort_values(
        ["ordem_frequencia_inferencia", "frequencia", "teste_estatistico"]
    ).reset_index(drop=True)

print(f"Base de gráficos de p-valores            : {len(df_base_grafico):,} linhas")
print("OK")

# ============================================================
# 12) Parâmetros e auditoria de validação
# ============================================================

print("\n[12/13] Parâmetros e auditoria de validação...")

PARAMETROS_14_5 = {
    "etapa": "14",
    "subetapa": "14.5",
    "nome_subetapa": "Testes entre Capitulação e Euforia",
    "comparacao_principal": "Capitulação contra Euforia",
    "diferenca_principal": "retorno_excesso_capitulacao_vs_euforia",
    "interpretacao_diferenca_positiva": "Capitulação superou Euforia no período",
    "interpretacao_diferenca_negativa": "Euforia superou Capitulação no período",
    "frequencia_principal_inferencia": "mensal",
    "frequencia_complementar_inferencia": "diaria",
    "frequencia_exploratoria": "anual",
    "teste_sinal": "binomial bilateral com hipótese nula de 50% de vitória para cada estratégia",
    "teste_wilcoxon": "Wilcoxon pareado bilateral sobre os excessos de retorno",
    "teste_permutacao": "permutação pareada bilateral por inversão de sinal sobre a média dos excessos",
    "n_permutacoes": N_PERMUTACOES,
    "semente_permutacao": SEMENTE_PERMUTACAO,
    "min_observacoes_testes": MIN_OBSERVACOES_TESTES,
    "alpha_referencia": ALPHA_REFERENCIA,
    "alpha_indicativo": ALPHA_INDICATIVO,
    "ajuste_multiplas_comparacoes": "Holm dentro da subetapa 14.5 por teste estatístico",
    "observacao_metodologica": "Esta subetapa compara diretamente as duas estratégias reais e não trata uma delas como benchmark externo.",
}

df_parametros = pd.DataFrame(
    [{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_14_5.items()]
)

frequencias_identificadas = int(df_base_testes["frequencia"].nunique()) if len(df_base_testes) > 0 else 0
comparacoes_identificadas = int(df_base_testes["comparacao_id"].nunique()) if len(df_base_testes) > 0 else 0
comparacoes_mensais = int((df_resumo_testes["frequencia"].astype(str) == "mensal").sum()) if len(df_resumo_testes) > 0 else 0
comparacoes_diarias = int((df_resumo_testes["frequencia"].astype(str) == "diaria").sum()) if len(df_resumo_testes) > 0 else 0
comparacoes_anuais = int((df_resumo_testes["frequencia"].astype(str) == "anual").sum()) if len(df_resumo_testes) > 0 else 0
valores_excesso_ausentes = int(df_base_testes["retorno_excesso_capitulacao_vs_euforia"].isna().sum()) if len(df_base_testes) > 0 else 0
duplicatas_comparacao_periodo = int(df_base_testes.duplicated(["comparacao_id", "frequencia", "periodo_comparacao"]).sum()) if len(df_base_testes) > 0 else 0

metricas_infinitas = 0
for df_verificacao in [df_base_testes, df_teste_sinal, df_teste_wilcoxon, df_teste_permutacao, df_resumo_testes, df_base_grafico]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

linhas_auditoria = [
    {
        "item": "linhas_base_testes_capitulacao_euforia",
        "valor": len(df_base_testes),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_testes) > 0 else "ERRO",
        "observacao": "A subetapa deve carregar a comparação direta preparada na 14.1 a partir da 11.7.",
    },
    {
        "item": "comparacoes_diretas_identificadas",
        "valor": comparacoes_identificadas,
        "valor_referencia": ">= 3",
        "status": "OK" if comparacoes_identificadas >= 3 else "ERRO",
        "observacao": "Espera-se uma comparação direta por frequência: diária, mensal e anual.",
    },
    {
        "item": "frequencias_identificadas",
        "valor": frequencias_identificadas,
        "valor_referencia": ">= 3",
        "status": "OK" if frequencias_identificadas >= 3 else "ERRO",
        "observacao": "A subetapa deve preservar as visões diária, mensal e anual.",
    },
    {
        "item": "comparacoes_mensais_principais",
        "valor": comparacoes_mensais,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_mensais > 0 else "ERRO",
        "observacao": "A visão mensal deve existir como referência principal dos testes formais.",
    },
    {
        "item": "comparacoes_diarias_complementares",
        "valor": comparacoes_diarias,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_diarias > 0 else "ERRO",
        "observacao": "A visão diária deve ser preservada como análise complementar.",
    },
    {
        "item": "comparacoes_anuais_exploratorias",
        "valor": comparacoes_anuais,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_anuais > 0 else "ERRO",
        "observacao": "A visão anual deve ser preservada como análise exploratória.",
    },
    {
        "item": "testes_sinal_calculados",
        "valor": len(df_teste_sinal),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_sinal) > 0 else "ERRO",
        "observacao": "O teste binomial de sinal deve ser calculado para as frequências aplicáveis.",
    },
    {
        "item": "testes_wilcoxon_calculados",
        "valor": len(df_teste_wilcoxon),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_wilcoxon) > 0 else "ERRO",
        "observacao": "O teste de Wilcoxon pareado deve ser calculado para as frequências aplicáveis.",
    },
    {
        "item": "testes_permutacao_calculados",
        "valor": len(df_teste_permutacao),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_teste_permutacao) > 0 else "ERRO",
        "observacao": "O teste de permutação pareado deve ser calculado para as frequências aplicáveis.",
    },
    {
        "item": "duplicatas_comparacao_frequencia_periodo",
        "valor": duplicatas_comparacao_periodo,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_comparacao_periodo == 0 else "ERRO",
        "observacao": "Cada frequência deve ter no máximo uma linha por período.",
    },
    {
        "item": "valores_excesso_ausentes",
        "valor": valores_excesso_ausentes,
        "valor_referencia": "0",
        "status": "OK" if valores_excesso_ausentes == 0 else "ERRO",
        "observacao": "A base de testes não deve conter excesso de retorno ausente.",
    },
    {
        "item": "base_grafico_linhas",
        "valor": len(df_base_grafico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Parâmetros metodológicos                 : {len(df_parametros):,} linhas")
print(f"Itens de auditoria                       : {len(df_auditoria):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 13) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[13/13] Salvamento dos outputs e validação final da subetapa...")

salvar_parquet_seguro(df_base_testes, CAMINHO_BASE_TESTES, index=False)
salvar_parquet_seguro(df_parametros, CAMINHO_PARAMETROS, index=False)
salvar_parquet_seguro(df_teste_sinal, CAMINHO_TESTE_SINAL, index=False)
salvar_parquet_seguro(df_teste_wilcoxon, CAMINHO_TESTE_WILCOXON, index=False)
salvar_parquet_seguro(df_teste_permutacao, CAMINHO_TESTE_PERMUTACAO, index=False)
salvar_parquet_seguro(df_resumo_testes, CAMINHO_RESUMO_TESTES, index=False)
salvar_parquet_seguro(df_base_grafico, CAMINHO_BASE_GRAFICO, index=False)
salvar_parquet_seguro(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação dos testes entre Capitulação e Euforia:")
print(df_auditoria.to_string(index=False))

print("\nResumo dos testes entre Capitulação e Euforia:")
colunas_print_resumo = [
    coluna for coluna in [
        "frequencia",
        "visao_inferencia",
        "n_periodos",
        "media_excesso_periodico",
        "media_excesso_anualizada_aproximada",
        "mediana_excesso_periodico",
        "pct_periodos_capitulacao_superou_euforia",
        "pct_periodos_euforia_superou_capitulacao",
        "p_valor_holm_teste_sinal",
        "p_valor_holm_wilcoxon",
        "p_valor_holm_permutacao",
        "n_testes_significativos_5pct",
        "direcao_economica",
        "classificacao_evidencia_capitulacao_vs_euforia",
    ] if coluna in df_resumo_testes.columns
]
print(df_resumo_testes[colunas_print_resumo].to_string(index=False))

print("\nBase gráfica de p-valores - amostra:")
if len(df_base_grafico) > 0:
    colunas_grafico_print = [
        coluna for coluna in [
            "frequencia",
            "visao_inferencia",
            "teste_estatistico",
            "p_valor_holm_14_5",
            "media_excesso_periodico",
            "pct_periodos_capitulacao_superou_euforia",
            "direcao_economica",
            "classificacao_evidencia_capitulacao_vs_euforia",
        ] if coluna in df_base_grafico.columns
    ]
    print(df_base_grafico[colunas_grafico_print].to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nArquivos salvos na subetapa 14.5:")
print(f"- {CAMINHO_BASE_TESTES}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_TESTE_SINAL}")
print(f"- {CAMINHO_TESTE_WILCOXON}")
print(f"- {CAMINHO_TESTE_PERMUTACAO}")
print(f"- {CAMINHO_RESUMO_TESTES}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 14.5 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 14.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 14.5 - TESTES ENTRE CAPITULAÇÃO E EUFORIA

[1/13] Validação inicial do ambiente...
OK

[2/13] Definição determinística dos caminhos de entrada e saída...
Entrada - base direta da 14.1           : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_base_inferencia_comparacao_direta_pareada.parquet
Entrada - base consolidada da 14.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_base_inferencia_comparacoes_pareadas.parquet
Entrada - mapa comparações da 14.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_mapa_comparacoes_inferencia.parquet
Entrada - parâmetros da 14.1            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultad

## Etapa 14.6) Bootstrap dos Excessos de Retorno

In [76]:
%%time
# ============================================================
# Etapa 14.6) Bootstrap dos Excessos de Retorno
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 14.6 - BOOTSTRAP DOS EXCESSOS DE RETORNO")
print("=" * 100)

# Esta subetapa estima intervalos de confiança por bootstrap para os excessos de retorno
# preparados na Etapa 14.1. A visão mensal permanece como referência principal de inferência,
# a visão diária é tratada como complementar com bootstrap por blocos e a visão anual é mantida
# como análise exploratória.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
DIR_ETAPA_14.mkdir(parents=True, exist_ok=True)

CAMINHO_BASE_INFERENCIA_14_1 = DIR_ETAPA_14 / "14_1_base_inferencia_comparacoes_pareadas.parquet"
CAMINHO_MAPA_COMPARACOES_14_1 = DIR_ETAPA_14 / "14_1_tbl_mapa_comparacoes_inferencia.parquet"
CAMINHO_PARAMETROS_14_1 = DIR_ETAPA_14 / "14_1_tbl_parametros_inferencia_estatistica.parquet"

CAMINHO_BASE_EXCESSOS = DIR_ETAPA_14 / "14_6_base_excessos_retorno_bootstrap.parquet"
CAMINHO_BASE_AMOSTRAS = DIR_ETAPA_14 / "14_6_base_amostras_bootstrap_excessos_retorno.parquet"
CAMINHO_INTERVALOS = DIR_ETAPA_14 / "14_6_tbl_intervalos_bootstrap_excessos_retorno.parquet"
CAMINHO_RESUMO = DIR_ETAPA_14 / "14_6_tbl_resumo_bootstrap_excessos_retorno.parquet"
CAMINHO_GRAFICO = DIR_ETAPA_14 / "14_6_tbl_base_grafico_bootstrap_excessos_retorno.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_14 / "14_6_tbl_parametros_bootstrap_excessos_retorno.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_14 / "14_6_tbl_auditoria_validacao_bootstrap_excessos_retorno.parquet"

CAMINHOS_OBRIGATORIOS = [
    CAMINHO_BASE_INFERENCIA_14_1,
    CAMINHO_MAPA_COMPARACOES_14_1,
    CAMINHO_PARAMETROS_14_1,
]

for caminho in CAMINHOS_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - base consolidada da 14.1       : {CAMINHO_BASE_INFERENCIA_14_1}")
print(f"Entrada - mapa comparações da 14.1      : {CAMINHO_MAPA_COMPARACOES_14_1}")
print(f"Entrada - parâmetros da 14.1            : {CAMINHO_PARAMETROS_14_1}")
print(f"Saída   - base de excessos              : {CAMINHO_BASE_EXCESSOS}")
print(f"Saída   - amostras bootstrap            : {CAMINHO_BASE_AMOSTRAS}")
print(f"Saída   - intervalos bootstrap          : {CAMINHO_INTERVALOS}")
print(f"Saída   - resumo bootstrap              : {CAMINHO_RESUMO}")
print(f"Saída   - base gráfico bootstrap        : {CAMINHO_GRAFICO}")
print(f"Saída   - parâmetros da 14.6            : {CAMINHO_PARAMETROS}")
print(f"Saída   - auditoria de validação        : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/12] Carga das bases oficiais da subetapa...")

df_base_inferencia = pd.read_parquet(CAMINHO_BASE_INFERENCIA_14_1)
df_mapa_comparacoes = pd.read_parquet(CAMINHO_MAPA_COMPARACOES_14_1)
df_parametros_14_1 = pd.read_parquet(CAMINHO_PARAMETROS_14_1)

print(f"Base consolidada da 14.1      : {len(df_base_inferencia):,} linhas x {df_base_inferencia.shape[1]:,} colunas")
print(f"Mapa comparações da 14.1      : {len(df_mapa_comparacoes):,} linhas x {df_mapa_comparacoes.shape[1]:,} colunas")
print(f"Parâmetros da 14.1            : {len(df_parametros_14_1):,} linhas x {df_parametros_14_1.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

N_BOOTSTRAP = 1000
SEMENTE_BOOTSTRAP = 20260426
BLOCO_BOOTSTRAP_DIARIO = 21
ALPHA_REFERENCIA = 0.05
ALPHA_INDICATIVO = 0.10
MIN_OBSERVACOES_BOOTSTRAP = 12

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}

MAPA_ANUALIZACAO = {
    "diaria": 252,
    "mensal": 12,
    "anual": 1,
}


def validar_colunas_obrigatorias(df, colunas, nome_base):
    ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(ausentes) > 0:
        raise KeyError(
            f"Colunas obrigatórias ausentes na base {nome_base}: {ausentes}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def converter_booleano(serie):
    if pd.api.types.is_bool_dtype(serie):
        return serie.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce").fillna(0.0).ne(0.0)
    texto = serie.astype(str).str.strip().str.lower()
    return texto.isin(["true", "1", "sim", "s", "yes", "y", "ok", "verdadeiro"])


def obter_coluna_opcional(df, coluna, valor_padrao=np.nan):
    if coluna in df.columns:
        return df[coluna]
    return pd.Series(valor_padrao, index=df.index)


def preparar_dataframe_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def salvar_dataframe_compat(df, caminho, index=False):
    salvar_dataframe(preparar_dataframe_para_parquet(df), caminho, index=index)


def gerar_indices_bootstrap(n_observacoes, frequencia, n_bootstrap, tamanho_bloco, rng):
    if str(frequencia) == "diaria":
        n_blocos = int(np.ceil(n_observacoes / tamanho_bloco))
        inicios = rng.integers(0, n_observacoes, size=(n_bootstrap, n_blocos))
        deslocamentos = np.arange(tamanho_bloco)
        indices = (inicios[:, :, None] + deslocamentos[None, None, :]) % n_observacoes
        indices = indices.reshape(n_bootstrap, n_blocos * tamanho_bloco)[:, :n_observacoes]
        return indices, "bootstrap_blocos", tamanho_bloco

    indices = rng.integers(0, n_observacoes, size=(n_bootstrap, n_observacoes))
    return indices, "bootstrap_simples", 1


def classificar_intervalo(media_observada, ic95_inf, ic95_sup, ic90_inf, ic90_sup):
    if pd.notna(ic95_inf) and pd.notna(ic95_sup):
        if ic95_inf > 0.0:
            return "favoravel_bootstrap_ic95"
        if ic95_sup < 0.0:
            return "desfavoravel_bootstrap_ic95"

    if pd.notna(ic90_inf) and pd.notna(ic90_sup):
        if ic90_inf > 0.0:
            return "favoravel_indicativo_bootstrap_ic90"
        if ic90_sup < 0.0:
            return "desfavoravel_indicativo_bootstrap_ic90"

    if pd.notna(media_observada):
        if media_observada > 0.0:
            return "favoravel_economica_sem_confirmacao_bootstrap"
        if media_observada < 0.0:
            return "desfavoravel_economica_sem_confirmacao_bootstrap"

    return "neutra_ou_indeterminada_bootstrap"


def calcular_intervalos_bootstrap(df_grupo, indice_comparacao):
    df_grupo = df_grupo.sort_values(["data_fim_periodo", "periodo_comparacao"]).reset_index(drop=True)

    excesso = converter_numero(df_grupo["retorno_excesso_real_vs_comparador"]).dropna().to_numpy(dtype=float)
    flags = converter_booleano(df_grupo.loc[converter_numero(df_grupo["retorno_excesso_real_vs_comparador"]).notna(), "flag_real_superou_comparador"]).to_numpy(dtype=float)

    n_observacoes = int(len(excesso))
    linha = df_grupo.iloc[0].to_dict()
    frequencia = str(linha.get("frequencia", ""))
    fator_anualizacao = MAPA_ANUALIZACAO.get(frequencia, np.nan)

    resultado_base = {
        "comparacao_id": linha.get("comparacao_id", np.nan),
        "origem_comparacao": linha.get("origem_comparacao", np.nan),
        "grupo_comparacao": linha.get("grupo_comparacao", np.nan),
        "tipo_comparacao": linha.get("tipo_comparacao", np.nan),
        "subtipo_comparacao": linha.get("subtipo_comparacao", np.nan),
        "estrategia_referencia": linha.get("estrategia_referencia", np.nan),
        "frequencia": frequencia,
        "visao_inferencia": linha.get("visao_inferencia", np.nan),
        "descricao_visao_inferencia": linha.get("descricao_visao_inferencia", np.nan),
        "ordem_frequencia_inferencia": MAPA_ORDEM_FREQUENCIA.get(frequencia, 99),
        "chave_estrategia_real": linha.get("chave_estrategia_real", np.nan),
        "nome_exibicao_estrategia_real": linha.get("nome_exibicao_estrategia_real", np.nan),
        "chave_comparador": linha.get("chave_comparador", np.nan),
        "nome_exibicao_comparador": linha.get("nome_exibicao_comparador", np.nan),
        "n_observacoes": n_observacoes,
        "fator_anualizacao_aproximado": fator_anualizacao,
    }

    if n_observacoes < MIN_OBSERVACOES_BOOTSTRAP:
        resultado = resultado_base.copy()
        resultado.update({
            "metodo_bootstrap": "nao_calculado",
            "tamanho_bloco_bootstrap": np.nan,
            "n_bootstrap": 0,
            "status_bootstrap": "observacoes_insuficientes",
            "media_excesso_observada": np.nanmean(excesso) if n_observacoes > 0 else np.nan,
            "mediana_excesso_observada": np.nanmedian(excesso) if n_observacoes > 0 else np.nan,
            "desvio_excesso_observado": np.nanstd(excesso, ddof=1) if n_observacoes > 1 else np.nan,
            "pct_periodos_real_superou_comparador_observado": np.nanmean(flags) if len(flags) > 0 else np.nan,
            "media_excesso_anualizada_aproximada": np.nan,
            "ic95_media_inf": np.nan,
            "ic95_media_sup": np.nan,
            "ic90_media_inf": np.nan,
            "ic90_media_sup": np.nan,
            "ic95_mediana_inf": np.nan,
            "ic95_mediana_sup": np.nan,
            "ic90_mediana_inf": np.nan,
            "ic90_mediana_sup": np.nan,
            "ic95_pct_superacao_inf": np.nan,
            "ic95_pct_superacao_sup": np.nan,
            "ic90_pct_superacao_inf": np.nan,
            "ic90_pct_superacao_sup": np.nan,
            "prob_bootstrap_media_acima_zero": np.nan,
            "prob_bootstrap_mediana_acima_zero": np.nan,
            "prob_bootstrap_pct_superacao_acima_50pct": np.nan,
            "classificacao_bootstrap_media": "observacoes_insuficientes",
        })
        return resultado, pd.DataFrame()

    rng = np.random.default_rng(SEMENTE_BOOTSTRAP + int(indice_comparacao))
    indices, metodo_bootstrap, tamanho_bloco = gerar_indices_bootstrap(
        n_observacoes=n_observacoes,
        frequencia=frequencia,
        n_bootstrap=N_BOOTSTRAP,
        tamanho_bloco=BLOCO_BOOTSTRAP_DIARIO,
        rng=rng,
    )

    amostras_excesso = excesso[indices]
    amostras_flags = flags[indices]

    bootstrap_media = np.nanmean(amostras_excesso, axis=1)
    bootstrap_mediana = np.nanmedian(amostras_excesso, axis=1)
    bootstrap_pct_superacao = np.nanmean(amostras_flags, axis=1)

    media_observada = float(np.nanmean(excesso))
    mediana_observada = float(np.nanmedian(excesso))
    desvio_observado = float(np.nanstd(excesso, ddof=1)) if n_observacoes > 1 else np.nan
    pct_superacao_observado = float(np.nanmean(flags))
    media_anualizada = media_observada * fator_anualizacao if pd.notna(fator_anualizacao) else np.nan

    ic95_media_inf, ic95_media_sup = np.nanpercentile(bootstrap_media, [2.5, 97.5])
    ic90_media_inf, ic90_media_sup = np.nanpercentile(bootstrap_media, [5.0, 95.0])
    ic95_mediana_inf, ic95_mediana_sup = np.nanpercentile(bootstrap_mediana, [2.5, 97.5])
    ic90_mediana_inf, ic90_mediana_sup = np.nanpercentile(bootstrap_mediana, [5.0, 95.0])
    ic95_pct_inf, ic95_pct_sup = np.nanpercentile(bootstrap_pct_superacao, [2.5, 97.5])
    ic90_pct_inf, ic90_pct_sup = np.nanpercentile(bootstrap_pct_superacao, [5.0, 95.0])

    resultado = resultado_base.copy()
    resultado.update({
        "metodo_bootstrap": metodo_bootstrap,
        "tamanho_bloco_bootstrap": tamanho_bloco,
        "n_bootstrap": N_BOOTSTRAP,
        "status_bootstrap": "calculado",
        "media_excesso_observada": media_observada,
        "mediana_excesso_observada": mediana_observada,
        "desvio_excesso_observado": desvio_observado,
        "pct_periodos_real_superou_comparador_observado": pct_superacao_observado,
        "media_excesso_anualizada_aproximada": media_anualizada,
        "ic95_media_inf": float(ic95_media_inf),
        "ic95_media_sup": float(ic95_media_sup),
        "ic90_media_inf": float(ic90_media_inf),
        "ic90_media_sup": float(ic90_media_sup),
        "ic95_mediana_inf": float(ic95_mediana_inf),
        "ic95_mediana_sup": float(ic95_mediana_sup),
        "ic90_mediana_inf": float(ic90_mediana_inf),
        "ic90_mediana_sup": float(ic90_mediana_sup),
        "ic95_pct_superacao_inf": float(ic95_pct_inf),
        "ic95_pct_superacao_sup": float(ic95_pct_sup),
        "ic90_pct_superacao_inf": float(ic90_pct_inf),
        "ic90_pct_superacao_sup": float(ic90_pct_sup),
        "prob_bootstrap_media_acima_zero": float(np.nanmean(bootstrap_media > 0.0)),
        "prob_bootstrap_mediana_acima_zero": float(np.nanmean(bootstrap_mediana > 0.0)),
        "prob_bootstrap_pct_superacao_acima_50pct": float(np.nanmean(bootstrap_pct_superacao > 0.5)),
        "classificacao_bootstrap_media": classificar_intervalo(
            media_observada,
            float(ic95_media_inf),
            float(ic95_media_sup),
            float(ic90_media_inf),
            float(ic90_media_sup),
        ),
    })

    df_amostras = pd.DataFrame({
        "comparacao_id": resultado["comparacao_id"],
        "grupo_comparacao": resultado["grupo_comparacao"],
        "tipo_comparacao": resultado["tipo_comparacao"],
        "estrategia_referencia": resultado["estrategia_referencia"],
        "frequencia": resultado["frequencia"],
        "visao_inferencia": resultado["visao_inferencia"],
        "metodo_bootstrap": metodo_bootstrap,
        "iteracao_bootstrap": np.arange(1, N_BOOTSTRAP + 1),
        "media_excesso_bootstrap": bootstrap_media,
        "mediana_excesso_bootstrap": bootstrap_mediana,
        "pct_superacao_bootstrap": bootstrap_pct_superacao,
    })

    return resultado, df_amostras

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização da base de excessos de retorno
# ============================================================

print("\n[5/12] Padronização da base de excessos de retorno...")

colunas_obrigatorias = [
    "comparacao_id",
    "grupo_comparacao",
    "tipo_comparacao",
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "periodo_comparacao",
    "retorno_excesso_real_vs_comparador",
    "flag_real_superou_comparador",
]

validar_colunas_obrigatorias(df_base_inferencia, colunas_obrigatorias, "14.1 base consolidada de inferência")

df_base_excessos = df_base_inferencia.copy()
df_base_excessos["retorno_excesso_real_vs_comparador"] = converter_numero(df_base_excessos["retorno_excesso_real_vs_comparador"])
df_base_excessos["flag_real_superou_comparador"] = converter_booleano(df_base_excessos["flag_real_superou_comparador"])
df_base_excessos["ordem_frequencia_inferencia"] = df_base_excessos["frequencia"].astype(str).map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)

if "data_fim_periodo" in df_base_excessos.columns:
    df_base_excessos["data_fim_periodo"] = pd.to_datetime(df_base_excessos["data_fim_periodo"], errors="coerce")
else:
    df_base_excessos["data_fim_periodo"] = pd.NaT

linhas_antes = len(df_base_excessos)
df_base_excessos = df_base_excessos.dropna(subset=["retorno_excesso_real_vs_comparador"]).copy()
linhas_depois = len(df_base_excessos)
linhas_descartadas = linhas_antes - linhas_depois

df_base_excessos = df_base_excessos.sort_values([
    "ordem_frequencia_inferencia",
    "grupo_comparacao",
    "tipo_comparacao",
    "estrategia_referencia",
    "comparacao_id",
    "data_fim_periodo",
    "periodo_comparacao",
]).reset_index(drop=True)

print(f"Base de excessos para bootstrap          : {len(df_base_excessos):,} linhas")
print(f"Comparações disponíveis                  : {df_base_excessos['comparacao_id'].nunique():,}")
print(f"Frequências preservadas                  : {df_base_excessos['frequencia'].nunique():,}")
print(f"Linhas descartadas por ausência de excesso: {linhas_descartadas:,}")
print("OK")

# ============================================================
# 6) Execução do bootstrap por comparação
# ============================================================

print("\n[6/12] Execução do bootstrap por comparação...")

resultados_intervalos = []
lista_amostras = []

comparacoes = list(df_base_excessos.groupby("comparacao_id", dropna=False, sort=False))

for indice_comparacao, (comparacao_id, df_grupo) in enumerate(comparacoes, start=1):
    resultado, df_amostras = calcular_intervalos_bootstrap(df_grupo, indice_comparacao)
    resultados_intervalos.append(resultado)

    if len(df_amostras) > 0:
        lista_amostras.append(df_amostras)

    if indice_comparacao % 20 == 0 or indice_comparacao == len(comparacoes):
        print(f"Comparações processadas                  : {indice_comparacao:,} / {len(comparacoes):,}")

df_intervalos_bootstrap = pd.DataFrame(resultados_intervalos)

if len(lista_amostras) > 0:
    df_amostras_bootstrap = pd.concat(lista_amostras, ignore_index=True)
else:
    df_amostras_bootstrap = pd.DataFrame()

if len(df_intervalos_bootstrap) > 0:
    df_intervalos_bootstrap = df_intervalos_bootstrap.sort_values([
        "ordem_frequencia_inferencia",
        "grupo_comparacao",
        "tipo_comparacao",
        "estrategia_referencia",
        "comparacao_id",
    ]).reset_index(drop=True)

print(f"Intervalos bootstrap calculados          : {len(df_intervalos_bootstrap):,} linhas")
print(f"Amostras bootstrap salvas                : {len(df_amostras_bootstrap):,} linhas")
print("OK")

# ============================================================
# 7) Consolidação do resumo por grupo de comparação
# ============================================================

print("\n[7/12] Consolidação do resumo por grupo de comparação...")

if len(df_intervalos_bootstrap) > 0:
    df_resumo_bootstrap = (
        df_intervalos_bootstrap
        .groupby(["grupo_comparacao", "tipo_comparacao", "estrategia_referencia", "frequencia", "visao_inferencia"], dropna=False)
        .agg(
            n_comparacoes=("comparacao_id", "nunique"),
            n_comparacoes_bootstrap_calculado=("status_bootstrap", lambda x: int((x.astype(str) == "calculado").sum())),
            n_observacoes_total=("n_observacoes", "sum"),
            media_excesso_observada_media=("media_excesso_observada", "mean"),
            mediana_excesso_observada_media=("mediana_excesso_observada", "mean"),
            pct_superacao_observado_media=("pct_periodos_real_superou_comparador_observado", "mean"),
            pct_comparacoes_ic95_media_acima_zero=("ic95_media_inf", lambda x: float((pd.to_numeric(x, errors="coerce") > 0.0).mean())),
            pct_comparacoes_ic95_media_abaixo_zero=("ic95_media_sup", lambda x: float((pd.to_numeric(x, errors="coerce") < 0.0).mean())),
            pct_comparacoes_ic90_media_acima_zero=("ic90_media_inf", lambda x: float((pd.to_numeric(x, errors="coerce") > 0.0).mean())),
            pct_comparacoes_ic90_media_abaixo_zero=("ic90_media_sup", lambda x: float((pd.to_numeric(x, errors="coerce") < 0.0).mean())),
            prob_bootstrap_media_acima_zero_media=("prob_bootstrap_media_acima_zero", "mean"),
            prob_bootstrap_pct_superacao_acima_50pct_media=("prob_bootstrap_pct_superacao_acima_50pct", "mean"),
        )
        .reset_index()
    )
    df_resumo_bootstrap["ordem_frequencia_inferencia"] = df_resumo_bootstrap["frequencia"].astype(str).map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
    df_resumo_bootstrap = df_resumo_bootstrap.sort_values([
        "ordem_frequencia_inferencia",
        "grupo_comparacao",
        "tipo_comparacao",
        "estrategia_referencia",
    ]).reset_index(drop=True)
else:
    df_resumo_bootstrap = pd.DataFrame()

print(f"Resumo bootstrap por grupo               : {len(df_resumo_bootstrap):,} linhas")
print("OK")

# ============================================================
# 8) Construção da base de gráficos dos intervalos
# ============================================================

print("\n[8/12] Construção da base de gráficos dos intervalos...")

registros_grafico = []

for _, linha in df_intervalos_bootstrap.iterrows():
    metadados = {
        "comparacao_id": linha.get("comparacao_id", np.nan),
        "grupo_comparacao": linha.get("grupo_comparacao", np.nan),
        "tipo_comparacao": linha.get("tipo_comparacao", np.nan),
        "subtipo_comparacao": linha.get("subtipo_comparacao", np.nan),
        "estrategia_referencia": linha.get("estrategia_referencia", np.nan),
        "frequencia": linha.get("frequencia", np.nan),
        "visao_inferencia": linha.get("visao_inferencia", np.nan),
        "ordem_frequencia_inferencia": linha.get("ordem_frequencia_inferencia", np.nan),
        "nome_exibicao_estrategia_real": linha.get("nome_exibicao_estrategia_real", np.nan),
        "nome_exibicao_comparador": linha.get("nome_exibicao_comparador", np.nan),
        "status_bootstrap": linha.get("status_bootstrap", np.nan),
        "classificacao_bootstrap_media": linha.get("classificacao_bootstrap_media", np.nan),
    }

    registros_grafico.append({
        **metadados,
        "metrica_bootstrap": "media_excesso",
        "valor_observado": linha.get("media_excesso_observada", np.nan),
        "ic95_inf": linha.get("ic95_media_inf", np.nan),
        "ic95_sup": linha.get("ic95_media_sup", np.nan),
        "ic90_inf": linha.get("ic90_media_inf", np.nan),
        "ic90_sup": linha.get("ic90_media_sup", np.nan),
    })

    registros_grafico.append({
        **metadados,
        "metrica_bootstrap": "mediana_excesso",
        "valor_observado": linha.get("mediana_excesso_observada", np.nan),
        "ic95_inf": linha.get("ic95_mediana_inf", np.nan),
        "ic95_sup": linha.get("ic95_mediana_sup", np.nan),
        "ic90_inf": linha.get("ic90_mediana_inf", np.nan),
        "ic90_sup": linha.get("ic90_mediana_sup", np.nan),
    })

    registros_grafico.append({
        **metadados,
        "metrica_bootstrap": "pct_superacao",
        "valor_observado": linha.get("pct_periodos_real_superou_comparador_observado", np.nan),
        "ic95_inf": linha.get("ic95_pct_superacao_inf", np.nan),
        "ic95_sup": linha.get("ic95_pct_superacao_sup", np.nan),
        "ic90_inf": linha.get("ic90_pct_superacao_inf", np.nan),
        "ic90_sup": linha.get("ic90_pct_superacao_sup", np.nan),
    })

df_base_grafico = pd.DataFrame(registros_grafico)

if len(df_base_grafico) > 0:
    df_base_grafico = df_base_grafico.sort_values([
        "ordem_frequencia_inferencia",
        "grupo_comparacao",
        "tipo_comparacao",
        "estrategia_referencia",
        "comparacao_id",
        "metrica_bootstrap",
    ]).reset_index(drop=True)

print(f"Base gráfica de intervalos bootstrap     : {len(df_base_grafico):,} linhas")
print("OK")

# ============================================================
# 9) Parâmetros metodológicos da subetapa
# ============================================================

print("\n[9/12] Parâmetros metodológicos da subetapa...")

PARAMETROS_BOOTSTRAP_14_6 = {
    "etapa": "14",
    "subetapa": "14.6",
    "nome_subetapa": "Bootstrap dos Excessos de Retorno",
    "n_bootstrap": N_BOOTSTRAP,
    "semente_bootstrap": SEMENTE_BOOTSTRAP,
    "min_observacoes_bootstrap": MIN_OBSERVACOES_BOOTSTRAP,
    "bootstrap_visao_mensal": "bootstrap_simples",
    "bootstrap_visao_diaria": "bootstrap_blocos",
    "tamanho_bloco_diario": BLOCO_BOOTSTRAP_DIARIO,
    "bootstrap_visao_anual": "bootstrap_simples_exploratorio",
    "frequencia_principal_inferencia": "mensal",
    "frequencia_complementar_inferencia": "diaria",
    "frequencia_exploratoria": "anual",
    "metrica_principal": "media do excesso de retorno período a período",
    "metrica_secundaria": "mediana do excesso de retorno período a período",
    "metrica_complementar": "frequência de superação período a período",
    "alpha_referencia": ALPHA_REFERENCIA,
    "alpha_indicativo": ALPHA_INDICATIVO,
    "observacao_metodologica": "Intervalos bootstrap estimam incerteza dos excessos de retorno sem substituir os testes formais das subetapas anteriores.",
}

df_parametros_bootstrap = pd.DataFrame(
    [{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_BOOTSTRAP_14_6.items()]
)

print(f"Parâmetros metodológicos                 : {len(df_parametros_bootstrap):,} linhas")
print("OK")

# ============================================================
# 10) Auditoria de validação
# ============================================================

print("\n[10/12] Auditoria de validação...")

comparacoes_identificadas = int(df_base_excessos["comparacao_id"].nunique()) if len(df_base_excessos) > 0 else 0
frequencias_identificadas = int(df_base_excessos["frequencia"].nunique()) if len(df_base_excessos) > 0 else 0
comparacoes_mensais = int((df_intervalos_bootstrap["frequencia"].astype(str) == "mensal").sum()) if len(df_intervalos_bootstrap) > 0 else 0
comparacoes_diarias = int((df_intervalos_bootstrap["frequencia"].astype(str) == "diaria").sum()) if len(df_intervalos_bootstrap) > 0 else 0
comparacoes_anuais = int((df_intervalos_bootstrap["frequencia"].astype(str) == "anual").sum()) if len(df_intervalos_bootstrap) > 0 else 0
comparacoes_calculadas = int((df_intervalos_bootstrap["status_bootstrap"].astype(str) == "calculado").sum()) if len(df_intervalos_bootstrap) > 0 else 0
comparacoes_nao_calculadas = int((df_intervalos_bootstrap["status_bootstrap"].astype(str) != "calculado").sum()) if len(df_intervalos_bootstrap) > 0 else 0
comparacoes_diretas = int((df_intervalos_bootstrap["grupo_comparacao"].astype(str) == "comparacao_direta").sum()) if len(df_intervalos_bootstrap) > 0 else 0
linhas_amostras_esperadas_minimas = comparacoes_calculadas * N_BOOTSTRAP
valores_excesso_ausentes = int(df_base_excessos["retorno_excesso_real_vs_comparador"].isna().sum()) if len(df_base_excessos) > 0 else 0
duplicatas_base = int(df_base_excessos.duplicated(["comparacao_id", "frequencia", "periodo_comparacao"]).sum()) if len(df_base_excessos) > 0 else 0

metricas_infinitas = 0
for df_verificacao in [df_base_excessos, df_amostras_bootstrap, df_intervalos_bootstrap, df_resumo_bootstrap, df_base_grafico]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

linhas_auditoria = [
    {
        "item": "linhas_base_excessos",
        "valor": len(df_base_excessos),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_excessos) > 0 else "ERRO",
        "observacao": "A subetapa deve carregar a base consolidada de inferência preparada na 14.1.",
    },
    {
        "item": "comparacoes_identificadas",
        "valor": comparacoes_identificadas,
        "valor_referencia": ">= 117",
        "status": "OK" if comparacoes_identificadas >= 117 else "ERRO",
        "observacao": "A base deve preservar benchmarks, controles aleatórios, controles mensais e comparação direta.",
    },
    {
        "item": "frequencias_identificadas",
        "valor": frequencias_identificadas,
        "valor_referencia": ">= 3",
        "status": "OK" if frequencias_identificadas >= 3 else "ERRO",
        "observacao": "A subetapa deve preservar as visões diária, mensal e anual.",
    },
    {
        "item": "comparacoes_mensais_principais",
        "valor": comparacoes_mensais,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_mensais > 0 else "ERRO",
        "observacao": "A visão mensal deve existir como referência principal para inferência.",
    },
    {
        "item": "comparacoes_diarias_complementares",
        "valor": comparacoes_diarias,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_diarias > 0 else "ERRO",
        "observacao": "A visão diária deve ser preservada com bootstrap por blocos.",
    },
    {
        "item": "comparacoes_anuais_exploratorias",
        "valor": comparacoes_anuais,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_anuais > 0 else "ERRO",
        "observacao": "A visão anual deve ser preservada como análise exploratória.",
    },
    {
        "item": "comparacoes_diretas_capitulacao_euforia",
        "valor": comparacoes_diretas,
        "valor_referencia": ">= 3",
        "status": "OK" if comparacoes_diretas >= 3 else "ERRO",
        "observacao": "O bootstrap deve incluir a comparação direta entre Capitulação e Euforia.",
    },
    {
        "item": "comparacoes_bootstrap_calculadas",
        "valor": comparacoes_calculadas,
        "valor_referencia": "> 0",
        "status": "OK" if comparacoes_calculadas > 0 else "ERRO",
        "observacao": "Ao menos uma comparação deve ter bootstrap calculado.",
    },
    {
        "item": "comparacoes_bootstrap_nao_calculadas",
        "valor": comparacoes_nao_calculadas,
        "valor_referencia": "0",
        "status": "OK" if comparacoes_nao_calculadas == 0 else "ALERTA",
        "observacao": "Comparações com poucas observações podem ser mantidas apenas de forma descritiva.",
    },
    {
        "item": "linhas_amostras_bootstrap",
        "valor": len(df_amostras_bootstrap),
        "valor_referencia": f">= {linhas_amostras_esperadas_minimas}",
        "status": "OK" if len(df_amostras_bootstrap) >= linhas_amostras_esperadas_minimas else "ERRO",
        "observacao": "A base de amostras deve ter n_bootstrap linhas por comparação calculada.",
    },
    {
        "item": "linhas_intervalos_bootstrap",
        "valor": len(df_intervalos_bootstrap),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_intervalos_bootstrap) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar intervalos de confiança por comparação.",
    },
    {
        "item": "linhas_resumo_bootstrap",
        "valor": len(df_resumo_bootstrap),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_resumo_bootstrap) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar resumo agregado por grupo de comparação.",
    },
    {
        "item": "linhas_base_grafico",
        "valor": len(df_base_grafico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "duplicatas_comparacao_frequencia_periodo",
        "valor": duplicatas_base,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_base == 0 else "ERRO",
        "observacao": "Cada comparação deve ter no máximo uma linha por frequência e período.",
    },
    {
        "item": "valores_excesso_ausentes",
        "valor": valores_excesso_ausentes,
        "valor_referencia": "0",
        "status": "OK" if valores_excesso_ausentes == 0 else "ERRO",
        "observacao": "A base de bootstrap não deve conter excesso de retorno ausente.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = pd.DataFrame(linhas_auditoria)
df_auditoria = df_auditoria[["item", "valor", "valor_referencia", "status", "observacao"]].copy()
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Itens de auditoria                       : {len(df_auditoria):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 11) Salvamento dos outputs
# ============================================================

print("\n[11/12] Salvamento dos outputs...")

salvar_dataframe_compat(df_base_excessos, CAMINHO_BASE_EXCESSOS, index=False)
salvar_dataframe_compat(df_amostras_bootstrap, CAMINHO_BASE_AMOSTRAS, index=False)
salvar_dataframe_compat(df_intervalos_bootstrap, CAMINHO_INTERVALOS, index=False)
salvar_dataframe_compat(df_resumo_bootstrap, CAMINHO_RESUMO, index=False)
salvar_dataframe_compat(df_base_grafico, CAMINHO_GRAFICO, index=False)
salvar_dataframe_compat(df_parametros_bootstrap, CAMINHO_PARAMETROS, index=False)
salvar_dataframe_compat(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")
print("OK")

# ============================================================
# 12) Validação final da subetapa
# ============================================================

print("\n[12/12] Validação final da subetapa...")

print("\nAuditoria de validação do bootstrap dos excessos de retorno:")
print(df_auditoria.to_string(index=False))

print("\nResumo bootstrap por grupo - amostra:")
if len(df_resumo_bootstrap) > 0:
    colunas_resumo_print = [
        coluna for coluna in [
            "grupo_comparacao",
            "tipo_comparacao",
            "estrategia_referencia",
            "frequencia",
            "visao_inferencia",
            "n_comparacoes",
            "media_excesso_observada_media",
            "pct_superacao_observado_media",
            "pct_comparacoes_ic95_media_acima_zero",
            "pct_comparacoes_ic95_media_abaixo_zero",
            "prob_bootstrap_media_acima_zero_media",
        ] if coluna in df_resumo_bootstrap.columns
    ]
    print(df_resumo_bootstrap[colunas_resumo_print].head(60).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nIntervalos bootstrap - amostra:")
if len(df_intervalos_bootstrap) > 0:
    colunas_intervalos_print = [
        coluna for coluna in [
            "grupo_comparacao",
            "tipo_comparacao",
            "estrategia_referencia",
            "frequencia",
            "nome_exibicao_comparador",
            "n_observacoes",
            "metodo_bootstrap",
            "media_excesso_observada",
            "ic95_media_inf",
            "ic95_media_sup",
            "prob_bootstrap_media_acima_zero",
            "classificacao_bootstrap_media",
        ] if coluna in df_intervalos_bootstrap.columns
    ]
    print(df_intervalos_bootstrap[colunas_intervalos_print].head(80).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nArquivos salvos na subetapa 14.6:")
print(f"- {CAMINHO_BASE_EXCESSOS}")
print(f"- {CAMINHO_BASE_AMOSTRAS}")
print(f"- {CAMINHO_INTERVALOS}")
print(f"- {CAMINHO_RESUMO}")
print(f"- {CAMINHO_GRAFICO}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 14.6 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 14.6 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 14.6 - BOOTSTRAP DOS EXCESSOS DE RETORNO

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - base consolidada da 14.1       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_base_inferencia_comparacoes_pareadas.parquet
Entrada - mapa comparações da 14.1      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_mapa_comparacoes_inferencia.parquet
Entrada - parâmetros da 14.1            : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_parametros_inferencia_estatistica.parquet
Saída   - base de excessos              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\e

## Etapa 14.7) Percentis Empíricos e Posicionamento Estatístico

In [77]:
%%time
# ============================================================
# Etapa 14.7) Percentis Empíricos e Posicionamento Estatístico
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 14.7 - PERCENTIS EMPÍRICOS E POSICIONAMENTO ESTATÍSTICO")
print("=" * 100)

# Esta subetapa consolida percentis empíricos, rankings e posicionamento estatístico das estratégias reais.
# As métricas de eficiência são carregadas da Etapa 11.3, preservando Sharpe com taxa livre de risco e Calmar oficial.
# A visão mensal permanece como principal para inferência, a visão diária como complementar e a visão anual como exploratória.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])
DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
DIR_ETAPA_14.mkdir(parents=True, exist_ok=True)

CAMINHO_METRICAS_EFICIENCIA_11_3 = DIR_ETAPA_11 / "11_3_tbl_metricas_eficiencia_multifrequencia.parquet"
CAMINHO_COMPARACAO_DIRETA_11_7 = DIR_ETAPA_11 / "11_7_tbl_resumo_comparacao_capitulacao_euforia.parquet"

CAMINHO_BASE_INFERENCIA_14_1 = DIR_ETAPA_14 / "14_1_base_inferencia_comparacoes_pareadas.parquet"
CAMINHO_MAPA_COMPARACOES_14_1 = DIR_ETAPA_14 / "14_1_tbl_mapa_comparacoes_inferencia.parquet"
CAMINHO_PARAMETROS_14_1 = DIR_ETAPA_14 / "14_1_tbl_parametros_inferencia_estatistica.parquet"

CAMINHO_RESUMO_BENCHMARKS_14_2 = DIR_ETAPA_14 / "14_2_tbl_resumo_testes_benchmarks.parquet"
CAMINHO_RESUMO_ALEATORIOS_PARES_14_3 = DIR_ETAPA_14 / "14_3_tbl_resumo_testes_aleatorios_pares.parquet"
CAMINHO_RESUMO_ALEATORIOS_GRUPOS_14_3 = DIR_ETAPA_14 / "14_3_tbl_resumo_testes_aleatorios_grupos.parquet"
CAMINHO_RESUMO_MENSAIS_PARES_14_4 = DIR_ETAPA_14 / "14_4_tbl_resumo_testes_mensais_pares.parquet"
CAMINHO_RESUMO_MENSAIS_GRUPOS_14_4 = DIR_ETAPA_14 / "14_4_tbl_resumo_testes_mensais_grupos.parquet"
CAMINHO_RESUMO_DIRETA_14_5 = DIR_ETAPA_14 / "14_5_tbl_resumo_testes_capitulacao_euforia.parquet"
CAMINHO_INTERVALOS_BOOTSTRAP_14_6 = DIR_ETAPA_14 / "14_6_tbl_intervalos_bootstrap_excessos_retorno.parquet"
CAMINHO_RESUMO_BOOTSTRAP_14_6 = DIR_ETAPA_14 / "14_6_tbl_resumo_bootstrap_excessos_retorno.parquet"

CAMINHO_METRICAS_OFICIAIS = DIR_ETAPA_14 / "14_7_tbl_metricas_oficiais_estrategias_inferencia.parquet"
CAMINHO_COMPARACOES_METRICAS = DIR_ETAPA_14 / "14_7_tbl_metricas_comparacoes_estatisticas.parquet"
CAMINHO_PERCENTIS = DIR_ETAPA_14 / "14_7_tbl_percentis_empiricos_controles.parquet"
CAMINHO_RANKING = DIR_ETAPA_14 / "14_7_tbl_ranking_estatistico_metricas.parquet"
CAMINHO_EVIDENCIAS_COMPACTAS = DIR_ETAPA_14 / "14_7_tbl_evidencias_estatisticas_compactas.parquet"
CAMINHO_POSICIONAMENTO = DIR_ETAPA_14 / "14_7_tbl_posicionamento_estatistico_consolidado.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_14 / "14_7_tbl_base_grafico_percentis_empiricos.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_14 / "14_7_tbl_parametros_percentis_posicionamento.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_14 / "14_7_tbl_auditoria_validacao_percentis_posicionamento.parquet"

CAMINHOS_OBRIGATORIOS = [
    CAMINHO_METRICAS_EFICIENCIA_11_3,
    CAMINHO_BASE_INFERENCIA_14_1,
    CAMINHO_MAPA_COMPARACOES_14_1,
    CAMINHO_PARAMETROS_14_1,
    CAMINHO_RESUMO_BENCHMARKS_14_2,
    CAMINHO_RESUMO_ALEATORIOS_PARES_14_3,
    CAMINHO_RESUMO_ALEATORIOS_GRUPOS_14_3,
    CAMINHO_RESUMO_MENSAIS_PARES_14_4,
    CAMINHO_RESUMO_MENSAIS_GRUPOS_14_4,
    CAMINHO_RESUMO_DIRETA_14_5,
    CAMINHO_INTERVALOS_BOOTSTRAP_14_6,
    CAMINHO_RESUMO_BOOTSTRAP_14_6,
]

for caminho in CAMINHOS_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo de entrada obrigatório não encontrado: {caminho}")

print(f"Entrada - métricas oficiais da 11.3       : {CAMINHO_METRICAS_EFICIENCIA_11_3}")
print(f"Entrada - base inferência da 14.1         : {CAMINHO_BASE_INFERENCIA_14_1}")
print(f"Entrada - mapa comparações da 14.1        : {CAMINHO_MAPA_COMPARACOES_14_1}")
print(f"Entrada - resumo benchmarks da 14.2       : {CAMINHO_RESUMO_BENCHMARKS_14_2}")
print(f"Entrada - resumo aleatórios pares da 14.3 : {CAMINHO_RESUMO_ALEATORIOS_PARES_14_3}")
print(f"Entrada - resumo mensais pares da 14.4    : {CAMINHO_RESUMO_MENSAIS_PARES_14_4}")
print(f"Entrada - resumo direta da 14.5           : {CAMINHO_RESUMO_DIRETA_14_5}")
print(f"Entrada - intervalos bootstrap da 14.6    : {CAMINHO_INTERVALOS_BOOTSTRAP_14_6}")
print(f"Saída   - métricas oficiais padronizadas  : {CAMINHO_METRICAS_OFICIAIS}")
print(f"Saída   - comparações por métrica         : {CAMINHO_COMPARACOES_METRICAS}")
print(f"Saída   - percentis empíricos             : {CAMINHO_PERCENTIS}")
print(f"Saída   - ranking estatístico             : {CAMINHO_RANKING}")
print(f"Saída   - evidências compactas            : {CAMINHO_EVIDENCIAS_COMPACTAS}")
print(f"Saída   - posicionamento consolidado      : {CAMINHO_POSICIONAMENTO}")
print(f"Saída   - base gráfico                    : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - parâmetros                      : {CAMINHO_PARAMETROS}")
print(f"Saída   - auditoria                       : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/12] Carga das bases oficiais da subetapa...")

df_metricas_11_3 = pd.read_parquet(CAMINHO_METRICAS_EFICIENCIA_11_3)
df_base_inferencia = pd.read_parquet(CAMINHO_BASE_INFERENCIA_14_1)
df_mapa_comparacoes = pd.read_parquet(CAMINHO_MAPA_COMPARACOES_14_1)
df_parametros_14_1 = pd.read_parquet(CAMINHO_PARAMETROS_14_1)

df_resumo_benchmarks = pd.read_parquet(CAMINHO_RESUMO_BENCHMARKS_14_2)
df_resumo_aleatorios_pares = pd.read_parquet(CAMINHO_RESUMO_ALEATORIOS_PARES_14_3)
df_resumo_aleatorios_grupos = pd.read_parquet(CAMINHO_RESUMO_ALEATORIOS_GRUPOS_14_3)
df_resumo_mensais_pares = pd.read_parquet(CAMINHO_RESUMO_MENSAIS_PARES_14_4)
df_resumo_mensais_grupos = pd.read_parquet(CAMINHO_RESUMO_MENSAIS_GRUPOS_14_4)
df_resumo_direta = pd.read_parquet(CAMINHO_RESUMO_DIRETA_14_5)
df_intervalos_bootstrap = pd.read_parquet(CAMINHO_INTERVALOS_BOOTSTRAP_14_6)
df_resumo_bootstrap = pd.read_parquet(CAMINHO_RESUMO_BOOTSTRAP_14_6)

if CAMINHO_COMPARACAO_DIRETA_11_7.exists():
    df_resumo_direta_11_7 = pd.read_parquet(CAMINHO_COMPARACAO_DIRETA_11_7)
else:
    df_resumo_direta_11_7 = pd.DataFrame()

print(f"Métricas oficiais da 11.3        : {len(df_metricas_11_3):,} linhas x {df_metricas_11_3.shape[1]:,} colunas")
print(f"Base consolidada da 14.1         : {len(df_base_inferencia):,} linhas x {df_base_inferencia.shape[1]:,} colunas")
print(f"Mapa de comparações da 14.1      : {len(df_mapa_comparacoes):,} linhas x {df_mapa_comparacoes.shape[1]:,} colunas")
print(f"Resumo benchmarks da 14.2        : {len(df_resumo_benchmarks):,} linhas x {df_resumo_benchmarks.shape[1]:,} colunas")
print(f"Resumo aleatórios pares da 14.3  : {len(df_resumo_aleatorios_pares):,} linhas x {df_resumo_aleatorios_pares.shape[1]:,} colunas")
print(f"Resumo mensais pares da 14.4     : {len(df_resumo_mensais_pares):,} linhas x {df_resumo_mensais_pares.shape[1]:,} colunas")
print(f"Resumo direta da 14.5            : {len(df_resumo_direta):,} linhas x {df_resumo_direta.shape[1]:,} colunas")
print(f"Intervalos bootstrap da 14.6     : {len(df_intervalos_bootstrap):,} linhas x {df_intervalos_bootstrap.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

MAPA_VISAO_INFERENCIA = {
    "diaria": "complementar",
    "mensal": "principal",
    "anual": "exploratoria",
}

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}

METRICAS_OFICIAIS = [
    {
        "metrica": "retorno_acumulado_total",
        "nome_metrica": "Retorno acumulado",
        "criterio": "maior_melhor",
        "familia_metrica": "retorno",
    },
    {
        "metrica": "retorno_anualizado",
        "nome_metrica": "Retorno anualizado",
        "criterio": "maior_melhor",
        "familia_metrica": "retorno",
    },
    {
        "metrica": "sharpe_anualizado",
        "nome_metrica": "Sharpe anualizado oficial",
        "criterio": "maior_melhor",
        "familia_metrica": "eficiencia_risco_retorno",
    },
    {
        "metrica": "sortino_anualizado",
        "nome_metrica": "Sortino anualizado oficial",
        "criterio": "maior_melhor",
        "familia_metrica": "eficiencia_risco_retorno",
    },
    {
        "metrica": "calmar",
        "nome_metrica": "Calmar oficial",
        "criterio": "maior_melhor",
        "familia_metrica": "eficiencia_risco_retorno",
    },
    {
        "metrica": "max_drawdown_abs",
        "nome_metrica": "Drawdown máximo absoluto",
        "criterio": "menor_melhor",
        "familia_metrica": "risco",
    },
    {
        "metrica": "volatilidade_sharpe_anualizada",
        "nome_metrica": "Volatilidade anualizada",
        "criterio": "menor_melhor",
        "familia_metrica": "risco",
    },
]

METRICAS_OBRIGATORIAS_OFICIAIS = [
    "retorno_acumulado_total",
    "retorno_anualizado",
    "sharpe_anualizado",
    "sortino_anualizado",
    "calmar",
]


def validar_colunas_obrigatorias(df, colunas, nome_base):
    ausentes = [coluna for coluna in colunas if coluna not in df.columns]
    if len(ausentes) > 0:
        raise KeyError(
            f"Colunas obrigatórias ausentes na base {nome_base}: {ausentes}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )


def obter_coluna_opcional(df, coluna, valor_padrao=np.nan):
    if coluna in df.columns:
        return df[coluna]
    return pd.Series(valor_padrao, index=df.index)


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce").replace([np.inf, -np.inf], np.nan)


def normalizar_texto(valor):
    if pd.isna(valor):
        return pd.NA
    return str(valor).strip()


def classificar_visao(frequencia):
    return MAPA_VISAO_INFERENCIA.get(str(frequencia), "nao_classificada")


def obter_ordem_frequencia(frequencia):
    return MAPA_ORDEM_FREQUENCIA.get(str(frequencia), 99)


def calcular_percentil_superacao(valor_real, valores_comparadores, criterio):
    valores = pd.to_numeric(pd.Series(valores_comparadores), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    valor = pd.to_numeric(pd.Series([valor_real]), errors="coerce").iloc[0]

    if pd.isna(valor) or len(valores) == 0:
        return np.nan, 0, np.nan, np.nan, np.nan

    if criterio == "menor_melhor":
        pct_superados = float((valor < valores).mean())
        pct_empatados = float(np.isclose(valor, valores, atol=0.0000000001, rtol=0.0).mean())
    else:
        pct_superados = float((valor > valores).mean())
        pct_empatados = float(np.isclose(valor, valores, atol=0.0000000001, rtol=0.0).mean())

    score_percentil = pct_superados + 0.5 * pct_empatados
    return score_percentil, int(len(valores)), float(valores.mean()), float(valores.median()), float(valores.std(ddof=1)) if len(valores) >= 2 else np.nan


def classificar_posicionamento(score):
    if pd.isna(score):
        return "nao_classificado"
    if score >= 0.80:
        return "favoravel_forte"
    if score >= 0.60:
        return "favoravel_moderado"
    if score >= 0.45:
        return "neutro"
    if score >= 0.30:
        return "desfavoravel_moderado"
    return "desfavoravel_forte"


def classificar_posicionamento_consolidado(score_metricas, prob_bootstrap):
    if pd.isna(score_metricas):
        return "nao_classificado"
    if score_metricas >= 0.80 and pd.notna(prob_bootstrap) and prob_bootstrap >= 0.70:
        return "favoravel_forte_com_suporte_bootstrap"
    if score_metricas >= 0.80:
        return "favoravel_forte_por_metricas"
    if score_metricas >= 0.60 and pd.notna(prob_bootstrap) and prob_bootstrap >= 0.55:
        return "favoravel_moderado_com_suporte_bootstrap"
    if score_metricas >= 0.60:
        return "favoravel_moderado_por_metricas"
    if score_metricas >= 0.45:
        return "neutro_ou_misto"
    if score_metricas >= 0.30:
        return "desfavoravel_moderado"
    return "desfavoravel_forte"


def preparar_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]) or str(df_saida[coluna].dtype) == "category":
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def preparar_auditoria(linhas):
    df = pd.DataFrame(linhas)
    if len(df) == 0:
        return pd.DataFrame(columns=["item", "valor", "valor_referencia", "status", "observacao"])
    df["status"] = df["status"].astype(str)
    return df[["item", "valor", "valor_referencia", "status", "observacao"]].copy()

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Padronização das métricas oficiais da Etapa 11.3
# ============================================================

print("\n[5/12] Padronização das métricas oficiais da Etapa 11.3...")

validar_colunas_obrigatorias(
    df_metricas_11_3,
    ["chave_estrategia", "frequencia"] + METRICAS_OBRIGATORIAS_OFICIAIS,
    "11.3 métricas de eficiência multifrequência",
)

if "max_drawdown_abs" not in df_metricas_11_3.columns:
    if "max_drawdown" in df_metricas_11_3.columns:
        df_metricas_11_3["max_drawdown_abs"] = converter_numero(df_metricas_11_3["max_drawdown"]).abs()
    else:
        df_metricas_11_3["max_drawdown_abs"] = np.nan

for metrica in METRICAS_OFICIAIS:
    coluna = metrica["metrica"]
    if coluna not in df_metricas_11_3.columns:
        df_metricas_11_3[coluna] = np.nan
    df_metricas_11_3[coluna] = converter_numero(df_metricas_11_3[coluna])

df_metricas_oficiais = df_metricas_11_3.copy()
df_metricas_oficiais["chave_estrategia"] = df_metricas_oficiais["chave_estrategia"].astype(str).str.strip()
df_metricas_oficiais["frequencia"] = df_metricas_oficiais["frequencia"].astype(str).str.strip()
df_metricas_oficiais["visao_inferencia"] = df_metricas_oficiais["frequencia"].map(MAPA_VISAO_INFERENCIA).fillna("nao_classificada")
df_metricas_oficiais["ordem_frequencia_inferencia"] = df_metricas_oficiais["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_metricas_oficiais["flag_estrategia_real"] = df_metricas_oficiais["chave_estrategia"].isin([
    "capitulacao__carteira_na__replica_na",
    "euforia__carteira_na__replica_na",
])

df_metricas_oficiais = df_metricas_oficiais.sort_values([
    "ordem_frequencia_inferencia",
    "flag_estrategia_real",
    "nome_exibicao_estrategia" if "nome_exibicao_estrategia" in df_metricas_oficiais.columns else "chave_estrategia",
], ascending=[True, False, True]).reset_index(drop=True)

metricas_oficiais_validas = int(df_metricas_oficiais[METRICAS_OBRIGATORIAS_OFICIAIS].notna().all(axis=1).sum())

print(f"Métricas oficiais padronizadas           : {len(df_metricas_oficiais):,} linhas")
print(f"Linhas com métricas obrigatórias completas: {metricas_oficiais_validas:,}")
print(f"Estratégias reais identificadas          : {int(df_metricas_oficiais['flag_estrategia_real'].sum()):,}")
print("OK")

# ============================================================
# 6) Construção das comparações por métricas oficiais
# ============================================================

print("\n[6/12] Construção das comparações por métricas oficiais...")

validar_colunas_obrigatorias(
    df_mapa_comparacoes,
    [
        "comparacao_id",
        "grupo_comparacao",
        "tipo_comparacao",
        "estrategia_referencia",
        "frequencia",
        "chave_estrategia_real",
        "chave_comparador",
        "nome_exibicao_comparador",
    ],
    "14.1 mapa de comparações",
)

df_mapa_aux = df_mapa_comparacoes.copy()
for coluna in ["comparacao_id", "grupo_comparacao", "tipo_comparacao", "estrategia_referencia", "frequencia", "chave_estrategia_real", "chave_comparador"]:
    df_mapa_aux[coluna] = df_mapa_aux[coluna].astype(str).str.strip()

colunas_metricas_base = [
    "chave_estrategia",
    "frequencia",
    "nome_exibicao_estrategia",
    "categoria_estrategia",
    "subcategoria_estrategia",
    "familia_estrategia",
    "estrategia_referencia",
    "tipo_estrategia",
    "visao_inferencia",
    "ordem_frequencia_inferencia",
] + [metrica["metrica"] for metrica in METRICAS_OFICIAIS]
colunas_metricas_base = [coluna for coluna in colunas_metricas_base if coluna in df_metricas_oficiais.columns]

df_metricas_join = df_metricas_oficiais[colunas_metricas_base].drop_duplicates(["chave_estrategia", "frequencia"]).copy()

df_comp_metricas = df_mapa_aux.merge(
    df_metricas_join.add_suffix("_real"),
    left_on=["chave_estrategia_real", "frequencia"],
    right_on=["chave_estrategia_real", "frequencia_real"],
    how="left",
)

df_comp_metricas = df_comp_metricas.merge(
    df_metricas_join.add_suffix("_comparador"),
    left_on=["chave_comparador", "frequencia"],
    right_on=["chave_estrategia_comparador", "frequencia_comparador"],
    how="left",
)

registros_metricas = []
for metrica in METRICAS_OFICIAIS:
    coluna = metrica["metrica"]
    coluna_real = f"{coluna}_real"
    coluna_comparador = f"{coluna}_comparador"

    if coluna_real not in df_comp_metricas.columns or coluna_comparador not in df_comp_metricas.columns:
        continue

    df_temp = df_comp_metricas.copy()
    df_temp["metrica"] = coluna
    df_temp["nome_metrica"] = metrica["nome_metrica"]
    df_temp["familia_metrica"] = metrica["familia_metrica"]
    df_temp["criterio"] = metrica["criterio"]
    df_temp["valor_real"] = converter_numero(df_temp[coluna_real])
    df_temp["valor_comparador"] = converter_numero(df_temp[coluna_comparador])

    if metrica["criterio"] == "menor_melhor":
        df_temp["diferenca_real_menos_comparador"] = df_temp["valor_real"] - df_temp["valor_comparador"]
        df_temp["flag_real_superou_comparador_metrica"] = df_temp["valor_real"] < df_temp["valor_comparador"]
    else:
        df_temp["diferenca_real_menos_comparador"] = df_temp["valor_real"] - df_temp["valor_comparador"]
        df_temp["flag_real_superou_comparador_metrica"] = df_temp["valor_real"] > df_temp["valor_comparador"]

    df_temp["flag_empate_metrica"] = np.isclose(
        df_temp["valor_real"],
        df_temp["valor_comparador"],
        atol=0.0000000001,
        rtol=0.0,
        equal_nan=False,
    )

    registros_metricas.append(df_temp)

if len(registros_metricas) > 0:
    df_metricas_comparacoes = pd.concat(registros_metricas, ignore_index=True)
else:
    df_metricas_comparacoes = pd.DataFrame()

if len(df_metricas_comparacoes) > 0:
    colunas_saida_metricas = [
        "comparacao_id",
        "origem_comparacao",
        "grupo_comparacao",
        "tipo_comparacao",
        "subtipo_comparacao",
        "estrategia_referencia",
        "frequencia",
        "visao_inferencia",
        "descricao_visao_inferencia",
        "ordem_frequencia_inferencia",
        "chave_estrategia_real",
        "nome_exibicao_estrategia_real",
        "chave_comparador",
        "nome_exibicao_comparador",
        "metrica",
        "nome_metrica",
        "familia_metrica",
        "criterio",
        "valor_real",
        "valor_comparador",
        "diferenca_real_menos_comparador",
        "flag_real_superou_comparador_metrica",
        "flag_empate_metrica",
    ]
    colunas_saida_metricas = [coluna for coluna in colunas_saida_metricas if coluna in df_metricas_comparacoes.columns]
    df_metricas_comparacoes = df_metricas_comparacoes[colunas_saida_metricas].dropna(subset=["valor_real", "valor_comparador"]).copy()
    df_metricas_comparacoes = df_metricas_comparacoes.sort_values([
        "ordem_frequencia_inferencia",
        "estrategia_referencia",
        "grupo_comparacao",
        "tipo_comparacao",
        "comparacao_id",
        "metrica",
    ]).reset_index(drop=True)

print(f"Comparações por métricas oficiais        : {len(df_metricas_comparacoes):,} linhas")
print(f"Comparações únicas com métricas oficiais : {df_metricas_comparacoes['comparacao_id'].nunique() if len(df_metricas_comparacoes) > 0 else 0:,}")
print("OK")

# ============================================================
# 7) Cálculo dos percentis empíricos por grupo de comparação
# ============================================================

print("\n[7/12] Cálculo dos percentis empíricos por grupo de comparação...")

registros_percentis = []

if len(df_metricas_comparacoes) > 0:
    agrupadores_percentis = [
        "grupo_comparacao",
        "tipo_comparacao",
        "subtipo_comparacao" if "subtipo_comparacao" in df_metricas_comparacoes.columns else "tipo_comparacao",
        "estrategia_referencia",
        "frequencia",
        "visao_inferencia",
        "ordem_frequencia_inferencia",
        "metrica",
        "nome_metrica",
        "familia_metrica",
        "criterio",
    ]
    agrupadores_percentis = list(dict.fromkeys(agrupadores_percentis))

    for chaves, grupo in df_metricas_comparacoes.groupby(agrupadores_percentis, dropna=False):
        dados_chave = dict(zip(agrupadores_percentis, chaves if isinstance(chaves, tuple) else (chaves,)))
        valores_reais = converter_numero(grupo["valor_real"]).dropna()
        valor_real = float(valores_reais.iloc[0]) if len(valores_reais) > 0 else np.nan
        valores_comparadores = converter_numero(grupo["valor_comparador"])
        criterio = dados_chave.get("criterio", "maior_melhor")
        score_percentil, n_comparadores, media_comparadores, mediana_comparadores, desvio_comparadores = calcular_percentil_superacao(
            valor_real,
            valores_comparadores,
            criterio,
        )

        registros_percentis.append({
            **dados_chave,
            "valor_real": valor_real,
            "n_comparadores_validos": n_comparadores,
            "media_comparadores": media_comparadores,
            "mediana_comparadores": mediana_comparadores,
            "desvio_comparadores": desvio_comparadores,
            "min_comparadores": float(valores_comparadores.dropna().min()) if valores_comparadores.notna().sum() > 0 else np.nan,
            "max_comparadores": float(valores_comparadores.dropna().max()) if valores_comparadores.notna().sum() > 0 else np.nan,
            "score_percentil_empirico": score_percentil,
            "pct_comparadores_superados": score_percentil,
            "classificacao_percentil_empirico": classificar_posicionamento(score_percentil),
        })

df_percentis_empiricos = pd.DataFrame(registros_percentis)

if len(df_percentis_empiricos) > 0:
    df_percentis_empiricos = df_percentis_empiricos.sort_values([
        "ordem_frequencia_inferencia",
        "estrategia_referencia",
        "grupo_comparacao",
        "tipo_comparacao",
        "familia_metrica",
        "metrica",
    ]).reset_index(drop=True)

print(f"Percentis empíricos calculados           : {len(df_percentis_empiricos):,} linhas")
print("OK")

# ============================================================
# 8) Construção do ranking estatístico por métricas oficiais e taxa de superação
# ============================================================

print("\n[8/12] Construção do ranking estatístico por métricas oficiais e taxa de superação...")

registros_ranking_metricas = []

for frequencia, grupo_freq in df_metricas_oficiais.groupby("frequencia", dropna=False):
    ordem_frequencia = obter_ordem_frequencia(frequencia)
    visao = classificar_visao(frequencia)

    for metrica in METRICAS_OFICIAIS:
        coluna = metrica["metrica"]
        if coluna not in grupo_freq.columns:
            continue

        df_rank = grupo_freq.copy()
        df_rank["valor_metrica"] = converter_numero(df_rank[coluna])
        df_rank = df_rank.dropna(subset=["valor_metrica"]).copy()

        if len(df_rank) == 0:
            continue

        ascending = metrica["criterio"] == "menor_melhor"
        df_rank["ranking_metrica"] = df_rank["valor_metrica"].rank(method="min", ascending=ascending)
        n_rank = int(len(df_rank))
        if n_rank > 1:
            df_rank["percentil_empirico_universo"] = 1.0 - ((df_rank["ranking_metrica"] - 1.0) / (n_rank - 1.0))
        else:
            df_rank["percentil_empirico_universo"] = 1.0

        for _, linha in df_rank.iterrows():
            registros_ranking_metricas.append({
                "origem_ranking": "metricas_oficiais_11_3",
                "frequencia": frequencia,
                "visao_inferencia": visao,
                "ordem_frequencia_inferencia": ordem_frequencia,
                "metrica": coluna,
                "nome_metrica": metrica["nome_metrica"],
                "familia_metrica": metrica["familia_metrica"],
                "criterio": metrica["criterio"],
                "chave_estrategia": linha.get("chave_estrategia", pd.NA),
                "nome_exibicao_estrategia": linha.get("nome_exibicao_estrategia", pd.NA),
                "estrategia_referencia": linha.get("estrategia_referencia", pd.NA),
                "categoria_estrategia": linha.get("categoria_estrategia", pd.NA),
                "subcategoria_estrategia": linha.get("subcategoria_estrategia", pd.NA),
                "familia_estrategia": linha.get("familia_estrategia", pd.NA),
                "flag_estrategia_real": bool(linha.get("flag_estrategia_real", False)),
                "valor_metrica": linha.get("valor_metrica", np.nan),
                "ranking_metrica": linha.get("ranking_metrica", np.nan),
                "n_entidades_rankeadas": n_rank,
                "percentil_empirico_universo": linha.get("percentil_empirico_universo", np.nan),
            })

registros_ranking_superacao = []
if "pct_periodos_real_superou_comparador" in df_mapa_comparacoes.columns:
    df_taxa = df_mapa_comparacoes.copy()
    df_taxa["valor_metrica"] = converter_numero(df_taxa["pct_periodos_real_superou_comparador"])
    df_taxa = df_taxa.dropna(subset=["valor_metrica"]).copy()

    for chaves, grupo in df_taxa.groupby(["frequencia", "grupo_comparacao", "tipo_comparacao"], dropna=False):
        frequencia, grupo_comparacao, tipo_comparacao = chaves
        grupo = grupo.copy()
        grupo["ranking_metrica"] = grupo["valor_metrica"].rank(method="min", ascending=False)
        n_rank = int(len(grupo))
        if n_rank > 1:
            grupo["percentil_empirico_universo"] = 1.0 - ((grupo["ranking_metrica"] - 1.0) / (n_rank - 1.0))
        else:
            grupo["percentil_empirico_universo"] = 1.0

        for _, linha in grupo.iterrows():
            registros_ranking_superacao.append({
                "origem_ranking": "taxa_superacao_comparacoes_14_1",
                "frequencia": frequencia,
                "visao_inferencia": linha.get("visao_inferencia", classificar_visao(frequencia)),
                "ordem_frequencia_inferencia": obter_ordem_frequencia(frequencia),
                "grupo_comparacao": grupo_comparacao,
                "tipo_comparacao": tipo_comparacao,
                "comparacao_id": linha.get("comparacao_id", pd.NA),
                "metrica": "pct_periodos_real_superou_comparador",
                "nome_metrica": "Taxa de superação período a período",
                "familia_metrica": "superacao",
                "criterio": "maior_melhor",
                "chave_estrategia": linha.get("chave_estrategia_real", pd.NA),
                "nome_exibicao_estrategia": linha.get("nome_exibicao_estrategia_real", pd.NA),
                "nome_exibicao_comparador": linha.get("nome_exibicao_comparador", pd.NA),
                "estrategia_referencia": linha.get("estrategia_referencia", pd.NA),
                "valor_metrica": linha.get("valor_metrica", np.nan),
                "ranking_metrica": linha.get("ranking_metrica", np.nan),
                "n_entidades_rankeadas": n_rank,
                "percentil_empirico_universo": linha.get("percentil_empirico_universo", np.nan),
            })

if len(registros_ranking_metricas) > 0:
    df_ranking_metricas = pd.DataFrame(registros_ranking_metricas)
else:
    df_ranking_metricas = pd.DataFrame()

if len(registros_ranking_superacao) > 0:
    df_ranking_superacao = pd.DataFrame(registros_ranking_superacao)
else:
    df_ranking_superacao = pd.DataFrame()

df_ranking_estatistico = pd.concat([df_ranking_metricas, df_ranking_superacao], ignore_index=True, sort=False)

if len(df_ranking_estatistico) > 0:
    df_ranking_estatistico = df_ranking_estatistico.sort_values([
        "ordem_frequencia_inferencia",
        "origem_ranking",
        "familia_metrica",
        "metrica",
        "ranking_metrica",
    ]).reset_index(drop=True)

print(f"Ranking estatístico consolidado          : {len(df_ranking_estatistico):,} linhas")
print(f"Linhas de métricas oficiais no ranking   : {len(df_ranking_metricas):,}")
print(f"Linhas de taxa de superação no ranking   : {len(df_ranking_superacao):,}")
print("OK")

# ============================================================
# 9) Consolidação das evidências estatísticas compactas
# ============================================================

print("\n[9/12] Consolidação das evidências estatísticas compactas...")

def padronizar_resumo_estatistico(df, origem):
    if len(df) == 0:
        return pd.DataFrame()

    df_temp = df.copy()
    df_temp["origem_evidencia"] = origem

    colunas_interesse = [
        "origem_evidencia",
        "comparacao_id",
        "grupo_comparacao",
        "tipo_comparacao",
        "tipo_controle_mensal",
        "estrategia_referencia",
        "frequencia",
        "visao_inferencia",
        "nome_exibicao_comparador",
        "n_periodos",
        "media_excesso_periodico",
        "media_excesso_anualizada_aproximada",
        "pct_periodos_real_superou_comparador",
        "pct_periodos_capitulacao_superou_euforia",
        "pct_periodos_euforia_superou_capitulacao",
        "p_valor_holm_teste_sinal",
        "p_valor_holm_wilcoxon",
        "p_valor_holm_permutacao",
        "n_testes_significativos_5pct",
        "classificacao_evidencia_benchmark",
        "classificacao_evidencia_aleatorio",
        "classificacao_evidencia_mensal",
        "classificacao_evidencia_capitulacao_vs_euforia",
        "direcao_economica",
    ]

    for coluna in colunas_interesse:
        if coluna not in df_temp.columns:
            df_temp[coluna] = np.nan

    return df_temp[colunas_interesse].copy()

lista_evidencias = [
    padronizar_resumo_estatistico(df_resumo_benchmarks, "14.2_benchmarks"),
    padronizar_resumo_estatistico(df_resumo_aleatorios_pares, "14.3_aleatorios_pares"),
    padronizar_resumo_estatistico(df_resumo_mensais_pares, "14.4_mensais_pares"),
    padronizar_resumo_estatistico(df_resumo_direta, "14.5_capitulacao_vs_euforia"),
]

df_evidencias_compactas = pd.concat([df for df in lista_evidencias if len(df) > 0], ignore_index=True, sort=False)

if len(df_evidencias_compactas) > 0:
    if "frequencia" in df_evidencias_compactas.columns:
        df_evidencias_compactas["ordem_frequencia_inferencia"] = df_evidencias_compactas["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
    else:
        df_evidencias_compactas["ordem_frequencia_inferencia"] = 99

    df_evidencias_compactas = df_evidencias_compactas.sort_values([
        "ordem_frequencia_inferencia",
        "origem_evidencia",
        "estrategia_referencia",
        "tipo_comparacao",
        "nome_exibicao_comparador",
    ]).reset_index(drop=True)

print(f"Evidências estatísticas compactas        : {len(df_evidencias_compactas):,} linhas")
print("OK")

# ============================================================
# 10) Construção do posicionamento estatístico consolidado
# ============================================================

print("\n[10/12] Construção do posicionamento estatístico consolidado...")

if len(df_percentis_empiricos) > 0:
    df_posicionamento = (
        df_percentis_empiricos
        .groupby([
            "grupo_comparacao",
            "tipo_comparacao",
            "estrategia_referencia",
            "frequencia",
            "visao_inferencia",
            "ordem_frequencia_inferencia",
        ], dropna=False)
        .agg(
            n_metricas_avaliadas=("metrica", "nunique"),
            n_linhas_percentis=("metrica", "size"),
            score_percentil_medio=("score_percentil_empirico", "mean"),
            score_percentil_mediano=("score_percentil_empirico", "median"),
            score_percentil_minimo=("score_percentil_empirico", "min"),
            score_percentil_maximo=("score_percentil_empirico", "max"),
            pct_metricas_score_acima_50=("score_percentil_empirico", lambda x: float((pd.to_numeric(x, errors="coerce") >= 0.50).mean())),
            pct_metricas_score_acima_60=("score_percentil_empirico", lambda x: float((pd.to_numeric(x, errors="coerce") >= 0.60).mean())),
            pct_metricas_score_acima_80=("score_percentil_empirico", lambda x: float((pd.to_numeric(x, errors="coerce") >= 0.80).mean())),
            n_comparadores_validos_total=("n_comparadores_validos", "sum"),
        )
        .reset_index()
    )
else:
    df_posicionamento = pd.DataFrame()

if len(df_posicionamento) > 0 and len(df_resumo_bootstrap) > 0:
    colunas_bootstrap_join = [
        "grupo_comparacao",
        "tipo_comparacao",
        "estrategia_referencia",
        "frequencia",
        "prob_bootstrap_media_acima_zero_media",
        "pct_comparacoes_ic95_media_acima_zero",
        "pct_comparacoes_ic95_media_abaixo_zero",
        "media_excesso_observada_media",
        "pct_superacao_observado_media",
    ]
    colunas_bootstrap_join = [coluna for coluna in colunas_bootstrap_join if coluna in df_resumo_bootstrap.columns]
    df_boot_aux = df_resumo_bootstrap[colunas_bootstrap_join].drop_duplicates([
        "grupo_comparacao",
        "tipo_comparacao",
        "estrategia_referencia",
        "frequencia",
    ]).copy()

    df_posicionamento = df_posicionamento.merge(
        df_boot_aux,
        on=["grupo_comparacao", "tipo_comparacao", "estrategia_referencia", "frequencia"],
        how="left",
    )
else:
    for coluna in [
        "prob_bootstrap_media_acima_zero_media",
        "pct_comparacoes_ic95_media_acima_zero",
        "pct_comparacoes_ic95_media_abaixo_zero",
        "media_excesso_observada_media",
        "pct_superacao_observado_media",
    ]:
        if coluna not in df_posicionamento.columns:
            df_posicionamento[coluna] = np.nan

if len(df_posicionamento) > 0:
    df_posicionamento["classificacao_percentil_medio"] = df_posicionamento["score_percentil_medio"].apply(classificar_posicionamento)
    df_posicionamento["classificacao_posicionamento_consolidado"] = df_posicionamento.apply(
        lambda linha: classificar_posicionamento_consolidado(
            linha.get("score_percentil_medio", np.nan),
            linha.get("prob_bootstrap_media_acima_zero_media", np.nan),
        ),
        axis=1,
    )
    df_posicionamento = df_posicionamento.sort_values([
        "ordem_frequencia_inferencia",
        "estrategia_referencia",
        "grupo_comparacao",
        "tipo_comparacao",
    ]).reset_index(drop=True)

print(f"Posicionamento estatístico consolidado   : {len(df_posicionamento):,} linhas")
print("OK")

# ============================================================
# 11) Base de gráficos, parâmetros e auditoria de validação
# ============================================================

print("\n[11/12] Base de gráficos, parâmetros e auditoria de validação...")

if len(df_percentis_empiricos) > 0:
    df_base_grafico = df_percentis_empiricos.copy()
    df_base_grafico["score_percentil_empirico_pct"] = df_base_grafico["score_percentil_empirico"] * 100.0
    df_base_grafico["valor_real_formatacao_base"] = df_base_grafico["valor_real"]
    df_base_grafico["media_comparadores_formatacao_base"] = df_base_grafico["media_comparadores"]
else:
    df_base_grafico = pd.DataFrame()

PARAMETROS_14_7 = {
    "etapa": "14",
    "subetapa": "14.7",
    "nome_subetapa": "Percentis Empíricos e Posicionamento Estatístico",
    "frequencia_principal_inferencia": "mensal",
    "frequencia_complementar_inferencia": "diaria",
    "frequencia_exploratoria": "anual",
    "fonte_metricas_oficiais": "Etapa 11.3",
    "sharpe_utilizado": "sharpe_anualizado oficial com taxa livre de risco",
    "calmar_utilizado": "calmar oficial calculado como retorno anualizado dividido pelo drawdown máximo absoluto",
    "metricas_ranking": ", ".join([metrica["metrica"] for metrica in METRICAS_OFICIAIS]),
    "taxa_superacao": "pct_periodos_real_superou_comparador da base de inferência 14.1",
    "comparacoes_incluidas": "benchmarks, controles aleatórios, controles mensais e comparação direta Capitulação vs Euforia",
    "interpretacao_percentil": "1.0 indica melhor posicionamento empírico dentro do grupo de comparação; 0.0 indica pior posicionamento",
    "observacao_metodologica": "Esta subetapa consolida posicionamento e rankings; não executa novos testes de hipótese.",
}

df_parametros_14_7 = pd.DataFrame([{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_14_7.items()])

frequencias_identificadas = int(df_percentis_empiricos["frequencia"].nunique()) if len(df_percentis_empiricos) > 0 else 0
estrategias_reais_metricas = int(df_metricas_oficiais.loc[df_metricas_oficiais["flag_estrategia_real"], "chave_estrategia"].nunique()) if len(df_metricas_oficiais) > 0 else 0
linhas_comparacao_direta = int((df_percentis_empiricos["tipo_comparacao"].astype(str) == "capitulacao_vs_euforia").sum()) if len(df_percentis_empiricos) > 0 else 0
linhas_benchmark = int((df_percentis_empiricos["grupo_comparacao"].astype(str) == "benchmark").sum()) if len(df_percentis_empiricos) > 0 else 0
linhas_aleatorios = int((df_percentis_empiricos["tipo_comparacao"].astype(str) == "controle_aleatorio").sum()) if len(df_percentis_empiricos) > 0 else 0
linhas_mensais = int((df_percentis_empiricos["tipo_comparacao"].astype(str) == "controle_mensal").sum()) if len(df_percentis_empiricos) > 0 else 0

metricas_oficiais_ausentes = [metrica for metrica in METRICAS_OBRIGATORIAS_OFICIAIS if metrica not in df_metricas_oficiais.columns]
sharpe_valido = int(df_metricas_oficiais["sharpe_anualizado"].notna().sum()) if "sharpe_anualizado" in df_metricas_oficiais.columns else 0
calmar_valido = int(df_metricas_oficiais["calmar"].notna().sum()) if "calmar" in df_metricas_oficiais.columns else 0

metricas_infinitas = 0
for df_verificacao in [
    df_metricas_oficiais,
    df_metricas_comparacoes,
    df_percentis_empiricos,
    df_ranking_estatistico,
    df_evidencias_compactas,
    df_posicionamento,
    df_base_grafico,
]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

linhas_auditoria = [
    {
        "item": "linhas_metricas_oficiais",
        "valor": len(df_metricas_oficiais),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_metricas_oficiais) > 0 else "ERRO",
        "observacao": "A subetapa deve carregar as métricas oficiais da Etapa 11.3.",
    },
    {
        "item": "metricas_obrigatorias_ausentes",
        "valor": len(metricas_oficiais_ausentes),
        "valor_referencia": "0",
        "status": "OK" if len(metricas_oficiais_ausentes) == 0 else "ERRO",
        "observacao": f"Métricas ausentes: {metricas_oficiais_ausentes}",
    },
    {
        "item": "sharpe_oficial_valido",
        "valor": sharpe_valido,
        "valor_referencia": "> 0",
        "status": "OK" if sharpe_valido > 0 else "ERRO",
        "observacao": "O ranking deve usar Sharpe oficial anualizado com taxa livre de risco.",
    },
    {
        "item": "calmar_oficial_valido",
        "valor": calmar_valido,
        "valor_referencia": "> 0",
        "status": "OK" if calmar_valido > 0 else "ERRO",
        "observacao": "O ranking deve usar Calmar oficial da Etapa 11.3.",
    },
    {
        "item": "estrategias_reais_identificadas_metricas",
        "valor": estrategias_reais_metricas,
        "valor_referencia": ">= 2",
        "status": "OK" if estrategias_reais_metricas >= 2 else "ERRO",
        "observacao": "Espera-se encontrar Capitulação e Euforia nas métricas oficiais.",
    },
    {
        "item": "linhas_comparacoes_metricas",
        "valor": len(df_metricas_comparacoes),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_metricas_comparacoes) > 0 else "ERRO",
        "observacao": "A subetapa deve comparar métricas oficiais entre estratégia real e comparadores.",
    },
    {
        "item": "linhas_percentis_empiricos",
        "valor": len(df_percentis_empiricos),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_percentis_empiricos) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar percentis empíricos por grupo de comparação e métrica.",
    },
    {
        "item": "frequencias_identificadas_percentis",
        "valor": frequencias_identificadas,
        "valor_referencia": ">= 3",
        "status": "OK" if frequencias_identificadas >= 3 else "ERRO",
        "observacao": "A subetapa deve preservar as visões diária, mensal e anual.",
    },
    {
        "item": "linhas_benchmarks_percentis",
        "valor": linhas_benchmark,
        "valor_referencia": "> 0",
        "status": "OK" if linhas_benchmark > 0 else "ERRO",
        "observacao": "Os percentis devem incluir comparações contra Ibovespa e CDI quando houver métricas oficiais disponíveis.",
    },
    {
        "item": "linhas_aleatorios_percentis",
        "valor": linhas_aleatorios,
        "valor_referencia": "> 0",
        "status": "OK" if linhas_aleatorios > 0 else "ERRO",
        "observacao": "Os percentis devem incluir controles aleatórios.",
    },
    {
        "item": "linhas_mensais_percentis",
        "valor": linhas_mensais,
        "valor_referencia": "> 0",
        "status": "OK" if linhas_mensais > 0 else "ERRO",
        "observacao": "Os percentis devem incluir controles mensais.",
    },
    {
        "item": "linhas_comparacao_direta_percentis",
        "valor": linhas_comparacao_direta,
        "valor_referencia": "> 0",
        "status": "OK" if linhas_comparacao_direta > 0 else "ERRO",
        "observacao": "Os percentis devem incluir Capitulação contra Euforia.",
    },
    {
        "item": "linhas_ranking_estatistico",
        "valor": len(df_ranking_estatistico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_ranking_estatistico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar ranking estatístico por métricas oficiais e taxa de superação.",
    },
    {
        "item": "linhas_posicionamento_consolidado",
        "valor": len(df_posicionamento),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_posicionamento) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar posicionamento estatístico consolidado por grupo de comparação.",
    },
    {
        "item": "linhas_base_grafico",
        "valor": len(df_base_grafico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = preparar_auditoria(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Base gráfica de percentis empíricos     : {len(df_base_grafico):,} linhas")
print(f"Parâmetros metodológicos                : {len(df_parametros_14_7):,} linhas")
print(f"Itens de auditoria                      : {len(df_auditoria):,}")
print(f"Erros bloqueantes                       : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_dataframe(preparar_para_parquet(df_metricas_oficiais), CAMINHO_METRICAS_OFICIAIS, index=False)
salvar_dataframe(preparar_para_parquet(df_metricas_comparacoes), CAMINHO_COMPARACOES_METRICAS, index=False)
salvar_dataframe(preparar_para_parquet(df_percentis_empiricos), CAMINHO_PERCENTIS, index=False)
salvar_dataframe(preparar_para_parquet(df_ranking_estatistico), CAMINHO_RANKING, index=False)
salvar_dataframe(preparar_para_parquet(df_evidencias_compactas), CAMINHO_EVIDENCIAS_COMPACTAS, index=False)
salvar_dataframe(preparar_para_parquet(df_posicionamento), CAMINHO_POSICIONAMENTO, index=False)
salvar_dataframe(preparar_para_parquet(df_base_grafico), CAMINHO_BASE_GRAFICO, index=False)
salvar_dataframe(preparar_para_parquet(df_parametros_14_7), CAMINHO_PARAMETROS, index=False)
salvar_dataframe(preparar_para_parquet(df_auditoria), CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação dos percentis e posicionamento estatístico:")
print(df_auditoria.to_string(index=False))

print("\nPercentis empíricos - amostra:")
if len(df_percentis_empiricos) > 0:
    colunas_print_percentis = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "grupo_comparacao",
            "tipo_comparacao",
            "nome_metrica",
            "criterio",
            "valor_real",
            "n_comparadores_validos",
            "media_comparadores",
            "score_percentil_empirico",
            "classificacao_percentil_empirico",
        ] if coluna in df_percentis_empiricos.columns
    ]
    print(df_percentis_empiricos[colunas_print_percentis].head(80).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nPosicionamento estatístico consolidado:")
if len(df_posicionamento) > 0:
    colunas_print_pos = [
        coluna for coluna in [
            "estrategia_referencia",
            "frequencia",
            "grupo_comparacao",
            "tipo_comparacao",
            "n_metricas_avaliadas",
            "score_percentil_medio",
            "pct_metricas_score_acima_60",
            "prob_bootstrap_media_acima_zero_media",
            "classificacao_posicionamento_consolidado",
        ] if coluna in df_posicionamento.columns
    ]
    print(df_posicionamento[colunas_print_pos].to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nRanking estatístico - estratégias reais e taxa de superação - amostra:")
if len(df_ranking_estatistico) > 0:
    colunas_print_rank = [
        coluna for coluna in [
            "origem_ranking",
            "frequencia",
            "metrica",
            "nome_metrica",
            "nome_exibicao_estrategia",
            "nome_exibicao_comparador",
            "valor_metrica",
            "ranking_metrica",
            "n_entidades_rankeadas",
            "percentil_empirico_universo",
        ] if coluna in df_ranking_estatistico.columns
    ]
    print(df_ranking_estatistico[colunas_print_rank].head(80).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nArquivos salvos na subetapa 14.7:")
print(f"- {CAMINHO_METRICAS_OFICIAIS}")
print(f"- {CAMINHO_COMPARACOES_METRICAS}")
print(f"- {CAMINHO_PERCENTIS}")
print(f"- {CAMINHO_RANKING}")
print(f"- {CAMINHO_EVIDENCIAS_COMPACTAS}")
print(f"- {CAMINHO_POSICIONAMENTO}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 14.7 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 14.7 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 14.7 - PERCENTIS EMPÍRICOS E POSICIONAMENTO ESTATÍSTICO

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - métricas oficiais da 11.3       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11\11_3_tbl_metricas_eficiencia_multifrequencia.parquet
Entrada - base inferência da 14.1         : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_base_inferencia_comparacoes_pareadas.parquet
Entrada - mapa comparações da 14.1        : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_mapa_comparacoes_inferencia.parquet
Entrada - resumo benchmarks da 14.2       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado

## Etapa 14.8) Consolidação da Evidência Estatística

In [78]:
%%time
# ============================================================
# Etapa 14.8) Consolidação da Evidência Estatística
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 14.8 - CONSOLIDAÇÃO DA EVIDÊNCIA ESTATÍSTICA")
print("=" * 100)

# Esta subetapa consolida os resultados estatísticos das subetapas 14.2 a 14.7.
# O objetivo é organizar uma leitura final, separando evidência estatística formal,
# evidência econômica, bootstrap, percentis empíricos e posicionamento por métricas oficiais.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos caminhos de entrada e saída
# ============================================================

print("\n[2/12] Definição determinística dos caminhos de entrada e saída...")

if "etapa_14" in DIRETORIOS_PROJETO:
    DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
else:
    DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["resultados"]) / "etapa_14"
    DIRETORIOS_PROJETO["etapa_14"] = DIR_ETAPA_14

DIR_ETAPA_14.mkdir(parents=True, exist_ok=True)

CAMINHO_MAPA_14_1 = DIR_ETAPA_14 / "14_1_tbl_mapa_comparacoes_inferencia.parquet"
CAMINHO_RESUMO_BASES_14_1 = DIR_ETAPA_14 / "14_1_tbl_resumo_bases_inferencia.parquet"
CAMINHO_PARAMETROS_14_1 = DIR_ETAPA_14 / "14_1_tbl_parametros_inferencia_estatistica.parquet"

CAMINHO_RESUMO_BENCHMARKS_14_2 = DIR_ETAPA_14 / "14_2_tbl_resumo_testes_benchmarks.parquet"

CAMINHO_RESUMO_ALEATORIOS_PARES_14_3 = DIR_ETAPA_14 / "14_3_tbl_resumo_testes_aleatorios_pares.parquet"
CAMINHO_RESUMO_ALEATORIOS_GRUPOS_14_3 = DIR_ETAPA_14 / "14_3_tbl_resumo_testes_aleatorios_grupos.parquet"
CAMINHO_POSICAO_ALEATORIOS_14_3 = DIR_ETAPA_14 / "14_3_tbl_posicionamento_empirico_aleatorios.parquet"

CAMINHO_RESUMO_MENSAIS_PARES_14_4 = DIR_ETAPA_14 / "14_4_tbl_resumo_testes_mensais_pares.parquet"
CAMINHO_RESUMO_MENSAIS_GRUPOS_14_4 = DIR_ETAPA_14 / "14_4_tbl_resumo_testes_mensais_grupos.parquet"
CAMINHO_POSICAO_MENSAIS_14_4 = DIR_ETAPA_14 / "14_4_tbl_posicionamento_empirico_mensais.parquet"

CAMINHO_RESUMO_DIRETA_14_5 = DIR_ETAPA_14 / "14_5_tbl_resumo_testes_capitulacao_euforia.parquet"

CAMINHO_INTERVALOS_BOOTSTRAP_14_6 = DIR_ETAPA_14 / "14_6_tbl_intervalos_bootstrap_excessos_retorno.parquet"
CAMINHO_RESUMO_BOOTSTRAP_14_6 = DIR_ETAPA_14 / "14_6_tbl_resumo_bootstrap_excessos_retorno.parquet"

CAMINHO_EVIDENCIAS_COMPACTAS_14_7 = DIR_ETAPA_14 / "14_7_tbl_evidencias_estatisticas_compactas.parquet"
CAMINHO_POSICIONAMENTO_14_7 = DIR_ETAPA_14 / "14_7_tbl_posicionamento_estatistico_consolidado.parquet"
CAMINHO_PERCENTIS_14_7 = DIR_ETAPA_14 / "14_7_tbl_percentis_empiricos_controles.parquet"
CAMINHO_RANKING_14_7 = DIR_ETAPA_14 / "14_7_tbl_ranking_estatistico_metricas.parquet"

CAMINHO_MATRIZ_EVIDENCIAS = DIR_ETAPA_14 / "14_8_tbl_matriz_evidencias_estatisticas.parquet"
CAMINHO_CONSOLIDACAO = DIR_ETAPA_14 / "14_8_tbl_consolidacao_evidencia_estatistica.parquet"
CAMINHO_RESUMO_EXECUTIVO = DIR_ETAPA_14 / "14_8_tbl_resumo_executivo_inferencia_estatistica.parquet"
CAMINHO_SINAIS_CONCLUSAO = DIR_ETAPA_14 / "14_8_tbl_sinais_conclusao_estatistica.parquet"
CAMINHO_BASE_GRAFICO = DIR_ETAPA_14 / "14_8_tbl_base_grafico_evidencia_estatistica.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_14 / "14_8_tbl_parametros_consolidacao_evidencia.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_14 / "14_8_tbl_auditoria_validacao_consolidacao_evidencia.parquet"

CAMINHOS_OBRIGATORIOS = [
    CAMINHO_MAPA_14_1,
    CAMINHO_RESUMO_BASES_14_1,
    CAMINHO_PARAMETROS_14_1,
    CAMINHO_RESUMO_BENCHMARKS_14_2,
    CAMINHO_RESUMO_ALEATORIOS_PARES_14_3,
    CAMINHO_RESUMO_ALEATORIOS_GRUPOS_14_3,
    CAMINHO_RESUMO_MENSAIS_PARES_14_4,
    CAMINHO_RESUMO_MENSAIS_GRUPOS_14_4,
    CAMINHO_RESUMO_DIRETA_14_5,
    CAMINHO_INTERVALOS_BOOTSTRAP_14_6,
    CAMINHO_RESUMO_BOOTSTRAP_14_6,
    CAMINHO_EVIDENCIAS_COMPACTAS_14_7,
    CAMINHO_POSICIONAMENTO_14_7,
    CAMINHO_PERCENTIS_14_7,
    CAMINHO_RANKING_14_7,
]

for caminho in CAMINHOS_OBRIGATORIOS:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado para a Etapa 14.8: {caminho}")

print(f"Entrada - mapa comparações 14.1              : {CAMINHO_MAPA_14_1}")
print(f"Entrada - resumo bases 14.1                  : {CAMINHO_RESUMO_BASES_14_1}")
print(f"Entrada - benchmarks 14.2                    : {CAMINHO_RESUMO_BENCHMARKS_14_2}")
print(f"Entrada - aleatórios pares 14.3              : {CAMINHO_RESUMO_ALEATORIOS_PARES_14_3}")
print(f"Entrada - aleatórios grupos 14.3             : {CAMINHO_RESUMO_ALEATORIOS_GRUPOS_14_3}")
print(f"Entrada - mensais pares 14.4                 : {CAMINHO_RESUMO_MENSAIS_PARES_14_4}")
print(f"Entrada - mensais grupos 14.4                : {CAMINHO_RESUMO_MENSAIS_GRUPOS_14_4}")
print(f"Entrada - direta 14.5                        : {CAMINHO_RESUMO_DIRETA_14_5}")
print(f"Entrada - intervalos bootstrap 14.6          : {CAMINHO_INTERVALOS_BOOTSTRAP_14_6}")
print(f"Entrada - resumo bootstrap 14.6              : {CAMINHO_RESUMO_BOOTSTRAP_14_6}")
print(f"Entrada - evidências compactas 14.7          : {CAMINHO_EVIDENCIAS_COMPACTAS_14_7}")
print(f"Entrada - posicionamento 14.7                : {CAMINHO_POSICIONAMENTO_14_7}")
print(f"Entrada - percentis 14.7                     : {CAMINHO_PERCENTIS_14_7}")
print(f"Entrada - ranking 14.7                       : {CAMINHO_RANKING_14_7}")
print(f"Saída   - matriz de evidências               : {CAMINHO_MATRIZ_EVIDENCIAS}")
print(f"Saída   - consolidação da evidência          : {CAMINHO_CONSOLIDACAO}")
print(f"Saída   - resumo executivo                   : {CAMINHO_RESUMO_EXECUTIVO}")
print(f"Saída   - sinais de conclusão                : {CAMINHO_SINAIS_CONCLUSAO}")
print(f"Saída   - base gráfico                       : {CAMINHO_BASE_GRAFICO}")
print(f"Saída   - parâmetros                         : {CAMINHO_PARAMETROS}")
print(f"Saída   - auditoria                          : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Carga das bases oficiais da subetapa
# ============================================================

print("\n[3/12] Carga das bases oficiais da subetapa...")

df_mapa_14_1 = pd.read_parquet(CAMINHO_MAPA_14_1)
df_resumo_bases_14_1 = pd.read_parquet(CAMINHO_RESUMO_BASES_14_1)
df_parametros_14_1 = pd.read_parquet(CAMINHO_PARAMETROS_14_1)

df_resumo_benchmarks_14_2 = pd.read_parquet(CAMINHO_RESUMO_BENCHMARKS_14_2)

df_resumo_aleatorios_pares_14_3 = pd.read_parquet(CAMINHO_RESUMO_ALEATORIOS_PARES_14_3)
df_resumo_aleatorios_grupos_14_3 = pd.read_parquet(CAMINHO_RESUMO_ALEATORIOS_GRUPOS_14_3)
df_posicao_aleatorios_14_3 = pd.read_parquet(CAMINHO_POSICAO_ALEATORIOS_14_3) if CAMINHO_POSICAO_ALEATORIOS_14_3.exists() else pd.DataFrame()

df_resumo_mensais_pares_14_4 = pd.read_parquet(CAMINHO_RESUMO_MENSAIS_PARES_14_4)
df_resumo_mensais_grupos_14_4 = pd.read_parquet(CAMINHO_RESUMO_MENSAIS_GRUPOS_14_4)
df_posicao_mensais_14_4 = pd.read_parquet(CAMINHO_POSICAO_MENSAIS_14_4) if CAMINHO_POSICAO_MENSAIS_14_4.exists() else pd.DataFrame()

df_resumo_direta_14_5 = pd.read_parquet(CAMINHO_RESUMO_DIRETA_14_5)

df_intervalos_bootstrap_14_6 = pd.read_parquet(CAMINHO_INTERVALOS_BOOTSTRAP_14_6)
df_resumo_bootstrap_14_6 = pd.read_parquet(CAMINHO_RESUMO_BOOTSTRAP_14_6)

df_evidencias_compactas_14_7 = pd.read_parquet(CAMINHO_EVIDENCIAS_COMPACTAS_14_7)
df_posicionamento_14_7 = pd.read_parquet(CAMINHO_POSICIONAMENTO_14_7)
df_percentis_14_7 = pd.read_parquet(CAMINHO_PERCENTIS_14_7)
df_ranking_14_7 = pd.read_parquet(CAMINHO_RANKING_14_7)

print(f"Mapa comparações 14.1              : {len(df_mapa_14_1):,} linhas x {df_mapa_14_1.shape[1]:,} colunas")
print(f"Resumo bases 14.1                  : {len(df_resumo_bases_14_1):,} linhas x {df_resumo_bases_14_1.shape[1]:,} colunas")
print(f"Resumo benchmarks 14.2             : {len(df_resumo_benchmarks_14_2):,} linhas x {df_resumo_benchmarks_14_2.shape[1]:,} colunas")
print(f"Resumo aleatórios pares 14.3       : {len(df_resumo_aleatorios_pares_14_3):,} linhas x {df_resumo_aleatorios_pares_14_3.shape[1]:,} colunas")
print(f"Resumo aleatórios grupos 14.3      : {len(df_resumo_aleatorios_grupos_14_3):,} linhas x {df_resumo_aleatorios_grupos_14_3.shape[1]:,} colunas")
print(f"Posicionamento aleatórios 14.3     : {len(df_posicao_aleatorios_14_3):,} linhas x {df_posicao_aleatorios_14_3.shape[1]:,} colunas")
print(f"Resumo mensais pares 14.4          : {len(df_resumo_mensais_pares_14_4):,} linhas x {df_resumo_mensais_pares_14_4.shape[1]:,} colunas")
print(f"Resumo mensais grupos 14.4         : {len(df_resumo_mensais_grupos_14_4):,} linhas x {df_resumo_mensais_grupos_14_4.shape[1]:,} colunas")
print(f"Posicionamento mensais 14.4        : {len(df_posicao_mensais_14_4):,} linhas x {df_posicao_mensais_14_4.shape[1]:,} colunas")
print(f"Resumo direta 14.5                 : {len(df_resumo_direta_14_5):,} linhas x {df_resumo_direta_14_5.shape[1]:,} colunas")
print(f"Intervalos bootstrap 14.6          : {len(df_intervalos_bootstrap_14_6):,} linhas x {df_intervalos_bootstrap_14_6.shape[1]:,} colunas")
print(f"Resumo bootstrap 14.6              : {len(df_resumo_bootstrap_14_6):,} linhas x {df_resumo_bootstrap_14_6.shape[1]:,} colunas")
print(f"Evidências compactas 14.7          : {len(df_evidencias_compactas_14_7):,} linhas x {df_evidencias_compactas_14_7.shape[1]:,} colunas")
print(f"Posicionamento 14.7                : {len(df_posicionamento_14_7):,} linhas x {df_posicionamento_14_7.shape[1]:,} colunas")
print(f"Percentis 14.7                     : {len(df_percentis_14_7):,} linhas x {df_percentis_14_7.shape[1]:,} colunas")
print(f"Ranking 14.7                       : {len(df_ranking_14_7):,} linhas x {df_ranking_14_7.shape[1]:,} colunas")
print("OK")

# ============================================================
# 4) Funções auxiliares da subetapa
# ============================================================

print("\n[4/12] Funções auxiliares da subetapa...")

FREQUENCIAS_ANALISE = ["diaria", "mensal", "anual"]

MAPA_VISAO_INFERENCIA = {
    "diaria": "complementar",
    "mensal": "principal",
    "anual": "exploratoria",
}

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}

ORDEM_GRUPO_COMPARACAO = {
    "benchmark": 1,
    "aleatorio_mesma_referencia": 2,
    "controle_mensal": 3,
    "comparacao_direta": 4,
}

MAPA_LABEL_GRUPO = {
    "benchmark": "Benchmarks",
    "aleatorio_mesma_referencia": "Controles Aleatórios",
    "controle_mensal": "Controles Mensais",
    "comparacao_direta": "Capitulação vs Euforia",
}


def normalizar_texto_coluna(valor):
    if pd.isna(valor):
        return ""
    return str(valor).strip()


def obter_serie(df, coluna, valor_padrao=np.nan):
    if coluna in df.columns:
        return df[coluna]
    return pd.Series(valor_padrao, index=df.index)


def obter_primeira_coluna(df, colunas, valor_padrao=np.nan):
    for coluna in colunas:
        if coluna in df.columns:
            return df[coluna]
    return pd.Series(valor_padrao, index=df.index)


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def converter_string(serie):
    return serie.astype("string")


def preparar_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def salvar_parquet_compat(df, caminho, index=False):
    salvar_dataframe(preparar_para_parquet(df), caminho, index=index)


def criar_chave_evidencia(df):
    componentes = []
    for coluna in [
        "estrategia_referencia",
        "frequencia",
        "grupo_comparacao",
        "tipo_comparacao",
        "subtipo_comparacao",
        "nome_exibicao_comparador",
    ]:
        if coluna in df.columns:
            componentes.append(df[coluna].astype(str).fillna(""))
        else:
            componentes.append(pd.Series("", index=df.index))
    chave = componentes[0]
    for componente in componentes[1:]:
        chave = chave + "__" + componente
    return chave.str.lower().str.replace(" ", "_", regex=False)


def classificar_score(score):
    if pd.isna(score):
        return "nao_classificado"
    if score >= 1.25:
        return "favoravel_forte"
    if score >= 0.50:
        return "favoravel_moderado"
    if score > -0.50:
        return "neutro_ou_misto"
    if score > -1.25:
        return "desfavoravel_moderado"
    return "desfavoravel_forte"


def classificar_sinal_score(score):
    if pd.isna(score):
        return "neutro"
    if score > 0.0000000001:
        return "favoravel"
    if score < -0.0000000001:
        return "desfavoravel"
    return "neutro"


def calcular_score_textual(texto, valor_metrica=np.nan, n_sig=np.nan):
    texto_norm = str(texto).strip().lower()
    valor = np.nan
    try:
        valor = float(valor_metrica)
    except Exception:
        valor = np.nan

    n_sig_num = 0.0
    try:
        n_sig_num = float(n_sig)
    except Exception:
        n_sig_num = 0.0

    # A ordem é intencional: "desfavoravel" contém "favoravel" como substring.
    # Classificações desfavoráveis devem ser avaliadas antes das favoráveis.
    if "desfavoravel_forte" in texto_norm:
        return -1.50
    if "desfavoravel" in texto_norm and n_sig_num > 0:
        return -1.50
    if "desfavoravel" in texto_norm:
        return -0.50
    if "favoravel_forte_com_suporte" in texto_norm:
        return 2.00
    if "favoravel_forte_por_metricas" in texto_norm:
        return 1.50
    if "favoravel_forte" in texto_norm:
        return 1.50
    if "favoravel_moderado" in texto_norm:
        return 1.00
    if "favoravel_economica" in texto_norm and n_sig_num > 0:
        return 1.50
    if "favoravel_economica" in texto_norm:
        return 0.50
    if "neutra_a_favoravel" in texto_norm:
        return 0.25
    if "neutro" in texto_norm or "mista" in texto_norm:
        return 0.00

    if not pd.isna(valor):
        if valor > 0:
            return 0.25
        if valor < 0:
            return -0.25

    return 0.00


def calcular_score_testes(media_excesso, n_sig, classificacao):
    try:
        media = float(media_excesso)
    except Exception:
        media = np.nan

    try:
        n_sig_num = float(n_sig)
    except Exception:
        n_sig_num = 0.0

    if not pd.isna(media):
        if n_sig_num > 0 and media > 0:
            return 2.00
        if n_sig_num > 0 and media < 0:
            return -2.00
        if media > 0:
            return 0.50
        if media < 0:
            return -0.50

    return calcular_score_textual(classificacao, media_excesso, n_sig)


def calcular_score_bootstrap(media_excesso, ic_inf, ic_sup, prob_acima, classificacao):
    try:
        media = float(media_excesso)
    except Exception:
        media = np.nan

    try:
        inf = float(ic_inf)
    except Exception:
        inf = np.nan

    try:
        sup = float(ic_sup)
    except Exception:
        sup = np.nan

    try:
        prob = float(prob_acima)
    except Exception:
        prob = np.nan

    if not pd.isna(inf) and not pd.isna(sup):
        if inf > 0 and sup > 0:
            return 2.00
        if inf < 0 and sup < 0:
            return -2.00

    if not pd.isna(prob):
        if prob >= 0.70:
            return 1.00
        if prob >= 0.60:
            return 0.50
        if prob <= 0.30:
            return -1.00
        if prob <= 0.40:
            return -0.50

    if not pd.isna(media):
        if media > 0:
            return 0.25
        if media < 0:
            return -0.25

    return calcular_score_textual(classificacao, media_excesso, np.nan)


def obter_classificacao_final(score_medio, n_sig, n_bootstrap_confirmado):
    try:
        score = float(score_medio)
    except Exception:
        score = np.nan

    try:
        sig = int(n_sig)
    except Exception:
        sig = 0

    try:
        boot = int(n_bootstrap_confirmado)
    except Exception:
        boot = 0

    if pd.isna(score):
        return "nao_classificado"

    if score >= 1.25 and (sig > 0 or boot > 0):
        return "favoravel_forte_com_confirmacao_estatistica"
    if score >= 1.25:
        return "favoravel_forte_por_metricas_sem_confirmacao_formal"
    if score >= 0.50 and (sig > 0 or boot > 0):
        return "favoravel_moderado_com_confirmacao_estatistica"
    if score >= 0.50:
        return "favoravel_moderado_sem_confirmacao_formal"
    if score > -0.50:
        return "neutro_ou_misto_sem_confirmacao_formal"
    if score > -1.25 and (sig > 0 or boot > 0):
        return "desfavoravel_moderado_com_confirmacao_estatistica"
    if score > -1.25:
        return "desfavoravel_moderado_sem_confirmacao_formal"
    if sig > 0 or boot > 0:
        return "desfavoravel_forte_com_confirmacao_estatistica"
    return "desfavoravel_forte_sem_confirmacao_formal"


def eh_classificacao_favoravel(serie):
    serie_str = serie.astype(str).str.strip().str.lower()
    return serie_str.str.startswith("favoravel")


def eh_classificacao_desfavoravel(serie):
    serie_str = serie.astype(str).str.strip().str.lower()
    return serie_str.str.startswith("desfavoravel")


def eh_classificacao_com_confirmacao_formal(serie):
    serie_str = serie.astype(str).str.strip().str.lower()
    return (
        serie_str.str.contains("confirmacao", case=False, na=False)
        & ~serie_str.str.contains("sem_confirmacao", case=False, na=False)
    )


def obter_leitura_final(row):
    classificacao = str(row.get("classificacao_evidencia_final", "nao_classificado")).strip().lower()
    grupo = str(row.get("grupo_comparacao", ""))
    frequencia = str(row.get("frequencia", ""))

    # A ordem é intencional: classificações desfavoráveis contêm a substring
    # "favoravel" em português, portanto devem ser avaliadas antes das favoráveis.
    if classificacao.startswith("desfavoravel_forte"):
        if "com_confirmacao" in classificacao and "sem_confirmacao" not in classificacao:
            return "Evidência desfavorável forte com confirmação estatística."
        return "Evidência desfavorável forte no conjunto de métricas e testes considerados, sem confirmação formal robusta."
    if classificacao.startswith("desfavoravel_moderado"):
        if "com_confirmacao" in classificacao and "sem_confirmacao" not in classificacao:
            return "Evidência desfavorável moderada com confirmação estatística."
        return "Evidência desfavorável/moderada, sem suporte suficiente para conclusão positiva."
    if classificacao.startswith("favoravel_forte"):
        if "com_confirmacao" in classificacao and "sem_confirmacao" not in classificacao:
            return "Evidência favorável forte com confirmação estatística."
        return "Evidência favorável forte por métricas oficiais, sem confirmação formal robusta."
    if classificacao.startswith("favoravel_moderado"):
        if "com_confirmacao" in classificacao and "sem_confirmacao" not in classificacao:
            return "Evidência favorável moderada com confirmação estatística."
        return "Evidência econômica favorável/moderada, sem confirmação formal robusta."
    if classificacao.startswith("neutro") or "misto" in classificacao:
        return "Evidência mista ou neutra; resultados dependem do critério analisado."
    return f"Leitura não classificada para {grupo} em frequência {frequencia}."


def montar_base_evidencia_testes(df, origem, grupo_padrao, tipo_padrao, coluna_classificacao):
    if len(df) == 0:
        return pd.DataFrame()

    base = pd.DataFrame(index=df.index)
    base["origem_evidencia"] = origem
    base["categoria_evidencia"] = "teste_estatistico_periodo_a_periodo"
    base["estrategia_referencia"] = converter_string(obter_serie(df, "estrategia_referencia", ""))
    base["frequencia"] = converter_string(obter_serie(df, "frequencia", ""))
    base["visao_inferencia"] = converter_string(obter_serie(df, "visao_inferencia", ""))
    base["grupo_comparacao"] = converter_string(obter_serie(df, "grupo_comparacao", grupo_padrao))
    base["tipo_comparacao"] = converter_string(obter_serie(df, "tipo_comparacao", tipo_padrao))
    base["subtipo_comparacao"] = converter_string(obter_primeira_coluna(df, ["subtipo_comparacao", "tipo_controle_mensal"], ""))
    base["nome_exibicao_comparador"] = converter_string(obter_serie(df, "nome_exibicao_comparador", ""))
    base["n_observacoes"] = converter_numero(obter_primeira_coluna(df, ["n_periodos", "n_observacoes"], np.nan))
    base["media_excesso_periodico"] = converter_numero(obter_serie(df, "media_excesso_periodico", np.nan))
    base["pct_periodos_real_superou_comparador"] = converter_numero(obter_serie(df, "pct_periodos_real_superou_comparador", np.nan))
    base["p_valor_minimo_ajustado"] = pd.concat(
        [
            converter_numero(obter_serie(df, "p_valor_holm_teste_sinal", np.nan)),
            converter_numero(obter_serie(df, "p_valor_holm_wilcoxon", np.nan)),
            converter_numero(obter_serie(df, "p_valor_holm_permutacao", np.nan)),
        ],
        axis=1,
    ).min(axis=1, skipna=True)
    base["n_testes_significativos_5pct"] = converter_numero(obter_serie(df, "n_testes_significativos_5pct", 0)).fillna(0)
    base["prob_bootstrap_media_acima_zero"] = np.nan
    base["score_evidencia"] = [
        calcular_score_testes(media, sig, cls)
        for media, sig, cls in zip(
            base["media_excesso_periodico"],
            base["n_testes_significativos_5pct"],
            obter_serie(df, coluna_classificacao, ""),
        )
    ]
    base["sinal_evidencia"] = base["score_evidencia"].map(classificar_sinal_score)
    base["classificacao_origem"] = converter_string(obter_serie(df, coluna_classificacao, ""))
    base["observacao_evidencia"] = "Teste formal aplicado sobre excessos de retorno pareados por período."
    return base


def montar_base_evidencia_posicionamento(df, origem, grupo_padrao, tipo_padrao, coluna_classificacao):
    if len(df) == 0:
        return pd.DataFrame()

    base = pd.DataFrame(index=df.index)
    base["origem_evidencia"] = origem
    base["categoria_evidencia"] = "posicionamento_empirico"
    base["estrategia_referencia"] = converter_string(obter_serie(df, "estrategia_referencia", ""))
    base["frequencia"] = converter_string(obter_serie(df, "frequencia", ""))
    base["visao_inferencia"] = converter_string(obter_serie(df, "visao_inferencia", ""))
    base["grupo_comparacao"] = converter_string(obter_serie(df, "grupo_comparacao", grupo_padrao))
    base["tipo_comparacao"] = converter_string(obter_serie(df, "tipo_comparacao", tipo_padrao))
    base["subtipo_comparacao"] = converter_string(obter_serie(df, "subtipo_comparacao", ""))
    base["nome_exibicao_comparador"] = converter_string(obter_serie(df, "nome_exibicao_comparador", "grupo_consolidado"))
    base["n_observacoes"] = converter_numero(obter_primeira_coluna(df, ["n_comparacoes", "n_pares_aleatorios", "n_pares_mensais", "n_controles_mensais", "n_metricas_avaliadas"], np.nan))
    base["media_excesso_periodico"] = converter_numero(
        obter_primeira_coluna(
            df,
            [
                "media_excesso_periodico_media_pares",
                "media_das_medias_excesso_periodico",
                "media_excesso_observada_media",
            ],
            np.nan,
        )
    )
    base["pct_periodos_real_superou_comparador"] = converter_numero(
        obter_primeira_coluna(
            df,
            [
                "pct_periodos_real_superou_comparador_medio",
                "pct_periodos_vencedores_medio",
                "pct_superacao_observado_media",
            ],
            np.nan,
        )
    )
    base["p_valor_minimo_ajustado"] = np.nan
    base["n_testes_significativos_5pct"] = converter_numero(obter_serie(df, "n_testes_significativos_5pct_total", 0)).fillna(0)
    base["prob_bootstrap_media_acima_zero"] = converter_numero(obter_serie(df, "prob_bootstrap_media_acima_zero_media", np.nan))
    base["score_evidencia"] = [
        calcular_score_textual(cls, media, sig)
        for cls, media, sig in zip(
            obter_serie(df, coluna_classificacao, ""),
            base["media_excesso_periodico"],
            base["n_testes_significativos_5pct"],
        )
    ]
    base["sinal_evidencia"] = base["score_evidencia"].map(classificar_sinal_score)
    base["classificacao_origem"] = converter_string(obter_serie(df, coluna_classificacao, ""))
    base["observacao_evidencia"] = "Posicionamento empírico agregado por grupo de comparação."
    return base


def montar_base_evidencia_bootstrap(df):
    if len(df) == 0:
        return pd.DataFrame()

    base = pd.DataFrame(index=df.index)
    base["origem_evidencia"] = "14.6_bootstrap_intervalos"
    base["categoria_evidencia"] = "bootstrap_intervalo_confianca"
    base["estrategia_referencia"] = converter_string(obter_serie(df, "estrategia_referencia", ""))
    base["frequencia"] = converter_string(obter_serie(df, "frequencia", ""))
    base["visao_inferencia"] = converter_string(obter_serie(df, "visao_inferencia", ""))
    base["grupo_comparacao"] = converter_string(obter_serie(df, "grupo_comparacao", ""))
    base["tipo_comparacao"] = converter_string(obter_serie(df, "tipo_comparacao", ""))
    base["subtipo_comparacao"] = converter_string(obter_primeira_coluna(df, ["subtipo_comparacao", "tipo_controle_mensal"], ""))
    base["nome_exibicao_comparador"] = converter_string(obter_serie(df, "nome_exibicao_comparador", ""))
    base["n_observacoes"] = converter_numero(obter_serie(df, "n_observacoes", np.nan))
    base["media_excesso_periodico"] = converter_numero(obter_serie(df, "media_excesso_observada", np.nan))
    base["pct_periodos_real_superou_comparador"] = converter_numero(obter_serie(df, "pct_superacao_observada", np.nan))
    base["p_valor_minimo_ajustado"] = np.nan
    base["n_testes_significativos_5pct"] = 0
    base["prob_bootstrap_media_acima_zero"] = converter_numero(obter_serie(df, "prob_bootstrap_media_acima_zero", np.nan))
    base["ic95_media_inf"] = converter_numero(obter_serie(df, "ic95_media_inf", np.nan))
    base["ic95_media_sup"] = converter_numero(obter_serie(df, "ic95_media_sup", np.nan))
    base["score_evidencia"] = [
        calcular_score_bootstrap(media, inf, sup, prob, cls)
        for media, inf, sup, prob, cls in zip(
            base["media_excesso_periodico"],
            base["ic95_media_inf"],
            base["ic95_media_sup"],
            base["prob_bootstrap_media_acima_zero"],
            obter_serie(df, "classificacao_bootstrap_media", ""),
        )
    ]
    base["sinal_evidencia"] = base["score_evidencia"].map(classificar_sinal_score)
    base["classificacao_origem"] = converter_string(obter_serie(df, "classificacao_bootstrap_media", ""))
    base["observacao_evidencia"] = "Intervalo de confiança por bootstrap dos excessos de retorno."
    return base


def montar_base_evidencia_percentis(df):
    if len(df) == 0:
        return pd.DataFrame()

    base = pd.DataFrame(index=df.index)
    base["origem_evidencia"] = "14.7_percentis_metricas_oficiais"
    base["categoria_evidencia"] = "percentil_empirico_metricas_oficiais"
    base["estrategia_referencia"] = converter_string(obter_serie(df, "estrategia_referencia", ""))
    base["frequencia"] = converter_string(obter_serie(df, "frequencia", ""))
    base["visao_inferencia"] = converter_string(obter_serie(df, "visao_inferencia", ""))
    base["grupo_comparacao"] = converter_string(obter_serie(df, "grupo_comparacao", ""))
    base["tipo_comparacao"] = converter_string(obter_serie(df, "tipo_comparacao", ""))
    base["subtipo_comparacao"] = converter_string(obter_primeira_coluna(df, ["subtipo_comparacao", "nome_metrica"], ""))
    base["nome_exibicao_comparador"] = converter_string(obter_serie(df, "nome_metrica", "metricas_oficiais"))
    base["n_observacoes"] = converter_numero(obter_serie(df, "n_comparadores_validos", np.nan))
    base["media_excesso_periodico"] = np.nan
    base["pct_periodos_real_superou_comparador"] = converter_numero(obter_serie(df, "score_percentil_empirico", np.nan))
    base["p_valor_minimo_ajustado"] = np.nan
    base["n_testes_significativos_5pct"] = 0
    base["prob_bootstrap_media_acima_zero"] = np.nan
    base["score_evidencia"] = np.select(
        [
            converter_numero(obter_serie(df, "score_percentil_empirico", np.nan)) >= 0.80,
            converter_numero(obter_serie(df, "score_percentil_empirico", np.nan)) >= 0.60,
            converter_numero(obter_serie(df, "score_percentil_empirico", np.nan)) <= 0.20,
            converter_numero(obter_serie(df, "score_percentil_empirico", np.nan)) <= 0.40,
        ],
        [1.50, 0.75, -1.50, -0.75],
        default=0.00,
    )
    base["sinal_evidencia"] = pd.Series(base["score_evidencia"], index=base.index).map(classificar_sinal_score)
    base["classificacao_origem"] = converter_string(obter_serie(df, "classificacao_percentil_empirico", ""))
    base["observacao_evidencia"] = "Percentil empírico com métricas oficiais da Etapa 11.3."
    return base


print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Montagem da matriz unificada de evidências
# ============================================================

print("\n[5/12] Montagem da matriz unificada de evidências...")

bases_evidencias = []

bases_evidencias.append(
    montar_base_evidencia_testes(
        df_resumo_benchmarks_14_2,
        origem="14.2_testes_benchmarks",
        grupo_padrao="benchmark",
        tipo_padrao="benchmark",
        coluna_classificacao="classificacao_evidencia_benchmark",
    )
)

bases_evidencias.append(
    montar_base_evidencia_testes(
        df_resumo_aleatorios_pares_14_3,
        origem="14.3_testes_aleatorios_pares",
        grupo_padrao="aleatorio_mesma_referencia",
        tipo_padrao="controle_aleatorio",
        coluna_classificacao="classificacao_evidencia_aleatorio",
    )
)

bases_evidencias.append(
    montar_base_evidencia_posicionamento(
        df_resumo_aleatorios_grupos_14_3,
        origem="14.3_resumo_aleatorios_grupos",
        grupo_padrao="aleatorio_mesma_referencia",
        tipo_padrao="controle_aleatorio",
        coluna_classificacao="classificacao_empirica_vs_aleatorias",
    )
)

bases_evidencias.append(
    montar_base_evidencia_posicionamento(
        df_posicao_aleatorios_14_3,
        origem="14.3_posicionamento_aleatorios",
        grupo_padrao="aleatorio_mesma_referencia",
        tipo_padrao="controle_aleatorio",
        coluna_classificacao="classificacao_empirica_vs_aleatorias",
    )
)

bases_evidencias.append(
    montar_base_evidencia_testes(
        df_resumo_mensais_pares_14_4,
        origem="14.4_testes_mensais_pares",
        grupo_padrao="controle_mensal",
        tipo_padrao="controle_mensal",
        coluna_classificacao="classificacao_evidencia_mensal",
    )
)

bases_evidencias.append(
    montar_base_evidencia_posicionamento(
        df_resumo_mensais_grupos_14_4,
        origem="14.4_resumo_mensais_grupos",
        grupo_padrao="controle_mensal",
        tipo_padrao="controle_mensal",
        coluna_classificacao="classificacao_empirica_vs_mensais",
    )
)

bases_evidencias.append(
    montar_base_evidencia_posicionamento(
        df_posicao_mensais_14_4,
        origem="14.4_posicionamento_mensais",
        grupo_padrao="controle_mensal",
        tipo_padrao="controle_mensal",
        coluna_classificacao="classificacao_empirica_vs_mensais",
    )
)

bases_evidencias.append(
    montar_base_evidencia_testes(
        df_resumo_direta_14_5,
        origem="14.5_testes_capitulacao_vs_euforia",
        grupo_padrao="comparacao_direta",
        tipo_padrao="capitulacao_vs_euforia",
        coluna_classificacao="classificacao_evidencia_capitulacao_vs_euforia",
    )
)

bases_evidencias.append(montar_base_evidencia_bootstrap(df_intervalos_bootstrap_14_6))
bases_evidencias.append(
    montar_base_evidencia_posicionamento(
        df_resumo_bootstrap_14_6,
        origem="14.6_resumo_bootstrap_grupos",
        grupo_padrao="",
        tipo_padrao="",
        coluna_classificacao="classificacao_bootstrap_media",
    )
)
bases_evidencias.append(
    montar_base_evidencia_posicionamento(
        df_posicionamento_14_7,
        origem="14.7_posicionamento_metricas_oficiais",
        grupo_padrao="",
        tipo_padrao="",
        coluna_classificacao="classificacao_posicionamento_consolidado",
    )
)
bases_evidencias.append(montar_base_evidencia_percentis(df_percentis_14_7))

bases_evidencias = [df for df in bases_evidencias if isinstance(df, pd.DataFrame) and len(df) > 0]

if len(bases_evidencias) == 0:
    raise ValueError("Nenhuma base de evidências foi gerada na Etapa 14.8.")

df_matriz_evidencias = pd.concat(bases_evidencias, ignore_index=True)

for coluna in ["estrategia_referencia", "frequencia", "grupo_comparacao", "tipo_comparacao"]:
    if coluna in df_matriz_evidencias.columns:
        df_matriz_evidencias[coluna] = df_matriz_evidencias[coluna].astype(str).str.strip()

df_matriz_evidencias["visao_inferencia"] = df_matriz_evidencias["frequencia"].map(MAPA_VISAO_INFERENCIA).fillna(df_matriz_evidencias["visao_inferencia"].astype(str))
df_matriz_evidencias["ordem_frequencia_inferencia"] = df_matriz_evidencias["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
df_matriz_evidencias["ordem_grupo_comparacao"] = df_matriz_evidencias["grupo_comparacao"].map(ORDEM_GRUPO_COMPARACAO).fillna(99).astype(int)
df_matriz_evidencias["label_grupo_comparacao"] = df_matriz_evidencias["grupo_comparacao"].map(MAPA_LABEL_GRUPO).fillna(df_matriz_evidencias["grupo_comparacao"])
df_matriz_evidencias["chave_evidencia"] = criar_chave_evidencia(df_matriz_evidencias)

df_matriz_evidencias = (
    df_matriz_evidencias
    .sort_values(
        [
            "ordem_frequencia_inferencia",
            "estrategia_referencia",
            "ordem_grupo_comparacao",
            "tipo_comparacao",
            "categoria_evidencia",
            "nome_exibicao_comparador",
        ]
    )
    .reset_index(drop=True)
)

print(f"Matriz unificada de evidências           : {len(df_matriz_evidencias):,} linhas")
print(f"Origens de evidência                     : {df_matriz_evidencias['origem_evidencia'].nunique():,}")
print(f"Categorias de evidência                  : {df_matriz_evidencias['categoria_evidencia'].nunique():,}")
print(f"Grupos de comparação                     : {df_matriz_evidencias['grupo_comparacao'].nunique():,}")
print("OK")

# ============================================================
# 6) Consolidação da evidência por estratégia, frequência e grupo
# ============================================================

print("\n[6/12] Consolidação da evidência por estratégia, frequência e grupo...")

agrupadores_consolidacao = [
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "ordem_frequencia_inferencia",
    "grupo_comparacao",
    "label_grupo_comparacao",
    "ordem_grupo_comparacao",
    "tipo_comparacao",
]

df_consolidacao = (
    df_matriz_evidencias
    .groupby(agrupadores_consolidacao, dropna=False)
    .agg(
        n_evidencias=("score_evidencia", "size"),
        n_origens_evidencia=("origem_evidencia", "nunique"),
        n_categorias_evidencia=("categoria_evidencia", "nunique"),
        score_evidencia_total=("score_evidencia", "sum"),
        score_evidencia_medio=("score_evidencia", "mean"),
        n_evidencias_favoraveis=("sinal_evidencia", lambda x: int((x == "favoravel").sum())),
        n_evidencias_neutras=("sinal_evidencia", lambda x: int((x == "neutro").sum())),
        n_evidencias_desfavoraveis=("sinal_evidencia", lambda x: int((x == "desfavoravel").sum())),
        n_testes_significativos_5pct=("n_testes_significativos_5pct", lambda x: int(pd.to_numeric(x, errors="coerce").fillna(0).sum())),
        p_valor_minimo_ajustado=("p_valor_minimo_ajustado", "min"),
        media_excesso_periodico_media=("media_excesso_periodico", "mean"),
        pct_periodos_superacao_media=("pct_periodos_real_superou_comparador", "mean"),
        prob_bootstrap_media_acima_zero_media=("prob_bootstrap_media_acima_zero", "mean"),
    )
    .reset_index()
)

df_consolidacao["pct_evidencias_favoraveis"] = np.where(
    df_consolidacao["n_evidencias"] > 0,
    df_consolidacao["n_evidencias_favoraveis"] / df_consolidacao["n_evidencias"],
    np.nan,
)

df_consolidacao["pct_evidencias_desfavoraveis"] = np.where(
    df_consolidacao["n_evidencias"] > 0,
    df_consolidacao["n_evidencias_desfavoraveis"] / df_consolidacao["n_evidencias"],
    np.nan,
)

df_consolidacao["n_bootstrap_confirmado"] = (
    df_matriz_evidencias
    .assign(
        flag_bootstrap_confirmado=lambda x: (
            x["categoria_evidencia"].astype(str).eq("bootstrap_intervalo_confianca")
            & x["classificacao_origem"].astype(str).str.contains("confirmacao", case=False, na=False)
            & ~x["classificacao_origem"].astype(str).str.contains("sem_confirmacao", case=False, na=False)
        )
    )
    .groupby(agrupadores_consolidacao, dropna=False)["flag_bootstrap_confirmado"]
    .sum()
    .reset_index(name="n_bootstrap_confirmado")
)["n_bootstrap_confirmado"].astype(int)

df_consolidacao["classificacao_score"] = df_consolidacao["score_evidencia_medio"].map(classificar_score)

df_consolidacao["classificacao_evidencia_final"] = [
    obter_classificacao_final(score, sig, boot)
    for score, sig, boot in zip(
        df_consolidacao["score_evidencia_medio"],
        df_consolidacao["n_testes_significativos_5pct"],
        df_consolidacao["n_bootstrap_confirmado"],
    )
]

df_consolidacao["leitura_final_estatistica"] = df_consolidacao.apply(obter_leitura_final, axis=1)

df_consolidacao = (
    df_consolidacao
    .sort_values(
        [
            "ordem_frequencia_inferencia",
            "estrategia_referencia",
            "ordem_grupo_comparacao",
            "tipo_comparacao",
        ]
    )
    .reset_index(drop=True)
)

print(f"Consolidação da evidência estatística    : {len(df_consolidacao):,} linhas")
print(f"Linhas na visão mensal principal         : {int((df_consolidacao['frequencia'] == 'mensal').sum()):,}")
print("OK")

# ============================================================
# 7) Construção do resumo executivo da inferência
# ============================================================

print("\n[7/12] Construção do resumo executivo da inferência...")

df_mensal = df_consolidacao.loc[df_consolidacao["frequencia"].eq("mensal")].copy()

if len(df_mensal) > 0:
    df_resumo_exec_estrategia = (
        df_mensal
        .groupby(["estrategia_referencia"], dropna=False)
        .agg(
            n_grupos_avaliados=("grupo_comparacao", "nunique"),
            score_medio_geral=("score_evidencia_medio", "mean"),
            pct_grupos_favoraveis=("classificacao_evidencia_final", lambda x: float(eh_classificacao_favoravel(x).mean())),
            pct_grupos_desfavoraveis=("classificacao_evidencia_final", lambda x: float(eh_classificacao_desfavoravel(x).mean())),
            n_testes_significativos_5pct_total=("n_testes_significativos_5pct", "sum"),
            prob_bootstrap_media=("prob_bootstrap_media_acima_zero_media", "mean"),
        )
        .reset_index()
    )
else:
    df_resumo_exec_estrategia = pd.DataFrame(
        columns=[
            "estrategia_referencia",
            "n_grupos_avaliados",
            "score_medio_geral",
            "pct_grupos_favoraveis",
            "pct_grupos_desfavoraveis",
            "n_testes_significativos_5pct_total",
            "prob_bootstrap_media",
        ]
    )

df_resumo_exec_estrategia["classificacao_geral_visao_mensal"] = [
    obter_classificacao_final(score, sig, 0)
    for score, sig in zip(
        df_resumo_exec_estrategia["score_medio_geral"],
        df_resumo_exec_estrategia["n_testes_significativos_5pct_total"],
    )
]

classificacao_geral_serie = df_resumo_exec_estrategia["classificacao_geral_visao_mensal"].astype("string").fillna("")
flag_geral_favoravel = eh_classificacao_favoravel(classificacao_geral_serie)
flag_geral_desfavoravel = eh_classificacao_desfavoravel(classificacao_geral_serie)
flag_geral_confirmacao = eh_classificacao_com_confirmacao_formal(classificacao_geral_serie)

df_resumo_exec_estrategia["leitura_geral_visao_mensal"] = np.select(
    [
        flag_geral_favoravel & flag_geral_confirmacao,
        flag_geral_favoravel,
        flag_geral_desfavoravel & flag_geral_confirmacao,
        flag_geral_desfavoravel,
    ],
    [
        "Leitura geral favorável com confirmação estatística na visão mensal.",
        "Leitura geral favorável por métricas e evidências econômicas, mas sem confirmação estatística formal robusta.",
        "Leitura geral desfavorável com confirmação estatística na visão mensal.",
        "Leitura geral desfavorável ou insuficiente na visão mensal.",
    ],
    default="Leitura geral neutra ou mista na visão mensal.",
)

registros_resumo_grupos = []

for _, linha in df_mensal.iterrows():
    registros_resumo_grupos.append(
        {
            "nivel_resumo": "grupo_comparacao_mensal",
            "estrategia_referencia": linha.get("estrategia_referencia", ""),
            "grupo_comparacao": linha.get("grupo_comparacao", ""),
            "label_grupo_comparacao": linha.get("label_grupo_comparacao", ""),
            "tipo_comparacao": linha.get("tipo_comparacao", ""),
            "score_evidencia_medio": linha.get("score_evidencia_medio", np.nan),
            "classificacao_evidencia_final": linha.get("classificacao_evidencia_final", ""),
            "leitura_final": linha.get("leitura_final_estatistica", ""),
            "n_evidencias": linha.get("n_evidencias", np.nan),
            "n_testes_significativos_5pct": linha.get("n_testes_significativos_5pct", np.nan),
            "prob_bootstrap_media_acima_zero_media": linha.get("prob_bootstrap_media_acima_zero_media", np.nan),
        }
    )

for _, linha in df_resumo_exec_estrategia.iterrows():
    registros_resumo_grupos.append(
        {
            "nivel_resumo": "estrategia_mensal",
            "estrategia_referencia": linha.get("estrategia_referencia", ""),
            "grupo_comparacao": "todos_os_grupos",
            "label_grupo_comparacao": "Todos os Grupos",
            "tipo_comparacao": "consolidado_mensal",
            "score_evidencia_medio": linha.get("score_medio_geral", np.nan),
            "classificacao_evidencia_final": linha.get("classificacao_geral_visao_mensal", ""),
            "leitura_final": linha.get("leitura_geral_visao_mensal", ""),
            "n_evidencias": linha.get("n_grupos_avaliados", np.nan),
            "n_testes_significativos_5pct": linha.get("n_testes_significativos_5pct_total", np.nan),
            "prob_bootstrap_media_acima_zero_media": linha.get("prob_bootstrap_media", np.nan),
        }
    )

df_resumo_executivo = pd.DataFrame(registros_resumo_grupos)

if len(df_resumo_executivo) > 0:
    df_resumo_executivo["ordem_nivel_resumo"] = np.select(
        [
            df_resumo_executivo["nivel_resumo"].eq("estrategia_mensal"),
            df_resumo_executivo["nivel_resumo"].eq("grupo_comparacao_mensal"),
        ],
        [1, 2],
        default=99,
    )
    df_resumo_executivo["ordem_grupo_comparacao"] = df_resumo_executivo["grupo_comparacao"].map(ORDEM_GRUPO_COMPARACAO).fillna(99).astype(int)
    df_resumo_executivo = (
        df_resumo_executivo
        .sort_values(["ordem_nivel_resumo", "estrategia_referencia", "ordem_grupo_comparacao"])
        .reset_index(drop=True)
    )

print(f"Resumo executivo da inferência           : {len(df_resumo_executivo):,} linhas")
print("OK")

# ============================================================
# 8) Sinais finais para conclusão estatística
# ============================================================

print("\n[8/12] Sinais finais para conclusão estatística...")

registros_sinais = []

for estrategia in sorted(df_consolidacao["estrategia_referencia"].dropna().astype(str).unique()):
    df_est = df_consolidacao.loc[df_consolidacao["estrategia_referencia"].astype(str).eq(estrategia)].copy()
    df_est_mensal = df_est.loc[df_est["frequencia"].eq("mensal")].copy()

    def extrair_classificacao(grupo, tipo=None):
        base = df_est_mensal.loc[df_est_mensal["grupo_comparacao"].astype(str).eq(grupo)].copy()
        if tipo is not None:
            base = base.loc[base["tipo_comparacao"].astype(str).eq(tipo)].copy()
        if len(base) == 0:
            return "nao_disponivel"
        return str(base.iloc[0]["classificacao_evidencia_final"])

    def extrair_score(grupo, tipo=None):
        base = df_est_mensal.loc[df_est_mensal["grupo_comparacao"].astype(str).eq(grupo)].copy()
        if tipo is not None:
            base = base.loc[base["tipo_comparacao"].astype(str).eq(tipo)].copy()
        if len(base) == 0:
            return np.nan
        return float(base.iloc[0]["score_evidencia_medio"])

    class_benchmark_ibov = extrair_classificacao("benchmark", "benchmark_ibovespa")
    class_benchmark_cdi = extrair_classificacao("benchmark", "benchmark_cdi")
    class_aleatorios = extrair_classificacao("aleatorio_mesma_referencia", "controle_aleatorio")
    class_mensais = extrair_classificacao("controle_mensal", "controle_mensal")
    class_direta = extrair_classificacao("comparacao_direta", "capitulacao_vs_euforia")

    score_benchmark_ibov = extrair_score("benchmark", "benchmark_ibovespa")
    score_benchmark_cdi = extrair_score("benchmark", "benchmark_cdi")
    score_aleatorios = extrair_score("aleatorio_mesma_referencia", "controle_aleatorio")
    score_mensais = extrair_score("controle_mensal", "controle_mensal")
    score_direta = extrair_score("comparacao_direta", "capitulacao_vs_euforia")

    n_sig_total = int(pd.to_numeric(df_est_mensal["n_testes_significativos_5pct"], errors="coerce").fillna(0).sum())
    n_confirmacao_bootstrap = int(pd.to_numeric(df_est_mensal["n_bootstrap_confirmado"], errors="coerce").fillna(0).sum())

    registros_sinais.append(
        {
            "estrategia_referencia": estrategia,
            "frequencia_base_conclusao": "mensal",
            "classificacao_vs_ibovespa": class_benchmark_ibov,
            "classificacao_vs_cdi": class_benchmark_cdi,
            "classificacao_vs_aleatorias": class_aleatorios,
            "classificacao_vs_mensais": class_mensais,
            "classificacao_vs_outra_estrategia": class_direta,
            "score_vs_ibovespa": score_benchmark_ibov,
            "score_vs_cdi": score_benchmark_cdi,
            "score_vs_aleatorias": score_aleatorios,
            "score_vs_mensais": score_mensais,
            "score_vs_outra_estrategia": score_direta,
            "n_testes_significativos_5pct_mensal": n_sig_total,
            "n_confirmacoes_bootstrap_mensal": n_confirmacao_bootstrap,
            "sinal_robustez_benchmarks": (
                "favoravel" if (
                    str(class_benchmark_ibov).startswith("favoravel")
                    and str(class_benchmark_cdi).startswith("favoravel")
                ) else "misto"
            ),
            "sinal_robustez_controles": (
                "favoravel" if str(class_mensais).startswith("favoravel") and not str(class_aleatorios).startswith("desfavoravel")
                else "misto_ou_fraco"
            ),
            "sinal_confirmacao_formal": (
                "com_confirmacao_formal" if n_sig_total > 0 or n_confirmacao_bootstrap > 0 else "sem_confirmacao_formal"
            ),
        }
    )

df_sinais_conclusao = pd.DataFrame(registros_sinais)

if len(df_sinais_conclusao) > 0:
    df_sinais_conclusao["leitura_sintese"] = np.select(
        [
            df_sinais_conclusao["sinal_confirmacao_formal"].eq("com_confirmacao_formal")
            & df_sinais_conclusao["sinal_robustez_benchmarks"].eq("favoravel")
            & df_sinais_conclusao["sinal_robustez_controles"].eq("favoravel"),
            df_sinais_conclusao["sinal_robustez_benchmarks"].eq("favoravel")
            & df_sinais_conclusao["sinal_robustez_controles"].eq("favoravel"),
            df_sinais_conclusao["sinal_robustez_benchmarks"].eq("favoravel"),
        ],
        [
            "Estratégia com leitura favorável ampla e algum suporte formal.",
            "Estratégia com leitura econômica e posicional favorável, mas sem confirmação formal robusta.",
            "Estratégia com leitura favorável contra benchmarks, mas evidência mista contra controles.",
        ],
        default="Estratégia com evidência mista ou insuficiente na consolidação estatística.",
    )

print(f"Sinais finais de conclusão estatística   : {len(df_sinais_conclusao):,} linhas")
if len(df_sinais_conclusao) > 0:
    print(df_sinais_conclusao.to_string(index=False))
else:
    print("Sem sinais finais para exibir.")
print("OK")

# ============================================================
# 9) Base de gráficos para o HTML final
# ============================================================

print("\n[9/12] Base de gráficos para o HTML final...")

df_base_grafico = df_consolidacao.copy()
df_base_grafico["metrica_grafico"] = "score_evidencia_medio"
df_base_grafico["valor_grafico"] = df_base_grafico["score_evidencia_medio"]
df_base_grafico["label_valor_grafico"] = df_base_grafico["valor_grafico"].map(lambda x: "" if pd.isna(x) else f"{x:.2f}")
df_base_grafico["flag_visao_principal"] = df_base_grafico["frequencia"].eq("mensal")
df_base_grafico["flag_evidencia_favoravel"] = eh_classificacao_favoravel(df_base_grafico["classificacao_evidencia_final"].astype("string"))
df_base_grafico["flag_evidencia_desfavoravel"] = eh_classificacao_desfavoravel(df_base_grafico["classificacao_evidencia_final"].astype("string"))

colunas_grafico = [
    "estrategia_referencia",
    "frequencia",
    "visao_inferencia",
    "grupo_comparacao",
    "label_grupo_comparacao",
    "tipo_comparacao",
    "score_evidencia_medio",
    "valor_grafico",
    "label_valor_grafico",
    "pct_evidencias_favoraveis",
    "pct_evidencias_desfavoraveis",
    "prob_bootstrap_media_acima_zero_media",
    "n_testes_significativos_5pct",
    "classificacao_evidencia_final",
    "flag_visao_principal",
    "flag_evidencia_favoravel",
    "flag_evidencia_desfavoravel",
    "ordem_frequencia_inferencia",
    "ordem_grupo_comparacao",
]

df_base_grafico = df_base_grafico[[coluna for coluna in colunas_grafico if coluna in df_base_grafico.columns]].copy()

print(f"Base gráfica de evidência estatística    : {len(df_base_grafico):,} linhas")
print("OK")

# ============================================================
# 10) Parâmetros metodológicos da consolidação
# ============================================================

print("\n[10/12] Parâmetros metodológicos da consolidação...")

PARAMETROS_14_8 = {
    "etapa": "14",
    "subetapa": "14.8",
    "versao_codigo": "v3",
    "nome_subetapa": "Consolidação da Evidência Estatística",
    "frequencia_principal_conclusao": "mensal",
    "frequencia_complementar": "diaria",
    "frequencia_exploratoria": "anual",
    "fontes_consolidadas": "14.2, 14.3, 14.4, 14.5, 14.6 e 14.7",
    "tratamento_benchmarks": "Ibovespa e CDI consolidados separadamente",
    "tratamento_controles_aleatorios": "testes pareados, posicionamento empírico, bootstrap e percentis oficiais",
    "tratamento_controles_mensais": "testes pareados, grupos de controle, bootstrap e percentis oficiais",
    "tratamento_comparacao_direta": "Capitulação contra Euforia consolidada em retorno pareado e métricas oficiais",
    "criterio_score_positivo": "valores positivos indicam evidência favorável à estratégia de referência",
    "criterio_score_negativo": "valores negativos indicam evidência desfavorável à estratégia de referência",
    "observacao_metodologica": "A consolidação não cria novos testes; apenas organiza evidências já calculadas.",
}

df_parametros = pd.DataFrame(
    [{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_14_8.items()]
)

print(f"Parâmetros metodológicos                : {len(df_parametros):,} linhas")
print("OK")

# ============================================================
# 11) Auditoria de validação
# ============================================================

print("\n[11/12] Auditoria de validação...")

def contar_linhas_grupo(df, grupo):
    if len(df) == 0 or "grupo_comparacao" not in df.columns:
        return 0
    return int(df["grupo_comparacao"].astype(str).eq(grupo).sum())


def contar_linhas_freq(df, freq):
    if len(df) == 0 or "frequencia" not in df.columns:
        return 0
    return int(df["frequencia"].astype(str).eq(freq).sum())


metricas_infinitas = 0
for df_verificacao in [
    df_matriz_evidencias,
    df_consolidacao,
    df_resumo_executivo,
    df_sinais_conclusao,
    df_base_grafico,
]:
    if len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

inconsistencias_leitura_desfavoravel = 0
if len(df_consolidacao) > 0:
    classif_tmp = df_consolidacao["classificacao_evidencia_final"].astype("string").fillna("").str.lower()
    leitura_tmp = df_consolidacao["leitura_final_estatistica"].astype("string").fillna("").str.lower()
    leitura_tmp_ascii = (
        leitura_tmp
        .str.replace("á", "a", regex=False)
        .str.replace("à", "a", regex=False)
        .str.replace("â", "a", regex=False)
        .str.replace("ã", "a", regex=False)
        .str.replace("é", "e", regex=False)
        .str.replace("ê", "e", regex=False)
        .str.replace("í", "i", regex=False)
        .str.replace("ó", "o", regex=False)
        .str.replace("ô", "o", regex=False)
        .str.replace("õ", "o", regex=False)
        .str.replace("ú", "u", regex=False)
        .str.replace("ç", "c", regex=False)
    )
    leitura_textualmente_favoravel = (
        leitura_tmp_ascii.str.startswith("evidencia favoravel")
        | leitura_tmp_ascii.str.startswith("leitura geral favoravel")
        | leitura_tmp_ascii.str.contains("economica favoravel", regex=False, na=False)
        | leitura_tmp_ascii.str.contains("leitura favoravel", regex=False, na=False)
    )
    inconsistencias_leitura_desfavoravel = int((
        classif_tmp.str.startswith("desfavoravel")
        & leitura_textualmente_favoravel
    ).sum())

linhas_auditoria = [
    {
        "item": "linhas_matriz_evidencias",
        "valor": len(df_matriz_evidencias),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_matriz_evidencias) > 0 else "ERRO",
        "observacao": "A subetapa deve consolidar evidências das subetapas estatísticas anteriores.",
    },
    {
        "item": "linhas_consolidacao",
        "valor": len(df_consolidacao),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_consolidacao) > 0 else "ERRO",
        "observacao": "A consolidação por estratégia, frequência e grupo deve ser gerada.",
    },
    {
        "item": "linhas_resumo_executivo",
        "valor": len(df_resumo_executivo),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_resumo_executivo) > 0 else "ERRO",
        "observacao": "O resumo executivo deve ser gerado para consumo do HTML final.",
    },
    {
        "item": "linhas_sinais_conclusao",
        "valor": len(df_sinais_conclusao),
        "valor_referencia": ">= 2",
        "status": "OK" if len(df_sinais_conclusao) >= 2 else "ERRO",
        "observacao": "Espera-se uma linha final para Capitulação e outra para Euforia.",
    },
    {
        "item": "frequencias_consolidacao",
        "valor": int(df_consolidacao["frequencia"].nunique()) if len(df_consolidacao) > 0 else 0,
        "valor_referencia": ">= 3",
        "status": "OK" if (len(df_consolidacao) > 0 and df_consolidacao["frequencia"].nunique() >= 3) else "ERRO",
        "observacao": "A consolidação deve preservar as visões diária, mensal e anual.",
    },
    {
        "item": "linhas_mensais_principais",
        "valor": contar_linhas_freq(df_consolidacao, "mensal"),
        "valor_referencia": "> 0",
        "status": "OK" if contar_linhas_freq(df_consolidacao, "mensal") > 0 else "ERRO",
        "observacao": "A visão mensal deve existir como principal para conclusão estatística.",
    },
    {
        "item": "linhas_benchmarks",
        "valor": contar_linhas_grupo(df_consolidacao, "benchmark"),
        "valor_referencia": "> 0",
        "status": "OK" if contar_linhas_grupo(df_consolidacao, "benchmark") > 0 else "ERRO",
        "observacao": "A consolidação deve incluir benchmarks.",
    },
    {
        "item": "linhas_aleatorios",
        "valor": contar_linhas_grupo(df_consolidacao, "aleatorio_mesma_referencia"),
        "valor_referencia": "> 0",
        "status": "OK" if contar_linhas_grupo(df_consolidacao, "aleatorio_mesma_referencia") > 0 else "ERRO",
        "observacao": "A consolidação deve incluir controles aleatórios.",
    },
    {
        "item": "linhas_mensais",
        "valor": contar_linhas_grupo(df_consolidacao, "controle_mensal"),
        "valor_referencia": "> 0",
        "status": "OK" if contar_linhas_grupo(df_consolidacao, "controle_mensal") > 0 else "ERRO",
        "observacao": "A consolidação deve incluir controles mensais.",
    },
    {
        "item": "linhas_comparacao_direta",
        "valor": contar_linhas_grupo(df_consolidacao, "comparacao_direta"),
        "valor_referencia": "> 0",
        "status": "OK" if contar_linhas_grupo(df_consolidacao, "comparacao_direta") > 0 else "ERRO",
        "observacao": "A consolidação deve incluir Capitulação contra Euforia.",
    },
    {
        "item": "origens_evidencia",
        "valor": int(df_matriz_evidencias["origem_evidencia"].nunique()) if len(df_matriz_evidencias) > 0 else 0,
        "valor_referencia": ">= 6",
        "status": "OK" if (len(df_matriz_evidencias) > 0 and df_matriz_evidencias["origem_evidencia"].nunique() >= 6) else "ERRO",
        "observacao": "A matriz deve consolidar múltiplas origens de evidência.",
    },
    {
        "item": "classificacoes_finais_ausentes",
        "valor": int(df_consolidacao["classificacao_evidencia_final"].isna().sum()) if len(df_consolidacao) > 0 else 0,
        "valor_referencia": "0",
        "status": "OK" if (len(df_consolidacao) > 0 and df_consolidacao["classificacao_evidencia_final"].isna().sum() == 0) else "ERRO",
        "observacao": "Todas as linhas consolidadas devem ter classificação final.",
    },
    {
        "item": "linhas_base_grafico",
        "valor": len(df_base_grafico),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_base_grafico) > 0 else "ERRO",
        "observacao": "A subetapa deve gerar base gráfica para o HTML final.",
    },
    {
        "item": "leituras_desfavoraveis_com_texto_favoravel",
        "valor": inconsistencias_leitura_desfavoravel,
        "valor_referencia": "0",
        "status": "OK" if inconsistencias_leitura_desfavoravel == 0 else "ERRO",
        "observacao": "Classificações desfavoráveis não devem receber leitura textual favorável.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As métricas numéricas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = pd.DataFrame(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Itens de auditoria                      : {len(df_auditoria):,}")
print(f"Erros bloqueantes                       : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final...")

salvar_parquet_compat(df_matriz_evidencias, CAMINHO_MATRIZ_EVIDENCIAS, index=False)
salvar_parquet_compat(df_consolidacao, CAMINHO_CONSOLIDACAO, index=False)
salvar_parquet_compat(df_resumo_executivo, CAMINHO_RESUMO_EXECUTIVO, index=False)
salvar_parquet_compat(df_sinais_conclusao, CAMINHO_SINAIS_CONCLUSAO, index=False)
salvar_parquet_compat(df_base_grafico, CAMINHO_BASE_GRAFICO, index=False)
salvar_parquet_compat(df_parametros, CAMINHO_PARAMETROS, index=False)
salvar_parquet_compat(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação da consolidação da evidência estatística:")
print(df_auditoria.to_string(index=False))

print("\nResumo executivo da inferência estatística:")
if len(df_resumo_executivo) > 0:
    print(df_resumo_executivo.to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nSinais finais para conclusão estatística:")
if len(df_sinais_conclusao) > 0:
    print(df_sinais_conclusao.to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nConsolidação da evidência estatística - visão mensal principal:")
df_print_mensal = df_consolidacao.loc[df_consolidacao["frequencia"].eq("mensal")].copy()
if len(df_print_mensal) > 0:
    colunas_print = [
        coluna for coluna in [
            "estrategia_referencia",
            "grupo_comparacao",
            "tipo_comparacao",
            "n_evidencias",
            "score_evidencia_medio",
            "pct_evidencias_favoraveis",
            "pct_evidencias_desfavoraveis",
            "n_testes_significativos_5pct",
            "prob_bootstrap_media_acima_zero_media",
            "classificacao_evidencia_final",
            "leitura_final_estatistica",
        ] if coluna in df_print_mensal.columns
    ]
    print(df_print_mensal[colunas_print].to_string(index=False))
else:
    print("Sem linhas mensais para exibir.")

print("\nArquivos salvos na subetapa 14.8:")
print(f"- {CAMINHO_MATRIZ_EVIDENCIAS}")
print(f"- {CAMINHO_CONSOLIDACAO}")
print(f"- {CAMINHO_RESUMO_EXECUTIVO}")
print(f"- {CAMINHO_SINAIS_CONCLUSAO}")
print(f"- {CAMINHO_BASE_GRAFICO}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 14.8 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 14.8 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 14.8 - CONSOLIDAÇÃO DA EVIDÊNCIA ESTATÍSTICA

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos caminhos de entrada e saída...
Entrada - mapa comparações 14.1              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_mapa_comparacoes_inferencia.parquet
Entrada - resumo bases 14.1                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_1_tbl_resumo_bases_inferencia.parquet
Entrada - benchmarks 14.2                    : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14\14_2_tbl_resumo_testes_benchmarks.parquet
Entrada - aleatórios pares 14.3              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resulta

# Etapa 15) Bases Finais para o HTML

## Etapa 15.1) Tabelas-Resumo das Estratégias

In [79]:
%%time
# ============================================================
# Etapa 15.1) Tabelas-Resumo das Estratégias
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 15.1 - TABELAS-RESUMO DAS ESTRATÉGIAS")
print("=" * 100)

# Esta subetapa consolida tabelas analíticas já produzidas ao longo do projeto
# para consumo posterior pela seção final do HTML. Não há recálculo pesado de backtest.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/12] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos diretórios e caminhos de saída
# ============================================================

print("\n[2/12] Definição determinística dos diretórios e caminhos de saída...")

DIR_RESULTADOS = Path(DIRETORIOS_PROJETO["resultados"])

for etapa in range(1, 16):
    chave_etapa = f"etapa_{etapa}"
    if chave_etapa not in DIRETORIOS_PROJETO:
        DIRETORIOS_PROJETO[chave_etapa] = DIR_RESULTADOS / chave_etapa
    Path(DIRETORIOS_PROJETO[chave_etapa]).mkdir(parents=True, exist_ok=True)

DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])
DIR_ETAPA_12 = Path(DIRETORIOS_PROJETO["etapa_12"])
DIR_ETAPA_13 = Path(DIRETORIOS_PROJETO["etapa_13"])
DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
DIR_ETAPA_15 = Path(DIRETORIOS_PROJETO["etapa_15"])

DIR_ETAPA_15.mkdir(parents=True, exist_ok=True)

CAMINHO_RESUMO_EXECUTIVO_HTML = DIR_ETAPA_15 / "15_1_tbl_resumo_executivo_html.parquet"
CAMINHO_METRICAS_PRINCIPAIS_HTML = DIR_ETAPA_15 / "15_1_tbl_metricas_principais_estrategias_html.parquet"
CAMINHO_COMPARACAO_BENCHMARKS_HTML = DIR_ETAPA_15 / "15_1_tbl_comparacao_benchmarks_html.parquet"
CAMINHO_COMPARACAO_CONTROLES_HTML = DIR_ETAPA_15 / "15_1_tbl_comparacao_controles_html.parquet"
CAMINHO_COMPARACAO_DIRETA_HTML = DIR_ETAPA_15 / "15_1_tbl_comparacao_capitulacao_euforia_html.parquet"
CAMINHO_INFERENCIA_ESTATISTICA_HTML = DIR_ETAPA_15 / "15_1_tbl_inferencia_estatistica_html.parquet"
CAMINHO_ROBUSTEZ_SENSIBILIDADE_HTML = DIR_ETAPA_15 / "15_1_tbl_robustez_sensibilidade_html.parquet"
CAMINHO_CATALOGO_TABELAS_HTML = DIR_ETAPA_15 / "15_1_tbl_catalogo_tabelas_html.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_15 / "15_1_tbl_parametros_tabelas_resumo_html.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_15 / "15_1_tbl_auditoria_validacao_tabelas_resumo_html.parquet"

print(f"Diretório da Etapa 11 : {DIR_ETAPA_11}")
print(f"Diretório da Etapa 12 : {DIR_ETAPA_12}")
print(f"Diretório da Etapa 13 : {DIR_ETAPA_13}")
print(f"Diretório da Etapa 14 : {DIR_ETAPA_14}")
print(f"Diretório da Etapa 15 : {DIR_ETAPA_15}")
print(f"Saída - resumo executivo HTML             : {CAMINHO_RESUMO_EXECUTIVO_HTML}")
print(f"Saída - métricas principais HTML          : {CAMINHO_METRICAS_PRINCIPAIS_HTML}")
print(f"Saída - comparação benchmarks HTML        : {CAMINHO_COMPARACAO_BENCHMARKS_HTML}")
print(f"Saída - comparação controles HTML         : {CAMINHO_COMPARACAO_CONTROLES_HTML}")
print(f"Saída - comparação Cap. vs Euforia HTML   : {CAMINHO_COMPARACAO_DIRETA_HTML}")
print(f"Saída - inferência estatística HTML       : {CAMINHO_INFERENCIA_ESTATISTICA_HTML}")
print(f"Saída - robustez e sensibilidade HTML     : {CAMINHO_ROBUSTEZ_SENSIBILIDADE_HTML}")
print(f"Saída - catálogo de tabelas HTML          : {CAMINHO_CATALOGO_TABELAS_HTML}")
print(f"Saída - parâmetros                        : {CAMINHO_PARAMETROS}")
print(f"Saída - auditoria                         : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Catálogo determinístico de entradas da subetapa
# ============================================================

print("\n[3/12] Catálogo determinístico de entradas da subetapa...")

ENTRADAS_15_1 = [
    {
        "chave": "metricas_oficiais_11_3",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_3_tbl_metricas_eficiencia_multifrequencia.parquet",
        "obrigatorio": True,
        "finalidade": "Métricas oficiais multifrequência, incluindo Sharpe oficial e Calmar oficial.",
    },
    {
        "chave": "comparacao_benchmarks_diaria_11_5",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_diaria.parquet",
        "obrigatorio": False,
        "finalidade": "Base diária de comparação contra Ibovespa e CDI.",
    },
    {
        "chave": "comparacao_benchmarks_mensal_11_5",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_mensal.parquet",
        "obrigatorio": False,
        "finalidade": "Base mensal de comparação contra Ibovespa e CDI.",
    },
    {
        "chave": "comparacao_benchmarks_anual_11_5",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_anual.parquet",
        "obrigatorio": False,
        "finalidade": "Base anual de comparação contra Ibovespa e CDI.",
    },
    {
        "chave": "pares_controles_11_6",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_6_tbl_pares_comparacao_controles.parquet",
        "obrigatorio": False,
        "finalidade": "Mapa de pares contra controles aleatórios e mensais.",
    },
    {
        "chave": "ranking_reais_vs_controles_11_6",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_6_tbl_ranking_estrategias_reais_vs_controles.parquet",
        "obrigatorio": False,
        "finalidade": "Ranking das estratégias reais contra controles.",
    },
    {
        "chave": "posicionamento_multifrequencia_11_6",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_6_tbl_posicionamento_relativo_multifrequencia.parquet",
        "obrigatorio": False,
        "finalidade": "Posicionamento relativo multifrequência contra controles.",
    },
    {
        "chave": "resumo_capitulacao_vs_euforia_11_7",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_7_tbl_resumo_comparacao_capitulacao_euforia.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo descritivo direto entre Capitulação e Euforia.",
    },
    {
        "chave": "ranking_capitulacao_vs_euforia_11_7",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_7_tbl_ranking_capitulacao_vs_euforia.parquet",
        "obrigatorio": True,
        "finalidade": "Ranking comparativo direto entre Capitulação e Euforia.",
    },
    {
        "chave": "resumo_sensibilidade_capitulacao_13_4",
        "etapa": 13,
        "caminho": DIR_ETAPA_13 / "13_4_tbl_resumo_sensibilidade_limiares_capitulacao.parquet",
        "obrigatorio": False,
        "finalidade": "Resumo de sensibilidade dos limiares de Capitulação.",
    },
    {
        "chave": "comparacao_limiar_capitulacao_13_4",
        "etapa": 13,
        "caminho": DIR_ETAPA_13 / "13_4_tbl_comparacao_limiar_capitulacao_referencia.parquet",
        "obrigatorio": False,
        "finalidade": "Comparação dos limiares alternativos de Capitulação contra referência.",
    },
    {
        "chave": "resumo_sensibilidade_euforia_13_5",
        "etapa": 13,
        "caminho": DIR_ETAPA_13 / "13_5_tbl_resumo_sensibilidade_limiares_euforia.parquet",
        "obrigatorio": False,
        "finalidade": "Resumo de sensibilidade dos limiares de Euforia.",
    },
    {
        "chave": "comparacao_limiar_euforia_13_5",
        "etapa": 13,
        "caminho": DIR_ETAPA_13 / "13_5_tbl_comparacao_limiar_euforia_referencia.parquet",
        "obrigatorio": False,
        "finalidade": "Comparação dos limiares alternativos de Euforia contra referência.",
    },
    {
        "chave": "posicionamento_aleatorias_13_6",
        "etapa": 13,
        "caminho": DIR_ETAPA_13 / "13_6_tbl_posicionamento_final_estrategias_reais_vs_aleatorias.parquet",
        "obrigatorio": False,
        "finalidade": "Posicionamento final das estratégias reais contra réplicas aleatórias.",
    },
    {
        "chave": "posicionamento_mensais_13_7",
        "etapa": 13,
        "caminho": DIR_ETAPA_13 / "13_7_tbl_posicionamento_final_estrategias_reais_vs_mensais.parquet",
        "obrigatorio": False,
        "finalidade": "Posicionamento final das estratégias reais contra controles mensais.",
    },
    {
        "chave": "resumo_benchmarks_14_2",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_2_tbl_resumo_testes_benchmarks.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo dos testes contra Ibovespa e CDI.",
    },
    {
        "chave": "resumo_aleatorios_pares_14_3",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_3_tbl_resumo_testes_aleatorios_pares.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo dos testes par a par contra controles aleatórios.",
    },
    {
        "chave": "resumo_aleatorios_grupos_14_3",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_3_tbl_resumo_testes_aleatorios_grupos.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo agregado contra controles aleatórios.",
    },
    {
        "chave": "resumo_mensais_pares_14_4",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_4_tbl_resumo_testes_mensais_pares.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo dos testes par a par contra controles mensais.",
    },
    {
        "chave": "resumo_mensais_grupos_14_4",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_4_tbl_resumo_testes_mensais_grupos.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo agregado contra controles mensais.",
    },
    {
        "chave": "resumo_capitulacao_vs_euforia_14_5",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_5_tbl_resumo_testes_capitulacao_euforia.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo dos testes formais entre Capitulação e Euforia.",
    },
    {
        "chave": "resumo_bootstrap_14_6",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_6_tbl_resumo_bootstrap_excessos_retorno.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo agregado de bootstrap dos excessos de retorno.",
    },
    {
        "chave": "intervalos_bootstrap_14_6",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_6_tbl_intervalos_bootstrap_excessos_retorno.parquet",
        "obrigatorio": True,
        "finalidade": "Intervalos de confiança por comparação.",
    },
    {
        "chave": "percentis_empiricos_14_7",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_7_tbl_percentis_empiricos_controles.parquet",
        "obrigatorio": True,
        "finalidade": "Percentis empíricos por métricas oficiais.",
    },
    {
        "chave": "ranking_estatistico_14_7",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_7_tbl_ranking_estatistico_metricas.parquet",
        "obrigatorio": True,
        "finalidade": "Ranking estatístico por métricas oficiais e taxa de superação.",
    },
    {
        "chave": "posicionamento_estatistico_14_7",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_7_tbl_posicionamento_estatistico_consolidado.parquet",
        "obrigatorio": True,
        "finalidade": "Posicionamento estatístico consolidado por métricas oficiais.",
    },
    {
        "chave": "consolidacao_evidencia_14_8",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_8_tbl_consolidacao_evidencia_estatistica.parquet",
        "obrigatorio": True,
        "finalidade": "Consolidação final da evidência estatística.",
    },
    {
        "chave": "resumo_executivo_inferencia_14_8",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_8_tbl_resumo_executivo_inferencia_estatistica.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo executivo da inferência estatística.",
    },
    {
        "chave": "sinais_conclusao_estatistica_14_8",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_8_tbl_sinais_conclusao_estatistica.parquet",
        "obrigatorio": True,
        "finalidade": "Sinais finais para conclusão estatística.",
    },
]

for entrada in ENTRADAS_15_1:
    print(f"{entrada['chave']:<45} | obrigatório={str(entrada['obrigatorio']):<5} | {entrada['caminho']}")

print("OK")

# ============================================================
# 4) Funções auxiliares de carga, padronização e salvamento
# ============================================================

print("\n[4/12] Funções auxiliares de carga, padronização e salvamento...")

def preparar_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def salvar_parquet_compat(df, caminho, index=False):
    salvar_dataframe(preparar_para_parquet(df), caminho, index=index)


def carregar_parquet_entrada(entrada):
    caminho = Path(entrada["caminho"])
    if caminho.exists():
        df = pd.read_parquet(caminho)
        status = "carregado"
    else:
        df = pd.DataFrame()
        status = "ausente"
    registro = {
        "chave": entrada["chave"],
        "etapa": entrada["etapa"],
        "caminho": str(caminho),
        "obrigatorio": bool(entrada["obrigatorio"]),
        "status_arquivo": status,
        "linhas": int(len(df)),
        "colunas": int(df.shape[1]) if isinstance(df, pd.DataFrame) else 0,
        "finalidade": entrada["finalidade"],
    }
    return df, registro


def obter_coluna_existente(df, candidatas):
    for coluna in candidatas:
        if coluna in df.columns:
            return coluna
    return None


def selecionar_colunas_existentes(df, colunas):
    return [coluna for coluna in colunas if coluna in df.columns]


def normalizar_nome_estrategia(texto):
    texto_norm = "" if pd.isna(texto) else str(texto).strip().lower()
    texto_norm = texto_norm.replace("ç", "c").replace("ã", "a").replace("á", "a").replace("é", "e")
    return texto_norm


def classificar_familia_estrategia(texto):
    texto_norm = normalizar_nome_estrategia(texto)
    if "capitulacao" in texto_norm:
        return "capitulacao"
    if "euforia" in texto_norm:
        return "euforia"
    if "ibovespa" in texto_norm or "ibov" in texto_norm:
        return "benchmark_ibovespa"
    if "cdi" in texto_norm:
        return "benchmark_cdi"
    if "aleatoria" in texto_norm or "aleatorio" in texto_norm:
        return "controle_aleatorio"
    if "aportes mensais" in texto_norm or "mensal" in texto_norm:
        return "controle_mensal"
    return "outros"


def adicionar_metadados_tabela(df, nome_tabela, tema, fonte):
    df_saida = df.copy()
    df_saida.insert(0, "nome_tabela_html", nome_tabela)
    df_saida.insert(1, "tema_tabela_html", tema)
    df_saida.insert(2, "fonte_analitica", fonte)
    return df_saida


def empilhar_tabelas(dicionario_tabelas, tema):
    registros = []
    for nome_tabela, df in dicionario_tabelas.items():
        if isinstance(df, pd.DataFrame) and len(df) > 0:
            temp = df.copy()
            temp.insert(0, "nome_tabela_origem", nome_tabela)
            temp.insert(1, "tema_tabela_html", tema)
            registros.append(temp)
    if len(registros) == 0:
        return pd.DataFrame()
    return pd.concat(registros, ignore_index=True, sort=False)


def converter_colunas_numericas_seguras(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if coluna.startswith("pct_") or coluna.startswith("score_") or coluna.startswith("n_"):
            df_saida[coluna] = pd.to_numeric(df_saida[coluna], errors="ignore")
    return df_saida


print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Carga das entradas determinísticas
# ============================================================

print("\n[5/12] Carga das entradas determinísticas...")

DADOS_ENTRADA = {}
REGISTROS_CATALOGO_ENTRADAS = []

for entrada in ENTRADAS_15_1:
    df_carregado, registro_catalogo = carregar_parquet_entrada(entrada)
    DADOS_ENTRADA[entrada["chave"]] = df_carregado
    REGISTROS_CATALOGO_ENTRADAS.append(registro_catalogo)

df_catalogo_entradas = pd.DataFrame(REGISTROS_CATALOGO_ENTRADAS)

arquivos_obrigatorios_ausentes = int(
    ((df_catalogo_entradas["obrigatorio"]) & (df_catalogo_entradas["status_arquivo"] != "carregado")).sum()
)

print(f"Entradas mapeadas                       : {len(df_catalogo_entradas):,}")
print(f"Entradas carregadas                     : {int((df_catalogo_entradas['status_arquivo'] == 'carregado').sum()):,}")
print(f"Entradas opcionais ausentes             : {int((~df_catalogo_entradas['obrigatorio'] & (df_catalogo_entradas['status_arquivo'] != 'carregado')).sum()):,}")
print(f"Entradas obrigatórias ausentes          : {arquivos_obrigatorios_ausentes:,}")

if arquivos_obrigatorios_ausentes > 0:
    print("\nEntradas obrigatórias ausentes:")
    print(
        df_catalogo_entradas.loc[
            (df_catalogo_entradas["obrigatorio"]) & (df_catalogo_entradas["status_arquivo"] != "carregado"),
            ["chave", "caminho"]
        ].to_string(index=False)
    )
    raise FileNotFoundError("Há arquivos obrigatórios ausentes para a Etapa 15.1.")

print("OK")

# ============================================================
# 6) Consolidação do resumo executivo para HTML
# ============================================================

print("\n[6/12] Consolidação do resumo executivo para HTML...")

df_sinais_conclusao_14_8 = DADOS_ENTRADA["sinais_conclusao_estatistica_14_8"].copy()
df_resumo_exec_14_8 = DADOS_ENTRADA["resumo_executivo_inferencia_14_8"].copy()
df_resumo_cap_euf_11_7 = DADOS_ENTRADA["resumo_capitulacao_vs_euforia_11_7"].copy()

colunas_sinais = selecionar_colunas_existentes(
    df_sinais_conclusao_14_8,
    [
        "estrategia_referencia",
        "frequencia_base_conclusao",
        "classificacao_vs_ibovespa",
        "classificacao_vs_cdi",
        "classificacao_vs_aleatorias",
        "classificacao_vs_mensais",
        "classificacao_vs_outra_estrategia",
        "score_vs_ibovespa",
        "score_vs_cdi",
        "score_vs_aleatorias",
        "score_vs_mensais",
        "score_vs_outra_estrategia",
        "n_testes_significativos_5pct_mensal",
        "n_confirmacoes_bootstrap_mensal",
        "sinal_robustez_benchmarks",
        "sinal_robustez_controles",
        "sinal_confirmacao_formal",
        "leitura_sintese",
    ],
)

df_resumo_executivo_html = df_sinais_conclusao_14_8[colunas_sinais].copy()

if len(df_resumo_exec_14_8) > 0:
    df_resumo_exec_estrategia = df_resumo_exec_14_8.loc[
        df_resumo_exec_14_8.get("nivel_resumo", pd.Series("", index=df_resumo_exec_14_8.index)).astype(str).eq("estrategia_mensal")
    ].copy()

    col_estrategia_resumo = obter_coluna_existente(df_resumo_exec_estrategia, ["estrategia_referencia"])
    if col_estrategia_resumo is not None and len(df_resumo_exec_estrategia) > 0 and len(df_resumo_executivo_html) > 0:
        colunas_merge_resumo = selecionar_colunas_existentes(
            df_resumo_exec_estrategia,
            [
                "estrategia_referencia",
                "score_evidencia_medio",
                "classificacao_evidencia_final",
                "leitura_final",
                "n_evidencias",
                "prob_bootstrap_media_acima_zero_media",
            ],
        )
        df_resumo_executivo_html = df_resumo_executivo_html.merge(
            df_resumo_exec_estrategia[colunas_merge_resumo],
            on="estrategia_referencia",
            how="left",
            suffixes=("", "_resumo_geral"),
        )

df_resumo_executivo_html["ordem_estrategia_html"] = df_resumo_executivo_html["estrategia_referencia"].map(
    {"capitulacao": 1, "euforia": 2}
).fillna(99).astype(int)

df_resumo_executivo_html = (
    df_resumo_executivo_html
    .sort_values(["ordem_estrategia_html", "estrategia_referencia"])
    .reset_index(drop=True)
)

print(f"Resumo executivo para HTML              : {len(df_resumo_executivo_html):,} linhas")
print("OK")

# ============================================================
# 7) Consolidação das métricas principais oficiais
# ============================================================

print("\n[7/12] Consolidação das métricas principais oficiais...")

df_metricas_11_3 = DADOS_ENTRADA["metricas_oficiais_11_3"].copy()

col_nome_estrategia = obter_coluna_existente(
    df_metricas_11_3,
    ["nome_exibicao_estrategia", "estrategia_nome", "estrategia", "chave_estrategia", "estrategia_id"],
)
col_frequencia = obter_coluna_existente(df_metricas_11_3, ["frequencia", "frequencia_retorno", "periodicidade"])

if col_nome_estrategia is None:
    raise KeyError("Não foi encontrada coluna de identificação da estratégia em 11_3_tbl_metricas_eficiencia_multifrequencia.parquet.")
if col_frequencia is None:
    raise KeyError("Não foi encontrada coluna de frequência em 11_3_tbl_metricas_eficiencia_multifrequencia.parquet.")

df_metricas_principais_html = df_metricas_11_3.copy()
df_metricas_principais_html["nome_estrategia_html"] = df_metricas_principais_html[col_nome_estrategia].astype(str)
df_metricas_principais_html["familia_estrategia_html"] = df_metricas_principais_html["nome_estrategia_html"].map(classificar_familia_estrategia)
df_metricas_principais_html["frequencia"] = df_metricas_principais_html[col_frequencia].astype(str)

colunas_metricas_prioritarias = [
    "nome_estrategia_html",
    "familia_estrategia_html",
    "frequencia",
    "retorno_acumulado_total",
    "retorno_anualizado",
    "volatilidade_anualizada",
    "downside_vol_anualizada",
    "max_drawdown_abs",
    "sharpe_anualizado",
    "sortino_anualizado",
    "calmar",
    "n_periodos",
    "data_inicio",
    "data_fim",
]

colunas_metricas_html = selecionar_colunas_existentes(df_metricas_principais_html, colunas_metricas_prioritarias)
colunas_adicionais = [
    coluna for coluna in df_metricas_principais_html.columns
    if coluna not in colunas_metricas_html
    and coluna in [
        "estrategia_id",
        "estrategia_referencia",
        "tipo_estrategia",
        "grupo_estrategia",
        "capital_inicial",
        "patrimonio_final",
        "cagr",
        "retorno_medio",
        "retorno_medio_anualizado",
        "volatilidade_periodica",
        "downside_vol_periodica",
        "pior_retorno_periodico",
        "melhor_retorno_periodico",
    ]
]

df_metricas_principais_html = df_metricas_principais_html[colunas_metricas_html + colunas_adicionais].copy()

ordem_frequencia = {"diaria": 1, "diário": 1, "mensal": 2, "anual": 3}
df_metricas_principais_html["ordem_frequencia_html"] = df_metricas_principais_html["frequencia"].map(ordem_frequencia).fillna(99).astype(int)
df_metricas_principais_html["flag_estrategia_real"] = df_metricas_principais_html["familia_estrategia_html"].isin(["capitulacao", "euforia"])

df_metricas_principais_html = (
    df_metricas_principais_html
    .sort_values(["ordem_frequencia_html", "flag_estrategia_real", "nome_estrategia_html"], ascending=[True, False, True])
    .reset_index(drop=True)
)

print(f"Métricas principais oficiais para HTML   : {len(df_metricas_principais_html):,} linhas")
print(f"Estratégias reais nas métricas           : {int(df_metricas_principais_html['flag_estrategia_real'].sum()):,}")
print("OK")

# ============================================================
# 8) Consolidação de benchmarks, controles e comparação direta
# ============================================================

print("\n[8/12] Consolidação de benchmarks, controles e comparação direta...")

df_comparacao_benchmarks_html = adicionar_metadados_tabela(
    DADOS_ENTRADA["resumo_benchmarks_14_2"],
    nome_tabela="comparacao_benchmarks_testes",
    tema="benchmarks",
    fonte="14.2",
)

tabelas_controles = {
    "aleatorios_pares_14_3": DADOS_ENTRADA["resumo_aleatorios_pares_14_3"],
    "aleatorios_grupos_14_3": DADOS_ENTRADA["resumo_aleatorios_grupos_14_3"],
    "mensais_pares_14_4": DADOS_ENTRADA["resumo_mensais_pares_14_4"],
    "mensais_grupos_14_4": DADOS_ENTRADA["resumo_mensais_grupos_14_4"],
    "posicionamento_aleatorias_13_6": DADOS_ENTRADA["posicionamento_aleatorias_13_6"],
    "posicionamento_mensais_13_7": DADOS_ENTRADA["posicionamento_mensais_13_7"],
}

df_comparacao_controles_html = empilhar_tabelas(tabelas_controles, tema="controles")

tabelas_comparacao_direta = {
    "resumo_descritivo_11_7": DADOS_ENTRADA["resumo_capitulacao_vs_euforia_11_7"],
    "ranking_descritivo_11_7": DADOS_ENTRADA["ranking_capitulacao_vs_euforia_11_7"],
    "testes_formais_14_5": DADOS_ENTRADA["resumo_capitulacao_vs_euforia_14_5"],
}

df_comparacao_direta_html = empilhar_tabelas(tabelas_comparacao_direta, tema="capitulacao_vs_euforia")

print(f"Comparação benchmarks para HTML          : {len(df_comparacao_benchmarks_html):,} linhas")
print(f"Comparação controles para HTML           : {len(df_comparacao_controles_html):,} linhas")
print(f"Comparação Cap. vs Euforia para HTML     : {len(df_comparacao_direta_html):,} linhas")
print("OK")

# ============================================================
# 9) Consolidação da inferência estatística e evidências
# ============================================================

print("\n[9/12] Consolidação da inferência estatística e evidências...")

tabelas_inferencia = {
    "consolidacao_evidencia_14_8": DADOS_ENTRADA["consolidacao_evidencia_14_8"],
    "resumo_executivo_inferencia_14_8": DADOS_ENTRADA["resumo_executivo_inferencia_14_8"],
    "sinais_conclusao_estatistica_14_8": DADOS_ENTRADA["sinais_conclusao_estatistica_14_8"],
    "posicionamento_estatistico_14_7": DADOS_ENTRADA["posicionamento_estatistico_14_7"],
    "percentis_empiricos_14_7": DADOS_ENTRADA["percentis_empiricos_14_7"],
    "intervalos_bootstrap_14_6": DADOS_ENTRADA["intervalos_bootstrap_14_6"],
    "resumo_bootstrap_14_6": DADOS_ENTRADA["resumo_bootstrap_14_6"],
}

df_inferencia_estatistica_html = empilhar_tabelas(tabelas_inferencia, tema="inferencia_estatistica")

print(f"Inferência estatística para HTML         : {len(df_inferencia_estatistica_html):,} linhas")
print("OK")

# ============================================================
# 10) Consolidação de robustez e sensibilidade
# ============================================================

print("\n[10/12] Consolidação de robustez e sensibilidade...")

tabelas_robustez = {
    "sensibilidade_limiares_capitulacao_13_4": DADOS_ENTRADA["resumo_sensibilidade_capitulacao_13_4"],
    "comparacao_limiar_capitulacao_13_4": DADOS_ENTRADA["comparacao_limiar_capitulacao_13_4"],
    "sensibilidade_limiares_euforia_13_5": DADOS_ENTRADA["resumo_sensibilidade_euforia_13_5"],
    "comparacao_limiar_euforia_13_5": DADOS_ENTRADA["comparacao_limiar_euforia_13_5"],
    "robustez_aleatorias_13_6": DADOS_ENTRADA["posicionamento_aleatorias_13_6"],
    "robustez_mensais_13_7": DADOS_ENTRADA["posicionamento_mensais_13_7"],
}

df_robustez_sensibilidade_html = empilhar_tabelas(tabelas_robustez, tema="robustez_sensibilidade")

print(f"Robustez e sensibilidade para HTML       : {len(df_robustez_sensibilidade_html):,} linhas")
print("OK")

# ============================================================
# 11) Catálogo final, parâmetros e auditoria
# ============================================================

print("\n[11/12] Catálogo final, parâmetros e auditoria...")

REGISTROS_OUTPUTS_15_1 = [
    {
        "arquivo": CAMINHO_RESUMO_EXECUTIVO_HTML.name,
        "tema": "resumo_executivo",
        "linhas": len(df_resumo_executivo_html),
        "colunas": df_resumo_executivo_html.shape[1],
        "uso_html": "Cards e tabela sintética de conclusão das estratégias.",
    },
    {
        "arquivo": CAMINHO_METRICAS_PRINCIPAIS_HTML.name,
        "tema": "metricas_principais",
        "linhas": len(df_metricas_principais_html),
        "colunas": df_metricas_principais_html.shape[1],
        "uso_html": "Tabela de métricas oficiais de retorno, risco e eficiência.",
    },
    {
        "arquivo": CAMINHO_COMPARACAO_BENCHMARKS_HTML.name,
        "tema": "benchmarks",
        "linhas": len(df_comparacao_benchmarks_html),
        "colunas": df_comparacao_benchmarks_html.shape[1],
        "uso_html": "Tabela de testes e evidências contra Ibovespa e CDI.",
    },
    {
        "arquivo": CAMINHO_COMPARACAO_CONTROLES_HTML.name,
        "tema": "controles",
        "linhas": len(df_comparacao_controles_html),
        "colunas": df_comparacao_controles_html.shape[1],
        "uso_html": "Tabela de comparação contra controles aleatórios e mensais.",
    },
    {
        "arquivo": CAMINHO_COMPARACAO_DIRETA_HTML.name,
        "tema": "capitulacao_vs_euforia",
        "linhas": len(df_comparacao_direta_html),
        "colunas": df_comparacao_direta_html.shape[1],
        "uso_html": "Tabela de comparação direta entre Capitulação e Euforia.",
    },
    {
        "arquivo": CAMINHO_INFERENCIA_ESTATISTICA_HTML.name,
        "tema": "inferencia_estatistica",
        "linhas": len(df_inferencia_estatistica_html),
        "colunas": df_inferencia_estatistica_html.shape[1],
        "uso_html": "Tabela consolidada de evidência estatística e bootstrap.",
    },
    {
        "arquivo": CAMINHO_ROBUSTEZ_SENSIBILIDADE_HTML.name,
        "tema": "robustez_sensibilidade",
        "linhas": len(df_robustez_sensibilidade_html),
        "colunas": df_robustez_sensibilidade_html.shape[1],
        "uso_html": "Tabela de robustez de limiares, réplicas aleatórias e controles mensais.",
    },
]

df_catalogo_outputs = pd.DataFrame(REGISTROS_OUTPUTS_15_1)

df_catalogo_tabelas_html = pd.concat(
    [
        df_catalogo_entradas.assign(tipo_registro="entrada")[["tipo_registro", "chave", "etapa", "caminho", "obrigatorio", "status_arquivo", "linhas", "colunas", "finalidade"]],
        df_catalogo_outputs.assign(
            tipo_registro="saida",
            chave=df_catalogo_outputs["arquivo"].str.replace(".parquet", "", regex=False),
            etapa=15,
            caminho=df_catalogo_outputs["arquivo"],
            obrigatorio=True,
            status_arquivo="gerado",
            finalidade=df_catalogo_outputs["uso_html"],
        )[["tipo_registro", "chave", "etapa", "caminho", "obrigatorio", "status_arquivo", "linhas", "colunas", "finalidade"]],
    ],
    ignore_index=True,
)

PARAMETROS_15_1 = {
    "etapa": "15",
    "subetapa": "15.1",
    "nome_subetapa": "Tabelas-Resumo das Estratégias",
    "objetivo": "Consolidar tabelas-resumo para consumo posterior pelo HTML final.",
    "tipo_processamento": "consolidacao_de_outputs_existentes",
    "recalculo_backtest": "nao",
    "frequencia_principal_inferencia": "mensal",
    "frequencias_preservadas": "diaria, mensal, anual",
    "metricas_oficiais": "retorno, volatilidade, drawdown, Sharpe oficial, Sortino oficial e Calmar oficial",
    "fontes_centrais": "Etapas 11, 13 e 14",
    "observacao_metodologica": "A subetapa amplia o escopo original da etapa final para incluir inferência estatística, comparação direta e robustez adicionadas ao longo do projeto.",
}

df_parametros = pd.DataFrame([{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_15_1.items()])

metricas_infinitas = 0
for df_verificacao in [
    df_resumo_executivo_html,
    df_metricas_principais_html,
    df_comparacao_benchmarks_html,
    df_comparacao_controles_html,
    df_comparacao_direta_html,
    df_inferencia_estatistica_html,
    df_robustez_sensibilidade_html,
]:
    if isinstance(df_verificacao, pd.DataFrame) and len(df_verificacao) > 0:
        numericas = df_verificacao.select_dtypes(include=[np.number])
        if numericas.shape[1] > 0:
            metricas_infinitas += int(np.isinf(numericas.to_numpy()).sum())

linhas_auditoria = [
    {
        "item": "arquivos_obrigatorios_ausentes",
        "valor": arquivos_obrigatorios_ausentes,
        "valor_referencia": "0",
        "status": "OK" if arquivos_obrigatorios_ausentes == 0 else "ERRO",
        "observacao": "Todos os arquivos obrigatórios da subetapa devem estar disponíveis.",
    },
    {
        "item": "linhas_resumo_executivo_html",
        "valor": len(df_resumo_executivo_html),
        "valor_referencia": ">= 2",
        "status": "OK" if len(df_resumo_executivo_html) >= 2 else "ERRO",
        "observacao": "Deve haver resumo executivo para Capitulação e Euforia.",
    },
    {
        "item": "linhas_metricas_principais_html",
        "valor": len(df_metricas_principais_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_metricas_principais_html) > 0 else "ERRO",
        "observacao": "A tabela de métricas oficiais deve ser gerada.",
    },
    {
        "item": "estrategias_reais_metricas",
        "valor": int(df_metricas_principais_html["flag_estrategia_real"].sum()) if "flag_estrategia_real" in df_metricas_principais_html.columns else 0,
        "valor_referencia": ">= 2",
        "status": "OK" if ("flag_estrategia_real" in df_metricas_principais_html.columns and int(df_metricas_principais_html["flag_estrategia_real"].sum()) >= 2) else "ERRO",
        "observacao": "A tabela de métricas deve conter Capitulação e Euforia.",
    },
    {
        "item": "linhas_comparacao_benchmarks_html",
        "valor": len(df_comparacao_benchmarks_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_comparacao_benchmarks_html) > 0 else "ERRO",
        "observacao": "A tabela de comparação contra benchmarks deve ser gerada.",
    },
    {
        "item": "linhas_comparacao_controles_html",
        "valor": len(df_comparacao_controles_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_comparacao_controles_html) > 0 else "ERRO",
        "observacao": "A tabela de comparação contra controles deve ser gerada.",
    },
    {
        "item": "linhas_comparacao_direta_html",
        "valor": len(df_comparacao_direta_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_comparacao_direta_html) > 0 else "ERRO",
        "observacao": "A tabela de comparação direta entre Capitulação e Euforia deve ser gerada.",
    },
    {
        "item": "linhas_inferencia_estatistica_html",
        "valor": len(df_inferencia_estatistica_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_inferencia_estatistica_html) > 0 else "ERRO",
        "observacao": "A tabela de inferência estatística deve ser gerada.",
    },
    {
        "item": "linhas_robustez_sensibilidade_html",
        "valor": len(df_robustez_sensibilidade_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_robustez_sensibilidade_html) > 0 else "ERRO",
        "observacao": "A tabela de robustez e sensibilidade deve ser gerada.",
    },
    {
        "item": "catalogo_tabelas_linhas",
        "valor": len(df_catalogo_tabelas_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_catalogo_tabelas_html) > 0 else "ERRO",
        "observacao": "O catálogo de entradas e saídas deve ser gerado.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As tabelas consolidadas não devem conter valores infinitos.",
    },
]

df_auditoria = pd.DataFrame(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Catálogo de tabelas HTML                 : {len(df_catalogo_tabelas_html):,} linhas")
print(f"Parâmetros metodológicos                 : {len(df_parametros):,} linhas")
print(f"Itens de auditoria                       : {len(df_auditoria):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 12) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[12/12] Salvamento dos outputs e validação final da subetapa...")

salvar_parquet_compat(df_resumo_executivo_html, CAMINHO_RESUMO_EXECUTIVO_HTML, index=False)
salvar_parquet_compat(df_metricas_principais_html, CAMINHO_METRICAS_PRINCIPAIS_HTML, index=False)
salvar_parquet_compat(df_comparacao_benchmarks_html, CAMINHO_COMPARACAO_BENCHMARKS_HTML, index=False)
salvar_parquet_compat(df_comparacao_controles_html, CAMINHO_COMPARACAO_CONTROLES_HTML, index=False)
salvar_parquet_compat(df_comparacao_direta_html, CAMINHO_COMPARACAO_DIRETA_HTML, index=False)
salvar_parquet_compat(df_inferencia_estatistica_html, CAMINHO_INFERENCIA_ESTATISTICA_HTML, index=False)
salvar_parquet_compat(df_robustez_sensibilidade_html, CAMINHO_ROBUSTEZ_SENSIBILIDADE_HTML, index=False)
salvar_parquet_compat(df_catalogo_tabelas_html, CAMINHO_CATALOGO_TABELAS_HTML, index=False)
salvar_parquet_compat(df_parametros, CAMINHO_PARAMETROS, index=False)
salvar_parquet_compat(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação das tabelas-resumo para HTML:")
print(df_auditoria.to_string(index=False))

print("\nResumo executivo para HTML:")
if len(df_resumo_executivo_html) > 0:
    print(df_resumo_executivo_html.head(20).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nCatálogo de outputs da Etapa 15.1:")
print(df_catalogo_outputs.to_string(index=False))

print("\nAmostra das métricas principais oficiais:")
if len(df_metricas_principais_html) > 0:
    colunas_print_metricas = selecionar_colunas_existentes(
        df_metricas_principais_html,
        [
            "nome_estrategia_html",
            "familia_estrategia_html",
            "frequencia",
            "retorno_acumulado_total",
            "retorno_anualizado",
            "volatilidade_anualizada",
            "max_drawdown_abs",
            "sharpe_anualizado",
            "sortino_anualizado",
            "calmar",
        ],
    )
    print(df_metricas_principais_html[colunas_print_metricas].head(30).to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nArquivos salvos na subetapa 15.1:")
print(f"- {CAMINHO_RESUMO_EXECUTIVO_HTML}")
print(f"- {CAMINHO_METRICAS_PRINCIPAIS_HTML}")
print(f"- {CAMINHO_COMPARACAO_BENCHMARKS_HTML}")
print(f"- {CAMINHO_COMPARACAO_CONTROLES_HTML}")
print(f"- {CAMINHO_COMPARACAO_DIRETA_HTML}")
print(f"- {CAMINHO_INFERENCIA_ESTATISTICA_HTML}")
print(f"- {CAMINHO_ROBUSTEZ_SENSIBILIDADE_HTML}")
print(f"- {CAMINHO_CATALOGO_TABELAS_HTML}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 15.1 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 15.1 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 15.1 - TABELAS-RESUMO DAS ESTRATÉGIAS

[1/12] Validação inicial do ambiente...
OK

[2/12] Definição determinística dos diretórios e caminhos de saída...
Diretório da Etapa 11 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11
Diretório da Etapa 12 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12
Diretório da Etapa 13 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_13
Diretório da Etapa 14 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14
Diretório da Etapa 15 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_15
Saída - resumo executivo HTML             : C:\Users\mht

## Etapa 15.2) Bases dos Gráficos de Performance

In [80]:
%%time
# ============================================================
# Etapa 15.2) Bases dos Gráficos de Performance
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 15.2 - BASES DOS GRÁFICOS DE PERFORMANCE")
print("=" * 100)

# Esta subetapa consolida bases analíticas de performance já produzidas ao longo do projeto
# para consumo posterior pela seção final do HTML. Não há recálculo pesado de backtest.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/13] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos diretórios e caminhos de saída
# ============================================================

print("\n[2/13] Definição determinística dos diretórios e caminhos de saída...")

DIR_RESULTADOS = Path(DIRETORIOS_PROJETO["resultados"])

for etapa in range(1, 16):
    chave_etapa = f"etapa_{etapa}"
    if chave_etapa not in DIRETORIOS_PROJETO:
        DIRETORIOS_PROJETO[chave_etapa] = DIR_RESULTADOS / chave_etapa
    Path(DIRETORIOS_PROJETO[chave_etapa]).mkdir(parents=True, exist_ok=True)

DIR_ETAPA_09 = Path(DIRETORIOS_PROJETO["etapa_9"])
DIR_ETAPA_11 = Path(DIRETORIOS_PROJETO["etapa_11"])
DIR_ETAPA_13 = Path(DIRETORIOS_PROJETO["etapa_13"])
DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
DIR_ETAPA_15 = Path(DIRETORIOS_PROJETO["etapa_15"])

DIR_ETAPA_15.mkdir(parents=True, exist_ok=True)

CAMINHO_CURVAS_PATRIMONIAIS_HTML = DIR_ETAPA_15 / "15_2_base_curvas_patrimoniais_html.parquet"
CAMINHO_CURVAS_INDEXADAS_HTML = DIR_ETAPA_15 / "15_2_base_curvas_indexadas_html.parquet"
CAMINHO_RETORNOS_PERIODICOS_HTML = DIR_ETAPA_15 / "15_2_base_retornos_periodicos_html.parquet"
CAMINHO_RETORNOS_ACUMULADOS_HTML = DIR_ETAPA_15 / "15_2_base_retornos_acumulados_html.parquet"
CAMINHO_DESEMPENHO_BENCHMARKS_HTML = DIR_ETAPA_15 / "15_2_base_desempenho_vs_benchmarks_html.parquet"
CAMINHO_DESEMPENHO_CONTROLES_HTML = DIR_ETAPA_15 / "15_2_base_desempenho_vs_controles_html.parquet"
CAMINHO_DESEMPENHO_DIRETO_HTML = DIR_ETAPA_15 / "15_2_base_desempenho_capitulacao_vs_euforia_html.parquet"
CAMINHO_DISTRIBUICAO_RETORNOS_HTML = DIR_ETAPA_15 / "15_2_tbl_distribuicao_retornos_html.parquet"
CAMINHO_RESUMO_GRAFICOS_HTML = DIR_ETAPA_15 / "15_2_tbl_resumo_graficos_performance_html.parquet"
CAMINHO_CATALOGO_GRAFICOS_HTML = DIR_ETAPA_15 / "15_2_tbl_catalogo_graficos_performance_html.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_15 / "15_2_tbl_parametros_graficos_performance_html.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_15 / "15_2_tbl_auditoria_validacao_graficos_performance_html.parquet"

print(f"Diretório da Etapa 09 : {DIR_ETAPA_09}")
print(f"Diretório da Etapa 11 : {DIR_ETAPA_11}")
print(f"Diretório da Etapa 13 : {DIR_ETAPA_13}")
print(f"Diretório da Etapa 14 : {DIR_ETAPA_14}")
print(f"Diretório da Etapa 15 : {DIR_ETAPA_15}")
print(f"Saída - curvas patrimoniais HTML         : {CAMINHO_CURVAS_PATRIMONIAIS_HTML}")
print(f"Saída - curvas indexadas HTML            : {CAMINHO_CURVAS_INDEXADAS_HTML}")
print(f"Saída - retornos periódicos HTML         : {CAMINHO_RETORNOS_PERIODICOS_HTML}")
print(f"Saída - retornos acumulados HTML         : {CAMINHO_RETORNOS_ACUMULADOS_HTML}")
print(f"Saída - desempenho vs benchmarks HTML    : {CAMINHO_DESEMPENHO_BENCHMARKS_HTML}")
print(f"Saída - desempenho vs controles HTML     : {CAMINHO_DESEMPENHO_CONTROLES_HTML}")
print(f"Saída - desempenho Cap. vs Euforia HTML  : {CAMINHO_DESEMPENHO_DIRETO_HTML}")
print(f"Saída - distribuição de retornos HTML    : {CAMINHO_DISTRIBUICAO_RETORNOS_HTML}")
print(f"Saída - resumo dos gráficos HTML         : {CAMINHO_RESUMO_GRAFICOS_HTML}")
print(f"Saída - catálogo de gráficos HTML        : {CAMINHO_CATALOGO_GRAFICOS_HTML}")
print(f"Saída - parâmetros                       : {CAMINHO_PARAMETROS}")
print(f"Saída - auditoria                        : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Catálogo determinístico de entradas da subetapa
# ============================================================

print("\n[3/13] Catálogo determinístico de entradas da subetapa...")

ENTRADAS_15_2 = [
    {
        "chave": "curvas_patrimoniais_9_7",
        "etapa": 9,
        "caminho": DIR_ETAPA_09 / "9_7_base_curvas_patrimoniais_consolidadas.parquet",
        "obrigatorio": True,
        "finalidade": "Base mestra diária de curvas patrimoniais consolidada na Etapa 9.7.",
    },
    {
        "chave": "retornos_diarios_11_1",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_1_base_retornos_diarios.parquet",
        "obrigatorio": True,
        "finalidade": "Base diária de retornos e patrimônio para gráficos de evolução diária.",
    },
    {
        "chave": "retornos_mensais_11_1",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_1_tbl_retornos_mensais.parquet",
        "obrigatorio": True,
        "finalidade": "Tabela mensal de retornos para barras, distribuições e curvas mensais acumuladas.",
    },
    {
        "chave": "retornos_anuais_11_1",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_1_tbl_retornos_anuais.parquet",
        "obrigatorio": True,
        "finalidade": "Tabela anual de retornos para barras anuais e leitura de longo prazo.",
    },
    {
        "chave": "resumo_rentabilidade_11_1",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_1_tbl_resumo_rentabilidade_multifrequencia.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo de rentabilidade multifrequência para anotações e ordenações do HTML.",
    },
    {
        "chave": "comparacao_benchmarks_diaria_11_5",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_diaria.parquet",
        "obrigatorio": True,
        "finalidade": "Base diária de excesso de retorno contra Ibovespa e CDI.",
    },
    {
        "chave": "comparacao_benchmarks_mensal_11_5",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_mensal.parquet",
        "obrigatorio": True,
        "finalidade": "Base mensal de excesso de retorno contra Ibovespa e CDI.",
    },
    {
        "chave": "comparacao_benchmarks_anual_11_5",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_5_base_comparacao_benchmarks_anual.parquet",
        "obrigatorio": True,
        "finalidade": "Base anual de excesso de retorno contra Ibovespa e CDI.",
    },
    {
        "chave": "comparacao_controles_diaria_11_6",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_6_base_comparacao_controles_diaria.parquet",
        "obrigatorio": True,
        "finalidade": "Base diária de comparação contra estratégias aleatórias e mensais.",
    },
    {
        "chave": "comparacao_controles_mensal_11_6",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_6_base_comparacao_controles_mensal.parquet",
        "obrigatorio": True,
        "finalidade": "Base mensal de comparação contra estratégias aleatórias e mensais.",
    },
    {
        "chave": "comparacao_controles_anual_11_6",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_6_base_comparacao_controles_anual.parquet",
        "obrigatorio": True,
        "finalidade": "Base anual de comparação contra estratégias aleatórias e mensais.",
    },
    {
        "chave": "capitulacao_vs_euforia_diaria_11_7",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_diaria.parquet",
        "obrigatorio": True,
        "finalidade": "Base diária pareada de comparação direta entre Capitulação e Euforia.",
    },
    {
        "chave": "capitulacao_vs_euforia_mensal_11_7",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_mensal.parquet",
        "obrigatorio": True,
        "finalidade": "Base mensal pareada de comparação direta entre Capitulação e Euforia.",
    },
    {
        "chave": "capitulacao_vs_euforia_anual_11_7",
        "etapa": 11,
        "caminho": DIR_ETAPA_11 / "11_7_base_comparacao_capitulacao_euforia_anual.parquet",
        "obrigatorio": True,
        "finalidade": "Base anual pareada de comparação direta entre Capitulação e Euforia.",
    },
    {
        "chave": "metricas_principais_html_15_1",
        "etapa": 15,
        "caminho": DIR_ETAPA_15 / "15_1_tbl_metricas_principais_estrategias_html.parquet",
        "obrigatorio": True,
        "finalidade": "Métricas oficiais já padronizadas para uso no HTML.",
    },
    {
        "chave": "resumo_executivo_html_15_1",
        "etapa": 15,
        "caminho": DIR_ETAPA_15 / "15_1_tbl_resumo_executivo_html.parquet",
        "obrigatorio": True,
        "finalidade": "Resumo executivo para integração das seções do HTML.",
    },
    {
        "chave": "base_grafico_bootstrap_14_6",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_6_tbl_base_grafico_bootstrap_excessos_retorno.parquet",
        "obrigatorio": False,
        "finalidade": "Base gráfica de intervalos de bootstrap que pode complementar gráficos de performance estatística.",
    },
    {
        "chave": "base_grafico_evidencia_14_8",
        "etapa": 14,
        "caminho": DIR_ETAPA_14 / "14_8_tbl_base_grafico_evidencia_estatistica.parquet",
        "obrigatorio": False,
        "finalidade": "Base gráfica de evidência estatística para eventual painel junto aos gráficos de performance.",
    },
]

for entrada in ENTRADAS_15_2:
    print(f"{entrada['chave']:<45} | obrigatório={str(entrada['obrigatorio']):<5} | {entrada['caminho']}")

print("OK")

# ============================================================
# 4) Funções auxiliares de carga, padronização e salvamento
# ============================================================

print("\n[4/13] Funções auxiliares de carga, padronização e salvamento...")

FREQUENCIAS_ANALISE = ["diaria", "mensal", "anual"]

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}

MAPA_LABEL_FREQUENCIA = {
    "diaria": "Diária",
    "mensal": "Mensal",
    "anual": "Anual",
}

MAPA_ORDEM_GRUPO_HTML = {
    "estrategia_real": 1,
    "benchmark": 2,
    "controle_mensal": 3,
    "controle_aleatorio": 4,
    "outros_controles": 5,
    "nao_classificado": 99,
}


def carregar_entrada(entrada):
    caminho = Path(entrada["caminho"])
    if caminho.exists():
        df = pd.read_parquet(caminho)
        return df, True
    if entrada["obrigatorio"]:
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado para a Etapa 15.2: {caminho}")
    return pd.DataFrame(), False


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def converter_data(serie):
    return pd.to_datetime(serie, errors="coerce")


def obter_serie(df, coluna, valor_padrao=np.nan):
    if coluna in df.columns:
        return df[coluna]
    return pd.Series(valor_padrao, index=df.index)


def obter_primeira_coluna(df, colunas, valor_padrao=np.nan):
    for coluna in colunas:
        if coluna in df.columns:
            return df[coluna]
    return pd.Series(valor_padrao, index=df.index)


def escolher_colunas(df, colunas):
    return [coluna for coluna in colunas if coluna in df.columns]


def preparar_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def salvar_parquet_compat(df, caminho, index=False):
    salvar_dataframe(preparar_para_parquet(df), caminho, index=index)


def normalizar_nome(valor):
    if pd.isna(valor):
        return ""
    return str(valor).strip()


def classificar_grupo_html(row):
    nome = normalizar_nome(row.get("nome_exibicao_estrategia", "")).lower()
    chave = normalizar_nome(row.get("chave_estrategia", "")).lower()
    familia = normalizar_nome(row.get("familia_estrategia", "")).lower()
    categoria = normalizar_nome(row.get("categoria_estrategia", "")).lower()
    subcategoria = normalizar_nome(row.get("subcategoria_estrategia", "")).lower()
    texto = " ".join([nome, chave, familia, categoria, subcategoria])

    if nome in ["capitulação", "capitulacao", "euforia"] or chave in ["capitulacao", "euforia"]:
        return "estrategia_real"
    if "ibovespa" in texto or "cdi" in texto or "benchmark" in texto:
        return "benchmark"
    if "aportes mensais" in texto or "mensal" in texto:
        return "controle_mensal"
    if "aleatória" in texto or "aleatoria" in texto or "random" in texto:
        return "controle_aleatorio"
    if "controle" in texto:
        return "outros_controles"
    return "nao_classificado"


def label_grupo_html(grupo):
    mapa = {
        "estrategia_real": "Estratégias Reais",
        "benchmark": "Benchmarks",
        "controle_mensal": "Controles Mensais",
        "controle_aleatorio": "Controles Aleatórios",
        "outros_controles": "Outros Controles",
        "nao_classificado": "Não Classificado",
    }
    return mapa.get(grupo, grupo)


def criar_periodo_html(df, frequencia):
    df_saida = df.copy()
    if frequencia == "diaria":
        if "data" in df_saida.columns:
            df_saida["data_periodo"] = converter_data(df_saida["data"])
            df_saida["periodo_html"] = df_saida["data_periodo"].dt.strftime("%Y-%m-%d")
        else:
            df_saida["data_periodo"] = pd.NaT
            df_saida["periodo_html"] = ""
    elif frequencia == "mensal":
        if "ano_mes" in df_saida.columns:
            df_saida["ano_mes"] = df_saida["ano_mes"].astype(str)
            df_saida["data_periodo"] = pd.to_datetime(df_saida["ano_mes"] + "-01", errors="coerce")
            df_saida["periodo_html"] = df_saida["ano_mes"]
        elif "data" in df_saida.columns:
            df_saida["data_periodo"] = converter_data(df_saida["data"])
            df_saida["periodo_html"] = df_saida["data_periodo"].dt.to_period("M").astype(str)
        else:
            df_saida["data_periodo"] = pd.NaT
            df_saida["periodo_html"] = ""
    else:
        if "ano" in df_saida.columns:
            df_saida["ano"] = pd.to_numeric(df_saida["ano"], errors="coerce").astype("Int64")
            df_saida["data_periodo"] = pd.to_datetime(df_saida["ano"].astype(str) + "-12-31", errors="coerce")
            df_saida["periodo_html"] = df_saida["ano"].astype(str)
        elif "data" in df_saida.columns:
            df_saida["data_periodo"] = converter_data(df_saida["data"])
            df_saida["periodo_html"] = df_saida["data_periodo"].dt.year.astype("Int64").astype(str)
        else:
            df_saida["data_periodo"] = pd.NaT
            df_saida["periodo_html"] = ""
    return df_saida


def padronizar_identificacao_estrategia(df):
    df_saida = df.copy()
    if "nome_exibicao_estrategia" not in df_saida.columns:
        if "nome_estrategia_html" in df_saida.columns:
            df_saida["nome_exibicao_estrategia"] = df_saida["nome_estrategia_html"]
        elif "estrategia_referencia" in df_saida.columns:
            df_saida["nome_exibicao_estrategia"] = df_saida["estrategia_referencia"]
        else:
            df_saida["nome_exibicao_estrategia"] = df_saida.get("chave_estrategia", "")
    if "chave_estrategia" not in df_saida.columns:
        if "chave_estrategia_real" in df_saida.columns:
            df_saida["chave_estrategia"] = df_saida["chave_estrategia_real"]
        elif "estrategia_referencia" in df_saida.columns:
            df_saida["chave_estrategia"] = df_saida["estrategia_referencia"]
        else:
            df_saida["chave_estrategia"] = df_saida["nome_exibicao_estrategia"].astype(str).str.lower().str.replace(" ", "_", regex=False)
    if "familia_estrategia" not in df_saida.columns:
        if "familia_estrategia_html" in df_saida.columns:
            df_saida["familia_estrategia"] = df_saida["familia_estrategia_html"]
        elif "estrategia_referencia" in df_saida.columns:
            df_saida["familia_estrategia"] = df_saida["estrategia_referencia"]
        else:
            df_saida["familia_estrategia"] = "nao_classificada"
    if "categoria_estrategia" not in df_saida.columns:
        if "familia_estrategia_html" in df_saida.columns:
            df_saida["categoria_estrategia"] = df_saida["familia_estrategia_html"]
        else:
            df_saida["categoria_estrategia"] = "nao_classificada"
    if "subcategoria_estrategia" not in df_saida.columns:
        df_saida["subcategoria_estrategia"] = "nao_classificada"
    if "ordem_exibicao" not in df_saida.columns:
        df_saida["ordem_exibicao"] = pd.factorize(df_saida["chave_estrategia"].astype(str))[0] + 1

    df_saida["grupo_html"] = df_saida.apply(classificar_grupo_html, axis=1)
    df_saida["label_grupo_html"] = df_saida["grupo_html"].map(label_grupo_html)
    df_saida["ordem_grupo_html"] = df_saida["grupo_html"].map(MAPA_ORDEM_GRUPO_HTML).fillna(99).astype(int)
    df_saida["flag_estrategia_real"] = df_saida["grupo_html"].eq("estrategia_real")
    return df_saida


def padronizar_curvas_patrimoniais(df_curvas):
    df = df_curvas.copy()
    df = padronizar_identificacao_estrategia(df)

    if "data" not in df.columns:
        raise KeyError("A base de curvas patrimoniais não possui a coluna obrigatória 'data'.")
    if "patrimonio_total" not in df.columns:
        raise KeyError("A base de curvas patrimoniais não possui a coluna obrigatória 'patrimonio_total'.")

    df["data"] = converter_data(df["data"])
    df["patrimonio_total"] = converter_numero(df["patrimonio_total"])

    if "retorno_diario" in df.columns:
        df["retorno_diario"] = converter_numero(df["retorno_diario"])
    else:
        df["retorno_diario"] = df.groupby("chave_estrategia", dropna=False)["patrimonio_total"].pct_change()

    if "flag_data_ativa_backtest" in df.columns:
        df["flag_data_ativa_backtest"] = df["flag_data_ativa_backtest"].fillna(False).astype(bool)
    else:
        df["flag_data_ativa_backtest"] = True

    df = df.loc[df["data"].notna()].copy()
    df = df.sort_values(["ordem_grupo_html", "ordem_exibicao", "chave_estrategia", "data"]).reset_index(drop=True)

    df_ativas = df.loc[df["flag_data_ativa_backtest"].astype(bool)].copy()
    if len(df_ativas) == 0:
        raise ValueError("A base de curvas patrimoniais não possui linhas ativas para o backtest.")

    primeiro_patrimonio = (
        df_ativas
        .sort_values(["chave_estrategia", "data"])
        .groupby("chave_estrategia", dropna=False)["patrimonio_total"]
        .transform("first")
    )

    df_ativas["patrimonio_base_100"] = np.where(
        primeiro_patrimonio > 0,
        df_ativas["patrimonio_total"] / primeiro_patrimonio * 100.0,
        np.nan,
    )
    df_ativas["retorno_acumulado_patrimonio"] = df_ativas["patrimonio_base_100"] / 100.0 - 1.0
    df_ativas["ano"] = df_ativas["data"].dt.year.astype(int)
    df_ativas["ano_mes"] = df_ativas["data"].dt.to_period("M").astype(str)
    df_ativas["frequencia"] = "diaria"
    df_ativas["label_frequencia"] = MAPA_LABEL_FREQUENCIA["diaria"]
    df_ativas["ordem_frequencia"] = MAPA_ORDEM_FREQUENCIA["diaria"]

    return df_ativas.reset_index(drop=True)


def padronizar_retornos_periodicos(df_base, frequencia):
    if len(df_base) == 0:
        return pd.DataFrame()

    df = df_base.copy()
    df = padronizar_identificacao_estrategia(df)
    df = criar_periodo_html(df, frequencia)

    if "retorno_periodo" not in df.columns:
        if "retorno_diario" in df.columns:
            df["retorno_periodo"] = df["retorno_diario"]
        else:
            df["retorno_periodo"] = np.nan

    df["retorno_periodo"] = converter_numero(df["retorno_periodo"])

    # A base acumulada para gráficos do HTML é recalculada a partir dos retornos
    # periódicos padronizados da própria subetapa. Isso evita desalinhamento visual
    # quando a base de origem possui uma coluna acumulada calculada em convenção
    # diferente da curva indexada principal.
    if "retorno_acumulado_ate_periodo" in df.columns:
        df["retorno_acumulado_ate_periodo_origem"] = converter_numero(df["retorno_acumulado_ate_periodo"])
    elif "retorno_acumulado" in df.columns:
        df["retorno_acumulado_ate_periodo_origem"] = converter_numero(df["retorno_acumulado"])
    else:
        df["retorno_acumulado_ate_periodo_origem"] = np.nan

    df = df.sort_values(["chave_estrategia", "data_periodo"]).copy()

    fator_acumulado = (
        df
        .groupby("chave_estrategia", dropna=False)["retorno_periodo"]
        .transform(lambda x: (1.0 + x.fillna(0.0)).cumprod())
    )

    primeiro_fator_acumulado = (
        fator_acumulado
        .groupby(df["chave_estrategia"], dropna=False)
        .transform("first")
    )

    df["retorno_acumulado_ate_periodo"] = np.where(
        primeiro_fator_acumulado > 0,
        fator_acumulado / primeiro_fator_acumulado - 1.0,
        np.nan,
    )

    if "patrimonio_fim" in df.columns:
        df["patrimonio_fim_observado"] = converter_numero(df["patrimonio_fim"])
    elif "patrimonio_fim_observado" in df.columns:
        df["patrimonio_fim_observado"] = converter_numero(df["patrimonio_fim_observado"])
    elif "patrimonio_total" in df.columns:
        df["patrimonio_fim_observado"] = converter_numero(df["patrimonio_total"])
    else:
        df["patrimonio_fim_observado"] = np.nan

    df["frequencia"] = frequencia
    df["label_frequencia"] = MAPA_LABEL_FREQUENCIA.get(frequencia, frequencia)
    df["ordem_frequencia"] = MAPA_ORDEM_FREQUENCIA.get(frequencia, 99)
    df["retorno_percentual"] = df["retorno_periodo"] * 100.0
    df["retorno_acumulado_base_100"] = (1.0 + df["retorno_acumulado_ate_periodo"]) * 100.0

    colunas_saida = escolher_colunas(
        df,
        [
            "frequencia",
            "label_frequencia",
            "ordem_frequencia",
            "data_periodo",
            "periodo_html",
            "ano",
            "ano_mes",
            "chave_estrategia",
            "estrategia_id",
            "nome_exibicao_estrategia",
            "familia_estrategia",
            "categoria_estrategia",
            "subcategoria_estrategia",
            "grupo_html",
            "label_grupo_html",
            "ordem_grupo_html",
            "ordem_exibicao",
            "flag_estrategia_real",
            "retorno_periodo",
            "retorno_percentual",
            "retorno_acumulado_ate_periodo",
            "retorno_acumulado_base_100",
            "retorno_acumulado_ate_periodo_origem",
            "patrimonio_fim_observado",
        ],
    )

    return df[colunas_saida].sort_values(["ordem_frequencia", "ordem_grupo_html", "ordem_exibicao", "chave_estrategia", "data_periodo"]).reset_index(drop=True)


def construir_base_benchmarks(df_base, frequencia):
    if len(df_base) == 0:
        return pd.DataFrame()

    df = df_base.copy()
    df = padronizar_identificacao_estrategia(df)
    df = criar_periodo_html(df, frequencia)

    if "retorno_periodo" not in df.columns and "retorno_diario" in df.columns:
        df["retorno_periodo"] = df["retorno_diario"]

    registros = []
    definicoes = [
        {
            "benchmark": "ibovespa",
            "nome": "Ibovespa Buy and Hold",
            "col_retorno": "retorno_periodo_ibovespa",
            "col_excesso": "retorno_excesso_vs_ibovespa",
            "col_relativo": "retorno_relativo_composto_vs_ibovespa",
            "col_superou": "flag_superou_ibovespa",
        },
        {
            "benchmark": "cdi",
            "nome": "CDI-Only",
            "col_retorno": "retorno_periodo_cdi",
            "col_excesso": "retorno_excesso_vs_cdi",
            "col_relativo": "retorno_relativo_composto_vs_cdi",
            "col_superou": "flag_superou_cdi",
        },
    ]

    for definicao in definicoes:
        if definicao["col_retorno"] not in df.columns:
            continue
        base = pd.DataFrame(index=df.index)
        base["frequencia"] = frequencia
        base["label_frequencia"] = MAPA_LABEL_FREQUENCIA.get(frequencia, frequencia)
        base["ordem_frequencia"] = MAPA_ORDEM_FREQUENCIA.get(frequencia, 99)
        base["data_periodo"] = df["data_periodo"]
        base["periodo_html"] = df["periodo_html"]
        base["ano"] = obter_serie(df, "ano", np.nan)
        base["ano_mes"] = obter_serie(df, "ano_mes", "")
        base["chave_estrategia"] = df["chave_estrategia"]
        base["nome_exibicao_estrategia"] = df["nome_exibicao_estrategia"]
        base["familia_estrategia"] = df["familia_estrategia"]
        base["grupo_html"] = df["grupo_html"]
        base["benchmark"] = definicao["benchmark"]
        base["nome_exibicao_benchmark"] = definicao["nome"]
        base["retorno_periodo_estrategia"] = converter_numero(obter_serie(df, "retorno_periodo", np.nan))
        base["retorno_periodo_comparador"] = converter_numero(df[definicao["col_retorno"]])
        if definicao["col_excesso"] in df.columns:
            base["retorno_excesso"] = converter_numero(df[definicao["col_excesso"]])
        else:
            base["retorno_excesso"] = base["retorno_periodo_estrategia"] - base["retorno_periodo_comparador"]
        if definicao["col_relativo"] in df.columns:
            base["retorno_relativo_composto"] = converter_numero(df[definicao["col_relativo"]])
        else:
            base["retorno_relativo_composto"] = np.where(
                (1.0 + base["retorno_periodo_comparador"]) != 0,
                (1.0 + base["retorno_periodo_estrategia"]) / (1.0 + base["retorno_periodo_comparador"]) - 1.0,
                np.nan,
            )
        if definicao["col_superou"] in df.columns:
            base["flag_estrategia_superou_benchmark"] = df[definicao["col_superou"]].fillna(False).astype(bool)
        else:
            base["flag_estrategia_superou_benchmark"] = base["retorno_excesso"] > 0.0
        base["retorno_excesso_acumulado"] = (
            base
            .sort_values(["chave_estrategia", "benchmark", "data_periodo"])
            .groupby(["chave_estrategia", "benchmark"], dropna=False)["retorno_excesso"]
            .transform(lambda x: x.fillna(0.0).cumsum())
        )
        registros.append(base)

    if len(registros) == 0:
        return pd.DataFrame()

    return pd.concat(registros, ignore_index=True).sort_values(["ordem_frequencia", "chave_estrategia", "benchmark", "data_periodo"]).reset_index(drop=True)


def construir_base_controles(df_base, frequencia):
    if len(df_base) == 0:
        return pd.DataFrame()

    df = df_base.copy()
    df = criar_periodo_html(df, frequencia)

    df["retorno_periodo_real"] = converter_numero(obter_serie(df, "retorno_periodo_real", np.nan))
    df["retorno_periodo_controle"] = converter_numero(obter_serie(df, "retorno_periodo_controle", np.nan))

    if "retorno_excesso_real_vs_controle" not in df.columns:
        df["retorno_excesso_real_vs_controle"] = df["retorno_periodo_real"] - df["retorno_periodo_controle"]
    else:
        df["retorno_excesso_real_vs_controle"] = converter_numero(df["retorno_excesso_real_vs_controle"])

    if "retorno_relativo_composto_real_vs_controle" not in df.columns:
        df["retorno_relativo_composto_real_vs_controle"] = np.where(
            (1.0 + df["retorno_periodo_controle"]) != 0,
            (1.0 + df["retorno_periodo_real"]) / (1.0 + df["retorno_periodo_controle"]) - 1.0,
            np.nan,
        )
    else:
        df["retorno_relativo_composto_real_vs_controle"] = converter_numero(df["retorno_relativo_composto_real_vs_controle"])

    if "flag_real_superou_controle" in df.columns:
        df["flag_real_superou_controle"] = df["flag_real_superou_controle"].fillna(False).astype(bool)
    else:
        df["flag_real_superou_controle"] = df["retorno_excesso_real_vs_controle"] > 0.0

    df["grupo_comparacao_controle"] = obter_serie(df, "grupo_comparacao_controle", obter_serie(df, "grupo_comparacao", "controle")).astype(str)
    df["tipo_comparacao_controle"] = obter_serie(df, "tipo_comparacao_controle", obter_serie(df, "tipo_comparacao", "controle")).astype(str)
    df["estrategia_referencia"] = obter_serie(df, "estrategia_referencia", obter_serie(df, "familia_estrategia_real", "")).astype(str)
    df["nome_exibicao_estrategia_real"] = obter_serie(df, "nome_exibicao_estrategia_real", df["estrategia_referencia"]).astype(str)
    df["nome_exibicao_estrategia_controle"] = obter_serie(df, "nome_exibicao_estrategia_controle", obter_serie(df, "nome_exibicao_comparador", "controle")).astype(str)
    df["chave_estrategia_real"] = obter_serie(df, "chave_estrategia_real", df["estrategia_referencia"]).astype(str)
    df["chave_estrategia_controle"] = obter_serie(df, "chave_estrategia_controle", df["nome_exibicao_estrategia_controle"]).astype(str)
    df["frequencia"] = frequencia
    df["label_frequencia"] = MAPA_LABEL_FREQUENCIA.get(frequencia, frequencia)
    df["ordem_frequencia"] = MAPA_ORDEM_FREQUENCIA.get(frequencia, 99)
    df["retorno_excesso_acumulado"] = (
        df
        .sort_values(["chave_estrategia_real", "chave_estrategia_controle", "data_periodo"])
        .groupby(["chave_estrategia_real", "chave_estrategia_controle"], dropna=False)["retorno_excesso_real_vs_controle"]
        .transform(lambda x: x.fillna(0.0).cumsum())
    )

    colunas_saida = escolher_colunas(
        df,
        [
            "frequencia",
            "label_frequencia",
            "ordem_frequencia",
            "data_periodo",
            "periodo_html",
            "ano",
            "ano_mes",
            "estrategia_referencia",
            "chave_estrategia_real",
            "nome_exibicao_estrategia_real",
            "chave_estrategia_controle",
            "nome_exibicao_estrategia_controle",
            "grupo_comparacao_controle",
            "tipo_comparacao_controle",
            "retorno_periodo_real",
            "retorno_periodo_controle",
            "retorno_excesso_real_vs_controle",
            "retorno_relativo_composto_real_vs_controle",
            "retorno_excesso_acumulado",
            "flag_real_superou_controle",
        ],
    )

    return df[colunas_saida].sort_values(["ordem_frequencia", "estrategia_referencia", "nome_exibicao_estrategia_controle", "data_periodo"]).reset_index(drop=True)


def construir_base_direta(df_base, frequencia):
    if len(df_base) == 0:
        return pd.DataFrame()

    df = df_base.copy()
    df = criar_periodo_html(df, frequencia)
    df["retorno_periodo_capitulacao"] = converter_numero(obter_serie(df, "retorno_periodo_capitulacao", np.nan))
    df["retorno_periodo_euforia"] = converter_numero(obter_serie(df, "retorno_periodo_euforia", np.nan))
    if "excesso_capitulacao_vs_euforia" not in df.columns:
        df["excesso_capitulacao_vs_euforia"] = df["retorno_periodo_capitulacao"] - df["retorno_periodo_euforia"]
    else:
        df["excesso_capitulacao_vs_euforia"] = converter_numero(df["excesso_capitulacao_vs_euforia"])
    if "excesso_euforia_vs_capitulacao" not in df.columns:
        df["excesso_euforia_vs_capitulacao"] = -df["excesso_capitulacao_vs_euforia"]
    else:
        df["excesso_euforia_vs_capitulacao"] = converter_numero(df["excesso_euforia_vs_capitulacao"])
    if "retorno_relativo_composto_capitulacao_vs_euforia" not in df.columns:
        df["retorno_relativo_composto_capitulacao_vs_euforia"] = np.where(
            (1.0 + df["retorno_periodo_euforia"]) != 0,
            (1.0 + df["retorno_periodo_capitulacao"]) / (1.0 + df["retorno_periodo_euforia"]) - 1.0,
            np.nan,
        )
    else:
        df["retorno_relativo_composto_capitulacao_vs_euforia"] = converter_numero(df["retorno_relativo_composto_capitulacao_vs_euforia"])

    df["flag_capitulacao_superou_euforia"] = obter_serie(df, "flag_capitulacao_superou_euforia", df["excesso_capitulacao_vs_euforia"] > 0.0).fillna(False).astype(bool)
    df["flag_euforia_superou_capitulacao"] = obter_serie(df, "flag_euforia_superou_capitulacao", df["excesso_capitulacao_vs_euforia"] < 0.0).fillna(False).astype(bool)
    df["frequencia"] = frequencia
    df["label_frequencia"] = MAPA_LABEL_FREQUENCIA.get(frequencia, frequencia)
    df["ordem_frequencia"] = MAPA_ORDEM_FREQUENCIA.get(frequencia, 99)
    df["excesso_capitulacao_vs_euforia_acumulado"] = df.sort_values("data_periodo")["excesso_capitulacao_vs_euforia"].fillna(0.0).cumsum()

    colunas_saida = escolher_colunas(
        df,
        [
            "frequencia",
            "label_frequencia",
            "ordem_frequencia",
            "data_periodo",
            "periodo_html",
            "ano",
            "ano_mes",
            "retorno_periodo_capitulacao",
            "retorno_periodo_euforia",
            "excesso_capitulacao_vs_euforia",
            "excesso_euforia_vs_capitulacao",
            "retorno_relativo_composto_capitulacao_vs_euforia",
            "excesso_capitulacao_vs_euforia_acumulado",
            "flag_capitulacao_superou_euforia",
            "flag_euforia_superou_capitulacao",
            "patrimonio_relativo_capitulacao",
            "patrimonio_relativo_euforia",
        ],
    )

    return df[colunas_saida].sort_values(["ordem_frequencia", "data_periodo"]).reset_index(drop=True)


def construir_distribuicao_retornos(df_retornos_periodicos):
    if len(df_retornos_periodicos) == 0:
        return pd.DataFrame()

    registros = []
    agrupadores = [
        "frequencia",
        "label_frequencia",
        "ordem_frequencia",
        "grupo_html",
        "label_grupo_html",
        "nome_exibicao_estrategia",
        "chave_estrategia",
    ]

    for chaves, grupo in df_retornos_periodicos.dropna(subset=["retorno_periodo"]).groupby(agrupadores, dropna=False):
        serie = converter_numero(grupo["retorno_periodo"]).dropna()
        if len(serie) == 0:
            continue
        registro = {coluna: valor for coluna, valor in zip(agrupadores, chaves)}
        registro.update(
            {
                "n_periodos": int(len(serie)),
                "retorno_medio": float(serie.mean()),
                "retorno_mediano": float(serie.median()),
                "retorno_desvio_padrao": float(serie.std(ddof=1)) if len(serie) > 1 else np.nan,
                "retorno_minimo": float(serie.min()),
                "retorno_p05": float(serie.quantile(0.05)),
                "retorno_p25": float(serie.quantile(0.25)),
                "retorno_p75": float(serie.quantile(0.75)),
                "retorno_p95": float(serie.quantile(0.95)),
                "retorno_maximo": float(serie.max()),
                "pct_periodos_positivos": float((serie > 0.0).mean()),
                "pct_periodos_negativos": float((serie < 0.0).mean()),
            }
        )
        registros.append(registro)

    if len(registros) == 0:
        return pd.DataFrame()

    return pd.DataFrame(registros).sort_values(["ordem_frequencia", "grupo_html", "nome_exibicao_estrategia"]).reset_index(drop=True)

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Carga das entradas determinísticas
# ============================================================

print("\n[5/13] Carga das entradas determinísticas...")

bases = {}
registros_catalogo_entradas = []
entradas_obrigatorias_ausentes = 0
entradas_opcionais_ausentes = 0

for entrada in ENTRADAS_15_2:
    df_entrada, carregado = carregar_entrada(entrada)
    bases[entrada["chave"]] = df_entrada
    if not carregado and entrada["obrigatorio"]:
        entradas_obrigatorias_ausentes += 1
    if not carregado and not entrada["obrigatorio"]:
        entradas_opcionais_ausentes += 1
    registros_catalogo_entradas.append(
        {
            "tipo_item": "entrada",
            "chave": entrada["chave"],
            "etapa_origem": entrada["etapa"],
            "arquivo": Path(entrada["caminho"]).name,
            "caminho": str(entrada["caminho"]),
            "obrigatorio": bool(entrada["obrigatorio"]),
            "carregado": bool(carregado),
            "linhas": int(len(df_entrada)),
            "colunas": int(df_entrada.shape[1]) if isinstance(df_entrada, pd.DataFrame) else 0,
            "finalidade": entrada["finalidade"],
        }
    )

print(f"Entradas mapeadas                       : {len(ENTRADAS_15_2):,}")
print(f"Entradas carregadas                     : {sum(1 for r in registros_catalogo_entradas if r['carregado']):,}")
print(f"Entradas opcionais ausentes             : {entradas_opcionais_ausentes:,}")
print(f"Entradas obrigatórias ausentes          : {entradas_obrigatorias_ausentes:,}")
print("OK")

# ============================================================
# 6) Consolidação das curvas patrimoniais diárias
# ============================================================

print("\n[6/13] Consolidação das curvas patrimoniais diárias...")

df_curvas_html = padronizar_curvas_patrimoniais(bases["curvas_patrimoniais_9_7"])

colunas_curvas_prioritarias = escolher_colunas(
    df_curvas_html,
    [
        "data",
        "ano",
        "ano_mes",
        "frequencia",
        "label_frequencia",
        "ordem_frequencia",
        "chave_estrategia",
        "estrategia_id",
        "nome_exibicao_estrategia",
        "familia_estrategia",
        "categoria_estrategia",
        "subcategoria_estrategia",
        "grupo_html",
        "label_grupo_html",
        "ordem_grupo_html",
        "ordem_exibicao",
        "flag_estrategia_real",
        "patrimonio_total",
        "patrimonio_base_100",
        "retorno_diario",
        "retorno_acumulado_patrimonio",
        "flag_data_ativa_backtest",
    ],
)

df_curvas_html = df_curvas_html[colunas_curvas_prioritarias].copy()

print(f"Curvas patrimoniais HTML                : {len(df_curvas_html):,} linhas")
print(f"Estratégias nas curvas                  : {df_curvas_html['chave_estrategia'].nunique():,}")
print(f"Data inicial                            : {df_curvas_html['data'].min().date()}")
print(f"Data final                              : {df_curvas_html['data'].max().date()}")
print("OK")

# ============================================================
# 7) Construção das curvas indexadas para gráficos de performance
# ============================================================

print("\n[7/13] Construção das curvas indexadas para gráficos de performance...")

df_curvas_indexadas_html = df_curvas_html.copy()
df_curvas_indexadas_html["metrica_grafico"] = "patrimonio_base_100"
df_curvas_indexadas_html["valor_grafico"] = converter_numero(df_curvas_indexadas_html["patrimonio_base_100"])
df_curvas_indexadas_html["label_valor_grafico"] = df_curvas_indexadas_html["valor_grafico"].map(lambda x: "" if pd.isna(x) else f"{x:.2f}")
df_curvas_indexadas_html["tipo_grafico_sugerido"] = "linha_curva_patrimonial_indexada"

df_curvas_indexadas_html = df_curvas_indexadas_html[
    escolher_colunas(
        df_curvas_indexadas_html,
        [
            "data",
            "ano",
            "ano_mes",
            "chave_estrategia",
            "nome_exibicao_estrategia",
            "familia_estrategia",
            "categoria_estrategia",
            "grupo_html",
            "label_grupo_html",
            "ordem_grupo_html",
            "ordem_exibicao",
            "flag_estrategia_real",
            "metrica_grafico",
            "valor_grafico",
            "label_valor_grafico",
            "tipo_grafico_sugerido",
        ],
    )
].copy()

print(f"Curvas indexadas HTML                   : {len(df_curvas_indexadas_html):,} linhas")
print("OK")

# ============================================================
# 8) Consolidação dos retornos periódicos e acumulados
# ============================================================

print("\n[8/13] Consolidação dos retornos periódicos e acumulados...")

df_retornos_diarios_html = padronizar_retornos_periodicos(bases["retornos_diarios_11_1"], "diaria")
df_retornos_mensais_html = padronizar_retornos_periodicos(bases["retornos_mensais_11_1"], "mensal")
df_retornos_anuais_html = padronizar_retornos_periodicos(bases["retornos_anuais_11_1"], "anual")

df_retornos_periodicos_html = pd.concat(
    [df_retornos_diarios_html, df_retornos_mensais_html, df_retornos_anuais_html],
    ignore_index=True,
)

df_retornos_acumulados_html = df_retornos_periodicos_html.copy()
df_retornos_acumulados_html["metrica_grafico"] = "retorno_acumulado_base_100"
df_retornos_acumulados_html["valor_grafico"] = converter_numero(df_retornos_acumulados_html["retorno_acumulado_base_100"])
df_retornos_acumulados_html["label_valor_grafico"] = df_retornos_acumulados_html["valor_grafico"].map(lambda x: "" if pd.isna(x) else f"{x:.2f}")
df_retornos_acumulados_html["tipo_grafico_sugerido"] = "linha_retorno_acumulado_multifrequencia"

df_retornos_acumulados_html = df_retornos_acumulados_html[
    escolher_colunas(
        df_retornos_acumulados_html,
        [
            "frequencia",
            "label_frequencia",
            "ordem_frequencia",
            "data_periodo",
            "periodo_html",
            "ano",
            "ano_mes",
            "chave_estrategia",
            "nome_exibicao_estrategia",
            "familia_estrategia",
            "categoria_estrategia",
            "grupo_html",
            "label_grupo_html",
            "ordem_grupo_html",
            "ordem_exibicao",
            "flag_estrategia_real",
            "retorno_periodo",
            "retorno_acumulado_ate_periodo",
            "retorno_acumulado_base_100",
            "retorno_acumulado_ate_periodo_origem",
            "metrica_grafico",
            "valor_grafico",
            "label_valor_grafico",
            "tipo_grafico_sugerido",
        ],
    )
].copy()

print(f"Retornos periódicos HTML                : {len(df_retornos_periodicos_html):,} linhas")
print(f"Retornos acumulados HTML                : {len(df_retornos_acumulados_html):,} linhas")
print("OK")

# ============================================================
# 9) Consolidação do desempenho contra benchmarks
# ============================================================

print("\n[9/13] Consolidação do desempenho contra benchmarks...")

df_benchmarks_html = pd.concat(
    [
        construir_base_benchmarks(bases["comparacao_benchmarks_diaria_11_5"], "diaria"),
        construir_base_benchmarks(bases["comparacao_benchmarks_mensal_11_5"], "mensal"),
        construir_base_benchmarks(bases["comparacao_benchmarks_anual_11_5"], "anual"),
    ],
    ignore_index=True,
)

print(f"Desempenho vs benchmarks HTML           : {len(df_benchmarks_html):,} linhas")
print(f"Benchmarks identificados                : {df_benchmarks_html['benchmark'].nunique() if len(df_benchmarks_html) > 0 else 0:,}")
print("OK")

# ============================================================
# 10) Consolidação do desempenho contra controles
# ============================================================

print("\n[10/13] Consolidação do desempenho contra controles...")

df_controles_html = pd.concat(
    [
        construir_base_controles(bases["comparacao_controles_diaria_11_6"], "diaria"),
        construir_base_controles(bases["comparacao_controles_mensal_11_6"], "mensal"),
        construir_base_controles(bases["comparacao_controles_anual_11_6"], "anual"),
    ],
    ignore_index=True,
)

print(f"Desempenho vs controles HTML            : {len(df_controles_html):,} linhas")
print(f"Controles identificados                 : {df_controles_html['chave_estrategia_controle'].nunique() if len(df_controles_html) > 0 else 0:,}")
print("OK")

# ============================================================
# 11) Consolidação da comparação direta Capitulação vs Euforia
# ============================================================

print("\n[11/13] Consolidação da comparação direta Capitulação vs Euforia...")

df_direto_html = pd.concat(
    [
        construir_base_direta(bases["capitulacao_vs_euforia_diaria_11_7"], "diaria"),
        construir_base_direta(bases["capitulacao_vs_euforia_mensal_11_7"], "mensal"),
        construir_base_direta(bases["capitulacao_vs_euforia_anual_11_7"], "anual"),
    ],
    ignore_index=True,
)

print(f"Desempenho Cap. vs Euforia HTML         : {len(df_direto_html):,} linhas")
print("OK")

# ============================================================
# 12) Distribuições, resumo, catálogo, parâmetros e auditoria
# ============================================================

print("\n[12/13] Distribuições, resumo, catálogo, parâmetros e auditoria...")

df_distribuicao_retornos_html = construir_distribuicao_retornos(df_retornos_periodicos_html)

registros_resumo = []

if len(df_curvas_html) > 0:
    registros_resumo.append(
        {
            "tema": "curvas_patrimoniais",
            "descricao": "Curvas patrimoniais diárias das estratégias, benchmarks e controles.",
            "linhas": len(df_curvas_html),
            "frequencias": "diaria",
            "uso_html": "Gráfico de evolução patrimonial e filtros por família de estratégia.",
        }
    )

if len(df_retornos_periodicos_html) > 0:
    registros_resumo.append(
        {
            "tema": "retornos_periodicos",
            "descricao": "Retornos diários, mensais e anuais padronizados.",
            "linhas": len(df_retornos_periodicos_html),
            "frequencias": ", ".join(sorted(df_retornos_periodicos_html["frequencia"].dropna().astype(str).unique())),
            "uso_html": "Barras de retorno por período, distribuição de retornos e filtros multifrequência.",
        }
    )

if len(df_benchmarks_html) > 0:
    registros_resumo.append(
        {
            "tema": "desempenho_vs_benchmarks",
            "descricao": "Excessos de retorno contra Ibovespa e CDI.",
            "linhas": len(df_benchmarks_html),
            "frequencias": ", ".join(sorted(df_benchmarks_html["frequencia"].dropna().astype(str).unique())),
            "uso_html": "Gráficos de excesso de retorno e frequência de superação contra benchmarks.",
        }
    )

if len(df_controles_html) > 0:
    registros_resumo.append(
        {
            "tema": "desempenho_vs_controles",
            "descricao": "Excessos de retorno contra controles aleatórios e mensais.",
            "linhas": len(df_controles_html),
            "frequencias": ", ".join(sorted(df_controles_html["frequencia"].dropna().astype(str).unique())),
            "uso_html": "Gráficos de comparação das estratégias reais contra controles.",
        }
    )

if len(df_direto_html) > 0:
    registros_resumo.append(
        {
            "tema": "capitulacao_vs_euforia",
            "descricao": "Comparação direta dos retornos pareados entre Capitulação e Euforia.",
            "linhas": len(df_direto_html),
            "frequencias": ", ".join(sorted(df_direto_html["frequencia"].dropna().astype(str).unique())),
            "uso_html": "Gráficos de diferença de performance entre Capitulação e Euforia.",
        }
    )

if len(df_distribuicao_retornos_html) > 0:
    registros_resumo.append(
        {
            "tema": "distribuicao_retornos",
            "descricao": "Resumo estatístico das distribuições de retornos por estratégia e frequência.",
            "linhas": len(df_distribuicao_retornos_html),
            "frequencias": ", ".join(sorted(df_distribuicao_retornos_html["frequencia"].dropna().astype(str).unique())),
            "uso_html": "Boxplots, tabelas de percentis e histogramas de retornos.",
        }
    )

df_resumo_graficos_html = pd.DataFrame(registros_resumo)

SAIDAS_15_2 = [
    {
        "arquivo": CAMINHO_CURVAS_PATRIMONIAIS_HTML.name,
        "tema": "curvas_patrimoniais",
        "linhas": len(df_curvas_html),
        "colunas": df_curvas_html.shape[1],
        "uso_html": "Curvas diárias de patrimônio nominal e base 100.",
    },
    {
        "arquivo": CAMINHO_CURVAS_INDEXADAS_HTML.name,
        "tema": "curvas_indexadas",
        "linhas": len(df_curvas_indexadas_html),
        "colunas": df_curvas_indexadas_html.shape[1],
        "uso_html": "Gráfico principal de performance com patrimônio indexado a 100.",
    },
    {
        "arquivo": CAMINHO_RETORNOS_PERIODICOS_HTML.name,
        "tema": "retornos_periodicos",
        "linhas": len(df_retornos_periodicos_html),
        "colunas": df_retornos_periodicos_html.shape[1],
        "uso_html": "Barras e heatmaps de retornos diários, mensais e anuais.",
    },
    {
        "arquivo": CAMINHO_RETORNOS_ACUMULADOS_HTML.name,
        "tema": "retornos_acumulados",
        "linhas": len(df_retornos_acumulados_html),
        "colunas": df_retornos_acumulados_html.shape[1],
        "uso_html": "Curvas acumuladas em visão diária, mensal e anual.",
    },
    {
        "arquivo": CAMINHO_DESEMPENHO_BENCHMARKS_HTML.name,
        "tema": "benchmarks",
        "linhas": len(df_benchmarks_html),
        "colunas": df_benchmarks_html.shape[1],
        "uso_html": "Excesso de retorno e superação contra Ibovespa e CDI.",
    },
    {
        "arquivo": CAMINHO_DESEMPENHO_CONTROLES_HTML.name,
        "tema": "controles",
        "linhas": len(df_controles_html),
        "colunas": df_controles_html.shape[1],
        "uso_html": "Excesso de retorno e superação contra controles aleatórios e mensais.",
    },
    {
        "arquivo": CAMINHO_DESEMPENHO_DIRETO_HTML.name,
        "tema": "capitulacao_vs_euforia",
        "linhas": len(df_direto_html),
        "colunas": df_direto_html.shape[1],
        "uso_html": "Comparação direta de retorno entre Capitulação e Euforia.",
    },
    {
        "arquivo": CAMINHO_DISTRIBUICAO_RETORNOS_HTML.name,
        "tema": "distribuicao_retornos",
        "linhas": len(df_distribuicao_retornos_html),
        "colunas": df_distribuicao_retornos_html.shape[1],
        "uso_html": "Distribuições, percentis e frequência de retornos positivos/negativos.",
    },
    {
        "arquivo": CAMINHO_RESUMO_GRAFICOS_HTML.name,
        "tema": "resumo_graficos",
        "linhas": len(df_resumo_graficos_html),
        "colunas": df_resumo_graficos_html.shape[1],
        "uso_html": "Tabela de apoio para navegação do bloco de performance no HTML.",
    },
]

df_catalogo_graficos_html = pd.concat(
    [
        pd.DataFrame(registros_catalogo_entradas),
        pd.DataFrame(
            [
                {
                    "tipo_item": "saida",
                    "chave": item["tema"],
                    "etapa_origem": 15,
                    "arquivo": item["arquivo"],
                    "caminho": str(DIR_ETAPA_15 / item["arquivo"]),
                    "obrigatorio": True,
                    "carregado": True,
                    "linhas": item["linhas"],
                    "colunas": item["colunas"],
                    "finalidade": item["uso_html"],
                }
                for item in SAIDAS_15_2
            ]
        ),
    ],
    ignore_index=True,
)

PARAMETROS_15_2 = {
    "etapa": "15",
    "subetapa": "15.2",
    "nome_subetapa": "Bases dos Gráficos de Performance",
    "frequencias_preservadas": "diaria, mensal e anual",
    "fonte_curvas_patrimoniais": "9_7_base_curvas_patrimoniais_consolidadas.parquet",
    "fonte_retornos": "Etapa 11.1",
    "fonte_benchmarks": "Etapa 11.5",
    "fonte_controles": "Etapa 11.6",
    "fonte_comparacao_direta": "Etapa 11.7",
    "base_indexacao_curvas": "primeiro patrimônio ativo de cada estratégia igual a 100",
    "objetivo_html": "gerar bases prontas para gráficos sem recálculo pesado na seção final do HTML",
    "observacao_metodologica": "A subetapa organiza bases para visualização; não altera resultados das etapas anteriores.",
    "observacao_v3": "Retorno acumulado dos gráficos é recalculado a partir do retorno periódico padronizado para alinhar a primeira observação em base 100.",
    "observacao_v4": "A série acumulada é normalizada pelo primeiro fator acumulado de cada estratégia e frequência, preservando a curva patrimonial oficial sem deslocamento visual.",
}

df_parametros = pd.DataFrame([{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_15_2.items()])

def contar_valores_infinitos(df):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        return 0

    colunas_numericas = list(df.select_dtypes(include=[np.number]).columns)
    if len(colunas_numericas) == 0:
        return 0

    total_infinitos = 0
    for coluna in colunas_numericas:
        serie_numerica = pd.to_numeric(df[coluna], errors="coerce")
        valores = serie_numerica.to_numpy(dtype="float64", na_value=np.nan)
        total_infinitos += int(np.isinf(valores).sum())

    return total_infinitos


metricas_infinitas = 0
for df_verificacao in [
    df_curvas_html,
    df_curvas_indexadas_html,
    df_retornos_periodicos_html,
    df_retornos_acumulados_html,
    df_benchmarks_html,
    df_controles_html,
    df_direto_html,
    df_distribuicao_retornos_html,
]:
    metricas_infinitas += contar_valores_infinitos(df_verificacao)

n_estrategias_reais_curvas = int(
    df_curvas_html.loc[df_curvas_html["flag_estrategia_real"], "nome_exibicao_estrategia"].nunique()
) if len(df_curvas_html) > 0 and "flag_estrategia_real" in df_curvas_html.columns else 0

duplicatas_curvas = int(df_curvas_html.duplicated(["chave_estrategia", "data"]).sum()) if len(df_curvas_html) > 0 else 0

duplicatas_retornos = int(
    df_retornos_periodicos_html.duplicated(["chave_estrategia", "frequencia", "periodo_html"]).sum()
) if len(df_retornos_periodicos_html) > 0 else 0

if len(df_retornos_periodicos_html) > 0:
    primeiros_acumulados = (
        df_retornos_periodicos_html
        .sort_values(["chave_estrategia", "frequencia", "data_periodo"])
        .groupby(["chave_estrategia", "frequencia"], dropna=False)["retorno_acumulado_base_100"]
        .first()
    )
    desvios_primeiro_base100 = (primeiros_acumulados - 100.0).abs()
    primeiro_acumulado_base100_ok = int((desvios_primeiro_base100 <= 0.000001).all())
else:
    primeiro_acumulado_base100_ok = 0

linhas_auditoria = [
    {
        "item": "arquivos_obrigatorios_ausentes",
        "valor": entradas_obrigatorias_ausentes,
        "valor_referencia": "0",
        "status": "OK" if entradas_obrigatorias_ausentes == 0 else "ERRO",
        "observacao": "Todos os arquivos obrigatórios da subetapa devem estar disponíveis.",
    },
    {
        "item": "linhas_curvas_patrimoniais_html",
        "valor": len(df_curvas_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_curvas_html) > 0 else "ERRO",
        "observacao": "A base de curvas patrimoniais deve ser gerada.",
    },
    {
        "item": "estrategias_reais_curvas",
        "valor": n_estrategias_reais_curvas,
        "valor_referencia": ">= 2",
        "status": "OK" if n_estrategias_reais_curvas >= 2 else "ERRO",
        "observacao": "As curvas devem conter Capitulação e Euforia.",
    },
    {
        "item": "linhas_curvas_indexadas_html",
        "valor": len(df_curvas_indexadas_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_curvas_indexadas_html) > 0 else "ERRO",
        "observacao": "A base indexada deve ser gerada para o gráfico principal de performance.",
    },
    {
        "item": "linhas_retornos_periodicos_html",
        "valor": len(df_retornos_periodicos_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_retornos_periodicos_html) > 0 else "ERRO",
        "observacao": "A base de retornos periódicos deve ser gerada.",
    },
    {
        "item": "frequencias_retornos_periodicos",
        "valor": int(df_retornos_periodicos_html["frequencia"].nunique()) if len(df_retornos_periodicos_html) > 0 else 0,
        "valor_referencia": ">= 3",
        "status": "OK" if len(df_retornos_periodicos_html) > 0 and df_retornos_periodicos_html["frequencia"].nunique() >= 3 else "ERRO",
        "observacao": "A base de retornos deve preservar as visões diária, mensal e anual.",
    },
    {
        "item": "linhas_desempenho_benchmarks_html",
        "valor": len(df_benchmarks_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_benchmarks_html) > 0 else "ERRO",
        "observacao": "A base de desempenho contra benchmarks deve ser gerada.",
    },
    {
        "item": "benchmarks_identificados",
        "valor": int(df_benchmarks_html["benchmark"].nunique()) if len(df_benchmarks_html) > 0 else 0,
        "valor_referencia": ">= 2",
        "status": "OK" if len(df_benchmarks_html) > 0 and df_benchmarks_html["benchmark"].nunique() >= 2 else "ERRO",
        "observacao": "A base de benchmarks deve conter Ibovespa e CDI.",
    },
    {
        "item": "linhas_desempenho_controles_html",
        "valor": len(df_controles_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_controles_html) > 0 else "ERRO",
        "observacao": "A base de desempenho contra controles deve ser gerada.",
    },
    {
        "item": "linhas_desempenho_direto_html",
        "valor": len(df_direto_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_direto_html) > 0 else "ERRO",
        "observacao": "A comparação direta Capitulação vs Euforia deve ser preservada.",
    },
    {
        "item": "linhas_distribuicao_retornos_html",
        "valor": len(df_distribuicao_retornos_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_distribuicao_retornos_html) > 0 else "ERRO",
        "observacao": "A base de distribuição de retornos deve ser gerada.",
    },
    {
        "item": "duplicatas_curvas_estrategia_data",
        "valor": duplicatas_curvas,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_curvas == 0 else "ERRO",
        "observacao": "Cada estratégia deve ter no máximo uma linha por data na base de curvas.",
    },
    {
        "item": "duplicatas_retornos_estrategia_frequencia_periodo",
        "valor": duplicatas_retornos,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_retornos == 0 else "ERRO",
        "observacao": "Cada estratégia deve ter no máximo uma linha por frequência e período na base de retornos.",
    },
    {
        "item": "primeiro_retorno_acumulado_base100",
        "valor": primeiro_acumulado_base100_ok,
        "valor_referencia": "1",
        "status": "OK" if primeiro_acumulado_base100_ok == 1 else "ERRO",
        "observacao": "O primeiro ponto acumulado de cada estratégia e frequência deve estar alinhado em base 100.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As bases de gráficos não devem conter valores infinitos.",
    },
]

df_auditoria = pd.DataFrame(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Distribuição de retornos HTML           : {len(df_distribuicao_retornos_html):,} linhas")
print(f"Resumo dos gráficos de performance      : {len(df_resumo_graficos_html):,} linhas")
print(f"Catálogo de gráficos HTML               : {len(df_catalogo_graficos_html):,} linhas")
print(f"Parâmetros metodológicos                : {len(df_parametros):,} linhas")
print(f"Itens de auditoria                      : {len(df_auditoria):,}")
print(f"Erros bloqueantes                       : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 13) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[13/13] Salvamento dos outputs e validação final da subetapa...")

salvar_parquet_compat(df_curvas_html, CAMINHO_CURVAS_PATRIMONIAIS_HTML, index=False)
salvar_parquet_compat(df_curvas_indexadas_html, CAMINHO_CURVAS_INDEXADAS_HTML, index=False)
salvar_parquet_compat(df_retornos_periodicos_html, CAMINHO_RETORNOS_PERIODICOS_HTML, index=False)
salvar_parquet_compat(df_retornos_acumulados_html, CAMINHO_RETORNOS_ACUMULADOS_HTML, index=False)
salvar_parquet_compat(df_benchmarks_html, CAMINHO_DESEMPENHO_BENCHMARKS_HTML, index=False)
salvar_parquet_compat(df_controles_html, CAMINHO_DESEMPENHO_CONTROLES_HTML, index=False)
salvar_parquet_compat(df_direto_html, CAMINHO_DESEMPENHO_DIRETO_HTML, index=False)
salvar_parquet_compat(df_distribuicao_retornos_html, CAMINHO_DISTRIBUICAO_RETORNOS_HTML, index=False)
salvar_parquet_compat(df_resumo_graficos_html, CAMINHO_RESUMO_GRAFICOS_HTML, index=False)
salvar_parquet_compat(df_catalogo_graficos_html, CAMINHO_CATALOGO_GRAFICOS_HTML, index=False)
salvar_parquet_compat(df_parametros, CAMINHO_PARAMETROS, index=False)
salvar_parquet_compat(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação das bases dos gráficos de performance:")
print(df_auditoria.to_string(index=False))

print("\nResumo dos gráficos de performance para HTML:")
if len(df_resumo_graficos_html) > 0:
    print(df_resumo_graficos_html.to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nCatálogo de outputs da Etapa 15.2:")
df_print_catalogo = pd.DataFrame(SAIDAS_15_2)
print(df_print_catalogo.to_string(index=False))

print("\nAmostra das curvas indexadas:")
colunas_amostra_curvas = escolher_colunas(
    df_curvas_indexadas_html,
    [
        "data",
        "nome_exibicao_estrategia",
        "grupo_html",
        "valor_grafico",
        "tipo_grafico_sugerido",
    ],
)
print(df_curvas_indexadas_html[colunas_amostra_curvas].head(30).to_string(index=False))

print("\nAmostra dos retornos periódicos:")
colunas_amostra_retornos = escolher_colunas(
    df_retornos_periodicos_html,
    [
        "frequencia",
        "periodo_html",
        "nome_exibicao_estrategia",
        "grupo_html",
        "retorno_periodo",
        "retorno_acumulado_base_100",
    ],
)
print(df_retornos_periodicos_html[colunas_amostra_retornos].head(30).to_string(index=False))

print("\nArquivos salvos na subetapa 15.2:")
print(f"- {CAMINHO_CURVAS_PATRIMONIAIS_HTML}")
print(f"- {CAMINHO_CURVAS_INDEXADAS_HTML}")
print(f"- {CAMINHO_RETORNOS_PERIODICOS_HTML}")
print(f"- {CAMINHO_RETORNOS_ACUMULADOS_HTML}")
print(f"- {CAMINHO_DESEMPENHO_BENCHMARKS_HTML}")
print(f"- {CAMINHO_DESEMPENHO_CONTROLES_HTML}")
print(f"- {CAMINHO_DESEMPENHO_DIRETO_HTML}")
print(f"- {CAMINHO_DISTRIBUICAO_RETORNOS_HTML}")
print(f"- {CAMINHO_RESUMO_GRAFICOS_HTML}")
print(f"- {CAMINHO_CATALOGO_GRAFICOS_HTML}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 15.2 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 15.2 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 15.2 - BASES DOS GRÁFICOS DE PERFORMANCE

[1/13] Validação inicial do ambiente...
OK

[2/13] Definição determinística dos diretórios e caminhos de saída...
Diretório da Etapa 09 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_9
Diretório da Etapa 11 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11
Diretório da Etapa 13 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_13
Diretório da Etapa 14 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14
Diretório da Etapa 15 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_15
Saída - curvas patrimoniais HTML         : C:\Users\mh

## Etapa 15.3) Bases dos Gráficos de Risco

In [81]:
%%time
# ============================================================
# Etapa 15.3) Bases dos Gráficos de Risco
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 15.3 - BASES DOS GRÁFICOS DE RISCO")
print("=" * 100)

# Esta subetapa consolida bases analíticas de risco já produzidas ou deriváveis
# dos outputs finais das etapas anteriores para consumo posterior pela seção final do HTML.
# Não há recálculo pesado de backtest.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/13] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

print("OK")

# ============================================================
# 2) Definição determinística dos diretórios e caminhos de saída
# ============================================================

print("\n[2/13] Definição determinística dos diretórios e caminhos de saída...")

DIR_RESULTADOS = Path(DIRETORIOS_PROJETO["resultados"])

def obter_diretorio_etapa(numero_etapa):
    chave_sem_zero = f"etapa_{numero_etapa}"
    chave_com_zero = f"etapa_{numero_etapa:02d}"

    if chave_sem_zero in DIRETORIOS_PROJETO:
        return Path(DIRETORIOS_PROJETO[chave_sem_zero])
    if chave_com_zero in DIRETORIOS_PROJETO:
        return Path(DIRETORIOS_PROJETO[chave_com_zero])

    caminho_sem_zero = DIR_RESULTADOS / chave_sem_zero
    caminho_com_zero = DIR_RESULTADOS / chave_com_zero

    if caminho_sem_zero.exists():
        DIRETORIOS_PROJETO[chave_sem_zero] = caminho_sem_zero
        return caminho_sem_zero

    if caminho_com_zero.exists():
        DIRETORIOS_PROJETO[chave_sem_zero] = caminho_com_zero
        return caminho_com_zero

    DIRETORIOS_PROJETO[chave_sem_zero] = caminho_sem_zero
    caminho_sem_zero.mkdir(parents=True, exist_ok=True)
    return caminho_sem_zero

DIR_ETAPA_11 = obter_diretorio_etapa(11)
DIR_ETAPA_15 = obter_diretorio_etapa(15)
DIR_ETAPA_15.mkdir(parents=True, exist_ok=True)

CAMINHO_DRAWDOWN_HTML = DIR_ETAPA_15 / "15_3_base_drawdown_html.parquet"
CAMINHO_EVENTOS_DRAWDOWN_HTML = DIR_ETAPA_15 / "15_3_tbl_eventos_drawdown_html.parquet"
CAMINHO_VOLATILIDADE_ROLLING_HTML = DIR_ETAPA_15 / "15_3_base_volatilidade_rolling_html.parquet"
CAMINHO_METRICAS_RISCO_HTML = DIR_ETAPA_15 / "15_3_base_metricas_risco_html.parquet"
CAMINHO_METRICAS_RISCO_LONG_HTML = DIR_ETAPA_15 / "15_3_base_metricas_risco_long_html.parquet"
CAMINHO_RISCO_BENCHMARKS_HTML = DIR_ETAPA_15 / "15_3_base_risco_vs_benchmarks_html.parquet"
CAMINHO_RISCO_CONTROLES_HTML = DIR_ETAPA_15 / "15_3_base_risco_vs_controles_html.parquet"
CAMINHO_RISCO_DIRETO_HTML = DIR_ETAPA_15 / "15_3_base_risco_capitulacao_vs_euforia_html.parquet"
CAMINHO_RESUMO_GRAFICOS_HTML = DIR_ETAPA_15 / "15_3_tbl_resumo_graficos_risco_html.parquet"
CAMINHO_CATALOGO_GRAFICOS_HTML = DIR_ETAPA_15 / "15_3_tbl_catalogo_graficos_risco_html.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_15 / "15_3_tbl_parametros_graficos_risco_html.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_15 / "15_3_tbl_auditoria_validacao_graficos_risco_html.parquet"

print(f"Diretório da Etapa 11 : {DIR_ETAPA_11}")
print(f"Diretório da Etapa 15 : {DIR_ETAPA_15}")
print(f"Saída - drawdown HTML                  : {CAMINHO_DRAWDOWN_HTML}")
print(f"Saída - eventos de drawdown HTML       : {CAMINHO_EVENTOS_DRAWDOWN_HTML}")
print(f"Saída - volatilidade rolling HTML      : {CAMINHO_VOLATILIDADE_ROLLING_HTML}")
print(f"Saída - métricas de risco HTML         : {CAMINHO_METRICAS_RISCO_HTML}")
print(f"Saída - métricas de risco long HTML    : {CAMINHO_METRICAS_RISCO_LONG_HTML}")
print(f"Saída - risco vs benchmarks HTML       : {CAMINHO_RISCO_BENCHMARKS_HTML}")
print(f"Saída - risco vs controles HTML        : {CAMINHO_RISCO_CONTROLES_HTML}")
print(f"Saída - risco Cap. vs Euforia HTML     : {CAMINHO_RISCO_DIRETO_HTML}")
print(f"Saída - resumo dos gráficos HTML       : {CAMINHO_RESUMO_GRAFICOS_HTML}")
print(f"Saída - catálogo de gráficos HTML      : {CAMINHO_CATALOGO_GRAFICOS_HTML}")
print(f"Saída - parâmetros                     : {CAMINHO_PARAMETROS}")
print(f"Saída - auditoria                      : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Catálogo determinístico de entradas da subetapa
# ============================================================

print("\n[3/13] Catálogo determinístico de entradas da subetapa...")

ENTRADAS_15_3 = [
    {
        "chave": "curvas_patrimoniais_html_15_2",
        "caminho": DIR_ETAPA_15 / "15_2_base_curvas_patrimoniais_html.parquet",
        "obrigatorio": True,
        "finalidade": "Curvas patrimoniais diárias para construção da base de drawdown.",
    },
    {
        "chave": "retornos_periodicos_html_15_2",
        "caminho": DIR_ETAPA_15 / "15_2_base_retornos_periodicos_html.parquet",
        "obrigatorio": True,
        "finalidade": "Retornos diários, mensais e anuais para volatilidade rolling e estatísticas de risco.",
    },
    {
        "chave": "metricas_principais_html_15_1",
        "caminho": DIR_ETAPA_15 / "15_1_tbl_metricas_principais_estrategias_html.parquet",
        "obrigatorio": True,
        "finalidade": "Métricas oficiais de risco e eficiência já padronizadas para o HTML.",
    },
    {
        "chave": "comparacao_benchmarks_html_15_1",
        "caminho": DIR_ETAPA_15 / "15_1_tbl_comparacao_benchmarks_html.parquet",
        "obrigatorio": False,
        "finalidade": "Tabela final de comparação com Ibovespa e CDI para recortes de risco relativo.",
    },
    {
        "chave": "comparacao_controles_html_15_1",
        "caminho": DIR_ETAPA_15 / "15_1_tbl_comparacao_controles_html.parquet",
        "obrigatorio": False,
        "finalidade": "Tabela final de comparação com controles aleatórios e mensais para recortes de risco relativo.",
    },
    {
        "chave": "comparacao_direta_html_15_1",
        "caminho": DIR_ETAPA_15 / "15_1_tbl_comparacao_capitulacao_euforia_html.parquet",
        "obrigatorio": False,
        "finalidade": "Tabela final de comparação direta entre Capitulação e Euforia.",
    },
    {
        "chave": "drawdown_origem_11_4",
        "caminho": DIR_ETAPA_11 / "11_4_base_drawdown_diario.parquet",
        "obrigatorio": False,
        "padroes_alternativos": ["11_4*drawdown*.parquet", "11_4*recuper*.parquet"],
        "finalidade": "Bases originais de drawdown e recuperação, quando disponíveis.",
    },
    {
        "chave": "risco_origem_11_2",
        "caminho": DIR_ETAPA_11 / "11_2_tbl_metricas_risco_multifrequencia.parquet",
        "obrigatorio": False,
        "padroes_alternativos": ["11_2*risco*.parquet", "11_2*volatil*.parquet"],
        "finalidade": "Tabelas originais de risco e volatilidade da Etapa 11.2, quando disponíveis.",
    },
    {
        "chave": "eficiencia_origem_11_3",
        "caminho": DIR_ETAPA_11 / "11_3_tbl_metricas_eficiencia_multifrequencia.parquet",
        "obrigatorio": False,
        "padroes_alternativos": ["11_3*eficiencia*.parquet", "11_3*metricas*.parquet"],
        "finalidade": "Métricas oficiais de Sharpe, Sortino e Calmar da Etapa 11.3, quando disponíveis.",
    },
]

def localizar_arquivo_entrada(entrada):
    caminho = Path(entrada["caminho"])
    if caminho.exists():
        return caminho

    for padrao in entrada.get("padroes_alternativos", []):
        candidatos = sorted(caminho.parent.glob(padrao))
        if len(candidatos) > 0:
            return candidatos[0]

    return caminho

for entrada in ENTRADAS_15_3:
    caminho_localizado = localizar_arquivo_entrada(entrada)
    entrada["caminho_localizado"] = caminho_localizado
    entrada["existe"] = caminho_localizado.exists()
    print(f"{entrada['chave']:<38} | obrigatório={str(entrada['obrigatorio']):<5} | existe={str(entrada['existe']):<5} | {caminho_localizado}")

print("OK")

# ============================================================
# 4) Funções auxiliares de carga, padronização e salvamento
# ============================================================

print("\n[4/13] Funções auxiliares de carga, padronização e salvamento...")

MAPA_ORDEM_FREQUENCIA = {
    "diaria": 1,
    "mensal": 2,
    "anual": 3,
}

MAPA_LABEL_FREQUENCIA = {
    "diaria": "Diária",
    "mensal": "Mensal",
    "anual": "Anual",
}

MAPA_JANELAS_VOLATILIDADE = {
    "diaria": [
        {"janela": 21, "label": "21 pregões", "fator_anualizacao": 252.0},
        {"janela": 63, "label": "63 pregões", "fator_anualizacao": 252.0},
        {"janela": 126, "label": "126 pregões", "fator_anualizacao": 252.0},
        {"janela": 252, "label": "252 pregões", "fator_anualizacao": 252.0},
    ],
    "mensal": [
        {"janela": 6, "label": "6 meses", "fator_anualizacao": 12.0},
        {"janela": 12, "label": "12 meses", "fator_anualizacao": 12.0},
        {"janela": 36, "label": "36 meses", "fator_anualizacao": 12.0},
    ],
    "anual": [
        {"janela": 3, "label": "3 anos", "fator_anualizacao": 1.0},
        {"janela": 5, "label": "5 anos", "fator_anualizacao": 1.0},
    ],
}

TERMOS_RISCO = [
    "volatil", "vol_", "downside", "drawdown", "risco", "sharpe",
    "sortino", "calmar", "desvio", "perda", "recuper", "pior",
    "max_dd", "max_drawdown", "mdd", "var_", "cvar",
]

def carregar_entrada(entrada):
    caminho = Path(entrada["caminho_localizado"])
    if caminho.exists():
        return pd.read_parquet(caminho), True
    if entrada["obrigatorio"]:
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado para a Etapa 15.3: {caminho}")
    return pd.DataFrame(), False

def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")

def converter_data(serie):
    return pd.to_datetime(serie, errors="coerce")

def obter_serie(df, coluna, valor_padrao=np.nan):
    if coluna in df.columns:
        return df[coluna]
    return pd.Series(valor_padrao, index=df.index)

def obter_primeira_coluna(df, colunas, valor_padrao=np.nan):
    for coluna in colunas:
        if coluna in df.columns:
            return df[coluna]
    return pd.Series(valor_padrao, index=df.index)

def escolher_colunas(df, colunas):
    return [coluna for coluna in colunas if coluna in df.columns]

def preparar_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida

def salvar_parquet_compat(df, caminho, index=False):
    salvar_dataframe(preparar_para_parquet(df), caminho, index=index)

def normalizar_nome(valor):
    if pd.isna(valor):
        return ""
    return str(valor).strip()

def contar_valores_infinitos(df):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        return 0

    colunas_numericas = list(df.select_dtypes(include=[np.number]).columns)
    if len(colunas_numericas) == 0:
        return 0

    total_infinitos = 0
    for coluna in colunas_numericas:
        serie_numerica = pd.to_numeric(df[coluna], errors="coerce")
        valores = serie_numerica.to_numpy(dtype="float64", na_value=np.nan)
        total_infinitos += int(np.isinf(valores).sum())

    return total_infinitos

def classificar_grupo_html(row):
    nome = normalizar_nome(row.get("nome_exibicao_estrategia", row.get("nome_estrategia_html", ""))).lower()
    chave = normalizar_nome(row.get("chave_estrategia", "")).lower()
    familia = normalizar_nome(row.get("familia_estrategia", row.get("familia_estrategia_html", ""))).lower()
    categoria = normalizar_nome(row.get("categoria_estrategia", row.get("categoria_estrategia_html", ""))).lower()
    texto = " ".join([nome, chave, familia, categoria])

    # A ordem das regras é intencional: carteiras aleatórias da família
    # Capitulação/Euforia não são as estratégias reais, mesmo quando alguma
    # chave ou família de origem contém capitulacao/euforia.
    if "ibovespa" in texto or "cdi" in texto or "benchmark" in texto:
        return "benchmark"
    if "aportes mensais" in texto or "mensal" in texto:
        return "controle_mensal"
    if "aleatória" in texto or "aleatoria" in texto or "random" in texto:
        return "controle_aleatorio"
    if "controle" in texto:
        return "outros_controles"
    if nome in ["capitulação", "capitulacao", "euforia"] or chave in ["capitulacao", "euforia"]:
        return "estrategia_real"
    return "nao_classificado"

def label_grupo_html(grupo):
    mapa = {
        "estrategia_real": "Estratégias Reais",
        "benchmark": "Benchmarks",
        "controle_mensal": "Controles Mensais",
        "controle_aleatorio": "Controles Aleatórios",
        "outros_controles": "Outros Controles",
        "nao_classificado": "Não Classificado",
    }
    return mapa.get(grupo, grupo)

def padronizar_identificacao_estrategia(df):
    df_saida = df.copy()

    if "nome_exibicao_estrategia" not in df_saida.columns:
        if "nome_estrategia_html" in df_saida.columns:
            df_saida["nome_exibicao_estrategia"] = df_saida["nome_estrategia_html"]
        elif "estrategia_referencia" in df_saida.columns:
            df_saida["nome_exibicao_estrategia"] = df_saida["estrategia_referencia"]
        else:
            df_saida["nome_exibicao_estrategia"] = obter_serie(df_saida, "chave_estrategia", "")

    if "chave_estrategia" not in df_saida.columns:
        if "chave_estrategia_real" in df_saida.columns:
            df_saida["chave_estrategia"] = df_saida["chave_estrategia_real"]
        elif "estrategia_referencia" in df_saida.columns:
            df_saida["chave_estrategia"] = df_saida["estrategia_referencia"]
        else:
            df_saida["chave_estrategia"] = (
                df_saida["nome_exibicao_estrategia"]
                .astype(str)
                .str.lower()
                .str.replace(" ", "_", regex=False)
            )

    if "familia_estrategia" not in df_saida.columns:
        if "familia_estrategia_html" in df_saida.columns:
            df_saida["familia_estrategia"] = df_saida["familia_estrategia_html"]
        elif "estrategia_referencia" in df_saida.columns:
            df_saida["familia_estrategia"] = df_saida["estrategia_referencia"]
        else:
            df_saida["familia_estrategia"] = "nao_classificada"

    if "categoria_estrategia" not in df_saida.columns:
        if "categoria_estrategia_html" in df_saida.columns:
            df_saida["categoria_estrategia"] = df_saida["categoria_estrategia_html"]
        elif "grupo_html" in df_saida.columns:
            df_saida["categoria_estrategia"] = df_saida["grupo_html"]
        else:
            df_saida["categoria_estrategia"] = "nao_classificada"

    # Reclassificação determinística para evitar herdar grupos inconsistentes de bases anteriores.
    df_saida["grupo_html"] = df_saida.apply(classificar_grupo_html, axis=1)
    df_saida["label_grupo_html"] = df_saida["grupo_html"].map(label_grupo_html)
    mapa_ordem = {
        "estrategia_real": 1,
        "benchmark": 2,
        "controle_mensal": 3,
        "controle_aleatorio": 4,
        "outros_controles": 5,
        "nao_classificado": 99,
    }
    df_saida["ordem_grupo_html"] = df_saida["grupo_html"].map(mapa_ordem).fillna(99).astype(int)
    if "ordem_exibicao" not in df_saida.columns:
        df_saida["ordem_exibicao"] = pd.factorize(df_saida["chave_estrategia"].astype(str))[0] + 1
    df_saida["flag_estrategia_real"] = df_saida["grupo_html"].eq("estrategia_real")

    return df_saida

def calcular_drawdown_curvas(df_curvas):
    df = df_curvas.copy()
    df = padronizar_identificacao_estrategia(df)

    if "data" not in df.columns:
        raise KeyError("A base de curvas patrimoniais não possui a coluna obrigatória 'data'.")
    if "patrimonio_base_100" in df.columns:
        coluna_patrimonio = "patrimonio_base_100"
    elif "patrimonio_total" in df.columns:
        coluna_patrimonio = "patrimonio_total"
    else:
        raise KeyError("A base de curvas patrimoniais não possui 'patrimonio_base_100' nem 'patrimonio_total'.")

    df["data"] = converter_data(df["data"])
    df[coluna_patrimonio] = converter_numero(df[coluna_patrimonio])
    df = df.loc[df["data"].notna() & df[coluna_patrimonio].notna()].copy()
    df = df.sort_values(["chave_estrategia", "data"]).reset_index(drop=True)

    df["pico_historico_base"] = (
        df
        .groupby("chave_estrategia", dropna=False)[coluna_patrimonio]
        .cummax()
    )

    df["drawdown"] = np.where(
        df["pico_historico_base"] > 0,
        df[coluna_patrimonio] / df["pico_historico_base"] - 1.0,
        np.nan,
    )

    df["drawdown_abs"] = -df["drawdown"]
    df["drawdown_percentual"] = df["drawdown"] * 100.0
    df["drawdown_abs_percentual"] = df["drawdown_abs"] * 100.0
    df["flag_em_drawdown"] = df["drawdown"] < -0.0000000001
    df["flag_novo_pico"] = df["drawdown"].abs() <= 0.0000000001
    df["ano"] = df["data"].dt.year.astype(int)
    df["ano_mes"] = df["data"].dt.to_period("M").astype(str)
    df["metrica_grafico"] = "drawdown"
    df["valor_grafico"] = df["drawdown"]
    df["label_valor_grafico"] = df["drawdown_percentual"].map(lambda x: "" if pd.isna(x) else f"{x:.2f}%")
    df["tipo_grafico_sugerido"] = "area_drawdown"

    colunas_saida = escolher_colunas(
        df,
        [
            "data",
            "ano",
            "ano_mes",
            "chave_estrategia",
            "estrategia_id",
            "nome_exibicao_estrategia",
            "familia_estrategia",
            "categoria_estrategia",
            "grupo_html",
            "label_grupo_html",
            "ordem_grupo_html",
            "ordem_exibicao",
            "flag_estrategia_real",
            coluna_patrimonio,
            "pico_historico_base",
            "drawdown",
            "drawdown_abs",
            "drawdown_percentual",
            "drawdown_abs_percentual",
            "flag_em_drawdown",
            "flag_novo_pico",
            "metrica_grafico",
            "valor_grafico",
            "label_valor_grafico",
            "tipo_grafico_sugerido",
        ],
    )

    return df[colunas_saida].reset_index(drop=True)

def calcular_eventos_drawdown(df_drawdown):
    if len(df_drawdown) == 0:
        return pd.DataFrame()

    registros = []
    tolerancia = 0.0000000001

    for chave, grupo in df_drawdown.sort_values(["chave_estrategia", "data"]).groupby("chave_estrategia", dropna=False):
        grupo = grupo.copy().reset_index(drop=True)
        grupo["flag_em_drawdown"] = converter_numero(grupo["drawdown"]).fillna(0.0) < -tolerancia
        grupo["id_bloco_drawdown"] = grupo["flag_em_drawdown"].ne(grupo["flag_em_drawdown"].shift()).cumsum()

        for id_bloco, bloco in grupo.loc[grupo["flag_em_drawdown"]].groupby("id_bloco_drawdown", dropna=False):
            if len(bloco) == 0:
                continue

            idx_fundo = bloco["drawdown"].idxmin()
            data_inicio = bloco["data"].min()
            data_fim_pre_recuperacao = bloco["data"].max()
            data_fundo = grupo.loc[idx_fundo, "data"] if idx_fundo in grupo.index else bloco["data"].iloc[0]
            drawdown_minimo = float(bloco["drawdown"].min())
            drawdown_abs_maximo = float((-bloco["drawdown"]).max())

            pos_final_bloco = int(bloco.index.max())
            proximas_linhas = grupo.loc[pos_final_bloco + 1:].copy()
            linhas_recuperacao = proximas_linhas.loc[converter_numero(proximas_linhas["drawdown"]).abs() <= tolerancia]
            data_recuperacao = linhas_recuperacao["data"].min() if len(linhas_recuperacao) > 0 else pd.NaT
            flag_recuperado = pd.notna(data_recuperacao)

            if flag_recuperado:
                duracao_ate_recuperacao_pregoes = int((grupo.loc[:linhas_recuperacao.index.min()].shape[0]) - (grupo.loc[:bloco.index.min()].shape[0]) + 1)
            else:
                duracao_ate_recuperacao_pregoes = np.nan

            registros.append(
                {
                    "chave_estrategia": chave,
                    "nome_exibicao_estrategia": bloco["nome_exibicao_estrategia"].iloc[0] if "nome_exibicao_estrategia" in bloco.columns else str(chave),
                    "familia_estrategia": bloco["familia_estrategia"].iloc[0] if "familia_estrategia" in bloco.columns else "",
                    "grupo_html": bloco["grupo_html"].iloc[0] if "grupo_html" in bloco.columns else "",
                    "label_grupo_html": bloco["label_grupo_html"].iloc[0] if "label_grupo_html" in bloco.columns else "",
                    "ordem_grupo_html": bloco["ordem_grupo_html"].iloc[0] if "ordem_grupo_html" in bloco.columns else 99,
                    "ordem_exibicao": bloco["ordem_exibicao"].iloc[0] if "ordem_exibicao" in bloco.columns else 99,
                    "id_evento_drawdown": int(id_bloco),
                    "data_inicio_drawdown": data_inicio,
                    "data_fundo_drawdown": data_fundo,
                    "data_fim_pre_recuperacao": data_fim_pre_recuperacao,
                    "data_recuperacao": data_recuperacao,
                    "flag_recuperado": bool(flag_recuperado),
                    "duracao_drawdown_pregoes": int(len(bloco)),
                    "duracao_ate_recuperacao_pregoes": duracao_ate_recuperacao_pregoes,
                    "drawdown_minimo": drawdown_minimo,
                    "drawdown_abs_maximo": drawdown_abs_maximo,
                    "drawdown_abs_maximo_percentual": drawdown_abs_maximo * 100.0,
                }
            )

    if len(registros) == 0:
        return pd.DataFrame()

    df_eventos = pd.DataFrame(registros)
    return df_eventos.sort_values(["ordem_grupo_html", "ordem_exibicao", "chave_estrategia", "data_inicio_drawdown"]).reset_index(drop=True)

def calcular_volatilidade_rolling(df_retornos):
    if len(df_retornos) == 0:
        return pd.DataFrame()

    df = df_retornos.copy()
    df = padronizar_identificacao_estrategia(df)

    if "retorno_periodo" not in df.columns:
        raise KeyError("A base de retornos periódicos não possui a coluna obrigatória 'retorno_periodo'.")

    if "data_periodo" in df.columns:
        df["data_periodo"] = converter_data(df["data_periodo"])
    elif "data" in df.columns:
        df["data_periodo"] = converter_data(df["data"])
    else:
        raise KeyError("A base de retornos periódicos não possui 'data_periodo' nem 'data'.")

    if "frequencia" not in df.columns:
        df["frequencia"] = "diaria"

    df["retorno_periodo"] = converter_numero(df["retorno_periodo"])
    df = df.loc[df["data_periodo"].notna()].copy()

    registros = []

    for frequencia, configs in MAPA_JANELAS_VOLATILIDADE.items():
        df_freq = df.loc[df["frequencia"].astype(str).eq(frequencia)].copy()
        if len(df_freq) == 0:
            continue

        df_freq = df_freq.sort_values(["chave_estrategia", "data_periodo"]).copy()

        for config in configs:
            janela = int(config["janela"])
            fator_anualizacao = float(config["fator_anualizacao"])
            min_periods = max(2, int(np.ceil(janela * 0.60)))

            base = df_freq.copy()
            base["janela_volatilidade"] = janela
            base["label_janela_volatilidade"] = config["label"]
            base["n_minimo_observacoes"] = min_periods
            base["volatilidade_periodica_rolling"] = (
                base
                .groupby("chave_estrategia", dropna=False)["retorno_periodo"]
                .transform(lambda x: x.rolling(window=janela, min_periods=min_periods).std(ddof=1))
            )
            base["volatilidade_anualizada_rolling"] = base["volatilidade_periodica_rolling"] * np.sqrt(fator_anualizacao)
            base["volatilidade_anualizada_percentual"] = base["volatilidade_anualizada_rolling"] * 100.0
            base["metrica_grafico"] = "volatilidade_anualizada_rolling"
            base["valor_grafico"] = base["volatilidade_anualizada_rolling"]
            base["label_valor_grafico"] = base["volatilidade_anualizada_percentual"].map(lambda x: "" if pd.isna(x) else f"{x:.2f}%")
            base["tipo_grafico_sugerido"] = "linha_volatilidade_rolling"

            colunas_saida = escolher_colunas(
                base,
                [
                    "frequencia",
                    "label_frequencia",
                    "ordem_frequencia",
                    "data_periodo",
                    "periodo_html",
                    "ano",
                    "ano_mes",
                    "chave_estrategia",
                    "estrategia_id",
                    "nome_exibicao_estrategia",
                    "familia_estrategia",
                    "categoria_estrategia",
                    "grupo_html",
                    "label_grupo_html",
                    "ordem_grupo_html",
                    "ordem_exibicao",
                    "flag_estrategia_real",
                    "janela_volatilidade",
                    "label_janela_volatilidade",
                    "n_minimo_observacoes",
                    "retorno_periodo",
                    "volatilidade_periodica_rolling",
                    "volatilidade_anualizada_rolling",
                    "volatilidade_anualizada_percentual",
                    "metrica_grafico",
                    "valor_grafico",
                    "label_valor_grafico",
                    "tipo_grafico_sugerido",
                ],
            )

            registros.append(base[colunas_saida].copy())

    if len(registros) == 0:
        return pd.DataFrame()

    df_vol = pd.concat(registros, ignore_index=True)
    return df_vol.sort_values(["ordem_frequencia", "ordem_grupo_html", "ordem_exibicao", "chave_estrategia", "janela_volatilidade", "data_periodo"]).reset_index(drop=True)

def selecionar_colunas_risco(df):
    if len(df) == 0:
        return pd.DataFrame()

    colunas_identificacao = [
        "frequencia",
        "label_frequencia",
        "ordem_frequencia",
        "estrategia_referencia",
        "chave_estrategia",
        "estrategia_id",
        "nome_exibicao_estrategia",
        "nome_estrategia_html",
        "familia_estrategia",
        "familia_estrategia_html",
        "categoria_estrategia",
        "grupo_html",
        "label_grupo_html",
        "ordem_grupo_html",
        "flag_estrategia_real",
        "nome_exibicao_comparador",
        "benchmark",
        "tipo_comparacao",
        "grupo_comparacao",
        "tipo_controle_mensal",
    ]

    colunas_risco = []
    for coluna in df.columns:
        coluna_norm = str(coluna).lower()
        if any(termo in coluna_norm for termo in TERMOS_RISCO):
            colunas_risco.append(coluna)

    colunas_saida = []
    for coluna in colunas_identificacao + colunas_risco:
        if coluna in df.columns and coluna not in colunas_saida:
            colunas_saida.append(coluna)

    if len(colunas_saida) == 0:
        return df.copy()

    return df[colunas_saida].copy()

def construir_metricas_risco(df_metricas):
    if len(df_metricas) == 0:
        return pd.DataFrame(), pd.DataFrame()

    df = df_metricas.copy()
    df = padronizar_identificacao_estrategia(df)

    if "frequencia" not in df.columns:
        df["frequencia"] = "nao_classificada"
    if "label_frequencia" not in df.columns:
        df["label_frequencia"] = df["frequencia"].map(MAPA_LABEL_FREQUENCIA).fillna(df["frequencia"])
    if "ordem_frequencia" not in df.columns:
        df["ordem_frequencia"] = df["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)

    colunas_metricas = []
    for coluna in df.columns:
        coluna_norm = str(coluna).lower()
        if any(termo in coluna_norm for termo in TERMOS_RISCO):
            if pd.api.types.is_numeric_dtype(df[coluna]) or pd.api.types.is_float_dtype(df[coluna]) or pd.api.types.is_integer_dtype(df[coluna]):
                colunas_metricas.append(coluna)
            else:
                serie_num = pd.to_numeric(df[coluna], errors="coerce")
                if serie_num.notna().sum() > 0:
                    df[coluna] = serie_num
                    colunas_metricas.append(coluna)

    colunas_prioritarias = [
        "frequencia",
        "label_frequencia",
        "ordem_frequencia",
        "chave_estrategia",
        "estrategia_id",
        "nome_exibicao_estrategia",
        "familia_estrategia",
        "categoria_estrategia",
        "grupo_html",
        "label_grupo_html",
        "ordem_grupo_html",
        "ordem_exibicao",
        "flag_estrategia_real",
    ]

    df_wide = df[escolher_colunas(df, colunas_prioritarias + colunas_metricas)].copy()

    registros_long = []
    for coluna in colunas_metricas:
        base = df_wide[escolher_colunas(df_wide, colunas_prioritarias)].copy()
        base["metrica_risco"] = coluna
        base["nome_metrica_risco"] = coluna.replace("_", " ").title()
        base["valor_metrica_risco"] = converter_numero(df_wide[coluna])
        base["tipo_grafico_sugerido"] = "barra_metricas_risco"
        registros_long.append(base)

    if len(registros_long) > 0:
        df_long = pd.concat(registros_long, ignore_index=True)
        df_long = df_long.loc[df_long["valor_metrica_risco"].notna()].copy()
        df_long = df_long.sort_values(["ordem_frequencia", "ordem_grupo_html", "ordem_exibicao", "metrica_risco"]).reset_index(drop=True)
    else:
        df_long = pd.DataFrame()

    df_wide = df_wide.sort_values(["ordem_frequencia", "ordem_grupo_html", "ordem_exibicao", "chave_estrategia"]).reset_index(drop=True)

    return df_wide, df_long

def filtrar_comparacao_risco(df, origem):
    if len(df) == 0:
        return pd.DataFrame()

    base = selecionar_colunas_risco(df)
    base["origem_comparacao"] = origem

    if "frequencia" in base.columns:
        base["ordem_frequencia"] = base["frequencia"].map(MAPA_ORDEM_FREQUENCIA).fillna(99).astype(int)
        base["label_frequencia"] = base["frequencia"].map(MAPA_LABEL_FREQUENCIA).fillna(base["frequencia"])

    return base.reset_index(drop=True)

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Carga das entradas determinísticas
# ============================================================

print("\n[5/13] Carga das entradas determinísticas...")

bases = {}
registros_catalogo_entradas = []
entradas_obrigatorias_ausentes = 0
entradas_opcionais_ausentes = 0

for entrada in ENTRADAS_15_3:
    df_entrada, carregado = carregar_entrada(entrada)
    bases[entrada["chave"]] = df_entrada

    if not carregado and entrada["obrigatorio"]:
        entradas_obrigatorias_ausentes += 1
    if not carregado and not entrada["obrigatorio"]:
        entradas_opcionais_ausentes += 1

    registros_catalogo_entradas.append(
        {
            "tipo_item": "entrada",
            "chave": entrada["chave"],
            "arquivo": Path(entrada["caminho_localizado"]).name,
            "caminho": str(entrada["caminho_localizado"]),
            "obrigatorio": bool(entrada["obrigatorio"]),
            "carregado": bool(carregado),
            "linhas": int(len(df_entrada)),
            "colunas": int(df_entrada.shape[1]) if isinstance(df_entrada, pd.DataFrame) else 0,
            "finalidade": entrada["finalidade"],
        }
    )

print(f"Entradas mapeadas                       : {len(ENTRADAS_15_3):,}")
print(f"Entradas carregadas                     : {sum(1 for r in registros_catalogo_entradas if r['carregado']):,}")
print(f"Entradas opcionais ausentes             : {entradas_opcionais_ausentes:,}")
print(f"Entradas obrigatórias ausentes          : {entradas_obrigatorias_ausentes:,}")
print("OK")

# ============================================================
# 6) Construção da base de drawdown para HTML
# ============================================================

print("\n[6/13] Construção da base de drawdown para HTML...")

df_drawdown_html = calcular_drawdown_curvas(bases["curvas_patrimoniais_html_15_2"])

print(f"Base de drawdown HTML                   : {len(df_drawdown_html):,} linhas")
print(f"Estratégias na base de drawdown         : {df_drawdown_html['chave_estrategia'].nunique():,}")
print(f"Menor drawdown observado                : {df_drawdown_html['drawdown'].min():.6f}")
print("OK")

# ============================================================
# 7) Construção dos eventos de drawdown e recuperação
# ============================================================

print("\n[7/13] Construção dos eventos de drawdown e recuperação...")

df_eventos_drawdown_html = calcular_eventos_drawdown(df_drawdown_html)

print(f"Eventos de drawdown HTML                : {len(df_eventos_drawdown_html):,} linhas")
if len(df_eventos_drawdown_html) > 0:
    print(f"Eventos recuperados                     : {int(df_eventos_drawdown_html['flag_recuperado'].sum()):,}")
    print(f"Maior drawdown absoluto em evento       : {df_eventos_drawdown_html['drawdown_abs_maximo'].max():.6f}")
else:
    print("Eventos recuperados                     : 0")
    print("Maior drawdown absoluto em evento       : n/a")
print("OK")

# ============================================================
# 8) Construção da volatilidade rolling multifrequência
# ============================================================

print("\n[8/13] Construção da volatilidade rolling multifrequência...")

df_volatilidade_rolling_html = calcular_volatilidade_rolling(bases["retornos_periodicos_html_15_2"])

print(f"Volatilidade rolling HTML               : {len(df_volatilidade_rolling_html):,} linhas")
print(f"Frequências preservadas                 : {df_volatilidade_rolling_html['frequencia'].nunique() if len(df_volatilidade_rolling_html) > 0 else 0:,}")
print(f"Janelas calculadas                      : {df_volatilidade_rolling_html['label_janela_volatilidade'].nunique() if len(df_volatilidade_rolling_html) > 0 else 0:,}")
print("OK")

# ============================================================
# 9) Consolidação das métricas oficiais de risco e eficiência
# ============================================================

print("\n[9/13] Consolidação das métricas oficiais de risco e eficiência...")

df_metricas_risco_html, df_metricas_risco_long_html = construir_metricas_risco(bases["metricas_principais_html_15_1"])

print(f"Métricas de risco HTML                  : {len(df_metricas_risco_html):,} linhas")
print(f"Métricas de risco em formato long       : {len(df_metricas_risco_long_html):,} linhas")
print(f"Métricas distintas em formato long      : {df_metricas_risco_long_html['metrica_risco'].nunique() if len(df_metricas_risco_long_html) > 0 else 0:,}")
print("OK")

# ============================================================
# 10) Consolidação de risco relativo contra benchmarks, controles e comparação direta
# ============================================================

print("\n[10/13] Consolidação de risco relativo contra benchmarks, controles e comparação direta...")

df_risco_benchmarks_html = filtrar_comparacao_risco(
    bases["comparacao_benchmarks_html_15_1"],
    origem="15_1_comparacao_benchmarks",
)

df_risco_controles_html = filtrar_comparacao_risco(
    bases["comparacao_controles_html_15_1"],
    origem="15_1_comparacao_controles",
)

df_risco_direto_html = filtrar_comparacao_risco(
    bases["comparacao_direta_html_15_1"],
    origem="15_1_comparacao_capitulacao_vs_euforia",
)

print(f"Risco vs benchmarks HTML                : {len(df_risco_benchmarks_html):,} linhas")
print(f"Risco vs controles HTML                 : {len(df_risco_controles_html):,} linhas")
print(f"Risco Cap. vs Euforia HTML              : {len(df_risco_direto_html):,} linhas")
print("OK")

# ============================================================
# 11) Catálogo, resumo e parâmetros metodológicos
# ============================================================

print("\n[11/13] Catálogo, resumo e parâmetros metodológicos...")

registros_resumo = []

if len(df_drawdown_html) > 0:
    registros_resumo.append(
        {
            "tema": "drawdown",
            "descricao": "Série diária de drawdown das estratégias, benchmarks e controles.",
            "linhas": len(df_drawdown_html),
            "frequencias": "diaria",
            "uso_html": "Gráficos de área de drawdown e tabela de piores quedas.",
        }
    )

if len(df_eventos_drawdown_html) > 0:
    registros_resumo.append(
        {
            "tema": "eventos_drawdown",
            "descricao": "Eventos de drawdown e recuperação por estratégia.",
            "linhas": len(df_eventos_drawdown_html),
            "frequencias": "diaria",
            "uso_html": "Tabela de episódios de perda, fundo e recuperação.",
        }
    )

if len(df_volatilidade_rolling_html) > 0:
    registros_resumo.append(
        {
            "tema": "volatilidade_rolling",
            "descricao": "Volatilidade rolling em frequência diária, mensal e anual quando aplicável.",
            "linhas": len(df_volatilidade_rolling_html),
            "frequencias": ", ".join(sorted(df_volatilidade_rolling_html["frequencia"].dropna().astype(str).unique())),
            "uso_html": "Linhas de volatilidade rolling com filtros por janela e frequência.",
        }
    )

if len(df_metricas_risco_long_html) > 0:
    registros_resumo.append(
        {
            "tema": "metricas_risco",
            "descricao": "Métricas oficiais de risco e eficiência em formato longo.",
            "linhas": len(df_metricas_risco_long_html),
            "frequencias": ", ".join(sorted(df_metricas_risco_long_html["frequencia"].dropna().astype(str).unique())) if "frequencia" in df_metricas_risco_long_html.columns else "",
            "uso_html": "Barras e rankings de volatilidade, drawdown, Sharpe, Sortino e Calmar.",
        }
    )

if len(df_risco_benchmarks_html) > 0:
    registros_resumo.append(
        {
            "tema": "risco_vs_benchmarks",
            "descricao": "Comparações de risco relativo contra Ibovespa e CDI.",
            "linhas": len(df_risco_benchmarks_html),
            "frequencias": ", ".join(sorted(df_risco_benchmarks_html["frequencia"].dropna().astype(str).unique())) if "frequencia" in df_risco_benchmarks_html.columns else "",
            "uso_html": "Tabela e gráficos de diferença de risco contra benchmarks.",
        }
    )

if len(df_risco_controles_html) > 0:
    registros_resumo.append(
        {
            "tema": "risco_vs_controles",
            "descricao": "Comparações de risco relativo contra controles aleatórios e mensais.",
            "linhas": len(df_risco_controles_html),
            "frequencias": ", ".join(sorted(df_risco_controles_html["frequencia"].dropna().astype(str).unique())) if "frequencia" in df_risco_controles_html.columns else "",
            "uso_html": "Tabela e gráficos de diferença de risco contra controles.",
        }
    )

if len(df_risco_direto_html) > 0:
    registros_resumo.append(
        {
            "tema": "risco_capitulacao_vs_euforia",
            "descricao": "Comparação direta de risco entre Capitulação e Euforia.",
            "linhas": len(df_risco_direto_html),
            "frequencias": ", ".join(sorted(df_risco_direto_html["frequencia"].dropna().astype(str).unique())) if "frequencia" in df_risco_direto_html.columns else "",
            "uso_html": "Painel direto de diferenças de risco entre as duas estratégias reais.",
        }
    )

df_resumo_graficos_html = pd.DataFrame(registros_resumo)

SAIDAS_15_3 = [
    {
        "arquivo": CAMINHO_DRAWDOWN_HTML.name,
        "tema": "drawdown",
        "linhas": len(df_drawdown_html),
        "colunas": df_drawdown_html.shape[1],
        "uso_html": "Gráficos de drawdown diário por estratégia.",
    },
    {
        "arquivo": CAMINHO_EVENTOS_DRAWDOWN_HTML.name,
        "tema": "eventos_drawdown",
        "linhas": len(df_eventos_drawdown_html),
        "colunas": df_eventos_drawdown_html.shape[1],
        "uso_html": "Tabela de eventos de drawdown e recuperação.",
    },
    {
        "arquivo": CAMINHO_VOLATILIDADE_ROLLING_HTML.name,
        "tema": "volatilidade_rolling",
        "linhas": len(df_volatilidade_rolling_html),
        "colunas": df_volatilidade_rolling_html.shape[1],
        "uso_html": "Gráficos de volatilidade rolling diária, mensal e anual.",
    },
    {
        "arquivo": CAMINHO_METRICAS_RISCO_HTML.name,
        "tema": "metricas_risco",
        "linhas": len(df_metricas_risco_html),
        "colunas": df_metricas_risco_html.shape[1],
        "uso_html": "Tabela consolidada de métricas de risco em formato amplo.",
    },
    {
        "arquivo": CAMINHO_METRICAS_RISCO_LONG_HTML.name,
        "tema": "metricas_risco_long",
        "linhas": len(df_metricas_risco_long_html),
        "colunas": df_metricas_risco_long_html.shape[1],
        "uso_html": "Base longa para gráficos e rankings de métricas de risco.",
    },
    {
        "arquivo": CAMINHO_RISCO_BENCHMARKS_HTML.name,
        "tema": "risco_vs_benchmarks",
        "linhas": len(df_risco_benchmarks_html),
        "colunas": df_risco_benchmarks_html.shape[1],
        "uso_html": "Risco relativo contra Ibovespa e CDI.",
    },
    {
        "arquivo": CAMINHO_RISCO_CONTROLES_HTML.name,
        "tema": "risco_vs_controles",
        "linhas": len(df_risco_controles_html),
        "colunas": df_risco_controles_html.shape[1],
        "uso_html": "Risco relativo contra controles aleatórios e mensais.",
    },
    {
        "arquivo": CAMINHO_RISCO_DIRETO_HTML.name,
        "tema": "risco_capitulacao_vs_euforia",
        "linhas": len(df_risco_direto_html),
        "colunas": df_risco_direto_html.shape[1],
        "uso_html": "Risco relativo entre Capitulação e Euforia.",
    },
    {
        "arquivo": CAMINHO_RESUMO_GRAFICOS_HTML.name,
        "tema": "resumo_graficos_risco",
        "linhas": len(df_resumo_graficos_html),
        "colunas": df_resumo_graficos_html.shape[1],
        "uso_html": "Tabela de apoio para navegação do bloco de risco no HTML.",
    },
]

df_catalogo_graficos_html = pd.concat(
    [
        pd.DataFrame(registros_catalogo_entradas),
        pd.DataFrame(
            [
                {
                    "tipo_item": "saida",
                    "chave": item["tema"],
                    "arquivo": item["arquivo"],
                    "caminho": str(DIR_ETAPA_15 / item["arquivo"]),
                    "obrigatorio": True,
                    "carregado": True,
                    "linhas": item["linhas"],
                    "colunas": item["colunas"],
                    "finalidade": item["uso_html"],
                }
                for item in SAIDAS_15_3
            ]
        ),
    ],
    ignore_index=True,
)

PARAMETROS_15_3 = {
    "etapa": "15",
    "subetapa": "15.3",
    "nome_subetapa": "Bases dos Gráficos de Risco",
    "frequencias_preservadas": "diaria, mensal e anual quando metodologicamente aplicável",
    "fonte_curvas": "15_2_base_curvas_patrimoniais_html.parquet",
    "fonte_retornos": "15_2_base_retornos_periodicos_html.parquet",
    "fonte_metricas": "15_1_tbl_metricas_principais_estrategias_html.parquet",
    "fonte_comparacoes": "15_1_tbl_comparacao_benchmarks_html, 15_1_tbl_comparacao_controles_html e 15_1_tbl_comparacao_capitulacao_euforia_html",
    "calculo_drawdown": "derivado da curva patrimonial indexada já consolidada na 15.2",
    "janelas_volatilidade_diaria": "21, 63, 126 e 252 pregões",
    "janelas_volatilidade_mensal": "6, 12 e 36 meses",
    "janelas_volatilidade_anual": "3 e 5 anos",
    "objetivo_html": "gerar bases prontas para gráficos de risco sem recálculo pesado na seção final do HTML",
    "observacao_metodologica": "A subetapa organiza bases para visualização; não altera o motor de backtest nem as métricas oficiais.",
    "observacao_v2": "Carteiras aleatórias das famílias Capitulação/Euforia são classificadas como controles aleatórios, não como estratégias reais.",
}

df_parametros = pd.DataFrame([{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_15_3.items()])

print(f"Resumo dos gráficos de risco             : {len(df_resumo_graficos_html):,} linhas")
print(f"Catálogo de gráficos de risco            : {len(df_catalogo_graficos_html):,} linhas")
print(f"Parâmetros metodológicos                 : {len(df_parametros):,} linhas")
print("OK")

# ============================================================
# 12) Auditoria de validação
# ============================================================

print("\n[12/13] Auditoria de validação...")

metricas_infinitas = 0
for df_verificacao in [
    df_drawdown_html,
    df_eventos_drawdown_html,
    df_volatilidade_rolling_html,
    df_metricas_risco_html,
    df_metricas_risco_long_html,
    df_risco_benchmarks_html,
    df_risco_controles_html,
    df_risco_direto_html,
]:
    metricas_infinitas += contar_valores_infinitos(df_verificacao)

duplicatas_drawdown = int(df_drawdown_html.duplicated(["chave_estrategia", "data"]).sum()) if len(df_drawdown_html) > 0 else 0

if len(df_volatilidade_rolling_html) > 0:
    duplicatas_volatilidade = int(
        df_volatilidade_rolling_html.duplicated(["chave_estrategia", "frequencia", "janela_volatilidade", "data_periodo"]).sum()
    )
else:
    duplicatas_volatilidade = 0

n_estrategias_reais_drawdown = int(
    df_drawdown_html.loc[df_drawdown_html["flag_estrategia_real"], "nome_exibicao_estrategia"].nunique()
) if len(df_drawdown_html) > 0 and "flag_estrategia_real" in df_drawdown_html.columns else 0

if len(df_metricas_risco_long_html) > 0 and "nome_exibicao_estrategia" in df_metricas_risco_long_html.columns and "grupo_html" in df_metricas_risco_long_html.columns:
    nomes_metricas_risco = df_metricas_risco_long_html["nome_exibicao_estrategia"].astype(str).str.lower()
    grupos_metricas_risco = df_metricas_risco_long_html["grupo_html"].astype(str)
    metricas_aleatorias_classificadas_como_reais = int((
        grupos_metricas_risco.eq("estrategia_real")
        & (nomes_metricas_risco.str.contains("aleatória", case=False, na=False) | nomes_metricas_risco.str.contains("aleatoria", case=False, na=False) | nomes_metricas_risco.str.contains("random", case=False, na=False))
    ).sum())
else:
    metricas_aleatorias_classificadas_como_reais = 0

if len(df_drawdown_html) > 0 and "nome_exibicao_estrategia" in df_drawdown_html.columns and "grupo_html" in df_drawdown_html.columns:
    nomes_drawdown = df_drawdown_html["nome_exibicao_estrategia"].astype(str).str.lower()
    grupos_drawdown = df_drawdown_html["grupo_html"].astype(str)
    drawdown_aleatorias_classificadas_como_reais = int((
        grupos_drawdown.eq("estrategia_real")
        & (nomes_drawdown.str.contains("aleatória", case=False, na=False) | nomes_drawdown.str.contains("aleatoria", case=False, na=False) | nomes_drawdown.str.contains("random", case=False, na=False))
    ).sum())
else:
    drawdown_aleatorias_classificadas_como_reais = 0

linhas_auditoria = [
    {
        "item": "arquivos_obrigatorios_ausentes",
        "valor": entradas_obrigatorias_ausentes,
        "valor_referencia": "0",
        "status": "OK" if entradas_obrigatorias_ausentes == 0 else "ERRO",
        "observacao": "Todos os arquivos obrigatórios da subetapa devem estar disponíveis.",
    },
    {
        "item": "linhas_drawdown_html",
        "valor": len(df_drawdown_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_drawdown_html) > 0 else "ERRO",
        "observacao": "A base de drawdown deve ser gerada.",
    },
    {
        "item": "estrategias_reais_drawdown",
        "valor": n_estrategias_reais_drawdown,
        "valor_referencia": ">= 2",
        "status": "OK" if n_estrategias_reais_drawdown >= 2 else "ERRO",
        "observacao": "A base de drawdown deve conter Capitulação e Euforia.",
    },
    {
        "item": "linhas_eventos_drawdown_html",
        "valor": len(df_eventos_drawdown_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_eventos_drawdown_html) > 0 else "ERRO",
        "observacao": "A tabela de eventos de drawdown deve ser gerada.",
    },
    {
        "item": "linhas_volatilidade_rolling_html",
        "valor": len(df_volatilidade_rolling_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_volatilidade_rolling_html) > 0 else "ERRO",
        "observacao": "A base de volatilidade rolling deve ser gerada.",
    },
    {
        "item": "frequencias_volatilidade_rolling",
        "valor": int(df_volatilidade_rolling_html["frequencia"].nunique()) if len(df_volatilidade_rolling_html) > 0 else 0,
        "valor_referencia": ">= 3",
        "status": "OK" if len(df_volatilidade_rolling_html) > 0 and df_volatilidade_rolling_html["frequencia"].nunique() >= 3 else "ERRO",
        "observacao": "A volatilidade rolling deve preservar as visões diária, mensal e anual quando aplicável.",
    },
    {
        "item": "linhas_metricas_risco_html",
        "valor": len(df_metricas_risco_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_metricas_risco_html) > 0 else "ERRO",
        "observacao": "A base de métricas oficiais de risco deve ser gerada.",
    },
    {
        "item": "linhas_metricas_risco_long_html",
        "valor": len(df_metricas_risco_long_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_metricas_risco_long_html) > 0 else "ERRO",
        "observacao": "A base longa de métricas de risco deve ser gerada para gráficos.",
    },
    {
        "item": "linhas_risco_benchmarks_html",
        "valor": len(df_risco_benchmarks_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_risco_benchmarks_html) > 0 else "ERRO",
        "observacao": "A base de risco relativo contra benchmarks deve ser gerada.",
    },
    {
        "item": "linhas_risco_controles_html",
        "valor": len(df_risco_controles_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_risco_controles_html) > 0 else "ERRO",
        "observacao": "A base de risco relativo contra controles deve ser gerada.",
    },
    {
        "item": "linhas_risco_direto_html",
        "valor": len(df_risco_direto_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_risco_direto_html) > 0 else "ERRO",
        "observacao": "A base de risco direto Capitulação vs Euforia deve ser gerada.",
    },
    {
        "item": "duplicatas_drawdown_estrategia_data",
        "valor": duplicatas_drawdown,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_drawdown == 0 else "ERRO",
        "observacao": "Cada estratégia deve ter no máximo uma linha por data na base de drawdown.",
    },
    {
        "item": "duplicatas_volatilidade_estrategia_frequencia_janela_periodo",
        "valor": duplicatas_volatilidade,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_volatilidade == 0 else "ERRO",
        "observacao": "Cada estratégia deve ter no máximo uma linha por frequência, janela e período na base de volatilidade.",
    },
    {
        "item": "linhas_resumo_graficos_risco",
        "valor": len(df_resumo_graficos_html),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_resumo_graficos_html) > 0 else "ERRO",
        "observacao": "A tabela de resumo dos gráficos de risco deve ser gerada.",
    },
    {
        "item": "aleatorias_metricas_classificadas_como_reais",
        "valor": metricas_aleatorias_classificadas_como_reais,
        "valor_referencia": "0",
        "status": "OK" if metricas_aleatorias_classificadas_como_reais == 0 else "ERRO",
        "observacao": "Carteiras aleatórias não devem ser classificadas como estratégias reais nas métricas de risco.",
    },
    {
        "item": "aleatorias_drawdown_classificadas_como_reais",
        "valor": drawdown_aleatorias_classificadas_como_reais,
        "valor_referencia": "0",
        "status": "OK" if drawdown_aleatorias_classificadas_como_reais == 0 else "ERRO",
        "observacao": "Carteiras aleatórias não devem ser classificadas como estratégias reais na base de drawdown.",
    },
    {
        "item": "metricas_infinitas",
        "valor": metricas_infinitas,
        "valor_referencia": "0",
        "status": "OK" if metricas_infinitas == 0 else "ERRO",
        "observacao": "As bases de risco não devem conter valores infinitos.",
    },
]

df_auditoria = pd.DataFrame(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Itens de auditoria                       : {len(df_auditoria):,}")
print(f"Erros bloqueantes                        : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 13) Salvamento dos outputs e validação final da subetapa
# ============================================================

print("\n[13/13] Salvamento dos outputs e validação final da subetapa...")

salvar_parquet_compat(df_drawdown_html, CAMINHO_DRAWDOWN_HTML, index=False)
salvar_parquet_compat(df_eventos_drawdown_html, CAMINHO_EVENTOS_DRAWDOWN_HTML, index=False)
salvar_parquet_compat(df_volatilidade_rolling_html, CAMINHO_VOLATILIDADE_ROLLING_HTML, index=False)
salvar_parquet_compat(df_metricas_risco_html, CAMINHO_METRICAS_RISCO_HTML, index=False)
salvar_parquet_compat(df_metricas_risco_long_html, CAMINHO_METRICAS_RISCO_LONG_HTML, index=False)
salvar_parquet_compat(df_risco_benchmarks_html, CAMINHO_RISCO_BENCHMARKS_HTML, index=False)
salvar_parquet_compat(df_risco_controles_html, CAMINHO_RISCO_CONTROLES_HTML, index=False)
salvar_parquet_compat(df_risco_direto_html, CAMINHO_RISCO_DIRETO_HTML, index=False)
salvar_parquet_compat(df_resumo_graficos_html, CAMINHO_RESUMO_GRAFICOS_HTML, index=False)
salvar_parquet_compat(df_catalogo_graficos_html, CAMINHO_CATALOGO_GRAFICOS_HTML, index=False)
salvar_parquet_compat(df_parametros, CAMINHO_PARAMETROS, index=False)
salvar_parquet_compat(df_auditoria, CAMINHO_AUDITORIA, index=False)

print("Outputs da subetapa salvos com sucesso.")

print("\nAuditoria de validação das bases dos gráficos de risco:")
print(df_auditoria.to_string(index=False))

print("\nResumo dos gráficos de risco para HTML:")
if len(df_resumo_graficos_html) > 0:
    print(df_resumo_graficos_html.to_string(index=False))
else:
    print("Sem linhas para exibir.")

print("\nCatálogo de outputs da Etapa 15.3:")
df_print_catalogo = pd.DataFrame(SAIDAS_15_3)
print(df_print_catalogo.to_string(index=False))

print("\nAmostra da base de drawdown:")
colunas_amostra_drawdown = escolher_colunas(
    df_drawdown_html,
    [
        "data",
        "nome_exibicao_estrategia",
        "grupo_html",
        "drawdown",
        "drawdown_abs_percentual",
        "tipo_grafico_sugerido",
    ],
)
print(df_drawdown_html[colunas_amostra_drawdown].head(30).to_string(index=False))

print("\nAmostra da volatilidade rolling:")
colunas_amostra_vol = escolher_colunas(
    df_volatilidade_rolling_html,
    [
        "frequencia",
        "data_periodo",
        "nome_exibicao_estrategia",
        "grupo_html",
        "label_janela_volatilidade",
        "volatilidade_anualizada_rolling",
    ],
)
print(df_volatilidade_rolling_html[colunas_amostra_vol].dropna(subset=["volatilidade_anualizada_rolling"]).head(30).to_string(index=False))

print("\nAmostra das métricas de risco em formato long:")
colunas_amostra_metricas = escolher_colunas(
    df_metricas_risco_long_html,
    [
        "frequencia",
        "nome_exibicao_estrategia",
        "grupo_html",
        "metrica_risco",
        "valor_metrica_risco",
    ],
)
print(df_metricas_risco_long_html[colunas_amostra_metricas].head(30).to_string(index=False))

print("\nArquivos salvos na subetapa 15.3:")
print(f"- {CAMINHO_DRAWDOWN_HTML}")
print(f"- {CAMINHO_EVENTOS_DRAWDOWN_HTML}")
print(f"- {CAMINHO_VOLATILIDADE_ROLLING_HTML}")
print(f"- {CAMINHO_METRICAS_RISCO_HTML}")
print(f"- {CAMINHO_METRICAS_RISCO_LONG_HTML}")
print(f"- {CAMINHO_RISCO_BENCHMARKS_HTML}")
print(f"- {CAMINHO_RISCO_CONTROLES_HTML}")
print(f"- {CAMINHO_RISCO_DIRETO_HTML}")
print(f"- {CAMINHO_RESUMO_GRAFICOS_HTML}")
print(f"- {CAMINHO_CATALOGO_GRAFICOS_HTML}")
print(f"- {CAMINHO_PARAMETROS}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 15.3 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 15.3 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 15.3 - BASES DOS GRÁFICOS DE RISCO

[1/13] Validação inicial do ambiente...
OK

[2/13] Definição determinística dos diretórios e caminhos de saída...
Diretório da Etapa 11 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_11
Diretório da Etapa 15 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_15
Saída - drawdown HTML                  : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_15\15_3_base_drawdown_html.parquet
Saída - eventos de drawdown HTML       : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_15\15_3_tbl_eventos_drawdown_html.parquet
Saída - volatilidade rolling HTML      : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andame

## Etapa 15.4) Bases de Composição e Auditoria

In [82]:
%%time
# ============================================================
# Etapa 15.4) Bases de Composição e Auditoria
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 15.4 - BASES DE COMPOSIÇÃO E AUDITORIA")
print("=" * 100)

# Esta subetapa consolida bases já produzidas de sinais, aportes, compras,
# posições, concentração e auditorias para consumo posterior pela seção final do HTML.
# A versão utiliza limpeza preventiva do kernel, leitura seletiva de colunas, descarte explícito de objetos
# e processamento incremental da base histórica de posições para gerar agregações úteis sem carregar a tabela bruta integral.

# ============================================================
# 1) Validação inicial do ambiente
# ============================================================

print("\n[1/14] Validação inicial do ambiente...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")


def estimar_memoria_objeto_mb(objeto):
    try:
        if isinstance(objeto, pd.DataFrame):
            return float(objeto.memory_usage(index=True, deep=False).sum() / (1024 ** 2))
        if isinstance(objeto, pd.Series):
            return float(objeto.memory_usage(index=True, deep=False) / (1024 ** 2))
        if isinstance(objeto, np.ndarray):
            return float(objeto.nbytes / (1024 ** 2))
    except Exception:
        return 0.0
    return 0.0


def limpeza_preventiva_memoria_global():
    # Remove objetos tabulares remanescentes de células anteriores antes da consolidação da 15.4.
    # A subetapa 15.4 consome arquivos já salvos em parquet; portanto, dataframes antigos
    # mantidos no kernel não são necessários para a execução e podem pressionar a memória.
    nomes_preservados = {
        "pd",
        "np",
        "Path",
        "DIRETORIOS_PROJETO",
        "salvar_dataframe",
        "gc",
        "pq",
        "db_engine",
        "get_ipython",
        "In",
        "Out",
        "exit",
        "quit",
    }

    nomes_exatos_limpeza = {
        "bases",
        "bases_evidencias",
        "bases_evidencias_completas",
        "timelines",
        "tops_empresas",
        "tops_classificacoes",
        "catalogo_entradas",
        "catalogo_saidas",
        "registros_catalogo_entradas",
        "registros_resumo",
        "registros_resumo_grupos",
        "saidas_info",
        "linhas_auditoria",
        "SAIDAS_15_1",
        "SAIDAS_15_2",
        "SAIDAS_15_3",
    }

    prefixos_limpeza = (
        "df_",
        "base_",
        "bases_",
        "tbl_",
        "tabela_",
        "dados_",
        "resultado_",
        "resultados_",
        "curvas_",
        "retornos_",
        "metricas_",
        "drawdown_",
        "volatilidade_",
        "comparacao_",
        "consolidacao_",
        "auditoria_",
        "catalogo_",
        "timelines",
        "tops_",
    )

    objetos_removidos = 0
    memoria_estimativa_mb = 0.0

    for nome, objeto in list(globals().items()):
        if nome in nomes_preservados or nome.startswith("__"):
            continue

        remover = False
        if isinstance(objeto, (pd.DataFrame, pd.Series, np.ndarray)):
            remover = True
        elif nome in nomes_exatos_limpeza:
            remover = True
        elif nome.startswith(prefixos_limpeza):
            remover = True

        if remover:
            memoria_estimativa_mb += estimar_memoria_objeto_mb(objeto)
            try:
                del globals()[nome]
                objetos_removidos += 1
            except Exception:
                pass

    gc.collect()
    return objetos_removidos, memoria_estimativa_mb


objetos_removidos_memoria, memoria_estimativa_liberada_mb = limpeza_preventiva_memoria_global()
print(f"Objetos tabulares/intermediários removidos do kernel: {objetos_removidos_memoria:,}")
print(f"Memória tabular estimada liberada.................: {memoria_estimativa_liberada_mb:,.2f} MB")
print("OK")

# ============================================================
# 2) Definição determinística dos diretórios e caminhos de saída
# ============================================================

print("\n[2/14] Definição determinística dos diretórios e caminhos de saída...")

DIR_RESULTADOS = Path(DIRETORIOS_PROJETO["resultados"])

for etapa in range(1, 16):
    chave_etapa = f"etapa_{etapa}"
    if chave_etapa not in DIRETORIOS_PROJETO:
        DIRETORIOS_PROJETO[chave_etapa] = DIR_RESULTADOS / chave_etapa
    Path(DIRETORIOS_PROJETO[chave_etapa]).mkdir(parents=True, exist_ok=True)

DIR_ETAPA_10 = Path(DIRETORIOS_PROJETO["etapa_10"])
DIR_ETAPA_12 = Path(DIRETORIOS_PROJETO["etapa_12"])
DIR_ETAPA_13 = Path(DIRETORIOS_PROJETO["etapa_13"])
DIR_ETAPA_14 = Path(DIRETORIOS_PROJETO["etapa_14"])
DIR_ETAPA_15 = Path(DIRETORIOS_PROJETO["etapa_15"])
DIR_ETAPA_15.mkdir(parents=True, exist_ok=True)

CAMINHO_SINAIS_APORTES_HTML = DIR_ETAPA_15 / "15_4_base_sinais_aportes_html.parquet"
CAMINHO_APORTES_HTML = DIR_ETAPA_15 / "15_4_base_aportes_html.parquet"
CAMINHO_COMPRAS_TICKER_HTML = DIR_ETAPA_15 / "15_4_base_compras_ticker_html.parquet"
CAMINHO_COMPRAS_APORTE_HTML = DIR_ETAPA_15 / "15_4_base_compras_aporte_html.parquet"
CAMINHO_POSICOES_HTML = DIR_ETAPA_15 / "15_4_base_posicoes_historicas_html.parquet"
CAMINHO_POSICOES_SETORIAL_MENSAL_HTML = DIR_ETAPA_15 / "15_4_base_posicoes_setorial_mensal_html.parquet"
CAMINHO_TOP_POSICOES_HISTORICAS_HTML = DIR_ETAPA_15 / "15_4_tbl_top_posicoes_historicas_html.parquet"
CAMINHO_RESUMO_POSICOES_HISTORICAS_HTML = DIR_ETAPA_15 / "15_4_tbl_resumo_posicoes_historicas_html.parquet"
CAMINHO_CONCENTRACAO_EMPRESA_HTML = DIR_ETAPA_15 / "15_4_base_concentracao_empresa_html.parquet"
CAMINHO_CONCENTRACAO_SETORIAL_HTML = DIR_ETAPA_15 / "15_4_base_concentracao_setorial_html.parquet"
CAMINHO_TOP_TICKERS_HTML = DIR_ETAPA_15 / "15_4_tbl_top_tickers_comprados_html.parquet"
CAMINHO_TOP_EMPRESAS_HTML = DIR_ETAPA_15 / "15_4_tbl_top_empresas_html.parquet"
CAMINHO_TOP_CLASSIFICACOES_HTML = DIR_ETAPA_15 / "15_4_tbl_top_classificacoes_setoriais_html.parquet"
CAMINHO_TIMELINE_SINAIS_APORTES_HTML = DIR_ETAPA_15 / "15_4_tbl_timeline_sinais_aportes_html.parquet"
CAMINHO_AUDITORIA_CONSOLIDADA_HTML = DIR_ETAPA_15 / "15_4_tbl_auditoria_consolidada_html.parquet"
CAMINHO_RESUMO_COMPOSICAO_HTML = DIR_ETAPA_15 / "15_4_tbl_resumo_composicao_auditoria_html.parquet"
CAMINHO_CATALOGO = DIR_ETAPA_15 / "15_4_tbl_catalogo_composicao_auditoria_html.parquet"
CAMINHO_PARAMETROS = DIR_ETAPA_15 / "15_4_tbl_parametros_composicao_auditoria_html.parquet"
CAMINHO_AUDITORIA = DIR_ETAPA_15 / "15_4_tbl_auditoria_validacao_composicao_auditoria_html.parquet"

print(f"Diretório da Etapa 10 : {DIR_ETAPA_10}")
print(f"Diretório da Etapa 12 : {DIR_ETAPA_12}")
print(f"Diretório da Etapa 13 : {DIR_ETAPA_13}")
print(f"Diretório da Etapa 14 : {DIR_ETAPA_14}")
print(f"Diretório da Etapa 15 : {DIR_ETAPA_15}")
print(f"Saída - sinais e aportes HTML           : {CAMINHO_SINAIS_APORTES_HTML}")
print(f"Saída - aportes HTML                    : {CAMINHO_APORTES_HTML}")
print(f"Saída - compras por ticker HTML         : {CAMINHO_COMPRAS_TICKER_HTML}")
print(f"Saída - compras por aporte HTML         : {CAMINHO_COMPRAS_APORTE_HTML}")
print(f"Saída - posições históricas HTML        : {CAMINHO_POSICOES_HTML}")
print(f"Saída - posições setoriais mensais HTML : {CAMINHO_POSICOES_SETORIAL_MENSAL_HTML}")
print(f"Saída - top posições históricas HTML    : {CAMINHO_TOP_POSICOES_HISTORICAS_HTML}")
print(f"Saída - resumo posições históricas HTML : {CAMINHO_RESUMO_POSICOES_HISTORICAS_HTML}")
print(f"Saída - concentração empresa HTML       : {CAMINHO_CONCENTRACAO_EMPRESA_HTML}")
print(f"Saída - concentração setorial HTML      : {CAMINHO_CONCENTRACAO_SETORIAL_HTML}")
print(f"Saída - top tickers HTML                : {CAMINHO_TOP_TICKERS_HTML}")
print(f"Saída - top empresas HTML               : {CAMINHO_TOP_EMPRESAS_HTML}")
print(f"Saída - top classificações HTML         : {CAMINHO_TOP_CLASSIFICACOES_HTML}")
print(f"Saída - timeline sinais/aportes HTML    : {CAMINHO_TIMELINE_SINAIS_APORTES_HTML}")
print(f"Saída - auditoria consolidada HTML      : {CAMINHO_AUDITORIA_CONSOLIDADA_HTML}")
print(f"Saída - resumo composição/auditoria     : {CAMINHO_RESUMO_COMPOSICAO_HTML}")
print(f"Saída - catálogo                        : {CAMINHO_CATALOGO}")
print(f"Saída - parâmetros                      : {CAMINHO_PARAMETROS}")
print(f"Saída - auditoria                       : {CAMINHO_AUDITORIA}")
print("OK")

# ============================================================
# 3) Catálogo determinístico de entradas da subetapa
# ============================================================

print("\n[3/14] Catálogo determinístico de entradas da subetapa...")

ENTRADAS_15_4 = [
    {"chave": "sinais_temporal_12_1", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_1_base_sinais_temporal.parquet"], "obrigatorio": True, "tema": "sinais", "finalidade": "Base temporal dos sinais finais."},
    {"chave": "timeline_sinais_12_1", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_1_tbl_base_grafico_timeline_sinais.parquet"], "obrigatorio": False, "tema": "sinais", "finalidade": "Timeline dos sinais."},
    {"chave": "intensidade_sinais_12_2", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_2_base_intensidade_sinais.parquet"], "obrigatorio": True, "tema": "sinais", "finalidade": "Base de intensidade dos sinais."},
    {"chave": "timeline_intensidade_12_2", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_2_tbl_base_grafico_intensidade_timeline.parquet"], "obrigatorio": False, "tema": "sinais", "finalidade": "Timeline de intensidade dos sinais."},
    {"chave": "perfil_compras_aporte_12_3", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_3_base_perfil_compras_aporte.parquet"], "obrigatorio": True, "tema": "aportes", "finalidade": "Perfil por aporte."},
    {"chave": "perfil_compras_ticker_12_3", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_3_base_perfil_compras_ticker.parquet"], "obrigatorio": True, "tema": "compras", "finalidade": "Perfil por ticker comprado."},
    {"chave": "top_tickers_comprados_12_3", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_3_tbl_top_tickers_comprados.parquet"], "obrigatorio": True, "tema": "compras", "finalidade": "Ranking de tickers comprados."},
    {"chave": "top_empresas_compradas_12_3", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_3_tbl_top_empresas_compradas.parquet"], "obrigatorio": True, "tema": "compras", "finalidade": "Ranking de empresas compradas."},
    {"chave": "top_setores_comprados_12_3", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_3_tbl_top_setores_comprados.parquet"], "obrigatorio": True, "tema": "compras", "finalidade": "Ranking de setores comprados."},
    {"chave": "timeline_compras_12_3", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_3_tbl_base_grafico_perfil_compras_timeline.parquet"], "obrigatorio": False, "tema": "aportes", "finalidade": "Timeline do perfil das compras."},
    {"chave": "concentracao_empresa_aporte_12_4", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_4_base_concentracao_empresa_aporte.parquet"], "obrigatorio": True, "tema": "concentracao_empresa", "finalidade": "Concentração por empresa e aporte."},
    {"chave": "top_empresas_concentracao_12_4", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_4_tbl_top_empresas_concentracao.parquet"], "obrigatorio": True, "tema": "concentracao_empresa", "finalidade": "Ranking de concentração por empresa."},
    {"chave": "concentracao_setorial_aporte_12_5", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_5_base_concentracao_setorial_aporte.parquet"], "obrigatorio": True, "tema": "concentracao_setorial", "finalidade": "Concentração setorial hierárquica."},
    {"chave": "top_setores_concentracao_12_5", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_5_tbl_top_setores_concentracao.parquet"], "obrigatorio": True, "tema": "concentracao_setorial", "finalidade": "Top setores por concentração."},
    {"chave": "top_subsetores_concentracao_12_5", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_5_tbl_top_subsetores_concentracao.parquet"], "obrigatorio": True, "tema": "concentracao_setorial", "finalidade": "Top subsetores com hierarquia setorial."},
    {"chave": "top_segmentos_concentracao_12_5", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_5_tbl_top_segmentos_concentracao.parquet"], "obrigatorio": True, "tema": "concentracao_setorial", "finalidade": "Top segmentos com hierarquia setorial."},
    {"chave": "timeline_concentracao_setorial_12_5", "etapa": 12, "caminhos": [DIR_ETAPA_12 / "12_5_tbl_base_grafico_concentracao_setorial_timeline.parquet"], "obrigatorio": False, "tema": "concentracao_setorial", "finalidade": "Timeline de concentração setorial."},
    {"chave": "posicoes_historicas_10_4", "etapa": 10, "caminhos": [DIR_ETAPA_10 / "10_4_base_posicoes_historicas.parquet", DIR_ETAPA_10 / "10_4_base_posicoes_historicas_diarias.parquet", DIR_ETAPA_10 / "10_4_base_historico_posicoes.parquet", DIR_ETAPA_10 / "10_4_base_posicoes_diarias.parquet"], "obrigatorio": False, "tema": "posicoes", "finalidade": "Posições históricas, quando a base for compatível com o limite de memória."},
]


def resolver_primeiro_caminho(caminhos):
    for caminho in caminhos:
        caminho_path = Path(caminho)
        if caminho_path.exists():
            return caminho_path
    return Path(caminhos[0])


def obter_linhas_parquet(caminho):
    if not Path(caminho).exists():
        return 0
    try:
        return int(pq.ParquetFile(caminho).metadata.num_rows)
    except Exception:
        return 0


for entrada in ENTRADAS_15_4:
    entrada["caminho_resolvido"] = resolver_primeiro_caminho(entrada["caminhos"])
    entrada["existe"] = entrada["caminho_resolvido"].exists()
    entrada["linhas_origem_estimadas"] = obter_linhas_parquet(entrada["caminho_resolvido"])
    print(
        f"{entrada['chave']:<48} | obrigatório={str(entrada['obrigatorio']):<5} "
        f"| existe={str(entrada['existe']):<5} | linhas={entrada['linhas_origem_estimadas']:<10,} | {entrada['caminho_resolvido']}"
    )

print("OK")

# ============================================================
# 4) Funções auxiliares de leitura seletiva, padronização e salvamento
# ============================================================

print("\n[4/14] Funções auxiliares de leitura seletiva, padronização e salvamento...")

LIMITE_LER_TODAS_COLUNAS = 50000
LIMITE_POSICOES_HTML = 200000
TAMANHO_LOTE_POSICOES = 250000
TOP_N_POSICOES_HISTORICAS = 25

COLUNAS_PRIORITARIAS = [
    "estrategia_referencia", "chave_estrategia", "chave_estrategia_real", "nome_exibicao_estrategia", "nome_estrategia", "nome_estrategia_html",
    "data", "data_referencia", "data_referencia_html", "data_sinal", "data_aporte", "data_aporte_efetiva", "ano", "ano_mes",
    "tipo_sinal", "sinal", "flag_sinal", "flag_capitulacao", "flag_euforia", "regime_mercado", "score_intensidade_sinal", "classificacao_intensidade_sinal", "flag_sinal_intenso",
    "pct_tickers_em_minima_52s", "pct_tickers_em_maxima_52s", "pct_tickers_acima_mm200", "pct_tickers_abaixo_mm200",
    "chave_aporte", "id_aporte", "ticker", "issuer_code", "nome", "nom_res", "empresa", "nome_empresa", "nome_companhia",
    "setor", "subsetor", "segmento", "setor_nome", "subsetor_nome", "segmento_nome", "classificacao", "classificacao_setorial", "nome_classificacao", "nivel_classificacao", "nivel_classificacao_html",
    "qtd_acoes", "quantidade", "preco", "preco_compra", "valor_compra", "valor_comprado", "valor_aporte", "valor_posicao", "valor_mercado", "peso", "peso_carteira", "peso_empresa", "peso_setor", "peso_subsetor", "peso_segmento", "participacao", "participacao_percentual",
    "n_tickers", "n_empresas", "n_compras", "ranking", "rank", "ordem", "drawdown", "retorno", "retorno_periodo", "patrimonio_total",
    "status", "item", "valor", "valor_referencia", "observacao",
]

COLUNAS_AUDITORIA = ["item", "valor", "valor_referencia", "status", "observacao"]


def esquema_parquet(caminho):
    if not Path(caminho).exists():
        return [], 0
    arquivo = pq.ParquetFile(caminho)
    return list(arquivo.schema.names), int(arquivo.metadata.num_rows)


def ler_parquet_controlado(caminho, colunas_preferenciais=None, obrigatorio=False, max_linhas=None, ler_todas_ate=LIMITE_LER_TODAS_COLUNAS):
    caminho = Path(caminho)
    if not caminho.exists():
        if obrigatorio:
            raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {caminho}")
        return pd.DataFrame(), False, 0, 0, "ausente"

    colunas_disponiveis, linhas_origem = esquema_parquet(caminho)
    if colunas_preferenciais is None:
        colunas_preferenciais = COLUNAS_PRIORITARIAS

    if linhas_origem <= ler_todas_ate:
        colunas_leitura = None
    else:
        colunas_leitura = [coluna for coluna in colunas_preferenciais if coluna in colunas_disponiveis]
        if len(colunas_leitura) == 0:
            return pd.DataFrame(), True, linhas_origem, 0, "sem_colunas_prioritarias"

    df = pd.read_parquet(caminho, columns=colunas_leitura)

    if max_linhas is not None and len(df) > max_linhas:
        df = df.head(max_linhas).copy()
        modo = "carregado_com_limite_linhas"
    elif colunas_leitura is None:
        modo = "carregado_todas_colunas"
    else:
        modo = "carregado_colunas_seletivas"

    return df, True, linhas_origem, int(df.shape[1]), modo


def preparar_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def salvar_parquet_compat(df, caminho, index=False):
    salvar_dataframe(preparar_para_parquet(df), caminho, index=index)


def converter_data(serie):
    return pd.to_datetime(serie, errors="coerce")


def converter_numero(serie):
    return pd.to_numeric(serie, errors="coerce")


def obter_serie(df, coluna, valor_padrao=np.nan):
    if isinstance(df, pd.DataFrame) and coluna in df.columns:
        return df[coluna]
    return pd.Series(valor_padrao, index=df.index if isinstance(df, pd.DataFrame) else None)


def obter_primeira_coluna(df, colunas, valor_padrao=np.nan):
    for coluna in colunas:
        if isinstance(df, pd.DataFrame) and coluna in df.columns:
            return df[coluna]
    return pd.Series(valor_padrao, index=df.index if isinstance(df, pd.DataFrame) else None)


def escolher_colunas(df, colunas):
    return [coluna for coluna in colunas if coluna in df.columns]


def ordenar_por_colunas(df, colunas):
    colunas_validas = escolher_colunas(df, colunas)
    if len(df) == 0 or len(colunas_validas) == 0:
        return df.reset_index(drop=True)
    return df.sort_values(colunas_validas).reset_index(drop=True)


def adicionar_metadados_html(df, origem, tema, tipo_base):
    df_saida = df.copy()
    df_saida["origem_base_html"] = origem
    df_saida["tema_html"] = tema
    df_saida["tipo_base_composicao"] = tipo_base
    return df_saida


def classificar_grupo_vetor(df):
    if len(df) == 0:
        return pd.Series(dtype="string")

    componentes = []
    for coluna in ["nome_exibicao_estrategia", "nome_estrategia", "nome_estrategia_html", "chave_estrategia", "estrategia_referencia", "categoria_estrategia"]:
        if coluna in df.columns:
            componentes.append(df[coluna].astype("string").fillna(""))
    if len(componentes) == 0:
        texto = pd.Series("", index=df.index)
    else:
        texto = componentes[0]
        for componente in componentes[1:]:
            texto = texto + " " + componente
        texto = texto.str.lower()

    condicoes = [
        texto.str.contains("ibovespa|cdi|benchmark", regex=True, na=False),
        texto.str.contains("aportes mensais|aporte mensal|20 aportes|30 aportes|ações elegíveis|acoes elegiveis", regex=True, na=False),
        texto.str.contains("aleatória|aleatoria|random", regex=True, na=False),
        texto.str.strip().isin(["capitulação", "capitulacao", "euforia"]) | texto.str.contains(r"\bcapitulação\b|\bcapitulacao\b|\beuforia\b", regex=True, na=False),
        texto.str.contains("controle", regex=False, na=False),
    ]
    escolhas = ["benchmark", "controle_mensal", "controle_aleatorio", "estrategia_real", "outros_controles"]
    return pd.Series(np.select(condicoes, escolhas, default="nao_classificado"), index=df.index)


def padronizar_identificacao(df):
    df_saida = df.copy()
    if len(df_saida) == 0:
        return df_saida

    if "nome_exibicao_estrategia" not in df_saida.columns:
        df_saida["nome_exibicao_estrategia"] = obter_primeira_coluna(
            df_saida,
            ["nome_estrategia_html", "nome_estrategia", "estrategia_referencia", "chave_estrategia"],
            "nao_identificada",
        )

    if "chave_estrategia" not in df_saida.columns:
        df_saida["chave_estrategia"] = obter_primeira_coluna(
            df_saida,
            ["chave_estrategia_real", "estrategia_referencia", "nome_exibicao_estrategia"],
            "nao_identificada",
        ).astype("string").str.lower().str.replace(" ", "_", regex=False)

    if "estrategia_referencia" not in df_saida.columns:
        df_saida["estrategia_referencia"] = obter_primeira_coluna(df_saida, ["familia_estrategia", "chave_estrategia"], "nao_identificada")

    df_saida["grupo_html"] = classificar_grupo_vetor(df_saida)
    df_saida["label_grupo_html"] = df_saida["grupo_html"].map(
        {
            "estrategia_real": "Estratégias Reais",
            "benchmark": "Benchmarks",
            "controle_mensal": "Controles Mensais",
            "controle_aleatorio": "Controles Aleatórios",
            "outros_controles": "Outros Controles",
            "nao_classificado": "Não Classificado",
        }
    )
    df_saida["flag_estrategia_real"] = df_saida["grupo_html"].eq("estrategia_real")
    return df_saida


def padronizar_datas(df):
    df_saida = df.copy()
    for coluna in ["data", "data_referencia", "data_sinal", "data_aporte", "data_aporte_efetiva"]:
        if coluna in df_saida.columns:
            df_saida[coluna] = converter_data(df_saida[coluna])

    if "data_referencia_html" not in df_saida.columns:
        data_base = obter_primeira_coluna(df_saida, ["data", "data_referencia", "data_sinal", "data_aporte", "data_aporte_efetiva"], pd.NaT)
        df_saida["data_referencia_html"] = converter_data(data_base)

    if "ano" not in df_saida.columns and "data_referencia_html" in df_saida.columns:
        df_saida["ano"] = df_saida["data_referencia_html"].dt.year.astype("Int64")
    if "ano_mes" not in df_saida.columns and "data_referencia_html" in df_saida.columns:
        df_saida["ano_mes"] = df_saida["data_referencia_html"].dt.to_period("M").astype(str)

    return df_saida


def padronizar_base_html(df, origem, tema, tipo_base):
    if len(df) == 0:
        return pd.DataFrame()
    df_saida = adicionar_metadados_html(df, origem, tema, tipo_base)
    df_saida = padronizar_identificacao(df_saida)
    df_saida = padronizar_datas(df_saida)
    return df_saida


def construir_classificacao_hierarquica(df, nivel_padrao):
    if len(df) == 0:
        return df.copy()

    df_saida = df.copy()
    if "nivel_classificacao_html" not in df_saida.columns:
        df_saida["nivel_classificacao_html"] = obter_primeira_coluna(df_saida, ["nivel_classificacao", "tipo_classificacao"], nivel_padrao).astype("string")

    setor = obter_primeira_coluna(df_saida, ["setor", "setor_nome", "classificacao_setor"], "").astype("string").fillna("")
    subsetor = obter_primeira_coluna(df_saida, ["subsetor", "subsetor_nome", "classificacao_subsetor"], "").astype("string").fillna("")
    segmento = obter_primeira_coluna(df_saida, ["segmento", "segmento_nome", "classificacao_segmento"], "").astype("string").fillna("")
    classificacao = obter_primeira_coluna(df_saida, ["classificacao", "nome_classificacao", "classificacao_setorial"], "").astype("string").fillna("")

    df_saida["setor_html"] = setor
    df_saida["subsetor_html"] = subsetor
    df_saida["segmento_html"] = segmento

    nivel = df_saida["nivel_classificacao_html"].astype("string").str.lower().fillna("")
    df_saida["classificacao_hierarquica_html"] = classificacao
    df_saida.loc[nivel.eq("setor"), "classificacao_hierarquica_html"] = setor.mask(setor.eq(""), classificacao)
    df_saida.loc[nivel.eq("subsetor"), "classificacao_hierarquica_html"] = (
        setor.mask(setor.eq(""), classificacao).astype("string") + " > " + subsetor.mask(subsetor.eq(""), classificacao).astype("string")
    ).str.replace(r"^ > | > $", "", regex=True)
    df_saida.loc[nivel.eq("segmento"), "classificacao_hierarquica_html"] = (
        setor.astype("string") + " > " + subsetor.astype("string") + " > " + segmento.mask(segmento.eq(""), classificacao).astype("string")
    ).str.replace(r"^ > | > $| >  > ", " > ", regex=True)

    return df_saida


def contar_valores_infinitos(df):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        return 0
    total_infinitos = 0
    for coluna in df.select_dtypes(include=[np.number]).columns:
        valores = pd.to_numeric(df[coluna], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        total_infinitos += int(np.isinf(valores).sum())
    return total_infinitos


def contar_duplicatas(df, colunas):
    colunas_validas = escolher_colunas(df, colunas)
    if len(df) == 0 or len(colunas_validas) != len(colunas):
        return 0
    return int(df.duplicated(colunas_validas).sum())


def salvar_saida(df, caminho, tema, uso_html, saidas_info):
    salvar_parquet_compat(df, caminho, index=False)
    saidas_info.append(
        {
            "arquivo": Path(caminho).name,
            "tema": tema,
            "linhas": int(len(df)),
            "colunas": int(df.shape[1]),
            "uso_html": uso_html,
        }
    )
    return len(df), int(df.shape[1])


def normalizar_nome_coluna(nome):
    return str(nome).strip().lower()


def selecionar_primeira_coluna_disponivel(colunas_disponiveis, candidatos):
    mapa = {normalizar_nome_coluna(coluna): coluna for coluna in colunas_disponiveis}
    for candidato in candidatos:
        candidato_norm = normalizar_nome_coluna(candidato)
        if candidato_norm in mapa:
            return mapa[candidato_norm]
    return None


def selecionar_colunas_posicoes(colunas_disponiveis):
    candidatos = [
        "data", "data_posicao", "data_referencia", "data_carteira", "dt_posicao",
        "chave_estrategia", "estrategia_referencia", "nome_exibicao_estrategia", "nome_estrategia", "nome_estrategia_html",
        "ticker", "codigo_negociacao", "ativo", "issuer_code", "codigo_emissor", "nome", "nom_res", "empresa", "nome_empresa", "nome_companhia",
        "setor", "setor_nome", "subsetor", "subsetor_nome", "segmento", "segmento_nome",
        "valor_posicao", "valor_mercado", "valor_posicao_total", "valor_total_posicao", "valor_carteira", "valor_financeiro",
        "peso", "peso_carteira", "peso_posicao", "participacao", "participacao_percentual",
        "qtd_acoes", "quantidade", "quantidade_acoes", "preco", "preco_fechamento", "preco_mercado",
    ]
    return [coluna for coluna in candidatos if coluna in colunas_disponiveis]


def padronizar_lote_posicoes(df_lote):
    if len(df_lote) == 0:
        return pd.DataFrame()

    colunas = list(df_lote.columns)
    col_data = selecionar_primeira_coluna_disponivel(colunas, ["data", "data_posicao", "data_referencia", "data_carteira", "dt_posicao"])
    col_chave = selecionar_primeira_coluna_disponivel(colunas, ["chave_estrategia", "estrategia_referencia", "nome_exibicao_estrategia", "nome_estrategia", "nome_estrategia_html"])
    col_nome_est = selecionar_primeira_coluna_disponivel(colunas, ["nome_exibicao_estrategia", "nome_estrategia_html", "nome_estrategia", "estrategia_referencia", "chave_estrategia"])
    col_ticker = selecionar_primeira_coluna_disponivel(colunas, ["ticker", "codigo_negociacao", "ativo"])
    col_issuer = selecionar_primeira_coluna_disponivel(colunas, ["issuer_code", "codigo_emissor"])
    col_empresa = selecionar_primeira_coluna_disponivel(colunas, ["nome_empresa", "empresa", "nome_companhia", "nome", "nom_res"])
    col_setor = selecionar_primeira_coluna_disponivel(colunas, ["setor", "setor_nome"])
    col_subsetor = selecionar_primeira_coluna_disponivel(colunas, ["subsetor", "subsetor_nome"])
    col_segmento = selecionar_primeira_coluna_disponivel(colunas, ["segmento", "segmento_nome"])
    col_valor = selecionar_primeira_coluna_disponivel(colunas, ["valor_posicao", "valor_mercado", "valor_posicao_total", "valor_total_posicao", "valor_carteira", "valor_financeiro"])
    col_peso = selecionar_primeira_coluna_disponivel(colunas, ["peso_carteira", "peso_posicao", "peso", "participacao", "participacao_percentual"])
    col_qtd = selecionar_primeira_coluna_disponivel(colunas, ["qtd_acoes", "quantidade_acoes", "quantidade"])
    col_preco = selecionar_primeira_coluna_disponivel(colunas, ["preco", "preco_fechamento", "preco_mercado"])

    if col_data is None:
        return pd.DataFrame()

    saida = pd.DataFrame(index=df_lote.index)
    saida["data_posicao"] = converter_data(df_lote[col_data])
    saida["chave_estrategia"] = df_lote[col_chave].astype("string") if col_chave is not None else "nao_identificada"
    saida["nome_exibicao_estrategia"] = df_lote[col_nome_est].astype("string") if col_nome_est is not None else saida["chave_estrategia"]
    saida["ticker"] = df_lote[col_ticker].astype("string") if col_ticker is not None else ""
    saida["issuer_code"] = df_lote[col_issuer].astype("string") if col_issuer is not None else ""
    saida["nome_empresa"] = df_lote[col_empresa].astype("string") if col_empresa is not None else ""
    saida["setor"] = df_lote[col_setor].astype("string") if col_setor is not None else ""
    saida["subsetor"] = df_lote[col_subsetor].astype("string") if col_subsetor is not None else ""
    saida["segmento"] = df_lote[col_segmento].astype("string") if col_segmento is not None else ""
    saida["valor_posicao"] = converter_numero(df_lote[col_valor]) if col_valor is not None else np.nan
    saida["peso_posicao"] = converter_numero(df_lote[col_peso]) if col_peso is not None else np.nan
    saida["quantidade"] = converter_numero(df_lote[col_qtd]) if col_qtd is not None else np.nan
    saida["preco"] = converter_numero(df_lote[col_preco]) if col_preco is not None else np.nan

    peso_abs_max = saida["peso_posicao"].abs().max(skipna=True)
    if not pd.isna(peso_abs_max) and peso_abs_max > 1.5:
        saida["peso_posicao"] = saida["peso_posicao"] / 100.0

    saida = saida.loc[saida["data_posicao"].notna()].copy()
    if len(saida) == 0:
        return saida

    saida["ano"] = saida["data_posicao"].dt.year.astype("Int64")
    saida["ano_mes"] = saida["data_posicao"].dt.to_period("M").astype(str)
    return saida


def processar_posicoes_historicas_agregadas(caminho_posicoes, linhas_origem):
    caminho_posicoes = Path(caminho_posicoes)

    if not caminho_posicoes.exists():
        tecnico = pd.DataFrame(
            [
                {
                    "origem_base_html": "10_4_posicoes_historicas",
                    "tema_html": "posicoes_historicas",
                    "tipo_base_composicao": "posicoes_historicas",
                    "modo_processamento_posicoes": "ausente",
                    "base_posicoes_disponivel": 0,
                    "linhas_origem_estimadas": 0,
                    "observacao": "Base histórica de posições não localizada.",
                }
            ]
        )
        vazio = pd.DataFrame()
        return tecnico, vazio, vazio, vazio, tecnico.copy(), "ausente", 0, 0

    arquivo = pq.ParquetFile(caminho_posicoes)
    colunas_disponiveis = list(arquivo.schema.names)
    colunas_leitura = selecionar_colunas_posicoes(colunas_disponiveis)

    if len(colunas_leitura) == 0:
        tecnico = pd.DataFrame(
            [
                {
                    "origem_base_html": "10_4_posicoes_historicas",
                    "tema_html": "posicoes_historicas",
                    "tipo_base_composicao": "posicoes_historicas",
                    "modo_processamento_posicoes": "sem_colunas_compativeis",
                    "base_posicoes_disponivel": 1,
                    "linhas_origem_estimadas": linhas_origem,
                    "observacao": "A base existe, mas não possui colunas compatíveis com o processamento agregado de posições.",
                }
            ]
        )
        vazio = pd.DataFrame()
        return tecnico, vazio, vazio, vazio, tecnico.copy(), "sem_colunas_compativeis", 0, 0

    registros_datas = []
    linhas_lidas = 0

    for lote in arquivo.iter_batches(batch_size=TAMANHO_LOTE_POSICOES, columns=colunas_leitura):
        df_lote = lote.to_pandas()
        df_pos = padronizar_lote_posicoes(df_lote)
        linhas_lidas += len(df_lote)
        del df_lote

        if len(df_pos) == 0:
            del df_pos
            gc.collect()
            continue

        df_datas = (
            df_pos[["chave_estrategia", "nome_exibicao_estrategia", "ano_mes", "data_posicao"]]
            .dropna(subset=["data_posicao"])
            .groupby(["chave_estrategia", "ano_mes"], dropna=False)
            .agg(
                nome_exibicao_estrategia=("nome_exibicao_estrategia", "first"),
                data_snapshot=("data_posicao", "max"),
            )
            .reset_index()
        )
        registros_datas.append(df_datas)
        del df_pos, df_datas
        gc.collect()

    if len(registros_datas) == 0:
        tecnico = pd.DataFrame(
            [
                {
                    "origem_base_html": "10_4_posicoes_historicas",
                    "tema_html": "posicoes_historicas",
                    "tipo_base_composicao": "posicoes_historicas",
                    "modo_processamento_posicoes": "sem_datas_validas",
                    "base_posicoes_disponivel": 1,
                    "linhas_origem_estimadas": linhas_origem,
                    "linhas_lidas_processamento": linhas_lidas,
                    "observacao": "Nenhuma data válida foi encontrada na base histórica de posições.",
                }
            ]
        )
        vazio = pd.DataFrame()
        return tecnico, vazio, vazio, vazio, tecnico.copy(), "sem_datas_validas", linhas_lidas, 0

    df_datas_snapshot = (
        pd.concat(registros_datas, ignore_index=True)
        .groupby(["chave_estrategia", "ano_mes"], dropna=False)
        .agg(
            nome_exibicao_estrategia=("nome_exibicao_estrategia", "first"),
            data_snapshot=("data_snapshot", "max"),
        )
        .reset_index()
    )
    del registros_datas
    gc.collect()

    chaves_snapshot = df_datas_snapshot[["chave_estrategia", "ano_mes", "data_snapshot"]].rename(columns={"data_snapshot": "data_posicao"})
    partes_snapshot = []
    linhas_snapshot = 0

    for lote in arquivo.iter_batches(batch_size=TAMANHO_LOTE_POSICOES, columns=colunas_leitura):
        df_lote = lote.to_pandas()
        df_pos = padronizar_lote_posicoes(df_lote)
        del df_lote

        if len(df_pos) == 0:
            del df_pos
            gc.collect()
            continue

        df_snapshot_lote = df_pos.merge(
            chaves_snapshot,
            on=["chave_estrategia", "ano_mes", "data_posicao"],
            how="inner",
        )
        del df_pos

        if len(df_snapshot_lote) > 0:
            linhas_snapshot += len(df_snapshot_lote)
            partes_snapshot.append(df_snapshot_lote)
        else:
            del df_snapshot_lote
        gc.collect()

    if len(partes_snapshot) == 0:
        tecnico = pd.DataFrame(
            [
                {
                    "origem_base_html": "10_4_posicoes_historicas",
                    "tema_html": "posicoes_historicas",
                    "tipo_base_composicao": "posicoes_historicas",
                    "modo_processamento_posicoes": "sem_snapshots_mensais",
                    "base_posicoes_disponivel": 1,
                    "linhas_origem_estimadas": linhas_origem,
                    "linhas_lidas_processamento": linhas_lidas,
                    "observacao": "Não foram encontrados registros nos últimos pregões mensais identificados.",
                }
            ]
        )
        vazio = pd.DataFrame()
        return tecnico, vazio, vazio, vazio, tecnico.copy(), "sem_snapshots_mensais", linhas_lidas, 0

    df_snapshot = pd.concat(partes_snapshot, ignore_index=True, sort=False)
    del partes_snapshot, chaves_snapshot
    gc.collect()

    agrupadores_empresa = [
        "chave_estrategia", "nome_exibicao_estrategia", "ano", "ano_mes", "data_posicao",
        "ticker", "issuer_code", "nome_empresa", "setor", "subsetor", "segmento",
    ]
    df_empresa = (
        df_snapshot
        .groupby(agrupadores_empresa, dropna=False)
        .agg(
            valor_posicao=("valor_posicao", "sum"),
            peso_posicao=("peso_posicao", "sum"),
            quantidade=("quantidade", "sum"),
            preco_medio=("preco", "mean"),
            n_linhas_posicao=("ticker", "size"),
        )
        .reset_index()
    )
    df_empresa["origem_base_html"] = "10_4_base_historico_posicoes"
    df_empresa["tema_html"] = "posicoes_historicas"
    df_empresa["tipo_base_composicao"] = "snapshot_mensal_empresa"
    df_empresa["modo_processamento_posicoes"] = "agregacao_incremental_snapshot_mensal"
    df_empresa["data_referencia_html"] = df_empresa["data_posicao"]
    df_empresa["linhas_origem_estimadas"] = int(linhas_origem)
    df_empresa["linhas_lidas_processamento"] = int(linhas_lidas)
    df_empresa["linhas_snapshot_mensal"] = int(linhas_snapshot)
    df_empresa = padronizar_identificacao(df_empresa)
    df_empresa = construir_classificacao_hierarquica(df_empresa, "empresa")

    agrupadores_setor = [
        "chave_estrategia", "nome_exibicao_estrategia", "ano", "ano_mes", "data_posicao",
        "setor", "subsetor", "segmento",
    ]
    df_setorial = (
        df_snapshot
        .groupby(agrupadores_setor, dropna=False)
        .agg(
            valor_posicao=("valor_posicao", "sum"),
            peso_posicao=("peso_posicao", "sum"),
            n_tickers=("ticker", "nunique"),
            n_empresas=("issuer_code", "nunique"),
            n_linhas_posicao=("ticker", "size"),
        )
        .reset_index()
    )
    df_setorial["origem_base_html"] = "10_4_base_historico_posicoes"
    df_setorial["tema_html"] = "posicoes_setorial_mensal"
    df_setorial["tipo_base_composicao"] = "snapshot_mensal_setor_subsetor_segmento"
    df_setorial["modo_processamento_posicoes"] = "agregacao_incremental_snapshot_mensal"
    df_setorial["data_referencia_html"] = df_setorial["data_posicao"]
    df_setorial["linhas_origem_estimadas"] = int(linhas_origem)
    df_setorial["linhas_lidas_processamento"] = int(linhas_lidas)
    df_setorial["linhas_snapshot_mensal"] = int(linhas_snapshot)
    df_setorial = padronizar_identificacao(df_setorial)
    df_setorial = construir_classificacao_hierarquica(df_setorial, "segmento")

    criterio_top = "peso_posicao" if df_empresa["peso_posicao"].notna().any() else "valor_posicao"
    df_top = df_empresa.copy()
    df_top["criterio_ranking_posicao"] = criterio_top
    df_top["valor_ranking_posicao"] = pd.to_numeric(df_top[criterio_top], errors="coerce")
    df_top = df_top.sort_values(["chave_estrategia", "ano_mes", "valor_ranking_posicao"], ascending=[True, True, False]).copy()
    df_top["rank_posicao_mes"] = df_top.groupby(["chave_estrategia", "ano_mes"], dropna=False).cumcount() + 1
    df_top = df_top.loc[df_top["rank_posicao_mes"] <= TOP_N_POSICOES_HISTORICAS].copy()
    df_top["tema_html"] = "top_posicoes_historicas"
    df_top["tipo_base_composicao"] = "top_posicoes_snapshot_mensal"

    resumo_base = (
        df_empresa
        .groupby(["chave_estrategia", "nome_exibicao_estrategia", "ano", "ano_mes", "data_posicao"], dropna=False)
        .agg(
            n_tickers=("ticker", "nunique"),
            n_empresas=("issuer_code", "nunique"),
            valor_posicao_total=("valor_posicao", "sum"),
            peso_posicao_total=("peso_posicao", "sum"),
            maior_peso_empresa=("peso_posicao", "max"),
            maior_valor_empresa=("valor_posicao", "max"),
        )
        .reset_index()
    )
    top5 = (
        df_empresa
        .sort_values(["chave_estrategia", "ano_mes", "peso_posicao"], ascending=[True, True, False])
        .assign(rank_top5=lambda x: x.groupby(["chave_estrategia", "ano_mes"], dropna=False).cumcount() + 1)
        .loc[lambda x: x["rank_top5"] <= 5]
        .groupby(["chave_estrategia", "ano_mes"], dropna=False)
        .agg(peso_top5_empresas=("peso_posicao", "sum"), valor_top5_empresas=("valor_posicao", "sum"))
        .reset_index()
    )
    df_resumo = resumo_base.merge(top5, on=["chave_estrategia", "ano_mes"], how="left")
    df_resumo["origem_base_html"] = "10_4_base_historico_posicoes"
    df_resumo["tema_html"] = "resumo_posicoes_historicas"
    df_resumo["tipo_base_composicao"] = "resumo_snapshot_mensal_posicoes"
    df_resumo["modo_processamento_posicoes"] = "agregacao_incremental_snapshot_mensal"
    df_resumo["linhas_origem_estimadas"] = int(linhas_origem)
    df_resumo["linhas_lidas_processamento"] = int(linhas_lidas)
    df_resumo["linhas_snapshot_mensal"] = int(linhas_snapshot)
    df_resumo["data_referencia_html"] = df_resumo["data_posicao"]
    df_resumo = padronizar_identificacao(df_resumo)

    df_empresa = ordenar_por_colunas(df_empresa, ["data_posicao", "chave_estrategia", "peso_posicao", "valor_posicao"])
    df_setorial = ordenar_por_colunas(df_setorial, ["data_posicao", "chave_estrategia", "setor", "subsetor", "segmento"])
    df_top = ordenar_por_colunas(df_top, ["data_posicao", "chave_estrategia", "rank_posicao_mes"])
    df_resumo = ordenar_por_colunas(df_resumo, ["data_posicao", "chave_estrategia"])

    del df_snapshot, resumo_base, top5
    gc.collect()

    return (
        df_empresa,
        df_setorial,
        df_top,
        df_resumo,
        pd.DataFrame(),
        "agregada_incremental_snapshot_mensal",
        linhas_lidas,
        linhas_snapshot,
    )

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Carga controlada das entradas e catálogo inicial
# ============================================================

print("\n[5/14] Carga controlada das entradas e catálogo inicial...")

catalogo_entradas = []
entradas_obrigatorias_ausentes = 0
entradas_opcionais_ausentes = 0

for entrada in ENTRADAS_15_4:
    if entrada["obrigatorio"] and not entrada["existe"]:
        entradas_obrigatorias_ausentes += 1
    if (not entrada["obrigatorio"]) and not entrada["existe"]:
        entradas_opcionais_ausentes += 1
    catalogo_entradas.append(
        {
            "tipo_item": "entrada",
            "chave": entrada["chave"],
            "etapa_origem": entrada["etapa"],
            "tema": entrada["tema"],
            "arquivo": Path(entrada["caminho_resolvido"]).name,
            "caminho": str(entrada["caminho_resolvido"]),
            "obrigatorio": bool(entrada["obrigatorio"]),
            "carregado": bool(entrada["existe"]),
            "linhas_origem_estimadas": int(entrada["linhas_origem_estimadas"]),
            "finalidade": entrada["finalidade"],
        }
    )

print(f"Entradas mapeadas                       : {len(ENTRADAS_15_4):,}")
print(f"Entradas existentes                     : {sum(1 for e in ENTRADAS_15_4 if e['existe']):,}")
print(f"Entradas opcionais ausentes             : {entradas_opcionais_ausentes:,}")
print(f"Entradas obrigatórias ausentes          : {entradas_obrigatorias_ausentes:,}")
print("OK")

if entradas_obrigatorias_ausentes > 0:
    raise FileNotFoundError("Há arquivos obrigatórios ausentes para a Etapa 15.4.")

entradas_por_chave = {entrada["chave"]: entrada for entrada in ENTRADAS_15_4}
saidas_info = []
metricas_infinitas = 0

# ============================================================
# 6) Consolidação leve de sinais, intensidade e aportes
# ============================================================

print("\n[6/14] Consolidação leve de sinais, intensidade e aportes...")

entrada_sinais = entradas_por_chave["intensidade_sinais_12_2"] if entradas_por_chave["intensidade_sinais_12_2"]["existe"] else entradas_por_chave["sinais_temporal_12_1"]
df_sinais, _, linhas_sinais_origem, _, modo_sinais = ler_parquet_controlado(entrada_sinais["caminho_resolvido"], obrigatorio=True)
df_sinais_aportes_html = padronizar_base_html(df_sinais, Path(entrada_sinais["caminho_resolvido"]).stem, "sinais_aportes", "sinais_aportes")
salvar_saida(df_sinais_aportes_html, CAMINHO_SINAIS_APORTES_HTML, "sinais_aportes", "Calendário de sinais, intensidade e metadados de mercado.", saidas_info)
metricas_infinitas += contar_valores_infinitos(df_sinais_aportes_html)
linhas_sinais_html = len(df_sinais_aportes_html)
del df_sinais, df_sinais_aportes_html
gc.collect()

entrada_aportes = entradas_por_chave["perfil_compras_aporte_12_3"]
df_aportes, _, linhas_aportes_origem, _, modo_aportes = ler_parquet_controlado(entrada_aportes["caminho_resolvido"], obrigatorio=True)
df_aportes_html = padronizar_base_html(df_aportes, "12_3_base_perfil_compras_aporte", "aportes", "aportes")
df_aportes_html = ordenar_por_colunas(df_aportes_html, ["data_aporte", "data_aporte_efetiva", "estrategia_referencia", "nome_exibicao_estrategia"])
salvar_saida(df_aportes_html, CAMINHO_APORTES_HTML, "aportes", "Tabela final de aportes e filtros de análise.", saidas_info)
salvar_saida(df_aportes_html.assign(origem_base_html="12_3_base_perfil_compras_aporte", tema_html="compras_aporte"), CAMINHO_COMPRAS_APORTE_HTML, "compras_aporte", "Resumo de compras por aporte.", saidas_info)
metricas_infinitas += contar_valores_infinitos(df_aportes_html)
linhas_aportes_html = len(df_aportes_html)
del df_aportes, df_aportes_html
gc.collect()

print(f"Sinais/aportes HTML                     : {linhas_sinais_html:,} linhas | modo={modo_sinais}")
print(f"Aportes HTML                            : {linhas_aportes_html:,} linhas | modo={modo_aportes}")
print("OK")

# ============================================================
# 7) Consolidação leve de compras por ticker e rankings
# ============================================================

print("\n[7/14] Consolidação leve de compras por ticker e rankings...")

entrada_compras = entradas_por_chave["perfil_compras_ticker_12_3"]
df_compras, _, _, _, modo_compras = ler_parquet_controlado(entrada_compras["caminho_resolvido"], obrigatorio=True)
df_compras_ticker_html = padronizar_base_html(df_compras, "12_3_base_perfil_compras_ticker", "compras_ticker", "compras_ticker")
df_compras_ticker_html = ordenar_por_colunas(df_compras_ticker_html, ["data_aporte", "estrategia_referencia", "ticker", "issuer_code"])
salvar_saida(df_compras_ticker_html, CAMINHO_COMPRAS_TICKER_HTML, "compras_ticker", "Tabela detalhada de tickers comprados por aporte.", saidas_info)
metricas_infinitas += contar_valores_infinitos(df_compras_ticker_html)
linhas_compras_ticker = len(df_compras_ticker_html)
del df_compras, df_compras_ticker_html
gc.collect()

entrada_top_tickers = entradas_por_chave["top_tickers_comprados_12_3"]
df_top_tickers, _, _, _, _ = ler_parquet_controlado(entrada_top_tickers["caminho_resolvido"], obrigatorio=True)
df_top_tickers_html = padronizar_base_html(df_top_tickers, "12_3_tbl_top_tickers_comprados", "top_tickers", "top_tickers_comprados")
salvar_saida(df_top_tickers_html, CAMINHO_TOP_TICKERS_HTML, "top_tickers", "Ranking de tickers mais comprados.", saidas_info)
metricas_infinitas += contar_valores_infinitos(df_top_tickers_html)
linhas_top_tickers = len(df_top_tickers_html)
del df_top_tickers, df_top_tickers_html
gc.collect()

print(f"Compras por ticker HTML                 : {linhas_compras_ticker:,} linhas | modo={modo_compras}")
print(f"Top tickers HTML                        : {linhas_top_tickers:,} linhas")
print("OK")

# ============================================================
# 8) Consolidação agregada de posições históricas
# ============================================================

print("\n[8/14] Consolidação agregada de posições históricas...")

entrada_posicoes = entradas_por_chave["posicoes_historicas_10_4"]
base_posicoes_disponivel = int(entrada_posicoes["existe"])
linhas_posicoes_origem = int(entrada_posicoes["linhas_origem_estimadas"])
posicoes_brutas_nao_copiadas_html = int(entrada_posicoes["existe"] and linhas_posicoes_origem > LIMITE_POSICOES_HTML)

(
    df_posicoes_html,
    df_posicoes_setorial_mensal_html,
    df_top_posicoes_historicas_html,
    df_resumo_posicoes_historicas_html,
    df_posicoes_tecnico,
    modo_posicoes,
    linhas_lidas_posicoes,
    linhas_snapshot_posicoes,
) = processar_posicoes_historicas_agregadas(entrada_posicoes["caminho_resolvido"], linhas_posicoes_origem)

if len(df_posicoes_html) == 0 and len(df_posicoes_tecnico) > 0:
    df_posicoes_html = df_posicoes_tecnico.copy()

salvar_saida(
    df_posicoes_html,
    CAMINHO_POSICOES_HTML,
    "posicoes_historicas",
    "Snapshot mensal agregado por empresa a partir da base histórica de posições.",
    saidas_info,
)
metricas_infinitas += contar_valores_infinitos(df_posicoes_html)
linhas_posicoes_html = len(df_posicoes_html)

salvar_saida(
    df_posicoes_setorial_mensal_html,
    CAMINHO_POSICOES_SETORIAL_MENSAL_HTML,
    "posicoes_setorial_mensal",
    "Snapshot mensal de posições agregado por setor, subsetor e segmento.",
    saidas_info,
)
metricas_infinitas += contar_valores_infinitos(df_posicoes_setorial_mensal_html)
linhas_posicoes_setorial_mensal = len(df_posicoes_setorial_mensal_html)

salvar_saida(
    df_top_posicoes_historicas_html,
    CAMINHO_TOP_POSICOES_HISTORICAS_HTML,
    "top_posicoes_historicas",
    "Top posições por estratégia e mês, derivado da base histórica de posições.",
    saidas_info,
)
metricas_infinitas += contar_valores_infinitos(df_top_posicoes_historicas_html)
linhas_top_posicoes_historicas = len(df_top_posicoes_historicas_html)

salvar_saida(
    df_resumo_posicoes_historicas_html,
    CAMINHO_RESUMO_POSICOES_HISTORICAS_HTML,
    "resumo_posicoes_historicas",
    "Resumo mensal de posições, diversificação e concentração derivado da base histórica.",
    saidas_info,
)
metricas_infinitas += contar_valores_infinitos(df_resumo_posicoes_historicas_html)
linhas_resumo_posicoes_historicas = len(df_resumo_posicoes_historicas_html)

# A base bruta diária de posições não é copiada para o HTML, mas suas informações são
# processadas em agregações mensais completas para evitar perda analítica e estouro de memória.
posicoes_omitidas_por_memoria = int(linhas_posicoes_html <= 1 and base_posicoes_disponivel == 1)

del df_posicoes_html, df_posicoes_setorial_mensal_html, df_top_posicoes_historicas_html, df_resumo_posicoes_historicas_html, df_posicoes_tecnico
gc.collect()

print(f"Posições históricas HTML                : {linhas_posicoes_html:,} linhas | modo={modo_posicoes}")
print(f"Posições setoriais mensais HTML         : {linhas_posicoes_setorial_mensal:,} linhas")
print(f"Top posições históricas HTML            : {linhas_top_posicoes_historicas:,} linhas")
print(f"Resumo posições históricas HTML         : {linhas_resumo_posicoes_historicas:,} linhas")
print(f"Linhas estimadas na origem              : {linhas_posicoes_origem:,}")
print(f"Linhas lidas no processamento           : {linhas_lidas_posicoes:,}")
print(f"Linhas retidas em snapshots mensais     : {linhas_snapshot_posicoes:,}")
print("OK")

# ============================================================
# 9) Consolidação leve de concentração por empresa
# ============================================================

print("\n[9/14] Consolidação leve de concentração por empresa...")

entrada_conc_empresa = entradas_por_chave["concentracao_empresa_aporte_12_4"]
df_conc_empresa, _, _, _, modo_conc_empresa = ler_parquet_controlado(entrada_conc_empresa["caminho_resolvido"], obrigatorio=True, ler_todas_ate=50000)
df_concentracao_empresa_html = padronizar_base_html(df_conc_empresa, "12_4_base_concentracao_empresa_aporte", "concentracao_empresa", "empresa_por_aporte")
df_concentracao_empresa_html = ordenar_por_colunas(df_concentracao_empresa_html, ["data_aporte", "estrategia_referencia", "issuer_code", "nome_empresa", "nome"])
salvar_saida(df_concentracao_empresa_html, CAMINHO_CONCENTRACAO_EMPRESA_HTML, "concentracao_empresa", "Concentração por empresa e aporte.", saidas_info)
metricas_infinitas += contar_valores_infinitos(df_concentracao_empresa_html)
linhas_conc_empresa = len(df_concentracao_empresa_html)
del df_conc_empresa, df_concentracao_empresa_html
gc.collect()

# Top empresas combina compras e concentração, que são bases pequenas.
tops_empresas = []
for chave, origem, tipo_base in [
    ("top_empresas_compradas_12_3", "12_3_tbl_top_empresas_compradas", "top_empresas_compradas"),
    ("top_empresas_concentracao_12_4", "12_4_tbl_top_empresas_concentracao", "top_empresas_concentracao"),
]:
    entrada = entradas_por_chave[chave]
    df_tmp, _, _, _, _ = ler_parquet_controlado(entrada["caminho_resolvido"], obrigatorio=True)
    tops_empresas.append(padronizar_base_html(df_tmp, origem, "top_empresas", tipo_base))
    del df_tmp

df_top_empresas_html = pd.concat(tops_empresas, ignore_index=True, sort=False) if len(tops_empresas) > 0 else pd.DataFrame()
salvar_saida(df_top_empresas_html, CAMINHO_TOP_EMPRESAS_HTML, "top_empresas", "Ranking de empresas por compras e concentração.", saidas_info)
metricas_infinitas += contar_valores_infinitos(df_top_empresas_html)
linhas_top_empresas = len(df_top_empresas_html)
del tops_empresas, df_top_empresas_html
gc.collect()

print(f"Concentração empresa HTML               : {linhas_conc_empresa:,} linhas | modo={modo_conc_empresa}")
print(f"Top empresas HTML                       : {linhas_top_empresas:,} linhas")
print("OK")

# ============================================================
# 10) Consolidação leve de concentração setorial hierárquica
# ============================================================

print("\n[10/14] Consolidação leve de concentração setorial hierárquica...")

entrada_conc_setorial = entradas_por_chave["concentracao_setorial_aporte_12_5"]
df_conc_setorial, _, _, _, modo_conc_setorial = ler_parquet_controlado(entrada_conc_setorial["caminho_resolvido"], obrigatorio=True, ler_todas_ate=50000)
df_concentracao_setorial_html = padronizar_base_html(df_conc_setorial, "12_5_base_concentracao_setorial_aporte", "concentracao_setorial", "classificacao_por_aporte")
df_concentracao_setorial_html = construir_classificacao_hierarquica(df_concentracao_setorial_html, "nao_classificado")
df_concentracao_setorial_html = ordenar_por_colunas(df_concentracao_setorial_html, ["data_aporte", "estrategia_referencia", "nivel_classificacao_html", "classificacao_hierarquica_html"])
salvar_saida(df_concentracao_setorial_html, CAMINHO_CONCENTRACAO_SETORIAL_HTML, "concentracao_setorial", "Concentração por setor, subsetor e segmento com hierarquia preservada.", saidas_info)
metricas_infinitas += contar_valores_infinitos(df_concentracao_setorial_html)
linhas_conc_setorial = len(df_concentracao_setorial_html)
del df_conc_setorial, df_concentracao_setorial_html
gc.collect()

tops_classificacoes = []
for chave, origem, nivel in [
    ("top_setores_comprados_12_3", "12_3_tbl_top_setores_comprados", "setor"),
    ("top_setores_concentracao_12_5", "12_5_tbl_top_setores_concentracao", "setor"),
    ("top_subsetores_concentracao_12_5", "12_5_tbl_top_subsetores_concentracao", "subsetor"),
    ("top_segmentos_concentracao_12_5", "12_5_tbl_top_segmentos_concentracao", "segmento"),
]:
    entrada = entradas_por_chave[chave]
    df_tmp, _, _, _, _ = ler_parquet_controlado(entrada["caminho_resolvido"], obrigatorio=True)
    df_tmp = padronizar_base_html(df_tmp, origem, "top_classificacoes_setoriais", chave)
    df_tmp = construir_classificacao_hierarquica(df_tmp, nivel)
    tops_classificacoes.append(df_tmp)
    del df_tmp

df_top_classificacoes_html = pd.concat(tops_classificacoes, ignore_index=True, sort=False) if len(tops_classificacoes) > 0 else pd.DataFrame()
df_top_classificacoes_html = ordenar_por_colunas(df_top_classificacoes_html, ["nivel_classificacao_html", "estrategia_referencia", "classificacao_hierarquica_html"])
salvar_saida(df_top_classificacoes_html, CAMINHO_TOP_CLASSIFICACOES_HTML, "top_classificacoes_setoriais", "Rankings hierárquicos de setor, subsetor e segmento.", saidas_info)
metricas_infinitas += contar_valores_infinitos(df_top_classificacoes_html)
linhas_top_classificacoes = len(df_top_classificacoes_html)
del tops_classificacoes, df_top_classificacoes_html
gc.collect()

print(f"Concentração setorial HTML              : {linhas_conc_setorial:,} linhas | modo={modo_conc_setorial}")
print(f"Top classificações setoriais HTML       : {linhas_top_classificacoes:,} linhas")
print("OK")

# ============================================================
# 11) Timeline leve de sinais, intensidade, aportes e compras
# ============================================================

print("\n[11/14] Timeline leve de sinais, intensidade, aportes e compras...")

timelines = []
for chave, origem, tema, tipo_base in [
    ("timeline_sinais_12_1", "12_1_tbl_base_grafico_timeline_sinais", "timeline_sinais_aportes", "timeline_sinais"),
    ("timeline_intensidade_12_2", "12_2_tbl_base_grafico_intensidade_timeline", "timeline_sinais_aportes", "timeline_intensidade"),
    ("timeline_compras_12_3", "12_3_tbl_base_grafico_perfil_compras_timeline", "timeline_sinais_aportes", "timeline_compras"),
    ("timeline_concentracao_setorial_12_5", "12_5_tbl_base_grafico_concentracao_setorial_timeline", "timeline_sinais_aportes", "timeline_concentracao_setorial"),
]:
    entrada = entradas_por_chave[chave]
    if entrada["existe"]:
        df_tmp, _, _, _, _ = ler_parquet_controlado(entrada["caminho_resolvido"], obrigatorio=False, ler_todas_ate=50000)
        timelines.append(padronizar_base_html(df_tmp, origem, tema, tipo_base))
        del df_tmp

df_timeline_sinais_aportes_html = pd.concat(timelines, ignore_index=True, sort=False) if len(timelines) > 0 else pd.DataFrame()
df_timeline_sinais_aportes_html = ordenar_por_colunas(df_timeline_sinais_aportes_html, ["data_referencia_html", "data_sinal", "data_aporte", "estrategia_referencia", "nome_exibicao_estrategia"])
salvar_saida(df_timeline_sinais_aportes_html, CAMINHO_TIMELINE_SINAIS_APORTES_HTML, "timeline_sinais_aportes", "Timeline integrada de sinais, intensidade, compras e concentração.", saidas_info)
metricas_infinitas += contar_valores_infinitos(df_timeline_sinais_aportes_html)
linhas_timeline = len(df_timeline_sinais_aportes_html)
del timelines, df_timeline_sinais_aportes_html
gc.collect()

print(f"Timeline sinais/aportes HTML            : {linhas_timeline:,} linhas")
print("OK")

# ============================================================
# 12) Auditoria consolidada com leitura seletiva
# ============================================================

print("\n[12/14] Auditoria consolidada com leitura seletiva...")

registros_auditoria = []
for diretorio in [DIR_ETAPA_10, DIR_ETAPA_12, DIR_ETAPA_13, DIR_ETAPA_14, DIR_ETAPA_15]:
    if not Path(diretorio).exists():
        continue
    for caminho in sorted(Path(diretorio).glob("*auditoria*.parquet")):
        if caminho.name == CAMINHO_AUDITORIA.name:
            continue
        try:
            df_aud, carregado, linhas_origem, _, modo = ler_parquet_controlado(
                caminho,
                colunas_preferenciais=COLUNAS_AUDITORIA,
                obrigatorio=False,
                max_linhas=1000,
                ler_todas_ate=20000,
            )
            if not carregado or len(df_aud) == 0:
                continue
            df_aud["arquivo_origem_auditoria"] = caminho.name
            df_aud["diretorio_origem_auditoria"] = caminho.parent.name
            df_aud["caminho_origem_auditoria"] = str(caminho)
            df_aud["linhas_origem_estimadas"] = linhas_origem
            df_aud["modo_leitura_auditoria"] = modo
            if "status" not in df_aud.columns:
                df_aud["status"] = "NAO_INFORMADO"
            registros_auditoria.append(df_aud)
            del df_aud
        except Exception as erro:
            registros_auditoria.append(
                pd.DataFrame(
                    [
                        {
                            "item": "erro_leitura_auditoria",
                            "valor": caminho.name,
                            "valor_referencia": "leitura válida",
                            "status": "ERRO",
                            "observacao": str(erro),
                            "arquivo_origem_auditoria": caminho.name,
                            "diretorio_origem_auditoria": caminho.parent.name,
                            "caminho_origem_auditoria": str(caminho),
                        }
                    ]
                )
            )

if len(registros_auditoria) > 0:
    df_auditoria_consolidada_html = pd.concat(registros_auditoria, ignore_index=True, sort=False)
else:
    df_auditoria_consolidada_html = pd.DataFrame(
        [
            {
                "item": "auditoria_consolidada_vazia",
                "valor": 0,
                "valor_referencia": "> 0",
                "status": "ERRO",
                "observacao": "Nenhum arquivo de auditoria foi localizado nas etapas monitoradas.",
            }
        ]
    )

df_auditoria_consolidada_html["status_normalizado"] = df_auditoria_consolidada_html["status"].astype(str).str.upper().str.strip()
df_auditoria_consolidada_html["flag_erro_bloqueante_origem"] = df_auditoria_consolidada_html["status_normalizado"].eq("ERRO")
salvar_saida(df_auditoria_consolidada_html, CAMINHO_AUDITORIA_CONSOLIDADA_HTML, "auditoria_consolidada", "Painel técnico de auditoria e rastreabilidade.", saidas_info)
linhas_auditoria_consolidada = len(df_auditoria_consolidada_html)
erros_auditoria_origem = int(df_auditoria_consolidada_html["flag_erro_bloqueante_origem"].sum())
del registros_auditoria, df_auditoria_consolidada_html
gc.collect()

print(f"Auditoria consolidada HTML              : {linhas_auditoria_consolidada:,} linhas")
print(f"Erros bloqueantes nas auditorias origem : {erros_auditoria_origem:,}")
print("OK")

# ============================================================
# 13) Resumo, catálogo, parâmetros e auditoria da subetapa
# ============================================================

print("\n[13/14] Resumo, catálogo, parâmetros e auditoria da subetapa...")

mapa_linhas_saida = {item["tema"]: item["linhas"] for item in saidas_info}

registros_resumo = [
    {"tema": "sinais_aportes", "descricao": "Sinais finais, intensidade e metadados temporais.", "linhas": mapa_linhas_saida.get("sinais_aportes", 0), "uso_html": "Bloco de calendário de sinais e cards de intensidade."},
    {"tema": "aportes", "descricao": "Perfil consolidado por aporte.", "linhas": mapa_linhas_saida.get("aportes", 0), "uso_html": "Tabela de aportes, filtros por estratégia, ano e regime."},
    {"tema": "compras_ticker", "descricao": "Compras por ticker, empresa e classificação setorial.", "linhas": mapa_linhas_saida.get("compras_ticker", 0), "uso_html": "Tabela detalhada de composição por aporte e ativo."},
    {"tema": "posicoes_historicas", "descricao": "Snapshots mensais por empresa derivados da base histórica de posições.", "linhas": mapa_linhas_saida.get("posicoes_historicas", 0), "uso_html": "Painel de composição histórica por empresa, sem carregar a base bruta diária."},
    {"tema": "posicoes_setorial_mensal", "descricao": "Snapshots mensais por setor, subsetor e segmento derivados das posições.", "linhas": mapa_linhas_saida.get("posicoes_setorial_mensal", 0), "uso_html": "Evolução histórica da exposição setorial preservando a hierarquia setor > subsetor > segmento."},
    {"tema": "top_posicoes_historicas", "descricao": "Top posições por estratégia e mês derivadas da base histórica.", "linhas": mapa_linhas_saida.get("top_posicoes_historicas", 0), "uso_html": "Tabela e gráficos de maiores posições mensais por estratégia."},
    {"tema": "resumo_posicoes_historicas", "descricao": "Resumo mensal de diversificação e concentração das posições.", "linhas": mapa_linhas_saida.get("resumo_posicoes_historicas", 0), "uso_html": "Cards e gráficos de número de empresas, top 5 e concentração ao longo do tempo."},
    {"tema": "concentracao_empresa", "descricao": "Concentração por empresa e rankings de exposição.", "linhas": mapa_linhas_saida.get("concentracao_empresa", 0), "uso_html": "Gráficos de concentração por empresa e top empresas."},
    {"tema": "concentracao_setorial", "descricao": "Concentração por setor, subsetor e segmento com hierarquia preservada.", "linhas": mapa_linhas_saida.get("concentracao_setorial", 0), "uso_html": "Gráficos hierárquicos por setor, subsetor e segmento."},
    {"tema": "auditoria", "descricao": "Auditorias consolidadas das etapas monitoradas.", "linhas": mapa_linhas_saida.get("auditoria_consolidada", 0), "uso_html": "Painel técnico de rastreabilidade e validação."},
]

df_resumo_composicao_html = pd.DataFrame(registros_resumo)
salvar_saida(df_resumo_composicao_html, CAMINHO_RESUMO_COMPOSICAO_HTML, "resumo_composicao_auditoria", "Resumo de navegação do bloco de composição e auditoria.", saidas_info)

PARAMETROS_15_4 = {
    "etapa": "15",
    "subetapa": "15.4",
    "nome_subetapa": "Bases de Composição e Auditoria",
    "modo_processamento": "limpeza preventiva do kernel, leitura seletiva de colunas e salvamento incremental",
    "limite_ler_todas_colunas": LIMITE_LER_TODAS_COLUNAS,
    "limite_posicoes_html": LIMITE_POSICOES_HTML,
    "tamanho_lote_posicoes": TAMANHO_LOTE_POSICOES,
    "top_n_posicoes_historicas": TOP_N_POSICOES_HISTORICAS,
    "tratamento_posicoes_historicas": "base bruta diária não é copiada para o HTML; a informação é extraída em snapshots mensais agregados por empresa e hierarquia setorial",
    "objetos_removidos_limpeza_preventiva": objetos_removidos_memoria,
    "memoria_tabular_estimativa_liberada_mb": round(memoria_estimativa_liberada_mb, 4),
    "fonte_sinais": "Etapas 12.1 e 12.2",
    "fonte_compras": "Etapa 12.3",
    "fonte_concentracao_empresa": "Etapa 12.4",
    "fonte_concentracao_setorial": "Etapa 12.5",
    "fonte_posicoes": "Etapa 10.4 processada incrementalmente em snapshots mensais quando disponível",
    "fonte_auditoria": "Arquivos de auditoria das etapas 10, 12, 13, 14 e 15",
    "tratamento_hierarquia_setorial": "Subsetor e segmento são identificados com hierarquia setor > subsetor > segmento.",
    "objetivo_html": "gerar bases finais para tabelas e painéis de sinais, composição, concentração e auditoria",
    "observacao_metodologica": "A subetapa organiza bases para visualização e rastreabilidade; não altera resultados das etapas anteriores.",
}

df_parametros = pd.DataFrame([{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_15_4.items()])
salvar_saida(df_parametros, CAMINHO_PARAMETROS, "parametros", "Parâmetros metodológicos da subetapa.", saidas_info)

catalogo_saidas = pd.DataFrame(
    [
        {
            "tipo_item": "saida",
            "chave": item["tema"],
            "etapa_origem": 15,
            "tema": item["tema"],
            "arquivo": item["arquivo"],
            "caminho": str(DIR_ETAPA_15 / item["arquivo"]),
            "obrigatorio": True,
            "carregado": True,
            "linhas": item["linhas"],
            "colunas": item["colunas"],
            "finalidade": item["uso_html"],
        }
        for item in saidas_info
    ]
)

df_catalogo = pd.concat([pd.DataFrame(catalogo_entradas), catalogo_saidas], ignore_index=True, sort=False)
salvar_parquet_compat(df_catalogo, CAMINHO_CATALOGO, index=False)

linhas_catalogo = len(df_catalogo)
linhas_resumo = len(df_resumo_composicao_html)
linhas_parametros = len(df_parametros)

del df_resumo_composicao_html, df_parametros, catalogo_saidas, df_catalogo
gc.collect()

linhas_sinais = mapa_linhas_saida.get("sinais_aportes", 0)
linhas_aportes = mapa_linhas_saida.get("aportes", 0)
linhas_compras = mapa_linhas_saida.get("compras_ticker", 0)
linhas_posicoes = mapa_linhas_saida.get("posicoes_historicas", 0)
linhas_posicoes_setorial_mensal = mapa_linhas_saida.get("posicoes_setorial_mensal", 0)
linhas_top_posicoes_historicas = mapa_linhas_saida.get("top_posicoes_historicas", 0)
linhas_resumo_posicoes_historicas = mapa_linhas_saida.get("resumo_posicoes_historicas", 0)
linhas_conc_empresa = mapa_linhas_saida.get("concentracao_empresa", 0)
linhas_conc_setorial = mapa_linhas_saida.get("concentracao_setorial", 0)
linhas_top_tickers = mapa_linhas_saida.get("top_tickers", 0)
linhas_top_empresas = mapa_linhas_saida.get("top_empresas", 0)
linhas_top_classificacoes = mapa_linhas_saida.get("top_classificacoes_setoriais", 0)
linhas_timeline = mapa_linhas_saida.get("timeline_sinais_aportes", 0)

linhas_auditoria = [
    {"item": "objetos_removidos_limpeza_preventiva", "valor": objetos_removidos_memoria, "valor_referencia": ">= 0", "status": "OK", "observacao": "Quantidade de objetos tabulares/intermediários removidos do kernel antes da subetapa."},
    {"item": "memoria_tabular_estimativa_liberada_mb", "valor": round(memoria_estimativa_liberada_mb, 4), "valor_referencia": ">= 0", "status": "OK", "observacao": "Estimativa de memória tabular liberada antes da execução da subetapa."},
    {"item": "arquivos_obrigatorios_ausentes", "valor": entradas_obrigatorias_ausentes, "valor_referencia": "0", "status": "OK" if entradas_obrigatorias_ausentes == 0 else "ERRO", "observacao": "Todos os arquivos obrigatórios da subetapa devem estar disponíveis."},
    {"item": "linhas_sinais_aportes_html", "valor": linhas_sinais, "valor_referencia": "> 0", "status": "OK" if linhas_sinais > 0 else "ERRO", "observacao": "A base de sinais e intensidade deve ser gerada."},
    {"item": "linhas_aportes_html", "valor": linhas_aportes, "valor_referencia": "> 0", "status": "OK" if linhas_aportes > 0 else "ERRO", "observacao": "A base de aportes deve ser gerada."},
    {"item": "linhas_compras_ticker_html", "valor": linhas_compras, "valor_referencia": "> 0", "status": "OK" if linhas_compras > 0 else "ERRO", "observacao": "A base de compras por ticker deve ser gerada."},
    {"item": "linhas_posicoes_historicas_html", "valor": linhas_posicoes, "valor_referencia": "> 1", "status": "OK" if linhas_posicoes > 1 else "ERRO", "observacao": "A base histórica de posições deve gerar snapshots mensais por empresa, não apenas registro técnico."},
    {"item": "linhas_posicoes_setorial_mensal_html", "valor": linhas_posicoes_setorial_mensal, "valor_referencia": "> 0", "status": "OK" if linhas_posicoes_setorial_mensal > 0 else "ERRO", "observacao": "A base histórica de posições deve gerar agregação setorial mensal."},
    {"item": "linhas_top_posicoes_historicas_html", "valor": linhas_top_posicoes_historicas, "valor_referencia": "> 0", "status": "OK" if linhas_top_posicoes_historicas > 0 else "ERRO", "observacao": "A base histórica de posições deve gerar ranking de maiores posições mensais."},
    {"item": "linhas_resumo_posicoes_historicas_html", "valor": linhas_resumo_posicoes_historicas, "valor_referencia": "> 0", "status": "OK" if linhas_resumo_posicoes_historicas > 0 else "ERRO", "observacao": "A base histórica de posições deve gerar resumo mensal de diversificação e concentração."},
    {"item": "posicoes_brutas_nao_copiadas_html", "valor": posicoes_brutas_nao_copiadas_html, "valor_referencia": "0 ou 1", "status": "OK", "observacao": "Quando igual a 1, a base bruta diária não foi copiada para o HTML, mas foi processada em agregações mensais."},
    {"item": "posicoes_omitidas_por_memoria", "valor": posicoes_omitidas_por_memoria, "valor_referencia": "0", "status": "OK" if posicoes_omitidas_por_memoria == 0 else "ERRO", "observacao": "A base histórica não deve ser simplesmente omitida; deve gerar agregações consolidadas."},
    {"item": "linhas_concentracao_empresa_html", "valor": linhas_conc_empresa, "valor_referencia": "> 0", "status": "OK" if linhas_conc_empresa > 0 else "ERRO", "observacao": "A base de concentração por empresa deve ser gerada."},
    {"item": "linhas_concentracao_setorial_html", "valor": linhas_conc_setorial, "valor_referencia": "> 0", "status": "OK" if linhas_conc_setorial > 0 else "ERRO", "observacao": "A base de concentração setorial deve ser gerada."},
    {"item": "linhas_top_tickers_html", "valor": linhas_top_tickers, "valor_referencia": "> 0", "status": "OK" if linhas_top_tickers > 0 else "ERRO", "observacao": "A tabela de top tickers deve ser gerada."},
    {"item": "linhas_top_empresas_html", "valor": linhas_top_empresas, "valor_referencia": "> 0", "status": "OK" if linhas_top_empresas > 0 else "ERRO", "observacao": "A tabela de top empresas deve ser gerada."},
    {"item": "linhas_top_classificacoes_html", "valor": linhas_top_classificacoes, "valor_referencia": "> 0", "status": "OK" if linhas_top_classificacoes > 0 else "ERRO", "observacao": "A tabela de top classificações setoriais deve ser gerada."},
    {"item": "linhas_timeline_sinais_aportes_html", "valor": linhas_timeline, "valor_referencia": "> 0", "status": "OK" if linhas_timeline > 0 else "ERRO", "observacao": "A timeline integrada deve ser gerada."},
    {"item": "linhas_auditoria_consolidada_html", "valor": linhas_auditoria_consolidada, "valor_referencia": "> 0", "status": "OK" if linhas_auditoria_consolidada > 0 else "ERRO", "observacao": "A auditoria consolidada deve ser gerada."},
    {"item": "erros_auditoria_origem", "valor": erros_auditoria_origem, "valor_referencia": "0", "status": "OK" if erros_auditoria_origem == 0 else "ALERTA", "observacao": "Erros de auditorias anteriores são preservados como rastreabilidade, mas não bloqueiam a geração da 15.4."},
    {"item": "linhas_resumo_composicao_html", "valor": linhas_resumo, "valor_referencia": "> 0", "status": "OK" if linhas_resumo > 0 else "ERRO", "observacao": "A tabela de resumo deve ser gerada."},
    {"item": "linhas_catalogo", "valor": linhas_catalogo, "valor_referencia": "> 0", "status": "OK" if linhas_catalogo > 0 else "ERRO", "observacao": "O catálogo de entradas e saídas deve ser gerado."},
    {"item": "metricas_infinitas", "valor": metricas_infinitas, "valor_referencia": "0", "status": "OK" if metricas_infinitas == 0 else "ERRO", "observacao": "As bases consolidadas não devem conter valores infinitos."},
]

df_auditoria = pd.DataFrame(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())
salvar_parquet_compat(df_auditoria, CAMINHO_AUDITORIA, index=False)

print(f"Resumo composição/auditoria             : {linhas_resumo:,} linhas")
print(f"Catálogo composição/auditoria           : {linhas_catalogo:,} linhas")
print(f"Parâmetros metodológicos                : {linhas_parametros:,} linhas")
print(f"Itens de auditoria                      : {len(df_auditoria):,}")
print(f"Erros bloqueantes                       : {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 14) Validação final e prints de controle
# ============================================================

print("\n[14/14] Validação final e prints de controle...")

print("\nAuditoria de validação das bases de composição e auditoria:")
print(df_auditoria.to_string(index=False))

print("\nCatálogo de outputs da Etapa 15.4:")
print(pd.DataFrame(saidas_info).to_string(index=False))

print("\nResumo executivo da subetapa:")
print(pd.DataFrame(registros_resumo).to_string(index=False))

print("\nArquivos salvos na subetapa 15.4:")
for item in saidas_info:
    print(f"- {DIR_ETAPA_15 / item['arquivo']}")
print(f"- {CAMINHO_CATALOGO}")
print(f"- {CAMINHO_AUDITORIA}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 15.4 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 15.4 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 15.4 - BASES DE COMPOSIÇÃO E AUDITORIA

[1/14] Validação inicial do ambiente...
Objetos tabulares/intermediários removidos do kernel: 588
Memória tabular estimada liberada.................: 3,373.38 MB
OK

[2/14] Definição determinística dos diretórios e caminhos de saída...
Diretório da Etapa 10 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_10
Diretório da Etapa 12 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_12
Diretório da Etapa 13 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_13
Diretório da Etapa 14 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_14
Diretório da Etapa 15 : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_an

## Etapa 15.5) Consolidação dos Arquivos Finais do HTML

In [83]:
%%time
# ============================================================
# Etapa 15.5) Consolidação dos Arquivos Finais do HTML
# ============================================================

print("=" * 100)
print("INICIANDO ETAPA 15.5 - CONSOLIDAÇÃO DOS ARQUIVOS FINAIS DO HTML")
print("=" * 100)

# Esta subetapa organiza os arquivos finais já produzidos nas subetapas 15.1 a 15.4
# em uma estrutura única para consumo pela seção final do HTML. Não há recálculo de
# backtest, inferência, risco, performance ou composição.

# ============================================================
# 1) Validação inicial do ambiente e limpeza preventiva leve
# ============================================================

print("\n[1/11] Validação inicial do ambiente e limpeza preventiva leve...")

if "pd" not in globals():
    raise NameError("A biblioteca pandas deve estar importada no ambiente antes da execução desta subetapa.")
if "np" not in globals():
    raise NameError("A biblioteca numpy deve estar importada no ambiente antes da execução desta subetapa.")
if "Path" not in globals():
    raise NameError("Path deve estar importado no ambiente antes da execução desta subetapa.")
if "DIRETORIOS_PROJETO" not in globals():
    raise NameError("O dicionário DIRETORIOS_PROJETO não está disponível no ambiente atual.")
if "salvar_dataframe" not in globals():
    raise NameError("A função salvar_dataframe não está disponível no ambiente atual.")

OBJETOS_PRESERVAR_15_5 = {
    "pd", "np", "Path", "DIRETORIOS_PROJETO", "salvar_dataframe",
    "os", "shutil", "json", "gc", "pq", "datetime", "time",
}

prefixos_limpeza_15_5 = (
    "df_",
    "base_",
    "bases_",
    "registros_",
    "linhas_",
    "catalogo_",
    "manifesto_",
)

objetos_removidos_15_5 = 0
memoria_liberada_mb_15_5 = 0.0

for nome_objeto in list(globals().keys()):
    if nome_objeto in OBJETOS_PRESERVAR_15_5:
        continue
    if nome_objeto.startswith("_"):
        continue
    if nome_objeto.startswith(prefixos_limpeza_15_5):
        objeto = globals().get(nome_objeto)
        try:
            if hasattr(objeto, "memory_usage"):
                memoria_objeto = objeto.memory_usage(deep=True)
                if hasattr(memoria_objeto, "sum"):
                    memoria_liberada_mb_15_5 += float(memoria_objeto.sum()) / (1024 ** 2)
                else:
                    memoria_liberada_mb_15_5 += float(memoria_objeto) / (1024 ** 2)
        except Exception:
            pass
        try:
            del globals()[nome_objeto]
            objetos_removidos_15_5 += 1
        except Exception:
            pass

gc.collect()

print(f"Objetos tabulares/intermediários removidos do kernel: {objetos_removidos_15_5:,}")
print(f"Memória tabular estimada liberada.................: {memoria_liberada_mb_15_5:,.2f} MB")
print("OK")

# ============================================================
# 2) Definição determinística dos diretórios finais
# ============================================================

print("\n[2/11] Definição determinística dos diretórios finais...")

DIR_RESULTADOS = Path(DIRETORIOS_PROJETO["resultados"])
DIR_ETAPA_15 = Path(DIRETORIOS_PROJETO.get("etapa_15", DIR_RESULTADOS / "etapa_15"))
DIR_ETAPA_15.mkdir(parents=True, exist_ok=True)

if "html_final" in DIRETORIOS_PROJETO:
    DIR_HTML_FINAL = Path(DIRETORIOS_PROJETO["html_final"])
else:
    DIR_HTML_FINAL = DIR_RESULTADOS.parent / "html_final"
    DIRETORIOS_PROJETO["html_final"] = DIR_HTML_FINAL

DIR_HTML_DADOS = DIR_HTML_FINAL / "dados"
DIR_HTML_ASSETS = DIR_HTML_FINAL / "assets"
DIR_HTML_MANIFESTOS = DIR_HTML_DADOS / "00_manifestos"
DIR_HTML_RESUMO = DIR_HTML_DADOS / "01_resumo"
DIR_HTML_PERFORMANCE = DIR_HTML_DADOS / "02_performance"
DIR_HTML_RISCO = DIR_HTML_DADOS / "03_risco"
DIR_HTML_COMPOSICAO = DIR_HTML_DADOS / "04_composicao_auditoria"
DIR_HTML_APOIO = DIR_HTML_DADOS / "05_apoio"

DIRETORIOS_PROJETO["html_final_dados"] = DIR_HTML_DADOS
DIRETORIOS_PROJETO["html_final_assets"] = DIR_HTML_ASSETS

DIRETORIOS_HTML_15_5 = [
    DIR_HTML_FINAL,
    DIR_HTML_DADOS,
    DIR_HTML_ASSETS,
    DIR_HTML_MANIFESTOS,
    DIR_HTML_RESUMO,
    DIR_HTML_PERFORMANCE,
    DIR_HTML_RISCO,
    DIR_HTML_COMPOSICAO,
    DIR_HTML_APOIO,
]

for diretorio in DIRETORIOS_HTML_15_5:
    diretorio.mkdir(parents=True, exist_ok=True)

CAMINHO_MANIFESTO_ETAPA = DIR_ETAPA_15 / "15_5_tbl_manifesto_arquivos_html.parquet"
CAMINHO_CATALOGO_ETAPA = DIR_ETAPA_15 / "15_5_tbl_catalogo_dados_html.parquet"
CAMINHO_RESUMO_ETAPA = DIR_ETAPA_15 / "15_5_tbl_resumo_estrutura_html.parquet"
CAMINHO_PARAMETROS_ETAPA = DIR_ETAPA_15 / "15_5_tbl_parametros_consolidacao_html.parquet"
CAMINHO_AUDITORIA_ETAPA = DIR_ETAPA_15 / "15_5_tbl_auditoria_validacao_consolidacao_html.parquet"

CAMINHO_MANIFESTO_HTML_PARQUET = DIR_HTML_MANIFESTOS / "manifesto_arquivos_html.parquet"
CAMINHO_MANIFESTO_HTML_JSON = DIR_HTML_MANIFESTOS / "manifesto_arquivos_html.json"
CAMINHO_CATALOGO_HTML_PARQUET = DIR_HTML_MANIFESTOS / "catalogo_dados_html.parquet"
CAMINHO_CATALOGO_HTML_JSON = DIR_HTML_MANIFESTOS / "catalogo_dados_html.json"
CAMINHO_RESUMO_HTML_PARQUET = DIR_HTML_MANIFESTOS / "resumo_estrutura_html.parquet"
CAMINHO_AUDITORIA_HTML_PARQUET = DIR_HTML_MANIFESTOS / "auditoria_consolidacao_html.parquet"
CAMINHO_PARAMETROS_HTML_PARQUET = DIR_HTML_MANIFESTOS / "parametros_consolidacao_html.parquet"

print(f"Diretório da Etapa 15              : {DIR_ETAPA_15}")
print(f"Diretório HTML final               : {DIR_HTML_FINAL}")
print(f"Diretório dados HTML               : {DIR_HTML_DADOS}")
print(f"Diretório manifestos HTML          : {DIR_HTML_MANIFESTOS}")
print(f"Diretório resumo HTML              : {DIR_HTML_RESUMO}")
print(f"Diretório performance HTML         : {DIR_HTML_PERFORMANCE}")
print(f"Diretório risco HTML               : {DIR_HTML_RISCO}")
print(f"Diretório composição/auditoria HTML: {DIR_HTML_COMPOSICAO}")
print(f"Diretório apoio HTML               : {DIR_HTML_APOIO}")
print("OK")

# ============================================================
# 3) Catálogo determinístico de arquivos a consolidar
# ============================================================

print("\n[3/11] Catálogo determinístico de arquivos a consolidar...")

ARQUIVOS_15_5 = [
    # 15.1 - Tabelas-resumo, inferência, robustez e apoio executivo
    {"subetapa": "15.1", "arquivo": "15_1_tbl_resumo_executivo_html.parquet", "subpasta": "01_resumo", "tema": "resumo_executivo", "secao_html": "Resumo Executivo", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Resumo executivo final por estratégia."},
    {"subetapa": "15.1", "arquivo": "15_1_tbl_metricas_principais_estrategias_html.parquet", "subpasta": "01_resumo", "tema": "metricas_principais", "secao_html": "Resumo Executivo", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Métricas oficiais de retorno, risco e eficiência."},
    {"subetapa": "15.1", "arquivo": "15_1_tbl_comparacao_benchmarks_html.parquet", "subpasta": "01_resumo", "tema": "comparacao_benchmarks", "secao_html": "Performance e Benchmarks", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Comparações finais contra Ibovespa e CDI."},
    {"subetapa": "15.1", "arquivo": "15_1_tbl_comparacao_controles_html.parquet", "subpasta": "01_resumo", "tema": "comparacao_controles", "secao_html": "Controles", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Comparações finais contra controles aleatórios e mensais."},
    {"subetapa": "15.1", "arquivo": "15_1_tbl_comparacao_capitulacao_euforia_html.parquet", "subpasta": "01_resumo", "tema": "comparacao_capitulacao_euforia", "secao_html": "Capitulação vs Euforia", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Comparação direta entre Capitulação e Euforia."},
    {"subetapa": "15.1", "arquivo": "15_1_tbl_inferencia_estatistica_html.parquet", "subpasta": "01_resumo", "tema": "inferencia_estatistica", "secao_html": "Inferência Estatística", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Evidências estatísticas consolidadas."},
    {"subetapa": "15.1", "arquivo": "15_1_tbl_robustez_sensibilidade_html.parquet", "subpasta": "01_resumo", "tema": "robustez_sensibilidade", "secao_html": "Robustez", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Robustez e sensibilidade das estratégias."},
    {"subetapa": "15.1", "arquivo": "15_1_tbl_catalogo_tabelas_html.parquet", "subpasta": "05_apoio", "tema": "catalogo_tabelas_15_1", "secao_html": "Apoio Técnico", "tipo_conteudo": "catalogo", "obrigatorio": True, "descricao": "Catálogo da subetapa 15.1."},
    {"subetapa": "15.1", "arquivo": "15_1_tbl_parametros_tabelas_resumo_html.parquet", "subpasta": "05_apoio", "tema": "parametros_15_1", "secao_html": "Apoio Técnico", "tipo_conteudo": "parametros", "obrigatorio": True, "descricao": "Parâmetros da subetapa 15.1."},
    {"subetapa": "15.1", "arquivo": "15_1_tbl_auditoria_validacao_tabelas_resumo_html.parquet", "subpasta": "05_apoio", "tema": "auditoria_15_1", "secao_html": "Apoio Técnico", "tipo_conteudo": "auditoria", "obrigatorio": True, "descricao": "Auditoria da subetapa 15.1."},

    # 15.2 - Performance
    {"subetapa": "15.2", "arquivo": "15_2_base_curvas_patrimoniais_html.parquet", "subpasta": "02_performance", "tema": "curvas_patrimoniais", "secao_html": "Performance", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Curvas patrimoniais diárias."},
    {"subetapa": "15.2", "arquivo": "15_2_base_curvas_indexadas_html.parquet", "subpasta": "02_performance", "tema": "curvas_indexadas", "secao_html": "Performance", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Curvas de patrimônio em base 100."},
    {"subetapa": "15.2", "arquivo": "15_2_base_retornos_periodicos_html.parquet", "subpasta": "02_performance", "tema": "retornos_periodicos", "secao_html": "Performance", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Retornos diários, mensais e anuais."},
    {"subetapa": "15.2", "arquivo": "15_2_base_retornos_acumulados_html.parquet", "subpasta": "02_performance", "tema": "retornos_acumulados", "secao_html": "Performance", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Retornos acumulados em base 100."},
    {"subetapa": "15.2", "arquivo": "15_2_base_desempenho_vs_benchmarks_html.parquet", "subpasta": "02_performance", "tema": "desempenho_vs_benchmarks", "secao_html": "Performance e Benchmarks", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Desempenho relativo contra benchmarks."},
    {"subetapa": "15.2", "arquivo": "15_2_base_desempenho_vs_controles_html.parquet", "subpasta": "02_performance", "tema": "desempenho_vs_controles", "secao_html": "Performance e Controles", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Desempenho relativo contra controles."},
    {"subetapa": "15.2", "arquivo": "15_2_base_desempenho_capitulacao_vs_euforia_html.parquet", "subpasta": "02_performance", "tema": "desempenho_capitulacao_vs_euforia", "secao_html": "Capitulação vs Euforia", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Desempenho direto Capitulação vs Euforia."},
    {"subetapa": "15.2", "arquivo": "15_2_tbl_distribuicao_retornos_html.parquet", "subpasta": "02_performance", "tema": "distribuicao_retornos", "secao_html": "Performance", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Distribuição de retornos por estratégia."},
    {"subetapa": "15.2", "arquivo": "15_2_tbl_resumo_graficos_performance_html.parquet", "subpasta": "05_apoio", "tema": "resumo_graficos_performance", "secao_html": "Apoio Técnico", "tipo_conteudo": "resumo", "obrigatorio": True, "descricao": "Resumo das bases de performance."},
    {"subetapa": "15.2", "arquivo": "15_2_tbl_catalogo_graficos_performance_html.parquet", "subpasta": "05_apoio", "tema": "catalogo_performance_15_2", "secao_html": "Apoio Técnico", "tipo_conteudo": "catalogo", "obrigatorio": True, "descricao": "Catálogo da subetapa 15.2."},
    {"subetapa": "15.2", "arquivo": "15_2_tbl_parametros_graficos_performance_html.parquet", "subpasta": "05_apoio", "tema": "parametros_15_2", "secao_html": "Apoio Técnico", "tipo_conteudo": "parametros", "obrigatorio": True, "descricao": "Parâmetros da subetapa 15.2."},
    {"subetapa": "15.2", "arquivo": "15_2_tbl_auditoria_validacao_graficos_performance_html.parquet", "subpasta": "05_apoio", "tema": "auditoria_15_2", "secao_html": "Apoio Técnico", "tipo_conteudo": "auditoria", "obrigatorio": True, "descricao": "Auditoria da subetapa 15.2."},

    # 15.3 - Risco
    {"subetapa": "15.3", "arquivo": "15_3_base_drawdown_html.parquet", "subpasta": "03_risco", "tema": "drawdown", "secao_html": "Risco", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Série diária de drawdown."},
    {"subetapa": "15.3", "arquivo": "15_3_tbl_eventos_drawdown_html.parquet", "subpasta": "03_risco", "tema": "eventos_drawdown", "secao_html": "Risco", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Eventos de drawdown e recuperação."},
    {"subetapa": "15.3", "arquivo": "15_3_base_volatilidade_rolling_html.parquet", "subpasta": "03_risco", "tema": "volatilidade_rolling", "secao_html": "Risco", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Volatilidade rolling multifrequência."},
    {"subetapa": "15.3", "arquivo": "15_3_base_metricas_risco_html.parquet", "subpasta": "03_risco", "tema": "metricas_risco", "secao_html": "Risco", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Métricas oficiais de risco em formato amplo."},
    {"subetapa": "15.3", "arquivo": "15_3_base_metricas_risco_long_html.parquet", "subpasta": "03_risco", "tema": "metricas_risco_long", "secao_html": "Risco", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Métricas de risco em formato longo."},
    {"subetapa": "15.3", "arquivo": "15_3_base_risco_vs_benchmarks_html.parquet", "subpasta": "03_risco", "tema": "risco_vs_benchmarks", "secao_html": "Risco e Benchmarks", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Risco relativo contra benchmarks."},
    {"subetapa": "15.3", "arquivo": "15_3_base_risco_vs_controles_html.parquet", "subpasta": "03_risco", "tema": "risco_vs_controles", "secao_html": "Risco e Controles", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Risco relativo contra controles."},
    {"subetapa": "15.3", "arquivo": "15_3_base_risco_capitulacao_vs_euforia_html.parquet", "subpasta": "03_risco", "tema": "risco_capitulacao_vs_euforia", "secao_html": "Capitulação vs Euforia", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Risco relativo entre Capitulação e Euforia."},
    {"subetapa": "15.3", "arquivo": "15_3_tbl_resumo_graficos_risco_html.parquet", "subpasta": "05_apoio", "tema": "resumo_graficos_risco", "secao_html": "Apoio Técnico", "tipo_conteudo": "resumo", "obrigatorio": True, "descricao": "Resumo das bases de risco."},
    {"subetapa": "15.3", "arquivo": "15_3_tbl_catalogo_graficos_risco_html.parquet", "subpasta": "05_apoio", "tema": "catalogo_risco_15_3", "secao_html": "Apoio Técnico", "tipo_conteudo": "catalogo", "obrigatorio": True, "descricao": "Catálogo da subetapa 15.3."},
    {"subetapa": "15.3", "arquivo": "15_3_tbl_parametros_graficos_risco_html.parquet", "subpasta": "05_apoio", "tema": "parametros_15_3", "secao_html": "Apoio Técnico", "tipo_conteudo": "parametros", "obrigatorio": True, "descricao": "Parâmetros da subetapa 15.3."},
    {"subetapa": "15.3", "arquivo": "15_3_tbl_auditoria_validacao_graficos_risco_html.parquet", "subpasta": "05_apoio", "tema": "auditoria_15_3", "secao_html": "Apoio Técnico", "tipo_conteudo": "auditoria", "obrigatorio": True, "descricao": "Auditoria da subetapa 15.3."},

    # 15.4 - Composição e auditoria
    {"subetapa": "15.4", "arquivo": "15_4_base_sinais_aportes_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "sinais_aportes", "secao_html": "Sinais e Composição", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Calendário de sinais, intensidade e metadados."},
    {"subetapa": "15.4", "arquivo": "15_4_base_aportes_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "aportes", "secao_html": "Sinais e Composição", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Base final de aportes."},
    {"subetapa": "15.4", "arquivo": "15_4_base_compras_aporte_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "compras_aporte", "secao_html": "Sinais e Composição", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Compras consolidadas por aporte."},
    {"subetapa": "15.4", "arquivo": "15_4_base_compras_ticker_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "compras_ticker", "secao_html": "Sinais e Composição", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Compras detalhadas por ticker."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_top_tickers_comprados_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "top_tickers", "secao_html": "Sinais e Composição", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Ranking de tickers comprados."},
    {"subetapa": "15.4", "arquivo": "15_4_base_posicoes_historicas_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "posicoes_historicas", "secao_html": "Composição Histórica", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Snapshots mensais por empresa derivados das posições históricas."},
    {"subetapa": "15.4", "arquivo": "15_4_base_posicoes_setorial_mensal_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "posicoes_setorial_mensal", "secao_html": "Composição Histórica", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Snapshots mensais por setor, subsetor e segmento."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_top_posicoes_historicas_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "top_posicoes_historicas", "secao_html": "Composição Histórica", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Top posições mensais por estratégia."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_resumo_posicoes_historicas_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "resumo_posicoes_historicas", "secao_html": "Composição Histórica", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Resumo mensal de diversificação e concentração das posições."},
    {"subetapa": "15.4", "arquivo": "15_4_base_concentracao_empresa_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "concentracao_empresa", "secao_html": "Sinais e Composição", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Concentração por empresa e aporte."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_top_empresas_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "top_empresas", "secao_html": "Sinais e Composição", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Ranking de empresas."},
    {"subetapa": "15.4", "arquivo": "15_4_base_concentracao_setorial_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "concentracao_setorial", "secao_html": "Sinais e Composição", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Concentração por setor, subsetor e segmento."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_top_classificacoes_setoriais_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "top_classificacoes_setoriais", "secao_html": "Sinais e Composição", "tipo_conteudo": "tabela", "obrigatorio": True, "descricao": "Rankings hierárquicos setoriais."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_timeline_sinais_aportes_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "timeline_sinais_aportes", "secao_html": "Sinais e Composição", "tipo_conteudo": "base_grafico", "obrigatorio": True, "descricao": "Timeline integrada de sinais, aportes, compras e concentração."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_auditoria_consolidada_html.parquet", "subpasta": "04_composicao_auditoria", "tema": "auditoria_consolidada", "secao_html": "Auditoria", "tipo_conteudo": "auditoria", "obrigatorio": True, "descricao": "Auditoria consolidada das etapas monitoradas."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_resumo_composicao_auditoria_html.parquet", "subpasta": "05_apoio", "tema": "resumo_composicao_auditoria", "secao_html": "Apoio Técnico", "tipo_conteudo": "resumo", "obrigatorio": True, "descricao": "Resumo da subetapa 15.4."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_parametros_composicao_auditoria_html.parquet", "subpasta": "05_apoio", "tema": "parametros_15_4", "secao_html": "Apoio Técnico", "tipo_conteudo": "parametros", "obrigatorio": True, "descricao": "Parâmetros da subetapa 15.4."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_catalogo_composicao_auditoria_html.parquet", "subpasta": "05_apoio", "tema": "catalogo_composicao_15_4", "secao_html": "Apoio Técnico", "tipo_conteudo": "catalogo", "obrigatorio": True, "descricao": "Catálogo da subetapa 15.4."},
    {"subetapa": "15.4", "arquivo": "15_4_tbl_auditoria_validacao_composicao_auditoria_html.parquet", "subpasta": "05_apoio", "tema": "auditoria_15_4", "secao_html": "Apoio Técnico", "tipo_conteudo": "auditoria", "obrigatorio": True, "descricao": "Auditoria da subetapa 15.4."},
]

df_catalogo_planejado = pd.DataFrame(ARQUIVOS_15_5)
df_catalogo_planejado["caminho_origem"] = df_catalogo_planejado["arquivo"].map(lambda x: str(DIR_ETAPA_15 / x))
df_catalogo_planejado["caminho_destino"] = df_catalogo_planejado.apply(
    lambda row: str(DIR_HTML_DADOS / row["subpasta"] / row["arquivo"]),
    axis=1,
)

print(f"Arquivos planejados para consolidação: {len(df_catalogo_planejado):,}")
print(f"Subetapas cobertas...................: {df_catalogo_planejado['subetapa'].nunique():,}")
print(f"Seções HTML cobertas.................: {df_catalogo_planejado['secao_html'].nunique():,}")
print("OK")

# ============================================================
# 4) Funções auxiliares de manifesto e cópia segura
# ============================================================

print("\n[4/11] Funções auxiliares de manifesto e cópia segura...")

def obter_metadata_parquet(caminho):
    caminho = Path(caminho)
    if not caminho.exists():
        return {
            "linhas": 0,
            "colunas": 0,
            "schema_colunas": "",
            "metadata_ok": False,
            "erro_metadata": "arquivo_inexistente",
        }

    try:
        parquet_file = pq.ParquetFile(caminho)
        schema_colunas = [campo.name for campo in parquet_file.schema_arrow]
        return {
            "linhas": int(parquet_file.metadata.num_rows),
            "colunas": int(len(schema_colunas)),
            "schema_colunas": "|".join(schema_colunas),
            "metadata_ok": True,
            "erro_metadata": "",
        }
    except Exception as erro:
        return {
            "linhas": np.nan,
            "colunas": np.nan,
            "schema_colunas": "",
            "metadata_ok": False,
            "erro_metadata": str(erro),
        }


def calcular_tamanho_mb(caminho):
    caminho = Path(caminho)
    if not caminho.exists():
        return 0.0
    return float(caminho.stat().st_size) / (1024 ** 2)


def caminho_relativo_posix(caminho, base):
    caminho = Path(caminho)
    base = Path(base)
    try:
        return caminho.relative_to(base).as_posix()
    except Exception:
        return caminho.as_posix()


def preparar_para_parquet(df):
    df_saida = df.copy()
    for coluna in df_saida.columns:
        if pd.api.types.is_object_dtype(df_saida[coluna]):
            df_saida[coluna] = df_saida[coluna].astype("string")
    return df_saida


def salvar_parquet_compat(df, caminho, index=False):
    salvar_dataframe(preparar_para_parquet(df), caminho, index=index)


def salvar_json_records(df, caminho):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    registros = json.loads(df.to_json(orient="records", force_ascii=False, date_format="iso"))
    caminho.write_text(json.dumps(registros, ensure_ascii=False, indent=2), encoding="utf-8")


def copiar_arquivo_seguro(origem, destino):
    origem = Path(origem)
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    if not origem.exists():
        return False, "arquivo_origem_inexistente"

    try:
        shutil.copy2(origem, destino)
        return True, ""
    except Exception as erro:
        return False, str(erro)

print("Funções auxiliares declaradas com sucesso.")
print("OK")

# ============================================================
# 5) Cópia física dos arquivos finais para html_final/dados
# ============================================================

print("\n[5/11] Cópia física dos arquivos finais para html_final/dados...")

registros_manifesto = []

for ordem, item in enumerate(ARQUIVOS_15_5, start=1):
    caminho_origem = DIR_ETAPA_15 / item["arquivo"]
    caminho_destino = DIR_HTML_DADOS / item["subpasta"] / item["arquivo"]
    copiado, erro_copia = copiar_arquivo_seguro(caminho_origem, caminho_destino)

    metadata_origem = obter_metadata_parquet(caminho_origem)
    metadata_destino = obter_metadata_parquet(caminho_destino)

    tamanho_origem_mb = calcular_tamanho_mb(caminho_origem)
    tamanho_destino_mb = calcular_tamanho_mb(caminho_destino)

    registros_manifesto.append(
        {
            "ordem_manifesto": ordem,
            "subetapa_origem": item["subetapa"],
            "arquivo_origem": item["arquivo"],
            "arquivo_destino": item["arquivo"],
            "subpasta_html": item["subpasta"],
            "tema": item["tema"],
            "secao_html": item["secao_html"],
            "tipo_conteudo": item["tipo_conteudo"],
            "obrigatorio": bool(item["obrigatorio"]),
            "descricao": item["descricao"],
            "caminho_origem": str(caminho_origem),
            "caminho_destino": str(caminho_destino),
            "caminho_relativo_html_final": caminho_relativo_posix(caminho_destino, DIR_HTML_FINAL),
            "caminho_relativo_dados": caminho_relativo_posix(caminho_destino, DIR_HTML_DADOS),
            "arquivo_origem_existe": bool(caminho_origem.exists()),
            "arquivo_destino_existe": bool(caminho_destino.exists()),
            "copiado_com_sucesso": bool(copiado),
            "erro_copia": erro_copia,
            "linhas_origem": metadata_origem["linhas"],
            "colunas_origem": metadata_origem["colunas"],
            "linhas_destino": metadata_destino["linhas"],
            "colunas_destino": metadata_destino["colunas"],
            "schema_colunas": metadata_destino["schema_colunas"],
            "metadata_destino_ok": bool(metadata_destino["metadata_ok"]),
            "erro_metadata_destino": metadata_destino["erro_metadata"],
            "tamanho_origem_mb": tamanho_origem_mb,
            "tamanho_destino_mb": tamanho_destino_mb,
            "tamanho_diferente": bool(abs(tamanho_origem_mb - tamanho_destino_mb) > 0.000001),
        }
    )

df_manifesto = pd.DataFrame(registros_manifesto)

print(f"Arquivos copiados com sucesso..........: {int(df_manifesto['copiado_com_sucesso'].sum()):,}")
print(f"Arquivos obrigatórios ausentes.........: {int((df_manifesto['obrigatorio'] & ~df_manifesto['arquivo_origem_existe']).sum()):,}")
print(f"Arquivos com erro de cópia.............: {int((~df_manifesto['copiado_com_sucesso']).sum()):,}")
print(f"Tamanho total consolidado em dados MB..: {df_manifesto['tamanho_destino_mb'].sum():,.2f}")
print("OK")

# ============================================================
# 6) Construção do catálogo consolidado por tema, seção e subpasta
# ============================================================

print("\n[6/11] Construção do catálogo consolidado por tema, seção e subpasta...")

df_catalogo_dados = df_manifesto.copy()

df_catalogo_dados["flag_base_grande"] = df_catalogo_dados["linhas_destino"].fillna(0).astype(float) >= 100000
df_catalogo_dados["flag_tabela_navegavel"] = df_catalogo_dados["linhas_destino"].fillna(0).astype(float) <= 100000
df_catalogo_dados["classe_tamanho_html"] = np.select(
    [
        df_catalogo_dados["linhas_destino"].fillna(0).astype(float) <= 5000,
        df_catalogo_dados["linhas_destino"].fillna(0).astype(float) <= 100000,
        df_catalogo_dados["linhas_destino"].fillna(0).astype(float) > 100000,
    ],
    [
        "pequena",
        "media",
        "grande",
    ],
    default="nao_classificada",
)

df_catalogo_dados["recomendacao_consumo_html"] = np.select(
    [
        df_catalogo_dados["tipo_conteudo"].isin(["parametros", "auditoria", "catalogo", "resumo"]),
        df_catalogo_dados["classe_tamanho_html"].eq("pequena"),
        df_catalogo_dados["classe_tamanho_html"].eq("media"),
        df_catalogo_dados["classe_tamanho_html"].eq("grande"),
    ],
    [
        "carregar_como_tabela_tecnica",
        "carregar_diretamente",
        "carregar_com_filtros",
        "carregar_com_filtros_e_amostragem_visual",
    ],
    default="avaliar_no_html",
)

df_catalogo_dados["id_dataset_html"] = (
    df_catalogo_dados["subpasta_html"].astype(str)
    + "__"
    + df_catalogo_dados["tema"].astype(str)
)

print(f"Linhas no catálogo consolidado.........: {len(df_catalogo_dados):,}")
print(f"Datasets grandes para filtros..........: {int(df_catalogo_dados['flag_base_grande'].sum()):,}")
print(f"Seções HTML no catálogo................: {df_catalogo_dados['secao_html'].nunique():,}")
print("OK")

# ============================================================
# 7) Resumo da estrutura consolidada
# ============================================================

print("\n[7/11] Resumo da estrutura consolidada...")

df_resumo_subpasta = (
    df_manifesto
    .groupby("subpasta_html", dropna=False)
    .agg(
        arquivos=("arquivo_destino", "nunique"),
        linhas_total=("linhas_destino", "sum"),
        colunas_total=("colunas_destino", "sum"),
        tamanho_total_mb=("tamanho_destino_mb", "sum"),
        arquivos_obrigatorios=("obrigatorio", "sum"),
        arquivos_copiados=("copiado_com_sucesso", "sum"),
    )
    .reset_index()
)

df_resumo_secao = (
    df_manifesto
    .groupby("secao_html", dropna=False)
    .agg(
        arquivos=("arquivo_destino", "nunique"),
        linhas_total=("linhas_destino", "sum"),
        tamanho_total_mb=("tamanho_destino_mb", "sum"),
        datasets_grandes=("linhas_destino", lambda x: int((pd.to_numeric(x, errors="coerce").fillna(0) >= 100000).sum())),
    )
    .reset_index()
)

df_resumo_estrutura = pd.concat(
    [
        pd.DataFrame(
            [
                {
                    "nivel_resumo": "geral",
                    "grupo": "html_final_dados",
                    "arquivos": int(df_manifesto["arquivo_destino"].nunique()),
                    "linhas_total": float(pd.to_numeric(df_manifesto["linhas_destino"], errors="coerce").fillna(0).sum()),
                    "tamanho_total_mb": float(df_manifesto["tamanho_destino_mb"].sum()),
                    "observacao": "Estrutura consolidada completa para consumo pela seção final do HTML.",
                }
            ]
        ),
        df_resumo_subpasta.rename(columns={"subpasta_html": "grupo"}).assign(
            nivel_resumo="subpasta",
            observacao="Resumo por subpasta de dados do HTML.",
        )[["nivel_resumo", "grupo", "arquivos", "linhas_total", "tamanho_total_mb", "observacao"]],
        df_resumo_secao.rename(columns={"secao_html": "grupo"}).assign(
            nivel_resumo="secao_html",
            observacao="Resumo por seção lógica do HTML.",
        )[["nivel_resumo", "grupo", "arquivos", "linhas_total", "tamanho_total_mb", "observacao"]],
    ],
    ignore_index=True,
)

print(f"Resumo da estrutura....................: {len(df_resumo_estrutura):,} linhas")
print(f"Subpastas consolidadas.................: {df_manifesto['subpasta_html'].nunique():,}")
print(f"Seções lógicas consolidadas............: {df_manifesto['secao_html'].nunique():,}")
print("OK")

# ============================================================
# 8) Parâmetros metodológicos da consolidação final
# ============================================================

print("\n[8/11] Parâmetros metodológicos da consolidação final...")

PARAMETROS_15_5 = {
    "etapa": "15",
    "subetapa": "15.5",
    "nome_subetapa": "Consolidação dos Arquivos Finais do HTML",
    "objetivo": "organizar arquivos finais para consumo pela seção final do HTML sem recálculo pesado",
    "origem_principal": "outputs das subetapas 15.1, 15.2, 15.3 e 15.4",
    "destino_principal": str(DIR_HTML_DADOS),
    "estrutura_resumo": "html_final/dados/01_resumo",
    "estrutura_performance": "html_final/dados/02_performance",
    "estrutura_risco": "html_final/dados/03_risco",
    "estrutura_composicao_auditoria": "html_final/dados/04_composicao_auditoria",
    "estrutura_apoio": "html_final/dados/05_apoio",
    "estrutura_manifestos": "html_final/dados/00_manifestos",
    "formato_principal": "parquet",
    "formato_manifestos_auxiliares": "parquet e json",
    "recaculo_pesado": "nao",
    "politica_bases_grandes": "copiar bases consolidadas já geradas e registrar recomendação de consumo com filtros",
    "observacao_posicoes_historicas": "a base bruta diária não é copiada; são usados snapshots e agregações mensais gerados na 15.4",
    "observacao_memoria": "a subetapa usa cópia física de arquivos e metadata parquet, evitando carregar bases grandes em memória",
    "quantidade_arquivos_planejados": int(len(df_manifesto)),
    "quantidade_arquivos_copiados": int(df_manifesto["copiado_com_sucesso"].sum()),
    "tamanho_total_mb": float(df_manifesto["tamanho_destino_mb"].sum()),
}

df_parametros = pd.DataFrame([{"parametro": chave, "valor": valor} for chave, valor in PARAMETROS_15_5.items()])

print(f"Parâmetros metodológicos...............: {len(df_parametros):,} linhas")
print("OK")

# ============================================================
# 9) Auditoria de validação da estrutura consolidada
# ============================================================

print("\n[9/11] Auditoria de validação da estrutura consolidada...")

arquivos_obrigatorios_ausentes = int((df_manifesto["obrigatorio"] & ~df_manifesto["arquivo_origem_existe"]).sum())
arquivos_nao_copiados = int((~df_manifesto["copiado_com_sucesso"]).sum())
arquivos_destino_ausentes = int((~df_manifesto["arquivo_destino_existe"]).sum())
arquivos_metadata_invalida = int((~df_manifesto["metadata_destino_ok"]).sum())
arquivos_tamanho_diferente = int(df_manifesto["tamanho_diferente"].sum())
arquivos_sem_linhas = int((pd.to_numeric(df_manifesto["linhas_destino"], errors="coerce").fillna(0) <= 0).sum())
duplicatas_destino = int(df_manifesto.duplicated(["subpasta_html", "arquivo_destino"]).sum())

tem_resumo = int((df_manifesto["subpasta_html"] == "01_resumo").any())
tem_performance = int((df_manifesto["subpasta_html"] == "02_performance").any())
tem_risco = int((df_manifesto["subpasta_html"] == "03_risco").any())
tem_composicao = int((df_manifesto["subpasta_html"] == "04_composicao_auditoria").any())
tem_apoio = int((df_manifesto["subpasta_html"] == "05_apoio").any())

tem_posicoes_derivadas = int(
    df_manifesto.loc[
        df_manifesto["tema"].isin(
            [
                "posicoes_historicas",
                "posicoes_setorial_mensal",
                "top_posicoes_historicas",
                "resumo_posicoes_historicas",
            ]
        ),
        "linhas_destino",
    ].fillna(0).astype(float).gt(0).all()
)

linhas_auditoria = [
    {
        "item": "objetos_removidos_limpeza_preventiva",
        "valor": objetos_removidos_15_5,
        "valor_referencia": ">= 0",
        "status": "OK" if objetos_removidos_15_5 >= 0 else "ERRO",
        "observacao": "Quantidade de objetos tabulares/intermediários removidos do kernel antes da subetapa.",
    },
    {
        "item": "memoria_tabular_estimativa_liberada_mb",
        "valor": memoria_liberada_mb_15_5,
        "valor_referencia": ">= 0",
        "status": "OK" if memoria_liberada_mb_15_5 >= 0 else "ERRO",
        "observacao": "Estimativa de memória tabular liberada antes da execução da subetapa.",
    },
    {
        "item": "arquivos_planejados",
        "valor": int(len(df_manifesto)),
        "valor_referencia": "> 0",
        "status": "OK" if len(df_manifesto) > 0 else "ERRO",
        "observacao": "A subetapa deve possuir um manifesto de arquivos finais.",
    },
    {
        "item": "arquivos_obrigatorios_ausentes",
        "valor": arquivos_obrigatorios_ausentes,
        "valor_referencia": "0",
        "status": "OK" if arquivos_obrigatorios_ausentes == 0 else "ERRO",
        "observacao": "Todos os arquivos obrigatórios da Etapa 15 devem existir antes da consolidação.",
    },
    {
        "item": "arquivos_nao_copiados",
        "valor": arquivos_nao_copiados,
        "valor_referencia": "0",
        "status": "OK" if arquivos_nao_copiados == 0 else "ERRO",
        "observacao": "Todos os arquivos planejados devem ser copiados para html_final/dados.",
    },
    {
        "item": "arquivos_destino_ausentes",
        "valor": arquivos_destino_ausentes,
        "valor_referencia": "0",
        "status": "OK" if arquivos_destino_ausentes == 0 else "ERRO",
        "observacao": "Todos os arquivos de destino devem existir após a cópia.",
    },
    {
        "item": "arquivos_metadata_invalida",
        "valor": arquivos_metadata_invalida,
        "valor_referencia": "0",
        "status": "OK" if arquivos_metadata_invalida == 0 else "ERRO",
        "observacao": "Todos os parquets consolidados devem ter metadata legível.",
    },
    {
        "item": "arquivos_tamanho_diferente",
        "valor": arquivos_tamanho_diferente,
        "valor_referencia": "0",
        "status": "OK" if arquivos_tamanho_diferente == 0 else "ERRO",
        "observacao": "O tamanho dos arquivos copiados deve coincidir com o tamanho na origem.",
    },
    {
        "item": "arquivos_sem_linhas",
        "valor": arquivos_sem_linhas,
        "valor_referencia": "0",
        "status": "OK" if arquivos_sem_linhas == 0 else "ERRO",
        "observacao": "Os arquivos finais de dados devem conter ao menos uma linha.",
    },
    {
        "item": "duplicatas_destino",
        "valor": duplicatas_destino,
        "valor_referencia": "0",
        "status": "OK" if duplicatas_destino == 0 else "ERRO",
        "observacao": "Não deve haver duplicidade de arquivo dentro da mesma subpasta final.",
    },
    {
        "item": "subpasta_resumo_presente",
        "valor": tem_resumo,
        "valor_referencia": "1",
        "status": "OK" if tem_resumo == 1 else "ERRO",
        "observacao": "A estrutura final deve conter a subpasta de resumo.",
    },
    {
        "item": "subpasta_performance_presente",
        "valor": tem_performance,
        "valor_referencia": "1",
        "status": "OK" if tem_performance == 1 else "ERRO",
        "observacao": "A estrutura final deve conter a subpasta de performance.",
    },
    {
        "item": "subpasta_risco_presente",
        "valor": tem_risco,
        "valor_referencia": "1",
        "status": "OK" if tem_risco == 1 else "ERRO",
        "observacao": "A estrutura final deve conter a subpasta de risco.",
    },
    {
        "item": "subpasta_composicao_presente",
        "valor": tem_composicao,
        "valor_referencia": "1",
        "status": "OK" if tem_composicao == 1 else "ERRO",
        "observacao": "A estrutura final deve conter a subpasta de composição e auditoria.",
    },
    {
        "item": "subpasta_apoio_presente",
        "valor": tem_apoio,
        "valor_referencia": "1",
        "status": "OK" if tem_apoio == 1 else "ERRO",
        "observacao": "A estrutura final deve conter a subpasta de apoio técnico.",
    },
    {
        "item": "posicoes_historicas_derivadas_presentes",
        "valor": tem_posicoes_derivadas,
        "valor_referencia": "1",
        "status": "OK" if tem_posicoes_derivadas == 1 else "ERRO",
        "observacao": "As bases derivadas da tabela histórica de posições devem estar presentes no HTML final.",
    },
]

df_auditoria = pd.DataFrame(linhas_auditoria)
erros_bloqueantes = int((df_auditoria["status"] == "ERRO").sum())

print(f"Itens de auditoria.....................: {len(df_auditoria):,}")
print(f"Erros bloqueantes......................: {erros_bloqueantes:,}")
print("OK")

# ============================================================
# 10) Salvamento dos manifestos e arquivos de controle
# ============================================================

print("\n[10/11] Salvamento dos manifestos e arquivos de controle...")

# Primeiro salvamento na Etapa 15.
salvar_parquet_compat(df_manifesto, CAMINHO_MANIFESTO_ETAPA, index=False)
salvar_parquet_compat(df_catalogo_dados, CAMINHO_CATALOGO_ETAPA, index=False)
salvar_parquet_compat(df_resumo_estrutura, CAMINHO_RESUMO_ETAPA, index=False)
salvar_parquet_compat(df_parametros, CAMINHO_PARAMETROS_ETAPA, index=False)
salvar_parquet_compat(df_auditoria, CAMINHO_AUDITORIA_ETAPA, index=False)

# Salvamento espelhado em html_final/dados/00_manifestos.
salvar_parquet_compat(df_manifesto, CAMINHO_MANIFESTO_HTML_PARQUET, index=False)
salvar_parquet_compat(df_catalogo_dados, CAMINHO_CATALOGO_HTML_PARQUET, index=False)
salvar_parquet_compat(df_resumo_estrutura, CAMINHO_RESUMO_HTML_PARQUET, index=False)
salvar_parquet_compat(df_parametros, CAMINHO_PARAMETROS_HTML_PARQUET, index=False)
salvar_parquet_compat(df_auditoria, CAMINHO_AUDITORIA_HTML_PARQUET, index=False)

# JSONs leves para a etapa final do HTML localizar rapidamente os datasets.
df_manifesto_json = df_manifesto[
    [
        "ordem_manifesto",
        "id_dataset_html" if "id_dataset_html" in df_manifesto.columns else "tema",
        "tema",
        "secao_html",
        "tipo_conteudo",
        "caminho_relativo_html_final",
        "caminho_relativo_dados",
        "linhas_destino",
        "colunas_destino",
        "tamanho_destino_mb",
    ]
].copy() if "id_dataset_html" in df_manifesto.columns else df_manifesto[
    [
        "ordem_manifesto",
        "tema",
        "secao_html",
        "tipo_conteudo",
        "caminho_relativo_html_final",
        "caminho_relativo_dados",
        "linhas_destino",
        "colunas_destino",
        "tamanho_destino_mb",
    ]
].copy()

salvar_json_records(df_manifesto_json, CAMINHO_MANIFESTO_HTML_JSON)
salvar_json_records(
    df_catalogo_dados[
        [
            "id_dataset_html",
            "tema",
            "secao_html",
            "tipo_conteudo",
            "classe_tamanho_html",
            "recomendacao_consumo_html",
            "caminho_relativo_html_final",
            "linhas_destino",
            "colunas_destino",
            "tamanho_destino_mb",
        ]
    ],
    CAMINHO_CATALOGO_HTML_JSON,
)

print("Manifestos e arquivos de controle salvos com sucesso.")
print(f"Manifesto Etapa 15.....................: {CAMINHO_MANIFESTO_ETAPA}")
print(f"Manifesto HTML parquet.................: {CAMINHO_MANIFESTO_HTML_PARQUET}")
print(f"Manifesto HTML json....................: {CAMINHO_MANIFESTO_HTML_JSON}")
print(f"Catálogo HTML parquet..................: {CAMINHO_CATALOGO_HTML_PARQUET}")
print(f"Catálogo HTML json.....................: {CAMINHO_CATALOGO_HTML_JSON}")
print("OK")

# ============================================================
# 11) Validação final e prints de controle
# ============================================================

print("\n[11/11] Validação final e prints de controle...")

print("\nAuditoria de validação da consolidação dos arquivos finais do HTML:")
print(df_auditoria.to_string(index=False))

print("\nResumo da estrutura consolidada:")
print(df_resumo_estrutura.to_string(index=False))

print("\nArquivos por subpasta:")
print(
    df_resumo_subpasta
    .sort_values("subpasta_html")
    .to_string(index=False)
)

print("\nMaiores bases consolidadas para o HTML:")
colunas_maiores = [
    "subpasta_html",
    "tema",
    "arquivo_destino",
    "linhas_destino",
    "colunas_destino",
    "tamanho_destino_mb",
    "recomendacao_consumo_html",
]
print(
    df_catalogo_dados[colunas_maiores]
    .sort_values(["linhas_destino", "tamanho_destino_mb"], ascending=[False, False])
    .head(20)
    .to_string(index=False)
)

print("\nArquivos salvos na Etapa 15.5:")
print(f"- {CAMINHO_MANIFESTO_ETAPA}")
print(f"- {CAMINHO_CATALOGO_ETAPA}")
print(f"- {CAMINHO_RESUMO_ETAPA}")
print(f"- {CAMINHO_PARAMETROS_ETAPA}")
print(f"- {CAMINHO_AUDITORIA_ETAPA}")
print(f"- {CAMINHO_MANIFESTO_HTML_PARQUET}")
print(f"- {CAMINHO_MANIFESTO_HTML_JSON}")
print(f"- {CAMINHO_CATALOGO_HTML_PARQUET}")
print(f"- {CAMINHO_CATALOGO_HTML_JSON}")
print(f"- {CAMINHO_RESUMO_HTML_PARQUET}")
print(f"- {CAMINHO_AUDITORIA_HTML_PARQUET}")
print(f"- {CAMINHO_PARAMETROS_HTML_PARQUET}")

if erros_bloqueantes > 0:
    raise ValueError("A auditoria da Etapa 15.5 identificou erros bloqueantes. Verifique a tabela de auditoria salva.")

print("\nETAPA 15.5 FINALIZADA COM SUCESSO.")
print("=" * 100)

INICIANDO ETAPA 15.5 - CONSOLIDAÇÃO DOS ARQUIVOS FINAIS DO HTML

[1/11] Validação inicial do ambiente e limpeza preventiva leve...
Objetos tabulares/intermediários removidos do kernel: 98
Memória tabular estimada liberada.................: 0.01 MB
OK

[2/11] Definição determinística dos diretórios finais...
Diretório da Etapa 15              : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\resultados\etapa_15
Diretório HTML final               : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\html_final
Diretório dados HTML               : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\html_final\dados
Diretório manifestos HTML          : C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro\html_final\dados\00_manifestos
Diretório resumo HTM

# HTML Final

In [84]:
%%time
# ==================================================================================================
# HTML FINAL - CAPITULAÇÃO E EUFORIA NO MERCADO BRASILEIRO
# ==================================================================================================

print("=" * 110)
print("INICIANDO SEÇÃO HTML FINAL")
print("=" * 110)

print("\n[1/9] Resolvendo diretórios e validando estrutura consolidada do projeto...")

DIR_RAIZ = Path(".")
DIR_RESULTADOS = DIR_RAIZ / "resultados"
DIR_HTML_FINAL = DIR_RAIZ / "html_final"

DIR_ASSETS = DIR_HTML_FINAL / "assets"
DIR_ASSETS_CSS = DIR_ASSETS / "css"
DIR_ASSETS_JS = DIR_ASSETS / "js"

DIR_DADOS = DIR_HTML_FINAL / "dados"
DIR_DADOS_MANIFESTOS = DIR_DADOS / "00_manifestos"
DIR_DADOS_RESUMO = DIR_DADOS / "01_resumo"
DIR_DADOS_PERFORMANCE = DIR_DADOS / "02_performance"
DIR_DADOS_RISCO = DIR_DADOS / "03_risco"
DIR_DADOS_COMPOSICAO = DIR_DADOS / "04_composicao_auditoria"
DIR_DADOS_APOIO = DIR_DADOS / "05_apoio"

DIR_TABELAS = DIR_HTML_FINAL / "tabelas"
DIR_GRAFICOS = DIR_HTML_FINAL / "graficos"
DIR_RELATORIOS = DIR_HTML_FINAL / "relatorios"

ARQUIVO_INDEX = DIR_RAIZ / "index.html"
ARQUIVO_CSS = DIR_ASSETS_CSS / "capitulacao_euforia_final.css"
ARQUIVO_JS = DIR_ASSETS_JS / "capitulacao_euforia_final.js"
ARQUIVO_DATA_JS = DIR_ASSETS_JS / "capitulacao_euforia_data.js"
ARQUIVO_RESUMO_JSON = DIR_RELATORIOS / "resumo_execucao_html_final.json"

MAX_BYTES_GITHUB = 100 * 1024 * 1024
MAX_BYTES_ASSET_SEGURO = 95 * 1024 * 1024
MAX_LINHAS_TABELA_HTML = 12000
MAX_PONTOS_GRAFICO = 35000

for diretorio in [
    DIR_HTML_FINAL,
    DIR_ASSETS,
    DIR_ASSETS_CSS,
    DIR_ASSETS_JS,
    DIR_DADOS,
    DIR_DADOS_MANIFESTOS,
    DIR_DADOS_RESUMO,
    DIR_DADOS_PERFORMANCE,
    DIR_DADOS_RISCO,
    DIR_DADOS_COMPOSICAO,
    DIR_DADOS_APOIO,
    DIR_TABELAS,
    DIR_GRAFICOS,
    DIR_RELATORIOS,
]:
    diretorio.mkdir(parents=True, exist_ok=True)

if not DIR_RESULTADOS.exists():
    raise FileNotFoundError("A pasta 'resultados' não foi encontrada na raiz do projeto.")

if not DIR_DADOS.exists():
    raise FileNotFoundError("A pasta de dados finais do dashboard não foi encontrada.")

pastas_dados_obrigatorias = [
    DIR_DADOS_MANIFESTOS,
    DIR_DADOS_RESUMO,
    DIR_DADOS_PERFORMANCE,
    DIR_DADOS_RISCO,
    DIR_DADOS_COMPOSICAO,
    DIR_DADOS_APOIO,
]

pastas_faltantes = [p.as_posix() for p in pastas_dados_obrigatorias if not p.exists()]
if pastas_faltantes:
    raise FileNotFoundError("Pastas obrigatórias ausentes nos dados finais: " + ", ".join(pastas_faltantes))

print("OK")
print(f" - Raiz do projeto....................: {DIR_RAIZ.resolve()}")
print(f" - Pasta resultados...................: {DIR_RESULTADOS}")
print(f" - Pasta html_final...................: {DIR_HTML_FINAL}")
print(f" - Index final........................: {ARQUIVO_INDEX}")

print("\n[2/9] Definindo funções auxiliares de leitura, formatação, gráficos e tabelas...")

def esc(texto):
    if texto is None:
        return ""
    return html.escape(str(texto))

def slugify(texto):
    texto = str(texto).strip().lower()
    texto = texto.replace("ç", "c")
    texto = texto.replace("ã", "a").replace("á", "a").replace("à", "a").replace("â", "a")
    texto = texto.replace("é", "e").replace("ê", "e")
    texto = texto.replace("í", "i")
    texto = texto.replace("õ", "o").replace("ó", "o").replace("ô", "o")
    texto = texto.replace("ú", "u")
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto or "item"

def caminho_relativo(destino, origem_index=ARQUIVO_INDEX):
    return os.path.relpath(Path(destino), start=Path(origem_index).parent).replace("\\", "/")

def format_int(x):
    if x is None or pd.isna(x):
        return "N/A"
    try:
        return f"{int(round(float(x))):,}".replace(",", ".")
    except Exception:
        return esc(x)

def format_ano(x):
    if x is None or pd.isna(x):
        return ""
    try:
        return str(int(round(float(x))))
    except Exception:
        return esc(x)

def format_num(x, casas=2):
    if x is None or pd.isna(x):
        return "N/A"
    try:
        return f"{float(x):,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return esc(x)

def format_pct_prop(x, casas=2):
    if x is None or pd.isna(x):
        return "N/A"
    try:
        return f"{float(x):.{casas}%}".replace(".", ",")
    except Exception:
        return esc(x)

def format_pct_val(x, casas=2):
    if x is None or pd.isna(x):
        return "N/A"
    try:
        return f"{float(x):,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return esc(x)

def format_money(x, casas=2):
    if x is None or pd.isna(x):
        return "N/A"
    try:
        return "R$ " + f"{float(x):,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return esc(x)

def primeira_linha(df):
    if df is None or len(df) == 0:
        return {}
    return df.iloc[0].to_dict()

def serie_para_texto_sem_na(serie):
    return serie.astype("object").where(pd.notna(serie), "").astype(str)

def dataframe_para_objeto_sem_na(df):
    return df.astype("object").where(pd.notna(df), "")

def normalizar_nome_coluna(coluna):
    texto = str(coluna).strip().lower()
    texto = texto.replace("ç", "c")
    texto = texto.replace("ã", "a").replace("á", "a").replace("à", "a").replace("â", "a")
    texto = texto.replace("é", "e").replace("ê", "e")
    texto = texto.replace("í", "i")
    texto = texto.replace("õ", "o").replace("ó", "o").replace("ô", "o")
    texto = texto.replace("ú", "u")
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto

def encontrar_coluna(df, candidatos=None, contem_todos=None, contem_algum=None, excluir=None, somente_numerica=False, somente_texto=False):
    if df is None or len(df.columns) == 0:
        return None

    candidatos = candidatos or []
    contem_todos = contem_todos or []
    contem_algum = contem_algum or []
    excluir = excluir or []

    mapa = {normalizar_nome_coluna(c): c for c in df.columns}

    for candidato in candidatos:
        chave = normalizar_nome_coluna(candidato)
        if chave in mapa:
            col = mapa[chave]
            if somente_numerica and not pd.api.types.is_numeric_dtype(df[col]):
                continue
            if somente_texto and pd.api.types.is_numeric_dtype(df[col]):
                continue
            return col

    for col in df.columns:
        nome = normalizar_nome_coluna(col)

        if excluir and any(normalizar_nome_coluna(x) in nome for x in excluir):
            continue

        if contem_todos and not all(normalizar_nome_coluna(x) in nome for x in contem_todos):
            continue

        if contem_algum and not any(normalizar_nome_coluna(x) in nome for x in contem_algum):
            continue

        if somente_numerica and not pd.api.types.is_numeric_dtype(df[col]):
            continue

        if somente_texto and pd.api.types.is_numeric_dtype(df[col]):
            continue

        return col

    return None

def colunas_numericas(df):
    if df is None:
        return []
    return [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

def carregar_parquet(caminho, obrigatorio=True):
    caminho = Path(caminho)
    if not caminho.exists():
        if obrigatorio:
            raise FileNotFoundError(f"Arquivo parquet obrigatório não encontrado: {caminho}")
        return None
    return pd.read_parquet(caminho)

def carregar_json(caminho, obrigatorio=False):
    caminho = Path(caminho)
    if not caminho.exists():
        if obrigatorio:
            raise FileNotFoundError(f"Arquivo JSON obrigatório não encontrado: {caminho}")
        return {}
    return json.loads(caminho.read_text(encoding="utf-8"))

def reduzir_dataframe_para_exibicao(df, max_linhas=MAX_LINHAS_TABELA_HTML):
    if df is None:
        return pd.DataFrame()

    base = df.copy()

    if len(base) <= max_linhas:
        return base

    colunas_prioridade = [
        "ranking",
        "rank",
        "ordem",
        "data",
        "ano",
        "mes",
        "estrategia",
        "estrategia_nome",
        "grupo",
        "familia",
        "ticker",
        "empresa",
        "setor",
        "subsetor",
        "segmento",
        "retorno",
        "retorno_acumulado",
        "valor_final",
        "sharpe",
        "sortino",
        "calmar",
        "drawdown",
        "volatilidade",
        "peso",
        "participacao",
        "aporte",
    ]

    col_sort = None
    for chave in colunas_prioridade:
        col = encontrar_coluna(base, contem_algum=[chave])
        if col is not None:
            col_sort = col
            break

    if col_sort is not None:
        try:
            base = base.sort_values(col_sort, ascending=True)
        except Exception:
            pass

    return base.head(max_linhas).copy()

def reduzir_dataframe_para_grafico(df, max_linhas=MAX_PONTOS_GRAFICO):
    if df is None or len(df) <= max_linhas:
        return df

    base = df.copy()

    col_data = encontrar_coluna(base, candidatos=["data", "date", "dt", "mes", "periodo", "ano"])
    col_grupo = encontrar_coluna(
        base,
        candidatos=["estrategia", "estrategia_nome", "estrategia_referencia", "grupo", "carteira", "portfolio", "ticker", "setor", "subsetor", "segmento"],
        somente_texto=True,
    )

    if col_data is not None and col_grupo is not None:
        try:
            base = base.sort_values([col_grupo, col_data]).copy()
            frac = max(1, int(np.ceil(len(base) / max_linhas)))
            base = base.groupby(col_grupo, group_keys=False).apply(lambda x: x.iloc[::frac]).reset_index(drop=True)
            return base.head(max_linhas)
        except Exception:
            return base.head(max_linhas)

    return base.head(max_linhas)

def col_percentual_valor(nome_coluna):
    nome = normalizar_nome_coluna(nome_coluna)
    return (
        nome.startswith("pct_")
        or nome.startswith("perc_")
        or nome.endswith("_pct")
        or nome.endswith("_perc")
        or "percentual" in nome
        or "participacao" in nome
        or "share" in nome
    )

def col_percentual_prop(nome_coluna):
    nome = normalizar_nome_coluna(nome_coluna)
    chaves = [
        "retorno",
        "drawdown",
        "volatilidade",
        "downside",
        "cagr",
        "taxa",
        "excesso",
        "alpha",
        "rentabilidade",
        "performance",
    ]
    excluir = ["sharpe", "sortino", "calmar", "rank", "ranking", "ordem", "score", "valor_final", "patrimonio"]
    return any(chave in nome for chave in chaves) and not any(chave in nome for chave in excluir) and not col_percentual_valor(nome_coluna)

def col_monetaria(nome_coluna):
    nome = normalizar_nome_coluna(nome_coluna)
    if any(chave in nome for chave in ["p_valor", "pvalue", "p_value", "valor_p", "estatistica", "valor_metrica", "valor_ranking", "valor_referencia"]):
        return False
    return any(chave in nome for chave in ["valor_investido", "patrimonio", "aporte", "preco", "financeiro", "volume_fin", "valor_posicao", "valor_total"])


def col_inteira(nome_coluna):
    nome = normalizar_nome_coluna(nome_coluna)

    if any(chave in nome for chave in [
        "retorno", "volatilidade", "drawdown", "sharpe", "sortino", "calmar",
        "cagr", "taxa", "peso", "participacao", "share", "percentual", "pct",
        "score", "valor", "patrimonio", "aporte", "preco", "volume", "financeiro",
        "media", "mediana", "std", "desvio", "p_value", "pvalor", "ic_inf", "ic_sup",
    ]):
        return False

    if nome == "ano" or nome.startswith("ano_") or nome.endswith("_ano"):
        return True

    if nome.startswith("flag_") or nome.startswith("n_"):
        return True

    if any(chave in nome for chave in [
        "qtd", "quantidade", "count", "ordem", "ranking", "rank",
        "seed", "replica", "cenario", "id", "periodos", "linhas", "aportes", "compras",
    ]):
        return True

    return False

def formatar_dataframe_exibicao(df):
    base = df.copy()

    for col in base.columns:
        if pd.api.types.is_datetime64_any_dtype(base[col]):
            base[col] = base[col].dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_numeric_dtype(base[col]):
            nome_col = normalizar_nome_coluna(col)

            if col_percentual_valor(col):
                base[col] = base[col].apply(lambda x: format_pct_val(x, casas=2) if pd.notna(x) else "")
            elif col_percentual_prop(col):
                base[col] = base[col].apply(lambda x: format_pct_prop(x, casas=2) if pd.notna(x) else "")
            elif nome_col == "ano" or nome_col.startswith("ano_") or nome_col.endswith("_ano"):
                base[col] = base[col].apply(lambda x: format_ano(x) if pd.notna(x) else "")
            elif col_inteira(col):
                base[col] = base[col].apply(lambda x: format_int(x) if pd.notna(x) else "")
            elif col_monetaria(col):
                base[col] = base[col].apply(lambda x: format_money(x, casas=2) if pd.notna(x) else "")
            else:
                base[col] = base[col].apply(lambda x: format_num(x, casas=4) if pd.notna(x) else "")
        else:
            base[col] = serie_para_texto_sem_na(base[col])

    return base


def limpar_rotulo_estrategia(valor):
    texto = "" if valor is None else str(valor)
    texto = texto.strip()

    if texto == "" or texto.lower() in ["nan", "none", "nat", "na", "<na>", "não informado", "nao informado"]:
        return ""

    texto = re.sub(r"\bcarteira_na\b", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\breplica_na\b", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\bcarteira nan\b", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\breplica nan\b", "", texto, flags=re.IGNORECASE)
    texto = texto.replace("_", " ")
    texto = re.sub(r"\s+", " ", texto).strip()

    if texto == "" or texto.lower() in ["nan", "none", "nat", "na", "<na>"]:
        return ""

    substituicoes = [
        ("capitulacao", "Capitulação"),
        ("capitulação", "Capitulação"),
        ("euforia", "Euforia"),
        ("aleatorias", "Aleatórias"),
        ("aleatórias", "Aleatórias"),
        ("aleatoria", "Aleatória"),
        ("aleatória", "Aleatória"),
        ("ibovespa", "Ibovespa"),
        ("ibov", "Ibovespa"),
        ("cdi only", "CDI-Only"),
        ("cdi-only", "CDI-Only"),
        ("cdi", "CDI"),
        ("buy and hold", "Buy and Hold"),
        ("aportes mensais", "Aportes Mensais"),
        ("mensal", "Mensal"),
    ]

    texto_final = texto
    for origem, destino in substituicoes:
        texto_final = re.sub(origem, destino, texto_final, flags=re.IGNORECASE)

    texto_final = re.sub(r"\s+", " ", texto_final).strip()
    texto_final = texto_final.replace(" - ", " — ")

    return texto_final


def valor_textual_valido(valor):
    texto = "" if valor is None else str(valor).strip()
    texto_norm = normalizar_nome_coluna(texto)
    return texto != "" and texto_norm not in ["nan", "none", "nat", "na", "nao_informado", "nao_informada"]

def limpar_nome_comparador(valor):
    texto = limpar_rotulo_estrategia(valor)
    texto_norm = normalizar_nome_coluna(texto)

    if not valor_textual_valido(texto):
        return ""
    if "ibov" in texto_norm or "ibovespa" in texto_norm:
        return "Ibovespa"
    if texto_norm == "cdi" or "cdi" in texto_norm:
        return "CDI"
    if "aleator" in texto_norm and "capitulacao" in texto_norm:
        return "Aleatórias — Capitulação"
    if "aleator" in texto_norm and "euforia" in texto_norm:
        return "Aleatórias — Euforia"
    if "mensal" in texto_norm and "capitulacao" in texto_norm:
        return "Mensais — Capitulação"
    if "mensal" in texto_norm and "euforia" in texto_norm:
        return "Mensais — Euforia"
    if "aleator" in texto_norm:
        return "Controles Aleatórios"
    if "mensal" in texto_norm:
        return "Controles Mensais"

    return texto


def melhor_rotulo_estrategia_row(row):
    candidatos = [
        "nome_exibicao_estrategia",
        "nome_estrategia_html",
        "nome_exibicao_estrategia_real",
        "estrategia_referencia",
        "chave_estrategia",
        "estrategia_id",
    ]

    for col in candidatos:
        if col in row.index and valor_textual_valido(row.get(col)):
            rotulo = limpar_rotulo_estrategia(row.get(col))
            if valor_textual_valido(rotulo):
                return rotulo

    textos_contexto = " ".join([
        str(row.get("familia_estrategia", "")),
        str(row.get("categoria_estrategia", "")),
        str(row.get("grupo_html", "")),
        str(row.get("label_grupo_html", "")),
        str(row.get("tipo_estrategia", "")),
        str(row.get("grupo_comparacao", "")),
        str(row.get("tipo_comparacao", "")),
    ])
    contexto = normalizar_nome_coluna(textos_contexto)

    if "aleator" in contexto and "capitulacao" in contexto:
        return "Aleatórias — Capitulação"
    if "aleator" in contexto and "euforia" in contexto:
        return "Aleatórias — Euforia"
    if "mensal" in contexto and "capitulacao" in contexto:
        return "Aportes Mensais — Capitulação"
    if "mensal" in contexto and "euforia" in contexto:
        return "Aportes Mensais — Euforia"
    if "capitulacao" in contexto:
        return "Capitulação"
    if "euforia" in contexto:
        return "Euforia"
    if "ibov" in contexto:
        return "Ibovespa Buy and Hold"
    if "cdi" in contexto:
        return "CDI-Only"

    return ""


def renomear_coluna_exibicao(coluna):
    nome_original = str(coluna)
    nome = normalizar_nome_coluna(nome_original)

    mapa = {
        "nome_exibicao_estrategia": "Estratégia",
        "nome_estrategia_html": "Estratégia",
        "estrategia_referencia": "Estratégia de referência",
        "familia_estrategia": "Família",
        "categoria_estrategia": "Categoria",
        "grupo_html": "Grupo",
        "label_grupo_html": "Grupo",
        "frequencia": "Frequência",
        "label_frequencia": "Frequência",
        "retorno_acumulado_total": "Retorno acumulado",
        "retorno_anualizado": "Retorno equivalente",
        "sharpe_anualizado": "Sharpe",
        "sortino_anualizado": "Sortino",
        "calmar": "Calmar",
        "volatilidade_anualizada": "Volatilidade",
        "downside_vol_anualizada": "Downside volatility",
        "max_drawdown": "Max drawdown",
        "drawdown_abs_maximo": "Max drawdown",
        "drawdown_abs_maximo_percentual": "Max drawdown",
        "retorno_periodo": "Retorno do período",
        "retorno_percentual": "Retorno",
        "retorno_acumulado_ate_periodo": "Retorno acumulado",
        "patrimonio_total": "Patrimônio",
        "patrimonio_base_100": "Patrimônio base 100",
        "valor_grafico": "Valor",
        "valor_metrica_risco": "Valor",
        "metrica_risco": "Métrica",
        "nome_metrica_risco": "Métrica",
        "p_valor_teste_sinal": "p-valor teste de sinal",
        "p_valor_wilcoxon": "p-valor Wilcoxon",
        "p_valor_permutacao": "p-valor permutação",
        "p_valor_minimo_ajustado": "p-valor mínimo ajustado",
        "score_evidencia_medio": "Score médio de evidência",
        "score_evidencia_total": "Score total de evidência",
        "classificacao_evidencia_final": "Classificação da evidência",
        "leitura_final": "Leitura final",
        "leitura_final_estatistica": "Leitura estatística",
        "ticker": "Ticker",
        "issuer_code": "Issuer",
        "nome": "Empresa",
        "nome_empresa": "Empresa",
        "setor": "Setor",
        "subsetor": "Subsetor",
        "segmento": "Segmento",
        "valor_investido_total": "Valor investido",
        "valor_investido_ticker": "Valor investido",
        "valor_investido_empresa": "Valor investido",
        "peso_efetivo_ticker": "Peso do ticker",
        "peso_empresa_aporte": "Peso da empresa",
        "peso_posicao": "Peso da posição",
        "valor_posicao": "Valor da posição",
        "data_sinal": "Data do sinal",
        "data_aporte": "Data do aporte",
        "data_periodo": "Data",
        "data_posicao": "Data da posição",
        "ano_mes": "Ano-mês",
    }

    if nome in mapa:
        return mapa[nome]

    texto = nome_original.replace("_", " ").strip()
    texto = re.sub(r"\banualizado\b", "equivalente", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\bsimplificado\b", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto[:1].upper() + texto[1:]

def tornar_colunas_unicas(colunas):
    vistos = {}
    saida = []

    for col in colunas:
        base = str(col)
        if base not in vistos:
            vistos[base] = 1
            saida.append(base)
        else:
            vistos[base] += 1
            saida.append(f"{base} ({vistos[base]})")

    return saida

def coluna_tecnica_para_ocultar(coluna):
    nome = normalizar_nome_coluna(coluna)

    manter = [
        "flag_estrategia_real",
        "flag_sinal_intenso",
        "flag_sinal_muito_intenso",
        "flag_capitulacao_superou_euforia",
        "flag_euforia_superou_capitulacao",
    ]
    if nome in manter:
        return False

    nomes_exatos_ocultar = {
        "nome_tabela_origem",
        "nome_tabela_html",
        "tema_tabela_html",
        "fonte_analitica",
        "visao_comparacao",
        "visao_inferencia",
        "descricao_visao_inferencia",
        "aplicacao_metodologica",
        "comparacao_id",
        "comparacao_id_original",
        "subtipo_comparacao",
        "chave_comparador",
        "chave_estrategia",
        "chave_aporte",
        "chave_classificacao",
        "ordem_frequencia",
        "ordem_frequencia_html",
        "ordem_frequencia_inferencia",
        "ordem_grupo_comparacao",
        "ordem_estrategia_html",
        "ordem_cenario",
        "ordem_sinal",
        "ordem_aporte",
        "id_dataset_html",
        "schema_colunas",
        "metadata_destino_ok",
        "erro_metadata_destino",
        "erro_copia",
        "request_id",
        "replica_id",
        "carteira_id",
    }
    if nome in nomes_exatos_ocultar:
        return True

    padroes_ocultar = [
        "arquivo_origem",
        "arquivo_destino",
        "caminho",
        "schema_colunas",
        "metadata",
        "erro_",
        "origem_base_html",
        "data_referencia_html",
        "modo_processamento_",
        "linhas_origem",
        "linhas_destino",
        "linhas_lidas",
        "linhas_snapshot",
        "colunas_origem",
        "colunas_destino",
        "tamanho_origem",
        "tamanho_destino",
        "tamanho_diferente",
        "classe_tamanho_html",
        "recomendacao_consumo_html",
        "tipo_grafico_sugerido",
        "label_valor_grafico",
        "ordem_global",
        "ordem_exibicao",
        "arquivo_origem_auditoria",
        "diretorio_origem_auditoria",
        "nome_exibicao_comparador_vazio",
        "coluna_auxiliar",
        "merge_",
        "_merge",
        "score_vs_outra_estrategia",
        "score_contra_outra_estrategia",
    ]

    if any(p in nome for p in padroes_ocultar):
        return True

    # Flags auxiliares normalmente são colunas técnicas. As poucas flags analíticas relevantes
    # foram preservadas na lista 'manter'.
    if nome.startswith("flag_") and nome not in manter:
        return True

    if nome.startswith("n_linhas_") or nome.startswith("desvio_") or nome.startswith("status_arquivo"):
        return True

    return False


    manter = [
        "flag_estrategia_real",
        "flag_sinal_intenso",
        "flag_sinal_muito_intenso",
        "flag_capitulacao_superou_euforia",
        "flag_euforia_superou_capitulacao",
    ]
    if nome in manter:
        return False

    padroes_ocultar = [
        "arquivo_origem",
        "arquivo_destino",
        "caminho",
        "schema_colunas",
        "metadata",
        "erro_",
        "origem_base_html",
        "tema_html",
        "tipo_base_composicao",
        "data_referencia_html",
        "id_dataset_html",
        "tipo_grafico_sugerido",
        "label_valor_grafico",
        "ordem_global",
        "ordem_exibicao",
        "ordem_grupo",
        "ordem_frequencia",
        "ordem_sinal",
        "ordem_aporte",
        "request_id",
        "replica_id",
        "carteira_id",
        "chave_aporte",
        "chave_estrategia",
        "chave_comparador",
        "chave_classificacao",
        "arquivo_origem_auditoria",
        "diretorio_origem_auditoria",
    ]

    if any(p in nome for p in padroes_ocultar):
        return True

    if nome.startswith("flag_") and nome not in manter:
        return True

    if nome.startswith("n_linhas_") or nome.startswith("desvio_") or nome.startswith("status_arquivo"):
        return True

    return False

def ordenar_colunas_exibicao(df):
    if df is None or len(df.columns) == 0:
        return df

    prioridade_tokens = [
        "frequencia",
        "label_frequencia",
        "nome_exibicao_estrategia",
        "nome_estrategia_html",
        "estrategia_referencia",
        "familia_estrategia",
        "categoria_estrategia",
        "grupo_html",
        "label_grupo_html",
        "tipo_comparacao",
        "grupo_comparacao",
        "nome_exibicao_comparador",
        "benchmark",
        "ticker",
        "issuer_code",
        "nome",
        "nome_empresa",
        "setor",
        "subsetor",
        "segmento",
        "data_sinal",
        "data_aporte",
        "data_periodo",
        "data_posicao",
        "ano",
        "ano_mes",
        "n_periodos",
        "n_periodos_validos_excesso",
        "n_vitorias_real",
        "n_vitorias",
        "n_empates_real",
        "n_empates",
        "n_derrotas_real",
        "n_derrotas",
        "retorno_acumulado_total",
        "retorno_anualizado",
        "retorno_periodo",
        "retorno_percentual",
        "media_retorno_real_periodico",
        "media_retorno_comparador_periodico",
        "media_retorno_real",
        "media_retorno_comparador",
        "media_excesso_periodico",
        "mediana_excesso_periodico",
        "pct_periodos_real_superou_comparador",
        "pct_periodos_real_superou_sem_empates",
        "sharpe_anualizado",
        "sortino_anualizado",
        "calmar",
        "volatilidade_anualizada",
        "downside_vol_anualizada",
        "max_drawdown",
        "drawdown_abs_maximo",
        "p_valor_teste_sinal",
        "p_valor_wilcoxon",
        "p_valor_permutacao",
        "p_valor_minimo_ajustado",
        "estatistica_wilcoxon",
        "estatistica_observada_permutacao",
        "score_evidencia_medio",
        "score_evidencia_total",
        "classificacao_evidencia_final",
        "classificacao_evidencia_benchmark",
        "classificacao_evidencia_aleatorio",
        "classificacao_evidencia_mensal",
        "classificacao_empirica_vs_aleatorias",
        "classificacao_empirica_vs_mensais",
        "classificacao_robustez_retorno_acumulado",
        "classificacao_robustez_periodos",
        "leitura_final",
        "leitura_final_estatistica",
        "valor_investido_total",
        "valor_investido_ticker",
        "valor_investido_empresa",
        "valor_posicao",
        "peso_posicao",
        "peso_empresa_aporte",
        "peso_efetivo_ticker",
        "n_compras",
        "n_aportes",
        "quantidade_total",
    ]

    cols = list(df.columns)
    prioridade = []

    for token in prioridade_tokens:
        for col in cols:
            if col not in prioridade and normalizar_nome_coluna(col) == normalizar_nome_coluna(token):
                prioridade.append(col)

    restantes = [c for c in cols if c not in prioridade and not coluna_tecnica_para_ocultar(c)]

    limite_colunas = 48
    selecionadas = (prioridade + restantes)[:limite_colunas]

    if len(selecionadas) == 0:
        selecionadas = [c for c in cols if not coluna_tecnica_para_ocultar(c)][:limite_colunas]

    if len(selecionadas) == 0:
        selecionadas = cols[:limite_colunas]

    return df[selecionadas].copy()


    prioridade_tokens = [
        "frequencia",
        "label_frequencia",
        "nome_exibicao_estrategia",
        "nome_estrategia_html",
        "estrategia_referencia",
        "familia_estrategia",
        "categoria_estrategia",
        "grupo_html",
        "label_grupo_html",
        "tipo_comparacao",
        "grupo_comparacao",
        "nome_exibicao_comparador",
        "benchmark",
        "ticker",
        "issuer_code",
        "nome",
        "nome_empresa",
        "setor",
        "subsetor",
        "segmento",
        "data_sinal",
        "data_aporte",
        "data_periodo",
        "data_posicao",
        "ano",
        "ano_mes",
        "retorno_acumulado_total",
        "retorno_anualizado",
        "retorno_periodo",
        "retorno_percentual",
        "media_retorno",
        "retorno_medio",
        "media_excesso_periodico",
        "mediana_excesso_periodico",
        "pct_periodos_real_superou_comparador",
        "sharpe_anualizado",
        "sortino_anualizado",
        "calmar",
        "volatilidade_anualizada",
        "downside_vol_anualizada",
        "max_drawdown",
        "drawdown_abs_maximo",
        "score_evidencia_medio",
        "score_evidencia_total",
        "classificacao_evidencia_final",
        "leitura_final",
        "valor_investido_total",
        "valor_posicao",
        "peso_posicao",
        "peso_empresa_aporte",
        "peso_efetivo_ticker",
    ]

    cols = list(df.columns)
    prioridade = []

    for token in prioridade_tokens:
        for col in cols:
            if col not in prioridade and normalizar_nome_coluna(col) == normalizar_nome_coluna(token):
                prioridade.append(col)

    restantes = [c for c in cols if c not in prioridade and not coluna_tecnica_para_ocultar(c)]
    ocultas = [c for c in cols if c not in prioridade and coluna_tecnica_para_ocultar(c)]

    limite_colunas = 36
    selecionadas = (prioridade + restantes)[:limite_colunas]

    if len(selecionadas) == 0:
        selecionadas = cols[:limite_colunas]

    return df[selecionadas].copy()

def preparar_dataframe_tabela_html(df, titulo=None):
    if df is None:
        return pd.DataFrame()

    base = df.copy()
    titulo_norm = normalizar_nome_coluna(titulo or "")

    def texto_invalido_series(serie):
        texto = serie_para_texto_sem_na(serie).str.strip().str.lower()
        return texto.isin(["", "nan", "none", "nat", "<na>", "não informado", "nao informado"])

    def coluna_tem_conteudo(serie):
        vazio = texto_invalido_series(serie)
        if pd.api.types.is_numeric_dtype(serie):
            numerica = pd.to_numeric(serie, errors="coerce")
            return numerica.notna().sum() > 0 and float(numerica.fillna(0).abs().sum()) != 0.0
        return (~vazio).sum() > 0

    def coluna_analitica(coluna):
        nome = normalizar_nome_coluna(coluna)
        tokens = [
            "retorno", "excesso", "risco", "drawdown", "volatil", "downside",
            "sharpe", "sortino", "calmar", "p_valor", "pvalor", "wilcoxon",
            "permutacao", "estatistica", "vitoria", "derrota", "empate",
            "percentil", "score", "classificacao", "evidencia", "bootstrap",
            "valor_investido", "valor_posicao", "valor_aporte", "peso", "participacao",
            "quantidade", "qtd", "n_compras", "n_aportes", "n_periodos", "n_testes",
            "media_", "mediana_", "prob_", "pct_", "taxa", "preco_medio",
        ]
        if nome.startswith("n_") and any(t in nome for t in ["period", "vitoria", "derrota", "empate", "compra", "aporte", "teste", "replica", "controle", "empresa", "ticker", "setor", "subsetor", "segmento"]):
            return True
        return any(t in nome for t in tokens)

    def coluna_chave(coluna):
        nome = normalizar_nome_coluna(coluna)
        tokens = [
            "frequencia", "estrategia", "familia", "categoria", "grupo_comparacao",
            "tipo_comparacao", "comparador", "benchmark", "ticker", "issuer", "nome", "empresa",
            "setor", "subsetor", "segmento", "data", "ano", "mes",
        ]
        return any(t in nome for t in tokens)

    # Remove colunas técnicas conhecidas e colunas estruturalmente vazias.
    colunas_remover = []
    for col in list(base.columns):
        nome = normalizar_nome_coluna(col)
        if coluna_tecnica_para_ocultar(col):
            colunas_remover.append(col)
            continue
        if nome in ["score_vs_outra_estrategia", "score_contra_outra_estrategia", "classificacao_vs_outra_estrategia"]:
            colunas_remover.append(col)
            continue
        if not coluna_tem_conteudo(base[col]):
            colunas_remover.append(col)
            continue

    base = base.drop(columns=colunas_remover, errors="ignore")

    if len(base) == 0 or len(base.columns) == 0:
        return pd.DataFrame()

    # Limpeza de rótulos antes de avaliar densidade informacional.
    for col in base.columns:
        nome = normalizar_nome_coluna(col)
        if pd.api.types.is_object_dtype(base[col]) or str(base[col].dtype).startswith("string"):
            if any(chave in nome for chave in ["estrategia", "grupo", "familia", "categoria"]):
                base[col] = base[col].apply(limpar_rotulo_estrategia)
            elif any(chave in nome for chave in ["comparador", "benchmark", "controle"]):
                base[col] = base[col].apply(limpar_nome_comparador)
            else:
                base[col] = serie_para_texto_sem_na(base[col])

    metric_cols = [c for c in base.columns if coluna_analitica(c)]
    key_cols = [c for c in base.columns if coluna_chave(c)]

    # Para tabelas de comparação/inferência/risco/robustez, remove linhas que são apenas chaves
    # e não carregam resultado analítico. Essa regra é semântica: não depende só de percentual de vazio.
    tabelas_exigem_metrica = any(t in titulo_norm for t in [
        "comparacao", "controle", "capitulacao_vs_euforia", "risco", "inferencia", "robustez",
        "superacao", "benchmark", "evidencia", "estatistica",
    ])

    if tabelas_exigem_metrica and metric_cols:
        mask_metrica = pd.Series(False, index=base.index)
        for col in metric_cols:
            serie = base[col]
            if pd.api.types.is_numeric_dtype(serie):
                numerica = pd.to_numeric(serie, errors="coerce")
                mask_metrica = mask_metrica | numerica.notna()
            else:
                vazio = texto_invalido_series(serie)
                mask_metrica = mask_metrica | (~vazio)
        base = base.loc[mask_metrica].copy()

    # Remove linhas quase vazias, mas preserva tabelas de eventos/composição quando há chaves + métrica.
    if len(base) > 0:
        matriz_texto = base.apply(serie_para_texto_sem_na)
        vazio_df = matriz_texto.apply(lambda s: s.str.strip().str.lower().isin(["", "nan", "none", "nat", "<na>", "não informado", "nao informado"]))
        n_validos = (~vazio_df).sum(axis=1)
        min_validos = 2 if len(base.columns) <= 8 else 3
        if tabelas_exigem_metrica:
            min_validos = max(min_validos, 4)
        base = base.loc[n_validos >= min_validos].copy()

    if len(base) == 0:
        return pd.DataFrame()

    # Remove colunas redundantes por assinatura de conteúdo. Mantém nomes-chave se forem únicos.
    colunas_finais = []
    assinaturas = set()
    for col in base.columns:
        nome = normalizar_nome_coluna(col)
        assinatura = tuple(serie_para_texto_sem_na(base[col]).head(1000).tolist())
        if assinatura in assinaturas and not coluna_chave(col):
            continue
        assinaturas.add(assinatura)
        colunas_finais.append(col)

    base = base[colunas_finais].copy()
    base = ordenar_colunas_exibicao(base)

    # Última passada: descarta colunas que ficaram sem conteúdo depois da filtragem de linhas.
    finais = []
    for col in base.columns:
        if coluna_tem_conteudo(base[col]) or coluna_chave(col):
            finais.append(col)

    base = base[finais].copy() if finais else base
    return base.copy()


    base = df.copy()

    def serie_vazia_ou_nao_informativa(serie):
        texto = serie_para_texto_sem_na(serie).str.strip().str.lower()
        return texto.isin(["", "nan", "none", "nat", "<na>", "não informado", "nao informado"])

    colunas_prioritarias = {
        "frequencia", "label_frequencia", "nome_exibicao_estrategia", "nome_estrategia_html",
        "estrategia_referencia", "familia_estrategia", "categoria_estrategia", "grupo_html",
        "label_grupo_html", "tipo_comparacao", "grupo_comparacao", "nome_exibicao_comparador",
        "benchmark", "ticker", "issuer_code", "nome", "nome_empresa", "setor", "subsetor", "segmento",
        "data_sinal", "data_aporte", "data_periodo", "data_posicao", "ano", "ano_mes",
    }

    colunas_remover = []
    for col in base.columns:
        nome = normalizar_nome_coluna(col)
        serie = base[col]
        vazio = serie_vazia_ou_nao_informativa(serie)
        pct_vazio = float(vazio.mean()) if len(vazio) else 1.0

        if nome in ["score_vs_outra_estrategia", "score_contra_outra_estrategia"]:
            colunas_remover.append(col)
            continue

        if pct_vazio >= 0.85 and nome not in colunas_prioritarias:
            colunas_remover.append(col)
            continue

        if pd.api.types.is_numeric_dtype(serie):
            numerica = pd.to_numeric(serie, errors="coerce")
            if numerica.notna().sum() == 0:
                colunas_remover.append(col)
                continue
            if numerica.notna().sum() > 0 and float(numerica.fillna(0).abs().sum()) == 0.0 and nome not in colunas_prioritarias:
                colunas_remover.append(col)
                continue

        if not pd.api.types.is_numeric_dtype(serie):
            valores_validos = serie_para_texto_sem_na(serie).str.strip().loc[~vazio]
            if len(valores_validos) == 0 and nome not in colunas_prioritarias:
                colunas_remover.append(col)
                continue

    base = base.drop(columns=colunas_remover, errors="ignore")

    padroes_tecnicos = [
        "caminho_", "arquivo_origem", "arquivo_destino", "schema_colunas", "metadata_", "erro_",
        "origem_base_html", "tipo_base_composicao", "data_referencia_html", "modo_processamento_",
        "linhas_origem_estimadas", "linhas_lidas_processamento", "linhas_snapshot_mensal",
        "nome_exibicao_comparador_vazio", "coluna_auxiliar", "merge_", "_merge",
        "score_vs_outra_estrategia", "score_contra_outra_estrategia",
    ]
    base = base[[c for c in base.columns if not any(p in normalizar_nome_coluna(c) for p in padroes_tecnicos)]].copy()
    base = ordenar_colunas_exibicao(base)

    for col in base.columns:
        nome = normalizar_nome_coluna(col)
        if pd.api.types.is_object_dtype(base[col]) or str(base[col].dtype).startswith("string"):
            if any(chave in nome for chave in ["estrategia", "grupo", "familia", "categoria"]):
                base[col] = base[col].apply(limpar_rotulo_estrategia)
            elif any(chave in nome for chave in ["comparador", "benchmark", "controle"]):
                base[col] = base[col].apply(limpar_nome_comparador)

    if len(base) > 0 and len(base.columns) > 0:
        matriz_texto = base.apply(serie_para_texto_sem_na)
        vazio_df = matriz_texto.apply(lambda s: s.str.strip().str.lower().isin(["", "nan", "none", "nat", "<na>", "não informado", "nao informado"]))
        n_validos = (~vazio_df).sum(axis=1)
        minimo_validos = max(2, min(5, int(np.ceil(len(base.columns) * 0.12))))
        base = base.loc[n_validos >= minimo_validos].copy()

    colunas_finais = []
    assinaturas = set()
    for col in base.columns:
        assinatura = tuple(serie_para_texto_sem_na(base[col]).head(500).tolist())
        nome = normalizar_nome_coluna(col)
        if assinatura in assinaturas and nome not in colunas_prioritarias:
            continue
        assinaturas.add(assinatura)
        colunas_finais.append(col)

    return base[colunas_finais].copy()


def formatar_valor_percentual_auto(x, casas=2):
    if x is None or pd.isna(x):
        return ""

    try:
        valor = float(x)
    except Exception:
        return esc(x)

    if abs(valor) <= 1.5:
        return format_pct_prop(valor, casas)
    return format_pct_val(valor, casas)

def formatar_dataframe_exibicao(df):
    base = df.copy()

    for col in base.columns:
        if pd.api.types.is_datetime64_any_dtype(base[col]):
            base[col] = base[col].dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_numeric_dtype(base[col]):
            nome_col = normalizar_nome_coluna(col)
            serie_abs = base[col].dropna().astype(float).abs() if base[col].dropna().size else pd.Series(dtype=float)
            max_abs = float(serie_abs.max()) if len(serie_abs) else np.nan

            if nome_col == "ano" or nome_col.startswith("ano_") or nome_col.endswith("_ano"):
                base[col] = base[col].apply(lambda x: format_ano(x) if pd.notna(x) else "")
            elif col_inteira(col):
                base[col] = base[col].apply(lambda x: format_int(x) if pd.notna(x) else "")
            elif col_percentual_valor(col) or col_percentual_prop(col):
                base[col] = base[col].apply(lambda x: formatar_valor_percentual_auto(x, casas=2) if pd.notna(x) else "")
            elif col_monetaria(col):
                base[col] = base[col].apply(lambda x: format_money(x, casas=2) if pd.notna(x) else "")
            elif max_abs <= 10:
                base[col] = base[col].apply(lambda x: format_num(x, casas=4) if pd.notna(x) else "")
            else:
                base[col] = base[col].apply(lambda x: format_num(x, casas=2) if pd.notna(x) else "")
        else:
            nome_col = normalizar_nome_coluna(col)
            if any(chave in nome_col for chave in ["estrategia", "grupo", "familia", "categoria"]):
                base[col] = base[col].apply(limpar_rotulo_estrategia)
            else:
                base[col] = serie_para_texto_sem_na(base[col])

    base.columns = tornar_colunas_unicas([renomear_coluna_exibicao(c) for c in base.columns])
    return base

def dataframe_para_html_tabela(df, titulo="Tabela", subtitulo=""):
    base_original = preparar_dataframe_tabela_html(df.copy(), titulo=titulo)
    n_original = len(base_original)

    base = reduzir_dataframe_para_exibicao(base_original, max_linhas=MAX_LINHAS_TABELA_HTML)
    n_exibido = len(base)

    base = formatar_dataframe_exibicao(base).copy()
    base = dataframe_para_objeto_sem_na(base)

    thead = "".join([f"<th data-col='{i}'>{esc(col)}</th>" for i, col in enumerate(base.columns)])

    linhas = []
    for _, row in base.iterrows():
        tds = "".join([f"<td>{esc(val)}</td>" for val in row.tolist()])
        linhas.append(f"<tr>{tds}</tr>")
    tbody = "\n".join(linhas)

    nota_amostra = ""
    if n_original > n_exibido:
        nota_amostra = f"""
        <div class="table-note">
          Esta visualização exibe {format_int(n_exibido)} de {format_int(n_original)} linhas para preservar a navegação.
        </div>
        """

    return f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{esc(titulo)}</title>
  <style>
    :root {{
      --bg: #ffffff;
      --ink: #0f172a;
      --muted: #475569;
      --line: #dbe3ee;
      --soft: #f8fafc;
      --soft-2: #f1f5f9;
      --brand: #172554;
      --radius: 14px;
    }}

    * {{ box-sizing: border-box; }}

    html, body {{
      margin: 0;
      padding: 0;
      background: var(--bg);
      color: var(--ink);
      font-family: Inter, Arial, Helvetica, sans-serif;
    }}

    body {{ padding: 16px; }}

    .table-title {{
      font-size: 1.05rem;
      font-weight: 800;
      margin-bottom: 4px;
    }}

    .table-subtitle {{
      font-size: 0.92rem;
      color: var(--muted);
      margin-bottom: 12px;
    }}

    .table-toolbar {{
      display: grid;
      grid-template-columns: minmax(260px, 1fr) auto auto auto;
      gap: 10px;
      align-items: center;
      margin-bottom: 12px;
    }}

    .table-search input,
    .page-size select {{
      width: 100%;
      padding: 10px 12px;
      border: 1px solid #cbd5e1;
      border-radius: 10px;
      font-size: 14px;
      outline: none;
      background: #fff;
    }}

    .table-meta {{
      font-size: 13px;
      color: var(--muted);
      white-space: nowrap;
    }}

    .pager {{
      display: flex;
      gap: 6px;
      align-items: center;
      justify-content: flex-end;
    }}

    .pager button {{
      border: 1px solid #cbd5e1;
      background: #ffffff;
      border-radius: 10px;
      padding: 9px 11px;
      font-weight: 700;
      cursor: pointer;
    }}

    .pager button:disabled {{
      opacity: 0.45;
      cursor: not-allowed;
    }}

    .table-note {{
      margin: 8px 0 12px 0;
      padding: 10px 12px;
      border-radius: 12px;
      background: #eff6ff;
      border: 1px solid #bfdbfe;
      color: #1e3a8a;
      font-size: 13px;
    }}

    .table-wrap {{
      border: 1px solid var(--line);
      border-radius: var(--radius);
      overflow: auto;
      box-shadow: 0 8px 22px rgba(15, 23, 42, 0.06);
      background: #fff;
      max-height: 78vh;
    }}

    table {{
      width: 100%;
      border-collapse: collapse;
      font-size: 14px;
      min-width: 980px;
    }}

    thead th {{
      position: sticky;
      top: 0;
      z-index: 2;
      background: var(--soft);
      border-bottom: 1px solid var(--line);
      padding: 10px 12px;
      text-align: left;
      white-space: nowrap;
      cursor: pointer;
      user-select: none;
    }}

    thead th:hover {{ background: var(--soft-2); }}

    tbody td {{
      padding: 9px 12px;
      border-bottom: 1px solid #eef2f7;
      white-space: nowrap;
      vertical-align: top;
    }}

    tbody tr:nth-child(even) {{ background: #fcfdff; }}
    tbody tr:hover {{ background: #f8fbff; }}

    .sort-indicator {{
      margin-left: 6px;
      color: #64748b;
      font-size: 11px;
    }}

    .empty-state {{
      padding: 18px 12px;
      color: var(--muted);
      font-size: 14px;
    }}

    @media (max-width: 880px) {{
      .table-toolbar {{ grid-template-columns: 1fr; }}
      .pager {{ justify-content: flex-start; }}
    }}
  </style>
</head>
<body>
  <div class="table-title">{esc(titulo)}</div>
  <div class="table-subtitle">{esc(subtitulo)}</div>
  {nota_amostra}

  <div class="table-toolbar">
    <div class="table-search">
      <input type="text" id="tableSearch" placeholder="Buscar na tabela...">
    </div>
    <div class="page-size">
      <select id="pageSize">
        <option value="25">25 linhas</option>
        <option value="50" selected>50 linhas</option>
        <option value="100">100 linhas</option>
        <option value="250">250 linhas</option>
        <option value="500">500 linhas</option>
      </select>
    </div>
    <div class="table-meta">
      Linhas: <span id="visibleCount">0</span> / <span id="totalCount">0</span>
    </div>
    <div class="pager">
      <button id="prevPage">Anterior</button>
      <span id="pageInfo" class="table-meta">1 / 1</span>
      <button id="nextPage">Próxima</button>
    </div>
  </div>

  <div class="table-wrap">
    <table id="tabela">
      <thead>
        <tr>{thead}</tr>
      </thead>
      <tbody>
        {tbody}
      </tbody>
    </table>
    <div id="emptyState" class="empty-state" style="display:none;">Nenhum registro encontrado.</div>
  </div>

  <script>
    (function() {{
      const input = document.getElementById("tableSearch");
      const pageSizeSelect = document.getElementById("pageSize");
      const prevPage = document.getElementById("prevPage");
      const nextPage = document.getElementById("nextPage");
      const pageInfo = document.getElementById("pageInfo");
      const table = document.getElementById("tabela");
      const tbody = table.querySelector("tbody");
      const rows = Array.from(tbody.querySelectorAll("tr"));
      const headers = Array.from(table.querySelectorAll("thead th"));
      const visibleCount = document.getElementById("visibleCount");
      const totalCount = document.getElementById("totalCount");
      const emptyState = document.getElementById("emptyState");

      let filteredRows = rows.slice();
      let currentPage = 1;
      let sortState = {{ col: null, dir: "asc" }};

      totalCount.textContent = rows.length;

      function normalize(text) {{
        return String(text || "")
          .toLowerCase()
          .normalize("NFD")
          .replace(/[\\u0300-\\u036f]/g, "");
      }}

      function parseSortableValue(value) {{
        const raw = String(value || "").trim();

        const numericCandidate = raw
          .replace(/R\\$\\s*/g, "")
          .replace(/\\./g, "")
          .replace(/,/g, ".")
          .replace(/%/g, "")
          .replace(/[^0-9\\-\\.]/g, "");

        const asNumber = Number(numericCandidate);
        if (!Number.isNaN(asNumber) && numericCandidate !== "") return asNumber;

        return raw.toLowerCase();
      }}

      function getCellValue(row, colIndex) {{
        const cell = row.children[colIndex];
        return cell ? cell.innerText.trim() : "";
      }}

      function clearIndicators() {{
        headers.forEach(th => {{
          th.querySelectorAll(".sort-indicator").forEach(el => el.remove());
        }});
      }}

      function addIndicator(th, dir) {{
        const span = document.createElement("span");
        span.className = "sort-indicator";
        span.textContent = dir === "asc" ? "▲" : "▼";
        th.appendChild(span);
      }}

      function applyFilter() {{
        const q = normalize(input.value);

        filteredRows = rows.filter(row => normalize(row.innerText).includes(q));

        if (sortState.col !== null) {{
          const multiplier = sortState.dir === "asc" ? 1 : -1;
          filteredRows.sort((a, b) => {{
            const av = parseSortableValue(getCellValue(a, sortState.col));
            const bv = parseSortableValue(getCellValue(b, sortState.col));
            if (av < bv) return -1 * multiplier;
            if (av > bv) return 1 * multiplier;
            return 0;
          }});
        }}

        currentPage = 1;
        renderPage();
      }}

      function renderPage() {{
        const pageSize = Number(pageSizeSelect.value) || 50;
        const totalPages = Math.max(1, Math.ceil(filteredRows.length / pageSize));

        if (currentPage > totalPages) currentPage = totalPages;

        const start = (currentPage - 1) * pageSize;
        const end = start + pageSize;
        const pageRows = new Set(filteredRows.slice(start, end));

        rows.forEach(row => {{
          row.style.display = pageRows.has(row) ? "" : "none";
        }});

        visibleCount.textContent = filteredRows.length;
        emptyState.style.display = filteredRows.length === 0 ? "block" : "none";
        pageInfo.textContent = `${{currentPage}} / ${{totalPages}}`;
        prevPage.disabled = currentPage <= 1;
        nextPage.disabled = currentPage >= totalPages;
      }}

      function sortByColumn(colIndex) {{
        if (sortState.col === colIndex) {{
          sortState.dir = sortState.dir === "asc" ? "desc" : "asc";
        }} else {{
          sortState.col = colIndex;
          sortState.dir = "asc";
        }}

        clearIndicators();
        addIndicator(headers[colIndex], sortState.dir);

        applyFilter();
      }}

      input.addEventListener("input", applyFilter);
      pageSizeSelect.addEventListener("change", () => {{
        currentPage = 1;
        renderPage();
      }});

      prevPage.addEventListener("click", () => {{
        currentPage = Math.max(1, currentPage - 1);
        renderPage();
      }});

      nextPage.addEventListener("click", () => {{
        currentPage += 1;
        renderPage();
      }});

      headers.forEach((th, idx) => {{
        th.addEventListener("click", () => sortByColumn(idx));
      }});

      applyFilter();
    }})();
  </script>
</body>
</html>"""

def salvar_texto(caminho, conteudo):
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(conteudo, encoding="utf-8")
    return caminho

def validar_tamanho_arquivo_gerado(caminho, rotulo):
    caminho = Path(caminho)
    if not caminho.exists():
        return
    tamanho = caminho.stat().st_size
    if tamanho > MAX_BYTES_GITHUB:
        raise ValueError(
            f"O arquivo gerado '{rotulo}' ultrapassou 100 MB: {caminho} ({tamanho / 1024 / 1024:.2f} MB)."
        )

def salvar_tabela_html(df, caminho_saida, titulo, subtitulo=""):
    caminho_saida = Path(caminho_saida)
    caminho_saida.parent.mkdir(parents=True, exist_ok=True)

    limite_linhas = MAX_LINHAS_TABELA_HTML
    df_base = df.copy()

    while True:
        df_html = reduzir_dataframe_para_exibicao(df_base, max_linhas=limite_linhas)
        caminho_saida.write_text(dataframe_para_html_tabela(df_html, titulo=titulo, subtitulo=subtitulo), encoding="utf-8")

        if caminho_saida.stat().st_size <= MAX_BYTES_ASSET_SEGURO:
            break

        if limite_linhas <= 1000:
            raise ValueError(
                f"A tabela HTML '{titulo}' permaneceu acima do limite seguro mesmo após redução para 1000 linhas."
            )

        limite_linhas = max(1000, limite_linhas // 2)

    validar_tamanho_arquivo_gerado(caminho_saida, titulo)
    return caminho_saida

def salvar_plot_html(fig, caminho_saida, titulo, subtitulo=""):
    caminho_saida = Path(caminho_saida)
    caminho_saida.parent.mkdir(parents=True, exist_ok=True)

    fig.update_layout(template="plotly_white")
    html_plot = pio.to_html(
        fig,
        include_plotlyjs="cdn",
        full_html=False,
        config={
            "responsive": True,
            "displaylogo": False,
            "modeBarButtonsToRemove": ["lasso2d", "select2d"],
        },
    )

    html_doc = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{esc(titulo)}</title>
  <style>
    html, body {{ margin: 0; padding: 0; background: #ffffff; color: #0f172a; font-family: Inter, Arial, Helvetica, sans-serif; }}
    body {{ padding: 14px; }}
    .chart-title {{ font-size: 1.05rem; font-weight: 800; margin-bottom: 4px; }}
    .chart-subtitle {{ font-size: 0.92rem; color: #475569; margin-bottom: 10px; }}
    .chart-wrap {{ border: 1px solid #dbe3ee; border-radius: 14px; overflow: hidden; box-shadow: 0 8px 22px rgba(15, 23, 42, 0.06); }}
  </style>
</head>
<body>
  <div class="chart-title">{esc(titulo)}</div>
  <div class="chart-subtitle">{esc(subtitulo)}</div>
  <div class="chart-wrap">{html_plot}</div>
</body>
</html>"""

    caminho_saida.write_text(html_doc, encoding="utf-8")
    validar_tamanho_arquivo_gerado(caminho_saida, titulo)
    return caminho_saida

def bloco_kpi(titulo, valor, detalhe=""):
    return f"""
    <div class="kpi-card">
        <div class="kpi-title">{esc(titulo)}</div>
        <div class="kpi-value">{esc(valor)}</div>
        <div class="kpi-detail">{esc(detalhe)}</div>
    </div>
    """

def bloco_takeaway(indice, titulo, texto):
    return f"""
    <div class="takeaway-card">
        <div class="takeaway-index">{esc(indice)}</div>
        <div>
            <h3>{esc(titulo)}</h3>
            <p>{esc(texto)}</p>
        </div>
    </div>
    """

def bloco_info(titulo, texto):
    return f"""
    <div class="info-card">
        <h4>{esc(titulo)}</h4>
        <p>{texto}</p>
    </div>
    """

def bloco_viewer_grafico(viewer, titulo):
    return f"""
    <div class="viewer-card">
        <h3>{esc(titulo)}</h3>
        <div class="viewer-toolbar">
            <div class="filter-box">
                <label>Selecione o gráfico</label>
                <select id="grafico_select_{viewer}"></select>
            </div>
            <div class="actions">
                <a id="grafico_open_{viewer}" class="btn-link-open" href="#" target="_blank">Abrir gráfico</a>
            </div>
        </div>
        <iframe id="grafico_frame_{viewer}" class="viewer-frame" src="about:blank" loading="lazy"></iframe>
        <div id="grafico_caption_{viewer}" class="viewer-caption"></div>
    </div>
    """

def bloco_viewer_tabela(viewer, titulo):
    return f"""
    <div class="viewer-card">
        <h3>{esc(titulo)}</h3>
        <div class="viewer-toolbar">
            <div class="filter-box">
                <label>Selecione a tabela</label>
                <select id="tabela_select_{viewer}"></select>
            </div>
            <div class="actions">
                <a id="tabela_open_{viewer}" class="btn-link-open" href="#" target="_blank">Abrir tabela</a>
            </div>
        </div>
        <iframe id="tabela_frame_{viewer}" class="viewer-frame" src="about:blank" loading="lazy"></iframe>
        <div id="tabela_caption_{viewer}" class="viewer-caption"></div>
    </div>
    """

def texto_valido(valor, padrao="N/A"):
    if valor is None:
        return padrao
    if isinstance(valor, float) and pd.isna(valor):
        return padrao
    texto = str(valor)
    if texto.strip() == "" or texto.lower() in ["nan", "none", "nat"]:
        return padrao
    return texto

def obter_top_por_metrica(df, padroes_metrica, maior=True, col_grupo=None):
    if df is None or len(df) == 0:
        return {}

    col_metrica = None
    for padrao in padroes_metrica:
        col_metrica = encontrar_coluna(df, contem_todos=padrao, somente_numerica=True)
        if col_metrica is not None:
            break

    if col_metrica is None:
        return {}

    base = df.dropna(subset=[col_metrica]).copy()
    if len(base) == 0:
        return {}

    if col_grupo is None:
        col_grupo = encontrar_coluna(
            base,
            candidatos=["estrategia", "estrategia_nome", "estrategia_referencia", "nome_estrategia", "carteira", "portfolio", "grupo", "familia"],
            somente_texto=True,
        )

    if maior:
        row = base.sort_values(col_metrica, ascending=False).iloc[0].to_dict()
    else:
        row = base.sort_values(col_metrica, ascending=True).iloc[0].to_dict()

    return {
        "col_grupo": col_grupo,
        "col_metrica": col_metrica,
        "grupo": row.get(col_grupo, "N/A") if col_grupo is not None else "N/A",
        "valor": row.get(col_metrica, np.nan),
        "linha": row,
    }

def obter_menor_drawdown(df, col_grupo=None):
    if df is None or len(df) == 0:
        return {}

    col = encontrar_coluna(df, contem_algum=["drawdown"], somente_numerica=True)
    if col is None:
        return {}

    base = df.dropna(subset=[col]).copy()
    if len(base) == 0:
        return {}

    if col_grupo is None:
        col_grupo = encontrar_coluna(
            base,
            candidatos=["estrategia", "estrategia_nome", "estrategia_referencia", "nome_estrategia", "carteira", "portfolio", "grupo", "familia"],
            somente_texto=True,
        )

    serie = base[col].astype(float)
    if serie.median() < 0:
        row = base.sort_values(col, ascending=False).iloc[0].to_dict()
    else:
        row = base.sort_values(col, ascending=True).iloc[0].to_dict()

    return {
        "col_grupo": col_grupo,
        "col_metrica": col,
        "grupo": row.get(col_grupo, "N/A") if col_grupo is not None else "N/A",
        "valor": row.get(col, np.nan),
        "linha": row,
    }

def contar_classificacoes(df):
    if df is None or len(df) == 0:
        return {}

    col = encontrar_coluna(
        df,
        candidatos=["classificacao_evidencia", "classificacao", "resultado", "decisao", "sinal_evidencia"],
        contem_algum=["evidencia", "classificacao", "resultado", "decisao"],
        somente_texto=True,
    )

    if col is None:
        return {}

    contagem = df[col].fillna("Não informado").astype(str).value_counts().to_dict()
    return contagem

def extrair_periodo(df):
    if df is None or len(df) == 0:
        return ("N/A", "N/A")

    for col in ["data", "dt", "date", "data_ref", "data_referencia"]:
        if col in df.columns:
            s = pd.to_datetime(df[col], errors="coerce").dropna()
            if len(s) > 0:
                return (s.min().strftime("%Y-%m-%d"), s.max().strftime("%Y-%m-%d"))

    for col in ["ano", "ano_ref", "ano_referencia"]:
        if col in df.columns:
            s = pd.to_numeric(df[col], errors="coerce").dropna()
            if len(s) > 0:
                return (str(int(s.min())), str(int(s.max())))

    return ("N/A", "N/A")

def inferir_nome_estrategia_col(df):
    return encontrar_coluna(
        df,
        candidatos=[
            "estrategia_nome",
            "estrategia",
            "estrategia_referencia",
            "nome_estrategia",
            "carteira",
            "portfolio",
            "grupo",
            "familia",
            "tipo_estrategia",
        ],
        somente_texto=True,
    )

def preparar_base_linha(df, col_y_padroes=None, col_grupo=None):
    if df is None or len(df) == 0:
        return None, None, None, None

    base = reduzir_dataframe_para_grafico(df).copy()

    col_x = encontrar_coluna(base, candidatos=["data", "date", "dt", "mes", "periodo", "ano"])
    if col_x is None:
        return None, None, None, None

    if col_grupo is None:
        col_grupo = inferir_nome_estrategia_col(base)

    col_y = None
    if col_y_padroes:
        for padrao in col_y_padroes:
            col_y = encontrar_coluna(base, contem_todos=padrao, somente_numerica=True)
            if col_y is not None:
                break

    if col_y is None:
        numericas = [c for c in colunas_numericas(base) if c != col_x]
        if len(numericas) == 0:
            return None, None, None, None
        col_y = numericas[0]

    return base, col_x, col_y, col_grupo

def adicionar_grafico_linha(catalogo, df, viewer, nome_arquivo, titulo, subtitulo, y_padroes=None, col_grupo=None, y_tickformat=None):
    try:
        base, col_x, col_y, col_grupo = preparar_base_linha(df, y_padroes, col_grupo=col_grupo)
        if base is None or col_x is None or col_y is None:
            return False

        if col_grupo is not None:
            fig = px.line(
                base.sort_values([col_grupo, col_x]),
                x=col_x,
                y=col_y,
                color=col_grupo,
                title=titulo,
                labels={col_x: "Período", col_y: col_y, col_grupo: "Série"},
            )
        else:
            fig = px.line(
                base.sort_values(col_x),
                x=col_x,
                y=col_y,
                title=titulo,
                labels={col_x: "Período", col_y: col_y},
            )

        fig.update_layout(height=580, legend_title_text="")
        if y_tickformat is not None:
            fig.update_yaxes(tickformat=y_tickformat)

        caminho = salvar_plot_html(fig, DIR_GRAFICOS / nome_arquivo, titulo, subtitulo)
        catalogo.append({
            "viewer": viewer,
            "titulo": titulo,
            "subtitulo": subtitulo,
            "caminho_relativo": caminho_relativo(caminho),
        })
        return True
    except Exception as exc:
        print(f"   - Aviso: gráfico não gerado ({titulo}): {exc}")
        return False

def adicionar_grafico_barra(catalogo, df, viewer, nome_arquivo, titulo, subtitulo, y_padroes=None, x_col=None, y_col=None, color_col=None, top_n=40, maior=True, y_tickformat=None):
    try:
        if df is None or len(df) == 0:
            return False

        base = reduzir_dataframe_para_grafico(df).copy()

        if x_col is None:
            x_col = inferir_nome_estrategia_col(base)
            if x_col is None:
                x_col = encontrar_coluna(
                    base,
                    candidatos=["ticker", "empresa", "nome", "setor", "subsetor", "segmento", "metrica", "indicador"],
                    somente_texto=True,
                )

        if y_col is None:
            if y_padroes:
                for padrao in y_padroes:
                    y_col = encontrar_coluna(base, contem_todos=padrao, somente_numerica=True)
                    if y_col is not None:
                        break

        if y_col is None:
            numericas = colunas_numericas(base)
            if len(numericas) == 0:
                return False
            y_col = numericas[0]

        if x_col is None or y_col is None:
            return False

        base = base.dropna(subset=[y_col]).copy()
        if len(base) == 0:
            return False

        if color_col is None:
            color_col = encontrar_coluna(base, candidatos=["familia", "grupo", "tipo", "classe", "metrica", "estrategia_tipo"], somente_texto=True)

        base = base.sort_values(y_col, ascending=not maior).head(top_n).copy()
        base = base.sort_values(y_col, ascending=True).copy()

        fig = px.bar(
            base,
            x=y_col,
            y=x_col,
            color=color_col if color_col is not None and color_col != x_col else None,
            orientation="h",
            title=titulo,
            labels={x_col: "", y_col: y_col},
        )
        fig.update_layout(height=max(540, min(900, 30 * len(base) + 220)), legend_title_text="")
        if y_tickformat is not None:
            fig.update_xaxes(tickformat=y_tickformat)

        caminho = salvar_plot_html(fig, DIR_GRAFICOS / nome_arquivo, titulo, subtitulo)
        catalogo.append({
            "viewer": viewer,
            "titulo": titulo,
            "subtitulo": subtitulo,
            "caminho_relativo": caminho_relativo(caminho),
        })
        return True
    except Exception as exc:
        print(f"   - Aviso: gráfico não gerado ({titulo}): {exc}")
        return False

def adicionar_grafico_scatter(catalogo, df, viewer, nome_arquivo, titulo, subtitulo, x_padroes=None, y_padroes=None, col_grupo=None):
    try:
        if df is None or len(df) == 0:
            return False

        base = reduzir_dataframe_para_grafico(df).copy()

        col_x = None
        col_y = None

        for padrao in x_padroes or []:
            col_x = encontrar_coluna(base, contem_todos=padrao, somente_numerica=True)
            if col_x is not None:
                break

        for padrao in y_padroes or []:
            col_y = encontrar_coluna(base, contem_todos=padrao, somente_numerica=True)
            if col_y is not None:
                break

        if col_x is None or col_y is None:
            numericas = colunas_numericas(base)
            if len(numericas) < 2:
                return False
            col_x = col_x or numericas[0]
            col_y = col_y or numericas[1]

        if col_grupo is None:
            col_grupo = inferir_nome_estrategia_col(base)

        fig = px.scatter(
            base.dropna(subset=[col_x, col_y]),
            x=col_x,
            y=col_y,
            color=col_grupo if col_grupo is not None else None,
            hover_data=[c for c in base.columns[:8]],
            title=titulo,
            labels={col_x: col_x, col_y: col_y},
        )
        fig.update_layout(height=580, legend_title_text="")

        caminho = salvar_plot_html(fig, DIR_GRAFICOS / nome_arquivo, titulo, subtitulo)
        catalogo.append({
            "viewer": viewer,
            "titulo": titulo,
            "subtitulo": subtitulo,
            "caminho_relativo": caminho_relativo(caminho),
        })
        return True
    except Exception as exc:
        print(f"   - Aviso: gráfico não gerado ({titulo}): {exc}")
        return False

def adicionar_grafico_histograma(catalogo, df, viewer, nome_arquivo, titulo, subtitulo, x_padroes=None, color_col=None):
    try:
        if df is None or len(df) == 0:
            return False

        base = reduzir_dataframe_para_grafico(df).copy()

        col_x = None
        for padrao in x_padroes or []:
            col_x = encontrar_coluna(base, contem_todos=padrao, somente_numerica=True)
            if col_x is not None:
                break

        if col_x is None:
            numericas = colunas_numericas(base)
            if len(numericas) == 0:
                return False
            col_x = numericas[0]

        if color_col is None:
            color_col = inferir_nome_estrategia_col(base)

        fig = px.histogram(
            base.dropna(subset=[col_x]),
            x=col_x,
            color=color_col if color_col is not None else None,
            nbins=50,
            marginal="box",
            title=titulo,
            labels={col_x: col_x},
        )
        fig.update_layout(height=580, legend_title_text="")

        caminho = salvar_plot_html(fig, DIR_GRAFICOS / nome_arquivo, titulo, subtitulo)
        catalogo.append({
            "viewer": viewer,
            "titulo": titulo,
            "subtitulo": subtitulo,
            "caminho_relativo": caminho_relativo(caminho),
        })
        return True
    except Exception as exc:
        print(f"   - Aviso: gráfico não gerado ({titulo}): {exc}")
        return False

def escrever_data_js_particionado(payload, caminho_base):
    caminho_base = Path(caminho_base)
    caminho_base.parent.mkdir(parents=True, exist_ok=True)

    json_text = json.dumps(payload, ensure_ascii=False)
    conteudo_unico = "window.HTML_FINAL_META = " + json_text + ";"
    tamanho = len(conteudo_unico.encode("utf-8"))

    if tamanho <= MAX_BYTES_ASSET_SEGURO:
        caminho_base.write_text(conteudo_unico, encoding="utf-8")
        validar_tamanho_arquivo_gerado(caminho_base, "metadata JS")
        return [caminho_base]

    prefixo = caminho_base.with_suffix("")
    arquivos = []
    chunk_size = int(MAX_BYTES_ASSET_SEGURO * 0.70)
    chunks = [json_text[i:i + chunk_size] for i in range(0, len(json_text), chunk_size)]

    for i, chunk in enumerate(chunks, start=1):
        caminho_part = prefixo.parent / f"{prefixo.name}_part_{str(i).zfill(3)}.js"
        conteudo = "window.HTML_FINAL_META_CHUNKS = window.HTML_FINAL_META_CHUNKS || [];\n"
        conteudo += f"window.HTML_FINAL_META_CHUNKS[{i - 1}] = {json.dumps(chunk, ensure_ascii=False)};\n"
        caminho_part.write_text(conteudo, encoding="utf-8")
        validar_tamanho_arquivo_gerado(caminho_part, f"metadata JS parte {i}")
        arquivos.append(caminho_part)

    caminho_loader = prefixo.parent / f"{prefixo.name}_loader.js"
    caminho_loader.write_text(
        "window.HTML_FINAL_META = JSON.parse((window.HTML_FINAL_META_CHUNKS || []).join(''));",
        encoding="utf-8",
    )
    arquivos.append(caminho_loader)

    return arquivos


print("OK")

def escolher_coluna_exata(df, nomes):
    if df is None:
        return None

    mapa = {normalizar_nome_coluna(c): c for c in df.columns}
    for nome in nomes:
        chave = normalizar_nome_coluna(nome)
        if chave in mapa:
            return mapa[chave]
    return None

def escolher_coluna_estrategia_grafico(df):
    return escolher_coluna_exata(df, [
        "nome_exibicao_estrategia",
        "nome_estrategia_html",
        "nome_exibicao_estrategia_real",
        "estrategia_referencia",
        "estrategia_id",
        "chave_estrategia",
    ])

def escolher_coluna_frequencia_grafico(df):
    return escolher_coluna_exata(df, [
        "frequencia",
        "label_frequencia",
        "frequencia_base_conclusao",
    ])

def valor_filtro_valido(valor):
    if valor is None:
        return False

    try:
        if pd.isna(valor):
            return False
    except Exception:
        pass

    texto = str(valor).strip()
    texto_norm = normalizar_nome_coluna(texto)

    valores_invalidos = {
        "", "nan", "none", "nat", "na", "n_a", "n_d", "nao_informado", "nao_informada",
        "null", "pd_na", "pd_nat", "missing", "sem_informacao", "sem_informacoes"
    }

    return texto_norm not in valores_invalidos and texto not in ["<NA>", "<NaN>", "<NaT>"]

def normalizar_frequencia_grafico(valor):
    if not valor_filtro_valido(valor):
        return "Total"

    texto = str(valor).strip().lower()
    texto_norm = normalizar_nome_coluna(texto)

    if "diar" in texto_norm:
        return "Diária"
    if "mens" in texto_norm:
        return "Mensal"
    if "anu" in texto_norm:
        return "Anual"
    if "total" in texto_norm or "geral" in texto_norm or "consolid" in texto_norm:
        return "Total"

    return str(valor).strip()

def ordem_frequencia_grafico(valor):
    mapa = {"Diária": 1, "Mensal": 2, "Anual": 3, "Total": 4}
    return mapa.get(normalizar_frequencia_grafico(valor), 99)

print("\n[3/9] Carregando bases finais do projeto...")

ARQUIVOS = {}

for pasta in [
    DIR_DADOS_MANIFESTOS,
    DIR_DADOS_RESUMO,
    DIR_DADOS_PERFORMANCE,
    DIR_DADOS_RISCO,
    DIR_DADOS_COMPOSICAO,
    DIR_DADOS_APOIO,
]:
    for caminho in sorted(pasta.glob("*.parquet")):
        chave = caminho.stem
        ARQUIVOS[chave] = carregar_parquet(caminho, obrigatorio=False)

MANIFESTO_ARQUIVOS = carregar_json(DIR_DADOS_MANIFESTOS / "manifesto_arquivos_html.json", obrigatorio=False)
CATALOGO_DADOS = carregar_json(DIR_DADOS_MANIFESTOS / "catalogo_dados_html.json", obrigatorio=False)

DF_RESUMO_EXECUTIVO = ARQUIVOS.get("15_1_tbl_resumo_executivo_html")
DF_METRICAS_PRINCIPAIS = ARQUIVOS.get("15_1_tbl_metricas_principais_estrategias_html")
DF_BENCHMARKS = ARQUIVOS.get("15_1_tbl_comparacao_benchmarks_html")
DF_CONTROLES = ARQUIVOS.get("15_1_tbl_comparacao_controles_html")
DF_CAP_VS_EUF = ARQUIVOS.get("15_1_tbl_comparacao_capitulacao_euforia_html")
DF_INFERENCIA = ARQUIVOS.get("15_1_tbl_inferencia_estatistica_html")
DF_ROBUSTEZ = ARQUIVOS.get("15_1_tbl_robustez_sensibilidade_html")

DF_CURVAS_PATRIMONIAIS = ARQUIVOS.get("15_2_base_curvas_patrimoniais_html")
DF_CURVAS_INDEXADAS = ARQUIVOS.get("15_2_base_curvas_indexadas_html")
DF_RETORNOS_PERIODICOS = ARQUIVOS.get("15_2_base_retornos_periodicos_html")
DF_RETORNOS_ACUMULADOS = ARQUIVOS.get("15_2_base_retornos_acumulados_html")
DF_DESEMPENHO_BENCH = ARQUIVOS.get("15_2_base_desempenho_vs_benchmarks_html")
DF_DESEMPENHO_CONTROLES = ARQUIVOS.get("15_2_base_desempenho_vs_controles_html")
DF_DESEMPENHO_CAP_EUF = ARQUIVOS.get("15_2_base_desempenho_capitulacao_vs_euforia_html")
DF_DISTRIBUICAO_RETORNOS = ARQUIVOS.get("15_2_tbl_distribuicao_retornos_html")

DF_DRAWDOWN = ARQUIVOS.get("15_3_base_drawdown_html")
DF_EVENTOS_DRAWDOWN = ARQUIVOS.get("15_3_tbl_eventos_drawdown_html")
DF_VOLATILIDADE_ROLLING = ARQUIVOS.get("15_3_base_volatilidade_rolling_html")
DF_METRICAS_RISCO = ARQUIVOS.get("15_3_base_metricas_risco_html")
DF_METRICAS_RISCO_LONG = ARQUIVOS.get("15_3_base_metricas_risco_long_html")
DF_RISCO_BENCH = ARQUIVOS.get("15_3_base_risco_vs_benchmarks_html")
DF_RISCO_CONTROLES = ARQUIVOS.get("15_3_base_risco_vs_controles_html")
DF_RISCO_CAP_EUF = ARQUIVOS.get("15_3_base_risco_capitulacao_vs_euforia_html")

DF_SINAIS_APORTES = ARQUIVOS.get("15_4_base_sinais_aportes_html")
DF_APORTES = ARQUIVOS.get("15_4_base_aportes_html")
DF_COMPRAS_APORTE = ARQUIVOS.get("15_4_base_compras_aporte_html")
DF_COMPRAS_TICKER = ARQUIVOS.get("15_4_base_compras_ticker_html")
DF_TOP_TICKERS = ARQUIVOS.get("15_4_tbl_top_tickers_comprados_html")
DF_POSICOES = ARQUIVOS.get("15_4_base_posicoes_historicas_html")
DF_POSICOES_SETORIAL = ARQUIVOS.get("15_4_base_posicoes_setorial_mensal_html")
DF_TOP_POSICOES = ARQUIVOS.get("15_4_tbl_top_posicoes_historicas_html")
DF_RESUMO_POSICOES = ARQUIVOS.get("15_4_tbl_resumo_posicoes_historicas_html")
DF_CONCENTRACAO_EMPRESA = ARQUIVOS.get("15_4_base_concentracao_empresa_html")
DF_TOP_EMPRESAS = ARQUIVOS.get("15_4_tbl_top_empresas_html")
DF_CONCENTRACAO_SETORIAL = ARQUIVOS.get("15_4_base_concentracao_setorial_html")
DF_TOP_SETORIAIS = ARQUIVOS.get("15_4_tbl_top_classificacoes_setoriais_html")
DF_TIMELINE = ARQUIVOS.get("15_4_tbl_timeline_sinais_aportes_html")
DF_AUDITORIA = ARQUIVOS.get("15_4_tbl_auditoria_consolidada_html")

DF_MANIFESTO_PARQUET = ARQUIVOS.get("manifesto_arquivos_html")
DF_CATALOGO_PARQUET = ARQUIVOS.get("catalogo_dados_html")
DF_ESTRUTURA_HTML = ARQUIVOS.get("resumo_estrutura_html")
DF_AUDITORIA_HTML = ARQUIVOS.get("auditoria_consolidacao_html")
DF_PARAMETROS_HTML = ARQUIVOS.get("parametros_consolidacao_html")

arquivos_carregados = {k: v.shape for k, v in ARQUIVOS.items() if v is not None}

print("OK")
print(f" - Arquivos parquet carregados........: {format_int(len(arquivos_carregados))}")
print(f" - Resumo executivo...................: {None if DF_RESUMO_EXECUTIVO is None else DF_RESUMO_EXECUTIVO.shape}")
print(f" - Métricas principais................: {None if DF_METRICAS_PRINCIPAIS is None else DF_METRICAS_PRINCIPAIS.shape}")
print(f" - Performance........................: {None if DF_CURVAS_PATRIMONIAIS is None else DF_CURVAS_PATRIMONIAIS.shape}")
print(f" - Risco..............................: {None if DF_METRICAS_RISCO is None else DF_METRICAS_RISCO.shape}")
print(f" - Composição/auditoria...............: {None if DF_AUDITORIA is None else DF_AUDITORIA.shape}")

print("\n[4/9] Derivando métricas, leituras executivas e textos do relatório...")

periodo_inicio, periodo_fim = extrair_periodo(DF_CURVAS_PATRIMONIAIS if DF_CURVAS_PATRIMONIAIS is not None else DF_CURVAS_INDEXADAS)

col_estrategia = inferir_nome_estrategia_col(DF_METRICAS_PRINCIPAIS)
if col_estrategia is None:
    col_estrategia = inferir_nome_estrategia_col(DF_RETORNOS_ACUMULADOS)

n_estrategias = 0
if DF_METRICAS_PRINCIPAIS is not None and col_estrategia is not None:
    n_estrategias = int(DF_METRICAS_PRINCIPAIS[col_estrategia].nunique())
elif DF_CURVAS_PATRIMONIAIS is not None:
    col_tmp = inferir_nome_estrategia_col(DF_CURVAS_PATRIMONIAIS)
    n_estrategias = int(DF_CURVAS_PATRIMONIAIS[col_tmp].nunique()) if col_tmp is not None else 0

n_observacoes_curvas = int(len(DF_CURVAS_PATRIMONIAIS)) if DF_CURVAS_PATRIMONIAIS is not None else 0
n_linhas_auditoria = int(len(DF_AUDITORIA)) if DF_AUDITORIA is not None else 0
n_sinais = int(len(DF_SINAIS_APORTES)) if DF_SINAIS_APORTES is not None else 0
n_aportes = int(len(DF_APORTES)) if DF_APORTES is not None else 0
n_compras = int(len(DF_COMPRAS_TICKER)) if DF_COMPRAS_TICKER is not None else 0
n_posicoes = int(len(DF_POSICOES)) if DF_POSICOES is not None else 0

col_ticker = encontrar_coluna(DF_COMPRAS_TICKER, candidatos=["ticker", "ativo", "codigo_negociacao"], somente_texto=True)
n_tickers_comprados = int(DF_COMPRAS_TICKER[col_ticker].nunique()) if DF_COMPRAS_TICKER is not None and col_ticker is not None else 0

col_empresa = encontrar_coluna(DF_CONCENTRACAO_EMPRESA, candidatos=["empresa", "nome", "nom_res", "issuer_code"], somente_texto=True)
n_empresas_concentracao = int(DF_CONCENTRACAO_EMPRESA[col_empresa].nunique()) if DF_CONCENTRACAO_EMPRESA is not None and col_empresa is not None else 0

best_retorno = obter_top_por_metrica(
    DF_METRICAS_PRINCIPAIS if DF_METRICAS_PRINCIPAIS is not None else DF_RETORNOS_ACUMULADOS,
    [["retorno", "acumul"], ["retorno", "total"], ["valor", "final"], ["patrimonio", "final"], ["cagr"]],
    maior=True,
    col_grupo=col_estrategia,
)

best_sharpe = obter_top_por_metrica(
    DF_METRICAS_PRINCIPAIS if DF_METRICAS_PRINCIPAIS is not None else DF_METRICAS_RISCO,
    [["sharpe"], ["sortino"], ["calmar"]],
    maior=True,
    col_grupo=col_estrategia,
)

best_drawdown = obter_menor_drawdown(
    DF_METRICAS_PRINCIPAIS if DF_METRICAS_PRINCIPAIS is not None else DF_METRICAS_RISCO,
    col_grupo=col_estrategia,
)

best_benchmark = obter_top_por_metrica(
    DF_BENCHMARKS if DF_BENCHMARKS is not None else DF_DESEMPENHO_BENCH,
    [["excesso"], ["diferenca"], ["superacao"], ["retorno", "relativo"], ["retorno"]],
    maior=True,
)

best_controle = obter_top_por_metrica(
    DF_CONTROLES if DF_CONTROLES is not None else DF_DESEMPENHO_CONTROLES,
    [["percentil"], ["posicionamento"], ["superacao"], ["retorno"], ["sharpe"]],
    maior=True,
)

best_cap_euf = obter_top_por_metrica(
    DF_CAP_VS_EUF if DF_CAP_VS_EUF is not None else DF_DESEMPENHO_CAP_EUF,
    [["retorno", "acumul"], ["retorno"], ["valor", "final"], ["sharpe"], ["calmar"]],
    maior=True,
)

contagem_inferencia = contar_classificacoes(DF_INFERENCIA)
contagem_robustez = contar_classificacoes(DF_ROBUSTEZ)

def buscar_contagem_chave(contagem, chaves):
    total = 0
    for k, v in contagem.items():
        nome = normalizar_nome_coluna(k)
        if any(normalizar_nome_coluna(chave) in nome for chave in chaves):
            total += int(v)
    return total

n_evidencias_robustas = buscar_contagem_chave(contagem_inferencia, ["robusta", "confirmada", "significativa"])
if n_evidencias_robustas == 0:
    n_evidencias_robustas = buscar_contagem_chave(contagem_robustez, ["robusta", "estavel", "confirmada"])

n_evidencias_sensiveis = buscar_contagem_chave(contagem_inferencia, ["sensivel", "nao_confirmada", "nao robusta", "fragil"])
if n_evidencias_sensiveis == 0:
    n_evidencias_sensiveis = buscar_contagem_chave(contagem_robustez, ["sensivel", "instavel", "fragil"])

col_aporte_valor = encontrar_coluna(DF_APORTES, contem_algum=["aporte", "valor"], somente_numerica=True)
valor_total_aportado = float(DF_APORTES[col_aporte_valor].sum()) if DF_APORTES is not None and col_aporte_valor is not None else np.nan

nome_best_retorno = texto_valido(best_retorno.get("grupo"))
valor_best_retorno = best_retorno.get("valor", np.nan)
nome_best_sharpe = texto_valido(best_sharpe.get("grupo"))
valor_best_sharpe = best_sharpe.get("valor", np.nan)
nome_best_drawdown = texto_valido(best_drawdown.get("grupo"))
valor_best_drawdown = best_drawdown.get("valor", np.nan)
nome_best_cap_euf = texto_valido(best_cap_euf.get("grupo"))
valor_best_cap_euf = best_cap_euf.get("valor", np.nan)

if best_retorno.get("col_metrica") is not None and col_percentual_prop(best_retorno.get("col_metrica")):
    valor_best_retorno_fmt = format_pct_prop(valor_best_retorno, 2)
elif best_retorno.get("col_metrica") is not None and col_monetaria(best_retorno.get("col_metrica")):
    valor_best_retorno_fmt = format_money(valor_best_retorno, 2)
else:
    valor_best_retorno_fmt = format_num(valor_best_retorno, 4)

if best_drawdown.get("col_metrica") is not None and col_percentual_prop(best_drawdown.get("col_metrica")):
    valor_best_drawdown_fmt = format_pct_prop(valor_best_drawdown, 2)
else:
    valor_best_drawdown_fmt = format_num(valor_best_drawdown, 4)

valor_best_sharpe_fmt = format_num(valor_best_sharpe, 4)

texto_estrategia_lider = (
    f"A estratégia com melhor métrica de retorno no consolidado foi {nome_best_retorno}, com {valor_best_retorno_fmt}."
    if nome_best_retorno != "N/A"
    else "A liderança de retorno deve ser lida diretamente na tabela consolidada de métricas principais."
)

texto_risco_lider = (
    f"A melhor leitura de risco ajustado apareceu em {nome_best_sharpe}, com métrica de eficiência de {valor_best_sharpe_fmt}."
    if nome_best_sharpe != "N/A"
    else "A leitura de risco ajustado deve ser consultada nas tabelas de métricas principais e risco."
)

texto_drawdown_lider = (
    f"A estratégia com menor perda máxima observada no consolidado foi {nome_best_drawdown}, com drawdown de {valor_best_drawdown_fmt}."
    if nome_best_drawdown != "N/A"
    else "A leitura de drawdown deve ser consultada no bloco de risco."
)

texto_cap_euf = (
    f"Na comparação direta entre Capitulação e Euforia, o principal destaque do consolidado foi {nome_best_cap_euf}."
    if nome_best_cap_euf != "N/A"
    else "A comparação direta entre Capitulação e Euforia foi preservada em tabela e gráficos próprios para leitura no dashboard."
)


# ============================================================
# Textos executivos e conclusão numericamente ancorada
# ============================================================

def normalizar_valor_relatorio(valor):
    return normalizar_nome_coluna(texto_valido(valor, ""))

def selecionar_coluna_texto(df, candidatos):
    if df is None or len(df) == 0:
        return None
    col = encontrar_coluna(df, candidatos=candidatos, somente_texto=True)
    if col is not None:
        return col
    for cand in candidatos:
        for c in df.columns:
            if normalizar_nome_coluna(c) == normalizar_nome_coluna(cand):
                return c
    return None

def selecionar_linha_estrategia(df, estrategia, preferencias_frequencia=None, metricas_obrigatorias=None):
    if df is None or len(df) == 0:
        return {}
    base = df.copy()
    col_est = selecionar_coluna_texto(base, [
        "estrategia_referencia", "nome_estrategia_html", "estrategia", "nome_estrategia",
        "serie", "carteira", "portfolio"
    ])
    alvo = normalizar_valor_relatorio(estrategia)
    if col_est is not None:
        serie_norm = base[col_est].map(normalizar_valor_relatorio)
        mask_exata = serie_norm.eq(alvo)
        if mask_exata.any():
            base = base.loc[mask_exata].copy()
        else:
            mask_contem = serie_norm.str.contains(alvo, na=False)
            if mask_contem.any():
                base = base.loc[mask_contem].copy()
    if metricas_obrigatorias:
        cols_metricas = [c for c in metricas_obrigatorias if c in base.columns]
        if cols_metricas:
            mask_metricas = base[cols_metricas].notna().any(axis=1)
            if mask_metricas.any():
                base = base.loc[mask_metricas].copy()
    col_freq = encontrar_coluna(base, candidatos=["frequencia", "periodicidade", "freq"], somente_texto=True)
    if col_freq is not None and preferencias_frequencia:
        serie_freq = base[col_freq].map(normalizar_valor_relatorio)
        for freq in preferencias_frequencia:
            freq_norm = normalizar_valor_relatorio(freq)
            mask = serie_freq.eq(freq_norm)
            if mask.any():
                return base.loc[mask].iloc[0].to_dict()
    if len(base) == 0:
        return {}
    return base.iloc[0].to_dict()

def selecionar_linha_metricas(df, preferencias_frequencia=None, metricas_obrigatorias=None, filtros_texto=None):
    if df is None or len(df) == 0:
        return {}
    base = df.copy()
    if filtros_texto:
        for col_candidatos, alvo in filtros_texto:
            col = selecionar_coluna_texto(base, col_candidatos if isinstance(col_candidatos, list) else [col_candidatos])
            if col is None:
                continue
            alvo_norm = normalizar_valor_relatorio(alvo)
            serie_norm = base[col].map(normalizar_valor_relatorio)
            mask = serie_norm.eq(alvo_norm)
            if not mask.any():
                mask = serie_norm.str.contains(alvo_norm, na=False)
            if mask.any():
                base = base.loc[mask].copy()
    if metricas_obrigatorias:
        cols_metricas = [c for c in metricas_obrigatorias if c in base.columns]
        if cols_metricas:
            mask_metricas = base[cols_metricas].notna().any(axis=1)
            if mask_metricas.any():
                base = base.loc[mask_metricas].copy()
    col_freq = encontrar_coluna(base, candidatos=["frequencia", "periodicidade", "freq"], somente_texto=True)
    if col_freq is not None and preferencias_frequencia:
        serie_freq = base[col_freq].map(normalizar_valor_relatorio)
        for freq in preferencias_frequencia:
            freq_norm = normalizar_valor_relatorio(freq)
            mask = serie_freq.eq(freq_norm)
            if mask.any():
                return base.loc[mask].iloc[0].to_dict()
    if len(base) == 0:
        return {}
    return base.iloc[0].to_dict()

def valor_por_colunas(row, candidatos):
    if not row:
        return (None, np.nan)
    for col in candidatos:
        if col in row and pd.notna(row.get(col)):
            return (col, row.get(col))
    candidatos_norm = [normalizar_nome_coluna(c) for c in candidatos]
    for col, val in row.items():
        col_norm = normalizar_nome_coluna(col)
        if col_norm in candidatos_norm and pd.notna(val):
            return (col, val)
    return (None, np.nan)

def formatar_valor_coluna_relatorio(col, val, casas=2):
    if col is None or val is None or pd.isna(val):
        return "N/A"
    nome = normalizar_nome_coluna(col)
    if any(chave in nome for chave in ["prob", "pct", "percentil", "participacao", "superou", "vitorias", "derrotas"]):
        return formatar_valor_percentual_auto(val, casas=casas)
    if col_percentual_valor(col) or col_percentual_prop(col):
        return formatar_valor_percentual_auto(val, casas=casas)
    if col_monetaria(col):
        return format_money(val, casas=2)
    if col_inteira(col):
        return format_int(val)
    try:
        return format_num(val, casas=4 if abs(float(val)) <= 10 else 2)
    except Exception:
        return esc(val)

def metrica_formatada(row, candidatos, casas=2):
    col, val = valor_por_colunas(row, candidatos)
    return formatar_valor_coluna_relatorio(col, val, casas=casas)

def texto_metrica_estrategia(nome_estrategia, row_metricas, row_risco=None):
    row_risco = row_risco or {}
    retorno_acum = metrica_formatada(row_metricas, ["retorno_acumulado_total", "retorno_acumulado", "retorno_total"])
    retorno_anual = metrica_formatada(row_metricas, ["retorno_anualizado", "cagr"])
    sharpe = metrica_formatada(row_metricas, ["sharpe_anualizado", "sharpe_simplificado", "sharpe"])
    sortino = metrica_formatada(row_metricas, ["sortino_anualizado", "sortino_simplificado", "sortino"])
    calmar = metrica_formatada(row_metricas, ["calmar", "calmar_simplificado"])
    drawdown = metrica_formatada(row_risco if row_risco else row_metricas, ["max_drawdown", "drawdown_maximo", "max_drawdown_abs"])
    n_periodos = metrica_formatada(row_metricas, ["n_periodos"])
    return (
        f"{nome_estrategia}: retorno acumulado de <b>{retorno_acum}</b>, retorno anualizado de <b>{retorno_anual}</b>, "
        f"Sharpe de <b>{sharpe}</b>, Sortino de <b>{sortino}</b>, Calmar de <b>{calmar}</b>, "
        f"drawdown máximo de <b>{drawdown}</b> e <b>{n_periodos}</b> períodos observados"
    )

PREFERENCIA_FREQ_PRINCIPAL = ["mensal", "anual", "diaria"]
linha_metricas_cap = selecionar_linha_estrategia(DF_METRICAS_PRINCIPAIS, "Capitulação", PREFERENCIA_FREQ_PRINCIPAL, ["retorno_acumulado_total", "retorno_anualizado", "sharpe_anualizado", "calmar"])
linha_metricas_euf = selecionar_linha_estrategia(DF_METRICAS_PRINCIPAIS, "Euforia", PREFERENCIA_FREQ_PRINCIPAL, ["retorno_acumulado_total", "retorno_anualizado", "sharpe_anualizado", "calmar"])
linha_risco_cap = selecionar_linha_estrategia(DF_METRICAS_RISCO, "Capitulação", PREFERENCIA_FREQ_PRINCIPAL, ["max_drawdown", "volatilidade_anualizada", "sharpe_anualizado", "calmar"])
linha_risco_euf = selecionar_linha_estrategia(DF_METRICAS_RISCO, "Euforia", PREFERENCIA_FREQ_PRINCIPAL, ["max_drawdown", "volatilidade_anualizada", "sharpe_anualizado", "calmar"])
linha_resumo_cap = selecionar_linha_estrategia(DF_RESUMO_EXECUTIVO, "Capitulação", None, ["score_evidencia_medio", "score_vs_ibovespa", "score_vs_cdi"])
linha_resumo_euf = selecionar_linha_estrategia(DF_RESUMO_EXECUTIVO, "Euforia", None, ["score_evidencia_medio", "score_vs_ibovespa", "score_vs_cdi"])

metricas_cap_texto = texto_metrica_estrategia("Capitulação", linha_metricas_cap, linha_risco_cap)
metricas_euf_texto = texto_metrica_estrategia("Euforia", linha_metricas_euf, linha_risco_euf)

linha_cap_euf_mensal = selecionar_linha_metricas(
    DF_CAP_VS_EUF,
    preferencias_frequencia=["mensal", "anual", "diaria"],
    metricas_obrigatorias=["retorno_acumulado_capitulacao", "retorno_acumulado_euforia", "pct_periodos_capitulacao_superou_euforia"],
)
linha_cap_euf_teste = selecionar_linha_metricas(
    DF_CAP_VS_EUF,
    preferencias_frequencia=["mensal", "anual", "diaria"],
    metricas_obrigatorias=["p_valor_teste_sinal", "p_valor_wilcoxon", "p_valor_permutacao", "classificacao_evidencia_capitulacao_vs_euforia"],
)

cap_euf_ret_cap = metrica_formatada(linha_cap_euf_mensal, ["retorno_acumulado_capitulacao"])
cap_euf_ret_euf = metrica_formatada(linha_cap_euf_mensal, ["retorno_acumulado_euforia"])
cap_euf_dif = metrica_formatada(linha_cap_euf_mensal, ["diferenca_retorno_acumulado_capitulacao_menos_euforia", "diferenca_capitulacao_menos_euforia"])
cap_euf_pct_cap = metrica_formatada(linha_cap_euf_mensal, ["pct_periodos_capitulacao_superou_euforia", "pct_capitulacao_superou_euforia_teste"])
cap_euf_pct_euf = metrica_formatada(linha_cap_euf_mensal, ["pct_periodos_euforia_superou_capitulacao"])
cap_euf_excesso = metrica_formatada(linha_cap_euf_mensal, ["media_excesso_capitulacao_vs_euforia", "media_excesso_periodico"])
cap_euf_p_perm = metrica_formatada(linha_cap_euf_teste, ["p_valor_permutacao", "p_valor_holm_permutacao"])
cap_euf_p_wil = metrica_formatada(linha_cap_euf_teste, ["p_valor_wilcoxon", "p_valor_holm_wilcoxon"])
cap_euf_classif = texto_valido(linha_cap_euf_teste.get("classificacao_evidencia_capitulacao_vs_euforia") or linha_cap_euf_teste.get("interpretacao_direcional"), "N/A")

score_cap_ibov = metrica_formatada(linha_resumo_cap, ["score_vs_ibovespa"])
score_cap_cdi = metrica_formatada(linha_resumo_cap, ["score_vs_cdi"])
score_cap_alea = metrica_formatada(linha_resumo_cap, ["score_vs_aleatorias"])
score_cap_mensais = metrica_formatada(linha_resumo_cap, ["score_vs_mensais"])
score_cap_medio = metrica_formatada(linha_resumo_cap, ["score_evidencia_medio"])
score_euf_ibov = metrica_formatada(linha_resumo_euf, ["score_vs_ibovespa"])
score_euf_cdi = metrica_formatada(linha_resumo_euf, ["score_vs_cdi"])
score_euf_alea = metrica_formatada(linha_resumo_euf, ["score_vs_aleatorias"])
score_euf_mensais = metrica_formatada(linha_resumo_euf, ["score_vs_mensais"])
score_euf_medio = metrica_formatada(linha_resumo_euf, ["score_evidencia_medio"])
prob_boot_cap = metrica_formatada(linha_resumo_cap, ["prob_bootstrap_media_acima_zero_media", "prob_bootstrap_media_acima_zero"])
prob_boot_euf = metrica_formatada(linha_resumo_euf, ["prob_bootstrap_media_acima_zero_media", "prob_bootstrap_media_acima_zero"])
class_final_cap = texto_valido(linha_resumo_cap.get("classificacao_evidencia_final"), "N/A")
class_final_euf = texto_valido(linha_resumo_euf.get("classificacao_evidencia_final"), "N/A")

linha_top_ticker = obter_top_por_metrica(
    DF_TOP_TICKERS if DF_TOP_TICKERS is not None else DF_COMPRAS_TICKER,
    [["valor", "investido"], ["n", "compras"], ["quantidade"]],
    maior=True,
    col_grupo=col_ticker,
)
ticker_mais_relevante = texto_valido(linha_top_ticker.get("grupo"), "N/A")
ticker_mais_relevante_valor = formatar_valor_coluna_relatorio(linha_top_ticker.get("col_metrica"), linha_top_ticker.get("valor"))

col_setor_nome = encontrar_coluna(DF_TOP_SETORIAIS, candidatos=["classificacao_hierarquica_html", "rotulo_classificacao_hierarquica", "nome_classificacao", "setor", "subsetor", "segmento"], somente_texto=True)
linha_top_setorial = obter_top_por_metrica(
    DF_TOP_SETORIAIS if DF_TOP_SETORIAIS is not None else DF_CONCENTRACAO_SETORIAL,
    [["valor", "investido"], ["peso", "classificacao"], ["valor"]],
    maior=True,
    col_grupo=col_setor_nome,
)
classificacao_setorial_top = texto_valido(linha_top_setorial.get("grupo"), "N/A")
classificacao_setorial_top_valor = formatar_valor_coluna_relatorio(linha_top_setorial.get("col_metrica"), linha_top_setorial.get("valor"))

valor_total_aportado_fmt = format_money(valor_total_aportado, 2) if pd.notna(valor_total_aportado) else "N/A"

badges_html = f"""
<div class="badges">
    <span class="badge">Período: {esc(periodo_inicio)} a {esc(periodo_fim)}</span>
    <span class="badge">{format_int(n_estrategias)} estratégias/séries</span>
    <span class="badge">{format_int(n_sinais)} sinais consolidados</span>
    <span class="badge">{format_int(n_aportes)} aportes</span>
    <span class="badge">{format_int(n_compras)} compras por ticker</span>
    <span class="badge">{format_int(n_evidencias_robustas)} evidências robustas/significativas</span>
</div>
"""

kpis_html = "\n".join([
    bloco_kpi("Estratégias/séries", format_int(n_estrategias), "séries finais analisadas"),
    bloco_kpi("Sinais", format_int(n_sinais), "eventos de capitulação, euforia e controles"),
    bloco_kpi("Aportes", format_int(n_aportes), "registros de alocação"),
    bloco_kpi("Compras por ticker", format_int(n_compras), "linhas de compras agregadas"),
    bloco_kpi("Tickers comprados", format_int(n_tickers_comprados), "ativos distintos nas compras"),
    bloco_kpi("Evidências robustas", format_int(n_evidencias_robustas), "inferência estatística consolidada"),
])

takeaways = [
    ("01", "Capitulação e Euforia são comparadas em bases equivalentes", "As estratégias de extremos de mercado são avaliadas com a mesma régua de performance, risco, benchmarks e controles."),
    ("02", "Performance e risco são analisados separadamente", texto_estrategia_lider + " " + texto_risco_lider),
    ("03", "A robustez define a força da evidência", f"O painel preserva {format_int(n_evidencias_robustas)} evidências robustas/significativas e {format_int(n_evidencias_sensiveis)} evidências sensíveis ou frágeis identificadas nos testes."),
    ("04", "Controles aleatórios e mensais são o contrafactual principal", "As estratégias reais são confrontadas com alternativas sem condicionamento ao sinal, reduzindo o risco de atribuir causalidade a padrões de calendário ou aleatoriedade."),
    ("05", "Sinais, aportes e compras ficam auditáveis", f"O estudo expõe {format_int(n_sinais)} sinais, {format_int(n_aportes)} aportes e {format_int(n_compras)} registros de compras por ticker."),
    ("06", "Composição e concentração fecham a leitura econômica", f"As posições históricas, a concentração por empresa e a concentração por setor, subsetor e segmento ajudam a interpretar a origem econômica dos resultados."),
]

takeaways_html = "\n".join([bloco_takeaway(*item) for item in takeaways])

conteudo_introducao = f"""
<p>
    Este estudo investiga se momentos extremos de mercado carregam informação útil para a construção de carteiras no Brasil.
    A hipótese central é simples: períodos de <b>capitulação</b>, marcados por estresse, quedas relevantes e deterioração ampla
    de preços, podem oferecer pontos de entrada diferentes daqueles observados em períodos de <b>euforia</b>, nos quais o mercado
    apresenta força, proximidade de máximas e maior complacência com risco.
</p>

<p>
    O projeto compara essas duas famílias de sinais em uma estrutura única de avaliação: retorno, risco, eficiência ajustada,
    drawdown, benchmarks, controles aleatórios, controles mensais, inferência estatística, robustez e composição das carteiras.
    A pergunta não é apenas qual família terminou com maior retorno, mas se o resultado foi persistente, estatisticamente
    defensável e economicamente diversificado.
</p>

<p>
    A análise cobre o período disponível nas curvas finais entre <b>{esc(periodo_inicio)}</b> e <b>{esc(periodo_fim)}</b>.
    Foram mapeadas <b>{format_int(n_estrategias)}</b> estratégias ou séries finais, <b>{format_int(n_sinais)}</b> sinais,
    <b>{format_int(n_aportes)}</b> aportes e <b>{format_int(n_compras)}</b> registros de compras por ticker.
</p>
"""

conteudo_metodologia = f"""
<p>
    A metodologia parte da identificação de janelas de mercado extremas. Os sinais de capitulação buscam representar momentos
    de estresse, desvalorização e piora ampla do mercado; os sinais de euforia capturam ambientes de força e proximidade de
    máximas. A partir desses sinais, as carteiras são formadas com regras padronizadas de elegibilidade, liquidez, filtros
    contábeis e distribuição dos aportes.
</p>

<p>
    A comparação foi desenhada para evitar que o resultado dependa apenas da data escolhida. Por isso, as estratégias reais são
    avaliadas contra benchmarks de mercado, controles aleatórios e controles mensais. Os controles funcionam como contrafactuais:
    eles ajudam a distinguir o que parece vir do sinal de mercado daquilo que poderia surgir de aportes em datas alternativas.
</p>

<p>
    A avaliação combina três níveis. O primeiro mede performance: curvas patrimoniais, curvas indexadas, retornos acumulados e
    retornos periódicos. O segundo mede risco: volatilidade, drawdown, eventos de queda e métricas de eficiência. O terceiro
    mede robustez: testes estatísticos, sensibilidade de parâmetros e posicionamento empírico frente aos controles.
</p>
"""

conteudo_resultados = f"""
<p>
    O primeiro bloco de resultado é a comparação de performance. {esc(texto_estrategia_lider)}
    As curvas patrimoniais e indexadas permitem verificar se a liderança foi estável ao longo do tempo ou concentrada em poucos
    episódios específicos.
</p>

<p>
    O segundo bloco é a leitura de risco. {esc(texto_risco_lider)} {esc(texto_drawdown_lider)}
    Essa separação é importante porque uma estratégia pode terminar com retorno superior, mas ainda assim carregar trajetória
    mais instável, maior drawdown ou pior eficiência ajustada ao risco.
</p>

<p>
    O terceiro bloco é a comparação com benchmarks e controles. O estudo confronta as estratégias reais com referências passivas,
    controles aleatórios e controles mensais, permitindo avaliar se o comportamento observado se distancia de alternativas que
    não dependem de sinais de capitulação ou euforia.
</p>

<p>
    O quarto bloco é a comparação direta entre Capitulação e Euforia. {esc(texto_cap_euf)}
    A leitura final deve combinar esse bloco com as métricas de risco, os testes estatísticos, a robustez por sensibilidade e
    a composição das carteiras.
</p>
"""

conteudo_estatistica = f"""
<p>
    A inferência estatística foi construída para separar resultado visualmente interessante de evidência mais confiável. O
    estudo considera testes, posicionamento empírico, percentis, sensibilidade de parâmetros e comparação com controles.
</p>

<p>
    Foram identificadas <b>{format_int(n_evidencias_robustas)}</b> evidências robustas, significativas ou confirmadas nos
    consolidados disponíveis, além de <b>{format_int(n_evidencias_sensiveis)}</b> evidências sensíveis, frágeis ou não
    confirmadas. Essa gradação é essencial para separar desempenho econômico promissor de evidência estatística efetivamente
    estável.
</p>

<p>
    Resultados que superam benchmarks, mas não se destacam contra controles aleatórios ou mensais, pedem cautela. Resultados que
    mantêm bom posicionamento em performance, risco, controles e testes estatísticos são metodologicamente mais fortes.
</p>
"""

conteudo_composicao = f"""
<p>
    A composição mostra como os sinais se transformaram em alocações reais dentro do backtest. A análise contempla
    <b>{format_int(n_aportes)}</b> registros de aportes, <b>{format_int(n_compras)}</b> registros de compras por ticker e
    <b>{format_int(n_posicoes)}</b> registros de posições históricas.
</p>

<p>
    A concentração por empresa e por classificação setorial permite verificar se o resultado veio de uma carteira diversificada
    ou de poucos nomes dominantes. Essa leitura é indispensável para interpretar estratégias baseadas em extremos de mercado,
    porque retornos elevados podem depender de concentração involuntária em empresas, setores ou subsetores específicos.
</p>
"""

conteudo_conclusao = f"""
<div class="conclusao">
    <h3>Conclusão</h3>
    <p>
        Este projeto não foi construído para responder apenas se <b>Capitulação</b> ou <b>Euforia</b> teve o maior retorno no fim da amostra.
        Ele foi desenhado para testar, sob a mesma régua metodológica, se comprar momentos de estresse extremo ou momentos de força do mercado
        produz trajetórias diferentes quando as duas abordagens são confrontadas com benchmarks, controles aleatórios, controles mensais,
        métricas de risco, inferência estatística, robustez e composição das carteiras.
    </p>

    <p>
        A base final cobre o período de <b>{esc(periodo_inicio)}</b> a <b>{esc(periodo_fim)}</b>, com <b>{format_int(n_estrategias)}</b>
        estratégias ou séries finais, <b>{format_int(n_sinais)}</b> sinais consolidados, <b>{format_int(n_aportes)}</b> aportes,
        <b>{format_int(n_compras)}</b> registros de compras por ticker, <b>{format_int(n_tickers_comprados)}</b> tickers distintos comprados
        e <b>{format_int(n_posicoes)}</b> registros de posições históricas. Em termos financeiros, os aportes somaram aproximadamente
        <b>{valor_total_aportado_fmt}</b> na base consolidada de alocações.
    </p>

    <p>
        Na leitura das métricas principais, a comparação operacional entre as duas famílias fica mais concreta: {metricas_cap_texto};
        {metricas_euf_texto}. Essa distinção é importante porque retorno acumulado, retorno anualizado, Sharpe, Sortino, Calmar e drawdown
        não respondem à mesma pergunta. O retorno mede crescimento patrimonial, enquanto as métricas de eficiência e queda máxima indicam
        se esse crescimento foi obtido com uma trajetória aceitável de risco.
    </p>

    <p>
        Na comparação direta entre as famílias, a linha consolidada principal mostra Capitulação com retorno acumulado de
        <b>{cap_euf_ret_cap}</b> contra <b>{cap_euf_ret_euf}</b> em Euforia, diferença de <b>{cap_euf_dif}</b> e excesso médio periódico de
        <b>{cap_euf_excesso}</b>. Em frequência de superação, Capitulação ficou à frente em <b>{cap_euf_pct_cap}</b> dos períodos, enquanto
        Euforia superou Capitulação em <b>{cap_euf_pct_euf}</b>. Nos testes formais, a leitura inclui p-valor de Wilcoxon de
        <b>{cap_euf_p_wil}</b>, p-valor de permutação de <b>{cap_euf_p_perm}</b> e classificação direcional de <b>{esc(cap_euf_classif)}</b>.
    </p>

    <p>
        A camada de evidência reforça que a conclusão não pode depender apenas da curva mais alta. No resumo executivo, Capitulação apresentou
        score vs Ibovespa de <b>{score_cap_ibov}</b>, score vs CDI de <b>{score_cap_cdi}</b>, score vs controles aleatórios de
        <b>{score_cap_alea}</b>, score vs controles mensais de <b>{score_cap_mensais}</b> e score médio de evidência de
        <b>{score_cap_medio}</b>. Para Euforia, os respectivos scores foram <b>{score_euf_ibov}</b>, <b>{score_euf_cdi}</b>,
        <b>{score_euf_alea}</b>, <b>{score_euf_mensais}</b> e <b>{score_euf_medio}</b>. A confirmação por bootstrap também aparece no resumo:
        <b>{prob_boot_cap}</b> para Capitulação e <b>{prob_boot_euf}</b> para Euforia, com classificações finais de
        <b>{esc(class_final_cap)}</b> e <b>{esc(class_final_euf)}</b>, respectivamente.
    </p>

    <p>
        A composição das carteiras fecha a interpretação econômica. O ticker mais relevante por valor ou contagem agregada foi
        <b>{esc(ticker_mais_relevante)}</b>, com métrica de <b>{ticker_mais_relevante_valor}</b>, enquanto a principal classificação
        setorial identificada foi <b>{esc(classificacao_setorial_top)}</b>, com métrica de <b>{classificacao_setorial_top_valor}</b>.
        Esses números ajudam a separar desempenho genuinamente diversificado de desempenho concentrado em poucos ativos, empresas ou blocos setoriais.
    </p>

    <h3>Interpretação dos resultados</h3>
    <p>
        A primeira leitura do projeto é que estratégias de extremos de mercado precisam ser avaliadas como uma arquitetura completa, não como
        uma aposta isolada em datas de compra. A diferença entre comprar capitulação e comprar euforia só é economicamente relevante se persistir
        depois de controlar risco, drawdown, benchmarks, controles aleatórios e controles mensais.
    </p>

    <p>
        O bloco de performance responde quem acumulou mais patrimônio. O bloco de risco responde se esse patrimônio foi construído com trajetória
        mais eficiente ou mais instável. O bloco de controles responde se o sinal superou alternativas de calendário e aleatoriedade. O bloco de
        inferência responde se a evidência é estatisticamente defensável. E o bloco de composição responde de onde veio o resultado: quais tickers,
        empresas, setores, subsetores e segmentos concentraram as compras e posições.
    </p>

    <p>
        A leitura estatística deve ser tratada com disciplina. Scores próximos de neutralidade, p-valores elevados ou baixa confirmação por bootstrap
        não anulam necessariamente o resultado econômico, mas reduzem a força da conclusão. Por outro lado, quando performance, risco, controles,
        bootstrap e testes convergem, a evidência deixa de ser apenas uma boa história visual e passa a ser uma hipótese quantitativamente mais robusta.
    </p>

    <h3>Principais aprendizados</h3>
    <ul>
        <li>A comparação entre Capitulação e Euforia precisa combinar retorno, risco, drawdown, eficiência, controles e inferência; uma única métrica não é suficiente.</li>
        <li>O projeto analisou <b>{format_int(n_sinais)}</b> sinais, <b>{format_int(n_aportes)}</b> aportes e <b>{format_int(n_compras)}</b> compras por ticker, preservando rastreabilidade operacional.</li>
        <li>As métricas principais mostram diferenças entre crescimento patrimonial e eficiência ajustada ao risco: retorno acumulado, Sharpe, Sortino e Calmar podem apontar lideranças diferentes.</li>
        <li>A comparação direta registrou superação de Capitulação em <b>{cap_euf_pct_cap}</b> dos períodos e de Euforia em <b>{cap_euf_pct_euf}</b>, o que impede uma leitura baseada apenas no valor final da curva.</li>
        <li>Benchmarks, controles aleatórios e controles mensais funcionam como contrafactuais essenciais para reduzir o risco de confundir sinal com calendário ou sorte amostral.</li>
        <li>A concentração por ticker, empresa, setor, subsetor e segmento é indispensável para entender se o resultado veio de uma carteira ampla ou de poucos blocos dominantes.</li>
        <li>A evidência final deve ser lida por convergência: desempenho econômico, robustez estatística, confirmação por bootstrap e coerência de composição.</li>
    </ul>

    <h3>Síntese</h3>
    <p>
        Em termos práticos, o estudo entrega uma leitura auditável de duas formas opostas de operar extremos de mercado no Brasil. A Capitulação
        representa a tentativa de comprar estresse; a Euforia representa a tentativa de comprar força. O resultado final não deve ser reduzido a
        um campeão absoluto, porque cada família precisa ser julgada por retorno, risco, estabilidade, controles e composição.
    </p>

    <p>
        A síntese é que o projeto permite identificar quando uma estratégia de extremo de mercado parece economicamente promissora, quando essa
        promessa sobrevive aos controles e quando ela se apoia em evidência estatística mais frágil. Essa distinção é o principal valor do relatório:
        ele transforma uma comparação intuitiva entre medo e euforia em uma avaliação quantitativa, auditável e interpretável por camadas.
    </p>
</div>
"""

cards_metodologia = "\n".join([
    bloco_info("Capitulação", "Sinais associados a estresse de mercado, quedas relevantes e possível assimetria de oportunidade após movimentos extremos."),
    bloco_info("Euforia", "Sinais associados a força de mercado, proximidade de máximas e possível risco de comprar em ambientes excessivamente otimistas."),
    bloco_info("Universo elegível", "Formação de carteiras com regras de elegibilidade, liquidez e filtros contábeis aplicados antes das compras."),
    bloco_info("Benchmarks", "Comparação contra referências de mercado para avaliar se a estratégia adiciona valor frente a alternativas passivas."),
    bloco_info("Controles", "Amostras aleatórias e mensais usadas como contrafactuais para testar se o sinal supera datas alternativas de compra."),
    bloco_info("Inferência e robustez", "Testes estatísticos, sensibilidade de parâmetros e percentis empíricos para avaliar estabilidade dos resultados."),
])

cards_estatistica = "\n".join([
    bloco_info("Evidências robustas/significativas", f"<b>{format_int(n_evidencias_robustas)}</b> registros identificados nos consolidados de inferência/robustez."),
    bloco_info("Evidências sensíveis/frágeis", f"<b>{format_int(n_evidencias_sensiveis)}</b> registros classificados como sensíveis, frágeis ou não confirmados."),
    bloco_info("Melhor retorno consolidado", f"<b>{esc(nome_best_retorno)}</b><br>{esc(valor_best_retorno_fmt)}"),
    bloco_info("Melhor eficiência", f"<b>{esc(nome_best_sharpe)}</b><br>{esc(valor_best_sharpe_fmt)}"),
])

resumo_execucao = {
    "periodo_inicio": periodo_inicio,
    "periodo_fim": periodo_fim,
    "n_arquivos_parquet_carregados": len(arquivos_carregados),
    "n_estrategias": n_estrategias,
    "n_observacoes_curvas": n_observacoes_curvas,
    "n_sinais": n_sinais,
    "n_aportes": n_aportes,
    "n_compras": n_compras,
    "n_tickers_comprados": n_tickers_comprados,
    "n_posicoes": n_posicoes,
    "n_empresas_concentracao": n_empresas_concentracao,
    "n_evidencias_robustas": n_evidencias_robustas,
    "n_evidencias_sensiveis": n_evidencias_sensiveis,
    "melhor_retorno": nome_best_retorno,
    "melhor_eficiencia": nome_best_sharpe,
    "menor_drawdown": nome_best_drawdown,
}
ARQUIVO_RESUMO_JSON.write_text(json.dumps(resumo_execucao, ensure_ascii=False, indent=2), encoding="utf-8")

print("OK")
print(f" - Período final......................: {periodo_inicio} a {periodo_fim}")
print(f" - Estratégias/séries.................: {format_int(n_estrategias)}")
print(f" - Sinais.............................: {format_int(n_sinais)}")
print(f" - Aportes............................: {format_int(n_aportes)}")
print(f" - Compras por ticker.................: {format_int(n_compras)}")
print(f" - Evidências robustas/significativas.: {format_int(n_evidencias_robustas)}")

print("\n[5/9] Gerando tabelas HTML externas com busca, ordenação e visualização em iframe...")


catalogo_tabelas = []

# Tabelas derivadas dos principais gráficos para consulta direta no dashboard.
DF_TABELA_GRAFICO_RETORNO_TOTAL = None
DF_TABELA_GRAFICO_EFICIENCIA = None
DF_TABELA_GRAFICO_SUPERACAO_BENCH = None
DF_TABELA_GRAFICO_SUPERACAO_CONTROLES = None
DF_TABELA_GRAFICO_RISCO_CONTROLES = None
DF_TABELA_GRAFICO_CONCENTRACAO_SETORIAL = None

try:
    if DF_METRICAS_PRINCIPAIS is not None and len(DF_METRICAS_PRINCIPAIS) > 0:
        col_freq = escolher_coluna_frequencia_grafico(DF_METRICAS_PRINCIPAIS) if "escolher_coluna_frequencia_grafico" in globals() else None
        base_tmp = DF_METRICAS_PRINCIPAIS.copy()
        if col_freq is not None:
            base_tmp["_freq_tmp"] = base_tmp[col_freq].apply(normalizar_frequencia_grafico)
            if "Diária" in set(base_tmp["_freq_tmp"]):
                base_tmp = base_tmp.loc[base_tmp["_freq_tmp"] == "Diária"].copy()
        col_est = escolher_coluna_estrategia_grafico(base_tmp) if "escolher_coluna_estrategia_grafico" in globals() else inferir_nome_estrategia_col(base_tmp)
        if col_est is not None:
            base_tmp["_estrategia_limpa"] = base_tmp[col_est].apply(limpar_rotulo_estrategia)
            base_tmp = base_tmp.drop_duplicates("_estrategia_limpa", keep="first")
        cols = [c for c in [
            col_est,
            "familia_estrategia",
            "categoria_estrategia",
            "retorno_acumulado_total",
            "retorno_anualizado",
            "sharpe_anualizado",
            "sortino_anualizado",
            "calmar",
            "n_periodos",
            "data_inicio",
            "data_fim",
        ] if c is not None and c in base_tmp.columns]
        DF_TABELA_GRAFICO_RETORNO_TOTAL = base_tmp[[c for c in [
            col_est,
            "familia_estrategia",
            "categoria_estrategia",
            "retorno_acumulado_total",
            "retorno_anualizado",
            "n_periodos",
            "data_inicio",
            "data_fim",
        ] if c is not None and c in base_tmp.columns]].copy()
        DF_TABELA_GRAFICO_EFICIENCIA = base_tmp[[c for c in [
            col_est,
            "familia_estrategia",
            "categoria_estrategia",
            "sharpe_anualizado",
            "sortino_anualizado",
            "calmar",
            "n_periodos",
            "data_inicio",
            "data_fim",
        ] if c is not None and c in base_tmp.columns]].copy()
except Exception as exc:
    print(f"   - Aviso: não foi possível criar tabelas derivadas de métricas principais: {exc}")

try:
    if DF_BENCHMARKS is not None and len(DF_BENCHMARKS) > 0:
        cols = [c for c in [
            "estrategia_referencia",
            "frequencia",
            "nome_exibicao_comparador",
            "n_periodos",
            "n_vitorias_real",
            "n_derrotas_real",
            "pct_periodos_real_superou_comparador",
            "media_excesso_periodico",
            "mediana_excesso_periodico",
            "p_valor_permutacao",
            "classificacao_evidencia_benchmark",
        ] if c in DF_BENCHMARKS.columns]
        DF_TABELA_GRAFICO_SUPERACAO_BENCH = DF_BENCHMARKS[cols].copy() if cols else None
except Exception as exc:
    print(f"   - Aviso: não foi possível criar tabela derivada de benchmarks: {exc}")

try:
    if DF_CONTROLES is not None and len(DF_CONTROLES) > 0:
        cols = [c for c in [
            "estrategia_referencia",
            "frequencia",
            "grupo_comparacao",
            "tipo_comparacao",
            "tipo_controle_mensal",
            "cenario_aleatorio",
            "pct_periodos_real_superou_comparador",
            "media_excesso_periodico",
            "pct_replicas_retorno_acumulado_real_superou_controle",
            "pct_replicas_sharpe_real_superou_controle",
            "pct_replicas_sortino_real_superou_controle",
            "pct_replicas_calmar_real_superou_controle",
            "pct_controles_retorno_acumulado_real_superou_controle",
            "pct_controles_sharpe_real_superou_controle",
            "pct_controles_calmar_real_superou_controle",
            "classificacao_empirica_vs_aleatorias",
            "classificacao_empirica_vs_mensais",
        ] if c in DF_CONTROLES.columns]
        DF_TABELA_GRAFICO_SUPERACAO_CONTROLES = DF_CONTROLES[cols].copy() if cols else None
except Exception as exc:
    print(f"   - Aviso: não foi possível criar tabela derivada de controles: {exc}")

try:
    if DF_RISCO_CONTROLES is not None and len(DF_RISCO_CONTROLES) > 0:
        cols = [c for c in [
            "estrategia_referencia",
            "frequencia",
            "nome_exibicao_comparador",
            "tipo_comparacao",
            "grupo_comparacao",
            "tipo_controle_mensal",
            "pct_replicas_sharpe_real_superou_controle",
            "pct_replicas_sortino_real_superou_controle",
            "pct_replicas_calmar_real_superou_controle",
            "pct_replicas_volatilidade_real_menor_controle",
            "pct_replicas_drawdown_abs_real_menor_controle",
            "pct_controles_sharpe_real_superou_controle",
            "pct_controles_calmar_real_superou_controle",
            "pct_controles_drawdown_abs_real_menor_controle",
        ] if c in DF_RISCO_CONTROLES.columns]
        DF_TABELA_GRAFICO_RISCO_CONTROLES = DF_RISCO_CONTROLES[cols].copy() if cols else None
except Exception as exc:
    print(f"   - Aviso: não foi possível criar tabela derivada de risco contra controles: {exc}")

try:
    if DF_TOP_SETORIAIS is not None and len(DF_TOP_SETORIAIS) > 0:
        cols = [c for c in [
            "estrategia_referencia",
            "nome_exibicao_estrategia",
            "nivel_classificacao_html",
            "setor_html",
            "subsetor_html",
            "segmento_html",
            "classificacao_hierarquica_html",
            "valor_investido_total",
            "peso_medio_quando_presente",
            "peso_maximo",
            "pct_aportes_com_exposicao",
            "n_tickers_distintos",
            "n_empresas_distintas",
            "ranking_valor_estrategia_nivel",
            "ranking_frequencia_estrategia_nivel",
        ] if c in DF_TOP_SETORIAIS.columns]
        DF_TABELA_GRAFICO_CONCENTRACAO_SETORIAL = DF_TOP_SETORIAIS[cols].copy() if cols else None
except Exception as exc:
    print(f"   - Aviso: não foi possível criar tabela derivada de concentração setorial: {exc}")


def valor_textual_vazio_tabela(valor):
    if valor is None:
        return True
    try:
        if pd.isna(valor):
            return True
    except Exception:
        pass
    texto = str(valor).strip()
    texto_norm = normalizar_nome_coluna(texto)
    return texto_norm in ["", "nan", "none", "nat", "na", "nao_informado", "nao_informada", "nao_aplicavel", "nao_se_aplica", "null"]

def coluna_tecnica_para_ocultar(coluna):
    nome = normalizar_nome_coluna(coluna)

    manter = {
        "flag_estrategia_real",
        "flag_sinal_intenso",
        "flag_sinal_muito_intenso",
        "flag_capitulacao_superou_euforia",
        "flag_euforia_superou_capitulacao",
    }
    if nome in manter:
        return False

    nomes_exatos_ocultar = {
        "nome_tabela_origem",
        "nome_tabela_html",
        "tema_tabela_html",
        "tema_html",
        "fonte_analitica",
        "fonte",
        "visao_comparacao",
        "visao_inferencia",
        "descricao_visao_inferencia",
        "aplicacao_metodologica",
        "comparacao_id",
        "comparacao_id_original",
        "subtipo_comparacao",
        "chave_comparador",
        "chave_estrategia",
        "chave_aporte",
        "chave_classificacao",
        "ordem_frequencia",
        "ordem_frequencia_html",
        "ordem_frequencia_inferencia",
        "ordem_grupo_comparacao",
        "ordem_estrategia_html",
        "ordem_cenario",
        "ordem_sinal",
        "ordem_aporte",
        "ordem_global",
        "ordem_exibicao",
        "ordem_manifesto",
        "id_dataset_html",
        "schema_colunas",
        "metadata_destino_ok",
        "erro_metadata_destino",
        "erro_copia",
        "request_id",
        "replica_id",
        "carteira_id",
        "arquivo_origem",
        "arquivo_destino",
        "caminho_origem",
        "caminho_destino",
        "caminho_relativo_html_final",
        "caminho_relativo_dados",
        "arquivo_origem_existe",
        "arquivo_destino_existe",
        "copiado_com_sucesso",
        "tipo_base_composicao",
        "classe_tamanho_html",
        "recomendacao_consumo_html",
        "nome_exibicao_comparador_vazio",
    }
    if nome in nomes_exatos_ocultar:
        return True

    padroes_ocultar = [
        "arquivo_",
        "caminho",
        "schema",
        "metadata",
        "erro_",
        "origem_base",
        "data_referencia_html",
        "modo_processamento",
        "linhas_origem",
        "linhas_destino",
        "linhas_lidas",
        "linhas_snapshot",
        "colunas_origem",
        "colunas_destino",
        "tamanho_origem",
        "tamanho_destino",
        "tamanho_diferente",
        "classe_tamanho",
        "recomendacao_consumo",
        "tipo_grafico_sugerido",
        "label_valor_grafico",
        "arquivo_origem_auditoria",
        "diretorio_origem_auditoria",
        "coluna_auxiliar",
        "merge_",
        "_merge",
        "score_vs_outra_estrategia",
        "score_contra_outra_estrategia",
        "classificacao_vs_outra_estrategia",
    ]
    if any(p in nome for p in padroes_ocultar):
        return True

    if nome.startswith("flag_") and nome not in manter:
        return True
    if nome.startswith("status_arquivo"):
        return True

    return False

def coluna_chave_tabela(coluna):
    nome = normalizar_nome_coluna(coluna)
    tokens = [
        "frequencia", "estrategia", "familia", "categoria", "grupo_comparacao",
        "tipo_comparacao", "comparador", "benchmark", "controle", "ticker",
        "issuer", "nome", "empresa", "setor", "subsetor", "segmento",
        "data", "ano", "mes", "periodo", "cenario",
    ]
    return any(t in nome for t in tokens)

def coluna_analitica_tabela(coluna):
    nome = normalizar_nome_coluna(coluna)
    tokens = [
        "retorno", "excesso", "risco", "drawdown", "volatil", "downside",
        "sharpe", "sortino", "calmar", "p_valor", "pvalor", "wilcoxon",
        "permutacao", "estatistica", "vitoria", "derrota", "empate",
        "percentil", "score", "classificacao", "evidencia", "bootstrap",
        "valor_investido", "valor_aporte", "valor_posicao", "valor_total",
        "peso", "participacao", "quantidade", "qtd", "n_compras", "n_aportes",
        "n_periodos", "n_testes", "n_replicas", "n_controles", "n_tickers",
        "n_empresas", "media", "mediana", "prob", "pct", "taxa", "preco_medio",
        "min_", "max_", "q05", "q25", "q75", "q95", "ic_",
    ]
    if nome.startswith("n_"):
        return True
    return any(t in nome for t in tokens)

def coluna_tem_conteudo_tabela(serie, coluna=None, permitir_zero=False):
    if serie is None:
        return False
    nome = normalizar_nome_coluna(coluna or getattr(serie, "name", ""))
    if pd.api.types.is_numeric_dtype(serie):
        numerica = pd.to_numeric(serie, errors="coerce")
        if numerica.notna().sum() == 0:
            return False
        if permitir_zero or nome.startswith("flag_") or nome.startswith("n_"):
            return True
        return float(numerica.fillna(0).abs().sum()) != 0.0
    texto = serie_para_texto_sem_na(serie).str.strip()
    validos = texto.loc[~texto.apply(valor_textual_vazio_tabela)]
    return len(validos) > 0

def linha_tem_metrica_analitica(row, metric_cols):
    for col in metric_cols:
        val = row.get(col)
        if not valor_textual_vazio_tabela(val):
            return True
    return False

def preparar_dataframe_tabela_html(df, titulo=None):
    if df is None:
        return pd.DataFrame()

    base = df.copy()
    if len(base) == 0 or len(base.columns) == 0:
        return pd.DataFrame()

    titulo_norm = normalizar_nome_coluna(titulo or "")

    # 1) Remove colunas técnicas de rastreabilidade que não pertencem à apresentação final.
    colunas_remover = []
    for col in list(base.columns):
        if coluna_tecnica_para_ocultar(col):
            colunas_remover.append(col)
            continue
        nome_col_limpeza = normalizar_nome_coluna(col)
        permitir_zero = coluna_chave_tabela(col) or nome_col_limpeza.startswith("n_") or nome_col_limpeza.startswith("flag_")
        if not coluna_tem_conteudo_tabela(base[col], col, permitir_zero=permitir_zero):
            colunas_remover.append(col)

    base = base.drop(columns=colunas_remover, errors="ignore")
    if len(base.columns) == 0:
        return pd.DataFrame()

    # 2) Limpa rótulos textuais relevantes.
    for col in base.columns:
        nome = normalizar_nome_coluna(col)
        if pd.api.types.is_object_dtype(base[col]) or str(base[col].dtype).startswith("string"):
            if any(chave in nome for chave in ["estrategia", "familia", "categoria"]):
                base[col] = base[col].apply(limpar_rotulo_estrategia)
            elif any(chave in nome for chave in ["comparador", "benchmark", "controle"]):
                base[col] = base[col].apply(limpar_nome_comparador)
            else:
                base[col] = serie_para_texto_sem_na(base[col]).str.strip()

    metric_cols = [c for c in base.columns if coluna_analitica_tabela(c)]
    key_cols = [c for c in base.columns if coluna_chave_tabela(c)]

    tabelas_exigem_metrica = any(t in titulo_norm for t in [
        "comparacao", "controle", "capitulacao_vs_euforia", "risco", "inferencia",
        "robustez", "superacao", "benchmark", "evidencia", "estatistica",
    ])

    # 3) Remove linhas estruturais sem conteúdo analítico real nas tabelas analíticas.
    if tabelas_exigem_metrica and metric_cols:
        mask = base.apply(lambda row: linha_tem_metrica_analitica(row, metric_cols), axis=1)
        base = base.loc[mask].copy()

    if len(base) == 0:
        return pd.DataFrame()

    # 4) Remove linhas praticamente vazias após descarte de colunas técnicas.
    matriz = base.apply(serie_para_texto_sem_na)
    vazio = matriz.apply(lambda s: s.str.strip().str.lower().isin(["", "nan", "none", "nat", "<na>", "nao informado", "não informado", "null"]))
    n_validos = (~vazio).sum(axis=1)
    min_validos = 2 if len(base.columns) <= 8 else 3
    if tabelas_exigem_metrica:
        min_validos = max(min_validos, 4)
    base = base.loc[n_validos >= min_validos].copy()

    if len(base) == 0:
        return pd.DataFrame()

    # 5) Remove colunas que ficaram sem conteúdo depois da filtragem de linhas.
    finais = []
    for col in base.columns:
        nome_col_limpeza = normalizar_nome_coluna(col)
        permitir_zero = coluna_chave_tabela(col) or nome_col_limpeza.startswith("n_") or nome_col_limpeza.startswith("flag_")
        if coluna_tem_conteudo_tabela(base[col], col, permitir_zero=permitir_zero):
            finais.append(col)
    base = base[finais].copy() if finais else pd.DataFrame()

    if len(base) == 0 or len(base.columns) == 0:
        return pd.DataFrame()

    # 6) Remove colunas redundantes por assinatura de conteúdo.
    colunas_finais = []
    assinaturas = set()
    for col in base.columns:
        assinatura = tuple(serie_para_texto_sem_na(base[col]).head(2000).tolist())
        if assinatura in assinaturas and not coluna_chave_tabela(col):
            continue
        assinaturas.add(assinatura)
        colunas_finais.append(col)

    base = base[colunas_finais].copy()
    base = ordenar_colunas_exibicao(base)

    # 7) Limite de colunas orientado a apresentação. A tabela de auditoria pode manter um pouco mais.
    limite = 42 if "auditoria" in titulo_norm else 34
    if len(base.columns) > limite:
        prioridade = []
        restantes = []
        for col in base.columns:
            if coluna_chave_tabela(col) or coluna_analitica_tabela(col):
                prioridade.append(col)
            else:
                restantes.append(col)
        base = base[(prioridade + restantes)[:limite]].copy()

    return base.reset_index(drop=True)

def salvar_tabela_html(df, caminho_saida, titulo, subtitulo=""):
    caminho_saida = Path(caminho_saida)
    caminho_saida.parent.mkdir(parents=True, exist_ok=True)

    df_base = preparar_dataframe_tabela_html(df.copy(), titulo=titulo)
    if df_base is None or len(df_base) == 0 or len(df_base.columns) == 0:
        print(f"   - Aviso: tabela não gerada ({titulo}): sem linhas analíticas após higienização.")
        return None

    limite_linhas = MAX_LINHAS_TABELA_HTML

    while True:
        df_html = reduzir_dataframe_para_exibicao(df_base, max_linhas=limite_linhas)
        caminho_saida.write_text(dataframe_para_html_tabela(df_html, titulo=titulo, subtitulo=subtitulo), encoding="utf-8")

        if caminho_saida.stat().st_size <= MAX_BYTES_ASSET_SEGURO:
            break

        if limite_linhas <= 1000:
            raise ValueError(
                f"A tabela HTML '{titulo}' permaneceu acima do limite seguro mesmo após redução para 1000 linhas."
            )

        limite_linhas = max(1000, limite_linhas // 2)

    validar_tamanho_arquivo_gerado(caminho_saida, titulo)
    return caminho_saida

def coluna_existe(df, coluna):
    return df is not None and coluna in df.columns

def colunas_existentes(df, colunas):
    if df is None:
        return []
    return [c for c in colunas if c in df.columns]

def primeira_coluna_existente(df, colunas):
    if df is None:
        return None
    for col in colunas:
        if col in df.columns:
            return col
    return None

def serie_valida_apresentacao(serie):
    texto = serie_para_texto_sem_na(serie).str.strip().str.lower()
    return ~texto.isin(["", "nan", "none", "nat", "<na>", "nao informado", "não informado", "nao aplicavel", "não aplicável", "null"])

def mascara_alguma_coluna_preenchida(df, colunas):
    if df is None or len(df) == 0 or not colunas:
        return pd.Series(False, index=df.index if df is not None else [])
    mask = pd.Series(False, index=df.index)
    for col in colunas:
        if col not in df.columns:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            mask = mask | pd.to_numeric(df[col], errors="coerce").notna()
        else:
            mask = mask | serie_valida_apresentacao(df[col])
    return mask

def limpar_frequencias_invalidas_tabela(df):
    if df is None or len(df) == 0:
        return df
    base = df.copy()
    for col in [c for c in base.columns if "frequencia" in normalizar_nome_coluna(c)]:
        base[col] = base[col].apply(normalizar_frequencia_grafico)
        # Para tabelas onde frequência é aplicável, valores inválidos devem ficar vazios,
        # não como "<NA>" ou "Não informado".
        base[col] = base[col].replace({"<NA>": "", "nan": "", "None": "", "N/A": "", "Não informado": "", "Nao informado": ""})
    return base

def finalizar_tabela_derivada(df):
    if df is None or len(df) == 0:
        return None
    base = df.copy()
    base = limpar_frequencias_invalidas_tabela(base)

    # Remove colunas técnicas e colunas sem conteúdo, mas sem apagar zeros legítimos de contagens/flags.
    manter = []
    for col in base.columns:
        nome = normalizar_nome_coluna(col)
        if coluna_tecnica_para_ocultar(col):
            continue
        permitir_zero = nome.startswith("n_") or nome.startswith("flag_") or nome.startswith("qtd_") or "quantidade" in nome
        if coluna_tem_conteudo_tabela(base[col], col, permitir_zero=permitir_zero):
            manter.append(col)
    base = base[manter].copy() if manter else pd.DataFrame()
    if len(base) == 0 or len(base.columns) == 0:
        return None

    # Remove duplicidades por conteúdo depois da seleção semântica.
    assinaturas = set()
    finais = []
    for col in base.columns:
        assinatura = tuple(serie_para_texto_sem_na(base[col]).head(5000).tolist())
        if assinatura in assinaturas and not coluna_chave_tabela(col):
            continue
        assinaturas.add(assinatura)
        finais.append(col)

    base = base[finais].copy()
    base = ordenar_colunas_exibicao(base)
    return base.reset_index(drop=True)


def texto_coluna_seguro(df, col):
    if df is None or col not in df.columns:
        return pd.Series("", index=df.index if df is not None else [])
    return serie_para_texto_sem_na(df[col]).str.strip()

def coluna_contem(df, col, padrao):
    if df is None or col not in df.columns:
        return pd.Series(False, index=df.index if df is not None else [])
    return texto_coluna_seguro(df, col).str.lower().str.contains(padrao, regex=True, na=False)

def coluna_preenchida(df, col):
    if df is None or col not in df.columns:
        return pd.Series(False, index=df.index if df is not None else [])
    if pd.api.types.is_numeric_dtype(df[col]):
        return pd.to_numeric(df[col], errors="coerce").notna()
    return serie_valida_apresentacao(df[col])

def mascara_metricas_validas(df, colunas):
    return mascara_alguma_coluna_preenchida(df, [c for c in colunas if c in df.columns])

def coalescer_colunas(df, colunas, nome_saida):
    base = pd.Series(pd.NA, index=df.index, dtype="object")
    for col in colunas:
        if col not in df.columns:
            continue
        atual = df[col].astype("object")
        mask_base_vazia = pd.isna(base) | serie_para_texto_sem_na(base).str.strip().isin(["", "nan", "None", "<NA>"])
        mask_atual_valida = coluna_preenchida(df, col)
        base.loc[mask_base_vazia & mask_atual_valida] = atual.loc[mask_base_vazia & mask_atual_valida]
    df[nome_saida] = base
    return df

def tabela_bloco(df, colunas, metricas, filtro=None, exigir_metricas=True):
    if df is None or len(df) == 0:
        return None
    base = df.copy()
    if filtro is not None:
        try:
            base = base.loc[filtro(base)].copy()
        except Exception:
            return None
    cols = colunas_existentes(base, colunas)
    if not cols:
        return None
    base = base[cols].copy()
    metricas_existentes = [c for c in metricas if c in base.columns]
    if exigir_metricas and metricas_existentes:
        base = base.loc[mascara_metricas_validas(base, metricas_existentes)].copy()
    elif exigir_metricas and not metricas_existentes:
        return None
    return finalizar_tabela_derivada(base)

def mask_controle_aleatorio_periodo(df):
    tipo = coluna_contem(df, "tipo_comparacao", "controle_aleatorio|aleatorio")
    grupo = coluna_contem(df, "grupo_comparacao", "aleatorio|aleatórias|aleatorias")
    metricas = mascara_metricas_validas(df, [
        "media_excesso_periodico",
        "mediana_excesso_periodico",
        "pct_periodos_real_superou_comparador",
        "media_excesso_anualizada_aproximada",
    ])
    return (tipo | grupo) & metricas

def mask_controle_mensal_periodo(df):
    tipo = coluna_contem(df, "tipo_comparacao", "controle_mensal")
    grupo = coluna_contem(df, "grupo_comparacao", "controle mensal|mensal")
    controle = mascara_metricas_validas(df, ["tipo_controle_mensal", "nome_tipo_controle_mensal", "n_periodos_unicos"])
    metricas = mascara_metricas_validas(df, [
        "media_excesso_periodico",
        "mediana_excesso_periodico",
        "pct_periodos_real_superou_comparador",
        "media_excesso_anualizada_aproximada",
    ])
    return (tipo | grupo | controle) & metricas

def derivar_controles_aleatorios_periodo(df):
    cols = [
        "frequencia", "estrategia_referencia", "grupo_comparacao", "tipo_comparacao",
        "n_periodos", "n_vitorias", "n_empates", "n_derrotas",
        "media_excesso_periodico", "mediana_excesso_periodico",
        "pct_periodos_real_superou_comparador", "media_excesso_anualizada_aproximada",
    ]
    metricas = ["media_excesso_periodico", "pct_periodos_real_superou_comparador"]
    return tabela_bloco(df, cols, metricas, filtro=mask_controle_aleatorio_periodo)

def derivar_controles_mensais_periodo(df):
    cols = [
        "frequencia", "estrategia_referencia", "tipo_controle_mensal", "nome_tipo_controle_mensal",
        "grupo_comparacao", "tipo_comparacao", "n_periodos", "n_periodos_unicos",
        "media_excesso_periodico", "mediana_excesso_periodico",
        "pct_periodos_real_superou_comparador", "media_excesso_anualizada_aproximada",
    ]
    metricas = ["media_excesso_periodico", "pct_periodos_real_superou_comparador"]
    return tabela_bloco(df, cols, metricas, filtro=mask_controle_mensal_periodo)

def derivar_controles_aleatorios_robustez(df):
    cols = [
        "frequencia", "estrategia_referencia", "cenario_aleatorio",
        "n_replicas_disponiveis", "n_sementes_aleatorias_distintas",
        "n_pares_comparacao", "n_periodos_comparaveis_total",
        "pct_replicas_retorno_acumulado_real_superou_controle",
        "pct_replicas_retorno_anualizado_real_superou_controle",
        "pct_replicas_retorno_equivalente_real_superou_controle",
        "pct_replicas_sharpe_real_superou_controle",
        "pct_replicas_sortino_real_superou_controle",
        "pct_replicas_calmar_real_superou_controle",
        "pct_replicas_volatilidade_real_menor_controle",
        "pct_replicas_drawdown_abs_real_menor_controle",
        "classificacao_robustez_retorno_acumulado",
        "classificacao_robustez_periodos",
    ]
    metricas = [
        "pct_replicas_retorno_acumulado_real_superou_controle",
        "pct_replicas_sharpe_real_superou_controle",
        "pct_replicas_sortino_real_superou_controle",
        "pct_replicas_calmar_real_superou_controle",
    ]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_controles_mensais_robustez(df):
    cols = [
        "frequencia", "estrategia_referencia", "cenario_mensal",
        "tipo_controle_mensal", "nome_tipo_controle_mensal",
        "n_controles_mensais", "n_tipos_controle_mensal",
        "n_pares_comparacao", "n_periodos_comparaveis_total",
        "pct_controles_retorno_acumulado_real_superou_controle",
        "pct_controles_retorno_anualizado_real_superou_controle",
        "pct_controles_retorno_equivalente_real_superou_controle",
        "pct_controles_sharpe_real_superou_controle",
        "pct_controles_sortino_real_superou_controle",
        "pct_controles_calmar_real_superou_controle",
        "pct_controles_volatilidade_real_menor_controle",
        "pct_controles_drawdown_abs_real_menor_controle",
        "classificacao_robustez_retorno_acumulado",
        "classificacao_robustez_periodos",
    ]
    metricas = [
        "pct_controles_retorno_acumulado_real_superou_controle",
        "pct_controles_sharpe_real_superou_controle",
        "pct_controles_sortino_real_superou_controle",
        "pct_controles_calmar_real_superou_controle",
    ]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_controles_classificacao(df):
    cols = [
        "frequencia", "estrategia_referencia", "tipo_comparacao", "grupo_comparacao",
        "classificacao_empirica_vs_aleatorias", "classificacao_empirica_vs_mensais",
        "classificacao_evidencia_aleatorio", "classificacao_evidencia_mensal",
        "n_replicas_aleatorias", "n_controles_mensais",
        "n_testes_significativos_5pct", "n_testes_significativos_10pct",
    ]
    metricas = [
        "classificacao_empirica_vs_aleatorias", "classificacao_empirica_vs_mensais",
        "classificacao_evidencia_aleatorio", "classificacao_evidencia_mensal",
    ]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_controles_testes(df):
    cols = [
        "frequencia", "estrategia_referencia", "grupo_comparacao", "tipo_comparacao",
        "tipo_controle_mensal", "cenario_aleatorio", "n_periodos",
        "n_validos_teste_sinal", "p_valor_teste_sinal", "p_valor_holm_teste_sinal",
        "status_teste_sinal", "estatistica_wilcoxon", "p_valor_wilcoxon",
        "p_valor_holm_wilcoxon", "status_wilcoxon", "estatistica_observada_permutacao",
        "p_valor_permutacao", "p_valor_holm_permutacao", "n_permutacoes",
        "status_permutacao", "n_testes_significativos_5pct",
        "n_testes_significativos_10pct",
    ]
    metricas = ["p_valor_teste_sinal", "p_valor_wilcoxon", "p_valor_permutacao", "status_teste_sinal", "status_wilcoxon", "status_permutacao"]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_cap_euf_descritiva_resumo(df):
    cols = [
        "frequencia", "n_periodos", "data_inicio", "data_fim",
        "retorno_acumulado_capitulacao", "retorno_acumulado_euforia",
        "diferenca_retorno_acumulado_capitulacao_menos_euforia",
        "retorno_anualizado_capitulacao", "retorno_anualizado_euforia",
        "diferenca_retorno_anualizado_capitulacao_menos_euforia",
        "volatilidade_anualizada_capitulacao", "volatilidade_anualizada_euforia",
        "sharpe_simplificado_capitulacao", "sharpe_simplificado_euforia",
        "sortino_simplificado_capitulacao", "sortino_simplificado_euforia",
        "max_drawdown_capitulacao", "max_drawdown_euforia",
        "calmar_simplificado_capitulacao", "calmar_simplificado_euforia",
        "media_excesso_capitulacao_vs_euforia", "mediana_excesso_capitulacao_vs_euforia",
        "pct_periodos_capitulacao_superou_euforia", "pct_periodos_euforia_superou_capitulacao",
        "vencedor_retorno_acumulado", "vencedor_sharpe_simplificado",
        "vencedor_sortino_simplificado", "vencedor_calmar_simplificado", "vencedor_max_drawdown",
    ]
    metricas = ["retorno_acumulado_capitulacao", "retorno_anualizado_capitulacao", "volatilidade_anualizada_capitulacao"]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_cap_euf_metricas_longas(df):
    cols = [
        "frequencia", "metrica", "nome_metrica", "criterio",
        "valor_capitulacao", "valor_euforia", "diferenca_capitulacao_menos_euforia",
        "vencedor",
    ]
    metricas = ["valor_capitulacao", "valor_euforia", "diferenca_capitulacao_menos_euforia", "vencedor"]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_cap_euf_testes(df):
    cols = [
        "frequencia", "estrategia_referencia", "grupo_comparacao", "tipo_comparacao",
        "n_observacoes_teste", "n_vitorias_capitulacao", "n_vitorias_euforia",
        "n_empates", "pct_capitulacao_superou_euforia_teste",
        "media_excesso_periodico", "mediana_excesso_periodico", "direcao_economica",
        "p_valor_teste_sinal", "p_valor_holm_teste_sinal", "status_teste_sinal",
        "estatistica_wilcoxon", "p_valor_wilcoxon", "p_valor_holm_wilcoxon", "status_wilcoxon",
        "estatistica_observada_media", "p_valor_permutacao", "p_valor_holm_permutacao",
        "n_permutacoes", "status_permutacao", "n_testes_significativos_5pct",
        "classificacao_evidencia_capitulacao_vs_euforia", "interpretacao_direcional",
    ]
    metricas = ["p_valor_teste_sinal", "p_valor_wilcoxon", "p_valor_permutacao", "n_observacoes_teste", "classificacao_evidencia_capitulacao_vs_euforia"]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_risco_cap_euf(df):
    cols = [
        "frequencia", "n_periodos", "data_inicio", "data_fim",
        "volatilidade_anualizada_capitulacao", "volatilidade_anualizada_euforia",
        "diferenca_volatilidade_anualizada_capitulacao_menos_euforia",
        "downside_vol_anualizada_capitulacao", "downside_vol_anualizada_euforia",
        "diferenca_downside_vol_anualizada_capitulacao_menos_euforia",
        "max_drawdown_capitulacao", "max_drawdown_euforia",
        "diferenca_max_drawdown_capitulacao_menos_euforia",
        "sharpe_simplificado_capitulacao", "sharpe_simplificado_euforia",
        "sortino_simplificado_capitulacao", "sortino_simplificado_euforia",
        "calmar_simplificado_capitulacao", "calmar_simplificado_euforia",
        "vencedor_volatilidade_anualizada", "vencedor_max_drawdown",
        "vencedor_sharpe_simplificado", "vencedor_sortino_simplificado", "vencedor_calmar_simplificado",
    ]
    metricas = ["volatilidade_anualizada_capitulacao", "max_drawdown_capitulacao", "sharpe_simplificado_capitulacao"]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_risco_controles_aleatorios(df):
    cols = [
        "frequencia", "estrategia_referencia", "nome_exibicao_comparador",
        "tipo_comparacao", "grupo_comparacao", "cenario_aleatorio",
        "n_replicas_disponiveis", "n_periodos_comparaveis_total",
        "pct_replicas_sharpe_real_superou_controle",
        "pct_replicas_sortino_real_superou_controle",
        "pct_replicas_calmar_real_superou_controle",
        "pct_replicas_volatilidade_real_menor_controle",
        "pct_replicas_drawdown_abs_real_menor_controle",
        "classificacao_robustez_retorno_acumulado",
        "classificacao_robustez_periodos",
    ]
    metricas = [
        "pct_replicas_sharpe_real_superou_controle",
        "pct_replicas_sortino_real_superou_controle",
        "pct_replicas_calmar_real_superou_controle",
        "pct_replicas_volatilidade_real_menor_controle",
        "pct_replicas_drawdown_abs_real_menor_controle",
    ]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_risco_controles_mensais(df):
    cols = [
        "frequencia", "estrategia_referencia", "nome_exibicao_comparador",
        "tipo_comparacao", "grupo_comparacao", "tipo_controle_mensal",
        "n_controles_mensais", "n_periodos_comparaveis_total",
        "pct_controles_sharpe_real_superou_controle",
        "pct_controles_sortino_real_superou_controle",
        "pct_controles_calmar_real_superou_controle",
        "pct_controles_volatilidade_real_menor_controle",
        "pct_controles_drawdown_abs_real_menor_controle",
        "classificacao_robustez_retorno_acumulado",
        "classificacao_robustez_periodos",
    ]
    metricas = [
        "pct_controles_sharpe_real_superou_controle",
        "pct_controles_sortino_real_superou_controle",
        "pct_controles_calmar_real_superou_controle",
        "pct_controles_volatilidade_real_menor_controle",
        "pct_controles_drawdown_abs_real_menor_controle",
    ]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_inferencia_scores(df):
    cols = [
        "estrategia_referencia", "frequencia", "grupo_comparacao", "label_grupo_comparacao",
        "tipo_comparacao", "n_evidencias", "n_origens_evidencia", "n_categorias_evidencia",
        "score_evidencia_total", "score_evidencia_medio",
        "n_evidencias_favoraveis", "n_evidencias_neutras", "n_evidencias_desfavoraveis",
        "classificacao_score", "classificacao_evidencia_final",
        "classificacao_vs_ibovespa", "classificacao_vs_cdi",
        "classificacao_vs_aleatorias", "classificacao_vs_mensais",
    ]
    metricas = ["score_evidencia_total", "score_evidencia_medio", "classificacao_evidencia_final"]
    def filtro(x):
        mask = mascara_metricas_validas(x, metricas)
        for col in ["grupo_comparacao", "label_grupo_comparacao", "tipo_comparacao", "visao_inferencia"]:
            if col in x.columns:
                mask = mask & ~coluna_contem(x, col, "outra_estrategia|outra estratégia|capitulacao_vs_euforia|capitulação_vs_euforia")
        return mask
    return tabela_bloco(df, cols, metricas, filtro=filtro)

def derivar_inferencia_testes(df):
    cols = [
        "estrategia_referencia", "frequencia", "grupo_comparacao", "label_grupo_comparacao",
        "tipo_comparacao", "n_testes_significativos_5pct",
        "p_valor_minimo_ajustado", "p_valor_teste_sinal", "p_valor_holm_teste_sinal",
        "p_valor_wilcoxon", "p_valor_holm_wilcoxon", "p_valor_permutacao",
        "p_valor_holm_permutacao", "status_teste_sinal", "status_teste_wilcoxon",
        "status_teste_permutacao", "media_excesso_periodico_media",
        "pct_periodos_superacao_media", "classificacao_evidencia_final",
    ]
    metricas = ["p_valor_minimo_ajustado", "p_valor_teste_sinal", "p_valor_wilcoxon", "p_valor_permutacao", "status_teste_sinal", "status_teste_wilcoxon", "status_teste_permutacao"]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_inferencia_bootstrap(df):
    cols = [
        "estrategia_referencia", "frequencia", "grupo_comparacao", "label_grupo_comparacao",
        "tipo_comparacao", "score_percentil_medio", "score_percentil_mediano",
        "prob_bootstrap_media_acima_zero", "prob_bootstrap_media_acima_zero_media",
        "n_confirmacoes_bootstrap", "classificacao_evidencia_final",
    ]
    metricas = ["score_percentil_medio", "score_percentil_mediano", "prob_bootstrap_media_acima_zero", "prob_bootstrap_media_acima_zero_media"]
    def filtro(x):
        mask = mascara_metricas_validas(x, metricas)
        for col in ["grupo_comparacao", "label_grupo_comparacao", "tipo_comparacao", "visao_inferencia"]:
            if col in x.columns:
                mask = mask & ~coluna_contem(x, col, "outra_estrategia|outra estratégia|capitulacao_vs_euforia|capitulação_vs_euforia")
        return mask
    return tabela_bloco(df, cols, metricas, filtro=filtro)

def preparar_robustez_unificada(df):
    if df is None or len(df) == 0:
        return None
    base = df.copy()
    base = coalescer_colunas(base, ["cenario_limiar_capitulacao", "cenario_limiar_euforia"], "cenario_limiar")
    base = coalescer_colunas(base, ["limiar_score_capitulacao", "limiar_score_euforia"], "limiar_score")
    return base

def derivar_robustez_limiares(df):
    base = preparar_robustez_unificada(df)
    cols = [
        "estrategia_referencia", "cenario_limiar", "tipo_cenario", "limiar_score",
        "n_sinais_brutos", "n_sinais_pos_cooldown", "n_sinais_rejeitados_cooldown",
        "pct_sinais_aceitos", "primeira_data_sinal", "ultima_data_sinal",
        "score_medio_sinais_aceitos", "score_mediano_sinais_aceitos",
        "intervalo_mediano_pregoes",
    ]
    metricas = ["n_sinais_brutos", "n_sinais_pos_cooldown", "pct_sinais_aceitos", "score_medio_sinais_aceitos"]
    return tabela_bloco(base, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_robustez_jaccard(df):
    base = preparar_robustez_unificada(df)
    cols = [
        "estrategia_referencia", "cenario_limiar", "n_sinais_cenario",
        "n_sinais_referencia", "n_sinais_comuns_com_referencia",
        "n_sinais_adicionados_vs_referencia", "n_sinais_removidos_vs_referencia",
        "pct_jaccard_vs_referencia",
    ]
    metricas = ["n_sinais_cenario", "n_sinais_referencia", "pct_jaccard_vs_referencia"]
    return tabela_bloco(base, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_robustez_controles(df):
    cols = [
        "frequencia", "estrategia_referencia", "cenario_aleatorio", "cenario_mensal",
        "tipo_cenario", "n_replicas_disponiveis", "n_sementes_aleatorias_distintas",
        "n_pares_comparacao", "n_periodos_comparaveis_total", "n_controles_mensais",
        "n_tipos_controle_mensal",
        "pct_replicas_retorno_acumulado_real_superou_controle",
        "pct_replicas_retorno_equivalente_real_superou_controle",
        "pct_replicas_sharpe_real_superou_controle",
        "pct_replicas_sortino_real_superou_controle",
        "pct_replicas_calmar_real_superou_controle",
        "pct_replicas_volatilidade_real_menor_controle",
        "pct_replicas_drawdown_abs_real_menor_controle",
        "pct_controles_retorno_acumulado_real_superou_controle",
        "pct_controles_retorno_anualizado_real_superou_controle",
        "pct_controles_sharpe_real_superou_controle",
        "pct_controles_sortino_real_superou_controle",
        "pct_controles_calmar_real_superou_controle",
        "pct_controles_volatilidade_real_menor_controle",
        "pct_controles_drawdown_abs_real_menor_controle",
        "classificacao_robustez_retorno_acumulado", "classificacao_robustez_periodos",
    ]
    metricas = [
        "pct_replicas_retorno_acumulado_real_superou_controle",
        "pct_replicas_sharpe_real_superou_controle",
        "pct_controles_retorno_acumulado_real_superou_controle",
        "pct_controles_sharpe_real_superou_controle",
        "classificacao_robustez_retorno_acumulado",
    ]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_top_empresas_compras(df):
    if df is None or len(df) == 0:
        return None
    tipo_col = primeira_coluna_existente(df, ["tipo_base_composicao"])
    base = df.copy()
    if tipo_col is not None:
        mask = serie_para_texto_sem_na(base[tipo_col]).str.contains("compr", case=False, na=False)
        if mask.any():
            base = base.loc[mask].copy()
    cols = [
        "estrategia_referencia", "nome_exibicao_estrategia", "grupo_html", "label_grupo_html",
        "nome_empresa", "issuer_code", "ticker", "valor_investido_total", "n_compras",
        "n_aportes", "peso_medio_quando_presente", "peso_maximo",
        "ranking_valor_estrategia", "ranking_frequencia_estrategia",
    ]
    metricas = ["valor_investido_total", "n_compras", "n_aportes"]
    return tabela_bloco(base, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_top_empresas_concentracao(df):
    if df is None or len(df) == 0:
        return None
    tipo_col = primeira_coluna_existente(df, ["tipo_base_composicao"])
    base = df.copy()
    if tipo_col is not None:
        mask = serie_para_texto_sem_na(base[tipo_col]).str.contains("concentr", case=False, na=False)
        if mask.any():
            base = base.loc[mask].copy()
    cols = [
        "estrategia_referencia", "nome_exibicao_estrategia", "grupo_html", "label_grupo_html",
        "nome_empresa", "issuer_code", "ticker", "valor_posicao", "valor_investido_total",
        "peso_medio_quando_presente", "peso_maximo", "pct_aportes_com_exposicao",
        "ranking_valor_estrategia", "ranking_frequencia_estrategia",
    ]
    metricas = ["valor_posicao", "valor_investido_total", "peso_medio_quando_presente", "peso_maximo", "pct_aportes_com_exposicao"]
    return tabela_bloco(base, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_top_setoriais_ranking(df):
    cols = [
        "frequencia", "ranking_frequencia_estrategia", "estrategia_referencia",
        "nome_exibicao_estrategia", "nivel_classificacao_html",
        "setor_html", "subsetor_html", "segmento_html",
        "rotulo_classificacao_hierarquica", "valor_investido_classificacao",
        "peso_classificacao", "n_tickers_classificacao", "n_empresas_classificacao",
        "valor_total_aporte_nivel",
    ]
    metricas = ["valor_investido_classificacao", "peso_classificacao", "n_tickers_classificacao", "n_empresas_classificacao"]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_timeline_eventos(df):
    cols = [
        "data", "data_sinal", "data_aporte", "ano", "ano_mes",
        "estrategia_referencia", "nome_exibicao_estrategia",
        "grupo_html", "label_grupo_html", "valor_aportado", "valor_aportado_mes",
        "n_aportes", "n_aportes_mes", "score_intensidade_sinal",
        "percentil_intensidade_sinal", "classe_intensidade_sinal",
    ]
    metricas = ["valor_aportado", "valor_aportado_mes", "n_aportes", "n_aportes_mes", "score_intensidade_sinal"]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas))

def derivar_auditoria_resumo(df):
    cols = [
        "etapa", "subetapa", "tema", "tipo_validacao", "status", "resultado",
        "flag_ok", "flag_alerta", "flag_erro", "n_linhas", "n_colunas",
        "arquivo", "tabela", "observacao", "mensagem", "erro",
    ]
    cols = colunas_existentes(df, cols)
    if not cols:
        # Fallback defensivo: evita expor 180+ colunas técnicas no dashboard.
        cols = [c for c in df.columns if any(t in normalizar_nome_coluna(c) for t in [
            "etapa", "subetapa", "status", "erro", "alerta", "ok", "linhas", "colunas", "arquivo", "tabela", "observacao"
        ])]
    metricas = [c for c in cols if any(t in normalizar_nome_coluna(c) for t in ["status", "erro", "alerta", "ok", "linhas", "colunas", "resultado"])]
    return tabela_bloco(df, cols, metricas, filtro=lambda x: mascara_metricas_validas(x, metricas), exigir_metricas=bool(metricas))

DF_CONTROLES_ALEATORIOS_PERIODO_HTML = derivar_controles_aleatorios_periodo(DF_CONTROLES)
DF_CONTROLES_MENSAIS_PERIODO_HTML = derivar_controles_mensais_periodo(DF_CONTROLES)
DF_CONTROLES_ALEATORIOS_ROBUSTEZ_HTML = derivar_controles_aleatorios_robustez(DF_CONTROLES)
DF_CONTROLES_MENSAIS_ROBUSTEZ_HTML = derivar_controles_mensais_robustez(DF_CONTROLES)
DF_CONTROLES_CLASSIFICACAO_HTML = derivar_controles_classificacao(DF_CONTROLES)
DF_CONTROLES_TESTES_HTML = derivar_controles_testes(DF_CONTROLES)
DF_CAP_EUF_DESCRITIVA_HTML = derivar_cap_euf_descritiva_resumo(DF_CAP_VS_EUF)
DF_CAP_EUF_METRICAS_LONGAS_HTML = derivar_cap_euf_metricas_longas(DF_CAP_VS_EUF)
DF_CAP_EUF_TESTES_HTML = derivar_cap_euf_testes(DF_CAP_VS_EUF)
DF_RISCO_CAP_EUF_LIMPA_HTML = derivar_risco_cap_euf(DF_RISCO_CAP_EUF)
DF_RISCO_CONTROLES_ALEATORIOS_HTML = derivar_risco_controles_aleatorios(DF_RISCO_CONTROLES)
DF_RISCO_CONTROLES_MENSAIS_HTML = derivar_risco_controles_mensais(DF_RISCO_CONTROLES)
DF_INFERENCIA_SCORES_HTML = derivar_inferencia_scores(DF_INFERENCIA)
DF_INFERENCIA_TESTES_HTML = derivar_inferencia_testes(DF_INFERENCIA)
DF_INFERENCIA_BOOTSTRAP_HTML = derivar_inferencia_bootstrap(DF_INFERENCIA)
DF_ROBUSTEZ_LIMIARES_HTML = derivar_robustez_limiares(DF_ROBUSTEZ)
DF_ROBUSTEZ_JACCARD_HTML = derivar_robustez_jaccard(DF_ROBUSTEZ)
DF_ROBUSTEZ_CONTROLES_HTML = derivar_robustez_controles(DF_ROBUSTEZ)
DF_TOP_EMPRESAS_COMPRAS_HTML = derivar_top_empresas_compras(DF_TOP_EMPRESAS)
DF_TOP_EMPRESAS_CONCENTRACAO_HTML = derivar_top_empresas_concentracao(DF_TOP_EMPRESAS)
DF_TOP_SETORIAIS_RANKING_HTML = derivar_top_setoriais_ranking(DF_TOP_SETORIAIS)
DF_TIMELINE_EVENTOS_HTML = derivar_timeline_eventos(DF_TIMELINE)
DF_AUDITORIA_RESUMO_HTML = derivar_auditoria_resumo(DF_AUDITORIA)

TABELAS_SPECS = [
    ("resumo", DF_RESUMO_EXECUTIVO, "resumo_executivo.html", "Resumo executivo", "Síntese dos principais indicadores finais do projeto."),
    ("resumo", DF_METRICAS_PRINCIPAIS, "metricas_principais.html", "Métricas principais das estratégias", "Retorno, risco e eficiência por estratégia e frequência."),
    ("resumo", DF_TABELA_GRAFICO_RETORNO_TOTAL, "grafico_retorno_total_tabela.html", "Tabela do gráfico — retorno acumulado", "Tabela executiva usada para a leitura de retorno acumulado por estratégia."),
    ("resumo", DF_TABELA_GRAFICO_EFICIENCIA, "grafico_eficiencia_tabela.html", "Tabela do gráfico — eficiência", "Tabela executiva com Sharpe, Sortino e Calmar."),

    ("performance", DF_BENCHMARKS, "comparacao_benchmarks.html", "Comparação com benchmarks", "Comparações finais contra Ibovespa e CDI."),
    ("performance", DF_CONTROLES_ALEATORIOS_PERIODO_HTML, "controles_aleatorios_periodo.html", "Controles aleatórios — período a período", "Excesso periódico e superação contra controles aleatórios."),
    ("performance", DF_CONTROLES_ALEATORIOS_ROBUSTEZ_HTML, "controles_aleatorios_robustez.html", "Controles aleatórios — robustez agregada", "Percentual de réplicas aleatórias superadas por métrica."),
    ("performance", DF_CONTROLES_MENSAIS_PERIODO_HTML, "controles_mensais_periodo.html", "Controles mensais — período a período", "Excesso periódico e superação contra controles mensais."),
    ("performance", DF_CONTROLES_MENSAIS_ROBUSTEZ_HTML, "controles_mensais_robustez.html", "Controles mensais — robustez agregada", "Percentual de controles mensais superados por métrica."),
    ("performance", DF_CONTROLES_CLASSIFICACAO_HTML, "controles_classificacao_empirica.html", "Controles — classificação empírica", "Classificações empíricas consolidadas contra controles."),
    ("performance", DF_CONTROLES_TESTES_HTML, "comparacao_controles_testes.html", "Comparação com controles — testes estatísticos", "Testes estatísticos aplicados às comparações com controles."),
    ("performance", DF_CAP_EUF_DESCRITIVA_HTML, "capitulacao_vs_euforia_descritiva.html", "Capitulação vs Euforia — resumo descritivo", "Comparação direta de retorno, risco e eficiência entre Capitulação e Euforia."),
    ("performance", DF_CAP_EUF_METRICAS_LONGAS_HTML, "capitulacao_vs_euforia_metricas.html", "Capitulação vs Euforia — métricas comparativas", "Métricas comparativas em formato longo."),
    ("performance", DF_CAP_EUF_TESTES_HTML, "capitulacao_vs_euforia_testes.html", "Capitulação vs Euforia — testes formais", "Testes formais e evidência estatística da comparação direta."),
    ("performance", DF_RETORNOS_ACUMULADOS, "retornos_acumulados.html", "Retornos acumulados", "Base consolidada dos retornos acumulados."),
    ("performance", DF_RETORNOS_PERIODICOS, "retornos_periodicos.html", "Retornos periódicos", "Retornos por frequência e período."),
    ("performance", DF_DESEMPENHO_BENCH, "desempenho_vs_benchmarks.html", "Desempenho vs benchmarks", "Comparação detalhada de performance contra benchmarks."),
    ("performance", DF_DESEMPENHO_CONTROLES, "desempenho_vs_controles.html", "Desempenho vs controles", "Comparação detalhada contra controles."),
    ("performance", DF_DESEMPENHO_CAP_EUF, "desempenho_capitulacao_vs_euforia.html", "Desempenho — Capitulação vs Euforia", "Base de desempenho da comparação direta."),
    ("performance", DF_DISTRIBUICAO_RETORNOS, "distribuicao_retornos.html", "Distribuição de retornos", "Estatísticas de distribuição dos retornos."),
    ("performance", DF_TABELA_GRAFICO_SUPERACAO_BENCH, "grafico_superacao_benchmarks_tabela.html", "Tabela do gráfico — superação de benchmarks", "Resumo usado nos gráficos de superação contra Ibovespa e CDI."),

    ("risco", DF_METRICAS_RISCO, "metricas_risco.html", "Métricas de risco", "Resumo de risco das estratégias."),
    ("risco", DF_METRICAS_RISCO_LONG, "metricas_risco_long.html", "Métricas de risco em formato longo", "Base longa de Sharpe, Sortino e Calmar."),
    ("risco", DF_EVENTOS_DRAWDOWN, "eventos_drawdown.html", "Eventos de drawdown", "Eventos relevantes de queda e recuperação."),
    ("risco", DF_RISCO_BENCH, "risco_vs_benchmarks.html", "Risco vs benchmarks", "Comparação de risco contra benchmarks."),
    ("risco", DF_RISCO_CONTROLES_ALEATORIOS_HTML, "risco_controles_aleatorios.html", "Risco vs controles aleatórios", "Robustez de risco contra réplicas aleatórias."),
    ("risco", DF_RISCO_CONTROLES_MENSAIS_HTML, "risco_controles_mensais.html", "Risco vs controles mensais", "Robustez de risco contra controles mensais."),
    ("risco", DF_RISCO_CAP_EUF_LIMPA_HTML, "risco_capitulacao_vs_euforia.html", "Risco — Capitulação vs Euforia", "Resumo de risco da comparação direta."),

    ("inferencia", DF_INFERENCIA_SCORES_HTML, "inferencia_scores.html", "Inferência — scores de evidência", "Scores e classificações de evidência por bloco comparativo."),
    ("inferencia", DF_INFERENCIA_TESTES_HTML, "inferencia_testes.html", "Inferência — testes estatísticos", "P-valores, status e estatísticas dos testes formais."),
    ("inferencia", DF_INFERENCIA_BOOTSTRAP_HTML, "inferencia_bootstrap_percentis.html", "Inferência — bootstrap e percentis", "Percentis empíricos e probabilidade bootstrap de excesso positivo."),
    ("inferencia", DF_ROBUSTEZ_LIMIARES_HTML, "robustez_limiares_sinais.html", "Robustez — sensibilidade dos limiares de sinais", "Cenários de limiar, quantidade de sinais e efeito do cooldown."),
    ("inferencia", DF_ROBUSTEZ_JACCARD_HTML, "robustez_similaridade_sinais.html", "Robustez — similaridade dos sinais", "Sobreposição dos sinais de cada cenário contra a referência oficial."),
    ("inferencia", DF_ROBUSTEZ_CONTROLES_HTML, "robustez_controles.html", "Robustez — controles e subamostras", "Robustez contra controles aleatórios, mensais e subamostras."),

    ("composicao", DF_SINAIS_APORTES, "sinais_aportes.html", "Sinais e aportes", "Rastreamento dos sinais que originaram aportes."),
    ("composicao", DF_APORTES, "aportes.html", "Aportes", "Base consolidada de aportes."),
    ("composicao", DF_COMPRAS_APORTE, "compras_por_aporte.html", "Compras por aporte", "Compras associadas a cada aporte."),
    ("composicao", DF_COMPRAS_TICKER, "compras_por_ticker.html", "Compras por ticker", "Compras agregadas por ativo."),
    ("composicao", DF_TOP_TICKERS, "top_tickers_comprados.html", "Top tickers comprados", "Ranking no nível do ticker negociado."),
    ("composicao", DF_POSICOES, "posicoes_historicas.html", "Posições históricas", "Base histórica de posições."),
    ("composicao", DF_TOP_POSICOES, "top_posicoes_historicas.html", "Top posições históricas", "Maiores posições históricas agregadas."),
    ("composicao", DF_RESUMO_POSICOES, "resumo_posicoes_historicas.html", "Resumo de posições históricas", "Resumo agregado das posições."),
    ("composicao", DF_CONCENTRACAO_EMPRESA, "concentracao_empresa.html", "Concentração por empresa", "Concentração histórica por empresa emissora."),
    ("composicao", DF_TOP_EMPRESAS_COMPRAS_HTML, "top_empresas_compradas.html", "Top empresas compradas", "Ranking de empresas por compras realizadas."),
    ("composicao", DF_TOP_EMPRESAS_CONCENTRACAO_HTML, "top_empresas_concentracao.html", "Top empresas por concentração", "Ranking de empresas por concentração e relevância nas posições."),
    ("composicao", DF_CONCENTRACAO_SETORIAL, "concentracao_setorial.html", "Concentração setorial", "Concentração por setor, subsetor e segmento."),
    ("composicao", DF_TOP_SETORIAIS_RANKING_HTML, "top_classificacoes_setoriais.html", "Top classificações setoriais", "Ranking setorial agregado."),
    ("composicao", DF_TIMELINE_EVENTOS_HTML, "timeline_sinais_aportes.html", "Timeline de sinais e aportes", "Linha do tempo dos sinais e aportes."),
    ("composicao", DF_AUDITORIA_RESUMO_HTML, "auditoria_consolidada.html", "Auditoria consolidada", "Resumo das auditorias finais para rastreabilidade."),
]

for viewer, df_tab, nome_html, titulo, subtitulo in TABELAS_SPECS:
    if df_tab is None or len(df_tab) == 0:
        continue

    df_tab_limpo = preparar_dataframe_tabela_html(df_tab.copy(), titulo=titulo)
    if df_tab_limpo is None or len(df_tab_limpo) == 0 or len(df_tab_limpo.columns) == 0:
        print(f"   - Tabela removida após higienização: {titulo}")
        continue

    caminho_html = salvar_tabela_html(df_tab_limpo, DIR_TABELAS / nome_html, titulo, subtitulo=subtitulo)
    if caminho_html is None:
        continue

    catalogo_tabelas.append({
        "viewer": viewer,
        "titulo": titulo,
        "subtitulo": subtitulo,
        "linhas": int(len(df_tab_limpo)),
        "colunas": int(len(df_tab_limpo.columns)),
        "caminho_relativo": caminho_relativo(caminho_html),
    })

print("OK")
print(f" - Tabelas HTML geradas...............: {format_int(len(catalogo_tabelas))}")
print(f" - Pasta de tabelas...................: {DIR_TABELAS}")


print("\n[6/9] Gerando gráficos HTML externos com filtros internos e foco analítico...")

catalogo_graficos = []

def classificar_categoria_estrategia(row):
    texto = " ".join([
        str(row.get("nome_exibicao_estrategia", "")),
        str(row.get("nome_estrategia_html", "")),
        str(row.get("estrategia_referencia", "")),
        str(row.get("familia_estrategia", "")),
        str(row.get("categoria_estrategia", "")),
        str(row.get("grupo_html", "")),
        str(row.get("label_grupo_html", "")),
        str(row.get("tipo_estrategia", "")),
    ]).lower()

    texto_norm = normalizar_nome_coluna(texto)

    if "cdi" in texto_norm:
        return "CDI-Only"

    if "ibov" in texto_norm or "ibovespa" in texto_norm:
        return "Ibovespa Buy and Hold"

    if "aleator" in texto_norm:
        return "Aleatórias"

    if "mensal" in texto_norm or "aportes_mensais" in texto_norm or "aportes_mensal" in texto_norm:
        return "Aportes Mensais"

    if "capitulacao" in texto_norm:
        return "Capitulação"

    if "euforia" in texto_norm:
        return "Euforia"

    return "Outras"

def limpar_nome_comparador(valor):
    texto = limpar_rotulo_estrategia(valor)
    texto_norm = normalizar_nome_coluna(texto)

    if not valor_textual_valido(texto):
        return ""
    if "ibov" in texto_norm or "ibovespa" in texto_norm:
        return "Ibovespa"
    if texto_norm == "cdi" or "cdi" in texto_norm:
        return "CDI"
    if "aleator" in texto_norm and "capitulacao" in texto_norm:
        return "Aleatórias — Capitulação"
    if "aleator" in texto_norm and "euforia" in texto_norm:
        return "Aleatórias — Euforia"
    if "mensal" in texto_norm and "capitulacao" in texto_norm:
        return "Mensais — Capitulação"
    if "mensal" in texto_norm and "euforia" in texto_norm:
        return "Mensais — Euforia"
    if "aleator" in texto_norm:
        return "Controles Aleatórios"
    if "mensal" in texto_norm:
        return "Controles Mensais"

    return texto


def escolher_coluna_exata(df, nomes):
    if df is None:
        return None

    mapa = {normalizar_nome_coluna(c): c for c in df.columns}
    for nome in nomes:
        chave = normalizar_nome_coluna(nome)
        if chave in mapa:
            return mapa[chave]
    return None

def escolher_coluna_estrategia_grafico(df):
    return escolher_coluna_exata(df, [
        "nome_exibicao_estrategia",
        "nome_estrategia_html",
        "nome_exibicao_estrategia_real",
        "estrategia_referencia",
        "estrategia_id",
        "chave_estrategia",
    ])

def escolher_coluna_frequencia_grafico(df):
    return escolher_coluna_exata(df, [
        "frequencia",
        "label_frequencia",
        "frequencia_base_conclusao",
    ])

def valor_filtro_valido(valor):
    if valor is None:
        return False

    try:
        if pd.isna(valor):
            return False
    except Exception:
        pass

    texto = str(valor).strip()
    texto_norm = normalizar_nome_coluna(texto)

    valores_invalidos = {
        "", "nan", "none", "nat", "na", "n_a", "n_d", "nao_informado", "nao_informada",
        "null", "pd_na", "pd_nat", "missing", "sem_informacao", "sem_informacoes"
    }

    return texto_norm not in valores_invalidos and texto not in ["<NA>", "<NaN>", "<NaT>"]


def normalizar_frequencia_grafico(valor):
    if not valor_filtro_valido(valor):
        return "Total"

    texto = str(valor).strip().lower()
    texto_norm = normalizar_nome_coluna(texto)

    if "diar" in texto_norm:
        return "Diária"
    if "mens" in texto_norm:
        return "Mensal"
    if "anu" in texto_norm:
        return "Anual"
    if "total" in texto_norm or "geral" in texto_norm or "consolid" in texto_norm:
        return "Total"

    return str(valor).strip()

def ordem_frequencia_grafico(valor):
    mapa = {"Diária": 1, "Mensal": 2, "Anual": 3, "Total": 4}
    return mapa.get(normalizar_frequencia_grafico(valor), 99)

def escolher_coluna_data_grafico(df, candidatos=None):
    if df is None or len(df) == 0:
        return None

    candidatos = candidatos or [
        "data",
        "data_periodo",
        "data_posicao",
        "data_sinal",
        "data_aporte",
        "ano_mes",
        "periodo_html",
        "ano",
    ]

    for nome in candidatos:
        col = escolher_coluna_exata(df, [nome])
        if col is not None:
            serie = pd.to_datetime(df[col], errors="coerce")
            if serie.notna().mean() >= 0.55:
                return col

    return None

def escolher_primeira_coluna_existente(df, candidatos, numerica=False):
    if df is None:
        return None

    for nome in candidatos:
        col = escolher_coluna_exata(df, [nome])
        if col is not None:
            if numerica and not pd.api.types.is_numeric_dtype(df[col]):
                continue
            return col

    for col in df.columns:
        nome_col = normalizar_nome_coluna(col)
        for nome in candidatos:
            if normalizar_nome_coluna(nome) in nome_col:
                if numerica and not pd.api.types.is_numeric_dtype(df[col]):
                    continue
                return col

    return None

def preparar_base_grafico_padrao(df, col_data=None, col_valor=None, col_estrategia=None, col_frequencia=None, col_categoria=None):
    if df is None or len(df) == 0:
        return pd.DataFrame()

    base = df.copy()

    if col_estrategia is None:
        col_estrategia = escolher_coluna_estrategia_grafico(base)

    if col_frequencia is None:
        col_frequencia = escolher_coluna_frequencia_grafico(base)

    if col_data is None:
        col_data = escolher_coluna_data_grafico(base)

    base["_estrategia_html"] = base.apply(melhor_rotulo_estrategia_row, axis=1)
    if col_estrategia is not None and col_estrategia in base.columns:
        fallback_estrategia = base[col_estrategia].apply(limpar_rotulo_estrategia)
        base["_estrategia_html"] = np.where(
            base["_estrategia_html"].astype(str).str.strip() != "",
            base["_estrategia_html"],
            fallback_estrategia,
        )

    if col_frequencia is not None and col_frequencia in base.columns:
        base["_frequencia_html"] = base[col_frequencia].apply(normalizar_frequencia_grafico)
    else:
        base["_frequencia_html"] = "Total"

    base["_categoria_html"] = base.apply(classificar_categoria_estrategia, axis=1)
    if col_categoria is not None and col_categoria in base.columns:
        fallback_categoria = base[col_categoria].apply(limpar_rotulo_estrategia)
        base["_categoria_html"] = np.where(
            base["_categoria_html"].astype(str).str.strip() != "Outras",
            base["_categoria_html"],
            fallback_categoria.replace("", "Outras"),
        )

    if col_data is not None and col_data in base.columns:
        base["_x_html"] = pd.to_datetime(base[col_data], errors="coerce")
    else:
        base["_x_html"] = np.nan

    if col_valor is not None and col_valor in base.columns:
        base["_valor_html"] = pd.to_numeric(base[col_valor], errors="coerce")
    else:
        base["_valor_html"] = np.nan

    base = base.loc[base["_estrategia_html"].notna()].copy()
    base = base.loc[base["_estrategia_html"].astype(str).str.strip() != ""].copy()
    base = base.loc[~base["_estrategia_html"].astype(str).str.lower().isin(["nan", "none", "nat", "não informado", "nao informado"])].copy()
    base["_categoria_html"] = base["_categoria_html"].replace("", "Outras").fillna("Outras")

    return base

def amostrar_series(base, grupo_cols, data_col="_x_html", max_pontos_por_serie=900):
    if base is None or len(base) == 0 or data_col not in base.columns:
        return pd.DataFrame()

    saidas = []
    for _, grupo in base.sort_values(data_col).groupby(grupo_cols, dropna=False):
        if len(grupo) <= max_pontos_por_serie:
            saidas.append(grupo)
        else:
            passo = int(np.ceil(len(grupo) / max_pontos_por_serie))
            saidas.append(grupo.iloc[::passo].copy())

    if not saidas:
        return pd.DataFrame()

    return pd.concat(saidas, ignore_index=True)

def serializar_registros(df, colunas):
    if df is None or len(df) == 0:
        return []

    base = df[colunas].copy()

    for col in base.columns:
        if pd.api.types.is_datetime64_any_dtype(base[col]):
            base[col] = base[col].dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_numeric_dtype(base[col]):
            base[col] = base[col].replace([np.inf, -np.inf], np.nan)
            base[col] = base[col].where(base[col].notna(), None)
        else:
            base[col] = base[col].where(base[col].notna(), None)
            base[col] = base[col].astype(object)

    registros = base.to_dict(orient="records")

    for row in registros:
        for k, v in list(row.items()):
            if pd.isna(v) if isinstance(v, (float, np.floating)) else False:
                row[k] = None

    return registros

def salvar_dashboard_plotly_interativo(
    registros,
    caminho_saida,
    titulo,
    subtitulo,
    tipo_grafico,
    metricas,
    viewer,
    catalogo,
    x_field="x",
    y_field="valor",
    label_field="estrategia",
    strategy_field="estrategia",
    category_field="categoria",
    frequency_field="frequencia",
    group_field=None,
    window_field=None,
    use_strategy=True,
    use_category=True,
    use_frequency=True,
    use_group=False,
    use_window=False,
    top_n=40,
    sort_asc=False,
    y_percent=False,
    initial_metric=None,
    aggregate_bars=True,
):
    caminho_saida = Path(caminho_saida)
    caminho_saida.parent.mkdir(parents=True, exist_ok=True)

    registros = registros or []

    metricas = metricas or [{"field": y_field, "label": "Valor"}]
    if initial_metric is None:
        initial_metric = metricas[0]["field"]

    categorias = sorted({str(r.get(category_field)).strip() for r in registros if valor_filtro_valido(r.get(category_field))})
    estrategias = sorted({str(r.get(strategy_field)).strip() for r in registros if valor_filtro_valido(r.get(strategy_field))})
    frequencias = sorted(
        {normalizar_frequencia_grafico(r.get(frequency_field)) for r in registros if valor_filtro_valido(r.get(frequency_field)) or normalizar_frequencia_grafico(r.get(frequency_field)) == "Total"},
        key=ordem_frequencia_grafico,
    )
    grupos = sorted({str(r.get(group_field)).strip() for r in registros if group_field and valor_filtro_valido(r.get(group_field))})
    janelas = sorted({str(r.get(window_field)).strip() for r in registros if window_field and valor_filtro_valido(r.get(window_field))})

    if len(frequencias) <= 1:
        use_frequency = False
    if len(categorias) <= 1:
        use_category = False
    if len(estrategias) <= 1:
        use_strategy = False
    if len(grupos) <= 1:
        use_group = False
    if len(janelas) <= 1:
        use_window = False

    payload = {
        "title": titulo,
        "subtitle": subtitulo,
        "chartType": tipo_grafico,
        "xField": x_field,
        "yField": y_field,
        "labelField": label_field,
        "strategyField": strategy_field,
        "categoryField": category_field,
        "frequencyField": frequency_field,
        "groupField": group_field,
        "windowField": window_field,
        "metrics": metricas,
        "initialMetric": initial_metric,
        "useStrategy": use_strategy,
        "useCategory": use_category,
        "useFrequency": use_frequency,
        "useGroup": use_group,
        "useWindow": use_window,
        "topN": top_n,
        "sortAsc": sort_asc,
        "yPercent": y_percent,
        "aggregateBars": aggregate_bars,
        "categories": categorias,
        "strategies": estrategias,
        "frequencies": frequencias,
        "groups": grupos,
        "windows": janelas,
        "windowByFrequency": {
            freq: sorted({str(r.get(window_field)).strip() for r in registros if window_field and normalizar_frequencia_grafico(r.get(frequency_field)) == str(freq) and valor_filtro_valido(r.get(window_field))})
            for freq in frequencias
        } if window_field else {},
        "data": registros,
    }

    html_doc = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{esc(titulo)}</title>
  <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
  <style>
    :root {{
      --ink: #0f172a;
      --muted: #475569;
      --line: #dbe3ee;
      --soft: #f8fafc;
      --soft-2: #f1f5f9;
      --brand: #172554;
      --brand-2: #1d4ed8;
    }}

    * {{ box-sizing: border-box; }}

    html, body {{
      margin: 0;
      padding: 0;
      background: #ffffff;
      color: var(--ink);
      font-family: Inter, Arial, Helvetica, sans-serif;
    }}

    body {{ padding: 14px; }}

    .chart-title {{
      font-size: 1.08rem;
      font-weight: 800;
      margin-bottom: 3px;
    }}

    .chart-subtitle {{
      color: var(--muted);
      font-size: 0.92rem;
      margin-bottom: 12px;
    }}

    .controls {{
      display: grid;
      grid-template-columns: repeat(4, minmax(210px, 1fr));
      gap: 10px;
      align-items: end;
      margin-bottom: 12px;
      padding: 12px;
      border: 1px solid var(--line);
      border-radius: 14px;
      background: var(--soft);
    }}

    .control-group {{
      min-width: 0;
    }}

    .control-group label {{
      display: block;
      font-size: 0.82rem;
      color: #334155;
      font-weight: 800;
      margin-bottom: 5px;
    }}

    select,
    .multi-button {{
      width: 100%;
      min-height: 38px;
      border: 1px solid #cbd5e1;
      border-radius: 10px;
      padding: 8px 10px;
      background: #ffffff;
      color: var(--ink);
      font-size: 0.92rem;
      text-align: left;
    }}

    .multi-wrap {{
      position: relative;
    }}

    .multi-button {{
      display: flex;
      justify-content: space-between;
      align-items: center;
      cursor: pointer;
    }}

    .multi-panel {{
      position: absolute;
      z-index: 50;
      top: calc(100% + 6px);
      left: 0;
      right: 0;
      max-height: 320px;
      overflow: auto;
      background: #ffffff;
      border: 1px solid #cbd5e1;
      border-radius: 12px;
      box-shadow: 0 18px 36px rgba(15, 23, 42, 0.14);
      padding: 10px;
      display: none;
    }}

    .multi-panel.open {{
      display: block;
    }}

    .multi-actions {{
      display: flex;
      gap: 6px;
      margin-bottom: 8px;
    }}

    .multi-actions button {{
      flex: 1;
      border: 1px solid #cbd5e1;
      background: #ffffff;
      border-radius: 8px;
      padding: 7px 8px;
      font-size: 0.78rem;
      font-weight: 800;
      cursor: pointer;
    }}

    .multi-option {{
      display: flex;
      gap: 8px;
      align-items: center;
      padding: 6px 4px;
      font-size: 0.88rem;
      border-bottom: 1px solid #f1f5f9;
    }}

    .multi-option:last-child {{
      border-bottom: 0;
    }}

    .chart-wrap {{
      border: 1px solid var(--line);
      border-radius: 14px;
      overflow: hidden;
      box-shadow: 0 8px 22px rgba(15, 23, 42, 0.06);
      background: #fff;
    }}

    #chart {{
      width: 100%;
      min-height: 640px;
    }}

    .empty {{
      padding: 24px;
      color: var(--muted);
      display: none;
    }}

    @media (max-width: 1050px) {{
      .controls {{ grid-template-columns: repeat(2, minmax(220px, 1fr)); }}
    }}

    @media (max-width: 680px) {{
      .controls {{ grid-template-columns: 1fr; }}
      #chart {{ min-height: 560px; }}
    }}
  </style>
</head>
<body>
  <div class="chart-title">{esc(titulo)}</div>
  <div class="chart-subtitle">{esc(subtitulo)}</div>

  <div class="controls">
    <div class="control-group" id="metricControl">
      <label>Métrica</label>
      <select id="metricSelect"></select>
    </div>

    <div class="control-group" id="frequencyControl">
      <label>Frequência</label>
      <select id="frequencySelect"></select>
    </div>

    <div class="control-group" id="windowControl">
      <label>Janela</label>
      <select id="windowSelect"></select>
    </div>

    <div class="control-group" id="categoryControl">
      <label>Categoria</label>
      <div id="categoryMulti" class="multi-wrap"></div>
    </div>

    <div class="control-group" id="strategyControl">
      <label>Estratégias</label>
      <div id="strategyMulti" class="multi-wrap"></div>
    </div>

    <div class="control-group" id="groupControl">
      <label>Comparador / grupo</label>
      <div id="groupMulti" class="multi-wrap"></div>
    </div>
  </div>

  <div class="chart-wrap">
    <div id="chart"></div>
    <div id="emptyState" class="empty">Nenhum dado disponível para os filtros selecionados.</div>
  </div>

  <script>
    const CONFIG = {json.dumps(payload, ensure_ascii=False)};

    const DATA = CONFIG.data || [];
    const METRICS = CONFIG.metrics || [];
    const chart = document.getElementById("chart");
    const emptyState = document.getElementById("emptyState");

    const metricControl = document.getElementById("metricControl");
    const frequencyControl = document.getElementById("frequencyControl");
    const windowControl = document.getElementById("windowControl");
    const categoryControl = document.getElementById("categoryControl");
    const strategyControl = document.getElementById("strategyControl");
    const groupControl = document.getElementById("groupControl");

    const metricSelect = document.getElementById("metricSelect");
    const frequencySelect = document.getElementById("frequencySelect");
    const windowSelect = document.getElementById("windowSelect");

    let selectedCategories = new Set(CONFIG.categories || []);
    let selectedStrategies = new Set(CONFIG.strategies || []);
    let selectedGroups = new Set(CONFIG.groups || []);

    function toNumber(v) {{
      if (v === null || v === undefined || v === "") return null;
      const n = Number(v);
      return Number.isFinite(n) ? n : null;
    }}

    function unique(arr) {{
      return Array.from(new Set(arr.filter(x => x !== null && x !== undefined && x !== "")));
    }}

    function makeOptions(select, items) {{
      select.innerHTML = "";
      items.forEach(item => {{
        const opt = document.createElement("option");
        opt.value = item.value || item;
        opt.textContent = item.label || item;
        select.appendChild(opt);
      }});
    }}

    function validFrequenciesForMetric(metric) {{
      if (!CONFIG.useFrequency) return CONFIG.frequencies || [];

      const allowed = new Set(CONFIG.frequencies || []);
      const freqs = unique(DATA
        .filter(row => {{
          if (CONFIG.useCategory && selectedCategories.size && !selectedCategories.has(String(row[CONFIG.categoryField]))) return false;
          if (CONFIG.useStrategy && selectedStrategies.size && !selectedStrategies.has(String(row[CONFIG.strategyField]))) return false;
          if (CONFIG.useGroup && selectedGroups.size && !selectedGroups.has(String(row[CONFIG.groupField]))) return false;
          return toNumber(row[metric]) !== null;
        }})
        .map(row => String(row[CONFIG.frequencyField] || ""))
        .filter(freq => allowed.has(freq)));

      return freqs;
    }}

    function updateFrequencyOptions() {{
      if (!CONFIG.useFrequency) return;

      const metric = activeMetric();
      const current = frequencySelect.value;
      const freqs = validFrequenciesForMetric(metric);

      makeOptions(frequencySelect, freqs);
      if (freqs.includes(current)) frequencySelect.value = current;
      else if (freqs.length) frequencySelect.value = freqs[0];

      updateWindowOptions();
    }}

    function updateWindowOptions() {{
      if (!CONFIG.useWindow) return;
      const freq = frequencySelect.value;
      let windows = CONFIG.windows || [];
      if (CONFIG.useFrequency && CONFIG.windowByFrequency && CONFIG.windowByFrequency[freq]) {{
        windows = CONFIG.windowByFrequency[freq];
      }}
      const current = windowSelect.value;
      makeOptions(windowSelect, windows);
      if (windows.includes(current)) windowSelect.value = current;
      else if (windows.length) windowSelect.value = windows[0];
    }}

    function metricDecimals(field) {{
      const label = metricLabel(field).toLowerCase();
      const f = String(field || "").toLowerCase();
      if (label.includes("número") || label.includes("aportes realizados") || f.startsWith("n_") || f.includes("quantidade") || f.includes("linhas")) return ".0f";
      if (label.includes("bootstrap") || f.includes("prob_bootstrap")) return ".2%";
      if (label.includes("score") || f.includes("score")) return ".4f";
      if (CONFIG.yPercent || label.includes("%") || label.includes("percentual") || label.includes("drawdown") || label.includes("volatilidade")) return ".2%";
      return ".4f";
    }}

    function buildMulti(rootId, items, selectedSet, labelAll, onChange) {{
      const root = document.getElementById(rootId);
      root.innerHTML = "";

      const button = document.createElement("button");
      button.type = "button";
      button.className = "multi-button";

      const label = document.createElement("span");
      const arrow = document.createElement("span");
      arrow.textContent = "▾";
      button.appendChild(label);
      button.appendChild(arrow);

      const panel = document.createElement("div");
      panel.className = "multi-panel";

      const actions = document.createElement("div");
      actions.className = "multi-actions";

      const allBtn = document.createElement("button");
      allBtn.type = "button";
      allBtn.textContent = "Selecionar tudo";

      const noneBtn = document.createElement("button");
      noneBtn.type = "button";
      noneBtn.textContent = "Deselecionar tudo";

      actions.appendChild(allBtn);
      actions.appendChild(noneBtn);
      panel.appendChild(actions);

      const updateLabel = () => {{
        if (selectedSet.size === 0) {{
          label.textContent = "Nenhum item";
        }} else if (selectedSet.size === items.length) {{
          label.textContent = labelAll;
        }} else {{
          label.textContent = `${{selectedSet.size}} selecionado(s)`;
        }}
      }};

      const renderChecks = () => {{
        panel.querySelectorAll(".multi-option").forEach(el => el.remove());

        items.forEach(item => {{
          const row = document.createElement("label");
          row.className = "multi-option";

          const checkbox = document.createElement("input");
          checkbox.type = "checkbox";
          checkbox.checked = selectedSet.has(item);
          checkbox.addEventListener("change", () => {{
            if (checkbox.checked) selectedSet.add(item);
            else selectedSet.delete(item);
            updateLabel();
            onChange();
          }});

          const txt = document.createElement("span");
          txt.textContent = item;

          row.appendChild(checkbox);
          row.appendChild(txt);
          panel.appendChild(row);
        }});
      }};

      allBtn.addEventListener("click", () => {{
        selectedSet.clear();
        items.forEach(item => selectedSet.add(item));
        renderChecks();
        updateLabel();
        onChange();
      }});

      noneBtn.addEventListener("click", () => {{
        selectedSet.clear();
        renderChecks();
        updateLabel();
        onChange();
      }});

      button.addEventListener("click", () => {{
        document.querySelectorAll(".multi-panel.open").forEach(p => {{
          if (p !== panel) p.classList.remove("open");
        }});
        panel.classList.toggle("open");
      }});

      document.addEventListener("click", (ev) => {{
        if (!root.contains(ev.target)) panel.classList.remove("open");
      }});

      root.appendChild(button);
      root.appendChild(panel);
      renderChecks();
      updateLabel();
    }}

    function metricLabel(field) {{
      const found = METRICS.find(m => m.field === field);
      return found ? found.label : field;
    }}

    function activeMetric() {{
      return metricSelect.value || CONFIG.initialMetric;
    }}

    function filteredData() {{
      const metric = activeMetric();
      const freq = frequencySelect.value;
      const windowValue = windowSelect.value;

      return DATA.filter(row => {{
        if (CONFIG.useFrequency && String(row[CONFIG.frequencyField]) !== String(freq)) return false;
        if (CONFIG.useWindow && String(row[CONFIG.windowField]) !== String(windowValue)) return false;
        if (CONFIG.useCategory && !selectedCategories.has(String(row[CONFIG.categoryField]))) return false;
        if (CONFIG.useStrategy && !selectedStrategies.has(String(row[CONFIG.strategyField]))) return false;
        if (CONFIG.useGroup && !selectedGroups.has(String(row[CONFIG.groupField]))) return false;

        const value = toNumber(row[metric]);
        return value !== null;
      }});
    }}

    function sortRows(rows, metric) {{
      const mult = CONFIG.sortAsc ? 1 : -1;
      return rows.slice().sort((a, b) => {{
        const av = toNumber(a[metric]);
        const bv = toNumber(b[metric]);
        if (av < bv) return -1 * mult;
        if (av > bv) return 1 * mult;
        return 0;
      }});
    }}

    function buildLine(rows, metric) {{
      const groups = unique(rows.map(r => r[CONFIG.strategyField])).sort();

      return groups.map(name => {{
        const rs = rows
          .filter(r => r[CONFIG.strategyField] === name)
          .sort((a, b) => String(a[CONFIG.xField]).localeCompare(String(b[CONFIG.xField])));

        return {{
          type: "scatter",
          mode: "lines",
          name: name,
          x: rs.map(r => r[CONFIG.xField]),
          y: rs.map(r => toNumber(r[metric])),
          hovertemplate: "%{{x}}<br>" + name + "<br>" + metricLabel(metric) + ": %{{y:" + metricDecimals(metric) + "}}<extra></extra>"
        }};
      }});
    }}

    function aggregateRowsByLabel(rows, metric) {{
      const grouped = new Map();

      rows.forEach(row => {{
        const label = String(row[CONFIG.labelField] || "").trim();
        const value = toNumber(row[metric]);

        if (!label || value === null) return;

        if (!grouped.has(label)) {{
          grouped.set(label, {{
            row: Object.assign({{}}, row),
            values: []
          }});
        }}

        grouped.get(label).values.push(value);
      }});

      const aggregated = [];
      grouped.forEach(item => {{
        const values = item.values.filter(v => v !== null && Number.isFinite(v));
        if (!values.length) return;

        const mean = values.reduce((acc, v) => acc + v, 0) / values.length;
        const row = Object.assign({{}}, item.row);
        row[metric] = mean;
        row._n_registros_agregados = values.length;
        aggregated.push(row);
      }});

      return aggregated;
    }}

    function buildBar(rows, metric) {{
      const aggregateBars = CONFIG.aggregateBars !== false;
      let rs = aggregateBars ? aggregateRowsByLabel(rows, metric) : rows.slice();
      rs = sortRows(rs, metric).slice(0, CONFIG.topN || 40);

      const hoverExtra = aggregateBars
        ? "<br>Registros consolidados: %{{customdata}}"
        : "";

      return [{{
        type: "bar",
        x: rs.map(r => r[CONFIG.labelField]),
        y: rs.map(r => toNumber(r[metric])),
        marker: {{ opacity: 1, line: {{ width: 0 }} }},
        hovertemplate: "%{{x}}<br>" + metricLabel(metric) + ": %{{y:" + metricDecimals(metric) + "}}" + hoverExtra + "<extra></extra>",
        customdata: rs.map(r => r._n_registros_agregados || 1)
      }}];
    }}

    function buildBox(rows, metric) {{
      const groups = unique(rows.map(r => r[CONFIG.strategyField])).sort();

      return groups.map(name => {{
        const vals = rows
          .filter(r => r[CONFIG.strategyField] === name)
          .map(r => toNumber(r[metric]))
          .filter(v => v !== null);

        return {{
          type: "box",
          name: name,
          y: vals,
          boxpoints: false,
          hovertemplate: name + "<br>" + metricLabel(metric) + ": %{{y:" + metricDecimals(metric) + "}}<extra></extra>"
        }};
      }});
    }}

    function render() {{
      const metric = activeMetric();
      const rows = filteredData();

      let traces = [];
      if (CONFIG.chartType === "line") traces = buildLine(rows, metric);
      else if (CONFIG.chartType === "bar") traces = buildBar(rows, metric);
      else if (CONFIG.chartType === "box") traces = buildBox(rows, metric);

      emptyState.style.display = traces.length === 0 || rows.length === 0 ? "block" : "none";

      const yAxis = {{
        title: {{ text: metricLabel(metric), standoff: 18 }},
        automargin: true,
        tickformat: (CONFIG.yPercent || metricDecimals(metric) === ".2%") ? ".1%" : undefined,
        showgrid: CONFIG.chartType === "bar" ? false : true,
        zeroline: true,
        zerolinecolor: "rgba(15, 23, 42, 0.35)",
        zerolinewidth: 1
      }};

      const layout = {{
        title: {{ text: CONFIG.title, x: 0.02, xanchor: "left", y: 0.98, yanchor: "top" }},
        margin: {{ l: 105, r: 40, t: 82, b: CONFIG.chartType === "bar" ? 180 : 112 }},
        height: 680,
        paper_bgcolor: "#ffffff",
        plot_bgcolor: "#ffffff",
        hovermode: CONFIG.chartType === "line" ? "x unified" : "closest",
        legend: {{ orientation: "v", x: 1.02, y: 1, tracegroupgap: 4 }},
        xaxis: {{
          title: CONFIG.chartType === "line" ? "Data" : "",
          automargin: true,
          tickangle: CONFIG.chartType === "bar" ? -32 : 0
        }},
        yaxis: yAxis,
        barmode: "group",
        bargap: 0.28,
        bargroupgap: 0.12
      }};

      Plotly.react(chart, traces, layout, {{
        responsive: true,
        displaylogo: false,
        modeBarButtonsToRemove: ["lasso2d", "select2d"]
      }});
    }}

    function init() {{
      if (METRICS.length <= 1) {{
        metricControl.style.display = "none";
      }} else {{
        makeOptions(metricSelect, METRICS.map(m => ({{ value: m.field, label: m.label }})));
        metricSelect.value = CONFIG.initialMetric;
        metricSelect.addEventListener("change", () => {{
          updateFrequencyOptions();
          render();
        }});
      }}

      if (CONFIG.useFrequency) {{
        updateFrequencyOptions();
        frequencySelect.addEventListener("change", () => {{
          updateWindowOptions();
          render();
        }});
      }} else {{
        frequencyControl.style.display = "none";
      }}

      if (CONFIG.useWindow) {{
        updateWindowOptions();
        windowSelect.addEventListener("change", render);
      }} else {{
        windowControl.style.display = "none";
      }}

      if (CONFIG.useCategory) {{
        buildMulti("categoryMulti", CONFIG.categories || [], selectedCategories, "Todas as categorias", () => {{ updateFrequencyOptions(); render(); }});
      }} else {{
        categoryControl.style.display = "none";
      }}

      if (CONFIG.useStrategy) {{
        buildMulti("strategyMulti", CONFIG.strategies || [], selectedStrategies, "Todas as estratégias", () => {{ updateFrequencyOptions(); render(); }});
      }} else {{
        strategyControl.style.display = "none";
      }}

      if (CONFIG.useGroup) {{
        buildMulti("groupMulti", CONFIG.groups || [], selectedGroups, "Todos os grupos", () => {{ updateFrequencyOptions(); render(); }});
      }} else {{
        groupControl.style.display = "none";
      }}

      render();
    }}

    init();
  </script>
</body>
</html>"""

    caminho_saida.write_text(html_doc, encoding="utf-8")
    validar_tamanho_arquivo_gerado(caminho_saida, titulo)

    catalogo.append({
        "viewer": viewer,
        "titulo": titulo,
        "subtitulo": subtitulo,
        "caminho_relativo": caminho_relativo(caminho_saida),
    })

    return caminho_saida

def gerar_linhas_por_estrategia(df, caminho_saida, titulo, subtitulo, valor_col, viewer, fixed_freq=None, use_frequency=True, use_category=True, use_strategy=True, window_col=None):
    if df is None or len(df) == 0 or valor_col is None or valor_col not in df.columns:
        return None

    col_data = escolher_coluna_data_grafico(df)
    if col_data is None:
        print(f"   - Aviso: gráfico não gerado ({titulo}): coluna de data não encontrada.")
        return None

    base = preparar_base_grafico_padrao(df, col_data=col_data, col_valor=valor_col)

    if fixed_freq is not None:
        base = base.loc[base["_frequencia_html"] == fixed_freq].copy()

    base = base.dropna(subset=["_x_html", "_valor_html"]).copy()
    if len(base) == 0:
        print(f"   - Aviso: gráfico não gerado ({titulo}): base vazia após filtros.")
        return None

    grupo_cols = ["_estrategia_html"]
    if window_col is not None and window_col in base.columns:
        grupo_cols.append(window_col)

    base = amostrar_series(base, grupo_cols=grupo_cols, data_col="_x_html", max_pontos_por_serie=950)

    registros = serializar_registros(
        base.assign(
            x=base["_x_html"],
            valor=base["_valor_html"],
            estrategia=base["_estrategia_html"],
            categoria=base["_categoria_html"],
            frequencia=base["_frequencia_html"],
            janela=base[window_col].astype(str) if window_col is not None and window_col in base.columns else "Total",
        ),
        ["x", "valor", "estrategia", "categoria", "frequencia", "janela"],
    )

    return salvar_dashboard_plotly_interativo(
        registros,
        DIR_GRAFICOS / caminho_saida,
        titulo,
        subtitulo,
        tipo_grafico="line",
        metricas=[{"field": "valor", "label": titulo}],
        viewer=viewer,
        catalogo=catalogo_graficos,
        use_strategy=use_strategy,
        use_category=use_category,
        use_frequency=use_frequency and fixed_freq is None,
        use_window=window_col is not None,
        window_field="janela",
        y_percent=("drawdown" in normalizar_nome_coluna(valor_col) or "volatilidade" in normalizar_nome_coluna(valor_col)),
    )

def gerar_barras_metricas(df, caminho_saida, titulo, subtitulo, metricas, viewer, usar_frequencia=True, usar_estrategia=True, usar_categoria=True, group_col=None, top_n=40, sort_asc=False, y_percent=False):
    if df is None or len(df) == 0:
        return None

    base = preparar_base_grafico_padrao(df)

    registros_base = []
    metricas_validas = []

    for label, col in metricas:
        if col in base.columns:
            serie = pd.to_numeric(base[col], errors="coerce")
            if serie.notna().sum() == 0:
                continue
            if float(serie.fillna(0).abs().sum()) == 0.0:
                continue
            base[col] = serie
            metricas_validas.append({"field": col, "label": label})

    if not metricas_validas:
        print(f"   - Aviso: gráfico não gerado ({titulo}): métricas numéricas úteis não encontradas.")
        return None

    if group_col is not None and group_col in base.columns:
        base["_grupo_html"] = base[group_col].apply(limpar_nome_comparador)
        base["_grupo_html"] = base["_grupo_html"].replace("", "Total")
    else:
        base["_grupo_html"] = "Total"

    for _, row in base.iterrows():
        estrategia = row.get("_estrategia_html", "")
        grupo = row.get("_grupo_html", "Total")
        if not valor_textual_valido(estrategia):
            continue

        label = estrategia
        if valor_textual_valido(grupo) and grupo != "Total":
            label = f"{estrategia} — {grupo}"

        item = {
            "estrategia": estrategia,
            "categoria": row.get("_categoria_html", "Outras"),
            "frequencia": row.get("_frequencia_html", "Total"),
            "grupo": grupo,
            "label": label,
        }

        tem_valor = False
        for m in metricas_validas:
            valor = pd.to_numeric(pd.Series([row.get(m["field"])]), errors="coerce").iloc[0]
            item[m["field"]] = None if pd.isna(valor) else float(valor)
            if pd.notna(valor):
                tem_valor = True

        if tem_valor:
            registros_base.append(item)

    if not registros_base:
        print(f"   - Aviso: gráfico não gerado ({titulo}): registros vazios após limpeza.")
        return None

    return salvar_dashboard_plotly_interativo(
        registros_base,
        DIR_GRAFICOS / caminho_saida,
        titulo,
        subtitulo,
        tipo_grafico="bar",
        metricas=metricas_validas,
        viewer=viewer,
        catalogo=catalogo_graficos,
        x_field="label",
        label_field="label",
        group_field="grupo",
        use_group=group_col is not None,
        use_strategy=usar_estrategia,
        use_category=usar_categoria,
        use_frequency=usar_frequencia,
        top_n=top_n,
        sort_asc=sort_asc,
        y_percent=y_percent,
    )

def gerar_boxplot_retornos(df, caminho_saida, titulo, subtitulo, viewer):
    if df is None or len(df) == 0:
        return None

    col_valor = escolher_primeira_coluna_existente(df, ["retorno_periodo", "retorno_percentual", "retorno"], numerica=True)
    if col_valor is None:
        print(f"   - Aviso: gráfico não gerado ({titulo}): coluna de retorno não encontrada.")
        return None

    base = preparar_base_grafico_padrao(df, col_valor=col_valor)
    base = base.dropna(subset=["_valor_html"]).copy()

    registros = serializar_registros(
        base.assign(
            valor=base["_valor_html"],
            estrategia=base["_estrategia_html"],
            categoria=base["_categoria_html"],
            frequencia=base["_frequencia_html"],
        ),
        ["valor", "estrategia", "categoria", "frequencia"],
    )

    return salvar_dashboard_plotly_interativo(
        registros,
        DIR_GRAFICOS / caminho_saida,
        titulo,
        subtitulo,
        tipo_grafico="box",
        metricas=[{"field": "valor", "label": "Retorno do período"}],
        viewer=viewer,
        catalogo=catalogo_graficos,
        use_strategy=True,
        use_category=True,
        use_frequency=True,
        y_percent=True,
    )

def gerar_capitulacao_euforia_wide(df, caminho_saida, titulo, subtitulo, metricas, viewer, y_percent=False):
    if df is None or len(df) == 0:
        return None

    col_freq = escolher_coluna_frequencia_grafico(df)
    base = df.copy()
    base["_frequencia_html"] = base[col_freq].apply(normalizar_frequencia_grafico) if col_freq in base.columns else "Total"

    registros = []
    metricas_payload = []

    for label, col_cap, col_euf in metricas:
        if col_cap in base.columns and col_euf in base.columns:
            campo = slugify(label)
            metricas_payload.append({"field": campo, "label": label})

            for _, row in base.iterrows():
                for familia, col_val in [("Capitulação", col_cap), ("Euforia", col_euf)]:
                    valor = pd.to_numeric(pd.Series([row.get(col_val)]), errors="coerce").iloc[0]
                    if pd.isna(valor):
                        continue

                    registros.append({
                        "label": familia,
                        "estrategia": familia,
                        "categoria": familia,
                        "frequencia": row["_frequencia_html"],
                        "grupo": familia,
                        campo: float(valor),
                    })

    if not registros or not metricas_payload:
        print(f"   - Aviso: gráfico não gerado ({titulo}): métricas não encontradas.")
        return None

    return salvar_dashboard_plotly_interativo(
        registros,
        DIR_GRAFICOS / caminho_saida,
        titulo,
        subtitulo,
        tipo_grafico="bar",
        metricas=metricas_payload,
        viewer=viewer,
        catalogo=catalogo_graficos,
        label_field="label",
        group_field="grupo",
        use_group=True,
        use_strategy=False,
        use_category=False,
        use_frequency=True,
        top_n=20,
        y_percent=y_percent,
    )

def gerar_superacao_cap_euf(df):
    if df is None or len(df) == 0:
        return None

    return gerar_capitulacao_euforia_wide(
        df,
        "capitulacao_euforia_superacao_direta.html",
        "Capitulação vs Euforia — frequência de superação direta",
        "Percentual de períodos em que Capitulação superou Euforia e vice-versa. Não compara a família contra ela mesma.",
        metricas=[
            ("Frequência de superação", "pct_periodos_capitulacao_superou_euforia", "pct_periodos_euforia_superou_capitulacao"),
        ],
        viewer="capitulacao_euforia",
        y_percent=True,
    )

def preparar_df_unico_por_estrategia(df, metrica_col):
    if df is None or len(df) == 0 or metrica_col not in df.columns:
        return pd.DataFrame()

    base = preparar_base_grafico_padrao(df)
    base = base.dropna(subset=[metrica_col]).copy()

    if len(base) == 0:
        return pd.DataFrame()

    if "_frequencia_html" in base.columns and "Diária" in set(base["_frequencia_html"]):
        base = base.loc[base["_frequencia_html"] == "Diária"].copy()

    base = base.sort_values(["_estrategia_html"]).drop_duplicates("_estrategia_html", keep="first").copy()

    return base

def gerar_retorno_acumulado_total():
    if DF_METRICAS_PRINCIPAIS is None or len(DF_METRICAS_PRINCIPAIS) == 0:
        return None

    col = escolher_primeira_coluna_existente(DF_METRICAS_PRINCIPAIS, ["retorno_acumulado_total", "retorno_acumulado"], numerica=True)
    if col is None:
        return None

    base = preparar_df_unico_por_estrategia(DF_METRICAS_PRINCIPAIS, col)
    if len(base) == 0:
        return None

    registros = serializar_registros(
        base.assign(
            label=base["_estrategia_html"],
            estrategia=base["_estrategia_html"],
            categoria=base["_categoria_html"],
            frequencia="Total",
            valor=pd.to_numeric(base[col], errors="coerce"),
        ),
        ["label", "estrategia", "categoria", "frequencia", "valor"],
    )

    return salvar_dashboard_plotly_interativo(
        registros,
        DIR_GRAFICOS / "performance_retorno_acumulado_total.html",
        "Retorno acumulado por estratégia",
        "Ranking do retorno acumulado no período completo. A frequência não é usada como filtro porque a métrica final é consolidada por estratégia.",
        tipo_grafico="bar",
        metricas=[{"field": "valor", "label": "Retorno acumulado"}],
        viewer="performance",
        catalogo=catalogo_graficos,
        label_field="label",
        use_frequency=False,
        use_strategy=True,
        use_category=True,
        top_n=40,
        y_percent=True,
    )

def gerar_max_drawdown_total():
    if DF_DRAWDOWN is None or len(DF_DRAWDOWN) == 0:
        return None

    col = escolher_primeira_coluna_existente(DF_DRAWDOWN, ["drawdown_abs_maximo", "drawdown_abs_percentual", "drawdown_abs", "drawdown"], numerica=True)
    if col is None:
        return None

    base = preparar_base_grafico_padrao(DF_DRAWDOWN, col_valor=col)
    base = base.dropna(subset=["_valor_html"]).copy()

    if len(base) == 0:
        return None

    # Se a coluna de drawdown vier negativa, usa o valor absoluto para ranking de severidade.
    base["_valor_dd"] = base["_valor_html"].astype(float).abs()
    agg = (
        base.groupby(["_estrategia_html", "_categoria_html"], as_index=False)
        .agg(valor=("_valor_dd", "max"))
        .sort_values("valor", ascending=False)
    )

    registros = serializar_registros(
        agg.assign(
            label=agg["_estrategia_html"],
            estrategia=agg["_estrategia_html"],
            categoria=agg["_categoria_html"],
            frequencia="Total",
        ),
        ["label", "estrategia", "categoria", "frequencia", "valor"],
    )

    return salvar_dashboard_plotly_interativo(
        registros,
        DIR_GRAFICOS / "risco_max_drawdown_total.html",
        "Máximo drawdown por estratégia",
        "Maior perda acumulada observada ao longo de todo o período, em valor absoluto.",
        tipo_grafico="bar",
        metricas=[{"field": "valor", "label": "Max drawdown"}],
        viewer="risco",
        catalogo=catalogo_graficos,
        label_field="label",
        use_frequency=False,
        use_strategy=True,
        use_category=True,
        top_n=40,
        y_percent=True,
    )

def gerar_timeline_sinais():
    # A timeline deve representar eventos efetivos de sinal/aporte, não um painel mensal contínuo
    # com controles sem aporte registrado. Por isso, prioriza a base de aportes e remove séries zeradas.
    fonte = DF_APORTES if DF_APORTES is not None and len(DF_APORTES) > 0 else DF_SINAIS_APORTES
    if fonte is None or len(fonte) == 0:
        return None

    col_data = escolher_primeira_coluna_existente(fonte, ["data_aporte", "data_sinal", "data"], numerica=False)
    if col_data is None:
        print("   - Aviso: timeline não gerada: coluna de data não encontrada.")
        return None

    base = preparar_base_grafico_padrao(fonte, col_data=col_data)
    base["_data_plot"] = pd.to_datetime(base["_x_html"], errors="coerce")
    base = base.dropna(subset=["_data_plot"]).copy()

    if len(base) == 0:
        return None

    col_valor_aporte = escolher_coluna_metrica_util(base, [
        "valor_aporte_total_realizado",
        "valor_total_aportado",
        "valor_aporte",
        "aporte_realizado",
        "valor_investido_total",
        "valor_investido",
    ])

    col_aporte_id = escolher_primeira_coluna_existente(base, ["chave_aporte", "id_aporte", "data_aporte", "data_sinal"], numerica=False)

    base["_valor_aporte_plot"] = pd.to_numeric(base[col_valor_aporte], errors="coerce") if col_valor_aporte is not None else np.nan
    if col_valor_aporte is not None:
        base = base.loc[base["_valor_aporte_plot"].fillna(0).abs() > 0].copy()

    if len(base) == 0:
        print("   - Aviso: timeline não gerada: não há eventos efetivos de aporte com valor útil.")
        return None

    base["_periodo_plot"] = base["_data_plot"].dt.to_period("M").dt.to_timestamp()

    if col_aporte_id is not None and col_aporte_id in base.columns:
        base["_id_aporte_plot"] = serie_para_texto_sem_na(base[col_aporte_id]).replace("", np.nan)
        agg_aportes = (
            base.dropna(subset=["_id_aporte_plot"])
            .groupby(["_periodo_plot", "_estrategia_html", "_categoria_html"], as_index=False)
            .agg(n_aportes_mes=("_id_aporte_plot", "nunique"))
        )
    else:
        agg_aportes = (
            base.groupby(["_periodo_plot", "_estrategia_html", "_categoria_html"], as_index=False)
            .size()
            .rename(columns={"size": "n_aportes_mes"})
        )

    if col_valor_aporte is not None:
        agg_valor = (
            base.groupby(["_periodo_plot", "_estrategia_html", "_categoria_html"], as_index=False)
            .agg(valor_aportado_mes=("_valor_aporte_plot", "sum"))
        )
        agg = agg_aportes.merge(agg_valor, on=["_periodo_plot", "_estrategia_html", "_categoria_html"], how="left")
    else:
        agg = agg_aportes.copy()
        agg["valor_aportado_mes"] = np.nan

    agg = agg.loc[agg["n_aportes_mes"].fillna(0) > 0].copy()
    if col_valor_aporte is not None:
        agg = agg.loc[agg["valor_aportado_mes"].fillna(0).abs() > 0].copy()

    if len(agg) == 0:
        print("   - Aviso: timeline não gerada: todas as séries ficaram zeradas após a limpeza.")
        return None

    # Visão 1: valor aportado no mês em linha, quando houver valor monetário válido.
    if col_valor_aporte is not None and agg["valor_aportado_mes"].fillna(0).abs().sum() > 0:
        registros_valor = serializar_registros(
            agg.assign(
                x=agg["_periodo_plot"],
                estrategia=agg["_estrategia_html"],
                categoria=agg["_categoria_html"],
                frequencia="Mensal",
            ),
            ["x", "estrategia", "categoria", "frequencia", "valor_aportado_mes"],
        )

        salvar_dashboard_plotly_interativo(
            registros_valor,
            DIR_GRAFICOS / "composicao_timeline_valor_aportado.html",
            "Timeline de valor aportado",
            "Série mensal de valor efetivamente aportado por estratégia. Séries sem aporte registrado foram removidas.",
            tipo_grafico="line",
            metricas=[{"field": "valor_aportado_mes", "label": "Valor aportado no mês"}],
            viewer="composicao",
            catalogo=catalogo_graficos,
            use_frequency=False,
            use_strategy=len(set(r["estrategia"] for r in registros_valor)) > 1,
            use_category=True,
            y_percent=False,
        )

    # Visão 2: número de aportes como heatmap. Evita linha horizontal em 1 e deixa claro quando houve evento.
    registros_heatmap = []
    for _, row in agg.iterrows():
        periodo_txt = pd.to_datetime(row["_periodo_plot"]).strftime("%Y-%m")
        n_aportes = int(row.get("n_aportes_mes", 0) or 0)
        valor_aportado = row.get("valor_aportado_mes", np.nan)
        texto = f"n={format_int(n_aportes)}"
        if pd.notna(valor_aportado):
            texto += f"<br>{formatar_moeda_curta(valor_aportado)}"
        hover = (
            f"Mês: {periodo_txt}<br>"
            f"Estratégia: {esc(row['_estrategia_html'])}<br>"
            f"N aportes: {format_int(n_aportes)}"
        )
        if pd.notna(valor_aportado):
            hover += f"<br>Valor aportado: {format_money(valor_aportado)}"

        registros_heatmap.append({
            "x": periodo_txt,
            "y": row["_estrategia_html"],
            "estrategia": row["_estrategia_html"],
            "categoria": row["_categoria_html"],
            "n_aportes_mes": float(n_aportes),
            "valor_aportado_mes": None if pd.isna(valor_aportado) else float(valor_aportado),
            "text": texto,
            "hover": hover,
        })

    salvar_heatmap_plotly_interativo(
        registros_heatmap,
        DIR_GRAFICOS / "composicao_aportes_mes_estrategia_heatmap.html",
        "Aportes por mês e estratégia",
        "Heatmap dos eventos efetivos de aporte. A cor mostra o número de aportes e o texto mostra a contagem e o valor aportado quando disponível.",
        metricas=[
            {"field": "n_aportes_mes", "label": "N aportes no mês"},
            {"field": "valor_aportado_mes", "label": "Valor aportado no mês"},
        ] if col_valor_aporte is not None else [
            {"field": "n_aportes_mes", "label": "N aportes no mês"},
        ],
        viewer="composicao",
        catalogo=catalogo_graficos,
        x_title="Mês",
        y_title="Estratégia",
        top_n=999,
        x_sort="text",
    )

    return True


    col_data = escolher_primeira_coluna_existente(fonte, ["data_aporte", "data_sinal", "data"], numerica=False)
    if col_data is None:
        print("   - Aviso: timeline não gerada: coluna de data não encontrada.")
        return None

    base = preparar_base_grafico_padrao(fonte, col_data=col_data)
    base["_data_plot"] = pd.to_datetime(base["_x_html"], errors="coerce")
    base = base.dropna(subset=["_data_plot"]).copy()

    if len(base) == 0:
        return None

    col_valor_aporte = escolher_coluna_metrica_util(base, [
        "valor_aporte_total_realizado",
        "valor_total_aportado",
        "valor_aporte",
        "aporte_realizado",
        "valor_investido_total",
        "valor_investido",
    ])

    col_aporte_id = escolher_primeira_coluna_existente(base, ["chave_aporte", "id_aporte", "data_aporte", "data_sinal"], numerica=False)

    base["_valor_aporte_plot"] = pd.to_numeric(base[col_valor_aporte], errors="coerce") if col_valor_aporte is not None else np.nan
    if col_valor_aporte is not None:
        base = base.loc[base["_valor_aporte_plot"].fillna(0).abs() > 0].copy()

    # Remove controles/séries sem aporte efetivo. Se a tabela não trouxer valor de aporte, mantém apenas eventos
    # com identificador/data de aporte válidos para contagem.
    if len(base) == 0:
        print("   - Aviso: timeline não gerada: não há eventos efetivos de aporte com valor útil.")
        return None

    base["_periodo_plot"] = base["_data_plot"].dt.to_period("M").dt.to_timestamp()

    if col_aporte_id is not None and col_aporte_id in base.columns:
        base["_id_aporte_plot"] = serie_para_texto_sem_na(base[col_aporte_id]).replace("", np.nan)
        agg_aportes = (
            base.dropna(subset=["_id_aporte_plot"])
            .groupby(["_periodo_plot", "_estrategia_html", "_categoria_html"], as_index=False)
            .agg(n_aportes_mes=("_id_aporte_plot", "nunique"))
        )
    else:
        agg_aportes = (
            base.groupby(["_periodo_plot", "_estrategia_html", "_categoria_html"], as_index=False)
            .size()
            .rename(columns={"size": "n_aportes_mes"})
        )

    agregacoes = {}
    metricas = [{"field": "n_aportes_mes", "label": "Aportes no mês"}]

    if col_valor_aporte is not None:
        agg_valor = (
            base.groupby(["_periodo_plot", "_estrategia_html", "_categoria_html"], as_index=False)
            .agg(valor_aportado_mes=("_valor_aporte_plot", "sum"))
        )
        agg = agg_aportes.merge(agg_valor, on=["_periodo_plot", "_estrategia_html", "_categoria_html"], how="left")
        if agg["valor_aportado_mes"].fillna(0).abs().sum() > 0:
            metricas.insert(0, {"field": "valor_aportado_mes", "label": "Valor aportado no mês"})
    else:
        agg = agg_aportes.copy()

    # Remove séries totalmente zeradas por segurança.
    metric_fields = [m["field"] for m in metricas]
    agg = agg.loc[agg[metric_fields].fillna(0).abs().sum(axis=1) > 0].copy()

    if len(agg) == 0:
        print("   - Aviso: timeline não gerada: todas as séries ficaram zeradas após a limpeza.")
        return None

    registros = serializar_registros(
        agg.assign(
            x=agg["_periodo_plot"],
            estrategia=agg["_estrategia_html"],
            categoria=agg["_categoria_html"],
            frequencia="Mensal",
        ),
        ["x", "estrategia", "categoria", "frequencia"] + metric_fields,
    )

    return salvar_dashboard_plotly_interativo(
        registros,
        DIR_GRAFICOS / "composicao_timeline_sinais_aportes.html",
        "Timeline de sinais e aportes",
        "Série mensal dos eventos efetivos de aporte. Séries sem aporte registrado foram removidas para evitar controles zerados no gráfico.",
        tipo_grafico="line",
        metricas=metricas,
        viewer="composicao",
        catalogo=catalogo_graficos,
        use_frequency=False,
        use_strategy=len(set(r["estrategia"] for r in registros)) > 1,
        use_category=True,
        y_percent=False,
    )


def escolher_coluna_label_util(df, candidatos):
    if df is None or len(df) == 0:
        return None

    for candidato in candidatos:
        col = escolher_primeira_coluna_existente(df, [candidato], numerica=False)
        if col is None or col not in df.columns:
            continue
        texto = serie_para_texto_sem_na(df[col]).str.strip()
        validos = texto.loc[~texto.str.lower().isin(["", "nan", "none", "nat", "<na>", "não informado", "nao informado"])]
        if len(validos) > 0 and validos.nunique() > 0:
            return col
    return None


def escolher_coluna_metrica_util(df, candidatos):
    if df is None or len(df) == 0:
        return None

    for candidato in candidatos:
        col = escolher_primeira_coluna_existente(df, [candidato], numerica=True)
        if col is None or col not in df.columns:
            continue
        serie = pd.to_numeric(df[col], errors="coerce")
        if serie.notna().sum() > 0 and float(serie.fillna(0).abs().sum()) != 0.0:
            return col
    return None


def gerar_top_bar_from_df(df, caminho_saida, titulo, subtitulo, label_candidates, metric_candidates, viewer, top_n=35, group_candidates=None, y_percent=False):
    if df is None or len(df) == 0:
        return None

    label_col = escolher_coluna_label_util(df, label_candidates)
    metric_col = escolher_coluna_metrica_util(df, metric_candidates)

    if label_col is None or metric_col is None:
        print(f"   - Aviso: gráfico não gerado ({titulo}): colunas úteis insuficientes.")
        return None

    base = preparar_base_grafico_padrao(df)
    base["_label_html"] = base[label_col].apply(limpar_rotulo_estrategia)
    base["_valor_plot"] = pd.to_numeric(base[metric_col], errors="coerce")
    base = base.dropna(subset=["_valor_plot"]).copy()
    base = base.loc[base["_valor_plot"].abs() > 0].copy()
    base = base.loc[base["_label_html"].astype(str).str.strip() != ""].copy()

    if len(base) == 0:
        print(f"   - Aviso: gráfico não gerado ({titulo}): não há valores positivos/úteis para plotar.")
        return None

    agg = (
        base.groupby(["_label_html", "_estrategia_html", "_categoria_html", "_frequencia_html"], as_index=False)
        .agg(valor=("_valor_plot", "max"))
        .sort_values("valor", ascending=False)
    )

    if agg["_estrategia_html"].nunique() > 1:
        agg["_label_plot"] = agg["_estrategia_html"].astype(str) + " — " + agg["_label_html"].astype(str)
    else:
        agg["_label_plot"] = agg["_label_html"].astype(str)

    registros = serializar_registros(
        agg.assign(
            label=agg["_label_plot"],
            estrategia=agg["_estrategia_html"],
            categoria=agg["_categoria_html"],
            frequencia=agg["_frequencia_html"],
            grupo="Total",
        ),
        ["label", "estrategia", "categoria", "frequencia", "grupo", "valor"],
    )

    return salvar_dashboard_plotly_interativo(
        registros,
        DIR_GRAFICOS / caminho_saida,
        titulo,
        subtitulo,
        tipo_grafico="bar",
        metricas=[{"field": "valor", "label": renomear_coluna_exibicao(metric_col)}],
        viewer=viewer,
        catalogo=catalogo_graficos,
        label_field="label",
        use_group=False,
        use_frequency=len(set(r["frequencia"] for r in registros)) > 1,
        use_strategy=True,
        use_category=True,
        top_n=top_n,
        y_percent=y_percent,
    )


def gerar_concentracao_setorial_por_nivel(nivel, caminho_saida, titulo, subtitulo):
    fonte = DF_TOP_SETORIAIS if DF_TOP_SETORIAIS is not None and len(DF_TOP_SETORIAIS) > 0 else DF_CONCENTRACAO_SETORIAL
    if fonte is None or len(fonte) == 0:
        return None

    base = fonte.copy()

    col_nivel = escolher_primeira_coluna_existente(base, ["nivel_classificacao_html", "nivel_classificacao"], numerica=False)
    if col_nivel is not None:
        alvo = normalizar_nome_coluna(nivel)
        filtro_nivel = base[col_nivel].astype(str).map(normalizar_nome_coluna).str.contains(alvo, na=False)
        if filtro_nivel.any():
            base = base.loc[filtro_nivel].copy()

    label_candidates = [
        f"{nivel}_html",
        f"{nivel}_hierarquico",
        nivel,
        "nome_classificacao",
        "rotulo_classificacao_hierarquica",
        "classificacao_hierarquica_html",
    ]

    return gerar_top_bar_from_df(
        base,
        caminho_saida,
        titulo,
        subtitulo,
        label_candidates=label_candidates,
        metric_candidates=[
            "valor_investido_total",
            "peso_medio_quando_presente",
            "peso_maximo",
            "pct_aportes_com_exposicao",
            "n_aportes_com_exposicao",
            "n_empresas_distintas",
            "n_tickers_distintos",
        ],
        viewer="composicao",
        top_n=35,
        group_candidates=None,
        y_percent=False,
    )


def gerar_metricas_compras_por_estrategia():
    fonte = DF_COMPRAS_TICKER if DF_COMPRAS_TICKER is not None and len(DF_COMPRAS_TICKER) > 0 else DF_COMPRAS_APORTE
    if fonte is None or len(fonte) == 0:
        return None

    base = preparar_base_grafico_padrao(fonte)
    if len(base) == 0:
        return None

    valor_col = escolher_coluna_metrica_util(base, [
        "valor_investido_total", "valor_investido_ticker", "valor_investido", "valor_compra", "valor_total_comprado"
    ])
    qtd_col = escolher_coluna_metrica_util(base, [
        "quantidade_total", "qtd_total", "quantidade", "qtd_acoes", "quantidade_comprada", "qtd_comprada"
    ])
    preco_col = escolher_coluna_metrica_util(base, [
        "preco_medio_compra", "preco_compra", "preco_medio", "preco_execucao", "preco"
    ])

    chave_aporte_col = escolher_primeira_coluna_existente(base, ["chave_aporte", "id_aporte", "data_aporte", "data_sinal"], numerica=False)
    empresa_col = escolher_coluna_label_util(base, ["issuer_code", "nome_empresa", "nome", "empresa"])
    setor_col = escolher_coluna_label_util(base, ["setor_html", "setor"])
    subsetor_col = escolher_coluna_label_util(base, ["subsetor_html", "subsetor"])
    segmento_col = escolher_coluna_label_util(base, ["segmento_html", "segmento"])

    rows = []
    for (estrategia, categoria), g in base.groupby(["_estrategia_html", "_categoria_html"], dropna=False):
        if not valor_textual_valido(estrategia):
            continue

        item = {
            "label": estrategia,
            "estrategia": estrategia,
            "categoria": categoria if valor_textual_valido(categoria) else "Outras",
            "frequencia": "Total",
            "grupo": "Total",
        }

        if preco_col is not None:
            item["preco_medio_compra"] = float(pd.to_numeric(g[preco_col], errors="coerce").mean())
        elif valor_col is not None and qtd_col is not None:
            valor_total = float(pd.to_numeric(g[valor_col], errors="coerce").fillna(0).sum())
            qtd_total = float(pd.to_numeric(g[qtd_col], errors="coerce").fillna(0).sum())
            item["preco_medio_compra"] = valor_total / qtd_total if qtd_total != 0 else None
        else:
            item["preco_medio_compra"] = None

        if chave_aporte_col is not None:
            item["n_aportes_compra"] = int(serie_para_texto_sem_na(g[chave_aporte_col]).replace("", np.nan).dropna().nunique())
        else:
            item["n_aportes_compra"] = int(len(g))

        item["n_empresas_compradas"] = int(serie_para_texto_sem_na(g[empresa_col]).replace("", np.nan).dropna().nunique()) if empresa_col is not None else None
        item["n_setores_comprados"] = int(serie_para_texto_sem_na(g[setor_col]).replace("", np.nan).dropna().nunique()) if setor_col is not None else None
        item["n_subsetores_comprados"] = int(serie_para_texto_sem_na(g[subsetor_col]).replace("", np.nan).dropna().nunique()) if subsetor_col is not None else None
        item["n_segmentos_comprados"] = int(serie_para_texto_sem_na(g[segmento_col]).replace("", np.nan).dropna().nunique()) if segmento_col is not None else None

        rows.append(item)

    metricas = []
    for field, label in [
        ("preco_medio_compra", "Preço médio de compra"),
        ("n_aportes_compra", "N aportes de compra"),
        ("n_empresas_compradas", "N empresas compradas"),
        ("n_setores_comprados", "N setores comprados"),
        ("n_subsetores_comprados", "N subsetores comprados"),
        ("n_segmentos_comprados", "N segmentos comprados"),
    ]:
        vals = [r.get(field) for r in rows if r.get(field) is not None]
        if vals and sum(abs(float(v)) for v in vals) != 0:
            metricas.append({"field": field, "label": label})

    if not rows or not metricas:
        print("   - Aviso: métricas de compras por estratégia não geradas por falta de colunas úteis.")
        return None

    return salvar_dashboard_plotly_interativo(
        rows,
        DIR_GRAFICOS / "composicao_metricas_compras_por_estrategia.html",
        "Compras por estratégia — preço médio e diversidade",
        "Preço médio de compra, número de aportes de compra e diversidade de empresas, setores, subsetores e segmentos comprados por estratégia.",
        tipo_grafico="bar",
        metricas=metricas,
        viewer="composicao",
        catalogo=catalogo_graficos,
        label_field="label",
        use_frequency=False,
        use_strategy=True,
        use_category=True,
        top_n=60,
        sort_asc=False,
        y_percent=False,
    )


def texto_limpo_generico(valor):
    if not valor_filtro_valido(valor):
        return ""
    texto = str(valor).strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto

def formatar_valor_curto(valor):
    if valor is None or pd.isna(valor):
        return ""
    try:
        v = float(valor)
    except Exception:
        return str(valor)

    sinal = "-" if v < 0 else ""
    v_abs = abs(v)

    if v_abs >= 1000000000:
        return f"{sinal}{v_abs / 1000000000:.1f} bi".replace(".", ",")
    if v_abs >= 1000000:
        return f"{sinal}{v_abs / 1000000:.1f} mi".replace(".", ",")
    if v_abs >= 1000:
        return f"{sinal}{v_abs / 1000:.1f} mil".replace(".", ",")
    if v_abs >= 10:
        return f"{sinal}{v_abs:.0f}".replace(".", ",")
    return f"{sinal}{v_abs:.2f}".replace(".", ",")

def formatar_moeda_curta(valor):
    texto = formatar_valor_curto(valor)
    return f"R$ {texto}" if texto else ""


def parte_hierarquia_valida(valor):
    if not valor_filtro_valido(valor):
        return ""
    texto = str(valor).strip()
    texto_norm = normalizar_nome_coluna(texto)
    if texto_norm in ["", "nan", "none", "nat", "na", "nao_informado", "nao_informada", "nao_se_aplica"]:
        return ""
    if texto_norm.startswith("nao_inform"):
        return ""
    return re.sub(r"\s+", " ", texto)


def montar_label_hierarquico(row, nivel):
    nivel_norm = normalizar_nome_coluna(nivel)

    def obter(candidatos):
        for col in candidatos:
            if col in row.index:
                texto = parte_hierarquia_valida(row.get(col))
                if texto:
                    return texto
        return ""

    def partes_de_hierarquia_pronta(candidatos):
        for col in candidatos:
            if col in row.index:
                bruto = parte_hierarquia_valida(row.get(col))
                if not bruto:
                    continue
                partes = []
                vistos = set()
                for parte in re.split(r"\s*>\s*", bruto):
                    parte = parte_hierarquia_valida(parte)
                    chave = normalizar_nome_coluna(parte)
                    if parte and chave not in vistos:
                        partes.append(parte)
                        vistos.add(chave)
                if partes:
                    return partes
        return []

    # Prioridade absoluta: colunas explícitas já presentes na tabela de concentração setorial.
    setor = obter(["setor_hierarquico", "setor_html", "setor"])
    subsetor = obter(["subsetor_hierarquico", "subsetor_html", "subsetor"])
    segmento = obter(["segmento_hierarquico", "segmento_html", "segmento"])

    # Fallback para a coluna hierárquica correta. Evita classificacao_hierarquica_html quando ela vier quebrada.
    partes_hierarquia = partes_de_hierarquia_pronta([
        "rotulo_classificacao_hierarquica",
        "rotulo_hierarquico",
        "hierarquia_setorial",
        "classificacao_hierarquica",
    ])

    if partes_hierarquia:
        if not setor and len(partes_hierarquia) >= 1:
            setor = partes_hierarquia[0]
        if not subsetor and len(partes_hierarquia) >= 2:
            subsetor = partes_hierarquia[1]
        if not segmento and len(partes_hierarquia) >= 3:
            segmento = partes_hierarquia[2]

    # Último fallback: usa nome_classificacao apenas para o nível específico.
    nome_classificacao = obter(["nome_classificacao", "classificacao", "classificacao_nome"])
    if nivel_norm == "setor" and not setor:
        setor = nome_classificacao
    elif nivel_norm == "subsetor" and not subsetor:
        subsetor = nome_classificacao
    elif nivel_norm == "segmento" and not segmento:
        segmento = nome_classificacao

    if nivel_norm == "setor":
        partes = [setor]
    elif nivel_norm == "subsetor":
        partes = [setor, subsetor]
    elif nivel_norm == "segmento":
        partes = [setor, subsetor, segmento]
    else:
        partes = [setor, subsetor, segmento]

    saida = []
    vistos = set()
    for parte in partes:
        parte = parte_hierarquia_valida(parte)
        chave = normalizar_nome_coluna(parte)
        if not parte or chave in vistos:
            continue
        vistos.add(chave)
        saida.append(parte)

    # Bloqueia explicitamente labels quebrados ou sem conteúdo real.
    label = " > ".join(saida)
    if not label:
        return ""

    label_limpo = re.sub(r"\s*>\s*", " > ", label).strip(" >")
    if not label_limpo or label_limpo.startswith(">") or label_limpo.endswith(">") or "> >" in label_limpo:
        return ""

    return label_limpo


def salvar_heatmap_plotly_interativo(
    registros,
    caminho_saida,
    titulo,
    subtitulo,
    metricas,
    viewer,
    catalogo,
    x_title="",
    y_title="Estratégia",
    initial_metric=None,
    top_n=30,
    x_sort="value",
):
    caminho_saida = Path(caminho_saida)
    caminho_saida.parent.mkdir(parents=True, exist_ok=True)

    registros = registros or []
    metricas = metricas or [{"field": "valor", "label": "Valor"}]
    if initial_metric is None:
        initial_metric = metricas[0]["field"]

    categorias = sorted({str(r.get("categoria")).strip() for r in registros if valor_filtro_valido(r.get("categoria"))})
    estrategias = sorted({str(r.get("estrategia")).strip() for r in registros if valor_filtro_valido(r.get("estrategia"))})

    payload = {
        "title": titulo,
        "subtitle": subtitulo,
        "metrics": metricas,
        "initialMetric": initial_metric,
        "categories": categorias,
        "strategies": estrategias,
        "topN": int(top_n),
        "xSort": x_sort,
        "xTitle": x_title,
        "yTitle": y_title,
        "data": registros,
    }

    html_doc = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>{esc(titulo)}</title>
  <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
  <style>
    :root {{
      --ink: #0f172a;
      --muted: #475569;
      --line: #dbe3ee;
      --soft: #f8fafc;
    }}

    * {{ box-sizing: border-box; }}

    html, body {{
      margin: 0;
      padding: 0;
      background: #ffffff;
      color: var(--ink);
      font-family: Inter, Arial, Helvetica, sans-serif;
    }}

    body {{ padding: 14px; }}

    .chart-title {{
      font-size: 1.08rem;
      font-weight: 800;
      margin-bottom: 3px;
    }}

    .chart-subtitle {{
      color: var(--muted);
      font-size: 0.92rem;
      margin-bottom: 12px;
    }}

    .controls {{
      display: grid;
      grid-template-columns: repeat(4, minmax(210px, 1fr));
      gap: 10px;
      align-items: end;
      margin-bottom: 12px;
      padding: 12px;
      border: 1px solid var(--line);
      border-radius: 14px;
      background: var(--soft);
    }}

    .control-group label {{
      display: block;
      font-size: 0.82rem;
      color: #334155;
      font-weight: 800;
      margin-bottom: 5px;
    }}

    select,
    .multi-button {{
      width: 100%;
      min-height: 38px;
      border: 1px solid #cbd5e1;
      border-radius: 10px;
      padding: 8px 10px;
      background: #ffffff;
      color: var(--ink);
      font-size: 0.92rem;
      text-align: left;
    }}

    .multi-wrap {{ position: relative; }}

    .multi-button {{
      display: flex;
      justify-content: space-between;
      align-items: center;
      cursor: pointer;
    }}

    .multi-panel {{
      position: absolute;
      z-index: 50;
      top: calc(100% + 6px);
      left: 0;
      right: 0;
      max-height: 320px;
      overflow: auto;
      background: #ffffff;
      border: 1px solid #cbd5e1;
      border-radius: 12px;
      box-shadow: 0 18px 36px rgba(15, 23, 42, 0.14);
      padding: 10px;
      display: none;
    }}

    .multi-panel.open {{ display: block; }}

    .multi-actions {{
      display: flex;
      gap: 6px;
      margin-bottom: 8px;
    }}

    .multi-actions button {{
      flex: 1;
      border: 1px solid #cbd5e1;
      background: #ffffff;
      border-radius: 8px;
      padding: 7px 8px;
      font-size: 0.78rem;
      font-weight: 800;
      cursor: pointer;
    }}

    .multi-option {{
      display: flex;
      gap: 8px;
      align-items: center;
      padding: 6px 4px;
      font-size: 0.88rem;
      border-bottom: 1px solid #f1f5f9;
    }}

    .chart-wrap {{
      border: 1px solid var(--line);
      border-radius: 14px;
      overflow: hidden;
      box-shadow: 0 8px 22px rgba(15, 23, 42, 0.06);
      background: #fff;
    }}

    #chart {{
      width: 100%;
      min-height: 680px;
    }}

    .empty {{
      padding: 24px;
      color: var(--muted);
      display: none;
    }}

    @media (max-width: 1050px) {{
      .controls {{ grid-template-columns: repeat(2, minmax(220px, 1fr)); }}
    }}

    @media (max-width: 680px) {{
      .controls {{ grid-template-columns: 1fr; }}
      #chart {{ min-height: 560px; }}
    }}
  </style>
</head>
<body>
  <div class="chart-title">{esc(titulo)}</div>
  <div class="chart-subtitle">{esc(subtitulo)}</div>

  <div class="controls">
    <div class="control-group" id="metricControl">
      <label>Métrica</label>
      <select id="metricSelect"></select>
    </div>
    <div class="control-group" id="categoryControl">
      <label>Categoria</label>
      <div id="categoryMulti" class="multi-wrap"></div>
    </div>
    <div class="control-group" id="strategyControl">
      <label>Estratégias</label>
      <div id="strategyMulti" class="multi-wrap"></div>
    </div>
    <div class="control-group" id="topNControl">
      <label>Top N colunas</label>
      <select id="topNSelect">
        <option value="10">10</option>
        <option value="20">20</option>
        <option value="30" selected>30</option>
        <option value="40">40</option>
        <option value="60">60</option>
      </select>
    </div>
  </div>

  <div class="chart-wrap">
    <div id="chart"></div>
    <div id="emptyState" class="empty">Nenhum dado disponível para os filtros selecionados.</div>
  </div>

  <script>
    const CONFIG = {json.dumps(payload, ensure_ascii=False)};
    const DATA = CONFIG.data || [];
    const METRICS = CONFIG.metrics || [];

    const chart = document.getElementById("chart");
    const emptyState = document.getElementById("emptyState");
    const metricSelect = document.getElementById("metricSelect");
    const topNSelect = document.getElementById("topNSelect");
    const metricControl = document.getElementById("metricControl");

    let selectedCategories = new Set(CONFIG.categories || []);
    let selectedStrategies = new Set(CONFIG.strategies || []);

    function toNumber(v) {{
      if (v === null || v === undefined || v === "") return null;
      const n = Number(v);
      return Number.isFinite(n) ? n : null;
    }}

    function unique(arr) {{
      return Array.from(new Set(arr.filter(x => x !== null && x !== undefined && x !== "")));
    }}

    function makeOptions(select, items) {{
      select.innerHTML = "";
      items.forEach(item => {{
        const opt = document.createElement("option");
        opt.value = item.value || item;
        opt.textContent = item.label || item;
        select.appendChild(opt);
      }});
    }}

    function buildMulti(rootId, items, selectedSet, labelAll, onChange) {{
      const root = document.getElementById(rootId);
      root.innerHTML = "";

      const button = document.createElement("button");
      button.type = "button";
      button.className = "multi-button";

      const label = document.createElement("span");
      const arrow = document.createElement("span");
      arrow.textContent = "▾";
      button.appendChild(label);
      button.appendChild(arrow);

      const panel = document.createElement("div");
      panel.className = "multi-panel";

      const actions = document.createElement("div");
      actions.className = "multi-actions";

      const allBtn = document.createElement("button");
      allBtn.type = "button";
      allBtn.textContent = "Selecionar tudo";

      const noneBtn = document.createElement("button");
      noneBtn.type = "button";
      noneBtn.textContent = "Deselecionar tudo";

      actions.appendChild(allBtn);
      actions.appendChild(noneBtn);
      panel.appendChild(actions);

      const updateLabel = () => {{
        if (selectedSet.size === 0) label.textContent = "Nenhum item";
        else if (selectedSet.size === items.length) label.textContent = labelAll;
        else label.textContent = `${{selectedSet.size}} selecionado(s)`;
      }};

      const renderChecks = () => {{
        panel.querySelectorAll(".multi-option").forEach(el => el.remove());
        items.forEach(item => {{
          const row = document.createElement("label");
          row.className = "multi-option";

          const checkbox = document.createElement("input");
          checkbox.type = "checkbox";
          checkbox.checked = selectedSet.has(item);
          checkbox.addEventListener("change", () => {{
            if (checkbox.checked) selectedSet.add(item);
            else selectedSet.delete(item);
            updateLabel();
            onChange();
          }});

          const txt = document.createElement("span");
          txt.textContent = item;

          row.appendChild(checkbox);
          row.appendChild(txt);
          panel.appendChild(row);
        }});
      }};

      allBtn.addEventListener("click", () => {{
        selectedSet.clear();
        items.forEach(item => selectedSet.add(item));
        renderChecks();
        updateLabel();
        onChange();
      }});

      noneBtn.addEventListener("click", () => {{
        selectedSet.clear();
        renderChecks();
        updateLabel();
        onChange();
      }});

      button.addEventListener("click", () => {{
        document.querySelectorAll(".multi-panel.open").forEach(p => {{
          if (p !== panel) p.classList.remove("open");
        }});
        panel.classList.toggle("open");
      }});

      document.addEventListener("click", (ev) => {{
        if (!root.contains(ev.target)) panel.classList.remove("open");
      }});

      root.appendChild(button);
      root.appendChild(panel);
      renderChecks();
      updateLabel();
    }}

    function metricLabel(field) {{
      const found = METRICS.find(m => m.field === field);
      return found ? found.label : field;
    }}

    function activeMetric() {{
      return metricSelect.value || CONFIG.initialMetric;
    }}

    function filteredRows(metric) {{
      return DATA.filter(row => {{
        if (selectedCategories.size && !selectedCategories.has(String(row.categoria))) return false;
        if (selectedStrategies.size && !selectedStrategies.has(String(row.estrategia))) return false;
        return toNumber(row[metric]) !== null;
      }});
    }}

    function selectedXValues(rows, metric) {{
      const totals = new Map();
      rows.forEach(row => {{
        const x = String(row.x || "");
        const value = toNumber(row[metric]);
        if (!x || value === null) return;
        totals.set(x, (totals.get(x) || 0) + Math.abs(value));
      }});

      let xs = Array.from(totals.keys());
      if (CONFIG.xSort === "numeric") {{
        xs.sort((a, b) => Number(a) - Number(b));
      }} else if (CONFIG.xSort === "text" || CONFIG.xSort === "date") {{
        xs.sort((a, b) => String(a).localeCompare(String(b)));
      }} else {{
        xs.sort((a, b) => (totals.get(b) || 0) - (totals.get(a) || 0));
        const topN = Number(topNSelect.value) || CONFIG.topN || 30;
        xs = xs.slice(0, topN);
      }}
      return xs;
    }}

    function render() {{
      const metric = activeMetric();
      const rows = filteredRows(metric);
      const xs = selectedXValues(rows, metric);
      const ys = unique(rows.map(r => String(r.y || ""))).sort();

      const z = ys.map(y => xs.map(x => {{
        const row = rows.find(r => String(r.y) === y && String(r.x) === x);
        return row ? toNumber(row[metric]) : null;
      }}));

      const text = ys.map(y => xs.map(x => {{
        const row = rows.find(r => String(r.y) === y && String(r.x) === x);
        return row ? (row.text || "") : "";
      }}));

      const custom = ys.map(y => xs.map(x => {{
        const row = rows.find(r => String(r.y) === y && String(r.x) === x);
        return row ? (row.hover || "") : "";
      }}));

      emptyState.style.display = rows.length === 0 || xs.length === 0 || ys.length === 0 ? "block" : "none";

      const trace = {{
        type: "heatmap",
        x: xs,
        y: ys,
        z: z,
        text: text,
        texttemplate: "%{{text}}",
        textfont: {{ size: 10 }},
        customdata: custom,
        colorscale: "Blues",
        colorbar: {{ title: metricLabel(metric) }},
        hovertemplate: "%{{customdata}}<extra></extra>",
        zsmooth: false
      }};

      const layout = {{
        title: {{ text: CONFIG.title, x: 0.02, xanchor: "left", y: 0.98, yanchor: "top" }},
        margin: {{ l: Math.max(285, 155 + Math.min(260, Math.max(...ys.map(y => String(y).length)) * 5)), r: 58, t: 82, b: 150 }},
        height: Math.max(650, Math.min(1150, 95 + ys.length * 48 + xs.length * 6)),
        paper_bgcolor: "#ffffff",
        plot_bgcolor: "#ffffff",
        xaxis: {{ title: {{ text: CONFIG.xTitle || "", standoff: 14 }}, automargin: true, tickangle: -45, tickfont: {{ size: xs.length > 18 ? 10 : 11 }} }},
        yaxis: {{ title: {{ text: CONFIG.yTitle || "", standoff: 30 }}, automargin: true, tickfont: {{ size: ys.length > 18 ? 10 : 11 }} }},
      }};

      Plotly.react(chart, [trace], layout, {{
        responsive: true,
        displaylogo: false,
        modeBarButtonsToRemove: ["lasso2d", "select2d"]
      }});
    }}

    function init() {{
      if (METRICS.length <= 1) {{
        metricControl.style.display = "none";
      }} else {{
        makeOptions(metricSelect, METRICS.map(m => ({{ value: m.field, label: m.label }})));
        metricSelect.value = CONFIG.initialMetric;
        metricSelect.addEventListener("change", render);
      }}

      if (CONFIG.xSort === "numeric") {{
        document.getElementById("topNControl").style.display = "none";
      }}

      buildMulti("categoryMulti", CONFIG.categories || [], selectedCategories, "Todas as categorias", render);
      buildMulti("strategyMulti", CONFIG.strategies || [], selectedStrategies, "Todas as estratégias", render);
      topNSelect.addEventListener("change", render);

      render();
    }}

    init();
  </script>
</body>
</html>"""

    caminho_saida.write_text(html_doc, encoding="utf-8")
    validar_tamanho_arquivo_gerado(caminho_saida, titulo)

    catalogo.append({
        "viewer": viewer,
        "titulo": titulo,
        "subtitulo": subtitulo,
        "caminho_relativo": caminho_relativo(caminho_saida),
    })

    return caminho_saida


def gerar_heatmap_top_tickers():
    fonte = DF_TOP_TICKERS if DF_TOP_TICKERS is not None and len(DF_TOP_TICKERS) > 0 else DF_COMPRAS_TICKER
    if fonte is None or len(fonte) == 0:
        return None

    base = preparar_base_grafico_padrao(fonte)
    col_ticker = escolher_coluna_label_util(base, ["ticker", "ativo", "codigo_negociacao"])
    valor_col = escolher_coluna_metrica_util(base, ["valor_investido_total", "valor_investido_ticker", "valor_investido", "valor_compra", "valor_total_comprado"])
    n_col = escolher_coluna_metrica_util(base, ["n_compras", "n_aportes_com_compra", "quantidade_compras", "quantidade_total", "quantidade"])

    if col_ticker is None or valor_col is None:
        print("   - Aviso: heatmap de top tickers não gerado por falta de ticker ou valor investido.")
        return None

    base["_ticker_heatmap"] = serie_para_texto_sem_na(base[col_ticker]).str.strip()
    base["_valor_heatmap"] = pd.to_numeric(base[valor_col], errors="coerce").fillna(0)
    if n_col is not None:
        base["_n_compras_heatmap"] = pd.to_numeric(base[n_col], errors="coerce").fillna(0)
    else:
        base["_n_compras_heatmap"] = 1

    base = base.loc[(base["_ticker_heatmap"] != "") & (base["_valor_heatmap"].abs() > 0)].copy()
    if len(base) == 0:
        return None

    agg = (
        base.groupby(["_estrategia_html", "_categoria_html", "_ticker_heatmap"], as_index=False)
        .agg(valor_investido=("_valor_heatmap", "sum"), n_compras=("_n_compras_heatmap", "sum"))
    )

    registros = []
    for _, row in agg.iterrows():
        texto = f"{formatar_moeda_curta(row['valor_investido'])}<br>n={format_int(row['n_compras'])}"
        hover = (
            f"Estratégia: {esc(row['_estrategia_html'])}<br>"
            f"Ticker: {esc(row['_ticker_heatmap'])}<br>"
            f"Valor investido: {format_money(row['valor_investido'])}<br>"
            f"N compras/aportes: {format_int(row['n_compras'])}"
        )
        registros.append({
            "x": row["_ticker_heatmap"],
            "y": row["_estrategia_html"],
            "estrategia": row["_estrategia_html"],
            "categoria": row["_categoria_html"],
            "valor_investido": float(row["valor_investido"]),
            "n_compras": float(row["n_compras"]),
            "text": texto,
            "hover": hover,
        })

    return salvar_heatmap_plotly_interativo(
        registros,
        DIR_GRAFICOS / "composicao_top_tickers_heatmap.html",
        "Top tickers comprados",
        "Heatmap por estratégia e ticker. A cor representa valor investido; o texto mostra valor investido e número de compras/aportes.",
        metricas=[
            {"field": "valor_investido", "label": "Valor investido"},
            {"field": "n_compras", "label": "N compras/aportes"},
        ],
        viewer="composicao",
        catalogo=catalogo_graficos,
        x_title="Ticker",
        y_title="Estratégia",
        top_n=30,
        x_sort="value",
    )


def gerar_heatmap_ticker_mais_comprado_por_ano():
    fonte = DF_COMPRAS_TICKER if DF_COMPRAS_TICKER is not None and len(DF_COMPRAS_TICKER) > 0 else DF_COMPRAS_APORTE
    if fonte is None or len(fonte) == 0:
        return None

    base = preparar_base_grafico_padrao(fonte)
    col_data = escolher_primeira_coluna_existente(base, ["data_aporte", "data_sinal", "data", "ano"], numerica=False)
    col_ticker = escolher_coluna_label_util(base, ["ticker", "ativo", "codigo_negociacao"])
    valor_col = escolher_coluna_metrica_util(base, ["valor_investido_total", "valor_investido_ticker", "valor_investido", "valor_compra", "valor_total_comprado"])
    n_col = escolher_coluna_metrica_util(base, ["n_compras", "n_aportes_com_compra", "quantidade_compras", "quantidade_total", "quantidade"])

    if col_data is None or col_ticker is None or valor_col is None:
        print("   - Aviso: heatmap de ticker mais comprado por ano não gerado por falta de data, ticker ou valor.")
        return None

    data = pd.to_datetime(base[col_data], errors="coerce")
    if data.notna().sum() == 0:
        base["_ano_heatmap"] = pd.to_numeric(base[col_data], errors="coerce")
    else:
        base["_ano_heatmap"] = data.dt.year

    base["_ticker_heatmap"] = serie_para_texto_sem_na(base[col_ticker]).str.strip()
    base["_valor_heatmap"] = pd.to_numeric(base[valor_col], errors="coerce").fillna(0)
    if n_col is not None:
        base["_n_compras_heatmap"] = pd.to_numeric(base[n_col], errors="coerce").fillna(0)
    else:
        base["_n_compras_heatmap"] = 1

    base = base.loc[
        (base["_ano_heatmap"].notna())
        & (base["_ticker_heatmap"] != "")
        & (base["_valor_heatmap"].abs() > 0)
    ].copy()

    if len(base) == 0:
        return None

    agg = (
        base.groupby(["_ano_heatmap", "_estrategia_html", "_categoria_html", "_ticker_heatmap"], as_index=False)
        .agg(valor_investido=("_valor_heatmap", "sum"), n_compras=("_n_compras_heatmap", "sum"))
        .sort_values(["_ano_heatmap", "_estrategia_html", "valor_investido"], ascending=[True, True, False])
    )

    idx = agg.groupby(["_ano_heatmap", "_estrategia_html"], dropna=False)["valor_investido"].idxmax()
    top = agg.loc[idx].copy()

    registros = []
    for _, row in top.iterrows():
        ano = str(int(row["_ano_heatmap"]))
        texto = f"{esc(row['_ticker_heatmap'])}<br>{formatar_moeda_curta(row['valor_investido'])}<br>n={format_int(row['n_compras'])}"
        hover = (
            f"Ano: {ano}<br>"
            f"Estratégia: {esc(row['_estrategia_html'])}<br>"
            f"Ticker: {esc(row['_ticker_heatmap'])}<br>"
            f"Valor investido: {format_money(row['valor_investido'])}<br>"
            f"N compras/aportes: {format_int(row['n_compras'])}"
        )
        registros.append({
            "x": ano,
            "y": row["_estrategia_html"],
            "estrategia": row["_estrategia_html"],
            "categoria": row["_categoria_html"],
            "valor_investido": float(row["valor_investido"]),
            "n_compras": float(row["n_compras"]),
            "text": texto,
            "hover": hover,
        })

    return salvar_heatmap_plotly_interativo(
        registros,
        DIR_GRAFICOS / "composicao_ticker_mais_comprado_por_ano_heatmap.html",
        "Ticker mais comprado por ano",
        "Heatmap ano × estratégia. O texto da célula mostra o ticker líder, valor investido e número de compras/aportes.",
        metricas=[
            {"field": "valor_investido", "label": "Valor investido"},
            {"field": "n_compras", "label": "N compras/aportes"},
        ],
        viewer="composicao",
        catalogo=catalogo_graficos,
        x_title="Ano",
        y_title="Estratégia",
        top_n=99,
        x_sort="numeric",
    )


def gerar_heatmap_concentracao(df, caminho_saida, titulo, subtitulo, label_candidates, viewer="composicao", top_n=30):
    if df is None or len(df) == 0:
        return None

    base = preparar_base_grafico_padrao(df)
    label_col = escolher_coluna_label_util(base, label_candidates)
    if label_col is None:
        print(f"   - Aviso: heatmap não gerado ({titulo}): coluna de classificação não encontrada.")
        return None

    valor_col = escolher_coluna_metrica_util(base, [
        "valor_investido_total",
        "valor_investido_empresa",
        "valor_investido_ticker",
        "valor_posicao",
        "valor_posicao_total",
        "valor_total",
    ])
    peso_medio_col = escolher_coluna_metrica_util(base, ["peso_medio_quando_presente", "peso_medio", "peso_posicao", "participacao_media"])
    peso_max_col = escolher_coluna_metrica_util(base, ["peso_maximo", "peso_max", "max_peso", "participacao_maxima"])
    pct_col = escolher_coluna_metrica_util(base, ["pct_aportes_com_exposicao", "participacao", "percentual", "pct_exposicao"])

    metric_cols = []
    if valor_col is not None:
        metric_cols.append(("valor_investido", "Valor investido", valor_col, "sum"))
    if peso_medio_col is not None:
        metric_cols.append(("peso_medio", "Peso médio", peso_medio_col, "mean"))
    if peso_max_col is not None:
        metric_cols.append(("peso_maximo", "Peso máximo", peso_max_col, "max"))
    if pct_col is not None and pct_col not in [peso_medio_col, peso_max_col]:
        metric_cols.append(("pct_exposicao", "% aportes com exposição", pct_col, "max"))

    if not metric_cols:
        print(f"   - Aviso: heatmap não gerado ({titulo}): métricas úteis não encontradas.")
        return None

    base["_classificacao_heatmap"] = base[label_col].apply(texto_limpo_generico)
    base = base.loc[base["_classificacao_heatmap"] != ""].copy()
    base = base.loc[~base["_classificacao_heatmap"].str.lower().isin(["não informado", "nao informado", "outros não informado"])].copy()

    if len(base) == 0:
        return None

    agg_dict = {}
    for field, _, col, agg_func in metric_cols:
        base[field] = pd.to_numeric(base[col], errors="coerce")
        if agg_func == "sum":
            agg_dict[field] = (field, "sum")
        elif agg_func == "mean":
            agg_dict[field] = (field, "mean")
        else:
            agg_dict[field] = (field, "max")

    agg = (
        base.groupby(["_estrategia_html", "_categoria_html", "_classificacao_heatmap"], as_index=False)
        .agg(**agg_dict)
    )

    metricas = []
    for field, label, _, _ in metric_cols:
        if field in agg.columns and pd.to_numeric(agg[field], errors="coerce").fillna(0).abs().sum() > 0:
            metricas.append({"field": field, "label": label})

    if not metricas:
        return None

    registros = []
    for _, row in agg.iterrows():
        valor = row.get("valor_investido", np.nan)
        peso = row.get("peso_medio", np.nan)
        texto_partes = []
        if pd.notna(valor):
            texto_partes.append(formatar_moeda_curta(valor))
        if pd.notna(peso):
            texto_partes.append(formatar_valor_curto(peso if abs(float(peso)) > 1.5 else float(peso) * 100) + "%")
        texto = "<br>".join(texto_partes[:2])

        hover = (
            f"Estratégia: {esc(row['_estrategia_html'])}<br>"
            f"Classificação: {esc(row['_classificacao_heatmap'])}<br>"
        )
        for met in metricas:
            v = row.get(met["field"], np.nan)
            if pd.notna(v):
                if "peso" in met["field"] or "pct" in met["field"]:
                    val_txt = formatar_valor_curto(v if abs(float(v)) > 1.5 else float(v) * 100) + "%"
                elif "valor" in met["field"]:
                    val_txt = format_money(v)
                else:
                    val_txt = format_num(v, 4)
                hover += f"{esc(met['label'])}: {val_txt}<br>"

        reg = {
            "x": row["_classificacao_heatmap"],
            "y": row["_estrategia_html"],
            "estrategia": row["_estrategia_html"],
            "categoria": row["_categoria_html"],
            "text": texto,
            "hover": hover,
        }
        for met in metricas:
            reg[met["field"]] = None if pd.isna(row.get(met["field"], np.nan)) else float(row.get(met["field"]))
        registros.append(reg)

    return salvar_heatmap_plotly_interativo(
        registros,
        DIR_GRAFICOS / caminho_saida,
        titulo,
        subtitulo,
        metricas=metricas,
        viewer=viewer,
        catalogo=catalogo_graficos,
        x_title=titulo.replace("Concentração por ", "").capitalize(),
        y_title="Estratégia",
        top_n=top_n,
        x_sort="value",
    )


def gerar_heatmaps_concentracao_setorial():
    fonte = None
    if DF_TOP_SETORIAIS is not None and len(DF_TOP_SETORIAIS) > 0:
        fonte = DF_TOP_SETORIAIS.copy()
    elif DF_CONCENTRACAO_SETORIAL is not None and len(DF_CONCENTRACAO_SETORIAL) > 0:
        fonte = DF_CONCENTRACAO_SETORIAL.copy()

    if fonte is None or len(fonte) == 0:
        print("   - Aviso: heatmaps setoriais não gerados: base de concentração setorial ausente.")
        return None

    def filtrar_nivel(nivel):
        base = fonte.copy()
        col_nivel = escolher_primeira_coluna_existente(base, ["nivel_classificacao_html", "nivel_classificacao"], numerica=False)
        if col_nivel is not None:
            alvo = normalizar_nome_coluna(nivel)
            nivel_norm = serie_para_texto_sem_na(base[col_nivel]).map(normalizar_nome_coluna)
            filtro = nivel_norm.eq(alvo)
            if not filtro.any():
                filtro = nivel_norm.str.contains(alvo, na=False)
            if filtro.any():
                base = base.loc[filtro].copy()

        base["_label_hierarquico_heatmap"] = base.apply(lambda row: montar_label_hierarquico(row, nivel), axis=1)
        base = base.loc[base["_label_hierarquico_heatmap"].apply(parte_hierarquia_valida).astype(str).str.strip() != ""].copy()
        base = base.loc[~base["_label_hierarquico_heatmap"].astype(str).str.contains(r"(^\s*>|>\s*>|>\s*$)", regex=True, na=False)].copy()

        if len(base) == 0:
            print(f"   - Aviso: heatmap de {nivel} não gerado: labels hierárquicos válidos não encontrados.")
        else:
            exemplos_invalidos = base["_label_hierarquico_heatmap"].astype(str).str.contains(r"(^\s*>|>\s*>|>\s*$)", regex=True, na=False).sum()
            if exemplos_invalidos > 0:
                raise ValueError(f"Labels hierárquicos inválidos persistiram em {nivel}.")
        return base

    gerar_heatmap_concentracao(
        filtrar_nivel("setor"),
        "composicao_concentracao_setor_heatmap.html",
        "Concentração por setor",
        "Heatmap estratégia × setor. A cor representa valor investido ou a métrica selecionada; o texto resume valor/peso quando disponível.",
        label_candidates=["_label_hierarquico_heatmap"],
        top_n=30,
    )

    gerar_heatmap_concentracao(
        filtrar_nivel("subsetor"),
        "composicao_concentracao_subsetor_heatmap.html",
        "Concentração por subsetor",
        "Heatmap estratégia × hierarquia setor > subsetor. A cor representa valor investido ou a métrica selecionada; o texto resume valor/peso quando disponível.",
        label_candidates=["_label_hierarquico_heatmap"],
        top_n=30,
    )

    gerar_heatmap_concentracao(
        filtrar_nivel("segmento"),
        "composicao_concentracao_segmento_heatmap.html",
        "Concentração por segmento",
        "Heatmap estratégia × hierarquia setor > subsetor > segmento. A cor representa valor investido ou a métrica selecionada; o texto resume valor/peso quando disponível.",
        label_candidates=["_label_hierarquico_heatmap"],
        top_n=30,
    )

# --------------------------------------------------------------------------------------------------
# Performance
# --------------------------------------------------------------------------------------------------

gerar_linhas_por_estrategia(
    DF_CURVAS_PATRIMONIAIS,
    "performance_curvas_patrimoniais_diarias.html",
    "Curvas patrimoniais por estratégia",
    "Evolução diária do patrimônio das estratégias. Use os filtros para focar em famílias específicas ou comparar conjuntos de estratégias.",
    valor_col="patrimonio_total" if DF_CURVAS_PATRIMONIAIS is not None and "patrimonio_total" in DF_CURVAS_PATRIMONIAIS.columns else escolher_primeira_coluna_existente(DF_CURVAS_PATRIMONIAIS, ["patrimonio", "valor"], numerica=True),
    viewer="performance",
    fixed_freq="Diária",
    use_frequency=False,
    use_category=True,
    use_strategy=True,
)

gerar_retorno_acumulado_total()

gerar_barras_metricas(
    DF_METRICAS_PRINCIPAIS,
    "performance_eficiencia_estrategia.html",
    "Eficiência por estratégia",
    "Compara Sharpe, Sortino e Calmar com seleção de métrica e frequência.",
    metricas=[
        ("Sharpe", "sharpe_anualizado"),
        ("Sortino", "sortino_anualizado"),
        ("Calmar", "calmar"),
    ],
    viewer="performance",
    usar_frequencia=True,
    usar_estrategia=True,
    usar_categoria=True,
    top_n=40,
    sort_asc=False,
)

gerar_boxplot_retornos(
    DF_RETORNOS_PERIODICOS,
    "performance_distribuicao_retornos.html",
    "Distribuição de retornos por estratégia",
    "Boxplots dos retornos por estratégia, com filtro de frequência e seleção de estratégias.",
    viewer="performance",
)

gerar_barras_metricas(
    DF_BENCHMARKS,
    "performance_excesso_medio_benchmarks.html",
    "Excesso médio versus benchmarks",
    "Excesso médio periódico das estratégias contra Ibovespa e CDI.",
    metricas=[
        ("Excesso médio", "media_excesso_periodico"),
        ("Excesso mediano", "mediana_excesso_periodico"),
        ("% períodos de superação", "pct_periodos_real_superou_comparador"),
    ],
    viewer="performance",
    usar_frequencia=True,
    usar_estrategia=True,
    usar_categoria=True,
    group_col=escolher_primeira_coluna_existente(DF_BENCHMARKS, ["nome_exibicao_comparador", "benchmark", "chave_comparador"], numerica=False),
    top_n=50,
    sort_asc=False,
)

gerar_barras_metricas(
    DF_CONTROLES,
    "performance_excesso_medio_controles.html",
    "Excesso médio contra controles mensais e aleatórios",
    "Excesso médio das estratégias reais contra controles aleatórios e aportes mensais.",
    metricas=[
        ("Excesso médio", "media_excesso_periodico"),
        ("Excesso médio das médias", "media_excesso_periodico_media_pares"),
        ("% períodos de superação", "pct_periodos_real_superou_comparador"),
        ("% réplicas com excesso positivo", "pct_replicas_media_excesso_positiva"),
    ],
    viewer="performance",
    usar_frequencia=True,
    usar_estrategia=True,
    usar_categoria=True,
    group_col=escolher_primeira_coluna_existente(DF_CONTROLES, ["nome_exibicao_comparador", "grupo_comparacao", "tipo_comparacao", "tipo_controle_mensal", "cenario_aleatorio"], numerica=False),
    top_n=60,
    sort_asc=False,
)

# --------------------------------------------------------------------------------------------------
# Risco
# --------------------------------------------------------------------------------------------------

gerar_linhas_por_estrategia(
    DF_DRAWDOWN,
    "risco_curvas_drawdown.html",
    "Curvas de drawdown por estratégia",
    "Séries de drawdown das estratégias, com controles aleatórios de Capitulação e Euforia tratados como uma categoria própria.",
    valor_col=escolher_primeira_coluna_existente(DF_DRAWDOWN, ["drawdown", "drawdown_percentual", "drawdown_abs_percentual"], numerica=True),
    viewer="risco",
    fixed_freq=None,
    use_frequency=False,
    use_category=True,
    use_strategy=True,
)

gerar_linhas_por_estrategia(
    DF_VOLATILIDADE_ROLLING,
    "risco_volatilidade_rolling.html",
    "Volatilidade rolling por estratégia",
    "Curvas de volatilidade móvel das estratégias, com filtros de frequência, janela e estratégia.",
    valor_col=escolher_primeira_coluna_existente(DF_VOLATILIDADE_ROLLING, ["volatilidade_anualizada_rolling", "volatilidade_periodica_rolling", "volatilidade"], numerica=True),
    viewer="risco",
    fixed_freq=None,
    use_frequency=True,
    use_category=True,
    use_strategy=True,
    window_col=escolher_primeira_coluna_existente(DF_VOLATILIDADE_ROLLING, ["label_janela_volatilidade", "janela_volatilidade"], numerica=False),
)

gerar_barras_metricas(
    DF_METRICAS_RISCO if DF_METRICAS_RISCO is not None else DF_METRICAS_PRINCIPAIS,
    "risco_ajustado_metricas.html",
    "Risco ajustado: Sharpe, Sortino e Calmar",
    "Métricas de eficiência por estratégia, com seleção de métrica e frequência.",
    metricas=[
        ("Sharpe", "sharpe_anualizado"),
        ("Sortino", "sortino_anualizado"),
        ("Calmar", "calmar"),
    ],
    viewer="risco",
    usar_frequencia=True,
    usar_estrategia=True,
    usar_categoria=True,
    top_n=40,
    sort_asc=False,
)

gerar_barras_metricas(
    preparar_df_unico_por_estrategia(DF_METRICAS_PRINCIPAIS, "retorno_acumulado_total") if DF_METRICAS_PRINCIPAIS is not None and "retorno_acumulado_total" in DF_METRICAS_PRINCIPAIS.columns else DF_METRICAS_PRINCIPAIS,
    "risco_metricas_totais_periodo_completo.html",
    "Métricas totais no período completo",
    "Retorno acumulado, Sharpe, Sortino e Calmar calculados no período completo de cada estratégia.",
    metricas=[
        ("Retorno acumulado", "retorno_acumulado_total"),
        ("Sharpe", "sharpe_anualizado"),
        ("Sortino", "sortino_anualizado"),
        ("Calmar", "calmar"),
    ],
    viewer="risco",
    usar_frequencia=False,
    usar_estrategia=True,
    usar_categoria=True,
    top_n=40,
    sort_asc=False,
)

gerar_max_drawdown_total()

gerar_barras_metricas(
    DF_RISCO_CONTROLES,
    "risco_vs_controles_percentual_superacao.html",
    "Risco versus controles",
    "Percentual de controles superados pela estratégia real. Use o filtro de métrica para alternar entre Sharpe, Sortino, Calmar, volatilidade e drawdown.",
    metricas=[
        ("Aleatórias — Sharpe maior", "pct_replicas_sharpe_real_superou_controle"),
        ("Aleatórias — Sortino maior", "pct_replicas_sortino_real_superou_controle"),
        ("Aleatórias — Calmar maior", "pct_replicas_calmar_real_superou_controle"),
        ("Aleatórias — Volatilidade menor", "pct_replicas_volatilidade_real_menor_controle"),
        ("Aleatórias — Drawdown menor", "pct_replicas_drawdown_abs_real_menor_controle"),
        ("Mensais — Sharpe maior", "pct_controles_sharpe_real_superou_controle"),
        ("Mensais — Sortino maior", "pct_controles_sortino_real_superou_controle"),
        ("Mensais — Calmar maior", "pct_controles_calmar_real_superou_controle"),
        ("Mensais — Volatilidade menor", "pct_controles_volatilidade_real_menor_controle"),
        ("Mensais — Drawdown menor", "pct_controles_drawdown_abs_real_menor_controle"),
    ],
    viewer="risco",
    usar_frequencia=True,
    group_col=None,
    y_percent=True,
)

# --------------------------------------------------------------------------------------------------
# Capitulação vs Euforia
# --------------------------------------------------------------------------------------------------

gerar_capitulacao_euforia_wide(
    DF_CAP_VS_EUF,
    "capitulacao_euforia_retorno_risco.html",
    "Capitulação vs Euforia — retorno e risco",
    "Compara retorno acumulado, retorno equivalente, volatilidade e drawdown entre as duas famílias.",
    metricas=[
        ("Retorno acumulado", "retorno_acumulado_capitulacao", "retorno_acumulado_euforia"),
        ("Retorno equivalente", "retorno_anualizado_capitulacao", "retorno_anualizado_euforia"),
        ("Volatilidade", "volatilidade_anualizada_capitulacao", "volatilidade_anualizada_euforia"),
        ("Max drawdown", "max_drawdown_capitulacao", "max_drawdown_euforia"),
    ],
    viewer="capitulacao_euforia",
    y_percent=True,
)

gerar_capitulacao_euforia_wide(
    DF_CAP_VS_EUF,
    "capitulacao_euforia_eficiencia.html",
    "Capitulação vs Euforia — eficiência",
    "Compara Sharpe, Sortino e Calmar entre Capitulação e Euforia.",
    metricas=[
        ("Sharpe", "sharpe_simplificado_capitulacao", "sharpe_simplificado_euforia"),
        ("Sortino", "sortino_simplificado_capitulacao", "sortino_simplificado_euforia"),
        ("Calmar", "calmar_simplificado_capitulacao", "calmar_simplificado_euforia"),
    ],
    viewer="capitulacao_euforia",
    y_percent=False,
)

gerar_superacao_cap_euf(DF_CAP_VS_EUF)

gerar_barras_metricas(
    DF_BENCHMARKS,
    "capitulacao_euforia_superacao_benchmarks.html",
    "Capitulação e Euforia versus Ibovespa e CDI",
    "Percentual e quantidade de períodos em que as estratégias de Capitulação e Euforia superaram Ibovespa e CDI.",
    metricas=[
        ("% períodos de superação", "pct_periodos_real_superou_comparador"),
        ("Número de vitórias", "n_vitorias_real"),
        ("Número de derrotas", "n_derrotas_real"),
    ],
    viewer="capitulacao_euforia",
    usar_frequencia=True,
    usar_estrategia=True,
    usar_categoria=True,
    group_col=escolher_primeira_coluna_existente(DF_BENCHMARKS, ["nome_exibicao_comparador", "benchmark", "chave_comparador"], numerica=False),
    top_n=40,
    sort_asc=False,
)

gerar_barras_metricas(
    DF_CONTROLES,
    "capitulacao_euforia_superacao_controles.html",
    "Capitulação e Euforia versus controles",
    "Percentual de controles superados por cada estratégia real. O gráfico não é empilhado: cada métrica mostra uma leitura objetiva de superação.",
    metricas=[
        ("Aleatórias — retorno acumulado maior", "pct_replicas_retorno_acumulado_real_superou_controle"),
        ("Aleatórias — retorno anualizado maior", "pct_replicas_retorno_anualizado_real_superou_controle"),
        ("Aleatórias — Sharpe maior", "pct_replicas_sharpe_real_superou_controle"),
        ("Aleatórias — Sortino maior", "pct_replicas_sortino_real_superou_controle"),
        ("Aleatórias — Calmar maior", "pct_replicas_calmar_real_superou_controle"),
        ("Mensais — retorno acumulado maior", "pct_controles_retorno_acumulado_real_superou_controle"),
        ("Mensais — retorno anualizado maior", "pct_controles_retorno_anualizado_real_superou_controle"),
        ("Mensais — Sharpe maior", "pct_controles_sharpe_real_superou_controle"),
        ("Mensais — Sortino maior", "pct_controles_sortino_real_superou_controle"),
        ("Mensais — Calmar maior", "pct_controles_calmar_real_superou_controle"),
    ],
    viewer="capitulacao_euforia",
    usar_frequencia=True,
    group_col=None,
    y_percent=True,
)


def classificar_grupo_evidencia(row):
    textos = []
    for col in [
        "visao_inferencia",
        "grupo_comparacao",
        "label_grupo_comparacao",
        "tipo_comparacao",
        "nome_exibicao_comparador",
        "nome_tabela_origem",
        "tema_tabela_html",
        "classificacao_vs_ibovespa",
        "classificacao_vs_cdi",
        "classificacao_vs_aleatorias",
        "classificacao_vs_mensais",
    ]:
        if col in row.index and valor_textual_valido(row.get(col)):
            textos.append(str(row.get(col)))

    texto = " ".join(textos)
    texto_norm = normalizar_nome_coluna(texto)

    if "outra_estrategia" in texto_norm or "outra" in texto_norm:
        return ""

    if "ibov" in texto_norm or "ibovespa" in texto_norm:
        return "vs Ibovespa"

    if "cdi" in texto_norm:
        return "vs CDI"

    if "aleator" in texto_norm or "replica" in texto_norm:
        return "vs aleatórias"

    if "mensal" in texto_norm:
        return "vs mensais"

    return ""


def adicionar_metricas_wide_por_coluna(df, metricas, aceitar_total=True):
    if df is None or len(df) == 0:
        return pd.DataFrame(), []

    base = preparar_base_grafico_padrao(df).copy()
    base["_frequencia_html"] = base["_frequencia_html"].apply(normalizar_frequencia_grafico)

    if not aceitar_total:
        base = base.loc[base["_frequencia_html"] != "Total"].copy()

    chaves = ["_estrategia_html", "_categoria_html", "_frequencia_html"]
    registros = {}

    metricas_validas = []

    for label, candidatos in metricas:
        col = escolher_primeira_coluna_existente(base, candidatos, numerica=True)
        if col is None:
            continue

        temp = base[chaves + [col]].copy()
        temp[col] = pd.to_numeric(temp[col], errors="coerce")
        temp = temp.dropna(subset=[col]).copy()

        if len(temp) == 0:
            continue

        campo = slugify(label)
        metricas_validas.append({"field": campo, "label": label})

        # A base executiva deveria ter uma linha por estratégia/frequência. Se houver duplicidade residual,
        # usa a média apenas dentro da mesma métrica final já selecionada, não entre blocos analíticos diferentes.
        temp = (
            temp.groupby(chaves, as_index=False)
            .agg(valor=(col, "mean"))
        )

        for _, row in temp.iterrows():
            key = (row["_estrategia_html"], row["_frequencia_html"])
            if key not in registros:
                registros[key] = {
                    "label": row["_estrategia_html"],
                    "estrategia": row["_estrategia_html"],
                    "categoria": row["_categoria_html"],
                    "frequencia": row["_frequencia_html"],
                }
            registros[key][campo] = float(row["valor"])

    if not registros:
        return pd.DataFrame(), []

    return pd.DataFrame(list(registros.values())), metricas_validas


def adicionar_metricas_wide_por_grupo_inferencia(df, metricas_grupo, coluna_valor_padrao, aceitar_total=True):
    if df is None or len(df) == 0:
        return pd.DataFrame(), []

    base = preparar_base_grafico_padrao(df).copy()
    base["_grupo_evidencia_html"] = base.apply(classificar_grupo_evidencia, axis=1)
    base = base.loc[base["_grupo_evidencia_html"].isin([m[0] for m in metricas_grupo])].copy()
    base["_frequencia_html"] = base["_frequencia_html"].apply(normalizar_frequencia_grafico)

    if not aceitar_total:
        base = base.loc[base["_frequencia_html"] != "Total"].copy()

    if len(base) == 0:
        return pd.DataFrame(), []

    col_valor = escolher_primeira_coluna_existente(base, [coluna_valor_padrao], numerica=True)
    if col_valor is None:
        return pd.DataFrame(), []

    base[col_valor] = pd.to_numeric(base[col_valor], errors="coerce")
    base = base.dropna(subset=[col_valor]).copy()

    if len(base) == 0:
        return pd.DataFrame(), []

    chaves = ["_estrategia_html", "_categoria_html", "_frequencia_html", "_grupo_evidencia_html"]
    base_agg = (
        base.groupby(chaves, as_index=False)
        .agg(valor=(col_valor, "mean"))
    )

    registros = {}
    metricas_validas = []

    for label, _ in metricas_grupo:
        campo = slugify(label)
        metricas_validas.append({"field": campo, "label": label})

    for _, row in base_agg.iterrows():
        key = (row["_estrategia_html"], row["_frequencia_html"])
        if key not in registros:
            registros[key] = {
                "label": row["_estrategia_html"],
                "estrategia": row["_estrategia_html"],
                "categoria": row["_categoria_html"],
                "frequencia": row["_frequencia_html"],
            }

        campo = slugify(row["_grupo_evidencia_html"])
        registros[key][campo] = float(row["valor"])

    if not registros:
        return pd.DataFrame(), []

    return pd.DataFrame(list(registros.values())), metricas_validas


def combinar_metricas_wide(*pares):
    bases = []
    metricas = []

    for base, mets in pares:
        if base is not None and len(base) > 0:
            bases.append(base)
        metricas.extend(mets or [])

    if not bases:
        return pd.DataFrame(), []

    base_final = bases[0].copy()
    for outra in bases[1:]:
        base_final = base_final.merge(
            outra,
            on=["label", "estrategia", "categoria", "frequencia"],
            how="outer",
        )

    metricas_unicas = []
    campos = set()
    for met in metricas:
        if met["field"] not in campos and met["field"] in base_final.columns:
            campos.add(met["field"])
            metricas_unicas.append(met)

    col_metricas = [m["field"] for m in metricas_unicas]
    if col_metricas:
        base_final = base_final.dropna(subset=col_metricas, how="all").copy()

    return base_final, metricas_unicas


def gerar_score_evidencia_estatistica():
    metricas_grupo = [
        ("vs Ibovespa", ["score_vs_ibovespa"]),
        ("vs CDI", ["score_vs_cdi"]),
        ("vs aleatórias", ["score_vs_aleatorias", "score_vs_aleatorias_controle", "score_vs_controles_aleatorios"]),
        ("vs mensais", ["score_vs_mensais", "score_vs_controles_mensais"]),
    ]

    metricas_totais = [
        ("score médio", ["score_evidencia_medio"]),
        ("score total", ["score_evidencia_total"]),
    ]

    base_grupos, mets_grupos = adicionar_metricas_wide_por_coluna(DF_RESUMO_EXECUTIVO, metricas_grupo)
    if len(base_grupos) == 0:
        base_grupos, mets_grupos = adicionar_metricas_wide_por_grupo_inferencia(
            DF_INFERENCIA,
            metricas_grupo,
            "score_evidencia_medio",
        )

    base_totais, mets_totais = adicionar_metricas_wide_por_coluna(DF_RESUMO_EXECUTIVO, metricas_totais)
    if len(base_totais) == 0:
        base_totais_medio, mets_medio = adicionar_metricas_wide_por_grupo_inferencia(
            DF_INFERENCIA,
            [("score médio", ["score_evidencia_medio"])],
            "score_evidencia_medio",
        )
        base_totais_total, mets_total = adicionar_metricas_wide_por_grupo_inferencia(
            DF_INFERENCIA,
            [("score total", ["score_evidencia_total"])],
            "score_evidencia_total",
        )
        base_totais, mets_totais = combinar_metricas_wide((base_totais_medio, mets_medio), (base_totais_total, mets_total))

    base_final, metricas_validas = combinar_metricas_wide((base_grupos, mets_grupos), (base_totais, mets_totais))

    if len(base_final) == 0 or not metricas_validas:
        print("   - Aviso: gráfico não gerado (Score de evidência estatística): métricas válidas não encontradas.")
        return None

    registros = serializar_registros(
        base_final,
        ["label", "estrategia", "categoria", "frequencia"] + [m["field"] for m in metricas_validas],
    )

    return salvar_dashboard_plotly_interativo(
        registros,
        DIR_GRAFICOS / "inferencia_score_evidencia.html",
        "Score de evidência estatística",
        "Score por estratégia, com seleção explícita do bloco de evidência. A comparação direta contra outra estratégia foi removida desta visão para preservar a simetria entre Capitulação e Euforia.",
        tipo_grafico="bar",
        metricas=metricas_validas,
        viewer="inferencia",
        catalogo=catalogo_graficos,
        label_field="label",
        use_frequency=True,
        use_strategy=False,
        use_category=False,
        top_n=20,
        y_percent=False,
        aggregate_bars=False,
    )


def gerar_percentis_bootstrap_inferencia():
    metricas_grupo = [
        ("vs Ibovespa", ["score_percentil_vs_ibovespa"]),
        ("vs CDI", ["score_percentil_vs_cdi"]),
        ("vs aleatórias", ["score_percentil_vs_aleatorias", "score_percentil_vs_controles_aleatorios"]),
        ("vs mensais", ["score_percentil_vs_mensais", "score_percentil_vs_controles_mensais"]),
    ]

    metricas_totais = [
        ("score percentil médio", ["score_percentil_medio"]),
        ("score percentil mediano", ["score_percentil_mediano"]),
        ("bootstrap: probabilidade de excesso positivo", ["prob_bootstrap_media_acima_zero_media", "prob_bootstrap_media_acima_zero"]),
    ]

    base_grupos, mets_grupos = adicionar_metricas_wide_por_coluna(DF_RESUMO_EXECUTIVO, metricas_grupo)
    if len(base_grupos) == 0:
        base_grupos, mets_grupos = adicionar_metricas_wide_por_grupo_inferencia(
            DF_INFERENCIA,
            metricas_grupo,
            "score_percentil_medio",
        )

    base_totais, mets_totais = adicionar_metricas_wide_por_coluna(DF_RESUMO_EXECUTIVO, metricas_totais)
    if len(base_totais) == 0:
        base_percentil_medio, mets_percentil_medio = adicionar_metricas_wide_por_grupo_inferencia(
            DF_INFERENCIA,
            [("score percentil médio", ["score_percentil_medio"])],
            "score_percentil_medio",
        )
        base_percentil_mediano, mets_percentil_mediano = adicionar_metricas_wide_por_grupo_inferencia(
            DF_INFERENCIA,
            [("score percentil mediano", ["score_percentil_mediano"])],
            "score_percentil_mediano",
        )
        base_bootstrap, mets_bootstrap = adicionar_metricas_wide_por_grupo_inferencia(
            DF_INFERENCIA,
            [("bootstrap: probabilidade de excesso positivo", ["prob_bootstrap_media_acima_zero_media", "prob_bootstrap_media_acima_zero"])],
            "prob_bootstrap_media_acima_zero_media" if DF_INFERENCIA is not None and "prob_bootstrap_media_acima_zero_media" in DF_INFERENCIA.columns else "prob_bootstrap_media_acima_zero",
        )
        base_totais, mets_totais = combinar_metricas_wide(
            (base_percentil_medio, mets_percentil_medio),
            (base_percentil_mediano, mets_percentil_mediano),
            (base_bootstrap, mets_bootstrap),
        )

    base_final, metricas_validas = combinar_metricas_wide((base_grupos, mets_grupos), (base_totais, mets_totais))

    if len(base_final) == 0 or not metricas_validas:
        print("   - Aviso: gráfico não gerado (Percentis empíricos e bootstrap): métricas válidas não encontradas.")
        return None

    registros = serializar_registros(
        base_final,
        ["label", "estrategia", "categoria", "frequencia"] + [m["field"] for m in metricas_validas],
    )

    return salvar_dashboard_plotly_interativo(
        registros,
        DIR_GRAFICOS / "inferencia_percentis_bootstrap.html",
        "Percentis empíricos e bootstrap",
        "Percentis e evidência bootstrap por estratégia, com seleção explícita do bloco analisado. A comparação direta contra outra estratégia foi removida desta visão.",
        tipo_grafico="bar",
        metricas=metricas_validas,
        viewer="inferencia",
        catalogo=catalogo_graficos,
        label_field="label",
        use_frequency=True,
        use_strategy=False,
        use_category=False,
        top_n=20,
        y_percent=False,
        aggregate_bars=False,
    )


# --------------------------------------------------------------------------------------------------
# Inferência estatística e robustez
# --------------------------------------------------------------------------------------------------

gerar_score_evidencia_estatistica()

gerar_percentis_bootstrap_inferencia()

gerar_barras_metricas(
    DF_ROBUSTEZ,
    "robustez_controles_sensibilidade.html",
    "Robustez contra controles e sensibilidades",
    "Leitura individual das métricas de robustez. Evita empilhamento para preservar interpretação direta.",
    metricas=[
        ("Aleatórias — retorno maior", "pct_replicas_retorno_acumulado_real_superou_controle"),
        ("Aleatórias — Sharpe maior", "pct_replicas_sharpe_real_superou_controle"),
        ("Aleatórias — Calmar maior", "pct_replicas_calmar_real_superou_controle"),
        ("Mensais — retorno maior", "pct_controles_retorno_acumulado_real_superou_controle"),
        ("Mensais — Sharpe maior", "pct_controles_sharpe_real_superou_controle"),
        ("Mensais — Calmar maior", "pct_controles_calmar_real_superou_controle"),
    ],
    viewer="inferencia",
    usar_frequencia=True,
    group_col=None,
    y_percent=True,
)

# --------------------------------------------------------------------------------------------------
# Composição, sinais e auditoria
# --------------------------------------------------------------------------------------------------

gerar_timeline_sinais()

gerar_metricas_compras_por_estrategia()

gerar_heatmap_ticker_mais_comprado_por_ano()

gerar_heatmap_top_tickers()

gerar_heatmap_concentracao(
    DF_TOP_EMPRESAS if DF_TOP_EMPRESAS is not None and len(DF_TOP_EMPRESAS) > 0 else DF_CONCENTRACAO_EMPRESA,
    "composicao_concentracao_empresa_heatmap.html",
    "Concentração por empresa",
    "Heatmap estratégia × empresa emissora. A cor representa valor investido ou a métrica selecionada; o texto resume valor/peso quando disponível.",
    label_candidates=["nome", "nome_empresa", "issuer_code"],
    viewer="composicao",
    top_n=30,
)

gerar_top_bar_from_df(
    DF_TOP_POSICOES if DF_TOP_POSICOES is not None and len(DF_TOP_POSICOES) > 0 else DF_POSICOES,
    "composicao_top_posicoes_historicas.html",
    "Top posições históricas",
    "Maiores posições históricas observadas nas carteiras ao longo do tempo.",
    label_candidates=["nome_empresa", "nome", "ticker", "issuer_code"],
    metric_candidates=["valor_posicao", "valor_posicao_total", "peso_posicao", "peso_medio_posicao", "participacao_posicao", "valor_ranking_posicao"],
    viewer="composicao",
    top_n=35,
    group_candidates=None,
)

gerar_heatmaps_concentracao_setorial()

gerar_barras_metricas(
    DF_AUDITORIA,
    "auditoria_resumo_estrategias.html",
    "Auditoria consolidada por estratégia",
    "Resumo de aportes, compras, conciliação e validações finais por estratégia. As carteiras aleatórias de Capitulação e Euforia aparecem como controles separados das estratégias reais.",
    metricas=[
        ("Aportes realizados", "n_aportes_realizados"),
        ("Compras efetivadas", "n_compras_efetivadas_total"),
        ("Valor total aportado", "valor_total_aportado_registrado"),
        ("Valor investido efetivo", "valor_total_investido_efetivo"),
        ("% capital aportado", "pct_capital_total_aportado"),
    ],
    viewer="auditoria",
    usar_frequencia=False,
    usar_estrategia=True,
    usar_categoria=True,
    top_n=60,
    sort_asc=False,
)


def deduplicar_catalogo_por_viewer_titulo(catalogo):
    saida = []
    vistos = set()
    for item in catalogo:
        chave = (
            str(item.get("viewer", "")).strip(),
            str(item.get("titulo", "")).strip(),
            str(item.get("caminho_relativo", "")).strip(),
        )
        if chave in vistos:
            continue
        vistos.add(chave)
        saida.append(item)
    return saida

catalogo_graficos = deduplicar_catalogo_por_viewer_titulo(catalogo_graficos)
catalogo_tabelas = deduplicar_catalogo_por_viewer_titulo(catalogo_tabelas)

print("OK")
print(f" - Gráficos HTML gerados..............: {format_int(len(catalogo_graficos))}")
print(f" - Pasta de gráficos..................: {DIR_GRAFICOS}")

print("\n[7/9] Escrevendo CSS, JS, manifestos do dashboard e index.html...")

meta_payload = {
    "hero": {
        "titulo": "Capitulação e Euforia no Mercado Brasileiro",
        "subtitulo": "Dashboard interativo com performance, risco, controles, inferência estatística, robustez, sinais, aportes, compras e composição.",
    },
    "grafico_catalogo": catalogo_graficos,
    "tabela_catalogo": catalogo_tabelas,
    "resumo_execucao": resumo_execucao,
}

arquivos_data_js = escrever_data_js_particionado(meta_payload, ARQUIVO_DATA_JS)

css_html = """
:root {
  --bg: #f5f7fb;
  --card: #ffffff;
  --ink: #0f172a;
  --muted: #475569;
  --line: #dbe3ee;
  --brand: #172554;
  --brand-2: #1d4ed8;
  --brand-3: #0f766e;
  --warn: #b45309;
  --soft-blue: #eff6ff;
  --soft-green: #ecfdf5;
  --soft-orange: #fff7ed;
  --shadow: 0 10px 30px rgba(15, 23, 42, 0.08);
  --radius: 18px;
}

* { box-sizing: border-box; }

html {
  scroll-behavior: smooth;
  scroll-padding-top: 132px;
}

body {
  margin: 0;
  background: linear-gradient(180deg, #f5f7fb 0%, #eef3fa 100%);
  color: var(--ink);
  font-family: Inter, Arial, Helvetica, sans-serif;
  line-height: 1.6;
}

a {
  color: var(--brand-2);
  text-decoration: none;
}

a:hover {
  text-decoration: underline;
}

code {
  background: #eef2ff;
  color: #1e1b4b;
  padding: 2px 5px;
  border-radius: 6px;
}

.container {
  width: min(1400px, calc(100vw - 32px));
  margin: 0 auto;
  padding: 24px 0 80px 0;
}

.topbar {
  position: sticky;
  top: 0;
  z-index: 70;
  backdrop-filter: blur(14px);
  background: linear-gradient(180deg, rgba(15, 23, 42, 0.97) 0%, rgba(23, 37, 84, 0.93) 100%);
  border-bottom: 1px solid rgba(255, 255, 255, 0.10);
  box-shadow: 0 10px 28px rgba(15, 23, 42, 0.20);
}

.topbar-inner {
  width: min(1400px, calc(100vw - 32px));
  margin: 0 auto;
  padding: 14px 0 16px 0;
}

.topbar-title {
  color: #f8fafc;
  font-size: 1.05rem;
  font-weight: 800;
  margin-bottom: 10px;
}

.nav-links {
  display: flex;
  flex-wrap: wrap;
  gap: 8px;
}

.nav-link {
  color: rgba(255, 255, 255, 0.82);
  border: 1px solid rgba(255, 255, 255, 0.12);
  padding: 8px 12px;
  border-radius: 999px;
  font-size: 0.88rem;
}

.nav-link.active,
.nav-link:hover {
  color: #ffffff;
  background: rgba(255, 255, 255, 0.12);
  text-decoration: none;
}

.hero {
  background: linear-gradient(135deg, #020617 0%, #172554 55%, #0f766e 100%);
  color: #f8fafc;
  border-radius: 24px;
  padding: 32px 30px 28px 30px;
  box-shadow: 0 18px 36px rgba(15, 23, 42, 0.18);
  margin-top: 18px;
}

.hero h1 {
  margin: 0 0 8px 0;
  font-size: clamp(1.8rem, 2.8vw, 3rem);
  line-height: 1.08;
}

.hero p {
  margin: 0;
  color: rgba(255,255,255,0.92);
  max-width: 1050px;
}

.badges {
  margin-top: 16px;
  display: flex;
  flex-wrap: wrap;
  gap: 10px;
}

.badge {
  display: inline-flex;
  align-items: center;
  gap: 8px;
  padding: 8px 12px;
  border-radius: 999px;
  background: rgba(255,255,255,0.10);
  border: 1px solid rgba(255,255,255,0.13);
  color: #f8fafc;
  font-size: 0.9rem;
}

.section {
  margin-top: 26px;
  background: var(--card);
  border-radius: var(--radius);
  padding: 24px 22px;
  box-shadow: var(--shadow);
  border: 1px solid rgba(15, 23, 42, 0.05);
}

.section-header {
  margin-bottom: 16px;
}

.section-eyebrow {
  color: var(--brand-2);
  font-size: 0.83rem;
  font-weight: 800;
  letter-spacing: 0.08em;
  text-transform: uppercase;
  margin-bottom: 6px;
}

.section h2 {
  margin: 0;
  font-size: 1.55rem;
}

.section p {
  color: var(--muted);
}

.kpi-grid {
  display: grid;
  grid-template-columns: repeat(6, minmax(0, 1fr));
  gap: 14px;
}

.kpi-card {
  background: linear-gradient(180deg, #ffffff 0%, #f8fbff 100%);
  border: 1px solid #dbe6f3;
  border-radius: 18px;
  padding: 16px;
  min-height: 122px;
}

.kpi-title {
  font-size: 0.87rem;
  color: #475569;
  font-weight: 700;
  margin-bottom: 10px;
}

.kpi-value {
  font-size: 1.8rem;
  font-weight: 900;
  color: #0f172a;
  line-height: 1.1;
  margin-bottom: 10px;
}

.kpi-detail {
  font-size: 0.85rem;
  color: #64748b;
}

.two-col {
  display: grid;
  grid-template-columns: 1.08fr 0.92fr;
  gap: 18px;
}

.method-cards,
.info-grid,
.takeaways-grid {
  display: grid;
  gap: 14px;
}

.method-cards {
  grid-template-columns: repeat(2, minmax(0, 1fr));
  margin-top: 0;
}

.method-card,
.info-card,
.takeaway-card {
  border: 1px solid #dbe6f3;
  border-radius: 18px;
  padding: 16px;
  background: #fbfdff;
}

.info-grid {
  grid-template-columns: repeat(2, minmax(0, 1fr));
}

.info-card h4,
.method-card h4 {
  margin: 0 0 8px 0;
  font-size: 1rem;
}

.info-card p,
.method-card p {
  margin: 0;
}

.takeaways-grid {
  grid-template-columns: repeat(2, minmax(0, 1fr));
  margin-top: 16px;
}

.takeaway-card {
  display: grid;
  grid-template-columns: 56px 1fr;
  gap: 14px;
  align-items: start;
}

.takeaway-index {
  width: 56px;
  height: 56px;
  border-radius: 16px;
  background: linear-gradient(135deg, #172554 0%, #0f766e 100%);
  color: #ffffff;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 1rem;
  font-weight: 900;
}

.takeaway-card h3 {
  margin: 0 0 6px 0;
  font-size: 1rem;
}

.takeaway-card p {
  margin: 0;
}

.viewer-grid {
  display: grid;
  grid-template-columns: 1fr;
  gap: 18px;
}

.viewer-card {
  border: 1px solid #dbe6f3;
  border-radius: 18px;
  padding: 16px;
  background: #fbfdff;
}

.viewer-card h3 {
  margin: 0 0 12px 0;
}

.viewer-toolbar {
  display: grid;
  grid-template-columns: minmax(280px, 1fr) auto;
  gap: 12px;
  align-items: end;
  margin-bottom: 12px;
}

.filter-box {
  min-width: 320px;
  flex: 1 1 320px;
}

.filter-box label {
  display: block;
  margin-bottom: 6px;
  font-size: 0.88rem;
  font-weight: 700;
  color: #475569;
}

.filter-box select {
  width: 100%;
  border: 1px solid #cbd5e1;
  border-radius: 12px;
  background: #fff;
  padding: 10px 12px;
  font-size: 0.95rem;
  color: #0f172a;
}

.actions {
  display: flex;
  gap: 10px;
  align-items: center;
  justify-content: flex-end;
}

.btn-link-open {
  display: inline-flex;
  align-items: center;
  justify-content: center;
  padding: 10px 14px;
  border-radius: 12px;
  border: 1px solid #cbd5e1;
  background: #fff;
  color: #0f172a;
  font-weight: 700;
}

.viewer-frame {
  width: 100%;
  min-height: 720px;
  border: 1px solid #dbe3ee;
  border-radius: 14px;
  background: #fff;
}

.viewer-caption {
  margin-top: 10px;
  color: #64748b;
  font-size: 0.92rem;
}

.note-box {
  border-left: 4px solid #1d4ed8;
  background: #eff6ff;
  border-radius: 12px;
  padding: 14px 16px;
  color: #1e293b;
}

.conclusao ul {
  margin: 0 0 18px 0;
  padding-left: 20px;
}

.conclusao li {
  margin-bottom: 8px;
}

.footer {
  color: #64748b;
  text-align: center;
  font-size: 0.92rem;
  padding: 18px 0 8px 0;
}

@media (max-width: 1200px) {
  .kpi-grid {
    grid-template-columns: repeat(3, minmax(0, 1fr));
  }
}

@media (max-width: 980px) {
  .two-col,
  .method-cards,
  .info-grid,
  .takeaways-grid {
    grid-template-columns: 1fr;
  }

  .kpi-grid {
    grid-template-columns: repeat(2, minmax(0, 1fr));
  }

  .viewer-frame {
    min-height: 620px;
  }
}

@media (max-width: 640px) {
  .container,
  .topbar-inner {
    width: min(100vw - 20px, 100%);
  }

  .hero {
    padding: 22px 18px;
  }

  .section {
    padding: 18px 16px;
  }

  .kpi-grid {
    grid-template-columns: 1fr;
  }

  .viewer-toolbar {
    grid-template-columns: 1fr;
  }

  .filter-box {
    min-width: 100%;
  }

  .viewer-frame {
    min-height: 560px;
  }
}
"""
salvar_texto(ARQUIVO_CSS, css_html)
validar_tamanho_arquivo_gerado(ARQUIVO_CSS, "CSS final")

js_html = """
const META = window.HTML_FINAL_META || {};
const GRAFICOS = META.grafico_catalogo || [];
const TABELAS = META.tabela_catalogo || [];

function byViewer(items, viewer) {
  return items.filter(x => x.viewer === viewer);
}

function initIframeViewer(viewer, items, tipo) {
  const select = document.getElementById(`${tipo}_select_${viewer}`);
  const frame = document.getElementById(`${tipo}_frame_${viewer}`);
  const openBtn = document.getElementById(`${tipo}_open_${viewer}`);
  const caption = document.getElementById(`${tipo}_caption_${viewer}`);

  if (!select || !frame || !openBtn || !caption) return;

  select.innerHTML = "";

  if (!items.length) {
    const opt = document.createElement("option");
    opt.value = "";
    opt.textContent = "Nenhum item disponível";
    select.appendChild(opt);
    frame.src = "about:blank";
    openBtn.href = "#";
    caption.textContent = "Nenhum item disponível para esta seção.";
    return;
  }

  items.forEach((item, idx) => {
    const opt = document.createElement("option");
    opt.value = idx;
    opt.textContent = item.titulo;
    select.appendChild(opt);
  });

  const sync = () => {
    const item = items[Number(select.value) || 0];
    const caminho = item.caminho_relativo || item.caminho_html_rel || "";

    frame.src = caminho || "about:blank";
    openBtn.href = caminho || "#";
    caption.textContent = item.subtitulo || item.titulo || "";
  };

  select.addEventListener("change", sync);
  select.value = 0;
  sync();
}

function initActiveSectionNav() {
  const sections = Array.from(document.querySelectorAll("section[id]"));
  const links = Array.from(document.querySelectorAll(".nav-link"));

  if (!sections.length || !links.length) return;

  const onScroll = () => {
    let currentId = sections[0].id;
    const offset = window.scrollY + 185;

    sections.forEach(sec => {
      if (sec.offsetTop <= offset) currentId = sec.id;
    });

    links.forEach(link => {
      link.classList.toggle("active", link.getAttribute("href") === `#${currentId}`);
    });
  };

  document.addEventListener("scroll", onScroll, { passive: true });
  onScroll();
}

document.addEventListener("DOMContentLoaded", () => {
  initIframeViewer("performance", byViewer(GRAFICOS, "performance"), "grafico");
  initIframeViewer("risco", byViewer(GRAFICOS, "risco"), "grafico");
  initIframeViewer("capitulacao_euforia", byViewer(GRAFICOS, "capitulacao_euforia"), "grafico");
  initIframeViewer("inferencia", byViewer(GRAFICOS, "inferencia"), "grafico");
  initIframeViewer("composicao", byViewer(GRAFICOS, "composicao"), "grafico");
  initIframeViewer("auditoria", byViewer(GRAFICOS, "auditoria"), "grafico");

  initIframeViewer("resumo", byViewer(TABELAS, "resumo"), "tabela");
  initIframeViewer("performance_tab", byViewer(TABELAS, "performance"), "tabela");
  initIframeViewer("risco_tab", byViewer(TABELAS, "risco"), "tabela");
  initIframeViewer("inferencia_tab", byViewer(TABELAS, "inferencia"), "tabela");
  initIframeViewer("composicao_tab", byViewer(TABELAS, "composicao"), "tabela");

  initActiveSectionNav();
});
"""
salvar_texto(ARQUIVO_JS, js_html)
validar_tamanho_arquivo_gerado(ARQUIVO_JS, "JS final")

data_scripts_html = "\n  ".join([f'<script src="{caminho_relativo(c)}"></script>' for c in arquivos_data_js])

index_html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Capitulação e Euforia no Mercado Brasileiro — Relatório Final</title>
  <meta name="description" content="Dashboard interativo do projeto de Capitulação e Euforia no mercado brasileiro, com performance, risco, controles, inferência estatística, robustez, sinais, aportes, compras e composição.">
  <link rel="stylesheet" href="{caminho_relativo(ARQUIVO_CSS)}">
</head>
<body>
  <div class="topbar">
    <div class="topbar-inner">
      <div class="topbar-title">Capitulação e Euforia no Mercado Brasileiro — Relatório Final</div>
      <div class="nav-links">
        <a class="nav-link" href="#resumo">Resumo</a>
        <a class="nav-link" href="#introducao">Introdução</a>
        <a class="nav-link" href="#metodologia">Metodologia</a>
        <a class="nav-link" href="#resultados">Resultados</a>
        <a class="nav-link" href="#performance">Performance</a>
        <a class="nav-link" href="#risco">Risco</a>
        <a class="nav-link" href="#capitulacao_euforia">Capitulação vs Euforia</a>
        <a class="nav-link" href="#inferencia">Inferência</a>
        <a class="nav-link" href="#composicao">Composição</a>
        <a class="nav-link" href="#tabelas">Tabelas</a>
        <a class="nav-link" href="#conclusao">Conclusão</a>
      </div>
    </div>
  </div>

  <main class="container">
    <section class="hero" id="resumo">
      <h1>Capitulação e Euforia no Mercado Brasileiro</h1>
      <p>Relatório interativo com estratégias de extremos de mercado, benchmarks, controles, inferência estatística, robustez e auditoria econômica das carteiras.</p>
      {badges_html}
    </section>

    <section class="section">
      <div class="section-header">
        <div class="section-eyebrow">Visão executiva</div>
        <h2>Resumo em números</h2>
      </div>
      <div class="kpi-grid">{kpis_html}</div>
    </section>

    <section id="introducao" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Introdução</div>
        <h2>Estratégias de extremos sob a mesma régua metodológica</h2>
      </div>
      {conteudo_introducao}
    </section>

    <section id="metodologia" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Metodologia</div>
        <h2>Como os sinais foram transformados em carteiras comparáveis</h2>
      </div>
      <div class="two-col">
        <div>
          {conteudo_metodologia}
        </div>
        <div class="method-cards">
          {cards_metodologia}
        </div>
      </div>
    </section>

    <section id="resultados" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Principais resultados</div>
        <h2>Performance, risco e robustez precisam ser lidos juntos</h2>
      </div>
      {conteudo_resultados}
      <div class="takeaways-grid">{takeaways_html}</div>
    </section>

    <section id="performance" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Performance</div>
        <h2>Curvas, retornos e comparação contra referências</h2>
      </div>
      <p>
        Esta seção concentra curvas patrimoniais, curvas indexadas, retornos acumulados, distribuição de retornos e comparação
        contra benchmarks e controles. Use os seletores para navegar entre os gráficos da análise.
      </p>
      <div class="viewer-grid">
        {bloco_viewer_grafico('performance', 'Gráficos de performance')}
      </div>
    </section>

    <section id="risco" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Risco</div>
        <h2>Drawdown, volatilidade e eficiência</h2>
      </div>
      <p>
        A análise de risco separa o retorno final da qualidade da trajetória. O objetivo é mostrar se a estratégia foi eficiente
        ou se o resultado veio acompanhado de perdas máximas e volatilidade incompatíveis.
      </p>
      <div class="viewer-grid">
        {bloco_viewer_grafico('risco', 'Gráficos de risco')}
      </div>
    </section>

    <section id="capitulacao_euforia" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Capitulação vs Euforia</div>
        <h2>Comparação direta entre famílias de estratégia</h2>
      </div>
      <p>
        Este bloco isola a comparação entre sinais de capitulação e sinais de euforia, preservando a análise de performance,
        risco e eficiência sob uma estrutura comparável.
      </p>
      <div class="viewer-grid">
        {bloco_viewer_grafico('capitulacao_euforia', 'Gráficos — Capitulação vs Euforia')}
      </div>
    </section>

    <section id="inferencia" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Inferência estatística</div>
        <h2>Significância, controles e sensibilidade</h2>
      </div>
      <div class="two-col">
        <div>
          {conteudo_estatistica}
        </div>
        <div class="info-grid">
          {cards_estatistica}
        </div>
      </div>
      <div class="viewer-grid" style="margin-top: 18px;">
        {bloco_viewer_grafico('inferencia', 'Gráficos de inferência e robustez')}
      </div>
    </section>

    <section id="composicao" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Composição e auditoria econômica</div>
        <h2>Sinais, aportes, compras, posições e concentração</h2>
      </div>
      {conteudo_composicao}
      <div class="viewer-grid">
        {bloco_viewer_grafico('composicao', 'Gráficos de composição e concentração')}
        {bloco_viewer_grafico('auditoria', 'Gráficos de auditoria')}
      </div>
    </section>

    <section id="tabelas" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Tabelas navegáveis</div>
        <h2>Bases analíticas do dashboard</h2>
      </div>
      <p>
        As tabelas abaixo permitem consultar métricas, rankings, testes, sinais, compras, posições e auditorias em detalhes.
        Use a busca e a ordenação para explorar os resultados por estratégia, período, ativo, setor ou métrica.
      </p>
      <div class="viewer-grid">
        {bloco_viewer_tabela('resumo', 'Resumo, métricas principais e comparações consolidadas')}
        {bloco_viewer_tabela('performance_tab', 'Tabelas de performance')}
        {bloco_viewer_tabela('risco_tab', 'Tabelas de risco')}
        {bloco_viewer_tabela('inferencia_tab', 'Tabelas de inferência e robustez')}
        {bloco_viewer_tabela('composicao_tab', 'Tabelas de composição, sinais, compras e auditoria')}
      </div>
    </section>

    <section id="conclusao" class="section">
      <div class="section-header">
        <div class="section-eyebrow">Conclusão</div>
        <h2>Leitura final do projeto</h2>
      </div>
      {conteudo_conclusao}
    </section>

    <div class="footer">
      Projeto de Capitulação e Euforia no Mercado Brasileiro — relatório interativo de pesquisa quantitativa.
    </div>
  </main>

  {data_scripts_html}
  <script src="{caminho_relativo(ARQUIVO_JS)}"></script>
</body>
</html>"""

salvar_texto(ARQUIVO_INDEX, index_html)
validar_tamanho_arquivo_gerado(ARQUIVO_INDEX, "index.html")

print("OK")
print(f" - CSS...............................: {ARQUIVO_CSS}")
print(f" - JS app............................: {ARQUIVO_JS}")
print(f" - JS data...........................: {', '.join([str(c) for c in arquivos_data_js])}")
print(f" - Index final.......................: {ARQUIVO_INDEX}")

print("\n[8/9] Validando assets finais gerados pela célula...")

arquivos_gerados = []
for pasta in [DIR_ASSETS_CSS, DIR_ASSETS_JS, DIR_TABELAS, DIR_GRAFICOS, DIR_RELATORIOS]:
    arquivos_gerados.extend([p for p in pasta.rglob("*") if p.is_file()])
arquivos_gerados.append(ARQUIVO_INDEX)

arquivos_acima_limite = []
for caminho in arquivos_gerados:
    tamanho = caminho.stat().st_size
    if tamanho > MAX_BYTES_GITHUB:
        arquivos_acima_limite.append((caminho, tamanho))

if arquivos_acima_limite:
    msg = "\n".join([f"{caminho} - {tamanho / 1024 / 1024:.2f} MB" for caminho, tamanho in arquivos_acima_limite])
    raise ValueError("Arquivos gerados acima de 100 MB:\n" + msg)

maiores_arquivos = sorted(
    [(p, p.stat().st_size) for p in arquivos_gerados if p.exists()],
    key=lambda x: x[1],
    reverse=True,
)[:10]

print("OK")
print(" - Nenhum arquivo gerado nesta célula ultrapassou 100 MB.")
print(" - Maiores arquivos gerados:")
for caminho, tamanho in maiores_arquivos:
    print(f"   {caminho} | {tamanho / 1024 / 1024:.2f} MB")

print("\n[9/9] Resumo da execução do HTML final...")

print("=" * 110)
print("SEÇÃO HTML FINAL CONCLUÍDA")
print("=" * 110)
print()
print(f"Index final..........................: {ARQUIVO_INDEX}")
print(f"Pasta html_final.....................: {DIR_HTML_FINAL}")
print(f"Tabelas HTML.........................: {format_int(len(catalogo_tabelas))}")
print(f"Gráficos HTML........................: {format_int(len(catalogo_graficos))}")
print(f"Período final........................: {periodo_inicio} a {periodo_fim}")
print(f"Estratégias/séries...................: {format_int(n_estrategias)}")
print(f"Sinais...............................: {format_int(n_sinais)}")
print(f"Aportes..............................: {format_int(n_aportes)}")
print(f"Compras por ticker...................: {format_int(n_compras)}")
print(f"Tickers comprados....................: {format_int(n_tickers_comprados)}")
print(f"Posições históricas..................: {format_int(n_posicoes)}")
print(f"Evidências robustas/significativas...: {format_int(n_evidencias_robustas)}")
print(f"Resumo da execução...................: {ARQUIVO_RESUMO_JSON}")

INICIANDO SEÇÃO HTML FINAL

[1/9] Resolvendo diretórios e validando estrutura consolidada do projeto...
OK
 - Raiz do projeto....................: C:\Users\mht-1\OneDrive\Documentos\Projetos\Dados\_projeto_em_andamento\capitulacao_euforia_mercado_brasileiro
 - Pasta resultados...................: resultados
 - Pasta html_final...................: html_final
 - Index final........................: index.html

[2/9] Definindo funções auxiliares de leitura, formatação, gráficos e tabelas...
OK

[3/9] Carregando bases finais do projeto...
OK
 - Arquivos parquet carregados........: 58
 - Resumo executivo...................: (2, 24)
 - Métricas principais................: (78, 16)
 - Performance........................: (93588, 22)
 - Risco..............................: (78, 16)
 - Composição/auditoria...............: (2501, 183)

[4/9] Derivando métricas, leituras executivas e textos do relatório...
OK
 - Período final......................: 2011-06-29 a 2025-12-30
 - Estratégias/séries...

# Conclusão

Este projeto não foi construído para responder apenas se **Capitulação** ou **Euforia** teve o maior retorno no fim da amostra. Ele foi desenhado para testar, sob a mesma régua metodológica, se comprar momentos de estresse extremo ou momentos de força do mercado produz trajetórias diferentes quando as duas abordagens são confrontadas com benchmarks, controles aleatórios, controles mensais, métricas de risco, inferência estatística, robustez e composição das carteiras.

A base final cobre o período de **2011-06-29** a **2025-12-30**, com **8 estratégias ou séries finais**, **46 sinais consolidados**, **276 aportes**, **20.845 registros de compras por ticker**, **239 tickers distintos comprados** e **526.253 registros de posições históricas**. Esses números são importantes porque mostram que a comparação não ficou restrita a uma curva agregada: o projeto preservou a trilha operacional completa entre sinal, aporte, compra, posição e concentração.

Na leitura de performance, a estratégia com melhor métrica de retorno no consolidado foi **Capitulação**, com **291,36%**. Na leitura de eficiência ajustada ao risco, a melhor métrica também apareceu em **Capitulação**, com valor de **0,0474**. Essa combinação sugere que, no consolidado principal do relatório, Capitulação não apareceu apenas como uma estratégia com maior crescimento patrimonial, mas também como a família com melhor leitura de eficiência dentro das métricas destacadas no HTML.

Mesmo assim, a conclusão não deve ser reduzida a “Capitulação venceu”. O projeto foi estruturado justamente para evitar esse tipo de leitura simplista. Retorno acumulado, retorno anualizado, Sharpe, Sortino, Calmar, volatilidade e drawdown não respondem à mesma pergunta. O retorno mede crescimento patrimonial; as métricas de eficiência avaliam a qualidade desse crescimento; o drawdown mede a profundidade das perdas; e os controles verificam se o resultado é realmente específico do sinal ou se poderia ter surgido de calendários alternativos.

A comparação com benchmarks, controles aleatórios e controles mensais é o principal filtro contra conclusões frágeis. Uma estratégia pode superar o Ibovespa ou o CDI e, ainda assim, não se destacar de forma convincente contra controles aleatórios ou mensais. Por isso, a leitura economicamente correta depende da convergência entre desempenho, risco, controles e inferência estatística, e não apenas de uma curva patrimonial mais alta.

A camada de inferência estatística reforça essa cautela. O painel consolidado preserva **0 evidências robustas/significativas** e **0 evidências sensíveis ou frágeis** identificadas nos testes destacados no HTML. Isso indica que, na versão final apresentada, a interpretação estatística deve ser tratada com disciplina: há sinais econômicos e diferenças de trajetória, mas a força formal da evidência precisa ser lida com cuidado, principalmente quando a comparação envolve múltiplas frequências, benchmarks e controles.

A composição das carteiras fecha a interpretação econômica. O projeto expõe **276 aportes**, **20.845 compras por ticker** e **526.253 posições históricas**, permitindo investigar se o resultado veio de uma carteira ampla ou de concentração involuntária em poucos nomes, setores, subsetores ou segmentos. Essa camada é essencial em estratégias de extremos de mercado, porque momentos de capitulação ou euforia podem alterar fortemente quais empresas entram na carteira e quais blocos setoriais dominam o resultado.

**📊 Interpretação dos resultados**

A primeira leitura do projeto é que estratégias de extremos de mercado precisam ser avaliadas como uma arquitetura completa, não como uma aposta isolada em datas de compra. A diferença entre comprar capitulação e comprar euforia só é economicamente relevante se persistir depois de controlar risco, drawdown, benchmarks, controles aleatórios e controles mensais.

O bloco de performance responde quem acumulou mais patrimônio. Nesse recorte, **Capitulação** apareceu como destaque consolidado, com **291,36%**. O bloco de risco responde se esse patrimônio foi construído com trajetória mais eficiente ou mais instável; nesse ponto, **Capitulação** também apareceu com a melhor métrica de eficiência destacada, em **0,0474**.

O bloco de controles responde se o sinal superou alternativas de calendário e aleatoriedade. Essa etapa é central porque impede atribuir ao sinal aquilo que poderia ser apenas resultado de datas alternativas de compra. O bloco de inferência responde se a evidência é estatisticamente defensável. E o bloco de composição responde de onde veio o resultado: quais tickers, empresas, setores, subsetores e segmentos concentraram as compras e posições.

A leitura estatística deve ser tratada com disciplina. Scores próximos de neutralidade, p-valores elevados ou baixa confirmação por bootstrap não anulam necessariamente o resultado econômico, mas reduzem a força da conclusão. Por outro lado, quando performance, risco, controles, bootstrap e testes convergem, a evidência deixa de ser apenas uma boa história visual e passa a ser uma hipótese quantitativamente mais robusta.

Neste projeto, a mensagem mais madura é que **Capitulação apresentou melhor destaque econômico consolidado**, mas essa leitura deve ser combinada com a análise dos controles e da inferência. O relatório não deve ser lido como uma recomendação mecânica de compra em todo evento de estresse, mas como uma investigação quantitativa sobre como sinais de medo e força se comportaram dentro de uma estrutura comparável de backtest.

**🧠 Principais aprendizados**

- A comparação entre **Capitulação** e **Euforia** precisa combinar retorno, risco, drawdown, eficiência, controles e inferência; uma única métrica não é suficiente.

- O projeto analisou **46 sinais**, **276 aportes** e **20.845 compras por ticker**, preservando rastreabilidade operacional entre sinal, aporte, compra e posição.

- A análise cobre um período longo, de **2011-06-29** a **2025-12-30**, permitindo avaliar diferentes regimes de mercado dentro da mesma estrutura metodológica.

- A estratégia com melhor métrica de retorno no consolidado foi **Capitulação**, com **291,36%**, e a melhor leitura de eficiência destacada também apareceu em **Capitulação**, com **0,0474**.

- O painel identificou **239 tickers distintos comprados**, o que reforça a importância de avaliar concentração por empresa, setor, subsetor e segmento antes de interpretar o resultado como diversificado.

- A existência de **526.253 registros de posições históricas** torna possível auditar a trajetória das carteiras ao longo do tempo, e não apenas o ponto final do backtest.

- Benchmarks, controles aleatórios e controles mensais funcionam como contrafactuais essenciais para reduzir o risco de confundir sinal com calendário ou sorte amostral.

- A inferência estatística deve ser lida com cautela: o painel final destacou **0 evidências robustas/significativas** e **0 evidências sensíveis ou frágeis**, o que exige separar resultado econômico observado de força estatística formal.

- A concentração por ticker, empresa, setor, subsetor e segmento é indispensável para entender se o resultado veio de uma carteira ampla ou de poucos blocos dominantes.

- A evidência final deve ser lida por convergência: desempenho econômico, robustez estatística, comparação com controles e coerência da composição.

**📘 Síntese**

Em termos práticos, o estudo entrega uma leitura auditável de duas formas opostas de operar extremos de mercado no Brasil. A **Capitulação** representa a tentativa de comprar estresse; a **Euforia** representa a tentativa de comprar força. O resultado final não deve ser reduzido a um campeão absoluto, porque cada família precisa ser julgada por retorno, risco, estabilidade, controles e composição.

A síntese é que o projeto permite identificar quando uma estratégia de extremo de mercado parece economicamente promissora, quando essa promessa sobrevive aos controles e quando ela se apoia em evidência estatística mais frágil. Nesta versão final, **Capitulação** apareceu como destaque econômico consolidado, com melhor retorno e melhor métrica de eficiência destacados no painel, mas a interpretação completa depende da leitura conjunta dos controles, da inferência e da composição das carteiras.

Esse é o principal valor do relatório: ele transforma uma comparação intuitiva entre medo e euforia em uma avaliação quantitativa, auditável e interpretável por camadas.